# LayerNorm backward with deterministic weight gradients

The forward pass $y_{ij} = \hat x_{ij} w_j + b_j$, with $\hat x_{ij} = (x_{ij} - \mu_i)\, r_i$, saved the per-row mean $\mu_i$ and inverse standard deviation $r_i$. Given the upstream gradient $\partial L/\partial y = dy$, the backward pass needs three things:

$$dx_{ij} = r_i\Bigl(g_{ij} - \tfrac1N\textstyle\sum_k g_{ik} - \hat x_{ij}\,\tfrac1N\sum_k g_{ik}\hat x_{ik}\Bigr), \quad g_{ij} = w_j\, dy_{ij}$$

$$dw_j = \sum_i dy_{ij}\,\hat x_{ij}, \qquad db_j = \sum_i dy_{ij}$$

`dx` is a per-row computation, but `dw` and `db` are sums **over rows**, so different programs contribute to the same output column. That is the interesting part.

Write `layer_norm_bwd(x, w, mean, rstd, dy)`.

## Inputs
- `x`, `dy`: contiguous `float32` CUDA tensors `(M, N)`, $M \ge 1$, $1 \le N \le 8192$.
- `w`: `float32` `(N,)`. `mean`, `rstd`: `float32` `(M,)`, exactly as saved by the forward pass (use them; do not recompute).

## Output
A tuple `(dx, dw, db)` of `float32` tensors with shapes `(M, N)`, `(N,)`, `(N,)`.

## Requirements
- Launch `G = min(M, 64)` programs. Program `pid` handles rows `pid, pid + G, pid + 2G, ...`, keeping `dw` and `db` accumulators for its rows in registers (one block of `BLOCK = next_power_of_2(N)` columns).
- Each program writes its partial sums to row `pid` of `(G, N)` buffers; the host sums those over the first axis. No atomics: the result must be identical run to run.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def layer_norm_bwd(x, w, mean, rstd, dy):
    """x, dy: (M, N), w: (N,), mean, rstd: (M,), float32 CUDA. Return (dx, dw, db)."""
    raise NotImplementedError


@triton.jit
def _ln_bwd_kernel(x_ptr, dy_ptr, w_ptr, mean_ptr, rstd_ptr, dx_ptr, dw_part_ptr, db_part_ptr,
                   M, N, stride_row, G, BLOCK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0, 3.0, 4.0]], "dtype": "float32"}, {"__nd__": [1.0, 1.0, 1.0, 1.0], "dtype": "float32"}, {"__nd__": [2.5], "dtype": "float32"}, {"__nd__": [0.8944230079650879], "dtype": "float32"}, {"__nd__": [[1.0, 0.0, 0.0, 0.0]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [[0.2683306634426117, -0.3577679395675659, -0.08944355696439743, 0.17888084053993225]], "dtype": "float32"}, {"__nd__": [-1.3416345119476318, 0.0, 0.0, 0.0], "dtype": "float32"}, {"__nd__": [1.0, 0.0, 0.0, 0.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.0, 2.0], [1.0, 1.0]], "dtype": "float32"}, {"__nd__": [2.0, -1.0], "dtype": "float32"}, {"__nd__": [1.0, 1.0], "dtype": "float32"}, {"__nd__": [0.9999949932098389, 316.227783203125], "dtype": "float32"}, {"__nd__": [[1.0, 1.0], [0.5, -0.5]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [[1.5020257706055418e-05, -1.5020257706055418e-05], [79.05694580078125, -79.05694580078125]], "dtype": "float32"}, {"__nd__": [-0.9999949932098389, 0.9999949932098389], "dtype": "float32"}, {"__nd__": [1.5, 0.5], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.3455841839313507], [0.8216181397438049], [0.3304370641708374]], "dtype": "float32"}, {"__nd__": [-1.3031572103500366], "dtype": "float32"}, {"__nd__": [0.3455841839313507, 0.8216181397438049, 0.3304370641708374], "dtype": "float32"}, {"__nd__": [316.2277526855469, 316.2277526855469, 316.2277526855469], "dtype": "float32"}, {"__nd__": [[0.9053558707237244], [0.4463745653629303], [-0.5369532108306885]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [[0.0], [0.0], [0.0]], "dtype": "float32"}, {"__nd__": [0.0], "dtype": "float32"}, {"__nd__": [0.8147772550582886], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.18905338644981384, -0.5227484703063965, -0.41306355595588684, -2.44146728515625, 1.7997074127197266, 1.1441658735275269, -0.32542282342910767, 0.7738065719604492, 0.28121066093444824, -0.5538228154182434, 0.9775674343109131, -0.310556560754776, -0.32882389426231384, -0.7921467423439026, 0.45495808124542236, -0.09919805079698563, 0.5452887415885925, -0.6071857213973999, 0.12682785093784332, -0.892274022102356, 0.8414649963378906, 0.18803508579730988, 0.3305709958076477, 0.41050392389297485, -1.0107574462890625, 0.7831810116767883, 2.0567028522491455, -1.6384425163269043, -1.7294114828109741, -1.5048314332962036, 0.8414589166641235, 0.1287156641483307, 1.0783424377441406, 0.7224308848381042, 0.21057181060314178, 0.28403815627098083, -0.16976049542427063], [0.8684602379798889, -1.129715919494629, -0.4218588173389435, 0.2429388463497162, 1.80142080783844, -0.7644641399383545, -1.079060435295105, -0.5632871985435486, 0.9692721962928772, -0.23500551283359528, 1.3243470191955566, -1.8725295066833496, 1.1285231113433838, 1.034866213798523, -1.4186656475067139, 0.1535818874835968, 1.2157576084136963, 0.08791904151439667, 0.9997014403343201, 2.374886989593506, 0.27393224835395813, -0.28038230538368225, -0.7710521817207336, 0.6480646133422852, -0.19673006236553192, -0.17874637246131897, -0.10525977611541748, 0.6498696804046631, -1.0663396120071411, -1.529876470565796, -2.4338767528533936, 1.1986708641052246, 0.07379335910081863, 1.5101194381713867, -0.008956298232078552, -0.7421554327011108, 0.4779292941093445], [-0.07658841460943222, -1.2541866302490234, -0.885066568851471, 1.7667793035507202, 0.35435059666633606, 0.41638654470443726, -0.2765517234802246, -0.689719557762146, 0.8916558027267456, -0.10462645441293716, -0.7591081261634827, -0.13412852585315704, -0.9056060314178467, 0.19004583358764648, 1.129227638244629, -0.8360851407051086, 1.428544044494629, -0.6676199436187744, 0.15341216325759888, -0.836386501789093, -0.22223517298698425, 0.04740441218018532, -0.4346849322319031, -0.7028878927230835, -0.6778861880302429, -0.8211599588394165, -1.570080041885376, -0.26296961307525635, 0.4011582136154175, 0.9084019064903259, 0.6471027135848999, 2.45733642578125, 0.31868162751197815, -0.4563339054584503, 1.871634602546692, -1.0472180843353271, 0.9684776067733765], [-0.9551447629928589, 0.3541116714477539, -1.968396782875061, 0.899273693561554, -0.15824761986732483, -0.9676814675331116, 1.6784189939498901, 0.7653546929359436, 0.04580783471465111, -0.7454434037208557, -0.04336259141564369, -0.16392876207828522, 0.7247765064239502, 0.7980751991271973, -0.6677563786506653, -0.5493705868721008, -0.5320653319358826, -1.349317193031311, -0.5915749073028564, -0.09240806847810745, 0.690951943397522, 1.3203794956207275, -0.8083458542823792, 0.5509765148162842, -0.4444568157196045, 2.0751876831054688, -0.04872354492545128, 0.5021269917488098, -0.9361621737480164, -0.8106894493103027, 0.2016953080892563, -0.3840544521808624, 0.35328859090805054, -1.561824083328247, 0.6620049476623535, -0.8994803428649902, 1.7224234342575073], [-0.2884672284126282, 1.0972650051116943, -1.3949832916259766, 0.42894625663757324, -0.8880296349525452, -0.4868890643119812, 0.024596381932497025, -0.3170749545097351, 0.2642556130886078, 0.74765545129776, 0.652782142162323, -0.08748148381710052, -1.3634305000305176, -0.8046461343765259, -0.1957562118768692, -2.121825933456421, 0.7520805597305298, -0.23963819444179535, -0.2513192892074585, 0.9483762383460999, 0.6679105162620544, 0.22100096940994263, -1.0013335943222046, 0.31042158603668213, 0.3461775481700897, -0.884773313999176, 1.0433118343353271, 0.5078871846199036, -1.969624400138855, 0.1219494491815567, -1.2646980285644531, 1.136315107345581, 0.8070279359817505, -1.103590726852417, -0.8848726749420166, 0.10185328125953674, -0.23881082236766815]], "dtype": "float32"}, {"__nd__": [1.5252031087875366, 0.7409738302230835, -0.28242725133895874, 0.5773959159851074, -2.0322792530059814, 0.30986469984054565, 0.866407036781311, -0.17895251512527466, -0.7102141976356506, 0.5658334493637085, 1.958614706993103, 0.20486892759799957, -1.1206777095794678, 1.4458048343658447, -1.8785474300384521, 0.503204882144928, -0.5819546580314636, 0.9305984377861023, -0.8494743704795837, 1.323847770690918, 0.3002730906009674, -1.726840615272522, -1.9738130569458008, -0.3160955011844635, 1.2942116260528564, 0.12703002989292145, 0.06712070107460022, -0.23590382933616638, 0.7379001975059509, 0.3773267865180969, 0.2803458869457245, -0.984722912311554, 1.1878533363342285, 1.363080382347107, -1.3013535737991333, 2.30149507522583, 0.722506582736969], "dtype": "float32"}, {"__nd__": [0.022397011518478394, 0.06043493002653122, 0.0089045949280262, -0.03604273125529289, -0.1515522301197052], "dtype": "float32"}, {"__nd__": [1.0599523782730103, 0.9410534501075745, 1.0843757390975952, 1.0849535465240479, 1.1763653755187988], "dtype": "float32"}, {"__nd__": [[-0.48800599575042725, -1.9043558835983276, 0.6568464040756226, 0.5575113892555237, -0.5956318378448486, -1.7905532121658325, 1.1244767904281616, -0.588318943977356, -0.4717450737953186, 2.8456451892852783, 2.12797212600708, 0.9537367224693298, -0.4554210305213928, 0.6494612693786621, -2.2814698219299316, 0.10653270035982132, -0.7598182559013367, -1.2856051921844482, -1.5828044414520264, 0.5251415967941284, 0.08930037170648575, 0.1597236692905426, 0.8423760533332825, -0.8460837602615356, -1.2355844974517822, -1.7763653993606567, -0.3351965844631195, 0.7601562142372131, -0.1494235247373581, 0.6411281824111938, -1.1810537576675415, -0.4691213369369507, -0.7659366726875305, -1.248377799987793, 1.8594735860824585, 0.8521407842636108, -0.7618005275726318], [-0.7660171985626221, -0.057196855545043945, -1.4420539140701294, 0.4358271658420563, -1.294238805770874, -0.37043297290802, -1.0248225927352905, -0.47984829545021057, -1.3311572074890137, -1.5728219747543335, -0.966140866279602, -0.6800098419189453, -0.2948894202709198, 0.06898362934589386, 1.5268961191177368, 0.8122518062591553, -0.2528991997241974, 0.7259183526039124, -0.6888972520828247, 1.4484913349151611, 0.5494769811630249, -1.3558393716812134, 0.7553855180740356, 0.8160923719406128, -2.6855952739715576, -0.7480949759483337, -0.47382253408432007, -1.9285122156143188, -1.0671775341033936, -0.9483581781387329, 0.5070958137512207, 0.30046316981315613, 2.626173734664917, -0.39425939321517944, 1.1532310247421265, 0.07884647697210312, 0.2709257900714874], [-1.1596298217773438, -1.9108997583389282, -0.7671481966972351, 0.7621685862541199, 1.5828056335449219, 0.6039910316467285, 0.49828091263771057, -0.838766872882843, 1.5342938899993896, 1.343806266784668, -0.5308324694633484, -0.046370670199394226, -0.9960519075393677, 1.4672068357467651, 0.406354159116745, -0.0950646698474884, 0.40689998865127563, 0.4406222999095917, -0.41149669885635376, -1.241565227508545, -1.2837454080581665, -1.7383664846420288, 0.46125301718711853, 2.2185418605804443, -1.6990692615509033, 0.13828745484352112, 0.41098394989967346, -0.9846256971359253, 0.20442579686641693, -1.0969369411468506, 1.2004021406173706, -0.5942490696907043, 0.10056038200855255, 0.6842415928840637, -0.42981937527656555, -1.7912307977676392, -0.5312556624412537], [0.7629952430725098, -0.08453265577554703, 0.22978918254375458, -1.0804017782211304, 0.14322291314601898, 0.7281669974327087, 0.03644802048802376, 1.9080145359039307, -0.20779357850551605, -1.041350245475769, -1.6176592111587524, 1.152005910873413, -0.05901824310421944, 1.0116218328475952, 0.4576689302921295, 1.6250842809677124, 1.2661073207855225, -0.3775545656681061, 0.2738848924636841, 1.085371971130371, -1.2360751628875732, 1.4508960247039795, 1.2119933366775513, 0.9829258918762207, -0.20093216001987457, 0.3021410405635834, -1.117239236831665, 0.057954173535108566, -0.587379515171051, 0.08656396716833115, -0.4946262538433075, 0.2838035225868225, 1.1793321371078491, -1.309603214263916, 0.925390899181366, -1.9918125867843628, -2.1610333919525146], [-0.11936867982149124, 0.006729381624609232, 0.4428008496761322, -1.6473852396011353, -0.059938643127679825, 0.12652470171451569, 1.2552050352096558, -0.10383958369493484, 0.5289917588233948, -0.9774255752563477, -0.18755462765693665, -0.4455353319644928, 0.7033043503761292, -0.9888193011283875, -0.834322452545166, 0.5759881734848022, 1.7894366979599, 0.38270333409309387, 0.3035602867603302, -0.34614238142967224, 1.0884252786636353, 0.5399169325828552, 1.7331926822662354, 0.3307115137577057, -0.9486769437789917, -0.569941520690918, -0.9874045252799988, -1.2508208751678467, 1.2637174129486084, 0.8073369264602661, -1.2386443614959717, 1.7813206911087036, 0.24456073343753815, -0.9568920135498047, 0.3862219452857971, -0.0691942572593689, -0.7726547718048096]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [[-0.9874082803726196, -1.5891659259796143, -0.3063015639781952, 0.5307125449180603, 0.8470233678817749, -0.9274427890777588, 0.9100686311721802, -0.17313149571418762, 0.14305688440799713, 1.6177870035171509, 4.102973461151123, 0.08231762051582336, 0.4188854694366455, 0.9415332674980164, 4.305099964141846, -0.09913995862007141, 0.21766997873783112, -1.349144697189331, 1.2358617782592773, 0.6978991031646729, -0.2662816345691681, -0.4906800091266632, -1.9817252159118652, 0.05233754590153694, -1.716490387916565, -0.5252872705459595, -0.4977913498878479, -0.11900712549686432, -0.03238508477807045, 0.307778924703598, -0.6456568241119385, 0.3000722825527191, -1.2940080165863037, -2.080803632736206, -2.7665581703186035, 1.8662896156311035, -0.7289581298828125], [-1.3293951749801636, 0.40880247950553894, 0.5915518999099731, 0.2193174958229065, 1.928426742553711, 0.2166212797164917, -0.4040919840335846, 0.3371241092681885, 0.6255080699920654, -0.6926703453063965, -2.165513515472412, 0.569858193397522, -0.007259209640324116, -0.1925894320011139, -2.152449131011963, 0.3974894881248474, -0.20938090980052948, 0.6708717346191406, 0.27620047330856323, 1.0630037784576416, 0.12724770605564117, 2.363541841506958, -1.076224446296692, -0.39783915877342224, -3.1390202045440674, 0.03628969192504883, 0.07083246111869812, 0.27243098616600037, -0.3138900101184845, 0.24784021079540253, 1.0253853797912598, -0.6205105781555176, 2.975576162338257, -0.9535810351371765, -1.344241976737976, 0.48783084750175476, 0.08690623193979263], [-1.651476502418518, -0.9100345373153687, 0.7477980852127075, 0.1817692071199417, -3.3530385494232178, 0.3191143870353699, 0.7955169677734375, 0.616075336933136, -1.2103153467178345, 1.0995007753372192, -0.652960479259491, 0.273664265871048, 1.7295516729354858, 2.4854371547698975, -0.9288723468780518, 0.4460504353046417, -0.4491177201271057, 0.8912152051925659, 0.5753726363182068, -1.2843114137649536, -0.10717801749706268, 3.48380184173584, -0.6116691827774048, -0.30311647057533264, -1.9347892999649048, 0.5124233365058899, 0.7515601515769958, 0.5751123428344727, 0.284382700920105, -0.4826270341873169, 0.41076725721359253, 0.12862618267536163, 0.27547791600227356, 1.393546223640442, 0.27914685010910034, -3.9080710411071777, -0.46833324432373047], [1.8319091796875, 0.3660874366760254, 0.6036063432693481, -0.29909902811050415, 0.1711883842945099, 0.8154210448265076, 0.33147257566452026, 0.0211025420576334, 0.626015841960907, -0.09163133800029755, -2.962421417236328, 0.7436317801475525, 0.46750545501708984, 1.9750339984893799, -0.39316216111183167, 1.4146186113357544, -0.27380287647247314, 0.22885173559188843, 0.2793349027633667, 2.039116144180298, -0.0034503876231610775, -2.3841090202331543, -2.0413222312927246, 0.07661066949367523, 0.23441645503044128, 0.2978564500808716, 0.39430755376815796, 0.40391838550567627, 0.09711407124996185, 0.5898360013961792, 0.299346923828125, 0.2071065604686737, 1.9540129899978638, -1.3047353029251099, -0.9043349623680115, -4.410013675689697, -1.4013397693634033], [0.14528541266918182, 0.40724626183509827, 0.1788642853498459, -0.737790048122406, 0.4846124053001404, 0.3995731770992279, 1.6482468843460083, 0.38045036792755127, -0.06577901542186737, -0.2597973644733429, -0.04420153424143791, 0.25816258788108826, -0.6002507209777832, -1.3379390239715576, 2.2059953212738037, 0.6449471116065979, -0.8340955376625061, 0.7798879742622375, 0.05723468214273453, -0.1421814113855362, 0.7728561758995056, -0.7219150066375732, -3.686455011367798, 0.2546021640300751, -1.0656712055206299, 0.2562469244003296, 0.32178428769111633, 0.730663537979126, 1.405551552772522, 0.7302290797233582, -0.07857044786214828, -1.6609090566635132, 0.7343364953994751, -1.199562430381775, -0.24984212219715118, 0.18392862379550934, -0.29574525356292725]], "dtype": "float32"}, {"__nd__": [-1.302795648574829, 3.755847454071045, -0.03446241468191147, -2.14961838722229, -2.6166720390319824, -2.3604986667633057, 0.8580363392829895, 2.1277015209198, 0.4410720765590668, -1.6985992193222046, 1.2827666997909546, 0.7141361236572266, -0.19045642018318176, 1.465907335281372, -2.947979211807251, -2.0954978466033936, 1.1511170864105225, 1.051750898361206, -1.0493056774139404, 3.2694993019104004, 0.583953320980072, 2.762165069580078, -3.2859151363372803, -0.8034090399742126, 2.8019487857818604, -0.20486629009246826, -2.725104808807373, -3.0541067123413086, -0.6331121325492859, -0.5015159845352173, 0.10946815460920334, 1.2827764749526978, -0.01654868759214878, 1.4302306175231934, -0.20494630932807922, 4.073431491851807, -4.334812164306641], "dtype": "float32"}, {"__nd__": [-1.770026445388794, -3.950255870819092, -0.8797656893730164, -0.9722799062728882, -0.22378073632717133, -0.7023034691810608, 1.8895881175994873, -0.10275916010141373, 0.052589789032936096, 0.5978536605834961, -1.1742150783538818, 0.9338268041610718, -1.1020762920379639, 2.2084543704986572, -0.7248730659484863, 3.024792194366455, 2.4497265815734863, -0.11391577124595642, -2.105753183364868, 1.471297264099121, -0.792617917060852, -0.9436691999435425, 5.004200458526611, 3.502187967300415, -6.769858360290527, -2.653973340988159, -2.502678871154785, -3.345848321914673, -0.33583736419677734, -0.5102660655975342, -1.2068264484405518, 1.3022170066833496, 3.384690284729004, -3.22489070892334, 3.894498109817505, -2.921250343322754, -3.9558186531066895], "dtype": "float32"}]}}, {"a": [{"__nd__": [[2.040919065475464, -2.5556650161743164, 0.4180988371372223, -0.5677695870399475, -0.4526492953300476, -0.21559716761112213, -2.019986152648926, -0.2319323718547821, -0.8652130961418152, 3.3229994773864746, 0.22578661143779755, -0.3526307940483093, -0.28128743171691895, -0.6680463552474976, -1.0551505088806152, -0.39080098271369934, 0.4819453954696655, -0.23855361342430115, 0.9577587246894836, -0.19980213046073914, 0.02425956539809704, 1.5458208322525024, 0.545105516910553, -0.5052287578582764, -0.18283897638320923, 0.5405251383781433, 1.935088038444519, -0.2696203291416168, -0.24355867505073547, 1.0023136138916016, -0.8864599466323853, -0.2917202413082123, 0.8825389742851257, 0.5803500413894653, 0.09151670336723328, 0.6701043844223022, -2.82816219329834, 1.0213068723678589, -0.9596447348594666, -1.6686198711395264, 0.27644574642181396, 0.7005448937416077, -0.44476744532585144, -1.0764058828353882, 0.026124833151698112, -0.05274730920791626, 1.4055981636047363, 0.7474079728126526, 0.19381564855575562, 1.1116331815719604, -0.2055230438709259, -0.9258995652198792, 0.5840582847595215, 0.5825384259223938, -0.21482890844345093, -0.7828086018562317, 0.22915390133857727, -2.493894338607788, 0.6901247501373291, 0.49136826395988464, -1.638857126235962, 0.06135350838303566, -0.964099645614624, 0.7572210431098938, -2.0341672897338867, -0.914494514465332, 0.7095800042152405, 1.1564010381698608, -2.158005475997925, -0.4980398416519165, 0.3280200958251953, -0.6092161536216736, 1.5906401872634888, -1.1912267208099365, 0.3545319437980652, -1.0484055280685425, 1.4059629440307617, -0.02165122888982296, -0.3722505569458008, -1.7181849479675293, 1.6818255186080933, 0.7527785897254944, 0.7535638213157654, 1.1378812789916992, 0.34922656416893005, -0.6392465829849243, -0.800241231918335, -0.8001999855041504, 1.370072364807129, -1.4603811502456665, -0.5963695049285889, -0.32124391198158264, 0.2246190309524536, 0.5753493905067444, -1.2490969896316528, -1.7300134897232056, -0.004414232447743416, 1.2135637998580933, 0.7570580840110779, 0.2156507819890976, -0.3171556293964386, 0.2932336926460266, -0.2433350831270218, 0.8172065615653992, -0.7944473624229431, 0.13423994183540344, -0.11078013479709625, 0.5433593988418579, 0.22463852167129517, 2.550034523010254, 1.4986547231674194, 1.4967371225357056, -2.039503812789917, -0.34031662344932556, -0.6086106300354004, 0.5327215790748596, -2.279026508331299, 1.1744986772537231, 1.0669833421707153, -1.3020708560943604, -0.978548526763916, -0.8011720180511475, 0.04329590126872063, 0.6409710645675659, 2.0478861331939697, -0.19744543731212616, 0.7675025463104248, 0.1554178148508072, 1.7599263191223145, 0.7421578764915466, 1.368550419807434, -1.077675223350525, -0.19224071502685547, -0.813772439956665, 1.5049474239349365, 0.6576399207115173, -0.30514442920684814, -0.4524678885936737, 0.4846648871898651, -0.7014955282211304, -0.9305889010429382, 0.48127448558807373, 2.46313214302063, -0.24613355100154877, -0.5558657646179199, -1.171156883239746, -1.3350110054016113, 0.5249832272529602, 0.850803017616272, 0.00917472131550312, 0.3325759768486023, 0.11591657996177673, 0.13865484297275543, -1.5261590480804443, -0.4581182599067688, 0.11147930473089218, -0.783166766166687, -0.4764297306537628, -0.8191201686859131, -0.33349665999412537, 0.853108286857605, -0.4065803587436676, -0.15387068688869476, 0.8137183785438538, 0.6447702050209045, 1.6952075958251953, -2.090485095977783, 0.8568589687347412, -0.4822840988636017, 0.13469181954860687, 0.8377193808555603, 1.0832531452178955, 1.0393507480621338, 0.15510669350624084, 1.609662652015686, -0.2829742133617401, -0.14098191261291504, 0.7993512153625488, -0.5513724088668823, 2.1609063148498535, 1.0192065238952637, 2.1755752563476562, -0.02658918686211109, -0.3830885589122772, 0.16704800724983215, 0.7345765233039856, -0.5874276757240295, 0.37970173358917236, -0.016804279759526253, 1.615681767463684, -0.6627017855644226, 1.0461617708206177, -0.6438635587692261, -0.960634171962738, -0.7102994322776794, -1.1901859045028687, 0.14635035395622253, 1.0312608480453491, 0.1643148809671402, 0.6243250966072083, 1.6321742534637451, 0.27002644538879395, 0.1951623558998108, -0.2750614583492279, -1.6082422733306885, 0.7597389221191406, -1.756460428237915, 0.6526764631271362, -0.01432033535093069, 1.1267814636230469, -0.06777025759220123, -0.8232001662254333, 0.35791027545928955, -0.5602361559867859, -0.18095682561397552, 0.04187078773975372, -0.13454970717430115, -0.18880227208137512, -0.832444965839386, -0.18913377821445465, -2.138340711593628, -0.15733209252357483, -1.1980559825897217, 1.120263695716858, 1.2699267864227295, -1.9510257244110107, 0.14491678774356842, -0.12636391818523407, -1.0467156171798706, 0.5316531658172607, -0.4616895318031311, -1.767599105834961, -0.266676664352417, -0.14825311303138733, 0.1064310148358345, -1.23123300075531, 0.6156809329986572, 0.7354570627212524, -1.145827293395996, -0.6588555574417114, -0.08033731579780579, -0.5659253001213074, 1.7437546253204346, 0.2083721160888672, -1.0105820894241333, -0.7881089448928833, -0.057470258325338364, 2.295778751373291, -0.178264319896698, 0.12747642397880554, 0.5140453577041626, -0.04012593626976013, 2.28066086769104, -0.5315249562263489, 0.7442031502723694, 0.16043797135353088, 0.6211963891983032, -1.1766430139541626, 1.7681126594543457, -0.12152750790119171, 0.03416326269507408, -0.9349762797355652, -0.7296952605247498, 0.5642394423484802, 0.9864999055862427, 0.7532547116279602, 1.208970308303833, 0.7144889235496521, 0.02846183255314827, 0.8365259766578674, 0.593603253364563, -0.1005752682685852, 0.7260727286338806, 1.285697340965271, 0.23457132279872894, -0.3562051057815552, 0.7187113761901855, 1.900721788406372, -0.21056205034255981, -0.0923096239566803, -0.13639752566814423, 1.2230174541473389, -1.837198257446289, 0.36584538221359253, 1.1922639608383179, -0.813383162021637, 1.4889774322509766, 0.5353090167045593, -0.549035906791687, 0.2041398137807846, -1.5344921350479126, -0.5655710101127625, 1.8269038200378418, -0.8938040733337402, 1.8445717096328735, -0.08157359063625336, 0.994662880897522, 0.04389702156186104, -2.2414815425872803, -0.5726513266563416, 0.18985359370708466, -1.1509497165679932, -1.3474106788635254, 0.36618584394454956, -0.6494150161743164, -1.6958272457122803, -0.6841151714324951, 0.8607375621795654, -0.47374606132507324, 0.9347725510597229, 1.5272728204727173, 0.032515399158000946, -1.1296833753585815, -1.3015650510787964, -0.026253854855895042, 0.8810556530952454, 0.2690153419971466, 0.5472571849822998, -0.44334179162979126, 0.3004227578639984, -0.47130876779556274, -0.32560989260673523, -1.2922464609146118, -1.572213053703308, -0.48195669054985046, -1.1075993776321411, -0.9587457180023193, 0.6710474491119385, -0.10708092153072357, 2.914125919342041, 0.9263420701026917, -0.6331875920295715, 0.7780073285102844, 0.35680389404296875, -0.6856811046600342, 1.1025949716567993, -0.7753286361694336, -3.0583813190460205, 0.7940647602081299, -0.61725914478302, 1.4860374927520752, -0.6496722102165222, -1.181354284286499, 1.548461675643921, -1.075498342514038, 0.20339369773864746, 1.449727177619934, 0.1144973412156105, -0.08058491349220276, 0.10242343693971634, 0.8390536308288574, -0.8068121671676636, 0.6106410026550293, 0.5770754218101501, 2.332711935043335, -0.3365255296230316, -0.9148298501968384, 0.7356870174407959, -0.3971572518348694, 0.28911617398262024, 0.13257652521133423, 1.1683939695358276, -0.6568374633789062, -0.09227953106164932, -2.552112579345703, -0.890113890171051, 1.5363388061523438, -0.6815059185028076, -0.7014768123626709, -1.217907190322876, 0.5798342823982239, 0.1779901683330536, -1.1868668794631958, 0.6674503684043884, 0.2601058781147003, 0.7540333867073059, -0.6414794921875, 1.738008737564087, 0.40986761450767517, -0.11688951402902603, -0.5188342928886414, -0.24482737481594086, 0.8587920069694519, -0.2689667046070099, 1.078143835067749, 0.5806761980056763, 1.3464951515197754, 0.8141041994094849, -0.06293926388025284, -1.9093399047851562, 0.29857057332992554, 0.789076030254364, 0.16289228200912476, 0.5564317107200623, -0.471620112657547, -0.38047757744789124, -0.9423890709877014, 1.3619004487991333, 0.15839426219463348, -0.4924860894680023, -1.3318525552749634, -0.15330727398395538, 1.4426312446594238, 0.4619216322898865, -0.7245215773582458, 1.4294949769973755, 1.4285101890563965, 0.9807761907577515, 0.7888168692588806, 1.1556518077850342, 0.5432590246200562, 0.775000810623169, -0.7800253629684448, 1.4995578527450562, 0.23118019104003906, 2.024211883544922, -2.042248010635376, 0.7054088711738586, 0.9814615249633789, -0.3429326117038727, 0.45244431495666504, 0.7483264207839966, 0.5773814916610718, -0.5396592617034912, -0.14193783700466156, 1.019039511680603, 1.2773200273513794, 0.09897684305906296, 0.07703247666358948, 0.2004963606595993, 1.0376652479171753, -1.0540385246276855, -1.3290657997131348, 0.12431496381759644, -1.1065287590026855, -0.5871094465255737, 0.08629573881626129, 0.4825049340724945, -1.072091817855835, 0.8016054630279541, -1.7737146615982056, 0.5454160571098328, -1.577731728553772, -1.2749234437942505, 0.6422219276428223, -0.40636223554611206, -0.7955582737922668, -0.4447767734527588, 0.4650326073169708, -0.05404679477214813, -0.029382774606347084, 1.0547764301300049, -0.7163906097412109, -0.4719336926937103, -0.5224300026893616, 1.1689221858978271, 0.9914056658744812, 1.2870765924453735, -0.03375060856342316, -0.15680254995822906, 0.8527730107307434, -0.39441168308258057, 0.8306663632392883, 0.7242927551269531, -0.6983180046081543, -0.08319466561079025, -0.11925099045038223, 1.7333403825759888, 2.5878233909606934, -0.09992724657058716, 0.01406467892229557, -2.2815845012664795, 0.21153229475021362, -0.007032972294837236, -1.0728437900543213, 0.7855942249298096, -1.5657538175582886, -0.2148529589176178, 0.13885676860809326, -1.2489999532699585, 0.8567407131195068, 1.0555617809295654, -1.3864208459854126, -1.4078385829925537, 0.6022594571113586, -0.030497003346681595, -1.2114031314849854, -0.29522672295570374, -0.10751668363809586, -0.14303892850875854, 0.8274886608123779, 0.38419824838638306, 1.7694637775421143, 1.37436044216156, 0.6784737706184387, 1.07707941532135, 1.323819637298584, 0.7059948444366455, -1.225742220878601, 0.8594171404838562, 0.07227258384227753, 1.618811011314392, 0.09347418695688248, 0.5259924530982971, 0.32963991165161133, 1.6812255382537842, 0.17718231678009033, -0.26953989267349243, 0.30909180641174316, -0.43950825929641724, 0.16203923523426056, -0.007472030818462372, 0.27162256836891174, 1.9232840538024902, -0.40569058060646057, 0.03697320818901062, -0.19949175417423248, 0.6726115345954895, -1.1815745830535889, -0.6927294731140137, -0.03948960080742836, 0.5461478233337402, -0.04628121107816696, -0.5002862811088562, -0.8668274283409119, 0.5117526054382324, 1.442198634147644, 0.3228754997253418, 0.188135027885437, 0.44387301802635193, -0.9426529407501221, -0.31167489290237427, -0.267926424741745, 0.29524046182632446, -0.032025691121816635, -1.2164000272750854, 1.125229001045227, 1.6924084424972534, -0.263197124004364, 0.39884087443351746, 0.2671615779399872, 0.7846636176109314, -1.479783058166504, 1.6130096912384033, -0.46785232424736023, 1.2642261981964111, 0.08067114651203156, -0.3895556330680847, -0.14045269787311554, 0.8971729278564453, -0.8632761240005493, -0.37517791986465454, -0.48218023777008057, -1.7154483795166016, 1.3004252910614014, 0.5788888335227966, -0.021315021440386772, 1.1636626720428467, -0.9564298391342163, -0.07344981282949448, 1.72540283203125, 0.9796664118766785, -2.2254505157470703, -3.3320813179016113, -1.037903904914856, 1.0612833499908447, -0.4724457859992981, -0.21774406731128693, 0.6082714200019836, -0.8852022886276245, 0.21287789940834045, 0.7920982241630554, 0.4670132100582123, 0.3959454596042633, -0.6020461320877075, 0.5142871737480164, -0.42763006687164307, 1.3049468994140625, 2.5700926780700684, -0.7399058938026428, -2.17450213432312, -0.42543989419937134, 0.376845121383667, 1.0501313209533691, 1.0790154933929443, -0.7839513421058655, -0.7932353615760803, 1.2673009634017944, -1.0923876762390137, 1.0902312994003296, -0.22688792645931244, 0.7664036154747009, 0.7061647176742554, 0.776652455329895, 0.8204267024993896, -0.9893909692764282, 0.29752111434936523, 0.013982411473989487, 0.3084080219268799, 1.2332483530044556, -0.2711082398891449, -0.7119704484939575, -0.7281154990196228, -0.3108183741569519, 0.7526105642318726, -0.8148156404495239, 1.1592578887939453, 0.2904514670372009, -0.22611092031002045, -1.331650972366333, 0.24493318796157837, 0.6914693117141724, -1.7657657861709595, 1.1919596195220947, 1.6477470397949219, 0.33850228786468506, 0.4917809069156647, 2.4136950969696045, 1.2318906784057617, -0.8977571725845337, -0.01664518564939499, -0.156931072473526, -0.15087679028511047, -2.3642678260803223, 0.7570048570632935, -0.6367360949516296, 0.1636514663696289, -0.13613934814929962, 0.8710953593254089, -0.29580870270729065, -0.7954280376434326, 0.5324224829673767, 1.4048856496810913, -0.319810688495636, 1.3699480295181274, 0.26218315958976746, -0.5327295064926147, 1.5251578092575073, -0.818626344203949, 1.0140482187271118, -0.4981859624385834, 0.45021864771842957, -1.2341482639312744, -0.837592601776123, 1.0650522708892822, 0.18811684846878052, 0.0382930152118206, -0.4398082494735718, 0.4464806616306305, -1.3972692489624023, -1.1199352741241455, -0.15779036283493042, -0.8342810273170471, -0.7005937695503235, -0.07150325924158096, -0.13752132654190063, -0.22250670194625854, 1.6652755737304688, 1.0110758543014526, 1.2297345399856567, -0.028162000700831413, -0.5046842694282532, -1.4538991451263428, -0.34813740849494934, -2.0859556198120117, -0.583741307258606, 0.7642389535903931, 1.621530294418335, 0.9656069278717041, -0.8616004586219788, 0.49261853098869324, -0.7891640067100525, 0.8229737877845764, -0.11136118322610855, 0.8582001328468323, 2.5728955268859863, 0.9225243330001831, 2.3324499130249023, 0.17744231224060059, -0.3297177255153656, 1.3337043523788452, 1.7243565320968628, -0.5550146102905273, 0.07663196325302124, -1.4345295429229736, -0.2000458836555481, 0.8883322477340698, -0.0863848403096199, -0.5692201852798462, -1.4072399139404297, 0.36825451254844666, 0.7975406646728516, 0.5965006947517395, 0.2553156912326813, -1.274065375328064, 1.7618086338043213, -0.056536365300416946, -0.1447964459657669, 0.7329285740852356, -2.1631863117218018, -0.06446723639965057, -0.13678960502147675, 0.37228313088417053, 1.3471598625183105, 0.4722001254558563, -1.832098364830017, -0.3390767574310303, 0.20553170144557953, 0.14716723561286926, -1.2812628746032715, -1.6637585163116455, 0.5490108728408813, 0.34353509545326233, -0.021301068365573883, -1.7254220247268677, 1.7740503549575806, -2.900233745574951, -0.3640972971916199, 2.091151475906372, -0.1574258655309677, 0.2320702224969864, -1.179146409034729, 1.3586400747299194, -1.2926229238510132, 0.2544191777706146, 0.1349056512117386, -1.0507245063781738, 0.8869794011116028, -1.415958046913147, -0.6834845542907715, -0.028702203184366226, -0.41249769926071167, 0.4678066670894623, 0.9240548610687256, -0.2937737703323364, -0.5414659380912781, -0.5169782042503357, -0.36559394001960754, 0.809648871421814, -0.2175850123167038, 0.4076395332813263, -0.09115277975797653, -0.8444651961326599, 0.8637197613716125, -1.0380412340164185, 0.006197796668857336, 1.4870238304138184, -1.093106985092163, -0.05518852174282074, -1.5847810506820679, -0.5094162821769714, -1.49459969997406, -0.36158716678619385, -0.5382974743843079, 0.13197161257266998, 1.5843008756637573, 0.28298452496528625, 0.658174455165863, -0.050701938569545746, -0.24452361464500427, -0.8339818120002747, -1.8008568286895752, 0.06862036883831024, 1.629123568534851, 0.47629988193511963, -0.5339889526367188, 1.6494061946868896, -0.4326235055923462], [-0.6188397407531738, -1.2126944065093994, -0.9620509147644043, -0.2463628649711609, 1.9990206956863403, -0.20617620646953583, -1.0315163135528564, -0.6054186224937439, -0.22286176681518555, -1.711751937866211, 0.12662842869758606, 0.7006685137748718, 0.32081225514411926, 0.6007326245307922, -1.289547085762024, -1.698591947555542, 1.2855745553970337, 1.0943313837051392, -1.5402644872665405, 0.2063816636800766, 0.025527870282530785, 0.5448862910270691, 0.2085862159729004, -0.5021318793296814, 0.19525663554668427, 0.15704688429832458, -0.04862762987613678, -1.8524186611175537, -2.9952011108398438, -0.7901887893676758, -1.634771466255188, 0.2525620460510254, 0.7160966992378235, 0.26311543583869934, -0.3695279061794281, 0.0704808458685875, 1.1673409938812256, -0.7288080453872681, 0.9222947359085083, 0.7076163291931152, 0.26168426871299744, 1.4519301652908325, -0.717818558216095, 0.9425060153007507, -0.9289515614509583, -0.7670315504074097, -1.0597110986709595, 0.4458315968513489, -0.9397600293159485, 0.05215989798307419, -1.1367119550704956, 1.0662575960159302, 1.8361735343933105, 0.2260766476392746, 1.6433653831481934, 0.560663640499115, 2.1693100929260254, 0.0997142642736435, -1.5704236030578613, 1.4905840158462524, -0.12636713683605194, -1.0358450412750244, -0.6562658548355103, -1.6263316869735718, -0.1677018105983734, -1.0712637901306152, 1.4841805696487427, 0.6644920110702515, -1.6604059934616089, -1.509372353553772, -0.5669757723808289, -1.9250410795211792, -0.10980580747127533, -0.04344815015792847, -0.7728063464164734, -0.5464943647384644, 0.49885284900665283, 1.4216846227645874, 1.4956731796264648, 0.03430883213877678, 0.8558490872383118, -0.7383553981781006, 0.41195377707481384, -0.9596719741821289, 0.7295733690261841, -1.238401174545288, -0.9650830626487732, -0.00834464468061924, 0.5569891929626465, -1.4381744861602783, -0.5366385579109192, 0.13515105843544006, 0.2290463000535965, -0.838799238204956, 0.09688255935907364, -0.2672370374202728, 1.4038292169570923, 0.556898295879364, -1.324095368385315, -2.201557159423828, 0.09988179802894592, 1.182084560394287, 0.17374005913734436, 0.882066011428833, -0.484673410654068, -1.3639205694198608, 0.7485937476158142, 0.09894227236509323, 0.683682918548584, -1.3815659284591675, 0.1815534383058548, -0.6255009770393372, 0.8065888285636902, -0.5806873440742493, -0.6073948740959167, -0.5857203602790833, 1.2637170553207397, 0.028864195570349693, -0.7599971294403076, -1.0225841999053955, -1.0441224575042725, 1.4626275300979614, -1.814774513244629, -0.8503575325012207, 0.6378903388977051, 1.1784957647323608, -0.1482774019241333, 0.785516083240509, 0.759617269039154, 0.9861871004104614, 0.04641503095626831, 1.6549681425094604, 0.21612536907196045, 1.5482001304626465, -1.0827884674072266, 0.9606130719184875, -1.2350369691848755, -1.4755833148956299, 0.5254991054534912, -0.7997444868087769, 0.47699373960494995, -1.0392570495605469, 1.6403754949569702, 0.04541300609707832, -1.0942738056182861, 0.21565909683704376, 0.924079954624176, -0.7268860936164856, 0.5617496967315674, 2.0933477878570557, 0.9154857993125916, 0.03924795985221863, -0.5195255875587463, 0.6291380524635315, -1.4644056558609009, -2.986478090286255, -0.6133970022201538, -1.0284662246704102, 1.799839735031128, 0.1958865523338318, -0.9284030795097351, -1.1052078008651733, 0.5581414699554443, -0.7712110280990601, -0.5223010182380676, -0.030369741842150688, 0.15036295354366302, 0.16054552793502808, 0.17942208051681519, 1.3678315877914429, 1.768678903579712, -1.5849144458770752, 2.6462831497192383, 0.19913043081760406, 1.314860224723816, -0.6682167649269104, 0.3973124921321869, 0.28750643134117126, 0.18613076210021973, 0.9257142543792725, 0.7057353854179382, -0.9314151406288147, -0.043422214686870575, 0.8034627437591553, 0.7619158625602722, -0.26096925139427185, -0.20561031997203827, -0.07871096581220627, -1.0290894508361816, -1.2122299671173096, 1.6446222066879272, -0.31982892751693726, 0.17068354785442352, -0.672717809677124, 1.3727784156799316, 1.8276342153549194, -0.14458514750003815, 0.5897448658943176, -0.6796556115150452, 1.8403476476669312, 1.001141905784607, -0.7646186947822571, 1.3231734037399292, 0.4352547824382782, 0.48870596289634705, 0.43224677443504333, -0.5942724347114563, -0.4640330970287323, 0.6962088942527771, 0.023503541946411133, -0.8758109211921692, 2.103349208831787, -0.13811509311199188, -0.4884083867073059, -0.6271817088127136, 0.7604384422302246, 1.0446656942367554, 0.3153950572013855, 0.9384251236915588, -1.4301458597183228, -0.3031352162361145, -0.40237241983413696, 0.07267668098211288, 0.5553210973739624, -1.9734467267990112, 3.1912713050842285, -0.8026558756828308, 0.40858787298202515, 0.8906661868095398, 0.9132422804832458, 0.3015294671058655, -2.851038694381714, -1.0294182300567627, 0.815044641494751, -0.8672524094581604, -1.0034019947052002, -2.304955244064331, 1.266568899154663, 0.5565275549888611, -0.8489745259284973, -0.09093708544969559, -0.06102960556745529, -1.0791676044464111, -0.2390943467617035, 0.28724735975265503, 0.5837607383728027, 1.2956645488739014, 0.05704819783568382, -0.4357512593269348, 0.4279448390007019, -0.6054584980010986, -1.2284018993377686, 1.165817379951477, -0.3596625328063965, 0.1418885886669159, 0.6701045036315918, -1.087502121925354, 0.6610705852508545, 0.4700078070163727, -0.07869519293308258, 0.413828581571579, -0.737575352191925, -2.1866278648376465, -0.26442113518714905, -0.8926159143447876, 0.8870077729225159, 0.2828402817249298, -0.5116117000579834, 0.0655950978398323, 1.3710854053497314, 1.1530663967132568, 0.38690221309661865, 0.44852638244628906, -0.26040804386138916, -0.11286324262619019, 0.6066693663597107, -0.10218464583158493, 0.3161347210407257, -1.0989238023757935, -1.6523231267929077, -1.682691216468811, -0.4212658107280731, 1.3442906141281128, 1.1995958089828491, -0.9814296364784241, -1.5408070087432861, 0.8088977932929993, 0.4789857864379883, -1.353267788887024, 1.0735628604888916, 0.19024643301963806, 1.3394519090652466, 0.3579726219177246, 0.9059391617774963, 0.33684447407722473, -0.9376466870307922, -0.2178904265165329, -0.6630188226699829, 0.8452523350715637, -0.981764554977417, -1.519432783126831, 0.6968106031417847, 1.1609312295913696, 1.4599090814590454, 1.1067994832992554, -1.4944950342178345, -0.6837617754936218, -0.5355051159858704, -0.010022813454270363, -2.1419713497161865, 0.002466826234012842, 0.2888009548187256, -0.9390360116958618, 0.12963184714317322, 1.2030428647994995, 0.05754097178578377, 1.068389892578125, 1.0254358053207397, 0.4216483533382416, 0.39493289589881897, -0.3604743480682373, 1.2524467706680298, -0.45974308252334595, -0.6860493421554565, 0.7173232436180115, -0.7147547006607056, 0.43736329674720764, 1.0686253309249878, 0.09235601127147675, 0.11751622706651688, 0.9307212233543396, 0.7662864923477173, -1.5274426937103271, -1.2976537942886353, 1.3449342250823975, 0.4041241705417633, -2.067204236984253, -1.3341705799102783, 0.04009436070919037, -0.9646182656288147, 0.48522254824638367, 0.09518151730298996, 0.5078786015510559, -1.1506434679031372, 0.6393343210220337, 1.978328824043274, -0.7758641839027405, 1.3338230848312378, 1.0021175146102905, 0.6649901270866394, -0.04457911476492882, -1.1405709981918335, -0.8561702966690063, 0.7861270904541016, 1.0016069412231445, 1.2456564903259277, 0.9835174679756165, 0.9265053272247314, -0.41126105189323425, -1.1844086647033691, 0.22776035964488983, 0.5240649580955505, 0.1097026914358139, -1.4964087009429932, -0.4962444603443146, -1.4530391693115234, 0.16998548805713654, -1.0258519649505615, -0.142476424574852, 0.3472633957862854, 0.1621048003435135, 1.3936094045639038, -1.0797241926193237, -2.1580617427825928, 0.9708937406539917, 1.428524136543274, 1.4587852954864502, 0.3118498623371124, 0.5667908787727356, 0.15065422654151917, -1.266661286354065, -0.5423584580421448, 0.5257276892662048, -0.5477734804153442, 1.9372409582138062, -1.5109803676605225, -0.4037870466709137, 1.1405305862426758, -0.5772757530212402, -1.1019749641418457, -1.3286824226379395, -0.6961856484413147, 1.5409595966339111, 0.4400498569011688, 0.5478272438049316, -0.4877318739891052, 0.8760697245597839, 0.13884399831295013, -0.8888521194458008, -0.41992905735969543, 0.23508520424365997, 1.2034703493118286, -0.025284837931394577, 0.5330597758293152, 2.052851438522339, -0.7653564214706421, -1.2071739435195923, 1.0169624090194702, 0.4721352756023407, -0.06420072913169861, 0.298200786113739, -0.29524552822113037, 0.5504210591316223, -0.30122679471969604, 0.8346636891365051, -1.1357535123825073, -0.7816544771194458, 0.1515355259180069, -0.4345260262489319, 0.7310964465141296, -0.006476094946265221, -0.5063658952713013, 0.24596531689167023, -0.49146533012390137, -1.520829439163208, -1.0989676713943481, -0.764117956161499, -0.4321034252643585, 0.13610687851905823, 0.9499886631965637, 0.591343343257904, 1.8206182718276978, 1.3160899877548218, -0.1465705782175064, 1.6935900449752808, 1.2241992950439453, -0.40974128246307373, 0.35544049739837646, 1.1388827562332153, 0.28662607073783875, 1.06144380569458, 0.44097617268562317, 0.7493008375167847, -0.2722027003765106, 0.4252520203590393, -0.31735706329345703, 0.7226077914237976, -1.6420605182647705, 0.38173675537109375, 1.297328233718872, 0.6274482011795044, 0.021638665348291397, 0.5861490368843079, -0.798447847366333, -0.7843064069747925, -0.31168362498283386, -0.1006540060043335, 1.013187050819397, -0.8469995260238647, 1.0477770566940308, -1.5258344411849976, -0.7400428056716919, 0.010092291980981827, 1.2045551538467407, -0.5668904781341553, -2.0968539714813232, 0.46663737297058105, 0.017535120248794556, -1.0356242656707764, -0.9973533749580383, -0.649196982383728, 0.144624724984169, 0.709801971912384, -1.4321069717407227, 2.3350119590759277, 0.00820211973041296, -0.32695433497428894, 0.7186461091041565, 0.1545155793428421, 2.3817873001098633, 1.1414964199066162, -0.32327789068222046, -0.3454863727092743, -0.45306384563446045, -0.29686376452445984, 0.7174218893051147, -0.6014658212661743, 1.657326102256775, -0.037853263318538666, -0.1016811951994896, 0.992730975151062, -1.9356498718261719, -0.8924553990364075, 0.6421632766723633, 0.05458779260516167, 0.0309041365981102, 0.43976300954818726, -0.558366060256958, -0.1283116638660431, -1.3187379837036133, -1.833094835281372, 1.3067233562469482, -0.8911605477333069, -0.6746698021888733, 1.4376716613769531, 1.100531816482544, -0.48571476340293884, -0.490479975938797, -0.04172052443027496, -0.9150159358978271, 0.5880361199378967, 0.1845998913049698, 0.1689651757478714, 0.7926115393638611, 0.757807195186615, 1.8810261487960815, 1.5556578636169434, -1.6711822748184204, 0.9366335868835449, -1.0798875093460083, -1.3894935846328735, 0.7497140169143677, -0.663746178150177, -0.27746066451072693, 1.1084840297698975, -0.6249480247497559, -0.8568428754806519, 0.3652093708515167, 1.2615996599197388, 0.5671926736831665, -1.5152347087860107, -0.6126543879508972, -0.11370231956243515, 1.9686957597732544, 0.9317599534988403, 0.8898720145225525, 1.2413420677185059, -0.2706436216831207, 0.5181459188461304, 0.5287922620773315, -0.938443660736084, 0.02170277014374733, 0.6026281714439392, -1.0412603616714478, 0.8466441631317139, 0.7972095608711243, 1.0722075700759888, -0.7363685369491577, 0.14570656418800354, 1.4916578531265259, 0.8310011029243469, -0.7500245571136475, 0.263302743434906, 1.458528995513916, -0.038349635899066925, 0.4187130331993103, 2.606179714202881, 0.0004128905711695552, -0.0708533301949501, -0.9691065549850464, -0.6225593090057373, 1.5211036205291748, -0.10413961857557297, -1.15211820602417, -0.8976781368255615, 0.03908656910061836, 0.7977871894836426, -0.7123287916183472, 0.7443654537200928, -0.45831558108329773, 0.3848016858100891, 0.5471117496490479, 0.6337680816650391, -0.40673431754112244, -0.14548945426940918, 0.5283980965614319, -1.273240566253662, 0.19109147787094116, 0.5307928323745728, 1.0592479705810547, -0.023392777889966965, -0.17200535535812378, 0.1524926722049713, -0.7232627272605896, -0.20064637064933777, 0.4159854054450989, 0.28739240765571594, -1.0392892360687256, -0.32265856862068176, -2.111755132675171, 0.801702618598938, -0.532426655292511, 0.2373330146074295, -1.102174997329712, -0.08905167877674103, 0.4396975636482239, 0.9738321900367737, -1.3697545528411865, 0.8326238393783569, -0.1757824867963791, -0.7873982191085815, 0.46329113841056824, -0.3082805871963501, -0.3110032379627228, -0.7189329862594604, 0.20695945620536804, 0.01984628476202488, -0.19113457202911377, -0.9731241464614868, -0.5300232768058777, 0.3835427761077881, 1.2592462301254272, -0.04607333987951279, 1.3936539888381958, -0.030390454456210136, -1.1096733808517456, 0.6112676858901978, -0.9851086735725403, 0.38207483291625977, 1.0499378442764282, -0.3277072310447693, -0.10802119970321655, 2.4462101459503174, 0.21949394047260284, 0.24460020661354065, 0.761101245880127, -0.11728048324584961, -0.10719336569309235, 0.36194998025894165, -2.4084315299987793, 1.3955397605895996, -1.0846890211105347, -1.0954275131225586, 0.5201892256736755, 0.008816112764179707, -1.266526222229004, 0.19369088113307953, -1.156624436378479, -0.7133705019950867, -1.5188568830490112, -0.8435394167900085, 0.3790154755115509, 0.10514663904905319, 0.2910861670970917, 0.9027455449104309, -0.06060607358813286, -0.6768487095832825, -0.2972157895565033, 1.6273870468139648, 0.015504619106650352, 0.30083930492401123, -0.11406422406435013, 0.22672437131404877, 0.4124878942966461, -0.027173079550266266, -0.172787144780159, -0.7982817888259888, -0.8626048564910889, 0.38799381256103516, 0.7287136912345886, 0.9030340909957886, 0.03557128831744194, -0.841515839099884, 0.6105508804321289, 0.5961484909057617, -1.5178544521331787, 0.7828530669212341, 0.5630512833595276, 1.6445955038070679, -1.1425522565841675, -1.2950359582901, 0.7669473886489868, -1.8088688850402832, -0.7460106015205383, -1.1059008836746216, -1.0267740488052368, -1.917872428894043, -0.252991646528244, 0.9571146965026855, 0.4253012537956238, 0.42965278029441833, -0.3706599771976471, 1.4285941123962402, 0.513006865978241, 1.0352050065994263, -1.5575213432312012, 0.33247533440589905, -1.1505359411239624, 0.3378314971923828, 0.18840168416500092, -1.1319832801818848, 0.16810542345046997, 1.1498833894729614, 0.49358564615249634, 0.5839363932609558, -0.8643968105316162, -0.6808387637138367, -1.111741304397583, 0.12878969311714172, 1.1826921701431274, 0.43184301257133484, -0.0034248074516654015, -1.1165767908096313, 0.44173526763916016, -0.23774702847003937, -1.8976415395736694, -1.7217886447906494, -0.3377601206302643, -0.8907064199447632, 1.64602792263031, 0.03591098636388779, -1.0325337648391724, -0.4844639003276825, 1.5880379676818848, -0.9419392943382263, 0.7695557475090027, 0.43886658549308777, 1.465262770652771, 0.5318866968154907, -0.39721816778182983, 0.5285932421684265, 0.07031619548797607, 0.9742986559867859, -0.05680607259273529, 1.9751167297363281, -0.09490149468183517, 0.44469738006591797, -0.6460459232330322, 0.13374091684818268, 0.036246493458747864, 0.17917950451374054, 0.9070638418197632, -0.5239865779876709, -0.8315182328224182, 1.1150941848754883, -0.272482693195343, -0.6534436941146851, 1.1615248918533325, 0.7875040173530579, -2.6181998252868652, -0.36186227202415466, 0.7963047027587891, 0.8459317088127136, -0.4945407211780548, 1.5728285312652588, -0.7711960673332214, -0.21201692521572113, 0.14625568687915802, 0.014140341430902481, 0.8223953247070312, 1.0362454652786255, -0.0231077391654253, 1.4129376411437988, -0.6593137383460999, -0.1894417554140091, 1.7349375486373901, 0.12129327654838562, 1.08705735206604, -0.4028674066066742, -0.23735825717449188, -0.07818762958049774, 0.19849228858947754, -0.30805739760398865, 0.04580876976251602, 0.6959390044212341, 1.2295970916748047, 0.1392023265361786, 0.6767752766609192], [0.9390639066696167, -1.0389410257339478, -1.5325171947479248, 1.6407750844955444, 1.345535159111023, 0.09034295380115509, 0.38273125886917114, -0.9037477970123291, -1.3427045345306396, 0.34758979082107544, -1.445356845855713, 0.9417096376419067, -0.643819272518158, 0.3217112421989441, 1.9572620391845703, 1.910284161567688, -0.23866190016269684, 1.0937919616699219, -0.9944708347320557, -0.5517165064811707, 0.3992884159088135, 1.4106777906417847, -0.5413222908973694, 0.5688710808753967, 0.13061578571796417, -1.2283730506896973, -0.10485062748193741, 1.5210869312286377, -0.0637921467423439, 0.5277710556983948, 0.8529094457626343, -0.7059682607650757, 1.2532442808151245, 1.5241742134094238, 1.6178134679794312, 1.3345118761062622, -0.43692368268966675, -0.5579150915145874, -0.7572281956672668, -0.8790299296379089, -0.7681241631507874, 1.0123260021209717, 1.4809725284576416, -0.6153769493103027, 0.8306652903556824, -1.2391290664672852, -0.30372118949890137, -0.9720001816749573, -1.603305697441101, 0.07313946634531021, 0.09654352813959122, -0.3884628117084503, -0.5046857595443726, -0.9667872190475464, -0.11661641299724579, -0.584105372428894, 0.25431448221206665, -1.189256191253662, 0.23966781795024872, 0.522671103477478, 1.2523924112319946, 1.0603009462356567, 0.20144441723823547, -0.6709134578704834, -1.8168476819992065, 0.0026918591465801, 1.4207584857940674, 1.665061116218567, -2.529310941696167, -0.7472798824310303, -0.3420387804508209, -1.0319069623947144, 0.1937917172908783, 0.04599625617265701, -0.5092268586158752, -0.028704989701509476, -1.58358895778656, -0.9637572765350342, -0.8882927894592285, 0.9028779864311218, -1.6191641092300415, -0.20645304024219513, 0.783644437789917, -0.04793410748243332, 0.8168278932571411, -0.6461483836174011, 0.441234827041626, 0.9096816778182983, 2.2376174926757812, 0.7338247299194336, 1.4342801570892334, 0.01947159692645073, 0.1071014329791069, 1.5773988962173462, 0.5929288268089294, 1.428602933883667, 0.7207574248313904, 2.1328043937683105, 2.394803047180176, -0.8218938708305359, 1.8732020854949951, 0.4500902593135834, 0.49367761611938477, -0.09755853563547134, 0.7372644543647766, -1.0916911363601685, -0.7062584757804871, -0.7429202795028687, 0.3118625581264496, 0.9444763660430908, 0.7609955072402954, 0.39364364743232727, -0.5666345357894897, -1.733282208442688, 0.43093422055244446, 0.17362281680107117, 0.3855600953102112, 1.4560154676437378, -0.15326151251792908, -0.1728496253490448, -2.2867226600646973, 0.5885845422744751, -0.019332516938447952, -0.7095016241073608, 0.41531333327293396, -1.0146247148513794, -0.5123655796051025, 1.9919570684432983, 0.7810935378074646, 0.9548371434211731, 0.7776476740837097, 0.6709345579147339, 0.22161290049552917, 2.3129711151123047, -0.0466807559132576, 0.7619715332984924, -0.1926318258047104, -0.3258672058582306, -2.4131600856781006, -1.8994190692901611, 1.571799635887146, 1.511946439743042, -0.4144875705242157, -0.010676424950361252, 0.6338039040565491, -0.9740737080574036, 1.7892078161239624, -1.0210282802581787, -0.7019392848014832, -2.745591402053833, 1.5320641994476318, 0.0748327374458313, 1.6393996477127075, -0.9860650897026062, 0.35382410883903503, 0.09005791693925858, 0.21417666971683502, 2.0897886753082275, 2.2807819843292236, 1.3934330940246582, -0.38719886541366577, 0.9069615602493286, 1.3587636947631836, 0.5713238716125488, 0.5823290944099426, 0.4946015179157257, -0.8889856338500977, 0.2572779059410095, 1.6154931783676147, -0.9842339158058167, 0.09306641668081284, 0.846075713634491, -1.3539173603057861, -0.12730292975902557, -1.686989426612854, -0.5965573787689209, -1.3980454206466675, -0.8877118229866028, -0.4822198450565338, -0.26986026763916016, -0.1984267383813858, -2.4011573791503906, 1.593066930770874, -0.27143871784210205, -1.0933544635772705, -0.6267236471176147, 0.31149446964263916, -1.001158356666565, -0.41353851556777954, -0.4698505997657776, 0.5397399663925171, -0.748589813709259, 0.4534892439842224, 0.34906983375549316, -0.14843380451202393, 1.2411036491394043, 0.02043960615992546, -1.1776244640350342, 0.297817587852478, 0.1231890395283699, -0.04050282761454582, 0.23232640326023102, 2.349553108215332, 0.23351901769638062, 0.45779475569725037, -0.24645480513572693, 1.3103865385055542, -0.056560587137937546, -0.8021083474159241, -1.2226428985595703, 0.5270918607711792, 0.7442358732223511, -1.5885518789291382, -0.7237141132354736, 0.641386866569519, 1.027116060256958, -1.3212260007858276, -0.40871185064315796, 0.8317688703536987, -0.5348942875862122, -0.45173537731170654, -0.5417974591255188, -1.3082160949707031, 0.045409709215164185, 0.30292508006095886, -1.4035862684249878, 0.098485566675663, 1.106058120727539, 0.20567147433757782, 0.9453840255737305, -0.5513148307800293, 1.36720609664917, -0.7881364822387695, 1.8833832740783691, -0.45110389590263367, -0.7207785248756409, -0.8949810862541199, -0.4150449335575104, -0.49557724595069885, -0.6917725801467896, -0.33178502321243286, 0.6620391011238098, 0.8797226548194885, -0.34002912044525146, 1.2321466207504272, -1.0006740093231201, -1.031279444694519, -1.1338053941726685, 0.5986278653144836, -1.553841233253479, -1.3998346328735352, 1.2750935554504395, 1.7088872194290161, 1.3391127586364746, -1.4963747262954712, 0.006539255380630493, 0.8261597156524658, 0.09072638303041458, -1.6242270469665527, -0.34872177243232727, -1.7051392793655396, -1.239441156387329, 2.1451520919799805, 0.6719362735748291, -1.475069284439087, -1.5086182355880737, 0.4481253921985626, -0.6179530620574951, 0.5686865448951721, -0.6563408374786377, -0.20918555557727814, 0.22913804650306702, -1.0425843000411987, -1.9301940202713013, -1.2045025825500488, -0.7088424563407898, -0.21507838368415833, -0.07068876922130585, -0.07921269536018372, 0.5019532442092896, -0.9050357341766357, 1.382136344909668, 1.9883475303649902, 0.3520614206790924, 0.17394126951694489, 1.2332775592803955, 0.6337528824806213, 0.17293903231620789, -0.18279434740543365, -0.75407874584198, -1.184386968612671, 0.6499515175819397, 1.172234296798706, -1.0725984573364258, -0.08787476271390915, 0.20212575793266296, -0.4414781332015991, -0.7774019241333008, -0.408725768327713, -1.9865738153457642, -0.07726774364709854, -0.3106481730937958, 2.10026478767395, 0.9197710752487183, -0.36233624815940857, -0.024983713403344154, 1.6989367008209229, 1.5887950658798218, 0.16741353273391724, -0.14574536681175232, -1.4623169898986816, -0.3444855213165283, -0.18632164597511292, -0.7531993389129639, 1.7125872373580933, -1.8930187225341797, -0.15312954783439636, 0.38952407240867615, 0.4192516505718231, 1.6691080331802368, -1.4161486625671387, -0.7655752301216125, 0.07940022647380829, 0.11466497927904129, -0.4034154415130615, -0.7818980813026428, 1.0552581548690796, -1.0677045583724976, -0.5929406881332397, 1.4195395708084106, 1.7160123586654663, 0.8537982106208801, -0.8782702088356018, -0.8709902763366699, -0.9172168374061584, -1.1524791717529297, -0.4340750575065613, 1.2524681091308594, 0.20403172075748444, -0.7745705246925354, -0.797057569026947, -1.19520103931427, 1.1866185665130615, -2.0405044555664062, 0.2517307996749878, -1.4622981548309326, -1.082761526107788, -1.512913465499878, 0.9379881024360657, -0.10783299058675766, 2.7162272930145264, -1.064976453781128, -1.1233488321304321, 0.5610081553459167, 0.5322912931442261, -0.3127525746822357, -2.052834987640381, 1.4116051197052002, 0.5343497395515442, 0.36350691318511963, -0.6297994256019592, 0.6034522652626038, -0.5992394089698792, 0.0027823620475828648, -0.23031124472618103, 1.2153445482254028, -0.5142291784286499, -0.3904013931751251, 1.0593440532684326, 0.17986184358596802, -0.06755071878433228, -0.29001107811927795, 0.901050865650177, -0.7011904716491699, 0.4197467565536499, 1.9785337448120117, 0.34653183817863464, -0.8393389582633972, -1.2570782899856567, 0.0913928896188736, -1.2842551469802856, -1.651800274848938, -0.8178593516349792, -1.154883861541748, 0.815321683883667, -1.5129321813583374, 1.5206859111785889, -0.8151857852935791, 0.054662980139255524, 0.6076080203056335, 0.15800535678863525, 0.6552466750144958, -1.0543657541275024, -2.341078996658325, -0.2707422375679016, -1.752894639968872, -0.652518630027771, 0.8720541000366211, -0.27137988805770874, -0.6681801676750183, -1.205816626548767, -0.3686716854572296, -0.25081679224967957, -1.6610217094421387, 0.21573515236377716, -1.155592679977417, 0.1384168416261673, -0.5325832366943359, 0.9094871282577515, 0.7757476568222046, -0.07191889733076096, -0.1894349902868271, 0.3586345314979553, 1.3057222366333008, 0.9909160137176514, 0.08859570324420929, 0.24926498532295227, 1.5156350135803223, -1.695516586303711, 0.48489823937416077, -0.9567307829856873, 1.3039183616638184, -0.5436124205589294, -0.15811483561992645, -1.1306650638580322, -0.02417014352977276, 0.7569795250892639, 1.639538049697876, -1.2954630851745605, -0.4699377417564392, 3.8864552974700928, 0.9217358827590942, -0.025641659274697304, 0.5070818662643433, -2.681901454925537, 1.6035196781158447, 0.14751459658145905, -1.445626974105835, 1.1621214151382446, -0.6370449662208557, -1.0342037677764893, 0.5195441842079163, 0.39870986342430115, -0.4621754288673401, -0.0758126750588417, -1.346692681312561, -1.4483717679977417, 0.8991607427597046, 1.3172838687896729, 0.43128523230552673, 0.37363964319229126, -0.3960976004600525, -0.13019312918186188, -0.9373248815536499, 1.7582709789276123, -0.5680426955223083, 0.4106023907661438, 0.0001667467731749639, -1.4002809524536133, 1.9391934871673584, -0.37582993507385254, -0.7687258720397949, 0.11466401070356369, -0.898296594619751, 0.029630368575453758, -0.960284411907196, 0.5653350949287415, 0.055658962577581406, -1.368286371231079, 1.0376981496810913, 0.1943228840827942, -1.1029078960418701, 0.361402302980423, 0.43308210372924805, 1.8657306432724, -0.8602581024169922, 0.8611261248588562, -0.08875591307878494, 0.6232821345329285, -0.2556106746196747, -0.050496410578489304, 0.5057615041732788, -1.1841301918029785, -2.0985774993896484, -0.0001655638770898804, -0.4367785155773163, 0.09812106192111969, 0.6559041738510132, -0.20338432490825653, 1.2112683057785034, -0.024050770327448845, 0.13227415084838867, 0.8550949692726135, 0.6115972995758057, -0.0645970031619072, 0.15147794783115387, -0.7325077056884766, -0.588540256023407, 0.13132265210151672, 0.8462540507316589, -0.4059438109397888, 0.8735383152961731, 1.9794363975524902, 2.1877756118774414, 0.06357245147228241, 1.095914363861084, 0.6528558731079102, 0.830687940120697, 0.8841544985771179, 0.5700922608375549, 0.8754392266273499, 0.24068191647529602, -0.20984597504138947, 0.30575820803642273, 0.5470753312110901, 1.2804758548736572, 0.4263070523738861, 0.7989928722381592, 1.4833457469940186, -0.879856288433075, -0.20736753940582275, 0.5968579649925232, 0.08149553090333939, -0.9605385661125183, 1.046521544456482, -0.015999235212802887, -0.9484673738479614, 1.9530872106552124, -1.2323038578033447, 0.03862753137946129, -0.6207221746444702, 0.16117168962955475, 0.6161870956420898, -0.7686342000961304, -0.05466770380735397, -0.08203095942735672, -0.9570086002349854, 1.2168262004852295, 0.22016793489456177, -0.24215134978294373, -0.6210333704948425, -0.2890438735485077, 0.1883605718612671, -0.41753193736076355, 1.296987771987915, -2.5053977966308594, -2.102890729904175, -0.2128668874502182, -1.8045495748519897, -0.024169601500034332, -0.6157132387161255, 0.6257994174957275, 0.45880621671676636, 1.3551604747772217, 0.96669602394104, 1.3010443449020386, 0.5754666328430176, 1.11487877368927, 0.5473601222038269, -0.5467557907104492, -1.2079026699066162, -1.1677119731903076, 0.528984785079956, -2.344456434249878, 0.2211552858352661, -0.7085970640182495, 0.7815117835998535, 0.2450992912054062, -0.7064484357833862, 0.059004005044698715, -2.2333641052246094, 0.31525757908821106, 0.5122212767601013, -1.7966276407241821, 0.9052960872650146, -0.07355114072561264, 0.21914319694042206, -1.3762871026992798, 0.6531754732131958, 0.6302028298377991, 0.03348367288708687, -0.9371272921562195, 0.8133684992790222, 1.2992558479309082, 0.4526406526565552, 1.05582857131958, -0.5307132601737976, 0.2122935950756073, 0.4359525442123413, 0.2067253589630127, 2.8865253925323486, 0.48701784014701843, -0.4927061200141907, 0.9004853367805481, -1.0137019157409668, 0.44453874230384827, 0.8487818837165833, -0.41249802708625793, -0.23828640580177307, -0.8553231954574585, 0.2178591787815094, -0.588180661201477, 1.8263884782791138, 0.7398642897605896, -0.12996773421764374, -1.322251796722412, 2.1339375972747803, 1.402572512626648, 0.5735793113708496, 0.299108624458313, 0.2745548188686371, -0.3655281066894531, 1.986129879951477, 0.3093262314796448, -1.335917592048645, -0.5531930327415466, -2.4793617725372314, 0.7640093564987183, 0.8690643310546875, 1.876631259918213, 2.2044386863708496, -0.6787261962890625, 0.5533410906791687, 1.1001068353652954, 0.8960789442062378, -0.4079291522502899, 0.9048473834991455, 0.5957071781158447, 1.4620180130004883, 0.06936802715063095, -0.14207717776298523, -0.41482821106910706, -0.2837981879711151, -0.6733856201171875, -0.24865064024925232, 0.5934916734695435, 0.5493717789649963, 1.4085243940353394, 1.1823714971542358, 0.7123072147369385, -1.0688021183013916, 0.6528002619743347, 1.2463040351867676, -1.2251017093658447, -1.0397617816925049, -0.8794495463371277, -0.9670960307121277, -0.21878889203071594, -0.8675790429115295, 0.4544622302055359, -1.1485435962677002, -0.06447743624448776, -0.4381052553653717, -0.9098559021949768, 1.094797968864441, 0.08675415813922882, 0.6002408266067505, -0.2757418751716614, 0.7611515522003174, 0.6721111536026001, -0.09537194669246674, -1.7179102897644043, 0.8777901530265808, 0.7497206330299377, 0.23805901408195496, 0.29991328716278076, 0.18103457987308502, 0.2500183880329132, -0.947805643081665, 0.0017189881764352322, 0.37575238943099976, -0.5547694563865662, -0.9129834175109863, 1.9173918962478638, 0.6651930212974548, -0.2487480342388153, -0.43437638878822327, -2.58504581451416, -1.3675827980041504, -0.1913093775510788, 1.3467440605163574, -1.9998259544372559, -0.6417015194892883, 1.9730949401855469, 0.7285888195037842, 0.4735696315765381, 1.2539916038513184, -0.06093714386224747, -0.11823772639036179, 0.774385392665863, 0.9110653400421143, -1.0072636604309082, 1.588135838508606, -1.683752417564392, -0.6347077488899231, 3.135681629180908, -0.4506058394908905, 0.7173622846603394, -0.9405189752578735, -0.2588171362876892, -0.6583632230758667, -0.3365699350833893, 0.7187452912330627, 1.8119304180145264, 0.9365940093994141, 1.6203142404556274, -0.4437192678451538, 1.529422402381897, 0.6273380517959595, 1.4351396560668945, -0.355573832988739, -0.17041146755218506, 1.428621530532837, 0.6709538102149963, 0.16643300652503967, -0.6267417669296265, -0.04386284947395325, 0.5398931503295898, 0.7751210331916809, -0.5037563443183899, -0.29154324531555176, 0.47608259320259094, -1.0634695291519165, 0.9797362685203552, 0.02994203194975853, -0.10251457989215851, -2.386324405670166, -1.573457956314087, -1.0216054916381836, -0.5714914202690125, -1.0890519618988037, 0.37822476029396057, 0.13628891110420227, -0.26628658175468445, -2.0343573093414307, 1.3978474140167236, -0.278790146112442, 0.5904706120491028, -0.6711690425872803, -0.0029405353125184774, 0.6943869590759277, 0.23325610160827637, 0.9592137336730957, -0.18441547453403473, 1.9883666038513184, -1.897481918334961, -0.5563602447509766, 0.9744526743888855, -0.25700584053993225, -0.4497135877609253, -1.7821519374847412, 1.190725326538086, -1.9066672325134277, 0.49793556332588196, 0.18430539965629578, 0.8637610673904419, 1.4469974040985107, -0.7158894538879395, -0.8238552212715149, -0.33265525102615356, -1.735034704208374, 0.8031315803527832, -0.4167933464050293, 0.1382836103439331, -1.1521764993667603, -0.014647300355136395, -0.6481513381004333, -0.10353849828243256], [-1.1224991083145142, 0.9942165017127991, -0.68248051404953, -2.5877280235290527, 0.6099529266357422, -0.33607131242752075, -0.9213085770606995, 2.467170238494873, -0.09132832288742065, -1.3786239624023438, -0.6557888388633728, 0.338618665933609, -0.2680213153362274, 1.1991018056869507, 1.0216524600982666, -0.8339216709136963, -0.004582902882248163, 0.5990666151046753, -0.45530712604522705, -0.5063516497612, 0.8370848894119263, 0.4084610044956207, -1.4448473453521729, -0.5911694169044495, -0.45565178990364075, -0.1603986918926239, -0.7598997354507446, 1.3000696897506714, -1.0117992162704468, -1.1277618408203125, 0.16890501976013184, -2.2950961589813232, -1.0725997686386108, 0.5848870277404785, -1.365986704826355, -0.8850347399711609, 1.0463696718215942, 0.417703241109848, 0.018991127610206604, 0.20674896240234375, -0.7132614254951477, 0.2741943299770355, -0.23711468279361725, 2.4666926860809326, 0.5068399310112, -0.36892151832580566, -0.08286812901496887, 0.02450433000922203, -1.4086029529571533, -0.8518156409263611, 1.4998193979263306, -1.4307782649993896, -0.6069023013114929, 0.9984365701675415, 1.4870041608810425, -1.0472524166107178, 1.4401201009750366, -0.38452863693237305, 1.635938048362732, 0.4814639985561371, -0.3014393448829651, 0.7112938165664673, 0.54352867603302, -0.7892229557037354, 2.1329989433288574, -0.3593624234199524, 1.1382944583892822, -0.4997788369655609, -1.8414429426193237, -1.136546015739441, -0.1143496185541153, 0.6690422892570496, 1.7481712102890015, 0.06817593425512314, -0.2956811785697937, -1.243565320968628, -0.05959330499172211, 0.2867417633533478, -2.227774143218994, 1.2913390398025513, 1.123835802078247, 2.2635679244995117, -0.9252534508705139, 0.024259066209197044, 1.7177671907120384e-05, -1.3433594703674316, -0.8674712777137756, -1.0028724670410156, 0.4974759817123413, 0.41373687982559204, 1.3314073085784912, -0.42221546173095703, -1.4182991981506348, 0.41974276304244995, 0.3502131402492523, 0.6464753746986389, 1.2269155979156494, -0.6066330075263977, 0.7783809304237366, -1.0265306234359741, 0.13558101654052734, 0.6694432497024536, -0.13722003996372223, 0.1963340789079666, -1.4165822267532349, -0.6728836297988892, 0.12781967222690582, -1.2268056869506836, -0.8775528073310852, 2.412447214126587, 0.23366864025592804, -0.851930558681488, -1.0626219511032104, 1.3499202728271484, -2.8426356315612793, -0.4077172577381134, 0.5435798168182373, -0.6556543707847595, 0.4250890612602234, -1.153435230255127, -0.12242576479911804, -0.06943148374557495, -0.10122037678956985, 0.029246743768453598, -0.08028572797775269, 1.0012798309326172, -0.38539427518844604, -0.7129018902778625, -0.40711838006973267, -0.43082723021507263, -1.6526169776916504, -2.0419864654541016, -1.4839376211166382, 0.15324482321739197, -0.7994989156723022, -0.5444471836090088, 2.737698554992676, 0.9931285977363586, -0.34309127926826477, -0.7650974988937378, 1.2438132762908936, -0.9154293537139893, 1.1042662858963013, 0.4549177587032318, 1.2527658939361572, -0.8592309355735779, 0.7707862854003906, -1.948150396347046, -0.3085043132305145, 0.9452505111694336, 0.5608177185058594, -1.973645806312561, 0.5494218468666077, 0.3177606165409088, -0.20235425233840942, -0.8623090386390686, -0.42053475975990295, -0.7138211727142334, 0.8081549406051636, 0.32695645093917847, 0.9301929473876953, 0.5505679845809937, -1.422502040863037, -0.0036481020506471395, 2.0051002502441406, -1.251207709312439, 0.14084790647029877, -0.9215608239173889, -1.0304752588272095, 0.07115548104047775, 0.7788941860198975, 0.6279189586639404, 0.32190531492233276, 0.8101217746734619, -1.7717849016189575, 0.6765504479408264, 0.26267164945602417, -0.32582399249076843, -0.15281446278095245, -0.9964066743850708, 1.0230422019958496, -0.8969496488571167, -1.619987964630127, -0.14953769743442535, -0.7231910228729248, -0.21331830322742462, 1.3711321353912354, -0.28041642904281616, -0.018401550129055977, 2.127685308456421, -1.5467555522918701, 0.2276153862476349, 1.0300360918045044, 1.379024624824524, 2.1122491359710693, -0.7967726588249207, 0.4749739170074463, -0.230729341506958, -0.5896416306495667, -0.8614416718482971, -1.6004722118377686, 0.2020159810781479, 0.6228167414665222, -1.1983243227005005, 0.6320664882659912, -0.6868362426757812, -1.1491180658340454, -2.765415906906128, 0.9779204726219177, 0.5900130271911621, -0.44342759251594543, -0.5181060433387756, -1.4363276958465576, 0.39181700348854065, -0.19110986590385437, 0.417136013507843, 0.05362837761640549, -1.5397968292236328, -1.6890698671340942, 0.2998860478401184, -1.8213775157928467, 0.1572043001651764, 0.7969985008239746, -1.4054841995239258, 0.5694829225540161, -0.06115039065480232, 0.43666011095046997, -0.7351228594779968, -0.10857977718114853, 0.34052541851997375, 1.7564114332199097, 0.6585507988929749, 0.4240565001964569, -1.3804391622543335, 0.2639543116092682, -0.7550179958343506, -0.13604216277599335, 0.2071915864944458, 0.9158821105957031, 0.5392610430717468, 0.13434144854545593, -0.11184676736593246, 1.3137245178222656, -0.9425020813941956, -0.009123696945607662, 1.3387553691864014, -0.2414776235818863, 0.7907440662384033, 0.852977454662323, 1.9500311613082886, 1.0958369970321655, -1.7058370113372803, -0.04420303925871849, 0.24934324622154236, 0.7854777574539185, 0.3401229679584503, 1.8472790718078613, 1.05498468875885, -1.2480685710906982, -0.7006041407585144, -2.1460163593292236, 1.673776626586914, -0.2036363184452057, -0.22646598517894745, -1.6842881441116333, -0.5421620011329651, 0.32484492659568787, -0.5238729119300842, -0.5194655060768127, -0.7537512183189392, 1.8672086000442505, -0.6957301497459412, -1.6134731769561768, -0.6759340167045593, -0.5135419964790344, -0.260604590177536, -0.4185172915458679, -1.3075921535491943, -0.3830552399158478, 0.2070455700159073, -1.60598886013031, 1.8144290447235107, 1.5321985483169556, -2.030519723892212, -0.5649585127830505, 0.8303572535514832, 1.1683423519134521, 1.618346095085144, -2.0661847591400146, -1.3190441131591797, -0.7134859561920166, 0.36679452657699585, -1.2806649208068848, 0.25236639380455017, 0.2753777801990509, 0.5494953989982605, -1.4874353408813477, 0.02401883713901043, -0.09884186089038849, -0.5925266146659851, 1.1594327688217163, -0.45735248923301697, 1.3156324625015259, -0.7565046548843384, 1.5841286182403564, 1.53162682056427, -0.9746177196502686, 0.2079622745513916, 0.1882728785276413, -1.1192102432250977, 0.675058126449585, 0.48549655079841614, -1.000077247619629, -0.011015204712748528, 0.7970510721206665, -0.26605284214019775, -0.827451765537262, 1.5677791833877563, -1.0920722484588623, 0.2560620605945587, 1.0545178651809692, 0.7209298610687256, -1.9468703269958496, -0.4081268012523651, -0.20663301646709442, -0.2830014228820801, 0.5728076100349426, 0.4919133186340332, 0.08075904101133347, 0.21863798797130585, 0.9812970757484436, 1.2237670421600342, -1.9649516344070435, 2.179419994354248, -0.5958083868026733, 0.7622644901275635, -1.6825244426727295, -0.5499566793441772, -1.3022358417510986, 1.0799956321716309, 0.8791921734809875, -0.014258560724556446, -0.19162176549434662, 0.3743670582771301, -0.9717922210693359, -0.6341529488563538, -0.7940674424171448, 0.8150408864021301, 0.009890737943351269, -1.310197114944458, 0.33402886986732483, 0.5182516574859619, 1.3930190801620483, -0.5800811052322388, 0.3956138789653778, 2.098142623901367, -0.34563836455345154, -0.3843082785606384, 0.5018574595451355, -0.5246061086654663, -0.014510304667055607, -0.5280489921569824, -0.8127743005752563, 0.873835563659668, -0.3993484377861023, -1.0392725467681885, -0.311465322971344, -0.16472668945789337, 0.28242820501327515, 0.9269111752510071, 1.7735437154769897, -1.2672785520553589, -0.2756551504135132, -1.8735724687576294, 0.19395290315151215, 0.4982829689979553, -0.6966447830200195, -0.9639397859573364, -1.0452154874801636, -1.5603387355804443, 0.5948299169540405, 1.8879982233047485, -1.2031261920928955, 1.6032475233078003, -0.16928572952747345, -1.4659799337387085, 0.5120036602020264, 0.707841157913208, 0.33625680208206177, -0.3289906680583954, 1.6249445676803589, -0.3556702136993408, 0.5742908120155334, 0.5873076319694519, -0.979933500289917, -1.949404239654541, -1.830085277557373, -0.48046889901161194, 0.2818545699119568, -1.8850449323654175, -0.8135586977005005, 1.470281958580017, -1.9059646129608154, 0.38231244683265686, 0.2689759135246277, 0.710246741771698, -1.5820320844650269, 0.25058460235595703, -1.5030516386032104, 1.259168267250061, -0.971124529838562, 0.28988388180732727, -0.9073076844215393, 0.04323318600654602, -1.6539281606674194, 0.1895781308412552, 0.5127954483032227, -0.11008884012699127, 1.5392088890075684, 0.27489832043647766, 0.6973913311958313, 0.4140518605709076, 1.8035929203033447, -0.7651493549346924, -0.3722771108150482, -0.31940120458602905, 0.5307744741439819, 0.17046454548835754, 0.7538608908653259, 0.20826905965805054, -0.614450991153717, -0.15290267765522003, 0.3557981550693512, -2.4089739322662354, 1.3274887800216675, -0.6586859226226807, -0.4417263865470886, 0.18642425537109375, -0.33076050877571106, -1.5513845682144165, -0.3046495020389557, 1.3409161567687988, 0.9983112812042236, 1.192882776260376, -0.6232307553291321, -0.2794046998023987, 1.0149625539779663, 1.4574615955352783, -0.01902424730360508, -1.034990668296814, -2.014125347137451, 1.4032734632492065, -1.5934399366378784, 0.4007355868816376, 0.8737286925315857, 3.293656587600708, 0.165798157453537, -0.1427440643310547, 0.4477011263370514, 0.5021828413009644, 1.3561495542526245, 0.6894013285636902, -0.7972708344459534, -2.1752102375030518, 0.29329758882522583, -0.7041776776313782, 1.439232349395752, -1.3562283515930176, 0.20056940615177155, -0.7563377618789673, -0.9060184359550476, 1.7109003067016602, 0.17168456315994263, -0.18815480172634125, 0.7606294751167297, -0.8097084760665894, -0.3796728253364563, -0.6578619480133057, 2.042267322540283, 0.3399350345134735, 0.6481192708015442, 1.150268793106079, -0.19239802658557892, -0.16039803624153137, -1.690752625465393, 0.493153840303421, 0.2656588852405548, -0.4273357093334198, 1.2419788837432861, -1.8489665985107422, 0.4308423399925232, -1.4276734590530396, -1.1860284805297852, -0.22169236838817596, 1.3743988275527954, -1.2347182035446167, -1.0384308099746704, -1.25812566280365, 2.0791664123535156, 0.5366870760917664, 0.41532546281814575, 2.1341354846954346, 0.25400543212890625, -2.1830615997314453, -0.5280473828315735, -1.2447540760040283, 2.0952446460723877, -1.4729804992675781, 0.3100181221961975, -0.539620578289032, -0.46294301748275757, 0.01605016365647316, -0.1454448699951172, -0.7164522409439087, -1.2891932725906372, -0.6830203533172607, -1.0910181999206543, -0.9495899677276611, 0.11488600075244904, -0.16705387830734253, 0.4878608286380768, -0.2565340995788574, 1.98790442943573, -0.7698174118995667, -1.0494575500488281, 1.0539101362228394, -0.744595468044281, 0.11776109039783478, 1.005301594734192, -0.1142963245511055, -0.38148102164268494, 0.9936009645462036, 0.33223679661750793, -0.5661107897758484, -1.2590702772140503, -0.4079930782318115, -1.7380188703536987, -0.9224035143852234, -1.1545697450637817, 0.28946781158447266, 0.7758850455284119, -0.6944417357444763, -0.47600167989730835, -1.1635700464248657, -1.45587158203125, -0.4699931740760803, 0.5646513104438782, 0.4658673107624054, 0.5901614427566528, -0.30599918961524963, -2.8606226444244385, 0.9418735504150391, 1.3517080545425415, 0.9267686605453491, 0.918768584728241, 1.1214500665664673, 2.769683361053467, 0.32091131806373596, 1.604892373085022, 0.831809937953949, 0.6970983147621155, -0.14309222996234894, -0.3627842366695404, -1.173919677734375, 0.6045864224433899, -0.32904183864593506, -1.251124382019043, -0.7932021617889404, -1.2373532056808472, 1.657955527305603, -0.2588428556919098, 0.10751022398471832, -1.2631741762161255, -1.0777074098587036, 1.0853383541107178, -0.24821606278419495, 0.22677087783813477, 1.9037195444107056, 2.1684157848358154, 0.7132322788238525, 0.5040022134780884, 0.2436208873987198, -0.29817768931388855, 2.101930618286133, -1.2637991905212402, -0.09444450587034225, -1.9996432065963745, -1.3159613609313965, 0.41797611117362976, 0.47379565238952637, 0.34948936104774475, -0.33738651871681213, -0.7354851365089417, -0.5605399012565613, -0.3985370993614197, 1.1365535259246826, 1.515030860900879, 0.4593721032142639, -0.5600021481513977, -0.9738859534263611, 0.31604811549186707, -0.23779690265655518, -1.8960106372833252, 1.4243413209915161, -0.008583035320043564, -0.3314610719680786, -0.9046298861503601, -0.9086011052131653, 0.934103786945343, -0.9809394478797913, 0.09944872558116913, 0.6594175100326538, 1.9164307117462158, 0.6280729174613953, -0.035543978214263916, -0.4572617709636688, 0.8893720507621765, 1.0439059734344482, -0.4426550269126892, 0.6105468273162842, -0.1436976045370102, -1.686521291732788, -1.0464136600494385, 0.8411470651626587, 1.4986090660095215, 0.7053757309913635, -0.21011368930339813, -0.7444566488265991, 0.5577568411827087, 0.3439517319202423, -1.6812189817428589, 0.6770060062408447, -0.7349183559417725, 0.6983442902565002, 0.5583033561706543, 0.7546725869178772, 1.114040493965149, 0.7328285574913025, -1.0901414155960083, -1.057875394821167, 1.0810468196868896, 0.49670085310935974, -1.6323219537734985, -2.08496356010437, 0.930036723613739, 2.3133718967437744, 0.8620755076408386, -0.24423688650131226, 0.28050634264945984, -2.8282532691955566, -1.034661889076233, 1.0564669370651245, 0.8981563448905945, -2.2078051567077637, 1.232280969619751, 0.5591688752174377, 0.9769022464752197, -0.20694513618946075, 0.007602143567055464, -1.1169463396072388, 0.3367112874984741, 0.9348941445350647, -0.974236249923706, 0.3841854929924011, -0.549252986907959, 1.0872433185577393, 1.0224074125289917, -2.4502804279327393, 0.47412246465682983, 1.0586367845535278, -0.578971803188324, -0.22136518359184265, -0.15436728298664093, -1.1877418756484985, -0.11791444569826126, 1.6200151443481445, 0.49924466013908386, 1.5867520570755005, 1.035065770149231, 1.2518367767333984, 0.1309123933315277, 0.8099439740180969, 2.100221872329712, 1.5347427129745483, 0.3401186764240265, -0.7217456102371216, -0.9549883008003235, 0.25375285744667053, 0.46012696623802185, -1.9045817852020264, 0.2322242259979248, 0.3651328682899475, 0.3184094727039337, -0.5668579936027527, 2.398263454437256, 0.9107897877693176, -1.422834038734436, -0.7405896186828613, 0.8339025378227234, 0.5329341292381287, 0.1318827122449875, -0.03434879332780838, -0.22626784443855286, -0.47137749195098877, -1.0728224515914917, 0.5879554748535156, -2.8986146450042725, -0.7740153074264526, 1.2674952745437622, -0.8363261818885803, -1.2632614374160767, 1.939246416091919, 1.0507760047912598, -0.18549136817455292, -0.6394923329353333, -0.27503010630607605, -0.39684250950813293, -0.28854236006736755, -0.8049989938735962, 0.5216492414474487, -0.6191295385360718, -1.9232642650604248, -1.3860514163970947, -0.04642735421657562, -1.1060513257980347, -2.617316484451294, 1.501259684562683, 0.45867031812667847, -0.5503271222114563, -0.6016889810562134, 0.46896231174468994, 1.3587477207183838, -0.7107702493667603, 0.3593028783798218, -0.7150284647941589, 0.2684755325317383, -1.249068260192871, -1.9662644863128662, 0.01761493645608425, -0.1395605057477951, 0.6751620769500732, -0.7658265233039856, 1.3478296995162964, 0.024913568049669266, 1.5240705013275146, -0.4270039200782776, 1.488090991973877, -0.14079385995864868, 0.07266763597726822, 0.9252985119819641, -1.9058650732040405, -1.1777334213256836, 1.8262896537780762, -1.634458065032959, -0.5896575450897217, 0.49004697799682617, 1.4412212371826172, -0.5967857241630554, 0.06297177076339722, -1.2414945363998413, -0.46837958693504333, 0.9917968511581421, 0.26963186264038086, -0.10051772743463516, 0.3737700879573822, 0.07718806713819504, 0.11133663356304169, -0.42843395471572876], [-1.075048565864563, 0.63822340965271, -1.2527737617492676, 0.9953588247299194, 2.1428661346435547, 1.5107449293136597, 2.3416497707366943, -1.1315574645996094, 0.19905848801136017, -0.9074109196662903, -0.8146839141845703, -1.8233400583267212, -1.3025991916656494, 0.8398433923721313, 0.6793009638786316, 1.203513264656067, 1.1259746551513672, 0.7732988595962524, -0.8645650744438171, -0.15018461644649506, 0.7290703058242798, 1.2627625465393066, -0.41189515590667725, -0.5328701138496399, -0.47255831956863403, 1.6263442039489746, 0.5661053657531738, -0.3423862159252167, -0.8065356016159058, 0.43704771995544434, -1.7451603412628174, -0.24539174139499664, -0.597165584564209, -0.611206591129303, 0.20146410167217255, 0.4217239320278168, 3.1294360160827637, 0.10353605449199677, 0.7172001004219055, -0.8291743993759155, -1.7049236297607422, 1.1798592805862427, 2.0261623859405518, -1.0221329927444458, -0.06696243584156036, 1.1544313430786133, 0.10801086574792862, -2.4552626609802246, 1.053354024887085, -0.13317951560020447, 0.7614005208015442, -0.9783194065093994, 0.9237542152404785, 1.0846787691116333, -0.727597177028656, 0.5229985117912292, -0.6384115815162659, -0.5028989911079407, -0.9179773926734924, -0.42985799908638, -0.6811418533325195, 1.2215508222579956, -0.21655601263046265, -1.1424050331115723, 1.4251651763916016, 0.5751367807388306, 1.0350061655044556, -0.04774909093976021, 0.7363600730895996, -0.08048992604017258, -0.9599102735519409, 0.13179528713226318, 0.17174561321735382, 0.16391243040561676, 0.8897827863693237, -1.7543033361434937, 0.3310903012752533, -0.7915576696395874, -0.2590629458427429, 0.488852322101593, 0.09934353083372116, -0.6553767919540405, 0.8129527568817139, -0.01988081820309162, 0.9991669058799744, 1.307692289352417, -0.32201722264289856, -1.0801235437393188, 0.13046401739120483, 0.8838015198707581, 0.37683209776878357, -1.1466997861862183, -1.6855201721191406, -0.9114753603935242, -0.9198753833770752, 0.6909968256950378, 0.11862365901470184, 0.9504358172416687, -0.03073877841234207, 1.7700449228286743, 0.07070908695459366, 0.6264184713363647, -0.3578101396560669, 0.5196537375450134, 1.1745411157608032, 0.8568023443222046, 1.3706889152526855, -0.5522089004516602, 0.5336425304412842, 1.8128997087478638, 0.3275827467441559, -0.0969712883234024, -0.1642497330904007, -0.464912086725235, 0.21931138634681702, 1.490288257598877, 0.5335245728492737, 0.04617292433977127, -0.7379665970802307, -1.0879125595092773, 2.5010814666748047, -0.4083156883716583, 0.17022646963596344, -0.42582041025161743, 0.23785382509231567, -0.3699093461036682, 0.17222860455513, 1.5378029346466064, -0.6907739043235779, 0.5964619517326355, -0.012028866447508335, 0.7870753407478333, -0.6381882429122925, 0.4642443358898163, -0.9482086896896362, -1.8067082166671753, -1.294722318649292, -0.9292933344841003, -1.1087137460708618, -0.30254927277565, 1.14255690574646, -1.1345710754394531, 0.47121214866638184, 0.16733068227767944, -0.7040384411811829, -0.9359508752822876, 0.4418719708919525, 0.4932915270328522, -0.34583714604377747, -0.14693519473075867, -0.005420089699327946, -1.3250213861465454, 0.11176911741495132, 0.31139612197875977, 1.7775471210479736, -1.2865819931030273, -0.877257764339447, 0.0858624130487442, -0.5916534066200256, -0.40923014283180237, 0.007817387580871582, -0.2966357469558716, -0.033919256180524826, 0.4440985918045044, -1.6335192918777466, 0.5664440989494324, -2.2656149864196777, 1.7177364826202393, 0.9356186985969543, -0.6099986433982849, 1.0294194221496582, -0.14371681213378906, 0.44219768047332764, 0.7676122188568115, 0.3576658368110657, 0.010484517551958561, 0.6346647143363953, -0.18099136650562286, -0.36767616868019104, -0.9409578442573547, -1.8045413494110107, 1.60584557056427, 0.7982544302940369, -1.392777919769287, 0.4129672348499298, 0.22455714643001556, -2.194443702697754, -0.8200986385345459, -0.0878778025507927, 1.7431988716125488, 1.118430495262146, 1.2829214334487915, 0.3384537994861603, 1.02362859249115, -1.156285047531128, 0.6414597630500793, 0.7556569576263428, -1.244286060333252, 0.2781482934951782, 1.5545119047164917, 1.9610120058059692, 1.4507445096969604, -0.6762915849685669, -0.05813281610608101, -0.48448294401168823, 1.2697128057479858, -0.8103867173194885, -1.0137444734573364, -1.587047815322876, 0.2358672022819519, 0.47559940814971924, -0.5610063076019287, -0.7183217406272888, -0.14615167677402496, 1.389201045036316, -1.305289626121521, 0.0013890627305954695, 0.2542071044445038, 0.42005255818367004, 1.3054454326629639, 0.8781855702400208, 0.3222116231918335, 1.5139625072479248, 1.1937681436538696, 0.34004923701286316, 0.287124902009964, -1.1359184980392456, 0.18073056638240814, -0.7350806593894958, -1.0919853448867798, -0.22658458352088928, -1.680747151374817, -0.8449296355247498, -0.47121432423591614, 0.3778185546398163, -0.2811327278614044, 0.43323978781700134, 1.6213853359222412, 0.8545868992805481, -1.1251932382583618, -1.7521800994873047, -0.8937216401100159, -0.8692778944969177, -1.968169927597046, -1.710193395614624, -1.7114152908325195, -0.6023600697517395, 1.4241218566894531, 1.7001700401306152, -1.0392520427703857, -0.4822259843349457, -1.402772307395935, -1.6681102514266968, -0.11685872822999954, -0.004366234876215458, -0.6912798881530762, 0.567509651184082, -0.4027634859085083, 1.017234206199646, -0.5783166885375977, 0.7761110663414001, -0.2839326560497284, 0.04075141251087189, 0.5383037328720093, -1.4240483045578003, -0.8297860622406006, -0.15960554778575897, -0.36899423599243164, 0.16789604723453522, -0.901286780834198, -1.4218040704727173, -0.19956038892269135, 0.1947323977947235, -0.8998470902442932, 1.122774600982666, 0.603072464466095, 0.7609068155288696, 0.6984923481941223, -0.9397217631340027, 2.049518585205078, -0.7793891429901123, 0.3184286952018738, 0.6760847568511963, 0.2927044630050659, 0.15510091185569763, -0.8365341424942017, -0.41523051261901855, 0.02215041220188141, 1.297265648841858, -0.6578723788261414, 0.696243166923523, 0.45191800594329834, 0.10751888155937195, -0.5220803618431091, -0.10516073554754257, -1.6870442628860474, 0.15506474673748016, 0.37995073199272156, -0.456035315990448, 0.5925941467285156, 0.3873142898082733, -0.6503883600234985, 0.9180743098258972, -0.44890594482421875, 2.495239734649658, -0.939577579498291, -0.7195900082588196, -0.5466392040252686, 0.3272325396537781, 1.2496540546417236, 0.18098397552967072, -0.3111860752105713, 1.9891071319580078, 1.1078020334243774, -0.19355271756649017, -0.027921944856643677, -1.4480386972427368, 0.11891230195760727, 1.798473834991455, -1.0865695476531982, -1.3405897617340088, 0.20577797293663025, -2.320896625518799, 1.4583220481872559, -1.4470795392990112, -0.6617780923843384, -0.10310116410255432, 1.2512719631195068, 0.4416014552116394, -1.8541611433029175, -1.6506880521774292, 0.2344156950712204, -2.068007230758667, -0.27538442611694336, -0.812221109867096, 0.8076767325401306, 1.2079275846481323, -2.5120229721069336, 0.42507532238960266, 0.6323766708374023, -0.7769470810890198, -0.6133223176002502, -1.2170047760009766, -0.36564919352531433, 0.3826982080936432, 1.313394546508789, 1.4115649461746216, -0.3144209682941437, -0.8625769019126892, 0.965941846370697, -2.7951712608337402, -1.4373412132263184, 0.604172945022583, 2.1898550987243652, -0.8220074772834778, 0.06294840574264526, -1.154808521270752, -0.1125173345208168, -0.11478716135025024, -0.7930143475532532, 0.3563525974750519, -0.20422978699207306, 0.4740638732910156, -0.23236599564552307, -1.234588384628296, 0.11370114982128143, 0.8318167924880981, -0.8111926317214966, 1.3190758228302002, 0.4305337965488434, -0.3789007067680359, 0.01756208948791027, -0.4366312325000763, -0.9168868064880371, 0.9046016335487366, 0.8903372287750244, -0.4643664062023163, -0.19497232139110565, -0.22038643062114716, 0.6865158081054688, 1.4569350481033325, -0.6681774258613586, 0.3016127645969391, -0.026155713945627213, 0.555271565914154, 1.4411557912826538, -0.8764351606369019, -2.061659336090088, -0.1684981733560562, -1.6012626886367798, 1.6766579151153564, -0.6937276124954224, -0.27694952487945557, 0.24981504678726196, 0.2155114710330963, -0.41337960958480835, 0.009026159532368183, 0.9583737850189209, -0.29745420813560486, 1.866323709487915, 0.37002265453338623, -2.3407201766967773, 0.9085569381713867, -1.7415587902069092, 0.35563310980796814, 2.279893636703491, 1.1001352071762085, -1.1123042106628418, -0.7926363348960876, 0.042897216975688934, 1.4778647422790527, -0.7088145613670349, 1.1337096691131592, 0.33421483635902405, -0.07489122450351715, -0.1389925181865692, 1.710214614868164, -1.4812196493148804, -1.1097111701965332, 0.9266383051872253, 1.0207372903823853, -1.2755413055419922, -0.8945516347885132, -1.1286672353744507, 0.5768077373504639, -0.6323090195655823, 1.4515880346298218, 0.7966792583465576, 0.2919238805770874, 0.2266910970211029, 0.5681440234184265, 0.8704697489738464, 0.03643528372049332, 0.2811005413532257, 0.8120283484458923, -0.04643815755844116, 1.0235629081726074, -0.1156497448682785, -0.7478850483894348, 0.20969557762145996, 1.6780147552490234, 1.5199753046035767, -1.582150936126709, 0.32524341344833374, -0.4802497327327728, -0.31170156598091125, -0.3323873281478882, -0.2193814069032669, -0.5157130360603333, -0.2698272466659546, 0.22915546596050262, -1.200433611869812, -1.2317246198654175, -1.1181411743164062, -1.132944941520691, 0.5965343117713928, -1.1064200401306152, -0.08018188923597336, 0.06325075775384903, 0.7153446674346924, 0.027215557172894478, 1.05536949634552, -0.059119898825883865, 0.24884887039661407, 0.7594911456108093, -1.4602408409118652, 0.3888775408267975, 1.5116913318634033, 0.11135940253734589, -0.2434501051902771, 1.9028993844985962, 0.10540128499269485, 1.234095573425293, -2.376509666442871, 0.10502316802740097, -0.5026662349700928, 0.38459187746047974, 1.1841169595718384, -0.14850111305713654, -0.22374920547008514, -2.820676565170288, 0.7887475490570068, 1.3310788869857788, 0.1515267789363861, -0.47520700097084045, 0.3672332465648651, 1.0364694595336914, 0.6486294865608215, 2.082751512527466, -0.5571320056915283, -1.3570626974105835, 0.49319374561309814, 1.2727442979812622, -0.4543045163154602, 0.00810531061142683, -0.640387773513794, -1.2548507452011108, -0.7337090373039246, -2.066926956176758, 0.27231284976005554, 0.46209532022476196, 0.8858374357223511, -0.8296325206756592, -0.6410772204399109, -1.6835861206054688, -1.4620620012283325, 0.8851058483123779, 2.8181865215301514, -1.0630215406417847, -0.4021267890930176, 0.9247698187828064, -0.7033829689025879, 0.5762680172920227, -0.30933821201324463, -0.3068234622478485, 0.8990276455879211, -0.26121285557746887, -1.2751420736312866, 0.8171394467353821, 0.018823761492967606, -0.5605177879333496, -0.8107805848121643, 0.5738301873207092, -0.7260957956314087, 0.11494380980730057, 0.996856153011322, -0.11418977379798889, 1.0850428342819214, -1.3847932815551758, 0.6034656167030334, 0.5276445150375366, 1.8699698448181152, 0.016824783757328987, 0.8918395042419434, -0.16264383494853973, -1.0650142431259155, -0.8826722502708435, -0.7101952433586121, 0.7751078009605408, -0.7822225689888, 0.0390169695019722, 0.8978480696678162, 1.2405638694763184, 0.9366365075111389, 1.4915721416473389, -1.1350641250610352, -0.4893532991409302, 0.3170454204082489, 1.114730954170227, 0.5029048323631287, -0.7674216032028198, -0.1902296245098114, -1.651578664779663, 0.8075640201568604, -1.786780834197998, 0.21576359868049622, 1.0769767761230469, 0.9038225412368774, -1.6122119426727295, 0.21409814059734344, -0.34767529368400574, -0.2968983054161072, 1.642608880996704, 0.28728264570236206, -1.4324098825454712, -1.079689383506775, 0.7099534273147583, 0.2247367650270462, -1.1357351541519165, -1.6788519620895386, -0.5573149919509888, -0.4441302418708801, 0.8121137022972107, -0.6177338361740112, 0.6607650518417358, 1.2025130987167358, -1.6137391328811646, -0.5980636477470398, 1.5388896465301514, -0.4610358476638794, -0.031306084245443344, 0.9480972290039062, -0.7794435024261475, -0.4357558786869049, -1.6774156093597412, 0.5296189188957214, -0.5158726572990417, -0.39977046847343445, -2.6065094470977783, 0.9347922801971436, 0.6579669117927551, -0.8335344195365906, 1.187525749206543, -2.4954049587249756, -0.11321136355400085, -1.73336923122406, 2.679318428039551, -2.668282985687256, -1.2742853164672852, -0.8150474429130554, 0.7482399344444275, 0.7725623846054077, 0.32697775959968567, -0.31227245926856995, 2.0796029567718506, -0.7877261638641357, 0.2467808574438095, 1.6861928701400757, -1.3054146766662598, -0.8808966875076294, 1.0356192588806152, 0.3859933614730835, -1.3872534036636353, -0.03912284970283508, 1.8571701049804688, -0.33031871914863586, 0.7347524762153625, 0.053481049835681915, -0.7659477591514587, 1.5432429313659668, 0.8973816633224487, 1.125185489654541, -0.22835735976696014, 0.0249031949788332, 0.1473134160041809, 0.3698478043079376, -0.4271032512187958, -0.5859496593475342, 1.234837293624878, -1.1874022483825684, -0.5280911326408386, -0.4637366831302643, -0.6006788611412048, -1.2985363006591797, -0.8573467135429382, 0.718403160572052, -0.4493582546710968, -0.8300719857215881, 1.5873746871948242, -1.4279767274856567, 0.952655017375946, 0.2660783529281616, -1.447710633277893, -0.49283432960510254, -0.4795284867286682, 1.2092522382736206, 0.8314024806022644, -0.369962215423584, -1.2715821266174316, -1.7904183864593506, -0.26150646805763245, -1.4990588426589966, -0.26957932114601135, -0.29795560240745544, 0.2862878441810608, 0.9127605557441711, 0.0176800899207592, 0.38158831000328064, -0.2642515301704407, 0.9530856013298035, 0.7274221777915955, 1.509709119796753, 1.2925992012023926, -0.8234419226646423, -1.5577775239944458, 0.5540980696678162, 1.0588551759719849, -0.7655169367790222, 1.4154642820358276, 0.006585346069186926, -0.1804799735546112, 0.26538434624671936, -1.5049426555633545, -1.747976541519165, -0.4461745619773865, -1.2806291580200195, -0.12209348380565643, -0.11171415448188782, -2.687425374984741, -0.9193198084831238, 0.4126332402229309, -2.4508540630340576, -0.45876380801200867, -1.4796396493911743, -2.0939011573791504, 0.9936670064926147, 0.9544943571090698, -0.552094578742981, -0.6313082575798035, 1.2621461153030396, 1.071995496749878, -0.14278019964694977, 0.73748379945755, -0.02943258360028267, -0.4482945501804352, 0.4842562675476074, -0.5648497939109802, 0.4548114538192749, -0.38470274209976196, -0.5965983271598816, -0.709530234336853, 0.31721562147140503, 0.7997533082962036, 0.28972935676574707, -1.0098627805709839, 0.47900187969207764, -1.0384902954101562, -1.2660861015319824, -0.33873629570007324, 0.6730307936668396, 0.5532600283622742, 0.42745715379714966, 1.2466511726379395, 0.7345409989356995, -0.4853536784648895, 0.659192681312561, -0.3920721709728241, 0.2756240963935852, 0.47298920154571533, -0.18497514724731445, -1.5960417985916138, -0.252077579498291, -0.4120783805847168, -0.2094714194536209, -0.5181533694267273, -1.2182109355926514, -1.6944867372512817, 0.5715779066085815, 0.6975346207618713, 0.28831106424331665, -0.12855735421180725, 1.8688393831253052, 0.5832528471946716, -0.05846976861357689, 0.7500482201576233, -0.14400997757911682, 0.6268437504768372, 0.20921814441680908, 0.4674629271030426, -0.003215221455320716, -0.5132883787155151, 0.44411078095436096, 1.0559078454971313, -0.9668839573860168, -0.5831406712532043, -1.1383634805679321, 0.8165098428726196, -0.27112653851509094, 0.8316695690155029, -0.8371488451957703, 0.7700929641723633, -0.6177929043769836, -0.5063008666038513, 0.48772376775741577, -0.4324682950973511, -0.6368050575256348, 0.8417524695396423, -0.10867390036582947, 1.1875962018966675, 2.4681990146636963, -0.5987967252731323, -0.37760624289512634, -1.5491162538528442, -1.212232232093811, 0.5549950003623962, 0.25763005018234253, 0.23611146211624146, 0.41737326979637146], [-0.8738108277320862, -1.7078951597213745, 1.0351831912994385, 0.20803223550319672, 1.0867526531219482, -0.06653641164302826, 0.7150542736053467, -0.5564828515052795, -0.03925619274377823, -0.19601891934871674, 0.3348346948623657, 1.3506625890731812, 0.6063337922096252, -2.8945705890655518, -0.13561637699604034, -0.024130836129188538, 0.7646659016609192, 0.10819713771343231, -0.7249911427497864, -0.8079013228416443, -0.36362096667289734, -0.7309289574623108, 1.3778239488601685, 0.5670459270477295, 0.02594025433063507, 0.9307919144630432, 2.103410243988037, -0.23275578022003174, -0.05549164116382599, -0.32208940386772156, -1.2405705451965332, -0.6487307548522949, -0.9483420252799988, 0.4506577253341675, 0.42232683300971985, 0.840303361415863, -0.7878500819206238, 0.9586920142173767, -0.9866753816604614, -0.20206882059574127, 1.254223346710205, 1.2005698680877686, 1.057490587234497, 2.0672218799591064, 1.0181862115859985, -0.25948566198349, -0.22855058312416077, -0.7760991454124451, 0.2225039005279541, -0.753409206867218, -0.6217378377914429, -0.9786962866783142, -1.6783263683319092, 2.4492745399475098, 0.8737998008728027, -2.0575733184814453, 0.4345192015171051, -0.5584351420402527, -2.1215178966522217, -0.21548084914684296, 0.20568832755088806, 0.5677739381790161, 1.0889240503311157, -0.8428160548210144, -0.05281146243214607, 0.6563656330108643, 0.5351579785346985, 0.38305604457855225, 0.09684956073760986, 1.221562385559082, -1.2456209659576416, -1.4770063161849976, 1.7225122451782227, 0.4210421144962311, 1.1057555675506592, 0.2322312593460083, -0.35190078616142273, -0.4424501657485962, 0.15168914198875427, -0.45110252499580383, -0.052263882011175156, -0.06247442215681076, 0.8320020437240601, 0.8079757690429688, 1.6054329872131348, -0.673474133014679, -0.00913101527839899, 0.9834729433059692, 0.3833865225315094, -2.3714606761932373, 0.400983989238739, 0.03660105541348457, -0.4223555028438568, -0.361738383769989, -0.6592142581939697, 1.2046122550964355, 0.7354822754859924, 0.27065595984458923, 1.167049527168274, 0.9891257882118225, -0.7435221076011658, -0.9665509462356567, -0.9225010871887207, 0.817533552646637, 0.06272897124290466, 1.2783334255218506, 0.6021775603294373, -0.10893407464027405, -1.1474075317382812, -0.1649673581123352, 2.2939345836639404, -0.5807489156723022, -0.7111479640007019, -0.11607719957828522, 1.4377297163009644, 0.06134083494544029, -0.9854557514190674, -0.4985949695110321, -0.03217325732111931, 0.18070727586746216, 1.1025962829589844, 2.555060625076294, 1.4432904720306396, 0.7302070260047913, 0.08171055465936661, -0.6862578988075256, 0.671830952167511, 1.3619053363800049, 0.914198637008667, -0.022776903584599495, 0.760011613368988, -0.11526306718587875, -0.05489616096019745, -0.12569276988506317, 0.17281150817871094, 0.019008127972483635, 1.0908279418945312, 0.09707535803318024, -0.0019835426937788725, 1.114388108253479, 0.7662209272384644, 1.1958239078521729, -0.660757839679718, 1.3718549013137817, 0.07610747218132019, -0.948220431804657, -0.07613930851221085, -1.1322195529937744, -0.9030418395996094, -0.3349325954914093, 0.9994789958000183, -1.2461342811584473, 0.3410845100879669, -0.19739219546318054, 1.578460931777954, 0.24653314054012299, 0.6430248618125916, 1.4271037578582764, -1.0152946710586548, -0.2821561396121979, -1.4042589664459229, -0.7726093530654907, 0.35843759775161743, -0.1217627301812172, 0.4331483542919159, -0.3546072542667389, -0.6394955515861511, -0.10496386885643005, -0.8295284509658813, -1.0615475177764893, -1.5288280248641968, -0.11316187679767609, 0.4298005700111389, -1.192842960357666, -0.4628010392189026, -2.2494418621063232, -1.1835840940475464, 0.031936440616846085, -1.3007732629776, 0.38172438740730286, -0.436742901802063, 0.30753961205482483, -0.5876467227935791, -1.4339147806167603, -0.074213407933712, 0.6420187950134277, -0.5649855732917786, 0.3142373263835907, -0.747180700302124, -0.39599931240081787, 1.1466668844223022, -0.0828341692686081, -0.9701007604598999, 0.47163844108581543, -0.013071060180664062, -0.5749338865280151, 0.6090652346611023, -2.872256278991699, 0.49623391032218933, 1.7820613384246826, 0.2300073355436325, -1.8874317407608032, 1.2584893703460693, 2.380866527557373, 0.5041162967681885, -1.0811549425125122, -0.7012913227081299, -0.17287307977676392, 0.13843725621700287, 1.0934730768203735, 0.5030826926231384, -2.2223808765411377, -0.334605872631073, -0.18027877807617188, 1.7615790367126465, -0.5988226532936096, -1.3843231201171875, -0.3337409496307373, 0.22311042249202728, -1.817671298980713, 1.014750361442566, -0.16800162196159363, 0.7610911130905151, 0.6285060048103333, 0.4815693199634552, 1.3205077648162842, 1.5786718130111694, -0.427695095539093, -0.10159711539745331, 0.9764545559883118, 0.12282758951187134, -0.6075292229652405, 0.0664922297000885, -0.8569101691246033, -0.5155556201934814, -3.3589019775390625, 0.10519295185804367, 1.7618622779846191, 0.683835506439209, 0.6506158709526062, 0.15740159153938293, -0.7548139691352844, 0.464645117521286, 0.04016491398215294, -0.1432207077741623, 0.22390463948249817, -0.5635124444961548, 0.11680827289819717, 0.6221003532409668, 0.9591206312179565, 0.12896700203418732, 0.7837471961975098, 0.1668056845664978, -2.211940050125122, -1.890803575515747, -0.2064514458179474, -1.0948420763015747, 0.5443927645683289, -1.2894481420516968, -0.4100140631198883, -0.5429268479347229, -0.7304664254188538, -0.5769822597503662, -0.03293001651763916, 0.0814148336648941, -0.5226753950119019, 0.1590309590101242, 0.5147376656532288, -0.1971539855003357, -0.6759917736053467, -0.6646979451179504, -0.47192010283470154, -0.5246622562408447, 0.07389518618583679, 2.6164917945861816, 1.0929549932479858, 0.03495863825082779, 0.5822433233261108, 0.5778390765190125, 1.2388113737106323, 1.3967636823654175, -0.9273389577865601, 2.1884920597076416, -0.8176535964012146, 0.10884109139442444, -0.7888332009315491, -1.636675238609314, -1.7185800075531006, 1.9511656761169434, 1.4759209156036377, -0.9319403767585754, -1.0577138662338257, 1.483432412147522, -0.906586766242981, 0.4818061590194702, 2.0228891372680664, 0.7802627682685852, 1.4910304546356201, 0.2527552545070648, -1.6233625411987305, -0.7611356973648071, -1.7109591960906982, 0.3474064767360687, -1.252945899963379, 0.15064486861228943, -0.4628445506095886, -0.6567877531051636, 0.5238730311393738, -0.441705584526062, -0.7945249080657959, 0.993445873260498, -0.022145099937915802, -1.4322373867034912, 1.1764404773712158, -1.3729610443115234, 0.23678578436374664, 0.5253486037254333, -0.25039178133010864, -0.8054301142692566, -0.3138764500617981, -0.3381282091140747, -1.0793348550796509, 0.8417276740074158, -0.5895808935165405, -0.7105648517608643, 1.350166916847229, -0.7239043712615967, -1.5054947137832642, -0.06826900690793991, 0.007401301059871912, 0.028562180697917938, 2.0474374294281006, -0.5515169501304626, 1.3226284980773926, 0.8693410158157349, 1.433271884918213, -0.0894344374537468, -0.30479031801223755, -0.18791262805461884, 0.3205343186855316, 0.9628768563270569, 1.643454909324646, -1.7214065790176392, 1.0744280815124512, 0.39189451932907104, 0.07276544719934464, -0.9940438270568848, 0.3661687672138214, -1.7617310285568237, 1.2563656568527222, -0.8892465233802795, -0.08782564848661423, -0.13432082533836365, 0.598170280456543, -1.2822297811508179, 0.8506312966346741, 0.4376862645149231, 1.5599100589752197, 0.328079491853714, -0.09637028723955154, -1.835825800895691, -0.33325815200805664, 1.656310796737671, -0.420222669839859, -0.458829402923584, -0.11094678193330765, 0.870471179485321, -1.7110354900360107, 1.8128650188446045, 0.6117827892303467, 0.7397779226303101, 0.020028116181492805, 1.972849726676941, -0.40096643567085266, 0.8347805738449097, 0.1333824247121811, -1.2139606475830078, -0.10354981571435928, 0.7374048233032227, -0.2437659651041031, -0.3699452877044678, 0.013190367259085178, -0.8258195519447327, -1.0631855726242065, 0.10156859457492828, -0.8947599530220032, 0.5937013030052185, 0.7349538207054138, 1.3038464784622192, 0.9938008189201355, -0.24017870426177979, 0.4435516595840454, -1.0992984771728516, 0.4556041955947876, -0.4578123092651367, 1.3687587976455688, -0.6559396982192993, 0.20132435858249664, -0.7234850525856018, 0.6464460492134094, 0.6584309339523315, -0.9566277861595154, 1.9638391733169556, 0.5200446248054504, -1.3717724084854126, -0.8615873456001282, -1.7716999053955078, -1.9491510391235352, -0.11367933452129364, -2.0300204753875732, 0.40172699093818665, -1.5871005058288574, -0.35297051072120667, 1.7136263847351074, -0.3297751843929291, -0.012211291119456291, 1.245577096939087, -0.29026588797569275, 0.5495078563690186, -0.7806057929992676, -0.353678822517395, 0.1473129242658615, 0.0291576087474823, -0.7383667826652527, 0.23870714008808136, 0.6116370558738708, 0.6464532017707825, 0.05340602621436119, -0.9185051321983337, 0.6898705363273621, 0.855553150177002, -0.96103835105896, -2.2622036933898926, -0.1785484254360199, 1.0126886367797852, 0.8648103475570679, -1.159956932067871, 1.3127034902572632, -0.8821960091590881, 0.9127151370048523, -0.4208749830722809, 0.9544802904129028, 0.4931268095970154, -0.23305730521678925, 0.6939300894737244, -0.6612278819084167, -0.38705703616142273, -0.32571080327033997, -0.858503520488739, 0.9029361605644226, 0.9057196974754333, -0.015065072104334831, -0.9739655256271362, 0.43798327445983887, 0.25791430473327637, 0.6024418473243713, -0.03472520411014557, -0.2459702342748642, 1.3492380380630493, -0.6283196210861206, -0.2164953500032425, -1.7914868593215942, -0.40334606170654297, -0.5087606310844421, -0.3698272407054901, 0.5387773513793945, -0.6374421119689941, 0.37466102838516235, -0.4149835407733917, -0.4844694137573242, -1.8764736652374268, 0.35192158818244934, 0.5640805959701538, -1.0262235403060913, -0.6486658453941345, -1.8523021936416626, -0.4296067953109741, -1.1653554439544678, 0.29080089926719666, -0.20787115395069122, 0.5006042718887329, -0.9212510585784912, -1.7760759592056274, -0.1808185577392578, -0.6868544816970825, 1.697546124458313, 0.1246882900595665, 1.3068982362747192, 0.41240981221199036, -0.42321187257766724, -1.6098606586456299, -0.33658942580223083, 0.5056493878364563, -0.024784626439213753, 0.22168323397636414, 0.19826757907867432, -1.415190577507019, 0.19847196340560913, -0.8919408321380615, -0.1436871886253357, 1.240815281867981, -2.0490798950195312, -0.35687878727912903, 0.23976729810237885, 0.9619829654693604, -0.031222224235534668, -0.9145708680152893, 0.6736761331558228, 0.46912869811058044, -1.833521842956543, -0.02852698601782322, -1.7765955924987793, -2.685892343521118, -1.0722161531448364, -1.0104820728302002, -2.18841814994812, 1.7617148160934448, -1.3748266696929932, 0.2945359945297241, 0.6439993381500244, 2.089850902557373, -0.3929203748703003, 1.5409775972366333, -1.1714773178100586, -0.07556214928627014, -0.43305790424346924, -1.0227315425872803, 0.304866224527359, 1.863934874534607, 2.072620153427124, 0.6504458785057068, -1.1441766023635864, -0.8223114013671875, 0.5802941918373108, 0.5515611171722412, 0.5414259433746338, -0.2892455458641052, -0.9182955026626587, 0.43425172567367554, -0.03982848674058914, 1.391278862953186, -0.7055122256278992, 0.8301284909248352, 1.3365111351013184, 1.9564088582992554, 1.940241813659668, -1.0338618755340576, -1.1828582286834717, 0.13868141174316406, -0.873724639415741, -2.3764071464538574, 1.4637596607208252, 0.49218249320983887, 1.02290940284729, 0.1841057538986206, 1.1669200658798218, 0.26629653573036194, 0.14660824835300446, 1.9176180362701416, -1.1499762535095215, 0.5419232249259949, -1.5757479667663574, 1.498006820678711, -0.22378072142601013, -1.2036187648773193, -1.8132925033569336, -0.22024032473564148, 1.9678069353103638, 0.493269681930542, -1.562528371810913, -2.045823812484741, 2.8292343616485596, -0.5814045667648315, -1.4836336374282837, 0.13049769401550293, 0.5720628499984741, 1.5748786926269531, -0.2120964229106903, -0.729774534702301, 0.6671910285949707, -0.8223658204078674, 0.3475848436355591, 0.16409263014793396, 0.9718093872070312, -0.7231408953666687, -0.6982526183128357, 0.27247098088264465, 0.4514361619949341, -1.493510127067566, -0.832598865032196, 0.15209339559078217, -0.3982197046279907, 2.2722296714782715, 0.6996307969093323, 1.2368419170379639, -0.09998886287212372, 0.06290191411972046, 0.09951411187648773, 1.048712134361267, -0.3510081171989441, 1.5643380880355835, -0.7610559463500977, 0.9865989089012146, 0.24904905259609222, -0.38538381457328796, 1.5785657167434692, 0.9697596430778503, -0.2012520283460617, -0.7913442254066467, 0.8187875151634216, -0.8263174891471863, 0.5343275666236877, -0.9226882457733154, 0.09031479060649872, -0.8862617015838623, -0.21281950175762177, 1.6474394798278809, 1.7315188646316528, -0.49466416239738464, -0.28933095932006836, -1.7013806104660034, 0.28266650438308716, 1.1210873126983643, -0.8716939091682434, -0.9341865181922913, 0.3421093821525574, -0.5889579057693481, 0.17577312886714935, 1.5467561483383179, 0.4522940516471863, 0.6687882542610168, 1.8848310708999634, 0.2359023541212082, 0.6884163022041321, 0.057088453322649, -0.7665666937828064, 1.6088340282440186, -1.4045636653900146, 0.07716149836778641, -1.1711769104003906, -0.8343628644943237, -1.3014222383499146, -0.2649272382259369, 0.4622752070426941, 1.30203378200531, -1.8987412452697754, 0.22351329028606415, -0.40686625242233276, -0.6295178532600403, -0.12120742350816727, 0.19561876356601715, -0.11122468113899231, -0.0021514075342565775, 2.192739963531494, 0.10439535975456238, 0.33992916345596313, -1.1502684354782104, 1.0938206911087036, -0.5242139101028442, -0.09611140191555023, -0.7749882936477661, -0.09399706870317459, -0.9330614805221558, -1.295305609703064, -1.398228645324707, -0.38501277565956116, 0.01382573414593935, 0.7211169004440308, 1.6588133573532104, 1.8620132207870483, -0.6334415674209595, -0.7081896066665649, 0.7583281993865967, -0.8759666085243225, 1.1321731805801392, 1.0547035932540894, -0.9419142603874207, 0.7710549235343933, 1.4606415033340454, 0.4808559715747833, -0.25728338956832886, -0.13486924767494202, 0.5092883706092834, -0.7009961605072021, 0.376225084066391, -0.19421398639678955, -0.6972314715385437, -0.9319680333137512, 0.5980426073074341, -0.8595236539840698, 0.264279842376709, -1.7991377115249634, -0.1533980816602707, 0.15202021598815918, 0.39423757791519165, 0.9823877215385437, 0.5642678141593933, 0.6005803346633911, -0.22460974752902985, -0.9871042966842651, -1.11186945438385, -0.7152895927429199, 0.28806474804878235, 1.049920678138733, 0.3930620551109314, 0.249283105134964, 0.8000555634498596, -0.2788122296333313, 0.22233089804649353, -0.06124002858996391, 0.25457003712654114, -0.1700826734304428, -1.2363481521606445, -0.33278411626815796, -0.7607414722442627, 1.761845350265503, 1.4637449979782104, -0.5750356912612915, -0.8096387982368469, 0.059427082538604736, -1.4493186473846436, 1.2714520692825317, -0.01864132285118103, -0.4347485601902008, 1.069745659828186, 1.3000767230987549, 0.17678508162498474, 1.024141550064087, -1.6244977712631226, -0.7306544780731201, -0.8146520256996155, -1.5887755155563354, 0.8758096098899841, 2.3954014778137207, -1.292073369026184, -0.5451972484588623, 1.0678068399429321, -0.2810017764568329, -0.0014359918422996998, -0.7871628403663635, 0.3350633680820465, -0.9584549069404602, 0.7529599070549011, 0.5210946798324585, 0.3749486804008484, -0.7552114129066467, -0.20959509909152985, 0.46702051162719727, -0.8824822306632996, -0.9687047600746155, -2.246901750564575, -0.5314314365386963, -0.15137438476085663, 0.9375474452972412, -1.195270299911499, -1.436255693435669, 1.1880325078964233, 1.0773248672485352, -1.1318925619125366, -0.4978930950164795, -1.811165452003479, -0.07387343049049377, 1.2701352834701538, 0.7744849920272827, -1.051837682723999, 0.27004778385162354, -0.7177926301956177, 0.30751317739486694], [-0.546955406665802, 0.6173134446144104, 1.2431395053863525, 0.31640195846557617, 0.3543555736541748, 0.3877732455730438, 0.8649905323982239, -0.41109782457351685, -0.16680438816547394, -0.14966659247875214, 1.305840253829956, 0.16333246231079102, 0.5672598481178284, -0.3570599853992462, -1.333630084991455, -0.29593223333358765, 0.4793241322040558, -0.29948288202285767, -0.5538923740386963, -0.35909542441368103, 0.5225579142570496, 0.4325774610042572, 0.009441223926842213, -0.9233226180076599, -0.6631020307540894, 3.570286512374878, -0.1595599353313446, 0.6182639002799988, 0.3098661005496979, -1.3553857803344727, 1.3121356964111328, 0.08807894587516785, -0.8749949336051941, 0.7565662264823914, -0.5514370799064636, 0.31906697154045105, -1.3012131452560425, -0.17265565693378448, 0.660244345664978, -0.47753599286079407, -1.3838521242141724, -0.8044763207435608, 0.6638932824134827, -1.5640664100646973, -1.3696116209030151, 0.7577922344207764, 0.1062699630856514, -0.052323486655950546, -0.4467623233795166, 0.37942981719970703, 0.8087664842605591, 0.4323456585407257, -1.732041835784912, 0.6127843856811523, -1.6365818977355957, 1.683210849761963, -0.4477090835571289, 0.5887224674224854, 0.3173922598361969, -0.637575626373291, 1.1734719276428223, 0.21714945137500763, 2.0935285091400146, 0.6572917699813843, 0.34146633744239807, 2.492621421813965, 1.3941092491149902, 0.040984924882650375, -0.7567098736763, 1.9340217113494873, 1.2497669458389282, 1.2493140697479248, -0.27568677067756653, -0.5556349158287048, -0.8355951905250549, 1.175079345703125, -0.10909204930067062, 0.697176456451416, -2.1045985221862793, 1.0617667436599731, -0.6707392930984497, -2.654589891433716, -1.237740397453308, 0.4359651207923889, 1.5310472249984741, -2.710397720336914, -1.344512939453125, 0.1391548365354538, -1.5407534837722778, -0.43253597617149353, -1.0329102277755737, 0.08004424721002579, -1.4884148836135864, -0.36509406566619873, -0.865286111831665, 0.1026621088385582, -0.09623314440250397, 0.9711006283760071, 0.20474882423877716, 1.7298612594604492, -1.6180338859558105, -0.2785531282424927, 0.7506858706474304, 0.08713827282190323, -2.6462433338165283, 0.30318209528923035, -0.5072449445724487, 2.439357042312622, -0.11781815439462662, 0.6676194071769714, -0.9634756445884705, 0.6814649105072021, -2.7957539558410645, -1.0296320915222168, -0.3909095227718353, -1.6114106178283691, 0.97292160987854, 0.006933615077286959, 1.650026798248291, -1.4236109256744385, -0.839648425579071, 0.6502833962440491, -0.512188196182251, 0.831801176071167, 0.6463663578033447, -0.7526623606681824, 0.12294923514127731, 0.5414919853210449, 0.24797643721103668, -0.20372332632541656, -0.48453640937805176, -0.22173917293548584, 0.9934295415878296, 2.080723762512207, -1.4162054061889648, -1.0058883428573608, -0.15636394917964935, 1.471055269241333, -0.8518376350402832, 2.3517117500305176, 2.3081536293029785, -2.3816113471984863, 0.702487587928772, 0.628286600112915, -0.40292420983314514, -1.844928503036499, 0.3021776080131531, -0.3693740665912628, 1.218359112739563, 0.6074314117431641, -0.9110127091407776, 2.0131146907806396, 1.892362117767334, 1.5539147853851318, 0.11474481225013733, 0.9943466186523438, 0.16560682654380798, -0.4895205497741699, -1.8240573406219482, 0.3438192903995514, -1.9332629442214966, 0.7804359197616577, -1.3083491325378418, 0.2848896086215973, -0.22053106129169464, -1.0875132083892822, -0.9770718216896057, -0.5174654126167297, 1.574944257736206, 0.44871628284454346, -0.20951911807060242, 0.8950755596160889, -1.1221411228179932, -1.4797409772872925, -0.8521938920021057, 1.094152569770813, -0.25451377034187317, 0.15299458801746368, 0.15020976960659027, -0.397347092628479, 1.2853150367736816, 0.7484840750694275, -0.8485654592514038, -1.0246504545211792, -0.030833039432764053, -0.9186582565307617, -0.00038168070022948086, 0.750859260559082, 0.06189509853720665, 0.19513818621635437, -0.5028058886528015, -0.8184067010879517, -1.0406965017318726, 1.31692636013031, -0.7820993661880493, 0.08159974962472916, 0.10883943736553192, 0.44854485988616943, -0.34249311685562134, -1.637119174003601, 2.1220216751098633, -1.0916191339492798, 0.6080378890037537, -1.1008683443069458, 0.6749172806739807, -1.0913232564926147, 1.7241355180740356, 0.46932414174079895, -1.6185442209243774, 1.716004490852356, -1.9333692789077759, -0.3497684895992279, 1.1288939714431763, -0.40299320220947266, 1.3060615062713623, 0.036716170608997345, -0.6142854690551758, 1.271960735321045, 0.036266110837459564, -0.49626216292381287, 0.9570521116256714, -0.273233026266098, -1.421105980873108, -0.7606175541877747, 2.5156099796295166, 2.3286781311035156, -1.6141629219055176, 0.8000585436820984, 1.2529182434082031, -0.3031063675880432, 0.6793004870414734, -1.0660061836242676, -0.36261045932769775, 1.0537540912628174, -0.08786819130182266, -0.18985199928283691, -0.2862972021102905, -1.4121413230895996, -1.0782663822174072, -0.5839353799819946, -0.17329417169094086, -0.9322655200958252, 1.9000128507614136, -1.8492469787597656, -1.1540828943252563, -0.708345353603363, 0.49857282638549805, -1.5290381908416748, 0.2203899472951889, -0.920688807964325, -0.859854519367218, 0.3991701900959015, 0.8085860013961792, 1.0212628841400146, -1.250342845916748, 1.023128628730774, 1.0429410934448242, 0.30733582377433777, 0.4388447105884552, 0.015101323835551739, -0.17662645876407623, 0.8669028878211975, 1.7064766883850098, -0.27521947026252747, 0.35839590430259705, 0.9557065367698669, -1.4407150745391846, 1.6849908828735352, 0.3810654580593109, 1.5009524822235107, -0.2847825288772583, -0.891306459903717, -0.13724027574062347, 0.32590821385383606, -0.942083477973938, 0.9572057723999023, 0.29764434695243835, -1.074100136756897, 0.5115475058555603, -0.13059723377227783, 0.062459830194711685, -0.3798774778842926, -0.7064502835273743, -1.2221592664718628, 0.1545306295156479, 0.0846627801656723, 0.5402213931083679, -0.6901994347572327, 0.49469950795173645, 0.9095367193222046, -1.1578755378723145, 0.2458556443452835, 2.319342851638794, -0.4839458465576172, -0.2907932996749878, -0.01719772070646286, -0.5131248235702515, 0.9138010144233704, -0.0656479224562645, -0.09169206768274307, -0.7893660664558411, 0.5728089809417725, -0.08502046018838882, -0.40456923842430115, -0.3139932155609131, 1.6392996311187744, 0.007254068274050951, 0.2850475013256073, 1.3913912773132324, -0.09627299755811691, 0.9927016496658325, -1.0264883041381836, 0.6821591854095459, -2.163881778717041, 0.15861396491527557, -0.43434834480285645, -0.7266392111778259, 1.1113063097000122, 0.760455846786499, 0.8844071626663208, 1.5432038307189941, 0.5457643270492554, 0.5949342846870422, -0.05530112236738205, 1.9313864707946777, 0.9157949686050415, -0.7917551398277283, -0.8811447024345398, -0.7560880184173584, -0.7535562515258789, 1.3189194202423096, 0.3058207333087921, 1.0828461647033691, -0.7783719897270203, -0.7866873145103455, 1.2410814762115479, -2.051985263824463, -1.2271124124526978, -2.162177085876465, -0.6988748908042908, 2.6182332038879395, -0.795486569404602, 1.0548917055130005, 1.6586053371429443, -0.6902245879173279, -0.7387391328811646, 0.32560694217681885, 0.6712690591812134, -1.6157554388046265, -0.8406364321708679, 1.6631064414978027, -0.8394145965576172, 0.36781546473503113, 1.0374282598495483, -0.3525281846523285, -0.19137704372406006, -0.5751323103904724, -0.6813009977340698, -0.6327389478683472, -1.6749026775360107, -0.5251346826553345, -0.23712575435638428, 0.713134229183197, -0.4400823712348938, -0.6851539015769958, 1.9100030660629272, -0.026851430535316467, -0.24558593332767487, -0.5448981523513794, -0.01831239089369774, -0.8920486569404602, -0.533396303653717, -0.881085991859436, -1.246883749961853, -0.46806973218917847, -1.4139965772628784, 1.73918616771698, -1.1884949207305908, 1.7720184326171875, 0.3122865557670593, 0.8840938806533813, 1.4663175344467163, -1.682665228843689, 1.6538702249526978, 0.49277955293655396, -0.49142083525657654, -0.7630104422569275, 0.3644086718559265, -1.6177973747253418, 1.8194037675857544, -0.1521380990743637, 0.8120907545089722, 0.1472570300102234, -0.39828580617904663, -0.3492928445339203, -0.8367205858230591, 1.3140381574630737, 0.03465619310736656, 0.5706873536109924, 0.5290014743804932, 0.7392350435256958, 0.5120236277580261, 1.3412905931472778, -0.5848289728164673, -0.9400897026062012, -0.05250561982393265, -0.2663779556751251, 0.1282561719417572, 1.2527656555175781, 1.0476500988006592, 0.6455318331718445, -0.7073768973350525, 0.5907521843910217, 0.2622315287590027, 0.8402878642082214, 0.07493194192647934, -0.3165616989135742, 0.21118608117103577, -0.644831657409668, -0.8053686022758484, 0.3150765895843506, 0.08684797585010529, 1.083223581314087, 0.23378674685955048, -1.0963890552520752, 1.4550319910049438, 1.7128291130065918, -1.0212355852127075, -0.5270932912826538, 1.0586167573928833, 0.2617328464984894, -1.4041814804077148, 1.2050063610076904, 0.15762192010879517, 0.3936457335948944, 1.2356406450271606, -0.6017642617225647, -0.668084442615509, 0.7158992290496826, 0.9835217595100403, 0.16535411775112152, 1.208230972290039, 0.4744436740875244, 1.1390475034713745, -0.7647711634635925, -0.16170145571231842, 1.104487419128418, -0.8053058385848999, -0.7512340545654297, 0.17443440854549408, -0.236123725771904, -0.902177095413208, -1.5975347757339478, 1.3845163583755493, 0.08590026944875717, 1.4438356161117554, -0.852173924446106, 0.09669540077447891, -0.13046741485595703, 0.11914198100566864, -1.541056513786316, 1.237330675125122, -0.2618248164653778, 0.23269023001194, -0.9678936004638672, 1.5392553806304932, -1.500089168548584, 1.8903999328613281, -0.39373689889907837, -0.34154775738716125, -0.6721630692481995, 1.3907724618911743, -0.8508502244949341, -0.8695841431617737, -0.944063127040863, 0.4801604449748993, 0.5300965309143066, -0.7220406532287598, 0.5061057209968567, 1.5214709043502808, 0.6821835041046143, 0.3744124472141266, -1.0155223608016968, 1.535480260848999, 0.6659669280052185, -1.4216700792312622, -0.7457132935523987, -2.749814748764038, 0.49121055006980896, -1.249894142150879, -0.28288814425468445, 1.3514976501464844, -1.4405348300933838, 0.8247525095939636, -0.337567538022995, -1.117944359779358, 1.2366554737091064, 0.36517366766929626, -0.7306250929832458, 0.47487035393714905, -0.6402168273925781, -0.9079326391220093, -0.4250197112560272, 0.9356526732444763, -1.6389334201812744, -1.3523606061935425, 0.8204214572906494, -1.4504297971725464, -0.2622453570365906, -0.3671126663684845, -1.346867322921753, -0.8074499368667603, -0.1908111721277237, -0.14782896637916565, 0.5629854202270508, 0.15250596404075623, 0.8446555733680725, 0.14276860654354095, -0.4922029376029968, 0.5652502775192261, 0.37791454792022705, 0.7965250015258789, -0.20952901244163513, -0.1902834177017212, -0.17570364475250244, -1.5349700450897217, 0.39327800273895264, -0.7128944396972656, -0.8087930083274841, -0.4188983738422394, 2.0145816802978516, -0.8388190865516663, 1.4401689767837524, -2.135014295578003, -0.3341568112373352, 1.8559691905975342, 0.1298631876707077, 0.3316729962825775, -0.604195237159729, -1.989652156829834, 0.47130200266838074, 0.8858662843704224, 1.0233968496322632, 1.878167748451233, -0.1048194169998169, 0.49894213676452637, -2.1443634033203125, -0.3601423501968384, 0.12572255730628967, 1.2481122016906738, 1.8234049081802368, 1.5926483869552612, 0.4448704123497009, -0.14433614909648895, -0.08417779952287674, -0.055939797312021255, 0.13451607525348663, 1.1668329238891602, -0.19751659035682678, -1.6594657897949219, -1.1954830884933472, 2.916414976119995, 1.3336786031723022, -0.49982407689094543, -0.26925310492515564, 0.7168418169021606, 0.29723161458969116, 0.3150562047958374, -1.9660652875900269, -1.378847599029541, 0.1436784267425537, 0.6942001581192017, 0.31493180990219116, 1.5697509050369263, 0.05464399233460426, 0.25792914628982544, 0.0668523907661438, 0.9762664437294006, -0.602423906326294, 2.038215398788452, -1.0597578287124634, 0.784909725189209, 0.7270870208740234, 1.5891849994659424, -1.1153937578201294, -0.8985681533813477, -0.27721095085144043, 0.7261066436767578, 0.8614540696144104, -2.569058656692505, -0.3087531328201294, 0.24839890003204346, 0.34746718406677246, 0.35757163166999817, -0.14505191147327423, 0.10848987102508545, -1.8763645887374878, 0.8646455407142639, -2.4240243434906006, -0.1935790330171585, -0.9335836172103882, 0.30713996291160583, -1.0088669061660767, 0.8451390266418457, 0.31153225898742676, 0.5844337344169617, 1.0418003797531128, 1.1036314964294434, -0.2290855497121811, 0.7738296389579773, 0.4421151578426361, 0.7228293418884277, 0.1122841089963913, -2.568896532058716, -0.2970739006996155, -0.0687665045261383, -0.5468004941940308, -0.7559257745742798, 0.094472236931324, -0.7402891516685486, 0.5936488509178162, 1.1893256902694702, -0.04538743942975998, 0.7913878560066223, -1.232759714126587, -1.0471168756484985, -0.37095344066619873, -1.0076968669891357, -0.006670055445283651, 0.024871570989489555, 1.5270124673843384, -0.5706924796104431, 1.202695369720459, -1.0510708093643188, -0.27462294697761536, 0.7770149111747742, 0.28569474816322327, -2.450955629348755, -0.0862598642706871, 0.4207497239112854, -1.529572606086731, -0.48604485392570496, -0.3419809937477112, 0.35744819045066833, 1.216838002204895, -1.3766244649887085, -0.9226536750793457, 1.5151138305664062, 1.1448619365692139, 1.662216067314148, -0.052602872252464294, 0.9870933294296265, -2.397474527359009, 0.4218154847621918, -0.9718543887138367, -1.412172794342041, -1.095406174659729, 0.9629751443862915, 1.4449180364608765, -0.5075094103813171, -0.37301570177078247, 1.1423559188842773, -0.6887819766998291, 0.23117507994174957, 0.5912147164344788, 1.1604474782943726, -0.29011282324790955, -2.7804548740386963, -0.21619412302970886, 0.6653842926025391, -0.47981855273246765, -0.5846770405769348, -0.9586490988731384, 0.4075322151184082, 0.14442561566829681, 0.2538650929927826, -0.015118052251636982, 1.2416189908981323, 0.31048789620399475, -0.08177822828292847, -1.1172020435333252, -0.41754692792892456, -0.017902741208672523, 1.4427471160888672, 0.4600135087966919, -0.3376632332801819, 0.7311762571334839, 0.6120651364326477, -0.45325252413749695, 0.4491333067417145, -1.0806190967559814, 0.09492747485637665, 1.9822105169296265, 0.9592180252075195, 0.6159011125564575, 0.8608480095863342, 0.695300281047821, -1.1480028629302979, -0.4475531578063965, -0.6745718121528625, 0.46219778060913086, 0.9061958193778992, 2.0098371505737305, -0.13756045699119568, -1.104506015777588, 0.29439517855644226, -0.025278137996792793, 0.2680017948150635, 0.030295953154563904, 0.5059695839881897, -0.5544782876968384, 0.30892840027809143, 0.9802942276000977, 0.19188739359378815, 0.5556138157844543, -0.6099476218223572, 0.8657662868499756, -0.08300809562206268, -0.631192147731781, -0.19826218485832214, -0.3597106337547302, -0.1471385508775711, -0.523465096950531, 1.6850649118423462, 0.9002399444580078, 0.47505947947502136, 0.4662785232067108, -0.07770387828350067, 0.8114545345306396, -0.21667027473449707, -0.06306801736354828, -0.7473912239074707, 1.1903975009918213, 1.1598162651062012, 1.0741612911224365, 1.095296025276184, -2.312026023864746, 0.09705091267824173, 0.9984279274940491, 0.48992064595222473, -1.5396058559417725, 0.2887379229068756, 0.6610885858535767, -0.5385056734085083, -1.1382567882537842, -0.6047691702842712, -2.92169189453125, 0.2239842265844345, -0.8611741065979004, -0.34814217686653137, -0.20817647874355316, -0.13226987421512604, 1.4966621398925781, -0.8784251809120178, -0.28378865122795105, -2.1414244174957275, -0.7248339653015137, -1.9569417238235474, 0.6731711626052856, -0.0986836701631546, -0.6767418384552002, 0.7787857055664062, -1.0858794450759888, -0.2546267509460449, -0.8847774863243103, -0.9944509863853455, -0.30859577655792236, -0.6748923063278198], [-0.21795545518398285, -1.1592957973480225, -1.6072081327438354, 1.2586127519607544, 0.2981046140193939, -0.4301948547363281, 0.1420215219259262, -1.0471539497375488, 0.3074738085269928, -1.4907586574554443, 0.3969714045524597, 0.3044697046279907, -0.11661414057016373, 0.6518802046775818, -0.12338438630104065, 1.6021801233291626, 0.5760362148284912, -1.5402849912643433, -1.0214461088180542, 1.579156517982483, -0.5751088261604309, 0.3096616864204407, -0.5687583088874817, 1.4532824754714966, 1.0483020544052124, 0.7671726942062378, -0.42278388142585754, -0.31329452991485596, 1.3447492122650146, -0.5519164800643921, -2.112281560897827, 1.5256413221359253, 0.9573380351066589, -1.2847871780395508, 0.2754720151424408, 0.5975174903869629, 0.21808400750160217, 1.6373897790908813, 1.2376309633255005, -0.7564480900764465, -0.12708312273025513, 0.9282832741737366, 0.9475933909416199, -0.3148077726364136, 1.1349914073944092, -0.7951467037200928, 0.8369782567024231, -1.7157403230667114, 0.4909124970436096, 2.0398099422454834, 1.5122525691986084, -0.7863904237747192, -2.8961586952209473, 0.2923137843608856, 1.0733124017715454, 0.12330663204193115, 0.25093814730644226, -1.350144863128662, -0.5247480273246765, -1.2595685720443726, -1.2799580097198486, -0.5543888211250305, 1.3848638534545898, 0.34593823552131653, -0.7895117402076721, -0.18683169782161713, -0.9644506573677063, -0.6024739742279053, 0.1475801318883896, -1.5267271995544434, 0.2409612536430359, -0.6705007553100586, 1.0232514142990112, -0.9748265743255615, 0.1566420942544937, -0.8996762037277222, 0.45644187927246094, -1.7826719284057617, -0.5809178352355957, -0.9138688445091248, -1.6285051107406616, -0.42413583397865295, -0.24876977503299713, -0.6491518616676331, -1.0675519704818726, -0.7434971928596497, -0.7764566540718079, 0.9737842082977295, 0.46493270993232727, 1.2509759664535522, 0.8840399980545044, 1.7512153387069702, 0.030972784385085106, 0.2767595946788788, 0.10047470033168793, -0.362280935049057, 0.6764311194419861, 0.41441091895103455, 0.5218102931976318, 0.30401942133903503, 1.0747910737991333, -1.3216980695724487, 0.6231931447982788, 0.022708851844072342, -0.5525521039962769, -0.2589882016181946, 0.5546296238899231, 0.872077465057373, -0.1502871960401535, 0.7703831195831299, 0.4366331100463867, 1.0833320617675781, 1.0756345987319946, 0.8331667184829712, -0.2641705274581909, -1.0115488767623901, 1.4344143867492676, -0.7175403833389282, 0.8899715542793274, 0.15810811519622803, 1.691528081893921, -0.022295404225587845, -0.21225623786449432, -1.9235199689865112, 0.6894814968109131, 0.07428566366434097, -0.5020357370376587, -0.296444296836853, 1.33477783203125, 0.4783438742160797, 0.9188759922981262, -0.13806043565273285, 0.40359607338905334, 1.0220285654067993, -0.16230559349060059, 1.3670581579208374, -0.12186826765537262, 1.9932668209075928, -0.41792261600494385, -0.31184545159339905, 0.25360214710235596, -0.198933944106102, 0.03983632102608681, 0.11492440104484558, 1.3325234651565552, -0.3309478759765625, 1.2841169834136963, 0.994513988494873, 0.2856611907482147, -0.04757365584373474, 1.9934837818145752, -2.320129156112671, -0.3068768084049225, 0.023454684764146805, 0.6652982234954834, 0.18769381940364838, 1.0439180135726929, 0.6865600943565369, 1.208250641822815, -0.4371156394481659, 0.6102017760276794, 1.7705351114273071, -0.47922253608703613, 0.34457340836524963, -1.2218971252441406, 0.5415335893630981, -0.613460898399353, 0.7513121366500854, -0.9836022853851318, 0.17430612444877625, -0.8716813921928406, 0.09554453939199448, 0.5316299796104431, 2.3642501831054688, 0.5340889096260071, 0.5881783366203308, 1.922155737876892, 1.1691478490829468, -0.9129571318626404, 0.6355729103088379, 0.22813309729099274, 0.9140201807022095, 0.6203571557998657, -1.632766842842102, -0.4589295983314514, 1.856222152709961, 0.9202741384506226, -0.7761565446853638, -0.414250910282135, -0.21564960479736328, -1.2829378843307495, -3.2390079498291016, 0.5209661722183228, -1.0445300340652466, -0.7054965496063232, -0.7828716039657593, -0.4287485182285309, -0.4834049940109253, 1.4897578954696655, -0.359178364276886, 0.5434268116950989, -0.07421767711639404, 1.232263445854187, 0.06839657574892044, -2.0478355884552, 1.7602680921554565, 1.6195762157440186, 0.6290784478187561, -1.632546305656433, 1.271926760673523, 0.3000664710998535, -0.8731777667999268, -0.6124951839447021, 1.1117973327636719, -0.6928720474243164, -0.22290585935115814, -0.8152444362640381, 0.4665774703025818, -0.9470061659812927, 0.5202494263648987, -0.36400914192199707, 1.9993058443069458, -0.5102139115333557, 0.04350727051496506, 0.9254182577133179, 0.8289088606834412, -1.5688652992248535, 0.14713901281356812, 0.3732875883579254, 0.6585323810577393, -0.18969376385211945, 1.4805667400360107, 1.3742753267288208, -0.3767971992492676, -1.0863932371139526, -0.07022836059331894, 0.5857216119766235, 1.1146835088729858, -1.130732774734497, -0.6186981797218323, -1.916623592376709, -0.9250350594520569, 1.1739128828048706, 2.1816914081573486, 0.8316411972045898, -0.03929794207215309, -0.1627182811498642, 2.0721590518951416, 1.3377366065979004, 0.342122346162796, -0.4014434814453125, -0.6343620419502258, 0.13158579170703888, 1.9483529329299927, -0.09304112941026688, -0.21014994382858276, 0.4085165560245514, -0.1314278095960617, 1.0766867399215698, -0.8976821899414062, 0.20812371373176575, -0.5863267183303833, 0.9476468563079834, 0.24950715899467468, -0.6668586730957031, -0.46295225620269775, 0.01278245635330677, 1.5112813711166382, 0.5898044109344482, 0.32922154664993286, -0.9738402366638184, -1.5076431035995483, -1.5534634590148926, -0.6211667656898499, 0.38403433561325073, 0.027503805235028267, 1.114670753479004, -1.1197870969772339, -0.23577281832695007, -0.2863295078277588, -1.0220237970352173, -2.209699869155884, -0.8497530221939087, 0.8886198997497559, -1.7853374481201172, 0.8176527619361877, 0.8446223139762878, -0.1924487203359604, 0.24313686788082123, -0.16929325461387634, -0.02441699616611004, 1.4093947410583496, 1.1155835390090942, 0.9670115113258362, 0.2318372130393982, -0.1476326584815979, 0.2328444868326187, -1.3103801012039185, -0.9038159847259521, 0.047381505370140076, -1.8194992542266846, -0.03149985149502754, -0.172993466258049, -0.22212882339954376, -0.17823055386543274, -0.7141103148460388, 2.0320358276367188, -1.9871256351470947, -0.797328770160675, -1.8512874841690063, 0.25703707337379456, 0.8264469504356384, 1.7175540924072266, -0.8143184781074524, 0.5786975622177124, -0.9879243969917297, -0.16460460424423218, 0.5630753636360168, -1.5796629190444946, -0.27067258954048157, -0.7335817813873291, 0.2735593020915985, 0.7141014933586121, -1.5735341310501099, -0.21802979707717896, -1.4893815517425537, -2.4122202396392822, -1.2119766473770142, -0.7661239504814148, -1.322095274925232, -1.224990963935852, 1.1877943277359009, 0.007600078824907541, 0.9865913987159729, 1.3404377698898315, 0.08216717094182968, 2.506687879562378, 0.38258421421051025, 0.08033277839422226, -0.8700048327445984, -0.3970571458339691, 0.3176994323730469, 0.042528219521045685, 0.19579698145389557, 0.30367806553840637, -1.412689208984375, -0.8442835211753845, -0.4524292051792145, -0.5966315865516663, -1.4032821655273438, -0.1515142172574997, -0.9476966261863708, 0.9292663335800171, -0.8807024955749512, 0.1605234593153, -2.0733981132507324, -0.24395613372325897, 0.9139878153800964, -0.03728187456727028, 0.5674391388893127, 0.12295264005661011, -0.39133909344673157, 0.4119398891925812, 0.3267303705215454, 0.3048764169216156, -0.675815761089325, -0.5780598521232605, -2.8459317684173584, 0.00031582728843204677, -1.1566011905670166, -0.855841875076294, 0.719936728477478, 0.08551790565252304, 0.2880842983722687, -0.8276312351226807, 0.37575197219848633, -0.07876527309417725, 0.09461095184087753, -0.21335454285144806, 1.54560124874115, -1.2197989225387573, 0.1454555094242096, 0.19011050462722778, 0.8498849868774414, 0.4317305386066437, -0.5563278794288635, 0.08224255591630936, 1.450370192527771, 1.0075387954711914, -0.4568757116794586, 0.13074713945388794, -0.18158596754074097, 1.273815393447876, -0.4710753262042999, 1.2688868045806885, 1.6044450998306274, -0.16545109450817108, 2.21448016166687, -0.2583432197570801, -0.21993988752365112, 0.7218661308288574, -0.5329793691635132, 0.08123308420181274, -1.1918468475341797, -0.42383745312690735, 0.06737471371889114, -1.364235758781433, 0.6924426555633545, 0.7283409833908081, -0.12546902894973755, 0.6393386721611023, 0.6432555913925171, -0.812708854675293, 0.5251984000205994, 0.19700033962726593, -0.6730917096138, 0.35934585332870483, -1.0825728178024292, -2.8366503715515137, 0.42184141278266907, 0.22632746398448944, 1.7928082942962646, -0.32221174240112305, 0.8788105845451355, -0.8740854263305664, -1.1256844997406006, -0.3049001395702362, 0.23474064469337463, -0.010306081734597683, 1.1369733810424805, 1.2154638767242432, -1.2155307531356812, -0.45480698347091675, 1.8726719617843628, 1.4341121912002563, -1.0681358575820923, -0.6473856568336487, 2.095716953277588, -0.8317267894744873, 1.3729077577590942, -1.8565852642059326, -0.06273168325424194, -0.8769047260284424, 0.4913386404514313, 1.4308311939239502, 0.627220630645752, 1.179795265197754, 1.3416520357131958, 1.8303284645080566, 0.49361690878868103, -1.7817912101745605, -1.4173986911773682, 1.0528526306152344, 0.7731364965438843, 1.2343432903289795, -0.19346782565116882, -0.3615552484989166, -1.7643001079559326, -1.0488007068634033, -0.08563003689050674, -0.6375126838684082, -1.6148853302001953, 1.4765770435333252, 0.029858361929655075, 0.2665244936943054, 0.11086724698543549, -0.028920425102114677, -1.070191740989685, 0.8543024063110352, -0.5800677537918091, -1.0328586101531982, 1.2379571199417114, 0.08964353054761887, 0.5981334447860718, -1.9157811403274536, 1.744152545928955, -0.8516539335250854, -1.1595442295074463, -0.42190128564834595, -0.9862055778503418, -0.9045524597167969, 0.4766611158847809, 1.4492028951644897, -0.2510985732078552, 0.005345165263861418, 0.06724829226732254, 0.19015845656394958, -1.2640454769134521, 1.6914395093917847, -1.4360071420669556, -0.6553221940994263, 2.0652248859405518, -0.11648282408714294, -1.0083272457122803, 0.12194812297821045, 2.360511302947998, 0.08412276208400726, -1.1081973314285278, 0.2513388395309448, 0.3310564160346985, -0.819326639175415, 0.2671341001987457, -2.2360525131225586, 0.48878705501556396, 0.2515065371990204, 0.1310998648405075, -0.6087847948074341, 1.4142669439315796, 0.7252517938613892, 1.412636637687683, -0.5597255825996399, 0.8192731738090515, 1.8046321868896484, 1.4892109632492065, -0.8845078349113464, 0.014962228015065193, -0.6082407832145691, -0.24800287187099457, 0.036632053554058075, 1.815423846244812, 1.5750423669815063, -0.10105457901954651, 0.9355946779251099, -0.564344048500061, -1.228369951248169, -1.8748444318771362, -0.8342218995094299, 2.047058582305908, 0.22917519509792328, 0.5761849880218506, 1.095882534980774, -1.9561913013458252, -0.2158418893814087, 0.050515249371528625, 1.3077490329742432, 0.43718472123146057, -1.681326150894165, -1.7240054607391357, 0.32490643858909607, 0.2207748293876648, -1.4149986505508423, -1.4006260633468628, 0.09053339809179306, -1.3257967233657837, 0.36523598432540894, -0.24323518574237823, -0.27990350127220154, 1.9807639122009277, -0.8382891416549683, -1.1439954042434692, -0.5341749787330627, -2.4348392486572266, 0.7746961116790771, 0.6457025408744812, 0.07481909543275833, -0.046212416142225266, -0.09973155707120895, -1.5914552211761475, -0.19205255806446075, -0.07239701598882675, 0.7995855808258057, 0.2576087713241577, 0.9932488203048706, 0.6517404913902283, -2.0609066486358643, -0.8009156584739685, 0.6050236821174622, -0.4604676067829132, -0.04647153988480568, 0.8254296779632568, -1.0351957082748413, 0.15554329752922058, 1.6202391386032104, 0.022849924862384796, -0.7388949394226074, -0.007565599400550127, -0.39536458253860474, -1.276892900466919, 0.6304140686988831, 0.004235670901834965, -1.9015190601348877, 1.1986329555511475, -0.963470995426178, 0.05679740756750107, 0.44539302587509155, -0.1426936239004135, -1.2888950109481812, -2.379431962966919, -0.5794515013694763, 0.2137448489665985, -1.245520830154419, -0.426196426153183, 1.276164174079895, -1.0234445333480835, 4.036861896514893, 0.36645764112472534, -1.2229506969451904, 0.414054274559021, -0.24901263415813446, 1.324722409248352, -1.285361409187317, 0.6888108253479004, -1.124700903892517, 0.3363484740257263, 1.5278546810150146, 0.14091891050338745, 1.3659188747406006, -0.3092799186706543, 0.9436260461807251, -0.6644560098648071, -0.2614412009716034, 0.015903571620583534, -1.509752631187439, -0.946412980556488, -0.6136409044265747, 1.7274078130722046, 0.14391441643238068, -1.1831753253936768, -1.2470006942749023, -0.11820031702518463, -1.1768467426300049, 0.17385804653167725, -1.1402838230133057, -1.093242883682251, 0.8201685547828674, -1.9197845458984375, 2.3038978576660156, -0.7901695966720581, -1.5226995944976807, 0.5894994139671326, 1.0887129306793213, 1.3148207664489746, 0.5304538607597351, -0.718711256980896, -0.17637406289577484, 0.13519801199436188, 0.8804975152015686, 0.9818411469459534, 0.7133980989456177, 0.8757429718971252, -2.277080774307251, -0.5607978105545044, 1.1874595880508423, -0.47382596135139465, -0.07698626816272736, -0.733051598072052, -0.133950874209404, -0.6794551014900208, -2.0792694091796875, 0.3853945732116699, 0.43081772327423096, -0.9136263728141785, -2.110457181930542, -0.3631592094898224, -1.3108571767807007, 0.9884555339813232, -0.3056862950325012, -1.6905783414840698, -0.7789376974105835, 1.8001834154129028, -0.2737944722175598, -0.9268800616264343, -0.37000006437301636, -1.2932252883911133, 0.7845473885536194, -1.537399172782898, 0.5140547752380371, 0.9770203828811646, 0.8702061176300049, -0.19542378187179565, -0.4933713376522064, -0.021411368623375893, 0.20489871501922607, 0.5278820991516113, -0.05066649615764618, 0.2149815857410431, -0.5082175731658936, -0.8633909225463867, 1.1045982837677002, 0.40414854884147644, 0.7050737142562866, 1.6793781518936157, -0.7543193101882935, -0.6481146216392517, -0.41274493932724, 0.8714406490325928, -1.4786388874053955, -0.7113156914710999, 0.7960575222969055, -1.6210150718688965, 0.6995994448661804, 1.614615559577942, 0.41320276260375977, -0.8708067536354065, -0.46646422147750854, -1.0207054615020752, -0.08438274264335632, 0.959157407283783, 1.0297125577926636, -2.2978312969207764, -0.7942619919776917, 0.33191195130348206, -0.3615522086620331, -2.505873441696167, 0.21162240207195282, 0.5489720702171326, -0.006984233390539885, -0.0752776637673378, 0.9512256979942322, -1.4551748037338257, -0.30825385451316833, 0.40839579701423645, 3.484982967376709, -0.7484485507011414, 0.7993671298027039, 1.7676334381103516, 0.7851652503013611, 0.888940691947937, 1.2867451906204224, -1.8161876201629639, 0.5249518752098083, -0.4530232548713684, 1.5785975456237793, 0.44907161593437195, 0.2757357358932495, 0.2008703500032425, -1.2742719650268555, 0.956822395324707, -1.6443063020706177, -0.19053404033184052, -0.3506781756877899, 1.0255675315856934, 0.023885836824774742, -0.8568097949028015, 0.7107119560241699, 1.7924931049346924, -1.0334968566894531, 1.4402668476104736, -0.6653204560279846, -0.21275721490383148, -0.18522782623767853, -1.0497725009918213, 1.6853686571121216, 0.8281928300857544, 0.6482651829719543, -1.375177025794983, 0.7059811949729919, -0.8937869668006897, 1.4381047487258911, -1.7980921268463135, 0.7693324685096741, -1.5853897333145142, -1.4961650371551514, 0.2483585625886917, 0.904735803604126, 1.1727416515350342, -0.3899655044078827, -0.5443766117095947, 0.9735826849937439, -1.1724793910980225, -1.1373041868209839, 0.5658564567565918, 0.4937354624271393, 0.11106667667627335, 1.3079437017440796, -1.6698001623153687, 0.5107475519180298, 2.2923803329467773]], "dtype": "float32"}, {"__nd__": [-0.3990460932254791, -1.2782952785491943, 1.314501404762268, 0.6080613136291504, 1.291145920753479, 0.7674800753593445, -0.42366504669189453, -0.17702126502990723, -1.1738595962524414, 2.7459514141082764, -0.9123489856719971, -0.3111708462238312, -0.6545354127883911, 0.5069372057914734, 1.5194206237792969, -0.473714679479599, 0.5140010714530945, 0.811817467212677, 2.3173434734344482, -0.17857663333415985, 0.3304615914821625, 1.7544214725494385, -1.4312540292739868, 0.041342224925756454, 0.14261819422245026, -1.0999292135238647, -0.15937365591526031, 0.27342361211776733, 1.2083377838134766, 0.9987149238586426, -0.5109549760818481, -0.9129632115364075, 1.0336456298828125, -1.5120999813079834, -0.5107467174530029, -1.1190903186798096, -1.5848567485809326, 0.3069054186344147, -1.4240621328353882, -0.6109195947647095, 1.3113574981689453, 1.6284644603729248, -1.3577311038970947, -0.9250946044921875, -0.8695799708366394, -1.3493984937667847, 1.3723739385604858, 0.9272035360336304, 0.5213491916656494, -0.9856082797050476, 0.7895569205284119, 0.21093609929084778, -0.015414925292134285, 0.6058946251869202, 1.4050202369689941, -0.052581723779439926, -0.4056719243526459, -1.035693883895874, 1.4494526386260986, 0.5110726356506348, -2.4512593746185303, 0.15277695655822754, 0.8364701271057129, -1.6097352504730225, -1.5070089101791382, 1.0727275609970093, -0.28616511821746826, -1.4939179420471191, 0.03732287511229515, 0.36660081148147583, -0.9711063504219055, -0.8147364854812622, 0.035007212311029434, -0.06772412359714508, 1.8021186590194702, 1.3489638566970825, 0.5208491683006287, -0.6823938488960266, -0.6271253824234009, -2.281367063522339, -0.08094191551208496, -0.41998371481895447, -0.6837112903594971, 0.7327607870101929, 1.270538568496704, -1.8895344734191895, 0.4057096540927887, 0.44624581933021545, -0.6414676904678345, -0.7393257021903992, 1.7395951747894287, 0.7403997778892517, 2.1564114093780518, -0.7910337448120117, -1.085361361503601, -0.28107118606567383, 0.8740528225898743, 1.3317290544509888, 1.98874032497406, -0.6612593531608582, -1.4727414846420288, 1.005771517753601, 0.9144768714904785, 0.5816971659660339, 1.4728405475616455, -0.8237137198448181, -1.655031442642212, 2.093451499938965, 0.31126147508621216, -0.3052235245704651, 0.8207834959030151, 2.2696313858032227, 0.1928243339061737, 1.7787336111068726, 0.10148503631353378, 1.968629002571106, 0.20561060309410095, 1.2392641305923462, -0.005673936568200588, 0.6271430850028992, 0.24211660027503967, -0.4159497916698456, 0.12067936360836029, 0.7307035326957703, -0.10527107119560242, -0.5388365983963013, 0.4382268190383911, 1.4897342920303345, -0.07796918600797653, -1.0161038637161255, -0.8898144960403442, 0.0325234979391098, 1.6848777532577515, -1.0555013418197632, 1.1689560413360596, -0.1913403868675232, -0.6864349246025085, -0.5414464473724365, 1.1800533533096313, -0.47580984234809875, 1.3906669616699219, 2.076165199279785, -0.8839995265007019, -1.3202062845230103, -1.795446753501892, 0.5422350168228149, -0.9140052199363708, -0.6517276167869568, -0.8150538206100464, 1.3709487915039062, 0.6678017973899841, 0.6048237681388855, -1.7871655225753784, -0.8825697302818298, -0.07611773163080215, -0.14268988370895386, 0.2945314347743988, 1.616516351699829, 0.6723906993865967, 0.7925763726234436, -0.15966854989528656, -0.4460895359516144, -0.5132283568382263, -0.13712790608406067, -1.7090779542922974, -0.62137371301651, -0.009337487630546093, -0.5262873768806458, -0.08273644745349884, -1.9148746728897095, -0.6019083857536316, -0.027512559667229652, 1.1423003673553467, 0.0812908187508583, -0.2684426009654999, -0.6280617713928223, -0.0009032441885210574, -0.5644707679748535, 0.6985507011413574, -0.5965278148651123, 0.6282672882080078, -0.7070732116699219, -1.5611156225204468, -1.5259274244308472, 0.8745753765106201, -0.6417968273162842, -0.5534705519676208, 2.5317113399505615, -1.2485449314117432, -0.09974800795316696, -1.4484683275222778, -0.6997363567352295, 0.4681895077228546, -0.37385329604148865, 1.86037278175354, -1.0216000080108643, -0.7720773220062256, -1.4923840761184692, -0.04905305802822113, -0.8543766736984253, -0.21356256306171417, -0.8947174549102783, 0.3927055597305298, -0.7271826863288879, 0.024538643658161163, -0.18843436241149902, -0.8201452493667603, -0.05400001257658005, -0.06360217928886414, 0.47011956572532654, -2.0591416358947754, 1.4529294967651367, 0.0776621401309967, 0.3861325979232788, 0.4918534755706787, -1.430822491645813, 0.7024717926979065, 0.257769912481308, -0.535028338432312, 0.7913720607757568, -0.29269543290138245, -1.052369475364685, 0.8573254346847534, 0.7611470818519592, 0.30841386318206787, -0.15712833404541016, 1.5901455879211426, -0.6004446148872375, 0.2072206288576126, -0.19361773133277893, -0.20576132833957672, 0.47145751118659973, -0.9937095046043396, 0.5011529922485352, 0.2716117203235626, -0.34256982803344727, -0.7300257086753845, -1.3856085538864136, 0.08400580286979675, 1.7011717557907104, -0.6354141235351562, -0.9344778656959534, -0.6780100464820862, 0.2671155333518982, 0.5004255175590515, 0.3539360463619232, -0.0516141876578331, -1.2299989461898804, 1.0293141603469849, -0.3343892991542816, 0.3955498933792114, 0.09100126475095749, 0.008612123318016529, 0.8169299364089966, 0.15213264524936676, 2.099600076675415, 1.4850690364837646, 0.9261180758476257, -0.24994516372680664, 0.8367198705673218, 0.3751840889453888, 0.7503980398178101, 0.11267036199569702, -0.986906886100769, -0.928646981716156, -0.0345439612865448, 0.897329568862915, -0.03561825305223465, -0.5636928081512451, -0.1442623734474182, 0.7014939785003662, -0.44616034626960754, -0.4867613911628723, 0.14104576408863068, 0.6908890604972839, -1.1773767471313477, -0.3603038787841797, 0.01724523864686489, -0.40638357400894165, 0.5402539372444153, 0.8768092393875122, -0.7163369655609131, 2.4293460845947266, 0.4194319546222687, 1.6129344701766968, 0.7852514386177063, -1.7921645641326904, -1.2335612773895264, 0.2935985028743744, 0.5514148473739624, 0.6386204361915588, -1.9395798444747925, -0.6098182201385498, 0.5035402774810791, 1.2945208549499512, 0.5498773455619812, -1.3581515550613403, -0.5292479395866394, -0.3099837303161621, -0.40480515360832214, 1.0550148487091064, 1.1893855333328247, 1.3035436868667603, -1.0808250904083252, -1.243548035621643, -1.4963301420211792, -0.24152463674545288, 0.10842785984277725, -1.3638567924499512, 0.27388855814933777, 0.25395455956459045, -0.19615726172924042, 0.46118855476379395, -0.983917236328125, -1.0650261640548706, 1.4651631116867065, -0.3668307662010193, 0.9832844734191895, -0.6488187909126282, -2.2623093128204346, 0.5431658029556274, -1.2313342094421387, -0.8678023815155029, 0.8795678615570068, -1.5248461961746216, -0.2655368149280548, 1.1448193788528442, 2.0264527797698975, -0.9775325059890747, 0.16719138622283936, -0.6573379039764404, 1.2321842908859253, -0.4846906363964081, 1.3390021324157715, -1.2423696517944336, 0.7883921265602112, 1.492455244064331, 0.6716196537017822, 0.022903714329004288, -0.4902096688747406, -1.318139672279358, 0.9580745697021484, 0.9423396587371826, 0.3498082458972931, -1.2637698650360107, -0.9038633704185486, -0.14465540647506714, 1.7052226066589355, 1.8681730031967163, 0.966923713684082, -1.4307969808578491, -2.078376293182373, 0.004857118707150221, 0.42842021584510803, 1.6243282556533813, -0.9289014339447021, 1.0762677192687988, -0.3847845494747162, 0.5200170874595642, -0.6264866590499878, -0.5661135315895081, -0.6753919720649719, -0.2821667790412903, -1.060333251953125, -0.9239559173583984, 0.18793578445911407, -1.3412362337112427, -0.4589356780052185, -0.6813256144523621, -0.3407086730003357, 1.1262061595916748, -0.8115136623382568, 1.5833477973937988, -1.9342855215072632, -0.9796671271324158, 0.6542446613311768, -2.5051684379577637, -0.2970242202281952, 0.3932947814464569, -1.4854943752288818, 1.4498211145401, 1.8380764722824097, 0.04116823151707649, 0.001527488580904901, 0.8024422526359558, 0.9817739129066467, -0.4581528604030609, -0.4150106906890869, 2.062080144882202, 0.24526450037956238, -0.16342918574810028, -0.8727446794509888, -2.970841884613037, -0.15362326800823212, 0.0581609345972538, -0.07512512058019638, -0.19801518321037292, 0.819071888923645, 0.2874387800693512, 0.19404400885105133, -0.19190378487110138, -0.3508937954902649, 0.11012996733188629, -0.7064120769500732, 1.1520624160766602, 0.2762795686721802, 0.004392136353999376, 2.054560899734497, 0.8199785351753235, -1.1596810817718506, -0.8625797033309937, -0.08207641541957855, -0.496858149766922, 1.3287544250488281, 0.7739410400390625, 0.9091895222663879, 1.0929731130599976, 0.9307626485824585, -1.8086295127868652, -1.3874152898788452, -1.013373613357544, 0.5302985906600952, 0.23235274851322174, 1.8649542331695557, 1.3090331554412842, 1.3746647834777832, -0.6593653559684753, -0.060819126665592194, 1.0122674703598022, -1.099778175354004, -0.6723588705062866, 1.8388270139694214, 2.161330223083496, -1.7733820676803589, -0.058083318173885345, -1.2222166061401367, -2.025723934173584, -0.8623833060264587, 0.5357939600944519, 0.634160041809082, 0.2488320767879486, 1.2963510751724243, -0.871598482131958, -0.29229363799095154, 1.4939208030700684, 0.7359438538551331, -0.8364891409873962, -1.281491756439209, 0.007843316532671452, 0.1593749076128006, 0.2853654623031616, -0.203835129737854, 0.0721101313829422, -0.6778606176376343, -0.10417953133583069, -0.7051849365234375, -1.3279759883880615, 0.07245223224163055, -0.8629637360572815, -0.8585455417633057, 0.5264712572097778, 0.30012479424476624, 1.863537073135376, 1.687742829322815, -0.41283276677131653, -0.32375770807266235, -0.7124590873718262, -1.434324860572815, 1.5358283519744873, 0.6421278715133667, -0.6146174669265747, 1.7755722999572754, 0.6587690711021423, 0.7480593323707581, -0.48836180567741394, -2.456220865249634, 0.6314692497253418, -0.7956108450889587, -1.5424354076385498, 0.5299584865570068, 0.5627626776695251, -0.994897186756134, -0.07554437965154648, 0.46044498682022095, -0.003632782958447933, -0.6194184422492981, 1.4885627031326294, -0.32573986053466797, 1.1530436277389526, -0.10686280578374863, -0.3367473781108856, 1.268477439880371, -0.6757029891014099, 0.34115439653396606, 0.5111801028251648, 1.1353431940078735, -0.07921165227890015, -1.6005401611328125, -1.7652040719985962, -1.8395315408706665, -1.203870177268982, -0.1703934371471405, -0.36493247747421265, -0.5494616031646729, -0.15997883677482605, -0.8053856492042542, -0.1064523383975029, 0.0476749949157238, 2.1362619400024414, -0.31386375427246094, -0.29333534836769104, -0.35465484857559204, -0.8874759078025818, 0.7780729532241821, -1.3689734935760498, 2.021597385406494, -1.6687272787094116, -0.1122661754488945, 0.9805395603179932, 0.5063618421554565, 0.7279291152954102, -1.5801314115524292, 0.870521068572998, 0.5589762330055237, 1.57964026927948, -0.036256641149520874, 0.07964684069156647, -1.6556074619293213, 0.5477957129478455, -1.0990852117538452, 0.3790096938610077, 1.8164076805114746, -0.9205844402313232, 0.24249279499053955, -0.5448453426361084, -0.6105738878250122, -1.956322193145752, -0.43978506326675415, 0.3271610140800476, 1.2972785234451294, 1.5898505449295044, -0.157427579164505, 0.9445458054542542, -0.10521458089351654, -0.9305113554000854, -0.48745545744895935, 0.8745076656341553, 0.6967790126800537, -1.7624396085739136, 0.5308611989021301, -0.34231770038604736, 0.624088704586029, 0.1588325798511505, -1.3003919124603271, -1.4020651578903198, -0.0969766229391098, -0.0043665640987455845, 0.1124027669429779, -1.2104681730270386, 1.4089258909225464, -2.4366631507873535, 1.3403542041778564, -0.20815172791481018, -0.5583915114402771, 1.7114160060882568, -0.138078972697258, 0.03952696919441223, -0.5565257668495178, 0.5313630700111389, 0.5481865406036377, 1.509291648864746, 1.1863081455230713, 0.005335490684956312, -0.6454775333404541, -0.4307372570037842, -0.7756108045578003, -0.4545840322971344, -0.5955211520195007, 0.3209695518016815, 0.12351097911596298, -0.7531969547271729, 0.26172536611557007, -2.2747037410736084, -0.12018220126628876, -2.385711193084717, -0.42523518204689026, 0.6081069111824036, 0.15935029089450836, -1.175630807876587, -0.2523946464061737, -0.20421984791755676, -0.6550514101982117, 0.07541905343532562, -0.7404826283454895, -0.9301268458366394, -1.0746978521347046, 0.38072875142097473, -1.024503469467163, -1.606255054473877, 0.7332476377487183, -0.8491495251655579, -1.5625104904174805, 0.5404251217842102, -0.7306989431381226, -0.6906322836875916, -0.9618486762046814, 0.5483133792877197, 0.5818698406219482, 0.9001405835151672, -0.8049007654190063, 2.2998623847961426, -0.7311121821403503, -1.2603882551193237, -0.4465976059436798, -1.9267491102218628, -0.907494306564331, -0.5362973213195801, 0.0725669115781784, 1.263525128364563, 0.38056012988090515, -0.3389480412006378, 0.0005729454569518566, -1.2482820749282837, 1.4355229139328003, 0.17702455818653107, 0.43181076645851135, -0.5804747939109802, -0.8798193335533142, -1.04619300365448, -0.9165594577789307, -0.6289050579071045, 0.7991191148757935, -0.7612695097923279, 0.552646279335022, 0.8052698373794556, 0.33990100026130676, 0.8291682004928589, -2.3447911739349365, 0.24081948399543762, -0.48186415433883667, 1.3376587629318237, 0.3419002294540405, -0.34540095925331116, 0.03017556667327881, 0.20906151831150055, 1.7649122476577759, -0.40067344903945923, 0.2894350588321686, -0.5523137450218201, 0.27944448590278625, 0.374349445104599, 0.2029723972082138, -0.1427917182445526, 1.556554913520813, 1.111990213394165, 0.5221898555755615, -0.5551561713218689, 0.4853259325027466, 0.18093492090702057, 0.9797019362449646, 0.6132094860076904, -0.6021135449409485, 0.6444925665855408, 1.9227378368377686, 0.6200080513954163, -1.0319639444351196, -0.36755526065826416, 1.046463966369629, 1.3343009948730469, -1.4333224296569824, 1.1671968698501587, 1.796816110610962, 1.5901994705200195, -1.1967910528182983, 0.1444716453552246, 1.4824817180633545, 0.9070338606834412, -0.23950013518333435, -0.8296528458595276, 0.07779628038406372, 0.6743477582931519, -0.8726616501808167, -0.034600064158439636, 1.4062508344650269, 1.4596526622772217, 0.32087773084640503, 1.5056960582733154, -0.011965411715209484, -0.5122495889663696, -0.5484695434570312, -0.9611060619354248, -0.3605343699455261, -0.24126167595386505, 2.239704132080078, 1.1068835258483887, 2.3788564205169678, -0.9086253046989441, 0.8155571818351746, -0.3627539873123169, -1.3489985466003418, 0.6888210773468018, -0.5805227756500244, 0.8552298545837402, 2.7027385234832764, 0.18823453783988953, -0.02364247664809227, -0.612963080406189, -1.4444798231124878, -0.16859735548496246, -0.7645784020423889, 0.23855897784233093, -0.2785816192626953, 0.3859015703201294, 0.5652822852134705, 1.2659257650375366, -1.0718467235565186, -0.7410696148872375, 1.5061473846435547, 0.9699507355690002, -0.6547232866287231, 2.6063551902770996, -2.783040761947632, -0.3566131591796875, -0.28216007351875305, 0.5264424681663513, -1.3176950216293335, 0.033745937049388885, 0.24637563526630402, 1.2035579681396484, -1.8967968225479126, -0.6682316660881042, -0.6169450879096985, 1.0043237209320068, 1.9569562673568726, 1.723488211631775, -0.7153761386871338, 0.21994532644748688, -0.1809999644756317, -0.04221063107252121, -0.7256357073783875, 0.17679466307163239, -0.10367380082607269, 0.5659136772155762, -0.17512647807598114, 2.5598249435424805, -1.6761751174926758, -0.60069340467453, -0.5788553357124329, 0.1753847450017929, 0.9307531118392944, -0.05086464434862137, 3.0514626502990723, 0.386836975812912, 1.479782223701477, -0.04443634673953056, -0.7152699828147888, 2.5251317024230957, -0.8221867680549622, -1.494250774383545, -0.7003505229949951, 0.22068218886852264, 1.4411213397979736, -0.24732591211795807, -0.23239867389202118, 0.3019033968448639, 0.8261533975601196, -0.8007031679153442, 0.3311728537082672], "dtype": "float32"}, {"__nd__": [0.051889315247535706, 0.024707874283194542, 0.02890041284263134, -0.04113021865487099, -0.026300625875592232, -0.007715282961726189, 0.005554567091166973, -0.0004977525095455348], "dtype": "float32"}, {"__nd__": [1.0050002336502075, 1.0455470085144043, 0.9782841801643372, 0.9571136832237244, 1.0051246881484985, 1.0125696659088135, 0.9845351576805115, 0.9699299931526184], "dtype": "float32"}, {"__nd__": [[-0.5354435443878174, 0.4687831699848175, 0.5422608852386475, 1.0670572519302368, 1.8672266006469727, -0.4736842215061188, -0.8562108874320984, 0.2054133117198944, 0.1193690150976181, 0.11804255843162537, 0.9109196662902832, 0.24267807602882385, 0.9694296717643738, 0.15575185418128967, -0.007920640520751476, -0.7401852011680603, 0.7849828004837036, 0.14992330968379974, 1.3251848220825195, 0.7372468709945679, -0.4273032248020172, 0.32448825240135193, -0.006565895397216082, -0.05090158060193062, -1.2859046459197998, 0.2897597849369049, 0.15993936359882355, -1.1311924457550049, 0.36434102058410645, 0.4347505271434784, -1.0589011907577515, -1.0534732341766357, 0.31622231006622314, 0.8094483017921448, 0.3483237624168396, -1.7141166925430298, -1.0368491411209106, 0.57792067527771, -0.9642291069030762, 1.0855422019958496, 1.1430068016052246, -0.7843714952468872, 0.27800440788269043, 0.7343724966049194, 0.504582405090332, -0.41711536049842834, -0.16924192011356354, -0.053139299154281616, 1.6075047254562378, 0.6784626245498657, 1.1805531978607178, 0.27834102511405945, 0.5731985569000244, 0.8924188017845154, -0.13760881125926971, -0.6938839554786682, 0.4223777949810028, -0.3704933226108551, -0.5144904255867004, -0.7294053435325623, -1.3442238569259644, 0.15042799711227417, -0.9051602482795715, -0.7513537406921387, -0.2879590690135956, -1.1843923330307007, 0.5357567667961121, 0.8793273568153381, -1.0881133079528809, -0.3431694805622101, -0.2099655270576477, -0.22903338074684143, 0.0907745510339737, 0.3542237877845764, -0.616798996925354, 0.7311879396438599, -1.8786473274230957, -1.0036325454711914, 0.923576831817627, -0.40360236167907715, 0.3532513976097107, 1.1688780784606934, 1.695472002029419, -1.0681794881820679, -0.7261345386505127, -0.71238112449646, -0.6879856586456299, 0.28544750809669495, -1.8882838487625122, 0.42292073369026184, -0.03800452873110771, 1.451102375984192, -1.5014880895614624, 0.6297032237052917, -0.38264673948287964, -0.6905807852745056, 0.846336841583252, 0.8568008542060852, -1.3381500244140625, 0.8143829703330994, -0.26906266808509827, -0.38254353404045105, -0.47693657875061035, -0.08145956695079803, -0.5915555357933044, 0.09256411343812943, -2.393141746520996, 0.895389974117279, 0.27658611536026, -0.5598190426826477, 0.36511287093162537, -0.2173769474029541, 0.5730311870574951, 1.024169683456421, 0.44652968645095825, -2.9936306476593018, 1.2776663303375244, 0.05469633266329765, 0.6509475111961365, -0.7911726236343384, -0.17500662803649902, -0.24041436612606049, 0.5309014320373535, 0.6903164386749268, 0.3312128484249115, 0.23278848826885223, 0.837731122970581, 1.071086049079895, 0.6402269005775452, -0.3575899302959442, 0.7913153767585754, -0.6153507828712463, 0.21778684854507446, 1.4048219919204712, -1.460753083229065, -1.1530992984771729, -2.2297072410583496, 1.797162413597107, 0.09333734959363937, 0.3884681463241577, 0.18319128453731537, -0.20741598308086395, -0.9344338774681091, 0.1363980770111084, -1.0165081024169922, 0.528692364692688, 1.4516034126281738, 2.1341211795806885, -0.972778856754303, -0.4837501347064972, -0.9734019637107849, -1.7613117694854736, -1.2656809091567993, -0.25929611921310425, 0.8466737866401672, 1.0796568393707275, -0.9903574585914612, 0.5308247208595276, -0.3139764368534088, 0.12163402885198593, 0.9010128378868103, 0.8625656366348267, 0.8286707997322083, 0.2179015725851059, 0.19095823168754578, 0.18192094564437866, 0.5139487385749817, -0.15091075003147125, 0.37772026658058167, 0.07799471169710159, 1.2764687538146973, -0.996125340461731, -0.4394712448120117, 0.029825305566191673, -0.13735581934452057, -2.231111526489258, 0.9947652220726013, 1.3877649307250977, -1.6870874166488647, -1.3796932697296143, 1.2231448888778687, -1.5228241682052612, 0.050099436193704605, -0.15340307354927063, -0.46309730410575867, 0.28908389806747437, -1.1158970594406128, -1.6366666555404663, -0.6461630463600159, -0.2500836253166199, 0.13131417334079742, 0.7220139503479004, 1.3789159059524536, 0.4393039345741272, -0.130629301071167, 0.39082831144332886, -1.245161771774292, -0.7769599556922913, -0.8237189650535583, -1.5997017621994019, 0.6764191389083862, -1.1050488948822021, -1.4713339805603027, -1.0606129169464111, 1.3554744720458984, 1.136262059211731, 0.16314665973186493, 0.5505777597427368, -1.8864744901657104, -0.9827779531478882, 0.1874512881040573, -1.125654697418213, -1.4943907260894775, -0.833364725112915, 0.6408116817474365, 1.3223586082458496, -1.1194660663604736, 1.211863398551941, -1.3444457054138184, 0.001998670632019639, -0.1603420227766037, -0.4746394157409668, 1.5477951765060425, 0.6671985387802124, -0.05667434632778168, 0.926959216594696, -0.7595123648643494, 0.3129051625728607, 1.3105800151824951, 0.3165878653526306, 1.8120276927947998, -1.134909749031067, -1.2792260646820068, -0.35198500752449036, -0.28561335802078247, -0.5818275809288025, 1.3178093433380127, 0.27901211380958557, -1.0331438779830933, -0.7341096997261047, 0.3557940125465393, -1.2739505767822266, 0.2947116196155548, -1.074306845664978, -0.1258293241262436, 0.16599637269973755, -1.5090265274047852, -0.5271863341331482, -0.46874502301216125, 0.03341799974441528, 0.6419749855995178, -0.1618071049451828, -1.422439694404602, -0.4933808445930481, -0.9687903523445129, -0.293128103017807, 0.860010027885437, 0.7915260791778564, 1.0400466918945312, -0.8873574137687683, -1.7351921796798706, -1.0134779214859009, -0.1056780219078064, -0.6658706068992615, -1.2508617639541626, -0.15099401772022247, 1.1157608032226562, -0.3824310898780823, 0.31377530097961426, 1.944779634475708, -0.618161141872406, 0.7211335301399231, -1.4932713508605957, -0.8855515122413635, -0.22533096373081207, 1.0109710693359375, 0.08940444886684418, -0.5234969854354858, 0.585303008556366, 1.4424819946289062, 1.4031636714935303, -0.8727417588233948, 0.5864878296852112, -0.5784208178520203, 0.6661890745162964, 1.0468974113464355, -1.121594786643982, 2.0807158946990967, -1.892486333847046, 0.13638898730278015, -1.7481904029846191, -0.5296163558959961, 1.5014469623565674, 1.263790488243103, 1.8532295227050781, -0.3368939757347107, -0.977336049079895, -0.9764273166656494, 2.8578908443450928, 0.9787566661834717, -0.9597652554512024, -0.002445325953885913, -0.10014744848012924, 0.9254592657089233, -0.608773410320282, -1.7538660764694214, -0.7795118093490601, 0.5242006778717041, 0.0556543804705143, -0.5817098617553711, -1.2829058170318604, 1.1758133172988892, 0.5782124996185303, -0.8773086071014404, -0.57065749168396, 1.7912510633468628, -0.3072395920753479, 1.3055815696716309, -0.16871339082717896, -0.1687186360359192, -0.47579291462898254, 0.579369843006134, 0.4698966443538666, -0.8763327598571777, 1.7015738487243652, -0.4070344567298889, -0.5473876595497131, 1.1575478315353394, -0.7492503523826599, -0.1965184360742569, 0.2582390308380127, 0.9046216607093811, -1.5987184047698975, -0.9472683668136597, -2.388170003890991, 0.5245774388313293, 0.2520569860935211, 0.5487064123153687, 0.27119746804237366, -0.6634455323219299, 0.551437258720398, -1.0850911140441895, -0.9641745686531067, -0.37154075503349304, 0.953434407711029, -1.3323073387145996, 2.9071974754333496, -0.7738221287727356, -1.5998307466506958, 0.09712269902229309, -0.9105037450790405, 0.6006659269332886, 0.7966787815093994, -1.0628905296325684, 0.4351896047592163, -0.6157110929489136, 0.1966170370578766, -0.5573701858520508, -0.2005770206451416, 0.257394939661026, 1.3534663915634155, -0.4807671010494232, 0.6927824020385742, 1.4696139097213745, -0.8647260069847107, 1.015681266784668, -0.3728271722793579, 0.27738022804260254, 0.398347944021225, -0.321277916431427, 1.4082576036453247, 1.193102478981018, 0.08081332594156265, -0.09383457154035568, 0.009509816765785217, 0.4990459382534027, 0.487729012966156, -1.4177701473236084, 0.6194242835044861, 0.43601903319358826, -0.40247154235839844, -0.8120660185813904, -1.7247025966644287, 0.263720840215683, -0.7193929553031921, -0.48239657282829285, 2.485227584838867, -1.5059465169906616, -1.993618130683899, 0.6613315343856812, -1.8435086011886597, -0.6372010111808777, -0.07644107937812805, -1.0435491800308228, 1.0508556365966797, -0.08618751168251038, 0.033768948167562485, -0.8145765066146851, -1.1498594284057617, -0.3591119647026062, 1.7379331588745117, -0.20468363165855408, 0.42058342695236206, 0.33417245745658875, -1.0194553136825562, 0.6219314932823181, -0.6501817107200623, -1.860425591468811, -1.9641778469085693, 0.5097112059593201, 1.7626289129257202, 0.5251356363296509, 0.9716977477073669, -1.256974458694458, -3.0134518146514893, 1.6378430128097534, 0.846518337726593, -1.7222651243209839, 0.12597869336605072, -0.1429564505815506, -0.24359117448329926, -1.1675422191619873, 0.2899375557899475, 0.5900917053222656, -1.456485629081726, 0.26476117968559265, -1.575052261352539, 0.5665566921234131, -0.3204837441444397, 0.8017621040344238, 0.3618961274623871, -1.7177035808563232, -1.1553285121917725, -0.024943700060248375, 0.478960245847702, 1.1065313816070557, -1.274240255355835, -0.23532648384571075, 2.286496877670288, 0.01263608317822218, -1.0130029916763306, -1.8532190322875977, 0.3480275869369507, 0.8260510563850403, -0.8480691909790039, 0.76231449842453, 1.0353409051895142, 1.3546031713485718, 0.6166759133338928, -0.32162824273109436, -0.7858481407165527, 0.8569878935813904, -1.4115411043167114, 1.4692384004592896, 0.5786890983581543, -0.3329566717147827, -1.8767188787460327, -0.9139052629470825, -1.56350839138031, -1.1333575248718262, 0.4065071642398834, 2.167449951171875, -1.1179108619689941, -0.895626962184906, 2.00111722946167, 0.016373705118894577, 0.27626147866249084, -0.7713962197303772, 1.5044795274734497, 0.018451938405632973, 0.6934525966644287, 0.7637917995452881, 1.4862737655639648, 0.34687289595603943, -0.6495770215988159, 0.4899497628211975, -0.13491934537887573, -1.8117743730545044, -0.671086311340332, -1.5895981788635254, -0.23199324309825897, -0.5478343963623047, -0.4306943714618683, -1.7244911193847656, 0.46312281489372253, 0.09958241879940033, 0.9824413657188416, -1.2784708738327026, -0.5837920904159546, -0.7525588274002075, 0.36435645818710327, -0.7865164875984192, 0.4508539140224457, -1.1941001415252686, 0.3486131429672241, 0.055900901556015015, -0.14091756939888, -0.27329984307289124, -0.1939658522605896, 0.8564445376396179, 0.9265218377113342, 0.8083868622779846, 0.04907398298382759, -0.603178083896637, -0.695859432220459, -0.3434448540210724, -0.1647671014070511, -0.6696131825447083, -1.2888689041137695, -1.1374318599700928, 0.6525192856788635, -0.7853220105171204, -0.1470777541399002, -0.531914472579956, 0.17941264808177948, -0.3647334575653076, 1.0077340602874756, 0.08253588527441025, -1.2336097955703735, -0.10815674811601639, -1.2079309225082397, -0.49813637137413025, 0.7663899064064026, -1.4956152439117432, 0.42025691270828247, -0.19599807262420654, 0.8943592309951782, -0.11099473387002945, 0.8049574494361877, -0.012487414292991161, 1.0451594591140747, 0.1163821667432785, -0.8547670841217041, 0.05695726349949837, -0.08352714776992798, 1.6156673431396484, -0.8951653838157654, -1.632948875427246, 1.1244776248931885, -0.7414587140083313, -1.7852996587753296, -1.0354658365249634, -0.8701393604278564, -0.5306159257888794, 2.0462472438812256, -0.3605806529521942, 0.4910145699977875, 0.3499988615512848, -0.6991572380065918, -1.8533275127410889, 1.7127516269683838, -0.6128252148628235, -0.9560985565185547, 0.4216102361679077, -0.9842842221260071, -0.6726956963539124, 0.8197473287582397, -0.604439377784729, 0.06600470840930939, -0.5004515051841736, 1.3474863767623901, -1.7117820978164673, -2.8801686763763428, -0.034656353294849396, -0.4565531313419342, 2.668346643447876, -0.03908208757638931, -0.022007513791322708, 0.6454780697822571, 0.16915041208267212, 0.4890744388103485, -1.01749849319458, -0.4294147491455078, 1.3238276243209839, -1.2961876392364502, -0.493846595287323, 0.2551254332065582, 0.6259955167770386, -0.0618908554315567, 0.5305620431900024, 0.7036232948303223, 0.010001414455473423, -0.8231812715530396, 0.3706135153770447, -0.6590616703033447, -0.6379029750823975, 1.146597146987915, 0.3806455731391907, -1.1855577230453491, -1.827617883682251, -0.8923346996307373, 0.15526293218135834, 0.497585266828537, -0.12240032106637955, 1.8967208862304688, -0.20936331152915955, -1.081843614578247, -0.1967717707157135, 0.824948787689209, 0.34316277503967285, 0.24716463685035706, 1.208105444908142, 0.9531487822532654, 1.461211919784546, -0.4901021420955658, 0.3357972502708435, 0.20189616084098816, 0.10811498016119003, 1.8285613059997559, 0.4842786192893982, 0.48804420232772827, -1.1399955749511719, -0.267088383436203, 0.5450031161308289, 0.1720922291278839, -0.9244106411933899, 0.6542150974273682, 0.004272859543561935, 1.0874037742614746, 0.7080245018005371, 1.0472064018249512, -0.789326548576355, -0.5659956932067871, 0.9636215567588806, 2.9888863563537598, 0.818343460559845, 0.719423770904541, -0.21605156362056732, 2.197730541229248, 0.4869183599948883, 0.16939251124858856, 0.5893929600715637, -0.2858429551124573, 0.8573213815689087, 0.34623444080352783, 0.033826254308223724, -1.205207347869873, -0.9197461605072021, -0.1883896440267563, -2.0437490940093994, -0.7241643071174622, 0.1034209355711937, 0.07579824328422546, 2.230302095413208, -1.894201397895813, -0.1264057755470276, -0.3251960575580597, 0.35198840498924255, -0.21489833295345306, -0.7115517854690552, 0.23306481540203094, -0.23182746767997742, 2.1520140171051025, -0.29979419708251953, 0.5032884478569031, 0.9575659036636353, 0.4722955524921417, -0.12298090010881424, -0.02412586845457554, -0.14743486046791077, -0.8024295568466187, 0.12421351671218872, 1.6595667600631714, 1.4928590059280396, -0.7984800338745117, 0.8912374973297119, 0.04383694380521774, -0.08573231846094131, -0.20368345081806183, 0.9018455743789673, -0.83552485704422, -0.8132481575012207, 1.2641416788101196, -1.6776349544525146, -0.45806002616882324, 0.48246878385543823, -0.011263099499046803, -0.23545220494270325, 1.8936632871627808, 1.365177035331726, 0.4123299717903137, 1.069063663482666, 0.020316822454333305, -0.2456233650445938, 1.274470329284668, 1.4366943836212158, -1.072190523147583, 1.603885293006897, 2.1403324604034424, 1.0511467456817627, -1.0100356340408325, 0.8446277976036072, -0.5013301372528076, 1.3299152851104736, 0.10630775988101959, 0.030717087909579277, -0.7454341053962708, 0.042867813259363174, 0.27471283078193665, -1.2494934797286987, -0.48938626050949097, -0.8807559609413147, 1.123692512512207, 0.2566780745983124, -0.39888688921928406, 1.1264088153839111, 0.19734084606170654, -0.14085671305656433, -0.3168952465057373, 1.232397198677063, 0.19003704190254211, -1.6373344659805298, -0.542034924030304, -0.20272256433963776, -2.2339274883270264, 1.0014278888702393, 1.3300750255584717, -0.8331894874572754, 1.1078757047653198, 0.5655213594436646, 0.29767850041389465, -0.795070230960846, 1.8893755674362183, 0.41634389758110046, -0.6495128273963928, 0.8613080978393555, 0.18685001134872437, -0.788052499294281, 0.783926784992218, 0.43908876180648804, -1.8530898094177246, 0.02702290564775467, 0.4691193699836731, 0.9492044448852539, 1.1903976202011108, 0.20981109142303467, 0.3499864935874939, -1.0445362329483032, -0.059370256960392, -0.6388620734214783, 0.8875414729118347, 1.2528129816055298, -0.5005710124969482, -2.104534387588501, 0.37312859296798706, 1.070427417755127, -0.19651274383068085, -1.35186767578125, 0.7117499113082886, -0.7700091600418091, 0.12448181211948395, -0.1370229721069336, 0.7707749009132385, -1.297911286354065, 1.2619658708572388, 0.3634452819824219, 2.1817891597747803, -0.9703449010848999, -2.1004245281219482, -0.07635363936424255, -1.3810765743255615, 0.05343792587518692, -1.9365161657333374, 0.8332490921020508, 0.10805122554302216, 0.7154265642166138, -0.08888257294893265, -1.323721170425415], [1.0867055654525757, 1.2784245014190674, 1.5493563413619995, -0.46915605664253235, 0.18947383761405945, 0.6076681613922119, -0.5045809149742126, 0.07174279540777206, 0.16100773215293884, 0.9479650259017944, -1.7915608882904053, 1.5388338565826416, 0.11617796868085861, 0.15888290107250214, -0.4711078405380249, 1.1245050430297852, 0.7542927861213684, 0.17949381470680237, 1.0654432773590088, -0.4785926043987274, 0.9013564586639404, -0.7541329264640808, -0.25993266701698303, 1.8968720436096191, -0.5276482701301575, -0.6510735750198364, -0.09663031250238419, 0.304864764213562, 0.643919825553894, 0.22110812366008759, 0.13279855251312256, 0.17734551429748535, 2.6952855587005615, -1.4516453742980957, 0.45589786767959595, 1.0056383609771729, -0.7798035740852356, 0.7619462013244629, 0.7441129088401794, -2.49627947807312, 2.116532564163208, 1.209014654159546, 0.6752270460128784, -0.6275039315223694, 0.009860610589385033, -0.5059984922409058, -0.5378624796867371, 1.2323687076568604, -1.972277283668518, -2.0832488536834717, -0.44921356439590454, 0.9052203297615051, -0.8623208403587341, 0.8986598253250122, -0.515605628490448, 0.04343581199645996, 0.08971168100833893, 0.4967164695262909, 0.7232087254524231, -0.03518795967102051, -0.8157806992530823, -1.2772353887557983, 1.3762617111206055, 0.5273616909980774, 1.2169959545135498, -0.5243619680404663, 0.7723337411880493, 1.0831482410430908, 1.4309183359146118, 0.12340940535068512, 1.4871481657028198, -1.1292555332183838, -0.06398084759712219, -0.8256797790527344, -1.1391786336898804, 0.26604241132736206, 1.1163603067398071, -0.5852137804031372, -0.34536731243133545, -0.4235819876194, -0.3969068229198456, 0.9794490337371826, 1.178123116493225, -1.3403587341308594, 0.6338253617286682, 0.5769339203834534, -0.3426724970340729, -1.5812665224075317, -0.07099278271198273, -1.1167491674423218, 0.5380673408508301, 0.5124229788780212, 1.5495591163635254, -0.4168234169483185, 0.3237552046775818, -0.20110933482646942, 0.5753999948501587, -0.47843167185783386, -0.3676352798938751, -0.9708173274993896, 0.1654462218284607, 0.23545633256435394, 1.8489266633987427, 0.514757513999939, -0.11227350682020187, 1.1593581438064575, 1.0983684062957764, -0.3066542446613312, 0.3081248998641968, -0.19796904921531677, -0.12407417595386505, 0.1844007670879364, -0.10535397380590439, -0.5482580065727234, 0.4842533767223358, -0.43507587909698486, -1.7824065685272217, -0.5566074848175049, 1.1351494789123535, 0.38950616121292114, 0.6453471779823303, 0.2672523558139801, -0.759345293045044, -1.4587411880493164, -1.3216253519058228, 0.026701342314481735, -0.2612505853176117, 0.16981637477874756, -0.5680928826332092, 2.275559186935425, -0.7809404134750366, 1.4128285646438599, -0.33230262994766235, 1.368714690208435, 1.9734169244766235, 1.8542194366455078, 1.5581591129302979, 0.6240490674972534, -1.44435715675354, -1.6084665060043335, 0.311471164226532, 0.9954422116279602, -1.2996279001235962, 1.2532001733779907, 0.11254657059907913, -1.0242154598236084, 0.29750338196754456, 2.3958969116210938, -0.532027006149292, 0.385820209980011, 0.4376291334629059, -0.31634506583213806, 1.2258275747299194, 1.5633655786514282, 0.727514922618866, 1.6325454711914062, 0.24961361289024353, 0.8760640025138855, -0.9889234900474548, 1.232572317123413, -0.07126645743846893, -1.7165855169296265, -0.023539261892437935, -1.448440670967102, -1.0981407165527344, 0.7907008528709412, 0.755804181098938, -1.076499104499817, -2.3691580295562744, 0.5509849190711975, -1.312650442123413, 0.6718279123306274, 0.6804459095001221, 0.3558708131313324, 2.0354437828063965, 0.6928777694702148, -0.2046106606721878, -0.547593891620636, -0.08770567178726196, 0.2626189589500427, -1.425778865814209, -0.39719682931900024, -0.06923463940620422, -1.6688536405563354, -1.4614193439483643, 0.203951895236969, -1.579924464225769, 1.1455109119415283, 0.4338573217391968, 0.20565669238567352, -0.7766925692558289, -1.1614645719528198, 0.11115686595439911, 0.28888702392578125, 0.9417546391487122, -2.8431854248046875, 0.4121614098548889, 0.7099647521972656, -0.7370193600654602, 0.2953961491584778, -0.6857980489730835, -0.5967335104942322, -1.5586283206939697, 0.6605083346366882, 0.014492183923721313, -0.4513337016105652, -0.45136499404907227, 0.2533683478832245, 1.4060606956481934, 1.4134942293167114, 0.10088140517473221, -0.9079704284667969, 0.543297529220581, -1.781048059463501, 1.5162098407745361, 0.7398636937141418, -0.5872739553451538, -0.25624075531959534, -0.7941716909408569, 0.33975306153297424, -0.990093469619751, 0.0042442213743925095, -0.7251123189926147, -1.900951862335205, -0.08640560507774353, -0.4937237799167633, -0.9898462891578674, -1.383514165878296, -0.6169363260269165, -0.5696225166320801, -0.9122312664985657, -0.7499688863754272, -0.3812601566314697, -1.028260588645935, 0.747689962387085, 0.2297155261039734, 0.5489402413368225, -0.9885371923446655, -0.5459762215614319, 0.0749150738120079, -0.12720999121665955, -0.7415739297866821, -0.6906618475914001, 0.5168371200561523, 0.8563808798789978, 1.5316452980041504, -0.33282729983329773, 0.6848935484886169, 0.7763163447380066, 0.2391977608203888, 0.1119682714343071, -0.018552973866462708, 0.838995635509491, 0.84661865234375, -0.8489099740982056, -0.33484768867492676, -0.864144504070282, 0.1645277887582779, -0.5136919617652893, 0.33084914088249207, 1.37104070186615, -0.5106077790260315, 1.0619045495986938, 0.21827638149261475, -0.16839981079101562, -0.15022802352905273, -0.1192423403263092, 0.6888893842697144, 1.8788459300994873, 0.1094270646572113, -0.3037714660167694, 1.1992064714431763, 0.6966105103492737, -0.729374885559082, 0.8606025576591492, -0.29687172174453735, 0.26444125175476074, 0.7383976578712463, -0.5295526385307312, -0.16492488980293274, 0.47479549050331116, -1.445091724395752, -0.3749501705169678, -1.814292073249817, 0.20337872207164764, -0.19183498620986938, -1.0569311380386353, -0.7521114945411682, -1.7632743120193481, -0.16254669427871704, -0.9335660338401794, -0.005041591357439756, 0.82036292552948, -0.5381419658660889, -1.1296502351760864, 1.2007660865783691, -0.6725906133651733, 1.0600894689559937, 0.399906188249588, 1.1169791221618652, -0.5033147931098938, -1.7148722410202026, -0.5634086728096008, 0.08310487121343613, 1.7325700521469116, -0.8841493129730225, 0.4452737271785736, -1.3569018840789795, -0.3912760317325592, 1.04180908203125, 0.9348971247673035, 0.4098690152168274, 0.05691298469901085, 0.4461587965488434, 0.9168674349784851, -0.166049525141716, -0.16438938677310944, 0.13268038630485535, 0.29127809405326843, -1.7646023035049438, -0.28687289357185364, -0.3795807957649231, -0.11015457659959793, -0.6964725852012634, 0.979057788848877, 0.7114121913909912, 0.04804787412285805, 2.95741605758667, 0.7799949049949646, 1.265699028968811, 0.3307632505893707, 0.5960196256637573, 1.0406795740127563, -0.6925880312919617, -1.4632467031478882, 2.012899398803711, -1.6523816585540771, -1.3419588804244995, -0.25534874200820923, -0.6288350224494934, 0.1088632270693779, 1.5640757083892822, 0.8046324253082275, -2.0810930728912354, -0.6055541634559631, 0.2036105841398239, -0.1690049171447754, -2.0052683353424072, -0.6418989300727844, 1.0629721879959106, -0.03021341934800148, -0.5749692916870117, -1.35654878616333, 0.9667123556137085, 2.441476583480835, -0.8007373213768005, 0.9815280437469482, 0.059402987360954285, -1.0871845483779907, -1.4566655158996582, 0.3417322635650635, -0.1174745038151741, -0.7203549146652222, 0.24668695032596588, 0.4527475833892822, 2.5529747009277344, -0.11294999718666077, -0.7321699261665344, -1.0113532543182373, 0.5204601883888245, 0.8246545791625977, -0.15854474902153015, 0.9949341416358948, 0.14067603647708893, -0.2035696804523468, -0.4583733081817627, 0.07764671742916107, 0.2761838138103485, -0.3142560124397278, -1.0097591876983643, -1.0368677377700806, 1.1874758005142212, -0.5345369577407837, 1.8546991348266602, -1.340250015258789, -0.04945939779281616, -0.28245609998703003, -1.2920069694519043, -1.7588521242141724, -0.8605932593345642, 1.1884735822677612, -0.9020102620124817, 0.3018525540828705, 0.37482133507728577, 0.5773208141326904, -2.360050678253174, -0.7009514570236206, -2.1005260944366455, 0.3891691267490387, 0.661761999130249, 1.0360552072525024, 0.6896098852157593, 1.174634575843811, 0.9921529293060303, 0.3795561194419861, 0.8721656203269958, -0.8940142393112183, 0.18695662915706635, -1.046280860900879, -0.6527648568153381, 0.12821511924266815, 1.3738882541656494, -0.2148071974515915, 0.63434237241745, 0.049480222165584564, 1.3298906087875366, 0.6997799873352051, -0.5140925645828247, 0.9759872555732727, 0.3168642222881317, -0.11729485541582108, 1.7073822021484375, 0.011633767746388912, -0.6350380182266235, -2.1679790019989014, -1.5238959789276123, 0.8872843980789185, -3.318207025527954, 0.9785914421081543, -1.195913314819336, -1.2089858055114746, 0.9347200989723206, -1.7994825839996338, -0.6514396667480469, -0.9606796503067017, -0.2923455238342285, 1.1065324544906616, 1.822791337966919, -1.7144705057144165, -1.0787526369094849, 2.5342705249786377, 0.7566525936126709, 2.03903865814209, 1.9315170049667358, 0.10748536139726639, -0.7671712636947632, 0.9419021010398865, 0.465327650308609, 1.6018707752227783, 0.5057203769683838, 2.515563726425171, -1.596454381942749, -1.6713742017745972, -0.47255590558052063, -1.1325061321258545, 0.13260510563850403, -0.6940637826919556, 1.3426803350448608, 0.17726820707321167, 0.46578723192214966, -0.5269616842269897, 0.009393708780407906, -0.36429035663604736, -0.9257249236106873, 0.292041540145874, -1.9813727140426636, -0.47209078073501587, -0.6459798812866211, 1.2038891315460205, -0.32037296891212463, -1.7900313138961792, -0.1644279807806015, 0.34769004583358765, 0.4562933146953583, 0.8684993386268616, 0.7828713655471802, 0.500903308391571, 1.3952851295471191, -0.34207436442375183, 0.5578741431236267, 0.20667842030525208, 0.4176352620124817, 0.5167784094810486, -0.2935008108615875, -0.23415564000606537, 0.05351148918271065, -2.2525107860565186, 0.36851024627685547, 0.9700836539268494, -0.19662129878997803, -0.3926700949668884, -0.22716741263866425, 1.1194394826889038, -0.5218908786773682, 1.025266408920288, -1.4576786756515503, -0.6311224102973938, 1.5303572416305542, -0.563112199306488, -1.4628405570983887, 0.2690751850605011, -0.48826566338539124, -0.08660133928060532, 0.5360593795776367, -0.49560126662254333, 0.4366922080516815, -0.2786898612976074, 0.027160149067640305, -0.2824920415878296, -1.38395357131958, -0.8901569843292236, -0.684282660484314, 0.7503078579902649, -0.856674075126648, -1.0401198863983154, 0.14473873376846313, 0.7038787603378296, 0.08703187108039856, -0.3441610038280487, -1.5623241662979126, 0.41530686616897583, 1.084520697593689, -0.28104883432388306, 0.20837999880313873, -0.858920156955719, -0.685768187046051, 0.12657158076763153, -0.161007821559906, 0.49448350071907043, 0.18308772146701813, -0.4820522665977478, -0.6312243342399597, 0.5201894640922546, -1.207971215248108, 0.4873019754886627, -0.37749508023262024, -1.4157623052597046, -0.9672249555587769, -0.40862056612968445, 2.4389820098876953, -0.17639559507369995, -1.2984966039657593, 0.9155725836753845, -1.0904974937438965, -1.4000341892242432, 0.3212434947490692, 0.9652705788612366, 1.9689420461654663, 0.3174821734428406, -0.8159971237182617, 0.02459694817662239, 1.3279169797897339, 1.603293776512146, -0.8524019718170166, 1.6706289052963257, -0.2314881682395935, -1.1944411993026733, 0.6653188467025757, 0.13061486184597015, -1.486008644104004, -0.9862831234931946, 0.017934363335371017, 0.4401722848415375, -1.1427700519561768, 1.0992437601089478, -0.8849238157272339, -0.977422297000885, 0.6899287700653076, 1.213960886001587, -0.33730924129486084, -0.2322370409965515, 0.3348209261894226, 1.1075410842895508, -0.4435546100139618, -0.20692668855190277, -0.6350822448730469, -1.6971114873886108, 0.6484492421150208, -0.4355412423610687, 0.9786455035209656, -0.5152884125709534, 1.3162997961044312, 0.20981739461421967, 1.000900149345398, -1.3026347160339355, -0.699661135673523, -0.5888822674751282, 1.8573037385940552, 0.4189995527267456, -0.5224697589874268, -2.257063388824463, -0.9998037815093994, 1.3192514181137085, -0.5174021124839783, 0.5796229839324951, -1.6926093101501465, 0.37162235379219055, 1.0309486389160156, 0.411387175321579, -1.0179460048675537, -1.1097710132598877, 0.5603137016296387, -1.0967857837677002, -1.4675540924072266, 0.11683053523302078, -1.1527843475341797, -1.207453727722168, -0.24649806320667267, -0.16238678991794586, -1.4249083995819092, -0.03483868017792702, 1.1879286766052246, 0.6601441502571106, 1.0070898532867432, 0.9389218688011169, 0.02158287912607193, 0.20455066859722137, -1.0270625352859497, -0.9577014446258545, -0.8138734698295593, -1.2097572088241577, 1.9267843961715698, 1.288076639175415, -1.6067025661468506, 0.5958104133605957, 0.0485258549451828, 0.3873905837535858, 0.37004318833351135, 0.9126068353652954, -1.2261561155319214, -1.0041929483413696, -0.4739840030670166, 0.8909783363342285, -0.0040915049612522125, -0.09522517770528793, -0.26173269748687744, 0.7525641918182373, 0.3517623841762543, 1.397526741027832, 1.9028090238571167, 0.6414739489555359, -0.8676348328590393, -1.229722499847412, -0.12288501113653183, 0.22080574929714203, 0.046938806772232056, 0.31497687101364136, 0.0467599593102932, 1.3551899194717407, -0.02771497704088688, 1.2516273260116577, 0.22487565875053406, -0.3376784324645996, 0.11301762610673904, -0.5219265818595886, 0.12705697119235992, -2.173874616622925, 0.07154014706611633, 0.9641303420066833, 1.1165989637374878, -0.7131199836730957, -1.0691282749176025, 1.2146930694580078, -1.3858007192611694, -0.09858199208974838, -0.32641884684562683, 0.26649922132492065, 0.8899468183517456, 1.3365172147750854, -1.964390754699707, -0.3831944763660431, -2.0792295932769775, -0.7440024614334106, -1.5841493606567383, -0.3483312726020813, -0.6066698431968689, 0.2043686807155609, 0.4897875189781189, 0.030248722061514854, 0.6247978210449219, 1.020386815071106, -0.2216227501630783, -0.12095677852630615, -0.8577427864074707, -1.0087170600891113, 0.6084603667259216, 1.461830973625183, 0.06525696069002151, -1.582666277885437, -0.8418800234794617, 0.1439545899629593, 0.011040208861231804, -1.0005865097045898, 0.06763122975826263, 0.018681855872273445, -1.5087103843688965, -1.0774493217468262, 0.5108246803283691, -0.9952746629714966, -0.3000502288341522, 1.1313873529434204, 0.7274208664894104, 0.596291184425354, -1.3109022378921509, -0.8217450976371765, -0.11586257070302963, -0.13312695920467377, 0.6800580024719238, 0.6435085535049438, 0.9830313324928284, 1.3931546211242676, -0.16015344858169556, -1.0427374839782715, 0.8471361398696899, 0.12141110002994537, 0.14870479702949524, -0.5077730417251587, -0.1819935441017151, 2.309354305267334, -1.8656221628189087, 0.6879533529281616, 0.2887348234653473, 0.652887225151062, -0.5002766251564026, 0.004345486871898174, -0.46375134587287903, -0.27500835061073303, -0.44483277201652527, 1.6371688842773438, 1.0553972721099854, 0.5773827433586121, 0.42880013585090637, -0.5583151578903198, -1.481461763381958, 0.33831077814102173, -0.9340448975563049, -0.5810448527336121, -0.9510481953620911, -0.49103814363479614, -0.5529094338417053, 1.264194369316101, 1.4465031623840332, -0.02124761790037155, 0.541317880153656, -0.06131754070520401, 0.26640504598617554, -0.45824408531188965, 0.36099186539649963, -1.5934765338897705, -2.510453701019287, 1.8234201669692993, 0.13090737164020538, -0.005060211755335331, 0.5215069055557251, 0.13616330921649933, 0.7791450619697571, -1.1288950443267822, -0.67979896068573, 1.2564243078231812, 0.24343685805797577, -0.7543305158615112, -1.3893976211547852, 1.7684932947158813, 1.0134586095809937, 0.5476924777030945, -1.4833333492279053, -0.6638368964195251, -1.1973196268081665], [-1.9561352729797363, -0.6186386346817017, -0.4509485065937042, 2.0621652603149414, -1.3919777870178223, 0.9517075419425964, 0.2964707911014557, 0.6648410558700562, 0.09623889625072479, 0.943835973739624, -0.13190694153308868, 1.136327862739563, 0.7013256549835205, -0.9884242415428162, -0.2347487062215805, 1.8009089231491089, 1.7164148092269897, -0.8626373410224915, 0.3358798027038574, 1.6762572526931763, 0.5070697665214539, -0.63663649559021, 0.6607042551040649, 0.17615056037902832, -1.6644023656845093, 0.5014516115188599, 0.1956189125776291, 0.6653293371200562, 1.8295105695724487, 1.8376010656356812, 1.607598900794983, 1.686248779296875, -0.9423644542694092, -1.036793828010559, 0.38926613330841064, 0.009735135361552238, 0.46871763467788696, -0.44741663336753845, 0.6148169636726379, 1.632997989654541, -0.0985686182975769, -0.07991823554039001, -0.5002676844596863, -2.0685150623321533, -0.20776642858982086, -0.20925085246562958, 0.26831915974617004, -0.6929259300231934, 1.4130582809448242, 1.0703016519546509, 1.9330663681030273, -2.6911892890930176, -0.5521084666252136, 0.7103496789932251, 0.49236875772476196, 0.4483449161052704, 0.032628875225782394, 1.1220213174819946, -0.2730626165866852, -0.11452760547399521, 2.589585065841675, -0.2044467329978943, 1.0634642839431763, 0.24870160222053528, 0.8891845941543579, 0.13373003900051117, 1.065890908241272, 1.290838599205017, -0.030330486595630646, -0.40796899795532227, -0.5143083333969116, -0.7964395880699158, 1.611721396446228, 1.9388848543167114, -1.7393443584442139, -0.7072798013687134, 0.23553074896335602, 0.9998807907104492, -1.1055439710617065, -0.4619038701057434, 0.2251373827457428, 0.9623318314552307, -0.8107923865318298, 1.1095657348632812, -0.7945343852043152, 1.0130478143692017, -0.5576779246330261, -0.2742217183113098, 0.8606982231140137, 0.488957017660141, 0.8228431344032288, -0.08047891408205032, -0.1931498497724533, 1.3112602233886719, 1.9687261581420898, 0.5054379105567932, 0.5023000240325928, -0.7023511528968811, 1.9364233016967773, -0.30222031474113464, 0.47881880402565, 1.4453110694885254, -2.7567243576049805, -0.27825024724006653, -1.31974196434021, -0.40723612904548645, 0.043483980000019073, -1.4395112991333008, 1.1671744585037231, 2.4111170768737793, -0.2733227014541626, 0.021679608151316643, 1.758480429649353, 0.4295736253261566, -1.0879656076431274, 0.6125932931900024, -0.5986605286598206, 1.3499176502227783, -0.6350637078285217, 1.4948408603668213, -1.2446765899658203, -0.1724323332309723, -2.096357583999634, 2.621976852416992, 0.07641811668872833, -0.27198949456214905, 0.4625967741012573, 0.1928776651620865, 0.5325186848640442, 2.3613147735595703, 1.6997705698013306, -0.4014967978000641, 0.3815673887729645, -0.033014461398124695, 0.5603039264678955, -0.9990184903144836, -0.32818126678466797, -1.5081464052200317, 0.6034889221191406, -0.6312136650085449, 1.2480789422988892, 0.21740387380123138, -1.400570273399353, 1.8029427528381348, -2.024548053741455, 2.9839236736297607, 1.8156884908676147, 1.7646232843399048, 1.8839285373687744, 1.3172839879989624, -0.013717986643314362, -0.6472011804580688, 0.2075345665216446, -2.451782703399658, 0.4809681475162506, -0.019390780478715897, 0.27464771270751953, -0.43593138456344604, -1.4770374298095703, 0.11259832978248596, -0.4155661463737488, -0.4049959182739258, 0.06912162899971008, 0.8862210512161255, 1.3444191217422485, -1.2153971195220947, 0.19670109450817108, -2.0028605461120605, -1.1944774389266968, -1.1534245014190674, -0.28362563252449036, -0.08792088180780411, 0.2973158061504364, 0.015298034064471722, -0.13488881289958954, -1.429579257965088, 0.45734918117523193, -1.0425994396209717, 0.6147720217704773, -0.3725418448448181, 2.0121264457702637, -1.060279130935669, 0.4756602346897125, 1.0848058462142944, -0.49623605608940125, -0.34185463190078735, 1.7097560167312622, 0.983443558216095, 0.3454320728778839, -0.5505116581916809, -0.5310323238372803, -1.6211519241333008, 1.718400478363037, 0.08746013045310974, 0.6862483024597168, 1.4492263793945312, 0.865128755569458, -0.0832860916852951, 1.4887546300888062, -1.8013560771942139, 1.3847578763961792, 0.5873798131942749, 0.011972538195550442, 0.7790274620056152, 1.2487391233444214, -0.5858578085899353, 0.41025853157043457, -0.26500335335731506, 0.0009971198160201311, 0.9690607786178589, -0.30675649642944336, -0.9217151999473572, -0.16509845852851868, 1.730567216873169, 1.4903066158294678, 0.9495775699615479, 1.7467710971832275, -0.031258609145879745, -0.5018876194953918, -0.6876056790351868, 0.5458019375801086, 0.6132803559303284, -1.0979254245758057, 2.6111979484558105, 0.961665689945221, -1.4689913988113403, -1.75819730758667, 0.2760692238807678, 0.21743394434452057, -1.1640727519989014, -0.018569961190223694, -0.07169289886951447, -0.19609400629997253, 1.2190656661987305, -1.3789595365524292, -0.18526101112365723, -1.3724366426467896, 0.009922435507178307, 0.13068629801273346, 2.171049118041992, -1.0387661457061768, 0.13689593970775604, -0.23585209250450134, -0.0004091660666745156, -0.7986379265785217, -1.3062998056411743, -0.7685028910636902, -0.8960750102996826, 0.8996874094009399, -0.27451518177986145, 1.3953421115875244, -0.9873231053352356, -1.913283109664917, -0.6734384298324585, -0.4551069736480713, 2.0527145862579346, -1.2158671617507935, 0.7838758826255798, -0.4044475853443146, -0.7057141065597534, 0.9159167408943176, -0.6659210920333862, 1.886397361755371, 0.6517815589904785, 0.8118091821670532, -0.21460866928100586, 0.4296376705169678, 1.0245953798294067, -0.7569696307182312, -0.06119510531425476, -0.8984799981117249, -0.3213730454444885, 0.17869827151298523, -1.9188278913497925, 0.01305533666163683, -0.6574828624725342, 0.8039892315864563, -1.2342537641525269, -0.41760924458503723, -0.30346670746803284, 1.0987108945846558, -0.047502290457487106, 1.5451606512069702, 0.8268771767616272, -0.9928683638572693, -1.4015086889266968, -0.2934398055076599, 0.5345712304115295, -0.26106536388397217, -1.1552506685256958, 0.3802092969417572, -0.4147556722164154, -0.9794937968254089, 0.12420783936977386, -0.3276590406894684, -1.7691361904144287, -0.27421483397483826, 0.29067957401275635, -0.957682192325592, -1.6462244987487793, 0.5102518200874329, 0.9099146127700806, -0.3700123131275177, 0.03470037877559662, -0.9439483284950256, -0.11613169312477112, -1.5975968837738037, -0.5305702686309814, 0.029791317880153656, -0.8327968120574951, -0.32907387614250183, -0.11399123817682266, 1.1146669387817383, -0.7159637212753296, -1.5991712808609009, -0.9493930339813232, -0.9363129734992981, 0.2689962089061737, 1.085500717163086, -1.4473892450332642, -0.08706393837928772, 0.28589242696762085, -0.5059764385223389, -0.38087767362594604, -1.056312918663025, 0.9849520921707153, 0.4286854565143585, -1.2945125102996826, 0.010541040450334549, -1.5170398950576782, 0.19278940558433533, 0.6971812844276428, -2.1530778408050537, -0.5779186487197876, -0.4445229768753052, 1.9542529582977295, 0.39522916078567505, 0.5080580711364746, -0.29508984088897705, 0.001000995864160359, 0.5233620405197144, 0.027851320803165436, 0.4692448079586029, -0.25449037551879883, -1.3646854162216187, 0.15975530445575714, -0.033886589109897614, -1.5859111547470093, -1.1383464336395264, -0.6909943222999573, 1.3107168674468994, 0.47095200419425964, 1.4923968315124512, -2.359754800796509, 0.24665425717830658, 1.4105793237686157, -0.5737679600715637, 0.33174756169319153, -1.027807593345642, 0.2620120942592621, -0.15724734961986542, -1.5682963132858276, 1.0209134817123413, -0.03489190712571144, 0.35307008028030396, -1.7690443992614746, 0.765885591506958, -0.6512317657470703, 0.6859904527664185, -1.1671257019042969, 1.148340106010437, -0.37358078360557556, 0.0398082509636879, -0.542968213558197, -1.7020930051803589, -0.22679145634174347, 1.166976809501648, 1.2212055921554565, -0.37223750352859497, 1.272823452949524, -0.1398024559020996, -1.1771469116210938, -0.09480076283216476, 0.04231884330511093, 1.028875708580017, -2.169290542602539, 1.203652262687683, 0.8576732873916626, 0.6141934394836426, -1.3708382844924927, -1.5402003526687622, 1.0172713994979858, -0.2900179624557495, 0.996673583984375, -1.1762717962265015, 0.6530174612998962, 0.5260802507400513, -1.3334910869598389, -0.30101290345191956, 1.7765775918960571, -0.771280825138092, 1.1031838655471802, -0.009343351237475872, -0.9562903642654419, -0.563671350479126, -1.0532844066619873, 1.7368940114974976, -1.5748997926712036, -1.2248843908309937, -0.008893350139260292, 0.958797812461853, -1.3419239521026611, -0.9059275388717651, -0.266335666179657, -0.38361304998397827, 0.013671759516000748, -0.6851789951324463, -0.06832952797412872, 0.6846504807472229, -1.1941864490509033, -1.6545677185058594, -0.7934851050376892, 0.07878555357456207, 2.5848820209503174, 0.8030512928962708, 0.4699844419956207, -2.9683918952941895, -1.9291274547576904, 0.5984709858894348, -0.6315966844558716, -0.7686052322387695, 0.7451121807098389, 1.1476413011550903, 0.5188243985176086, 0.1336398869752884, -1.8686552047729492, -0.2683287560939789, 0.8022715449333191, -0.2539771795272827, -1.5430728197097778, -2.4806768894195557, -0.4104160964488983, 0.5079609155654907, -0.7989580035209656, -0.26173049211502075, 0.3666597008705139, -0.5630687475204468, -0.17526043951511383, -0.6815651059150696, -0.7875736355781555, 0.4361855685710907, 0.5218626260757446, -0.2642683684825897, -1.4791181087493896, 0.3758366107940674, 0.7722018361091614, 0.6928211450576782, 0.6756012439727783, -0.024023108184337616, -0.26115936040878296, 0.5191236734390259, 0.5968677401542664, 0.5899055600166321, -0.873755156993866, -1.834809422492981, 0.3539878726005554, 0.3213264048099518, 0.25850000977516174, 0.5130177736282349, -1.545249104499817, -0.6434493064880371, 0.625659704208374, -0.4889014959335327, -0.18505659699440002, 1.4145488739013672, 0.737933874130249, -0.5062530636787415, 0.28059783577919006, -0.7274827361106873, 0.42446067929267883, -1.336965799331665, -1.6473642587661743, 1.5080560445785522, 0.998719334602356, -0.8954718112945557, 0.47810253500938416, -2.393439531326294, 0.05299091339111328, 1.1232452392578125, 0.7584363222122192, 0.17059296369552612, -0.04808465391397476, 0.2397274225950241, 2.068917989730835, -0.4332948625087738, 0.24840931594371796, 0.7830125093460083, -0.8386057019233704, 1.5290390253067017, -1.8767285346984863, -0.20121344923973083, -0.4181830585002899, -0.05255497246980667, -1.2578223943710327, -0.36031702160835266, 0.3055993914604187, -0.7970383763313293, 0.8911043405532837, -1.9436858892440796, -0.14532341063022614, 1.6022260189056396, -0.9211487770080566, 0.08654060959815979, -0.3517085909843445, -0.5520389080047607, 0.6562939286231995, -1.2986652851104736, 0.522024393081665, -1.3743579387664795, -0.2568081021308899, 0.7702167630195618, -0.5766310095787048, 0.03511640056967735, 2.21675968170166, 0.15450114011764526, 0.2733666002750397, 0.6449499726295471, 0.5715423226356506, -0.001013062079437077, -0.2657351493835449, 0.22902081906795502, 0.34397074580192566, 1.3181095123291016, 0.07623742520809174, 0.7872352004051208, 1.9347259998321533, -1.630581021308899, -0.2226509302854538, -0.2612513601779938, -0.16851627826690674, 1.3848055601119995, -0.28453269600868225, 0.17696955800056458, 1.6363568305969238, -2.115297555923462, -1.2642204761505127, -1.1577556133270264, 0.0112222321331501, 0.5108578205108643, -0.8758467435836792, 1.3603806495666504, 1.594428300857544, -0.7878562808036804, -0.5531755685806274, -1.0200974941253662, 0.6221686601638794, -0.4489533603191376, 0.9251927137374878, 1.094044804573059, 0.011566142551600933, -0.8388446569442749, -2.5072414875030518, 0.3161245286464691, 0.49922454357147217, 0.048331841826438904, -0.8405028581619263, 1.0554578304290771, -0.06476625055074692, 0.6571632623672485, 0.7843882441520691, 0.9735424518585205, 0.6596660614013672, -1.5768284797668457, 0.3574007451534271, -0.09761934727430344, -0.5344736576080322, -0.8937587738037109, -0.9605467915534973, -0.4825194776058197, -0.30066484212875366, 0.04643252119421959, -1.8899720907211304, -0.9040243625640869, 0.6517100930213928, 1.0687012672424316, -0.2940533459186554, -0.49651435017585754, 1.8432635068893433, -1.0362122058868408, 1.0136630535125732, 0.8229461908340454, 0.35546815395355225, 1.047365665435791, 0.6581311821937561, -0.15559585392475128, 0.7194499969482422, 2.492011547088623, 0.18992504477500916, 1.0135672092437744, 1.9028509855270386, -0.89543217420578, -0.009730338118970394, -0.9435257911682129, 0.13680818676948547, -0.4958488345146179, 1.1798063516616821, -0.2554091215133667, 0.7040203213691711, 0.21009676158428192, 0.46009770035743713, -0.8798585534095764, 0.9744028449058533, -0.07933405786752701, -0.18753351271152496, -0.7630581259727478, 0.47712063789367676, -0.4740854799747467, -0.7128042578697205, -0.02701110579073429, -2.22501540184021, 1.1788321733474731, -0.4501776695251465, 0.9694119095802307, -1.2082538604736328, -2.0163350105285645, -0.9255067706108093, 0.38360825181007385, 1.2636175155639648, 1.3740959167480469, 1.1140544414520264, -0.7519196271896362, -0.5291501879692078, 0.37532296776771545, 0.7309287786483765, -0.9443048238754272, -0.9530680179595947, -1.3230432271957397, 1.1864603757858276, -0.9254799485206604, 0.2899118661880493, 0.07196220755577087, 0.4772261679172516, -1.525555968284607, 0.05075256898999214, 0.8878450989723206, 0.6089035868644714, -0.3634834289550781, 0.7572380900382996, 0.2526753544807434, -0.31332916021347046, 0.08753372728824615, 2.27520751953125, -0.46087726950645447, -1.4980342388153076, 0.7024935483932495, 0.8037551045417786, 0.535528302192688, -0.0928567498922348, 1.0214471817016602, 1.983779788017273, 1.484076976776123, -2.036027193069458, -0.81255042552948, 0.2745230197906494, -2.125659704208374, 1.799070119857788, 0.82892906665802, -0.7470197677612305, -0.9733206629753113, 1.1900867223739624, -0.28954556584358215, 0.34309542179107666, -1.0202194452285767, -1.0697063207626343, -0.4216064214706421, 0.07177141308784485, 0.8490346670150757, -0.48254096508026123, 0.809961199760437, -0.4288567900657654, -0.8414871692657471, -0.39709821343421936, -0.44674891233444214, -1.2165982723236084, -1.8341470956802368, 0.5311822891235352, 0.23078875243663788, 0.7491198778152466, -1.5551023483276367, -0.33416950702667236, -1.3346447944641113, 1.323528528213501, 1.6895853281021118, 0.62850022315979, 1.4196950197219849, -1.4341109991073608, -0.12736019492149353, 0.23806753754615784, 0.5747923851013184, 0.11989305168390274, 0.7405656576156616, 0.32571977376937866, -0.2594550549983978, 1.2769094705581665, -0.3746349513530731, 0.7734950184822083, 0.6396242380142212, -1.0351791381835938, -0.3653961718082428, 2.14564847946167, 0.11328727751970291, 0.7695674896240234, -0.34169280529022217, 1.193582534790039, 0.17962823808193207, 1.0963046550750732, 1.012523889541626, -0.7127367258071899, 0.1357974261045456, 0.7785962224006653, -0.8609858155250549, 0.19649116694927216, -0.08978724479675293, 0.7811301350593567, -0.9017329812049866, 0.56773442029953, 1.0248819589614868, -0.36805739998817444, -0.27396878600120544, 0.8566177487373352, -0.37141311168670654, 0.6328177452087402, 1.3248978853225708, -1.1808547973632812, 0.5893692970275879, 1.3422967195510864, -1.2961814403533936, 0.33067587018013, 0.5967631936073303, -0.2614883482456207, 0.3748628795146942, -0.9127757549285889, 0.1516885608434677, -0.622958242893219, 0.8088592886924744, 0.12791456282138824, 0.16486133635044098, 1.053086280822754, 0.49333351850509644, -0.5030860304832458, -0.13058894872665405, -0.5847992300987244, 0.6429122686386108, 1.3863160610198975, 2.0209808349609375, 0.9259190559387207, -0.618033766746521, 0.8591115474700928, -0.9111837148666382, -1.2478030920028687, -0.9006377458572388, 0.47496098279953003, -1.5056792497634888, 0.634263813495636, -0.013837608508765697, 1.7764109373092651, 0.09751413762569427, -0.2857936918735504], [-0.9881327748298645, -0.719590961933136, 0.3226592540740967, -1.2821581363677979, -2.5190320014953613, -0.4705454409122467, -0.09168461710214615, -0.17305399477481842, -0.30059537291526794, -0.8019385933876038, -1.4530211687088013, 1.483402132987976, 0.7742848992347717, 0.40878963470458984, -0.4222627580165863, -0.723238468170166, -0.18999935686588287, 0.668760359287262, 0.139714315533638, 0.03098159097135067, 0.6264334321022034, -0.6061606407165527, -0.8082452416419983, -0.3295974135398865, -0.14803488552570343, 1.7806276082992554, 2.278878688812256, -1.0242700576782227, -0.7542235851287842, 0.028527405112981796, -1.8943989276885986, -0.5827973484992981, 1.3332651853561401, -0.7301682233810425, -0.457870215177536, 1.0689845085144043, 1.766133427619934, -0.20969733595848083, -0.0071473801508545876, 0.30371755361557007, 0.9110375046730042, -1.711135745048523, 0.31814098358154297, -0.4007216989994049, 0.2990729808807373, 0.6389296054840088, -2.3141350746154785, 1.017167091369629, 0.10049862414598465, -0.31702226400375366, 0.5351295471191406, 0.535685122013092, 2.0775949954986572, 0.7908558249473572, 0.875169575214386, 0.30275532603263855, 0.7424070835113525, -0.8176949620246887, 0.8128767609596252, 0.2511305809020996, 0.8681150674819946, 0.9213606119155884, -1.3527017831802368, 0.4838619530200958, 0.7507984042167664, 1.0595717430114746, -1.8941679000854492, 0.4269596040248871, -1.3025435209274292, -0.6847702860832214, 0.35628923773765564, 0.5777961015701294, 0.1192890852689743, -0.3090815842151642, -0.3921477198600769, 0.7716019749641418, 0.8379608392715454, 0.14864026010036469, -1.7088377475738525, -2.221191167831421, -1.5511735677719116, -0.4694649577140808, 0.7527967095375061, -0.38479524850845337, -0.5387306809425354, -0.16804999113082886, -0.28126123547554016, -0.4739505648612976, -0.8127437233924866, -0.7317058444023132, -0.2302631139755249, -0.02386036328971386, -0.5976517796516418, 0.8103948831558228, -1.1052262783050537, 0.43995988368988037, -0.7270091772079468, 0.5776807069778442, -1.0319244861602783, -0.039792001247406006, 0.9158586859703064, 0.7681058645248413, -0.8581251502037048, -0.6286773085594177, -1.1886467933654785, -0.38422709703445435, -1.5789512395858765, -0.4395332634449005, -0.8751083612442017, -0.15675358474254608, -0.2987304627895355, -0.13184893131256104, -0.041748445481061935, 0.8800116777420044, -0.3035595715045929, -0.5773972272872925, 1.0130571126937866, -3.0971109867095947, -1.019729495048523, -1.0572736263275146, 1.4847759008407593, 0.5819591283798218, 0.5084769129753113, -0.01966206356883049, 0.336016446352005, 1.3687275648117065, -1.6939301490783691, -0.906567394733429, 0.4282998740673065, -0.8430169224739075, 0.7515931725502014, 0.7778748869895935, 1.1019514799118042, -1.632545828819275, 0.16527636349201202, 1.048998475074768, -0.41798093914985657, 1.0076035261154175, -0.2671416103839874, -1.1490713357925415, 1.6766353845596313, -1.4191858768463135, 0.11228486150503159, -0.12970218062400818, -0.5926908850669861, -0.5835670232772827, -1.7263965606689453, 0.2556520700454712, -1.0831040143966675, 1.840080738067627, -0.41977426409721375, -1.638999581336975, 0.34753647446632385, 0.8471285700798035, 1.1387896537780762, -0.35189640522003174, -0.06916315108537674, -0.15696501731872559, 0.5754462480545044, 1.3689063787460327, 0.9642802476882935, -0.38158077001571655, -0.17306064069271088, -0.09135939925909042, -0.5886223316192627, -1.061176061630249, -0.4469180405139923, 0.4961455166339874, 0.491525262594223, -1.0655330419540405, 0.2914392352104187, -0.24413561820983887, 0.2832781672477722, 0.6673806309700012, 0.4403979480266571, 0.8925960659980774, 0.6661436557769775, 0.7923353910446167, -0.15531106293201447, -1.7028580904006958, -2.2327675819396973, -0.21667206287384033, -0.29054492712020874, -1.6113299131393433, -1.4713199138641357, -2.279813528060913, 1.109352707862854, -2.393707036972046, -2.201585292816162, 0.47504401206970215, 0.7667658925056458, -0.6595613360404968, 0.40642428398132324, 0.4855404496192932, -0.6273621320724487, 2.556462287902832, 0.5122396349906921, 0.4091632664203644, 0.02029004693031311, 0.4552302062511444, -0.9181354641914368, -0.06615927815437317, 0.16647689044475555, 0.33903518319129944, -0.3498587906360626, 0.24571675062179565, -0.48562997579574585, 1.3965805768966675, -1.1302235126495361, -1.4082945585250854, -0.033376410603523254, 0.6972333192825317, -0.5449796915054321, 0.057453688234090805, -0.002383718965575099, 0.3030148446559906, 1.4646986722946167, -1.0689305067062378, -0.7277539372444153, -0.021302977576851845, -0.32653284072875977, 0.3123958110809326, 1.6418626308441162, 0.6397521495819092, -0.4425518214702606, 0.7762947082519531, 0.6204574108123779, 2.0173356533050537, -0.9313530325889587, 0.35496747493743896, 1.057826042175293, 0.8622568845748901, 0.2554866373538971, -0.6251978278160095, -0.8561926484107971, -0.9530501961708069, 0.23290792107582092, 0.5000895261764526, -0.26311859488487244, -0.6120239496231079, 0.18965527415275574, -1.501322865486145, 2.425767421722412, -0.717445433139801, 0.29125019907951355, 0.03459315001964569, 0.03395271301269531, 0.36576855182647705, -1.0095770359039307, 1.9647681713104248, 0.3223738372325897, -1.4534173011779785, 1.5097935199737549, 0.7723209261894226, 0.013544132001698017, 0.3050192594528198, -0.7297810316085815, 2.175368547439575, 0.2674781382083893, -0.41943588852882385, -1.4646966457366943, -0.8922489881515503, -1.7795804738998413, -0.11409483850002289, -0.46279609203338623, -0.8534331321716309, -0.24168464541435242, 0.32696712017059326, -0.01265833992511034, -0.5646469593048096, 0.6135839223861694, -0.18182338774204254, 0.22043044865131378, -0.17548872530460358, -0.4313609004020691, 1.8690146207809448, -0.8516082167625427, -1.8459765911102295, -1.5370651483535767, -1.114999532699585, 1.8916741609573364, 2.9882140159606934, -1.9543182849884033, 0.9558712840080261, -0.03397338464856148, 1.9980748891830444, 0.46614140272140503, -0.5081937909126282, 1.00330650806427, 0.23991839587688446, 0.2005196064710617, -1.7144794464111328, 0.37340396642684937, 0.27805519104003906, 0.14899663627147675, -0.17256613075733185, 0.5070222020149231, -0.09324616193771362, -1.2329517602920532, 0.40586793422698975, -1.386931300163269, 0.3102174401283264, 0.22359879314899445, 1.8553746938705444, 0.995359480381012, 1.2268757820129395, 1.0719975233078003, -0.3248131573200226, 0.8674632906913757, 0.9680169224739075, -1.0820595026016235, -0.4298873245716095, 1.5905954837799072, 1.7430678606033325, 1.930702805519104, 1.3842220306396484, -1.9675213098526, -0.7491607666015625, -1.6806347370147705, -1.442848563194275, 0.6762962341308594, -0.4931815266609192, -0.7472770810127258, -0.8275762796401978, -0.7203295230865479, -1.1252087354660034, -1.2876790761947632, 0.9612085819244385, -0.0555557943880558, -1.7300422191619873, -0.5790344476699829, -0.03262937068939209, -0.8385641574859619, 0.49544021487236023, -0.5639927983283997, 0.3287719786167145, 0.5996101498603821, 0.10115988552570343, -0.34357333183288574, 0.5827063322067261, 0.9541518688201904, 1.4253884553909302, 0.4618019461631775, -1.4597084522247314, 1.4503017663955688, -1.5191454887390137, -1.0523018836975098, 2.2104945182800293, 0.3605567514896393, 2.130758285522461, 0.09129612892866135, -0.061534322798252106, -2.2275328636169434, -0.02872343361377716, 0.6772514581680298, 0.3700277805328369, 2.5382254123687744, 0.9162164330482483, 0.6070560216903687, 1.1540567874908447, 0.571623682975769, 0.6848692297935486, -0.6428627371788025, -1.783159852027893, -0.1738455444574356, -0.3439140319824219, 0.29602551460266113, 0.8280916810035706, 0.01394633762538433, -1.2732388973236084, 0.2603145241737366, 0.9248871207237244, -1.0545185804367065, 0.9055227041244507, -0.658136248588562, -0.48140838742256165, 1.4489654302597046, -0.08055926859378815, 0.9471999406814575, 0.8172805309295654, 0.3478477895259857, 1.9477280378341675, 0.770301342010498, 1.1680691242218018, 0.5553953051567078, -1.6523301601409912, -1.8539999723434448, -0.26231929659843445, 0.25813838839530945, 1.81028151512146, 1.1078909635543823, -0.3214717209339142, 0.14279890060424805, 0.3252955377101898, 0.5582308173179626, 0.4928613305091858, -0.690991222858429, 0.7089021801948547, 0.5907830595970154, -0.21083630621433258, 1.7743695974349976, 1.0026956796646118, 1.4298827648162842, 0.02870769239962101, -2.145831346511841, 0.8566006422042847, -0.9256680607795715, -1.110885500907898, -1.210333228111267, -1.4868301153182983, -0.1298336386680603, -1.2591043710708618, -1.7631205320358276, -1.500781536102295, -0.7586806416511536, -1.626328468322754, -0.9956400394439697, 0.40872976183891296, -0.8984672427177429, -0.4855295717716217, 0.001703457091934979, -2.499281167984009, 0.8875236511230469, 1.2757543325424194, -2.159212112426758, -0.37901055812835693, 0.3237491846084595, -0.38668787479400635, 0.4167930483818054, -0.38058018684387207, -0.3344692587852478, 0.5438068509101868, -0.029589124023914337, 0.5155104994773865, -0.7269397377967834, -0.4900762736797333, -0.4974652826786041, 0.08921527862548828, 0.7960538268089294, -0.5178375244140625, -0.6481157541275024, -1.0594258308410645, 0.6961745619773865, -0.37752068042755127, -0.9530309438705444, 0.118794746696949, -0.18815116584300995, 0.7051593661308289, 0.08931359648704529, 0.23699991405010223, -0.718363881111145, -0.8636738061904907, 1.362691879272461, -0.2115524411201477, -1.2597527503967285, 0.7619953155517578, 0.4889882504940033, 0.24664857983589172, -0.47680312395095825, 0.2793143689632416, 0.36643508076667786, -1.7215020656585693, 0.4035674035549164, -1.183614730834961, 1.9788821935653687, 0.9978394508361816, -0.6381261944770813, -0.37241214513778687, -0.1617724597454071, -0.5866013765335083, -0.3950662612915039, -1.1256643533706665, 1.6926770210266113, 0.4455350935459137, -0.1525246948003769, 1.1984343528747559, -0.3917912244796753, 3.196833610534668, -0.789657473564148, 1.7896150350570679, -1.1609171628952026, -0.674431324005127, -0.36239469051361084, 1.5381053686141968, -0.41322848200798035, 0.5208322405815125, -2.419229745864868, -1.6712666749954224, 0.06527438014745712, -0.5147956013679504, 1.0607490539550781, -0.04059777036309242, 0.8945168852806091, -0.10625836253166199, -1.152929425239563, 1.286584496498108, -1.388570785522461, -1.630589485168457, 0.29200488328933716, -0.720959484577179, -0.6203187108039856, -0.7243924140930176, 0.8659632205963135, -1.4433492422103882, -1.7074881792068481, -0.42951688170433044, -1.682005524635315, -0.046196404844522476, 0.40244191884994507, 1.3770711421966553, -0.6346554756164551, -1.3980556726455688, -0.6960002779960632, 1.2463996410369873, 0.35632652044296265, 2.10442852973938, -0.43571075797080994, 0.047186657786369324, 1.7884931564331055, -0.5177680850028992, -0.5380910634994507, 0.309291809797287, 0.4577150344848633, 0.48327428102493286, -0.03072563000023365, 0.43798309564590454, 0.2490352839231491, -0.6633133292198181, 0.7658238410949707, 1.1536881923675537, 1.4750205278396606, -0.7083858251571655, -0.8801367282867432, -1.076808214187622, -0.7264198064804077, -1.650653600692749, -0.17533789575099945, 2.232602596282959, -0.22066760063171387, 0.18364134430885315, -0.7645716667175293, -0.04396240785717964, -0.1990624964237213, 0.37930575013160706, 0.20340387523174286, -1.0824476480484009, 1.1207131147384644, 1.6594185829162598, -0.49815019965171814, -0.16111348569393158, -0.2517407536506653, 0.5525582432746887, 1.6375305652618408, -0.47344934940338135, -0.7383447289466858, 2.7828590869903564, 0.9607905745506287, 1.149875521659851, -0.6676532030105591, 0.9076538681983948, 1.2531640529632568, -0.04918168857693672, -1.1009987592697144, 2.6995456218719482, 0.5388391613960266, -0.7911364436149597, -0.425385057926178, 0.30193933844566345, -0.8039065599441528, -0.793915867805481, -0.563205361366272, 0.5980572700500488, 0.807390034198761, 0.3779589831829071, -0.4307738244533539, 0.9451679587364197, 0.37569665908813477, -1.1806252002716064, 0.3912506401538849, 0.18600326776504517, -0.7733114361763, -0.3619808256626129, -1.375003695487976, -0.738122820854187, -0.442141592502594, 0.06249396875500679, 0.5616331100463867, 0.5397085547447205, -0.03550932928919792, -0.49140220880508423, -0.9512001872062683, 0.04038062319159508, 0.9332349896430969, 0.7226928472518921, -1.8539106845855713, 0.30420342087745667, -0.23530147969722748, 0.4921668469905853, 0.3852613568305969, 1.408639907836914, 1.4956207275390625, 0.8304891586303711, 0.3037680983543396, -1.127558946609497, 1.4079653024673462, 0.18372821807861328, 0.9929913878440857, -0.31192439794540405, 0.7969760894775391, 1.6134625673294067, 0.5727787017822266, -0.230446919798851, 0.9267260432243347, 0.06225699931383133, -1.3927639722824097, 0.8984447717666626, -1.0428756475448608, -0.46532297134399414, 1.5505410432815552, 1.3808609247207642, -0.13525177538394928, -0.6446143984794617, -0.16094407439231873, 0.7565294504165649, -0.8172081112861633, 0.3946399390697479, -2.112943410873413, 0.7031930088996887, -1.4791827201843262, -1.2249219417572021, 0.450893372297287, 0.36718589067459106, -0.4708763659000397, -0.9141148328781128, 1.376530647277832, -1.8798034191131592, -0.15514293313026428, -0.3788355588912964, 0.0068940962664783, -0.7458012104034424, -0.23351198434829712, -0.3055916130542755, -1.0026413202285767, -0.21047015488147736, -0.03187389671802521, -0.06284041702747345, 0.7857437133789062, -0.7823168039321899, 0.6532485485076904, -0.08287299424409866, -0.3578195571899414, 1.5877374410629272, 0.7077108025550842, -1.1715797185897827, 0.2623783349990845, 0.532753050327301, 1.6886768341064453, -0.16532227396965027, -0.2846795916557312, -0.15350040793418884, 1.065163016319275, -1.1178566217422485, -1.159295678138733, -1.3137352466583252, -0.4327959418296814, 0.41833847761154175, -0.7150732278823853, 1.6961193084716797, 0.6369820833206177, -0.6197000741958618, -0.7239825129508972, -0.6306047439575195, -0.03602662682533264, -2.0486481189727783, 0.6144443154335022, 0.9322928190231323, -1.8329616785049438, -1.8874629735946655, -0.703802764415741, -0.4989798367023468, -0.6294662952423096, -1.659883737564087, -0.660914421081543, -1.415658712387085, -1.1472655534744263, 0.13700374960899353, -1.8952137231826782, 0.17358523607254028, -1.5839996337890625, -1.612566351890564, -1.3802435398101807, -0.6234017014503479, 0.7783971428871155, 0.03914880007505417, 0.9184171557426453, 2.109015464782715, -0.8942842483520508, -0.743682861328125, -0.12170983850955963, -1.9875394105911255, 0.9202451705932617, -0.43346261978149414, 0.2417117804288864, -0.480343759059906, 0.15603385865688324, -1.9312970638275146, 1.4814404249191284, -0.6682287454605103, -0.7200725078582764, 1.4024535417556763, 0.15114682912826538, 0.6754053831100464, -0.1453486829996109, 2.0540452003479004, -0.1579592525959015, -1.081985592842102, 0.11767079681158066, -0.9456418752670288, 0.11237119138240814, -0.051984094083309174, 0.6876486539840698, 0.5596652030944824, 0.48449328541755676, 1.1903351545333862, 0.6623539924621582, -0.9671978950500488, -0.17227418720722198, -0.4095202684402466, 2.900468349456787, 1.2859814167022705, 0.5896905660629272, -0.9023555517196655, 0.3317165970802307, 2.0063722133636475, 0.23996785283088684, -0.5283768177032471, 0.9840270280838013, -0.5596197843551636, 0.45515915751457214, 0.07852290570735931, -0.43938711285591125, 1.8203116655349731, -1.7563434839248657, 0.9882360696792603, 0.3382788598537445, 0.3779890835285187, -0.11652319878339767, 1.572782039642334, 0.3221452236175537, -3.6518449783325195, 0.34220850467681885, -0.3205110430717468, 0.4253382384777069, 0.4884290397167206, -0.8480269908905029, -0.5249989032745361, -0.4594961106777191, -0.05514857545495033, 0.47191664576530457, 1.2586685419082642, -0.09246214479207993, -0.6438656449317932, -0.8502578139305115, -0.2442200779914856, 0.827149510383606, -0.5952632427215576, 0.35796111822128296, -0.5555786490440369], [-0.5877032279968262, 0.0826316848397255, 0.27099931240081787, 0.6350783109664917, 0.20248065888881683, -0.848400354385376, -0.7909390330314636, -2.0960264205932617, -1.3601946830749512, -0.8982388377189636, 1.0615938901901245, -0.5535134673118591, 1.0439419746398926, -0.7724981307983398, 0.4671417772769928, 0.14837969839572906, 0.04313927888870239, -0.7833214402198792, -0.0008121214923448861, 1.1993907690048218, 0.011997269466519356, -1.0199110507965088, -1.102041244506836, -0.10429972410202026, -0.6220577955245972, -0.44768160581588745, -0.43095213174819946, 1.207769751548767, 0.25999394059181213, -0.5996502637863159, -0.8143736124038696, -0.2980901002883911, 2.0554215908050537, -0.27597174048423767, -2.4413485527038574, -0.3552999794483185, 0.408149778842926, -0.7504522800445557, 0.12496326863765717, 2.022061824798584, 1.0981602668762207, 0.956015408039093, -1.269910454750061, -0.32118988037109375, -0.13009683787822723, -0.03470582515001297, 1.8733106851577759, 1.4918582439422607, -0.21308307349681854, 0.5366378426551819, -0.9550389051437378, 0.8958097100257874, -1.812831997871399, 1.0055385828018188, -1.0847126245498657, 1.7460757493972778, 1.1803934574127197, -0.2961002290248871, -0.31774240732192993, -1.13429856300354, -0.9780536890029907, -0.9967585802078247, -0.8516044616699219, 0.5573511719703674, -0.5448479056358337, 0.7501227259635925, -0.4038865864276886, 0.11421485245227814, 0.3205254375934601, 0.04881307855248451, -1.1357712745666504, 0.9193231463432312, -1.2090661525726318, 0.4503325819969177, 0.3417980968952179, -0.3393259048461914, -0.0505293607711792, 0.26700860261917114, 1.9762036800384521, 0.20757746696472168, -0.6735451817512512, 1.0493533611297607, 1.2651076316833496, 0.12217390537261963, 1.0423299074172974, 1.4743919372558594, -2.0151941776275635, 0.7997177839279175, -0.44855910539627075, -0.7305944561958313, -0.43407657742500305, -0.07905884832143784, -0.9706024527549744, -0.46559572219848633, -1.237406611442566, 0.18187308311462402, 0.2780511975288391, -0.545799732208252, -0.16135630011558533, 1.4877182245254517, -1.0386621952056885, 1.207231879234314, 1.0891324281692505, 0.482093870639801, 1.0558785200119019, -1.6286120414733887, 1.6355286836624146, -0.8697181940078735, 1.402134656906128, 1.4082553386688232, -0.383298397064209, 0.8633766770362854, 0.7577259540557861, 0.4382680654525757, 0.2228212058544159, 0.6038089394569397, -1.2693463563919067, -1.5435386896133423, 1.3522049188613892, -1.145909070968628, -1.0147552490234375, 0.5640782713890076, -0.33426907658576965, -0.2722738981246948, -0.023453380912542343, 1.5745583772659302, 0.6191698312759399, 0.13374054431915283, 2.193121910095215, 0.503546953201294, 0.19601553678512573, 0.9844714999198914, -0.25492867827415466, -1.1505602598190308, -0.7416772246360779, -0.3199114203453064, 1.662881851196289, 1.2476266622543335, -2.084928512573242, -1.3502174615859985, -0.04720265045762062, 0.13837110996246338, 0.0385623425245285, -0.44120147824287415, 1.0903213024139404, 0.27141207456588745, -0.9918729066848755, 0.010471954010426998, 0.7539780735969543, 0.8967874050140381, -0.2162122130393982, -0.8974509835243225, 1.996423363685608, -0.27065080404281616, 0.2998392581939697, -0.5647873878479004, 2.4529759883880615, 1.6182366609573364, -0.4810754656791687, 0.7805948257446289, -0.24690085649490356, 0.7191961407661438, 0.42747581005096436, 0.5032474994659424, 1.7382017374038696, -1.0437572002410889, 0.32860231399536133, -1.026257872581482, 0.6831524968147278, -1.0161707401275635, 1.8251745700836182, 0.5079380869865417, -1.9877538681030273, 0.0430205836892128, -1.6269631385803223, 0.10704457014799118, -0.24975180625915527, -1.657169222831726, -0.35501208901405334, 0.042504843324422836, 0.8279479146003723, -1.150490164756775, 1.0486565828323364, 0.08318613469600677, -0.8146984577178955, 0.22892332077026367, -0.6558173894882202, -0.18017953634262085, 0.883869469165802, 1.0568124055862427, 0.7480083107948303, -1.271851897239685, -0.5186411738395691, 0.2154070883989334, -0.06068268418312073, -0.04068725183606148, -0.6633334755897522, -1.5299115180969238, -1.0049104690551758, -2.6512067317962646, -0.21543218195438385, 0.6075392961502075, -1.085250735282898, 0.9211417436599731, 0.1284361630678177, -1.046064019203186, 0.10449135303497314, 0.35724616050720215, 0.062388814985752106, -0.5160682201385498, 0.06266902387142181, -0.35467302799224854, 0.5631104707717896, -0.8254377841949463, -0.12769097089767456, -0.6297628283500671, 0.4868045747280121, 1.2699214220046997, -1.1309584379196167, -0.5763018131256104, -1.0601236820220947, -0.12117084115743637, 0.65446937084198, 1.516654133796692, -0.4673607051372528, 0.626556396484375, 0.05291379615664482, 0.1857663094997406, -0.8654783964157104, -1.1425426006317139, 0.11046401411294937, 0.48932379484176636, -0.5228866338729858, 0.6234016418457031, -0.6976694464683533, 0.7562820911407471, 0.5093309283256531, -0.11909569799900055, -0.24698692560195923, 0.7334330081939697, 0.7610599398612976, 0.14368963241577148, -0.6451826691627502, 1.3288768529891968, 0.913184404373169, -1.4519920349121094, -0.533193051815033, 0.4970240890979767, 0.515684187412262, -1.257516622543335, -0.24846893548965454, -0.20661896467208862, 2.0315747261047363, -0.35268673300743103, 0.11103183776140213, -0.609226644039154, 0.735634446144104, -0.3038085997104645, 0.3586483299732208, -2.0245866775512695, 0.43863677978515625, 0.6177243590354919, 0.719229519367218, -1.6965911388397217, -0.26910749077796936, 0.9172617197036743, 0.08127179741859436, -0.1101205125451088, -0.2314131259918213, -0.28813138604164124, -0.12911337614059448, 0.030654899775981903, 0.3656562566757202, -0.692528486251831, -0.7158613801002502, 0.1565278023481369, 1.0608363151550293, 0.3448162376880646, 1.7046140432357788, -2.0851712226867676, -0.1293591856956482, 0.06883327662944794, -0.9759795069694519, -0.34775489568710327, 0.48653385043144226, 1.3665553331375122, -0.8970914483070374, 1.4367468357086182, 0.6250324249267578, 1.2360918521881104, 0.6250888705253601, -0.14373821020126343, 2.4405019283294678, -0.7277624011039734, 0.6092057228088379, -0.7299343943595886, 0.580409049987793, -0.3878139555454254, 0.9444729089736938, 2.331791639328003, 0.8797932863235474, 0.9685366153717041, -0.8828604221343994, -0.9636096358299255, -1.586044192314148, -0.7299932837486267, -0.8683289289474487, 0.7024204730987549, 1.2056431770324707, -0.5711462497711182, -0.16021323204040527, -1.2405173778533936, -0.2715819180011749, 0.6284934282302856, 0.8539855480194092, -0.6983948945999146, 1.08051598072052, 0.5495074987411499, 0.3316258490085602, 0.9359042644500732, -0.5126401782035828, -0.9382205009460449, 0.6523670554161072, 1.5140085220336914, 1.568246841430664, -3.5367507934570312, 0.24416594207286835, -0.4837157726287842, -1.5654648542404175, -0.02848339080810547, -0.03712569549679756, 0.7847025394439697, -2.0545802116394043, -1.2414089441299438, -0.4545206129550934, 1.6480270624160767, 1.8038997650146484, 1.0207276344299316, 1.1811596155166626, -0.32965782284736633, 0.15773308277130127, 0.3950284719467163, -0.03153672441840172, -0.7727863788604736, 0.9771885275840759, 1.388202428817749, -1.6108980178833008, -0.04683134704828262, 2.18829083442688, -0.7274758815765381, -0.6362422704696655, -0.7161933183670044, -1.6599136590957642, 1.679973840713501, 1.2376726865768433, 0.17251063883304596, 2.523258924484253, -1.5469850301742554, 0.053069107234478, 0.35357511043548584, -0.5078761577606201, 0.19655953347682953, 0.277332067489624, -0.46500203013420105, 0.9913117289543152, 1.6076527833938599, -0.6997571587562561, -0.30754563212394714, -0.8549047708511353, -2.3130438327789307, -0.7369852662086487, -0.24625074863433838, 1.3574062585830688, -0.7771835923194885, -0.7420952916145325, 1.2371704578399658, -0.5897462368011475, 0.5209718942642212, -0.6899796724319458, -2.8880465030670166, -1.0891340970993042, -1.2456117868423462, 0.2712049186229706, -0.0989936962723732, 2.9955103397369385, -1.1479955911636353, -0.485886812210083, -0.0015163037460297346, 2.724930763244629, -1.1507282257080078, 2.068493366241455, -1.1238257884979248, -0.5592308044433594, 1.6693599224090576, -0.9875428080558777, -1.1114659309387207, -0.159878671169281, 0.8160409927368164, -1.1256487369537354, 0.3799724876880646, -0.8940539956092834, -1.5892455577850342, 0.09375568479299545, 1.8592662811279297, 0.9873727560043335, -0.9801833033561707, -1.1685371398925781, 0.4110542833805084, 0.6455370187759399, -1.1601203680038452, 1.3684614896774292, -0.18424955010414124, 0.6006526947021484, 0.9944568276405334, 1.2334568500518799, 0.4122390151023865, -0.15334369242191315, -0.8496698141098022, 0.06048837676644325, 1.6718542575836182, -0.8979787230491638, -1.1242480278015137, -1.9462991952896118, 0.40292903780937195, -0.36220550537109375, -1.2633895874023438, 0.6312986016273499, 0.23361831903457642, -0.05856223404407501, -1.1102941036224365, 0.21029172837734222, -1.6030195951461792, -0.5154950022697449, 0.7124845385551453, 0.29616692662239075, -0.8198035955429077, -0.6430083513259888, -0.5863555073738098, -1.0039139986038208, -0.5156927704811096, -0.7482802271842957, -0.972662627696991, 0.11798905581235886, -1.7128119468688965, -1.2812477350234985, -1.0525466203689575, -0.7743269801139832, -1.9323127269744873, 0.9042506814002991, 1.748435139656067, 2.5594494342803955, -1.4481040239334106, -0.23128943145275116, -0.6691489815711975, -0.2723831832408905, 0.8506926894187927, -0.20209020376205444, -1.502925992012024, 0.025562429800629616, 2.1202359199523926, 0.27947965264320374, 1.8463574647903442, 0.16644014418125153, 0.6654872894287109, 2.043811321258545, -1.083339810371399, 1.8540432453155518, 1.6921427249908447, -0.20414642989635468, 1.1167733669281006, -0.08072419464588165, 0.679093599319458, 0.2677091360092163, -0.7640908360481262, 1.932965636253357, 1.839335322380066, 0.272763192653656, 1.139988899230957, 0.990871250629425, -0.07871965318918228, 1.918291687965393, 0.9047101736068726, -1.3266386985778809, -2.3816945552825928, -0.5092493295669556, 0.364678293466568, -0.009063957259058952, -0.23486317694187164, -0.843963086605072, -0.42161184549331665, 1.083521842956543, 0.8452929854393005, -0.37633177638053894, -0.6266149282455444, 0.843645453453064, -0.7994807362556458, 0.41184961795806885, 0.3371798098087311, -0.8921062350273132, -0.22528980672359467, 0.08259847015142441, -0.826298713684082, -1.5141369104385376, -0.20883603394031525, -0.33841434121131897, 0.6795595288276672, 0.0037877624854445457, 0.38132578134536743, -2.4307968616485596, -1.2987980842590332, 0.2850798964500427, 0.595710039138794, -1.5692483186721802, -0.3094683587551117, 1.285721778869629, -1.2845544815063477, -1.744677186012268, -0.24760466814041138, 1.5193989276885986, 0.23413367569446564, 0.5356814861297607, 1.0455214977264404, -0.45646727085113525, 0.014435102231800556, -0.9242308139801025, 0.22042958438396454, 0.09860453754663467, 0.23472337424755096, -0.37570086121559143, 0.6852111220359802, -0.6941906213760376, 0.5294783711433411, 1.4448920488357544, -0.450479120016098, 0.3381670415401459, -1.0489054918289185, -0.4346904456615448, 0.7238126397132874, -1.0004405975341797, -0.7010570168495178, -1.8781481981277466, -0.9248014092445374, 0.8410851955413818, 0.9691550135612488, 0.6565114259719849, -1.140475869178772, -0.2417908012866974, 0.822114109992981, 0.6172361373901367, 0.25157374143600464, 1.0114808082580566, -0.9975172281265259, 0.3730984926223755, 0.5038080215454102, -0.9893071055412292, -0.557686984539032, 1.3804560899734497, 0.5162340402603149, -1.1930652856826782, -0.13508909940719604, -0.1441875696182251, -0.20621025562286377, 0.9270473718643188, 0.9005844593048096, 1.4376156330108643, -1.064685583114624, 0.9274474382400513, -0.92522132396698, 0.9891881942749023, 0.7656145691871643, 1.67154061794281, 0.9044482111930847, -0.049040473997592926, 0.46779847145080566, 0.9272975921630859, -1.634045958518982, -0.9927528500556946, 0.24633339047431946, 0.6353446245193481, 0.7122240662574768, -1.0575774908065796, 1.2811839580535889, 0.017295939847826958, 0.5810567140579224, 0.5013132095336914, -0.5642091035842896, -0.9966331124305725, 1.9445866346359253, 0.12946176528930664, 1.7665932178497314, -1.088820457458496, -0.6470896601676941, -0.7983394265174866, 1.035443663597107, -0.07133256644010544, -0.8810502290725708, 0.8482173681259155, -0.9921136498451233, -1.6806840896606445, -0.8541999459266663, -1.0061836242675781, -1.0097520351409912, -0.6416950225830078, -1.5603713989257812, -0.1026802659034729, -0.8858567476272583, 0.2752700746059418, 0.2723906338214874, -0.00897715799510479, -0.6608016490936279, -0.3242669105529785, -0.7144652605056763, -0.2645014524459839, 0.8080707788467407, 1.0229519605636597, 1.2955725193023682, 0.10758401453495026, 1.0257281064987183, -1.663912296295166, 1.463429570198059, 0.9402675628662109, -0.8900086283683777, 0.08762884885072708, 0.2756994664669037, -0.44703003764152527, -0.6139171719551086, -0.6308621764183044, -0.09570587426424026, -0.4565677046775818, -1.0560033321380615, 0.373444139957428, 0.5299738049507141, 0.22345107793807983, 0.48372992873191833, 1.8942159414291382, 0.6306276917457581, -0.14359912276268005, -0.6924879550933838, 0.2744394540786743, 1.0096462965011597, -0.7956940531730652, 0.5086081027984619, -0.7870272994041443, -1.5979290008544922, -0.5602918863296509, 1.4551082849502563, 0.028768768534064293, -1.2396363019943237, 0.3257319927215576, 0.5362436175346375, 0.40495437383651733, -1.2182800769805908, 0.8642805218696594, -0.7343359589576721, 1.8940309286117554, -1.7056102752685547, -0.9273290634155273, -0.37300434708595276, -1.2879096269607544, -0.19287987053394318, -1.151254415512085, -1.1622287034988403, 0.4017118811607361, -1.8244798183441162, 0.7201316952705383, -0.13801945745944977, -1.7765123844146729, 0.7162402868270874, 1.3340246677398682, -0.4111005365848541, 1.559584617614746, 0.28265121579170227, 0.19673599302768707, 0.9639763832092285, -0.9870074987411499, 0.5659297704696655, -0.5811846256256104, 0.08680196106433868, 0.12255576252937317, -1.310813546180725, -0.040930815041065216, -0.7899596691131592, 0.07021845132112503, 0.010498023591935635, -1.2428511381149292, -0.9621251225471497, 1.5675143003463745, 1.6675599813461304, -0.17285796999931335, 0.33654171228408813, 0.4455892741680145, -1.1542682647705078, 0.9958256483078003, 0.0840073823928833, -0.273642361164093, 1.2068499326705933, 0.2481287270784378, -1.4824968576431274, -0.5468732714653015, -0.647239089012146, -0.9537843465805054, -0.7646051049232483, -0.7406564950942993, -0.6886059641838074, 1.6450108289718628, 0.14332732558250427, 0.8161700367927551, 0.973572850227356, 1.5778306722640991, 0.1856623888015747, 0.05604221671819687, 0.3583318591117859, -1.1993800401687622, -0.6137336492538452, 1.2466051578521729, 0.711138904094696, -0.17295685410499573, -0.5730998516082764, 0.4185279309749603, -1.7071373462677002, -1.0057158470153809, 1.7251685857772827, 0.41414618492126465, 0.18110167980194092, -0.664694607257843, -0.15798602998256683, -0.877715528011322, 0.41480305790901184, 0.34093549847602844, -0.05974028259515762, 1.560912847518921, 0.6748831868171692, -1.426287293434143, 0.7383451461791992, 1.932879090309143, 0.04575417563319206, 1.2606852054595947, 0.4199831783771515, -0.7103785276412964, -0.5512009859085083, -0.40501388907432556, -1.3553355932235718, -0.04277867078781128, -0.7823934555053711, 1.6640000343322754, 0.0865367129445076, -1.1315784454345703, -1.289035439491272, 1.2562612295150757, -0.062014833092689514, -1.2441189289093018, -0.5893151760101318, 0.21708691120147705, 0.06375880539417267, 2.0592877864837646, 0.4270263612270355, -0.4951971471309662, -1.9862221479415894, 0.5777851939201355, 0.4634423553943634, -0.0933958888053894, 0.9514537453651428, 0.09017129242420197, 0.2945552468299866, 2.2129669189453125, -1.1156723499298096, 0.9783809185028076, 1.0404696464538574], [-0.5839176177978516, 2.796064615249634, -1.5474294424057007, 0.5200605988502502, -1.0185657739639282, -1.1204761266708374, 0.09555662423372269, -0.297442227602005, 0.5305417776107788, 0.01569552533328533, -0.4715377390384674, -0.3673073947429657, -0.8340559005737305, 0.2711638808250427, 1.1173981428146362, -0.16207903623580933, -0.19838778674602509, 0.15836060047149658, -1.7386316061019897, -1.8596081733703613, -0.3464709222316742, 0.9855260848999023, 1.383841872215271, 1.1108745336532593, 0.11863992363214493, -0.3956076204776764, 0.06843239814043045, -0.5414628386497498, 0.01043765526264906, 0.44356364011764526, -0.24307742714881897, -1.6646125316619873, 1.5030040740966797, 2.409996747970581, -0.8034069538116455, -1.0092679262161255, 1.2081841230392456, -1.4518648386001587, -0.957606315612793, 0.5569695234298706, 2.4612607955932617, 1.118066430091858, -1.2511417865753174, 2.1264801025390625, -0.8117713928222656, 0.6410897374153137, -1.4855866432189941, 0.5672357082366943, 0.46535035967826843, -1.3010666370391846, -1.2827726602554321, 0.9608470797538757, 0.727810263633728, -0.8432033658027649, -0.03747784346342087, 0.47655653953552246, -0.9785563945770264, -1.7011003494262695, 0.03222768381237984, 0.005584741476923227, 0.5058760643005371, -0.7881384491920471, -0.8150101900100708, -0.7450195550918579, -0.041864342987537384, 1.3089879751205444, 0.13479657471179962, -0.6599065065383911, 1.397247314453125, -0.22584134340286255, 0.9033990502357483, 1.6558339595794678, 1.8537657260894775, -0.634947657585144, -0.9179152250289917, -0.9122977256774902, 0.19870953261852264, 1.0855051279067993, -1.2179539203643799, -0.44805479049682617, -0.30520695447921753, -0.6011412739753723, 0.5135771632194519, 1.7749568223953247, -0.4277186691761017, -0.5702415704727173, -0.7720230221748352, -0.2657378613948822, 0.7246763110160828, -0.5392472147941589, 1.2492703199386597, -0.11655460298061371, -1.1024675369262695, -0.29187774658203125, 0.5129658579826355, -0.15804795920848846, -1.136824131011963, 0.3802189826965332, -0.9092440009117126, -0.49360498785972595, 0.7059529423713684, 0.7252012491226196, 0.3949006199836731, -0.7573437690734863, 1.4785457849502563, -0.39301547408103943, 2.6834254264831543, -0.7501972913742065, -0.943978488445282, -0.24130521714687347, 0.8937933444976807, -1.174442172050476, 0.258134126663208, -1.0611923933029175, -0.10890838503837585, 0.9408141374588013, 0.9190828204154968, -0.8101726770401001, 0.17539933323860168, 0.4402290880680084, -0.6108365654945374, -0.5101075768470764, -0.0609450601041317, 0.2824288308620453, 0.11721636354923248, 2.936758279800415, -0.5473152995109558, -0.23214703798294067, 1.1085457801818848, 0.4756435453891754, -2.3422250747680664, -0.18748758733272552, 1.0018876791000366, 0.016927605494856834, -0.6539515256881714, 0.1420321762561798, 1.3336753845214844, 0.39581969380378723, -0.7403610944747925, -0.9588177800178528, -1.3304970264434814, 0.35107868909835815, 1.1820299625396729, 0.5119249820709229, 1.4410769939422607, -0.35725656151771545, 1.825779914855957, -1.2949784994125366, 1.113531231880188, -0.12345916032791138, -0.20230913162231445, -0.8926509618759155, -0.4220227003097534, 0.7704566717147827, -0.7228363752365112, 0.3273560404777527, -0.6824595332145691, -1.7751786708831787, 0.6581061482429504, -0.6124827265739441, -0.05038636177778244, 1.5400443077087402, 1.5540965795516968, -0.827839195728302, 0.030187806114554405, 1.0559406280517578, -0.2453324943780899, -1.3967450857162476, 1.195875883102417, 2.1131677627563477, 0.6795874834060669, -0.43242546916007996, -1.044381856918335, -0.6882398128509521, -0.41303443908691406, -0.37347355484962463, -0.9489995837211609, -0.7065227627754211, 0.7541643381118774, -0.24744948744773865, 0.04590725526213646, 0.0836116150021553, 1.2639269828796387, 1.2896610498428345, 0.11290343850851059, -1.5026392936706543, 1.3154526948928833, 0.6696085929870605, -0.2584955096244812, 1.2120836973190308, 0.27187204360961914, -0.34383806586265564, 2.0809099674224854, 1.8732844591140747, -2.7562994956970215, 1.4087406396865845, -0.12491171807050705, 1.8642230033874512, -1.1960930824279785, 0.5483894944190979, -0.5223768949508667, -0.036508675664663315, -0.005775514990091324, 0.5632385015487671, -1.017295479774475, -0.5695107579231262, 0.23893438279628754, -0.9105603694915771, 0.13907265663146973, -1.1755530834197998, 0.036659255623817444, -1.3291689157485962, -1.7245993614196777, -0.320789635181427, 1.3862725496292114, 0.7937653660774231, 1.2911163568496704, 0.45351049304008484, -0.039580874145030975, 0.6247383952140808, -1.1190754175186157, -0.08561733365058899, -0.8479124307632446, 1.2600891590118408, 0.3777611255645752, -0.5516225695610046, 0.0006956930155865848, 0.9029250144958496, -0.08403051644563675, 0.16402693092823029, 0.5257033109664917, -0.058178503066301346, 2.353140354156494, 0.40430837869644165, -0.6508431434631348, -0.5761693716049194, 2.4679534435272217, 0.21923433244228363, -1.0489031076431274, -1.751774549484253, 0.6681013703346252, 2.620075225830078, 0.6766191720962524, -0.21402117609977722, 1.2501310110092163, -0.2653684914112091, -0.27702364325523376, -0.20224174857139587, -0.5196956396102905, -1.3691210746765137, 0.23029781877994537, 2.192519187927246, -1.4159573316574097, -1.076983094215393, 0.5480534434318542, -0.2649409770965576, -0.0751633420586586, 0.3155289888381958, 0.6811484694480896, 0.2145485281944275, 0.6929923892021179, 0.803051233291626, -0.5196067094802856, 0.939633846282959, 0.11645569652318954, -1.0605665445327759, 1.6205720901489258, -0.5971072316169739, 0.7212260365486145, -0.5321394205093384, 1.1349875926971436, -1.472944974899292, -0.3448421061038971, -0.3623587489128113, -0.6011108756065369, 2.2101221084594727, -1.517700433731079, -0.751102089881897, -0.37072688341140747, 1.3837536573410034, 0.0659257173538208, -0.8823648691177368, 0.23675474524497986, -0.6420255303382874, -0.45657527446746826, 2.105245351791382, -0.704488217830658, -0.08116213232278824, 0.21421130001544952, -1.9144103527069092, -0.06896188855171204, -1.4260770082473755, -2.3698878288269043, 0.38611459732055664, -0.1317918300628662, 0.5490804314613342, -1.9877885580062866, 1.7500159740447998, 1.3817569017410278, -0.999082088470459, 0.36750754714012146, -0.3982219398021698, 0.6931172609329224, 0.260563462972641, 0.12983135879039764, 0.18368308246135712, -0.7153245210647583, -0.9403003454208374, -0.4300597310066223, 0.6187267899513245, -0.29886865615844727, 0.11865159869194031, -1.014106035232544, 0.09670655429363251, 0.6557224988937378, 0.03943448141217232, -0.9475647211074829, 1.6035356521606445, 0.6411387324333191, -0.82625412940979, 0.9816039800643921, -0.22776472568511963, 0.6090492010116577, -0.1569603830575943, -1.5935333967208862, -0.3690710961818695, 2.325486898422241, 1.3659584522247314, 0.24371017515659332, -0.4772581160068512, -1.1016814708709717, -0.4345988929271698, 0.7941039800643921, 1.2432888746261597, 1.883293867111206, -1.3906919956207275, -0.7081348896026611, 0.5212379693984985, -0.678414523601532, 0.8168045878410339, -0.6868788599967957, 0.7033562660217285, -0.571514904499054, 1.538068175315857, -0.09274876117706299, -0.5594701170921326, 0.6836684346199036, -0.4839741885662079, 3.506115198135376, 0.5331635475158691, 0.003804673906415701, -1.7211037874221802, 1.0444358587265015, 1.1213208436965942, -0.9318666458129883, -0.739163339138031, 0.867568850517273, 1.5371311902999878, -1.4869543313980103, -1.0934195518493652, 0.6092524528503418, 1.7772270441055298, -1.268657922744751, 0.16481398046016693, 1.8029900789260864, -0.1634330004453659, 1.1119146347045898, 0.9736723303794861, 0.2681760787963867, -1.1420068740844727, 0.6932086944580078, 0.3924880921840668, 1.1720914840698242, -1.1775422096252441, 0.8293356895446777, 2.503267288208008, -0.014920014888048172, -1.414707064628601, 0.3266642987728119, -1.5407003164291382, 0.07727031409740448, 0.652540922164917, 0.47946977615356445, 1.4129515886306763, -0.40761077404022217, 0.37516871094703674, 0.72648024559021, -0.158040851354599, -0.30445149540901184, 0.5266232490539551, 0.22633883357048035, -0.1319170445203781, -0.2589973509311676, 0.037454746663570404, 0.20358014106750488, 0.6056633591651917, -1.8014932870864868, -0.6097589731216431, -0.4229277968406677, 1.1128885746002197, 0.9256511330604553, -0.5935396552085876, -0.7311745882034302, -0.2962592542171478, 0.20730911195278168, -0.9386100172996521, -1.0321192741394043, -0.3838776648044586, 0.44164496660232544, 1.2641745805740356, 0.9089990258216858, -0.9965055584907532, -1.4432796239852905, 1.372743844985962, -1.6007814407348633, -0.36880210041999817, -0.6795968413352966, 0.8551036715507507, 0.20729795098304749, 0.13611796498298645, 0.33128267526626587, -1.1011420488357544, -0.029130086302757263, -0.31350475549697876, 1.1936637163162231, -0.8699985146522522, -0.6352912783622742, 3.2895023822784424, 0.09617523103952408, 1.0097161531448364, 0.00614504050463438, -0.20481917262077332, -0.21272413432598114, -1.4987398386001587, -1.4787555932998657, -0.9117355942726135, -0.36706802248954773, 0.098939910531044, -0.8615943193435669, -0.35254067182540894, -0.8244408965110779, 0.40809983015060425, -0.8056463599205017, -0.046492788940668106, -0.024081170558929443, -1.466741919517517, -0.12957961857318878, -1.4720591306686401, 1.9908676147460938, -0.6262732148170471, -0.10863503813743591, -1.5806852579116821, 0.13596515357494354, 1.201399326324463, 0.07981152832508087, -0.3421527147293091, 0.508978009223938, 0.38370105624198914, 1.1272873878479004, -0.975843071937561, 1.3029588460922241, 0.18328848481178284, 1.3256253004074097, 0.41444510221481323, -0.04546251520514488, 1.4431017637252808, -0.7003583312034607, -0.1388346254825592, -0.9141151905059814, 1.0861778259277344, 1.0430015325546265, 1.7342805862426758, -1.0904853343963623, 0.30336281657218933, -0.8889000415802002, 0.7469557523727417, -0.44346949458122253, -1.4929687976837158, 2.134972095489502, 1.0956220626831055, 1.867072343826294, 1.5939654111862183, 0.85914146900177, 0.22635981440544128, 0.7865124344825745, 0.17356941103935242, 0.0692000538110733, -2.061284065246582, 0.6166687607765198, 0.5059787631034851, -0.655001163482666, 0.5826405882835388, 0.176731139421463, -1.119398593902588, 0.8488093018531799, -0.820732057094574, 1.379767656326294, 1.1129403114318848, 0.4506869912147522, -0.7006354331970215, 0.20384642481803894, 0.43939507007598877, -0.833432674407959, -0.10160297900438309, -0.5118871927261353, -0.828948438167572, 1.5543098449707031, 0.12234214693307877, -0.19267314672470093, -0.6823992729187012, 1.4146100282669067, -1.017913818359375, 1.7760906219482422, 1.4039157629013062, -1.9848958253860474, -0.547568678855896, 1.880827784538269, 0.18423044681549072, -0.6133361458778381, 0.7618906497955322, -0.2716769278049469, 0.46013614535331726, -0.6086834073066711, 0.9442059993743896, 1.0154448747634888, 0.7895936965942383, -0.8518662452697754, -0.5524824261665344, -0.6402040719985962, 2.7099878787994385, -1.0815831422805786, 0.05545007064938545, -1.5673655271530151, 1.3328911066055298, -1.36276376247406, -0.36236417293548584, 0.24830302596092224, -0.5447466373443604, 1.8159211874008179, 0.22508420050144196, 1.122940182685852, 0.16504159569740295, 0.2709173262119293, 1.1698530912399292, 0.3955039083957672, -0.5762081146240234, -0.9458226561546326, 1.7086100578308105, 0.33973273634910583, 0.3781549036502838, 0.6381163001060486, 1.5183593034744263, 0.3309737741947174, -1.0158990621566772, -1.2875256538391113, 0.013618066906929016, -1.5432740449905396, 1.0273045301437378, 1.4675438404083252, -2.0890259742736816, -0.48912331461906433, -1.0056629180908203, 2.5277411937713623, -0.713499128818512, 1.41917884349823, 1.9542230367660522, -0.012767712585628033, 0.26624196767807007, 1.3117445707321167, -2.542872190475464, -0.49348312616348267, -0.3590090572834015, -1.8466581106185913, -1.3710774183273315, 0.050396762788295746, -0.1961432546377182, -2.429131031036377, 0.36137229204177856, 0.7007247805595398, 0.525566041469574, -0.5462015271186829, -1.705047369003296, -0.6798537969589233, -0.42307165265083313, -0.391167551279068, -1.0625618696212769, -0.3485499918460846, -1.1506344079971313, -1.438439130783081, 0.8360992074012756, -0.5881630182266235, -1.2413355112075806, 0.4466787874698639, -0.08441387116909027, -0.913279116153717, -0.3194009065628052, 0.9229016900062561, -1.116990327835083, 0.2926354706287384, 1.257426381111145, 0.44441983103752136, 1.0583302974700928, -1.4639798402786255, -0.22680556774139404, 2.057893991470337, 0.3699129521846771, 1.435631275177002, -0.1785230189561844, 0.4689123034477234, 0.7938748598098755, 1.3722283840179443, -0.01062766369432211, 1.2100805044174194, -0.8585001230239868, -1.3248189687728882, -1.0725483894348145, 0.07843223214149475, -0.2934703826904297, -1.1070724725723267, -0.20904168486595154, 0.18972207605838776, 0.9239144325256348, 0.34253451228141785, -1.3786563873291016, -0.4479069709777832, 1.1133464574813843, 1.2321217060089111, -0.9592640399932861, 0.01801794394850731, -0.6901292204856873, -0.6774841547012329, -0.42447060346603394, 0.2910897135734558, 0.08754652738571167, -0.5489351749420166, -1.0997307300567627, -0.7742263674736023, 2.5595510005950928, 0.34159356355667114, 2.2094576358795166, 1.5934165716171265, 0.864742636680603, 0.19887107610702515, -1.483976125717163, 0.2887224853038788, 1.9366835355758667, -1.2259210348129272, -0.4304855167865753, 1.3072830438613892, 0.01539615262299776, 1.0884929895401, -0.05543563514947891, 0.35971689224243164, -0.23990188539028168, -0.3466477394104004, -0.13473708927631378, 0.5629335641860962, -0.08107862621545792, -2.0858547687530518, 0.049939680844545364, 1.4901032447814941, -0.26185914874076843, -1.391861081123352, -0.29700034856796265, -1.4736754894256592, 1.1813043355941772, 0.1907128244638443, 0.2574215531349182, -0.33540958166122437, 0.5428804755210876, -0.04704377055168152, 1.2886909246444702, 0.6749991178512573, -1.4545007944107056, -1.0920079946517944, -0.232520192861557, 0.5086432099342346, -0.3160690665245056, -1.156107783317566, -0.8982300758361816, -0.30236515402793884, 1.398597002029419, -1.0044711828231812, -0.6742178201675415, 0.3051773011684418, 0.10867467522621155, 0.35020628571510315, 0.38794514536857605, 0.5961377620697021, 1.074716329574585, -0.32618945837020874, 1.458117961883545, -0.04166603088378906, -0.41022032499313354, -0.4560752809047699, 0.7828580141067505, 1.1069937944412231, -0.7540169954299927, 0.7795153260231018, 0.45502519607543945, -0.795555591583252, 0.040308594703674316, 0.7580757141113281, 1.7182834148406982, 0.343365877866745, -0.3120296895503998, -0.44405779242515564, -0.554999053478241, 1.0694465637207031, -0.7027478814125061, -0.19690026342868805, -0.6291764974594116, 0.2881583273410797, 0.7006592154502869, 1.2202669382095337, 1.5566136837005615, -0.1793001890182495, -0.702651858329773, 1.0918296575546265, 0.512694239616394, 0.19607298076152802, -0.06985591351985931, 0.45170238614082336, -0.8868038058280945, -0.4790067970752716, -0.04045544192194939, 0.4294811189174652, 0.581886887550354, -0.34865036606788635, 0.15116235613822937, 0.251848965883255, 0.9104933142662048, 0.6495924592018127, -0.6939979791641235, 0.7463437914848328, 2.214308500289917, -1.450210690498352, 0.6734189987182617, -0.3320496678352356, -1.3621366024017334, 0.3973310887813568, 1.1463677883148193, -1.1517666578292847, 0.1414720118045807, 1.1846977472305298, -0.4591203033924103, 0.4339296221733093, 0.16235068440437317, -1.4433118104934692, 0.9192861914634705, -0.4252298176288605, 0.9335656762123108, -1.3096686601638794, -1.3720089197158813, -0.6082119345664978, 1.875728726387024, 0.41937434673309326, 0.5659946799278259, -0.5476245284080505, -1.5373789072036743, -1.8738003969192505, -1.0028116703033447, 0.2647077739238739, 1.7946735620498657, -1.6198756694793701, -2.0702688694000244, 0.8496018648147583, 1.1468586921691895], [0.20313987135887146, -1.075600266456604, 1.1634544134140015, -0.9396464824676514, 2.5249953269958496, -0.2310248613357544, -0.7809872627258301, -0.6712377071380615, 2.5521743297576904, 0.640145480632782, 0.4615764617919922, -1.1167634725570679, 0.29266685247421265, -0.1393326073884964, 1.4176980257034302, -0.8030950427055359, 1.489815354347229, -0.9481914043426514, -0.8970127701759338, 0.18089377880096436, 0.15504395961761475, -1.702558994293213, -1.4731968641281128, -0.9732030034065247, 0.6805655360221863, 1.071445345878601, 0.8194394111633301, 0.39394018054008484, 0.637917697429657, -0.6139837503433228, 0.1904846876859665, -0.3025640547275543, -0.4864412844181061, 1.9227827787399292, 0.016698040068149567, 1.1466126441955566, -1.0500547885894775, -0.3056796193122864, -0.6404657959938049, -0.425436794757843, -0.18996457755565643, 0.34238773584365845, 0.08924593776464462, -0.5352374911308289, 1.610087513923645, -0.7339664697647095, 1.0734467506408691, -0.5387275815010071, -1.0232229232788086, 1.301312804222107, -1.3671008348464966, -0.9443212747573853, 0.562061071395874, -0.3198715150356293, -1.25115966796875, 0.10474646836519241, 1.3995327949523926, 0.588312566280365, 0.24584200978279114, 0.5633948445320129, -0.5222573280334473, 0.9357953667640686, -0.03383545204997063, -1.4725615978240967, -0.8300255537033081, 0.04346414655447006, 0.09605979174375534, -0.10831271857023239, -1.0272785425186157, 0.2693692147731781, -0.9330117702484131, -1.8520616292953491, -0.5845918655395508, 1.321129560470581, 1.0728341341018677, -0.04261666163802147, 0.9979974627494812, 0.574230432510376, -0.6686820387840271, 0.8008933663368225, -0.1386842131614685, -0.22431284189224243, 0.6672856211662292, -0.8392317295074463, 0.4162485599517822, 1.3891047239303589, 1.2415136098861694, -0.020227884873747826, 1.1674162149429321, -1.3102375268936157, 0.657924234867096, -0.5315548777580261, 1.2836781740188599, 0.04890110343694687, 0.2846449613571167, 0.3957344591617584, -0.9817530512809753, -2.888334035873413, -0.755208432674408, -0.0025603624526411295, 1.5394593477249146, 0.013722959905862808, 0.7135823965072632, -0.014722361229360104, 0.15147653222084045, 0.7671987414360046, -0.17785872519016266, 1.0586323738098145, 2.009891986846924, 1.3938673734664917, -0.7727487683296204, 0.9979193806648254, 0.8960654735565186, 1.206149697303772, -0.07555881887674332, 1.350879430770874, 0.4550231695175171, -0.2610028386116028, 1.128585696220398, -0.0702224150300026, -1.3606723546981812, 0.1698692888021469, -1.2287483215332031, -0.5888563394546509, 0.22142496705055237, -2.0267627239227295, -0.6015469431877136, -0.355290949344635, -0.9277233481407166, -0.9321287274360657, -1.6036407947540283, -0.7964826822280884, -0.8381372690200806, 0.5013593435287476, -0.0889144316315651, 1.1643424034118652, 0.7921421527862549, 0.4361116290092468, 0.7434359788894653, -0.5069146752357483, 1.8041900396347046, 1.047780156135559, -0.08036424219608307, -0.06044945865869522, -0.059291549026966095, -0.9677571654319763, 0.9436582326889038, -0.9471096396446228, 0.5461894869804382, 0.3600673973560333, -0.14991843700408936, 0.5783041715621948, 0.7821505665779114, -1.207737922668457, -1.9751346111297607, -0.44330623745918274, 0.7064408659934998, 0.3689936101436615, 0.4519987106323242, 0.3983112871646881, 1.5206221342086792, -0.9167868494987488, -0.11246553808450699, 2.4987967014312744, -1.419619083404541, -1.439673662185669, -0.6336644291877747, 0.0280892476439476, 0.3578948974609375, -0.12253470718860626, 0.7881909608840942, 1.5781539678573608, 0.07409827411174774, -1.1692293882369995, -0.32504889369010925, -0.13391119241714478, 0.4733155369758606, -0.9472783803939819, 1.329697608947754, 0.2583334147930145, 1.417024850845337, -0.5288952589035034, -0.5169742703437805, 2.7700355052948, -2.033677101135254, -0.330036461353302, -0.21914835274219513, -0.6251352429389954, -1.0752325057983398, 1.0014139413833618, -0.42603906989097595, -0.6401760578155518, 0.2556679844856262, -0.23905429244041443, -1.1728013753890991, -1.4299665689468384, 0.06950365751981735, 0.8120620846748352, 0.038787033408880234, 0.6280543208122253, 0.6359273791313171, -0.9289060235023499, 0.8548980951309204, 0.22516992688179016, 2.226271152496338, 0.6387186050415039, -0.6932483911514282, 0.0626356303691864, -2.1060667037963867, -0.5855450630187988, 0.37642529606819153, 0.4549987316131592, 0.8220441341400146, -0.4176500737667084, -0.894442617893219, -0.6390635967254639, -1.1127086877822876, 1.559025764465332, -1.0624313354492188, -1.2696025371551514, 1.886849284172058, -0.41671401262283325, 0.6062943935394287, 0.3378629684448242, -0.6005632877349854, 0.051506832242012024, 1.2843437194824219, -0.4414013624191284, -1.9701560735702515, -2.348335027694702, -1.2497445344924927, -0.30102190375328064, 0.7527587413787842, 1.0004489421844482, 1.2233021259307861, 1.1226329803466797, 1.749029278755188, -0.1559094488620758, 0.2001800686120987, -2.3928310871124268, -0.038285400718450546, 1.5720880031585693, -0.25077396631240845, 0.7174420952796936, -0.11560932546854019, 0.30828866362571716, -0.6549186110496521, 0.36342334747314453, -0.9245873093605042, 0.28034141659736633, -0.8446357846260071, -0.6065889000892639, 0.9843772053718567, -0.9218365550041199, 0.5391356945037842, 0.8198424577713013, -0.9163621664047241, -1.051850438117981, -0.9259433150291443, 0.7457025051116943, -0.18449178338050842, 0.261726051568985, 1.673288345336914, -0.45387452840805054, 1.0863796472549438, -0.07134819775819778, -0.2681407928466797, -1.3152114152908325, -0.8504343032836914, 2.3882009983062744, 0.23946821689605713, -0.8283657431602478, 1.3063011169433594, 0.6679354310035706, 0.8107677698135376, 0.8587501645088196, -1.8561965227127075, -1.7199857234954834, -0.29996657371520996, 0.2524610459804535, -0.019781136885285378, 0.8203749656677246, -1.2517753839492798, 1.0478432178497314, -2.185358762741089, 0.09760542958974838, 0.533918559551239, 0.567483127117157, 0.6458538174629211, 0.10296420753002167, 0.024368418380618095, -1.3444478511810303, 0.05655045434832573, -0.2615676820278168, 0.2725834548473358, -0.6978421807289124, -1.443569540977478, 1.2898499965667725, -1.9091005325317383, -0.3699147403240204, 1.3758126497268677, 1.4322824478149414, -1.807528018951416, 0.750200629234314, -0.6365975141525269, 0.3699677288532257, 0.6453078389167786, 0.873731791973114, -1.949281096458435, -0.9166214466094971, 0.6258437037467957, 0.6033297777175903, 1.6677427291870117, 0.8078658580780029, 1.2237193584442139, 1.128374695777893, 0.7716090083122253, -1.413130521774292, 1.9174449443817139, 0.3751521408557892, -0.4880635738372803, -0.26756736636161804, -0.6770778298377991, -0.03939466550946236, -1.7488455772399902, 0.9101247191429138, -1.0750771760940552, 1.0190784931182861, -0.5961913466453552, 0.6326571702957153, 1.3447954654693604, 0.4431767761707306, -0.004686305765062571, 0.1603747308254242, -0.25913333892822266, -0.2500024437904358, 2.0348641872406006, -1.1824864149093628, -0.8133600950241089, -0.4963669180870056, -0.06426604092121124, 0.6218498945236206, -0.13867473602294922, 1.0276765823364258, -1.0874618291854858, 0.3590082824230194, 0.19314375519752502, -1.0609867572784424, -0.13595902919769287, -0.38342148065567017, -0.7087506651878357, -0.8616306185722351, 0.8797003626823425, -0.8314269185066223, 0.9069012403488159, 1.0239306688308716, 0.5824205875396729, 0.6068452000617981, -1.5714576244354248, 0.13480238616466522, 0.7072891592979431, 0.8230388760566711, 1.2952220439910889, 0.7208809852600098, -0.09737975895404816, -0.5446614027023315, -0.7292628884315491, 0.048702824860811234, -0.2382012903690338, 1.3802729845046997, -0.5265732407569885, -0.9647971391677856, -0.6687635779380798, 2.3563385009765625, 0.11777795106172562, -1.2619754076004028, 1.4961624145507812, -0.26560547947883606, 0.20721587538719177, -0.6919222474098206, 0.652760922908783, 0.7630888223648071, -0.7913463115692139, 0.06050426512956619, -1.39030921459198, 1.4037553071975708, -0.9820578098297119, -0.8865704536437988, -0.9962871670722961, -1.0784451961517334, 0.1327226608991623, -0.6368729472160339, 0.6350025534629822, 0.4973960220813751, 1.1565189361572266, 1.3919037580490112, -0.7032210826873779, 0.06383143365383148, -0.931801974773407, -0.22960636019706726, 0.6441109776496887, 0.3948991000652313, -0.6743289828300476, 0.47970035672187805, 0.0526345819234848, 0.5680440664291382, -2.421506881713867, -0.9434317350387573, 1.005179524421692, -1.6100029945373535, -0.452265202999115, -0.2842598259449005, -0.7604522705078125, -0.38941147923469543, 0.14827999472618103, -0.18513359129428864, -0.5806896686553955, 0.556186318397522, 0.8508696556091309, 0.8295640349388123, -1.9394482374191284, 1.3632736206054688, 0.41503292322158813, -0.19868065416812897, 1.020389437675476, 0.8304176330566406, 0.8951219916343689, -0.777593731880188, -1.7143545150756836, 1.5267441272735596, 0.5768613219261169, -0.4503306746482849, -0.4258701205253601, -0.1974683552980423, 0.8079025745391846, -1.22199285030365, -0.8563238382339478, 2.4190754890441895, -1.3039499521255493, 0.5009877681732178, 0.23017145693302155, 0.15076526999473572, 1.296686053276062, -0.38385140895843506, 1.2127106189727783, 0.044750768691301346, -0.6527836918830872, -0.7355043292045593, -0.5309150218963623, 0.9441918730735779, 1.0740773677825928, 0.09287507832050323, -0.7829422354698181, -0.9059673547744751, -0.16792307794094086, 1.3812329769134521, 0.7675371170043945, -0.3057364225387573, 1.5012964010238647, -1.613687515258789, -2.0499746799468994, -1.5443974733352661, 0.9040413498878479, -1.1538856029510498, -0.5085256695747375, 0.0026104908902198076, -0.5612697005271912, 0.8516573309898376, 0.37691396474838257, -0.25648966431617737, 0.7769456505775452, -2.0145034790039062, 1.1824088096618652, -0.5216083526611328, -0.9361113905906677, -0.010008575394749641, 0.5792025327682495, 0.020904913544654846, 0.3922869563102722, -0.20806817710399628, -0.6321131587028503, 1.5720707178115845, 1.0710529088974, -1.4577512741088867, 0.5546501874923706, -0.8498597741127014, -0.8075747489929199, 0.40600308775901794, 0.9808505773544312, 1.5923848152160645, 1.0851777791976929, -0.5353401899337769, -0.5761337876319885, -0.7427382469177246, -0.31643879413604736, 1.6235020160675049, 0.3715224266052246, 0.246229887008667, -3.174307107925415, 0.29268962144851685, -0.6954219937324524, -0.23719163239002228, 0.36714160442352295, 0.7084499597549438, -0.6782594919204712, -0.793034553527832, 0.09421011060476303, -0.6777404546737671, 1.5014313459396362, -0.44366753101348877, -1.7598484754562378, -0.028692232444882393, -0.6250943541526794, -0.1600687950849533, 0.020316462963819504, 1.1008994579315186, 1.2239471673965454, -0.414341002702713, -1.8282039165496826, -2.1543970108032227, 1.785525918006897, -0.14187443256378174, 0.5879095792770386, -0.46032291650772095, 0.17608729004859924, 0.7237237095832825, -0.48287639021873474, -0.1339513659477234, 0.4740351140499115, -1.2251001596450806, -0.9602874517440796, 0.5196647047996521, -0.4706956148147583, -0.2582699954509735, -1.5966445207595825, 0.3959295451641083, 0.3661576509475708, 1.9547468423843384, -0.05534158647060394, -0.24083152413368225, 0.8280977606773376, 0.047208599746227264, 1.5650429725646973, 0.032938312739133835, -0.09795787185430527, 0.826858401298523, 1.255361557006836, 0.5483384132385254, 1.3822224140167236, -1.2251803874969482, 0.07856377959251404, 0.5065398812294006, 1.6911777257919312, -1.2196186780929565, 1.6678640842437744, -0.0003382449212949723, 0.9093248844146729, 0.4181351065635681, -1.9970226287841797, -0.5498265624046326, 0.21705204248428345, -0.581890881061554, -0.5293996930122375, 0.21176007390022278, -0.18828608095645905, -0.1980505883693695, 0.6134865880012512, -0.4632808268070221, -1.3659629821777344, -0.9836968183517456, 0.2632957398891449, -1.3274171352386475, -0.5249884724617004, 0.5244264006614685, -0.689629316329956, -0.5756297707557678, -1.3888087272644043, 0.844525933265686, 0.7245283722877502, -0.2703724503517151, 0.6887357831001282, -0.6421647071838379, -1.186708927154541, -1.4518399238586426, 0.5924875140190125, -0.8201508522033691, 0.4004804193973541, -0.2549649775028229, 1.0084348917007446, 0.22316507995128632, -1.0548417568206787, -1.456915259361267, -0.1702953428030014, 0.9577697515487671, 0.011441228911280632, 1.3621207475662231, 1.1220569610595703, -0.34257808327674866, -0.5652394890785217, -0.42009857296943665, -0.12935949862003326, 0.3690343201160431, 0.2537211775779724, -1.0616375207901, 0.10684766620397568, -0.17195037007331848, 0.7673531174659729, -1.1560642719268799, 1.4907505512237549, 0.11641968041658401, -0.3658483028411865, -0.23098917305469513, -0.6894827485084534, 0.3873290717601776, 1.0478399991989136, 0.3358295261859894, 0.2674199640750885, 0.9429970383644104, 1.6675899028778076, 1.1559191942214966, -1.9966527223587036, 0.25967469811439514, 0.43644487857818604, 0.18731659650802612, 0.4014277160167694, 1.7736929655075073, 0.45130258798599243, 0.14854292571544647, 0.020402444526553154, -1.4072933197021484, -1.1799602508544922, -1.5458072423934937, 1.698012113571167, 0.19940191507339478, -1.0274252891540527, -0.22506116330623627, 0.12213967740535736, -0.4461352825164795, 1.2698646783828735, 0.4469345808029175, 1.0222736597061157, -0.6153554320335388, -1.1455284357070923, -0.18760980665683746, -0.0657796636223793, 0.80008864402771, -1.8588272333145142, 0.9788088202476501, 0.02467905916273594, 0.29458633065223694, 0.974183201789856, -0.5745551586151123, 0.011948115192353725, 0.5335407257080078, -0.2176821231842041, -1.1705853939056396, 0.609496533870697, -0.1701427698135376, -0.28549185395240784, 0.5286744236946106, -0.5606153011322021, 1.4638910293579102, 0.9722857475280762, 1.508067011833191, -0.5400266647338867, 0.09651423245668411, -0.20238818228244781, -0.47549116611480713, -0.8481131196022034, -0.1857476383447647, -0.0402691513299942, -0.8091081380844116, -2.2228453159332275, -0.891063392162323, 1.7245615720748901, -0.3866286277770996, 1.0234789848327637, -0.8149884343147278, -1.812621831893921, 0.32193106412887573, 0.7003971934318542, 0.4310087561607361, 0.007563864346593618, -0.00783684104681015, 0.9472079277038574, -0.4826298952102661, 0.1983519196510315, 1.9046251773834229, -1.8918079137802124, 0.38919326663017273, 2.099806070327759, -1.6537306308746338, -1.6778984069824219, -0.6481953859329224, -0.20383338630199432, 1.5960966348648071, -0.6872761249542236, -1.6532806158065796, 1.5817108154296875, -0.1429557502269745, 0.18662016093730927, 0.03669142723083496, -1.1301591396331787, -0.6461141109466553, 0.6306129097938538, 0.819435715675354, -1.506881594657898, 0.1884486973285675, -0.9672539234161377, 0.8599482178688049, -0.13739600777626038, -0.2903268039226532, 0.3699176013469696, 1.8037314414978027, 0.2735344171524048, 2.415268659591675, 0.6653304100036621, 0.372285395860672, 0.36848098039627075, 0.38733339309692383, 0.22090871632099152, 2.5045225620269775, -0.997568666934967, 0.25185519456863403, 0.1315745711326599, 0.3846410810947418, 0.09879091382026672, 1.0921865701675415, -1.050122618675232, 1.3304955959320068, 1.4455502033233643, -0.32593491673469543, 0.6332345604896545, -1.514504313468933, 1.2691106796264648, 1.234407663345337, 1.8036022186279297, 0.16141892969608307, 2.3420796394348145, 1.0902388095855713, 1.380439043045044, 1.0684943199157715, 0.24721890687942505, -0.6029185652732849, 0.5244542956352234, -0.24119094014167786, 1.3726534843444824, -0.7798293828964233, -0.8610816597938538, 0.6569234132766724, -0.5155271887779236, 0.8908116817474365, 0.5935653448104858, 1.115140676498413, 0.8013765215873718, 0.057941146194934845, -0.24916112422943115, -0.0990372747182846, 0.8486425280570984, 0.004171975888311863, 1.79574453830719, 2.022317886352539, -0.6197938323020935, 0.9167895913124084, -1.2366828918457031, -0.3754108250141144, 1.4752908945083618, -1.1912459135055542], [-0.32617971301078796, -0.14927871525287628, -1.54719877243042, -1.0150724649429321, 0.6906181573867798, 0.19164973497390747, 1.1453312635421753, -0.2984282970428467, -2.311521530151367, -0.039928898215293884, -1.2179712057113647, 0.7360718846321106, -0.906365156173706, 2.0085766315460205, -1.7341434955596924, -0.4382059574127197, 2.3357224464416504, 0.8186226487159729, 0.34400951862335205, 0.9242737293243408, 1.4903833866119385, -0.824518620967865, 0.2792227864265442, 1.5693399906158447, 1.0914430618286133, -0.521561324596405, 0.46899446845054626, 0.865727424621582, -1.1557679176330566, 1.2995928525924683, 1.3881504535675049, -1.4173293113708496, -0.36480724811553955, -0.7342181205749512, -1.3655438423156738, -0.6649008393287659, -1.6504648923873901, 0.8633127212524414, 0.4664930999279022, -1.5655295848846436, -1.4273449182510376, 0.8286135196685791, -0.45190516114234924, 1.416000247001648, -0.5016205906867981, -0.31103605031967163, 0.15022234618663788, -0.9969701766967773, -0.3676964044570923, 0.40239304304122925, 2.560121774673462, -0.2873799502849579, 0.26928892731666565, -1.0292538404464722, -0.20960187911987305, -0.9483274221420288, -0.919377326965332, -0.844023585319519, -1.1211966276168823, 0.9026616811752319, 0.6253151297569275, -1.5830163955688477, 1.2692381143569946, 0.4386492073535919, -0.3475271761417389, -1.2053484916687012, 0.09038082510232925, 0.3770468235015869, -1.4485679864883423, -1.6671500205993652, 1.2366365194320679, -0.9386356472969055, 0.3097990155220032, -0.6785504817962646, -0.6599479913711548, 1.0851304531097412, -0.8094278573989868, 1.6435123682022095, 0.6912254095077515, 2.1363441944122314, -0.003006810788065195, -0.09107450395822525, -0.06309468299150467, 0.3809841275215149, 1.6560624837875366, -1.1591871976852417, 2.0282557010650635, 0.32699671387672424, -0.3731776773929596, -0.4217158854007721, 1.0223500728607178, 0.702307939529419, 0.7548294067382812, 0.3036291301250458, 1.3827804327011108, 0.46156805753707886, -0.41391870379447937, 0.48253554105758667, 0.26181256771087646, 0.23804382979869843, 1.2655866146087646, 0.25805261731147766, -1.177024006843567, -1.4321883916854858, -0.901594877243042, -1.2071504592895508, -0.5539895296096802, -0.43282249569892883, 1.7323517799377441, 0.6978372931480408, 1.3214071989059448, 0.07362422347068787, -0.2390313446521759, -1.743773341178894, -0.44657450914382935, 0.7409195899963379, -0.9767245054244995, 1.1277836561203003, 0.8197848200798035, 0.006039902102202177, -0.046014126390218735, 1.4025917053222656, -0.37469619512557983, -0.38562098145484924, -0.35002344846725464, -1.4227564334869385, -0.44405627250671387, -1.1489900350570679, 1.1141490936279297, -0.027296703308820724, -1.6411621570587158, 1.23264741897583, -0.49864518642425537, -0.5042688250541687, -0.4725611209869385, -1.2501375675201416, -0.37135180830955505, 0.5463628768920898, -1.3787562847137451, -0.25642165541648865, -0.7192246317863464, 0.49852874875068665, -0.05182979255914688, -2.4473648071289062, 0.39149922132492065, 0.6253200173377991, -1.1263761520385742, -0.5875007510185242, -0.3013913333415985, -0.4139549732208252, -0.7925102710723877, 1.7007321119308472, -0.5413146615028381, -0.28155088424682617, -0.08327765017747879, -0.055116936564445496, 1.8041938543319702, 0.7934857606887817, 1.8644465208053589, -0.6172692179679871, 1.0739035606384277, 0.2691221535205841, 0.5315267443656921, -1.6609437465667725, -0.5169835686683655, -0.11706855148077011, -0.4684400260448456, 0.3242899477481842, 0.48931363224983215, 0.7011098265647888, 0.3204008936882019, -1.1626924276351929, 0.5541406273841858, -1.0098140239715576, 0.9887136816978455, -1.0713852643966675, 0.8147621154785156, -2.5474188327789307, 1.361038088798523, 0.05198041349649429, 0.41318732500076294, -0.44082486629486084, -0.3912615478038788, -0.6988546252250671, 0.8313417434692383, -0.276138573884964, -0.35255464911460876, -0.6558283567428589, -1.540932536125183, -0.9906859993934631, -0.3207358717918396, -1.3069041967391968, 1.3587138652801514, -0.7574526071548462, -0.09430418163537979, 0.24115177989006042, -1.2634072303771973, 0.41039395332336426, -0.4350056052207947, 0.05003426969051361, -0.29467687010765076, -0.023918945342302322, 0.767245352268219, 0.13925312459468842, -0.2102949321269989, 2.1005759239196777, 0.03450121730566025, 0.22016838192939758, -0.11063922941684723, 0.328766405582428, -1.286694049835205, 1.1543582677841187, 1.7156461477279663, -0.9442220330238342, 1.122574806213379, 0.6372643709182739, -0.10865114629268646, -1.23435640335083, -1.3668147325515747, 0.10037699341773987, 1.7375712394714355, 1.16423499584198, 1.6213535070419312, -0.9021843075752258, -0.6753729581832886, -0.4312799870967865, 1.5239267349243164, -0.8847332000732422, -0.834437906742096, 0.6156591773033142, -0.07394442707300186, -1.2363593578338623, 1.0011963844299316, -2.082218885421753, -0.3820728063583374, -0.11905062198638916, -0.6439498066902161, -0.1346030980348587, 0.37967172265052795, -1.1222540140151978, 0.3676680326461792, -0.5415120124816895, -0.3572787344455719, -0.8168017268180847, -0.15530231595039368, -0.29958629608154297, -0.11256483942270279, -1.0367586612701416, -0.17093005776405334, -0.059269774705171585, 1.5506352186203003, 0.4375338852405548, -0.37696248292922974, 0.7131194472312927, 0.6831359267234802, -0.25972622632980347, -0.4356919229030609, 1.9108861684799194, 0.043740447610616684, 0.4232657253742218, -0.9885511994361877, -0.11194286495447159, -1.6248286962509155, 0.04431184381246567, 0.5924383997917175, -1.3552559614181519, -0.9989222288131714, -0.093989297747612, 0.5909005403518677, 0.03550980985164642, -1.1149230003356934, -0.8112338185310364, -0.22376210987567902, 0.06906501948833466, -0.6951424479484558, -0.8341448903083801, 0.13831180334091187, 0.4224035143852234, -0.06577226519584656, -0.5969136953353882, -2.0549705028533936, 0.07145509868860245, 0.006800655275583267, -0.539033055305481, -1.252885341644287, -0.3148682415485382, 1.9824575185775757, -1.7266002893447876, -1.0478858947753906, 0.157459557056427, -1.393567681312561, -0.7200751304626465, 0.05378430709242821, -0.5051009654998779, -0.4798695147037506, -0.20258618891239166, -0.9695115685462952, 0.7789375185966492, -0.951137900352478, -0.9285954236984253, -0.4082508981227875, -0.3607700765132904, 0.8224391341209412, 0.272234171628952, -0.32901236414909363, -0.5319584608078003, -1.0698868036270142, 0.8358714580535889, -0.8403887152671814, 0.8829288482666016, -1.4808751344680786, 0.43965306878089905, -0.508335292339325, -1.3408783674240112, -2.509638786315918, 0.12500685453414917, 0.7186859250068665, -0.00905691273510456, -1.7037409543991089, 1.7827681303024292, -0.46175065636634827, -0.9043912887573242, 0.44215449690818787, -0.8486358523368835, 1.344854712486267, -0.018126197159290314, -0.3256239891052246, -0.0060583315789699554, -0.526534378528595, 0.07123719900846481, 1.6341781616210938, 1.1445006132125854, 1.087816596031189, 0.755795419216156, -0.037990640848875046, 1.503899097442627, 0.21201331913471222, 1.0913376808166504, 0.12168990075588226, -0.5788062214851379, -1.249621033668518, 0.5808078646659851, -1.1473486423492432, 0.007867047563195229, 1.0299136638641357, 0.4169272780418396, -0.5892786383628845, 0.040352243930101395, 1.7931205034255981, -0.24756093323230743, 1.057126522064209, 0.5829051733016968, 1.5201308727264404, -0.13864605128765106, 1.1773301362991333, 0.24348515272140503, 0.6766063570976257, -0.9610138535499573, -0.5664607286453247, -0.355289101600647, 0.02736815996468067, 0.0373401939868927, -0.3693905174732208, 0.6289555430412292, -0.4656226634979248, 0.618336021900177, -0.9558016657829285, -0.025451060384511948, -0.19680018723011017, 0.5684874057769775, -1.4221659898757935, 0.16428032517433167, -1.2930686473846436, 0.6699332594871521, -0.7601678967475891, 1.6111068725585938, 0.3239963948726654, 1.2642252445220947, 0.5073423981666565, 0.8026838302612305, -0.9044198393821716, -0.2202538251876831, -1.2854949235916138, 0.38874128460884094, -0.47790956497192383, 0.8010618090629578, 0.7316423654556274, 0.38682541251182556, -0.2655317783355713, 0.444835901260376, -0.021985230967402458, -0.21606886386871338, 0.0766945630311966, 0.8348917365074158, 0.2222265601158142, 0.06731710582971573, -0.3279549181461334, -1.0258405208587646, -0.07046562433242798, 0.09225071966648102, 0.5638959407806396, -0.7065956592559814, -0.7710735201835632, -1.3671029806137085, 0.3842991590499878, -1.0516481399536133, 1.1567214727401733, -0.7719455361366272, -0.5328934192657471, -0.3269735872745514, 0.28735342621803284, 1.4981951713562012, -0.7210862040519714, -0.6856248378753662, 1.7892476320266724, 1.8209950923919678, 1.7599209547042847, -0.46207883954048157, 2.107201099395752, -0.02993505261838436, 1.6144676208496094, -0.8420647978782654, -0.04883426055312157, -1.265917420387268, 0.18104583024978638, 1.41506826877594, 0.7026144862174988, 0.8696553111076355, 1.1584910154342651, 0.9890459179878235, -0.3880910277366638, -0.2989305555820465, -0.03526288643479347, -0.2515586316585541, 1.0185987949371338, 0.6275083422660828, 0.390814870595932, 0.7465614080429077, 0.3761078417301178, -0.2737594246864319, 1.0665907859802246, 0.6112574934959412, 1.160912036895752, -0.1630202829837799, 2.4740231037139893, 0.9906359910964966, 0.5877669453620911, 1.3195730447769165, -1.0081812143325806, -1.8416550159454346, -0.610526442527771, 1.2558363676071167, -0.13602855801582336, -1.8709094524383545, -0.6271805763244629, -0.44897252321243286, 1.5484998226165771, 0.06058374047279358, 1.5960859060287476, -0.3344630300998688, -0.07134811580181122, 0.07877229899168015, -0.3502759039402008, -2.8800129890441895, -1.36708402633667, 1.5114535093307495, -0.41531410813331604, 0.8984308242797852, -1.7077487707138062, 0.2813897728919983, -0.5874650478363037, -2.414994239807129, 0.36507856845855713, -0.5385885238647461, -0.5253926515579224, 1.5539710521697998, 0.8943729996681213, -0.5372474193572998, 1.5607483386993408, -1.4437862634658813, -0.4209657609462738, -1.098444938659668, -0.6970869302749634, 0.8578500151634216, 0.6059119701385498, 1.3254773616790771, 1.0876893997192383, 0.3527935743331909, -0.6450039148330688, 0.06373175978660583, -0.42626768350601196, -0.20916692912578583, -0.2402706742286682, 0.7909700274467468, -0.5784673094749451, 1.1032829284667969, 0.7562487125396729, 0.1995832771062851, 0.6970626711845398, -0.8355973958969116, 1.4081648588180542, 0.4708620309829712, 0.5236989855766296, 0.2946663796901703, 0.21605059504508972, -0.8662965297698975, -1.35696280002594, -0.4180373549461365, 1.2519489526748657, -0.6086483597755432, -0.5956190228462219, 0.19327139854431152, -1.6619000434875488, 0.18575425446033478, 0.5290964841842651, 1.31858491897583, 0.2181076854467392, 0.6201506853103638, -0.49741309881210327, -0.6855758428573608, -1.948272466659546, 1.703507661819458, -0.5047262907028198, -0.7478125095367432, -0.2721562087535858, 0.3787209689617157, 0.23800460994243622, 0.5600739121437073, 0.8714345693588257, 2.4667532444000244, -0.990811824798584, 1.081518530845642, -1.7189483642578125, 0.8321517705917358, 0.682258665561676, 0.9565033912658691, -0.753421425819397, -0.2923193573951721, 0.8761852383613586, -2.6145706176757812, -0.4665283262729645, -0.2019680291414261, -1.3528004884719849, 0.6187406778335571, -0.41120705008506775, 1.1827011108398438, 0.4387317895889282, 0.1892811506986618, 0.6244680285453796, 0.15857389569282532, -0.8292313814163208, -2.0339789390563965, 0.6534440517425537, 0.23743890225887299, -0.776679515838623, -0.1708085983991623, -0.9077247977256775, -0.1503024399280548, -0.14233613014221191, -0.40465018153190613, -1.1208980083465576, -0.029389187693595886, -1.4458184242248535, -0.9983586668968201, 0.5426714420318604, 0.058246906846761703, 0.5954392552375793, 1.3080118894577026, 0.626380443572998, 1.0517451763153076, 0.5999457836151123, 0.23551645874977112, -0.054075099527835846, 0.8959712386131287, 0.9540044069290161, -1.0437533855438232, 0.3940906822681427, 0.8623314499855042, 0.030649244785308838, -1.800113320350647, -0.019689731299877167, -1.8904482126235962, 0.3556062579154968, -0.910856306552887, 0.7444168925285339, -0.9329853653907776, 1.1476422548294067, 1.3134841918945312, -0.973352313041687, 0.46975597739219666, 1.1902800798416138, -0.93993079662323, -1.9746352434158325, 0.2541891932487488, -0.4268108606338501, -0.09487096965312958, 0.5238611698150635, -1.8217573165893555, 1.1330331563949585, -0.5452636480331421, -0.09618942439556122, 1.6793655157089233, 0.644263744354248, 0.08621610701084137, -1.7177164554595947, 1.46415114402771, -0.7489407062530518, 0.6601549983024597, -0.1871398389339447, 1.1306288242340088, -1.2267426252365112, -0.27425116300582886, -0.011093898676335812, -0.5838776230812073, -2.242546796798706, 0.9813042879104614, 0.7076417803764343, -0.8067735433578491, 0.09018845856189728, 0.7189261317253113, 0.6733496189117432, 1.4565967321395874, 0.20384232699871063, -0.23226670920848846, -1.0563331842422485, 0.06297251582145691, -0.21429936587810516, 1.2292691469192505, -1.222732424736023, -0.22375182807445526, 1.0564066171646118, 1.1717381477355957, -0.3614741861820221, 0.8568149209022522, -1.1567846536636353, 1.7272536754608154, -0.832133412361145, 0.6708118319511414, -0.27058738470077515, -1.965981125831604, -0.6902666091918945, 1.990035057067871, 0.22228814661502838, -1.3534481525421143, -1.2626479864120483, 0.35444727540016174, 0.04317198693752289, 0.15633390843868256, -0.18960799276828766, 0.3343106806278229, 0.2551833689212799, -0.17198483645915985, -2.024754762649536, -0.32506123185157776, 0.18654850125312805, 0.9394649863243103, 1.5700557231903076, 0.19759736955165863, 1.636437177658081, 0.14885789155960083, 1.3744792938232422, 0.3383893370628357, 0.6799372434616089, -1.2165435552597046, -0.45582881569862366, 1.0569032430648804, 0.9980818629264832, 0.7561705112457275, -0.34243735671043396, -0.7603070139884949, -0.5652067065238953, -0.2589614689350128, 0.2941918671131134, -0.860085666179657, -0.1453855186700821, -0.5789157152175903, -2.7525811195373535, -1.3570764064788818, -0.00672368798404932, -0.40309014916419983, 0.4369201958179474, -2.727189779281616, 0.3740667998790741, -0.7710449695587158, -0.3389413356781006, -1.428425669670105, 0.5602115392684937, -0.34594666957855225, 0.940296471118927, -1.2723881006240845, 0.21390798687934875, 0.710085391998291, 1.2612143754959106, 1.0836840867996216, 0.47760218381881714, 0.35039740800857544, -1.0212595462799072, -1.3641576766967773, 1.823736310005188, 1.4342418909072876, -0.4057863652706146, -1.6083935499191284, 1.1950972080230713, 0.8667015433311462, -0.37736397981643677, 0.11475156992673874, -0.06345001608133316, 0.9459690451622009, -0.10307972133159637, 1.2630467414855957, 0.5612764954566956, 1.0289561748504639, -0.6195361018180847, 0.4862310290336609, -2.115266799926758, 0.9611249566078186, -1.787000298500061, 0.2160053700208664, 0.023175135254859924, 1.077210783958435, -0.2681546211242676, 1.3934485912322998, 1.0568805932998657, -0.4655199646949768, 0.6092107892036438, 0.5929430723190308, 0.42757052183151245, -1.6805317401885986, 0.1295839250087738, -0.06795621663331985, -0.518487811088562, -1.4578666687011719, 0.3853212893009186, 0.421421617269516, -1.3042430877685547, -1.2343292236328125, -0.04095456376671791, 0.38780662417411804, -0.13918140530586243, -1.054925560951233, -0.31633374094963074, 0.06919194757938385, -0.7546601891517639, -0.8212921619415283, 0.9159188866615295, 1.2507919073104858, 1.1917498111724854, 0.8176285028457642, 0.4121674597263336, 0.7935105562210083, 0.4599859416484833, -0.2383018285036087, 1.3075770139694214, 0.7517992854118347, -0.6932148933410645, -0.4829784333705902, -0.009923332370817661, 0.47518211603164673, -1.404350996017456, -0.4871804714202881, 0.7175902128219604, 0.09214618802070618, 1.5408071279525757, 0.523646891117096, -2.1049184799194336, -2.576308012008667, 2.022498369216919, -0.5105258822441101]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [[0.1969742774963379, -0.5572201609611511, 0.7207706570625305, 0.6699493527412415, 2.439213275909424, -0.352302223443985, 0.4022636413574219, -0.023262446746230125, -0.11889184266328812, 0.29048845171928406, -0.8282018303871155, -0.0609615296125412, -0.6237428188323975, 0.09858956187963486, 0.012430472299456596, 0.36784160137176514, 0.4090312421321869, 0.13569127023220062, 3.083296775817871, -0.11947030574083328, -0.13213054835796356, 0.5611370205879211, 0.012113548815250397, 0.014899681322276592, -0.17169903218746185, -0.3175772428512573, -0.04193289950489998, -0.29704463481903076, 0.4558890461921692, 0.4327874481678009, 0.5659776926040649, 0.9806899428367615, 0.32655662298202515, -1.227899193763733, -0.169930562376976, 1.9288049936294556, 1.7002153396606445, 0.17441898584365845, 1.4032090902328491, -0.633590817451477, 1.5127238035202026, -1.2831618785858154, -0.3631537854671478, -0.6579452753067017, -0.43121138215065, 0.5765038132667542, -0.24250811338424683, -0.04961127042770386, 0.8497288823127747, -0.6771105527877808, 0.9496959447860718, 0.08176586776971817, -0.006742916535586119, 0.5455732941627502, -0.18126162886619568, 0.05747383087873459, -0.16521921753883362, 0.4298121929168701, -0.7487698793411255, -0.3712400496006012, 3.344015121459961, 0.03237304091453552, -0.7376435399055481, 1.2153003215789795, 0.47402364015579224, -1.2542788982391357, -0.15365880727767944, -1.325891375541687, -0.0012263287790119648, -0.1095188707113266, 0.21055245399475098, 0.20596975088119507, -0.008417190983891487, 0.002274517435580492, -1.1118310689926147, 1.0157114267349243, -0.9924731254577637, 0.6987070441246033, -0.5668962597846985, 0.9589503407478333, -0.0415920726954937, -0.493531733751297, -1.1651877164840698, -0.7920608520507812, -0.9218506217002869, 1.371644377708435, -0.2594742178916931, 0.14905999600887299, 1.2087314128875732, -0.28417953848838806, -0.048183683305978775, 1.094269871711731, -3.246969699859619, -0.49835118651390076, 0.4445610046386719, 0.22881564497947693, 0.753616452217102, 1.1402713060379028, -2.6747653484344482, -0.5340422987937927, 0.4126870334148407, -0.3805660605430603, -0.4248906672000885, -0.0486690029501915, -0.8546587824821472, -0.06834687292575836, 3.992156505584717, 1.8865211009979248, 0.09356711804866791, 0.14701005816459656, 0.2908225655555725, -0.5061607956886292, 0.1490166038274765, 1.8455963134765625, 0.06396925449371338, -5.919978141784668, 0.3052564859390259, 0.06219496205449104, -0.008170627988874912, -0.47076156735420227, -0.01910467818379402, 0.12155692279338837, 0.07391203194856644, 0.5082986354827881, -0.05289739742875099, -0.11325128376483917, 0.3685835003852844, 1.6116036176681519, -0.06409052759408951, 0.3651430904865265, -0.7162222266197205, 0.0047196452505886555, 0.38151881098747253, -1.4689770936965942, -1.7265347242355347, 0.22286978363990784, 1.5524837970733643, -0.9616388082504272, 0.11418838798999786, -0.1660659909248352, 0.278856098651886, -0.42924219369888306, 0.8066421151161194, -0.1674981564283371, 1.8519182205200195, 0.31421881914138794, -1.305059552192688, -1.394876480102539, 0.795325517654419, -0.6565239429473877, -0.647718071937561, -1.0620789527893066, 2.2815120220184326, 0.26094967126846313, -0.048397984355688095, -0.14623485505580902, -0.27233925461769104, 0.8789986968040466, -0.19086873531341553, 0.11155536770820618, -0.14612038433551788, -0.3710382580757141, -0.41520822048187256, -0.03102942183613777, -0.3266865313053131, -0.12664514780044556, 0.03384307399392128, 0.07823070883750916, -0.01470628660172224, -0.14182235300540924, -0.7734864354133606, 0.022862054407596588, -0.5085996389389038, 0.010432356037199497, 0.02518587000668049, 1.4222615957260132, 0.01113668642938137, -0.7880730628967285, -1.1667641401290894, 0.8077411651611328, 0.7684982419013977, 1.0625323057174683, -0.06812477111816406, 0.25059881806373596, -0.3992060422897339, -0.186379536986351, 0.6388416290283203, -4.159358024597168, 0.8211413025856018, 0.013117213733494282, -0.17199012637138367, -0.5119199752807617, 0.6677301526069641, -0.14182190597057343, -0.22441889345645905, -0.37489691376686096, 0.9742835164070129, 1.1613497734069824, 0.04847791790962219, 1.375169038772583, -0.15735824406147003, 1.000076413154602, -0.5732415318489075, 0.7889868021011353, 0.06550739705562592, -0.21544378995895386, -0.10036906599998474, -0.02867997996509075, 0.13089358806610107, -0.4696088135242462, -0.3768790066242218, -1.6223173141479492, -0.11141208559274673, -0.30563241243362427, 0.3293471336364746, -1.8919788599014282, -0.778373658657074, 0.32663673162460327, 0.7443970441818237, 0.014286955818533897, 0.08648572117090225, 0.5142566561698914, 1.3600765466690063, 0.505189061164856, -0.02479708567261696, -0.10961852967739105, -1.205639362335205, -0.1769816279411316, 0.29734739661216736, -0.0587506964802742, -0.35828959941864014, -0.5034810900688171, 1.2912917137145996, -0.16514137387275696, -0.06930331140756607, 0.227243572473526, -0.9651399254798889, -0.38846540451049805, -0.06146015226840973, -1.235978364944458, -0.21599556505680084, 1.2142748832702637, -0.21451865136623383, -0.28113073110580444, -0.03936634212732315, 0.07992404699325562, 0.08917572349309921, 0.6304393410682678, -0.47234952449798584, -0.002857412677258253, 0.2582961916923523, -0.00413607619702816, -0.033346645534038544, -0.3876992166042328, -0.1481717824935913, -0.61060631275177, 1.285190224647522, 0.7628967761993408, -0.27528923749923706, -0.7344081401824951, -0.6446241736412048, -0.7414305210113525, 0.008114039897918701, 0.6628459692001343, 1.1640578508377075, 0.0050682006403803825, 0.999813437461853, 0.014045290648937225, -0.16803185641765594, -0.28327250480651855, -0.4337978959083557, -0.31186214089393616, 0.7306987643241882, -0.13297376036643982, -0.14954674243927002, -1.1812663078308105, -0.03207585960626602, -0.024918917566537857, -0.22605684399604797, 0.7945781946182251, 1.2484357357025146, 0.621713399887085, 1.4671127796173096, -0.23870383203029633, 1.0737223625183105, 0.8474116325378418, 2.009911060333252, -2.576721668243408, -0.5407965183258057, 0.0829089805483818, -1.090940237045288, 1.050208330154419, -0.935025691986084, 0.6618731617927551, 2.395961046218872, -0.17494836449623108, 1.3305367231369019, 0.5288707613945007, -0.8496009707450867, -0.380251407623291, -1.010108470916748, 0.022910883650183678, -0.1026822105050087, -1.0001482963562012, 0.7798082232475281, 2.6707608699798584, 0.20867052674293518, 0.05548034608364105, -0.05969955027103424, -0.16277331113815308, -0.3381742238998413, -0.2221275269985199, 0.2935417890548706, 0.8954062461853027, 0.62127685546875, 2.635678768157959, 0.11970842629671097, 1.2928168773651123, 0.12618109583854675, 0.3896132707595825, -0.24317535758018494, -0.702403724193573, -0.3820527195930481, -0.7430607676506042, -2.5909154415130615, 0.1338648498058319, -0.6065847277641296, 2.358394145965576, 0.7476555109024048, -0.06270752847194672, -0.17313693463802338, 1.1389963626861572, 0.7782465815544128, -1.2694957256317139, 3.001304864883423, 0.4106955826282501, 0.3987683951854706, 0.42225033044815063, 0.005511291790753603, 0.34539833664894104, -0.7406880259513855, -1.0258090496063232, -0.8868739008903503, -0.1416526585817337, -1.186143159866333, 1.2175813913345337, -0.4323308765888214, -1.3175866603851318, -2.992490291595459, 0.10309506952762604, 1.30791437625885, -1.2335186004638672, 0.005662922281771898, -0.4554082751274109, 0.6886793375015259, 0.589505136013031, 0.2352796196937561, 0.2156059443950653, -0.08928647637367249, -0.15589539706707, -0.7617434859275818, 0.32048603892326355, -0.17737236618995667, -1.5546977519989014, 0.8479347229003906, 0.2141086757183075, 0.49168047308921814, -0.10851402580738068, -0.2530668079853058, 0.13675788044929504, 1.5961134433746338, -0.9653770923614502, 0.15491983294487, 0.18340851366519928, -0.0028014113195240498, 0.327946275472641, -1.2090773582458496, 0.4095941185951233, 0.2493506520986557, -0.6392318606376648, -0.569229006767273, -1.486644983291626, -0.07297322899103165, 0.014192559756338596, -0.5847689509391785, -0.473789244890213, -1.1525837182998657, 0.6271041035652161, -4.120582580566406, 0.19920416176319122, 0.30882588028907776, 0.5582314133644104, 0.23611925542354584, 0.1636294424533844, 0.0779799148440361, 0.021817956119775772, 0.016265135258436203, -0.679019570350647, -0.3242160379886627, -0.0531914122402668, -0.3068789839744568, 0.08438938111066818, 0.036961108446121216, -0.2334386110305786, -1.160339117050171, 0.16327622532844543, -0.012266433797776699, -3.8447515964508057, -1.6192964315414429, -0.5997278690338135, -1.5253159999847412, -0.04378758743405342, -0.46444228291511536, -1.6889286041259766, -2.3369390964508057, 1.4790229797363281, 0.9678553342819214, -1.61055588722229, -0.2322787493467331, 0.21412965655326843, 0.2520178258419037, -0.6223483681678772, 0.06993291527032852, 1.1234816312789917, -1.9040685892105103, 0.3619738221168518, 1.0363962650299072, -0.025867445394396782, -0.3169753849506378, -0.8787937164306641, -0.24859905242919922, -3.1498429775238037, -2.4812657833099365, 0.05287215858697891, -0.0027315455954521894, -1.3410505056381226, 2.6031014919281006, 0.20748047530651093, 1.2559738159179688, 0.0072191799990832806, -0.21898843348026276, -2.4117703437805176, -0.27319422364234924, -0.21512967348098755, -1.2719405889511108, 0.5794901847839355, -0.8494019508361816, -1.7284038066864014, 0.008623702451586723, -0.040663447231054306, -0.2148597687482834, -0.17984969913959503, -0.0823967307806015, -0.9843589067459106, -0.04333948716521263, 0.23011904954910278, 2.5012729167938232, -0.07401032745838165, 1.3665727376937866, 0.9901602268218994, 0.21355141699314117, 0.6692591905593872, -2.094916343688965, -1.5189244747161865, -0.8106057643890381, 0.005922815762460232, -0.18606628477573395, 1.098405361175537, 2.296945571899414, 0.023386718705296516, -0.41841715574264526, 1.4042248725891113, 0.9912320375442505, 0.27098917961120605, 0.3435819149017334, -1.2100577354431152, -0.05412415787577629, 1.4617236852645874, 1.0485007762908936, -0.8194603323936462, -0.13279718160629272, 0.5434615015983582, 0.061749160289764404, -0.7686612010002136, 0.00019764831813517958, -0.051460929214954376, 1.496397614479065, 0.43267762660980225, -0.6649209856987, 0.09289045631885529, -0.12449729442596436, -0.9978002905845642, -0.32021984457969666, -0.4180663526058197, 0.17994271218776703, 0.05918743833899498, 0.0032515255734324455, 0.4400862753391266, 0.3709567189216614, -1.5849581956863403, -1.111862301826477, -0.1504281759262085, -0.009160749614238739, 0.33601054549217224, 0.11749134957790375, 0.2651405930519104, 0.02532174624502659, -0.018287887796759605, -2.7612364292144775, 0.3749006986618042, -0.1844630241394043, 0.29012712836265564, 0.13758496940135956, -0.4320918023586273, -0.231184184551239, -0.731421947479248, -1.6772030591964722, -0.008384705521166325, -1.1893993616104126, -0.0354648157954216, -0.8730311989784241, 0.7937116026878357, 0.6812407970428467, -0.8232465386390686, 0.689127504825592, 0.010266395285725594, 0.06200564280152321, 0.19038689136505127, 0.4507017135620117, 0.017845090478658676, 0.42109522223472595, 0.2268255203962326, 0.8045933842658997, 0.019962547346949577, 0.05628850311040878, -0.9646891355514526, 1.7547342777252197, 0.7087365984916687, 0.38343366980552673, -0.9620212912559509, -2.8460867404937744, 0.16322317719459534, -0.7956710457801819, 0.04419131949543953, -1.8970729112625122, 0.16949321329593658, 0.4405554234981537, 0.26052600145339966, 1.2504162788391113, -0.9909186363220215, -0.5673320293426514, -0.36913132667541504, -0.13591913878917694, -0.5174561738967896, 1.3792840242385864, 0.06776963174343109, 0.006807904224842787, -0.07405955344438553, -0.05711899697780609, -0.697507381439209, -3.313239574432373, -2.3091330528259277, 0.6430191993713379, 0.07507239282131195, -0.7609694004058838, -0.3746657371520996, 0.015014316886663437, 0.025397105142474174, 0.3465045690536499, 0.11539390683174133, 0.7490535974502563, -1.2138067483901978, 0.0014330876292660832, -0.8540673851966858, 0.579444944858551, 0.3880380094051361, -0.10060111433267593, -0.38236647844314575, -0.044952642172575, 0.0860777199268341, -0.49280330538749695, 0.018555542454123497, 1.8868236541748047, -0.048992425203323364, 1.575569748878479, 0.29343652725219727, 0.7216882705688477, 0.05376460775732994, 1.4257813692092896, 0.4588112235069275, 0.1963566690683365, -0.1025671511888504, 0.03818440064787865, 0.09059513360261917, -1.7741036415100098, 0.2497546523809433, -0.4078978896141052, 0.21252456307411194, -1.325801968574524, 0.2461518943309784, -0.19711221754550934, -1.8772779703140259, 0.5377399921417236, -1.0586854219436646, 0.34000784158706665, -0.3033582270145416, 0.10553689301013947, 0.0693705752491951, 1.6673948764801025, -0.36344343423843384, 1.1348158121109009, 0.8383022546768188, 0.3725493550300598, -0.25077977776527405, -0.3456273376941681, 0.8485829830169678, -0.3492107391357422, -0.022540530189871788, 1.374121069908142, 0.29316893219947815, -0.34638187289237976, 0.011803027242422104, 0.7222298979759216, 1.432623028755188, 0.5315269231796265, 0.3739471733570099, -0.4118165075778961, 0.20301038026809692, -2.312530279159546, -0.4343668520450592, -0.08608635514974594, 0.4761925935745239, 0.20961779356002808, 0.49064669013023376, 0.27160966396331787, 0.018088323995471, -0.986926257610321, 2.15667986869812, -0.02429976686835289, 0.9859979748725891, -0.9566099643707275, 0.03950151056051254, 0.000658493721857667, 0.08919116854667664, -0.4024170935153961, -0.21666577458381653, 0.14054009318351746, 0.11850827187299728, 0.12330090254545212, -0.1706353724002838, 0.11309421062469482, -0.03502059355378151, -0.28731751441955566, -0.44929659366607666, 0.573540985584259, 0.5145239233970642, -0.25035569071769714, -0.07261447608470917, -0.008082178421318531, -0.1518460512161255, -0.484019011259079, -0.058157458901405334, 1.1048980951309204, 2.8995981216430664, -0.45893529057502747, -0.9062368869781494, -0.016516899690032005, -0.10219712555408478, -0.2762081027030945, -1.2772170305252075, -0.9767123460769653, -1.4476714134216309, 2.019163131713867, 2.0294530391693115, -0.0681147426366806, 0.6938010454177856, -0.012752795591950417, 0.03493029996752739, -1.5712482929229736, 0.12135419994592667, 0.27134251594543457, -0.9510328769683838, 0.016988085582852364, -0.3380676507949829, 1.8992928266525269, 0.47615474462509155, -1.6244840621948242, -0.007993101142346859, -1.0839780569076538, -0.5500704646110535, 0.9806898832321167, -0.3068186938762665, 0.12352367490530014, 3.0001375675201416, 0.14577406644821167, 0.05948824808001518, 0.6915934085845947, 0.04722776263952255, -0.10004766285419464, 1.733652114868164, -0.3277904987335205, 0.5258379578590393, 0.9708501696586609, 0.6889171600341797, -0.07179488986730576, 0.008372751995921135, -0.10682221502065659, 0.2117890566587448, 0.06179899722337723, -0.9193621277809143, 0.07839954644441605, 0.4610278010368347, -0.20479588210582733, -0.10476338863372803, -2.8084468841552734, -1.092860221862793, -0.9408811926841736, -1.246093988418579, 1.0615111589431763, -0.3598470985889435, 0.7866799235343933, 2.2499959468841553, -0.6855874061584473, -0.09029418975114822, -0.3370015621185303, -1.1323447227478027, 0.030801940709352493, -0.19712799787521362, 0.9776723980903625, -0.8175774812698364, 1.2549911737442017, -0.0010069460840895772, 0.47722840309143066, 1.864333152770996, 2.076021432876587, -0.13333483040332794, 0.09453786164522171, 0.20511382818222046, 0.001574542373418808, 0.47898510098457336, 0.1622437983751297, -0.11917420476675034, -0.26304855942726135, 0.36872002482414246, 0.9842115640640259, -1.7931658029556274, 0.10843850672245026, 0.8114925026893616, 0.13632218539714813, -0.6885128617286682, 0.010708398185670376, -0.3896837532520294, 0.31470784544944763, -1.9127633571624756, -0.04804586246609688, -0.27278563380241394, 5.543101787567139, 0.8029187321662903, 3.1650612354278564, 0.06719556450843811, -0.284798264503479, 0.11210591346025467, 0.49052244424819946, -0.20675070583820343, 0.03639299049973488, 0.611414909362793, 0.05911094695329666, -0.42454957962036133], [-0.45597097277641296, -1.708988904953003, 2.128103733062744, -0.30223706364631653, 0.24341042339801788, 0.483496755361557, 0.22248022258281708, -0.015902772545814514, -0.2016647756099701, 2.7231435775756836, 1.7036129236221313, -0.5081618428230286, -0.08559651672840118, 0.07707426697015762, -0.7484782338142395, -0.5554907321929932, 0.3956655263900757, 0.14336812496185303, 2.5823276042938232, 0.08369596302509308, 0.30644509196281433, -1.390257716178894, 0.38330405950546265, 0.078981913626194, -0.08430031687021255, 0.7432750463485718, 0.011393852531909943, 0.08919601142406464, 0.8198301196098328, 0.22894896566867828, -0.06971698254346848, -0.17511944472789764, 2.9052932262420654, 2.289135694503784, -0.24696093797683716, -1.1818124055862427, 1.2829092741012573, 0.24233378469944, -1.1162687540054321, 1.5869485139846802, 2.8960788249969482, 2.0481889247894287, -0.9607369303703308, 0.5985236763954163, -0.010378758423030376, 0.7118732333183289, -0.7726932168006897, 1.188143014907837, -1.0764517784118652, 2.14170241355896, -0.3714704215526581, 0.19076059758663177, 0.0021369564346969128, 0.5635573267936707, -0.7684718370437622, -0.009375919587910175, -0.05105886235833168, -0.543140709400177, 1.096988558769226, -0.029270553961396217, 2.086353063583374, -0.20503343641757965, 1.2012015581130981, -0.8863820433616638, -1.9218204021453857, -0.5889986753463745, -0.24152544140815735, -1.6992123126983643, 0.057162050157785416, 0.04806101694703102, -1.5127252340316772, 0.9642648696899414, -0.00682078255340457, 0.0537380613386631, -2.148437976837158, 0.3723827600479126, 0.601182222366333, 0.40732529759407043, 0.21596667170524597, 1.00534188747406, 0.0254970733076334, -0.4322153925895691, -0.8486154675483704, -1.0281955003738403, 0.8343585133552551, -1.1400445699691772, -0.14663614332675934, -0.742631733417511, 0.04063955321907997, 0.8637388348579407, 0.9757705330848694, 0.3912826478481293, 3.4879345893859863, 0.3429882228374481, -0.3726486563682556, 0.05521078407764435, 0.515693724155426, -0.6731351017951965, -0.764366865158081, 0.6745501160621643, -0.2600211203098297, 0.23828808963298798, 1.762271523475647, 0.3048805594444275, -0.17596884071826935, -0.998261570930481, -1.9083223342895508, -0.6764655709266663, 0.09282738715410233, 0.06345710158348083, -0.11204573512077332, 0.43503502011299133, -0.029148336499929428, -1.0223394632339478, 0.04876592382788658, -0.898212194442749, -0.3927927315235138, -0.7261991500854492, -0.008779996074736118, 0.25433892011642456, 0.1623833179473877, -0.12658996880054474, -0.0939098671078682, -1.116163969039917, 0.13818886876106262, -0.02434295043349266, -0.12403656542301178, 0.2566744387149811, 0.038578711450099945, -2.4260988235473633, 0.7214787602424622, 0.036960020661354065, -0.5910892486572266, -1.5211645364761353, 2.41106915473938, -0.37943118810653687, -1.1185590028762817, -0.3526470363140106, -1.7889060974121094, 0.7982853055000305, 0.44620659947395325, 2.1598336696624756, 1.1901696920394897, -1.7348992824554443, -0.21207007765769958, -0.5863576531410217, -0.2926526963710785, -1.6347624063491821, 0.44638919830322266, 0.540307879447937, 0.2972448468208313, -0.2050844132900238, -2.293484926223755, -1.4498682022094727, -0.05730891227722168, -0.23727187514305115, 0.07427316159009933, 1.4796329736709595, -0.7068542242050171, 1.015779972076416, 0.010481617413461208, 0.7998744249343872, 0.005652746185660362, 0.20566435158252716, 1.9593557119369507, -0.5184751749038696, -0.012831314466893673, 0.586861789226532, 0.19938234984874725, -1.1131306886672974, 0.8145732879638672, -0.018284503370523453, 0.7978834509849548, 0.02461160346865654, -0.5810969471931458, -0.4573800563812256, -0.006183450110256672, 0.31721359491348267, -0.0696437880396843, -0.1721489131450653, -0.9441007375717163, 0.2922345995903015, 0.10827875137329102, 2.6546406745910645, -1.3440771102905273, -0.14077088236808777, 0.9101493954658508, 3.0275986194610596, -0.567401647567749, -0.021801745519042015, 1.1652113199234009, 0.8460428714752197, 0.048884253948926926, -0.115292988717556, 1.8217867612838745, 3.025164842605591, -0.3370632827281952, -1.1148958206176758, 0.0354531966149807, -0.2756514251232147, 0.14449535310268402, 0.5561972260475159, -0.6498019099235535, -0.5087055563926697, -0.006346872076392174, 0.08241299539804459, 0.38437962532043457, -0.017458472400903702, -0.10099691897630692, 0.689799964427948, -0.2188030183315277, -1.3920643329620361, 0.03974240645766258, -0.7221065163612366, 0.7771770358085632, -1.1145658493041992, -0.44013267755508423, -0.07512972503900528, 0.435855895280838, 0.28157925605773926, 0.2992396652698517, -0.008054076693952084, -0.6551337242126465, -1.519774079322815, -0.025367433205246925, 0.06427938491106033, -1.647576928138733, 0.8621417284011841, -0.1418876349925995, 0.10700499266386032, 0.19023293256759644, -0.36390382051467896, 0.39508020877838135, -0.5467268824577332, 0.210686594247818, -0.08341288566589355, -0.41525745391845703, 1.422482967376709, -0.05492667853832245, 0.13153517246246338, 0.07996305078268051, 0.719886302947998, 0.48875272274017334, 0.14034806191921234, 0.44210943579673767, 0.5597213506698608, 0.0082226088270545, -0.885891318321228, 0.832209587097168, -0.09011957049369812, 0.04368209466338158, -0.002058281097561121, -0.0016978950006887317, 0.7195857167243958, -0.14045003056526184, -0.7424653768539429, -1.3425856828689575, 0.15194855630397797, 0.12759409844875336, 0.2848413586616516, 0.531383216381073, -0.40274059772491455, 0.1283874213695526, -0.22913052141666412, 0.16195720434188843, -0.0027833511121571064, -0.11782150715589523, -0.028629988431930542, -1.112465739250183, -0.026525894179940224, -0.23200446367263794, -0.56574547290802, -0.361095666885376, -0.11147630214691162, 0.6171947121620178, 0.35828977823257446, -0.10412638634443283, 0.007240929640829563, 0.224225714802742, -0.09186628460884094, 0.43667349219322205, 1.0790082216262817, -0.9622921347618103, -0.8050110340118408, 0.34176039695739746, -0.15662381052970886, 1.9725525379180908, 0.9633507132530212, -0.5410999059677124, -0.10262037068605423, -0.6289510130882263, 0.0003216593759134412, -0.5292876362800598, -0.2915983498096466, -1.5351121425628662, 0.6889665126800537, 0.9510117769241333, -0.5890130996704102, -0.1376635581254959, -0.47396937012672424, -0.5543941855430603, -2.1400413513183594, -0.7771130800247192, -0.10426601022481918, -2.2616982460021973, 1.3839397430419922, -0.11477406322956085, -0.15671306848526, 0.5530980229377747, 0.3014616072177887, 0.24333611130714417, -0.09003132581710815, 0.026067232713103294, -0.46435269713401794, -1.03035569190979, -0.259475976228714, 0.054161764681339264, 0.1276775449514389, -0.20406222343444824, 4.167535781860352, -0.16645759344100952, 0.47910237312316895, 0.09677688777446747, -0.642819344997406, -1.5684844255447388, -0.19972540438175201, 0.0509849451482296, 6.25714111328125, -0.8024340867996216, 0.21592271327972412, -0.23569896817207336, 0.7600985765457153, -0.5265558958053589, -0.9696499705314636, 1.8907700777053833, 1.6528328657150269, -2.5755834579467773, -0.9422340989112854, -0.011154735460877419, 0.3210212290287018, -0.15673844516277313, 1.5615073442459106, 0.7859820127487183, -0.7617250084877014, 0.7928550243377686, -0.20471149682998657, 0.02357448823750019, -3.585054874420166, -1.2624372243881226, 1.0672484636306763, 0.04047517105937004, 1.248809576034546, -0.008575002662837505, 0.42519113421440125, 4.137749671936035, 0.7681328654289246, 1.095931887626648, -0.03225542977452278, -0.5944556593894958, 0.9536892175674438, -0.20801293849945068, 0.07610408961772919, 0.20721770823001862, -0.27277424931526184, -0.44040486216545105, 0.5021961331367493, 0.1528666764497757, 0.350272536277771, 0.7160888314247131, -0.1915913224220276, 0.9655354619026184, 0.12441640347242355, 1.6462287902832031, -0.2813154458999634, 0.1999908834695816, -0.32378292083740234, -0.2137266844511032, -0.09182652086019516, -0.1362355351448059, 1.5628581047058105, -1.5718921422958374, 2.2792251110076904, -0.02986547350883484, 0.00012202625657664612, -1.1365971565246582, -0.05000520870089531, 0.1319233626127243, 0.5514609813690186, -3.7948179244995117, -0.22145286202430725, -0.20299570262432098, 0.8207957148551941, -0.9482571482658386, -0.06674054265022278, 0.02816694602370262, 0.1823098063468933, 0.13695265352725983, -1.804254174232483, 0.11539361625909805, 0.1309412270784378, -0.21364830434322357, -0.2623947262763977, 0.1304592788219452, -0.7396759390830994, 0.4446168541908264, 0.24991080164909363, -0.004477954935282469, 0.3929135203361511, -0.9036605954170227, 0.7868284583091736, -0.12163885682821274, -0.12168491631746292, 0.10464024543762207, 0.8775134682655334, 0.03202567994594574, 1.2635548114776611, 0.7977120876312256, -0.5057492852210999, -1.8488627672195435, -0.46727147698402405, 0.11941173672676086, 0.9436666369438171, -0.00298406183719635, -1.241309642791748, -2.9664156436920166, -2.19103741645813, -0.6137220859527588, 0.20772941410541534, 1.0303157567977905, 1.3666996955871582, 0.8427934646606445, 1.7853713035583496, -4.076235771179199, 1.2035282850265503, 0.047113433480262756, 0.3641129434108734, -2.3469808101654053, -1.6497621536254883, -0.9695942997932434, -0.7212231159210205, 0.6504681706428528, 1.0190237760543823, -1.8658640384674072, -0.5941558480262756, 0.16140708327293396, -0.5940128564834595, -0.8313709497451782, -0.622218906879425, 0.006817836780101061, 0.07452552765607834, 0.7433132529258728, 0.3352642357349396, -0.13309581577777863, 0.33301499485969543, 0.12140287458896637, -0.10149379819631577, 0.95916748046875, 0.09302964061498642, -0.1616639345884323, -0.42692458629608154, -0.2892462909221649, 0.0008271511178463697, -0.7147166728973389, -1.6429450511932373, -0.1288241595029831, 0.6736592054367065, 0.3450287878513336, 0.9637908339500427, 1.9321675300598145, -0.216248020529747, 1.1478341817855835, -0.310682475566864, 0.2319338172674179, 0.35735049843788147, -0.4570881128311157, -2.0154080390930176, 0.32704541087150574, -1.1682450771331787, 0.546191394329071, 0.2953132688999176, 0.11244699358940125, -0.43810924887657166, -0.04441487416625023, -0.14449070394039154, -0.002889599185436964, -0.0422302782535553, -3.508362054824829, -0.13659769296646118, 1.1647471189498901, 0.017459135502576828, 0.12964852154254913, -0.29892784357070923, -0.7924109697341919, -0.1934477537870407, 0.5428726077079773, -1.7353497743606567, 0.04573370888829231, -2.5637614727020264, 1.03487229347229, 2.8135507106781006, -0.3367159068584442, 0.07720683515071869, 0.03148805350065231, -0.3103248178958893, 0.07262709736824036, -0.3767330050468445, 0.027946164831519127, -0.0017006046837195754, -0.6356973052024841, 0.4526914656162262, 0.2659170925617218, 0.24815702438354492, -0.7017311453819275, -0.7047704458236694, 1.4810250997543335, 0.2940017282962799, -1.2387917041778564, -0.008851856924593449, -0.3612288236618042, -0.8279824256896973, 0.31639325618743896, -1.7994338274002075, -0.2582084834575653, 0.11793316900730133, -1.4276202917099, 0.023444872349500656, 0.008856724947690964, 0.2724504768848419, 0.27360257506370544, -0.21740677952766418, -0.1902437061071396, -1.2013803720474243, -0.5051541924476624, -0.3185231387615204, -0.28597378730773926, 0.23276680707931519, 2.886302947998047, 0.4408683776855469, -0.14660251140594482, 3.3012828826904297, -0.29459404945373535, 0.20875875651836395, 0.89704430103302, 0.11896878480911255, 1.3540258407592773, -0.17159733176231384, 0.8736822009086609, 1.4322699308395386, -0.5904638171195984, -0.46338319778442383, -0.016803078353405, 0.8644013404846191, 0.2603788673877716, 1.1485955715179443, -2.453763008117676, 0.017014658078551292, -0.009189401753246784, 0.07329842448234558, -0.1699310839176178, -2.1902999877929688, 2.5101399421691895, 0.01455121673643589, -0.10029593855142593, 0.6665987372398376, 1.9654185771942139, 0.12271858006715775, -0.04826954007148743, -0.40367576479911804, 0.6667588353157043, -0.19650521874427795, -0.3728081285953522, 0.40835484862327576, -0.001083104987628758, 0.29597702622413635, 0.08884524554014206, 0.5081447958946228, 0.8064931631088257, -0.4093588888645172, -0.15303891897201538, 0.11752515286207199, 0.40098872780799866, 0.35595420002937317, -0.5044713020324707, -0.12795259058475494, 3.2451181411743164, 0.30462515354156494, -0.38037925958633423, 0.3084413409233093, -0.5187071561813354, 0.14088737964630127, 0.47404149174690247, 0.6818552017211914, 0.09825045615434647, 0.3998121917247772, -0.5682349801063538, 1.8953603506088257, 0.1393975466489792, -1.1040817499160767, -0.6988955140113831, -0.7846351861953735, 0.9833399653434753, -0.9219958186149597, -0.6234639286994934, 1.117456078529358, -0.08656153082847595, 1.1536424160003662, -0.6971821784973145, -0.15413722395896912, -0.15407690405845642, 1.1962417364120483, -0.09009873867034912, -0.9176692962646484, -0.8746522665023804, -0.4803544580936432, -1.8962405920028687, -0.02121577598154545, -0.12187381088733673, -0.07912888377904892, -1.2715150117874146, -0.33265385031700134, 0.425057590007782, -0.0033314304891973734, -1.69516122341156, -2.417221784591675, 0.1044718474149704, 0.014170350506901741, -0.23956364393234253, -0.34488868713378906, -1.0044938325881958, 1.1791555881500244, 0.6501947045326233, -0.3968523442745209, -0.7099586725234985, -0.009200646542012691, -0.08509746938943863, -0.09316558390855789, 0.6468090415000916, -0.8629385828971863, 0.3496602475643158, -0.957863450050354, 0.8954226970672607, -0.31646403670310974, 0.43880993127822876, -0.009856186807155609, 0.039996445178985596, 0.0819530338048935, -0.13430805504322052, 0.010372741147875786, -0.7935612797737122, -0.013045454397797585, 0.483871191740036, 0.04325943812727928, 0.04467553645372391, 0.17749722301959991, -0.6115999221801758, 0.06512651592493057, 1.259905219078064, 0.03463979437947273, 0.17604847252368927, 1.1361427307128906, -0.46547845005989075, 0.6680338978767395, 0.8167768120765686, -2.793067455291748, -0.0710262879729271, 0.3529852330684662, -0.1102340891957283, 0.9667181968688965, 1.853496789932251, 2.9432334899902344, -0.46767857670783997, -3.9139163494110107, -1.235120415687561, 1.9801499843597412, -0.053367551416158676, -0.9413884282112122, 0.1960993856191635, -0.12659019231796265, -0.03471051901578903, 0.04433942958712578, 0.7129385471343994, 0.198707714676857, -0.00586014473810792, -1.2679500579833984, -1.548202633857727, 0.20507261157035828, 2.2951912879943848, -0.0014008136931806803, 0.8414919376373291, 0.4771817624568939, -0.1453111618757248, -0.009680625982582569, 0.2432054728269577, 0.15163618326187134, 0.014545436017215252, -3.754128932952881, 1.021246075630188, 0.43485233187675476, 0.37211230397224426, 0.4138875901699066, 0.8083134293556213, -0.44639524817466736, 0.5324819684028625, -3.7109432220458984, -0.16572625935077667, 0.005075391381978989, 0.0868719145655632, -1.0306981801986694, -0.11499214917421341, -0.7968873977661133, 0.34246280789375305, 0.04562192037701607, -0.4237988293170929, 0.48984965682029724, 0.15933287143707275, -0.17441809177398682, 0.3869021236896515, -0.2969670593738556, 2.3351030349731445, 1.273697018623352, 1.86785089969635, -0.8453136086463928, -0.251968652009964, 0.14291010797023773, -0.009889193810522556, 0.6343810558319092, -0.016257110983133316, -0.11706002056598663, 2.0547842979431152, -2.098079204559326, -0.4089590311050415, -0.2848796546459198, -0.5891977548599243, -3.0329818725585938, 0.6005692481994629, 0.6947574019432068, -0.13606353104114532, 0.1707436740398407, 0.013834253884851933, 0.42439255118370056, 0.23014695942401886, -0.16466467082500458, -0.02062746323645115, -0.1021561548113823, -0.17488694190979004, -0.46888411045074463, 0.2837051749229431, -0.22391685843467712, -0.2971433699131012, -2.451005697250366, -0.10573965311050415, 0.41284969449043274, -0.012223921716213226, 0.8044434189796448, -0.010507154278457165, -0.5940646529197693, -2.9857890605926514, 0.5754210948944092, -1.9663058519363403, -0.18225814402103424, -0.17864669859409332, -2.099121570587158, -0.4610533118247986, -0.25131526589393616, 0.16538724303245544, -1.2907681465148926, 0.5503354072570801, -0.42200252413749695], [0.8133169412612915, 0.7348896861076355, -0.6407037973403931, 1.307740569114685, -1.6903679370880127, 0.7262956500053406, -0.09806578606367111, -0.14783209562301636, -0.16283664107322693, 2.5586862564086914, 0.06082409620285034, -0.29611536860466003, -0.4701516032218933, -0.46810466051101685, -0.2537412941455841, -0.7414957880973816, 0.8601140975952148, -0.6284998655319214, 0.7246940732002258, -0.3098004162311554, 0.1894792765378952, -1.021911859512329, -0.9415962100028992, 0.04025599732995033, -0.2186783403158188, -0.5867918133735657, -0.027484280988574028, 0.2536635994911194, 2.167510986328125, 1.826680302619934, -0.7577443718910217, -1.5299077033996582, -0.8891938328742981, 1.6095266342163086, -0.11447777599096298, 0.05669919773936272, -0.7385477423667908, -0.15157023072242737, -0.8826717138290405, -1.007557988166809, -0.15308597683906555, -0.07436296343803406, 0.7383831143379211, 1.8522111177444458, 0.22158046066761017, 0.22854210436344147, 0.3543630540370941, -0.6642791628837585, 0.656730592250824, -1.0210185050964355, 1.505139708518982, -0.565004289150238, -0.006532208062708378, 0.3855358064174652, 0.6792545914649963, -0.04147109389305115, 0.006121303886175156, -1.1822948455810547, -0.36878064274787903, -0.026194388046860695, -6.146212100982666, 0.02454295940697193, 0.8869457840919495, -0.4139387011528015, -1.3844236135482788, 0.14816316962242126, -0.22718454897403717, -1.8043967485427856, -0.10646966844797134, -0.17201636731624603, 0.4810146689414978, 0.5963714718818665, 0.07156163454055786, -0.11869948357343674, -3.0814976692199707, -0.9269568920135498, 0.056925106793642044, -0.7028748989105225, 0.6462528109550476, 1.078951120376587, -0.08250447362661362, -0.3969135582447052, 0.5850421786308289, 0.80094975233078, -0.9433490037918091, -1.893802285194397, -0.19391587376594543, -0.07134643197059631, -0.4323931336402893, -0.313142865896225, 1.4721465110778809, -0.04971998929977417, -0.394975870847702, -0.936511754989624, -2.0561702251434326, -0.06741585582494736, 0.46942397952079773, -0.8119876384735107, 3.8821685314178467, 0.16646835207939148, -0.5984253883361816, 1.4499074220657349, -2.436445474624634, -0.15500134229660034, -1.860899806022644, 0.28706327080726624, -0.09427326172590256, -2.9736130237579346, 0.37705013155937195, -0.6700264811515808, -0.17774710059165955, 0.07343492656946182, 0.3140871524810791, 0.677725613117218, -0.0810488611459732, 1.1952437162399292, -0.0954805538058281, 1.7093645334243774, 0.0043762121349573135, 0.9170964956283569, -0.38933077454566956, 0.10417857766151428, -0.24065527319908142, 1.8502689599990845, 0.01839747466146946, 0.10572203993797302, 0.18311862647533417, 0.3778427243232727, 0.001999977044761181, -2.2968523502349854, -1.4371713399887085, 0.024919578805565834, 0.6465420722961426, 0.1451859474182129, 0.6463629007339478, 0.22876514494419098, 0.21947428584098816, 0.7919831871986389, 0.596514105796814, 0.21661058068275452, 1.7759350538253784, 0.5168532729148865, 1.2003909349441528, -2.3213422298431396, 3.5920662879943848, 1.5470114946365356, -1.5358275175094604, -1.163018822669983, -1.525834083557129, 1.6516813039779663, 0.06722606718540192, -0.37189480662345886, -0.2818581461906433, 2.080502510070801, -0.012296458706259727, 0.014434718526899815, 0.09641198813915253, -0.5882680416107178, -0.8619222640991211, 0.15729577839374542, 0.05530569702386856, 0.22498606145381927, 0.033736396580934525, -0.08564528077840805, -2.2140870094299316, 0.768627405166626, -0.03383375704288483, 1.050392985343933, 0.17659816145896912, 2.1244056224823, 0.17887181043624878, 0.047889593988657, 0.2794285714626312, 0.0032281260937452316, -0.032285477966070175, 0.859401285648346, -0.05519689992070198, 0.5437567830085754, 0.40626972913742065, 0.21304462850093842, 1.2355332374572754, 0.6337811946868896, -0.6475203633308411, -1.6238195896148682, -0.4657438397407532, 0.1943233460187912, -0.9041234254837036, 2.3986759185791016, -0.4327053725719452, 0.04041906073689461, 0.7843095064163208, 1.0839838981628418, 0.8150398135185242, -0.008681008592247963, 1.2500205039978027, -1.3851971626281738, -0.6448253989219666, 0.07665608823299408, -0.05042696371674538, 1.5188241004943848, -0.2834187150001526, -0.49603888392448425, 0.11733072996139526, -0.5360524654388428, 0.05814332887530327, 0.10468365997076035, -0.2628861665725708, 0.01917327009141445, -0.028215430676937103, 0.39872926473617554, 0.6492022275924683, -1.2691348791122437, -0.07585219293832779, 0.6290682554244995, 0.7534677982330322, -1.275556206703186, 1.1490520238876343, -0.018450461328029633, 0.3075765371322632, -0.5485436916351318, -0.16877567768096924, -0.6478992700576782, -0.9716159701347351, 1.9540774822235107, 0.31139376759529114, 0.17076726257801056, -2.7229721546173096, -0.10501972585916519, 0.06097440794110298, 0.27045315504074097, -0.01320449449121952, 0.03575243800878525, 0.16310009360313416, 0.6895639300346375, -0.37887099385261536, 0.0375688336789608, 0.9478517174720764, -0.024301055818796158, -0.003710933495312929, 3.589902639389038, 0.6385840773582458, -0.08785175532102585, 0.20346477627754211, -0.007604595739394426, -0.3281986713409424, -0.48933595418930054, 0.00040656010969541967, 1.0352559089660645, 0.940412700176239, 0.028044430539011955, 0.4850691258907318, -0.023195525631308556, 0.06797270476818085, -0.47064223885536194, -0.12692135572433472, 4.2242817878723145, -1.7218027114868164, 0.7219545841217041, 0.033991001546382904, -0.5855484008789062, 0.2676546275615692, -0.5365572571754456, 0.31151965260505676, -0.5915401577949524, -0.7957488298416138, -0.05248311907052994, 0.40488871932029724, -0.0556231327354908, 0.450555682182312, -0.013000897131860256, -0.6182400584220886, 0.1582152545452118, -0.12399756908416748, -0.3433460295200348, -0.03731716796755791, 0.7333104610443115, -0.28530195355415344, -0.016280489042401314, 0.1701854169368744, -0.13024844229221344, 0.9096853137016296, 0.1027747094631195, 3.768799304962158, 0.36272719502449036, -1.5511775016784668, -1.013805866241455, 0.5505042672157288, -0.6296735405921936, -0.07545293122529984, -0.6491953134536743, 0.19229473173618317, 0.8237385153770447, 0.6444450616836548, 0.020940713584423065, -0.4111764132976532, -0.9349450469017029, 0.35230517387390137, -0.17754973471164703, 0.27985069155693054, 0.5708277821540833, 0.5308812856674194, 1.0525532960891724, -0.37026554346084595, 0.012126914225518703, 1.1398591995239258, 0.17658314108848572, 0.46112728118896484, 0.022444674745202065, -0.024562893435359, -0.2219531387090683, -0.13942083716392517, 0.014177755452692509, 0.5022814273834229, 0.6631842851638794, 1.7504314184188843, -1.4377275705337524, 0.3368667662143707, 0.28387054800987244, -0.6625555157661438, 3.285649061203003, -0.10186522454023361, -0.3709048330783844, 0.4408039450645447, -0.3149048686027527, 1.5654056072235107, -0.28311142325401306, 0.5349839925765991, -2.606328010559082, -0.028883690014481544, -0.17697009444236755, -0.03956496715545654, 0.8862691521644592, 0.9893565773963928, -0.7882623672485352, 0.5069701075553894, 1.4634441137313843, 0.5653508305549622, 0.39750149846076965, 0.010210871696472168, -0.027402443811297417, -0.7028108835220337, -0.0196211040019989, 0.49333155155181885, -0.17060139775276184, 1.7061511278152466, -0.19892624020576477, -0.035903625190258026, -2.7055323123931885, -2.0308148860931396, -0.6507477164268494, -1.7055224180221558, -0.9974637031555176, -0.03542203828692436, -0.956232488155365, 0.42344358563423157, -1.2881134748458862, -0.6881807446479797, -0.054075974971055984, -0.4912823736667633, -0.13663093745708466, 0.06663582473993301, 1.0708905458450317, -0.3008970320224762, 0.04402020201086998, -0.3217301368713379, -0.2632168233394623, -1.0202109813690186, 0.28263381123542786, -0.4021766185760498, 0.4047568738460541, 1.269865870475769, 0.2913206219673157, 0.10964222997426987, 1.003806233406067, 1.6577394008636475, -0.04900900647044182, -2.8367950916290283, -0.38466838002204895, -0.1917112022638321, -1.837924838066101, -0.24799315631389618, -2.182835817337036, -0.03267551213502884, -0.043859805911779404, 0.851832926273346, -2.1434319019317627, -0.4638019800186157, -0.3769519627094269, 1.2491583824157715, -0.29405343532562256, 0.26101282238960266, -0.8315457105636597, 0.803457498550415, -0.24673524498939514, -0.07132785767316818, -0.11864780634641647, -0.12337617576122284, -1.0218219757080078, -0.0890728086233139, 0.31508147716522217, 0.0985976904630661, -0.3874722123146057, -0.004516371991485357, 0.5943170189857483, -0.6179367303848267, -0.32863637804985046, 0.021352672949433327, -3.1815662384033203, -0.9342101812362671, 0.05246889963746071, -0.8045923709869385, 0.10698265582323074, 0.46407657861709595, -0.28013941645622253, -0.23844878375530243, 0.023822860792279243, -0.7137748599052429, 0.013236138969659805, -1.2794790267944336, 1.6502310037612915, 1.6052191257476807, -0.3456568419933319, 0.0013103208038955927, 4.716635704040527, 0.9855522513389587, 0.6386628746986389, 1.9562917947769165, 0.19577226042747498, 0.5424498319625854, 0.6662272810935974, 0.6869895458221436, 1.3892841339111328, 2.4331231117248535, -0.8697240948677063, -0.11977697163820267, 2.3136868476867676, 0.5460524559020996, -0.7337608933448792, -0.07347339391708374, -0.9780797958374023, -0.6423958539962769, -0.48956289887428284, -0.4075983464717865, 0.21550121903419495, -0.3782004117965698, 0.2114839106798172, 0.40373027324676514, 0.2676137387752533, 0.061357345432043076, -0.0958123505115509, 0.14617379009723663, -0.11406797170639038, -0.01676025614142418, 0.9466650485992432, 0.047995638102293015, -0.5504101514816284, -0.8740134239196777, 0.055595505982637405, -0.03461182489991188, 0.3137355446815491, 0.25827065110206604, 0.14858360588550568, 1.0882662534713745, -1.4751033782958984, 0.7500470876693726, -0.14734165370464325, -0.19098685681819916, -0.3525311052799225, 0.7173346877098083, -0.9166110157966614, 0.40327587723731995, 1.0451802015304565, -0.29122161865234375, -0.10836543142795563, -0.5847066044807434, -1.803920865058899, -0.2665450870990753, -0.2146640419960022, 1.1332918405532837, 0.21633760631084442, -0.7306106686592102, 1.633677363395691, -0.1566815823316574, 0.36376145482063293, 0.010877206921577454, -0.30153700709342957, -3.47332763671875, 0.020135799422860146, 1.2656360864639282, -0.017440680414438248, -0.04957214370369911, -0.04605468735098839, -0.11254053562879562, 0.7255346179008484, -0.21186724305152893, 0.2903788387775421, -0.08571894466876984, 1.2944681644439697, -2.6268808841705322, 3.422863006591797, 0.22653037309646606, 0.11645909398794174, 0.11494883894920349, 0.7816162109375, 0.06693532317876816, -0.18408915400505066, 0.11989003419876099, 0.08639617264270782, -4.014827728271484, 0.07780753076076508, -0.41294753551483154, 0.33805662393569946, -0.07681313902139664, -0.24634219706058502, 0.7714728116989136, 1.3628919124603271, 2.1468162536621094, -0.013914497569203377, -1.244337558746338, -0.1588428020477295, 0.5469204783439636, 0.9257490634918213, 0.04125090688467026, 1.1769720315933228, 0.2932398021221161, -0.0027091181837022305, 0.01555684208869934, -0.830692708492279, -0.04792674630880356, 0.29515203833580017, 0.06487102806568146, 0.6261301040649414, -1.1518336534500122, -0.008571472950279713, -0.4143486022949219, -1.1516051292419434, 3.085592031478882, 0.1578885316848755, -0.06607110798358917, -0.21698756515979767, 2.1337645053863525, 0.03860202059149742, 0.17964816093444824, -0.1793919950723648, 1.9912446737289429, 0.49857550859451294, -1.0767802000045776, 0.005836301017552614, -0.953768253326416, -0.4482344388961792, -0.4753909111022949, 1.0091325044631958, -0.09420818835496902, 0.7720040082931519, 1.4500993490219116, 0.006835480220615864, 0.03534432873129845, 0.15927504003047943, -1.2633779048919678, -0.0007967421552166343, 1.953302025794983, -3.3321101665496826, -0.033024225383996964, -0.3698081076145172, 0.0985078513622284, 0.08956224471330643, 0.08345016837120056, 0.05391990393400192, 0.3177320659160614, 0.4309932291507721, 1.3453181982040405, 0.7873678803443909, 0.022368889302015305, -0.2982945144176483, 0.08930562436580658, 0.4099557399749756, 0.41496384143829346, 0.5057837963104248, -0.11461061239242554, -0.0004556666826829314, -0.025014396756887436, -0.5181007981300354, 2.0557925701141357, -0.010841979645192623, -2.4663095474243164, 0.17722594738006592, -0.3113986551761627, 0.30453819036483765, 1.2189387083053589, -0.23334412276744843, -0.027677634730935097, -0.1983206868171692, 0.06295337527990341, -0.4287964403629303, 0.10396910458803177, -0.7288270592689514, 0.9738210439682007, -0.20109058916568756, -1.5956426858901978, 1.3344296216964722, 0.7612849473953247, -0.003716869978234172, -0.40948721766471863, -0.057019393891096115, 0.3369050621986389, -1.1615568399429321, -0.03391028568148613, 0.4711518883705139, 0.21835194528102875, -0.3412180542945862, -1.9596352577209473, -0.7055650353431702, 0.19430583715438843, 0.10346286743879318, 1.3862789869308472, -0.44060811400413513, 0.1456000804901123, -0.008748038671910763, 0.013162758201360703, -0.7367734909057617, -0.28464174270629883, -0.022890325635671616, -1.1513837575912476, -1.6399316787719727, -0.3014311194419861, -0.40149810910224915, -0.16968896985054016, -1.0532811880111694, -1.3332946300506592, -0.9881201386451721, 0.46396806836128235, -0.42451271414756775, -0.284501314163208, 0.37277379631996155, -0.7473199367523193, -0.2826817035675049, -1.0409425497055054, -2.6509227752685547, -0.15747754275798798, -0.09712095558643341, 0.054095447063446045, 0.19650396704673767, 0.5788998603820801, -0.04556366801261902, 0.14280658960342407, 1.0197135210037231, 0.10694689303636551, 0.21233373880386353, -0.16760553419589996, -0.05763929337263107, -0.011583009734749794, 0.4565958082675934, 0.052498508244752884, -2.314105987548828, 0.8208438158035278, 0.42217859625816345, -0.25631171464920044, -0.04871107637882233, 0.22252899408340454, 1.9390546083450317, 0.8937265276908875, 1.1302064657211304, -0.4653696119785309, 0.5575893521308899, -1.2709624767303467, -1.795149564743042, -0.28226616978645325, -0.7458749413490295, -1.3051666021347046, -1.660956621170044, -0.3061187267303467, 0.585995078086853, -1.6202313899993896, 1.3458268642425537, -0.02215002290904522, 0.10067199170589447, 0.7416645884513855, 0.005205283407121897, -0.7108249068260193, -0.03348860144615173, -0.4872283339500427, 0.2573137879371643, -0.005861067213118076, -1.577787160873413, -2.578808307647705, 0.1956147700548172, 0.4037092328071594, -0.0037906737998127937, 0.7817184925079346, 0.22162020206451416, 1.3033078908920288, -0.5041394233703613, -0.3200855851173401, 1.3095217943191528, 1.5166417360305786, -3.189587354660034, 0.10076904296875, 0.2297108918428421, -0.23832087218761444, -0.16209068894386292, 0.4773118793964386, -0.19232459366321564, -0.17724379897117615, 3.4649062156677246, -0.01941833831369877, 0.06224273890256882, -0.3956848084926605, 1.538893699645996, 0.09601228684186935, -1.5330359935760498, 0.01824626326560974, -0.20964723825454712, -0.0574323795735836, 0.6977540850639343, 0.2375999540090561, -1.1698671579360962, -0.7283147573471069, -1.0183385610580444, 0.17120787501335144, -0.5135116577148438, -2.200634002685547, -0.5059838891029358, -0.00851268321275711, -0.1641194373369217, -0.45536115765571594, -0.7287355661392212, -0.06513594835996628, -0.1513451784849167, -0.36054208874702454, -1.6073896884918213, 0.20182010531425476, -0.3573261797428131, 1.3155252933502197, -2.264899730682373, 0.9104757308959961, -0.8692061901092529, -0.28365829586982727, -0.0244553592056036, -0.04694291949272156, 0.19319555163383484, 0.10357702523469925, 0.11070507764816284, 0.13455913960933685, 0.10618587583303452, 2.1221601963043213, -0.2868695855140686, -0.11404861509799957, -0.5450851321220398, 0.08085785806179047, -0.4704812467098236, -0.06546486169099808, -1.6848117113113403, 0.1657724529504776, 2.036857843399048, -0.07191386073827744, -0.6015863418579102, -1.454341173171997, -0.7153104543685913, 1.302844524383545, 0.8477540612220764, -0.2642953395843506, 0.7132159471511841, 0.35337749123573303, -0.13031740486621857, -0.047888919711112976, 1.442765474319458, -0.09765556454658508, -0.08951792120933533], [0.40315189957618713, 0.8394521474838257, 0.41783276200294495, -0.6742720603942871, -3.1417927742004395, -0.3446759879589081, 0.056589845567941666, -0.058042850345373154, 0.33098238706588745, -2.073822259902954, 1.2798548936843872, -0.4620853662490845, -0.486236035823822, 0.15093852579593658, -0.6558922529220581, 0.3445739150047302, -0.1029466986656189, 0.4911309778690338, 0.31460875272750854, 0.0010409801034256816, 0.16213703155517578, -1.040343999862671, 1.143102765083313, -0.004032893106341362, -0.015468355268239975, -1.8791338205337524, -0.33329129219055176, -0.31863531470298767, -0.850008487701416, 0.05318663269281387, 0.9114987254142761, 0.5719559192657471, 1.3432005643844604, 1.0286870002746582, 0.2572510242462158, -1.1267166137695312, -2.72161865234375, -0.08437909185886383, -0.0004761445743497461, -0.19372402131557465, 1.1563160419464111, -2.6852798461914062, -0.4155728816986084, 0.26745903491973877, -0.27450522780418396, -0.8231896162033081, -3.046665906906128, 0.892282247543335, 0.08491528034210205, 0.31628185510635376, 0.3475143015384674, 0.14361557364463806, -0.0211477167904377, 0.4175436794757843, 1.1204196214675903, 0.008144119754433632, -0.3432570695877075, 0.8130595088005066, 1.0665265321731567, 0.09805060178041458, -2.036834955215454, 0.10269255936145782, -1.1097158193588257, -0.7302359938621521, -1.15976881980896, 1.089590311050415, 0.47330981492996216, -0.6043587327003479, 0.001877018017694354, -0.21407672762870789, -0.3371725082397461, -0.4812646806240082, -0.06070980429649353, 0.008266745135188103, -0.6766912937164307, 1.0257912874221802, 0.4099917709827423, -0.11573631316423416, 1.0862767696380615, 4.799720764160156, 0.07513730973005295, 0.10776431858539581, -0.47308576107025146, -0.28025445342063904, -0.6647434234619141, 0.33662980794906616, -0.09150097519159317, -0.18044574558734894, 0.47369447350502014, 0.4951118528842926, -0.4349597692489624, -0.013223514892160892, -1.1984390020370483, -0.6364036202430725, 1.127469539642334, -0.14834752678871155, -0.6564738154411316, 0.7458174228668213, -1.9983645677566528, 0.047912195324897766, -1.3048688173294067, 0.7086930871009827, -0.7563766837120056, -0.36582255363464355, -1.6405878067016602, 0.31450384855270386, 2.4874956607818604, -0.8516413569450378, -0.24267248809337616, -0.03984591364860535, -0.2516603171825409, -0.26918885111808777, 0.016160231083631516, 1.4460188150405884, 0.05047011747956276, -1.0847046375274658, 0.17261384427547455, -3.662493944168091, -0.0174768827855587, -0.6078988313674927, 0.33831003308296204, -0.2391161024570465, 0.05230112746357918, -0.024292102083563805, -0.04094540327787399, -0.7470621466636658, -0.7079652547836304, -1.2797796726226807, -0.02875271439552307, 0.8238130807876587, -0.5976404547691345, 0.07894039899110794, 1.8141697645187378, 1.6348061561584473, 0.20048888027668, -0.1845710724592209, 0.17872391641139984, -0.5630804300308228, -0.3005300760269165, 0.5377815365791321, 2.1828322410583496, -2.8008744716644287, -0.13941927254199982, 0.1399354785680771, 0.9694118499755859, -0.28540393710136414, 1.4763556718826294, -0.10770071297883987, 0.8450304269790649, 2.375063419342041, -0.2955954670906067, -0.896219789981842, -0.6214008331298828, -0.7352186441421509, -0.08620767295360565, 0.06561163067817688, -0.015865003690123558, -0.22998075187206268, 0.33524540066719055, 1.0185104608535767, -0.1862933486700058, 0.1359504759311676, 0.12021588534116745, 0.0024860911071300507, 0.8900548815727234, 0.6609159111976624, -0.01006369199603796, -0.23049677908420563, -0.01607087254524231, 1.9409968852996826, -0.20206010341644287, -0.022977329790592194, 0.289948046207428, 0.016777735203504562, -0.06693971902132034, -0.5675016641616821, -0.01847252808511257, -0.4274216294288635, -0.10864392668008804, 0.9940167665481567, -1.3844718933105469, 0.16527725756168365, 0.4755505323410034, 2.3484175205230713, -1.2184257507324219, 1.3975292444229126, -0.640487790107727, -5.801059722900391, 2.6218535900115967, -0.12201821804046631, -1.0238841772079468, 0.4249342083930969, 0.14004571735858917, -0.2268105298280716, -1.1932529211044312, -2.4841885566711426, -0.40311411023139954, -0.5867900252342224, 0.008008268661797047, -0.35473230481147766, 0.22848372161388397, 0.04066995158791542, 0.033327359706163406, -0.20782603323459625, -0.0377536378800869, -0.03229217603802681, 0.4077967703342438, 0.005341308191418648, 0.028366733342409134, -0.6618848443031311, 0.0701327919960022, 0.9762923717498779, -0.004868088755756617, -0.000732862506993115, -0.0047195469960570335, -0.43773066997528076, 0.9734739661216736, -0.22481979429721832, 0.41627568006515503, -0.035204894840717316, 0.13925032317638397, -0.3292299211025238, 1.3125094175338745, 0.500731348991394, -0.15820026397705078, -0.1244392916560173, 0.9209259748458862, -1.14580500125885, -0.19091671705245972, -0.08613042533397675, -0.2732914090156555, 0.3587121367454529, -0.26597362756729126, -0.26600268483161926, -0.24051572382450104, 0.3266565501689911, -0.1680695116519928, -0.6793595552444458, -0.059635717421770096, -1.0231187343597412, -0.1291942149400711, 1.3366904258728027, -1.6251764297485352, -0.16334207355976105, 0.13016638159751892, -0.040086787194013596, -0.0036875198129564524, -0.4651375114917755, -1.0311036109924316, -0.6998888254165649, 0.07789550721645355, -0.08245688676834106, 0.0042182812467217445, 0.586397111415863, -0.03239881619811058, 0.5926160216331482, -1.1051257848739624, 1.8853838443756104, -0.034278932958841324, -0.32344192266464233, -0.4679592251777649, -0.7031901478767395, -0.19510968029499054, 0.10528866946697235, 0.4547973871231079, 0.035681359469890594, -0.22742579877376556, -0.004258129745721817, 0.013578934594988823, 0.09209609031677246, 0.34350836277008057, 0.08994720876216888, -0.06147230044007301, -0.0120103619992733, -0.2786785662174225, -2.1075689792633057, 0.29724711179733276, 0.0011153900995850563, 0.600300669670105, -0.5926926732063293, 1.6284918785095215, -2.1155617237091064, -4.602004528045654, 0.4380936920642853, -0.04426373168826103, 1.465917706489563, -0.8460103273391724, 0.5393875241279602, 0.33742499351501465, 0.15856625139713287, 0.13542744517326355, 3.161579132080078, -0.18720702826976776, 0.11643548309803009, 0.16631034016609192, -0.11775555461645126, -0.6218293905258179, 0.03685750812292099, 0.3592991828918457, -0.14819954335689545, -1.4466352462768555, 0.35793671011924744, 0.2278938591480255, -1.9051151275634766, -1.244231104850769, -1.8149631023406982, -0.2267179787158966, -0.049881067126989365, -1.1479092836380005, 0.2794066369533539, -0.2939005196094513, 0.05579096078872681, 0.7239987850189209, -1.6507554054260254, -2.0027997493743896, 1.939897060394287, 0.707248866558075, -0.7640687823295593, 1.0684560537338257, 3.1064932346343994, 0.3087383210659027, 0.5488907098770142, 0.6724064946174622, -0.6934510469436646, 1.0481774806976318, 0.2852688133716583, -1.4386080503463745, 1.8391876220703125, 0.03981420770287514, -0.2933524549007416, 0.3237563669681549, -0.08666316419839859, 0.4413120150566101, 0.5566490292549133, 0.6797928214073181, 0.2144455760717392, 0.8999117612838745, 0.07273757457733154, 0.02388402819633484, -0.31704947352409363, -1.2410911321640015, 1.2978914976119995, 0.4129300117492676, -0.5101358294487, -1.7332406044006348, 1.3245762586593628, 0.1610955446958542, 3.572427749633789, 0.6347637176513672, 2.003589391708374, -0.14516934752464294, 0.096456378698349, -0.06387076526880264, -0.00311834504827857, 1.030814528465271, -0.4047132432460785, 2.615924835205078, -0.3349359929561615, 0.2767074704170227, -0.6850827932357788, -0.31888800859451294, -0.43569788336753845, 0.18960724771022797, 1.7725014686584473, 0.15670146048069, -0.038732513785362244, -0.379817396402359, -0.3681710362434387, -0.02761368826031685, 0.3763715922832489, 0.21508872509002686, -0.6880557537078857, -1.5989969968795776, -1.6270031929016113, 0.6013720631599426, -0.3267725110054016, -3.4618966579437256, 0.043657369911670685, 0.3798690140247345, -1.1224499940872192, 0.4543253779411316, 3.357424020767212, 0.05864451453089714, -0.058432262390851974, 0.42227429151535034, -1.5160683393478394, 0.7872336506843567, 0.07227195799350739, 0.48925817012786865, 0.4257037937641144, -0.23412030935287476, 0.2701185643672943, -0.4337553381919861, -0.07595615833997726, 0.05233431234955788, 0.016370439901947975, 0.1790076494216919, 0.5612609386444092, 0.1440301388502121, 0.01062376145273447, -0.3098883032798767, -0.3927006125450134, 0.20115944743156433, -0.041076429188251495, -2.3842062950134277, 0.19451139867305756, 0.0363411046564579, -2.202014684677124, -0.9121412038803101, 1.6010044813156128, 0.12817084789276123, 0.08015664666891098, 0.8574224710464478, -1.9196293354034424, -0.5194944143295288, -1.4308207035064697, -1.0673168897628784, 0.3579646646976471, 1.4971823692321777, 0.6264596581459045, -0.033247437328100204, -1.2911920547485352, 0.13092149794101715, 2.2916786670684814, -2.7031514644622803, -0.4982229173183441, -0.23065900802612305, 0.007518342696130276, 0.3704373836517334, 0.3844211995601654, 0.22498160600662231, 0.9522804617881775, -0.08204039186239243, -0.8087000846862793, -0.011038154363632202, 0.5844278931617737, 0.9688094854354858, -0.08913217484951019, 0.40903210639953613, -0.27504169940948486, -0.15437522530555725, -1.3663618564605713, -0.6218395233154297, 0.05840570107102394, -1.352673888206482, 0.0828617736697197, 0.10903436690568924, -0.9204474687576294, -0.008349561132490635, 0.05914635211229324, -0.14235647022724152, 0.11465854197740555, 0.1346416026353836, 0.11500582098960876, 0.08846017718315125, -0.6277097463607788, -0.6363598108291626, 0.011982392519712448, 0.3700904846191406, -0.254963755607605, 0.13229043781757355, -0.5258510112762451, 0.7353131771087646, -1.8530417680740356, -0.8007732629776001, -0.29663339257240295, 0.3801693916320801, 0.5443688631057739, -0.2537391483783722, -0.3463052809238434, 0.25133153796195984, -1.9765138626098633, 1.0522319078445435, 0.3153027296066284, 0.03770480304956436, -2.801482915878296, -0.2344493865966797, -2.423246383666992, 1.09178626537323, 0.8874160647392273, -0.6553449034690857, 0.5963475704193115, 0.022645991295576096, 0.6732751131057739, 0.04509512335062027, -0.3339368402957916, -3.4647247791290283, 0.5248973369598389, 0.023280499503016472, 0.10129708051681519, -0.36508119106292725, -0.013920462690293789, -0.5507522225379944, -0.037329550832509995, -0.6170079708099365, 1.4273583889007568, 0.12837669253349304, 2.527923822402954, -0.5684792995452881, 1.2428194284439087, 0.6920494437217712, 0.041269246488809586, -0.32008880376815796, 0.8182249665260315, 0.2684670090675354, 0.3606955409049988, 0.09573104232549667, 0.034688085317611694, 0.8034625053405762, -0.40629205107688904, 0.18315130472183228, 0.4644377529621124, 0.586156964302063, 0.9411559700965881, -0.43587687611579895, 4.08375883102417, 0.7206604480743408, 0.015233042649924755, 1.6652393341064453, -0.25528958439826965, -0.3998865783214569, -0.46929800510406494, 0.30910149216651917, 0.27319201827049255, -0.023003656417131424, -0.05802838131785393, 0.03282790258526802, 1.0377589464187622, 0.3602256774902344, -1.219639539718628, 0.5374730825424194, -1.2724640369415283, 0.7554032206535339, -0.2417004555463791, 0.40886807441711426, 0.9678600430488586, 0.37345442175865173, -0.9203100204467773, -0.042335204780101776, 0.2092759758234024, -1.1974927186965942, 0.018887346610426903, -0.17458008229732513, -0.011150852777063847, -0.14489561319351196, 0.5102068781852722, 0.910628080368042, 1.0823612213134766, 0.8120909333229065, -0.08183795213699341, 0.16300205886363983, 0.29075703024864197, 0.19672514498233795, 0.5504426956176758, 0.9522400498390198, -0.3032563328742981, -0.10091131925582886, 0.10397431999444962, 0.7133216261863708, 1.1881422996520996, -2.954169750213623, -0.06820426881313324, 0.22115862369537354, -1.4153841733932495, 0.8539583086967468, 0.10530346632003784, 0.01371198333799839, -0.14545519649982452, -0.37947556376457214, -0.4784131348133087, -0.8150490522384644, 0.6660459041595459, 0.034307848662137985, -0.20916084945201874, 0.1337728053331375, -0.7034411430358887, -0.18022669851779938, 0.6033262014389038, 0.042244747281074524, -0.010106202214956284, 0.5319751501083374, -0.10797286033630371, 2.99336576461792, 0.009050696156919003, 1.0397891998291016, -0.03207838907837868, 0.38027775287628174, 0.11416254192590714, 0.0171650443226099, 0.09415877610445023, 0.1652907431125641, -0.02430499903857708, 0.08092179894447327, -0.5041474103927612, 1.653359055519104, -0.3583398759365082, -0.14310415089130402, -0.5066971778869629, -0.5842618346214294, 1.0096542835235596, -1.2351183891296387, -1.244122862815857, 0.20724980533123016, 0.7340694665908813, -0.9400332570075989, -0.16831466555595398, 0.540006697177887, -0.15470385551452637, 0.6475699543952942, -1.2216905355453491, 1.2480642795562744, 0.13085833191871643, -1.1879507303237915, -0.056022342294454575, 2.5599215030670166, -0.7755774259567261, 0.49766066670417786, -0.07483327388763428, 1.8794560432434082, 0.47410517930984497, 0.03878586366772652, 0.04317150637507439, 0.21564194560050964, 1.0033149719238281, -0.19530437886714935, 0.13125470280647278, 1.1709113121032715, -0.578310489654541, 1.4539483785629272, 1.0541068315505981, -0.22804997861385345, 0.24988846480846405, 0.3566293716430664, -0.515142023563385, 1.0337276458740234, -0.6449452042579651, -0.16784711182117462, 0.8174828886985779, 0.02632131241261959, 0.36767813563346863, -0.34264811873435974, -0.1252722144126892, 0.37327829003334045, 0.050001755356788635, -0.04530402272939682, -0.18866831064224243, -0.3381095230579376, -0.218642458319664, -0.36378347873687744, 0.05733725428581238, -0.10522089898586273, 0.2655358612537384, -0.13464295864105225, -1.685468077659607, 0.23079922795295715, 0.23902767896652222, -0.9376773238182068, -0.0798925906419754, -0.05915853753685951, -0.11835826188325882, 0.604926347732544, 0.6051315665245056, -0.6940346360206604, -2.4393646717071533, -0.24914081394672394, -0.45707574486732483, 0.20972023904323578, 1.766399621963501, 0.7889159321784973, 0.8071585297584534, -0.8001652359962463, -1.087131142616272, -0.05958778038620949, 2.374462604522705, 0.07905890047550201, 1.2621653079986572, -1.6166086196899414, 0.3730408251285553, 0.5166343450546265, -0.08622051775455475, -0.42001956701278687, 1.3512533903121948, -0.05391330271959305, -1.9633756875991821, -1.6231285333633423, 0.055199749767780304, -2.710761308670044, -0.019603542983531952, 0.7524863481521606, 0.8969093561172485, 1.2527319192886353, 0.19399337470531464, -0.19939647614955902, 0.09216431528329849, 0.8877914547920227, 4.7635626792907715, 0.8129370808601379, -0.5667872428894043, 0.006360386498272419, 2.5397884845733643, 0.5929239988327026, 0.2323060929775238, 0.19536419212818146, -1.2373324632644653, 0.052297886461019516, 0.015555663965642452, -0.7874049544334412, 0.9386179447174072, 0.066635861992836, -1.0095651149749756, 0.06469858437776566, -0.25081396102905273, -0.09641563147306442, 1.107545018196106, -0.18085721135139465, 1.1090335845947266, -0.08057694882154465, -1.363721251487732, 0.12006721645593643, 0.006518185138702393, 1.7252833843231201, -1.43978750705719, -0.1313101351261139, -0.32961755990982056, 0.3589708209037781, 1.2926700115203857, -0.06249041482806206, -0.12064158171415329, 3.3488926887512207, -2.3252947330474854, -0.40154770016670227, 0.4803934097290039, 0.3316410183906555, 3.7370529174804688, 0.40875673294067383, 0.343698114156723, 0.23689045011997223, 0.14928722381591797, -0.028563180938363075, -0.05975718051195145, -0.10524459183216095, -0.1661124974489212, -1.0034037828445435, -0.17604869604110718, 0.771152913570404, -0.6025682091712952, 0.010481679812073708, -0.8765519261360168, 0.04216694459319115, -3.291973829269409, 0.033776987344026566, -0.9085910320281982, 0.09031188488006592, 0.7336561679840088, 0.04502850025892258, 0.33434993028640747, -1.1655620336532593, 0.05258385092020035, -0.6865238547325134, -0.8142028450965881, -0.014389918185770512, -0.9289671182632446, 0.18315622210502625, 0.04787028208374977, 0.21761220693588257, -0.48273998498916626, -0.28745636343955994, -0.17222079634666443], [0.19195328652858734, -0.12731091678142548, 0.31193807721138, 0.3717203438282013, 0.26150184869766235, -0.6640856862068176, 0.33816584944725037, 0.32842716574668884, 1.5779180526733398, -2.5207154750823975, -1.0138381719589233, 0.11946793645620346, -0.7335731983184814, -0.41209402680397034, 0.6928228735923767, -0.08432594686746597, 0.007587117608636618, -0.6585308313369751, -0.04288097098469734, -0.24683527648448944, -0.015957143157720566, -1.8114173412322998, 1.5503731966018677, -0.0409427210688591, -0.12498350441455841, 0.486849844455719, 0.04694000631570816, 0.297832190990448, 0.27554750442504883, -0.6257476210594177, 0.3656212091445923, 0.24072806537151337, 2.0980074405670166, 0.3817921280860901, 1.2263906002044678, 0.3756491541862488, -0.6384141445159912, -0.25970175862312317, -0.19896629452705383, -1.2821698188781738, 1.3953726291656494, 1.550826907157898, 1.7302216291427612, 0.25558334589004517, 0.08325399458408356, 0.03274766728281975, 2.555913209915161, 1.3283472061157227, -0.12731924653053284, -0.5629551410675049, -0.7774369716644287, 0.1474352329969406, 0.010716985911130905, 0.5971270799636841, -1.5710338354110718, -0.11494588106870651, -0.5193089842796326, 0.27202796936035156, -0.5046075582504272, -0.6179279088973999, 2.371182680130005, -0.1665000021457672, -0.7484232783317566, -0.9464444518089294, 0.8145497441291809, 0.786825954914093, 0.10026897490024567, -0.20170363783836365, -0.0078215841203928, -0.012647401541471481, 1.0663584470748901, -0.7806748747825623, -0.06984566897153854, -0.05806085467338562, 0.6012977957725525, -0.5128243565559387, -0.05165129899978638, -0.2231639176607132, -1.278671145439148, -0.4991019070148468, 0.026538478210568428, -0.48119643330574036, -0.888235330581665, 0.060149479657411575, 1.3147321939468384, -2.8124914169311523, -0.8555974960327148, 0.31486329436302185, 0.26136279106140137, 0.5250167846679688, -0.7835813760757446, -0.10355066508054733, -2.15557599067688, 0.32858043909072876, 1.3081960678100586, -0.07182618975639343, 0.21627257764339447, -0.747600793838501, -0.3525172173976898, -0.9950029850006104, 1.5088828802108765, 1.1991239786148071, 0.9667938947677612, 0.25916188955307007, 1.5490515232086182, 1.3301297426223755, -2.732191324234009, -1.8669074773788452, 0.4161439538002014, -0.4376632869243622, -0.34146183729171753, 1.938737154006958, 0.11511651426553726, 0.7478460073471069, -0.003945620264858007, 1.1848787069320679, -0.2848532497882843, -1.9516160488128662, -0.04702901095151901, -0.7662709355354309, -0.2434871792793274, -0.2707942724227905, -0.06786889582872391, -0.23516589403152466, -0.02394801564514637, -0.8872340321540833, 0.24543099105358124, 0.19099780917167664, -0.21056632697582245, -0.5359715223312378, -0.20504115521907806, 0.013006520457565784, -0.4697244167327881, 1.1972017288208008, -0.9135251641273499, 0.00809334684163332, -1.19398033618927, -0.7208291292190552, -2.5171492099761963, 0.6121723055839539, -0.08046071231365204, 0.24419815838336945, -0.05761143937706947, 0.5581009387969971, -2.0065152645111084, 0.10599116235971451, 0.8874877691268921, -0.029915882274508476, -0.6518204808235168, 1.2042385339736938, -0.17476941645145416, -0.5926520228385925, -3.614318370819092, 0.2146339863538742, -0.029034828767180443, 0.03443947061896324, 0.6850240230560303, 2.600874662399292, -0.36251336336135864, 0.586875855922699, 0.01015658862888813, -0.3559586703777313, -0.2505359947681427, -0.0930686667561531, -3.0370912551879883, 0.6297970414161682, -0.06257718056440353, 0.5359897613525391, -0.074025459587574, 1.9181841611862183, -1.1201931238174438, -0.045515358448028564, -2.305979013442993, -0.01591797173023224, 0.4141371250152588, -0.09700765460729599, -0.020962176844477654, 0.9082559943199158, -0.28369173407554626, -0.06748354434967041, 0.46943461894989014, 0.8092871308326721, -1.6644920110702515, -0.17555202543735504, -0.7402835488319397, -0.17428043484687805, 0.30628257989883423, -0.4989023506641388, -1.1399376392364502, -0.11250357329845428, -1.1038185358047485, 0.8818944692611694, -0.2691677510738373, -0.09699518978595734, -0.1583130806684494, 0.020679933950304985, 0.49517834186553955, 2.2489120960235596, 0.02364908903837204, 2.2676968574523926, 0.042572278529405594, -0.5567725896835327, -0.46687087416648865, -0.7036097645759583, -0.0328017883002758, 0.18532276153564453, -0.1264110952615738, -0.062349747866392136, -0.0545196495950222, -0.27031296491622925, -0.15299542248249054, -0.5549359917640686, 0.004898563027381897, -0.3518630564212799, -0.07435080409049988, 0.8588865399360657, 0.3141663074493408, 0.3028114438056946, 0.5841724276542664, -0.47073623538017273, 0.29391080141067505, 0.10285446792840958, 0.5543926954269409, 1.146507978439331, -0.16995912790298462, -0.12473319470882416, 0.03999868407845497, -0.13929806649684906, -0.21954332292079926, 0.17835725843906403, -0.055409207940101624, 0.18010887503623962, 0.48153015971183777, 0.2782263159751892, -0.21504740417003632, -0.29369106888771057, -0.3975793123245239, 0.15770846605300903, -0.03913905471563339, 1.2096580266952515, -0.5387786626815796, -0.176337331533432, 0.3986304700374603, 0.3012179732322693, 0.4071650207042694, -0.568719744682312, -0.009865041822195053, -0.6252345442771912, 0.5264045000076294, 0.37935855984687805, -0.13472530245780945, -0.06699645519256592, -0.03401601314544678, -0.320711225271225, -0.012650519609451294, -1.3243882656097412, 1.0759906768798828, -0.31769484281539917, -0.10623809695243835, -1.7399020195007324, 0.14609208703041077, 0.4325937330722809, 0.05241835489869118, 1.6604965925216675, 0.20280805230140686, -0.07237827032804489, 0.04162241145968437, -0.030501920729875565, 0.10376083850860596, 0.00030518349376507103, -0.13938526809215546, -0.04595368728041649, -0.20589861273765564, -0.1396341174840927, -0.511857807636261, -0.2068427950143814, -0.40370383858680725, -0.01436903141438961, -0.7382591366767883, -1.1347979307174683, -0.15386903285980225, -0.0749259814620018, -2.4037845134735107, -0.17231221497058868, 0.7612462639808655, 1.0379695892333984, 1.580919623374939, -1.8106762170791626, 0.1720111072063446, 0.6468328237533569, 0.38086462020874023, 0.2566179931163788, -1.5240403413772583, -0.4048018157482147, 0.7617111206054688, -0.4552832245826721, -0.8198462128639221, 0.181748628616333, -0.3298654854297638, -0.9705031514167786, 0.9084960222244263, 1.119706153869629, -1.1741907596588135, 1.0113310813903809, 1.9858134984970093, 1.0559287071228027, 0.17172293365001678, 0.03976176306605339, -1.6780004501342773, -0.17029878497123718, -0.06807610392570496, 0.21090267598628998, -0.1291930377483368, -0.6364946961402893, -0.9463052153587341, -1.0584460496902466, -0.44709309935569763, 0.5150906443595886, -0.2220861315727234, -2.172076940536499, -0.3271518349647522, 1.134330153465271, -0.629250168800354, 1.3281868696212769, -2.452272653579712, 0.9056391716003418, 0.25002574920654297, -0.9982958436012268, 1.5143963098526, -0.05884561687707901, -0.02684256061911583, 0.9453781247138977, 0.9440557956695557, -1.703975796699524, 0.5272783637046814, 1.2870460748672485, 2.6924188137054443, 0.6263064742088318, 0.003234597621485591, 0.14121046662330627, -0.24881191551685333, 0.34273508191108704, -0.07551466673612595, -0.30611252784729004, -1.2657867670059204, -1.2734004259109497, 0.22329095005989075, -0.11399068683385849, 4.0680928230285645, -0.7238321900367737, 0.8485116958618164, 1.447593331336975, -0.029695341363549232, 0.7227737903594971, 1.980262041091919, -0.18980620801448822, 2.6847965717315674, 0.5672494173049927, -0.0033487204927951097, -0.26268965005874634, 0.26412439346313477, -0.16570332646369934, -0.10196491330862045, 0.46294403076171875, -0.9664984941482544, 0.27561450004577637, 0.9247642755508423, 0.10158251225948334, 0.5733036398887634, 0.7682278156280518, -0.8688259720802307, 0.1715209037065506, 2.124922752380371, 1.4693185091018677, 0.7131081819534302, 0.7957479357719421, 1.4492809772491455, -0.18768027424812317, -0.30523771047592163, 4.291658401489258, -1.5974708795547485, -2.339658498764038, -0.014365318231284618, -0.03006841242313385, 2.3938050270080566, -1.1433855295181274, 0.1826051026582718, -0.05616697296500206, 5.616024971008301, -0.3343981206417084, -0.3472118377685547, 0.9471062421798706, 1.6366716623306274, -0.2840384244918823, -0.0844554677605629, 0.04889639839529991, 0.0023688676301389933, 0.6549078822135925, -0.35871270298957825, 0.06918693333864212, 0.14776752889156342, 0.5000291466712952, -0.007193379104137421, -1.3727108240127563, 1.118470311164856, -0.2716526687145233, -0.020200090482831, 0.8046027421951294, 0.4919997453689575, 1.3232598304748535, -1.1965090036392212, -0.023732313886284828, -0.31456658244132996, 1.3030036687850952, 0.9289549589157104, 0.3453174829483032, -0.17544348537921906, -0.8440272808074951, -0.15418873727321625, -2.348775863647461, 0.8985612988471985, -0.6456595659255981, -0.49593088030815125, 0.7108177542686462, -0.4985218644142151, -1.7835593223571777, -0.428789347410202, -0.03333045542240143, -0.08529996126890182, 1.2007578611373901, -0.16418355703353882, -2.98085618019104, -1.1489543914794922, -1.295840859413147, -0.03613697737455368, 0.9769281148910522, 1.2931798696517944, 0.4771561920642853, -0.5800959467887878, -0.3555091917514801, -0.19455963373184204, -1.276870608329773, -0.15383267402648926, 0.47793421149253845, -1.9598053693771362, -0.8122725486755371, 0.6170746088027954, 2.4564645290374756, -0.02925337664783001, 0.24695010483264923, 0.7075769305229187, 0.25126194953918457, -0.06260225176811218, 0.41157588362693787, -0.01601170189678669, -0.624662458896637, 0.2255626916885376, -0.1400783210992813, -0.050908125936985016, -1.8497710227966309, 0.11868049204349518, 0.5413448214530945, 0.28140509128570557, 1.1026428937911987, -0.867616593837738, 0.30368053913116455, -1.352134346961975, -2.4491264820098877, -0.343240886926651, 0.6880000233650208, 0.04542951285839081, 1.1837800741195679, 0.1639900952577591, -0.6354722380638123, -0.9770081639289856, -4.577176094055176, 0.14863260090351105, -0.9255678057670593, -1.5677194595336914, -0.07445799559354782, 1.0182514190673828, -0.9238602519035339, 0.08874236792325974, -1.1298290491104126, -0.033987607806921005, -0.25176697969436646, -0.029443666338920593, 0.05589187517762184, -0.9801771640777588, 0.008356444537639618, -0.41423681378364563, 1.0546725988388062, 0.24282990396022797, -0.25043898820877075, 0.4040008783340454, -0.9503652453422546, -0.07893437147140503, -0.581696629524231, 1.52595055103302, 0.3905769884586334, -0.12341547757387161, 0.12364575266838074, 0.5148614048957825, 0.07729768753051758, 0.002610341412946582, -0.5989928245544434, -0.018286606296896935, 0.025921892374753952, -5.263041019439697, 0.3748528063297272, -0.10141003876924515, -0.25121477246284485, 1.3778469562530518, -0.2756793797016144, -1.8027622699737549, -2.6278574466705322, 2.893289566040039, -0.01847180165350437, 1.4786866903305054, 0.08984154462814331, 0.35496267676353455, -1.700806736946106, -0.42139309644699097, -0.031050359830260277, -1.4954869747161865, -0.024438422173261642, -0.023185310885310173, -0.40584200620651245, -0.25472208857536316, -0.7785658240318298, -0.287055641412735, 0.9618030786514282, -1.366310477256775, -0.127590149641037, -0.21691198647022247, 0.6000795960426331, 0.813524067401886, -0.3589038848876953, -0.34831658005714417, -0.9540287852287292, -3.0303328037261963, 0.12862230837345123, 0.7853280305862427, -0.11969255656003952, -0.6238936185836792, 0.514218270778656, -0.2485654205083847, 0.5503836870193481, -1.1082649230957031, 0.11130606383085251, -0.38772857189178467, -0.6578128933906555, 0.008180373348295689, -0.6774107217788696, 1.3410122394561768, 0.027638377621769905, -0.021406009793281555, 0.040689367800951004, 1.4007048606872559, -0.2180492877960205, 0.31897425651550293, -0.31130337715148926, -0.20183232426643372, -0.5312326550483704, 2.424478054046631, 0.10393355041742325, 0.016652477905154228, 0.49094539880752563, 0.48374098539352417, 0.3701067268848419, 2.4988396167755127, 1.0430161952972412, -0.019108332693576813, -0.34123027324676514, -0.42231273651123047, 1.2601892948150635, 0.40271854400634766, -0.18491818010807037, 0.19572459161281586, 0.05275837704539299, 0.7706618309020996, 0.31998735666275024, -0.07940991967916489, -0.10551659017801285, -1.2538423538208008, 0.21857482194900513, -0.645549476146698, 0.2766076624393463, -0.21697457134723663, -0.46538877487182617, 0.2026171088218689, 0.3854697048664093, -0.07440580427646637, -0.8331852555274963, 0.03562217205762863, 0.8992514610290527, 0.3304106891155243, 0.9568215012550354, 2.667041301727295, -0.6698852777481079, 0.8390899300575256, 1.566465973854065, -0.3738178610801697, 1.112309455871582, 0.0691722109913826, 0.816452145576477, 0.1253960281610489, 0.1520068347454071, -0.054933812469244, 0.4934004247188568, -0.7654845714569092, 0.5005580186843872, 0.28719040751457214, -0.3928195834159851, -1.986115574836731, -1.2156834602355957, -0.07785975188016891, 0.045950714498758316, -2.1528561115264893, 0.5505877733230591, -0.3380542993545532, -0.015223112888634205, -0.14253300428390503, 0.3685590326786041, -0.10716624557971954, -0.29114094376564026, 0.3328644931316376, 0.047325730323791504, 0.4668433964252472, 0.9275968670845032, -0.27261000871658325, 0.38998696208000183, -0.20848242938518524, 0.22198067605495453, 1.4922778606414795, 0.1953664869070053, -0.15518370270729065, 1.591526985168457, 0.05782265588641167, -0.5374359488487244, -1.086810827255249, 0.14872750639915466, 0.2245420664548874, -0.08454540371894836, -0.15363991260528564, 2.5676989555358887, -0.030176542699337006, -0.3950900137424469, -0.22719308733940125, 0.09740107506513596, 0.11934659630060196, -0.2979136109352112, -0.15717588365077972, -1.1823980808258057, 2.0911471843719482, -0.9127327799797058, 0.4881132245063782, -0.2064877152442932, -0.26728299260139465, -0.2069166749715805, -0.7295418381690979, 0.6937476992607117, 0.24772757291793823, -3.566420078277588, 0.39863088726997375, 0.12090805172920227, 0.6407260894775391, 0.7136794924736023, 1.7782355546951294, 0.5627752542495728, 1.7977163791656494, 0.4844089150428772, 0.26500585675239563, -1.2122472524642944, -0.17878900468349457, 0.7967957258224487, -0.5610390901565552, -0.051942020654678345, -0.16726383566856384, -0.1442115306854248, -0.05186435952782631, 0.6309607028961182, -0.038071926683187485, -0.03427419066429138, -1.8806531429290771, -0.3267541825771332, 2.355330467224121, -0.056917835026979446, 0.051091473549604416, -0.19843053817749023, -0.44586631655693054, 0.38682934641838074, -0.26131677627563477, 0.1591562032699585, -0.3399340808391571, 2.862459897994995, -0.26364243030548096, -1.238821268081665, 0.16474531590938568, 0.8401487469673157, -0.6992954015731812, 0.42076388001441956, -0.6556862592697144, -1.8964040279388428, 0.26832637190818787, -0.026650749146938324, -0.5461323261260986, -1.4598054885864258, -0.3014260530471802, -0.16336312890052795, -0.008826149627566338, -0.1242617666721344, -0.4783209264278412, -0.36858052015304565, 1.5502158403396606, -0.7870030999183655, 0.09408080577850342, -0.8935271501541138, 0.38470765948295593, 1.0914164781570435, -2.685335874557495, -4.858719348907471, -0.18346017599105835, -0.08369902521371841, -0.3881309926509857, 0.16358426213264465, -0.08172132819890976, 0.08069898188114166, 0.3920798897743225, 0.08813264220952988, -1.0796655416488647, -0.42338868975639343, -1.461663007736206, 1.4219707250595093, 3.3287010192871094, -0.06437212228775024, 0.25741058588027954, -0.103214330971241, 0.006742026191204786, 0.3724074065685272, -0.10832127183675766, 0.1175272986292839, -0.03995862603187561, 0.0953792855143547, 4.244104862213135, -0.19039946794509888, 0.6644278168678284, 0.7168371081352234, 0.20287111401557922, -0.09864357113838196, 0.04420565813779831, -1.8452190160751343, 0.04814986139535904, 0.07170307636260986, -0.1272587925195694, -0.3449857234954834, -1.2753000259399414, 1.6104081869125366, -0.8816667199134827, -0.3232087194919586, -0.058195773512125015, 1.343629002571106, -0.07244627177715302, -0.11438608914613724, 0.6492848992347717, -0.9526084661483765, -0.8138599991798401, 0.32228243350982666], [0.2417512685060501, -3.622163772583008, -2.03359055519104, 0.33749935030937195, -1.3050246238708496, -0.8563710451126099, -0.018315311521291733, 0.0624961219727993, -0.6159388422966003, 0.056647758930921555, 0.45425620675086975, 0.14515630900859833, 0.5743046998977661, 0.12355370074510574, 1.7327865362167358, 0.09257569909095764, -0.08004911243915558, 0.14641186594963074, -4.072257995605469, 0.34276869893074036, -0.10470665991306305, 1.7580903768539429, -1.975812554359436, 0.06760980933904648, 0.03249577432870865, 0.46557748317718506, 0.02637101709842682, -0.13729272782802582, 0.027269257232546806, 0.46023058891296387, 0.1276821494102478, 1.5470340251922607, 1.5781227350234985, -3.670090675354004, 0.4350656270980835, 1.167665958404541, -1.9321423768997192, -0.42592236399650574, 1.3854464292526245, -0.3315979540348053, 3.296563148498535, 1.8714485168457031, 1.746378779411316, -1.9548919200897217, 0.7406682372093201, -0.8636261820793152, -2.051745653152466, 0.5394034385681152, 0.26310887932777405, 1.3055510520935059, -1.0170648097991943, 0.20992396771907806, -0.014087417162954807, -0.4762284755706787, -0.028956426307559013, -0.03212595358490944, 0.42166244983673096, 1.7931246757507324, 0.03986809775233269, 0.01569037325680256, -1.238349437713623, -0.10080866515636444, -0.6636547446250916, 1.2205002307891846, 0.07840991765260696, 1.443892240524292, -0.018290910869836807, 1.0173914432525635, 0.06892034411430359, -0.055780358612537384, -0.886458158493042, -1.3666160106658936, 0.09908222407102585, 0.06309852004051208, -1.6481600999832153, -1.228572964668274, 0.1161508709192276, -0.7396618127822876, 0.790108323097229, 1.0453251600265503, 0.039547331631183624, 0.27006739377975464, -0.3316332697868347, 1.3406310081481934, -0.5181352496147156, 1.0989736318588257, -0.30216363072395325, -0.09454825520515442, -0.4515424966812134, 0.3936059772968292, 2.2198851108551025, -0.07190568745136261, -2.3966519832611084, 0.2450351119041443, -0.5556613206863403, 0.0728551372885704, -0.9832397103309631, 0.5306737422943115, -1.8035039901733398, 0.35609039664268494, -1.0455594062805176, 0.7433828711509705, 0.3709622621536255, -0.42231687903404236, 2.22078800201416, 0.35645800828933716, -4.475497722625732, -1.5763111114501953, -0.2946091592311859, 0.08791428059339523, 0.7822688221931458, -2.6901328563690186, 0.057939160615205765, -1.8974493741989136, 0.019156984984874725, 1.8911330699920654, 0.19597579538822174, -1.0068429708480835, 0.013738313689827919, 0.2965626120567322, -0.12296147644519806, 0.2570546865463257, 0.022960251197218895, 0.23180417716503143, 0.0034602652303874493, -1.5945203304290771, -0.2206442952156067, -0.32064083218574524, -0.0627274364233017, -0.47453242540359497, 2.133497714996338, 0.007689662277698517, 1.7237814664840698, -0.004338339902460575, -0.7571274638175964, -0.012228792533278465, -0.9003223776817322, -0.20089107751846313, -0.8695808053016663, 0.4888657331466675, -1.850314974784851, 0.7658401131629944, -1.0499743223190308, -0.6546924710273743, -2.604004144668579, -0.1911294311285019, -1.675468921661377, 0.8576511740684509, -0.9134938716888428, -0.1598513275384903, -0.11110397428274155, -0.5448225140571594, 0.7824127674102783, -0.675536572933197, 0.08755455166101456, -0.02959313988685608, -0.18161936104297638, -2.8754396438598633, 0.45237717032432556, -0.4794485867023468, 0.008328106254339218, -0.6887463927268982, -0.7887398600578308, 0.12874183058738708, -0.03255658596754074, -0.6530576348304749, 0.010619117878377438, 0.7583025097846985, -0.0939037874341011, -4.093494415283203, -0.4153313934803009, 0.02593303844332695, -1.1883436441421509, -0.054224856197834015, 0.12244479358196259, 0.2287234365940094, 0.0033921964932233095, 0.4192509055137634, 0.5347243547439575, 0.16860531270503998, 0.03965623676776886, -0.041510697454214096, -1.9890879392623901, -1.9927983283996582, 0.11428351700305939, 0.9984135031700134, -0.7281254529953003, 1.734987497329712, 0.3339565396308899, -0.11153849214315414, -0.3714889883995056, 0.2578284740447998, 0.991296648979187, -0.6890427470207214, -5.177249908447266, -1.4482744932174683, 0.1192062720656395, -2.832507610321045, 0.07976438105106354, -0.4404170513153076, 0.13049139082431793, 0.02812870778143406, 0.026149330660700798, -0.37436601519584656, -0.004838292952626944, 0.11227580159902573, -0.19078046083450317, 0.06304087489843369, 0.007600485812872648, -0.5329027771949768, -0.05600787699222565, -1.9639652967453003, -0.12408378720283508, -0.11225036531686783, 0.7241995334625244, -1.1412819623947144, 0.9187666177749634, 0.1299157738685608, 0.03889884054660797, 0.496408611536026, 0.35752415657043457, 0.10453786700963974, -0.7129080295562744, 0.9929277896881104, 0.13817042112350464, 0.1168692335486412, 0.032964687794446945, -0.538423478603363, -0.0036226101219654083, -0.006705491337925196, -0.09313776344060898, -0.01913461647927761, -2.3519368171691895, 0.21115899085998535, -0.16938354074954987, 0.17929330468177795, -1.8081117868423462, -0.273802250623703, -0.06687532365322113, -2.9955341815948486, -0.41309884190559387, -2.472102403640747, -0.44450142979621887, -0.04237309843301773, 0.6470283269882202, -0.0776398628950119, 0.023584142327308655, 0.26821133494377136, -0.5199630856513977, 0.48884230852127075, 0.10869577527046204, 0.22543665766716003, 0.004510434344410896, -0.8992701172828674, 0.07944221794605255, -0.5503661036491394, -0.10955960303544998, 0.3167562186717987, -0.17098921537399292, 0.19250887632369995, 0.2725923955440521, 0.6175165772438049, -0.05031706765294075, -0.9242493510246277, -0.09355387091636658, 0.04663620889186859, 1.4892414808273315, 0.04208652302622795, -0.39866530895233154, 0.08564478158950806, 0.8142268061637878, 0.6755083203315735, 0.17948415875434875, -0.03587968647480011, -0.3776605427265167, -2.608165740966797, 0.5691654682159424, 0.008152075111865997, 0.17377209663391113, 0.7852122187614441, 0.08844450116157532, 0.6452596783638, 0.62070631980896, -0.266262412071228, -0.7294397950172424, 1.6806408166885376, 1.2761436700820923, 0.09822238981723785, 0.09948106110095978, -1.038149356842041, -0.03939869999885559, 2.8046181201934814, 1.4942001104354858, 0.20233255624771118, -0.15255002677440643, 0.3422817289829254, 2.7570223808288574, -0.9069200158119202, -0.41593557596206665, 0.4073733389377594, 0.3996078670024872, -0.482666552066803, 0.9336405396461487, -0.28337544202804565, -0.14679434895515442, -0.26813068985939026, 0.18305611610412598, -0.08258804678916931, 0.604311466217041, 0.17824622988700867, -0.05122050270438194, -0.00871445145457983, -0.4736879765987396, -0.06877226382493973, -0.7066256999969482, 0.07610512524843216, 0.37262895703315735, 1.608980417251587, -0.41467341780662537, 1.904494047164917, 0.5513739585876465, 0.2876105308532715, -0.5111557245254517, -0.1309632807970047, 2.4679813385009766, 0.12865285575389862, 2.703129768371582, 2.8019511699676514, -0.22686626017093658, -0.06563032418489456, 0.7486703395843506, -0.505416750907898, -0.38049936294555664, 1.7148187160491943, -2.344841718673706, -1.0798909664154053, -1.056005835533142, 0.3663262724876404, -0.0026406084652990103, -0.3869485855102539, 0.9420909881591797, 0.7148702144622803, -0.5485152006149292, 0.5712839365005493, 0.13793376088142395, 0.527900755405426, -0.09560328722000122, -0.8166829943656921, 6.628748893737793, 0.5504318475723267, 0.00013638229575008154, 3.6362195014953613, 0.01879848539829254, 0.5078718066215515, -1.5312063694000244, 0.7193570137023926, 0.965206503868103, -0.5672534704208374, -0.7643910646438599, 0.7076877355575562, -0.35364046692848206, -1.2038624286651611, 0.3951413631439209, -0.1663273572921753, -1.6766057014465332, -0.01719110831618309, -1.4857585430145264, -0.45554420351982117, -0.1506814956665039, 0.41556376218795776, 0.813448965549469, -0.3072128891944885, 1.9151841402053833, 2.317164182662964, -0.7987369894981384, 1.6748385429382324, 0.04004877805709839, 0.4394724667072296, 0.15300506353378296, 2.329969882965088, 0.12459699809551239, 1.2297239303588867, 0.026308812201023102, 0.005987623706459999, -0.3150297999382019, 0.37855064868927, -0.31563323736190796, 0.08930185437202454, -0.6067672967910767, 0.15642185509204865, -0.024917198345065117, 0.13637271523475647, 0.782530665397644, 0.014097352512180805, 0.022217273712158203, -0.01645607501268387, 0.36933189630508423, -0.48848965764045715, -0.11568587273359299, 0.24061308801174164, -0.15779225528240204, 0.2158205509185791, -0.04560346156358719, 0.23251931369304657, 0.24236178398132324, -0.25663620233535767, -0.00830856990069151, -0.804215669631958, 0.38057222962379456, -1.4909273386001587, -0.7745880484580994, 0.08105874061584473, 0.7374598979949951, 1.8802440166473389, -1.2428960800170898, -0.3245678246021271, -0.7238087058067322, 0.8179091811180115, -0.35871753096580505, -0.1844242513179779, -0.3285995423793793, -0.5746227502822876, 0.008543498814105988, -0.5847710967063904, 1.5998073816299438, -1.1894092559814453, 0.4461037516593933, -0.1869249939918518, 0.10391673445701599, -1.1020116806030273, 0.019985295832157135, -0.37647461891174316, -0.4744711220264435, 2.7044386863708496, 0.1128074899315834, 1.1526124477386475, 0.7557000517845154, -0.05737524479627609, -0.46171629428863525, -0.20160171389579773, -0.19710586965084076, 0.5609095096588135, 0.73134845495224, 0.026374083012342453, -0.013974224217236042, -1.0849390029907227, 0.12073346972465515, 1.9217736721038818, 0.021786129102110863, -0.07639499008655548, -0.006688954774290323, 0.34117674827575684, 0.014676968567073345, -0.80488121509552, 0.009405954740941525, 0.2657959759235382, -0.6696876287460327, 0.04062609001994133, -0.9556268453598022, 0.8567548394203186, 0.707382321357727, 0.051772456616163254, 2.51220965385437, 0.7179561853408813, 0.03016628883779049, -0.4522816240787506, 0.5135699510574341, 0.2207014262676239, -1.4108881950378418, 0.7161770462989807, -0.6539351940155029, 3.1368699073791504, -0.7063327431678772, 0.2339804321527481, 0.447763592004776, -1.8623236417770386, -0.2730298936367035, 1.2054704427719116, -3.316274881362915, 0.600813627243042, 1.0843271017074585, -1.6004562377929688, -0.06948395073413849, 0.11870454251766205, 0.004903694614768028, -0.07575715333223343, 0.1207144558429718, 0.7088419198989868, 0.7394487261772156, -0.04415465146303177, 0.22134189307689667, 0.7598710060119629, -0.10046399384737015, -0.3718634843826294, 0.45678895711898804, -0.9263330698013306, -0.11060163378715515, -1.7865018844604492, -0.799934446811676, 1.3186036348342896, -0.22023096680641174, -0.08247379213571548, 0.319269061088562, 0.07416123151779175, 0.10821907967329025, 0.6907711029052734, -0.1621599793434143, 0.02814432419836521, -0.39670681953430176, 0.21249797940254211, -0.4053862392902374, 0.3617754876613617, -1.609472632408142, 1.1097856760025024, 2.755786657333374, -1.1290192604064941, -3.144252061843872, -0.02044854685664177, -0.5907458066940308, 0.4125649631023407, -0.16297684609889984, -0.7252978682518005, -0.5050876140594482, 0.5370755791664124, 1.638485312461853, -0.01849706843495369, -0.06446965783834457, 0.9445149898529053, -0.3202367424964905, -2.9788589477539062, -0.3930914103984833, 0.10492848604917526, 1.4673880338668823, 0.34852635860443115, 0.7727705240249634, 0.24486570060253143, -0.47984927892684937, 0.24792294204235077, 0.6212632060050964, 0.31033197045326233, 1.837603211402893, -0.01870989240705967, 0.2830093502998352, -0.09535866230726242, -0.3367927074432373, 0.32008877396583557, -0.8334124088287354, 1.2080200910568237, -0.5897250175476074, 0.2090843766927719, -0.231321319937706, 0.9901265501976013, 0.07354205846786499, 1.3636176586151123, 1.8449273109436035, 0.02613670937716961, 0.024737680330872536, 0.13356703519821167, -1.7633017301559448, -2.977397918701172, 1.227649450302124, -1.3665261268615723, -0.5017789602279663, 0.4161319434642792, 2.461646318435669, -0.2773887515068054, 0.012238755822181702, -0.11405792087316513, 0.7260972261428833, -1.4129881858825684, -0.7608000636100769, -0.3861299455165863, -0.0010605257702991366, 0.8954631686210632, -0.005507905501872301, 0.17520281672477722, 1.1499284505844116, -0.20507368445396423, 0.23507963120937347, 0.0878986269235611, 0.4229269027709961, -0.4330863952636719, 1.5827338695526123, 0.07688771188259125, 0.9523546695709229, 0.4651940166950226, -0.19664016366004944, -0.1657792627811432, 1.711564064025879, -0.2074296921491623, 0.13832630217075348, 0.8342200517654419, 0.07331792265176773, 0.08580654114484787, 0.8883590698242188, 0.3616003096103668, 0.3715471029281616, 1.1748884916305542, -0.4497363269329071, 0.9449559450149536, -0.35043004155158997, -1.6674288511276245, -0.7755563259124756, 0.1855408102273941, -1.4281158447265625, -0.3284291625022888, 0.8224515318870544, -0.09223154187202454, 0.43408021330833435, -0.6232437491416931, 3.2019219398498535, 0.028626898303627968, -1.5390485525131226, 0.4042695462703705, 2.5903592109680176, 0.9983945488929749, -0.010017241351306438, 0.011903016828000546, -1.406559705734253, -0.06853649020195007, -0.06808619946241379, 0.01862393692135811, -0.40596678853034973, -1.9981344938278198, -0.07511568814516068, 0.5055168271064758, -0.7153696417808533, 0.871540904045105, 0.012418569065630436, 0.6603838205337524, 0.4536152780056, -0.3083719313144684, -0.20679157972335815, 0.0713852196931839, -0.43190374970436096, -0.3715474009513855, -0.6178684830665588, -6.0768723487854, 0.0992029681801796, -1.0753848552703857, 2.1644704341888428, 0.3006454408168793, -0.05727824196219444, -0.025348279625177383, 0.09002760052680969, 3.4559736251831055, 0.5148282051086426, -0.1153949424624443, -0.7227006554603577, 0.018157413229346275, 0.4297625422477722, 0.002513590734452009, -0.03694552928209305, -0.33975160121917725, -0.374118447303772, -0.052546970546245575, -0.3135663866996765, -0.01314614899456501, -0.37262460589408875, 0.0636083334684372, 0.9320922493934631, 0.17374058067798615, -0.9031361937522888, -0.5768933892250061, -0.9249295592308044, -1.2233859300613403, -0.055744316428899765, 0.29551035165786743, -0.4204675257205963, -0.7530514001846313, -0.04723573848605156, 2.3522164821624756, 1.110012173652649, 1.76840341091156, -0.13264212012290955, -0.32275697588920593, 0.4722451865673065, 0.0999220758676529, 1.0018162727355957, -0.050565678626298904, -0.19410564005374908, -1.2221872806549072, 0.05568505823612213, -0.952390193939209, 0.47013306617736816, 0.04833563044667244, 0.5416189432144165, 0.0004947806592099369, -0.2877742052078247, -0.590894341468811, 0.3353360593318939, -0.5363191366195679, 0.02363804541528225, -0.9136196374893188, -0.4918954372406006, 1.9112305641174316, -0.9974086284637451, -0.6012111306190491, -0.2736232578754425, -0.6169341206550598, -0.5515981316566467, -0.016199178993701935, 0.6746234893798828, 4.728677272796631, 0.08470547199249268, 0.025203462690114975, 0.2991921901702881, 0.8238899111747742, -0.16512475907802582, 0.5584971308708191, -0.029773054644465446, 0.19076235592365265, 0.11456268280744553, 0.41260388493537903, 1.5711971521377563, -1.6556342840194702, 0.16516873240470886, -1.062615990638733, 1.0788260698318481, -0.32417383790016174, 0.5171629786491394, 0.22543905675411224, -0.14821811020374298, 0.26383864879608154, -0.22889673709869385, 0.08286534994840622, 0.0316394567489624, 0.1711231768131256, -0.427051305770874, -0.28299638628959656, -0.1639685332775116, -0.5705617666244507, 0.6849855184555054, -1.3346810340881348, 1.3038558959960938, -1.5946741104125977, -0.29655447602272034, -0.11131610721349716, 0.02926449477672577, 1.0075711011886597, 0.08977308869361877, -0.11542828381061554, -0.6369134783744812, -0.004468160215765238, 3.0898053646087646, 0.7863104939460754, -0.2510722875595093, -0.07511404901742935, -0.25059637427330017, 0.8711885809898376, 0.01313856989145279, 2.8939950466156006, -0.4995156526565552, -2.030755043029785, 0.029766632243990898, -1.3586742877960205, 1.0999845266342163, -0.44467973709106445, 0.8316469192504883, 1.1000405550003052, -0.4228494167327881, -1.4490352869033813, -0.03772248327732086, -0.3990139067173004, -0.49127042293548584, -1.7139043807983398, -0.6813613176345825, 0.402934193611145], [-0.08609382808208466, 1.3587852716445923, 1.5169520378112793, -0.5603590607643127, 3.21225905418396, -0.17169860005378723, 0.3332986831665039, 0.11203062534332275, -2.9521267414093018, 1.7282285690307617, -0.4027514159679413, 0.3427990674972534, -0.1839747130870819, -0.0739666223526001, 2.1067798137664795, 0.37072691321372986, 0.7576869130134583, -0.7617164850234985, -2.0528934001922607, -0.036249831318855286, 0.05462970212101936, -2.937507390975952, 2.0750732421875, -0.04958232119679451, 0.08813756704330444, -1.12626314163208, -0.13106979429721832, 0.11116980016231537, 0.7610030174255371, -0.6179121136665344, -0.08390743285417557, 0.27189013361930847, -0.5045290589332581, -2.855999708175659, -0.014725729823112488, -1.2611255645751953, 1.6247800588607788, -0.0949845239520073, 0.9034922122955322, 0.2502826452255249, -0.2597380578517914, 0.540136992931366, -0.11372832208871841, 0.4712444841861725, -1.392787218093872, 0.9815857410430908, 1.4504982233047485, -0.4932277798652649, -0.5305109620094299, -1.2599650621414185, -1.055722951889038, -0.19280824065208435, -0.026418210938572884, -0.1857418566942215, -1.747672438621521, 0.010126990266144276, -0.5642845630645752, -0.5950550436973572, 0.3530029356479645, 0.2763102948665619, 1.270949125289917, 0.14195282757282257, -0.008297801949083805, 2.3392810821533203, 1.2339246273040771, 0.06937837600708008, -0.014344785362482071, 0.15877901017665863, -0.04608696326613426, 0.11522897332906723, 0.9033475518226624, 1.4969080686569214, -0.023777814581990242, -0.0944589152932167, 1.8943638801574707, -0.04602467641234398, 0.5097691416740417, -0.37989577651023865, 0.3913266956806183, -1.7894102334976196, 0.0035545662976801395, 0.06583039462566376, -0.46222370862960815, -0.6021077036857605, 0.5347408652305603, -2.6116366386413574, 0.481810599565506, -0.008454803377389908, -0.7532939910888672, 0.9485465884208679, 1.115778923034668, -0.38762325048446655, 2.709826707839966, -0.04258893430233002, -0.31356683373451233, -0.10943448543548584, -0.846705973148346, -3.7784156799316406, -1.4776121377944946, 0.017673183232545853, -2.2489352226257324, 0.009931281208992004, 0.6488824486732483, -0.008508582599461079, 0.19281183183193207, -0.6201409697532654, 0.2839130461215973, 2.2048752307891846, 0.6138433814048767, -0.4132554829120636, -0.6348140239715576, 2.2356245517730713, 0.1418086290359497, 2.101229429244995, -0.012306981720030308, 2.6015465259552, 0.100706085562706, -0.31931164860725403, 0.008920175023376942, -0.05822676047682762, -0.333497554063797, -0.06412789225578308, -0.15193624794483185, -0.41641151905059814, -0.017551008611917496, 1.0669056177139282, -0.25926369428634644, -0.516732394695282, 0.07271285355091095, 0.9295674562454224, 1.3992012739181519, -0.028604980558156967, -1.381523847579956, -0.5015602111816406, -0.1171255111694336, -0.23011891543865204, -0.5378060936927795, -0.21900694072246552, 0.8544566035270691, 0.25955960154533386, 2.491894006729126, 2.1174752712249756, 0.0758909210562706, 0.08379271626472473, 0.09993340075016022, -0.53562992811203, -0.847141683101654, 0.6031650900840759, -0.4272909462451935, 0.4910169541835785, -0.10841716825962067, 0.3631424903869629, -1.3586180210113525, 1.063712477684021, 0.14821095764636993, 0.07108219712972641, 0.20554247498512268, 0.5815367102622986, 0.28043070435523987, 0.3132460117340088, -0.25889888405799866, 0.40935516357421875, 0.04308803379535675, -0.33549660444259644, 2.3856289386749268, 0.8691632151603699, -0.00467107305303216, -0.02055094577372074, -0.014663463458418846, 0.23447299003601074, -0.4700634777545929, -0.034914400428533554, 0.07141677290201187, -0.10899559408426285, 0.07663378864526749, 0.09258612245321274, -0.003842979436740279, 0.5270094275474548, 0.9150369763374329, -0.1565403938293457, 0.8881563544273376, 0.3745822608470917, 0.7853373885154724, -4.172467231750488, -1.7523301839828491, 0.1986161470413208, 0.11848244816064835, -1.5517652034759521, 1.3213907480239868, -0.09736394882202148, 0.6017076373100281, 0.43208396434783936, 0.10673058032989502, 0.09995251148939133, -2.15669322013855, 1.4381307363510132, -0.052696917206048965, -1.1897052526474, -0.006156647577881813, -0.5452553033828735, -0.11386417597532272, 0.8066375851631165, 0.33555421233177185, -0.17291592061519623, 0.059462521225214005, -0.1301102340221405, 0.5757218599319458, 0.00033477373654022813, 0.11510206013917923, -0.25514844059944153, -0.7829852104187012, 0.6465029120445251, 0.07297682762145996, -0.16365011036396027, -0.42127418518066406, 0.8996750116348267, -0.7765029072761536, 0.40717822313308716, 0.559065043926239, -0.9949789643287659, -0.5352913737297058, 0.42814984917640686, 0.49690932035446167, 0.24480925500392914, -0.15865829586982727, 0.013901102356612682, 1.9939757585525513, 0.2678412199020386, -0.3906066417694092, 0.44374993443489075, 0.2588929235935211, -0.15109141170978546, -0.7409358620643616, 0.5030109882354736, 0.3253343105316162, -0.3814217150211334, -1.2608234882354736, 0.19793257117271423, 0.005069058854132891, -4.014312744140625, 0.021323993802070618, -1.4564200639724731, 0.18507002294063568, 0.1696406602859497, -0.06918861716985703, 0.09956163913011551, 0.03723149001598358, -0.45599791407585144, -0.9357454776763916, -0.10223782062530518, -0.3382776379585266, -0.05136875435709953, 0.015332868322730064, -0.7323610186576843, 0.06757979840040207, 1.7038079500198364, -1.3305346965789795, -0.9569939970970154, 0.23122231662273407, 0.6135125756263733, -0.07080743461847305, 0.2009187936782837, 0.2013917714357376, 0.4373798966407776, -0.9906825423240662, 0.010853338055312634, -0.25192543864250183, 0.06168806552886963, 0.47477078437805176, -0.3254343569278717, 0.16166922450065613, 0.354211688041687, -0.6282973885536194, 0.09501317888498306, 0.5413340926170349, -0.9869949221611023, 0.6604359149932861, -0.04064922779798508, 0.12409448623657227, 0.13207493722438812, -0.017394741997122765, -0.5832262635231018, -3.0018138885498047, 0.41980618238449097, -3.4697465896606445, 0.07535819709300995, -0.9377131462097168, -0.6968871355056763, 0.19060248136520386, 0.06387273222208023, 0.0030549089424312115, 2.5688138008117676, -0.012174458242952824, -0.1353413611650467, 0.34363070130348206, -0.3788919150829315, 1.9243119955062866, -0.6640768647193909, 0.5810652375221252, 0.14559969305992126, 1.4203969240188599, 1.6818690299987793, -2.321516275405884, -0.8031874299049377, 0.7753925919532776, -0.5299130082130432, -0.15430662035942078, 0.09513237327337265, 2.6301188468933105, -0.24904243648052216, 0.1652669608592987, -0.12749750912189484, 0.7629977464675903, -0.8046966195106506, -1.2825151681900024, 1.6225029230117798, -0.28671708703041077, -1.3580704927444458, -1.2183197736740112, -0.8278563022613525, -0.24682092666625977, 0.32878291606903076, 0.5833776593208313, -0.03558606654405594, 2.6434595584869385, -0.22989816963672638, -1.2204176187515259, 2.02362060546875, 0.5654507279396057, 0.0958310067653656, -0.8583314418792725, 0.5396944880485535, 0.011907869018614292, 0.20287016034126282, 0.308328241109848, -0.1828310191631317, 2.9689574241638184, -0.7948436737060547, -0.040440261363983154, 0.23178823292255402, 0.1081056296825409, 0.5778464078903198, -0.11925988644361496, 0.369238942861557, 1.3453601598739624, -0.3276391327381134, -0.025249509140849113, -1.775597333908081, -0.26681655645370483, -0.3741665184497833, 1.0137485265731812, 1.7539496421813965, 0.0068776835687458515, -0.3414646089076996, 1.4459422826766968, -0.9392254948616028, 0.6105853319168091, -0.23749415576457977, -0.8116723299026489, -0.10047449916601181, -0.40028539299964905, -0.5505291819572449, -0.3537648618221283, -0.757792055606842, 0.08094482868909836, -0.08300834894180298, 0.9617943167686462, -0.02534046396613121, 0.1535176783800125, -0.46410781145095825, -0.5935229063034058, 0.7646853923797607, -1.052066445350647, -4.500483512878418, -0.11911173164844513, -0.8276464343070984, -3.674076795578003, 0.06510484218597412, 0.09665563702583313, 1.0140783786773682, 0.939476490020752, 1.3943504095077515, -0.04947914928197861, 0.015353289432823658, -1.094495177268982, 1.3511155843734741, 0.43457385897636414, 0.36488375067710876, -2.0394222736358643, -0.24353082478046417, -0.02377505600452423, 0.5542523860931396, -1.8568063974380493, -0.08005960285663605, 0.061873968690633774, -0.11207209527492523, 0.14903010427951813, 0.05088308826088905, -0.2590367794036865, -0.03961564972996712, -0.11538836359977722, -0.13234186172485352, -0.06091341748833656, -0.3402816951274872, 0.04956623911857605, 0.1530676633119583, -0.014009340666234493, -1.9080363512039185, 0.8228143453598022, 1.8475427627563477, 0.38947170972824097, 0.015114309266209602, 0.37684738636016846, -0.5077930092811584, 0.12028200924396515, -0.16591502726078033, -0.6288925409317017, 0.5108091235160828, -1.5223525762557983, -1.1419658660888672, 1.9371459484100342, 0.7116819620132446, 0.10461798310279846, -0.3634414076805115, 1.303402304649353, 1.1372076272964478, -0.5652453303337097, 0.03563240170478821, -1.7146387100219727, -1.6436787843704224, -0.38022732734680176, -0.8299521803855896, -0.8953437805175781, 0.34538426995277405, -0.043276116251945496, 1.4816101789474487, 1.701027274131775, -2.0613791942596436, -0.681765079498291, 0.3214922845363617, 0.05707702785730362, 0.20332138240337372, -1.1089965105056763, 0.12068405002355576, 1.775258183479309, 0.029911357909440994, 0.5474854111671448, 0.919151782989502, -0.012385035865008831, 0.14893095195293427, 0.2985225319862366, -0.02840156853199005, -0.07215610891580582, 0.617247462272644, 0.017134414985775948, -0.945755660533905, -1.0127816200256348, -0.02179221622645855, -1.2777361869812012, 1.3642351627349854, -1.0785804986953735, -0.4451597332954407, 1.6551671028137207, -1.91599702835083, 0.19628289341926575, 0.013308046385645866, 0.37808048725128174, -1.1850841045379639, 0.5651378035545349, -0.16642631590366364, -0.4776506721973419, -3.5088939666748047, 0.7576276659965515, -0.39360374212265015, 0.4399178922176361, 0.027973981574177742, 0.36435213685035706, -0.02437443844974041, -0.5916949510574341, -0.09459620714187622, -0.3444795608520508, -1.5371254682540894, -0.09053359925746918, -0.6467308402061462, 0.003606294048950076, 0.5034285187721252, -1.1917662620544434, -0.15805889666080475, 1.1173523664474487, -0.18070262670516968, -0.3634793162345886, -0.6562634110450745, 0.36824092268943787, -0.24232512712478638, -0.163490891456604, 1.8028510808944702, -0.017796307802200317, -0.3853610157966614, 5.508562088012695, -0.5263662338256836, 0.8170523047447205, 0.02997138723731041, -0.13700132071971893, -0.37501564621925354, 0.08985263854265213, 0.614650547504425, -0.0027715195901691914, -0.04694248363375664, 3.1543500423431396, 0.1325731724500656, 0.49412545561790466, 0.001182780833914876, 0.5433785915374756, -0.12499668449163437, -0.022800780832767487, 2.1917202472686768, -2.0035085678100586, 0.04626470059156418, -1.7706527709960938, -1.0694297552108765, 1.282405972480774, 0.22758165001869202, 0.5008913278579712, -0.25612354278564453, 0.2712025046348572, -0.041792601346969604, -0.0349445603787899, 0.210431307554245, 0.24680973589420319, 1.3206347227096558, -0.33953577280044556, 0.9201825261116028, 0.4397839307785034, -0.08349351584911346, 0.8522692322731018, -0.22076472640037537, -0.7049033045768738, -0.8440567255020142, -0.024671299383044243, -0.32800421118736267, 1.2998756170272827, 0.00042601904715411365, 1.464483380317688, 0.014046328142285347, 0.08778481185436249, -0.39286866784095764, 1.0589206218719482, 0.3717058002948761, -2.398109197616577, -0.6290527582168579, -0.009555705823004246, 0.3258998692035675, 0.2678854167461395, 1.5591119527816772, -2.304044723510742, -0.001445603440515697, -0.0035224263556301594, 0.056766577064991, 2.377084732055664, -0.7798622846603394, -0.5333384275436401, -0.7402547597885132, 0.12061870843172073, -0.12224021553993225, -0.32081887125968933, 0.033011868596076965, 0.025854144245386124, 0.25599485635757446, -0.7347768545150757, -0.5453400015830994, 0.3917209804058075, -1.5445064306259155, -0.0006045669433660805, -0.31883180141448975, 0.2920600175857544, 0.44115525484085083, 0.6212911605834961, -0.4865271747112274, 0.2221268266439438, -0.013852234929800034, -0.5220372080802917, -0.15871702134609222, 2.663853645324707, 0.1864159107208252, -1.403495192527771, 0.33363571763038635, 0.23612439632415771, -0.03382156044244766, -1.159708857536316, -0.0815376341342926, 0.20813508331775665, 0.9410972595214844, -0.010173161514103413, -0.6956733465194702, -0.012827586382627487, -1.4410977363586426, 0.40129169821739197, 0.3530799448490143, 0.8692148923873901, -0.3060980439186096, 0.09807618707418442, -0.565625786781311, 0.1241891011595726, 0.7710850238800049, -0.07053135335445404, 0.16762429475784302, 0.42351287603378296, -0.6524009704589844, 1.3179599046707153, -0.08561103045940399, -0.8249902129173279, 0.17241409420967102, 0.8557459115982056, -0.19638657569885254, -1.9915411472320557, -0.30165374279022217, -0.14748245477676392, 0.05904088169336319, 2.074451446533203, 0.4249156713485718, 0.6711775064468384, 0.010860572569072247, -0.53775554895401, 0.2715568244457245, 0.056963831186294556, 0.7428730130195618, -0.26248055696487427, -0.1394660621881485, -0.022010313346982002, 1.2692337036132812, 0.744627058506012, -1.2226983308792114, -1.2618093490600586, 0.09727361798286438, -0.818178653717041, -0.06863818317651749, 0.10157507658004761, 1.0049896240234375, 0.29930412769317627, -0.20884200930595398, 1.3304001092910767, -0.21282543241977692, 0.38526925444602966, -0.003004231955856085, -0.0025558657944202423, 1.3758403062820435, 0.7233011722564697, 0.2928241491317749, -0.0031410250812768936, 0.096391461789608, 0.3575999140739441, -0.10608124732971191, -0.02608281560242176, 0.8208417892456055, -0.24876227974891663, -0.6165710091590881, -0.34478795528411865, -0.07279972732067108, -0.03764008730649948, 0.5040343999862671, -0.34304049611091614, -0.8575432300567627, 0.6092663407325745, 2.8561084270477295, -0.3247847557067871, -0.08762756735086441, 0.06946782022714615, -0.5180420279502869, -1.1171845197677612, 0.2677032947540283, -0.05190310627222061, -1.4379900693893433, -3.4904189109802246, 1.052984595298767, 0.24578098952770233, -0.5627505779266357, 0.9128953814506531, 0.20339733362197876, 1.4826997518539429, 0.022926848381757736, 0.45313867926597595, -0.37532636523246765, -0.0013631522888317704, 0.0023451282177120447, 1.3647866249084473, -0.15670637786388397, 0.3002675175666809, -0.017374863848090172, 0.948723316192627, -0.2066924124956131, -1.9984363317489624, 0.5870053768157959, 0.4170292019844055, -1.4208533763885498, -0.21703076362609863, 3.745664596557617, 0.6206963062286377, -1.3396629095077515, -0.5702106952667236, 0.18233002722263336, 0.13015495240688324, -0.013028743676841259, -0.9328508973121643, -1.7215485572814941, 0.10512333363294601, -0.01712176389992237, 0.9082008004188538, -0.26630693674087524, 0.15992091596126556, -0.6433061361312866, -0.03862908482551575, 0.08172332495450974, 0.14921168982982635, 1.0047976970672607, 0.34542882442474365, -2.555664539337158, -0.4778848886489868, 0.5503023862838745, 0.3447711169719696, -0.25254568457603455, 0.5624104142189026, -6.8647661209106445, 0.34418925642967224, -0.05439677834510803, 0.07607899606227875, -0.49528050422668457, 0.006917207967489958, 0.26323577761650085, -1.2373231649398804, -2.4877030849456787, -0.9525716304779053, 0.1897265464067459, 0.6368618607521057, -2.9075584411621094, 2.163057565689087, -0.8596158623695374, 0.3669925630092621, -0.028745003044605255, -0.08848671615123749, -0.7750152945518494, 0.22427600622177124, -0.10716502368450165, 0.14328324794769287, 0.09775157272815704, 1.3096749782562256, 0.391174852848053, -0.8413280248641968, 0.44569018483161926, -0.15804676711559296, 0.5976390242576599, 0.022848188877105713, 2.674015522003174, 0.23978522419929504, 1.6151151657104492, -0.038768231868743896, -0.0626988485455513, -0.6274615526199341, 0.060077887028455734, -1.2428134679794312, -0.004773049149662256, 0.38260433077812195, 2.8760292530059814, 0.1393587440252304, -0.21318891644477844, -0.3771776258945465, -0.31601712107658386, -1.1669535636901855, -0.39594516158103943], [0.15421180427074432, 0.21708841621875763, -1.9387125968933105, -0.5770370364189148, 0.8906263709068298, 0.17154012620449066, -0.44422537088394165, 0.08276259154081345, 2.657520294189453, -0.07291942089796066, 1.1031270027160645, -0.1964334100484848, 0.602939248085022, 1.0118368864059448, -2.5281028747558594, 0.2214975655078888, 1.1890209913253784, 0.678227424621582, 0.8046294450759888, -0.13983668386936188, 0.5072020292282104, -1.3773542642593384, -0.3581513464450836, 0.08372300863265991, 0.1735106110572815, 0.5801678895950317, -0.04365387558937073, 0.2579665184020996, -1.3333039283752441, 1.2882921695709229, -0.6518609523773193, 1.2755433320999146, -0.3428206145763397, 1.1093697547912598, 0.7023225426673889, 0.7461754083633423, 2.5631887912750244, 0.2769921123981476, -0.6226199269294739, 0.9579289555549622, -1.7879005670547485, 1.331838846206665, 0.6180795431137085, -1.242163896560669, 0.44524240493774414, 0.437532514333725, 0.22340038418769836, -0.8622059226036072, -0.1610105186700821, -0.3663988709449768, 1.9811204671859741, -0.028391918167471886, 0.03543056547641754, -0.5790916085243225, -0.2632150650024414, 0.07486595213413239, 0.387703537940979, 0.880687415599823, -1.546972632408142, 0.47988805174827576, -1.4541960954666138, -0.20516745746135712, 1.050842523574829, -0.6593309640884399, 0.5383955836296082, -1.226298213005066, 0.006081998813897371, -0.5167243480682373, -0.026042457669973373, -0.5592196583747864, -1.138798475265503, 0.7716517448425293, 0.033158235251903534, 0.07578502595424652, -1.127184510231018, 1.450675368309021, -0.3838413655757904, -1.0531195402145386, -0.39092767238616943, -4.696279525756836, 0.03425354138016701, 0.06594932824373245, 0.06993859261274338, 0.30059078335762024, 2.072431802749634, 2.1546812057495117, 0.8285002708435059, 0.16438448429107666, 0.25721803307533264, 0.3240720331668854, 1.7482329607009888, 0.5238681435585022, 1.6056740283966064, -0.20711639523506165, -1.4290882349014282, -0.09724811464548111, -0.3267805576324463, 0.6485350131988525, 0.5298112034797668, -0.12695014476776123, -1.7854169607162476, 0.2844385504722595, -1.019639253616333, -0.781116247177124, -1.2585746049880981, 0.9925874471664429, 0.913949728012085, -0.8555588722229004, 0.5506748557090759, -0.1828673779964447, 1.0771318674087524, 0.18445651233196259, -0.022290721535682678, -2.984985113143921, -0.015794534236192703, 1.44610595703125, -0.1739111989736557, 1.3857040405273438, 0.01869949698448181, 0.030025344341993332, 0.008965883404016495, -0.5387392044067383, -0.015917843207716942, -0.23801830410957336, 0.05981065705418587, 0.7702916264533997, -0.1595618724822998, -1.6319175958633423, -0.06295470893383026, 0.051879528909921646, 1.4395047426223755, 0.06650668382644653, -0.7895946502685547, 0.538895845413208, -0.5080661773681641, 0.25317293405532837, 0.2747964560985565, -0.26845380663871765, -1.549258828163147, 0.14670704305171967, -0.9441842436790466, 1.0317881107330322, 0.07129862904548645, 3.1604061126708984, -0.6604672074317932, 0.35732439160346985, 1.020076036453247, 0.3941394090652466, 0.26406750082969666, -0.5232120752334595, -0.4948495328426361, 1.0346970558166504, 0.9666751623153687, 0.26794537901878357, 0.030323373153805733, 0.033852554857730865, 0.537963330745697, 1.2681962251663208, 1.2377849817276, -0.44561636447906494, -0.14190107583999634, -0.09700994193553925, -0.2355055958032608, 0.24646420776844025, 0.8892691731452942, 0.0952620655298233, 0.03390458971261978, -0.14173166453838348, -0.008016425184905529, -1.2758854627609253, -0.15628287196159363, 0.05764656141400337, 0.6387094855308533, -0.06273510307073593, -0.23269382119178772, 0.6771681308746338, 0.01806827262043953, 1.4167176485061646, 0.9531120657920837, -0.00577273266389966, 0.27783703804016113, 0.32543060183525085, 0.6168055534362793, 1.0683704614639282, 0.7342069745063782, 0.19096069037914276, 0.21234211325645447, -1.5800806283950806, 1.8948783874511719, 0.12380257248878479, 0.48314064741134644, 0.9279176592826843, 0.6418013572692871, 0.30617275834083557, -0.14010845124721527, -0.20856361091136932, 0.9749858379364014, -0.5649445652961731, 0.04133414477109909, -0.012891706079244614, 0.08573759347200394, 0.04810542240738869, 0.313983678817749, -0.07148119807243347, 0.03081159107387066, -0.3644416332244873, -0.007364748045802116, 0.012798902578651905, 0.040860116481781006, 0.17148403823375702, 2.595557451248169, 1.6575442552566528, 0.15889212489128113, -0.33137232065200806, 0.5655422210693359, -0.8564077615737915, -0.04350140318274498, -0.28358450531959534, 0.7403881549835205, 0.10184438526630402, -0.4646944999694824, -1.1699124574661255, 1.377448558807373, -0.6392029523849487, -0.1789720356464386, 0.08920162171125412, 2.3841593265533447, 0.5416575074195862, -0.14228519797325134, -0.09141400456428528, 0.04260112717747688, -0.544686496257782, -0.9438487887382507, -0.9834852814674377, -0.06896357238292694, 0.06688792258501053, 0.4804805517196655, 0.20314575731754303, 0.06281716376543045, -1.8220542669296265, -0.19134269654750824, 0.5218136310577393, 0.2569471299648285, -0.1939513087272644, -0.051918961107730865, -0.07564752548933029, 0.0333632230758667, 1.2550048828125, -0.14936044812202454, 0.04478500038385391, 0.6236623525619507, 0.06837061047554016, 0.023316340520977974, 0.5837204456329346, 0.1282311975955963, -0.5009918808937073, -0.6022993922233582, 1.7440848350524902, 0.011805899441242218, 0.3743870258331299, -0.3335992991924286, -0.05193014070391655, -0.1546015590429306, -0.016457470133900642, -0.5037314295768738, 0.07442440837621689, -0.8424338698387146, 0.023792147636413574, -0.29857146739959717, 0.020648449659347534, -0.7273852825164795, 0.3845556974411011, 0.13933905959129333, 0.039143532514572144, -0.4404427707195282, 0.9794827103614807, -0.026088880375027657, 0.03890004754066467, 0.05396648868918419, -0.2845294773578644, -1.7162213325500488, -0.013135457411408424, 0.046700313687324524, -0.19607239961624146, -1.9253654479980469, -0.2162945568561554, -3.4226491451263428, 2.0936777591705322, -0.27241992950439453, 0.11197089403867722, -0.8360651135444641, 1.3756283521652222, -0.00956942979246378, -0.22381024062633514, -0.5764865875244141, -0.08038457483053207, 1.3031798601150513, -0.3672022223472595, 0.3168794512748718, 0.39142337441444397, -0.3829222619533539, -0.3890268802642822, 1.067619800567627, -0.25740689039230347, 0.42463430762290955, 0.8021441102027893, 0.2689444124698639, 0.1234627291560173, 1.1421555280685425, 0.2695258557796478, -0.3388395309448242, -0.060164257884025574, -0.20772886276245117, 1.310165524482727, 2.6170055866241455, 0.2089167833328247, -0.22797249257564545, 0.015975970774888992, 1.1059870719909668, -3.88370418548584, -0.21308790147304535, 1.1059776544570923, -0.34819909930229187, -0.690205991268158, -1.9610668420791626, 0.03808896243572235, -0.32419082522392273, 0.02032245695590973, 0.5295444130897522, 0.04425482079386711, -1.0096197128295898, 1.3897629976272583, -0.4844028055667877, 1.0043768882751465, 0.06705731153488159, 1.1766865253448486, 0.32317954301834106, 0.7363117933273315, 0.029388427734375, 0.3059673011302948, 1.6263779401779175, 0.5653911232948303, -1.0218334197998047, 0.02885885164141655, -1.2367087602615356, -0.3324221074581146, 0.11333159357309341, 0.09571155160665512, 3.278718948364258, -0.19912351667881012, -1.4393715858459473, -1.1439703702926636, 0.030203862115740776, -0.026803888380527496, 1.8812066316604614, -0.18344619870185852, 0.7343888878822327, 0.38177189230918884, -0.25852179527282715, 0.2404857575893402, 0.011474614962935448, 0.004248043987900019, 0.12635774910449982, -0.6212208867073059, 0.44299983978271484, 0.14264272153377533, 1.2729175090789795, 0.05057046562433243, 0.1570814996957779, -0.15587176382541656, -1.5227882862091064, -0.1053662896156311, -1.959149956703186, -1.231082558631897, 0.7528990507125854, 1.0477800369262695, -0.7598909735679626, -0.3375902473926544, 0.2214803546667099, -1.1361294984817505, -1.2395540475845337, -0.36626407504081726, -0.025116154924035072, 0.02395898476243019, -0.34678584337234497, 0.7922295928001404, -0.29844754934310913, -0.13490290939807892, -0.5083764791488647, 0.13481192290782928, 0.02995375171303749, 0.2107115089893341, -0.19943203032016754, -0.0953509509563446, 0.03412141278386116, 0.015240190550684929, 0.09072701632976532, -0.7974434494972229, 0.008492802269756794, 0.04533594101667404, -0.08102742582559586, 0.26980119943618774, -0.05568358302116394, 0.9688421487808228, 0.45827195048332214, -0.25507140159606934, 0.037811245769262314, -1.5142592191696167, -0.3999171555042267, 0.3953511416912079, -0.2161254584789276, -0.09499922394752502, 0.37802115082740784, -0.858856201171875, 1.369316577911377, 1.6357628107070923, 1.891193151473999, -0.38547807931900024, -3.6573433876037598, 0.06550318747758865, -1.5608038902282715, -0.4137810468673706, 0.017406774684786797, -2.266627550125122, 0.2606489062309265, 1.91861093044281, -0.4210107624530792, -0.02527868002653122, 1.1645135879516602, -1.0328718423843384, 0.2749044895172119, -0.5009071826934814, -0.04494166374206543, 0.4516894519329071, -0.03650834783911705, -0.7122758626937866, -0.738069474697113, -0.6064249873161316, 0.2260553240776062, -0.14724808931350708, 0.2924174964427948, 0.7958756685256958, -0.9506303668022156, 0.07113847881555557, 3.605746269226074, 0.7314683198928833, -0.45490899682044983, -1.618899941444397, 0.011506384238600731, -0.25977590680122375, -0.1343090534210205, -0.21517449617385864, 0.012998055666685104, 1.2537928819656372, 0.08510800451040268, 0.3349481523036957, -1.965954303741455, 0.0388575978577137, -1.3044168949127197, 0.3059143126010895, -0.006667954847216606, 0.05688966065645218, -0.6124298572540283, -4.687657833099365, 0.5732924342155457, -0.448076069355011, 0.3141506314277649, -1.2182704210281372, -2.5205769538879395, 0.2047736942768097, 0.38167065382003784, -4.137339115142822, 0.25991567969322205, -0.3663178086280823, 0.2841164469718933, -3.6825761795043945, 0.5784705281257629, 0.44659197330474854, -2.306124210357666, -0.7108773589134216, -0.1988690197467804, 1.0849626064300537, 0.07188893109560013, 0.41122251749038696, 0.024871891364455223, -0.7695956826210022, 1.5966215133666992, -0.07900968194007874, -0.7015820145606995, 0.026585737243294716, 0.1690698266029358, -0.23917706310749054, 0.18499930202960968, 0.29308509826660156, -0.26030272245407104, 1.2318401336669922, -0.031433578580617905, -0.2780505418777466, -1.1675066947937012, 1.516496181488037, -1.6137261390686035, -0.05193568021059036, -0.14874355494976044, -0.13210682570934296, -0.007573593873530626, 0.7031900882720947, 0.16975019872188568, 0.0016307189362123609, 2.6179866790771484, 0.2062566578388214, 0.19889388978481293, -0.04296910762786865, 1.4498324394226074, 0.16082410514354706, -0.6717137098312378, 2.612457275390625, -0.32337823510169983, -0.03943448141217232, -0.44619446992874146, -0.31747061014175415, -1.3552873134613037, -2.5833611488342285, -0.4031475782394409, -0.37598857283592224, -0.38468098640441895, 0.021756235510110855, 0.048995617777109146, -0.8811336159706116, 0.48906007409095764, -2.6050896644592285, -0.34190768003463745, 1.9408302307128906, 1.5628091096878052, 0.22253598272800446, -0.3391292691230774, -0.5413007736206055, 1.463857889175415, 0.15911921858787537, 0.30366963148117065, -3.26375150680542, -0.6863057017326355, 0.06387893110513687, -1.2127177715301514, -0.030424367636442184, 0.3965896666049957, -0.5311048626899719, 0.4003680348396301, 0.14645187556743622, -1.0368657112121582, 0.11358793079853058, 0.304646760225296, -1.1937355995178223, 0.12437295913696289, -0.27521994709968567, 1.0829191207885742, 0.04329442232847214, 0.03130217269062996, 0.01747211068868637, 0.19496634602546692, -0.5256381034851074, 2.6727209091186523, -0.01228298619389534, 0.3146679401397705, 0.5649449825286865, 0.9366823434829712, 0.02266557700932026, 0.04726189374923706, -0.6770474910736084, 0.3500562608242035, 0.5827035903930664, 0.9097366333007812, 0.2973560392856598, 0.019797714427113533, -0.53400719165802, -0.3683684468269348, 0.8122655749320984, -0.14503413438796997, -0.4655857980251312, 0.03386642411351204, -0.18863657116889954, 0.04957324638962746, -0.458013653755188, -0.7534113526344299, 0.13296306133270264, -1.697441816329956, 0.4124503433704376, 0.709463894367218, 0.24024991691112518, 1.1394098997116089, -0.08888604491949081, -0.2033952921628952, 0.6260473132133484, -0.12289300560951233, -0.15114158391952515, 0.394758939743042, 0.12434912472963333, 0.22572888433933258, 1.8355273008346558, -1.7371163368225098, -0.36644530296325684, 0.1117682084441185, -2.521047592163086, 0.36956238746643066, -0.03551710769534111, 1.171112060546875, -1.3395193815231323, -0.37713703513145447, 0.4009305238723755, -0.14040599763393402, -0.8527981042861938, -2.7083499431610107, 0.2214406281709671, 0.04707009345293045, 0.28400805592536926, 4.220560550689697, -0.8441321849822998, -0.3416823744773865, -0.024677982553839684, 0.1429092437028885, 0.2929045855998993, -0.1892881691455841, 0.02709323912858963, -0.21487873792648315, -0.2916773855686188, -0.15786336362361908, 0.061641864478588104, 0.13779887557029724, -1.0185927152633667, 1.2743116617202759, 0.22341535985469818, -0.6220433115959167, 0.929590106010437, 0.29165831208229065, 0.48939064145088196, -0.8757262825965881, 0.5958909392356873, -0.6459792256355286, -1.5027992725372314, -0.03923458233475685, 0.9421215057373047, -0.8587758541107178, 0.6893700957298279, -0.05253525450825691, -0.010550065897405148, -0.22867336869239807, 0.6369327306747437, 0.010826827958226204, 0.07383312284946442, 0.13752572238445282, 0.11598820239305496, 0.11783643811941147, -0.0029084528796374798, 0.3165099024772644, -0.4621729254722595, 0.23385687172412872, 0.4986158311367035, -0.8170748949050903, 0.1272992640733719, 0.31755733489990234, 0.16075649857521057, 0.8457040190696716, -0.16661517322063446, 0.45365479588508606, -2.2361788749694824, -0.2504558265209198, -1.024262547492981, -0.33099499344825745, 0.7903481721878052, -0.4198792278766632, 1.0848641395568848, -0.6107233166694641, -0.42419272661209106, 0.4799068868160248, 1.0231552124023438, 0.006874695420265198, -0.8063175678253174, -2.3923985958099365, 0.3459811210632324, 0.02770068868994713, -0.0051203141920268536, 0.3097808361053467, 2.3281736373901367, 0.017712920904159546, -1.0218675136566162, -0.4510590732097626, -0.42127683758735657, 0.851518452167511, 0.034096866846084595, -0.44356876611709595, 0.7108666300773621, -0.17537832260131836, -0.22821027040481567, -0.2698761224746704, 2.3849141597747803, 0.5417848229408264, 0.8398898839950562, 0.9274308681488037, -1.0561801195144653, -0.6190628409385681, -1.8397215604782104, -0.24067135155200958, 0.931238055229187, 1.0199298858642578, 2.30981183052063, -0.0427752323448658, 0.022042788565158844, 0.0647827684879303, -1.2979917526245117, 0.03980458527803421, -0.9033859372138977, 0.1582237035036087, -0.25275135040283203, -0.21981462836265564, 0.2968340814113617, -2.573650360107422, -0.9797560572624207, 1.3081308603286743, 0.33876854181289673, 0.043311335146427155, -0.6492445468902588, -0.6531127691268921, -3.732438325881958, -0.34530797600746155, 0.15250429511070251, 0.3369171917438507, -0.731656014919281, 0.04649240896105766, -0.3786676526069641, 0.18535727262496948, 0.15287047624588013, 0.36458611488342285, 0.8950071930885315, 0.4022778868675232, 0.8306110501289368, -2.1562745571136475, 0.8757960200309753, 0.0227274801582098, -0.0472324900329113, 0.03558285906910896, 0.7704160213470459, -0.02641984634101391, 0.02457652986049652, -0.3944323658943176, 0.16298115253448486, 2.2983381748199463, -2.000572681427002, -0.6703493595123291, -0.4281919300556183, 0.09097325801849365, 0.7510989904403687, 0.0010352091630920768, -0.6714706420898438, 0.5240587592124939, 1.1050105094909668, 0.053025346249341965, 0.3570697009563446, 0.004398838616907597, -0.3495748043060303, 2.058204412460327, 0.36299803853034973, 0.18550731241703033, 0.15340249240398407, -0.34471118450164795, -0.0914822667837143, -0.5949555039405823, -2.030229091644287, -1.5458866357803345, -0.14679482579231262]], "dtype": "float32"}, {"__nd__": [-1.4306751489639282, -8.187704086303711, 0.9524465203285217, 5.0838751792907715, -3.538583278656006, -1.2401047945022583, 0.20484396815299988, 1.9441933631896973, -1.7150013446807861, 0.7402282357215881, 0.1298995465040207, 3.081618547439575, -2.512202262878418, 0.04062899947166443, -1.6921197175979614, 1.9099007844924927, 2.7735841274261475, -1.8807231187820435, 0.4883164167404175, 1.4652589559555054, 0.1051710993051529, -4.051390171051025, 2.873145818710327, 3.052698850631714, 1.0476926565170288, 1.4845759868621826, -1.718849539756775, -0.8617749214172363, -3.1278440952301025, 0.8011583089828491, 0.7953808307647705, -0.5254325866699219, -2.7607414722442627, 1.7310370206832886, -0.2717878818511963, -2.9262430667877197, 5.035461902618408, 0.03397654742002487, 2.3953311443328857, -5.428962707519531, 2.0004069805145264, 3.6998281478881836, -5.7602949142456055, 4.091742515563965, -3.5993072986602783, 0.01337923388928175, 1.1982742547988892, -1.146386981010437, -0.012075407430529594, 3.155167579650879, 3.8739328384399414, -0.8810470104217529, -6.843623638153076, -0.6983773112297058, 2.9098615646362305, 0.08001969754695892, -0.6770556569099426, 2.451611280441284, 0.573703944683075, -1.3059502840042114, 4.672746658325195, 1.2108917236328125, 0.2606756091117859, -2.7524213790893555, -0.48588141798973083, 3.0829386711120605, 0.5358690023422241, 3.0918543338775635, 3.172419786453247, 3.7150473594665527, -1.736473798751831, -0.3018343150615692, 4.090217113494873, -0.6618630290031433, 0.06424855440855026, -2.443333148956299, -2.9482617378234863, -4.83808708190918, 3.984633207321167, -3.2650680541992188, -1.8330087661743164, -1.2204900979995728, 1.139206051826477, 0.9776784777641296, -1.096634030342102, -1.2443569898605347, -1.6353353261947632, -0.7972524762153625, -2.8297057151794434, 1.7595219612121582, 1.0833206176757812, 0.7625213861465454, 1.0598920583724976, 3.6341419219970703, 1.8614304065704346, 2.092712163925171, -0.7489567995071411, -4.00433874130249, 2.1462812423706055, 4.936724662780762, -0.6364884972572327, 1.0634634494781494, -1.6621441841125488, -0.11140646785497665, 2.6804020404815674, -2.4267523288726807, 4.681155681610107, 4.7111358642578125, 2.666668176651001, 4.749177932739258, 3.4804186820983887, 1.0306428670883179, -5.283987045288086, -2.3416359424591064, -0.19108542799949646, -2.7705514430999756, -7.49414587020874, 3.271120071411133, 0.9545665979385376, 2.887080192565918, -0.0436829999089241, -0.9644137620925903, 2.0732345581054688, 0.40857580304145813, -0.25677064061164856, 0.4653223156929016, 0.8563792109489441, 1.230472207069397, 1.6959444284439087, 4.980190753936768, -1.392473578453064, 2.203930616378784, -2.4714102745056152, 0.5354099869728088, -3.8133163452148438, -2.3506951332092285, -3.01187801361084, 0.22643905878067017, 0.11446844786405563, 1.2713549137115479, 6.7269415855407715, -1.978649377822876, -4.548717498779297, 0.17308977246284485, -1.5332468748092651, -1.670426845550537, -1.679469108581543, -1.884595274925232, -2.832702398300171, -0.8185705542564392, -1.7373303174972534, 2.491792917251587, 1.4573355913162231, 2.0371603965759277, -2.4729373455047607, -4.44163179397583, 0.1307259500026703, -4.072071075439453, -3.3730790615081787, 1.0698801279067993, -0.34156152606010437, -0.5588049292564392, 0.6058425307273865, 2.3040120601654053, -1.5491485595703125, 1.4252537488937378, -0.9528917670249939, -2.5799949169158936, -3.153667449951172, 0.2155914157629013, -0.7220613360404968, -1.1297613382339478, -0.06978809088468552, 0.8632787466049194, 2.3771257400512695, 1.833176612854004, 1.6449356079101562, -1.187158465385437, -1.1635527610778809, -1.2181854248046875, -2.1705071926116943, -2.7936129570007324, 1.4939104318618774, -5.123149871826172, -0.33883994817733765, -0.3954494297504425, 3.442955732345581, -0.7091127634048462, 0.06677854806184769, 2.39241623878479, -0.9894859194755554, 5.180227756500244, -1.6108667850494385, 1.6246397495269775, 1.073204517364502, -7.104187488555908, -0.01280617993324995, -3.693613052368164, -0.729215681552887, -5.134965419769287, 2.2639870643615723, 2.3282809257507324, -0.06878971308469772, 1.5092651844024658, -0.6576054096221924, 2.7759721279144287, -0.343987375497818, -3.4556455612182617, 3.474943161010742, -5.127715587615967, -1.2972815036773682, 0.08836619555950165, 0.5068089365959167, -0.4773668646812439, 0.11270707845687866, 1.7934365272521973, -3.6120855808258057, 2.768587112426758, 1.895877480506897, -1.383420467376709, 0.024619679898023605, 2.158869504928589, -0.5274814963340759, 1.2468407154083252, -1.96531081199646, -2.2304277420043945, -3.474905252456665, -2.477694272994995, -4.005309104919434, 1.203900933265686, -0.13698779046535492, 2.5769360065460205, 2.7694807052612305, 3.4001739025115967, 0.025773702189326286, 2.843904972076416, 0.2775723338127136, -0.7097036838531494, -0.9510361552238464, -1.3308647871017456, -1.5186160802841187, -2.0293946266174316, 4.243117332458496, -5.33546257019043, -2.2722041606903076, 4.3082170486450195, 0.6785705089569092, -1.789455771446228, -0.4818120300769806, 3.937460422515869, -1.1127058267593384, 2.6902477741241455, -8.206888198852539, 2.162569522857666, -1.858766794204712, 1.1431456804275513, -1.3573508262634277, 1.1979609727859497, 1.0426691770553589, 1.3189483880996704, 1.7970366477966309, 0.3117903172969818, 3.465873956680298, -0.8291845321655273, -1.0969226360321045, 0.84357088804245, 1.988516092300415, -4.030825138092041, -0.48060429096221924, 6.254687786102295, 1.045243740081787, 2.9989216327667236, -0.8069923520088196, 3.6507203578948975, -3.425629138946533, 2.830789089202881, 0.5301463603973389, 2.782348155975342, -0.90431147813797, -2.797558069229126, -2.7235524654388428, 5.271822452545166, -0.9024302363395691, -5.286196231842041, 2.3089606761932373, -3.6013317108154297, 1.2982876300811768, 0.06512327492237091, 3.112602472305298, -3.160748243331909, 2.5681841373443604, -0.45532044768333435, -1.6372666358947754, -2.0894055366516113, 2.7514841556549072, 0.8316066861152649, -2.293560028076172, 1.6716734170913696, -4.842126369476318, 4.31920862197876, -1.2382498979568481, -0.7379458546638489, -1.5531102418899536, -1.1037598848342896, 0.5569318532943726, 7.886650562286377, -4.354013919830322, -1.9099161624908447, -1.213381052017212, -6.451300144195557, -2.2836616039276123, 0.008323273621499538, -1.2620491981506348, 2.0056111812591553, -2.186250686645508, 2.3764922618865967, -1.3251073360443115, -2.1271228790283203, 6.434933662414551, -4.341896057128906, 1.136251449584961, -1.6433961391448975, -0.30808091163635254, 5.855558395385742, -4.348183631896973, 1.9969027042388916, 0.1508273184299469, 2.2520458698272705, 0.24886655807495117, -3.455576181411743, 0.11000648140907288, 1.113762617111206, 4.381986618041992, 4.814442157745361, 2.6455764770507812, -1.4845525026321411, 0.9734686017036438, -0.11613436043262482, 2.967402219772339, 2.6067311763763428, 0.7860637307167053, 1.515329122543335, 2.904736042022705, 5.5562663078308105, -1.6457769870758057, 1.8317887783050537, 2.5529747009277344, 0.5273517966270447, -10.324005126953125, -1.1659785509109497, 3.216343641281128, 0.981858491897583, -2.544299840927124, 2.6988353729248047, 3.9727160930633545, -1.810689926147461, -1.741911768913269, 1.1319999694824219, 0.8907918930053711, 0.861537754535675, -1.3911889791488647, -2.633261203765869, -1.9536341428756714, -2.2551205158233643, -0.11272239685058594, -7.813727378845215, -0.11300953477621078, -0.6095464825630188, 0.03826979920268059, -3.2148900032043457, 3.64052152633667, -0.4176749587059021, 1.2878212928771973, -3.1328728199005127, -0.4302197992801666, 2.8474533557891846, 2.0370724201202393, -2.8078646659851074, 0.23944984376430511, -2.0236668586730957, 2.0042848587036133, 6.953567981719971, -1.68695867061615, -0.49707865715026855, -1.8599034547805786, 2.199507713317871, 5.794503688812256, -3.4417991638183594, 1.9847198724746704, -1.3316785097122192, 2.897305727005005, 1.1537634134292603, 1.47200345993042, -2.579477310180664, -0.8571455478668213, -0.5093920230865479, 0.4980749189853668, -4.244592189788818, -0.1112581193447113, 0.3714175224304199, -1.4405748844146729, 6.742717266082764, -4.342626094818115, 0.19474922120571136, 0.7463130354881287, 0.4148980379104614, 1.4420045614242554, -1.3147445917129517, -0.9928677678108215, -5.512382507324219, 3.2672653198242188, 2.664785146713257, 1.573797345161438, -0.19379425048828125, 1.2354048490524292, 0.028989221900701523, -3.3788135051727295, 0.26976728439331055, -9.155280113220215, 0.43140774965286255, 0.20543184876441956, -3.96056866645813, 3.8317453861236572, -2.7038800716400146, 2.3782553672790527, 1.988904356956482, -2.7945404052734375, -0.12498094886541367, 0.42821282148361206, 2.2851216793060303, -3.255941390991211, 3.987051010131836, -0.9329707026481628, -0.8179636001586914, -3.0928306579589844, -6.5847344398498535, -0.37054502964019775, -0.5109230279922485, -4.806411266326904, 1.1347228288650513, 4.528164863586426, -3.160731792449951, 0.869821310043335, -2.8638999462127686, 3.182063579559326, 1.2710754871368408, 0.683672308921814, 2.529120683670044, -3.5577871799468994, -1.5594171285629272, 4.509085178375244, -0.6302321553230286, -0.630766749382019, -0.41541773080825806, -0.2833136022090912, 3.921818256378174, -2.3104608058929443, 0.06794252246618271, -0.48920413851737976, 2.4340357780456543, 4.371143341064453, -2.9690465927124023, -2.3883135318756104, 1.0939043760299683, 0.706913948059082, 5.9442138671875, 1.0625667572021484, 2.3871700763702393, 0.5456979274749756, -1.622531771659851, -0.6025176048278809, -9.319032669067383, -0.5840086936950684, 0.7773942351341248, 7.940225124359131, -0.8266087770462036, -0.3871913254261017, 3.304677963256836, -2.023684024810791, 4.851041316986084, -5.087183952331543, -2.912863254547119, -6.719820976257324, -2.447355031967163, 1.0657682418823242, 1.8042709827423096, 0.15300428867340088, -2.711625576019287, -2.261263847351074, -0.10345900058746338, -0.067217618227005, -0.6291780471801758, -0.6338971257209778, -1.8095614910125732, -1.006289005279541, -0.3388300836086273, -1.6744197607040405, 1.1586076021194458, 4.982193946838379, 0.7458315491676331, -1.067065715789795, -2.565425395965576, 0.4553704857826233, 0.5152370929718018, 1.069180965423584, 0.7313112020492554, -4.3454203605651855, 2.5364744663238525, 1.120829701423645, 2.268296957015991, 4.040092468261719, 1.259526014328003, -8.425824165344238, -3.7217302322387695, -0.3748626410961151, -0.009513203985989094, 5.469367980957031, 0.10034238547086716, 1.4910176992416382, 0.2947080731391907, -4.1409735679626465, -1.08999764919281, -0.7952278852462769, -2.343703031539917, -1.0861127376556396, -1.0773056745529175, 0.838678240776062, 2.1818251609802246, 0.6317346096038818, 8.049603462219238, -4.954916954040527, -1.8170771598815918, 1.6918928623199463, -0.9963799715042114, 1.8332669734954834, -0.010253101587295532, 0.6674099564552307, -3.4144904613494873, -0.06405401974916458, 1.5669890642166138, 0.8002326488494873, -1.3029398918151855, 4.752850532531738, 1.0564360618591309, -3.51857852935791, 3.656442642211914, 2.454869031906128, -2.5025291442871094, -0.6276252865791321, -4.612129211425781, -3.447216749191284, 8.658981323242188, 3.545609951019287, -3.492142677307129, -1.6629955768585205, 3.3073062896728516, 0.36602744460105896, 2.042257785797119, 4.611652851104736, 5.672054290771484, 3.482034683227539, 1.3496633768081665, 10.34942626953125, -2.859015703201294, -5.001864910125732, 0.8885042667388916, 1.715258002281189, -1.1823402643203735, 3.9079558849334717, 3.658069133758545, -2.485569715499878, -2.5796406269073486, -0.377300500869751, 2.24342942237854, 0.2713966369628906, -2.9669675827026367, -0.4332507848739624, -0.4529092311859131, 0.21791154146194458, 0.09240667521953583, -2.1819188594818115, -0.6381630301475525, -1.9028117656707764, -4.0880022048950195, -0.29350119829177856, 1.2374438047409058, -3.070678949356079, -6.40557861328125, 4.007668972015381, -0.26580163836479187, -1.593601107597351, 1.8973811864852905, -2.979457378387451, -1.8281080722808838, -0.6288183927536011, 1.7659549713134766, 1.4226412773132324, 0.14919355511665344, 3.604116678237915, -2.661161422729492, 1.3714275360107422, -1.2066773176193237, -5.22147798538208, -1.9637556076049805, -3.8335158824920654, -0.7477313280105591, 0.17094944417476654, 0.3429344892501831, -0.3618929386138916, -0.5846781730651855, -1.3830649852752686, -0.19932129979133606, 1.2079672813415527, 1.512643814086914, 5.369864463806152, 0.5254883170127869, 0.8166337609291077, -0.11740650236606598, 2.1334574222564697, -3.271395444869995, 1.5190640687942505, 1.0156054496765137, 4.155175685882568, -1.6830695867538452, -1.6479012966156006, -1.6570584774017334, -0.42352935671806335, -0.9969989657402039, 4.4644365310668945, 3.2979319095611572, -3.469426155090332, 2.6419172286987305, 0.5236843228340149, -2.8326971530914307, 0.3688490390777588, -0.22425341606140137, -1.887729287147522, -1.8182430267333984, -1.7925713062286377, -9.641091346740723, -2.6476173400878906, -1.9130172729492188, 1.2294079065322876, -1.0790512561798096, -2.030477523803711, -3.9460761547088623, 2.237701892852783, 1.7328709363937378, 1.344322919845581, 0.780197262763977, 0.5056585073471069, 2.8826749324798584, 2.7649900913238525, 4.076838493347168, 1.5463248491287231, 1.020815372467041, 1.8673760890960693, -0.9395166039466858, 0.8201646208763123, 1.8523024320602417, -2.5708014965057373, 1.9163339138031006, -2.322666883468628, 1.9191874265670776, 1.7401386499404907, -1.111688256263733, -1.7383664846420288, -2.0278942584991455, 4.583887100219727, 1.922792673110962, 0.5106714963912964, -1.3690414428710938, 5.749844074249268, 0.6409395933151245, -0.8680259585380554, 2.172956705093384, -0.8482474088668823, -2.501398801803589, -4.161160469055176, -0.24065402150154114, -0.6421781182289124, -1.5535399913787842, -2.081447124481201, -3.961981773376465, -1.3828321695327759, -3.280362844467163, 2.1426734924316406, -0.1884474754333496, -0.33846575021743774, 2.6256163120269775, -3.706326484680176, -2.492344856262207, -0.5950160026550293, -2.3576741218566895, 0.683998703956604, 1.4195513725280762, 2.241635322570801, -0.5306606292724609, 0.5752110481262207, -1.8878734111785889, 2.134075403213501, -0.7525688409805298, -3.604201555252075, 1.7940236330032349, -2.6834962368011475, -1.5727273225784302, -5.3292412757873535, -2.4110987186431885, -2.12716007232666, 0.7097793221473694, -0.6421904563903809, 1.0773873329162598, -3.8560664653778076, 1.143662929534912, 0.983610212802887, 4.851905345916748, -5.280043125152588, 0.32545629143714905, 2.6335299015045166, -1.3973443508148193, 1.3903182744979858, -1.3949073553085327, 4.5745110511779785, 0.7641948461532593, -0.714235246181488, 2.870995283126831, -1.9352082014083862, -0.7610836625099182, -2.322805881500244, -1.912984013557434, 0.6496724486351013, -1.991136908531189, 0.8980576992034912, -0.5155021548271179, -0.3990689218044281, -0.22655926644802094, -5.361639499664307, 0.990199089050293, 1.2045602798461914, 3.500973701477051, -1.9579979181289673, -1.9751743078231812, -3.7722384929656982, -1.3402307033538818, 1.0627686977386475, -1.9235355854034424, -4.62157678604126, 1.944726586341858, 3.50980281829834, -6.425814151763916, 1.899046778678894, -0.00884809996932745, -1.7245264053344727, -1.7811928987503052, -0.12273133546113968, -1.205096960067749, -0.8579740524291992, -1.2159770727157593, 1.0213627815246582, 1.4297428131103516, 4.380715370178223, 0.6926135420799255, 1.364856243133545, 2.3170599937438965, 2.1931893825531006, 1.207261085510254, -0.04072722792625427, 0.5639827251434326], "dtype": "float32"}, {"__nd__": [-3.687666654586792, 2.062795400619507, 0.30315348505973816, 0.5783282518386841, 0.5452190041542053, -1.3931055068969727, -1.487044095993042, -2.594191551208496, -0.5129798650741577, 0.925578236579895, -2.6319079399108887, 3.099729537963867, 2.1574058532714844, 1.1029099225997925, 0.13205450773239136, 0.20698995888233185, 6.73598051071167, -0.6189894676208496, 0.5737752318382263, 2.4108431339263916, 2.9185101985931396, -4.2339043617248535, -1.326213002204895, 3.2952353954315186, -2.3573994636535645, 1.627360224723816, 3.4637207984924316, 0.7407060861587524, 1.8361291885375977, 3.0515096187591553, -0.691718578338623, -3.4552721977233887, 6.109585762023926, 0.9134305715560913, -3.8579838275909424, -0.5126147866249084, -0.6659874320030212, -0.9619311094284058, -0.6190623641014099, 1.1140432357788086, 6.014119625091553, 1.8786722421646118, -2.1126067638397217, 0.32368478178977966, 0.7723482847213745, -0.9320536851882935, -1.1415271759033203, 2.0268666744232178, 0.010132305324077606, 0.28777021169662476, 2.154744863510132, -0.346987247467041, 0.9826924800872803, 2.1054940223693848, -1.8686281442642212, 1.479703426361084, 1.9691179990768433, -1.8223620653152466, -0.41233688592910767, -0.29064762592315674, 0.9285757541656494, -2.8420114517211914, -0.24934802949428558, -0.713009238243103, 0.804754912853241, 0.38177382946014404, 0.39716410636901855, 3.4033162593841553, -1.7481427192687988, -2.887308359146118, 1.1904160976409912, -1.7924726009368896, 2.1277108192443848, 1.6163113117218018, -4.050700664520264, 0.8524426817893982, 0.6479543447494507, 3.1299312114715576, -1.4553790092468262, -0.8135191202163696, -2.4901347160339355, 2.7740187644958496, 5.198475360870361, -0.24488461017608643, 1.261348009109497, 1.8436185121536255, -1.38704514503479, -1.203242540359497, -0.8409664034843445, -3.9383723735809326, 3.58811092376709, 1.8343256711959839, -0.7772930264472961, 1.9295916557312012, 1.7475929260253906, 0.9348353147506714, -1.0574170351028442, -2.317680597305298, -2.3652825355529785, 0.7311500310897827, 3.763397693634033, 4.270538330078125, -1.222267985343933, -2.1957902908325195, -1.4279118776321411, -2.001120090484619, 0.7568652629852295, -2.284414529800415, 5.077177047729492, 4.755229949951172, 0.7281389236450195, 0.6173325777053833, 3.8573033809661865, 0.6249489784240723, -0.8689626455307007, 0.24291163682937622, -0.9623085260391235, -3.736035108566284, 3.60727858543396, -0.7339617013931274, -2.32183837890625, 2.0627965927124023, -3.8149831295013428, 0.8695676326751709, -0.612813413143158, 2.418025493621826, -1.6286015510559082, -1.075474739074707, 4.521046161651611, 3.4560322761535645, -2.929273843765259, 2.4070045948028564, 0.7791796326637268, -0.028565743938088417, -0.7188601493835449, 0.48742571473121643, 1.999637246131897, 4.546589374542236, -4.475282192230225, -6.072654724121094, 3.1266424655914307, 1.6220029592514038, -2.4339489936828613, 0.6257480382919312, -0.6575945019721985, 1.47655189037323, 2.4895877838134766, 3.7311766147613525, 1.4083261489868164, 3.7788753509521484, -2.3302152156829834, -3.8749232292175293, 2.3304543495178223, -1.2900676727294922, 0.7125371098518372, 1.6050605773925781, 3.745891809463501, 1.81952965259552, 0.2889848053455353, 2.784865140914917, 3.67569899559021, -0.02902081608772278, 3.1018261909484863, 0.07758381217718124, -0.3195987939834595, -2.848510265350342, 0.0007013380527496338, -4.80474853515625, 0.031846970319747925, 0.08559422940015793, 3.5849857330322266, -0.16537976264953613, -1.582328200340271, -1.7558878660202026, 0.8272643089294434, -3.5469424724578857, 2.0029735565185547, -5.268482208251953, 1.7745558023452759, -3.0871050357818604, 2.280792236328125, -5.2335710525512695, 1.570327877998352, 1.0952472686767578, -5.796203136444092, -4.008523464202881, 0.21121950447559357, -2.6929538249969482, -4.0592498779296875, 2.159729480743408, -0.13653941452503204, -6.282934188842773, 6.791548252105713, 2.393376111984253, -3.214076280593872, 1.7325701713562012, -1.437780737876892, 1.8156495094299316, -2.6489157676696777, -4.075160026550293, 0.060684934258461, -1.5623562335968018, -2.3203957080841064, 2.5667612552642822, 3.3959639072418213, 1.4685070514678955, -0.6789112091064453, 1.665013074874878, -3.624884605407715, -1.9569175243377686, -0.862740695476532, -2.332592010498047, -0.2849699556827545, -3.334491491317749, 5.13165807723999, 3.477018117904663, 2.061290740966797, 1.9035347700119019, -6.968044281005859, -1.4879459142684937, 0.5140550136566162, 0.9960137605667114, 3.400825023651123, 4.229618549346924, -0.9895018935203552, -0.5643005967140198, -0.025218607857823372, 0.985352635383606, -3.7743782997131348, -3.7733314037323, 1.2515308856964111, -2.2005505561828613, 1.9831151962280273, -0.840437650680542, -2.2803587913513184, -0.30672818422317505, 4.809584617614746, -0.3898870348930359, -2.4275906085968018, -3.633596181869507, 1.1380170583724976, 0.414389431476593, 1.2173489332199097, -0.25982826948165894, 2.115567684173584, -1.2827231884002686, -4.154104232788086, -0.7511522173881531, -0.9018471240997314, -0.44269731640815735, 3.15948748588562, -0.8042552471160889, 0.23609846830368042, -1.1862666606903076, -0.37790626287460327, 1.4157066345214844, -2.6413657665252686, 4.786054611206055, 0.5469791889190674, -2.322728157043457, -0.9543452262878418, -1.5116970539093018, 1.3111258745193481, -1.0764274597167969, 0.4559177756309509, -2.9391727447509766, 1.619252324104309, -0.4584076702594757, 1.6532723903656006, 3.0318045616149902, -1.0764092206954956, -1.6647460460662842, 0.7058210372924805, -4.027129650115967, -0.9843809008598328, 4.3168864250183105, -1.8685221672058105, -4.569197654724121, -0.9307757019996643, -1.1867793798446655, 2.730159044265747, 0.7011767029762268, -2.181819438934326, -1.0909347534179688, -3.5655593872070312, 4.7061662673950195, -1.0910279750823975, 1.5514494180679321, -2.4763078689575195, -1.3593847751617432, -2.914099931716919, -6.29823112487793, 1.8966689109802246, 0.019595101475715637, 0.8150447010993958, -2.1591124534606934, -5.237579822540283, 3.712085008621216, 0.5331544876098633, 0.8895785808563232, -0.12489694356918335, 1.1446415185928345, -1.9848016500473022, 3.2180278301239014, -1.246614933013916, -2.235572338104248, -2.868069887161255, -0.21636125445365906, -1.452453374862671, 0.609207272529602, -3.0732553005218506, 0.9629208445549011, 3.2141072750091553, 0.7881417870521545, 0.9015300869941711, 2.654451608657837, -1.7522170543670654, 1.688953161239624, 0.7138992547988892, -2.555988311767578, -0.6542839407920837, -2.3454439640045166, 0.13298147916793823, -2.312241792678833, 0.4747117757797241, -2.8497023582458496, -0.18938173353672028, 5.676922798156738, -2.4587507247924805, -1.9797486066818237, 2.04292368888855, 4.1029744148254395, -3.7270267009735107, -0.8042847514152527, -3.728283166885376, 6.331733703613281, 2.937156915664673, 1.2667824029922485, -0.8117395639419556, -1.2965996265411377, 0.29478153586387634, 4.233266830444336, -1.117570400238037, -2.3660073280334473, 1.2603881359100342, -0.8834190368652344, 0.3286396265029907, -3.705947160720825, 4.332047939300537, 1.7745641469955444, 0.17723409831523895, -2.2809083461761475, 0.4893473982810974, -0.653434693813324, 6.190609455108643, 1.0649218559265137, 7.792457103729248, 0.38597428798675537, -5.2803168296813965, -0.7435328364372253, 2.945608615875244, 1.3524969816207886, 0.28498390316963196, 0.9478975534439087, 1.998545527458191, 2.9735920429229736, -0.6967731714248657, 0.4114486873149872, -0.9347987174987793, -3.7474727630615234, 1.6490509510040283, 0.9315845966339111, 0.6287026405334473, 1.480941891670227, -3.10943865776062, 3.422041893005371, 4.396810531616211, 0.10394458472728729, 1.2313735485076904, -2.801621437072754, -2.494816541671753, 1.0957551002502441, -2.909905433654785, 5.092011451721191, 0.24430882930755615, -2.9214863777160645, 1.5426018238067627, -2.8619017601013184, -1.7214781045913696, -1.0175328254699707, 1.3182252645492554, -2.9520962238311768, -0.02833862602710724, 3.6923437118530273, 1.6049182415008545, -0.34694111347198486, -3.824641466140747, -4.295418739318848, -3.0217642784118652, 3.2238967418670654, 5.01668119430542, 0.09632402658462524, 0.9329360127449036, 1.0743471384048462, -1.7177863121032715, -1.1049840450286865, -4.19841194152832, -6.047454833984375, -3.8862082958221436, -3.4716994762420654, 4.832356929779053, -0.6688241958618164, -4.23632287979126, -1.0975849628448486, -2.536062240600586, 3.0343754291534424, 0.21177004277706146, -1.796416163444519, 4.114006042480469, 1.1017929315567017, -2.9054269790649414, -4.4571075439453125, -0.34135007858276367, 2.440516233444214, -2.9477319717407227, -1.056063175201416, -1.7386667728424072, -1.4522833824157715, 1.1551212072372437, 1.0088839530944824, -1.544957160949707, -2.051164388656616, -3.026111125946045, -0.8773307800292969, -0.4311065673828125, -3.8705692291259766, -2.4090871810913086, 5.157173156738281, -1.6792473793029785, -3.7680320739746094, -1.8834840059280396, -2.368948459625244, 5.361142635345459, -0.72508704662323, 0.4260595440864563, -1.003304123878479, 0.2570997476577759, -0.47047340869903564, 2.9823172092437744, -0.14178280532360077, 4.361902713775635, -2.761355400085449, -2.9987869262695312, -2.9386146068573, -2.425408363342285, 2.774646043777466, 1.233951449394226, -0.013352721929550171, 2.3631656169891357, -1.4531729221343994, 0.22621993720531464, 1.6162301301956177, 1.5571026802062988, -6.832687854766846, 2.5599706172943115, 1.2606542110443115, -0.3355279862880707, 1.7721080780029297, 0.6105175614356995, -0.20592020452022552, -0.983336329460144, -1.9167643785476685, 3.7624504566192627, -0.4572801887989044, 1.0635080337524414, 7.339442729949951, 0.7708089351654053, 2.191619873046875, 2.5485777854919434, 0.547399640083313, 0.2090873271226883, 0.5203066468238831, 1.138214111328125, -2.236403226852417, 0.35809212923049927, 2.165894031524658, -5.742487907409668, -4.065587043762207, 1.6833635568618774, 1.0349440574645996, 2.290459156036377, -0.41893985867500305, 1.6885325908660889, -1.4511131048202515, 0.5852038860321045, 1.2397881746292114, 1.5417901277542114, 0.6837955117225647, -1.854697823524475, -4.672916889190674, 1.3732526302337646, -0.9756875038146973, -0.6808481216430664, -2.075613021850586, -4.571776390075684, -1.8393901586532593, -2.6271560192108154, -0.3896549940109253, -2.982694625854492, -4.323151111602783, 0.07415527105331421, -4.046433925628662, -2.086381673812866, 0.6262353658676147, -1.2061806917190552, 3.12808895111084, 1.555442214012146, 0.8812142610549927, -2.5831897258758545, -4.289005756378174, -0.4592403471469879, 3.386336088180542, -0.003794882446527481, 1.1633044481277466, -0.3197425901889801, 1.9420514106750488, 0.916782557964325, -0.2554100751876831, 2.3938164710998535, 5.69540548324585, -1.4597245454788208, 1.306854486465454, -2.2087223529815674, -0.6952909827232361, -1.4743924140930176, 1.4634079933166504, -4.690497875213623, 1.251271367073059, 1.8702623844146729, -1.6187266111373901, -1.7350999116897583, -3.576976776123047, 1.3475853204727173, 3.1852362155914307, -0.7228302359580994, -2.0931296348571777, 1.925722599029541, 7.257925510406494, 2.594691753387451, -4.143162727355957, 3.744917869567871, 1.8554813861846924, 4.545563697814941, -2.9516875743865967, -2.4577457904815674, 1.785628080368042, -0.024074524641036987, 3.9473190307617188, -1.2418689727783203, -4.2458319664001465, -0.7616342306137085, -6.073424339294434, -1.7453001737594604, 1.4218305349349976, 4.441041469573975, 0.9015172719955444, 1.7371739149093628, 1.090846061706543, 3.274045944213867, -1.8868956565856934, 2.7084734439849854, 0.02858424186706543, -2.5660743713378906, 2.13275146484375, -0.7894380688667297, -4.167807102203369, -6.376649856567383, 1.4478358030319214, 1.5025460720062256, 0.5618503093719482, -1.473276972770691, -3.8821280002593994, -4.38605260848999, -0.9196102619171143, 0.11191335320472717, -4.949030876159668, 0.8257732391357422, 6.473392963409424, -3.1121745109558105, 1.4677866697311401, -4.819197177886963, -4.733963489532471, 1.3008460998535156, 2.90401029586792, -0.9331439733505249, -0.8115205764770508, 4.963525772094727, -2.7571706771850586, 1.81490957736969, 2.0745718479156494, -1.0497305393218994, 4.686156272888184, -1.9910845756530762, -3.7596912384033203, 0.8831897974014282, 1.3228044509887695, 1.4613385200500488, -0.14832881093025208, 4.436793327331543, 2.513052225112915, -0.39850378036499023, -0.43844422698020935, 1.4854501485824585, 1.1798385381698608, -2.5413818359375, 2.0128955841064453, 0.5028825998306274, -1.3324357271194458, 0.6400258541107178, 2.179229736328125, 0.6877161264419556, 1.792842984199524, 2.600999355316162, -2.2427124977111816, -0.7986888289451599, 2.6720972061157227, 0.21574194729328156, 2.4436442852020264, 1.3643708229064941, -4.227283000946045, -1.563420295715332, -0.31512123346328735, 2.3606607913970947, 2.297537088394165, -0.15569409728050232, -2.027688503265381, -3.5586485862731934, 2.331380605697632, 1.905861496925354, 1.103227972984314, 0.8854894638061523, 2.1554510593414307, -5.193518161773682, -2.6752636432647705, -2.417421579360962, 5.012923717498779, -2.60076642036438, -0.16130943596363068, 3.5143015384674072, 0.045067865401506424, 3.937220335006714, 1.8357374668121338, 1.272359848022461, -3.6221489906311035, 2.4624834060668945, 0.34966325759887695, 2.337782859802246, -0.7362481355667114, -0.3460470139980316, 3.334035634994507, 2.186403274536133, -3.7206060886383057, 1.5644874572753906, -2.762110710144043, -5.204916954040527, 4.97892951965332, -0.36591386795043945, 3.007655143737793, 0.6045597791671753, -1.3064332008361816, -1.3251824378967285, -2.6767144203186035, -1.5930252075195312, -8.621811866760254, -1.1133928298950195, 0.2477284073829651, -2.5924649238586426, -4.517000675201416, -0.8228271007537842, -1.2280645370483398, 0.7557847499847412, -2.8970847129821777, -1.7609254121780396, -5.178224563598633, -3.0450665950775146, -0.051165223121643066, 1.4014995098114014, 6.20603084564209, -3.1096675395965576, -1.2094058990478516, -1.147855520248413, 0.9159992933273315, 1.5035414695739746, 2.174471378326416, 2.007153034210205, 3.133113145828247, -3.1979410648345947, -5.205874919891357, 3.370609998703003, -2.31811785697937, 0.3343057632446289, -2.5570764541625977, 1.7845977544784546, 0.9926064014434814, 0.8023925423622131, 0.6182291507720947, 0.9870591759681702, 0.3887849450111389, 0.8180882334709167, 7.369440078735352, 2.130648136138916, 0.11526969075202942, -3.232654094696045, 6.268929481506348, -1.4657076597213745, 6.808598041534424, 0.47856974601745605, -3.361022472381592, 5.567412376403809, -0.3033876121044159, -0.04459407925605774, 5.803104877471924, 3.8621292114257812, 0.9681656360626221, -1.2874623537063599, 0.7772359848022461, 0.8425764441490173, -1.6021181344985962, 4.119341850280762, 4.99104642868042, 1.6823946237564087, -0.012139849364757538, 1.8092796802520752, -0.7554754018783569, 5.00213623046875, 2.349827289581299, 1.029909372329712, -0.5819005966186523, 1.6619839668273926, -3.2527613639831543, 3.143634080886841, 4.5355706214904785, -3.828460216522217, -3.263070821762085, 5.748020172119141, 2.4797534942626953, 1.2603358030319214, 0.5461068749427795, -0.7022126913070679, -5.127688407897949, 0.4346315264701843, 0.08533447980880737, 2.6425259113311768, 1.6570305824279785, 4.1303205490112305, 3.4212284088134766, -0.3595430254936218, -1.890263557434082, -1.8088151216506958, -1.3789963722229004, -2.5823686122894287, 0.5582417845726013, -1.2480676174163818, 5.766416549682617, -1.2794544696807861, -5.724419116973877, 5.028527736663818, -2.876856565475464], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.6517911553382874, -0.17471729218959808, 1.6637239456176758, 0.6591477394104004, -1.64139723777771, -0.00520326429978013, -0.6234637498855591, 0.1486315280199051, -1.6081877946853638, 0.2417718768119812, 0.23538091778755188, 1.575626015663147, 0.3166450262069702, 0.5105466842651367, -1.4931167364120483, 2.2527291774749756, -1.9156455993652344, 1.101801872253418, -0.32990407943725586, -0.8806465268135071, -0.6562849879264832, -0.6720146536827087, 0.38018909096717834, -0.11005869507789612, 1.4825711250305176, -1.8296040296554565, -0.0030809645541012287, -0.8920558094978333, 0.7758718729019165, -2.1180601119995117, -0.3436943590641022, 0.21017292141914368, -1.4842685461044312, 0.9852156043052673, 0.17871342599391937, 1.0068567991256714, 0.9591848850250244, -0.9798545837402344, -0.7977957725524902, -0.20333248376846313, 0.7479597330093384, 0.850967288017273, -0.7096848487854004, -0.6071971654891968, -0.797822892665863, -0.5842382311820984, -0.23792338371276855, -0.13181504607200623, 2.058114528656006, -0.5064036846160889, -0.2887243628501892, 0.4585777223110199, -0.953074038028717, -0.3686271607875824, 0.013318480923771858, 0.7741459608078003, -1.3160148859024048, 1.371469497680664, -0.3524573743343353, 0.1694164276123047, 0.8470882773399353, 0.6607929468154907, 1.0592318773269653, 0.17319782078266144], [-0.01961335353553295, 0.31647855043411255, -0.9957153797149658, 1.2141873836517334, -0.7753862738609314, -1.2602438926696777, 2.0564088821411133, -0.13644494116306305, -1.1791248321533203, 1.8522697687149048, -0.3297126293182373, 1.0620757341384888, -0.8294886946678162, -0.2487800270318985, -1.688529372215271, -1.9089393615722656, -0.976434588432312, -0.0780390202999115, 0.7979857325553894, 2.426220417022705, -1.0318629741668701, 1.009040355682373, -0.6043409705162048, -0.1525292992591858, -1.481191635131836, -1.4789857864379883, -0.7784606218338013, -1.1931105852127075, -0.0035543907433748245, -0.7025144696235657, 1.2506970167160034, 0.1530197113752365, 0.1652381867170334, 1.1374351978302002, 0.5181122422218323, -0.03733298182487488, -0.09072335809469223, -0.12094863504171371, 1.0767219066619873, 0.7830856442451477, 0.36591795086860657, 0.7988293170928955, -0.5445333123207092, 0.5110228061676025, -1.412333607673645, 1.3472706079483032, 0.09004826098680496, 0.18339300155639648, 0.5993838310241699, -0.009709574282169342, -0.25410789251327515, 1.1897846460342407, 0.11575067043304443, -0.9158716797828674, 0.4395589232444763, -1.7051204442977905, 1.6555424928665161, -0.17113111913204193, 0.04588385298848152, 2.257770538330078, 2.1517410278320312, 0.11580667644739151, 0.6248663663864136, 1.0901799201965332], [1.6820210218429565, -0.13109290599822998, 0.135260209441185, 1.2246341705322266, 0.26955029368400574, 0.7544998526573181, 0.6098595261573792, -1.1349118947982788, -0.8589410185813904, -2.64133882522583, -0.260120689868927, 1.3388373851776123, -1.1092067956924438, 1.233803629875183, 0.9995461702346802, -0.054842185229063034, -0.007611643988639116, 0.08283314108848572, -0.861086905002594, -0.28233039379119873, -0.04656430706381798, 0.611937940120697, 0.7918128371238708, -0.7427380681037903, 0.8257131576538086, -0.7778705358505249, -0.16891653835773468, 0.3695053458213806, -0.022653397172689438, -0.12696082890033722, 0.26944124698638916, -0.5527990460395813, -0.6526661515235901, -0.20411410927772522, 0.9612712264060974, -0.10166808217763901, 1.9045213460922241, 0.2142532765865326, -2.974398612976074, -0.43955370783805847, 1.6650340557098389, 0.006345835048705339, -0.43783366680145264, -0.8736597299575806, 1.3237978219985962, 1.0028245449066162, 0.6720348596572876, 0.42810192704200745, 1.440906286239624, 0.8212016820907593, -1.2298004627227783, 0.7684435248374939, 0.16962634027004242, -0.11031979322433472, 0.7633881568908691, 0.02838650532066822, -2.9335548877716064, 1.5573880672454834, 1.037719964981079, -0.12258922308683395, -0.3723966181278229, 0.22664842009544373, -1.2843937873840332, 0.3064056634902954], [-0.23019705712795258, -0.3529508411884308, -0.923118531703949, -0.6786361932754517, 1.106296420097351, 1.3350210189819336, 0.5028380751609802, -0.04553661867976189, -0.877903401851654, 0.7177537083625793, 1.610936164855957, -1.6145039796829224, 1.1742593050003052, -2.1656696796417236, -0.34000906348228455, 0.3846246004104614, -1.3649871349334717, -1.5169907808303833, 0.1856737732887268, 0.8767150044441223, -0.47545304894447327, 1.343891978263855, -0.7799220681190491, 1.7543513774871826, 0.12040501087903976, 0.23530331254005432, 0.6149356961250305, 0.32102686166763306, 1.8457802534103394, 2.2064831256866455, 0.5461177825927734, -0.16570477187633514, -1.176100730895996, -0.7543747425079346, 0.009248188696801662, -0.10133885592222214, -0.043673597276210785, 0.005048451013863087, 0.49553972482681274, -1.3506871461868286, -1.0254756212234497, 1.1219208240509033, 0.8382925391197205, -0.5357815027236938, 1.295333981513977, -0.6763912439346313, -0.2517920732498169, -1.825471043586731, 0.8471222519874573, -0.008751091547310352, 1.069783091545105, 0.7092646360397339, 0.0994710698723793, 0.16600695252418518, 1.202069640159607, -0.1548120379447937, -0.142278790473938, 0.08018752187490463, 0.8187363147735596, 0.9381846189498901, 1.9344972372055054, -0.05894193798303604, 1.2884653806686401, -0.6258324384689331], [0.6350175142288208, 0.1596672236919403, 1.7719188928604126, 0.6483385562896729, -0.4074447751045227, 0.6204893589019775, 0.3898468315601349, -0.27225053310394287, -0.24905620515346527, -0.8976904153823853, 2.8043320178985596, 0.18290133774280548, 1.0226532220840454, -1.116990566253662, -1.0368651151657104, -0.10387420654296875, -0.09927146881818771, 0.8522030115127563, -0.9537264704704285, 0.5900864601135254, 1.5655534267425537, -0.8636458516120911, 0.9604621529579163, -0.9099019765853882, 0.3969624936580658, 0.41418585181236267, 1.788287878036499, -0.6398277878761292, 0.21922658383846283, 1.0945159196853638, -1.1212269067764282, 0.04168013483285904, -0.819409966468811, -0.6514171957969666, 0.7324393391609192, 0.7509714365005493, 2.628106117248535, 1.4274554252624512, 1.3393956422805786, -0.11676380038261414, -0.25091788172721863, 0.18602922558784485, -2.2267565727233887, 1.7047678232192993, -0.6246207356452942, -1.0151551961898804, -0.41472163796424866, 0.9168834090232849, 1.2834434509277344, -0.3383313715457916, 0.4730301797389984, -0.021620124578475952, -0.962156355381012, 0.20267261564731598, 0.05913431569933891, -1.507153868675232, 1.1297919750213623, -1.5431582927703857, 0.261273056268692, -1.0235143899917603, -0.7883965373039246, -0.6774150729179382, 0.72923344373703, -1.2306691408157349], [-0.1676996648311615, 1.574211597442627, -0.6745728254318237, -0.011189796030521393, 0.6841811537742615, -0.5636994242668152, 0.493717223405838, -0.64302659034729, 0.7172439694404602, 0.0794234424829483, -0.9195491671562195, 1.4256374835968018, -0.0785793885588646, -0.2637220025062561, -0.9088499546051025, -0.4542069733142853, -0.0807734802365303, 0.18730174005031586, 1.113384485244751, -1.3032360076904297, -0.3348938524723053, 1.2104649543762207, 0.19580209255218506, 0.4219990074634552, 1.6066585779190063, 0.350972056388855, 0.47962820529937744, -0.027718929573893547, 1.7806576490402222, 1.0627810955047607, -0.7088897824287415, 0.5604350566864014, -1.1366225481033325, 1.0983202457427979, 0.6820676922798157, -0.1501685380935669, -0.39291104674339294, -1.78682279586792, 0.7709866166114807, -0.11046920716762543, -0.30567485094070435, 0.7284855246543884, -0.6602470874786377, -0.5830860137939453, -0.6914806365966797, -1.3511924743652344, 2.4107327461242676, -0.1940978616476059, -0.3505159616470337, 0.899648129940033, -3.1509628295898438, -0.03327301889657974, 0.009213264100253582, -0.9763487577438354, 0.4531588554382324, -2.035080909729004, -0.00905892625451088, 0.860968828201294, -2.2549219131469727, -1.4915499687194824, -0.21255236864089966, -0.08649378269910812, -0.7290129661560059, 0.2673395276069641], [0.13792045414447784, -1.4635001420974731, -0.22432641685009003, -1.4088387489318848, -2.2176811695098877, -2.5605154037475586, 0.8732165098190308, -0.28830626606941223, 1.7932034730911255, -1.6309247016906738, -0.5253295302391052, 0.16670386493206024, -0.3159458637237549, 0.3289758861064911, -0.024797827005386353, -0.7837550640106201, -2.4976282119750977, -1.114784836769104, 1.5594944953918457, 0.6082985401153564, 1.2233881950378418, -0.7364241480827332, -0.3381264805793762, -1.0197584629058838, 0.2523939907550812, -0.07351211458444595, 0.32040631771087646, 1.6324464082717896, -0.9643092751502991, 0.6115328073501587, -1.5280566215515137, 0.9799928665161133, -0.5650072693824768, -0.19473381340503693, 0.014454003423452377, 1.5250083208084106, -0.15312272310256958, -0.6194480061531067, -1.3061418533325195, 0.15439479053020477, -0.9767104983329773, -0.6251292824745178, -0.14722518622875214, 0.5501624345779419, -0.8369993567466736, 0.5488540530204773, 1.649262547492981, -1.6116336584091187, 1.2413233518600464, 2.0329701900482178, -0.7120074033737183, 0.4937293529510498, -1.4944212436676025, 0.22011545300483704, -0.4456501305103302, 0.18508723378181458, -1.0388376712799072, -0.9445831179618835, -0.5717085599899292, -0.8385680317878723, -0.7484551668167114, -0.33476921916007996, -1.2459324598312378, -0.28374379873275757], [0.3418424725532532, -2.4557788372039795, 0.05376087874174118, -0.8923131823539734, -1.2692457437515259, -1.4974932670593262, -1.0065075159072876, -0.9047656655311584, 0.18334238231182098, -1.4267792701721191, 0.8781827092170715, -0.9830530285835266, 0.5210263729095459, 0.9275922775268555, -1.3804208040237427, 0.9191116094589233, 2.043945074081421, -0.571794331073761, 0.4664337933063507, -0.23452316224575043, -0.013529309071600437, -0.27494993805885315, -0.5803139209747314, 0.6552587747573853, 0.8204806447029114, -0.05562277510762215, 1.465050458908081, 1.0454494953155518, 0.664487361907959, -0.7530927658081055, 1.8713735342025757, 1.2943223714828491, -2.1742208003997803, -1.1311862468719482, -0.49832385778427124, 0.9203640818595886, 1.2284717559814453, -0.3029577434062958, 0.23387475311756134, 0.6867866516113281, -1.2836401462554932, 0.5840255618095398, -0.31217166781425476, 0.18032720685005188, -0.05193053558468819, -1.1837236881256104, -0.9972339868545532, 0.04028345271945, 0.12867140769958496, -0.2641412019729614, 1.0624080896377563, -0.35878679156303406, -1.2175036668777466, -0.3456953167915344, -0.17534466087818146, -0.13114114105701447, 1.1635318994522095, 0.5108031034469604, 1.0547808408737183, 0.6244425177574158, -0.16389226913452148, 1.03900945186615, -0.258688747882843, 0.25090110301971436], [-0.998500406742096, 0.10473381727933884, -0.32048937678337097, 0.3766418993473053, 1.821940302848816, 2.3774471282958984, 1.025654911994934, -0.02200225368142128, 0.8332908749580383, 0.23758003115653992, -0.5994011759757996, -0.05441593751311302, -0.2414710372686386, -0.5954602360725403, 1.389548420906067, -0.5312719941139221, -0.4878784418106079, 0.7128491997718811, -0.6850510239601135, -1.1403285264968872, -0.841480016708374, 0.4919573962688446, 0.38664567470550537, 0.30959391593933105, -0.453536719083786, -0.9021627902984619, 0.9184895753860474, -1.4341645240783691, -0.20012585818767548, -1.3128063678741455, -0.3929530680179596, -0.13902996480464935, 1.2421635389328003, -2.852917194366455, 1.3821525573730469, -0.8681005835533142, 0.31279027462005615, 0.8243922591209412, -0.744997501373291, -0.06737960129976273, 0.5144665837287903, -0.1014498919248581, -0.7133065462112427, 0.21277637779712677, 0.1347413957118988, -1.4728920459747314, 1.3858156204223633, 0.2541366517543793, -2.393120050430298, -0.8004530668258667, 1.7509589195251465, 1.6196867227554321, 0.30309200286865234, -1.651560664176941, -0.20373058319091797, 1.0941351652145386, 0.42923709750175476, -0.296917587518692, -0.6996406316757202, 0.8631637096405029, -0.17404134571552277, 0.3626221716403961, 0.531775951385498, 1.6801828145980835], [0.22430631518363953, 0.3000706732273102, -1.74651038646698, 1.1536922454833984, 0.9306327700614929, 0.027342740446329117, -2.2312724590301514, -0.5222504138946533, 0.11706233024597168, 2.139956474304199, 0.6320170164108276, 0.8352308869361877, 0.8617412447929382, 1.376841425895691, -1.095662236213684, -0.8127559423446655, -1.1434448957443237, -0.24716682732105255, -0.3175815939903259, 0.13861298561096191, -0.9912421107292175, 0.12732145190238953, -0.82101970911026, 2.1509017944335938, -0.4971640706062317, 2.584930419921875, 1.6686891317367554, -1.1673105955123901, 0.9903547763824463, 0.6394600868225098, -1.5062772035598755, 0.640000581741333, -0.06043462082743645, 0.37895336747169495, -0.8245453238487244, -0.032452601939439774, 0.442485511302948, -0.15307483077049255, -0.5803561210632324, -0.1301661729812622, -0.9533864855766296, 0.3704303801059723, -2.536355972290039, 1.5367698669433594, 0.09073319286108017, 0.5015658736228943, 0.8825592398643494, -1.217719554901123, 2.3076090812683105, -1.0952460765838623, 0.4940866231918335, 0.3518518805503845, 1.340849757194519, 0.015679700300097466, -1.388330101966858, 0.34883275628089905, -2.4476072788238525, -0.1789979487657547, -0.4394744336605072, 1.583648681640625, -0.2608485519886017, -0.43224501609802246, -0.740471363067627, -0.8401613235473633], [-0.5382840633392334, 0.6154676675796509, -1.3065720796585083, -0.19611571729183197, -1.4536749124526978, -0.1556876003742218, 0.17414464056491852, -1.2265311479568481, -0.5656226277351379, 0.1116790696978569, 0.7782198190689087, 0.958109438419342, -0.10823607444763184, -0.9991463422775269, -1.6270428895950317, 0.7823622822761536, 1.3989601135253906, 0.36695998907089233, 1.569003701210022, -0.45514973998069763, -0.3934612572193146, 1.5737149715423584, -1.0127875804901123, 0.6056233048439026, 0.3863377273082733, 0.15290498733520508, 1.7363947629928589, -0.8829541206359863, -0.4308055639266968, -2.098907470703125, -1.053222417831421, -0.5164452791213989, -0.3906514048576355, -0.20533648133277893, 0.2990294396877289, -0.6718030571937561, -1.9897387027740479, 0.15182262659072876, 0.4218922257423401, -1.167873740196228, -0.7917993068695068, 1.7680031061172485, 1.0307691097259521, 1.0925626754760742, 0.6718866229057312, 0.14615051448345184, -0.5139859914779663, 0.5739297866821289, -0.200730100274086, -0.44390419125556946, -0.012523401528596878, -0.0822213739156723, 0.8927301168441772, -0.9659674167633057, 2.50480580329895, -1.0145426988601685, -0.4308573007583618, 0.9475775957107544, -0.8429614305496216, -0.4858073890209198, 0.027048384770751, -2.9853670597076416, -0.5113680362701416, -0.971637487411499], [-0.2493814378976822, 0.5014220476150513, 0.6430748105049133, 1.5115034580230713, 1.4521006345748901, 1.0407071113586426, -0.03902619704604149, 1.0511815547943115, -1.0008419752120972, -0.09575169533491135, 1.4512178897857666, 0.7985697388648987, -1.405401587486267, -0.5939798951148987, 0.9150710701942444, 0.6251831650733948, 1.0907542705535889, -1.7754569053649902, -1.2700579166412354, -0.2452031373977661, -0.9744390845298767, -0.021924108266830444, 0.023384414613246918, 0.36136916279792786, -0.11831995099782944, 0.5811876654624939, 0.8012047410011292, 0.17182983458042145, -1.2134807109832764, 1.319388747215271, 0.23831216990947723, 0.4465811550617218, 0.19302666187286377, -1.6024588346481323, 0.6033034324645996, -0.6402491927146912, -1.2091223001480103, 1.3147752285003662, -0.28822898864746094, 0.31009310483932495, 0.9171768426895142, 1.6490528583526611, -1.8068283796310425, -0.7302215099334717, 0.6992377638816833, -0.655087947845459, -3.266982078552246, 0.16260835528373718, 0.24436090886592865, 0.8172025680541992, 0.2489216923713684, -0.4266014099121094, 0.23299556970596313, -0.017353765666484833, -0.4912850260734558, 0.27561110258102417, 0.3758176863193512, -1.0029608011245728, 1.0515519380569458, 0.8516427278518677, 0.8102356195449829, -2.747506856918335, 0.1355685442686081, -0.10766950994729996], [-0.991314709186554, 0.6013715863227844, -1.4985389709472656, 1.2558213472366333, 0.4043612480163574, -0.493579238653183, 1.4489936828613281, 0.47952407598495483, 0.3513450622558594, -1.2969244718551636, 0.5442907810211182, 1.4315910339355469, 1.122946858406067, -1.2870787382125854, 1.4112497568130493, 1.624860405921936, 0.8613985776901245, -0.37917593121528625, -1.5068439245224, -0.9116092324256897, -0.5812304019927979, 0.27515706419944763, 0.5537098050117493, -0.7519630789756775, -1.6557376384735107, -0.4256852865219116, -1.3509544134140015, 1.3267661333084106, -1.2587836980819702, 0.46165093779563904, 0.8067646622657776, -1.0158286094665527, 0.9813439249992371, -0.17622923851013184, -0.8347627520561218, -0.9924518465995789, 0.06965366005897522, 1.0393328666687012, -1.9587538242340088, 0.5727066993713379, -0.199795201420784, -0.811048150062561, -0.15502026677131653, -0.06002044677734375, -0.10828954726457596, 0.7690322995185852, -0.9581045508384705, 1.9421019554138184, 0.8556663990020752, -0.6457382440567017, -1.9873558282852173, -0.06323305517435074, -0.45154353976249695, 0.7115729451179504, 0.6936917901039124, 1.897495150566101, -0.16112516820430756, 0.7700633406639099, -0.45305901765823364, -0.3999653160572052, 1.5251184701919556, -0.15826477110385895, -0.12652985751628876, -0.372982919216156], [0.31485411524772644, 0.5885961055755615, -0.13075260818004608, -1.1635242700576782, 0.33642086386680603, -0.8347936868667603, -0.3631742596626282, 0.21567034721374512, 0.25112563371658325, -2.34112811088562, 1.7185163497924805, 0.8080207705497742, 1.9559962749481201, 2.544229030609131, -0.9484394192695618, -0.5719333291053772, 0.24441663920879364, 0.9373209476470947, -0.5268083810806274, -0.3314462900161743, 1.0683927536010742, -0.9123714566230774, 0.1944064050912857, 1.0718390941619873, -0.28999122977256775, 0.7566619515419006, -0.14852678775787354, -1.636642336845398, -0.5885325074195862, -0.2493143528699875, -0.38857609033584595, -0.028433609753847122, 0.34126999974250793, 0.22846545279026031, 0.11266938596963882, -0.5082535743713379, -0.2986820936203003, -0.4582267105579376, -0.678790271282196, 1.29404878616333, -2.0083417892456055, 0.13540701568126678, -1.7528836727142334, -0.7146692872047424, 0.6885014772415161, -0.408258318901062, 1.280826449394226, -1.0874276161193848, -0.4360039234161377, -0.7810027599334717, -0.1378163844347, -0.47157981991767883, -0.5433803796768188, -0.08022476732730865, 0.9382159113883972, -0.02792220748960972, -0.1291312426328659, -0.48543494939804077, -0.9107785224914551, -1.6592645645141602, -1.264088749885559, -0.18403887748718262, -2.144617795944214, 0.4311109483242035], [-0.6539977192878723, -1.1777422428131104, -0.7862472534179688, 0.4208882451057434, 0.12973617017269135, 0.027185609564185143, 1.5396665334701538, 2.133976459503174, -1.4914040565490723, 0.6958007216453552, -0.9457076787948608, -1.5497461557388306, -0.683032751083374, 0.8811109662055969, -0.43786585330963135, -0.5920459032058716, 1.3152291774749756, 0.4274468719959259, -0.15070554614067078, 1.764712929725647, 0.44458723068237305, -0.7653193473815918, -1.1191133260726929, -0.4411645233631134, 0.1593409776687622, 0.9414842128753662, 0.8499870896339417, 0.11976394802331924, -1.1718817949295044, 0.9426450729370117, 0.10009041428565979, -1.0935513973236084, -1.24043607711792, 0.22070248425006866, 0.3430028557777405, 0.80997633934021, -1.2384294271469116, 1.9164962768554688, -1.8455731868743896, -0.05838962644338608, 1.3295844793319702, 1.0270769596099854, -0.9879001975059509, 0.5883155465126038, 1.2120684385299683, 0.7136140465736389, -1.0224370956420898, 1.6464797258377075, -1.6773860454559326, 0.002395205432549119, -1.1230067014694214, -0.47976526618003845, -0.3855418562889099, 0.4472455382347107, -1.3316494226455688, 0.1603763848543167, 1.246193766593933, 0.017694327980279922, 0.48454099893569946, 0.36932623386383057, -0.8167076706886292, -0.43937307596206665, -0.018025686964392662, 0.4140413701534271], [1.9808205366134644, 1.5781420469284058, -0.5381981730461121, -0.5048323273658752, -1.0924770832061768, -2.4034385681152344, 1.2511831521987915, 0.46509605646133423, -2.0359578132629395, -0.05619727447628975, 2.310926675796509, 0.4264736771583557, 0.44579747319221497, 0.14373105764389038, -0.8353951573371887, -1.2438520193099976, -0.8995811343193054, 0.09109167009592056, -0.2996174097061157, 0.6498637199401855, 1.7266138792037964, 0.5529460310935974, 0.8024965524673462, -0.4001161754131317, -1.3523653745651245, 2.150698184967041, 0.6963133215904236, 1.161105751991272, -0.4163956642150879, -1.7532471418380737, 1.3021107912063599, -2.216024398803711, -0.203194722533226, -0.5766212344169617, 0.1902502179145813, 0.7870512008666992, 0.7977679371833801, -2.163555860519409, 0.9684877395629883, -1.9659343957901, 0.6479964256286621, -0.4814930856227875, -1.1887954473495483, -0.5742208957672119, -1.376534104347229, -0.04845772683620453, 0.9190923571586609, 0.7777649760246277, 0.009958052076399326, 1.0823917388916016, 0.4787008464336395, 2.1565282344818115, 0.4936029613018036, -0.6365970373153687, -0.3264255225658417, -0.8031188249588013, -0.3281681537628174, -0.17894574999809265, 0.6644898653030396, 0.760880172252655, 0.021548079326748848, 0.3993336260318756, -0.7816249132156372, -0.5902450084686279], [-2.5370914936065674, -0.8232187032699585, -0.5398075580596924, 2.5780372619628906, -1.4102091789245605, 0.032143790274858475, -0.37265321612358093, -1.6397303342819214, 0.23075781762599945, -0.022397426888346672, -1.5203841924667358, -2.4061169624328613, -0.09394374489784241, -1.4899595975875854, 0.6510894298553467, -0.3136555254459381, 0.5108970999717712, -0.33943501114845276, 1.5419821739196777, 0.6785516738891602, 1.5305678844451904, 0.10123423486948013, -1.057013988494873, -1.4571579694747925, -0.5240218639373779, 0.2853142023086548, -0.6495400071144104, 2.492025375366211, -0.06969134509563446, 0.46645236015319824, 0.08646079897880554, -1.2774641513824463, 1.0847766399383545, 0.6693270206451416, 0.8926535844802856, -0.9380829334259033, 0.03507905825972557, -0.022120419889688492, 1.2134368419647217, -1.3874715566635132, 0.7107558250427246, 1.321585774421692, -0.3985714614391327, -0.7986955642700195, 1.3864192962646484, 0.2852390706539154, 1.1910886764526367, 0.8098971843719482, 1.1349947452545166, -0.14134418964385986, -2.1221983432769775, 0.655442476272583, -0.6518208980560303, 0.6804951429367065, -0.4883160889148712, 1.7165638208389282, -0.19969826936721802, 0.9015975594520569, 1.1896884441375732, -0.026515843346714973, 0.3888673186302185, 0.03823145851492882, -0.8929641246795654, 1.5617280006408691], [-0.8839654326438904, -1.551430106163025, -0.468553751707077, 0.18047592043876648, 0.11496160924434662, -0.30306947231292725, -0.42091500759124756, -2.1533117294311523, 0.6805887222290039, 1.8171292543411255, -0.7102707624435425, -1.8106591701507568, 0.8008977174758911, -0.14180471003055573, -4.056928634643555, -0.7399261593818665, -1.4625312089920044, -1.1742103099822998, 2.082556962966919, 1.052930474281311, -0.12358371168375015, 0.058492355048656464, 1.665522575378418, -1.08347487449646, 0.8117064237594604, -0.6075767874717712, 0.913720965385437, 0.28226473927497864, -0.7147799730300903, 1.1522921323776245, 0.6170856952667236, 1.954527497291565, -0.3615010380744934, 0.200712651014328, -0.6544866561889648, 1.31687593460083, -0.7251472473144531, 0.039694517850875854, 1.4034738540649414, 0.6023518443107605, 0.35431942343711853, 0.3290475904941559, 0.4040552079677582, 0.7218317985534668, -0.2625349462032318, -2.004173517227173, 1.4298079013824463, -1.0461386442184448, 1.952264666557312, 0.34388354420661926, 3.334820508956909, 1.2668548822402954, 0.4852364659309387, 0.9994964599609375, -1.3987770080566406, 0.22682730853557587, -0.3339264690876007, 0.058755647391080856, -2.0836102962493896, -2.240954875946045, 0.7756854891777039, 0.07585200667381287, -0.4362134039402008, -0.07668609917163849], [-0.6581864953041077, -0.48297417163848877, 0.7222656607627869, -0.23232555389404297, 0.7343197464942932, -0.7264589667320251, -1.8727902173995972, -0.6799766421318054, -0.14714378118515015, -1.1630852222442627, -0.40620434284210205, -0.5283982753753662, 0.9556424021720886, 0.8656560778617859, 1.1279419660568237, -0.805351972579956, 0.341193288564682, 1.325830340385437, 0.17285242676734924, -0.08541935682296753, -0.8083139657974243, 0.8537705540657043, 0.17018039524555206, 1.025681734085083, -0.1693335473537445, 2.12789249420166, -0.03957027569413185, -1.287367343902588, 0.19928015768527985, 0.4883364737033844, 1.3198213577270508, -0.5068645477294922, -0.5758610963821411, -0.29270321130752563, -0.42117711901664734, -2.2112393379211426, 0.730895459651947, 1.284152865409851, -0.8035095930099487, -1.0358920097351074, -0.1604616940021515, -0.8581238985061646, 0.808384358882904, -0.6544860005378723, 0.6261543035507202, -1.293967366218567, -1.4702423810958862, -1.6149319410324097, -1.4760358333587646, 0.439557284116745, 0.871579647064209, 0.14714545011520386, 0.21869023144245148, -2.2875776290893555, 1.1497570276260376, -1.2808055877685547, 1.346217393875122, 2.4987308979034424, 0.06317181885242462, 0.6430473327636719, 0.6398900747299194, -1.3597071170806885, -1.1686592102050781, -0.7133437991142273], [0.15684774518013, 0.8682001829147339, 0.1191287711262703, 1.1380358934402466, -0.5256055593490601, -1.4597352743148804, 0.030655555427074432, -0.056686609983444214, -1.4643917083740234, 1.0622469186782837, -0.9843227863311768, -0.7449828386306763, 0.23148256540298462, -1.1070477962493896, 0.6417399644851685, 0.403968870639801, 0.4729851186275482, 0.002629642840474844, -0.4840308725833893, -0.7766695022583008, 0.3248544931411743, 1.4188708066940308, 0.8222401142120361, 0.39653539657592773, 1.049294352531433, 1.3901448249816895, -1.269675850868225, -0.017729377374053, 1.2847023010253906, -0.6394259929656982, -0.4494880437850952, 0.666745662689209, -0.19215987622737885, 1.279613733291626, -0.28203046321868896, 0.654703676700592, 2.625676393508911, 1.224740743637085, -1.3504133224487305, -1.0277502536773682, -0.9930204153060913, -0.008470739237964153, -1.3988908529281616, 0.6146456003189087, 1.8120763301849365, 1.6470540761947632, -0.7429807186126709, -0.2504066526889801, -0.27368345856666565, -0.8156312108039856, 1.7756304740905762, -0.6753050088882446, -1.1158865690231323, 0.39783775806427, 0.3694157004356384, 1.5914860963821411, 1.6001797914505005, -0.16586922109127045, -2.129179000854492, 0.7127141356468201, 0.8455799221992493, 1.857504963874817, -1.599302887916565, -1.1836820840835571], [1.6045761108398438, 1.6747850179672241, 1.3372012376785278, -0.019490348175168037, 1.3234343528747559, -0.5689653754234314, -1.2793643474578857, -0.0698021873831749, 0.09744080901145935, 0.8151116371154785, 1.1072620153427124, -1.8200533390045166, 0.2723481059074402, 0.47940734028816223, -1.316344141960144, -0.2948501408100128, 0.032063692808151245, 0.42941826581954956, -0.9083303809165955, -0.6448862552642822, -1.1592937707901, 0.3161161541938782, -0.3737691342830658, -0.6360786557197571, 0.6202044486999512, -0.20963653922080994, -1.2628284692764282, 1.1241353750228882, 0.958145260810852, 0.4314214885234833, -1.098345398902893, 1.0942116975784302, -0.639690637588501, -1.9499764442443848, -0.16987693309783936, 1.7996141910552979, -0.49854493141174316, 0.5580097436904907, 1.3558454513549805, 0.14790241420269012, -0.35983696579933167, -0.02807418815791607, 0.5508117079734802, -0.5598260164260864, 0.3134033977985382, 0.5249447822570801, -0.42262017726898193, 0.6898813247680664, 0.5293309092521667, -0.36958083510398865, -0.2272946834564209, 0.07392840087413788, -0.1904640644788742, -0.2510813772678375, -1.0744822025299072, 0.06986108422279358, 0.7054221034049988, 0.6088868975639343, -0.8685728311538696, 0.5297482013702393, 0.5792762041091919, -0.9667626619338989, 0.4907868206501007, -0.21028879284858704], [-0.4096407890319824, 0.19949021935462952, -0.24571208655834198, -2.277491569519043, 0.17691297829151154, -1.160054326057434, -0.12884896993637085, 0.6482376456260681, -0.2724015712738037, -0.48662441968917847, 0.09797091782093048, 0.4575705826282501, -0.8787835240364075, -0.6738266944885254, 0.45920586585998535, 1.5609488487243652, -0.042902275919914246, 0.8358983993530273, -0.784143328666687, -0.03466227650642395, 0.02759763039648533, -1.3328070640563965, -0.06473182141780853, -0.23466436564922333, -2.6006031036376953, -0.1424185186624527, 0.23715932667255402, 0.1806858479976654, 0.2557254433631897, -1.177059531211853, -0.8042497634887695, -0.2641931474208832, -0.9400496482849121, -0.5432702302932739, 1.1395519971847534, -0.9584609270095825, -0.41962727904319763, 1.7463783025741577, -0.20936864614486694, -1.0716692209243774, 0.1851881742477417, 0.11130169034004211, 0.07326280325651169, -1.7104793787002563, 0.2247437685728073, 0.3241453766822815, 1.99857759475708, 0.6361198425292969, -0.46373558044433594, 0.28605759143829346, 0.6435467004776001, -0.1608121246099472, -0.05336722731590271, 1.349284291267395, 0.4833506643772125, 0.5147797465324402, -0.9465725421905518, 0.43958786129951477, -1.8620463609695435, 1.2440277338027954, -0.4273810088634491, 0.6641139984130859, 1.4225069284439087, -1.0297907590866089], [1.194760799407959, 0.27660447359085083, 1.195839285850525, -0.29973554611206055, -0.6215580105781555, 1.344626784324646, 0.48329803347587585, 0.11822215467691422, -1.062160611152649, 1.3768185377120972, -0.2579600512981415, -0.14429432153701782, -0.6111034154891968, 0.08294723182916641, 0.6578225493431091, 0.009341505356132984, -0.017949266359210014, -0.47746407985687256, 1.5372143983840942, 1.6658936738967896, -0.2258560061454773, -0.8074666261672974, 0.3187408745288849, -1.5368279218673706, 0.29902565479278564, 0.3221258819103241, 0.21481800079345703, 1.622923731803894, 1.8610175848007202, 0.49520692229270935, -0.29966920614242554, 1.0679316520690918, 1.177173376083374, -1.1244443655014038, -0.665765106678009, -0.2025100141763687, -1.5847727060317993, -1.0447524785995483, 1.6622042655944824, -1.1893806457519531, 0.8827515244483948, -0.7410960793495178, 1.0291446447372437, 0.3698422908782959, 0.9378909468650818, -1.3129342794418335, 0.338229775428772, -0.5057026743888855, 0.8060771226882935, 0.7570741176605225, 0.22141170501708984, -0.07196210324764252, 0.04133280739188194, 0.2992013692855835, -0.44924086332321167, -0.3782009184360504, -0.5819128155708313, -0.9852313995361328, 1.0751851797103882, -1.248609185218811, 1.3263866901397705, 0.7449713945388794, -1.6313836574554443, -1.7527397871017456], [0.07044553756713867, -1.0008671283721924, 0.1829850673675537, 0.9793502688407898, 0.3970954418182373, -0.16605523228645325, -0.13316597044467926, 1.4933151006698608, -0.30669471621513367, 1.7558982372283936, 2.060687780380249, -0.30169111490249634, 0.35196375846862793, 1.749318242073059, -0.616104245185852, 0.4938150644302368, 1.1082431077957153, -0.9423758387565613, 0.4442552626132965, 0.21004733443260193, -0.4334500730037689, 1.6016985177993774, -0.04610934853553772, 2.330007553100586, 0.7942575216293335, -0.5991833209991455, 0.35299038887023926, -0.6573371291160583, 0.8994134068489075, 0.7441602349281311, -1.0707191228866577, -0.30972760915756226, 0.5618857145309448, 0.15486615896224976, -0.8635398149490356, 1.1650906801223755, 1.4314062595367432, -1.363048791885376, 1.534898281097412, -1.915714144706726, -0.052759237587451935, -0.5741614103317261, -0.4669197201728821, 0.2510126829147339, -1.0700734853744507, -0.031345125287771225, 1.7176891565322876, 0.5130857229232788, 1.5257980823516846, 0.9147213697433472, -1.523422360420227, 1.806976318359375, 0.9043644070625305, 0.712020218372345, -0.3002943694591522, -0.4425995945930481, -0.9585437774658203, -0.07926609367132187, 0.1806633621454239, -0.08449893444776535, -0.08079227805137634, -1.8343840837478638, -0.671749472618103, -0.7078303098678589], [0.3558412492275238, -0.38004636764526367, -0.3807245194911957, 0.14571541547775269, -0.4058477282524109, -0.2206432819366455, 1.1253091096878052, -0.10527173429727554, -0.8319641947746277, 0.6358107924461365, 0.8338444232940674, 0.6638489961624146, -1.2344887256622314, 0.9707334637641907, 0.09684234857559204, 0.18029873073101044, 0.13859429955482483, 0.5018673539161682, 0.5308799743652344, 1.490127682685852, 0.2977508306503296, -0.02305813878774643, 0.6862725019454956, -0.0413052923977375, 1.1229326725006104, 0.21459218859672546, 0.7864783406257629, -0.43539854884147644, 1.02769935131073, 0.6719330549240112, -0.7550415992736816, -0.29833418130874634, 0.14902286231517792, -1.2124539613723755, -0.9889416694641113, 0.5349758863449097, 0.5510594844818115, 1.3636003732681274, -1.183117389678955, -0.294556587934494, -1.9033114910125732, 1.750677466392517, -1.2226182222366333, -0.18249385058879852, 1.9188705682754517, 1.4350568056106567, -0.7848740816116333, 2.2336416244506836, 0.48557183146476746, -0.9479118585586548, 0.2991798222064972, 0.6630575656890869, 0.6509283781051636, -2.0159289836883545, 1.8170804977416992, -1.5160490274429321, -0.17281006276607513, 1.4208030700683594, 0.41412997245788574, -0.0019530453719198704, -0.554492712020874, 0.5903645753860474, 1.3694396018981934, -0.49913859367370605], [0.44451671838760376, -0.32575103640556335, -0.3694938123226166, -1.9714903831481934, 0.2888577878475189, 0.9499403238296509, -0.1006985604763031, 1.0615605115890503, 0.22071364521980286, 1.2405798435211182, 0.20273718237876892, -0.2920507788658142, -0.4171392321586609, 0.15460067987442017, 0.34883859753608704, -0.6892102360725403, -0.0770893469452858, -1.21988046169281, -1.2277617454528809, 0.6945211887359619, 0.4236812889575958, 0.9285114407539368, 0.21064399182796478, 0.31383004784584045, 1.8471705913543701, -0.7316835522651672, 0.35296332836151123, 1.5919983386993408, 2.0344436168670654, 0.617806077003479, -0.032039664685726166, -0.9420714378356934, -0.08071361482143402, 0.215158149600029, 0.08494379371404648, 0.5524805188179016, 0.6118045449256897, -0.38797640800476074, -0.7948181629180908, -0.21808628737926483, 0.4790209233760834, 1.6286793947219849, -1.5084378719329834, 1.0849438905715942, -0.21877385675907135, 1.106679916381836, 1.8609063625335693, 0.531453549861908, -1.045102834701538, -0.3517317771911621, 0.39515674114227295, -0.269873708486557, -0.020509611815214157, -0.697784960269928, -0.16930289566516876, -0.5552653074264526, 0.863319456577301, 0.33417826890945435, 0.7485215067863464, -1.5217533111572266, -0.3921799957752228, -0.6719117760658264, -0.37408122420310974, -0.8004969358444214], [-0.6542180180549622, 1.5447204113006592, -0.22057731449604034, -1.16388738155365, -0.24575848877429962, 2.1860015392303467, 0.7113757729530334, -0.852151095867157, -0.20420971512794495, -0.03491653501987457, 0.0670107752084732, -1.7363990545272827, -0.8727511763572693, -0.13044193387031555, -1.6152608394622803, -0.14554071426391602, 0.0971236601471901, 0.7687095403671265, 1.0425013303756714, -0.14851690828800201, 0.2437920868396759, 0.675127387046814, 0.996008038520813, 0.0998280942440033, -0.6882039904594421, 0.5946337580680847, 0.8431693911552429, 0.17050230503082275, 0.5768548846244812, 1.5503178834915161, -0.44924846291542053, -0.6378794312477112, -1.0032347440719604, 0.36632677912712097, 0.937567949295044, -0.4488820433616638, 1.7624857425689697, -0.5192183256149292, 0.2944362759590149, -0.07171712070703506, 1.359406590461731, -0.5618307590484619, 1.612497329711914, -0.12603510916233063, -1.2383230924606323, -0.5023880004882812, -0.11242654919624329, 1.64452064037323, -0.4161367118358612, 0.031733445823192596, 2.02093768119812, 0.18873322010040283, -0.562533438205719, -0.6687373518943787, -1.453460931777954, 0.37230414152145386, 1.187497615814209, -0.31722643971443176, -0.9802911877632141, 0.2870371341705322, 0.02936454676091671, -0.113981693983078, 0.4456055760383606, 0.044735174626111984], [-0.33686405420303345, 0.9647309184074402, -1.7639986276626587, -1.0740947723388672, -0.26381799578666687, -1.2936110496520996, 1.4379055500030518, -2.0984232425689697, 0.6644913554191589, 0.5333335995674133, 0.315697580575943, 0.39486780762672424, -0.5581148266792297, -0.702983558177948, 1.3413875102996826, -0.1568915694952011, -1.4180808067321777, -1.2013001441955566, -1.4923855066299438, 0.23786577582359314, 1.0764158964157104, -0.3181138336658478, 0.11783542484045029, -0.5011501908302307, 0.8371400237083435, 0.8303892612457275, 0.8806267976760864, 1.7203352451324463, -1.1765276193618774, 0.7737180590629578, -1.4766135215759277, 0.705592930316925, 0.7988339066505432, 0.7172263264656067, 0.7969013452529907, -0.09207332879304886, -0.5692492127418518, 0.772386372089386, -0.5555465221405029, 0.7228623032569885, -0.7619388699531555, 0.37185928225517273, 0.32558226585388184, -0.7450562715530396, -0.25173768401145935, 0.9267471432685852, 0.08572368323802948, 0.5522924661636353, -0.46480533480644226, 0.16611385345458984, -1.3402923345565796, -0.6761208176612854, 1.2465225458145142, 0.5779885053634644, 0.07237755507230759, 0.19044731557369232, -1.2195439338684082, 0.4548109173774719, -1.1084253787994385, -0.6401540040969849, 0.7106108665466309, -0.6526829600334167, -0.4429561197757721, -0.041313137859106064], [0.1337144523859024, 1.3093301057815552, -1.5864766836166382, -0.3577919006347656, 0.5854482650756836, -0.4398266077041626, -0.35245412588119507, -0.7525768280029297, -0.27547135949134827, 0.6300758719444275, 1.1931036710739136, 0.345582515001297, 1.3191388845443726, 0.10953973233699799, -1.5579413175582886, 0.728140115737915, -1.8989514112472534, -0.04428429901599884, 0.009725349023938179, -1.0650272369384766, -0.18504704535007477, -0.5067101120948792, 0.14224094152450562, -0.2776812016963959, 0.5861374139785767, -0.3499591648578644, 0.5847291946411133, 0.008836092427372932, 0.2409697324037552, 0.12265034019947052, -0.44015344977378845, 1.065782070159912, 0.5615694522857666, -2.1333682537078857, -1.1429105997085571, 1.6184803247451782, -0.251345694065094, 2.6632230281829834, 0.42048531770706177, 0.4391385614871979, -0.42586812376976013, -0.19727730751037598, 0.16147716343402863, 0.8697928786277771, 0.1011873334646225, -0.03285270556807518, -0.923890233039856, -0.4377623200416565, -0.4019087255001068, -0.12847210466861725, -0.701387882232666, -1.0439695119857788, -0.31254056096076965, -2.344146728515625, -1.454393982887268, -0.8162989616394043, 0.31928151845932007, -0.713323712348938, -1.0920404195785522, 0.5623684525489807, -0.20293481647968292, 1.1939750909805298, -0.29317837953567505, -0.7618297338485718], [-0.4487181007862091, 1.5605541467666626, 1.338356852531433, 0.036332253366708755, 0.8642590045928955, -1.4204562902450562, 1.0612678527832031, 0.20219485461711884, -0.20243234932422638, -0.10097043216228485, 0.11011054366827011, -0.5542452931404114, -0.4936358332633972, -1.2809019088745117, 1.3181291818618774, -0.9945483207702637, -0.31253910064697266, 0.791681170463562, 0.9644564986228943, 1.1556925773620605, -0.7497903108596802, -0.6211870908737183, -0.2766314446926117, -1.9545756578445435, 0.5958967804908752, -1.7667171955108643, -0.36430394649505615, 0.6031523942947388, 0.5375517010688782, -0.6689380407333374, 0.5989653468132019, 0.7519721984863281, -1.8457281589508057, -0.5096639394760132, 0.3543342351913452, 0.14266063272953033, -0.46484091877937317, -0.3674616515636444, 1.9051783084869385, -0.6065216064453125, 0.7322214245796204, -1.3783154487609863, 0.2386884093284607, 0.7227267622947693, -0.6362733840942383, -0.8202070593833923, 1.5132651329040527, -0.40041670203208923, 0.7570304274559021, -1.552558183670044, 1.4157291650772095, -2.0378329753875732, -1.0297116041183472, -0.7127087116241455, -0.19682228565216064, 0.04929313436150551, 1.3809823989868164, 1.8469058275222778, 1.6633180379867554, 0.6798480749130249, 0.10890147089958191, -2.0305399894714355, -0.16070452332496643, -0.6238964200019836], [0.5149666666984558, -1.0224210023880005, -0.8549779057502747, -0.7686167359352112, 0.5409824252128601, 0.09486845135688782, -1.0297296047210693, -1.6911461353302002, -0.6009995341300964, -0.6665095686912537, -0.5817986130714417, 0.012275384739041328, 0.6210324168205261, 0.6754908561706543, -0.6955143809318542, -0.7795884609222412, 1.368773102760315, -0.28108176589012146, 0.628198504447937, 0.1318901926279068, 0.3161768615245819, 1.464065432548523, -0.25831034779548645, 1.0090714693069458, 0.4744827449321747, -0.16211427748203278, -0.45971810817718506, -0.5164734125137329, 1.0748565196990967, 0.9231835007667542, -0.8114818930625916, 0.3683984577655792, -1.722678303718567, 0.04549231752753258, 1.932561993598938, -1.1327754259109497, 0.2820988595485687, -0.5869224667549133, 0.07804211974143982, -1.0060886144638062, 0.17372587323188782, -2.21146559715271, -0.25099748373031616, -0.30847418308258057, -0.0982418805360794, 1.9015392065048218, -1.1123886108398438, -0.6708791255950928, -1.3130199909210205, 0.45260611176490784, -0.010404611006379128, -1.1416927576065063, 0.85713791847229, 0.2301211804151535, 1.3263911008834839, 0.6882798671722412, 0.04537045955657959, -0.7792214751243591, -2.0847864151000977, -0.6125203967094421, 1.030393362045288, -0.273001492023468, -0.8540878891944885, 0.18126124143600464], [0.6459529399871826, 2.5945746898651123, -0.9979245662689209, -0.39223209023475647, -1.144885778427124, -0.759040355682373, 0.804496169090271, -0.171786829829216, -1.4629578590393066, 1.2023247480392456, 1.6156302690505981, 0.6829213500022888, -1.0243282318115234, -1.0976706743240356, 1.1507086753845215, -1.048475742340088, -0.6212055087089539, -1.476340413093567, 1.7447391748428345, -0.7267996072769165, -0.24944236874580383, -0.8034290075302124, -0.7407545447349548, -1.2098726034164429, 0.9639564752578735, -0.7521321773529053, 1.2939753532409668, -1.332101583480835, 0.35296744108200073, 0.8178145289421082, -1.216821312904358, 0.12127744406461716, -0.6609360575675964, -1.172697901725769, -0.5272700190544128, -1.9235047101974487, -0.40824252367019653, -1.6107395887374878, 0.5044994354248047, 1.3308440446853638, 1.2190049886703491, -0.8357124924659729, -0.2614522874355316, 1.7738224267959595, 1.101265788078308, 0.48207777738571167, -2.1787307262420654, -0.44703322649002075, 2.0152587890625, 0.32893452048301697, -0.5496657490730286, -0.03683273494243622, 2.203587532043457, 1.1325403451919556, -1.3577332496643066, -1.2167466878890991, -0.07500475645065308, 1.1423829793930054, -0.8504819273948669, -0.6697827577590942, -2.6019556522369385, -0.7286055684089661, 0.4575592577457428, 0.4588838517665863], [-1.563784122467041, 0.6307452321052551, -0.9577623009681702, -1.5402354001998901, 0.3271673023700714, -1.7657558917999268, -0.6372073292732239, -1.039345145225525, -0.3382653594017029, 0.01548489648848772, -0.6369341611862183, -1.026592493057251, -0.7104371190071106, -2.5539627075195312, -0.32754001021385193, 1.8152782917022705, 0.16520246863365173, 0.9610005617141724, -0.14694659411907196, -0.12774349749088287, 0.3045547306537628, 0.52989262342453, 0.501397967338562, -0.9407569766044617, -0.09624006599187851, -1.5807067155838013, 0.9059613347053528, -0.14209474623203278, -0.3875827491283417, -1.353060245513916, 0.3540743291378021, 0.8778784275054932, 0.39739829301834106, 0.9853181838989258, -0.8563488721847534, 0.8149518370628357, -0.09076306968927383, -2.7152669429779053, 0.1887567639350891, -1.0645633935928345, -0.2243192493915558, -0.7282349467277527, -0.9353457093238831, 0.9722643494606018, -0.6058982610702515, 0.1177617534995079, 0.6331967711448669, -0.4267060458660126, -1.2680426836013794, 0.025779657065868378, -0.12724623084068298, -0.9187394380569458, 0.08911020308732986, 1.2403210401535034, 0.9427482485771179, 1.5245082378387451, 1.3670929670333862, 0.19953878223896027, 0.02409357763826847, -1.0632140636444092, 1.1832237243652344, -0.5470365285873413, 0.5850498676300049, 1.6602468490600586], [0.1390886902809143, -1.450374960899353, -1.007551670074463, -0.3308548331260681, 1.3350363969802856, -0.3079076409339905, -0.12360046058893204, -1.0534597635269165, 0.35312291979789734, 0.10662614554166794, 0.8017815947532654, -0.9342911243438721, 0.6022279262542725, -1.4073841571807861, -0.6753049492835999, -0.558630108833313, -0.21668298542499542, 0.3738895356655121, -2.333953857421875, -0.11173960566520691, -0.42236992716789246, -1.3518913984298706, -0.33887603878974915, 0.40970122814178467, -0.48380258679389954, -0.5186631679534912, 0.6437298655509949, -0.7769017815589905, 1.017932653427124, 0.35706451535224915, 0.40561723709106445, 0.5709677338600159, 1.2256265878677368, -0.7204920649528503, 1.1299540996551514, -0.8828766942024231, 0.3630097806453705, 0.5980525016784668, 0.14708155393600464, -1.1016180515289307, -1.2042137384414673, -1.0936559438705444, -0.04256564378738403, 0.17596003413200378, 0.5589755177497864, -1.5078086853027344, 0.31968531012535095, 1.0204769372940063, 0.17157647013664246, 0.00018627224199008197, 0.8211896419525146, 1.5827009677886963, 1.48978853225708, 1.5406414270401, -1.2323044538497925, 1.4613349437713623, -0.27362188696861267, 0.8737511038780212, -0.554864227771759, 0.40827733278274536, -0.5256174206733704, -0.24041634798049927, 1.9533870220184326, -1.138882040977478], [1.0179572105407715, 0.11340604722499847, -1.0460208654403687, 1.3509917259216309, 0.08528634905815125, -0.49663418531417847, 0.38747623562812805, -0.7365942001342773, 1.7094897031784058, -1.4119067192077637, -0.44704410433769226, -1.9229083061218262, 0.3051687777042389, 0.8721323609352112, 0.9794617295265198, 1.595033884048462, 0.059426531195640564, -0.38054752349853516, 0.9851451516151428, -0.520288348197937, 0.4121715724468231, -0.3858761787414551, -0.6370244026184082, 0.030978607013821602, 1.1923854351043701, 0.9866033792495728, 0.2280595451593399, -2.502596616744995, 0.10621289163827896, 0.4764416515827179, 0.6402673721313477, -0.6063845157623291, 1.1016606092453003, 0.4332406222820282, -1.77329421043396, 0.8500957489013672, 1.7305316925048828, 0.37533554434776306, -0.6694790124893188, 1.9141908884048462, 1.1115094423294067, -1.6988108158111572, 1.1606943607330322, 0.5138728618621826, 0.3937334418296814, -1.6396005153656006, -1.5619773864746094, 0.2720009982585907, 2.004218816757202, 0.2638160288333893, 0.18140427768230438, -0.5254349708557129, -0.310617595911026, 1.1005152463912964, -0.6964213848114014, 1.9311860799789429, 0.532659649848938, 0.364359587430954, -0.08398845046758652, -0.6842778325080872, -0.2243642807006836, -1.4286856651306152, 0.07461576163768768, -1.8923650979995728], [2.17773175239563, -0.08649507164955139, -1.5544583797454834, 0.2111993283033371, 0.06469117850065231, -0.6094876527786255, 0.18932373821735382, -0.8698752522468567, -0.20401868224143982, -0.25759318470954895, 0.408180832862854, -0.6297513842582703, 0.20582132041454315, -2.8852169513702393, 0.9926231503486633, 0.10681323707103729, 0.47323673963546753, -0.3120717704296112, 0.8131988048553467, -0.09915035963058472, 0.1477489322423935, 0.026138486340641975, -0.5847780704498291, -0.9508326053619385, 0.489681601524353, 0.5896188020706177, -0.0849541425704956, 1.216679573059082, 0.1082240417599678, 0.4664347469806671, 1.0750641822814941, 0.695271372795105, -0.19475242495536804, -1.0956079959869385, -0.7702208161354065, -1.0567878484725952, 0.7772903442382812, -3.554746150970459, 1.9906444549560547, -0.8946410417556763, 1.0658806562423706, 0.2954869866371155, 0.5312764644622803, -1.2971631288528442, 1.028520941734314, -0.09369056671857834, 0.6235244274139404, -0.9303983449935913, -0.029990682378411293, -0.6974140405654907, 0.3193218410015106, 0.6895738840103149, -0.21552954614162445, 0.44939517974853516, 0.51973956823349, -0.053584177047014236, -0.2619827687740326, -1.9140862226486206, 1.0623514652252197, 0.17894986271858215, -0.2374780774116516, 0.21903833746910095, -0.508624792098999, -0.2998589277267456], [0.5044350028038025, -1.4916640520095825, -0.29455167055130005, 0.2528025805950165, -1.5410423278808594, -1.0637258291244507, 1.2620654106140137, 2.1931514739990234, 0.3244841694831848, 0.6922556161880493, -0.8235588073730469, 1.374129056930542, -0.017272112891077995, 0.13199499249458313, 0.44345736503601074, -0.5230931639671326, -0.6530646681785583, 0.013848617672920227, -1.0109742879867554, -0.5401878952980042, -2.5268619060516357, -0.9684253931045532, 0.488838255405426, 1.1266262531280518, 0.6299227476119995, -0.8036858439445496, 0.10141125321388245, -0.9163286685943604, -1.036463737487793, 0.9994712471961975, -0.35005730390548706, -0.30268269777297974, 0.20776070654392242, 0.9462091326713562, 0.1504984349012375, -0.2091352641582489, 0.368549644947052, -0.03908759355545044, -0.2617416977882385, -2.1370368003845215, -0.8101083636283875, -0.0683714970946312, -0.9445797801017761, 1.9048528671264648, 2.348064422607422, 1.7746473550796509, -0.793096125125885, 0.5879466533660889, 0.5189412832260132, 0.45907726883888245, -1.4562811851501465, 1.036228895187378, 1.799828052520752, 0.9347269535064697, 2.2269937992095947, -0.15812301635742188, 0.8229620456695557, 1.4250407218933105, -0.7627618908882141, 0.032040249556303024, -0.5845131278038025, 1.0272101163864136, -0.9589815139770508, 0.46754124760627747], [-0.3613424003124237, 1.0478401184082031, -0.34430113434791565, 0.010315949097275734, 0.41394901275634766, -0.5798282027244568, -0.2662961184978485, -0.07857925444841385, -0.33710598945617676, 0.40952593088150024, -0.6066229939460754, 0.22344177961349487, -0.17228877544403076, -0.5101341605186462, -0.9837468862533569, -1.7622995376586914, -0.6148024797439575, 0.6061075925827026, -0.28180837631225586, 1.525972843170166, -0.9271951913833618, -0.35227441787719727, -1.541527509689331, 1.126190423965454, -1.3194383382797241, 0.4337562322616577, 0.27929043769836426, 2.5522360801696777, 1.3065377473831177, 0.7744492292404175, 0.14071904122829437, 1.852089762687683, -0.34576016664505005, -1.2037783861160278, 0.8776870369911194, -0.044230300933122635, -0.671359658241272, 0.3544609546661377, 0.5200506448745728, -1.5053954124450684, 0.9767979979515076, 1.9802933931350708, -0.3611297309398651, -1.4554065465927124, 0.28413471579551697, -0.0939192995429039, 0.7950661182403564, 0.9391974210739136, -0.09059801697731018, 1.532605767250061, -1.868545413017273, -0.6619069576263428, 0.2135445773601532, 0.425455242395401, -0.05679866299033165, 0.4829036295413971, -1.2412060499191284, -2.243788003921509, 0.8566943407058716, 1.3657634258270264, 0.04931824281811714, -1.1050410270690918, -0.6441173553466797, 0.6512966752052307], [0.2652692198753357, 1.3594363927841187, 1.6920921802520752, 1.2844501733779907, -0.16425280272960663, 1.1007137298583984, 0.5644261837005615, 0.2714419960975647, 0.23800547420978546, -0.031042233109474182, -0.952472984790802, 0.41613128781318665, -0.7154602408409119, -0.004438161849975586, 0.8328277468681335, 0.8100336790084839, -1.7963809967041016, -0.6397698521614075, -1.056563377380371, 0.2965407371520996, -0.7096341848373413, 0.9482331275939941, -0.432424396276474, 0.4746869206428528, -0.9528499245643616, -0.7234006524085999, -0.26880955696105957, 0.43551844358444214, -2.0383670330047607, 0.07403415441513062, -1.444252848625183, 0.15992121398448944, -1.7892663478851318, -0.44549089670181274, 1.164401650428772, 1.4120866060256958, -1.0302605628967285, 1.6498724222183228, -0.45645871758461, -0.002623716602101922, 0.46181362867355347, 0.06280510872602463, 0.3082680106163025, 0.14072640240192413, 0.8060595393180847, 0.16527163982391357, -0.4867963194847107, -1.0528335571289062, -1.0938966274261475, -0.1487428843975067, 0.6367091536521912, 0.2440837323665619, -0.374321311712265, 0.3507709503173828, -0.6351199746131897, 0.04028337076306343, 0.37178969383239746, -0.44717133045196533, 1.2372543811798096, 1.0262069702148438, -0.36538147926330566, 0.5370815396308899, 0.11364755034446716, 0.24651333689689636], [0.24454563856124878, 1.227865219116211, 0.26019659638404846, 0.24169479310512543, 0.8706080913543701, 0.8389414548873901, 1.653521180152893, -0.8268328309059143, 0.6343085765838623, -1.8171908855438232, 0.016431685537099838, -1.2698054313659668, -1.300014853477478, -0.1294713169336319, -1.0593019723892212, -0.2953795790672302, -2.671903371810913, 1.3326524496078491, -2.6328935623168945, -0.6675107479095459, 1.6472702026367188, -1.3501017093658447, 1.6573699712753296, 1.0740697383880615, -0.08565919101238251, -0.35994240641593933, -0.284171462059021, 0.2693173587322235, 0.9559041857719421, 1.300061583518982, -2.0660789012908936, 0.8573405742645264, 0.61078280210495, -1.1997498273849487, 0.8814881443977356, 0.9182392954826355, 0.903153657913208, -0.058422964066267014, 0.7823741436004639, 1.2001852989196777, -0.6361798644065857, -0.48199716210365295, -0.5687602162361145, 1.165716290473938, 0.027820318937301636, -0.7104313373565674, -1.0033183097839355, -1.2480584383010864, 0.22305428981781006, 0.8284783959388733, 1.250952124595642, -0.9802612662315369, -0.9507783055305481, -0.9385566711425781, -0.8522101044654846, -1.492647647857666, 1.4891419410705566, 0.4373342990875244, -0.5216147303581238, -2.5370142459869385, 1.30644953250885, 0.7254448533058167, 0.08538614213466644, 0.6630703806877136], [-0.7113270163536072, 0.5921401977539062, -0.5786644816398621, 0.4062473177909851, -0.19288916885852814, -1.3304829597473145, -1.0995365381240845, 1.0141128301620483, 0.935150146484375, -0.22625797986984253, -0.7389237880706787, -0.4939488470554352, 0.3248478174209595, -1.402109980583191, 0.8995650410652161, -0.8522325754165649, -1.1588789224624634, -0.08074089139699936, 1.8312448263168335, -0.7398539185523987, -0.42191967368125916, -1.1967514753341675, 0.19633646309375763, 0.21877390146255493, -0.0610838383436203, -0.4336572289466858, 0.3111650347709656, 0.7377752661705017, -0.49347904324531555, -0.29908832907676697, -0.6871767044067383, -0.6072854399681091, 1.1176615953445435, -1.0506967306137085, 0.10900500416755676, 0.9535261392593384, -0.5669568181037903, 0.4979861378669739, 0.6537268757820129, -1.506990909576416, 0.8831441402435303, -1.0744162797927856, 0.04263150319457054, -1.585762619972229, -1.6770671606063843, -0.33119115233421326, 1.5095657110214233, 2.4439589977264404, 2.634476661682129, 1.2737997770309448, 1.1094286441802979, 0.7027309536933899, 0.6087801456451416, -0.4038937985897064, -0.49281972646713257, 0.37715989351272583, 0.42195379734039307, -0.6512054204940796, -1.2163630723953247, -0.7632922530174255, 3.2729103565216064, -0.7321873903274536, 1.3968441486358643, 1.1757279634475708], [1.0342050790786743, -0.6378816366195679, 1.9718178510665894, 0.26213645935058594, -0.6225683093070984, -0.7389327883720398, -0.16933660209178925, 2.070805788040161, -1.313761591911316, -0.7002822160720825, -0.020309163257479668, 0.45528602600097656, -1.0917638540267944, -1.0223976373672485, -0.5894440412521362, 0.6257040500640869, -0.2091183364391327, -0.26799142360687256, 0.15285003185272217, 0.06703397631645203, 0.8954302668571472, -0.7692878842353821, -0.04059527814388275, -0.6836593151092529, 0.6779800057411194, 0.7771325707435608, -0.28256022930145264, 1.133314847946167, -0.22839514911174774, 1.3503522872924805, -0.42053231596946716, -0.47340771555900574, -1.5125489234924316, -1.0431066751480103, -0.42202964425086975, -2.1419806480407715, -0.06228658929467201, 0.8072910308837891, 0.49392861127853394, 0.15237733721733093, 0.6259250640869141, -0.6049961447715759, -0.5162051320075989, 0.7889293432235718, -0.176864355802536, 0.9436370134353638, -0.5146250128746033, 0.6350678205490112, 1.1945393085479736, -0.12260175496339798, 1.650699496269226, 1.51536226272583, -2.4450113773345947, 2.6023061275482178, -1.6424120664596558, 2.4157204627990723, -0.9255171418190002, -2.1410374641418457, -1.3866381645202637, 0.42096221446990967, -1.0518169403076172, -0.6065541505813599, 1.1674516201019287, -1.0464043617248535], [0.45132964849472046, 1.2280012369155884, -1.8896901607513428, -1.2113397121429443, 0.17634855210781097, -0.045468367636203766, 0.24732734262943268, -2.040391445159912, -0.882433295249939, 0.358968049287796, 0.554478645324707, 1.2351161241531372, -0.41580942273139954, -0.7560499906539917, -1.0067261457443237, -0.9597005248069763, -2.0790419578552246, 1.6565247774124146, -0.48395851254463196, -0.7493939399719238, -0.6818282604217529, 0.5080596208572388, 0.050074126571416855, -0.8366444110870361, 0.947858452796936, 0.2061554193496704, -0.25268369913101196, -0.06898494809865952, 1.1157500743865967, 0.6338908076286316, -0.16376540064811707, 0.9703584313392639, -2.232139825820923, 0.6480903625488281, -0.9135241508483887, -0.6969531178474426, -0.3095930814743042, 1.807201862335205, -0.23227596282958984, -0.2119794487953186, 0.5291433334350586, -2.079483985900879, -0.30020561814308167, 0.6172598004341125, -1.0651119947433472, 0.5136927366256714, -0.8275601267814636, -1.5732711553573608, 0.8181838393211365, 0.017338348552584648, 0.6995562314987183, 0.11709807068109512, -1.1289362907409668, 0.9665898680686951, 0.378579705953598, 0.37453678250312805, -0.42201167345046997, -1.130639910697937, 1.4130445718765259, 0.7728211283683777, 0.17507626116275787, -0.5988548994064331, 1.3341851234436035, -0.8466301560401917], [-0.7653025984764099, 2.134082555770874, 0.6101077795028687, 0.9218233227729797, 0.7234711050987244, 2.118293285369873, 0.41660186648368835, 0.5139140486717224, -0.8560548424720764, 0.05907498300075531, -1.510719656944275, -0.3659335970878601, -0.005974443629384041, 0.730309009552002, 1.363209843635559, -0.6714779734611511, -1.0606197118759155, 0.27907803654670715, -0.8636341094970703, 1.6991536617279053, 1.266513705253601, -0.9706504344940186, 0.230197936296463, 1.2866748571395874, -0.9063968658447266, -0.48025649785995483, -0.07101643085479736, 1.3647757768630981, 0.4740580916404724, -0.2386937290430069, 0.32423412799835205, 0.03671567887067795, 0.17200244963169098, 0.47125160694122314, 0.2282903641462326, 0.6388665437698364, -0.0035331144463270903, -0.2953691780567169, -0.321569561958313, -1.6116198301315308, -0.5098362565040588, 1.7540695667266846, -0.08782520145177841, 0.47727781534194946, 0.8158150315284729, -0.5248494744300842, -0.3820326626300812, 0.46743717789649963, -0.970657467842102, 0.8434706330299377, 0.622447669506073, 0.18221515417099, 0.1984870582818985, 0.6743929386138916, 0.05373905971646309, -2.522955894470215, 0.45884400606155396, -0.9776561856269836, -0.5367795825004578, -0.46328920125961304, -0.45748764276504517, -1.701861023902893, -1.9258549213409424, -0.11008346080780029], [0.01511063240468502, 0.39358392357826233, -1.2821155786514282, -0.3193916082382202, 0.7691450715065002, -0.3771837055683136, -0.5905759334564209, 2.0101730823516846, -1.653630018234253, -0.799828827381134, -1.0908325910568237, -0.4976908266544342, 3.1448659896850586, -1.0306857824325562, -0.2706841230392456, -0.530424952507019, -0.7599231004714966, 0.8308976888656616, -0.4409581422805786, 1.0848530530929565, 1.4330800771713257, -0.45753270387649536, 1.4304332733154297, -1.1497831344604492, 0.36687299609184265, 1.0726262331008911, -0.9700847864151001, -0.007191389799118042, 0.5348916053771973, 0.6372493505477905, 0.4068767726421356, 2.0485870838165283, -0.07809954881668091, 1.2659608125686646, 2.1848859786987305, -0.5460962653160095, -0.34932586550712585, -0.9626780152320862, 0.7793025374412537, 1.4851199388504028, -1.6693166494369507, -0.8521966338157654, -0.5385360717773438, 2.2937533855438232, -0.29368409514427185, 0.45880427956581116, -1.1235946416854858, 0.9530777335166931, -0.9662408232688904, 0.014498711563646793, -0.9488843679428101, -0.009235391393303871, -0.06325589865446091, 0.306782990694046, 0.2932000160217285, 1.8878344297409058, -0.6466166973114014, 1.4125832319259644, 0.4030396044254303, -0.0874469205737114, 0.39531010389328003, 0.3436284363269806, 0.9355557560920715, 0.9014030694961548], [0.7030925750732422, 0.354044646024704, -1.4859064817428589, 0.9464613795280457, 1.2046705484390259, 0.47090432047843933, 0.4011225700378418, -0.7405857443809509, 0.3746853470802307, 0.9582241773605347, -1.206286072731018, -0.36951953172683716, -0.43416866660118103, 1.3219963312149048, -0.36300814151763916, -1.7184407711029053, 0.22245532274246216, 1.3859927654266357, -0.0911448746919632, -0.36552590131759644, 0.2800864279270172, 1.7444465160369873, -0.19062648713588715, -0.19479233026504517, 0.477705180644989, -0.7058749794960022, -1.6174408197402954, 0.21149806678295135, -0.29440534114837646, 0.42042630910873413, 0.1979341208934784, 0.586952805519104, -1.0094729661941528, 0.38367295265197754, -0.8423148393630981, 0.4095006287097931, -0.3680028021335602, 1.7242345809936523, -0.8222585320472717, -2.1910035610198975, 0.17904719710350037, -0.4053349494934082, -0.8649823665618896, 0.2579737901687622, 1.673122763633728, -0.12058116495609283, -0.654821515083313, 1.439102292060852, 1.1479191780090332, 0.04498792812228203, -0.5136915445327759, 0.729758083820343, -0.6684750914573669, -0.6055442690849304, -0.7955200672149658, 0.539344847202301, 1.4960845708847046, 0.06451791524887085, 1.7543017864227295, -1.9407761096954346, -0.030530713498592377, -1.6936043500900269, 0.8730499744415283, 1.2570077180862427], [-1.7778193950653076, -0.7769983410835266, 0.03004041686654091, -1.5037965774536133, -0.9072403311729431, -1.0443719625473022, 0.007306438405066729, -0.8178585171699524, -1.213813066482544, -0.8806869387626648, -0.21260546147823334, -0.03521569445729256, -0.30528852343559265, 1.071234107017517, -0.4816991686820984, -0.023107880726456642, 0.47014984488487244, -0.4598517119884491, 1.1980171203613281, 0.6406381726264954, 0.666985273361206, -0.9696097373962402, -0.23651884496212006, 0.4561689794063568, 0.9379943013191223, -0.7045305371284485, -0.5930295586585999, -1.3792128562927246, -1.5535537004470825, 0.9888076782226562, -0.08355198055505753, 0.4682712256908417, 0.9965956211090088, -0.08322730660438538, 0.057277191430330276, -1.062685489654541, -0.6355735659599304, 0.3745555281639099, 1.32062828540802, 0.9610233902931213, -0.7078297734260559, -0.7796693444252014, 0.6549652218818665, 0.4453355669975281, 0.5072994232177734, 0.23585963249206543, 1.2295080423355103, 0.2602604627609253, 1.4048316478729248, 0.527582049369812, 0.023452656343579292, -1.0806162357330322, 0.32209512591362, 0.8835400342941284, 0.18797512352466583, -0.929045557975769, -1.788906455039978, -0.2580881118774414, 1.7302638292312622, -1.071961760520935, -0.43808823823928833, 0.1799582839012146, 0.9550010561943054, -0.9333059191703796], [0.058406513184309006, 0.30296432971954346, -1.1207122802734375, 0.410995215177536, -0.980427086353302, 0.17978806793689728, 0.6231619715690613, 0.10108711570501328, -1.1887165307998657, 0.6884428262710571, 0.7158186435699463, 0.16481706500053406, 0.8116161227226257, -0.4264776110649109, -0.7857580184936523, -0.15784794092178345, 0.48407915234565735, 0.8819596171379089, -0.6555714011192322, 0.22287727892398834, 0.17485010623931885, -0.46016791462898254, -1.9420377016067505, -1.6457934379577637, 0.9132375121116638, -1.467771053314209, -1.74402916431427, -1.410599946975708, -0.34452030062675476, 1.2267228364944458, 0.34279948472976685, -0.4827176034450531, -0.3244844377040863, 0.40198633074760437, 0.02753700129687786, -1.7211086750030518, 1.160588264465332, 0.8714012503623962, -1.047785997390747, -0.24323470890522003, 0.4107038974761963, 0.2625584602355957, -0.3287716507911682, 0.06690409779548645, 1.7549947500228882, -0.28924185037612915, 0.9588476419448853, 0.25678783655166626, -1.1003743410110474, 0.640428364276886, 0.36717456579208374, 1.1558074951171875, 1.5810160636901855, 1.0582640171051025, 0.3502785563468933, 0.5615181922912598, 0.860477089881897, -0.7312160730361938, -0.9243530035018921, 0.7965637445449829, -1.0382075309753418, -0.22572341561317444, 1.773127555847168, 0.9259857535362244], [1.2459019422531128, 0.005370558239519596, 1.1563159227371216, -0.7713994383811951, -0.7442688345909119, -0.13435286283493042, -0.6506817936897278, 1.279180884361267, -0.13535110652446747, -1.1121679544448853, 0.882354736328125, 0.7109274864196777, 1.1630293130874634, -1.1359609365463257, -0.8165166974067688, -2.1784627437591553, 0.3680175542831421, -0.07755258679389954, -0.555927574634552, 1.2966305017471313, -0.7457011342048645, -0.502550482749939, -0.8723145127296448, -1.0159870386123657, -0.07704684883356094, -0.21828868985176086, -0.38761916756629944, -1.1270610094070435, -1.3564335107803345, -0.6460341215133667, -0.12744837999343872, -0.9146677255630493, -0.48847466707229614, -0.41783657670021057, -0.9866988658905029, -1.5356090068817139, -0.22405347228050232, 1.466928243637085, 0.39774757623672485, -0.17915168404579163, -0.7394818663597107, -0.6761707663536072, 0.9843191504478455, -1.1531169414520264, 1.7012395858764648, 0.03328324109315872, 0.29462379217147827, -0.3361751139163971, 0.9471736550331116, 0.9294290542602539, -0.6883518099784851, 1.8102684020996094, 0.21960623562335968, -1.4130669832229614, 0.5197658538818359, 0.35772576928138733, 0.22498133778572083, 0.889275074005127, 1.108497977256775, -1.107724905014038, -0.15363726019859314, -0.0940338522195816, 0.08702392131090164, -1.6992360353469849], [0.2901429235935211, 0.10476259142160416, 1.917757272720337, -0.784536600112915, -0.43113917112350464, 0.48223185539245605, 0.03423497453331947, 0.8994408249855042, 1.2265487909317017, -0.8197942972183228, 0.14912045001983643, 0.6783930659294128, -0.6899111866950989, 2.22320556640625, 0.3376694619655609, 0.20439574122428894, -1.4411991834640503, -0.6228167414665222, -1.909064531326294, 0.6928718090057373, -0.7456469535827637, 0.0976114422082901, 0.3915776014328003, 0.21206021308898926, -0.38248133659362793, 0.7418116927146912, -1.0831516981124878, -1.8266572952270508, -1.4432053565979004, 0.7318328619003296, 1.3696128129959106, -0.6715482473373413, -0.5536804795265198, 0.3708382248878479, -1.3389297723770142, -0.5255574584007263, -1.4486922025680542, 0.009107086807489395, -2.9180359840393066, -1.5470445156097412, -1.7777600288391113, -0.14170114696025848, 0.40070265531539917, -0.1115216538310051, 1.9471004009246826, -0.03813815489411354, -1.0200226306915283, -0.5319411158561707, 0.5785613656044006, 0.37446898221969604, 2.0454940795898438, 0.1280043125152588, -0.9921558499336243, -1.1465433835983276, 0.719286322593689, -2.814579486846924, 0.3162977993488312, 0.7620257139205933, -0.5083141326904297, 0.1717526614665985, 1.2213413715362549, 1.401712417602539, 1.3002771139144897, 0.910219669342041], [-0.21743284165859222, -1.2293932437896729, -2.142787218093872, 1.6996886730194092, 0.9703792333602905, -1.317464828491211, 1.0270339250564575, -0.48020902276039124, -0.5683815479278564, -1.2518651485443115, 1.9933620691299438, -1.4130455255508423, -1.1743355989456177, 0.6383433938026428, 0.770093560218811, 0.46775758266448975, -1.0703279972076416, -0.24911487102508545, -0.9058767557144165, -0.5385908484458923, 1.7521636486053467, 0.23539398610591888, 0.7457355260848999, -0.9357568621635437, -0.5746681690216064, 0.4108710289001465, 1.1649457216262817, -0.1878572553396225, 1.4584234952926636, 2.3580994606018066, -1.8095539808273315, -0.018501507118344307, 0.6306939721107483, -0.2430812567472458, 1.6374502182006836, 0.09382545948028564, 0.31361547112464905, -1.6153372526168823, 0.9619265198707581, -0.3534252941608429, 0.40514233708381653, -0.2953792214393616, -0.7252362370491028, -0.2068251520395279, -1.304510474205017, -0.28751546144485474, 1.0614547729492188, 0.9015455842018127, -1.006800651550293, 0.7869716286659241, -0.4286283850669861, -0.6884938478469849, 0.5438366532325745, 0.6678006649017334, -0.6715621948242188, 0.8858497142791748, -0.05566619709134102, 2.1001532077789307, -1.5255942344665527, -0.2777721583843231, -0.3251795768737793, -0.8873591423034668, -1.3949463367462158, 1.1861562728881836], [0.24857205152511597, -1.4132200479507446, -0.5877131223678589, -0.4194532036781311, 0.2879604697227478, -1.0198558568954468, 1.4398869276046753, -0.012274173088371754, 1.3965857028961182, 0.5203741192817688, -0.28505557775497437, -0.1905302256345749, 0.37785470485687256, 0.147408127784729, 0.25012508034706116, 0.7560352087020874, 1.2187519073486328, -1.019693374633789, -1.635212779045105, -0.9225778579711914, 1.21278715133667, -0.19485080242156982, -0.6384797692298889, 0.3394176959991455, 0.7215956449508667, -1.1222760677337646, -0.9927576780319214, -0.05644812807440758, -0.11730357259511948, -0.7500013113021851, 2.245058298110962, 1.2250704765319824, -0.16817885637283325, 0.6565681099891663, -1.2136026620864868, 0.9158809185028076, -0.20079316198825836, -0.7118805646896362, -1.3757761716842651, 0.8111481070518494, 0.06034949794411659, -0.5942399501800537, 0.9925751686096191, -0.26464951038360596, 0.2478426694869995, -0.4895932376384735, 0.3966805040836334, -1.169594407081604, -0.4215700328350067, 0.03449403867125511, -0.8388142585754395, -0.5678041577339172, 0.7001901865005493, -0.5019306540489197, 1.033705234527588, 0.40476536750793457, 0.007063048426061869, -1.5016926527023315, 1.4858876466751099, 1.4485331773757935, 0.287942498922348, 0.13692858815193176, 0.06825946271419525, -0.4128236472606659], [-0.2490922510623932, 0.40331244468688965, -1.4959352016448975, -0.04521864652633667, 0.407443106174469, -1.752368688583374, -0.6655256152153015, -1.561270833015442, -1.4482313394546509, 0.5256979465484619, -1.201670527458191, 2.5859920978546143, -0.22600306570529938, -0.8640179634094238, -0.3325556218624115, -0.06564287096261978, 0.24009187519550323, 0.264392614364624, 0.252675861120224, 1.0314592123031616, 1.1424059867858887, 0.02710249461233616, 0.3283303380012512, -0.780301034450531, -0.6394658088684082, -0.4116581082344055, 1.3049103021621704, -0.7325677275657654, 0.16822366416454315, 0.09424814581871033, 0.7658219337463379, -0.6208220720291138, -0.4094955325126648, 1.0551828145980835, 0.24003291130065918, -0.5411466360092163, 1.1603946685791016, 0.1126178726553917, -0.5105281472206116, 0.913459300994873, 1.764615774154663, -0.39221665263175964, -0.9795605540275574, -2.336249351501465, 1.8257155418395996, -0.5282270908355713, -0.2938110828399658, -0.9839368462562561, -0.1699438840150833, -1.032644510269165, 1.5265940427780151, 0.7065678238868713, -0.5217834115028381, -1.20038640499115, 0.5974896550178528, -0.818317711353302, 1.2872637510299683, 0.2986975908279419, -0.017753755673766136, -1.1680320501327515, 0.26442110538482666, -1.2936855554580688, 1.1734704971313477, -1.9388374090194702], [-0.33640578389167786, 2.3696935176849365, -0.4751906096935272, 0.751101016998291, 1.0759345293045044, 0.3778078556060791, -0.09642288088798523, 1.053020715713501, 1.0942051410675049, -0.5147836208343506, 1.6125856637954712, 1.3482553958892822, -0.9550859928131104, 0.5717318058013916, 0.6182335615158081, 0.21462853252887726, -0.5494194030761719, -1.1289583444595337, 0.5647481083869934, -1.51902174949646, -0.49905794858932495, -1.857667326927185, -0.6321185231208801, 0.2266651690006256, -1.3426238298416138, 0.8407356142997742, 0.9986767768859863, 0.48780667781829834, 0.7262337803840637, 1.5321077108383179, 0.8174672722816467, -1.6001191139221191, -0.07169655710458755, -0.7518540024757385, 0.10915450006723404, -1.590057611465454, 0.13705314695835114, -0.17507681250572205, 0.6110079288482666, -0.8730646967887878, -0.18204249441623688, -2.015982151031494, -0.08705884218215942, 0.022383466362953186, -1.2521729469299316, 0.06518378108739853, -1.2943722009658813, 0.5226342678070068, -0.7592354416847229, -0.3641889691352844, 1.682213306427002, 0.6115570068359375, 1.5532997846603394, 1.074225664138794, -0.03658552095293999, 1.6050034761428833, 0.33705076575279236, 0.06348863989114761, -0.3554020822048187, -1.0955584049224854, 0.6188479065895081, -0.13761006295681, 0.5290243029594421, -0.019659191370010376], [0.6955142617225647, -0.4803679883480072, 0.3479084074497223, 1.11305570602417, 1.394142508506775, 0.12714755535125732, -0.7911264896392822, 1.599739670753479, 1.3681167364120483, -0.4805786609649658, 1.4529592990875244, 0.7087130546569824, -0.7117063403129578, 0.3307596743106842, -1.0170150995254517, -0.16611896455287933, -0.19190101325511932, 0.1317678987979889, 1.280351161956787, -1.2459533214569092, -2.8220596313476562, 0.871695876121521, -0.10095302015542984, 1.1108049154281616, 1.0224343538284302, 1.8280657529830933, -1.757277011871338, 1.7181562185287476, 0.14891327917575836, -0.2908581793308258, 1.7344895601272583, 1.4117554426193237, -1.161501407623291, -0.41881412267684937, 1.6218065023422241, -1.1769739389419556, 0.5737377405166626, 0.7153661847114563, 0.03248441591858864, -0.30368512868881226, 1.0028868913650513, -0.00216473825275898, -0.7557578682899475, 0.829297661781311, -1.2749089002609253, -0.2491944134235382, 0.6398245692253113, -1.1951448917388916, 1.65189790725708, 0.5428805351257324, 0.13557766377925873, -0.8282708525657654, 0.382417231798172, -0.1438610851764679, 0.5578873157501221, -0.3723515570163727, 0.4562256932258606, -1.3024137020111084, -1.3581794500350952, -0.6239506006240845, -1.8608875274658203, -0.15698333084583282, 0.14203332364559174, -1.6017571687698364], [-0.13977068662643433, -0.6085445284843445, -0.6569364666938782, 1.2739540338516235, -0.3546694815158844, -0.022908398881554604, -0.5908593535423279, -0.49905022978782654, -0.11558322608470917, 0.03231003135442734, -0.2279895395040512, 1.0416984558105469, -0.22896993160247803, -0.06950286030769348, 1.7283881902694702, 0.254675954580307, 0.8652471899986267, 0.7275245785713196, -0.3364546597003937, 1.3389077186584473, 0.3138626515865326, 0.21540650725364685, 0.1604911834001541, 0.45134204626083374, 0.8877471685409546, -2.024045944213867, 1.4116075038909912, -1.954027771949768, -1.1575956344604492, 0.6769077777862549, -0.12428438663482666, 0.7233880162239075, -1.1305984258651733, 0.08791819959878922, 0.6127446889877319, -0.7783092260360718, -0.6749391555786133, 0.18294301629066467, 0.5355671644210815, -2.418156385421753, -0.30695217847824097, 0.3286221921443939, 1.4778212308883667, -1.080745816230774, 0.1493913233280182, -1.0401153564453125, -0.9078402519226074, -0.5859872698783875, 1.6739073991775513, 0.5867608785629272, 1.0048822164535522, -0.05988259240984917, 1.1401787996292114, 0.09990445524454117, 0.7998098134994507, -1.3805752992630005, -0.7882091999053955, 0.7608423233032227, 2.239457845687866, 1.989878535270691, -0.916988730430603, -0.09832654148340225, -0.16767586767673492, 2.6431193351745605], [-0.6428559422492981, 0.30832186341285706, 1.2835578918457031, 0.7765576839447021, -1.1730109453201294, 2.0837249755859375, -2.0256199836730957, 0.3254161775112152, -0.349824458360672, -1.3647308349609375, -0.32432734966278076, -0.3517628312110901, -1.1232657432556152, -0.5051066279411316, -0.18559598922729492, 0.19626116752624512, -2.161363363265991, -0.5259295701980591, 0.07707560807466507, 1.0374900102615356, 1.1502444744110107, -1.7529748678207397, -0.5862610340118408, 0.2711590528488159, 0.7778112292289734, 1.7609871625900269, -0.2843391001224518, -1.2280809879302979, 0.08182080835103989, 0.2883497476577759, -0.616300642490387, -0.8904899954795837, 0.653274416923523, -1.2717701196670532, -0.5363722443580627, 0.332181841135025, 0.19294606149196625, 0.09717335551977158, -0.9167097806930542, -3.544102430343628, -0.8663249611854553, 0.6626661419868469, 0.9075347781181335, 1.5713833570480347, 0.5520488619804382, 1.441225290298462, -0.024446573108434677, 0.4669099450111389, 2.155850410461426, -0.6653643250465393, -0.5135539770126343, 1.238499402999878, 0.9184213876724243, -1.630070447921753, 1.3694369792938232, -0.7202033996582031, -0.6579744219779968, 1.0808689594268799, 0.5418058037757874, 0.5956353545188904, 1.49648916721344, 1.046005368232727, -2.52331805229187, 0.3440607488155365], [0.7718600034713745, 0.05550219118595123, -0.2107851356267929, 0.3027380406856537, 0.2874304950237274, -0.004821016453206539, -0.07881060987710953, -1.0429941415786743, 0.14852434396743774, -0.6743006110191345, 0.8579676747322083, -0.4972364008426666, -0.37295302748680115, -1.2877676486968994, -0.2518218755722046, 0.5660709738731384, -1.1665472984313965, 1.1531857252120972, 0.07544353604316711, -0.23535718023777008, 0.7229679822921753, 0.31493762135505676, -1.9708133935928345, 0.9746288657188416, 0.5896518230438232, -0.07532912492752075, -3.1901845932006836, -3.9104437828063965, -1.8770440816879272, 0.7669802904129028, 0.9116788506507874, 0.056902576237916946, -0.09879358857870102, -2.3325517177581787, -0.10993450880050659, -1.020541787147522, -0.2469068020582199, -1.1096755266189575, 0.7192466259002686, -0.3796186149120331, 0.2953891158103943, -0.10891879349946976, -0.7858846783638, -1.583203673362732, -1.343801498413086, -0.9772675633430481, 0.025294238701462746, 1.9602819681167603, 0.346724271774292, -0.1527586281299591, 0.8582541346549988, -1.1749377250671387, 0.24845550954341888, 0.26324865221977234, -2.270846366882324, 0.25399577617645264, -0.6347495317459106, -0.1465778648853302, -0.3152943551540375, 0.0913006067276001, -0.8277367353439331, 0.8276783227920532, 0.9478552937507629, -1.03597092628479], [0.19704028964042664, -0.2715904116630554, 1.260356068611145, -0.8227164149284363, 0.10476454347372055, -0.22223018109798431, -1.4742802381515503, 1.0484403371810913, 0.24384230375289917, -1.2729984521865845, 0.8040962815284729, 0.9490574598312378, -1.599008321762085, -0.283278226852417, -0.32949134707450867, -0.25408220291137695, -0.14357365667819977, -0.4776330292224884, -1.8250176906585693, -0.019625188782811165, -1.385306477546692, -0.6549652814865112, 0.4613856375217438, 1.842363953590393, 1.3672574758529663, -0.7221747636795044, -0.2150876522064209, 1.3815686702728271, 0.5711366534233093, 0.9123778343200684, -0.789232075214386, 0.6136534214019775, 0.7191489934921265, 1.4580743312835693, 1.8666582107543945, -0.5760060548782349, 1.5200212001800537, -0.17211726307868958, 0.19146351516246796, 1.5631186962127686, 0.3296182155609131, 0.6728848814964294, 1.807166337966919, 2.4169278144836426, 0.8457608819007874, -0.8771536946296692, 0.3198063373565674, -0.46360713243484497, -0.28370678424835205, 0.43889766931533813, -0.21117860078811646, 0.04193072021007538, -0.08277788758277893, -0.1475391536951065, -1.9115309715270996, -0.2853041887283325, -0.20920789241790771, 1.5110222101211548, 0.7743525505065918, 0.22493530809879303, -0.6386659741401672, 0.4413376748561859, -1.526331901550293, 0.598527729511261], [-0.7847092151641846, -0.7489288449287415, 1.0901453495025635, -0.05772959813475609, -0.2217073291540146, -1.264236330986023, 1.2628185749053955, -2.645334482192993, 0.4227188229560852, 0.2288271188735962, 0.40101611614227295, -0.6739775538444519, -1.0967334508895874, 1.7394965887069702, 0.4323604702949524, 0.900527834892273, -1.6902071237564087, 0.9754328727722168, 1.4777153730392456, 0.8177587985992432, -1.7329930067062378, -1.208626389503479, -2.6809725761413574, 0.21776965260505676, 0.9494053721427917, 0.23591116070747375, -0.5575857758522034, -1.026323914527893, 0.7569328546524048, -1.1175520420074463, 1.8475689888000488, 0.4828318953514099, 1.381308913230896, 0.11711851507425308, -0.27873218059539795, 0.48514071106910706, -0.8438606262207031, -0.2516700029373169, -0.1595851629972458, -1.000706672668457, 0.4605334401130676, -1.17830491065979, -2.00358247756958, -0.39766526222229004, -0.22477930784225464, -0.42730939388275146, -0.14114592969417572, 0.021201737225055695, 0.16510853171348572, -0.08808138966560364, -0.1620948761701584, -1.7365909814834595, 0.3702685832977295, 0.9226768612861633, -0.5080974698066711, -0.9173363447189331, 0.8282597661018372, -0.693962812423706, 0.14772604405879974, 0.46439892053604126, 1.1086300611495972, 3.1366193294525146, -0.19235564768314362, -3.3072056770324707], [0.15138280391693115, -0.7559022903442383, 0.6912758946418762, 0.41574299335479736, 0.974147617816925, 0.014197043143212795, 0.9632473587989807, -0.4664626717567444, 0.8047041296958923, -1.0896172523498535, -0.9067375063896179, -0.09810598939657211, 0.6645275950431824, -0.978165328502655, 0.0017663466278463602, 0.6442158818244934, 0.7084980607032776, -1.169154167175293, 1.8837456703186035, -0.38795214891433716, 0.41229766607284546, 1.5357121229171753, -0.7754860520362854, -1.960928201675415, 1.233335256576538, -1.468839168548584, 0.1486038863658905, -0.8926503658294678, 1.958260416984558, 0.3830638527870178, 1.8024460077285767, 0.5535885691642761, -0.5403238534927368, 0.9349802136421204, -1.0333110094070435, -0.5694132447242737, 0.9457348585128784, 0.7858719825744629, -0.9676395654678345, -0.1605275720357895, -0.20711499452590942, -1.3023492097854614, 1.1727581024169922, -1.2891219854354858, -0.12117909640073776, -0.03675183653831482, 0.06828609853982925, -3.219722032546997, 0.1493358165025711, -0.6512821912765503, 1.3324804306030273, -1.5339592695236206, 0.987765371799469, 0.5508432388305664, -0.8443484902381897, 0.20791226625442505, 0.6662673354148865, -1.3432822227478027, -1.8225929737091064, -0.3785351812839508, 0.6607716679573059, 0.3629908561706543, 0.08321041613817215, -0.6077446937561035], [-0.4978940188884735, 0.9045659303665161, 0.5485441088676453, 1.7307837009429932, 0.6936165690422058, 0.19306766986846924, -0.33985233306884766, 0.028712080791592598, 1.6053133010864258, -0.429147869348526, -0.259644091129303, 0.3381408751010895, 0.569282591342926, 1.447727084159851, -0.0635467991232872, 2.304657220840454, 0.15699680149555206, 0.32387927174568176, 0.3833795487880707, 1.5111867189407349, 1.0421537160873413, 0.31582924723625183, -0.9546984434127808, -1.2469661235809326, 0.3837369680404663, 0.3839583694934845, -0.06052680313587189, 0.3189666271209717, -0.6315559148788452, 0.3347545564174652, 1.417232871055603, 0.5420961380004883, 0.16908197104930878, 0.2675228714942932, 1.0742236375808716, -0.5669962167739868, -0.3576779067516327, -0.7150416374206543, -1.7192118167877197, 1.383471965789795, 0.7806445360183716, -0.7407025098800659, 1.9219915866851807, -0.6834026575088501, 0.48833081126213074, -0.7103869915008545, 1.4471628665924072, 0.5725393891334534, 0.7678186893463135, 0.7246755957603455, 2.4057278633117676, -2.242130756378174, -0.33628666400909424, 0.2133752852678299, 0.67266845703125, 0.03482700511813164, 0.120730459690094, 1.0222991704940796, 1.0817745923995972, -0.4012242257595062, 1.4276185035705566, 0.22823381423950195, 0.4296935796737671, 2.3438639640808105], [1.7453049421310425, -1.7532869577407837, 0.9788436889648438, 1.1958049535751343, 0.5696874260902405, -0.42726531624794006, 1.7046815156936646, -0.31007489562034607, 0.16915732622146606, -0.5904706716537476, -0.04744693264365196, -1.3818296194076538, 1.325877070426941, -0.5068851113319397, 0.1683557629585266, -0.2723569869995117, 1.673909068107605, 2.278367280960083, 0.6292116045951843, 0.3140445947647095, 0.3751981854438782, -0.8553434610366821, -0.8894170522689819, -0.4858321249485016, 0.18614675104618073, -0.5879045128822327, -0.19891665875911713, 0.10991392284631729, 0.7187957763671875, 0.19175659120082855, -0.5572534799575806, -0.01150062121450901, -1.3037292957305908, 1.2093039751052856, 0.7829534411430359, 0.5068773031234741, -1.7227095365524292, -0.30996617674827576, 0.15795087814331055, -0.38580697774887085, 0.7062859535217285, -1.5116016864776611, 0.5707992911338806, 1.17099928855896, -0.48895373940467834, -0.5491855144500732, -1.161563515663147, 0.5442107319831848, -0.1937839686870575, -0.03690234199166298, -0.7117732167243958, 0.4931565523147583, 0.018824495375156403, -0.19368335604667664, -0.40455377101898193, 0.3514867126941681, -0.4387180507183075, -0.7028501629829407, -0.5377707481384277, -0.5483747720718384, -1.204535961151123, 1.7621181011199951, 1.5597692728042603, -1.6817103624343872], [-0.7488538026809692, -0.9976834654808044, 1.5524994134902954, -0.5235592722892761, 0.41814589500427246, -0.8707385063171387, -0.2960669994354248, 0.8465887308120728, 2.155883312225342, 0.1661612093448639, 1.4430073499679565, 1.6444270610809326, 1.3654221296310425, 1.338216781616211, -0.6832548975944519, 1.1174696683883667, 0.24468694627285004, -1.9256614446640015, 0.6554766297340393, 0.1275450736284256, -0.748367190361023, 0.7410660982131958, -0.7656180262565613, -0.3792267441749573, -0.4889090955257416, 1.4548524618148804, -0.5657849907875061, 1.846406102180481, 0.5955743193626404, 1.4037787914276123, 0.24048808217048645, 0.18727421760559082, 0.29392361640930176, 0.9430297613143921, 0.49052655696868896, 0.962161660194397, -0.9612084627151489, 1.2063393592834473, 1.6253974437713623, 0.8412357568740845, 0.742868959903717, 0.5825632214546204, -0.42885926365852356, -0.9365711212158203, -0.044880036264657974, 0.21192872524261475, -0.11418123543262482, 0.5393258929252625, -0.517224133014679, 0.03397263213992119, -0.8794730305671692, -1.1122522354125977, 1.5411999225616455, 0.36180737614631653, -0.06385266780853271, -0.9069535732269287, 0.501323938369751, 1.9558018445968628, -0.6564464569091797, -0.29833534359931946, 0.46727824211120605, -0.9103295803070068, -1.973219871520996, 0.7317926287651062], [-0.1679157167673111, 1.089983582496643, 0.9324187636375427, 0.14007224142551422, -1.4130728244781494, 1.3212018013000488, 0.5542932152748108, 0.25003501772880554, 0.1848578304052353, 0.1121363565325737, 0.575228750705719, -1.0867739915847778, 0.2260144054889679, -0.9523346424102783, -0.7779097557067871, 0.016141517087817192, -0.43162646889686584, 0.1344955861568451, -0.02127107046544552, -0.6041783094406128, 0.6894945502281189, 0.49032026529312134, -0.2905610501766205, -0.7405431270599365, 1.4696602821350098, -1.4094576835632324, 0.5726775527000427, -0.7015329003334045, 0.8047022223472595, -0.15970109403133392, 1.8229714632034302, 0.4754154682159424, -0.3024352490901947, -0.701322615146637, 0.6418784260749817, -0.04330574348568916, -1.679797649383545, -1.240347981452942, -1.7823386192321777, 1.6140258312225342, 1.3968905210494995, -0.12712691724300385, 1.1320394277572632, -0.2014319747686386, -0.20932941138744354, -0.14756393432617188, 0.6313324570655823, -0.5985164642333984, 0.3798770308494568, -0.03776432201266289, -1.894792079925537, -1.4272451400756836, 0.7607215642929077, -0.2310125082731247, -0.5182844400405884, -0.1207830160856247, -0.3651183247566223, -0.08901048451662064, 0.8452476859092712, -0.5284963250160217, 0.28293341398239136, 0.977182924747467, 0.18716657161712646, -2.1656665802001953], [1.239595890045166, -1.1505557298660278, -0.5374793410301208, 0.5651557445526123, 1.8617243766784668, 0.6641152501106262, 0.09685487300157547, -0.3833921253681183, -0.5907062888145447, -1.0215051174163818, 0.8021915555000305, -0.12429886311292648, 0.07846216857433319, -1.4015092849731445, 0.15477092564105988, -1.6916089057922363, 0.4591277837753296, 1.8033264875411987, -0.9854552149772644, -1.234723687171936, 1.2443934679031372, -0.595129668712616, 1.177524447441101, -1.3572381734848022, 0.2964990437030792, 0.9284809231758118, 0.3403260409832001, -0.9714869856834412, -2.7127227783203125, -0.12902598083019257, -0.7928311228752136, 0.06406371295452118, 1.644700050354004, -1.5799241065979004, -0.3702520430088043, 0.6573033928871155, 1.9695991277694702, 0.07004540413618088, 1.1432632207870483, -0.8937533497810364, 0.07768679410219193, 0.01447369810193777, 0.6736140251159668, -0.024161046370863914, -0.11630449444055557, 1.059117078781128, 0.0654134750366211, -0.11883347481489182, 0.24080897867679596, -0.1285368800163269, 0.987695574760437, 0.6242689490318298, -0.8588796854019165, 0.026076847687363625, -1.6903759241104126, -0.906306266784668, 0.0007619870593771338, 1.3099180459976196, 0.3976052403450012, 1.0661548376083374, -0.6549022197723389, 1.3444342613220215, 0.6012154817581177, -0.4844861328601837], [-0.4058254361152649, -0.15760193765163422, -1.3988382816314697, -1.4558179378509521, -0.7166967391967773, 0.2759740650653839, -0.09521413594484329, 1.6920337677001953, 1.0378938913345337, -0.33791258931159973, 1.1775721311569214, -0.3655931353569031, -0.1971648931503296, 0.5928215384483337, 2.5122761726379395, -0.0716175064444542, 1.993648886680603, -0.11584880203008652, 1.2201952934265137, 1.344341516494751, 0.8034933805465698, 0.841892659664154, -0.10140611976385117, -1.4842292070388794, 0.16893674433231354, 0.17496350407600403, 1.9120043516159058, 0.23845946788787842, 0.45311859250068665, 0.6758703589439392, -0.5482726693153381, 0.3311893045902252, 1.1772078275680542, -0.014616644941270351, -1.0075476169586182, 0.17019720375537872, 1.1582670211791992, 0.00909485761076212, -0.36735936999320984, 0.9167810678482056, -1.424713373184204, 0.12341637164354324, 0.06201212853193283, -0.7598693370819092, -0.0038726315833628178, 1.0771584510803223, 0.9908074736595154, 0.2680380642414093, 1.1842795610427856, 0.6827685236930847, -1.1485366821289062, 1.0309126377105713, 0.01489365566521883, 0.16935665905475616, 0.24038586020469666, 1.5073939561843872, -0.3833990693092346, 0.6597449779510498, 0.03751538693904877, -0.43562954664230347, -1.4969143867492676, 0.7815931439399719, -0.4180912375450134, -0.29483503103256226], [-0.012720943428575993, -0.5119730234146118, -1.540350317955017, 1.6431101560592651, 0.738966166973114, 0.8043603897094727, 0.3138423264026642, 0.002794760512188077, 0.29695290327072144, 0.4513992369174957, 0.5375564098358154, 1.7015153169631958, -0.9306124448776245, -1.29706871509552, -0.3904826045036316, 0.673148512840271, 0.25380897521972656, -1.8264473676681519, 0.636698305606842, 1.4820691347122192, -0.2200731337070465, 0.7555672526359558, 0.26340141892433167, -0.5993993878364563, 1.256392478942871, 0.037958864122629166, -1.3037983179092407, 0.9856727123260498, -2.024372100830078, 1.9824869632720947, -2.2494349479675293, -0.7581626176834106, -1.9407297372817993, 0.476260781288147, 1.2489439249038696, -1.4130526781082153, 0.33939364552497864, -1.3188658952713013, -1.8634206056594849, 0.05646594241261482, 1.0665216445922852, -0.027802126482129097, -0.004223188851028681, 0.8630621433258057, 0.29255542159080505, 0.8515079617500305, 0.8904851078987122, 0.47221142053604126, 0.5179676413536072, 0.6837490797042847, -0.10169122368097305, -0.25327351689338684, -0.6345058083534241, 0.8398132920265198, -0.4795442819595337, 1.1221160888671875, 1.334020733833313, 0.06632577627897263, 0.06803232431411743, 1.5645486116409302, 0.3606167435646057, 2.0572316646575928, -1.0213793516159058, -1.3410437107086182], [0.33462345600128174, 0.3221227526664734, -0.22197848558425903, -1.37058687210083, 0.11715292185544968, -0.2950149476528168, 0.6697841882705688, 0.3612147271633148, -0.6393048167228699, -0.28095248341560364, 0.9199987649917603, -0.10500021278858185, -0.6742979884147644, -0.3190770149230957, 1.807350754737854, 2.226234197616577, 0.5995028018951416, 0.8228980302810669, 0.5643972754478455, 0.3844892680644989, -1.965330719947815, 2.277534246444702, -0.1616845726966858, 0.4623546898365021, 0.6374619603157043, -0.7887746691703796, -1.3813191652297974, 0.6340887546539307, 0.35835009813308716, 0.6820247173309326, 1.6575989723205566, -0.9965251088142395, -0.6666397452354431, 0.400870680809021, 1.130726933479309, 1.2856624126434326, -0.7667145729064941, -0.24219690263271332, -0.47120997309684753, -0.8380370140075684, 0.4374995529651642, -1.2071049213409424, 0.647350013256073, -0.7789433002471924, 1.9181413650512695, -0.4850176274776459, 0.9719905853271484, 1.151240587234497, -0.37814345955848694, 0.5607367753982544, 0.44671669602394104, -0.6608986854553223, -0.15039785206317902, 2.209218740463257, 0.4994288980960846, 0.6800793409347534, -1.6931267976760864, -1.860894799232483, 0.5345877408981323, -0.31394991278648376, -1.323603630065918, -0.821376621723175, -0.6797543168067932, -1.888464093208313], [0.779152512550354, -2.0893869400024414, 1.9833425283432007, -1.872209906578064, -0.820263147354126, 0.659699559211731, 0.241444930434227, 0.9707791805267334, -0.7246680855751038, -2.0236687660217285, 0.9392349123954773, -0.7564033269882202, 0.30847612023353577, 1.3834402561187744, 1.735711693763733, -2.0666089057922363, 0.487035870552063, -1.4292770624160767, -0.7311173677444458, -1.0181365013122559, 1.569253921508789, 0.21470794081687927, -0.8689084649085999, 0.9563145637512207, -0.9194539189338684, 1.3949265480041504, -0.5097775459289551, 0.5578798055648804, 1.295548915863037, -0.991031289100647, 0.8504559397697449, 0.5819832682609558, 0.4936288893222809, -0.9650479555130005, 0.3640717566013336, -0.5019009113311768, 0.6081439852714539, -1.8100050687789917, 0.0067426059395074844, -0.6859971284866333, -1.115251898765564, 1.377021074295044, -0.9472389221191406, -0.40064769983291626, 0.6452496647834778, -0.8515647053718567, 1.2107514142990112, -0.8468510508537292, -0.9159485101699829, -0.4071764647960663, -0.30004966259002686, 1.7231789827346802, 1.059920072555542, -0.22880063951015472, 0.9754233360290527, -0.8136385083198547, 0.5384808778762817, -0.04393045976758003, -0.5053569674491882, 1.4281960725784302, 0.5064286589622498, 0.35649141669273376, -1.4706425666809082, -0.1357267200946808], [0.3688597083091736, -2.118947744369507, 0.9354637265205383, -0.11407589167356491, 0.08752992004156113, -1.002595067024231, -1.8979294300079346, -0.37101033329963684, -0.351360023021698, -0.19439958035945892, 2.041937828063965, -0.2594328224658966, 1.9843220710754395, -0.41659170389175415, 1.0083609819412231, -0.5677335262298584, -0.2874128818511963, 1.7875229120254517, 0.7075613141059875, -1.3449883460998535, 0.8855414390563965, 1.131870150566101, 0.47307050228118896, -0.675909698009491, -0.19222567975521088, 0.5875634551048279, 0.23853710293769836, 0.11886948347091675, -1.0904279947280884, 1.815051555633545, 0.855818510055542, -0.9293294548988342, -1.4326344728469849, 0.90639328956604, -0.0763404443860054, -2.1023221015930176, 0.2140442281961441, 0.29503706097602844, -0.837588369846344, -0.8099626898765564, 0.8322733044624329, -1.0708656311035156, -1.264496922492981, -2.047499418258667, 0.0920584425330162, 1.4157342910766602, 0.10185104608535767, -1.4426307678222656, 0.2776952385902405, 0.8935564756393433, -0.5788476467132568, 0.43345925211906433, -0.10504390299320221, 2.205002546310425, -0.7865537405014038, -1.7237910032272339, -0.20203714072704315, 1.2634556293487549, 0.07498941570520401, 1.172786831855774, 0.3490811884403229, 1.276026725769043, -0.13643889129161835, -0.7486056685447693], [-0.8980655074119568, -0.538712203502655, 0.305934339761734, 0.9544858932495117, -0.58784419298172, 1.3073323965072632, -0.470144122838974, 1.1154688596725464, -0.6181808710098267, -0.12019490450620651, 0.7061737775802612, -0.14552952349185944, 3.1280083656311035, 0.5277076363563538, 0.9227467775344849, -0.9344235062599182, -0.25544387102127075, 1.379380226135254, 1.1621087789535522, 0.5513565540313721, 0.4381919205188751, -0.6112215518951416, 0.26271048188209534, -0.5603724718093872, -1.1632554531097412, -0.12750238180160522, 1.747166633605957, -0.13417363166809082, 0.5943737030029297, -0.7519693970680237, -0.12350286543369293, 1.262505292892456, 0.3652966618537903, -0.21732068061828613, 1.3867570161819458, 0.23902568221092224, 0.9618837237358093, 2.2424585819244385, 2.4121286869049072, -0.9293201565742493, 0.48072686791419983, -0.8433758020401001, 0.08847169578075409, 0.5482527613639832, -0.3819828927516937, 1.446595549583435, 0.05295303463935852, -1.5438987016677856, 0.1868337094783783, -1.1975117921829224, -0.4342506229877472, -0.731046199798584, 0.07619564235210419, -1.6914551258087158, -0.4626196622848511, 0.27981558442115784, -0.6676101684570312, -1.6389918327331543, 0.4406901001930237, -0.10719218105077744, -1.11521315574646, -1.3368102312088013, -0.3408418595790863, 0.5938577651977539], [-1.204285979270935, -0.48966631293296814, 1.4435585737228394, -0.0883326381444931, 0.18450261652469635, -0.01567094400525093, 1.380183219909668, -0.9202985167503357, 0.057334017008543015, 0.09646054357290268, 0.626815676689148, -0.20193363726139069, -2.273944854736328, -0.9989725947380066, -0.041623085737228394, 1.3725581169128418, -0.9719815850257874, 1.4006271362304688, -0.3288121223449707, 0.5807254910469055, 0.35435718297958374, -0.9554437398910522, 0.2447066456079483, -0.4999900460243225, 1.2689192295074463, 1.3105459213256836, 0.37402379512786865, 0.2714778780937195, 0.27002623677253723, 1.3235199451446533, 0.2935515344142914, -1.164194941520691, 0.3965010941028595, 3.1930735111236572, -0.9588873386383057, 0.7550808191299438, 0.17819054424762726, 1.3709982633590698, 1.3460776805877686, -0.19902898371219635, 0.8912044167518616, -1.0557254552841187, 0.37650299072265625, -1.4892348051071167, 1.0666624307632446, -0.488728404045105, 0.5272826552391052, -0.5909225940704346, 0.7483917474746704, 0.023246370255947113, -0.7502177953720093, 0.3581332266330719, -0.023546572774648666, -0.6911682486534119, -0.5101116299629211, 0.3951863646507263, -0.850276529788971, 0.4877966642379761, 0.5183761119842529, -0.32586702704429626, 0.5025731325149536, -0.9275931715965271, 0.19974112510681152, -0.466175377368927], [0.8735255599021912, 0.7433040142059326, 2.3096625804901123, -0.41811975836753845, 0.7850568294525146, -0.22344186902046204, 0.7402560114860535, 2.104127883911133, 1.2401491403579712, 0.17750880122184753, 0.13038679957389832, -1.0408109426498413, -1.4114857912063599, 0.16280119121074677, -0.5733213424682617, 0.7582622170448303, -0.3129637837409973, 1.027293086051941, -0.5385079979896545, -0.6435689926147461, 1.78190279006958, -0.5851146578788757, 0.058679718524217606, 0.25397515296936035, 0.5339705348014832, -2.1291801929473877, 0.5696544647216797, -0.9369184970855713, -1.5886310338974, 0.8405712246894836, -1.1153438091278076, 0.7527512907981873, 1.2391526699066162, -0.1727638691663742, -0.08357684314250946, -1.6724976301193237, -0.6614345908164978, 1.9275016784667969, 0.6449678540229797, 0.796139121055603, 0.8547583818435669, 0.6329076886177063, 0.7086613774299622, 0.7745687365531921, -1.4489567279815674, -1.452773094177246, -0.38780584931373596, 0.6209960579872131, -1.3533533811569214, 0.687223494052887, -0.9307399392127991, 1.83098566532135, 0.8918197751045227, 2.3453874588012695, 1.2508161067962646, -1.4510557651519775, -0.03980749845504761, 0.5897002220153809, 0.5633353590965271, -0.4446631371974945, -1.4527580738067627, -0.951375424861908, -0.36810529232025146, -0.15819790959358215], [-0.6349449753761292, -0.610802412033081, -0.23294946551322937, 0.5411930084228516, -0.9024756550788879, -0.0975944995880127, 0.718439519405365, -0.7669912576675415, -0.3825101852416992, -1.474297285079956, -0.251699835062027, -0.22304420173168182, 0.6633242964744568, -0.00259179575368762, -1.2387765645980835, 0.16510197520256042, -1.0951709747314453, -0.720379650592804, -0.6603640913963318, 1.9881525039672852, 1.3481788635253906, -1.5216453075408936, 0.34275373816490173, 1.0216323137283325, 0.8332457542419434, -0.029591284692287445, 0.06510084867477417, -0.10590081661939621, -0.0582297258079052, -0.3044835329055786, -1.1563373804092407, 0.7878985404968262, -0.5059637427330017, 2.0351319313049316, 0.2745468020439148, 0.2323700487613678, 1.1458940505981445, 0.4769165813922882, 0.2445267140865326, -1.0612584352493286, -1.1046192646026611, -1.249090552330017, 1.0883207321166992, -0.49654683470726013, 0.06546211987733841, 0.2705591917037964, -0.8409057855606079, -0.02085658721625805, -0.45654943585395813, -0.9608109593391418, -0.3916650414466858, -1.0784448385238647, -0.16220557689666748, -0.2439766377210617, 0.048364024609327316, 1.5042372941970825, -0.13098624348640442, 1.2666155099868774, 1.1423795223236084, -0.5310923457145691, -0.792504072189331, -0.59560227394104, 0.7059982419013977, 0.1841692179441452], [0.45510128140449524, 0.6216502785682678, 0.5149010419845581, 1.276091456413269, -1.7563122510910034, -0.17393866181373596, -1.2571665048599243, 1.3012590408325195, 0.09052563458681107, 0.2329915165901184, -1.0558699369430542, 0.19433899223804474, 0.7286348938941956, -0.20710305869579315, 0.3694552779197693, 0.32501867413520813, -1.0872786045074463, 0.2304157018661499, -0.14895673096179962, -0.6836302876472473, 1.3777505159378052, 0.8016704320907593, 0.035954054445028305, -0.2443569451570511, 1.1211557388305664, -0.7664949297904968, 0.6531597375869751, 1.3361437320709229, -1.2866853475570679, -0.04469618573784828, -1.5179805755615234, 2.8596227169036865, -0.1780553162097931, 0.45768460631370544, 1.4794195890426636, -0.7708335518836975, 0.4656151533126831, -0.8555161952972412, 1.8532991409301758, -0.0230332612991333, 0.5240815877914429, 0.11905791610479355, 2.1379759311676025, 0.4297516644001007, -0.21102963387966156, -0.690841555595398, 0.52231365442276, -1.0357656478881836, 0.3094537854194641, -0.16520634293556213, 0.0997900441288948, -0.8664878010749817, -0.5420101881027222, -0.8774803280830383, 0.08334735780954361, 1.5487453937530518, 0.891058623790741, 0.9409211277961731, 1.7796027660369873, -1.7655048370361328, 1.4612830877304077, -0.9570900201797485, 1.1670063734054565, -0.04206031933426857], [0.6885382533073425, -2.6224098205566406, -0.9219408631324768, -0.024473946541547775, 0.7959248423576355, -0.8231287002563477, -2.496417760848999, -1.0668429136276245, -0.829038679599762, 0.7116065621376038, -1.2755091190338135, 1.0415961742401123, 0.01798383891582489, -0.01741475984454155, 0.3733251690864563, 0.8674929738044739, 0.29802146553993225, -0.37494009733200073, -0.3643505871295929, 0.01016323547810316, 0.04370898753404617, -0.6814308166503906, 0.26032450795173645, -0.10561468452215195, 0.05131016671657562, -0.10995150357484818, -1.5693439245224, -0.6561205387115479, -0.20953786373138428, -0.6590895652770996, -0.07486546784639359, 0.4276563227176666, 1.926489233970642, 1.2932080030441284, -0.7741463780403137, -0.24580290913581848, -0.418236643075943, -0.7472456693649292, -0.9244194030761719, -0.49628570675849915, 0.3042007386684418, 1.1241589784622192, 2.1354053020477295, 1.1366852521896362, -0.6884372234344482, -0.3222215175628662, 0.7197851538658142, 0.7793883681297302, 1.0237067937850952, -1.6111195087432861, 0.6185188889503479, -0.255295068025589, 0.9255388975143433, -0.6993184685707092, 0.5043393969535828, -0.7426681518554688, 1.097493052482605, -0.18307973444461823, 1.9389562606811523, -1.6510995626449585, -1.414368748664856, 1.1413391828536987, 0.27357620000839233, 0.014303082600235939], [-0.560226321220398, 0.4262963831424713, 0.5973750948905945, -0.06158985197544098, 0.6718559265136719, 1.9774210453033447, 0.3974953591823578, 1.4916621446609497, -0.27321311831474304, -0.45767346024513245, -0.7702882289886475, -1.1916440725326538, -0.7892977595329285, -0.5890079736709595, 0.4295820891857147, -1.4627596139907837, 1.3454746007919312, -1.5609506368637085, 0.6758267879486084, 0.39331772923469543, 0.331868976354599, -0.35968539118766785, 0.5915435552597046, -2.1821985244750977, 1.6857858896255493, 0.514832615852356, 0.5486070513725281, 0.02185094729065895, -1.0352779626846313, -0.4506445825099945, -1.4109174013137817, -0.16860923171043396, 0.48846355080604553, -1.6458665132522583, 1.0380887985229492, -0.06277479976415634, -0.2995910346508026, 0.6968269944190979, -0.21337619423866272, -0.18259279429912567, 0.05348000302910805, -0.7774991393089294, -0.11293522268533707, -1.4923418760299683, 0.11018918454647064, 1.7751617431640625, 0.2698170244693756, 0.4684292674064636, 0.5123128890991211, -1.0737415552139282, -0.5036811828613281, 0.0506557896733284, -0.018635109066963196, -0.8623625636100769, -0.40561288595199585, -0.15385285019874573, 0.21116255223751068, -1.675337791442871, 0.4574349820613861, 0.3729030191898346, -0.5947275161743164, -2.2029483318328857, -0.20397593080997467, -1.530463457107544], [0.7408282160758972, -0.9877933263778687, 1.0151687860488892, 2.053481340408325, 1.0101792812347412, 0.22901077568531036, -0.8285350799560547, -0.05237556993961334, -0.0037826227489858866, -0.9605765342712402, -2.531017303466797, -0.35313114523887634, 0.49979040026664734, 0.1225600391626358, -0.3187483549118042, -0.8267237544059753, 0.0938522145152092, 1.1395630836486816, -2.9319984912872314, 1.5317429304122925, -0.4078439474105835, -0.21051307022571564, -0.1687563955783844, -1.2906215190887451, 1.091477394104004, -1.897218942642212, 0.4443553686141968, -2.020448923110962, 0.8975933790206909, -0.8586275577545166, 0.6456456184387207, 0.7339526414871216, 1.3392940759658813, 0.012365416623651981, 0.5543485283851624, 0.8655493259429932, -0.5772322416305542, -0.21952573955059052, 2.208322763442993, -2.085803747177124, 1.4962718486785889, 0.03845534473657608, -0.450868159532547, -1.5276840925216675, 0.07820596545934677, 0.8887808322906494, -0.0679255872964859, 1.637393832206726, -0.7405232191085815, -0.386062353849411, -0.008429737761616707, -0.5585298538208008, 0.08621103316545486, -0.7359004020690918, -1.1595407724380493, -0.5338783264160156, 0.5031505823135376, 0.8138847947120667, -0.890171229839325, 1.158599853515625, 0.10778548568487167, -0.07891817390918732, -0.88176429271698, 0.08247694373130798], [-0.42372187972068787, -0.8742559552192688, 0.7371283173561096, 0.6549756526947021, 0.01959696225821972, 1.067132830619812, 0.8887523412704468, 0.6364611387252808, 0.1884726881980896, -0.7739898562431335, 0.01796935871243477, 0.7379568815231323, 0.0962248146533966, 0.08714312314987183, 0.276236891746521, 0.9392668008804321, -0.8081039786338806, 0.8010907769203186, 1.5707281827926636, -1.3779792785644531, -1.1010470390319824, 0.16823279857635498, 1.4569144248962402, 1.4383164644241333, 0.8034958839416504, 0.7138764262199402, 0.18926295638084412, 0.27088624238967896, 1.345102071762085, -0.8762068152427673, -0.23273970186710358, 0.5664641261100769, 1.5876036882400513, -0.7745543718338013, -0.12870673835277557, 0.42596855759620667, 0.7732598781585693, 1.8412946462631226, 0.9302246570587158, -0.4461645781993866, -0.896781325340271, -1.186198353767395, -1.403123378753662, -0.1417728215456009, 1.9689122438430786, 0.21138951182365417, -0.4368605613708496, 0.11216370761394501, -0.3652362823486328, 1.6356713771820068, 0.5272584557533264, -0.5792283415794373, 0.3564678430557251, 0.5308248996734619, -0.68625408411026, -0.9956921935081482, 0.03423866257071495, -2.3302090167999268, -1.243821144104004, 0.33941200375556946, 0.7153282761573792, 1.6375941038131714, 0.46839889883995056, 0.13564400374889374], [-0.3631933331489563, -0.3787458539009094, 0.9827166199684143, 0.44969913363456726, 1.060912847518921, -0.43948182463645935, 0.9431242942810059, -1.1646404266357422, -1.5215657949447632, 1.0915430784225464, 0.8688568472862244, 1.0463250875473022, -0.35700705647468567, 0.7265738844871521, 0.2601923644542694, -1.160097360610962, 0.5105418562889099, -1.4604603052139282, 0.322078138589859, 0.5101746320724487, -0.86750328540802, 0.7780122756958008, -0.9409985542297363, -1.0071461200714111, 0.6792535781860352, 0.0519600585103035, -1.3550738096237183, 3.0418100357055664, 0.6297574639320374, 0.5652603507041931, 0.03042677789926529, -0.6027135848999023, -0.014714791439473629, -0.1145443394780159, -0.47918248176574707, -0.6289268732070923, 0.7024282217025757, -0.7261700630187988, 1.4400829076766968, 0.9931338429450989, -0.2808867394924164, 0.10745272785425186, -0.6709014177322388, 0.11234030872583389, 0.20895063877105713, 0.2685401141643524, 0.3286803960800171, 0.5154962539672852, 0.7632694244384766, -0.5242480039596558, 0.17774920165538788, -0.971342146396637, -2.397355794906616, -0.9638537764549255, -0.835313081741333, -0.7917397618293762, 1.013176441192627, -0.39987972378730774, 1.2220449447631836, -0.4957621693611145, -0.8054631948471069, -0.06944010406732559, 1.8061811923980713, -1.1886757612228394], [0.3447401225566864, 0.9133418798446655, 0.9766843914985657, -0.48541900515556335, 0.5515332818031311, 0.28314992785453796, 1.8314495086669922, 0.23360949754714966, -0.21088968217372894, -0.7450950145721436, 0.2752246856689453, -1.66032075881958, -2.1738739013671875, -0.8412914872169495, 0.9190300107002258, 0.17249073088169098, -0.3019329011440277, 2.1934871673583984, 0.24183791875839233, 1.0398952960968018, 0.3577570915222168, -0.5500319600105286, 0.49567559361457825, -0.502948522567749, -1.8501836061477661, -0.503791093826294, -1.6048623323440552, 0.38357481360435486, 0.0740654394030571, -0.7874664664268494, -1.6953823566436768, -0.9798838496208191, 0.6811385154724121, 0.38751643896102905, 0.7130044102668762, -1.2215745449066162, 1.4710111618041992, 1.0722616910934448, 0.15589749813079834, -0.3056025207042694, 1.485876441001892, -0.4461425244808197, 0.8899770379066467, -1.0988069772720337, 1.098104476928711, 1.4638773202896118, -0.3796892464160919, -1.235819697380066, -1.6776596307754517, -0.10767824202775955, -0.11632774770259857, 0.44374653697013855, -0.9433631896972656, -0.704586923122406, 0.822644829750061, -0.15661588311195374, 0.23451988399028778, 1.567888855934143, -1.6627217531204224, 1.885576844215393, 0.2419654279947281, -0.2807934582233429, -0.5353670716285706, 1.2824673652648926], [-0.8114326000213623, 1.599832534790039, 0.5262374877929688, -0.7273099422454834, 0.09767726063728333, 1.6231541633605957, -0.6864725351333618, 0.027253108099102974, 0.4126356244087219, -1.7544500827789307, 0.2820635437965393, -0.7956653237342834, -1.8032876253128052, -0.2650828957557678, 0.032454971224069595, 0.39081764221191406, 0.41814273595809937, -2.0272703170776367, -0.8666491508483887, -0.21074798703193665, 1.3675305843353271, 1.4864622354507446, 0.7640385031700134, 0.59914630651474, 0.613558292388916, -0.9304370284080505, -0.9994388222694397, -0.8440707325935364, 1.7363861799240112, 0.3840389549732208, 1.9338011741638184, 0.9472395181655884, -0.3148956894874573, -0.9020364284515381, -1.1571778059005737, 0.8162082433700562, 1.360990047454834, -1.1046069860458374, 0.23207272589206696, -2.0946731567382812, 1.883009433746338, 1.3138419389724731, -0.8262576460838318, 1.436781406402588, -2.0275938510894775, 0.6066951155662537, 2.215472936630249, -0.1159684807062149, 0.6257598996162415, -0.9938229322433472, -1.3892934322357178, -0.10193012654781342, -2.186152696609497, 0.4027389585971832, 0.6332551836967468, -0.7474673986434937, -0.1312326341867447, 0.5718339085578918, -0.569966733455658, -2.523205518722534, 0.20894362032413483, -0.27728256583213806, 0.009944080375134945, 0.39087530970573425], [0.7993833422660828, -0.12863300740718842, 0.18955908715724945, -1.3321443796157837, 0.23256541788578033, 0.2282535433769226, 1.5289649963378906, 0.016786551102995872, -0.994415283203125, 0.22074763476848602, -0.8770140409469604, 0.8260025382041931, -1.1259996891021729, 0.728998064994812, -1.09579598903656, 0.34256961941719055, -0.29400721192359924, 0.7252078056335449, -0.3186534345149994, 1.6956067085266113, 0.21274703741073608, 2.016953229904175, -0.04880733788013458, -0.6921507716178894, -1.5560009479522705, -0.3102198541164398, 1.455230474472046, 1.1596143245697021, -0.09182517230510712, -0.4307383596897125, -0.8538825511932373, -2.195333957672119, -0.1443566381931305, 0.02250385843217373, -2.6999027729034424, -0.08752203732728958, 0.9183416962623596, -0.367982417345047, -0.946708083152771, -1.3483190536499023, 0.5247573852539062, -1.5310492515563965, -0.2105513960123062, 0.8871996402740479, 0.5806781649589539, -0.6121891736984253, 2.0676956176757812, 0.95213782787323, -0.37730327248573303, 1.1607152223587036, -0.44793635606765747, -0.1565670222043991, -0.36013084650039673, 0.06367137283086777, -0.26042622327804565, 0.07463255524635315, -0.5831764936447144, -1.5802112817764282, 1.6537481546401978, 1.0578902959823608, -0.680096447467804, 0.8417392373085022, -1.7909201383590698, -0.6824930906295776], [-0.3860360383987427, 2.2393791675567627, 0.7981628179550171, -1.5698333978652954, 1.207802176475525, -0.293691486120224, -1.160921573638916, -0.31022703647613525, 0.12323755025863647, -0.02564331889152527, 0.47082018852233887, -1.3165810108184814, -1.4121360778808594, -1.511619210243225, 0.03180401399731636, -1.5008854866027832, -2.716010332107544, -1.720386266708374, 0.055718179792165756, 1.3873976469039917, -0.46281710267066956, 0.04629076272249222, -3.116281509399414, -0.9826186895370483, -0.050372421741485596, -2.130275011062622, -1.2835805416107178, -1.5847707986831665, -1.0427964925765991, -0.5448059439659119, 0.777920126914978, 0.6889644265174866, 0.03978734463453293, 1.0700914859771729, 0.8550854921340942, -0.3704870045185089, 0.8823718428611755, -0.8714134693145752, -0.6722355484962463, -0.6445229053497314, 1.3318589925765991, 1.6684036254882812, -0.20422741770744324, -0.7559636831283569, 1.630163311958313, -1.3599754571914673, 1.0196317434310913, 1.0099365711212158, -0.935931384563446, 0.7374682426452637, -0.9768359661102295, 0.15888841450214386, -0.5316709280014038, 0.24945595860481262, -0.5163000226020813, -1.0036265850067139, 0.08553727716207504, 0.5021854043006897, 0.26352304220199585, -1.1763967275619507, 1.45401930809021, -0.49531230330467224, 0.4139789938926697, -1.509590983390808], [0.02922930382192135, -0.7845267653465271, -0.26429134607315063, 0.7711001634597778, -0.32711610198020935, -0.24364882707595825, 0.7697823643684387, 0.654293417930603, -0.7209075689315796, -0.48013821244239807, -0.30949243903160095, -0.7779231071472168, -0.5071263313293457, 0.669116735458374, -0.8301469087600708, 0.32363635301589966, -0.8429864645004272, -1.8412413597106934, 0.014578239060938358, 0.2792896628379822, 0.3946053683757782, 0.2727809250354767, 0.2043139934539795, -1.8427318334579468, -0.19553670287132263, 0.7851943969726562, 0.6651443839073181, -0.48408615589141846, 0.18360790610313416, 0.6163252592086792, -0.18408794701099396, 0.19549451768398285, 0.7876718044281006, 0.37199991941452026, -0.376059353351593, -1.5351347923278809, -0.6926277875900269, 1.2717437744140625, 0.16190160810947418, 1.3409281969070435, -0.16671840846538544, 1.6927084922790527, -0.9671788215637207, -2.030660390853882, 0.5594179630279541, -0.2294597029685974, 0.5863131284713745, 2.3262641429901123, 0.23985205590724945, 0.6924425959587097, -2.3226137161254883, 0.17602641880512238, -0.659589409828186, -0.12207882851362228, 1.175907850265503, 1.0523064136505127, -1.2019530534744263, 0.0763273611664772, 1.4885449409484863, 1.1559340953826904, 2.6572163105010986, -1.1624103784561157, 0.4612484276294708, -1.321036458015442], [1.7450510263442993, 0.986880362033844, 1.653126835823059, -0.011849156580865383, 1.52272629737854, -1.8542643785476685, 0.4662266671657562, -0.9300990104675293, -0.471099317073822, -0.944743275642395, 0.2866274118423462, -0.11995033919811249, -1.0095988512039185, -0.5882830619812012, 1.5538690090179443, -0.16357415914535522, 0.9499030709266663, -1.2265547513961792, 0.5321350693702698, 2.1639459133148193, -0.5302110314369202, 1.5192726850509644, -1.1161305904388428, -0.5493622422218323, -0.4560864567756653, 1.2562434673309326, -0.5186563730239868, 0.2802062928676605, 0.2860613465309143, 0.8250306248664856, 0.70155268907547, 0.7440950274467468, -0.6963512897491455, 0.5773136019706726, -0.8141161203384399, 1.112352967262268, 2.310034990310669, 0.8686843514442444, -0.6404116749763489, 0.385394811630249, 1.7288856506347656, -0.35008805990219116, -0.20412391424179077, -1.8933372497558594, 0.39188599586486816, 0.44304341077804565, 0.7949419617652893, 0.3255631923675537, -0.5739547610282898, -0.32388579845428467, 1.1215075254440308, -0.9675177931785583, 0.870956301689148, 1.193484902381897, -0.24408946931362152, -0.811702311038971, 1.2436951398849487, -0.8345035314559937, 0.43327125906944275, 0.13400191068649292, 2.1111207008361816, -1.4219677448272705, 0.343851238489151, -0.0033722633961588144], [-1.696720838546753, 1.030527949333191, 0.7584152221679688, 1.3706743717193604, 0.15924565494060516, -0.19785307347774506, -2.7488269805908203, -0.20940187573432922, 0.8139193654060364, 0.5383821725845337, 0.9178776741027832, 0.804352879524231, 0.07362484931945801, 0.9257495403289795, 1.2125438451766968, 1.78835129737854, 1.1556674242019653, -1.2842984199523926, -0.10716695338487625, -0.43689343333244324, -0.1864020824432373, -0.1813475340604782, -0.431944340467453, 0.4579634368419647, -0.732151985168457, 0.9046237468719482, -0.37622570991516113, 0.5984915494918823, 1.7246321439743042, -0.6182004809379578, -0.7257846593856812, 0.41318395733833313, -0.5916987657546997, 1.2419788837432861, -0.6401644349098206, -1.2300944328308105, -1.7153570652008057, 0.5050442218780518, 0.025320174172520638, 0.4976181089878082, -0.5456374883651733, 1.0944087505340576, 0.5327829122543335, 0.4804919958114624, -0.36796173453330994, 0.8680396676063538, 0.9810174703598022, 0.030857644975185394, -0.41371411085128784, -1.4472064971923828, 0.13510458171367645, 0.3576219379901886, -2.1001596450805664, 0.9628671407699585, 0.017105937004089355, 0.43521183729171753, -0.05658021569252014, 0.31416499614715576, -0.9781015515327454, 0.5292739272117615, 1.3921562433242798, 0.01897219382226467, 0.9779782295227051, 1.0067816972732544], [-0.030249545350670815, 0.4753277599811554, 0.9507842063903809, 1.3076082468032837, 0.06295165419578552, 0.04176028445363045, 0.5392575263977051, 1.2231507301330566, -1.2283525466918945, -0.16824668645858765, -0.3854289650917053, -0.28476548194885254, -0.07915052771568298, 2.8785505294799805, 1.4612194299697876, -0.19875295460224152, -0.9231060743331909, -0.013384596444666386, 1.2409605979919434, 0.12866102159023285, 2.8026037216186523, -0.21104800701141357, 0.14552561938762665, -0.5822535753250122, -0.2028440237045288, -0.30192112922668457, -0.23937352001667023, -0.7505449056625366, 0.11602012813091278, 1.5550940036773682, 0.28497713804244995, 0.7474026679992676, -0.42779606580734253, 0.24771054089069366, 2.0823683738708496, 0.23117922246456146, 0.18233439326286316, 0.04364030063152313, 1.8159847259521484, 0.08293744921684265, -0.695847749710083, -0.4547079801559448, -0.9184409379959106, 0.15080678462982178, 0.3154030442237854, 0.5079617500305176, -0.11693218350410461, 1.32901930809021, -1.9290220737457275, -1.4990497827529907, 0.9187886714935303, -0.4144882261753082, 1.1387877464294434, -0.4035370349884033, 0.22219032049179077, -0.740283727645874, 0.9294962882995605, -1.4107085466384888, -1.2234513759613037, 1.6961268186569214, 1.690972089767456, -0.5836588740348816, -1.1422126293182373, 0.13303498923778534], [-1.4108716249465942, 0.1828860640525818, -1.1240254640579224, 1.533991813659668, -2.324866771697998, 0.6490567922592163, -0.9062716364860535, 0.004979224875569344, -0.9451817274093628, 0.5633812546730042, 2.16072940826416, 0.20505498349666595, -1.6588581800460815, 0.7584505677223206, -1.5419265031814575, -0.17256535589694977, -0.6881001591682434, 0.6118456125259399, -0.22925905883312225, -0.1647464781999588, -1.3837560415267944, -0.6096127033233643, 0.2735481560230255, -0.19856107234954834, -1.1467297077178955, -0.5320961475372314, 1.492703914642334, 0.7237442135810852, -0.9457449913024902, -0.6220853328704834, 0.3145240545272827, -1.5579167604446411, 0.32071584463119507, -0.01639442890882492, -0.8546575307846069, -0.3661997616291046, -0.3350721001625061, 0.7678708434104919, 1.7610300779342651, 1.6762068271636963, -0.6168201565742493, -0.9616671204566956, 0.020887333899736404, 1.8675955533981323, -0.37542182207107544, 2.3292996883392334, 0.2577473223209381, 0.4593338370323181, 0.6257226467132568, -0.3728519678115845, 2.048265218734741, 0.28472328186035156, -0.628505527973175, -0.5651715397834778, 0.06459107249975204, -0.48527729511260986, 0.7135089635848999, -1.0739818811416626, 0.18412794172763824, 0.45235130190849304, -0.4580146074295044, 0.05716221407055855, -1.4560284614562988, -0.5829432010650635], [-0.1326970010995865, -0.04888685420155525, -0.7640264630317688, 0.2721378207206726, 1.4974644184112549, 1.5196727514266968, -0.3584873378276825, -0.5694906115531921, 0.6128942966461182, -0.6946452260017395, 0.11597925424575806, 1.1633453369140625, -0.45803266763687134, 0.4177353084087372, -0.5394439697265625, 1.7205623388290405, -0.3551100194454193, -1.7219406366348267, 1.212986946105957, -0.11504779011011124, -0.6407174468040466, 0.3851131200790405, -0.46501436829566956, -0.8415886163711548, -0.0029428377747535706, -0.6661713719367981, 0.45660606026649475, 1.3017572164535522, 0.8571949005126953, -2.702549457550049, 0.2621246576309204, -0.7406402826309204, 0.6054743528366089, 0.17128266394138336, -0.16874398291110992, -1.1683365106582642, 0.2780674695968628, -0.344367653131485, -1.0418620109558105, 1.1940155029296875, 0.6807681918144226, -1.6215894222259521, 0.00312986527569592, 0.5036628246307373, -1.1232820749282837, -0.3601806163787842, 0.13324379920959473, -0.14531825482845306, 1.867323637008667, 0.5497026443481445, -0.5591031908988953, -1.1575978994369507, 0.32991573214530945, 0.9778433442115784, 0.16626101732254028, 0.8016313314437866, -0.27017948031425476, -0.7845965027809143, 0.6285000443458557, -2.5242464542388916, 0.489499568939209, 0.19479916989803314, -1.638956904411316, -0.6373911499977112], [1.3341833353042603, -0.33823320269584656, -0.4304102957248688, 0.5842949151992798, 0.11967534571886063, -0.05082176253199577, 1.311588168144226, 2.0556139945983887, -0.9137547612190247, 0.35753464698791504, -0.5414468050003052, 0.513632595539093, 0.1600390523672104, 1.3463879823684692, -0.6788040995597839, -0.9495512247085571, 0.1965666264295578, -0.6439684629440308, 0.7696995735168457, 1.351690649986267, -0.7573937773704529, 1.8528207540512085, 0.7676196694374084, 0.6956220865249634, 0.36828136444091797, 1.9273571968078613, -1.488244891166687, -0.4807227551937103, 0.3741452395915985, 1.3334451913833618, 0.12409937381744385, 1.4745960235595703, -0.3255132734775543, 2.3261396884918213, -0.12635037302970886, -1.435968041419983, -2.9773473739624023, -1.1007760763168335, 0.09939097613096237, 0.3396559953689575, -0.20973795652389526, -0.3958894908428192, -1.6541575193405151, 1.9945327043533325, 0.4492500126361847, 0.7341824173927307, 0.16945375502109528, -0.789523184299469, 1.9698611497879028, 0.7816551327705383, -0.2218179553747177, 0.055068593472242355, -1.172121286392212, 0.16201405227184296, 0.5623667240142822, -1.7108783721923828, -0.19245648384094238, -0.3593538999557495, 0.41179919242858887, -0.7942135334014893, 0.16719220578670502, 0.863255500793457, 0.43674954771995544, 0.7900652289390564], [-0.3593394458293915, 0.6849796175956726, 2.027381420135498, 0.08470486104488373, 1.0952885150909424, 0.15916475653648376, 1.288326621055603, 0.7110824584960938, -1.6824524402618408, -0.5200340151786804, 0.08822253346443176, 1.9891879558563232, -0.9272665977478027, -0.6769561171531677, 0.15273502469062805, -0.43885836005210876, -1.9973843097686768, 1.8778172731399536, -1.6148600578308105, -0.17423538863658905, 1.0551207065582275, -0.35503098368644714, 1.8380532264709473, -0.10064506530761719, 0.23217163980007172, -0.6372149586677551, 0.3006344437599182, -0.23237207531929016, 0.2948583960533142, -0.9045148491859436, -1.2208260297775269, 0.1841665655374527, -0.36234647035598755, 0.2816247045993805, 1.4159290790557861, 1.0530415773391724, 0.23093906044960022, -0.25032126903533936, -0.021411672234535217, -0.22464625537395477, 0.009439623914659023, -1.2974756956100464, 0.2785795331001282, 0.7165442705154419, 0.4636942446231842, -0.26643848419189453, 0.5187588930130005, -0.9527766704559326, 0.1260945349931717, -0.558419406414032, 0.3176270127296448, -0.7333213090896606, 0.6527814865112305, -1.4303261041641235, -1.1741182804107666, 0.9959042072296143, 1.1485334634780884, 1.2802047729492188, -0.1810648888349533, -0.243263378739357, 0.15394245088100433, -1.2952247858047485, -1.4079605340957642, -0.43756088614463806], [-0.1823255866765976, 0.566652774810791, -1.4033396244049072, -0.23363667726516724, -0.6998448967933655, -0.6065141558647156, 0.368181973695755, 1.0989269018173218, 0.07774760574102402, 1.4425138235092163, 1.2940518856048584, 0.567947506904602, -0.4261511564254761, 0.6659110188484192, 0.25724878907203674, 1.0453555583953857, 0.47016358375549316, 0.5183021426200867, 1.084848403930664, -0.19121944904327393, 1.5314849615097046, 1.6265658140182495, 0.40836846828460693, 1.1471290588378906, 0.39397668838500977, -0.1763550490140915, -0.04941317066550255, 0.8571588397026062, 0.22629055380821228, -1.0085809230804443, -2.7553162574768066, 0.769681990146637, 1.136125087738037, 0.9178394079208374, 1.1822010278701782, -0.9524401426315308, 0.6143597364425659, 0.22020725905895233, -2.7223455905914307, -1.0516537427902222, 0.10464563220739365, -0.4595419466495514, 1.4443159103393555, -1.3389824628829956, -1.08345627784729, 0.2032710164785385, -1.23228919506073, -1.9264575242996216, -0.2959553897380829, 1.2515672445297241, 0.5184919834136963, 0.589664101600647, -0.7510051131248474, 0.7518792152404785, -0.14790616929531097, -0.40494561195373535, -1.5509827136993408, 0.362247109413147, -1.121033787727356, 0.34028181433677673, -0.3613974452018738, 0.12113828957080841, -2.42637038230896, 0.5681187510490417], [-0.9831698536872864, 0.8180142641067505, -1.1568617820739746, -0.2548094689846039, -1.2119660377502441, -0.8769432306289673, -0.42397260665893555, -0.9600486159324646, 0.7923812866210938, -0.04175981879234314, 1.1901270151138306, 1.166817307472229, 1.0373886823654175, 0.3208216428756714, -0.7837279438972473, 0.0049916538409888744, -0.3516146242618561, -1.0272222757339478, -2.063110589981079, 1.0523825883865356, 0.43575748801231384, 1.8247833251953125, -1.2926851511001587, -0.8804047107696533, 0.10600446909666061, 0.055950019508600235, -0.5436933040618896, 1.0549659729003906, 0.41185322403907776, -0.7521557211875916, 2.1552233695983887, 1.319221019744873, 1.660681962966919, -0.43176934123039246, 1.827772855758667, -1.9261739253997803, 0.13642817735671997, -0.51446932554245, 0.668509840965271, 0.6007004380226135, -1.5210232734680176, -1.2904536724090576, 1.3276559114456177, -0.13188612461090088, -1.5529429912567139, 0.8864544630050659, 1.8336540460586548, 0.26186925172805786, 0.4453616440296173, 0.7272136211395264, -1.0892794132232666, 0.05510652810335159, 1.2252418994903564, -0.09268414229154587, 0.7361107468605042, -0.6160492300987244, -0.7043461203575134, -0.557452917098999, 0.3072155714035034, 1.298877477645874, -0.6266658306121826, 2.686412811279297, -1.3900623321533203, -0.7630143165588379], [0.10852570086717606, -0.15451815724372864, 0.4369381070137024, 1.5798003673553467, 1.8518882989883423, -1.1099140644073486, -0.8544309139251709, -0.30829697847366333, -0.1807887703180313, 1.300588607788086, -1.7545559406280518, -1.0318951606750488, 0.601005494594574, -0.7848004102706909, 0.04705428332090378, 1.6680967807769775, 0.9454721212387085, 0.04532109946012497, -0.06156693771481514, 1.363105297088623, -0.27266499400138855, 0.3739325702190399, -0.432334303855896, 2.6637775897979736, -1.4016810655593872, 1.371817946434021, 0.7511764168739319, 1.9803444147109985, 0.9088612198829651, -0.3274039924144745, 0.17849485576152802, 0.2394736260175705, -0.8872256875038147, -0.3318813741207123, -0.0014689977979287505, -0.11766355484724045, -0.12439125031232834, -1.3288079500198364, 1.5689016580581665, -1.3673175573349, 1.1046546697616577, 1.02963387966156, -0.5319191217422485, -0.6822406649589539, 0.23081821203231812, -1.1566314697265625, -0.920793354511261, 0.404547780752182, -0.04319775104522705, 0.5235705971717834, 1.5969218015670776, -0.3234572410583496, 0.09155069291591644, -0.6510645747184753, -0.8257325887680054, -0.6150218844413757, 1.036547064781189, 0.3083893060684204, -0.3624905049800873, -0.12275195121765137, 0.15562812983989716, 0.11209411174058914, 1.4960583448410034, 1.5582234859466553], [0.21812964975833893, -0.441549688577652, 0.20917697250843048, 1.5308929681777954, 1.4942100048065186, 0.49443838000297546, 1.2314292192459106, 0.9002730250358582, 2.0333192348480225, 1.7955374717712402, -0.48749569058418274, -0.7938954830169678, -1.7716175317764282, 0.35368770360946655, 0.05963873490691185, 1.3118823766708374, -0.7580902576446533, -1.112557291984558, -0.276101291179657, 0.3890974223613739, 0.8318049311637878, -1.0260417461395264, 0.217512845993042, -1.540846347808838, -0.7282459735870361, -1.014757513999939, 1.412809133529663, 0.842007040977478, 1.7209523916244507, -1.3916325569152832, 1.1012295484542847, 0.053958844393491745, 0.0334920696914196, -0.0749221220612526, -0.08375166356563568, -2.0369505882263184, -0.24179083108901978, -0.9612694978713989, 0.2739901542663574, 0.45940712094306946, -0.28379231691360474, -0.29342207312583923, 0.22649312019348145, -1.0971643924713135, 1.466886281967163, 0.5270691514015198, -0.3349114656448364, -0.6029007434844971, 0.8118980526924133, 0.050912804901599884, 1.4239758253097534, 0.7207133769989014, 0.41670066118240356, 0.5475277900695801, 1.318458080291748, 0.5576300621032715, 0.9516969323158264, 0.20598965883255005, -0.4996925890445709, 0.3867937922477722, -0.38521862030029297, -1.4148280620574951, 0.4999352991580963, 1.8863214254379272], [1.0988746881484985, 0.01394615601748228, 0.9485700130462646, -0.881503701210022, -0.21733662486076355, -1.3761931657791138, -0.47942376136779785, 1.0341265201568604, -1.100775122642517, -0.6454827785491943, -2.5477070808410645, -1.1736348867416382, -0.38565772771835327, 1.7251455783843994, -0.43765848875045776, 2.384702205657959, -0.5148245692253113, -0.6294180154800415, 0.8988837599754333, 1.487357258796692, 0.008784450590610504, -1.196340799331665, 1.9510810375213623, -0.35638922452926636, 0.04764612764120102, -0.4901234209537506, 0.6321189403533936, -1.125218152999878, 0.7493025660514832, 0.14226290583610535, 0.01887231506407261, -0.47572842240333557, -0.5875129699707031, -0.5841472148895264, 0.44408977031707764, 1.3709959983825684, -0.4901117980480194, -1.3046151399612427, -0.2489718198776245, 0.5654173493385315, 0.2082262933254242, 0.19990937411785126, -0.206661194562912, 1.1269748210906982, 0.4872479736804962, 0.7555330991744995, 1.1149970293045044, 0.23148061335086823, 1.8431349992752075, -0.18134216964244843, 0.12369119375944138, -0.08229298889636993, 0.8013916015625, 0.6105961799621582, -0.5956799983978271, 0.7207967042922974, -0.7511367797851562, -0.7107258439064026, -0.05938693881034851, -0.4378662407398224, -0.31975802779197693, 0.3190523087978363, 0.7521347403526306, -0.23075781762599945], [0.2337714433670044, 0.20004433393478394, -1.5914441347122192, 0.45048728585243225, -2.425462484359741, -0.8825897574424744, -0.4680009186267853, -0.16826994717121124, -0.41224512457847595, -0.981753408908844, 1.489058256149292, -1.088421106338501, 1.7720924615859985, -2.4048564434051514, 0.7117307186126709, 1.0567317008972168, -1.0092179775238037, -0.9402629733085632, -0.7378314137458801, -1.1200981140136719, 0.45418617129325867, 0.06135914474725723, -1.7230324745178223, 0.45245596766471863, 0.7364000678062439, -0.20411477982997894, 0.5108510255813599, -2.3857405185699463, 0.01076574344187975, 1.0602858066558838, -1.5544394254684448, 0.279234379529953, -1.3906902074813843, -1.0130910873413086, -1.0994336605072021, -0.45089080929756165, 0.44929373264312744, -0.18182608485221863, 0.977759599685669, 0.2671984136104584, 0.20045562088489532, 0.18085168302059174, -0.7990477681159973, 0.6696006655693054, 0.16295836865901947, -0.5598335266113281, -0.9772486686706543, 0.4092325270175934, -0.20618204772472382, 0.7634448409080505, 0.3492611348628998, -0.5505361557006836, 0.5347740054130554, -0.10675269365310669, 0.9697577953338623, -0.1773938536643982, 0.21192991733551025, -0.008450384251773357, -0.4180390238761902, -0.28666922450065613, -1.3800841569900513, -0.16390427947044373, -0.1582326889038086, -0.7189168930053711], [0.5255079865455627, -1.704345464706421, -1.2278920412063599, 0.5220317244529724, -1.576123595237732, -1.2631161212921143, -0.008618533611297607, -0.01671990379691124, -0.9897001385688782, -1.0264384746551514, -1.9754852056503296, -0.5916667580604553, -0.4100803732872009, -0.8783711791038513, 1.0412031412124634, -0.4501771628856659, -1.3943977355957031, -0.7655030488967896, 1.6958861351013184, 0.36734628677368164, -0.6173068284988403, -1.9638880491256714, -0.3601709306240082, -1.5536929368972778, -0.19317124783992767, 1.3364022970199585, -0.20156200230121613, -0.4159125089645386, 1.2450406551361084, -0.6394249200820923, 2.1132185459136963, -0.009291826747357845, 1.5250544548034668, 0.6452893018722534, -0.44909584522247314, 1.1460636854171753, 2.5056114196777344, 1.4515118598937988, 1.3114279508590698, 0.5381142497062683, 0.47975218296051025, -3.499114513397217, -0.7131935954093933, -0.007389349397271872, 0.8957571387290955, 0.5654467940330505, -1.031478762626648, -0.5646036267280579, -1.0852233171463013, -0.4995107352733612, -0.2872120440006256, 0.22715847194194794, -1.1801979541778564, 0.6335995197296143, -0.7647733092308044, -0.8940539360046387, 0.691973090171814, -1.404420256614685, -1.222594141960144, 0.0285302996635437, 0.01351602841168642, -0.6734514832496643, -0.5255559086799622, 1.1186774969100952]], "dtype": "float32"}, {"__nd__": [0.8603241443634033, -0.922868549823761, 0.003818538738414645, 0.7188116908073425, -1.056021809577942, -0.2892063558101654, -0.8454289436340332, 0.8680481910705566, -0.4438724219799042, -0.7771818041801453, 1.600706934928894, -1.968608021736145, -0.747082531452179, -0.4042791426181793, 0.06803059577941895, 0.44428756833076477, -1.6275495290756226, -0.5979975461959839, 0.3490310609340668, -0.8931693434715271, -0.8910256028175354, 1.5708844661712646, -0.9941556453704834, 1.0027328729629517, -0.3177110552787781, -0.15680018067359924, -0.8451589941978455, 0.33538344502449036, -0.4040396511554718, -0.5229804515838623, 0.7958060503005981, -0.6546374559402466, 0.526322066783905, 0.9255647659301758, 0.32572612166404724, 1.1696741580963135, 2.2190299034118652, -0.40847718715667725, -1.317344069480896, 0.951530933380127, -1.4800529479980469, 1.8681094646453857, 0.3284796476364136, -2.0653088092803955, 0.3096579313278198, 0.17249566316604614, -0.27384838461875916, -0.2773526906967163, -0.5630919933319092, -0.6604528427124023, 1.1516281366348267, 1.6742509603500366, -0.4782508611679077, 0.39718320965766907, 0.32221001386642456, 0.37382179498672485, -1.6131799221038818, -0.7826327085494995, 0.46795496344566345, 1.593390703201294, -0.37965863943099976, -1.3443999290466309, -0.920605480670929, 0.5095983743667603], "dtype": "float32"}, {"__nd__": [-0.03748594596982002, 0.09823299944400787, 0.07782219350337982, 0.14903593063354492, 0.1109219640493393, -0.057933297008275986, -0.25481271743774414, -0.020162872970104218, 0.023306654766201973, 0.012088891118764877, -0.12440051883459091, 0.03002195432782173, 0.004844697192311287, -0.15887850522994995, 0.0018536988645792007, 0.00968166533857584, 0.03815765678882599, 0.007435292936861515, -0.09969453513622284, 0.11415177583694458, 0.04368631914258003, -0.09669564664363861, 0.09345895051956177, 0.18421879410743713, 0.1801884025335312, 0.09296876937150955, 0.09150753170251846, -0.04801948368549347, -0.123061403632164, -0.02473653294146061, -0.12353740632534027, -0.11248951405286789, -0.1422606259584427, -0.030698996037244797, 0.08688428997993469, -0.047290075570344925, 0.08641493320465088, 0.005861243698745966, 0.030326958745718002, -0.03773575276136398, 0.04364445433020592, -0.027447110041975975, -0.11828815937042236, 0.03813914209604263, 0.17391034960746765, 0.045807573944330215, -0.08649589121341705, 0.027483204379677773, -0.12682807445526123, -0.07536403834819794, -0.00796503946185112, 0.004150786437094212, -0.09000425785779953, 0.06648865342140198, 0.07559528946876526, 0.10891739279031754, -0.029357116669416428, -0.2829529047012329, 0.14611835777759552, -0.12769462168216705, -0.04258176311850548, 0.3765619397163391, 0.018841169774532318, 0.23093651235103607, -0.03839299455285072, 0.03506850451231003, 0.2265794724225998, 0.09257975220680237, 0.06698612123727798, -0.0244307778775692, -0.005448489915579557, 0.10136901587247849, 0.10478559136390686, 0.11961698532104492, -0.0614585354924202, 0.18101349472999573, -0.05486595258116722, -0.1364152878522873, -0.05361206829547882, 0.18469837307929993, 0.0036206000950187445, 0.022170308977365494, -0.03492164984345436, -0.06294628977775574, -0.2491702437400818, 0.026245927438139915, 0.21239154040813446, 0.12551768124103546, 0.18939122557640076, -0.061658550053834915, -0.06238264963030815, 0.16550108790397644, 0.016076065599918365, 0.018521897494792938, 0.056555185467004776, 0.16506727039813995, 0.17678801715373993, 0.062390025705099106, -0.2362348586320877, -0.19391882419586182], "dtype": "float32"}, {"__nd__": [1.0293121337890625, 0.9659781455993652, 0.9952743053436279, 1.031930685043335, 0.984890341758728, 1.008832335472107, 0.9749612212181091, 1.0585885047912598, 0.9990276098251343, 0.9029942154884338, 0.9945592880249023, 0.9966213703155518, 1.0144174098968506, 1.0622481107711792, 1.032760739326477, 0.9056804180145264, 0.9108545184135437, 0.8081198930740356, 0.9848690629005432, 0.9613773822784424, 1.1902580261230469, 1.1149364709854126, 1.0750608444213867, 1.0219860076904297, 1.0794204473495483, 1.1873011589050293, 1.138077735900879, 1.1418383121490479, 1.1151378154754639, 0.9898149371147156, 1.1149417161941528, 0.8729206323623657, 1.0346508026123047, 1.10420823097229, 0.9593539834022522, 1.051639199256897, 0.9585997462272644, 1.022442102432251, 1.1993850469589233, 0.901010274887085, 0.9549848437309265, 0.9283050894737244, 1.0427175760269165, 1.0697953701019287, 0.9633549451828003, 1.0496302843093872, 1.1710618734359741, 1.1200281381607056, 1.1124601364135742, 0.9096775650978088, 0.9564008712768555, 1.1794493198394775, 1.0186370611190796, 1.0458661317825317, 0.9643588662147522, 0.9997382760047913, 0.8852866888046265, 0.9618487358093262, 1.0369242429733276, 0.8809661865234375, 0.9866356253623962, 1.0856798887252808, 1.086366891860962, 1.0521491765975952, 1.140674352645874, 1.0281480550765991, 1.1354551315307617, 0.9623162746429443, 0.992501437664032, 0.955039381980896, 0.9513437151908875, 1.0154426097869873, 1.103010892868042, 0.9512400031089783, 1.2173817157745361, 1.0295056104660034, 1.031889796257019, 1.0958456993103027, 0.9538917541503906, 1.1089112758636475, 1.0793931484222412, 0.9924684762954712, 0.8959013223648071, 1.0001280307769775, 0.912843644618988, 1.0279736518859863, 1.008358120918274, 1.0797622203826904, 1.0199167728424072, 1.0020346641540527, 1.0725717544555664, 0.9733332395553589, 1.0839735269546509, 0.9806080460548401, 0.9365794658660889, 1.0450453758239746, 1.050179362297058, 1.1077322959899902, 1.1110731363296509, 0.91323322057724], "dtype": "float32"}, {"__nd__": [[-0.4986637532711029, 1.6839343309402466, -0.76068514585495, -0.7407041192054749, -2.035062313079834, 0.7977936863899231, -0.45297181606292725, -0.028914261609315872, 0.7584291696548462, 0.35421690344810486, 0.8438050746917725, -0.4912603497505188, -0.057247359305620193, -0.12540876865386963, -1.0507124662399292, 0.5232448577880859, -1.0888134241104126, 0.48500072956085205, 1.2719812393188477, -1.1559383869171143, -0.8196009397506714, -1.3237276077270508, 1.198297142982483, -0.5365490317344666, 1.4615797996520996, -1.5836358070373535, 0.11584536731243134, -1.0965322256088257, 0.6946904063224792, 0.10688728839159012, 0.45206496119499207, -0.20633408427238464, 0.27713462710380554, -0.354095458984375, 1.2834690809249878, -0.1508815586566925, -0.47697722911834717, -1.3242958784103394, -0.5087386965751648, 0.34784141182899475, -0.24037276208400726, -0.3116915822029114, -0.8957418203353882, 0.5973321795463562, 0.4843323528766632, -0.18806958198547363, 1.8868526220321655, -0.27497154474258423, 2.000925302505493, -0.9551772475242615, -2.8395535945892334, -2.297354221343994, 2.1889853477478027, 0.2606988847255707, -0.5047484636306763, -1.3369407653808594, -0.6377161741256714, -0.01367354579269886, 1.8580092191696167, 1.3118202686309814, -1.0211268663406372, -0.5650491118431091, 1.7513989210128784, -0.5885075330734253], [-1.0708189010620117, 0.584163248538971, -0.15644071996212006, 0.04107052460312843, -1.159082293510437, -0.9809304475784302, 0.6127468943595886, -0.5020880103111267, -0.6234365701675415, -0.9813068509101868, 1.1484053134918213, -0.6101510524749756, 1.0002140998840332, -0.781317949295044, -0.12057775259017944, 0.7588281035423279, 0.2441624253988266, -1.0360286235809326, 0.49434641003608704, -0.714539110660553, 1.0080037117004395, 1.5019899606704712, 1.2440401315689087, -0.26679182052612305, -0.8125927448272705, 0.8139907121658325, -0.41996532678604126, -0.9316489100456238, -0.3076941668987274, -0.27185967564582825, -0.1432885378599167, 0.8865298628807068, -0.5428823828697205, 0.2559036314487457, 0.6993889808654785, -0.8593087792396545, 2.07646107673645, -0.23902501165866852, 1.6305549144744873, -1.0372891426086426, -1.3632313013076782, 0.3908993899822235, -1.507952094078064, -1.3831520080566406, 0.18862706422805786, 1.2018489837646484, 0.07115955650806427, 1.5162020921707153, -1.0045191049575806, -0.9791062474250793, -0.05932645499706268, -0.5833611488342285, -0.8108490109443665, 0.8097708225250244, 1.1629379987716675, -1.0346884727478027, 0.5198600888252258, -0.5130004286766052, -1.3111674785614014, 1.188890814781189, -0.9534302353858948, -0.48829129338264465, -0.09882408380508423, 1.2216585874557495], [-0.3119463622570038, -0.9672948718070984, 0.0017842971719801426, 0.8144246339797974, 0.20135405659675598, -0.41400575637817383, -0.995505154132843, -0.9230552911758423, 0.4215821921825409, -1.0136677026748657, 0.5998817682266235, -0.26707684993743896, 0.4714749753475189, -0.8799521327018738, -0.5178195834159851, 0.29721277952194214, 0.6539813280105591, -2.041633367538452, 0.5827357172966003, -0.7140840291976929, 1.1685079336166382, 0.99421226978302, 0.19189877808094025, 1.1757484674453735, -1.1744390726089478, -0.8178650140762329, 0.4984246492385864, 0.6590138673782349, 0.13890211284160614, 0.06091063469648361, 0.07509768754243851, 1.3789199590682983, -1.117756962776184, -1.0889288187026978, -0.24581986665725708, -0.604679524898529, 0.20483218133449554, 0.13446849584579468, -0.7125581502914429, -1.9421504735946655, -1.4226447343826294, 0.38086628913879395, 0.39658957719802856, -0.41020819544792175, 0.15794748067855835, 1.0344892740249634, 0.34168991446495056, -1.1865190267562866, -0.4124129116535187, -0.742810070514679, -0.574309229850769, 0.04698781296610832, -0.36336180567741394, -0.1669977605342865, 0.6821762323379517, 0.8943319916725159, 2.0660274028778076, -0.3740576505661011, 0.24321328103542328, -1.0661782026290894, -0.6377027034759521, -1.4961531162261963, 2.8325445652008057, -0.8150985240936279], [0.6778287887573242, 0.8656532764434814, 2.6720869541168213, 0.9694733619689941, -0.5555788278579712, 0.25926029682159424, -0.5834571719169617, -0.6411130428314209, -0.9612820744514465, 0.4200136363506317, -0.6804896593093872, 0.4094107151031494, 0.540887176990509, -1.0891106128692627, -0.7442001104354858, -0.1098119467496872, -0.619917631149292, 0.5407082438468933, -1.759971022605896, 0.6049638390541077, 1.082212209701538, -0.5344545841217041, -2.461717367172241, 0.4146782457828522, 1.1679540872573853, 1.4072469472885132, 2.987536907196045, 1.480688214302063, -0.5786539912223816, -1.7642219066619873, -0.4639089107513428, -0.7986058592796326, -1.3315541744232178, -0.2759804427623749, -0.5104199051856995, 1.6431115865707397, -1.184980034828186, -2.369922399520874, -1.528916358947754, -0.5967735052108765, -1.5573116540908813, 1.0404765605926514, 0.7558208703994751, 0.33436018228530884, -1.605792760848999, -1.0072417259216309, 0.5460047721862793, 1.8331149816513062, 0.11316683888435364, 1.1530652046203613, 0.5049335360527039, -0.33863991498947144, 0.9796260595321655, 1.1041162014007568, -0.7353397607803345, -1.623012661933899, -0.7053000330924988, 1.5997626781463623, 0.4711214005947113, 0.37558799982070923, 0.4024781882762909, 0.5166311860084534, -0.10964139550924301, -1.147903561592102], [-1.5267314910888672, -0.6587466597557068, -0.06518329679965973, 0.8636943101882935, -1.4625437259674072, 1.9399183988571167, -0.3141854405403137, -0.4421583116054535, 0.8350759148597717, -0.2502939999103546, -0.047893721610307693, 0.6554197072982788, -1.065766453742981, -1.9461807012557983, 0.8060727715492249, -0.7639166712760925, 0.2074180692434311, 1.1817525625228882, -1.742408037185669, -0.28031349182128906, 0.82566899061203, -0.4562968611717224, -0.11965759843587875, -0.5304467082023621, -0.8275839686393738, 1.461957573890686, 1.331900954246521, -0.0026566858869045973, -0.07083894312381744, 0.5683669447898865, -0.1289125233888626, 0.6234299540519714, -1.0047649145126343, 0.1154930591583252, 0.5330614447593689, -0.622489333152771, -0.37833771109580994, -0.4330607056617737, -0.9347638487815857, -0.8686068058013916, -0.5821312069892883, 0.6468599438667297, -0.9755319356918335, 0.3812335729598999, -0.26302552223205566, 0.5580178499221802, 1.4298417568206787, 1.1459332704544067, 0.6310956478118896, 1.1481090784072876, -0.551575779914856, 0.16189533472061157, 0.7280798554420471, -0.007528582587838173, -0.28414320945739746, 1.338873028755188, -0.9735424518585205, 0.21705609560012817, -1.109005093574524, -1.0358790159225464, -0.5991775393486023, 0.900761067867279, -0.5540767312049866, -0.532671332359314], [0.9028558135032654, -1.6112918853759766, 1.8674836158752441, 0.23196060955524445, -0.9853804111480713, 0.5452989935874939, -0.2897813320159912, 0.602933406829834, 0.2838706374168396, -0.8175175786018372, -0.4325616657733917, -0.03266724944114685, -0.27722635865211487, -0.0979381799697876, 0.904879629611969, 0.5930275917053223, -0.2703589200973511, -2.112623691558838, -0.5012879967689514, -0.5006358623504639, -1.6585993766784668, 0.1083817109465599, 0.8861939311027527, 0.9132491946220398, 0.8251289129257202, -1.5430054664611816, -0.5872118473052979, 0.5578975677490234, -0.21171154081821442, -1.052587866783142, 1.0996348857879639, 1.5112985372543335, 2.6191086769104004, -0.330099493265152, -1.9317636489868164, 1.0697094202041626, -0.24773472547531128, -0.17977885901927948, 0.22842846810817719, -0.32382404804229736, -0.6959913372993469, -0.37528952956199646, 0.48960214853286743, 0.1940508931875229, 1.3088083267211914, -0.7055346965789795, -0.19824808835983276, 0.41056764125823975, 0.329941064119339, 1.205467700958252, 0.14172498881816864, 0.734559953212738, -0.44525375962257385, -0.7781782150268555, 0.8118801116943359, 0.31056174635887146, 0.1775444895029068, 0.12658251821994781, 0.6851694583892822, -0.4074365198612213, 2.201052188873291, -1.2121820449829102, -0.6683761477470398, -0.4221489131450653], [1.0431098937988281, -2.0286731719970703, 0.03851250931620598, 1.399580478668213, 1.0324301719665527, 0.20375312864780426, 0.6476112604141235, 0.3978542983531952, 0.2170647829771042, -0.4522673487663269, -0.7525295615196228, -0.9450430870056152, 0.9701336026191711, -1.1141828298568726, -0.5180421471595764, -1.1815638542175293, 0.9351151585578918, 0.31289777159690857, -0.3359384834766388, -1.0670344829559326, 0.4559401571750641, 0.498782753944397, 0.11296915262937546, 0.4940173029899597, 0.1288042515516281, 0.29693105816841125, -0.7410931587219238, -0.47988608479499817, -0.4674985110759735, 0.23522992432117462, 2.209430456161499, -0.9795589447021484, 0.899896502494812, 0.10241890698671341, 0.41106539964675903, 0.506463885307312, -0.005914297886192799, -0.8464260101318359, 0.47229671478271484, 0.11619150638580322, 0.8752642869949341, -1.8630322217941284, -0.2393590360879898, 0.33122074604034424, 0.3824084401130676, 0.5386377573013306, 0.38462960720062256, 0.9202245473861694, 0.6781898736953735, 2.1827352046966553, -1.2427653074264526, -0.9400539994239807, -0.7990818619728088, -0.3843561112880707, -0.7082895040512085, -0.8854137659072876, -0.7523916363716125, 0.39598333835601807, -0.21646974980831146, -0.6380995512008667, -1.4830400943756104, 0.08145026117563248, -1.539095401763916, 1.107287049293518], [0.24759109318256378, 1.9328190088272095, -1.2895959615707397, 1.4729105234146118, -1.4774483442306519, 2.031486749649048, -0.8448042273521423, -0.18838226795196533, -0.1410355120897293, 1.703027606010437, -1.2309991121292114, 1.5533955097198486, 0.8884772658348083, -0.19356614351272583, 0.9986573457717896, 0.4053385853767395, 0.7847076058387756, 0.23031969368457794, 0.3110339939594269, -0.8564901351928711, -0.8387864232063293, -0.7864319086074829, -0.9912790656089783, 0.9027934670448303, 1.662131428718567, -0.2611893117427826, 1.185157060623169, -1.641722321510315, -0.2120552361011505, 0.19661664962768555, -0.12546004354953766, 0.834559440612793, 0.8357574939727783, 0.9542919993400574, -0.27161118388175964, 2.5310637950897217, 0.02349579520523548, 0.516741156578064, 0.007041377481073141, 0.9389208555221558, 0.06740698218345642, -0.669476330280304, 0.7793492674827576, 1.0423132181167603, 0.40756991505622864, -0.668481171131134, 0.773278534412384, -0.13876834511756897, -0.43081924319267273, -0.41839921474456787, -0.9611107110977173, -1.941518783569336, 0.34544244408607483, -0.013896103948354721, -1.631589651107788, 0.8181398510932922, 0.3810681998729706, -0.15722724795341492, 0.612347424030304, -0.8123104572296143, -0.6378552913665771, -0.758106529712677, -0.7863688468933105, -0.04440390318632126], [-0.7535128593444824, 0.6917173862457275, 0.8044829368591309, 1.0412027835845947, -0.5619716644287109, -0.09073865413665771, 0.57145756483078, -0.691278338432312, 0.9369285702705383, 0.6033861041069031, 2.3900725841522217, -0.3680310547351837, 1.1384752988815308, 0.7929950952529907, -0.8182755708694458, -1.2712671756744385, 0.28152933716773987, 0.724796712398529, -0.08147849142551422, -1.4068946838378906, 0.9820443391799927, 0.6844727396965027, 0.6212498545646667, 1.5061949491500854, 2.3100509643554688, 0.5280278325080872, 0.9252557158470154, 0.6530547142028809, 0.3419317603111267, 1.1644690036773682, 0.42078936100006104, 0.7052872776985168, 1.9724540710449219, 2.06144642829895, -1.9094233512878418, 1.667413353919983, 0.2201102375984192, -0.6791919469833374, -0.055538203567266464, -2.4793975353240967, -0.7226278185844421, 2.10421085357666, 0.7168921232223511, -0.14160247147083282, 1.629138708114624, -1.2263439893722534, 0.4668344557285309, -1.1588813066482544, 0.6859505772590637, 0.06361504644155502, 1.1367452144622803, -0.8433688879013062, 0.25119084119796753, 1.2764686346054077, 1.0440537929534912, -1.1312843561172485, 0.21866847574710846, 0.10509439557790756, -0.5678649544715881, 0.015659909695386887, -0.9141236543655396, -0.7377257347106934, -0.02734431065618992, 0.18852104246616364], [-0.5049445629119873, 0.4364780783653259, -0.31807154417037964, 1.6842286586761475, -1.3499517440795898, 1.219839334487915, 0.4737536907196045, -1.946498990058899, 0.5850788354873657, 0.22746573388576508, -0.0398033931851387, -0.6016440391540527, -0.4907790720462799, 0.6736002564430237, 1.2647104263305664, 0.5916103720664978, -1.0650866031646729, -0.1921243965625763, -2.1552555561065674, 0.0372849777340889, 0.5274841785430908, -1.4049789905548096, 0.3218563497066498, -0.5868608951568604, -0.45289844274520874, -0.9334269762039185, 0.8560352325439453, 0.7509400248527527, 0.14730414748191833, 0.9651439189910889, -1.267733097076416, 0.30316367745399475, -1.439918041229248, 0.4480513632297516, 2.6690709590911865, 0.096580371260643, -0.297429621219635, 0.9105005264282227, -0.567939043045044, -0.29367443919181824, -0.16454249620437622, -0.6668858528137207, 0.032323677092790604, 0.34105581045150757, -1.2944258451461792, 0.5401921272277832, -1.06290864944458, 2.1059064865112305, -1.5671122074127197, 1.4185330867767334, -0.3945377469062805, 0.9072819352149963, -0.722842276096344, 0.8367048501968384, 1.337127685546875, -0.44681745767593384, -1.612214207649231, 0.6040608882904053, -1.2490825653076172, -0.7876575589179993, -0.2121482640504837, -0.5671723484992981, -2.0427069664001465, -1.1949254274368286], [-0.3762473165988922, 0.6825595498085022, -0.5830124616622925, -0.35800793766975403, 1.373177409172058, 2.0645575523376465, 0.1241174042224884, -1.51414155960083, 0.7799797654151917, 0.20221038162708282, 0.28890570998191833, 0.9462727904319763, -0.3431486189365387, 1.9332830905914307, -0.887484610080719, -1.3593448400497437, 0.18342037498950958, -1.2976844310760498, -0.7137370109558105, -1.8494380712509155, 1.0222854614257812, 0.541334331035614, -0.06575388461351395, -0.6097418665885925, -1.7367546558380127, -1.6907256841659546, 0.18245017528533936, 1.3308271169662476, 0.9589008092880249, -2.204303026199341, -1.338801622390747, 2.693423271179199, 0.11277167499065399, 0.37979260087013245, 0.19657766819000244, 1.4990935325622559, -0.2697252631187439, 0.08150962740182877, 0.5411503911018372, 0.4130100905895233, 1.0835236310958862, 0.41610321402549744, 0.5815452933311462, -0.8859228491783142, -0.3579062223434448, -0.33139413595199585, 0.5347055196762085, 0.5416439175605774, -0.21518003940582275, -0.4902524948120117, -1.9550466537475586, -1.5978047847747803, -0.37826722860336304, 0.7560204863548279, -0.043458256870508194, -1.248677134513855, -0.8793207406997681, -0.9931828379631042, -0.09312334656715393, -0.12308783829212189, 1.0093894004821777, 0.714966356754303, -0.027925843372941017, 0.5145460367202759], [-0.9707809686660767, -1.8767032623291016, 0.5439742803573608, 1.0729738473892212, 0.46991878747940063, -0.7104008197784424, -0.9074998497962952, 1.3971903324127197, 0.072604238986969, 1.1108858585357666, -1.6821361780166626, -0.30587252974510193, -0.6297770738601685, -0.2810169756412506, 0.6753760576248169, 2.6192092895507812, -1.0975695848464966, 0.07982317358255386, 0.6585550308227539, -0.1676025390625, -0.04629325866699219, -1.3832629919052124, -0.3686824142932892, 2.296797275543213, -1.0171327590942383, -0.3821408450603485, 0.47450509667396545, 0.22485287487506866, 0.8190909624099731, 0.2798413634300232, 0.31364092230796814, -0.6384261846542358, -1.2475388050079346, -0.3701971173286438, 1.5963605642318726, -0.07565710693597794, -0.7539263367652893, 0.5068064332008362, -1.2161051034927368, -0.5496158599853516, 2.3245725631713867, -0.04285052791237831, -0.3961479663848877, 0.3762160539627075, 2.171459674835205, 0.7189722061157227, 0.0243830606341362, 0.030016610398888588, 0.9496725797653198, 1.1325126886367798, -0.9076924920082092, -0.6095688343048096, 0.37998250126838684, -2.0871922969818115, -1.0827244520187378, 1.5316766500473022, 0.6911093592643738, 0.7827688455581665, 0.903714656829834, 2.856726884841919, 1.2018271684646606, -0.19980697333812714, 1.345000147819519, 2.154304027557373], [-0.3100582957267761, 2.1820530891418457, 1.6969656944274902, -0.32622823119163513, -0.5211365222930908, -0.45471635460853577, -0.38690900802612305, -0.28738078474998474, 0.4074217677116394, -0.8143348097801208, 0.5344476699829102, -0.31770461797714233, -1.1726477146148682, 0.7610005140304565, 1.0934373140335083, 1.327623724937439, -0.4556773006916046, 0.777649998664856, 0.08283627033233643, 0.35274946689605713, 1.446123719215393, -0.6349261999130249, 0.27635112404823303, -0.0005663168849423528, 1.6259005069732666, 0.8372743129730225, 0.23827490210533142, -0.22989648580551147, 0.892676591873169, -0.12576431035995483, 1.1018431186676025, -0.7316418886184692, -0.23425690829753876, -0.35547778010368347, -0.2884165942668915, 0.04900788888335228, 0.9964386820793152, 1.7195196151733398, -0.31402039527893066, -0.19115208089351654, -0.16936732828617096, -0.38241100311279297, 1.368917465209961, 0.40911492705345154, -1.5247869491577148, -0.2908742427825928, -1.5233017206192017, 1.5859750509262085, -0.4725320637226105, -1.3136235475540161, -0.3690778911113739, -0.3488515615463257, 0.4496142864227295, -0.757932186126709, -1.417147159576416, 1.8527809381484985, -0.518815279006958, -0.5243289470672607, -0.9550809860229492, 0.4287051856517792, 0.9078624844551086, 0.810934841632843, -0.22838395833969116, 1.371127724647522], [-1.9931628704071045, -0.42447197437286377, -1.3148044347763062, 0.38206008076667786, -0.23446528613567352, 0.47691309452056885, -0.47948458790779114, -2.2180283069610596, 0.5233218669891357, -1.6203820705413818, 1.3982553482055664, 0.2974618971347809, -1.3396337032318115, 1.4614484310150146, 0.5407962799072266, -0.8516799211502075, 1.151060938835144, -1.5950666666030884, -2.258255958557129, -2.401606559753418, 0.4241667091846466, 0.3105875253677368, -0.8209551572799683, 0.5387073755264282, -0.7779198288917542, -0.9261022806167603, 0.3896915912628174, -0.272786021232605, -0.336306631565094, 2.0584189891815186, 1.3112943172454834, 0.37954428791999817, 1.533966064453125, 1.9384549856185913, -0.10440357029438019, 1.033371925354004, -0.14096325635910034, -0.06540564447641373, 0.15520499646663666, -0.2835034132003784, -1.3143253326416016, -0.14007435739040375, -0.3488989472389221, -1.6183574199676514, 0.28733137249946594, -0.9358722567558289, -0.05324525013566017, -0.7203090786933899, -0.9026764035224915, -0.7626274824142456, 0.72042316198349, 0.8793607354164124, 1.0649340152740479, 0.3815540373325348, -0.5525866746902466, -1.190327763557434, 0.574639081954956, 0.8976516127586365, -0.003002740675583482, 0.02846621535718441, -0.9061148166656494, -1.0728882551193237, 1.3332620859146118, 0.006962287705391645], [0.2776263356208801, 1.7348389625549316, -1.131062626838684, -0.000609026406891644, 1.3740577697753906, 0.8640313148498535, 0.5891847610473633, 0.2562401592731476, 1.0747109651565552, 0.8642715811729431, -0.44105470180511475, 1.0689644813537598, 0.3411531448364258, -0.6051064133644104, -0.08816162496805191, -0.4836407005786896, -0.707653820514679, 0.3681226968765259, -1.5257060527801514, 1.4974831342697144, 0.2185737043619156, -0.08458676934242249, 1.553844928741455, -0.5837078094482422, -0.5889756083488464, 1.5253504514694214, -0.48992615938186646, 0.6134951710700989, 0.6064773797988892, 0.8554896116256714, 1.601040244102478, 0.11471036076545715, -0.3478204011917114, -1.497157096862793, -0.09618059545755386, -0.7213459014892578, 0.7767981886863708, 0.6413211822509766, -0.058619484305381775, 1.1086533069610596, 2.0570731163024902, -0.5320966839790344, 0.5523231625556946, 0.025527048856019974, 0.22717221081256866, -0.6603750586509705, -1.2114417552947998, 1.3742432594299316, -0.10130299627780914, -1.5898816585540771, -0.40377306938171387, -1.8069379329681396, -0.827273428440094, -0.08996180444955826, 1.4016706943511963, -0.6466784477233887, 0.4530487358570099, -1.7617138624191284, 0.5920215845108032, -0.4986250400543213, 0.2859112322330475, -0.7415037155151367, -0.38718196749687195, 1.6890842914581299], [0.5824159979820251, 0.710729718208313, -0.7447122931480408, -1.2602055072784424, 0.7129333019256592, -1.2894572019577026, -0.1816389560699463, 1.11735200881958, 0.555294930934906, 0.51450514793396, 0.25848329067230225, -1.4825667142868042, 0.6072076559066772, 1.374225378036499, -0.6888113617897034, -0.6304967999458313, 1.0746943950653076, -0.472372442483902, 0.5253634452819824, 0.8095459938049316, -0.14241309463977814, -0.1749679446220398, -1.1715632677078247, -0.6142212152481079, -0.26118773221969604, -1.3139442205429077, 0.017215048894286156, 1.691500186920166, 1.332950234413147, 0.8020298480987549, -0.6756475567817688, -1.3928892612457275, 0.6482056975364685, 1.2855138778686523, 1.642564296722412, -0.7220407128334045, 0.8943542838096619, 1.646557092666626, 0.2800758481025696, -0.29300835728645325, -0.49355822801589966, -0.18235038220882416, 1.242664098739624, -1.4126393795013428, 0.6649361848831177, -0.4612148106098175, 2.3597922325134277, 1.3305381536483765, 0.6714395880699158, 1.5814878940582275, 0.24507533013820648, 0.08265554904937744, -0.09358233213424683, 0.8372592329978943, 0.3272024989128113, -0.780409574508667, 0.8361621499061584, 0.04104520380496979, -1.0485010147094727, 0.5145613551139832, -1.1317741870880127, -0.24209390580654144, -0.7219883799552917, -0.2093549519777298], [1.489014983177185, 0.74554044008255, 0.950192928314209, -0.4545770585536957, -1.8291375637054443, 0.5533158183097839, -0.8070078492164612, -0.606623649597168, 0.01934988424181938, 0.21581318974494934, 0.8795570135116577, -1.2718701362609863, -0.9415736198425293, -0.4608526825904846, 0.5105717182159424, -0.3669159412384033, 0.6847625374794006, 0.5374073386192322, -0.6242361664772034, 0.3752520680427551, -0.759393572807312, -0.44859933853149414, 0.7670719027519226, 0.09928078204393387, -1.2677782773971558, -0.0800202488899231, -0.5313652753829956, -0.530924379825592, -0.1913457065820694, 1.3458718061447144, 0.1388276219367981, 0.6912184357643127, -0.3431740701198578, -0.3141738772392273, -1.3777194023132324, -0.25055399537086487, -0.8420364856719971, -0.07230817526578903, 0.6170048713684082, 0.14634642004966736, -1.7769577503204346, -0.08821158111095428, -1.4453909397125244, 1.8396515846252441, -1.766999363899231, 1.2702257633209229, -0.5997061133384705, -0.08511455357074738, -0.281360924243927, 1.572014570236206, -0.7540624141693115, 0.7903040647506714, 2.207731246948242, -1.1397169828414917, 0.0985323116183281, -0.12958136200904846, -1.0904985666275024, -0.9914771914482117, -1.6408039331436157, 0.12232884019613266, 0.11788483709096909, 0.3314450979232788, 1.2696616649627686, -1.038697600364685], [-1.3167877197265625, 0.7682983875274658, 0.6952904462814331, 1.2201744318008423, 0.8077843189239502, -0.9252941608428955, 0.8536869883537292, -0.2932637631893158, 0.6803067922592163, 0.9451421499252319, -0.42371484637260437, -0.503476083278656, -0.32540544867515564, -0.08205831050872803, -0.7931509613990784, -1.3853349685668945, -0.4991455674171448, -0.02504359371960163, -0.26649782061576843, 0.031036391854286194, 2.275146722793579, -2.4210102558135986, 0.3889789879322052, 2.2455689907073975, 0.7582986950874329, 1.4587905406951904, -0.5266454815864563, 1.524483323097229, -1.1676661968231201, 0.5392940640449524, 0.5005495548248291, -0.41297823190689087, -1.854468822479248, -0.603823721408844, -0.17556717991828918, -1.0149465799331665, -1.339005470275879, -1.257607102394104, 1.2868355512619019, -0.949573278427124, 0.6388511061668396, 1.3133848905563354, 1.2048958539962769, 0.5784009099006653, 0.08268024027347565, 0.8729240298271179, -0.6699942946434021, 0.4634610116481781, 0.8720115423202515, -0.004360598046332598, -0.16056913137435913, 0.8158092498779297, -0.14759919047355652, -0.15394672751426697, -0.23927681148052216, -1.7804821729660034, 0.44793757796287537, 1.5366053581237793, 0.1878758668899536, -0.6907833218574524, -1.0296727418899536, 1.3079336881637573, -1.1645236015319824, 0.7528743147850037], [0.0599236935377121, 1.2325336933135986, 1.2379975318908691, 0.5423815846443176, -0.43981072306632996, -0.2965724766254425, 0.718681812286377, -0.09856307506561279, -0.524254560470581, 0.17019745707511902, -1.6476503610610962, 2.083998918533325, 0.9713178277015686, -1.6763505935668945, 0.7610358595848083, 0.2758413553237915, -1.1807204484939575, 0.6466273069381714, -0.21198929846286774, 0.3824675977230072, 1.0778149366378784, -0.5353207588195801, -1.1420758962631226, 1.712540864944458, -0.22803251445293427, -1.5146403312683105, 1.0183862447738647, 0.5467472672462463, 0.7025301456451416, 0.24119813740253448, -0.17777256667613983, 0.49838048219680786, 1.0760048627853394, -1.0490440130233765, -0.1618742197751999, 0.5542043447494507, 0.11294522136449814, 0.3073529005050659, 0.07589114457368851, 0.41888242959976196, -1.471428394317627, 1.9011567831039429, -0.020932527258992195, -0.7772420644760132, -0.6492389440536499, 0.10046708583831787, -2.548133134841919, 0.900135338306427, 1.6853384971618652, 1.3001904487609863, 1.369575023651123, 1.1546310186386108, 0.17306621372699738, -0.051730651408433914, 0.05832973122596741, -0.37817928194999695, -0.836637556552887, 0.9376344084739685, -1.4038701057434082, 0.6575560569763184, -0.04566042125225067, 0.37438488006591797, -1.7454898357391357, -2.0714311599731445], [-0.005191534757614136, 0.16023437678813934, 0.16261012852191925, 1.548972487449646, -0.7242040634155273, -1.0328030586242676, -0.25518810749053955, -0.0634230524301529, -0.9355069398880005, 1.0426907539367676, -1.1111741065979004, -0.03031100705265999, 0.1263400912284851, 0.6006981134414673, -0.9531271457672119, -0.1616687774658203, -0.13165362179279327, -0.7802435755729675, 2.586932420730591, 0.01925111562013626, -0.4293302297592163, -1.311971664428711, 1.8629411458969116, 0.5513858199119568, -0.30721768736839294, -1.0927717685699463, -1.2570340633392334, 0.3161742687225342, 1.720959186553955, -0.08749242126941681, 0.23045656085014343, -0.12617285549640656, -0.15778197348117828, -0.9353390336036682, -0.4208052456378937, 0.033745624125003815, 2.491258382797241, 0.895114004611969, 0.7015278935432434, -1.361014723777771, 0.2319260835647583, -1.0501185655593872, 0.1640348583459854, 1.1528455018997192, -0.5569077730178833, 0.6004670858383179, 0.24902163445949554, -1.156157374382019, -1.0338590145111084, 1.401983618736267, -0.7835802435874939, 1.158589482307434, 1.161505937576294, -0.0311119444668293, 0.8650035858154297, 0.07529115676879883, 2.3966329097747803, -0.25161561369895935, 1.2215358018875122, -0.5629268288612366, 0.0022343790624290705, 0.6676074862480164, 1.5940277576446533, 0.09327035397291183], [-0.36764851212501526, -0.748420774936676, 0.6139315366744995, 0.9550343155860901, -0.6771532297134399, 0.5669738054275513, -0.116892509162426, 0.9047802686691284, 0.03740434721112251, 0.3503718078136444, 1.7576501369476318, 0.15605974197387695, -0.40099573135375977, -0.4302597939968109, -0.5958272218704224, 0.8777638673782349, -0.5672856569290161, -1.6492565870285034, -1.2458142042160034, 0.7861859798431396, -1.1640363931655884, 0.2724263370037079, -0.8224100470542908, 1.1639437675476074, 0.006425114348530769, 0.047527920454740524, -0.9500556588172913, 0.840846836566925, 0.6058167815208435, -1.4686534404754639, 0.08957380801439285, -1.4680325984954834, -1.725267767906189, -0.016952741891145706, 0.3510870635509491, 0.1034184992313385, 0.34883254766464233, 0.15829558670520782, 1.3180723190307617, 0.30902621150016785, 1.7018194198608398, -0.9527862668037415, -0.5376760959625244, -2.403168201446533, 0.29274290800094604, 0.5899218916893005, 0.876749575138092, 0.921652615070343, -0.3961520195007324, -0.056367479264736176, -0.7451279759407043, 0.39743849635124207, 0.16258521378040314, 1.4688489437103271, 0.2645314931869507, 0.09578490257263184, -0.8455745577812195, 0.10237252712249756, 0.6528241038322449, -0.21855676174163818, 0.2522212862968445, 1.287927508354187, 0.8902367949485779, -0.3221186697483063], [0.21564525365829468, -0.772733747959137, -0.583386242389679, 0.8107603192329407, -0.8922905921936035, -0.7993948459625244, -1.2435040473937988, -0.547200083732605, 0.5151256322860718, 1.0599305629730225, 1.530756950378418, -0.13397961854934692, 0.1570931375026703, 1.1341338157653809, 0.14225097000598907, -0.8914200067520142, -0.397735059261322, 1.5480523109436035, -0.3363432288169861, 0.37066271901130676, 0.09692662209272385, 1.4043078422546387, 1.7366324663162231, 1.4452147483825684, -0.1506013125181198, -0.06489737331867218, -0.5023596882820129, -1.1201269626617432, -0.11167916655540466, -0.24968673288822174, 0.7254006862640381, -0.6205217242240906, -1.4949009418487549, 0.6223455667495728, -2.2795681953430176, 0.01567997969686985, -0.3759079873561859, 0.08071798831224442, 0.44690242409706116, -1.5018737316131592, -0.08960847556591034, 0.02111082151532173, 0.018537962809205055, 1.9069559574127197, -0.3586181402206421, -1.6748067140579224, 0.01018809899687767, 0.2064782977104187, -0.8497147560119629, -0.34795039892196655, -0.747107207775116, -0.951323390007019, -0.26340824365615845, 0.8686738014221191, 1.0409637689590454, 1.0140217542648315, -0.6167269945144653, -0.6055619716644287, 0.8387276530265808, 0.4338054656982422, -1.1852320432662964, -0.6809225082397461, -1.1764791011810303, 2.857769727706909], [-0.10933289676904678, 0.8021326065063477, 1.099481463432312, 1.2394713163375854, -0.29167240858078003, 1.860979676246643, 0.9423726201057434, -0.5199859142303467, -1.4473093748092651, -0.6394374370574951, 0.898448646068573, 1.2382068634033203, 0.6026447415351868, 1.2188963890075684, 0.7198696136474609, 0.42836877703666687, 1.3319206237792969, 0.0007011808338575065, 1.5567142963409424, 1.8064666986465454, 0.7132800221443176, 3.068225383758545, 3.4103384017944336, 0.20431041717529297, -0.9645917415618896, -0.6939183473587036, -0.6189809441566467, -1.8678709268569946, 1.4841339588165283, 0.47728097438812256, -0.10149434953927994, 0.8753455877304077, -1.9929509162902832, -0.7132324576377869, 2.0252723693847656, 0.7752923369407654, 0.9428369998931885, 0.5404330492019653, -0.5684064626693726, -1.6552091836929321, 0.6392802000045776, -1.6832044124603271, 0.20715735852718353, 0.8813648819923401, -1.0435258150100708, -1.534624695777893, 0.1983872950077057, -0.10081358253955841, 0.17717094719409943, -0.13637247681617737, -0.5044585466384888, 0.16424497961997986, 0.1943991631269455, 2.8281829357147217, 1.5927799940109253, 0.30096253752708435, -1.0002622604370117, -0.1192830353975296, -0.08241938054561615, 1.020719051361084, -0.8065415620803833, 0.2972501218318939, -3.671078681945801, -0.7440915107727051], [0.44165411591529846, -0.08603612333536148, 1.5026260614395142, -0.8077366352081299, 0.5951808094978333, 0.8764154314994812, 2.3092195987701416, 1.2766327857971191, 0.600548267364502, -0.22065448760986328, -0.6559363007545471, 0.6006434559822083, 0.8177468776702881, -0.26068130135536194, 0.4744066596031189, -0.05637230724096298, -0.9109246134757996, 0.22965437173843384, -0.035412028431892395, -0.9599084258079529, -0.3044527471065521, 3.3990631103515625, 0.676665723323822, 1.3478378057479858, -0.002057594945654273, 1.6049394607543945, 0.5280224680900574, 1.4356591701507568, -0.050477225333452225, -1.089341402053833, -0.3393048048019409, 0.4397708475589752, -0.28928858041763306, -0.3244624733924866, -0.5297653079032898, 0.5668423175811768, 0.6723832488059998, -0.14331291615962982, 2.255523920059204, 0.43566080927848816, -0.6400941014289856, 1.9673081636428833, 0.3449975848197937, -1.5712170600891113, 0.07718911021947861, -0.14251232147216797, 1.4992003440856934, 0.12031745165586472, -0.32206016778945923, -0.8924258351325989, -0.7918815016746521, 0.22429338097572327, -0.039778199046850204, -0.693387508392334, 1.5465059280395508, 2.1197948455810547, 0.6496648192405701, 0.559490978717804, 0.11151564121246338, 0.39527955651283264, 1.1049798727035522, 0.9805720448493958, -1.0525693893432617, -0.056755922734737396], [-0.9895891547203064, -1.955126166343689, -0.9196825623512268, 1.6518192291259766, -0.6347599029541016, 0.15536929666996002, 0.5829049944877625, 2.4231905937194824, 0.4252732992172241, 1.2587895393371582, 0.6372986435890198, 0.038775864988565445, 0.6060433387756348, 0.4556107819080353, 1.7470682859420776, -2.1080198287963867, 2.134563684463501, -0.5075750946998596, 0.1548669934272766, -0.08035916090011597, 0.12285152077674866, 1.2275112867355347, -0.5482448935508728, -0.2784309983253479, -1.201629400253296, -1.3527061939239502, 0.6770623326301575, -0.8271100521087646, -1.4608569145202637, -0.2839200794696808, 0.2809714078903198, -0.31748607754707336, -0.22195710241794586, 1.0125991106033325, 1.2375487089157104, -0.039186812937259674, 1.1513514518737793, -0.14358073472976685, -1.4689290523529053, -0.4005486071109772, 0.14995478093624115, -0.4539366364479065, -2.169935941696167, -1.7142055034637451, 1.0925524234771729, 1.2143019437789917, 1.1949996948242188, 2.3492538928985596, 1.3351221084594727, 0.44145700335502625, -0.14787045121192932, -1.5499401092529297, -0.9584769010543823, 1.309607982635498, -0.9616430401802063, -0.7913634777069092, -0.19468948245048523, 0.07531017065048218, 0.6982004642486572, 2.0630362033843994, -0.21774601936340332, -1.4442814588546753, -0.946774959564209, 0.2489759773015976], [0.21993964910507202, 0.8140836358070374, 0.6146473288536072, 0.7523585557937622, 0.6521628499031067, 0.01851879619061947, -0.43859952688217163, 0.5092995762825012, -0.9336528182029724, 0.4744952321052551, -1.1193090677261353, 1.6702773571014404, 1.3273316621780396, -0.0031497152522206306, 0.9489170908927917, 0.5509794354438782, 0.2870113253593445, 0.17933744192123413, -0.3913692235946655, 0.43130019307136536, -0.9172523617744446, -0.3482387065887451, -0.12986329197883606, -0.059732843190431595, -0.5997418761253357, -0.8496546149253845, -2.209812879562378, 0.4029088318347931, -0.0735432580113411, 0.4899595081806183, 0.8119474649429321, 0.4168742895126343, 0.9233576655387878, 0.5341622829437256, 1.1168432235717773, -0.927262008190155, 1.054292917251587, 2.2994070053100586, -0.18783250451087952, -0.8936008810997009, -0.38309696316719055, 1.8321977853775024, 1.3158810138702393, 1.5928815603256226, 0.14570751786231995, 0.07508251816034317, 0.9802024960517883, -0.7174213528633118, -0.5643685460090637, -0.3889840245246887, -0.0715048536658287, -1.0459290742874146, 1.6927921772003174, 0.16345037519931793, 0.31814447045326233, 1.1861722469329834, -0.14765778183937073, 0.2457224279642105, -0.6669971942901611, -0.2704610228538513, 0.8846216797828674, -1.2439712285995483, 0.15821883082389832, 0.19550269842147827], [-0.5176438093185425, -1.6319280862808228, -0.3835044205188751, -0.2498754858970642, -0.08635125309228897, 0.8336935043334961, 0.07983484119176865, -0.6707327365875244, -1.492943525314331, -1.152432918548584, -0.021864473819732666, -0.8079067468643188, -0.6852995753288269, -0.8191466331481934, -0.1401364952325821, -0.34206342697143555, 1.6589720249176025, -1.4951688051223755, -1.1189366579055786, -1.0344123840332031, 0.0025698035024106503, 1.106347680091858, 1.9446938037872314, 0.8697893023490906, 0.3402077853679657, 0.17663165926933289, -0.13617271184921265, -1.1456509828567505, -1.1665257215499878, 0.17358821630477905, -1.1204311847686768, -0.39849498867988586, -1.1780519485473633, -2.085951805114746, -1.4923352003097534, 0.44689860939979553, 0.6985971331596375, -1.2832459211349487, 0.1410714089870453, 0.192976713180542, -0.8583094477653503, 0.24116010963916779, 1.3055332899093628, -1.0651873350143433, -0.69869065284729, 1.5398635864257812, -0.7492533326148987, -0.3813808858394623, 0.3882087767124176, 0.5324994325637817, -0.3640935719013214, -0.5398935079574585, -0.24001005291938782, 0.46709951758384705, -1.518979787826538, 0.019607074558734894, 1.039428472518921, 0.968109667301178, -0.9125165343284607, 1.0538814067840576, 1.836524248123169, -0.189934641122818, 1.3272508382797241, -1.2922145128250122], [-0.9395565390586853, -1.0993205308914185, -0.6443929076194763, -0.24460606276988983, -0.48417848348617554, 0.6108430027961731, 0.6592439413070679, 0.35059088468551636, -0.6522258520126343, -0.3572206199169159, 0.5190439820289612, 0.10061656683683395, 2.4344472885131836, -0.07114583998918533, -0.6755999326705933, -1.0720387697219849, 0.3176102340221405, 0.5104945302009583, 0.20259180665016174, -0.8557060360908508, -0.9379059672355652, 0.1551809161901474, 2.56748104095459, 0.11436974257230759, 0.6489632725715637, 0.6913948655128479, -0.6041947603225708, 0.06615190207958221, -1.089374303817749, -1.4014774560928345, -0.1565971076488495, 0.24242819845676422, 0.11577979475259781, 0.003856053575873375, -0.6503254175186157, 0.5626681447029114, 1.5621325969696045, -0.593730628490448, -2.4083492755889893, 0.18510334193706512, 0.1178981363773346, 1.2052865028381348, -2.778815984725952, 0.9758533239364624, 2.427661180496216, -1.1149048805236816, -0.04400727525353432, -1.8676131963729858, 1.165650486946106, -0.8244604468345642, 0.5884588360786438, 2.134448289871216, -1.4464783668518066, 0.6305859684944153, 0.4554409682750702, 1.2009434700012207, 0.4312751293182373, -1.8835952281951904, -0.07326582819223404, -0.7574443221092224, 0.3535202741622925, 0.2291470468044281, -2.0777342319488525, -0.4098097085952759], [-2.030625104904175, 0.08896264433860779, 2.1964690685272217, -0.33574149012565613, -0.25180089473724365, 1.0631206035614014, -0.2006644457578659, 0.4459132254123688, -0.1977524608373642, -0.3308999240398407, -0.4181770980358124, 1.062973141670227, 0.6994314789772034, 0.830245316028595, 0.1687026023864746, 0.25281769037246704, -1.2122856378555298, 0.11337098479270935, 1.6548925638198853, -1.0781099796295166, -1.5333223342895508, 0.6513715982437134, 0.06157468631863594, -0.043921101838350296, -1.4706981182098389, -0.8108146786689758, 0.36682748794555664, -1.8259837627410889, 0.8582958579063416, -0.3373611271381378, -1.2668640613555908, -0.02573416940867901, 0.1718769520521164, 1.3170377016067505, 0.004682222381234169, 1.3215124607086182, 0.4102841317653656, 0.7223969101905823, -0.36240124702453613, -1.9250231981277466, 0.0646776482462883, 1.7618526220321655, -0.5890821218490601, 0.16075247526168823, -0.5208746194839478, 0.6825352311134338, 0.6988918781280518, -1.2501959800720215, 0.2798740863800049, -0.022266635671257973, 0.8260267376899719, 1.5886319875717163, 0.8122138977050781, 1.360312819480896, 0.10033509135246277, 0.6975463628768921, -0.979198694229126, -1.1082894802093506, 1.1467167139053345, -0.7877415418624878, -1.1693968772888184, -2.1433684825897217, -0.2301660180091858, 0.28727415204048157], [-0.9662294387817383, -1.2320332527160645, -0.06618846952915192, 0.6359229683876038, 0.11300364136695862, 0.5870014429092407, -0.6436039209365845, -0.7308836579322815, 0.6374114155769348, 2.609980583190918, -0.5310887694358826, -0.48801323771476746, -0.17083758115768433, 0.13836205005645752, 1.800671100616455, -0.4990939795970917, -0.1223372146487236, 0.8566451668739319, -1.0485327243804932, 1.295967936515808, 0.329980731010437, 0.048126645386219025, -0.9461707472801208, 0.25389403104782104, -0.5176682472229004, -0.6635957360267639, -1.1584885120391846, 0.6534727811813354, 0.6070513725280762, 0.0269570741802454, -0.24197402596473694, 0.33616015315055847, -0.009772919118404388, -0.481681764125824, 1.4194637537002563, 0.47089096903800964, 0.5085356831550598, 0.6355651617050171, 0.8926495313644409, 1.2163379192352295, 0.9949597120285034, -2.4326882362365723, -0.5819445848464966, -0.0981191024184227, 0.7364903688430786, -0.990717887878418, 1.4994516372680664, 1.0741075277328491, -1.4557350873947144, 0.7981904149055481, 0.4823653995990753, -1.0816200971603394, 0.5961010456085205, -0.7632451057434082, 1.425091028213501, -2.251554489135742, 0.810502827167511, 0.5067641139030457, -1.607429027557373, 2.453089475631714, -0.39413318037986755, -1.891124963760376, 1.7333862781524658, -0.5122122168540955], [1.1211600303649902, 0.7715761065483093, 1.5580521821975708, -0.8562206625938416, 1.064537525177002, 1.1317030191421509, 0.08620066940784454, 0.7483087182044983, 0.2979383170604706, -1.6921842098236084, 0.13345174491405487, 0.8623356223106384, -0.8672453165054321, -0.28725317120552063, -1.4706686735153198, 1.387616515159607, 0.3518017828464508, -0.4863019287586212, -0.30830198526382446, 0.35658177733421326, 0.2264331877231598, 1.5639021396636963, -0.005224925931543112, -0.26695793867111206, 0.1428958773612976, -0.31418320536613464, 0.6438532471656799, 0.8503235578536987, -2.3998184204101562, 1.4291343688964844, 1.6456060409545898, 0.5512340068817139, -0.07639427483081818, 0.3603661358356476, 0.6183878779411316, -0.6272041201591492, -2.3017659187316895, -0.47524699568748474, 0.743740439414978, -0.6606365442276001, 0.6279721260070801, 0.3309619128704071, -0.15397663414478302, 0.6579146981239319, 0.30718159675598145, 0.5703994631767273, -1.629980206489563, 1.554380178451538, -0.4377148449420929, -2.438187837600708, 0.9679545164108276, -0.5553487539291382, -0.9320126175880432, 0.8350995182991028, -0.9083091020584106, -0.44108453392982483, -2.3542182445526123, 0.26653748750686646, 0.47604668140411377, -1.0966126918792725, 1.0040010213851929, -0.46052077412605286, -1.5925084352493286, -1.2999213933944702], [0.11990047246217728, -0.21475638449192047, -0.7296620011329651, 1.4346734285354614, -0.06474976986646652, 0.41793036460876465, 0.048593632876873016, -1.2506179809570312, 0.3542580306529999, 0.5439929366111755, 0.5870917439460754, 0.9798504710197449, 1.801859736442566, 2.2896475791931152, 1.750706672668457, 0.08163923025131226, -1.5712876319885254, -0.11989686638116837, 2.2041783332824707, -0.30619603395462036, -0.5195703506469727, -0.9455865621566772, 1.234918236732483, -0.423721581697464, 0.42887669801712036, 1.6112695932388306, 0.7232844233512878, 0.6673322916030884, -0.12282062321901321, 1.1837698221206665, -0.5319857597351074, -0.9110257625579834, -0.08096788823604584, 1.0680557489395142, 1.1225855350494385, 0.011423557065427303, 0.3157607614994049, 0.28013983368873596, -0.06920819729566574, 0.14228560030460358, -1.7883647680282593, 2.05725359916687, -0.6936338543891907, -0.9813706278800964, -0.6411904692649841, 0.7180748581886292, -0.2054794579744339, -0.2586645781993866, 0.8080642223358154, 0.2919621467590332, -0.2431231588125229, 0.3911534249782562, 0.5982146859169006, 0.040639668703079224, 0.13253240287303925, -0.5976299047470093, -0.4060920774936676, -0.28693532943725586, -1.2095228433609009, 1.3843543529510498, 1.091314673423767, 0.2836377024650574, -0.5503378510475159, 0.5635160207748413], [-1.1806819438934326, -0.20158670842647552, 0.30459219217300415, -0.06657421588897705, -0.1988285928964615, 1.6231414079666138, 0.3981320858001709, 0.5561200380325317, -0.5058081150054932, 0.5531999468803406, 0.5508334636688232, -1.578033447265625, 0.2831990122795105, 0.4388009309768677, 0.2819654047489166, 1.3052639961242676, -1.1256461143493652, -0.9469921588897705, 0.4197326600551605, 0.974764883518219, -0.294685959815979, 1.4290740489959717, 0.5725998282432556, 0.7370831370353699, 0.5665773749351501, -0.8517390489578247, -0.8745825886726379, 0.7637864947319031, 0.3412001430988312, 0.5557474493980408, 2.0271315574645996, 0.4779796004295349, -0.1049167737364769, 1.1126861572265625, 0.21247808635234833, 0.6128897666931152, -1.0076830387115479, 0.9391922950744629, 0.44638893008232117, 0.024744447320699692, -1.2683407068252563, 0.43285828828811646, 1.9830278158187866, 0.8991475701332092, -0.5171324014663696, 1.1404340267181396, -0.38963082432746887, -0.610694169998169, -1.1089673042297363, -0.04265621677041054, -0.12757597863674164, 0.5690173506736755, -0.7514771819114685, -1.2262934446334839, -1.2496196031570435, -0.2144271284341812, -0.3318999409675598, -0.9411572813987732, 1.2149109840393066, 0.8522229790687561, 0.1794712096452713, 0.33223509788513184, -1.0415973663330078, -0.07336635142564774], [1.489754557609558, 0.5324379801750183, -0.6711021065711975, 0.6505088210105896, 0.8441556096076965, 1.1482678651809692, -1.4306658506393433, -0.40481436252593994, -1.431078553199768, 0.9920727610588074, 1.5974200963974, 1.029371976852417, 0.2990175783634186, 0.2183513045310974, 0.23607614636421204, 0.2825509309768677, -0.10476377606391907, 1.1439098119735718, 1.6666085720062256, 0.6786251068115234, -0.45419013500213623, -0.20966938138008118, -2.6141164302825928, 1.081000804901123, 0.8812465071678162, -0.886530876159668, -0.6866079568862915, -0.23992541432380676, 0.4284156262874603, 0.3636276423931122, 1.2351466417312622, 1.5373185873031616, -1.521210789680481, 0.916347324848175, 0.3671484887599945, 1.357987403869629, -1.2596123218536377, -1.3074158430099487, 0.047336023300886154, 0.724637508392334, -0.34540754556655884, 0.11550581455230713, -2.337360382080078, 0.40866678953170776, 1.8613290786743164, 0.3674350082874298, -1.2275232076644897, 1.0269453525543213, -0.5835563540458679, -1.1430230140686035, 1.7775368690490723, -0.43456873297691345, 1.4594266414642334, 0.5890777111053467, 0.8305111527442932, -0.3376859128475189, 2.335620641708374, 1.8670802116394043, -0.9701024889945984, -0.8278540968894958, -0.25383326411247253, -0.3398475646972656, -1.4181396961212158, -0.0597374252974987], [-0.33396461606025696, 1.221576452255249, -0.552045464515686, 0.8434811234474182, -0.2505086660385132, 0.3749564588069916, 1.3854018449783325, -0.8318127393722534, -2.200460433959961, -0.9457483291625977, 0.5517138838768005, 2.522817611694336, 1.9836366176605225, -0.2956089377403259, -0.6364396214485168, 0.7377808094024658, 0.4372723698616028, -0.6696076989173889, 0.0645340159535408, -0.5477735996246338, 0.16054002940654755, 2.515287399291992, -1.514564037322998, -0.30517297983169556, -1.4555331468582153, -0.17315495014190674, 0.09585005789995193, 0.06393732130527496, -0.8893592953681946, 0.05806611478328705, 1.495866060256958, -0.3456561267375946, 1.8800888061523438, -0.6535171866416931, -0.9173299670219421, 0.1364736557006836, -1.712135910987854, -0.07736755162477493, -0.5444450378417969, -0.8259844779968262, -0.20744560658931732, -0.67751544713974, -1.8673648834228516, -0.3290294408798218, -0.7677758932113647, 0.851752519607544, 0.5991247296333313, -0.5316915512084961, 0.36771300435066223, 0.072136290371418, 0.5760484933853149, 2.248112201690674, -1.9317879676818848, -0.42775094509124756, -0.702716052532196, -0.2809901237487793, 0.4997606873512268, 2.1113944053649902, 0.43917012214660645, -1.7661298513412476, -1.1089059114456177, 2.171877145767212, -1.159440040588379, -0.6482251882553101], [-0.9692772626876831, 0.24736391007900238, 0.7098855376243591, -0.8246060013771057, 1.4502103328704834, -0.6677802205085754, -0.18113921582698822, 1.1743468046188354, 2.1368496417999268, 0.32140836119651794, -0.4123077392578125, 0.22917284071445465, 1.654420018196106, 0.784193754196167, 1.8577336072921753, -0.8560832142829895, -0.8503546714782715, -0.621876060962677, 0.9743163585662842, 0.5745905637741089, -0.9839150905609131, -0.817683756351471, -1.4604324102401733, 0.2226802408695221, 0.6928484439849854, 1.0554072856903076, 0.48797959089279175, 0.822548508644104, -0.632725179195404, -0.6638813018798828, -0.5450995564460754, 0.1335752308368683, 0.1599847674369812, 0.23656417429447174, 0.7283815145492554, -0.4551052451133728, 0.8816052675247192, 1.9562894105911255, -0.9104055166244507, 0.08670467138290405, -0.19469887018203735, -0.03499171882867813, -1.1749303340911865, 0.1609252244234085, -1.241245150566101, 1.6502703428268433, 0.25969552993774414, -0.14897507429122925, -0.27790161967277527, -1.564200758934021, -0.32764682173728943, 0.03768842667341232, 0.04112919792532921, 0.4518638551235199, 0.41747722029685974, 2.82672381401062, 0.7279794812202454, 0.11842771619558334, -1.4339187145233154, 0.8630379438400269, 1.0609631538391113, -0.5050487518310547, -0.22610194981098175, -0.1512550264596939], [-0.2252790927886963, 0.19627262651920319, -1.0818111896514893, -0.44263675808906555, -0.33468541502952576, -0.09067032486200333, 0.4183383285999298, -0.5678296685218811, -0.523655891418457, 0.81234210729599, -0.0474541075527668, -0.8032233715057373, -1.4627408981323242, 0.720635175704956, 1.3694077730178833, -1.3161325454711914, -0.22636742889881134, -1.2843672037124634, -1.6976767778396606, 1.371094822883606, 1.7646315097808838, 0.6276115775108337, -0.9020537734031677, -1.4152816534042358, -1.3187590837478638, 1.3215863704681396, 1.4110000133514404, -0.5127860903739929, -0.6278588175773621, 1.4311680793762207, 0.05955113098025322, 0.0992484986782074, -0.6926605105400085, 1.9120450019836426, -0.3759075403213501, -0.7790912389755249, -0.2659434974193573, 0.09381695836782455, 0.44061392545700073, 0.17282931506633759, -0.2702096104621887, -0.9817005395889282, 0.07275810092687607, 1.6162759065628052, 2.0108375549316406, 1.9004021883010864, -0.4709676504135132, -1.8215464353561401, -0.6024182438850403, 0.7339454293251038, -2.431187391281128, 0.6553423404693604, -0.6923304796218872, 0.5204047560691833, -1.3310285806655884, -0.44917014241218567, -0.42457929253578186, 1.7640388011932373, 0.2429320067167282, -0.7381249666213989, -0.2630544602870941, 0.21173328161239624, 1.5918711423873901, -0.477378249168396], [-0.367279052734375, 1.0269466638565063, -0.04537658393383026, -1.375447154045105, 1.859501600265503, 1.459295630455017, 0.7611691355705261, -0.6901114583015442, -3.131180763244629, -0.4183073341846466, -0.7097482681274414, -1.9260811805725098, -1.247175931930542, 0.39818090200424194, 0.7458785176277161, 1.8492648601531982, 1.16456139087677, -0.19802170991897583, 1.0156060457229614, -1.3438209295272827, -0.8744326829910278, -1.3408602476119995, 0.9008194208145142, -0.591542661190033, -0.23936821520328522, 0.5137943029403687, -0.6887799501419067, 0.4510234296321869, -1.5227197408676147, -0.5909823775291443, 1.0272319316864014, -0.5653009414672852, 1.7795214653015137, 0.7571547627449036, -0.6395025849342346, -0.3037194013595581, 0.8256992697715759, -0.4395882189273834, -0.34288614988327026, -3.020322561264038, -0.14952318370342255, -1.7178901433944702, -0.38483142852783203, -0.6109727025032043, -0.6472542881965637, 0.29043424129486084, -1.112109661102295, -0.5188649892807007, 1.5475382804870605, -0.4046035706996918, 0.7871285676956177, 0.5600888133049011, 0.19205699861049652, 0.08982338011264801, -3.8576860427856445, 0.031917691230773926, 0.2810807228088379, -0.5775269865989685, -0.21195679903030396, -0.47190967202186584, -1.5514169931411743, 1.0846842527389526, -1.0477145910263062, -0.7875425219535828], [-0.9711814522743225, 0.335114061832428, -0.7380949854850769, -0.08071453124284744, 1.520907998085022, 0.6892627477645874, 0.4466446340084076, -0.21759630739688873, -0.04337480664253235, -1.2589502334594727, -1.6467766761779785, -0.6938152313232422, -0.21259473264217377, -0.5132790803909302, 0.07865932583808899, -0.5951947569847107, -0.0850822702050209, -0.7973960638046265, -0.2868165671825409, -1.50335693359375, 1.4974740743637085, 0.5295490622520447, -0.7779219746589661, -1.0914809703826904, 0.7600945234298706, 0.23485448956489563, 0.9653111696243286, -1.5913649797439575, 0.053648047149181366, -1.3081151247024536, 0.012283450923860073, 1.1630357503890991, 2.6254706382751465, -0.7525081634521484, 0.385270357131958, 0.08970862627029419, 0.023119062185287476, 1.023604393005371, -0.7521458268165588, 0.7682533860206604, 0.5562170147895813, -0.4533332288265228, -0.7656001448631287, 0.8103084564208984, -0.030845677480101585, 0.5063416361808777, -0.8101028203964233, 0.05937202647328377, 1.1261262893676758, 0.745758593082428, -0.1182502880692482, -0.764744222164154, 1.7082984447479248, 1.2740957736968994, -0.5839910507202148, -0.8790749907493591, -0.8550882935523987, 0.8345792293548584, -0.207235187292099, 0.6170209050178528, 0.0046417806297540665, 2.108675241470337, 0.5480619668960571, 0.6812520027160645], [0.24061302840709686, -2.5290119647979736, -0.3018189072608948, 0.45695778727531433, 0.6668445467948914, -1.083912968635559, 1.068460464477539, -0.4871392548084259, -1.942655324935913, 0.46937257051467896, 0.04105126112699509, 0.9608655571937561, -1.7418677806854248, 2.4361813068389893, -1.7300459146499634, 0.03989697992801666, 1.5418013334274292, 0.6416512131690979, -0.6469107270240784, -1.0481816530227661, 1.1622612476348877, 0.16328975558280945, 2.0144171714782715, -0.9749903082847595, -0.12469944357872009, 2.2665843963623047, 0.4659729599952698, -1.3696304559707642, 0.34757018089294434, 1.3727037906646729, -1.6455377340316772, -0.39089342951774597, 1.9983338117599487, -1.1604715585708618, -1.3946095705032349, 0.6736552119255066, 0.5831884145736694, 0.6020960211753845, 0.9004731178283691, -0.017431115731596947, 0.7205097675323486, -0.16487281024456024, -0.005222050938755274, -0.9811811447143555, 0.8964861631393433, 0.5969240069389343, -0.5190797448158264, 1.034305214881897, 0.27913835644721985, 0.010504920035600662, -0.5037726163864136, 2.3365883827209473, 0.5954858064651489, 0.13849182426929474, 0.6976620554924011, 0.11886896193027496, -2.2205731868743896, 0.5115353465080261, 0.17183835804462433, 1.07429838180542, 0.38689765334129333, 1.3008942604064941, 0.2398102730512619, -0.24767976999282837], [1.641581654548645, -0.19957628846168518, 0.29193031787872314, -0.41163551807403564, -0.017833342775702477, -0.8892642855644226, -0.14929990470409393, -0.7255143523216248, 1.055018424987793, -1.0873310565948486, 1.8806641101837158, 0.9489676356315613, -0.1417185515165329, 1.603361964225769, 0.5088559985160828, 0.25061461329460144, 0.0743282288312912, 0.2752866744995117, 1.7179465293884277, 0.6776494383811951, 0.18368305265903473, -1.1527183055877686, 0.5765977501869202, 1.0109827518463135, -0.17247697710990906, 1.509198784828186, -0.6085300445556641, -1.1989692449569702, 0.19568637013435364, 0.8438500165939331, 0.8497440814971924, -3.103517770767212, 0.11059368401765823, -1.596988558769226, 1.0280991792678833, -1.0687862634658813, 0.8796129822731018, 0.568930447101593, 0.07817735522985458, -0.6371104717254639, 1.2098501920700073, 0.09734471142292023, -0.18049553036689758, -0.7178133130073547, -1.614806890487671, -0.06578461825847626, -0.5121729969978333, -1.553248643875122, -1.712912678718567, -2.016927480697632, -1.1341012716293335, -0.8100433349609375, -0.9137840867042542, 0.28943657875061035, 2.081540822982788, 0.6217209696769714, 0.6030637621879578, 0.9030770063400269, 0.2883528470993042, 0.4919087886810303, -0.8528424501419067, -2.2684593200683594, 0.6177543997764587, -2.342334270477295], [-1.6901087760925293, -0.8805580735206604, -0.4796319603919983, 1.6669152975082397, 0.16718152165412903, -0.05491752550005913, -0.040940966457128525, -1.9678916931152344, -1.5733954906463623, 0.5608323216438293, -0.5936223864555359, -0.9798809289932251, 0.6983811259269714, -0.37609145045280457, 0.06993557512760162, 0.14130407571792603, -0.3485737144947052, -0.41562503576278687, 0.4561528265476227, 1.0578559637069702, -2.3057918548583984, 0.5712496042251587, -0.2252790480852127, 2.4671902656555176, 1.192312479019165, 1.0446902513504028, 0.03401161730289459, -2.078470230102539, 0.05332781374454498, 0.028200263157486916, 0.6694157123565674, -0.43594250082969666, -0.22443093359470367, 2.133976459503174, -1.3734098672866821, 1.8140684366226196, 0.30858123302459717, -0.43268123269081116, 1.3131746053695679, 0.7319998145103455, 1.1708884239196777, 0.8951999545097351, 0.011956915259361267, 0.1113334372639656, 1.2044531106948853, 1.1562628746032715, -0.7093423008918762, -1.7418025732040405, -1.692345380783081, -0.8787694573402405, 0.14655080437660217, -0.6820902228355408, -0.6170711517333984, -0.7593843936920166, -2.292710304260254, -0.8056504130363464, -0.5940111875534058, 0.41854792833328247, 0.4346543252468109, -1.135695219039917, -0.9454401731491089, 0.19157099723815918, -0.2264641672372818, 0.837430477142334], [0.05883419141173363, 0.027192994952201843, -0.2176913470029831, -1.1265820264816284, 0.8515478372573853, -0.8703203201293945, 0.6399151086807251, -0.572492778301239, -2.3040390014648438, -0.26842236518859863, -0.9904963970184326, -1.130147099494934, -0.4786592423915863, -1.3422434329986572, 0.3144630789756775, 1.1101198196411133, 0.2708059549331665, 0.180761456489563, 0.625212550163269, -0.20825780928134918, 0.4721505045890808, 0.25936055183410645, 0.49951618909835815, -1.199803113937378, -1.071103572845459, 1.2172890901565552, -0.508910059928894, 0.025767069309949875, 0.13520991802215576, 0.8470828533172607, -0.6480658650398254, -0.3156057894229889, 0.7380061745643616, 0.8144782185554504, 0.18800735473632812, -0.5788266062736511, -0.6985207200050354, 1.2652910947799683, 0.5393510460853577, 0.835152268409729, 3.220710515975952, 1.2812339067459106, -0.48550015687942505, 0.2185123711824417, 1.3262213468551636, 0.12219101190567017, -0.7352975606918335, 0.15089108049869537, 0.23236340284347534, 0.05679011717438698, -1.4701346158981323, 0.4504186511039734, 0.689055860042572, -0.5598273873329163, -1.158039927482605, -0.9089294672012329, -0.5729995369911194, 0.211748406291008, 1.8737820386886597, -0.6229227781295776, -0.5531882047653198, -1.9181307554244995, -0.28075268864631653, 0.48426222801208496], [0.11419014632701874, -0.9509900212287903, 0.19039373099803925, -0.6662189364433289, -0.7423005104064941, -0.904850423336029, -0.069747693836689, 0.23174212872982025, 0.31424859166145325, 0.23829670250415802, 1.6094242334365845, -1.2700834274291992, 1.320786476135254, -0.9486400485038757, 0.5542856454849243, 0.624634861946106, 1.097015380859375, -0.6178911328315735, 0.8295484185218811, 1.1243177652359009, 0.5763933062553406, -0.24296748638153076, -1.0201691389083862, 0.5556036233901978, 0.3918362855911255, 0.008874140679836273, 0.42216306924819946, -1.3131072521209717, 0.5237424969673157, -0.8995700478553772, 2.070230722427368, -1.1295703649520874, -1.485008716583252, -1.0509098768234253, 0.05524945259094238, 0.465891033411026, 1.2009185552597046, 0.470796674489975, -0.2329501509666443, 1.0400207042694092, -1.4273149967193604, -0.8075730204582214, 1.2215005159378052, 1.4325045347213745, 0.6840128898620605, 0.15230399370193481, 0.49162226915359497, -0.3593463599681854, -0.6936546564102173, 0.04687125235795975, -0.38467052578926086, 0.027869701385498047, -0.3819836378097534, -1.9241211414337158, -0.182509183883667, 0.0351606123149395, -1.0244133472442627, 1.765384554862976, -0.7546483278274536, 0.9962120056152344, 0.6008957624435425, -0.7150041460990906, 0.5813295245170593, 0.10247272998094559], [-0.6203571557998657, -0.3009348213672638, 1.105691909790039, 1.129726767539978, 0.4424729645252228, -0.8414804935455322, 0.03874551132321358, -0.09805306047201157, -0.9031856656074524, -0.9267089366912842, -0.742615818977356, -0.7786840796470642, -0.3865816295146942, 2.7200427055358887, 1.5349340438842773, -1.0823172330856323, 2.5782222747802734, 0.8893948793411255, -0.5681726336479187, -0.18653565645217896, -0.42886343598365784, -0.5261234641075134, 0.8221057057380676, -0.048034340143203735, -1.374757170677185, 0.18851925432682037, 0.6477566957473755, 2.0859124660491943, -0.01438425574451685, 0.7019582390785217, 0.06792858988046646, 0.20637241005897522, -0.43030381202697754, -1.4884437322616577, 0.11465323716402054, 0.1560874879360199, -0.23528121411800385, 1.3080729246139526, 1.545422077178955, -0.4505780041217804, -0.04353662207722664, -1.5118913650512695, -1.0658499002456665, -0.0110059455037117, -0.10818999260663986, 0.5645575523376465, 1.2752338647842407, -0.05553716421127319, -0.099515400826931, 0.22809146344661713, 0.0139106884598732, 0.6056172251701355, -1.6874852180480957, 0.7201536297798157, -0.47337740659713745, 1.0600069761276245, -0.20251527428627014, -0.32458192110061646, -0.9161562323570251, -0.0663728266954422, 1.1073598861694336, -1.5321046113967896, 1.4619821310043335, -0.7386976480484009], [-0.5817633271217346, -1.0593764781951904, -0.87493497133255, -0.9209156632423401, -0.4047550857067108, -0.2329200804233551, -0.5936210751533508, -3.334834337234497, 0.3854435682296753, 0.9473255276679993, -0.43534818291664124, -0.1174330785870552, 2.5223803520202637, 0.25609081983566284, -0.3308820128440857, 0.5876303315162659, 0.057276953011751175, 0.5362042188644409, -0.2580270767211914, 0.4104488790035248, 1.1286879777908325, -1.3237478733062744, 0.6939003467559814, -0.8283604979515076, 1.3195968866348267, 1.183618187904358, -0.11518046259880066, -0.23715080320835114, 1.3981469869613647, 0.702612042427063, 1.4889891147613525, 0.21095941960811615, -0.47115492820739746, -1.4226716756820679, -1.7427946329116821, 0.6652576923370361, 0.7251291275024414, 1.4704396724700928, 0.5818817615509033, 0.5675559639930725, 0.4142858386039734, 1.0015172958374023, 0.21925239264965057, -1.256142258644104, -0.22720789909362793, 0.579775333404541, 0.8062484264373779, -0.05226586014032364, 1.9044926166534424, 0.42147791385650635, 0.20205558836460114, -0.26768723130226135, -0.34449049830436707, -0.145461305975914, -0.8810474276542664, -0.8380252122879028, -0.3310096859931946, -0.6702325940132141, -0.2929803133010864, -0.07744070887565613, -0.6366963982582092, 0.7137086987495422, 0.7200973033905029, 0.21063202619552612], [-0.19770899415016174, -0.5034482479095459, -1.5411791801452637, 1.458527684211731, -0.7398486733436584, 0.21482862532138824, -0.7125023603439331, -0.5405467748641968, -0.4080449044704437, -1.549214243888855, 0.14104339480400085, 1.0881619453430176, 1.4447330236434937, 0.07798872888088226, -0.15585698187351227, 0.806047260761261, 1.3878921270370483, -1.6776469945907593, 0.5797883868217468, 0.701424241065979, -0.7211011052131653, -3.2035536766052246, 1.9823781251907349, -0.5856566429138184, -0.8082476854324341, 0.8564857840538025, -2.1657955646514893, -1.0382026433944702, 0.2881808280944824, 1.0592501163482666, -1.192397952079773, -1.6361809968948364, 0.08885300159454346, -0.411984920501709, -2.450087308883667, 0.45398300886154175, -0.9387314915657043, 0.4579842686653137, 0.8803761005401611, -0.5776896476745605, -0.1261662393808365, -0.9588549137115479, 1.3349106311798096, -0.4660356640815735, -0.11621181666851044, 0.4424530565738678, 1.111235499382019, 0.6493889689445496, 1.8218785524368286, 0.5087251663208008, -0.3950778543949127, -1.0443127155303955, -0.3081314265727997, 0.6159508228302002, -0.1371002197265625, -0.06160246953368187, -0.8784634470939636, -0.7991523742675781, -1.1359659433364868, -0.2838796079158783, -0.5452480912208557, 1.6442748308181763, -1.9156659841537476, -0.1963810920715332], [-0.022752905264496803, -0.12258394807577133, 0.6284568905830383, 0.19631582498550415, -0.37512314319610596, -0.3232361674308777, 0.8039205074310303, -1.6140305995941162, -0.896895706653595, -1.3204628229141235, 1.6036430597305298, -1.6795696020126343, -0.8969311118125916, -0.9368236660957336, -0.42165109515190125, 0.3747164011001587, 0.29432037472724915, 0.3103283643722534, 0.019002661108970642, 0.4201553761959076, -0.843695878982544, -0.31691208481788635, 1.1287921667099, 1.9085960388183594, 0.3943988084793091, -1.9856091737747192, -2.0025463104248047, 0.184609517455101, 0.30048176646232605, 0.3958498239517212, -0.4269959330558777, -0.7482669353485107, 1.893347978591919, -0.29903098940849304, -0.15277576446533203, -0.9339772462844849, -1.7193306684494019, -1.2121964693069458, -0.1852266639471054, 1.1813088655471802, -1.4662370681762695, -1.1729259490966797, -0.0844159722328186, 0.32120364904403687, -0.016402576118707657, 0.3630983531475067, -0.5140687823295593, 1.2514139413833618, 0.3499998152256012, -0.7942836284637451, 2.2070515155792236, -0.9596003890037537, -0.5148974061012268, -0.12800754606723785, 0.6328862309455872, 0.35055336356163025, 1.2893239259719849, 0.5480868816375732, 0.01032663881778717, 0.46900585293769836, -1.6122530698776245, -1.489931583404541, -0.19713187217712402, -0.19577912986278534], [0.4438362717628479, -0.46304750442504883, -0.483990877866745, 0.7301009893417358, 0.3045656085014343, -0.066392682492733, 0.008767679333686829, 0.17552241683006287, 0.7230660915374756, 0.5024712681770325, -1.3455318212509155, 0.4694821834564209, -1.6166651248931885, 0.14373576641082764, 0.299154669046402, -1.053848385810852, 1.6052372455596924, -0.2801576554775238, 0.0445641353726387, -0.7604163289070129, 0.456955224275589, 0.42721647024154663, 0.30686426162719727, 1.67521333694458, 1.2311073541641235, -1.825959324836731, -0.5950235724449158, -0.6893143057823181, -0.25378987193107605, -1.0163650512695312, -0.33972764015197754, 2.299818515777588, -0.41757330298423767, 0.524280309677124, 0.5995684862136841, 0.7021981477737427, 0.6891907453536987, -0.6626216769218445, -0.4498720169067383, -0.6580235362052917, 1.5399487018585205, 0.9134664535522461, -0.7638081312179565, -0.0029104435816407204, 0.25852882862091064, 0.013238336890935898, 0.0060668992809951305, -1.0734957456588745, 0.44953471422195435, -0.5784920454025269, -0.8441372513771057, 0.7637314796447754, 0.11217409372329712, 0.9053244590759277, -1.394482970237732, 0.3692588210105896, 0.6206972002983093, -0.11744456738233566, 0.08550555258989334, -0.8622800707817078, -0.4222005307674408, 0.8743452429771423, 1.511289119720459, 0.4346886873245239], [-1.7914937734603882, -0.5249895453453064, -1.6094188690185547, 0.7789353132247925, 0.5575284957885742, 0.14461858570575714, -0.16547518968582153, -1.2339204549789429, -1.2311573028564453, -0.5274003744125366, -0.4546901285648346, -2.022679328918457, -0.8787721991539001, 0.1383429318666458, -1.8618018627166748, -0.8394224643707275, -1.9523836374282837, -2.141202926635742, 0.6797478795051575, 0.798185408115387, -2.3348588943481445, -0.9318486452102661, 1.7777621746063232, -1.0539228916168213, -0.5852488279342651, -0.6013057827949524, -0.4939865469932556, -1.0501697063446045, 1.6920150518417358, 1.3011531829833984, 0.42384910583496094, -1.5184097290039062, -1.5115861892700195, 0.8413809537887573, -0.8256857395172119, 0.487708181142807, -0.5261497497558594, -1.0121331214904785, -0.5100756287574768, -0.7421213984489441, 1.6281347274780273, -1.717126488685608, -0.4590620994567871, 1.1087288856506348, -0.41753894090652466, -0.3004024922847748, 0.7514603734016418, 0.4798088073730469, -0.007947329431772232, -1.2023792266845703, -0.07221876084804535, -0.4103590250015259, 0.6097990870475769, 1.216538429260254, -0.8296995759010315, 1.9238191843032837, 1.9293876886367798, 0.5934134721755981, -1.148558259010315, 0.07063671201467514, -1.0895746946334839, -1.2920773029327393, 0.425374299287796, 0.4640953838825226], [-1.2258329391479492, 0.2527492046356201, 0.672005832195282, -0.8165509104728699, 0.03844255581498146, -1.124884843826294, -0.44391536712646484, 2.088799238204956, 0.11451022326946259, -0.19626113772392273, 0.26234370470046997, -0.3537489175796509, 0.28941842913627625, -0.08557889610528946, -0.8642295598983765, 0.4863698482513428, -1.0796046257019043, -0.5316996574401855, 0.8096424341201782, 1.0969945192337036, 0.7468740344047546, -0.3556866943836212, -0.41144701838493347, -0.38334211707115173, 1.494755744934082, -1.5120692253112793, 1.8217673301696777, 0.7523412704467773, -1.1173356771469116, -2.7067432403564453, -2.0746641159057617, 0.46320581436157227, 0.4941489100456238, -1.1245437860488892, -0.4532984793186188, -1.0294557809829712, 0.7252838611602783, -0.7932572960853577, -0.01910540834069252, -0.055671848356723785, 0.14524663984775543, -0.160872682929039, 0.3435985743999481, 1.6021302938461304, 0.4518272280693054, -0.30943602323532104, -1.601358413696289, -0.6983569264411926, -0.26051247119903564, 0.48746225237846375, -0.5995385646820068, 0.6528972387313843, -0.0005262568593025208, 1.0922925472259521, 0.48995906114578247, -3.0405433177948, -1.7362737655639648, 1.2278944253921509, 0.6879473924636841, 0.03673558682203293, -0.7033979892730713, -1.8883246183395386, 1.2403771877288818, -0.7689746618270874], [-0.24586926400661469, 0.38188186287879944, -0.6042295098304749, 1.174045205116272, -0.4920441806316376, -0.4702906906604767, 1.9513375759124756, 0.7614578604698181, 0.7906026244163513, 0.3467685580253601, -0.006412971299141645, 1.3298910856246948, 0.5849981904029846, -1.856622338294983, 1.352518081665039, 0.44109439849853516, 0.21623265743255615, 0.8800690770149231, -0.3981701731681824, 1.5186452865600586, 0.05335956811904907, 1.0874814987182617, -0.885701596736908, 0.46012136340141296, -0.49030327796936035, -0.7331271171569824, 0.7499296069145203, -0.2292349934577942, 0.14075860381126404, 0.4597815275192261, 0.1104622408747673, 0.11049776524305344, 0.06237216666340828, 0.0856148824095726, 0.31598135828971863, -0.09283295273780823, -1.1751981973648071, -0.20908723771572113, -0.651276707649231, -0.6483281850814819, 1.1639913320541382, 0.7344597578048706, -0.030193617567420006, -2.364885091781616, -1.7508045434951782, -0.160696879029274, 1.781955599784851, 0.7476072907447815, -0.6308718919754028, 0.11007422208786011, 0.8935330510139465, 1.689607858657837, -1.1042039394378662, 1.9899269342422485, 0.36784037947654724, 0.3018798232078552, 0.5478684902191162, 0.8977257013320923, 0.7938640117645264, 0.6712519526481628, 0.32252758741378784, -0.3866264820098877, 1.874241590499878, -1.7497743368148804], [-0.7490288019180298, -0.39059755206108093, -0.1416458934545517, 0.30759939551353455, -0.9471992254257202, -0.8676565289497375, -1.1331018209457397, 1.1673451662063599, -0.22863249480724335, 1.7552489042282104, 0.35765913128852844, -0.10481792688369751, -1.1580232381820679, 0.41833093762397766, -0.8020393252372742, 2.1262495517730713, -0.5906755328178406, 0.11448097229003906, -1.6706507205963135, -1.0481374263763428, -0.48640453815460205, 0.5427734851837158, 1.0174493789672852, 1.6123106479644775, -0.5430620312690735, -0.6599802374839783, 0.6466727256774902, -1.1854248046875, 0.7064377069473267, -0.9236192107200623, -0.9628567695617676, -0.9080510139465332, 0.41153278946876526, 0.8333173990249634, -1.0345786809921265, 2.1030426025390625, 0.7777711153030396, 1.5200531482696533, 0.24936380982398987, -2.6009674072265625, -0.440856397151947, 1.115976333618164, 0.95228511095047, 1.9592465162277222, -0.18064656853675842, -0.4305868446826935, -0.818510115146637, -0.14646455645561218, 0.48569342494010925, -0.4097777307033539, 0.2877408564090729, 0.7286865711212158, -1.0119811296463013, 0.17227746546268463, 1.4816970825195312, 1.5201427936553955, 0.030666841194033623, 0.5547120571136475, -1.2244433164596558, -0.06220218539237976, -1.231328010559082, 0.8632740378379822, -2.4608922004699707, 0.2588433623313904], [0.07511364668607712, -0.34227225184440613, 0.11879763007164001, 1.3844027519226074, -1.3841502666473389, 2.1029107570648193, -0.18979285657405853, -1.3478566408157349, -0.5455702543258667, -0.19075803458690643, -0.0985214039683342, -0.1689690202474594, -0.03571876883506775, 0.19896453619003296, -1.3039458990097046, 1.1135605573654175, 0.018315453082323074, -0.40297290682792664, 0.453012615442276, 0.8977062702178955, -2.131763219833374, -0.452155202627182, 0.05632967874407768, -0.6352542042732239, 0.5239723920822144, 0.6363576650619507, 0.09264229238033295, -0.3259817659854889, 0.7699428200721741, -1.4153910875320435, -0.08354244381189346, 0.021039385348558426, -0.9473873376846313, -0.5351155400276184, 0.21766826510429382, -1.8569316864013672, 0.7981926202774048, 0.6773824095726013, 0.09323325753211975, 1.6545430421829224, -0.18260350823402405, 0.06351272016763687, -1.20561683177948, 0.36591702699661255, 0.8017486929893494, 0.20564471185207367, 0.09419088065624237, 0.016229210421442986, -2.943176507949829, 2.0825283527374268, -0.8149814009666443, -0.19649814069271088, 0.451088011264801, 0.024528536945581436, -0.4848444163799286, -1.7053507566452026, -0.6050053238868713, 0.5696271657943726, 0.3884616792201996, -0.08459702879190445, -2.1481246948242188, 1.2673392295837402, -0.2042558789253235, 0.2645760178565979], [-0.13025131821632385, 0.6154218316078186, -1.4072010517120361, 0.03626292943954468, 0.9927248954772949, 0.11085274815559387, -1.1063178777694702, 0.4367733597755432, -1.381924033164978, 0.17386025190353394, 0.29040324687957764, 0.672407865524292, -2.1454837322235107, 1.1871585845947266, 0.21600213646888733, -0.14047738909721375, -1.2703592777252197, 0.8296791315078735, -1.9835196733474731, 2.690539598464966, 0.7236834764480591, 0.979173481464386, -0.4125431776046753, -0.3111618757247925, 0.2514546513557434, -0.029687510803341866, 0.7747538089752197, 0.521715521812439, 1.0396287441253662, 1.503090262413025, -1.4537651538848877, -1.1198036670684814, 0.11436769366264343, -1.4957951307296753, 0.5930060148239136, -0.5164698362350464, 0.6720640659332275, -0.8176060914993286, 0.1635696440935135, -0.31976696848869324, 0.9261486530303955, -0.5123379826545715, 0.04429001361131668, -0.7173202037811279, -0.13695386052131653, -0.11000736057758331, -1.700307846069336, -0.3162483274936676, 0.42179179191589355, 1.3600026369094849, 0.059002481400966644, -1.035928726196289, 0.09484384953975677, -0.17435747385025024, -0.5063686966896057, 0.5249042510986328, -1.9411969184875488, -1.4811865091323853, -0.44659072160720825, -0.39237692952156067, 0.22329136729240417, 0.07710891216993332, -0.6383252143859863, 2.118067979812622], [0.9761237502098083, 1.8717138767242432, -0.606698215007782, -0.1980842798948288, -0.05896836891770363, -0.6962889432907104, 2.0075368881225586, -0.25947973132133484, 0.7129302024841309, 0.7633489370346069, -0.10132843255996704, -0.5682870745658875, -0.2931384742259979, -0.5818008184432983, -0.8020113110542297, 1.3011250495910645, -0.22472387552261353, -0.5412582159042358, -0.9637432098388672, 0.41830283403396606, -0.8955237865447998, -1.3749107122421265, -0.7278775572776794, -0.10334338992834091, 1.2691128253936768, -0.03316289559006691, 2.044543743133545, -0.31606316566467285, -0.15110430121421814, 0.04223790764808655, 0.2298925668001175, -0.8387771844863892, 1.476084589958191, -0.8935188055038452, 1.7571790218353271, 0.2912278473377228, -0.858966052532196, -0.4667053520679474, 0.009415173903107643, -0.4721584618091583, 0.4817836880683899, -1.1918917894363403, -0.4909546673297882, 0.6401885747909546, -0.489413857460022, -0.04154808074235916, -0.31855309009552, 1.269134283065796, -1.4270589351654053, -1.947633147239685, 0.10561440140008926, 0.9522995948791504, -1.425829291343689, -2.0878682136535645, 1.291167140007019, 0.6021127104759216, -0.4351694881916046, -0.1535036414861679, -0.9194101691246033, -0.5883599519729614, -1.0721312761306763, -1.6036864519119263, 1.5975650548934937, -0.19606176018714905], [0.5715668797492981, -0.31024327874183655, -1.096191167831421, -1.7216291427612305, -1.7918113470077515, -1.8067799806594849, 0.07455649971961975, 1.6271307468414307, -0.40679219365119934, 1.3568639755249023, 0.5471295118331909, 0.2073686569929123, 1.43778395652771, 0.04844014719128609, -0.2373330444097519, -0.5102391242980957, -0.35207557678222656, -1.6569615602493286, -1.6984528303146362, -1.5539567470550537, 1.5020147562026978, 0.27311596274375916, 1.5837671756744385, -1.0394458770751953, 1.2591091394424438, -0.017425892874598503, -0.2261151820421219, -0.16027437150478363, 0.05126408487558365, -2.4675323963165283, -0.7691163420677185, -0.5025075078010559, -1.257835865020752, -0.4895124137401581, -0.5337324142456055, -0.12394967675209045, 0.9879169464111328, -0.5290858149528503, -0.22319534420967102, 0.8080146312713623, 0.13265414535999298, -0.9100081324577332, 1.8441729545593262, 1.1990565061569214, 0.4427777826786041, -1.1855862140655518, 0.13700279593467712, -2.4954240322113037, 0.5047240853309631, 0.5049228072166443, -1.526893138885498, -0.9231360554695129, -0.019775671884417534, 1.1740942001342773, -0.0014897065702825785, 0.010585521347820759, 0.1992809921503067, -0.24674972891807556, 1.4539049863815308, 1.1841299533843994, -2.3521299362182617, -1.5155142545700073, -1.2001807689666748, -0.42621999979019165], [0.6543092727661133, 0.7576210498809814, -1.2386400699615479, 1.342565894126892, 0.6321501731872559, 2.4687135219573975, 0.15112966299057007, -0.6063323616981506, -1.3615556955337524, -1.0737149715423584, -0.3149246871471405, 0.9750505089759827, -0.10433801263570786, 0.16426923871040344, -0.15161798894405365, 0.677821695804596, 0.6174991726875305, -0.746349036693573, 0.6715782284736633, 0.6173519492149353, 0.6903802752494812, -0.5970401167869568, -1.0251421928405762, -0.22663165628910065, 0.307533860206604, -0.003819113364443183, -0.46588417887687683, 0.9681828618049622, 1.598087191581726, 1.3687199354171753, 0.01604492962360382, -0.28175196051597595, 1.1086496114730835, 0.034112680703401566, -0.27259787917137146, 1.27610445022583, 2.2687020301818848, 1.999474287033081, 0.6341585516929626, 1.8217346668243408, 0.3509911894798279, 0.2803947329521179, -0.4201720654964447, 0.4898795485496521, 0.8895827531814575, 0.31678900122642517, -0.494157612323761, -0.7533237338066101, -0.490939199924469, 1.3643243312835693, -0.6757083535194397, -1.5687097311019897, -0.9966614246368408, 1.345187783241272, -0.7253291606903076, -0.7830469012260437, -1.7116268873214722, -1.2518774271011353, -1.3022732734680176, 0.5148212909698486, -0.2218063473701477, -1.105320692062378, 0.24453651905059814, 0.3220876455307007], [-0.2224535048007965, -1.790144443511963, -0.2953571081161499, -0.09359192848205566, 0.22327013313770294, -1.9296526908874512, 1.5339419841766357, 0.23486889898777008, 3.608455181121826, 0.5542171001434326, 0.6895992159843445, -0.3926904797554016, -1.116058349609375, -0.03548402711749077, -1.4448487758636475, -1.2953238487243652, 1.0121169090270996, -0.059764135628938675, -0.6045379638671875, -0.9654756784439087, -0.1715603470802307, -1.3107858896255493, 0.5592815279960632, -0.3706659972667694, -0.031736861914396286, 0.7512344121932983, -1.1215311288833618, 1.3216149806976318, 0.2923884093761444, 0.7823492288589478, 0.8358693718910217, -0.5798563957214355, 0.818115770816803, 0.9007077217102051, -0.65929114818573, 0.4720979928970337, 0.8857013583183289, -1.5041847229003906, -1.1242868900299072, -0.9354969263076782, 0.7636212706565857, -1.2979373931884766, 1.1552823781967163, 0.8552295565605164, 1.4628641605377197, 2.4277424812316895, 0.47830790281295776, 2.016820192337036, -0.46101507544517517, 2.981276273727417, 0.6057673096656799, -1.428486943244934, -0.3920690417289734, -0.44752752780914307, 0.4079665541648865, -0.3174598813056946, -0.40853819251060486, 1.7978365421295166, -0.11465828865766525, 0.9213106036186218, -0.1413724571466446, 0.531686544418335, -0.9816693663597107, -0.753863513469696], [-0.41053199768066406, -0.45848360657691956, 0.04665430635213852, -0.5094877481460571, -1.0471060276031494, 0.9430567026138306, -0.8058047890663147, -0.43084076046943665, 1.2938770055770874, -0.9943022131919861, 1.057211995124817, 0.07486572861671448, 0.9628263711929321, 0.558803915977478, -0.5879017114639282, 1.2501916885375977, -0.4495692253112793, 1.8444442749023438, -0.847438633441925, 0.7607428431510925, 1.0222018957138062, 1.6864138841629028, -0.005863893777132034, -2.257197618484497, -0.01886424608528614, -1.7435991764068604, -1.1576895713806152, -0.41470998525619507, 0.20319435000419617, -0.7632526159286499, -0.22798767685890198, -0.36800864338874817, -1.2251219749450684, 2.3353335857391357, 1.8507713079452515, 0.10791408270597458, -0.1890268325805664, -0.9309121966362, 1.5163663625717163, 0.9482980966567993, -0.8166226744651794, 1.7161568403244019, 0.8313274383544922, -0.6886805295944214, -1.2450309991836548, -0.200607031583786, 0.5437256097793579, -0.3067364990711212, 0.8652973175048828, 0.6872016191482544, -0.40958601236343384, 0.575631856918335, 1.7347044944763184, 0.3958183228969574, 1.1986817121505737, -2.1539084911346436, -0.45646241307258606, 0.8280422687530518, 0.025150159373879433, 0.8323354125022888, 0.5856761336326599, 1.5515488386154175, -2.1690356731414795, 0.6922992467880249], [0.16386333107948303, -0.9260514378547668, 1.6228430271148682, -1.4082355499267578, 0.9310970306396484, 0.05245368927717209, 1.098437786102295, -0.273902952671051, 1.050886631011963, 0.30760008096694946, -0.1257316917181015, -0.9923574924468994, -1.3132303953170776, 0.14812389016151428, -1.9556334018707275, 0.9347942471504211, 0.39779379963874817, 1.0360088348388672, -0.8856682777404785, -0.13100537657737732, 0.27897754311561584, 0.8141103982925415, 0.37009820342063904, -0.07235012203454971, -0.19570481777191162, -0.3995928168296814, 0.3802478015422821, 0.7053307890892029, 1.5583925247192383, 0.2541276812553406, 0.02103373408317566, -0.17297115921974182, 0.6440848708152771, 0.5883525013923645, 1.0609923601150513, -0.22689668834209442, -0.6898037195205688, 0.5759937167167664, -0.6393754482269287, -0.11021880060434341, 1.5930792093276978, 0.3633137047290802, -0.595263659954071, 0.3272407650947571, -0.574837327003479, 1.170669436454773, 1.4678229093551636, -1.6182308197021484, 0.9893897771835327, 1.0902719497680664, -0.3346785008907318, -0.7320676445960999, -0.23704683780670166, -0.39944833517074585, 0.053717032074928284, -0.3097054064273834, -1.4385038614273071, -0.9367570281028748, 0.3417702913284302, 0.38353094458580017, 1.099899411201477, -0.1158318817615509, 0.6997837424278259, -0.14839011430740356], [1.1101881265640259, -0.46576789021492004, 1.4499688148498535, 1.220104455947876, 2.6995720863342285, 0.6924726963043213, 0.4861322343349457, 1.5744339227676392, -0.6272302865982056, -0.6428750157356262, -0.407270222902298, 0.9618326425552368, 0.42818278074264526, -1.4067999124526978, 1.1279423236846924, -1.087166428565979, 0.20327095687389374, 1.3309653997421265, 0.17501205205917358, 0.8525246381759644, 1.3875694274902344, 1.8145148754119873, -0.19096437096595764, -0.5542101263999939, 0.10562724620103836, 1.8785176277160645, 0.9936856627464294, -1.3948520421981812, 0.6481882333755493, -0.8065370321273804, -0.21089990437030792, 0.8154523968696594, -1.1392014026641846, -3.1862316131591797, 2.508481502532959, 0.8061290383338928, -0.4261837303638458, 0.20210230350494385, -0.8852640986442566, 0.9663023948669434, 1.608414888381958, -0.03146222606301308, -0.08921430259943008, 0.433022141456604, -1.107861876487732, 0.30567666888237, 1.197189450263977, -1.1157827377319336, 0.42704257369041443, 0.5456826686859131, -0.640609860420227, 1.6118406057357788, 1.1666312217712402, -0.7215842604637146, -0.6759857535362244, -2.511768341064453, -0.783642590045929, 1.6177510023117065, 1.0762708187103271, 1.3014456033706665, 2.1810131072998047, 0.3783740699291229, 0.2910582721233368, 0.277572900056839], [-1.2549940347671509, -1.5111981630325317, 0.4947372078895569, 0.4077257513999939, 1.0536856651306152, -1.4600176811218262, -0.13843950629234314, -0.65591961145401, 1.8315651416778564, 0.6740314960479736, -0.6665927767753601, -1.020308017730713, -0.21209537982940674, -0.2745690643787384, -0.7023305296897888, 0.45828574895858765, -1.5615155696868896, -1.8764681816101074, -0.5612879991531372, -0.16727504134178162, 0.9414217472076416, 0.5822699069976807, 0.7327990531921387, -0.4081743657588959, 0.9155645370483398, -0.8780858516693115, 0.290558397769928, -0.23252342641353607, -0.7162019610404968, 0.6837677359580994, 0.34084832668304443, -0.2335902452468872, -0.8657335638999939, -0.32758045196533203, -0.9309068918228149, 0.8175143599510193, 1.0694084167480469, 1.0393428802490234, 0.24456022679805756, 1.5435794591903687, -1.0065944194793701, -0.3044857084751129, -0.5782831311225891, -1.0432465076446533, -0.08973665535449982, -0.6805381178855896, 0.5893496870994568, 0.0928037166595459, -1.231316328048706, 0.1632157564163208, -1.0463870763778687, -0.7055664658546448, 0.44076505303382874, 1.294508457183838, 1.0823144912719727, 0.4575604498386383, -0.7681968808174133, -0.3625916838645935, 0.8588414788246155, -0.27765214443206787, -1.88765549659729, -0.811979353427887, -0.6167774200439453, 0.924407422542572], [-0.18212270736694336, 0.23756666481494904, -1.0723820924758911, 0.19584989547729492, -0.4698255956172943, 1.4521849155426025, -0.7617563009262085, -2.5567312240600586, 1.0747342109680176, -0.5232354998588562, 0.16635121405124664, -1.471177339553833, -1.7075773477554321, -0.5166046023368835, 0.6713181138038635, 0.41269612312316895, 1.330129861831665, -1.1759179830551147, -0.512165904045105, 0.18389977514743805, 0.2614355981349945, -0.6269432306289673, 0.11355624347925186, 0.6321814060211182, -1.0993043184280396, -1.6097831726074219, 1.2185351848602295, -0.5733100771903992, -0.818777322769165, -2.81133770942688, 0.07323908060789108, -1.2899854183197021, 0.1737760603427887, -0.7185903191566467, -0.005581486504524946, -0.962651789188385, -0.2287895679473877, -1.0536459684371948, -1.1942819356918335, -0.48473304510116577, 0.6999061703681946, -1.3362369537353516, 1.73774254322052, -0.8722366094589233, 0.33240729570388794, 0.7380886673927307, 0.2152954339981079, 0.20483899116516113, 0.07635039836168289, 1.1595981121063232, 0.7146376967430115, -0.3316306173801422, 0.8052828907966614, 0.47080889344215393, -0.5404073596000671, -1.5795083045959473, -0.748691201210022, -0.3924093246459961, 0.03674672171473503, 1.9006133079528809, -1.4755744934082031, 0.36182358860969543, -1.2245614528656006, 0.3039955496788025], [-1.3906458616256714, 1.2142643928527832, 0.7143062949180603, 2.2052700519561768, -0.3410245180130005, 0.7812025547027588, -0.5570215582847595, 0.27260640263557434, -0.43317660689353943, 1.033478856086731, 2.823352813720703, -1.0794974565505981, -0.8912840485572815, 0.3149884045124054, -0.4966195523738861, -1.325354814529419, -0.10454007238149643, 0.009568294510245323, -0.04509778320789337, -2.5226857662200928, 0.9670321941375732, 1.1063971519470215, 0.19675669074058533, 1.2884693145751953, 0.7288883328437805, 1.303696632385254, -0.5454303026199341, -0.5440130829811096, -1.4022324085235596, -0.4189482629299164, 2.946984052658081, 1.5551115274429321, -0.49419426918029785, -1.3334195613861084, 1.2337363958358765, -0.8422899842262268, -0.6298730969429016, -0.7899460196495056, 0.5325087308883667, 0.3260281980037689, -0.047051526606082916, -0.6810287237167358, 0.976076066493988, 0.3939688801765442, 0.6711961030960083, -0.43090587854385376, -1.2118265628814697, -0.9495766758918762, -0.17556622624397278, -0.8922065496444702, -0.8453736901283264, 0.8649332523345947, -1.1000173091888428, 0.806476891040802, -0.3782971203327179, 1.4158791303634644, 0.16262130439281464, 1.1629701852798462, -0.3749309480190277, -1.3675072193145752, 1.2814831733703613, 1.9235737323760986, -0.9239577651023865, 0.45598798990249634], [0.08640303462743759, -1.57748544216156, -1.6639577150344849, 0.06848204135894775, 0.03523982688784599, 0.26627129316329956, -0.6040270924568176, -0.389430969953537, -0.43078404664993286, -0.932189404964447, 0.2967352271080017, -0.9055285453796387, 0.9541403651237488, 2.4685704708099365, 0.02794474922120571, 1.579997181892395, -0.6103180050849915, 0.5092421174049377, -0.9899243116378784, 0.05595962330698967, 2.1325364112854004, 0.7907156348228455, -2.976303815841675, 0.9454606175422668, -1.6882891654968262, -0.5475831031799316, -1.1798925399780273, -0.1629616767168045, -0.3275872468948364, 1.1900606155395508, -0.7702796459197998, -0.5028803944587708, 0.1514599472284317, 1.6969352960586548, 0.8952962756156921, 1.3541258573532104, 0.11908093094825745, 2.802114248275757, 0.3047575354576111, 0.8781078457832336, 1.5637292861938477, 0.2893168330192566, 0.6840434670448303, -0.016348812729120255, -0.612693727016449, 0.13428428769111633, 0.2983185350894928, 0.6497153043746948, -1.1484029293060303, 0.14449279010295868, 1.2364675998687744, -1.3574720621109009, 0.7496646046638489, -0.9915451407432556, 2.2509381771087646, 0.9643754363059998, -0.6543278098106384, 0.8124423027038574, -1.8330286741256714, -0.44988778233528137, -1.5112682580947876, -1.272841215133667, -0.28587794303894043, -0.39162665605545044], [-1.0282219648361206, -1.102934718132019, -1.0957834720611572, 0.675015389919281, -1.681288480758667, -0.2589033842086792, 0.48741981387138367, 0.003910008352249861, 0.1846321076154709, 0.36610496044158936, 0.882758617401123, 0.6801280379295349, 0.440393328666687, -1.0167127847671509, -0.07536370307207108, 1.505924105644226, 0.8265978693962097, 0.4042660593986511, -1.5510621070861816, -0.41225314140319824, -0.7280681133270264, 1.0762712955474854, -1.0948524475097656, -0.7800604701042175, -0.7384238839149475, -0.38380900025367737, -1.4615763425827026, -0.5586195588111877, -0.9737793207168579, 0.773594081401825, -0.032885629683732986, 0.4746766984462738, -0.4515646994113922, 0.5699443817138672, 0.22685998678207397, -0.46519747376441956, 0.5059727430343628, 0.459886759519577, -1.7859095335006714, 1.061638355255127, 0.3115606904029846, 0.28225111961364746, -0.12334633618593216, 0.4800126254558563, -0.27063894271850586, -0.27885520458221436, -1.3361010551452637, -2.077536106109619, 1.4272105693817139, 1.4332475662231445, -1.492795467376709, 0.3718201220035553, 0.008168267086148262, -0.6571290493011475, -0.43440771102905273, 0.02827301062643528, -0.32713162899017334, 0.5692614316940308, 1.2920567989349365, 0.9403735995292664, -0.5646699070930481, -0.1962471753358841, -0.08526448160409927, 0.27560552954673767], [0.9259889125823975, 0.8298569917678833, 1.821384072303772, -0.5922359824180603, 0.40854164958000183, 0.01912776008248329, 0.11634690314531326, -1.1610610485076904, 0.5061966180801392, 0.5318661332130432, 0.3729589283466339, 1.7578264474868774, 0.5630368590354919, 1.0987194776535034, -0.36755648255348206, -1.8046505451202393, -0.3549310863018036, -0.9221607446670532, 0.7129902243614197, -1.24116849899292, -0.2476750612258911, 0.5740823745727539, -1.0087376832962036, -0.09486754238605499, -0.06376758962869644, 0.4822826683521271, -0.5000002384185791, 0.8619412183761597, 1.3634849786758423, -0.01696067303419113, -1.1596543788909912, 2.2324905395507812, -1.374827265739441, -0.39868247509002686, 0.3823860287666321, -1.419577956199646, 0.7136016488075256, -0.7175231575965881, -1.5745819807052612, -0.8515582084655762, 0.8558799028396606, -0.1800488829612732, 0.06691935658454895, -0.3183223307132721, 0.3000175654888153, -0.6485025882720947, -0.31553471088409424, -0.5022682547569275, -0.760145902633667, -0.5413587689399719, 0.8034203052520752, 0.35249364376068115, -0.4890781342983246, -1.3263434171676636, -0.34147804975509644, -0.9709673523902893, -0.48219582438468933, 0.6611588001251221, -1.466007947921753, 0.2514383792877197, -1.6035293340682983, 1.0666882991790771, -0.6703324317932129, 0.5296713709831238], [1.1078490018844604, -0.6937779188156128, 0.07816978543996811, 0.07176626473665237, -0.24612896144390106, -1.3048263788223267, 1.476926565170288, 1.4427509307861328, 0.2866097390651703, -0.09514170140028, -1.120919942855835, -0.5621344447135925, 0.18191978335380554, 0.18183663487434387, 0.9482868313789368, 0.4992356300354004, -1.2227765321731567, 1.3930598497390747, -1.4578593969345093, -0.7229653596878052, -0.9257276058197021, -0.1941891461610794, 0.7899471521377563, 0.12859123945236206, -1.4817332029342651, -2.3404533863067627, 0.2799103856086731, 1.0332459211349487, 0.7066388726234436, 0.07711990922689438, -0.7053825259208679, -1.172431230545044, 0.17334234714508057, 0.22576269507408142, -1.53328537940979, 0.7029954195022583, 1.1275633573532104, -0.9643405675888062, 0.5796478986740112, -0.3718956708908081, -0.7870745658874512, -0.5247900485992432, -0.21189801394939423, 0.8417251706123352, -0.9180371165275574, 0.81682950258255, -0.22934983670711517, 1.699772596359253, -2.181534767150879, 2.00087833404541, -0.10762717574834824, 1.4364686012268066, -1.5025224685668945, -0.49525758624076843, -0.10718879103660583, 0.1766240894794464, -0.01744733192026615, -0.1194150522351265, 0.5674687027931213, -1.6740981340408325, -0.06364735215902328, -2.8570778369903564, -0.11167334020137787, -0.9143378138542175], [-0.014676126651465893, -0.5099941492080688, 0.03325274586677551, -0.5608831644058228, 0.06369586288928986, 1.479987382888794, 1.0564606189727783, -0.8850421905517578, 0.74582439661026, -0.383369117975235, -1.3735947608947754, 0.6066171526908875, -0.7777072787284851, -0.5437060594558716, 0.9984888434410095, 1.692155122756958, -0.34153202176094055, 1.0742403268814087, 1.0264686346054077, -0.36789417266845703, -0.6775566339492798, 0.07333870977163315, -0.9314805269241333, 0.20892661809921265, 0.4657428562641144, -1.4323647022247314, 2.3284108638763428, 0.7170843482017517, -0.018786923959851265, 0.019036060199141502, -2.5290467739105225, 2.349538564682007, 0.8715699911117554, 1.0552486181259155, 0.5308427810668945, -1.5499094724655151, -0.9076846241950989, 0.532082200050354, 1.1842855215072632, 0.9664205312728882, 0.9980819225311279, -1.1576430797576904, -0.4864918887615204, -0.2509157955646515, 0.7428920865058899, 0.39557069540023804, 1.5035895109176636, 2.066492795944214, -1.0170953273773193, 0.5592542886734009, 0.24981802701950073, 0.9670004844665527, -0.39672717452049255, 0.45129555463790894, 0.7687627077102661, 1.4506535530090332, -2.5269761085510254, 1.1996748447418213, -2.3983898162841797, 0.43446314334869385, -0.13089311122894287, 0.7655120491981506, -0.42638295888900757, -2.04437518119812], [1.3924881219863892, -0.24742631614208221, 1.5649975538253784, -0.49572062492370605, 0.3799569606781006, 0.7550903558731079, 0.029677972197532654, 0.6654388308525085, 0.5879470705986023, -0.7232145071029663, -0.43004679679870605, -1.8686338663101196, -2.3745508193969727, -0.19318386912345886, 1.1745710372924805, -1.7793855667114258, 0.8522966504096985, -0.3736581802368164, -0.04827122390270233, 0.6775088310241699, 0.5531646013259888, 0.010653584264218807, 0.9271038770675659, -0.427731990814209, -0.8909690976142883, -2.4029877185821533, 0.4588788151741028, -1.5789754390716553, -0.6228023767471313, -1.6705743074417114, -0.8181768655776978, -0.9324976801872253, 1.8660807609558105, -0.0057223401963710785, 0.9130178689956665, 0.24895602464675903, -1.0988759994506836, 2.2243738174438477, -0.0971696674823761, -0.8469922542572021, 0.11725305020809174, 0.928888738155365, 0.3656570017337799, 0.4243558347225189, 0.21868135035037994, -0.6326394081115723, -1.462365984916687, -0.35759463906288147, -0.40910831093788147, 1.4984945058822632, -0.8111273050308228, 0.28440842032432556, 0.20592322945594788, 0.3135719299316406, -0.4473380148410797, 0.18153443932533264, -1.7214107513427734, 0.010535859502851963, -0.5011196732521057, -0.2275892198085785, -0.3537755608558655, 0.5034620761871338, -0.7619304060935974, 0.07759340852499008], [-0.9241093397140503, -0.8003089427947998, -0.09530047327280045, -0.07066499441862106, 2.0693633556365967, 1.762523889541626, -1.1723030805587769, -1.0753108263015747, -1.774727702140808, 0.2783370614051819, 1.705336570739746, -1.4322166442871094, 1.3691411018371582, -0.021179012954235077, -0.23421254754066467, 0.05909359082579613, -0.5347613096237183, -0.07848148792982101, 0.4209756553173065, -0.4495483636856079, 0.8521244525909424, 0.7763799428939819, -1.2697806358337402, 0.09481439739465714, -1.5315190553665161, -0.06147534400224686, -0.1151283010840416, 0.6310123205184937, 0.46178263425827026, -0.4463558495044708, 0.9866670966148376, -0.5550408363342285, 0.7105293273925781, -1.2294222116470337, -1.9512269496917725, -0.49738481640815735, 0.6580201387405396, -0.1964540332555771, 0.8089419603347778, 1.2829512357711792, 1.9227497577667236, -1.1274216175079346, -1.5367529392242432, -0.3483143150806427, 0.027619043365120888, 0.028773056343197823, -1.6426498889923096, -0.866752028465271, -0.1980852484703064, -0.15704070031642914, 0.9202064871788025, 0.1251991242170334, -0.23643338680267334, -0.16092486679553986, 0.17823922634124756, 1.092943787574768, -2.093965768814087, 0.10619691014289856, 0.20239530503749847, -0.14302513003349304, 0.5056195259094238, -0.2751685082912445, -0.5543383359909058, -0.28817155957221985], [-0.26484325528144836, 1.7203049659729004, 0.33705833554267883, 0.6708934307098389, 1.1288173198699951, -1.0272783041000366, -0.6877986788749695, -0.9267303943634033, 0.5578539967536926, 0.7478273510932922, -1.0074414014816284, -1.4337973594665527, 0.30908459424972534, 0.45677056908607483, -0.5318670868873596, -1.4209089279174805, -1.9362932443618774, -1.300729513168335, 0.7101044654846191, 0.9615042805671692, 0.5864700675010681, -0.4853779971599579, 1.5195001363754272, 2.5907962322235107, -0.2742919921875, -0.69978266954422, -0.4693450331687927, 0.7458983063697815, -0.9434770941734314, 1.5026224851608276, -1.4707952737808228, 0.11565838009119034, 0.4342120587825775, 0.33379310369491577, 0.20726290345191956, -0.26767081022262573, -1.9806241989135742, -3.0158567428588867, 0.22799734771251678, -0.1601533740758896, 0.16483911871910095, 0.6561458706855774, -2.572906732559204, 1.8163971900939941, 1.5134230852127075, -0.643115222454071, 1.0381492376327515, -0.31379976868629456, -1.2614206075668335, -1.6539864540100098, 0.7495545744895935, -0.0669916495680809, 0.15897399187088013, 0.5438433289527893, -0.009578781202435493, 1.7193195819854736, -0.436670184135437, -0.08820202201604843, 0.4108304977416992, -0.6968396902084351, 1.0883156061172485, 0.5590581893920898, -0.44821596145629883, 1.322748064994812], [0.5591835379600525, -1.970102310180664, -0.5460670590400696, -1.3682814836502075, 0.6095677018165588, 0.4102197289466858, -0.5392506718635559, -0.7395789623260498, 1.6612451076507568, 0.4223342835903168, 1.7662556171417236, -1.2691487073898315, -0.48687684535980225, -0.3482087552547455, -0.056787215173244476, -1.2990870475769043, -0.038778938353061676, -1.6527141332626343, -0.5317732691764832, 0.3179124891757965, -0.5590627789497375, -0.0847078487277031, -0.4223739802837372, 0.7651517391204834, 0.7447249889373779, -0.22975987195968628, 0.25810205936431885, -2.9537792205810547, 1.310523271560669, -0.6786451935768127, 0.45201390981674194, 0.7967584729194641, -1.1212981939315796, -1.4344345331192017, -1.5414823293685913, 0.19580352306365967, 0.16537894308567047, 0.6581461429595947, -0.5237462520599365, -1.3361939191818237, -0.8876993060112, -0.97158282995224, -0.3496652841567993, -0.41680100560188293, -1.6229526996612549, -0.22058658301830292, 0.2681732773780823, 0.34681442379951477, -0.7512233853340149, -1.2523417472839355, -1.2829952239990234, -0.23704113066196442, -1.2533535957336426, 1.0296425819396973, 1.029863953590393, -1.326396107673645, 0.24564121663570404, 0.45012447237968445, 0.005667408462613821, 0.7554566264152527, -2.120676040649414, 0.5923840403556824, 0.6613917946815491, -1.4510802030563354], [-1.9685736894607544, -0.10208653658628464, 0.2541615068912506, 0.5955690741539001, -0.9262365698814392, 0.4822898805141449, -0.9202108979225159, -0.4751196801662445, -2.0716428756713867, 0.2511953115463257, 1.3931348323822021, -0.6029160022735596, -1.7301217317581177, -0.7596976161003113, 0.37178662419319153, 1.3215453624725342, -0.6622955203056335, -0.22244934737682343, -1.3946828842163086, -0.22168104350566864, 0.9367417693138123, -0.5289894938468933, 0.6151794195175171, 0.2908059060573578, 0.2112809121608734, 0.8073297142982483, -0.825383186340332, -2.0576589107513428, 0.1959843784570694, 0.7507236003875732, 0.7073281407356262, -1.0221105813980103, 0.9029943943023682, -0.32682475447654724, 1.0864838361740112, 1.0818967819213867, -0.27517828345298767, 0.07978980988264084, -0.5044687390327454, -0.5276710987091064, -1.0289479494094849, 1.1613506078720093, 1.6478217840194702, 0.3243795931339264, 0.7863476872444153, -0.14434805512428284, 0.27893969416618347, -0.3621784448623657, -0.40502896904945374, 1.0775957107543945, 1.0474891662597656, -0.26825496554374695, -1.6241074800491333, -0.035583578050136566, -1.5543015003204346, -1.5639256238937378, -0.22866959869861603, 0.4384252727031708, 0.6393473744392395, -1.0661849975585938, -0.6987355947494507, 0.5970210433006287, 1.7135391235351562, 0.027889687567949295], [-0.7612841725349426, -0.4366787075996399, 0.2550102174282074, 0.011335543356835842, 0.7531824111938477, 0.0051565514877438545, 0.6561573147773743, -0.4859267771244049, 3.046038866043091, -0.1146925613284111, -1.9846878051757812, -0.8523533344268799, 0.509190022945404, 0.1355528086423874, 0.6741713881492615, -0.49742352962493896, -0.8592541813850403, 1.0788716077804565, 1.027118444442749, 0.29051750898361206, 0.38667088747024536, -0.10022084414958954, 0.485724538564682, -0.2821906805038452, 1.4366254806518555, -0.4711954891681671, 1.1649919748306274, -0.6645775437355042, -0.5847131609916687, 1.58147394657135, -0.9131119251251221, -0.5709129571914673, 1.1634397506713867, -0.2624419927597046, 1.6213536262512207, -0.5915422439575195, 0.6443504691123962, 0.7924596667289734, 2.5002832412719727, -0.747180700302124, 1.2977399826049805, -1.8637847900390625, 1.344611644744873, -0.621097981929779, 3.3922879695892334, -1.3393807411193848, 1.0866854190826416, -0.6776052713394165, -0.5834656953811646, -1.361290454864502, 0.5228806734085083, -2.361025333404541, -0.2602260708808899, -0.10761458426713943, -0.10915794223546982, 2.0836687088012695, -0.36940306425094604, -0.7327318787574768, -2.0846757888793945, 0.6160919070243835, -0.15036910772323608, -0.3660956621170044, 1.2104129791259766, -1.4404053688049316], [-1.3993463516235352, -1.0041909217834473, -0.18715184926986694, 0.14335116744041443, -0.4056037366390228, 1.4904860258102417, -0.8358663320541382, -0.721562922000885, -0.875622570514679, -0.32378286123275757, 0.2527371048927307, 0.6232154965400696, 0.6879622340202332, 0.2623111307621002, -1.4465792179107666, -0.1914759874343872, 0.626506507396698, -2.0388240814208984, 0.09058468788862228, -0.320467084646225, 1.3344929218292236, -0.9034352898597717, -0.7676700353622437, -2.288079261779785, 1.0865579843521118, 0.2472783774137497, 1.6844630241394043, 0.30890122056007385, -0.4462987780570984, 0.2951135039329529, -0.7794882655143738, -0.434911847114563, -0.3898014724254608, -1.1968551874160767, -0.3411049246788025, -0.3835888206958771, 1.6796423196792603, -0.16564799845218658, 0.39633122086524963, 0.09533684700727463, -0.36404895782470703, 0.4015530049800873, -1.088327169418335, 1.4851577281951904, -1.299564242362976, 0.1804654747247696, -0.10799828916788101, -0.3760048449039459, -1.9177285432815552, 0.4827020764350891, -0.5661914944648743, 2.378092050552368, -1.2519164085388184, 0.5087918043136597, 1.8184934854507446, -0.2256443351507187, 0.1433427780866623, -0.6949695944786072, -1.9238677024841309, -0.9889053702354431, -1.6752276420593262, 0.7719980478286743, 0.5715705156326294, -0.28473639488220215], [1.4669256210327148, 1.1253057718276978, 0.7922295331954956, -0.1752028614282608, 0.519771933555603, -0.04656897485256195, -0.31866034865379333, 0.8604191541671753, 0.22974562644958496, 0.8305652737617493, 0.37441784143447876, 0.7312660217285156, 0.7853390574455261, 0.267426460981369, -0.5974261164665222, -1.4779094457626343, 1.0519273281097412, -0.20655888319015503, 1.159929871559143, -1.3016877174377441, 0.914222776889801, -0.9705234169960022, 0.4563705027103424, 0.7694453597068787, -0.5959404706954956, -1.1011009216308594, 1.0087974071502686, -0.9996708631515503, -0.38817694783210754, 0.11017777770757675, 0.8799296617507935, -0.45227962732315063, 0.3120017945766449, 2.1521685123443604, 0.3387816250324249, -1.2187299728393555, 0.8213021755218506, 0.3342202305793762, -1.5112974643707275, 1.087200403213501, -0.28305697441101074, -1.0919466018676758, 0.13407866656780243, 1.156750202178955, 0.741350531578064, 0.42471596598625183, -0.39237430691719055, -0.23980270326137543, -0.7771337628364563, -2.617687702178955, -1.3766082525253296, 1.7071868181228638, -0.49496912956237793, -1.7209267616271973, 0.2582332491874695, -0.21374395489692688, -0.02775643765926361, 0.632560670375824, 0.517722487449646, 0.8264788389205933, 0.3807257413864136, -1.112779140472412, -0.9051665663719177, -0.06709922105073929], [-0.4159059226512909, -0.917352020740509, 2.9727237224578857, 0.30627864599227905, -1.641894817352295, 0.8543736934661865, -0.17821280658245087, -1.3474375009536743, -1.9784914255142212, 1.9009214639663696, 1.583390712738037, 1.1216473579406738, 0.631325364112854, 0.4228622019290924, 1.0314663648605347, 1.4627301692962646, -0.12268730252981186, 0.14457903802394867, -0.6339568495750427, -0.37696048617362976, -0.8518315553665161, -0.7628359198570251, -0.6811292171478271, -0.6094161868095398, -0.4428222179412842, -0.7096826434135437, -0.0553944893181324, -0.5635973215103149, 0.4555228650569916, -0.3950454592704773, 1.6774952411651611, 0.625159502029419, 1.1656651496887207, 1.0909498929977417, 2.625589609146118, 1.356461763381958, -0.6851714849472046, 0.11251135170459747, -1.2694470882415771, 0.3403591811656952, -0.4045088589191437, -1.2105053663253784, -0.18426308035850525, 0.07778813689947128, -1.1066606044769287, -0.48365283012390137, 0.20544248819351196, 0.09098267555236816, 0.8551322221755981, 0.17461350560188293, 1.7705280780792236, 0.7243339419364929, 0.47685322165489197, 0.312031626701355, 0.05791086703538895, 0.4613828659057617, 1.8855890035629272, 0.3445178270339966, -1.6013875007629395, 0.7258005142211914, -0.10725881159305573, -1.364344835281372, -0.5406809449195862, -2.160691499710083], [0.5209314227104187, -0.6452214121818542, 1.999635934829712, 0.5895044207572937, 1.0138863325119019, 0.5374676585197449, -0.03427568078041077, -0.9571505188941956, 0.6023433208465576, -0.7945728302001953, -0.14948417246341705, -0.7719845175743103, 0.6457850933074951, 0.03618727624416351, 1.0609066486358643, -1.5169426202774048, -0.07055842131376266, 0.4140782654285431, -0.5499810576438904, -0.38069599866867065, -0.1318625509738922, 0.5699912309646606, -2.3333282470703125, -0.8577178120613098, 1.5088982582092285, -1.0407413244247437, 2.041515588760376, -1.2613213062286377, 0.7124778628349304, -0.43804851174354553, -1.292856216430664, -0.025910548865795135, -0.6409082412719727, 0.6549813747406006, 0.2200968712568283, 1.0327515602111816, 0.5573307275772095, 0.4036066234111786, -0.9634202718734741, -1.790326714515686, 0.8101244568824768, 1.9867303371429443, 1.3170263767242432, 0.471019446849823, 0.851428747177124, 0.5509372353553772, 0.19389109313488007, -0.18534258008003235, -0.039231520146131516, -0.12348063290119171, -0.2992037832736969, -1.2182070016860962, -0.8772361278533936, -0.38393139839172363, -0.6816831231117249, -1.0491052865982056, 0.8581984043121338, 1.134406328201294, -1.3261140584945679, -1.7371481657028198, 0.503393292427063, -0.26514720916748047, -0.23049768805503845, 1.412469744682312], [-0.48853445053100586, 1.4281420707702637, 1.1627111434936523, -0.23789551854133606, 1.550467610359192, -1.4723460674285889, -0.7101208567619324, 1.1311815977096558, 0.1626831442117691, -0.8968467712402344, -0.27578386664390564, -0.517704427242279, -0.12993969023227692, 0.4618363082408905, -0.0348355658352375, 0.32369762659072876, 0.30354830622673035, 0.5028684735298157, 0.6305423378944397, -0.42482855916023254, -1.6710093021392822, -1.1511387825012207, 0.3725071847438812, -1.3945436477661133, 2.1416890621185303, -0.25713714957237244, 0.378775417804718, 0.8449340462684631, 0.2838839292526245, 0.06774856895208359, 0.7142276763916016, 0.12142294645309448, -1.0102524757385254, 0.007813718169927597, 0.5070194005966187, 0.4279598593711853, 0.5679309964179993, -0.2142299860715866, 0.3352203965187073, -1.1297478675842285, 0.5008020401000977, -1.0269057750701904, -0.5054630041122437, -0.1482592225074768, 1.2777838706970215, 0.05031242221593857, 0.8090788125991821, 0.2744584083557129, -1.112726092338562, 0.3258554935455322, 0.2200806736946106, 0.6882832646369934, 1.6753360033035278, 0.7211422920227051, 0.15261276066303253, -0.005054818000644445, 1.7432875633239746, 0.27957966923713684, 2.210658550262451, -0.026315025985240936, -0.8282840847969055, -0.49802958965301514, -2.399851083755493, -0.1132008358836174], [-0.16073480248451233, 0.827024519443512, -0.42166265845298767, -0.050371069461107254, -0.5766175985336304, 0.7602887153625488, 1.6472667455673218, -0.3734576106071472, 1.1873435974121094, 1.034349799156189, -0.364319384098053, -1.1655688285827637, 0.3817448914051056, -1.042871356010437, 0.782631516456604, 1.4525529146194458, 1.0646591186523438, -0.6495596766471863, 0.5291461944580078, 1.644837737083435, 0.8856788277626038, -0.664090096950531, 0.5244727730751038, 0.4862965941429138, 1.8592774868011475, -1.746801495552063, 0.029156295582652092, 0.3994531035423279, -0.2237238883972168, 0.8623514175415039, 0.9002599716186523, -0.4359894096851349, -0.644959568977356, 0.7699050903320312, -1.4572761058807373, 0.9369848370552063, -0.32319527864456177, -0.5608139038085938, 0.16168609261512756, 0.35449498891830444, -0.6920430660247803, 1.455109715461731, -2.2132227420806885, -0.654341995716095, -0.5356711745262146, 1.873924732208252, -1.3031195402145386, 0.5209614038467407, -0.27523505687713623, -0.7060773372650146, 0.2509431838989258, 0.7198915481567383, 0.48397451639175415, 0.453628271818161, -0.6267358064651489, 0.26257985830307007, 1.0807300806045532, -1.2533031702041626, -0.014513051137328148, 0.034892089664936066, 0.40138059854507446, -1.076018214225769, -0.7178155183792114, 0.31860676407814026], [0.0932760089635849, 0.0148660559207201, 1.3629646301269531, 0.33728939294815063, 0.5829479098320007, 0.44924911856651306, 0.7471662759780884, -0.4133651554584503, -1.244789958000183, -0.4033370018005371, -3.0000290870666504, 1.0448377132415771, -0.43625834584236145, 0.15138588845729828, 0.032821983098983765, 0.2502340078353882, 0.5982444882392883, -0.2535143494606018, 1.0944784879684448, -0.8916631937026978, 1.653668999671936, 1.6523422002792358, -0.7776502966880798, 1.3694185018539429, 0.26123589277267456, -1.4249069690704346, 0.6369993090629578, -0.8599062561988831, -0.49511468410491943, 1.6364516019821167, -0.557604968547821, 2.213873863220215, 1.2141011953353882, 1.151930570602417, 1.1431552171707153, 0.8180180191993713, 0.3246365487575531, 1.1496644020080566, 1.333639144897461, -0.9774903059005737, -1.0190995931625366, -0.9105333685874939, -1.996478796005249, -1.1652357578277588, 1.1632012128829956, 0.14813809096813202, -1.9601954221725464, 0.6094963550567627, -0.2180531769990921, 0.3436274826526642, -0.31247478723526, 2.072012424468994, -0.14937451481819153, 2.515801191329956, -0.2593647837638855, -0.5446115136146545, 0.6813866496086121, -1.116626262664795, -0.6135614514350891, -1.981726050376892, -1.3045960664749146, 0.07166195660829544, -0.22920554876327515, 1.5466246604919434], [0.9073582887649536, -2.132052421569824, 0.867127001285553, -0.39321351051330566, 0.628044068813324, 0.799320638179779, -0.6295420527458191, 2.344461679458618, 1.0392322540283203, 0.8915991187095642, -1.459946870803833, 0.6492729187011719, -0.6749804615974426, 0.5096831321716309, -0.19957774877548218, -0.8749990463256836, 0.9307422041893005, 0.3464187979698181, -0.5491204857826233, -0.03438805416226387, 0.7182279229164124, 1.1942267417907715, 0.6097551584243774, 1.495876669883728, -1.3403462171554565, 0.5293750166893005, -0.08615464717149734, -0.5713925957679749, 1.0708783864974976, -0.43944627046585083, 0.5253294110298157, 0.1283845603466034, 0.61701899766922, 1.8674654960632324, 1.3318408727645874, 1.544823408126831, 1.138317584991455, 0.7010888457298279, -0.7157345414161682, 0.32740265130996704, 1.2395092248916626, -0.26572033762931824, 1.0189396142959595, 0.7188576459884644, 0.5646156668663025, -0.09583533555269241, -0.8997251391410828, -1.2593128681182861, 0.5455091595649719, 0.0398581437766552, 0.6506158709526062, -0.9012563824653625, -0.25806280970573425, 0.6744737029075623, 0.7643333077430725, 1.4911388158798218, -1.9129102230072021, 0.498359739780426, -2.0436437129974365, -1.0498504638671875, 1.5388948917388916, -0.9375601410865784, -0.6284431219100952, -1.297692894935608], [-0.8381327390670776, 0.29954445362091064, -0.006587489973753691, 0.12052391469478607, -0.07499917596578598, -2.5582892894744873, 0.6925846934318542, 0.43158701062202454, -0.2462586909532547, -0.09768544137477875, 0.9516302347183228, -0.4013412892818451, 0.7574272155761719, 0.7166186571121216, 1.4781830310821533, 1.8177125453948975, 1.4400639533996582, 0.6825277805328369, -0.18230940401554108, 0.6485052108764648, -0.9610077142715454, 0.2289944887161255, -0.7757936716079712, -0.16062678396701813, -0.5190728306770325, -1.2154271602630615, 0.4884474575519562, -1.5466943979263306, -0.8670827746391296, 0.12073246389627457, -0.38313204050064087, -0.9452948570251465, -0.8672897815704346, -1.4667437076568604, -1.2499113082885742, 0.6207647323608398, -1.1809003353118896, 0.5035497546195984, -0.5089888572692871, 1.2349556684494019, 1.2952505350112915, -0.3869419991970062, -1.0598708391189575, -0.3151077628135681, -0.027676792815327644, -1.9599157571792603, 1.474733591079712, 1.026021122932434, -0.6675513982772827, 0.11821170896291733, -0.39698681235313416, 0.2888440191745758, 0.1312733292579651, 0.7004501819610596, 0.294739305973053, -0.1597055196762085, -1.923017978668213, 0.45358312129974365, -1.67618727684021, 0.9097963571548462, 0.5105501413345337, 0.05906473845243454, 0.3815256953239441, -0.8258714079856873], [-0.3656763434410095, -0.19557130336761475, 1.1543926000595093, 1.7033939361572266, 0.06899544596672058, 0.07556183636188507, 1.0103625059127808, 2.041508197784424, -1.3846975564956665, -1.6695458889007568, 0.05939219146966934, 0.1631091833114624, -0.5792710185050964, -0.29669904708862305, 0.44955316185951233, -1.5809332132339478, -0.7039545774459839, 1.0746396780014038, -0.08006783574819565, 0.927200198173523, 1.2457489967346191, 0.6239538788795471, 0.30559009313583374, -0.16979095339775085, -1.5167229175567627, 1.2738171815872192, -1.2358715534210205, -0.28355515003204346, 1.7747095823287964, 0.8706040978431702, -0.05491877719759941, -0.31797468662261963, -0.22683221101760864, 1.2778445482254028, 0.3010789453983307, 0.6733945608139038, -0.6230660676956177, -1.010877251625061, 1.807567834854126, 0.6199731826782227, -1.2443517446517944, -0.33819380402565, 0.33035755157470703, -0.2400692254304886, 0.7460806369781494, -0.2132052779197693, -1.9330496788024902, 0.21836219727993011, 0.510860800743103, 2.7021372318267822, -1.8492635488510132, 1.353212594985962, 1.2877600193023682, 0.5651974081993103, -0.2699131667613983, -1.228074073791504, -0.7850884795188904, -2.7587785720825195, -0.03979215770959854, -1.8277552127838135, 1.2549022436141968, 1.0415550470352173, -0.08672448247671127, -0.04187297448515892], [2.425440788269043, 1.120985984802246, -1.141601800918579, -2.5819549560546875, 0.10019820183515549, 0.9115660786628723, -0.13248185813426971, -2.2794265747070312, -2.4260451793670654, 1.8759268522262573, 0.1882515549659729, 0.5794792175292969, 0.08154139667749405, 0.5175632834434509, 2.423630714416504, 0.5227450132369995, 0.20685161650180817, -1.6624207496643066, 0.7287341356277466, -0.19568681716918945, -0.11900678277015686, -1.3515781164169312, -0.06460226327180862, -0.1558152139186859, -0.0638115182518959, 0.9443120956420898, 1.4057003259658813, 0.1273246854543686, -0.2287929207086563, 1.111171007156372, -0.786213755607605, -0.09048383682966232, 0.21709485352039337, 0.2378169745206833, -0.7055800557136536, -0.2639983594417572, -0.15421006083488464, -0.28001418709754944, -1.3975584506988525, -0.002865409478545189, -2.0537819862365723, -0.4049147069454193, 1.523851752281189, 0.5309441089630127, 0.12014614045619965, -1.2247297763824463, 2.627500534057617, 0.6437212824821472, -1.188758134841919, -1.8802108764648438, 0.7343384623527527, -1.403472900390625, -0.24368256330490112, -0.001400607987307012, 0.7764076590538025, 0.3010728359222412, 1.0824499130249023, -0.8729903101921082, 0.7796144485473633, 0.8461856842041016, 0.10803025960922241, -0.06799757480621338, 2.598776340484619, 0.053513213992118835], [1.1214107275009155, 0.33079326152801514, -1.035256028175354, -1.625463843345642, -0.5077635645866394, -0.13402070105075836, 1.623136281967163, -0.8114088773727417, 0.1792953610420227, -1.6102384328842163, 1.142262578010559, 2.2390778064727783, -1.2641295194625854, -2.6242847442626953, 0.19380402565002441, -0.21506786346435547, 0.19955718517303467, -0.08073507994413376, -0.23715539276599884, 0.5292698740959167, -0.1447383314371109, 1.512381911277771, -1.3676763772964478, 1.3745529651641846, 0.2427348494529724, -0.37087151408195496, -2.015359401702881, 1.2326768636703491, -1.0562875270843506, -0.20448051393032074, 0.07718642801046371, 0.014605105854570866, -0.061382096260786057, 0.27778396010398865, -0.18600401282310486, -0.01701037585735321, -0.19043810665607452, -0.9677376747131348, -1.1700955629348755, -0.816515326499939, -1.5373332500457764, -0.2327255755662918, 0.22987036406993866, 0.5602249503135681, 1.6468793153762817, -0.5226758718490601, 0.6397508978843689, -0.5995793342590332, -0.685878336429596, -1.131052851676941, 0.11459924280643463, 0.49098798632621765, -0.6894207000732422, -0.879734992980957, 0.38395926356315613, -0.986425518989563, 0.9366899728775024, -0.6128010153770447, 1.2802047729492188, 1.3706591129302979, -0.8300632834434509, 1.9068632125854492, 0.030253617092967033, 0.029406875371932983], [0.822030782699585, 0.13139450550079346, 1.3662703037261963, 0.6131179332733154, 0.7739613056182861, 0.027688847854733467, -0.8290128707885742, 0.22466346621513367, -0.981505274772644, -0.6388256549835205, 0.8804931640625, 1.5991061925888062, 0.9396981000900269, 0.8106171488761902, -0.1682131141424179, 1.0971503257751465, 1.7360581159591675, 0.7077094912528992, -0.29367226362228394, 0.37386149168014526, 1.8636066913604736, -0.12244166433811188, -2.0511319637298584, -0.386025071144104, -0.09995409101247787, 0.5916690230369568, 1.3467018604278564, 0.5967196226119995, 0.07767163217067719, -1.4280247688293457, 2.3309149742126465, -3.645404577255249, -1.1209542751312256, 0.7020728588104248, -0.2017209380865097, -0.5875264406204224, -0.2449103146791458, 0.9929823875427246, -0.059163302183151245, 1.3611111640930176, 0.4560951888561249, -0.6112517714500427, 0.8270249366760254, 0.14964787662029266, 0.12663821876049042, -0.9005928635597229, -0.3406173884868622, -1.6242820024490356, -1.766160488128662, 1.1035343408584595, -0.05735108256340027, -0.9061632752418518, -0.6885454058647156, 1.183929204940796, -0.3146207630634308, 0.5625413060188293, 0.8130235075950623, 1.8816763162612915, -1.0797821283340454, 0.3079349100589752, 0.6925870180130005, 1.0889290571212769, 0.24285542964935303, -0.46790650486946106], [0.6053913235664368, -0.5660372972488403, -0.36548498272895813, -0.5920677781105042, 0.8273406624794006, -0.24722009897232056, 0.11782520264387131, -0.33638516068458557, 2.1683216094970703, 1.5805500745773315, -0.6957290172576904, -0.3293519914150238, -0.9610724449157715, 0.7744240760803223, 0.03690985217690468, -0.6995065212249756, 1.4435572624206543, -0.27497079968452454, 0.22382059693336487, -0.2379893660545349, -0.4904358386993408, 0.09058056026697159, -1.5363761186599731, 0.0037550306878983974, 1.022587537765503, 0.7795016765594482, 0.5573732256889343, -0.24570472538471222, 0.589680552482605, 1.1417444944381714, 0.1104678362607956, 0.048892077058553696, -1.3741122484207153, 2.539113998413086, -0.8780603408813477, 1.5031436681747437, -0.304782897233963, -0.9690166711807251, -0.7728255987167358, 0.5081741213798523, 0.11478621512651443, -1.0254418849945068, 0.5492731332778931, 0.4792460799217224, 0.3291778564453125, 1.1395304203033447, 0.07735181599855423, -1.4220203161239624, -0.1261458545923233, 0.21354062855243683, 1.5972719192504883, -1.7050681114196777, 2.270512104034424, 0.7601231932640076, 1.6017762422561646, 1.412076711654663, -0.19844599068164825, 0.15725237131118774, 1.5668338537216187, -0.5351474285125732, 1.6677181720733643, 2.1943416595458984, -0.27415841817855835, 0.07615228742361069], [-1.191138505935669, 1.1237789392471313, 1.2226428985595703, 0.4500772953033447, 0.06629875302314758, 0.23395024240016937, 0.9199733138084412, 0.26251617074012756, 0.7714253664016724, -0.7744025588035583, 1.0231529474258423, -1.044816255569458, 2.7688539028167725, -0.687030553817749, -0.9776480793952942, -1.3060450553894043, 1.285718321800232, 0.8940331935882568, -0.5555530786514282, 0.5896967649459839, 1.1599445343017578, 1.5884629487991333, -0.6327314972877502, 0.49399957060813904, 1.6816118955612183, -1.3262856006622314, 0.0718696340918541, -0.8376181721687317, -1.1897971630096436, 0.6253055334091187, 0.2803502082824707, -0.0842583104968071, -0.24081099033355713, -1.099059820175171, 0.18297487497329712, -1.818006992340088, 0.8532857298851013, -0.9927385449409485, 0.6564247608184814, -2.3276302814483643, 0.15992781519889832, -0.34695225954055786, -0.8674182891845703, 0.22343569993972778, -1.1681255102157593, 1.0743272304534912, 0.26854076981544495, -0.36694958806037903, 0.49859121441841125, -2.270827054977417, 0.5146902203559875, -0.2228052020072937, 0.03245015814900398, 1.7149949073791504, 2.252436637878418, -0.7386325001716614, 0.1683099865913391, -1.1938444375991821, 0.8736521005630493, 0.3054478168487549, -0.7998802065849304, 0.25808772444725037, 1.3605449199676514, 1.191527247428894], [-1.230443000793457, 0.40041449666023254, -0.9724366068840027, -0.09919323027133942, 0.15593719482421875, 1.7554309368133545, 0.2702166736125946, -0.24656391143798828, -0.6324533820152283, -0.219173863530159, -1.3986274003982544, -1.2122774124145508, 1.8779140710830688, -1.003915548324585, -1.8760532140731812, 0.7956093549728394, -1.1338844299316406, 0.3601550757884979, -0.44457539916038513, 0.6904814839363098, -1.0473337173461914, 0.9697302579879761, -0.5987713932991028, -0.2065768837928772, 1.158388376235962, -0.9384933710098267, -0.6135002374649048, -1.1727205514907837, 0.3608485758304596, 1.5683680772781372, -2.0501644611358643, -0.5782585740089417, -0.42062506079673767, -1.3075950145721436, -0.7689003348350525, 2.052135944366455, 0.07896525412797928, 1.940159559249878, 0.6770420670509338, -0.9322941899299622, -0.22639116644859314, -0.4801795482635498, -0.5087189674377441, -0.761123538017273, -0.06649825721979141, -0.26571258902549744, -1.9285252094268799, 0.4707005023956299, -0.17800922691822052, -0.03549379110336304, 0.12323950231075287, 1.4327646493911743, 0.5100658535957336, 0.33527395129203796, 1.4530214071273804, 1.3302680253982544, 0.4555126428604126, 0.4903702735900879, 0.4298939108848572, -1.340337872505188, 0.36076420545578003, -1.5117372274398804, -1.0599223375320435, -0.9917548298835754], [-0.6579581499099731, 0.6281953454017639, -0.0017820079810917377, -0.9755605459213257, 1.6285431385040283, -1.0393587350845337, -1.0941963195800781, 1.1036981344223022, 1.081631064414978, -1.814060926437378, -0.36158883571624756, 0.5302487015724182, 0.03817922994494438, -1.3245092630386353, -0.6864011883735657, 1.3223758935928345, 0.7779486775398254, -0.6884164810180664, 1.4193172454833984, -0.9854461550712585, 0.8226463794708252, 0.32012975215911865, -0.5257553458213806, -0.4077412486076355, 1.4713910818099976, -0.6420926451683044, -0.6620833277702332, -0.3830227851867676, 0.027054857462644577, -0.12510237097740173, -0.44384607672691345, -0.3911902904510498, 2.238515615463257, 0.17637579143047333, 1.3015797138214111, -0.41191938519477844, -0.4253607392311096, 0.014073380269110203, 0.2539507746696472, 0.6754218339920044, 0.8555478453636169, 0.6948527097702026, 0.9687394499778748, 1.2331035137176514, 0.04295732080936432, 0.4902893304824829, 0.7061589360237122, -0.07233955711126328, 0.3654235005378723, -0.9157355427742004, -0.03655020147562027, 1.30647611618042, 0.48516586422920227, -0.4839416444301605, 0.20826739072799683, -0.2063726782798767, -0.14307956397533417, -0.24280945956707, -1.6265367269515991, 1.9276334047317505, 0.29668861627578735, -0.09983745962381363, 0.4691830277442932, -0.553458571434021], [0.12954777479171753, 1.4443647861480713, 0.24403879046440125, -0.006478003691881895, -1.2905086278915405, -0.33861979842185974, 1.3280495405197144, 0.2470341920852661, 0.2917550504207611, 0.6439493894577026, -1.2947264909744263, -0.0194084495306015, -0.03227156400680542, 0.1694890558719635, -0.9096937775611877, 0.24952955543994904, 0.16890111565589905, 1.2619704008102417, 0.5587553381919861, 1.1272473335266113, 1.5852348804473877, 0.29562482237815857, -0.69781494140625, 1.240439534187317, 0.11909769475460052, 0.19824573397636414, 1.9262727499008179, 0.5382506847381592, 1.0143253803253174, -1.5455206632614136, 0.7716735601425171, -0.8561365008354187, -1.1656193733215332, 0.9000931978225708, -0.13521116971969604, 1.0067932605743408, 0.6151115894317627, 0.5956501364707947, 0.32911884784698486, 1.5245914459228516, 2.3818435668945312, -0.24764521420001984, 0.9125037789344788, 0.20150715112686157, 0.7973120808601379, 0.3780018091201782, 0.17419742047786713, -0.4178093671798706, -0.7237171530723572, 0.3891902565956116, -1.0834356546401978, -0.1261965036392212, -0.27501043677330017, 0.7439132928848267, -1.4539774656295776, 0.41001424193382263, -0.5216374397277832, 0.9850649833679199, -0.6638427972793579, 0.4389295279979706, -1.1995878219604492, 1.5249278545379639, -2.015186071395874, -1.473191499710083], [-0.4198533594608307, -1.4939804077148438, 2.3942482471466064, 1.5726100206375122, -1.5334047079086304, -0.22766953706741333, 0.2489219456911087, 0.44914671778678894, -0.6753026247024536, 0.6410418748855591, -0.3041033446788788, 0.7780518531799316, -1.1637037992477417, -0.0038128579035401344, 1.3152625560760498, 0.3795568645000458, -0.13585250079631805, -2.0373966693878174, 0.492508202791214, -0.3772163391113281, 0.23734720051288605, -0.5959292054176331, -0.5175074934959412, 0.31883177161216736, 1.541825532913208, 1.1293421983718872, 1.4649696350097656, 1.2096699476242065, 0.24131210148334503, 0.10793886333703995, 0.33546018600463867, 1.8367176055908203, 0.3125958740711212, -0.14637508988380432, -1.2632759809494019, -1.016401767730713, -1.0892136096954346, -0.41671913862228394, -0.4813278019428253, 0.9735355973243713, -0.2586767077445984, 0.0261172354221344, -0.21704337000846863, -0.13641829788684845, 0.3498283326625824, -1.4037561416625977, 2.094782829284668, 1.2702381610870361, 0.6816046237945557, 1.0098228454589844, -0.8590010404586792, 1.0273308753967285, 1.2186237573623657, -0.8345503211021423, -0.21453043818473816, 1.1226383447647095, 0.689458429813385, 0.7463758587837219, -0.9057138562202454, -1.555011510848999, -1.304059386253357, 0.16313624382019043, -0.5330415368080139, 0.9969508051872253], [2.2122228145599365, 0.2450757771730423, 0.8810397982597351, -0.5534512996673584, -2.241178512573242, -0.49557599425315857, -0.49067234992980957, -1.7348878383636475, -1.5662826299667358, -1.3916594982147217, 0.00642769131809473, 0.5262840390205383, 1.1481086015701294, -0.4724818468093872, 0.6519076228141785, 0.3389102518558502, 1.2563797235488892, 1.4981539249420166, -1.980294108390808, -0.11745171248912811, -0.40893733501434326, -0.7291566729545593, 1.7684643268585205, 0.06117444112896919, 0.09053061157464981, -0.8152259588241577, 0.4571816921234131, 2.830582857131958, -1.1274605989456177, -1.5233643054962158, 0.611809492111206, -0.1724959760904312, -0.16418853402137756, -0.12632787227630615, 0.18354976177215576, 1.0534170866012573, -0.7067756056785583, 0.7353199124336243, -0.15891969203948975, -2.179621696472168, -0.004287797026336193, 0.1817326545715332, 1.2193574905395508, 0.1176038533449173, 0.6986904144287109, -0.43302327394485474, -0.4333672523498535, 1.8656574487686157, -0.7254084944725037, -0.5594418048858643, -0.6395698189735413, -0.7584556937217712, 1.0783603191375732, -1.3727055788040161, -0.5316321849822998, 1.9801853895187378, 0.19117794930934906, 2.194364070892334, 1.6146529912948608, -1.0584022998809814, -2.462451457977295, 0.30668115615844727, 0.4524090588092804, 0.5359140634536743], [0.7907838225364685, 0.3817201256752014, -0.5562113523483276, -0.29288628697395325, -0.8613305687904358, 0.3433395326137543, -0.4962383508682251, 1.3946881294250488, -2.387018918991089, 0.30667173862457275, 1.1693073511123657, 0.6829594969749451, 1.3630613088607788, 0.8112773895263672, 0.2948116958141327, 1.4468035697937012, -1.5538978576660156, -1.1018468141555786, 0.3282790780067444, -0.809581995010376, -1.081329345703125, -0.20088326930999756, 1.0289167165756226, -1.93153715133667, -0.6561463475227356, -1.0588330030441284, -1.034842610359192, 0.8184056878089905, 0.11284704506397247, 0.48728349804878235, -1.818222165107727, 0.8565713167190552, -1.0696724653244019, -0.9658238291740417, 1.004358172416687, -1.8111525774002075, 1.1671046018600464, 0.5980156660079956, -0.11808690428733826, 0.5494990348815918, 0.07663755863904953, -2.152423143386841, -0.3990311324596405, -1.4236085414886475, 0.5033579468727112, -0.32205089926719666, -1.0634220838546753, 0.11274384707212448, 0.7548874616622925, 0.8707060813903809, 0.8667463660240173, -0.7551340460777283, -0.014694533310830593, -0.21721114218235016, 1.501824140548706, 0.6670723557472229, 0.22360730171203613, -0.47720441222190857, 0.48902881145477295, 0.065901979804039, -0.08227365463972092, 0.08298490196466446, 2.284231185913086, 0.7587933540344238], [0.4645632207393646, -0.1455819010734558, -1.1619428396224976, -0.062220826745033264, 0.9075092673301697, -0.5292356014251709, -0.3774171471595764, -1.407531976699829, -0.884110152721405, 0.23646396398544312, -1.6210649013519287, 1.3449410200119019, 0.41255027055740356, 0.807071328163147, 0.9742608070373535, -1.4870928525924683, 0.21790041029453278, -1.420741081237793, 0.3720877468585968, 1.38596510887146, -0.5613527894020081, -0.10189536958932877, 1.3964226245880127, 0.8020164370536804, -2.219611883163452, -0.2029878944158554, -1.9768227338790894, 0.9231155514717102, -1.2712124586105347, -0.14638768136501312, 0.958864152431488, 1.0383933782577515, 1.4149596691131592, -2.6917614936828613, 0.053993694484233856, 1.6153823137283325, 0.2228400558233261, -1.290359616279602, -0.07262729108333588, 0.7950978875160217, -0.15626585483551025, -0.15244068205356598, 1.0195249319076538, 1.1999467611312866, 0.15331529080867767, 0.7775668501853943, 0.7853338718414307, -0.8620997071266174, 0.20164719223976135, -0.5714019536972046, 0.01903652772307396, -0.8620527386665344, -0.7659308314323425, 0.25946521759033203, -0.13351629674434662, -1.067565679550171, 1.005530834197998, 0.44276976585388184, 0.4785958230495453, -0.2603277266025543, -0.20910775661468506, 1.0235520601272583, 0.6086777448654175, 0.9781576991081238], [1.247278094291687, 1.1774412393569946, 0.37142786383628845, 0.3490941524505615, -0.7395159006118774, 1.4283676147460938, 0.06685587018728256, 0.019986791536211967, -0.8336385488510132, 1.4222731590270996, -1.472741723060608, -0.6483193635940552, -1.647976040840149, 0.5844672322273254, 0.41862109303474426, -1.4889241456985474, -0.26556146144866943, 0.5469363927841187, 0.8632583022117615, 0.6536400318145752, -0.7681937217712402, 0.03624983876943588, 0.7772354483604431, 0.692782461643219, 1.8527883291244507, 0.3080218434333801, 0.35533711314201355, -1.9005157947540283, -2.5346388816833496, -1.3565973043441772, 0.40541955828666687, -1.0900224447250366, 1.7859781980514526, -1.0920214653015137, 0.9124972820281982, -0.07559232413768768, -0.11774168163537979, -0.42632055282592773, -0.09530231356620789, -0.2796371579170227, -1.6372092962265015, -0.07310914248228073, -0.7416453957557678, -0.7656430006027222, -1.7113556861877441, -0.31230926513671875, -0.8880488276481628, 0.6571779251098633, -1.8863487243652344, -0.037466902285814285, 0.34739795327186584, -1.6084907054901123, -1.1440297365188599, -0.2512291967868805, -1.7950325012207031, 0.08446543663740158, -0.23582248389720917, 0.9463046193122864, 0.3536410331726074, -1.4725492000579834, 0.7867456674575806, 0.8371542096138, 0.23256435990333557, 0.0748942643404007], [0.5920512080192566, 0.4285503923892975, -0.49396079778671265, -0.02852596715092659, 0.5030889511108398, 0.7209482192993164, 1.5777757167816162, 0.3438509702682495, 0.9619605541229248, 2.899982213973999, -1.5353026390075684, -1.7940832376480103, -1.9360634088516235, -1.6895432472229004, 0.2862067222595215, 0.5088698267936707, -0.9462260007858276, -0.532900869846344, 0.8120892643928528, 1.8536567687988281, 0.8216487169265747, -2.6594114303588867, -2.691311836242676, 0.96623295545578, -1.5356472730636597, 0.3285403847694397, 0.13282829523086548, -0.28623658418655396, 0.9956168532371521, -0.7673290967941284, -0.751060426235199, -0.6425979733467102, -0.5292396545410156, -0.23715819418430328, -1.4483840465545654, -0.26007938385009766, 0.5761959552764893, 0.006334671750664711, -0.10965410619974136, 0.7967095375061035, -0.7512342929840088, -1.1484744548797607, 1.3663417100906372, 1.5394755601882935, -1.8728631734848022, 1.0237860679626465, 0.2759009897708893, -0.11052578687667847, -1.4604086875915527, 0.1348305195569992, -0.35694214701652527, 0.4708530008792877, -1.2734837532043457, 1.3735589981079102, 1.8866994380950928, -0.15959812700748444, -0.5755258202552795, -0.15762504935264587, -0.011130391620099545, 0.5919040441513062, 0.4361748695373535, 1.1535817384719849, -1.224056601524353, 0.6683579087257385]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [[-0.47655895352363586, -1.5269395112991333, 0.48444995284080505, -0.28723931312561035, 1.953824520111084, -0.12658248841762543, 0.3656005561351776, 0.11978012323379517, -0.597260594367981, -0.11673180013895035, 1.5554627180099487, 1.463010549545288, 0.22754301130771637, 0.27945396304130554, -0.29836127161979675, 0.8596126437187195, 1.5039281845092773, 0.062132079154253006, 0.4946252107620239, 0.9761080145835876, 0.7157065868377686, -2.1799094676971436, -1.0283559560775757, -0.46653488278388977, -0.031402137130498886, -0.04510844126343727, 0.010608803480863571, -0.46771666407585144, -0.0017813119338825345, -0.4233194887638092, 0.40484076738357544, 0.2985326945781708, -0.07265157252550125, -0.0029857000336050987, 0.5827152729034424, 0.1575865000486374, -0.7609650492668152, 0.4478142261505127, 0.621916651725769, 0.4068906903266907, 0.6470238566398621, -0.2952705919742584, -0.3508903384208679, -1.294747233390808, 0.08645560592412949, -0.053122472018003464, -0.4734545946121216, 0.16084188222885132, -0.5833116173744202, 0.6471711993217468, -3.3190221786499023, -3.743549346923828, -1.180515170097351, 0.13549470901489258, -0.052315812557935715, -0.22768796980381012, 0.8740769028663635, 0.43251833319664, 0.9275130033493042, 2.3018157482147217, 0.7022392749786377, 1.0430842638015747, -1.308550477027893, -0.15753598511219025], [-1.0724108219146729, -0.7403552532196045, -0.07536399364471436, -0.29013752937316895, 1.083272099494934, 0.22844403982162476, -0.9120057821273804, -0.5906181335449219, 0.21276429295539856, 0.3476378917694092, 1.6274373531341553, 0.8584137558937073, -0.8149503469467163, 0.1479109674692154, -0.006257127039134502, 0.3516573905944824, -0.4607812464237213, 0.4224098324775696, -0.10605418682098389, 0.16408562660217285, -0.9383981227874756, 1.9831631183624268, -1.3126684427261353, -0.42625394463539124, 0.22817282378673553, -0.14474841952323914, 0.24410030245780945, -0.35483306646347046, -0.0641833022236824, 0.030197782441973686, -0.4328344166278839, -0.762163519859314, -0.478910893201828, -0.08138818293809891, -0.02178274467587471, -1.1514623165130615, 4.276310920715332, -0.07700508832931519, -2.378408193588257, -1.2245147228240967, 1.7239636182785034, 0.43257930874824524, -0.603055477142334, 2.5183887481689453, 0.027610547840595245, -0.1330806165933609, -0.2134275585412979, -0.6111202239990234, 0.2955826222896576, 0.44105806946754456, -0.22262270748615265, -1.259426236152649, 0.177156001329422, 0.2270871251821518, 0.12878896296024323, -0.370132178068161, -1.1774559020996094, 0.22204962372779846, -0.7824226021766663, 1.396100640296936, -0.0724535658955574, 0.4366787374019623, -0.16573934257030487, 0.2964041829109192], [-0.7886101603507996, 0.9213676452636719, -0.04853696748614311, 0.20100514590740204, -0.3012356460094452, -0.1187240332365036, 0.6439875960350037, -0.4576277732849121, 0.06921303272247314, 1.5845500230789185, 1.0280506610870361, 0.10671830177307129, -0.01858391799032688, -0.030384937301278114, -0.3478812873363495, 0.14100860059261322, -1.0642142295837402, 1.1826093196868896, 0.4585452675819397, 0.7139286398887634, -1.0291967391967773, 1.3601125478744507, -0.4391763508319855, 1.3933157920837402, 0.11170245707035065, 0.3583035171031952, -0.3747917115688324, 0.099807970225811, -0.056114573031663895, -6.78310971125029e-05, -0.030092088505625725, -0.7365797162055969, -0.3931361436843872, -0.9478830695152283, -0.3808082342147827, -0.6800327897071838, -0.13715754449367523, -0.12736554443836212, 1.8365583419799805, -1.7120635509490967, 1.5793298482894897, 0.6990125179290771, 0.2563495934009552, 1.1031609773635864, -0.3632894456386566, -0.13622097671031952, -0.30580466985702515, 0.18944187462329865, -0.21664777398109436, 0.22998562455177307, -0.289408415555954, -0.16385729610919952, 0.11390484869480133, -0.03946695476770401, -0.021843839436769485, 0.3168763816356659, -2.427302837371826, -0.19202758371829987, -0.2112177163362503, -1.6605103015899658, 0.34764987230300903, 1.9254348278045654, -2.2097830772399902, -0.5142855644226074], [0.6145383715629578, -0.8121278882026672, 0.020470840856432915, 0.7300581932067871, 0.6236492991447449, -0.05822966620326042, 0.5247749090194702, -0.5607689619064331, 0.4504368305206299, -0.32022133469581604, -1.1037768125534058, -0.8245804309844971, -0.39850109815597534, 0.45924171805381775, -0.03992704674601555, -0.035074878484010696, 1.0493037700653076, -0.32614538073539734, -0.6194388270378113, -0.5403119325637817, -0.983302891254425, -0.8471939563751221, 2.5360002517700195, 0.4499417841434479, -0.3687262535095215, -0.21303978562355042, -2.5893571376800537, 0.527466893196106, 0.26249024271965027, 0.9748102426528931, -0.3650405704975128, 0.5525188446044922, -0.7142935991287231, -0.2529648244380951, -0.1578245311975479, 1.9965635538101196, -2.6999425888061523, 1.0126941204071045, 2.094143867492676, -0.5777811408042908, 2.3880252838134766, 2.024064302444458, 0.27331915497779846, -0.7010868787765503, -0.4941413998603821, -0.16834555566310883, -0.14161944389343262, -0.5183893442153931, -0.04860220104455948, -0.772193968296051, 0.6181265115737915, -0.5684778094291687, -0.46935781836509705, 0.46691930294036865, -0.2258968949317932, -0.613017737865448, 1.1872295141220093, -1.277974247932434, 0.24454334378242493, 0.6350945830345154, -0.13609619438648224, -0.7032732963562012, 0.12311392277479172, -0.5924951434135437], [-1.1640645265579224, 0.7295028567314148, 0.12650862336158752, 0.7409931421279907, 1.6532999277114868, -0.4229504466056824, 0.39178919792175293, -0.24619227647781372, -0.23330090939998627, 0.3249589204788208, 0.04868803173303604, -1.140074610710144, 0.9127968549728394, 0.9088299870491028, 0.18772834539413452, -0.20286491513252258, -0.20108826458454132, -0.5669727921485901, -0.4654524028301239, 0.37626907229423523, -0.5973106026649475, -0.5726694464683533, 0.24592697620391846, -0.39045512676239014, 0.3891232907772064, -0.09565071761608124, -0.9819464087486267, 0.13185714185237885, 0.1587935835123062, -0.16431985795497894, 0.032889239490032196, -0.2709094285964966, -0.3876594603061676, 0.23804457485675812, 0.30034011602401733, -0.5878225564956665, -0.702226996421814, 0.30183058977127075, 1.3406258821487427, -0.6825804710388184, 0.9803369641304016, 1.3208332061767578, -0.17893047630786896, -0.6485479474067688, 0.052479688078165054, 0.22846673429012299, -0.2534670829772949, -0.18415123224258423, -0.22203031182289124, -0.6148276925086975, -0.4956374764442444, 0.3981594741344452, -0.20940952003002167, 0.1277003139257431, 0.040830884128808975, 0.6278231143951416, 1.6751155853271484, -0.03233357518911362, -0.3806230127811432, -1.4919344186782837, 0.3571488559246063, -1.059857726097107, 0.6317183971405029, -0.13314703106880188], [0.5993414521217346, 1.3682034015655518, -0.19230008125305176, -0.011357526294887066, 0.8910966515541077, -0.35526034235954285, 0.08275450021028519, 0.32945138216018677, -0.28479939699172974, 0.4641273021697998, -0.905373215675354, -0.07152708619832993, 0.027349676936864853, -0.14720815420150757, -0.1444285660982132, 0.07292751222848892, 0.26225247979164124, 1.1008985042572021, -0.32229456305503845, 0.2327229231595993, 1.30161714553833, 0.028891049325466156, -1.0621442794799805, 0.7572798728942871, -0.3954342305660248, 0.07539356499910355, 0.3358486592769623, 0.00869949720799923, -0.0394439622759819, 0.4080406725406647, 0.6823002099990845, -1.1604856252670288, 1.1772959232330322, -0.4544636607170105, -0.7935242652893066, 1.078521490097046, -0.7456192374229431, -0.15882180631160736, -0.45964643359184265, -0.49339863657951355, 0.8507897853851318, -0.864620566368103, -0.03681899234652519, -0.601060688495636, 0.20886044204235077, -0.34259629249572754, -0.05204254016280174, -0.29993852972984314, -0.3771876096725464, -0.9553912281990051, -0.10922849923372269, 1.060470461845398, 0.03587048873305321, -0.5203686356544495, 0.09828918427228928, -0.12324347347021103, -0.4684430658817291, -0.2533092796802521, 0.07649274170398712, -0.8789755702018738, -1.028644323348999, 1.4622228145599365, 0.41961586475372314, -0.3882256746292114], [0.9252810478210449, 1.8119300603866577, 0.03606608510017395, 0.969629168510437, -1.106373906135559, -0.11449869722127914, -0.45420020818710327, 0.3700859844684601, 0.022275008261203766, 0.32263919711112976, -1.150473952293396, 1.8653208017349243, -0.674345850944519, 0.49710410833358765, 0.009502793662250042, -0.4981500506401062, -1.5383833646774292, -0.1819407045841217, -0.007406078744679689, 0.9982370138168335, -0.30254679918289185, 0.7794548869132996, -0.07810288667678833, 0.487231969833374, 0.014996219426393509, -0.0034686082508414984, 0.6682594418525696, -0.04710177704691887, 0.19063301384449005, -0.05075410008430481, 1.698292851448059, 0.7090491056442261, 0.4841412305831909, 0.12952212989330292, 0.17596730589866638, 0.6831034421920776, 0.025960907340049744, 0.3572872579097748, -0.6137275695800781, 0.15878568589687347, -1.2570202350616455, -3.373231887817383, -0.03766490891575813, -0.6002007722854614, 0.12699180841445923, 0.1572781801223755, 0.007790536154061556, -0.26812228560447693, -0.27807220816612244, -1.2797446250915527, -1.3788527250289917, -1.4699798822402954, 0.35797119140625, -0.09522782266139984, -0.19538868963718414, -0.2704838812351227, 1.1868611574172974, -0.2948901355266571, -0.0766633078455925, -0.9798053503036499, 0.5640155673027039, -0.0752323716878891, 1.3766894340515137, 0.5837011337280273], [0.38779258728027344, -1.7381864786148071, 0.15583021938800812, 1.2776774168014526, 1.8068803548812866, -0.46768805384635925, 0.9124690294265747, -0.01625829190015793, 0.2278798371553421, -1.2465453147888184, -1.9212636947631836, -3.0806875228881836, -0.5395663976669312, 0.24770775437355042, 0.22668486833572388, 0.3554689586162567, -1.1822220087051392, 0.01250476948916912, 0.27777042984962463, 0.9695915579795837, 0.9519166350364685, -1.1481692790985107, 1.201491355895996, 1.1219745874404907, -0.39461749792099, 0.20391827821731567, -0.8931102752685547, -0.4174816310405731, 0.25441497564315796, 0.04866007715463638, 0.06330800801515579, -0.41186925768852234, 0.6169397830963135, 1.0908640623092651, 0.06497218459844589, 3.2988088130950928, 0.22137778997421265, -0.06396204978227615, 0.15201213955879211, 1.109570026397705, 0.04957776516675949, -1.1605647802352905, 0.43044033646583557, -2.117224931716919, 0.2941820025444031, 0.03356022387742996, -0.06772533804178238, 0.20172680914402008, 0.4181749224662781, 0.4521743953227997, -1.0062320232391357, -3.281799554824829, -0.019409138709306717, 0.1534518599510193, -0.3964751362800598, 0.4839908480644226, -0.4848468005657196, 0.29330432415008545, 0.4687645435333252, -1.2066195011138916, 0.41644585132598877, 1.2442675828933716, 0.9260251522064209, 0.13795214891433716], [-0.8976770043373108, -0.7758002281188965, -0.17815066874027252, 0.6372446417808533, 0.6291279792785645, 0.11885489523410797, -0.5272343158721924, -0.7504006624221802, -0.47957590222358704, -0.5930567383766174, 3.6125545501708984, 0.5695928931236267, -1.0229066610336304, -0.5294105410575867, -0.0632534772157669, -0.766874372959137, -0.6559680700302124, -0.5093343257904053, -0.24663583934307098, 0.9909347295761108, -1.1082793474197388, 0.975430965423584, -0.7264593839645386, 1.3915804624557495, -0.9279398322105408, -0.32297754287719727, -0.8366830945014954, -0.07545402646064758, -0.3070216178894043, -0.890348494052887, 0.14596594870090485, -0.6240588426589966, 1.0145362615585327, 1.4678696393966675, -0.6297348737716675, 1.7116281986236572, 0.37101855874061584, 0.21215783059597015, -0.15121814608573914, -2.5124566555023193, 0.9720211625099182, 3.7680881023406982, 0.014162403531372547, 0.16507871448993683, 0.36897438764572144, -0.509529173374176, -0.13573628664016724, 0.19821490347385406, -0.7774826288223267, -0.27191290259361267, 1.3368804454803467, -1.3949174880981445, -0.2379375398159027, 0.19016702473163605, 0.16670961678028107, -0.4601137042045593, -0.45752617716789246, -0.2609972655773163, -0.4851827323436737, -0.03614316135644913, 0.1803632527589798, 0.8789545893669128, -0.06956039369106293, 0.11783801764249802], [-0.35056230425834656, -0.31649598479270935, -0.10318075865507126, 1.2027275562286377, 1.3805376291275024, -0.2912213206291199, -0.49912557005882263, -1.5385068655014038, -0.20061995089054108, 0.021851196885108948, 0.013927658088505268, 1.1557952165603638, 0.41930708289146423, -0.12010035663843155, 0.023096587508916855, 0.20339320600032806, 1.5072410106658936, 0.11105762422084808, -0.6771034002304077, 0.005389038473367691, -0.4713863134384155, -1.9583256244659424, -0.32349321246147156, -0.3490971028804779, 0.11900472640991211, 0.34611478447914124, -0.5062041282653809, 0.16759784519672394, 0.04386281222105026, -0.38378387689590454, -0.9955593347549438, -0.10716716200113297, -0.6634065508842468, 0.42746853828430176, 0.7502360343933105, 0.12498795986175537, -0.5383490324020386, -0.3216624855995178, 0.658596396446228, -0.23648357391357422, 0.1756923794746399, -1.0725899934768677, -0.1501258760690689, -0.4985819160938263, -0.3299795687198639, 0.1460842490196228, 0.3525807559490204, -0.5909216403961182, 0.9905446767807007, -0.9005577564239502, -0.3488883376121521, 1.4226831197738647, 0.43534430861473083, 0.32657843828201294, 0.31309226155281067, -0.10002856701612473, 2.1952614784240723, -0.4146110713481903, -0.5345317721366882, -0.9924048781394958, 0.0790451243519783, 0.6823477745056152, 1.6694248914718628, -0.5858161449432373], [-0.1297311633825302, -0.5789477825164795, 0.28632092475891113, -0.1066410094499588, -1.1352359056472778, -0.4496045708656311, -0.0014879096997901797, -1.0286978483200073, -0.148697629570961, -0.045593250542879105, 0.487068235874176, -1.8481297492980957, 0.3932458162307739, -0.5273455381393433, 0.26867008209228516, -0.5740426182746887, -0.34760287404060364, 0.8504872918128967, -0.319783478975296, 1.8246525526046753, -0.7318831086158752, 0.7731338739395142, 0.31671249866485596, -0.5593095421791077, 0.6250516176223755, 0.36920052766799927, -0.24637143313884735, 0.679328203201294, -0.20659999549388885, 1.5344183444976807, -0.802861213684082, -1.56415855884552, 0.23272255063056946, 0.5000654458999634, 0.15089690685272217, 1.9528547525405884, -0.22107726335525513, 0.07255882769823074, -0.6371932029724121, 0.66199791431427, -1.370957612991333, 0.6761229038238525, 0.1854512244462967, 1.8074660301208496, -0.06976186484098434, 0.04953066632151604, 0.04352414608001709, -0.0966629683971405, 0.27038466930389404, 0.5023954510688782, -2.112957715988159, -2.5255532264709473, 0.19269528985023499, 0.544472873210907, -0.20328561961650848, -0.21232448518276215, 1.5895174741744995, 0.7789641618728638, 0.1870642602443695, -0.00943743996322155, -0.25982093811035156, -0.4569382965564728, 0.21439626812934875, 0.5073240995407104], [-0.7920671105384827, 1.7066737413406372, -0.028622658923268318, 0.668888509273529, -0.5896145105361938, 0.14243587851524353, 0.788199245929718, 1.1455752849578857, 0.06795407086610794, -0.8323663473129272, -2.7784860134124756, 0.5570470690727234, 0.601158618927002, 0.1809338927268982, -0.006537716370075941, 1.1304808855056763, 1.7140100002288818, 0.11411600559949875, 0.3505665063858032, 0.18915674090385437, 0.13908110558986664, -2.1434004306793213, 0.38388967514038086, 2.287008285522461, 0.35193467140197754, 0.03394727781414986, -0.4429485499858856, 0.08194940537214279, -0.21284057199954987, -0.2303471863269806, 0.2502579689025879, 0.4014628529548645, -0.6492826342582703, -0.1935560554265976, 0.49069032073020935, -0.01680625043809414, -1.5506929159164429, -0.2904425263404846, 1.6400036811828613, -0.5254154205322266, -3.4813625812530518, -0.1904921680688858, 0.03449733927845955, -0.6958313584327698, 0.6349775791168213, 0.19616983830928802, 0.27367717027664185, -0.0007712796796113253, -0.5319235324859619, -0.7899879217147827, -1.0411324501037598, -0.96272873878479, -0.1791861206293106, -0.804356038570404, -0.28814637660980225, 0.5691766142845154, -1.1205508708953857, -0.5103099346160889, 0.3582838773727417, 4.489219665527344, -0.4987320601940155, 0.5067232251167297, -1.2243543863296509, 1.1231456995010376], [-0.19994838535785675, -2.075709104537964, 0.11020736396312714, -0.3133656084537506, 0.5381501317024231, 0.17168211936950684, 0.2437695562839508, -0.2780607044696808, -0.20011858642101288, 0.73253333568573, 0.8386123776435852, 0.5475342273712158, 0.8218482136726379, -0.22220996022224426, -0.010136282071471214, 0.49886325001716614, 0.7024919390678406, -0.44089797139167786, 0.13350337743759155, -0.25414276123046875, -1.2631311416625977, -1.0234888792037964, -0.3085252344608307, 0.054506491869688034, -0.41015711426734924, -0.09931357204914093, -0.11024729162454605, -0.15831689536571503, -0.27783456444740295, 0.042879097163677216, 0.8432103991508484, 0.558107852935791, -0.18271039426326752, -0.3161199986934662, -0.034832071512937546, 0.12887176871299744, 2.244657039642334, -0.773920476436615, 0.5531997680664062, -0.21557310223579407, 0.2734605073928833, -0.6657602190971375, 0.4724065065383911, -0.8470466136932373, -0.4657467007637024, -0.09472906589508057, 0.4916563630104065, -0.5663347244262695, 0.22045005857944489, 0.9282692670822144, -0.2957456111907959, -0.5821928381919861, -0.18258297443389893, -0.3454725444316864, -0.502133846282959, 0.5853779911994934, 0.865667998790741, 0.37237510085105896, -0.41773438453674316, 0.7251346707344055, -0.4426482617855072, -1.089465856552124, 0.22769205272197723, 0.7392348647117615], [-1.9806162118911743, 0.2743368148803711, -0.19265185296535492, 0.03902501240372658, 0.10526932030916214, -0.3784007132053375, 0.22857128083705902, -2.2105932235717773, -0.40989112854003906, 1.010473608970642, 2.307271718978882, -0.7499261498451233, 1.0079010725021362, -0.6455875635147095, -0.20000237226486206, -0.6171930432319641, -2.153593063354492, 0.8935178518295288, -1.0496569871902466, 2.0785422325134277, -0.5128751993179321, 0.28146788477897644, 0.7002272605895996, 0.4626173675060272, 0.06513911485671997, 0.02311224676668644, -0.5382967591285706, -0.37983348965644836, -0.07195937633514404, -1.338348150253296, 0.9048535823822021, -0.4447716176509857, 0.7001799941062927, 1.7412710189819336, -0.20803245902061462, 1.0727314949035645, -0.5302226543426514, -0.17967000603675842, -0.4391983151435852, -0.38367462158203125, 1.760179042816162, -0.44843214750289917, -0.41174963116645813, 3.3261823654174805, -0.04094203561544418, -0.3763693571090698, -0.0824684351682663, -0.03566644713282585, 0.33328697085380554, 0.3065491318702698, 0.6935384273529053, 1.3550214767456055, -0.7544494271278381, -0.023139620199799538, -0.30877870321273804, -0.6534779667854309, -1.1719160079956055, -0.9560350179672241, -0.23819167912006378, -0.23590150475502014, 0.10636282712221146, 1.3414849042892456, -1.618620753288269, -0.14798051118850708], [0.4332086443901062, -1.4891879558563232, 0.17645788192749023, 0.2317345142364502, -1.2787508964538574, -0.042604025453329086, -0.23473015427589417, 0.5346589684486389, -0.34169289469718933, -0.4498389661312103, -0.5549824237823486, -2.0248210430145264, -0.07791727036237717, 0.5043789744377136, 0.18952031433582306, -0.03274872899055481, 1.4596437215805054, 0.005116880871355534, -0.3420540392398834, -1.092063546180725, 0.032058075070381165, 0.04457804933190346, -1.4285900592803955, -0.408903568983078, 0.4143328368663788, 0.007286080624908209, 0.6780414581298828, 0.4318944215774536, -0.08852853626012802, -0.2077154666185379, 1.5344206094741821, 0.09031303972005844, -0.027434390038251877, -1.2074283361434937, 0.19652388989925385, -0.6226695775985718, 1.9419231414794922, 0.025159360840916634, 0.21567963063716888, 1.3013089895248413, -2.873539924621582, -0.7686474323272705, 0.35972458124160767, 0.18484900891780853, 0.33843910694122314, 0.12697532773017883, 0.5135069489479065, -0.10939809679985046, 0.2019824981689453, 1.2988548278808594, -0.3136141896247864, -2.9304425716400146, 0.6065425872802734, 0.19640414416790009, 0.6241827607154846, -0.02853972837328911, -0.4875541031360626, 1.6390072107315063, 0.5210055112838745, -0.5905362963676453, 0.0675198882818222, 1.2251864671707153, 0.5816646814346313, 1.120849370956421], [0.30133867263793945, -0.7264809012413025, -0.02973281964659691, -0.8492281436920166, -0.6814467906951904, 0.40337586402893066, 0.02290763147175312, 0.8013665676116943, -0.17588254809379578, -0.4132837951183319, 0.20584188401699066, 2.568166732788086, -0.4869544506072998, -0.5642498135566711, -0.054813481867313385, -0.24575504660606384, -1.5933223962783813, 0.1973726600408554, 0.12704728543758392, -0.7411209940910339, -0.024896662682294846, -0.33036723732948303, 0.961009681224823, -0.5918346643447876, 0.08849931508302689, 0.02567618526518345, -0.10174602270126343, 0.4021033048629761, -0.5209839940071106, -0.34659790992736816, -0.6056748628616333, 0.8821405172348022, 0.2651641368865967, 1.052356481552124, 0.4211682081222534, -0.8579772114753723, 1.703796625137329, -0.5554463267326355, -0.4362648129463196, -0.20864330232143402, 0.5754272937774658, -0.33849862217903137, 0.3748966157436371, 2.616990327835083, 0.20102831721305847, -0.12357348948717117, -0.6849247217178345, -0.42684248089790344, -0.3968473970890045, -1.0537561178207397, 0.1778716892004013, -0.035874564200639725, -0.037950437515974045, 0.2789173722267151, 0.05779191851615906, -0.2781965434551239, -1.259259581565857, -0.07412201911211014, -0.5313590168952942, 0.6507841348648071, 0.3341576159000397, 0.2209773063659668, 0.5869274735450745, -0.12119249999523163], [0.9685606360435486, -0.6539995670318604, 0.004279185086488724, 0.01438648160547018, 1.6735572814941406, -0.08772455155849457, 0.6390964388847351, -0.5883913636207581, 0.0700232982635498, -0.10018251836299896, 1.185555100440979, 2.095399856567383, 0.6861793398857117, 0.07589049637317657, 0.15141721069812775, -0.12494863569736481, -0.9093402028083801, -0.271756112575531, 0.010200338438153267, -0.18276652693748474, 0.8238364458084106, -0.5769519805908203, -0.7452301383018494, 0.00013614629278890789, 0.3694295883178711, 0.09471777081489563, 0.3990812599658966, 0.14124269783496857, 0.11829276382923126, -0.5397586822509766, 0.16408240795135498, -0.48477432131767273, -0.0014744906220585108, -0.14326727390289307, -0.2648765742778778, -0.305698961019516, -1.6436100006103516, 0.07952239364385605, -0.5644701719284058, 0.04324999451637268, 2.521271228790283, 0.03656969964504242, -0.417392760515213, -3.485597610473633, -0.305251806974411, 0.2828575074672699, 0.32323822379112244, 0.15712425112724304, 0.31236252188682556, -0.9049616456031799, -0.9478719234466553, 1.3254263401031494, -0.9719254374504089, -0.2896097004413605, 0.03502901643514633, 0.18194963037967682, 1.6372522115707397, 0.8515589833259583, -0.5258642435073853, 0.2297222912311554, 0.05285344645380974, -0.34723207354545593, -1.0989160537719727, -0.27150705456733704], [-0.7422612309455872, -0.4150814116001129, 0.1849106103181839, 0.9064463973045349, -0.49319666624069214, 0.4028175473213196, -0.39938709139823914, -0.061630841344594955, -0.034882813692092896, -0.3583672344684601, -0.3708856701850891, 0.952921450138092, 0.40836426615715027, 0.21707482635974884, 0.05678563192486763, -0.3208519220352173, 0.8164511919021606, 0.17866846919059753, 0.16616006195545197, 0.1952904462814331, -1.4475477933883667, -2.878519058227539, -0.08074977993965149, 1.9882972240447998, 0.017462465912103653, -0.005274574272334576, 0.5741899609565735, 0.6131815314292908, 0.5583702921867371, -0.007949265651404858, 0.5295937061309814, 0.4568651616573334, -0.6035411357879639, -0.2535117268562317, 0.13228298723697662, -0.7356137633323669, -2.2242863178253174, 0.6095666289329529, -1.1441898345947266, -0.5228267312049866, -0.5624495148658752, 2.1838345527648926, 0.5226373076438904, -0.7552695870399475, 0.2081843763589859, 0.26919662952423096, 0.37461501359939575, 0.06562858819961548, -0.15846803784370422, 0.20374266803264618, 0.12064079195261002, 1.326389193534851, 0.2617049217224121, 0.1670529693365097, 0.09910698980093002, -0.33914270997047424, -0.398095041513443, -0.7769731283187866, 0.2167377769947052, -0.7474097013473511, 0.5272412896156311, -1.225724220275879, 1.049865961074829, 0.5018068552017212], [0.01728489063680172, -1.164160132408142, -0.11089514195919037, 0.32516661286354065, 0.34115463495254517, 0.05504251644015312, -0.5596866607666016, -0.11645645648241043, 0.16531293094158173, -0.1337476670742035, -2.6459670066833496, -4.081705570220947, -0.8440989851951599, 0.5433841943740845, -0.08867637813091278, 0.09595806151628494, 1.7997063398361206, -0.5322610139846802, -0.1557619422674179, -0.4039769470691681, -0.97039395570755, -0.9515714049339294, 1.0354896783828735, 1.557650089263916, 0.00880312267690897, 0.03479275107383728, -0.9179381132125854, 0.18450874090194702, -0.3640167713165283, -0.22587810456752777, -0.2904050052165985, -0.36380624771118164, 0.5193732976913452, -1.0114820003509521, -0.0995069295167923, 0.6972636580467224, 0.13077260553836823, -0.27259981632232666, -0.12331204116344452, 0.3815128803253174, 2.081763505935669, 3.476227045059204, -0.12744222581386566, 1.5472471714019775, -0.307837575674057, 0.021373575553297997, 0.702028214931488, -0.2224917858839035, -0.9195124506950378, -0.9444664716720581, 1.428948998451233, 1.8225293159484863, -0.1671321988105774, 0.04313597455620766, -0.1224534809589386, -0.13570933043956757, 1.176583170890808, -0.9438742995262146, -0.7233783602714539, 0.9210485816001892, -0.09358099848031998, -0.487493634223938, 1.5794501304626465, -1.0698357820510864], [0.13338789343833923, -0.010879579000175, 0.13861796259880066, 1.199273943901062, 0.879056990146637, 0.43937718868255615, 0.3462275266647339, 0.08667425811290741, 0.5514700412750244, -0.6495228409767151, -1.5620226860046387, 0.20315814018249512, 0.04626955837011337, -0.08442177623510361, 0.07098354399204254, 0.06640608608722687, 0.34083569049835205, 0.5876318216323853, 1.011492133140564, 0.12954683601856232, 0.503940224647522, -1.8550244569778442, -1.6488245725631714, 0.6670647263526917, 0.2234923541545868, 0.2913192808628082, 1.1718722581863403, 0.2411959171295166, -0.5409414768218994, 0.18883253633975983, 0.31945016980171204, 0.21250344812870026, 0.06098379194736481, -0.7046962380409241, 0.009855417534708977, 0.1711512953042984, 5.430142879486084, -0.22343406081199646, -0.7372232675552368, -1.0966942310333252, -0.18198221921920776, -1.746800422668457, 0.20347413420677185, -2.1554574966430664, -0.042994651943445206, 0.22385737299919128, 0.08021393418312073, 0.44962286949157715, 0.7012270092964172, -0.7437540292739868, -0.7444167733192444, 2.0100162029266357, -0.3849087655544281, 0.12363461405038834, 0.4037182927131653, 0.151838019490242, -3.5921759605407715, 0.3299010396003723, 0.7077869772911072, -0.7296366691589355, 0.13067224621772766, -0.7404792308807373, -1.2573176622390747, 0.19543232023715973], [-0.6568403244018555, 0.5377275347709656, -0.262299507856369, 0.6295201778411865, 0.586836576461792, -0.35135698318481445, 0.002025197958573699, 0.7501146793365479, -0.21402190625667572, -0.5593723654747009, 3.0968174934387207, -0.45038241147994995, 0.1523207426071167, -0.009043196216225624, -0.16173501312732697, 0.29232820868492126, 0.9084227681159973, 0.9606668949127197, -0.6543563604354858, -0.9876458644866943, 1.112058162689209, 0.3026176691055298, 0.8058198690414429, 1.2368263006210327, -0.22655637562274933, -0.18558676540851593, 0.839169442653656, 0.08274298161268234, -0.5347779989242554, 0.7008686661720276, -0.04109752178192139, 0.8926651477813721, -1.2329564094543457, -0.09596425294876099, -0.042872242629528046, -0.1475275605916977, 0.7611321806907654, -0.29753562808036804, -2.3328652381896973, 0.1528497040271759, -3.1661367416381836, -2.305640459060669, -0.43038034439086914, 5.750879764556885, -0.09870154410600662, -0.09756489843130112, -0.4503251314163208, -0.532364547252655, 0.04657597839832306, -0.12326763570308685, -1.197080373764038, 0.5990946888923645, -0.2703621983528137, 0.5200504064559937, -0.02585519850254059, -0.15006570518016815, 1.3945927619934082, -0.31884366273880005, 0.2245442271232605, -0.6334617137908936, -0.3357650935649872, -2.194380044937134, -1.1922169923782349, -0.37206146121025085], [0.15372274816036224, 0.7040545344352722, -0.06581336259841919, 0.7129087448120117, 0.960943877696991, 0.25134724378585815, 1.101522445678711, -0.6485653519630432, -0.3165985941886902, -0.9667723774909973, 2.6471986770629883, 0.18696166574954987, -0.1547737866640091, -0.5478867888450623, -0.09641864150762558, -0.6173551082611084, 0.6457818746566772, -1.162787914276123, -0.16070125997066498, -0.4455827474594116, -0.17663271725177765, 2.4638941287994385, -1.999513864517212, 1.5517088174819946, 0.13660134375095367, -0.058413803577423096, 0.37998610734939575, -0.5087218880653381, -0.044233329594135284, 0.1402333527803421, 0.6150659918785095, 0.39072662591934204, -0.8973408341407776, 0.5974193215370178, -0.9774144887924194, 0.0014849710278213024, -0.9825294613838196, -0.22409147024154663, -0.7219820618629456, -1.605249285697937, 0.05771707743406296, -0.04158240184187889, -0.07639549672603607, -4.363277912139893, -0.21642641723155975, -0.4209029972553253, -0.20613959431648254, -0.1820700466632843, 0.48370251059532166, 0.15978731215000153, -1.0779627561569214, -1.8444340229034424, 0.06515192240476608, 0.22206567227840424, 0.2652483880519867, 0.3119640052318573, 1.0895373821258545, 0.422417551279068, 0.4748794138431549, 0.6146073341369629, 0.44968172907829285, 0.9006855487823486, 1.0403871536254883, 1.6091750860214233], [0.36365219950675964, -0.6705847978591919, 0.4696866571903229, 0.8699358105659485, 0.1242373064160347, -0.05841116979718208, -0.654833197593689, -0.418579638004303, 0.3208130896091461, 1.0663599967956543, 1.4736645221710205, -2.650912284851074, -0.6870483756065369, -0.4761333763599396, 0.3188652992248535, 0.23101432621479034, -2.3141634464263916, -0.15406008064746857, 1.1755379438400269, -1.0955921411514282, -0.7438211441040039, 4.9059648513793945, -3.5040690898895264, -0.32511237263679504, 0.4629989266395569, 0.2590502202510834, 0.6647989153862, -0.05036746338009834, 0.06649485975503922, -0.06226302683353424, -0.1746937781572342, -0.1981731653213501, -0.6693974137306213, -1.1025543212890625, 0.48595812916755676, 0.9229754209518433, 1.68613600730896, -0.6007153987884521, 1.4426249265670776, -2.1100761890411377, -0.6677945256233215, -3.6315317153930664, 0.4766845703125, -1.7972012758255005, 0.02239370159804821, -0.7471495270729065, 0.08962581306695938, -0.13399215042591095, 0.21378839015960693, 0.39974725246429443, -0.5197223424911499, 0.29197216033935547, -0.06170990690588951, 1.3412222862243652, 0.4085591733455658, 0.0040490892715752125, 1.5424872636795044, -0.24101798236370087, 0.3790920376777649, 1.3097071647644043, 0.8426440358161926, -0.13117553293704987, 3.052966833114624, -1.032852292060852], [0.25397413969039917, 0.16495315730571747, -0.15139809250831604, -0.9128028750419617, -0.8432043194770813, -0.3452240824699402, -2.088087797164917, 0.7084567546844482, -0.329975426197052, -0.3022976219654083, -1.6126675605773926, -1.2669951915740967, -0.8160276412963867, -0.3685116469860077, 0.03844136744737625, -0.24615299701690674, 1.1694971323013306, -0.06845538318157196, -0.22309653460979462, 0.7134379148483276, 0.24550293385982513, 5.010773658752441, -0.7981125116348267, 0.7867724895477295, -0.28106826543807983, -0.2551756799221039, -0.6479550004005432, 0.5059364438056946, -0.2823061943054199, 0.3106948137283325, -0.17792600393295288, -0.3511503338813782, -0.39002469182014465, -0.4584498703479767, -0.12050899118185043, 0.3203483521938324, 1.1133626699447632, 0.2173861265182495, -3.469182252883911, 0.693757951259613, 0.8589450120925903, 3.7528655529022217, 0.09089520573616028, 3.1452794075012207, 0.12232775241136551, -0.13874094188213348, -0.8893560767173767, -0.25858500599861145, -0.24536435306072235, 0.2960972785949707, -0.7417871356010437, -0.1041780561208725, -0.28471535444259644, -0.5464470982551575, 0.4504041075706482, 0.7799744009971619, -0.9958791136741638, -0.5513625741004944, -0.10345768183469772, 0.5408883690834045, -0.5322873592376709, -1.0937269926071167, 1.007094383239746, -0.005422693211585283], [-1.1112967729568481, 1.6581811904907227, -0.2933237850666046, 1.0615986585617065, 0.4307083785533905, -0.31690627336502075, -0.6226944327354431, 2.017327070236206, -0.5528514385223389, -1.211366057395935, 0.9719259142875671, -0.2340562492609024, -0.8909449577331543, -0.30997613072395325, -0.09820520877838135, -1.2264329195022583, -3.971011161804199, 0.154595747590065, -0.1108635738492012, 0.03487636521458626, -0.3181382715702057, 1.8390986919403076, 0.43962857127189636, -0.5460984110832214, 0.3210268020629883, 0.017993126064538956, -0.7531450390815735, -0.5961796641349792, 0.53348708152771, 0.009685128927230835, -0.09758228063583374, -0.05431341007351875, -0.34571030735969543, 0.6123475432395935, 0.06530455499887466, -0.21814538538455963, 2.5912468433380127, 0.004008694551885128, 1.6933289766311646, -0.6895639300346375, -0.7300677299499512, -0.923560380935669, -1.1700440645217896, 3.5581724643707275, 0.379178524017334, 0.17623020708560944, -0.6961159110069275, -0.6477815508842468, -0.9866947531700134, -0.6791138648986816, -0.38360854983329773, -2.952847719192505, 0.34143346548080444, 0.05610070005059242, -0.3339031934738159, -0.7587088942527771, 0.07692237198352814, -0.1153697744011879, 0.16805532574653625, 3.3087565898895264, -0.22323401272296906, 1.9345433712005615, 0.8823015689849854, -0.16820812225341797], [0.24319498240947723, -0.8090072274208069, 0.08945061266422272, 0.8628419041633606, -0.7861295342445374, -0.030126214027404785, 0.5044237971305847, 0.4917922914028168, 0.5293105840682983, -0.4859325885772705, -2.0885000228881836, -3.823807954788208, -1.086707353591919, 0.04431124031543732, 0.1031891480088234, 0.4040662944316864, -0.49242734909057617, 0.030507704243063927, -0.0036878674291074276, -0.45976680517196655, 0.9906541109085083, -0.6714783310890198, 0.19139449298381805, -0.041642170399427414, 0.12737201154232025, 0.2751571536064148, 2.243652105331421, 0.08293379098176956, -0.07925590872764587, -0.3002021908760071, 0.8255982398986816, -0.18942992389202118, 0.639503002166748, 0.6247347593307495, 0.48055174946784973, -1.278241753578186, 2.782233238220215, -1.026968002319336, 0.41604772210121155, -0.9355571866035461, 0.688850998878479, 3.983255386352539, 0.6951881647109985, -3.9410407543182373, 0.12761972844600677, -0.021508565172553062, -0.4187150001525879, 0.2475060075521469, 0.5205237865447998, 0.3902016580104828, -0.07510488480329514, -2.0008137226104736, -0.9037590622901917, 0.19121988117694855, 0.19161823391914368, 0.6286818385124207, 0.2662959396839142, -0.20056091248989105, -0.37749534845352173, -0.3285641372203827, -0.31019747257232666, 2.097611427307129, -0.08589097112417221, 0.2410254329442978], [-0.5164608955383301, 1.664064884185791, -0.019244670867919922, -0.20469745993614197, 0.08666358143091202, -0.3361015021800995, -0.11147864162921906, -0.6686198711395264, 0.7362992167472839, 0.9983372688293457, -0.06268181651830673, 1.8202706575393677, 0.5770466327667236, 0.3576596677303314, -0.002857896499335766, -0.19191259145736694, -3.0962798595428467, 0.9818480014801025, -0.4852043092250824, 1.0325770378112793, -0.028697699308395386, 1.9439159631729126, -2.2401607036590576, 0.9691407084465027, -0.1320168823003769, -0.06404415518045425, 0.09389788657426834, -0.4620344042778015, 0.5042036771774292, -0.15336397290229797, -1.0281479358673096, 0.2869628071784973, -0.7088765501976013, -2.2256062030792236, -0.5920225381851196, 0.5815102458000183, 1.7103216648101807, 0.5844510793685913, -0.23852017521858215, 0.18867036700248718, 1.399204134941101, 0.5013978481292725, 0.43686893582344055, 2.4843921661376953, -0.24514764547348022, 0.28988462686538696, 0.2139519900083542, 0.06860977411270142, -0.262773334980011, -0.4224553406238556, -0.535870373249054, -1.0538103580474854, 0.11932564526796341, 0.2017793506383896, -0.5519843697547913, -0.02010631002485752, -1.9517054557800293, -0.8780983090400696, -0.4896275997161865, 1.8842265605926514, -0.8156877756118774, 0.2710736393928528, -1.420379877090454, -0.7718783020973206], [-1.1388185024261475, 0.8666266202926636, -0.13537117838859558, -0.3735864460468292, 0.36371853947639465, -0.3617284893989563, -0.9558101892471313, 0.23444974422454834, 0.05628820136189461, 0.050377413630485535, 0.6947565078735352, -0.4847146272659302, -2.2796318531036377, -0.16163623332977295, -0.36626166105270386, -0.7701968550682068, -0.7429944276809692, -0.5139727592468262, -0.06767158210277557, 0.6233126521110535, 0.6559140086174011, 0.06140905246138573, -3.156893491744995, -0.07530833035707474, -0.5197823643684387, -0.40774866938591003, 0.29617634415626526, -0.31056204438209534, 0.33573630452156067, 0.5562511682510376, -0.29162925481796265, -0.4578910171985626, -0.21253882348537445, -0.2732819616794586, -0.5238800644874573, 0.5213583707809448, 3.7558069229125977, -0.003651344683021307, 3.419541835784912, -0.07657234370708466, -0.39028388261795044, 2.3137683868408203, -1.2967544794082642, -2.493328332901001, 0.6375588774681091, -0.5091781616210938, -0.22674424946308136, 0.32372578978538513, -0.957844614982605, 0.3765544593334198, 0.6165199875831604, 3.884429454803467, 0.4816552698612213, 0.01675095036625862, -0.07216399163007736, 0.26599934697151184, -0.9587387442588806, 1.4212125539779663, -0.20996655523777008, -1.576235294342041, -0.43022528290748596, -0.5491747856140137, 1.9744250774383545, -0.4715511202812195], [-2.1247150897979736, 0.021696647629141808, -0.5912994146347046, -0.5668666362762451, 0.23131512105464935, -0.6608306169509888, -0.10724864155054092, 0.03656983748078346, -0.17956717312335968, 0.2325732707977295, -0.6618527173995972, -2.4578545093536377, -0.46702781319618225, -0.5568307042121887, -0.5808191895484924, 0.09522617608308792, 1.5225414037704468, -0.2960573732852936, 0.43697255849838257, 0.6017054319381714, 1.2683759927749634, 0.8065842390060425, -0.2427348792552948, -0.32710719108581543, 0.4560178220272064, -0.1540399044752121, -0.4111088216304779, -0.8902758955955505, -0.5368460416793823, 0.017445825040340424, -1.4423069953918457, 0.07199475169181824, 0.029783371835947037, 0.623879611492157, -0.48959940671920776, 1.9131771326065063, 0.7437552213668823, 0.11796756088733673, 0.4264983534812927, -2.14389705657959, -0.4212758243083954, 3.4121172428131104, -0.3855104446411133, -0.3653383255004883, -0.36445674300193787, -0.08634768426418304, -0.6507317423820496, 0.06920867413282394, -0.48436105251312256, -0.22481147944927216, 0.6783518195152283, 2.49910044670105, -0.7197554707527161, -0.18493907153606415, -0.5320388078689575, -0.1199994683265686, 1.6306719779968262, 0.5818580985069275, 0.11963697522878647, -1.4705966711044312, 0.23552151024341583, 3.298130989074707, -0.045526888221502304, -0.23410235345363617], [-0.708154022693634, 1.195181965827942, 0.07447195053100586, 0.5562658905982971, -0.032804202288389206, -0.03167441859841347, 0.6194924712181091, -0.5278735160827637, -0.1709004044532776, -1.9008893966674805, -0.739294707775116, 1.0679296255111694, 0.24198338389396667, 0.07787589728832245, 0.1964271068572998, -0.09263760596513748, 0.30868929624557495, -0.42011794447898865, -0.2791670560836792, -1.066926121711731, -0.16964998841285706, 0.19333474338054657, 1.041865587234497, 0.4002895653247833, 0.2541043758392334, 0.24708957970142365, 1.0818986892700195, 0.3080802261829376, -0.150160551071167, 0.10561579465866089, -0.09936109185218811, -0.12999822199344635, 0.14077141880989075, -0.3252752423286438, 0.5543550252914429, 0.6466162204742432, 1.2319726943969727, -0.14413520693778992, -1.1018917560577393, 1.2637708187103271, -1.369329810142517, -4.36282205581665, -0.08991847932338715, 0.2890591621398926, 0.34457769989967346, -0.04620400443673134, -0.3356259763240814, -0.18130163848400116, 0.899074375629425, -0.3824853301048279, 0.6228411197662354, -1.6423044204711914, -0.15455138683319092, -0.1795123815536499, 0.5635237693786621, -0.7295844554901123, -1.220400333404541, -0.3292108476161957, -0.6770814657211304, 3.9583542346954346, 0.25030332803726196, 2.666525363922119, -1.4712969064712524, -0.13980010151863098], [1.0880578756332397, -0.7757490873336792, 0.024190867319703102, -0.6689568161964417, -1.2408545017242432, -0.3507756292819977, -0.06306679546833038, 0.7447965741157532, -0.13080352544784546, 1.4831780195236206, 0.2547453045845032, -1.8782888650894165, 0.7346218824386597, 0.1415296196937561, -0.09456665068864822, 0.7046487331390381, -0.6288310885429382, 0.33972591161727905, -0.1077544242143631, -0.3410886824131012, -0.21160443127155304, 2.7483015060424805, 0.021202173084020615, -0.28760528564453125, -0.037843767553567886, 0.0699910894036293, -0.5905694365501404, 0.3343036472797394, 1.0916857719421387, -0.82215815782547, 1.4775099754333496, -0.3891802728176117, -0.02415004000067711, 0.3861977458000183, 0.23210519552230835, -0.7993914484977722, -5.681307315826416, 0.23303398489952087, -1.0781770944595337, -0.6827690601348877, -1.0224062204360962, 0.711776614189148, -0.04100731387734413, -1.4993882179260254, 0.12088941782712936, 0.11734030395746231, 0.5161573886871338, -0.4637690782546997, 0.29400941729545593, 1.8082525730133057, 1.2573707103729248, -1.0180760622024536, 0.5083668231964111, 0.38346558809280396, -0.3165968954563141, -0.17183451354503632, 4.2486186027526855, -0.21529294550418854, 0.2703561782836914, -1.9314886331558228, -0.41421690583229065, 0.7057508826255798, 1.6521391868591309, -0.7247503995895386], [-0.07844121009111404, -0.12828156352043152, -0.05888523533940315, 0.8024762272834778, 0.013250404968857765, -0.17824167013168335, -0.21515251696109772, -1.060395359992981, -0.16202306747436523, -0.5754580497741699, 0.5857666730880737, -1.8548181056976318, -1.2297252416610718, -0.857680082321167, -0.09891943633556366, -0.021345965564250946, 2.150235176086426, 0.03873839229345322, 0.4281913638114929, 0.1637999713420868, 0.2966557741165161, -1.3663508892059326, -1.1456650495529175, -0.412894606590271, -0.3091014325618744, -0.2937452793121338, -0.746257483959198, 0.16169188916683197, -0.10520025342702866, -0.7206135392189026, -0.4110924005508423, 0.38787412643432617, -0.11661902070045471, 0.8183878660202026, 0.23065927624702454, 0.018290456384420395, 0.5149994492530823, -0.11457773298025131, -0.07926049828529358, -0.09697821736335754, 2.102971315383911, 3.287278652191162, -0.3055350184440613, 1.523912787437439, -0.3728342354297638, -0.04919334501028061, 0.07314010709524155, -0.03137267753481865, -0.658997654914856, -0.31520339846611023, -0.33140942454338074, 0.44971317052841187, -0.5243806838989258, -0.18755733966827393, 0.005345006473362446, -0.23655666410923004, 0.45249849557876587, -0.006290891207754612, -0.5605764985084534, 1.8466870784759521, -0.3088105320930481, -0.40767183899879456, 0.2866135239601135, 0.09493708610534668], [-1.244996190071106, -0.051357001066207886, -0.2065819501876831, -0.24407686293125153, -0.019707903265953064, -0.6751341819763184, -0.5633169412612915, 0.29353269934654236, 0.01044713519513607, -0.674710750579834, 0.6972078084945679, 3.0079500675201416, -0.43230336904525757, -0.35509973764419556, -0.20224322378635406, 0.32927900552749634, 1.6622521877288818, 0.33458414673805237, -0.07461323589086533, -1.1274232864379883, 0.03523358330130577, 2.081146001815796, -0.8298839330673218, 0.5565364956855774, -0.4135855436325073, -0.05546517297625542, 0.5146855711936951, 0.03873811289668083, -0.3633626103401184, -0.49952900409698486, 1.431540846824646, -0.573196530342102, -0.2956783175468445, 0.8136591911315918, -0.13847941160202026, 0.493699312210083, -2.541025400161743, -0.5648254156112671, -0.8422332406044006, -0.181000217795372, 1.717825174331665, 0.6236510276794434, 0.46566110849380493, -2.1729576587677, -0.3814549148082733, -0.028661243617534637, -0.13350003957748413, -0.044592563062906265, 0.4453457295894623, -0.20096123218536377, -0.37864744663238525, 0.7770177721977234, 0.14030058681964874, -0.76161789894104, -0.6675150394439697, -0.3470631539821625, 0.2934117913246155, 0.5280497074127197, 0.3581518530845642, 1.1995855569839478, -0.32688042521476746, -0.6792418956756592, 0.7493227124214172, -0.3058919608592987], [1.401710867881775, -0.6138702630996704, -0.05802922323346138, 0.4857172966003418, -0.954394519329071, -0.396461546421051, 1.3124990463256836, -0.4448851943016052, 0.6956690549850464, -0.8660692572593689, 2.8340277671813965, -2.29013729095459, -0.2433575987815857, -0.16720642149448395, -0.025389088317751884, 0.09973328560590744, 0.16182337701320648, -0.7603266835212708, 0.5389045476913452, -0.6919293403625488, 0.41293811798095703, -0.43140459060668945, 2.8387644290924072, 1.1932265758514404, -0.34532058238983154, 0.11606446653604507, 0.6455850601196289, -0.13566827774047852, -0.17271271347999573, -0.21558621525764465, 1.0815333127975464, -1.109083890914917, -0.8581092953681946, 0.8917562961578369, 0.15454614162445068, 1.7032595872879028, -3.0917742252349854, 0.5928624868392944, -0.08208721876144409, 0.702749490737915, 0.5021471381187439, 0.1799340546131134, -0.867908239364624, -0.9441587924957275, 0.6381785869598389, -0.0033970153890550137, 0.36422795057296753, -0.29599249362945557, 0.35049745440483093, 0.8150104284286499, 2.271653175354004, -0.7644462585449219, -0.7351332306861877, 0.2957751750946045, 0.2321162223815918, -0.10485000908374786, -4.188932418823242, -1.6003293991088867, -0.5400164723396301, -1.460292935371399, 0.06873023509979248, 0.47718697786331177, 1.4940199851989746, -0.09358752518892288], [-0.28422456979751587, -1.0025850534439087, 0.18912065029144287, 0.5408479571342468, 0.33545809984207153, 0.03394750878214836, -1.0712262392044067, -0.5315046906471252, 0.8615201711654663, 0.9316920042037964, 0.9804173111915588, -4.488574981689453, -1.3613148927688599, 0.12017716467380524, -0.04639741778373718, 0.25003570318222046, -0.5985848903656006, 0.510894238948822, 0.016198959201574326, 0.6096369624137878, -0.08719515800476074, 3.9178860187530518, 1.5960761308670044, -0.2066456526517868, 0.4181784391403198, 0.020496156066656113, -0.009863725863397121, 0.3526655435562134, 0.42437389492988586, 0.014683450572192669, 1.1699968576431274, 0.36568212509155273, 0.9326267838478088, -0.5322897434234619, -0.025133254006505013, 0.1608000248670578, -3.7223949432373047, 0.08391880989074707, 0.8427748084068298, -0.8493162989616394, 0.27691295742988586, -0.9599159955978394, -0.6108569502830505, 0.6921209096908569, -0.17626427114009857, 0.38953453302383423, 0.0836707353591919, 0.20507216453552246, -0.3026643395423889, 0.018685901537537575, 0.7087962627410889, 3.751682996749878, 1.0063064098358154, -0.1795639991760254, -0.05990622937679291, -0.19772760570049286, -0.7350584864616394, -1.53061842918396, 0.295207142829895, -2.543613910675049, 0.5155259966850281, -2.5730173587799072, 1.106702446937561, -0.04386529326438904], [-1.0466442108154297, -0.24323603510856628, 0.10765160620212555, -0.6484026312828064, -1.624819278717041, 0.2384004145860672, 0.13759952783584595, 1.1264820098876953, -0.9919875860214233, -0.25327134132385254, -0.7336100339889526, -0.43765702843666077, -1.324475884437561, -0.13073045015335083, 0.05037999898195267, -0.41736841201782227, 1.4111329317092896, 0.4045111835002899, 0.2882945239543915, -0.5419407486915588, 0.9015741348266602, -1.3622636795043945, 1.5603559017181396, 0.2952253520488739, -0.27703195810317993, -0.22692303359508514, -0.43699395656585693, 0.1911071389913559, 0.25136318802833557, 0.32129642367362976, -0.5447865724563599, -0.15261858701705933, 0.09335047006607056, 0.3013160228729248, 0.29662710428237915, -0.4916148781776428, 1.9906387329101562, -0.5884485840797424, 1.1053186655044556, 0.1430359184741974, 0.2151283323764801, -0.100000761449337, -0.4544687867164612, -0.2636454701423645, -0.4893787205219269, 0.29673027992248535, -0.13017259538173676, 0.10235552489757538, 0.157245934009552, 1.1281962394714355, -0.42982199788093567, 0.006117616780102253, -0.014358972199261189, 0.14616426825523376, 0.09371169656515121, 1.105673909187317, -1.2252616882324219, 0.03377880901098251, -0.7933170795440674, 1.4234827756881714, -0.4156632125377655, 0.6884155869483948, 0.24678252637386322, -0.0685306191444397], [-0.026143332943320274, 0.12471053004264832, 0.21120372414588928, -0.12786950170993805, 0.6405792236328125, 0.2937467694282532, -0.23203147947788239, -0.43019238114356995, 0.39496323466300964, -0.45860493183135986, 0.17910651862621307, 1.6149834394454956, 1.243443250656128, -0.09375140070915222, 0.2531875669956207, -0.32948896288871765, 0.5932464003562927, 0.9299854040145874, -0.30306604504585266, -0.9416885375976562, -1.136960506439209, 1.2070765495300293, 1.020384430885315, -1.2439836263656616, 0.5525645017623901, 0.051895178854465485, -0.9554986953735352, 0.09350806474685669, 0.5098930597305298, -0.5922378897666931, 0.2644496262073517, 0.15344668924808502, -0.16920922696590424, 1.8254035711288452, 0.0668654590845108, -0.664327085018158, -0.39661693572998047, 0.16067710518836975, -0.3435259461402893, 0.5008333921432495, 0.634354829788208, -1.5585516691207886, 0.2832411527633667, -3.137573003768921, 0.6284336447715759, 0.38562434911727905, 0.37343934178352356, 0.638137936592102, 0.48381105065345764, -0.3018706440925598, -2.388023614883423, 1.1744768619537354, 0.38703471422195435, 0.32788559794425964, -0.3711621165275574, 0.04472576826810837, 0.7940796613693237, -1.2277603149414062, 0.3566721975803375, -0.9349594116210938, 0.331047922372818, -0.14954884350299835, -1.1434820890426636, -0.07099031656980515], [-0.31158170104026794, -0.8690996170043945, 0.012380637228488922, -0.9760677814483643, -1.947609782218933, -0.4337291121482849, -0.6405040621757507, -0.5832636952400208, 1.4340450763702393, 0.39225277304649353, -1.1654967069625854, 3.9249727725982666, 0.9760041832923889, -0.1624361127614975, 0.024317707866430283, 0.763630747795105, -1.9423322677612305, 0.19326430559158325, 0.3789121210575104, 1.357102394104004, 0.7726121544837952, -2.141549587249756, -0.978213906288147, -0.5016486048698425, 0.02913038432598114, -0.020994771271944046, 0.6468772292137146, 0.3489587903022766, 0.7451845407485962, 0.39876118302345276, 0.8788132071495056, 0.5287405252456665, 0.9700869917869568, 0.6751537919044495, -0.12374715507030487, -0.3318398892879486, 1.8654077053070068, 0.23999246954917908, 0.5286268591880798, -2.9987213611602783, 0.32171761989593506, -3.1228158473968506, -0.11774428188800812, 1.2330083847045898, -0.15293702483177185, 0.07949073612689972, 0.39543047547340393, 0.24022790789604187, -0.8624864220619202, 0.4035411477088928, 0.8437432646751404, 0.951403796672821, -0.04635346308350563, 0.0973324254155159, -1.2402833700180054, 0.07665927708148956, -0.5073274374008179, 0.3555114269256592, -0.013499059714376926, -0.6489573121070862, 0.6394825577735901, -1.5261497497558594, 0.9799240231513977, -0.3353114426136017], [-0.8131105899810791, -0.19354680180549622, 0.17046621441841125, 0.1085929423570633, -1.7327686548233032, -0.05895192176103592, -0.26706254482269287, -0.03759775310754776, 0.21239401400089264, 1.3656803369522095, -2.9596309661865234, 1.8255894184112549, 0.38993027806282043, 0.44076138734817505, 0.18939810991287231, -0.13394035398960114, 0.37701186537742615, 0.7705482840538025, 0.08299471437931061, 1.7991572618484497, -1.4009497165679932, 1.179474115371704, 1.124003529548645, -1.12589693069458, -0.08767978847026825, 0.15535373985767365, -0.7838185429573059, -0.4529297649860382, 0.18750061094760895, 1.0115669965744019, 0.21890772879123688, -0.7230395674705505, 1.8682126998901367, -0.6387983560562134, 0.32996952533721924, 0.30267414450645447, 0.2643137574195862, -0.3271903395652771, 1.3850765228271484, 1.0686308145523071, -0.8004470467567444, -0.8245658874511719, -0.11307066679000854, -1.816879153251648, 0.1718083769083023, 0.29483187198638916, 0.46308445930480957, 0.18327271938323975, -0.5570858716964722, -0.3973299562931061, 0.021731805056333542, -1.3464233875274658, -0.7840809226036072, 0.7950519323348999, -0.02710258774459362, -0.20273476839065552, 1.8423255681991577, -0.5868158340454102, 0.0623687319457531, 1.360106348991394, 0.19360224902629852, -3.2140166759490967, -0.4145241379737854, 0.6055963635444641], [0.1962912678718567, 2.0485575199127197, 0.007718326058238745, 0.30591514706611633, -0.6655458807945251, 0.25345662236213684, -0.8960006237030029, -0.30135223269462585, 0.76129150390625, -0.18444031476974487, 0.08386062830686569, -1.5957833528518677, 1.283010482788086, -0.8532329201698303, -0.01123516634106636, 0.06096069887280464, -2.060978412628174, -0.4069082736968994, -0.006006099749356508, 0.9127879738807678, -1.0147939920425415, 0.3448927700519562, -1.8867653608322144, -0.925198495388031, 0.06700892746448517, -0.2710196077823639, -0.310578316450119, -0.4057183861732483, -0.16314585506916046, -0.7058926820755005, -1.0194300413131714, 0.20037560164928436, 0.933545708656311, -0.8637977838516235, -0.4410555064678192, 0.675798773765564, 1.1328338384628296, -0.19206061959266663, -1.0941060781478882, -0.06749065965414047, -0.893615186214447, -0.22035135328769684, 0.06127310171723366, 1.7755467891693115, 0.2740356922149658, 0.16483211517333984, 0.21923695504665375, -0.15135014057159424, -0.13044226169586182, -0.03455597162246704, -0.5785861015319824, 3.6144394874572754, -0.16886752843856812, 0.1364971101284027, 0.2838457524776459, 0.1631087213754654, 3.156193494796753, -0.3635099232196808, 0.1321965456008911, 1.7335104942321777, -0.1918250322341919, -1.5973820686340332, -0.17875930666923523, -0.1312403827905655], [1.2524012327194214, 0.1691364049911499, -0.08613669872283936, -0.3020969033241272, -0.04270998015999794, 0.10674512386322021, -0.0024498512502759695, -0.5791932344436646, -0.4304003119468689, 0.744026243686676, 2.7766661643981934, -1.8654309511184692, 0.0759885385632515, -0.7628069519996643, 0.04742735996842384, 0.0003346169542055577, -0.24259506165981293, -0.2101992517709732, 0.6510084867477417, -0.6762866377830505, -0.23273016512393951, -1.8589444160461426, -0.5813748836517334, 0.9357024431228638, 0.0006928202346898615, -0.303227961063385, 0.4650920629501343, -0.38076186180114746, -0.15685409307479858, -0.4894426763057709, 0.5511336922645569, 1.8510547876358032, 0.08494098484516144, -1.5312132835388184, 0.27985265851020813, -1.17578125, 1.7776261568069458, -0.23515838384628296, -0.1008741557598114, -0.7299273610115051, -1.6967966556549072, 0.052400168031454086, -0.10113220661878586, 1.2593718767166138, -0.6402010321617126, -0.0810341164469719, 0.19022509455680847, 0.531888484954834, 1.054682731628418, 1.312203049659729, -1.2184826135635376, -1.29432213306427, 0.4117336571216583, 0.0345916710793972, 0.5591990947723389, 0.2004241943359375, -0.9475061297416687, -0.767147421836853, -0.0021555668208748102, 0.648632824420929, 0.4866541028022766, 2.814683198928833, -0.49457162618637085, -1.106574296951294], [-1.2281928062438965, 0.49140238761901855, 0.3355497419834137, 1.0563206672668457, -0.42334261536598206, -0.2714725732803345, -0.12307945638895035, -1.2257368564605713, 0.22989042103290558, -0.6819459199905396, -1.0030244588851929, 1.7791529893875122, -0.8517083525657654, -0.21026761829853058, -0.2474176585674286, 0.08592557162046432, 0.36228665709495544, 0.0528225377202034, 0.06668844074010849, -0.977949857711792, 1.9969042539596558, 0.5398324728012085, 0.08230508863925934, 2.0230607986450195, -0.31198179721832275, -0.08958832919597626, -0.207936093211174, -0.5027108788490295, -0.18879568576812744, 0.18062350153923035, 0.28154638409614563, 0.03977745771408081, -0.5738004446029663, 1.4773507118225098, -0.6286117434501648, 1.3608276844024658, 0.5050684809684753, 0.2334805130958557, -1.6085386276245117, 0.5653663873672485, -1.58103346824646, 1.2970225811004639, -0.23134352266788483, -0.14826419949531555, 0.18928657472133636, 0.28592151403427124, -0.05430076643824577, 0.47825881838798523, 1.0431023836135864, 0.39431339502334595, 0.42006540298461914, -0.8278490900993347, -0.4046538770198822, 0.2022702842950821, -1.1797852516174316, 0.15976914763450623, 0.5604151487350464, -0.9127814173698425, -0.2463711053133011, -1.6993098258972168, -0.024968966841697693, -0.494852751493454, 0.3457844853401184, 0.03922305628657341], [0.18882226943969727, 0.21660761535167694, -0.18653036653995514, -0.9368364810943604, -0.8394117951393127, 0.33022695779800415, -0.4561038613319397, -0.724553108215332, 1.0191420316696167, 0.340875506401062, -1.5030044317245483, 2.563608407974243, 0.38975343108177185, 0.5359446406364441, -0.042018115520477295, 0.45641854405403137, -0.6712625622749329, 0.18895120918750763, 0.23505495488643646, 0.16499333083629608, -0.458346426486969, 0.5686691999435425, -0.43690794706344604, -1.2954269647598267, 0.5591157674789429, -0.0966734066605568, 0.4877799451351166, 0.07355201244354248, 0.17038559913635254, -0.30080053210258484, -0.48624956607818604, 0.422802597284317, 0.17229802906513214, 0.9491378664970398, 0.012337783351540565, -0.7277154922485352, -1.5847760438919067, -0.216550812125206, -0.6987609267234802, 0.8735101819038391, -4.823712348937988, 2.2839837074279785, -0.1335231512784958, -0.31172776222229004, 0.3558694124221802, 0.16659142076969147, 0.17025789618492126, -0.18581850826740265, 0.050026122480630875, 0.03729429841041565, -1.5952160358428955, 0.8764406442642212, -0.42473819851875305, -0.02500089444220066, -0.2630251944065094, -0.22880156338214874, 0.9798648357391357, -0.2541539669036865, 1.1825065612792969, -0.8547355532646179, 0.3170746862888336, 2.680617570877075, 0.5268705487251282, 0.21499665081501007], [-0.16837158799171448, 1.1423238515853882, -0.046459853649139404, -0.5082758069038391, 0.810005784034729, 0.48078596591949463, -0.01598367840051651, 0.15214377641677856, -0.43761783838272095, -0.3359989523887634, 2.359947443008423, 2.4670259952545166, -1.2041786909103394, 0.3828163743019104, 0.11697491258382797, 0.03885060176253319, -2.2321062088012695, 0.29360029101371765, 0.02010388672351837, -0.9424034953117371, -0.48869696259498596, -0.7155579328536987, 0.9752676486968994, 0.6600525379180908, -0.4298560321331024, -0.22807230055332184, -0.5409683585166931, -0.3942396938800812, -0.2959977686405182, 0.31644248962402344, 1.6682312488555908, 0.649519145488739, -0.9554427862167358, -1.1106511354446411, -0.09078775346279144, 0.5404677391052246, 2.7026991844177246, -0.40190497040748596, 0.12781153619289398, 0.6460087895393372, 2.0284955501556396, -1.473006248474121, 0.2672070860862732, -3.2341442108154297, 0.21319104731082916, -0.20581305027008057, -0.3544541001319885, 0.035917412489652634, 0.11060665547847748, -0.04197032004594803, -0.5191245079040527, -0.06770098209381104, 0.08049238473176956, -0.8542320728302002, -0.2016613632440567, -0.548511266708374, 1.6957863569259644, -1.7864776849746704, -0.6136698722839355, 1.4743521213531494, -0.4668966829776764, 0.6008259057998657, -1.036887526512146, -0.10983199626207352], [-0.32997408509254456, 0.4247225224971771, 0.28078678250312805, 0.9903422594070435, -0.31975406408309937, 0.4466058313846588, 0.19582931697368622, -0.040145393460989, 0.6894311308860779, 0.9361422061920166, -0.8820765018463135, 1.6975088119506836, 0.23912732303142548, -0.8005769848823547, 0.30516064167022705, -0.2401445358991623, -3.802948474884033, -0.3863872289657593, 0.02566910721361637, 0.26836445927619934, 0.45114484429359436, -0.5782984495162964, -0.7041425108909607, 0.22087830305099487, 0.5798518657684326, 0.08025749772787094, -0.2729364335536957, 0.8597117066383362, 0.1526944935321808, -0.21386411786079407, 0.20830515027046204, -0.09103846549987793, -0.02735316753387451, -1.2322251796722412, 0.06536351144313812, 0.40009355545043945, -0.29278886318206787, -0.2608068585395813, -1.8315891027450562, -0.3337205648422241, 0.36641716957092285, -2.47482967376709, -0.11360714584589005, 0.043517690151929855, 0.1739313155412674, 0.24633875489234924, -0.07101289927959442, 0.13210153579711914, 0.3081680238246918, 0.03909672051668167, 0.2683798372745514, 1.1627119779586792, 0.967234194278717, 0.43891996145248413, 0.017400138080120087, 0.4323108494281769, 0.5461053252220154, 0.32920143008232117, -0.25650754570961, 0.08961039781570435, -0.2479591965675354, 2.145021438598633, -1.1780751943588257, -0.24169544875621796], [-0.3218914270401001, 1.2131683826446533, 0.09664272516965866, -0.4798794388771057, 0.6757681965827942, 0.26320144534111023, 0.7159752249717712, -2.9031434059143066, 0.008376565761864185, -0.5572900772094727, -0.6181049346923828, 0.39548781514167786, -1.8281673192977905, 0.12399092316627502, 0.12951458990573883, 0.3632098138332367, 0.08292368799448013, -0.10088111460208893, 0.07144264131784439, -0.231771782040596, -0.872111439704895, -1.9300609827041626, -0.56280517578125, -0.7107680439949036, -0.24724081158638, -0.05784136429429054, 0.19611918926239014, 0.09677038341760635, -0.4365638792514801, -0.1955750286579132, 1.4233698844909668, 0.05301692336797714, -0.1376529186964035, -1.1937470436096191, -0.46532532572746277, 1.0063523054122925, 1.841848611831665, -0.378807008266449, -0.6731132864952087, 0.6337239742279053, -0.46487370133399963, 2.114943742752075, 0.20504678785800934, 2.9055261611938477, 0.17538340389728546, 0.2695547938346863, -0.092377208173275, 0.2534034848213196, -0.9011819958686829, -0.11978545784950256, 0.39027225971221924, -0.26570644974708557, 0.311655730009079, 0.08105383068323135, -0.16524089872837067, -0.1330937147140503, 0.8013567328453064, 0.7238972187042236, 0.1091570183634758, -0.05083426460623741, 0.422556608915329, -0.9167829155921936, -0.484352171421051, 0.34225940704345703], [0.004737539682537317, 0.7835608124732971, 0.26123151183128357, 1.4414074420928955, 1.1497876644134521, 0.15721409022808075, 0.9727286696434021, -0.31147122383117676, 0.43605783581733704, 1.6457674503326416, 0.5238967537879944, -2.2428009510040283, -1.0077518224716187, 0.26817235350608826, 0.23753507435321808, 0.6856129169464111, -2.3615174293518066, 1.4255706071853638, 0.5465779304504395, -0.44385430216789246, 1.043170690536499, -5.660639762878418, -2.0492618083953857, -0.40445980429649353, 0.6010798811912537, 0.0847691223025322, 2.389559030532837, -0.18967698514461517, 0.0755370631814003, -0.34656161069869995, -0.8471513986587524, 1.5380150079727173, 0.3572097420692444, -0.18244658410549164, -0.6654842495918274, 0.851169764995575, -2.1949195861816406, 0.06127893552184105, -1.044197916984558, -0.3425389528274536, 0.46059778332710266, -1.8582885265350342, 0.8038138151168823, 1.4100258350372314, 0.24292853474617004, 0.364808589220047, -0.04565071314573288, 0.06537845730781555, -0.8844343423843384, -0.10767341405153275, -0.264923632144928, -1.8188483715057373, 0.45106634497642517, 0.5849249958992004, 0.2219996303319931, 0.20709943771362305, 1.86306893825531, 0.9903243184089661, -0.29401761293411255, -0.30071550607681274, 0.49391981959342957, -2.3152592182159424, 2.366220235824585, 0.11672119051218033], [-0.16369499266147614, 0.026650600135326385, -0.34020236134529114, 0.07642143964767456, 0.1247238740324974, -0.01636439375579357, -0.8066772222518921, -1.7037131786346436, 0.09140180796384811, 1.1151106357574463, 2.845432996749878, 3.5796566009521484, 0.7372128963470459, 0.1997213512659073, -0.31788602471351624, 0.0078077237121760845, -0.6056815385818481, -0.20914889872074127, -0.2561235725879669, -0.534030020236969, 0.7200772762298584, -0.7878087759017944, -1.739871859550476, 1.7110711336135864, -0.13630898296833038, -0.05337332934141159, 1.4464061260223389, -0.32298916578292847, -0.34647566080093384, -0.17436367273330688, -0.47385454177856445, 0.3145696818828583, 0.9090407490730286, -0.39316001534461975, -0.20277196168899536, -1.6688721179962158, -4.226956844329834, 0.551487147808075, -0.05715635046362877, 1.065748929977417, 2.348902463912964, -2.561103105545044, -0.2388678640127182, -0.883330762386322, 0.14192378520965576, -0.13091757893562317, 0.16948971152305603, -0.49667587876319885, -0.5601586699485779, 0.545055627822876, 2.757673740386963, -1.7540398836135864, 0.3937445282936096, -0.028172137215733528, 0.13641205430030823, 0.09081754088401794, -2.3345224857330322, -0.7568916082382202, -0.30398520827293396, 0.8211429715156555, 0.356758177280426, 2.0532538890838623, 0.35396894812583923, -0.10553334653377533], [0.43303635716438293, 0.5451179146766663, 0.01063480880111456, 0.6920472979545593, -0.25092023611068726, 0.09801296889781952, 0.09399405866861343, 0.1760985255241394, -0.28034117817878723, -0.30931854248046875, -2.369751453399658, -0.9934021234512329, 1.3559677600860596, 0.06164471060037613, 0.1330988109111786, -0.3429145812988281, -2.8546602725982666, 0.2602120041847229, 0.11484788358211517, 0.7612985372543335, -0.34599897265434265, 0.8414797782897949, -0.2261555939912796, 1.9890445470809937, -0.3613111674785614, 0.3993213176727295, 0.6486480832099915, -0.13133564591407776, 0.2512879967689514, 0.6933258175849915, -0.22445148229599, -1.5595389604568481, -0.15029269456863403, 0.6305282711982727, 0.3361501395702362, 1.0598057508468628, 1.7824218273162842, 0.29840272665023804, 0.7095673084259033, -0.6176719069480896, -2.428884506225586, 2.0018668174743652, -0.2578967213630676, 0.13382622599601746, 0.0747445821762085, 0.07088524103164673, 0.053545381873846054, 0.4178745746612549, -0.2585411071777344, 0.4489685893058777, -0.9773514270782471, 1.4027613401412964, -0.0005692687118425965, 0.5400396585464478, -0.4556102752685547, 0.20582714676856995, -1.0550572872161865, 0.12817735970020294, 0.059573087841272354, -1.403573751449585, 0.25592663884162903, -1.233008861541748, -1.4820853471755981, 0.40063172578811646], [-1.304774522781372, 0.5325222611427307, 0.1399194896221161, 0.5747703313827515, -0.4596770405769348, 0.06492515653371811, 0.21695850789546967, -0.8590229153633118, 0.6221455931663513, 0.4372541010379791, -0.5689854621887207, 3.7309958934783936, 0.6654560565948486, 0.10368366539478302, -0.016530944034457207, -0.24452053010463715, 2.9365687370300293, 1.2350090742111206, 0.24793624877929688, -0.5393087267875671, 1.9590977430343628, -1.2400352954864502, -1.507453203201294, -0.8663838505744934, 0.2464940845966339, 0.19643284380435944, 0.43637391924858093, -0.28584277629852295, -0.5759761333465576, -0.5086469650268555, 0.4361030161380768, 0.9730095267295837, -0.6514467597007751, 0.8080849647521973, -0.19564861059188843, 0.5920432806015015, -1.0163325071334839, 0.4650432765483856, 0.6134676337242126, -0.5995302200317383, -2.1560699939727783, -2.833562135696411, -0.036616433411836624, -1.9976633787155151, 0.028763458132743835, 0.040414512157440186, -0.12874232232570648, -0.048136722296476364, 0.1098974198102951, 0.8221672177314758, 0.07363805174827576, -0.5325130224227905, -0.2060130387544632, 0.494253545999527, -0.13319438695907593, 0.6594886779785156, -2.7332704067230225, -0.31121379137039185, -0.4153077304363251, 0.196157768368721, 0.5011767148971558, 1.7103908061981201, -0.22901882231235504, 0.33079513907432556], [-1.0208772420883179, -0.27166831493377686, -0.07892980426549911, -0.5047553181648254, -0.008415297605097294, 0.2593936324119568, 0.3913816213607788, 1.7124451398849487, -0.07345841079950333, 0.09648971259593964, 0.4687731862068176, 0.6108521223068237, -0.2533990442752838, 0.051576726138591766, -0.03301209211349487, 0.21902842819690704, 1.6376298666000366, 0.290710985660553, 0.23330356180667877, -0.9608601331710815, -0.5779842734336853, -0.5303646922111511, 0.4135523736476898, -0.40566980838775635, -0.4792674779891968, 0.23707447946071625, -1.4351563453674316, 0.23013998568058014, 0.4797019362449646, 1.4341005086898804, -1.6484642028808594, -0.29511135816574097, 0.26695483922958374, -1.0086233615875244, -0.08684801310300827, -1.1526952981948853, 1.5460833311080933, 0.2474571019411087, 0.05417845770716667, -0.06779126077890396, -0.19548703730106354, -0.302467942237854, 0.07746507972478867, -3.1764917373657227, 0.0825306624174118, -0.06580974161624908, 0.4530913829803467, 0.21318548917770386, 0.0997065007686615, -0.28408458828926086, -0.6801705956459045, 1.0162951946258545, 0.01533433422446251, 0.43447041511535645, 0.12243495136499405, -1.059690237045288, 2.6723694801330566, -0.8481104969978333, 0.248672217130661, 0.04157146066427231, 0.2392948716878891, 2.391678810119629, -1.1466410160064697, -0.33662357926368713], [-0.25835707783699036, -0.5663564801216125, -0.08296039700508118, 0.929477870464325, 0.6073428988456726, 0.043300461024045944, -1.8529636859893799, 0.7484631538391113, -0.32480135560035706, -0.30354028940200806, -0.06651799380779266, -3.1341826915740967, -0.5133073329925537, 0.8677825927734375, 0.09978535771369934, 0.26557493209838867, -0.3411600887775421, -0.7378234267234802, -0.3335441052913666, -1.708629846572876, 0.017337104305624962, 1.9681497812271118, 0.9539634585380554, 0.5430542230606079, 0.21522440016269684, 0.009724843315780163, -0.862351655960083, -0.12557946145534515, -0.10717253386974335, -0.37769514322280884, 0.26518794894218445, -0.010854654014110565, -0.005717686377465725, 0.11940862983465195, -0.012258923612535, -0.07999363541603088, -3.1229875087738037, 0.009898260235786438, 0.8644235134124756, -0.6884691119194031, -2.0568525791168213, 1.5374687910079956, 0.04292329400777817, 5.7080183029174805, -0.6483727097511292, -0.10455939918756485, -0.5717863440513611, -0.3744560778141022, 0.35292568802833557, -0.11288516968488693, 1.1120065450668335, 3.2579185962677, 0.652520477771759, 0.859277606010437, 0.19792157411575317, 0.1375579684972763, -1.071891188621521, -0.9869058132171631, 0.5348765254020691, 1.3550316095352173, -0.14993557333946228, 0.5946559906005859, -2.0593247413635254, -1.117008090019226], [-0.8068535327911377, 0.11556224524974823, 0.042402077466249466, 0.0431692898273468, 0.7666382789611816, 0.3383350670337677, 0.8899645805358887, 1.0852837562561035, 0.13892903923988342, -1.660179615020752, 0.5804889798164368, -0.37997522950172424, 0.7272452712059021, -0.227333664894104, -0.19307078421115875, 0.7833824157714844, 0.7529604434967041, -0.2998144030570984, -0.8222382664680481, 0.6045580506324768, 0.0752144604921341, 0.6752502918243408, -1.2703505754470825, 1.5788037776947021, 0.08586441725492477, -0.01980796456336975, -0.9481938481330872, -0.4804289937019348, -0.5059114694595337, 0.28834789991378784, -1.088380217552185, 0.5127432346343994, 0.0950784906744957, 0.4329306483268738, -0.5695707201957703, 2.4005818367004395, 1.3890111446380615, -0.8390173316001892, -0.44450676441192627, -2.8517773151397705, 0.2018829733133316, 1.9953827857971191, 0.2814978361129761, -3.9485747814178467, -0.5292263031005859, -0.1828000545501709, 0.0848250538110733, 0.004919953644275665, -0.4412993788719177, 0.24677802622318268, -0.08830421417951584, 0.944079577922821, 0.38485994935035706, 0.06681349128484726, 0.20457176864147186, 0.5167057514190674, -0.43912273645401, -0.6776267886161804, -0.7699804902076721, -0.10886528342962265, 0.24611327052116394, -1.170632004737854, 1.9366533756256104, 0.24601301550865173], [0.10276123881340027, 0.18807005882263184, 0.04475110396742821, 1.0046242475509644, 1.4712893962860107, -0.6477325558662415, 0.18725305795669556, -1.2796111106872559, 0.1946268081665039, 0.20192666351795197, -0.2575763165950775, 0.2725859582424164, 0.10365687310695648, -0.10850678384304047, -0.12020672112703323, 0.5164715051651001, 0.01796802319586277, 0.339180052280426, 0.14144538342952728, -0.7258484959602356, 2.0324182510375977, -0.6079224348068237, -0.004001038148999214, -0.6679574251174927, -0.0729445368051529, -0.1463789939880371, -0.13426783680915833, -0.13321982324123383, -0.35986822843551636, 0.6868115663528442, -0.11002842336893082, 0.10364428907632828, -0.503685474395752, -0.4555806815624237, 0.08010762929916382, -2.1542370319366455, 1.8565778732299805, -0.26479092240333557, -0.15540951490402222, 1.7169272899627686, 0.3077108561992645, 0.2694120407104492, -0.39536145329475403, -0.7787482738494873, 0.35488587617874146, 0.045938875526189804, 0.07102128118276596, -0.025868095457553864, 1.7961963415145874, -1.4014993906021118, -1.0788087844848633, -0.37106853723526, -0.3143792748451233, -0.04714448004961014, -0.14787395298480988, -0.7588787078857422, 1.0117570161819458, -0.4573061466217041, 0.2265411913394928, -0.05601847544312477, 0.8254902958869934, -1.759819746017456, 0.1750842183828354, 0.15541456639766693], [-0.13710322976112366, -0.4883318245410919, -0.008083243854343891, -0.035297691822052, -1.092545747756958, -0.017218368127942085, 0.9847234487533569, 0.2685973346233368, 0.5119221210479736, -0.07091020047664642, 0.36228951811790466, -1.3065602779388428, 1.622499942779541, -0.46444958448410034, 0.11390213668346405, -0.024438058957457542, 2.0315704345703125, -0.46511197090148926, -0.7406495809555054, -2.200512409210205, -0.3863808810710907, 1.441059947013855, 0.42636439204216003, -0.3611574172973633, -0.13066336512565613, -0.10970938950777054, -0.4760591983795166, 0.06280436366796494, -0.3930184245109558, -0.7129404544830322, -1.2228435277938843, 0.6240431666374207, 0.16864429414272308, -1.2803610563278198, 0.08758391439914703, -0.4708116948604584, 1.4182955026626587, 0.2915387451648712, -0.18698133528232574, -0.24732986092567444, -1.3740450143814087, -0.8995695114135742, 0.09411630034446716, 1.3895875215530396, 0.0782255306839943, 0.023697054013609886, 0.4241797924041748, 0.19771134853363037, -0.32999446988105774, -0.8837659358978271, 0.07859157770872116, -1.5870496034622192, -0.049238625913858414, -0.03270742669701576, -0.1760321706533432, 0.24048417806625366, 3.0088436603546143, 1.2390999794006348, -0.07615168392658234, -0.5327510833740234, 0.08143121749162674, -0.06490755081176758, 0.5792803764343262, 1.1845937967300415], [0.9325516223907471, -1.6148099899291992, 0.11173895001411438, -0.10694681107997894, 0.16399793326854706, 0.2895467281341553, -1.585422158241272, -0.11755833029747009, -0.22436393797397614, -0.5071266889572144, -0.06557255983352661, 1.1633037328720093, 0.3155633509159088, 0.32527461647987366, -0.03765694424510002, 0.6548433899879456, 0.41770318150520325, 0.3812461197376251, -0.2352878898382187, -0.34076210856437683, 0.8722358345985413, -2.080739974975586, 0.8041917681694031, -0.034688800573349, -0.3519724905490875, 0.17494212090969086, -1.6977182626724243, 0.06091677397489548, 0.19548527896404266, 0.03763777017593384, 0.27525976300239563, 0.6067795157432556, 0.9100419878959656, -0.7430793046951294, 0.6345445513725281, 0.4595516324043274, -1.7907841205596924, 0.2704315781593323, 0.05307924747467041, -0.26335838437080383, -0.6130801439285278, -2.152092695236206, -0.1341296136379242, -1.190522313117981, -0.07030151784420013, 0.12249864637851715, 0.21148788928985596, -0.24074071645736694, 0.8224644660949707, 1.349376916885376, 0.16795840859413147, 1.6837055683135986, 0.7225760817527771, -0.7458236217498779, 0.470633327960968, 0.36855605244636536, 0.8212253451347351, 0.1764080822467804, -0.4340580105781555, -0.9310035109519958, 0.5315855145454407, 2.246731996536255, -1.376217007637024, -0.12025762349367142], [0.348219633102417, 0.24159680306911469, 0.0615561343729496, -1.0704048871994019, 1.546095609664917, 0.591140627861023, -0.2522740066051483, 1.2398827075958252, 0.09592275321483612, -1.0777617692947388, 0.7134174108505249, -0.4254797697067261, -1.0760269165039062, -0.09354668110609055, -0.06523237377405167, -0.2214244306087494, 0.30007854104042053, 0.7993374466896057, -0.5549722909927368, 1.274530053138733, -1.1300913095474243, 0.2049097716808319, -1.4765223264694214, -0.9375346899032593, -0.32888323068618774, 0.10544227063655853, 0.11043264716863632, -0.18097947537899017, -0.04812439903616905, 1.1289843320846558, -0.6268596649169922, 0.1845330148935318, -0.5706729888916016, -0.5379495024681091, -0.23259031772613525, -0.13833530247211456, 1.9197423458099365, 0.16275428235530853, 0.15153156220912933, 0.3640790581703186, -0.27859270572662354, -1.4888293743133545, 0.571803867816925, -2.104316473007202, 0.12878559529781342, -0.10331636667251587, -0.07140693068504333, 0.6133883595466614, -0.11734945327043533, -0.3841085135936737, -1.6335777044296265, -1.3065669536590576, 0.04475471377372742, 0.24764913320541382, 0.0716293454170227, -0.08971938490867615, -0.3728991448879242, 0.2201920598745346, 0.608909547328949, 1.6811943054199219, 0.8726699948310852, 1.850206732749939, 0.7423099875450134, -0.20133142173290253], [0.46870166063308716, -0.7418416142463684, -0.07261399924755096, 0.857725977897644, -0.7125312685966492, -0.7557735443115234, -0.1915874481201172, -0.5703504681587219, 0.5115256905555725, 0.7367764711380005, -0.5580200552940369, -1.912962794303894, 0.0076822396367788315, -0.1268170326948166, -0.0777905285358429, 0.217897430062294, -1.0301849842071533, 0.35473260283470154, 0.15603233873844147, -0.5983109474182129, -0.6641855835914612, -0.9726659059524536, 0.9205790758132935, -0.29228630661964417, -0.1658526211977005, -0.06813319772481918, 0.32483649253845215, 0.26186370849609375, -0.6811928153038025, -0.761221706867218, -0.06112414598464966, 0.10807017236948013, 0.4926477074623108, -0.027599748224020004, -0.15394924581050873, 1.371469259262085, 4.774359226226807, -0.8493674397468567, -0.8760232329368591, 1.6000428199768066, -0.5701397061347961, 0.43527474999427795, -0.19808056950569153, -1.0346871614456177, 0.20228353142738342, -0.01185754407197237, 0.06097348779439926, 0.1225699633359909, 0.1951798051595688, -0.9350355863571167, -0.821628212928772, -2.5896897315979004, 0.3882193863391876, 0.44358208775520325, -0.2830543518066406, -0.35182857513427734, 2.589772939682007, 0.8740109801292419, -0.6537230014801025, 0.719513475894928, 0.015868933871388435, 1.3562946319580078, -0.2901107668876648, 0.09373518824577332], [-0.12622660398483276, 1.746392846107483, 0.15931318700313568, -0.08218178898096085, -0.1799212545156479, 0.6160946488380432, -1.4112294912338257, 0.3542981445789337, -1.5847281217575073, -0.4964301288127899, 1.2672157287597656, 0.9362413883209229, 0.7877158522605896, 0.04722846671938896, -0.07340595126152039, -0.5619693398475647, -1.6641451120376587, 0.0532793365418911, -0.314412385225296, 0.9484121203422546, 0.09938855469226837, -2.133622169494629, -0.48237910866737366, -0.17661072313785553, 0.1798114776611328, -0.12622042000293732, 1.0208851099014282, 0.6301583051681519, -0.01922459714114666, -0.29266229271888733, 0.6801081895828247, 0.5004151463508606, 0.5620514750480652, 1.0413399934768677, -0.011869007721543312, 0.5806458592414856, 2.2200047969818115, 0.6786925792694092, 1.6075193881988525, -0.7374090552330017, -1.0887049436569214, -2.402499198913574, 0.5993696451187134, -1.5750490427017212, 0.5957828760147095, 0.41729670763015747, -0.05340798199176788, -0.5626397132873535, 0.3014969527721405, -1.949398398399353, 0.7617156505584717, -2.4206080436706543, 0.24342676997184753, -0.14069338142871857, 0.03350486233830452, -0.0908701941370964, 0.7218819260597229, -1.2777105569839478, 0.06450586766004562, 1.5967503786087036, 0.05850948765873909, -0.6486927270889282, 0.8662734031677246, -0.29280510544776917], [-0.5334487557411194, 0.1562250852584839, 0.08021806180477142, -0.4276910722255707, 0.8426396250724792, -0.5399068593978882, 0.7080672383308411, -0.8518388867378235, -0.5335291028022766, 0.6219246983528137, 1.4597713947296143, -0.3342798054218292, -0.9063088893890381, -0.014238446019589901, -0.06125544756650925, 0.5388091206550598, 0.2762686610221863, -0.9101207852363586, -0.11800912022590637, -0.5624591112136841, -1.1776221990585327, 2.0431559085845947, -0.5229775905609131, -2.054577589035034, 0.06264371424913406, 0.18315057456493378, 0.6762940883636475, -0.38379621505737305, -0.04600268229842186, 0.07567403465509415, 0.042374882847070694, 0.19435420632362366, -0.4410371482372284, 1.8273500204086304, 0.3903856575489044, 0.09369037300348282, -0.6013659238815308, 0.19865736365318298, -1.881282091140747, 0.5377925634384155, 1.0432977676391602, 2.538574457168579, -0.1783003956079483, 1.0931510925292969, -0.4716411232948303, -0.1951461285352707, -0.24968570470809937, -0.01738169975578785, -0.49836480617523193, -0.5097925066947937, -0.537433385848999, 0.4732217788696289, -0.7669052481651306, 0.19155199825763702, 0.1625629961490631, -0.9530233144760132, 0.686531126499176, -0.7785776853561401, -0.06155618652701378, 1.1475145816802979, -0.11284682899713516, -1.4275099039077759, 1.632366418838501, -0.3183068633079529], [0.21313762664794922, 0.8680154085159302, 0.10945693403482437, -0.9103366732597351, -0.8514255285263062, 0.051634639501571655, -0.7981391549110413, -0.19406412541866302, -0.35072770714759827, -0.2291618436574936, -0.18194164335727692, 1.9879626035690308, 1.069871187210083, -0.04633082076907158, -0.06533756852149963, 0.5105562210083008, -0.5344992280006409, -0.6088626384735107, -0.13694339990615845, 0.16022583842277527, -0.15704968571662903, 1.4109474420547485, -0.3392675220966339, -0.11215467751026154, 0.19410380721092224, 0.04794520512223244, -0.24317964911460876, 0.25078749656677246, -0.4491429328918457, -0.04450959339737892, 0.180154487490654, 0.20759150385856628, 0.37097692489624023, 0.653847873210907, 0.350734144449234, -0.22691555321216583, -1.3930875062942505, -0.1236598938703537, 0.8443443775177002, -0.04635440558195114, -2.271738052368164, 0.6647999882698059, -0.06344881653785706, -0.6709423661231995, -0.11636824905872345, 0.26307380199432373, -0.3270520567893982, 0.333938866853714, -0.47573691606521606, -0.6799595952033997, -0.24213701486587524, -1.2266920804977417, 0.23128433525562286, -0.06081143394112587, 0.037090566009283066, -0.03711376711726189, 2.391538381576538, 0.7162783145904541, 0.12472593784332275, 0.6482380628585815, -0.3103179633617401, 0.23917201161384583, -0.5652682781219482, -0.04175597056746483], [0.9497362971305847, 0.7489896416664124, 0.19452014565467834, 1.452189564704895, -2.8683292865753174, -0.12258294224739075, -0.4917810559272766, 1.5353188514709473, 0.7692245841026306, 0.4733331501483917, -0.7322196364402771, -1.9226346015930176, -0.15332239866256714, 1.0429067611694336, 0.11053656786680222, 0.1268332153558731, -0.2738402783870697, -0.7347964644432068, 0.21130765974521637, -0.3845312297344208, -1.0237194299697876, 3.2218048572540283, -0.0014702973421663046, -0.8879343867301941, 0.10864920169115067, -0.1746472716331482, -0.8837551474571228, -0.3798752725124359, -0.40677177906036377, 0.5901200771331787, 0.23518678545951843, -0.39275380969047546, -0.5624359846115112, -3.0872771739959717, 1.2141069173812866, 0.9182685613632202, -1.0770173072814941, -0.23406432569026947, 0.8570884466171265, 1.4067529439926147, -2.3348376750946045, -0.21500907838344574, 0.5185867547988892, -1.1070502996444702, -0.19980819523334503, -0.08596482872962952, 0.06935026496648788, 0.5308119058609009, -0.014780914410948753, -0.15635810792446136, -0.12309042364358902, 2.3830294609069824, -0.6503844857215881, -0.21096272766590118, -0.015256385318934917, -0.9662550091743469, 1.4482524394989014, -1.0612444877624512, 0.8758100867271423, 2.189638376235962, -0.47884801030158997, -0.44816073775291443, -0.13371503353118896, 0.815133273601532], [-1.362589955329895, 1.337347149848938, -0.18497908115386963, 0.1306210607290268, -1.3944568634033203, 0.27646470069885254, -0.06235048174858093, -0.8011928796768188, -1.0674738883972168, -0.7507826685905457, -1.3427162170410156, 2.0030581951141357, -0.016073711216449738, -0.06138956546783447, -0.2361818552017212, 0.03841928765177727, 2.5715456008911133, 1.0275870561599731, -0.3986692726612091, -0.02246212586760521, -1.0962567329406738, 0.8128826022148132, -0.9721142053604126, -0.626689612865448, -0.5003435611724854, -0.03212818130850792, -0.44980332255363464, -0.26879632472991943, 0.12821921706199646, -0.5728365778923035, 0.11286833137273788, -0.017539774999022484, -0.6742774844169617, -0.5171982049942017, -0.5157741904258728, 0.8533852100372314, 2.4001576900482178, -0.6438634395599365, -0.5342347025871277, 1.4132254123687744, 1.4323790073394775, -0.796501636505127, -0.3920038342475891, 2.153029680252075, -0.21222764253616333, -0.3093636929988861, -0.3550841808319092, -0.21351580321788788, 0.5701825618743896, -0.3006836175918579, -1.4904074668884277, -1.4686999320983887, -0.4127688705921173, 0.3755190372467041, 0.1965244710445404, 0.000920755323022604, 1.1640582084655762, 0.12697231769561768, 0.25473588705062866, -0.6624557375907898, 0.5989542007446289, 0.9962102174758911, 0.4278416335582733, 0.3337779939174652], [-0.24993424117565155, -0.31502586603164673, -0.0961294174194336, 0.062381964176893234, 0.4335220456123352, -0.526604413986206, 0.5911902189254761, -2.4248569011688232, -0.5955110192298889, 0.340096116065979, 0.18866564333438873, 2.9551141262054443, 1.2509551048278809, 0.12855078279972076, -0.037219058722257614, 0.10237116366624832, -2.365736961364746, 0.6582361459732056, -0.27727705240249634, -0.2604656219482422, -0.33017387986183167, -1.1256585121154785, -0.20380942523479462, 0.5808053016662598, 0.28163453936576843, 0.17404262721538544, -1.1691771745681763, -0.29498788714408875, 0.25905296206474304, 1.4555630683898926, -0.0266535934060812, 0.8006900548934937, 0.008098138496279716, -0.7898218035697937, -0.09062254428863525, -1.2747997045516968, -0.6186231374740601, 0.362028032541275, 1.5632909536361694, -0.5750288367271423, -1.1793683767318726, -2.715392827987671, 0.5145642757415771, 1.8108526468276978, 0.021159276366233826, 0.04606308043003082, -0.14897103607654572, -0.14862804114818573, -0.1309916228055954, -0.893170952796936, 0.781219482421875, -0.6682018637657166, -0.4969991445541382, 0.10841655731201172, -0.27029067277908325, -0.7058618664741516, 1.1820167303085327, 0.2301255464553833, -0.0672568753361702, 3.0999505519866943, 0.5007877349853516, -0.596408486366272, 1.1046360731124878, 0.07357782870531082], [-1.4320889711380005, -1.5188482999801636, -0.21579214930534363, 1.6983758211135864, 0.5148787498474121, -0.5301529765129089, 0.37033611536026, 0.1449899822473526, 0.10336466878652573, -1.0221351385116577, 4.985403060913086, 2.4832100868225098, 0.6379063129425049, -0.10461581498384476, -0.021916326135396957, -0.7643983364105225, 0.1630149632692337, -0.11555012315511703, -0.10552796721458435, 2.5628485679626465, -1.1683135032653809, 1.8244941234588623, -0.27361351251602173, 1.485216498374939, -0.5570383667945862, -0.1295858472585678, 0.3564583659172058, -0.20201538503170013, 0.4449416697025299, 0.1814134567975998, 2.333601713180542, -1.317216157913208, -0.34555020928382874, -1.401706576347351, 0.27949774265289307, -1.2083367109298706, -1.4535102844238281, 0.44836997985839844, -0.6452407240867615, 0.04110082983970642, -0.20342959463596344, -1.5242031812667847, 0.11933155357837677, -0.9908931851387024, 0.17540302872657776, -0.1549672931432724, 0.20109888911247253, 0.292334645986557, -0.030047286301851273, 0.5868522524833679, -0.9400856494903564, 1.7578681707382202, 0.40483132004737854, 0.306689590215683, -0.1581687182188034, 0.5298739075660706, -0.33946433663368225, -1.116462230682373, -0.4070315659046173, -2.503223419189453, -0.684432327747345, -3.1749114990234375, 0.8539816737174988, 0.47278568148612976], [0.012849908322095871, 1.2247775793075562, -0.22508122026920319, -0.07178092747926712, -0.04758535325527191, -0.19293732941150665, 0.3618053197860718, -0.5526713132858276, -0.026594743132591248, 0.48411500453948975, 0.3866347670555115, 1.6502916812896729, -0.8977208733558655, -1.3199800252914429, -0.15622557699680328, 0.40253692865371704, 0.8896443843841553, -0.32751381397247314, -0.6128544807434082, -0.3307405710220337, -2.0167887210845947, 1.053510308265686, 2.9732067584991455, 0.6846938133239746, 0.4056657552719116, -0.0024301819503307343, 0.8832672238349915, -0.312589555978775, -0.2721586227416992, -0.8228261470794678, -0.8710644245147705, 0.17238041758537292, 0.05371095985174179, 1.305377721786499, 0.0958646833896637, 1.514112949371338, 0.2717655897140503, -1.3423867225646973, -0.4847490191459656, 0.6094481348991394, -2.544450521469116, 0.38527390360832214, 0.11808504164218903, -0.1390683352947235, -0.37688544392585754, -0.05550043284893036, -0.24996626377105713, -0.3673126697540283, 0.5141814351081848, -0.2810031473636627, 1.3784880638122559, -2.4539594650268555, -0.6151960492134094, -0.5743135809898376, 0.4266020655632019, 0.11993970721960068, 0.9136494398117065, -0.7111862897872925, -1.0189231634140015, -0.8157269954681396, 0.3611280024051666, 1.7049401998519897, 0.15134122967720032, -0.4191170334815979], [-1.09217369556427, 1.0467755794525146, -0.00761560769751668, 0.5529389977455139, 1.95480215549469, -0.0610053576529026, -0.5821928381919861, -0.26321250200271606, -0.3042057752609253, -0.416621595621109, 1.3813499212265015, -1.611451506614685, -0.47915753722190857, 0.29360491037368774, -0.343000590801239, 0.6433786153793335, -1.8204084634780884, -0.38702839612960815, -0.8414345383644104, 0.1807418018579483, 0.5454851984977722, 1.725313663482666, 1.1221238374710083, -0.883709728717804, 0.12950901687145233, -0.06905689090490341, 1.1167182922363281, -0.35554710030555725, 0.28557342290878296, -0.6395829319953918, -0.10528476536273956, -0.5035765767097473, -0.4929218590259552, 0.4777906537055969, 0.047592271119356155, -0.7548172473907471, 1.0534124374389648, -0.33651089668273926, 2.580303430557251, 0.9462168216705322, -0.524241030216217, 0.46571430563926697, -0.1737401783466339, -1.1831433773040771, -0.21726089715957642, -0.269124835729599, 0.20832382142543793, 0.5089149475097656, -1.136173963546753, -1.2556095123291016, -1.9762721061706543, 0.4962887763977051, -0.12814325094223022, -0.4332653284072876, -0.30191248655319214, -0.23928296566009521, 0.5095425248146057, -0.6846991777420044, 0.560882568359375, 1.6161483526229858, 0.24893975257873535, 0.11032883822917938, 0.0024305914994329214, 0.06223994493484497], [0.814478874206543, -0.7125416994094849, -0.01706380769610405, -0.2841998040676117, -0.33208757638931274, 0.08082534372806549, -0.03150004521012306, -0.9213008284568787, -0.15385505557060242, -0.32817575335502625, 0.648143470287323, -3.2018659114837646, -0.399959921836853, -0.4398038685321808, 0.006079122424125671, -0.6915712356567383, 0.6162415742874146, 0.4935019910335541, 0.3177677094936371, 1.1847139596939087, 0.2504982650279999, 0.951694905757904, 1.0258433818817139, -0.07119286805391312, 0.12683342397212982, -0.022547688335180283, 0.39398667216300964, 0.372834712266922, -0.5765866637229919, 0.14990785717964172, -0.9450774192810059, -1.3934909105300903, -0.7388579845428467, -0.28433144092559814, 0.2268475592136383, -1.6156617403030396, 1.5881850719451904, 0.2686721086502075, 1.9571999311447144, -0.7286573648452759, -1.1205745935440063, -0.27653393149375916, 0.06940066814422607, 0.7215597033500671, 0.15156041085720062, -0.01928969845175743, 0.17333821952342987, 0.20463557541370392, 0.4846266508102417, 0.4245636761188507, 0.9340535402297974, 0.6044967174530029, 0.2437913715839386, -0.41913893818855286, -0.07991413027048111, -0.24824757874011993, 0.859531581401825, -0.4463909864425659, -0.6085394024848938, 0.5073236227035522, 0.6512009501457214, -1.2351394891738892, 0.5944260358810425, 0.24533361196517944], [0.8592367768287659, 0.5471424460411072, -0.1575274020433426, -0.25334277749061584, 0.14346419274806976, 0.20738133788108826, -1.283186912536621, 1.1596574783325195, -0.33739548921585083, -0.091968834400177, -1.7927619218826294, 0.9554429650306702, -0.35049113631248474, -0.24318815767765045, 0.16542363166809082, 0.3750423491001129, 1.9223153591156006, -0.8511549830436707, -0.5623977780342102, 0.5605325698852539, 0.43814921379089355, -0.1413070410490036, -0.9295635223388672, 0.057566139847040176, 0.4191908538341522, 0.13400769233703613, -0.5407081246376038, 0.29546207189559937, -0.367063969373703, -0.08237962424755096, -0.4748722314834595, 0.5050009489059448, -0.12407316267490387, 0.12912775576114655, -0.48072049021720886, 0.8508661389350891, 2.2559285163879395, 0.230551078915596, -0.9475293755531311, -0.587733268737793, 1.0825923681259155, -1.256673812866211, -0.11586131900548935, -1.9543548822402954, -0.16659899055957794, -0.051580242812633514, 0.05702536180615425, -0.4503154754638672, 1.041421890258789, -1.3694192171096802, -0.19542424380779266, 2.1730852127075195, 0.5645141005516052, -0.04250507429242134, -0.09995224326848984, 0.02293248660862446, -0.31780773401260376, -0.2744150459766388, 0.20237568020820618, -2.81706166267395, -0.27455785870552063, 3.577864408493042, -0.11426196247339249, -0.8331451416015625], [0.14276079833507538, 0.5498858690261841, 0.17778988182544708, -0.2805323898792267, 0.06023074686527252, -0.2562249004840851, -0.7083888053894043, -0.5752624273300171, -0.18988201022148132, 0.38618794083595276, -1.9420098066329956, -1.0148173570632935, 0.7007772326469421, 0.37621185183525085, 0.2378435879945755, 0.8188234567642212, 0.6801449656486511, -0.5005958676338196, 0.468323677778244, 0.4345342218875885, 0.7463880181312561, 0.25413599610328674, 1.0079491138458252, 0.3582594096660614, -0.018730157986283302, 0.3810006380081177, -1.7490382194519043, 0.3803060054779053, 0.171867236495018, 0.11172255128622055, -1.7659692764282227, -1.3178640604019165, 0.5875034928321838, 1.054511308670044, 0.3120785057544708, -1.600868821144104, -1.7720460891723633, -0.1018809899687767, -1.3498048782348633, 1.0052525997161865, -1.291924238204956, -1.8992081880569458, -0.03055647388100624, 0.6273519992828369, 0.37197813391685486, 0.18904316425323486, -0.23023393750190735, -0.4234120547771454, 0.6696235537528992, -0.2204454094171524, 0.4091040790081024, 1.7189422845840454, 0.34135136008262634, 0.3068815767765045, 0.39510971307754517, 0.6425008177757263, 4.0434393882751465, -0.7574895620346069, -0.9414314031600952, 0.8282794952392578, 0.19710484147071838, -0.8360835313796997, 0.48701196908950806, -0.8575101494789124], [1.110367774963379, 0.12964314222335815, -0.010380569845438004, -0.37963414192199707, -0.4176410436630249, -0.26919931173324585, -0.10628294199705124, 0.5028690099716187, -0.2944738268852234, 0.49219802021980286, -0.645041286945343, 3.4555745124816895, 1.6961643695831299, 0.02657431922852993, 0.061659976840019226, -0.8033581972122192, -1.364362120628357, 0.21646079421043396, -0.03743080422282219, -0.6451520919799805, -0.4861367344856262, 0.004454552195966244, -0.9037330746650696, -0.4618302583694458, 0.2268255352973938, 0.33424344658851624, -0.4013406038284302, -0.5389830470085144, 0.17588862776756287, 0.8356986045837402, -0.6373602747917175, 0.5210135579109192, 0.8628535270690918, -0.02178521640598774, 0.2431652694940567, 0.18982960283756256, -2.3527519702911377, -0.8954587578773499, 0.06419362127780914, -0.8236624002456665, -0.18357901275157928, 1.5877878665924072, 0.04668617993593216, -0.9196977019309998, 0.028606422245502472, -0.1086374968290329, 0.3453955054283142, 0.02260233834385872, 0.18768863379955292, -0.9585776925086975, -0.9401916265487671, 0.42518150806427, -0.13412170112133026, 0.13214758038520813, -0.19351255893707275, -0.013775842264294624, 2.599127769470215, -0.01622997783124447, -0.25930631160736084, -0.35550248622894287, 0.09798207134008408, -0.652012288570404, 0.6261422634124756, -0.017882799729704857], [-1.1155527830123901, 0.5010064840316772, -0.11005367338657379, -0.0543074831366539, -2.476121187210083, -0.46214330196380615, 0.7687329649925232, -0.9240158200263977, 0.537833034992218, -0.3996182680130005, 2.728219747543335, 2.6788759231567383, -0.682938814163208, -0.06441593915224075, -0.02414218708872795, -0.2875778079032898, 0.6815271377563477, 0.1149996891617775, 0.18071448802947998, 0.338513046503067, -0.8588626384735107, 0.9775001406669617, 1.1650413274765015, -0.15600945055484772, 0.14211900532245636, -0.17137616872787476, 0.22680100798606873, 0.032634664326906204, -0.25157591700553894, -0.04710812121629715, 0.6168163418769836, 0.41703036427497864, 0.2798478901386261, -1.3514589071273804, -0.5768206119537354, -0.7114807367324829, 1.4812068939208984, 0.29116401076316833, -0.844452440738678, 0.9262234568595886, -2.9705758094787598, -2.437894105911255, -0.658134400844574, 0.6607635021209717, -0.2144474983215332, 0.08346789330244064, 0.30537912249565125, -0.17064104974269867, -0.016062937676906586, -0.2523053288459778, 0.8443490266799927, -0.06784605234861374, -0.032750871032476425, -0.5039873719215393, -0.17811289429664612, 0.3008840084075928, 3.1598715782165527, -0.5148283243179321, 0.00871358998119831, -0.4092291295528412, -0.5389799475669861, -0.004948084708303213, 0.3018604516983032, -0.21132078766822815], [-0.11484570056200027, -1.646788477897644, 0.01892823725938797, 0.6182592511177063, -1.2407723665237427, 0.41077136993408203, 0.661740243434906, -0.7635965347290039, -0.19333137571811676, -0.5630336999893188, -1.724530816078186, 3.204782724380493, -0.07016176730394363, -0.07643336057662964, 0.04432743042707443, -0.6756234765052795, 3.6020820140838623, 0.8773958086967468, 0.3705209791660309, -0.8909747004508972, -0.5099427103996277, -0.7157197594642639, -1.594856858253479, 2.970318555831909, 0.12147769331932068, 0.14451387524604797, 0.5030955076217651, 0.34610164165496826, 0.4907056391239166, -0.8438905477523804, -1.2218601703643799, 0.05116148293018341, 0.3166300058364868, 0.279674768447876, 0.19991600513458252, -0.29689711332321167, -4.773442268371582, 1.379577398300171, -0.3094019591808319, -0.07677861303091049, -0.2267763465642929, 1.481819748878479, -0.8667554259300232, -3.988579034805298, 0.5513613820075989, -0.018035640940070152, -0.2549040913581848, 0.2049165964126587, 0.8322065472602844, 1.286230444908142, 1.068203091621399, -0.057437777519226074, -0.000435555906733498, 0.35167840123176575, 0.10188286006450653, 0.7735382318496704, 0.8975620865821838, 0.13659165799617767, 0.27113115787506104, -1.127705693244934, -0.3959641754627228, -0.7049759030342102, 0.5285288691520691, 0.8468199372291565], [0.426078736782074, 1.7004826068878174, -0.061459776014089584, -0.9419994950294495, -0.6421513557434082, -0.1230599507689476, 0.40471771359443665, -0.666165292263031, -0.740100085735321, -0.3302312195301056, 2.672311782836914, 2.3823230266571045, 0.35890260338783264, 0.11619027704000473, -0.00707622105255723, -0.5783265233039856, 0.0515718087553978, 0.9055947661399841, -0.18063372373580933, -0.2721392810344696, 0.4246388077735901, -0.12974987924098969, 0.3837369382381439, 0.7103395462036133, -0.25000888109207153, 0.06113036349415779, -0.233133465051651, -0.9386738538742065, -0.48733800649642944, 0.3067103922367096, 0.34931686520576477, -0.5253491997718811, -0.6000421047210693, -1.2741175889968872, -0.4905456304550171, 0.23583699762821198, 0.34739935398101807, -0.30777227878570557, 0.6292144060134888, -1.23947274684906, 1.218601107597351, -1.7533854246139526, -0.13759374618530273, 0.7892308831214905, -0.46442529559135437, -0.0224906112998724, -0.07686760276556015, -0.1181308701634407, 0.4141525626182556, 0.7588633298873901, -1.4019383192062378, -0.42768099904060364, 0.5382910370826721, 0.3288448750972748, 0.2767709493637085, -0.4579879343509674, -0.390720009803772, -0.3611273169517517, -0.02298688143491745, 1.1391395330429077, 0.779578685760498, -0.7536166310310364, -0.586584746837616, -0.714887261390686], [-2.2514750957489014, -0.07302394509315491, -0.15543633699417114, 0.4282603859901428, 0.9790305495262146, -0.31527963280677795, 0.8687748312950134, -0.7026521563529968, 0.9505102038383484, -0.49644777178764343, 2.55660080909729, 1.289114236831665, 1.490667700767517, 0.23623573780059814, -0.20860986411571503, 0.5909242033958435, 1.0846569538116455, -0.03479386866092682, -0.7844019532203674, 0.2672266662120819, -1.042586326599121, -1.2743035554885864, -0.8537667393684387, 0.3016279339790344, -0.15058453381061554, -0.2939884066581726, 0.7171339392662048, -0.9862821698188782, -0.23863671720027924, -0.6404661536216736, 0.45264342427253723, 0.7419660687446594, 0.3994921147823334, -0.3382016718387604, 0.3159776031970978, 1.4222356081008911, -0.7865025401115417, -0.137870654463768, 0.6917020082473755, -0.8360322713851929, 1.6255881786346436, 2.4008963108062744, 0.6110667586326599, -0.9938904643058777, 0.1643734723329544, -0.1454893946647644, -0.29964715242385864, -0.016874434426426888, 0.10262560844421387, -1.0829353332519531, 1.2988687753677368, -0.7729630470275879, 0.7947825193405151, -0.17473091185092926, -0.7431434988975525, -0.7253584861755371, 0.30084818601608276, -0.4509136378765106, 0.320799857378006, -2.2493040561676025, 0.12027709931135178, -1.1635786294937134, -1.9997508525848389, -0.10498541593551636], [-0.43569934368133545, 0.6477078199386597, 0.23751218616962433, 0.21858911216259003, -0.5038352608680725, 0.25878283381462097, -0.2733443081378937, -0.22492285072803497, -1.1407684087753296, 0.338020384311676, -2.979839563369751, 1.9750478267669678, -0.16250872611999512, 0.20504628121852875, 0.28875428438186646, 0.015552814118564129, 1.7316277027130127, -0.4178560972213745, 0.6285285949707031, 0.010797175578773022, -0.14801239967346191, 0.06451745331287384, -0.24407003819942474, -0.028558824211359024, -0.2543434500694275, 0.35686224699020386, -0.7819237112998962, -0.021339772269129753, 0.5419960618019104, -0.5956323742866516, -0.44132813811302185, 0.540237307548523, 0.8908720016479492, -0.011586912907660007, 0.74687260389328, -0.4313782751560211, 1.7102341651916504, -0.049377135932445526, -3.2006664276123047, -0.47684064507484436, -1.7412036657333374, -3.334294080734253, 0.635120689868927, 1.5600605010986328, 1.3430430889129639, 0.04032963886857033, -0.07011404633522034, 0.4835863709449768, 0.5818493962287903, 1.1856120824813843, 0.8707885146141052, -3.7853283882141113, 0.401165246963501, 0.2406308650970459, 0.21521598100662231, 1.0026801824569702, 0.8370048999786377, 0.8121653199195862, -0.8115204572677612, 1.3259676694869995, 0.26257315278053284, 0.794087290763855, -0.933220386505127, -0.4999261796474457], [-1.0335441827774048, 0.6263189315795898, -0.0540323406457901, 0.1990565061569214, 0.6681970953941345, -0.4820219874382019, 0.41972988843917847, -0.7231968641281128, 0.3628799617290497, 0.47215527296066284, 0.30663686990737915, -0.9998079538345337, -0.43071866035461426, -0.015552164055407047, 0.05590217560529709, 0.15007273852825165, -0.9069876670837402, 1.2937973737716675, 0.07005328685045242, 0.39372289180755615, -1.1231647729873657, -1.4786127805709839, 0.9265875816345215, -2.2879722118377686, -0.25116169452667236, 0.03881055489182472, -1.6276696920394897, 0.09685970842838287, 0.2486899048089981, -0.1697879433631897, -0.555574357509613, 0.4600810408592224, 0.19845698773860931, -0.8359712362289429, -0.1438978612422943, -0.406265527009964, 3.8746960163116455, 0.04495018720626831, -0.5924525856971741, 0.1095704510807991, 0.7021995186805725, 1.0536839962005615, 0.0752587616443634, -2.883470058441162, -0.4305560290813446, 0.07640502601861954, 0.2443414330482483, 0.33113253116607666, 1.377566933631897, -0.4943966269493103, -0.4754889905452728, 4.163664817810059, 0.8651239275932312, 0.19145497679710388, 0.7833911180496216, -0.11116692423820496, 0.03666651248931885, 0.6281729340553284, -0.5168041586875916, -1.797894835472107, 0.5228813290596008, -0.7885593771934509, -0.4017486274242401, -0.050691213458776474], [1.2927508354187012, -1.1835373640060425, -0.03441594913601875, -0.2056308090686798, -0.6358518004417419, 0.03962806239724159, 0.24842816591262817, 0.8213056921958923, -0.18897366523742676, -0.7929577231407166, 0.5570071339607239, -1.6964360475540161, -0.7435789108276367, -0.21002383530139923, -0.0898810476064682, -0.8507314324378967, -1.8799543380737305, -0.00027407598099671304, 0.4094824194908142, 1.227073073387146, -0.9424461126327515, -1.751848816871643, -0.535184383392334, 0.6816825270652771, 0.21912431716918945, 0.14772559702396393, -0.9742549061775208, -0.43124493956565857, 0.06008142605423927, -0.1484137773513794, 0.6385404467582703, 0.2519802451133728, 0.13728119432926178, 2.0434064865112305, 0.10318674147129059, -1.6298227310180664, 1.918753981590271, -0.18282614648342133, 2.107210397720337, 1.0605465173721313, 0.39669016003608704, -2.3354856967926025, -0.021688062697649002, -2.750548839569092, 0.1917376071214676, 0.09597796201705933, 0.06516038626432419, 0.029304450377821922, 0.43795010447502136, 1.7810285091400146, -1.8249648809432983, 3.0696802139282227, 0.1937340795993805, -0.8529821634292603, 0.007952648214995861, -0.15936791896820068, -0.0061823055148124695, -0.6833358407020569, 0.2214125394821167, 1.3952099084854126, -0.2502059042453766, 1.4746508598327637, 0.8390885591506958, -0.17172284424304962], [-0.47252580523490906, 0.7958415150642395, -0.13934582471847534, -0.01194179616868496, 1.5041017532348633, -0.331527978181839, 0.12099175900220871, -1.1920902729034424, 0.757966935634613, -1.4228432178497314, 2.5126378536224365, -2.161863088607788, -0.5644533634185791, -0.25154411792755127, 0.008969155140221119, 0.5970555543899536, 0.10398505628108978, -0.24124416708946228, -0.08839350193738937, 0.1352832168340683, 0.6721994876861572, -1.2085227966308594, 0.5775914788246155, -0.5736914873123169, -0.02124617248773575, 0.1572929322719574, -0.06605782359838486, -0.12064207345247269, -0.317609578371048, 0.17642764747142792, 1.1487776041030884, -0.5211176872253418, 0.4126480519771576, 0.8823322653770447, 0.6974706649780273, 1.3736306428909302, -1.4904111623764038, -0.10866468399763107, 1.3625409603118896, 0.37311094999313354, 0.38765954971313477, -2.2397477626800537, -0.10656972974538803, -0.12764768302440643, -0.4122917354106903, -0.22101859748363495, -0.12897084653377533, -0.2172560840845108, -0.48812752962112427, -0.1633201539516449, 1.8655582666397095, 1.1154088973999023, -0.3034997582435608, 0.08908842504024506, 0.017952386289834976, 0.12142670899629593, -3.0163230895996094, -0.3934588134288788, -0.7332916855812073, 0.9430717825889587, -0.048606742173433304, 1.6751072406768799, 0.45575520396232605, -1.1360172033309937], [0.5962270498275757, 0.7850301861763, 0.04206814244389534, 0.5081395506858826, -1.1131190061569214, -0.15742889046669006, 0.05716021731495857, -0.882048487663269, -0.23185542225837708, 0.8038371801376343, -0.19107171893119812, 1.718805193901062, -0.46515586972236633, 0.054134104400873184, 0.13969872891902924, -0.7251904606819153, 0.24832743406295776, -0.24460236728191376, -0.22640599310398102, 0.5302678942680359, 0.2678380310535431, 1.0586800575256348, 2.5652291774749756, -0.9597833156585693, -0.5017572641372681, 0.2158772200345993, -1.8487389087677002, -0.4091320037841797, -0.3200013339519501, 0.37887561321258545, -1.0524711608886719, 0.06206119433045387, -0.38855570554733276, 0.7913385033607483, 0.16206271946430206, 1.390742540359497, 1.4029854536056519, -0.21165892481803894, 1.4300626516342163, -1.7885717153549194, -1.203616976737976, 4.258010387420654, 0.6343642473220825, -0.9954472780227661, 0.2563103139400482, 0.1687157303094864, 0.04110978543758392, 0.12594617903232574, 0.12043597549200058, 0.07322382926940918, -0.33663031458854675, -2.1536777019500732, 0.5203585028648376, -0.12383189797401428, -0.129474475979805, -0.30330002307891846, -1.4618593454360962, -0.7774600982666016, -0.5425251722335815, -3.0133256912231445, -0.1770988553762436, 0.3779667317867279, 0.284105509519577, 0.8658003211021423], [-0.3913051187992096, -1.35969078540802, 0.017470335587859154, -0.15222439169883728, -1.7575287818908691, 0.524796187877655, 0.6621604561805725, 1.1518245935440063, 0.027178864926099777, 0.7610117793083191, -0.45961612462997437, 1.1104010343551636, 0.1669161021709442, -0.1794009655714035, 0.0367916077375412, 0.24701175093650818, -0.5031551718711853, -0.22172199189662933, 0.2746167778968811, 0.4396907389163971, 1.6880995035171509, -1.9316385984420776, -0.3160407245159149, -1.4232416152954102, -0.7105802297592163, 0.09055665880441666, -0.24656474590301514, 0.2425425499677658, -0.0980994701385498, -0.010155974887311459, 0.661344051361084, -0.01459538098424673, -0.5244352221488953, 0.06098916381597519, 0.2449045330286026, 0.6124867796897888, 1.3833363056182861, 0.1702163815498352, -0.4808661639690399, -1.1480433940887451, -0.7407374978065491, -2.0256900787353516, -0.10549574345350266, 0.37531888484954834, 0.46833106875419617, 0.04840919375419617, -0.2023342251777649, -0.05224011093378067, 0.6970910429954529, -0.16399164497852325, 0.3159663677215576, 1.3286595344543457, -0.7273913025856018, 0.39370059967041016, 0.13286712765693665, 0.07614162564277649, -3.0239553451538086, -0.17246416211128235, 1.120463252067566, 0.021996285766363144, 0.4181194305419922, 0.7742266058921814, 2.3670008182525635, 0.030567051842808723], [-0.14550666511058807, -0.6813992857933044, 0.08388609439134598, -0.16735193133354187, 0.6267480850219727, -0.2356252521276474, -1.1698707342147827, -0.3464878797531128, -0.6137509346008301, -0.967763364315033, -0.5973531007766724, 1.9715540409088135, -0.6649447083473206, 0.23422516882419586, 0.12977302074432373, 0.6066735982894897, -1.8239322900772095, 0.6515049934387207, 0.15976791083812714, -1.3631925582885742, -0.7895519733428955, -1.1763544082641602, -0.5033535957336426, 0.34993523359298706, -0.920142650604248, 0.1376931220293045, -0.3219415843486786, 0.13045792281627655, 0.04129410535097122, -0.6238219738006592, 0.4001224935054779, 0.07849477231502533, -0.29525962471961975, 0.7053117752075195, -0.42473042011260986, 0.8470875024795532, -0.5529615879058838, 0.32701724767684937, -0.2476707547903061, 0.23003073036670685, 1.1775683164596558, 2.5722405910491943, -0.6489020586013794, 1.1188273429870605, -0.061128903180360794, 0.4785679876804352, 0.23843708634376526, -0.38614022731781006, -0.15446947515010834, 0.38743871450424194, 0.21015363931655884, 1.2026249170303345, -0.4290708005428314, 0.014885924756526947, -0.1377871334552765, 0.014779061079025269, -1.754896640777588, 1.1466771364212036, -0.31280916929244995, 0.2754952907562256, -0.17475053668022156, 1.3346424102783203, 0.5170208215713501, 0.29198533296585083], [0.2507418692111969, -0.2922784388065338, -0.07103205472230911, 0.380049467086792, -0.5456668734550476, -0.4008256196975708, -0.41084983944892883, -0.30221208930015564, 0.4409327805042267, 0.6391277313232422, -4.3314995765686035, -1.6669102907180786, 0.6595792174339294, 0.02005297690629959, 0.02026660367846489, 0.04967658594250679, -0.9274401664733887, 0.546026885509491, 0.5315955281257629, 0.778046190738678, -1.5558589696884155, 2.0670220851898193, 0.5716806650161743, 1.1406482458114624, -0.16666851937770844, 0.40166041254997253, -0.2676992416381836, -0.07331785559654236, -0.12675029039382935, -0.8153772950172424, -0.7410895824432373, -1.454222559928894, 0.6568511724472046, 1.1512864828109741, 0.5782329440116882, 0.7263386249542236, 0.4108494818210602, -0.18609018623828888, -1.5936933755874634, -0.41025543212890625, 1.0174347162246704, -1.749473214149475, -0.40586549043655396, 1.9070903062820435, 0.7329676747322083, -0.06811180710792542, 0.08378031104803085, -0.10493867844343185, 0.01536960992962122, 0.010230067186057568, -0.03358636423945427, 3.1517813205718994, 0.5044426918029785, 0.8430205583572388, -0.170929417014122, -0.015718335285782814, -0.9353601336479187, 0.6985642910003662, -0.1243305578827858, -2.3243772983551025, 0.4284234344959259, -0.009107697755098343, 0.2115919440984726, 0.6561743021011353], [0.6959771513938904, 1.8785629272460938, -0.08442214876413345, -0.37787407636642456, -0.7508361339569092, -0.3187423050403595, 0.45113176107406616, 1.946785807609558, -0.5548818707466125, -0.7806041836738586, -2.4302072525024414, -1.362942099571228, 0.41014954447746277, -0.2911706268787384, -0.107612244784832, -0.4757848083972931, -1.6051266193389893, -0.29229268431663513, -0.2819090783596039, -0.04963446408510208, -0.7276614904403687, 1.797454595565796, -0.695171058177948, 1.4081019163131714, 0.3296087980270386, -0.1731998175382614, -0.00870711263269186, -0.274639755487442, -0.5218454599380493, 0.13907749950885773, 0.3252663314342499, -0.18347793817520142, 0.2354220747947693, 1.6401290893554688, 0.3319762349128723, 1.718079686164856, 2.442122220993042, -0.3768816590309143, 0.8496869206428528, 0.21630293130874634, -1.9208651781082153, -0.5926244854927063, 0.2450491487979889, -1.5691677331924438, 0.08904320746660233, -0.10819631069898605, 0.16789063811302185, 0.26532211899757385, -0.3977230191230774, -0.10930199176073074, 0.6585054397583008, -1.5985525846481323, 0.03300686180591583, 0.17957323789596558, 0.15636831521987915, 0.46919459104537964, 2.9947426319122314, -0.48648974299430847, -1.0370135307312012, -1.756514310836792, -0.6763253808021545, 1.1760797500610352, 0.48118239641189575, -0.7533944249153137], [-0.6392678618431091, 0.005230257287621498, 0.12656155228614807, -0.009559730999171734, 0.23611462116241455, 0.7027345299720764, -0.5859753489494324, 0.36782899498939514, 0.16502106189727783, 0.1210111454129219, 1.4873497486114502, 0.655597984790802, -0.5908522009849548, -0.34781548380851746, 0.14872713387012482, 0.6548243761062622, -2.332319736480713, -0.4749038815498352, 0.001017867005430162, -0.34860166907310486, 0.7936258912086487, 0.38661840558052063, 0.47483953833580017, -0.18229883909225464, 0.2000027894973755, 0.03438735753297806, -0.43946170806884766, -0.5635252594947815, 0.2790612578392029, -0.05311572551727295, -0.15358027815818787, 0.6815503239631653, -0.3590334951877594, -1.087945580482483, -0.23988667130470276, 0.6831746697425842, -2.2578256130218506, -0.2129250466823578, 0.6050137281417847, 1.0681397914886475, -1.5748649835586548, -0.4541682004928589, -0.28232574462890625, 0.5794036388397217, 0.19438178837299347, -0.3781804144382477, -0.22194115817546844, -0.11393499374389648, 0.31210383772850037, 0.049799300730228424, -0.4520803391933441, 0.5099295377731323, -0.051594045013189316, 0.3306719958782196, 0.09380368143320084, -0.0916784480214119, 2.893613576889038, -0.22436635196208954, -0.6380254030227661, 1.270430326461792, 0.00925457663834095, -0.06346547603607178, -0.22895632684230804, -0.46734943985939026], [-0.3268601596355438, 0.16258560121059418, -0.005958224646747112, 1.2729802131652832, -0.086893230676651, -0.03245948627591133, -0.8638068437576294, 1.833211898803711, 0.6103963255882263, 1.3181781768798828, 0.0861566886305809, -0.3528720438480377, 0.4285634160041809, 0.13517124950885773, 0.007396363653242588, -0.7184454202651978, 1.1534209251403809, -0.7088680863380432, -0.032538291066884995, -0.8487833142280579, -1.1357531547546387, 1.0099519491195679, -0.31156861782073975, -0.22331276535987854, 0.48651668429374695, -0.190675288438797, 1.0854980945587158, -0.11351396143436432, -0.7368735671043396, -0.4574434161186218, -0.05349593609571457, 0.21450386941432953, -0.1080203577876091, 1.2205634117126465, 0.08764602988958359, 0.7687578797340393, -1.4420274496078491, 0.45077139139175415, -2.4480810165405273, 0.6343832612037659, 1.8850730657577515, -0.6130735278129578, 0.08422670513391495, 0.45689064264297485, 0.2467317134141922, -0.047461140900850296, 0.5540540814399719, -0.01070029754191637, -0.2941233813762665, -1.822131633758545, -2.2490272521972656, 2.3290514945983887, -0.6530579924583435, 0.223683163523674, -0.06539671868085861, -0.4508785903453827, 1.2689695358276367, 2.217177152633667, 0.012351029552519321, -2.9702701568603516, -0.4302781820297241, -1.471437692642212, 0.08896032720804214, -0.05773523077368736], [2.0376107692718506, -1.0280901193618774, -0.06100035086274147, -1.748915433883667, -0.14927026629447937, 0.054934125393629074, 0.18403837084770203, -1.7738559246063232, 1.2578035593032837, -1.2472121715545654, 0.39427614212036133, -1.0161349773406982, 0.16846033930778503, -0.026432132348418236, 0.12033398449420929, 0.37305107712745667, -0.3204173743724823, 1.255663275718689, 0.32048365473747253, 0.06467676907777786, 0.2852329611778259, -2.183121681213379, 0.30611130595207214, 0.02282221056520939, 0.19077631831169128, -0.16320842504501343, -1.0209046602249146, 0.13417235016822815, 0.1836969256401062, -0.5534845590591431, -0.5851244926452637, 0.10093056410551071, 0.31140103936195374, 0.2811007499694824, -0.022891808301210403, -0.3097938299179077, -0.4723406434059143, 0.14313174784183502, 2.046618700027466, 0.07704591006040573, 3.000356435775757, -0.6038187742233276, 0.6479616761207581, -0.7807564735412598, 0.11661197990179062, -0.13943371176719666, -0.6898201704025269, -0.09379734098911285, 0.8579900860786438, 1.4082789421081543, 0.8533473610877991, -2.14404559135437, 0.14506736397743225, -0.007711067330092192, 0.3997809588909149, 0.3220835328102112, -1.7733336687088013, 0.8999897837638855, 0.4425177276134491, 1.4664171934127808, -0.1472393125295639, 0.3664359152317047, -2.328179121017456, 0.14912226796150208], [0.7486400604248047, -0.4102911353111267, -0.10612620413303375, -1.315765380859375, 0.43045133352279663, -0.1344906985759735, -1.8567382097244263, -0.9377633333206177, -0.1834665685892105, 1.2322701215744019, 1.8848299980163574, -4.857727527618408, 0.8645408749580383, 1.056742787361145, -0.052249208092689514, -0.12480895966291428, -0.42161068320274353, -0.20883534848690033, -0.2586551904678345, -0.7053942680358887, -0.036197829991579056, 2.3902182579040527, 1.2735592126846313, 1.362986445426941, -0.30122944712638855, -0.027678370475769043, 1.6489226818084717, 0.3320804238319397, 0.4342244565486908, -0.09361390769481659, -0.15113785862922668, -0.13907159864902496, -0.24190232157707214, 0.21342173218727112, -0.276212602853775, -0.2782272398471832, -0.7508353590965271, 0.30523577332496643, 1.5054066181182861, -0.961081326007843, 2.253389358520508, -0.5451210737228394, -0.037901926785707474, -1.3728307485580444, 0.36105379462242126, -0.19066965579986572, -0.27368780970573425, 0.021031484007835388, 0.22386236488819122, 0.5329354405403137, -0.007905283011496067, 0.7545289397239685, 0.03150152787566185, -0.4632190465927124, -0.02601529471576214, -0.5251916646957397, -1.7969121932983398, 0.38139134645462036, 0.4097461998462677, 2.238490104675293, 0.28778165578842163, -2.9275174140930176, -0.11482901871204376, -0.06633136421442032], [0.8638726472854614, -0.04228784143924713, 0.029166357591748238, 0.43015486001968384, -0.7023037075996399, 0.12569208443164825, 0.7884806394577026, 0.1897851526737213, 0.7319141030311584, 0.6656473278999329, 1.623042106628418, -3.037334680557251, -0.5675217509269714, -0.543704092502594, -0.04960137978196144, 0.6601260304450989, -2.631164073944092, -0.291104257106781, -0.11581535637378693, -0.21723000705242157, -1.893863558769226, -0.0317172035574913, 2.201058864593506, -0.18540915846824646, 0.19585146009922028, 0.0808316320180893, -0.9929625988006592, 0.43386343121528625, 0.09286465495824814, 0.7124125361442566, 1.996325135231018, 2.482407569885254, -0.4110463857650757, 0.7716904282569885, -0.18011842668056488, -0.5899665355682373, -0.4374408721923828, -0.28005728125572205, -0.0013743680901825428, 1.449810266494751, -0.4653618335723877, -0.970683753490448, 0.5271385908126831, -0.1945626586675644, 0.14073747396469116, -0.0810047909617424, 0.24820032715797424, 0.43754109740257263, 1.386690378189087, -0.4230134189128876, -0.039645034819841385, -1.358284831047058, 0.3369482457637787, 0.6673535704612732, 0.008629921823740005, 0.4429842531681061, -1.3112534284591675, -1.192348599433899, -0.22837230563163757, 0.43407365679740906, -0.3339191973209381, -1.2835631370544434, 0.049122050404548645, -0.12038054317235947], [0.7343919277191162, 0.6077979207038879, 0.18803778290748596, -0.4507266581058502, -0.5894803404808044, 0.11851874738931656, 0.07211273908615112, -0.1939314305782318, -0.7893596887588501, -1.1771098375320435, -1.1905910968780518, 0.7322587966918945, 0.9518995881080627, -0.27564042806625366, 0.2255527228116989, -0.198478102684021, -2.199854612350464, 0.21463395655155182, 0.1957738995552063, 0.32530492544174194, 0.6481993198394775, 0.2906571924686432, 1.607572317123413, 0.1187993660569191, -0.13428667187690735, 0.019368840381503105, -0.49298331141471863, -0.041702382266521454, -0.06363707780838013, -0.4492460787296295, 0.1618628352880478, 0.1922505646944046, -0.6514219641685486, 2.455275774002075, -0.11881078034639359, 1.8902705907821655, -0.5516961812973022, 0.4339492619037628, 0.9776186943054199, 0.44881725311279297, -0.021579772233963013, -1.7431532144546509, 0.2781743109226227, -1.042903184890747, 0.23138642311096191, 0.1087435930967331, 0.05711222067475319, 0.4573335349559784, 0.1199280247092247, -0.0122805992141366, 1.7775814533233643, -2.784351348876953, -0.9384886622428894, 0.447025328874588, 0.6110267043113708, 0.6670182347297668, 0.3624739646911621, 0.06209135055541992, 0.8189566731452942, -0.7917425632476807, -0.4985647201538086, -2.861609697341919, 0.46903544664382935, 0.1848178207874298], [-1.0169302225112915, -1.0432549715042114, 0.18585070967674255, 0.3659517765045166, -0.24758687615394592, -0.24853290617465973, -0.716734766960144, 0.39486292004585266, -0.4015514850616455, 0.8155320286750793, 1.7999744415283203, 2.0858116149902344, -2.085646867752075, 0.2941133677959442, 0.0744175985455513, -0.8297199010848999, -2.1274778842926025, -0.2429206222295761, -0.3360234498977661, -0.4854760468006134, -0.9469687938690186, 2.677680730819702, 0.8088091015815735, 0.7242594957351685, -0.5111081600189209, 0.38860824704170227, -0.0750182569026947, -0.4432259798049927, 0.4431552588939667, 0.13296355307102203, 0.2598124146461487, 0.2363509237766266, -0.169070303440094, -1.05636465549469, 0.1517609804868698, -2.036785125732422, 2.0489048957824707, 0.5502152442932129, -0.7032404541969299, -2.500627040863037, -0.29877156019210815, -0.3803526759147644, -0.24462467432022095, -0.5121740102767944, -0.15099652111530304, 0.31651172041893005, -0.03822299465537071, 0.1933363974094391, -0.531407356262207, 1.5842010974884033, 0.7845731973648071, -0.15776847302913666, -0.00672032730653882, 0.6392934918403625, 0.8139218688011169, -0.35993099212646484, -0.18753421306610107, 1.1862056255340576, 0.4017736613750458, 0.9778788089752197, 0.31071195006370544, -0.34111765027046204, -1.0258804559707642, 0.8123235702514648], [-1.0840445756912231, -0.3566744029521942, 0.0025119048077613115, -0.097672238945961, -0.17280352115631104, -0.5008854269981384, -0.27528461813926697, -0.28647199273109436, 0.2957540452480316, 0.145210400223732, -2.169200897216797, 2.2969820499420166, -1.379434585571289, 0.3409317433834076, -0.10967884212732315, 0.3677782416343689, 1.7811124324798584, -0.19626279175281525, -0.18559031188488007, -0.6545581221580505, 0.9255266189575195, 1.4114364385604858, 0.5449109077453613, -0.23366399109363556, -0.3791680335998535, 0.06942955404520035, 0.5466651916503906, -0.37499096989631653, -0.16305747628211975, -0.8520216345787048, -1.6006972789764404, 0.3100014328956604, -0.2129104882478714, -1.2653110027313232, -0.24795368313789368, 2.3765358924865723, 0.26302093267440796, -0.7425256371498108, -0.8799474239349365, -0.8834295868873596, 0.32478103041648865, -0.8681511878967285, -0.1150364801287651, 1.4539567232131958, -0.043736930936574936, -0.07796556502580643, 0.4998309314250946, -0.10876859724521637, 0.022319525480270386, -0.01214602217078209, 0.13719625771045685, 2.324507474899292, -0.2061641663312912, 0.11565706878900528, 0.4281642735004425, 0.533556342124939, -0.717168927192688, -0.3698282837867737, 0.17338165640830994, -2.0602710247039795, -0.1474473476409912, 1.940453290939331, 0.9264791011810303, -0.5271673202514648], [-0.6897231340408325, -0.5830920338630676, 0.20145997405052185, -0.7846155166625977, -1.7711371183395386, 0.31000587344169617, 1.1182541847229004, 1.086883783340454, -0.7504456043243408, 1.4334232807159424, -0.651477575302124, -0.9344826936721802, -0.17310580611228943, 0.4673629105091095, -0.06719019263982773, 0.5514700412750244, -1.639125943183899, 0.630312442779541, 0.31482356786727905, 0.8994787335395813, -0.7061677575111389, 0.4694851338863373, 0.7460207343101501, -0.4892318844795227, -0.5140663385391235, 0.000683021149598062, 0.6071838736534119, -0.2006111443042755, -0.011891468428075314, -0.06862117350101471, -0.5592076182365417, 0.2646755576133728, 1.2006343603134155, 0.17537415027618408, 0.5899091958999634, -0.43412891030311584, -1.0306265354156494, -0.0696835070848465, -0.3994607627391815, 0.6361874341964722, -1.4058274030685425, 1.221817135810852, 0.3429968059062958, -2.711594343185425, 0.03401408717036247, 0.026347951963543892, -0.18361976742744446, -0.12340628355741501, -0.24271725118160248, 0.5563003420829773, -0.043020956218242645, 2.251420259475708, -0.2099277377128601, -0.4090527594089508, -0.09815793484449387, -0.0021285787224769592, 0.34944385290145874, 0.32055145502090454, -0.8804589509963989, 3.2667644023895264, -0.13853143155574799, -0.03949225693941116, -0.6663004159927368, -0.3909555673599243], [0.15157660841941833, -1.1032532453536987, -0.22020256519317627, 0.026649663224816322, 1.267021656036377, 0.04680939018726349, -0.9399566650390625, 0.5289600491523743, -0.02860204502940178, -0.09796017408370972, -1.6715143918991089, 0.24160292744636536, 0.013327959924936295, 0.15807710587978363, 0.07642436772584915, 0.41583728790283203, -0.08652222901582718, -0.5465943813323975, 0.5068848729133606, -0.9469322562217712, -0.9731003046035767, 0.8878918290138245, 0.8499950766563416, 1.5487884283065796, 0.12950177490711212, 0.013090644031763077, -1.5254793167114258, 0.4435455799102783, -0.27144744992256165, 0.6566452980041504, 0.08942460268735886, 0.7972456812858582, -0.2748890221118927, 1.096555233001709, 0.2934572696685791, 1.03093683719635, 1.5526304244995117, -0.10947069525718689, -0.9308118224143982, 1.27731454372406, -3.3527019023895264, -0.47117429971694946, 0.6871157288551331, -0.615337073802948, 0.08999618142843246, 0.18940624594688416, -0.23099541664123535, -0.22033168375492096, 0.4173889756202698, 0.09955167025327682, -1.030051827430725, 0.0016336049884557724, 0.048580870032310486, 0.53355473279953, -0.40969133377075195, 0.14456051588058472, 0.5722103714942932, -0.5962321162223816, -0.4648408591747284, 0.8408471941947937, 0.45025697350502014, -1.9026081562042236, 1.3774075508117676, -0.5320045948028564], [-0.4096466600894928, 1.4678436517715454, -0.08668588101863861, 1.087612509727478, 1.4137792587280273, 0.004941907245665789, -0.1914864182472229, 0.2969917058944702, 0.453747421503067, -0.40839603543281555, -0.22816066443920135, -1.2099993228912354, 1.020973801612854, 0.10955629497766495, 0.03990604355931282, 0.22258520126342773, 0.2226548045873642, 1.0636818408966064, -0.05896937847137451, 0.5243417620658875, -0.07413985580205917, -0.561675488948822, 0.3679133951663971, 0.24222524464130402, -0.3802390396595001, -0.09418994933366776, -1.1704726219177246, 0.589120090007782, 0.029324330389499664, -0.09242211282253265, 0.6105963587760925, -0.8806143403053284, 0.4465981125831604, -0.1223476305603981, -0.06988392770290375, -1.314583420753479, -2.180974245071411, 0.1525833010673523, 0.7498229742050171, 1.014229655265808, 0.21320927143096924, -0.06793814152479172, 0.17990194261074066, 0.30968764424324036, -0.04830096662044525, -0.04082918539643288, -0.22096048295497894, -0.22996120154857635, -0.2342129349708557, -0.4606017470359802, -1.0124588012695312, 1.6824702024459839, -0.31326550245285034, -0.25924214720726013, 0.10052536427974701, 0.3722301125526428, -1.074655532836914, -0.5598487854003906, -0.29071345925331116, -2.0778870582580566, 0.4414159059524536, 0.2282581329345703, 0.33225715160369873, 0.4347774088382721], [2.11669659614563, -0.06179438531398773, 0.07278434932231903, -0.5499445199966431, 2.2907066345214844, 0.49443677067756653, 0.7326876521110535, -1.3718591928482056, 0.9057899117469788, 1.0458037853240967, 0.4701758623123169, -0.7519489526748657, -0.8563094139099121, 0.48639535903930664, 0.18502753973007202, 0.007439716719090939, -2.158196449279785, -0.797259509563446, -0.5643006563186646, 0.014011280611157417, 0.5763872265815735, -1.1165313720703125, -1.613297939300537, -0.2630743384361267, 0.3665424883365631, 0.03641605004668236, -0.3904721736907959, 0.786587119102478, 0.4613100290298462, 1.0379225015640259, 0.6240930557250977, 0.22243237495422363, 0.21470272541046143, 0.08395107835531235, 0.2097988873720169, 1.455662727355957, -1.469800591468811, 0.06973567605018616, 0.0865255668759346, -1.7769176959991455, -0.04297405108809471, 0.31853923201560974, 0.6603319644927979, 0.014688353054225445, 0.3320651650428772, 0.27491486072540283, 0.43501028418540955, -0.4657185971736908, 0.5816193222999573, 0.43997371196746826, -0.9069697856903076, -1.1224030256271362, -0.4081994891166687, -0.3068077564239502, 0.11505034565925598, 1.030131220817566, -0.35977739095687866, -1.7025868892669678, 1.001212477684021, -1.5935040712356567, 1.0963540077209473, -0.303773432970047, -0.5545395016670227, 0.1550472378730774], [0.6914204955101013, -0.2605465054512024, -0.023481331765651703, -0.5077314972877502, 0.6759545803070068, -0.1828068345785141, 0.2140781432390213, 1.1113909482955933, 0.7251795530319214, -0.590074360370636, 2.08427357673645, -1.2317893505096436, -0.6929465532302856, -0.3947075307369232, 0.02983745001256466, 0.4323897659778595, 2.8289177417755127, 0.9361041188240051, 0.19651693105697632, 0.7020012736320496, 0.8655717372894287, -0.1046314612030983, -1.0971566438674927, -1.7038689851760864, 0.3858991861343384, 0.3988582491874695, 0.6555665135383606, 0.1399366408586502, -0.3726816475391388, 0.03254702314734459, -1.7199246883392334, -0.5789661407470703, -0.5772179961204529, -0.902996301651001, 0.3811277151107788, -1.7950139045715332, 2.789095163345337, -0.04277116060256958, 0.1291024088859558, 0.4776080250740051, -0.04138728976249695, -4.143065929412842, -0.16237853467464447, 3.3287765979766846, -0.11009514331817627, -0.14342013001441956, 0.3938257396221161, 0.10896670818328857, -0.5886716246604919, -0.5933895707130432, 0.7830876708030701, -1.4516867399215698, -0.05553904548287392, -0.17978976666927338, 0.26420217752456665, 0.1706627607345581, -0.5491603016853333, 0.37160569429397583, 0.3614092171192169, 0.05336243286728859, 0.13090069591999054, 0.18766872584819794, -2.2880287170410156, 0.04807983338832855], [0.5129944086074829, 0.3186977803707123, 0.0791463628411293, 0.2025400549173355, -0.8704896569252014, 0.46704933047294617, 0.568622350692749, -1.2772272825241089, 0.7069248557090759, 0.02683991566300392, -2.4693386554718018, -2.6539957523345947, -0.13485419750213623, -0.3486706614494324, 0.2847532629966736, -0.7796676754951477, -0.17443227767944336, 1.1700680255889893, 0.23248445987701416, -1.3366692066192627, 0.724409818649292, 0.10367905348539352, -1.5458029508590698, 1.0947192907333374, 0.947944164276123, 0.251407653093338, 1.9638361930847168, 0.6174106001853943, 0.6713100671768188, 0.24289534986019135, 1.0146955251693726, -0.5381760001182556, 1.0500470399856567, -2.5350210666656494, 0.14986032247543335, 2.1383056640625, 0.7639102339744568, 0.8747962713241577, 0.29999205470085144, 0.957305371761322, 0.4082317352294922, -0.16265836358070374, 0.5610970258712769, -2.677567720413208, 0.1790059506893158, 0.25036102533340454, -0.16945023834705353, 0.41476354002952576, -0.12384872138500214, 0.6058409214019775, 0.1840796172618866, -1.4200752973556519, 0.5033441781997681, 0.22924883663654327, 0.17818652093410492, -0.33709952235221863, -1.5567407608032227, -0.14745411276817322, 0.42469266057014465, -0.24813920259475708, 0.28845101594924927, -1.3824514150619507, -0.5186237692832947, 0.7445060014724731], [1.1889722347259521, -1.2083184719085693, 0.12165376543998718, 0.2608782947063446, 1.0441322326660156, -0.3868067264556885, -0.018650256097316742, 0.04316885396838188, 0.45151087641716003, -1.1492693424224854, -2.7073934078216553, 1.5041288137435913, 1.2606768608093262, -0.08748053759336472, -0.003940497990697622, -0.7938863039016724, 0.5609498620033264, -0.28732728958129883, 0.3971570134162903, -0.5604351162910461, 0.7423310875892639, 0.07164204120635986, -0.7295470833778381, 0.7537758946418762, -0.6912847757339478, -0.02734760195016861, -0.3556797206401825, -0.5344390869140625, 1.149636149406433, 0.7291373610496521, 0.47604840993881226, 0.7864755392074585, 1.1509183645248413, -1.0420124530792236, 0.4170630872249603, -0.05524599179625511, -0.30813857913017273, 0.218293234705925, 0.0859282836318016, -0.30117708444595337, 2.6912758350372314, -0.15144897997379303, -0.2041509598493576, 1.7241926193237305, -0.5872902274131775, -0.009499497711658478, 0.34877005219459534, -0.21765375137329102, 1.2066229581832886, -0.011584269814193249, 0.4334254562854767, -2.942411184310913, 0.584281861782074, -0.09113270789384842, -0.695641279220581, 0.05959109216928482, 0.4208753705024719, -0.8097798824310303, 0.22464236617088318, -2.5750679969787598, -0.22607865929603577, -1.2268823385238647, -0.21466685831546783, 0.10351313650608063], [0.3149261474609375, -0.028170710429549217, 0.2280317097902298, -0.16820694506168365, -0.1799546480178833, 0.04697617143392563, -1.2526404857635498, 0.23985566198825836, -0.21180501580238342, -1.8721611499786377, -1.8525649309158325, 3.3172740936279297, 1.373421549797058, 0.7777889966964722, -0.24421235918998718, 0.2676793336868286, 1.672244906425476, 0.4205705523490906, -0.14502285420894623, -1.6279343366622925, -0.5711562037467957, -3.4258928298950195, 2.4851367473602295, 1.185167908668518, 0.451073557138443, -0.37301334738731384, -0.09518782794475555, -0.03388374671339989, -0.6735336184501648, 0.46870312094688416, -1.0401540994644165, 0.3498333990573883, -0.6212334036827087, -0.3766525983810425, -0.36986491084098816, -0.5625316500663757, 0.588302731513977, -0.3532770574092865, -0.18863710761070251, 0.5393497943878174, 0.8750748038291931, -1.2373566627502441, 0.5280843377113342, -2.9383656978607178, -0.7601001858711243, 0.0023857690393924713, 0.11818908900022507, 0.1140042245388031, 0.949828028678894, -0.009419918060302734, -0.3495035171508789, 0.6343479156494141, 0.7756180763244629, 0.324558824300766, 0.6845560073852539, 0.1029217392206192, 0.6615598797798157, 0.38066941499710083, 0.22384952008724213, 0.8187697529792786, -0.19050724804401398, -1.3067123889923096, 1.1066452264785767, 0.03225821256637573]], "dtype": "float32"}, {"__nd__": [-3.232149124145508, -9.73578929901123, -11.330177307128906, -6.813002109527588, 9.150886535644531, 4.301830291748047, 0.1105230450630188, 4.10883092880249, 3.746734857559204, 6.7778778076171875, 2.355398178100586, -8.343978881835938, -22.453235626220703, -14.583157539367676, 15.794702529907227, -15.078067779541016, 0.9040567874908447, -0.6843499541282654, -1.2487908601760864, 4.452037334442139, 14.039796829223633, 18.43507957458496, 5.1168928146362305, -6.268769264221191, 2.3154053688049316, -4.876611232757568, 7.4178690910339355, 13.936201095581055, -11.898320198059082, -5.651806831359863, 0.9419501423835754, -1.99897301197052, -13.390667915344238, -15.935830116271973, -14.812530517578125, 5.626122951507568, 6.318438529968262, -13.058571815490723, 25.356550216674805, -3.521663188934326, -2.3368327617645264, -8.831690788269043, -1.1690081357955933, -2.1713693141937256, -2.6261746883392334, 14.849700927734375, 10.694581985473633, -2.5246424674987793, 12.044888496398926, 5.127988815307617, 16.840787887573242, -7.301268100738525, -19.687362670898438, -11.21673583984375, -9.736666679382324, -16.461978912353516, 6.579482555389404, -2.1489980220794678, -14.865242958068848, 8.83786392211914, 13.913656234741211, -1.1257823705673218, 6.815154075622559, -8.214492797851562], "dtype": "float32"}, {"__nd__": [-10.042717933654785, -4.314990043640137, 13.247325897216797, 18.169052124023438, -0.6334586143493652, 17.850099563598633, 8.973774909973145, -16.65108871459961, -8.019332885742188, 8.747055053710938, 2.438148021697998, -3.767655611038208, 5.255278587341309, 5.881141662597656, 9.443489074707031, 5.633002281188965, 7.85603141784668, -13.711409568786621, -5.079843997955322, 2.425062417984009, 14.301046371459961, 6.144451141357422, 3.0505237579345703, 17.590730667114258, 5.142502784729004, -16.202112197875977, 9.958539009094238, -9.804447174072266, 4.967775344848633, 4.544595718383789, 5.757635593414307, -5.002596855163574, 0.9610780477523804, 0.4494593143463135, 6.244607925415039, 17.63231086730957, 10.816354751586914, 9.350515365600586, 0.049793291836977005, -10.434926986694336, 7.9613356590271, -11.697539329528809, -1.6505136489868164, 11.0586519241333, 9.685624122619629, 10.282404899597168, 2.7611591815948486, 3.0488009452819824, -16.849437713623047, 6.154178142547607, -13.70361614227295, 1.0752840042114258, -1.9283186197280884, 20.93246078491211, 5.077756404876709, -0.9063835740089417, -15.076342582702637, 17.547016143798828, -12.577510833740234, 3.3641679286956787, -20.19097137451172, -2.5818440914154053, -12.822949409484863, -4.511178493499756], "dtype": "float32"}]}}, {"a": [{"__nd__": [[46.792274475097656, 44.70256423950195, 49.006553649902344, 51.68178176879883, 54.544185638427734, 50.43882369995117, 47.78940963745117, 46.860877990722656, 52.9949836730957, 56.53913116455078, 51.0910758972168, 45.06668472290039, 46.16693878173828, 56.40007781982422, 50.81153106689453, 43.07146072387695, 49.66521453857422, 45.34709548950195, 47.48284912109375, 48.047977447509766, 47.14674758911133, 52.2135124206543, 49.747657775878906, 47.642276763916016, 51.6385498046875, 53.319419860839844, 43.42790603637695, 48.973079681396484, 46.07701110839844, 49.30738067626953, 44.842323303222656, 50.082759857177734, 49.84845733642578, 48.782649993896484, 45.808292388916016, 48.41523742675781, 45.63468551635742, 44.579166412353516, 50.89914321899414, 45.562599182128906, 54.681182861328125, 52.86635208129883, 42.008731842041016, 51.08851623535156, 45.59313201904297, 50.13222885131836, 50.17452621459961, 42.04627990722656, 49.06631088256836, 48.97684097290039, 53.84800338745117, 45.27421188354492, 52.95216751098633, 45.60411071777344, 48.674835205078125, 46.63810729980469, 55.794925689697266, 52.27285385131836, 59.7269287109375, 52.567665100097656, 53.37997055053711, 53.36273193359375, 47.57355499267578, 49.719886779785156, 55.401554107666016, 48.413795471191406, 50.75519943237305, 49.91510772705078, 52.436866760253906, 48.54036331176758, 49.39055252075195, 50.96952438354492, 50.41209411621094, 46.54010772705078, 53.583133697509766, 44.80607604980469, 45.19553756713867, 44.870033264160156, 53.867889404296875, 48.55756759643555, 46.1158561706543, 45.45591354370117, 51.68452453613281, 45.780635833740234, 44.911685943603516, 52.455970764160156, 45.21316909790039, 48.71024703979492, 49.97295379638672, 48.218658447265625, 49.78362274169922, 55.35509490966797, 47.93242263793945, 44.962772369384766, 42.653018951416016, 49.18093490600586, 48.590972900390625, 51.06036376953125, 48.14302062988281, 48.085445404052734, 47.11473846435547, 47.92095947265625, 50.640907287597656, 48.47858810424805, 50.40176773071289, 57.6047477722168, 51.916473388671875, 43.695640563964844, 56.934120178222656, 51.39124298095703, 46.23434829711914, 53.628196716308594, 50.070621490478516, 47.539127349853516, 47.465965270996094, 46.02627182006836, 50.192474365234375, 54.2752685546875, 48.699790954589844, 51.68329620361328, 57.50258255004883, 45.14163589477539, 51.03009033203125, 48.774253845214844, 45.76297378540039, 45.896610260009766, 49.93885803222656, 51.73545455932617, 47.87364196777344, 50.21756362915039, 52.88883590698242, 51.18624496459961, 53.47468185424805, 51.8455810546875, 51.952022552490234, 57.30653762817383, 52.4926872253418, 50.51788330078125, 52.83977508544922, 46.32350540161133, 48.6491813659668, 53.172882080078125, 52.52314758300781, 56.19344711303711, 50.041893005371094, 44.15068817138672, 57.78898620605469, 54.3715705871582, 45.76504898071289, 55.5032958984375, 50.12961196899414, 42.744285583496094, 48.31858444213867, 47.964385986328125, 56.3633918762207, 46.83172607421875, 48.985530853271484, 48.8929557800293, 48.484107971191406, 46.34370422363281, 50.876346588134766, 54.307220458984375, 52.49591064453125, 46.29000473022461, 45.40073776245117, 50.475894927978516, 47.17367935180664, 47.47933578491211, 43.28122329711914, 57.80196762084961, 53.66647720336914, 46.1043701171875, 53.63283920288086, 55.36994934082031, 40.44178771972656, 47.804222106933594, 48.4481086730957, 52.593017578125, 49.514137268066406, 49.07827377319336, 49.76651382446289, 57.41330337524414, 58.639923095703125, 47.90069580078125, 46.29502487182617, 60.77021408081055, 46.08102035522461, 47.706390380859375, 50.146324157714844, 51.9329833984375, 54.11614227294922, 51.563907623291016, 46.510467529296875, 52.02552795410156, 50.999671936035156, 57.50767517089844, 49.94000244140625, 44.65210723876953, 45.81991958618164, 55.80061340332031, 47.839473724365234, 41.58213424682617, 47.67720031738281, 50.00006103515625, 54.75532150268555, 45.9421272277832, 52.66673278808594, 53.18119812011719, 47.20244598388672, 49.24964141845703, 57.07780075073242, 56.88193893432617, 53.422088623046875, 51.32778549194336, 54.55323791503906, 49.43729019165039, 49.61962890625, 46.55855941772461, 50.022254943847656, 49.671932220458984, 61.09452819824219, 49.2286262512207, 55.08327102661133, 55.28199005126953, 49.254085540771484, 54.677589416503906, 41.29274368286133, 50.37970733642578, 53.429046630859375, 40.408538818359375, 45.36331558227539, 54.22429275512695, 48.97195816040039, 45.60720443725586, 48.505619049072266, 47.829647064208984, 52.897789001464844, 51.81554412841797, 48.879085540771484, 47.33863067626953, 49.78449630737305, 55.30062484741211, 51.39610290527344, 52.555782318115234, 49.397274017333984, 44.832733154296875, 47.1566276550293, 52.60494613647461, 50.412864685058594, 46.384639739990234, 43.52851867675781, 50.442726135253906, 48.641876220703125, 48.597877502441406, 48.68220138549805, 51.1356201171875, 51.177791595458984, 53.848514556884766, 53.713687896728516, 55.328372955322266, 53.2254524230957, 48.65756607055664, 49.78292465209961, 51.97686004638672, 48.75883102416992, 52.19775390625, 54.614437103271484, 46.79501724243164, 42.91846466064453, 54.774620056152344, 50.071041107177734, 52.834678649902344, 44.670780181884766, 56.931968688964844, 52.9544792175293, 49.495704650878906, 48.59204864501953, 50.45558166503906, 49.33307647705078, 48.88523864746094, 45.52159881591797, 55.051239013671875, 49.84169006347656, 50.29854202270508, 50.37166213989258, 40.56713104248047, 52.10856628417969, 48.683380126953125, 48.84308624267578, 50.279151916503906, 46.1100959777832, 53.3735237121582, 47.2165412902832, 46.38536834716797, 46.55124282836914, 50.635009765625, 52.983116149902344, 47.61978530883789, 43.959251403808594, 54.249961853027344, 45.73966598510742, 54.779319763183594, 49.78281784057617, 47.206600189208984, 47.43983840942383, 51.84746551513672, 53.55748748779297, 49.20731735229492, 53.78773880004883, 58.3479118347168, 49.16676712036133, 45.40221405029297, 45.78239822387695, 44.43865966796875, 47.44133758544922, 47.104881286621094, 44.6995849609375, 52.12073516845703, 39.55464553833008, 44.71680450439453, 58.06100845336914, 54.15121078491211, 52.09116744995117, 51.829036712646484, 57.08374786376953, 52.14251708984375, 50.310306549072266, 45.07936096191406, 49.7053337097168, 47.0644645690918, 47.17530059814453, 47.143043518066406, 49.8214111328125, 48.681156158447266, 53.48444366455078, 44.695274353027344, 49.322120666503906, 57.43798828125, 50.15559005737305, 54.79446029663086, 54.547454833984375, 55.2665901184082, 52.952476501464844, 44.478214263916016, 48.930450439453125, 48.47865676879883, 43.80458068847656, 48.144474029541016, 44.99967956542969, 43.33779525756836, 50.84358596801758, 49.82563400268555, 47.248741149902344, 46.197654724121094, 47.83490753173828, 48.42363739013672, 48.367557525634766, 49.78916931152344, 48.25783920288086, 51.547889709472656, 52.09148406982422, 51.431480407714844, 47.34758758544922, 47.533973693847656, 51.73945617675781, 49.211830139160156, 49.827392578125, 40.032020568847656, 39.1995849609375, 51.348594665527344, 51.318233489990234, 55.573726654052734, 52.88465118408203, 50.88695526123047, 50.06544876098633, 44.29303741455078, 53.5327033996582, 47.585655212402344, 52.00993728637695, 50.800167083740234, 51.08838653564453, 53.570701599121094, 48.04053497314453, 48.45105743408203, 46.57655334472656, 48.491390228271484, 56.663761138916016, 48.54672622680664, 54.74246597290039, 53.304046630859375, 52.9171257019043, 53.212059020996094, 45.942893981933594, 51.49378967285156, 56.3536491394043, 47.21681594848633, 49.81157302856445, 47.24336242675781, 51.26804733276367, 45.68587875366211, 56.438720703125, 48.49266052246094, 43.82935333251953, 57.16316223144531, 53.8820915222168, 52.07404708862305, 52.13035202026367, 48.568809509277344, 49.10681915283203, 52.89607620239258, 53.670501708984375, 53.73044967651367, 50.85020446777344, 48.13444900512695, 49.231651306152344, 54.43840026855469, 55.759674072265625, 48.63475036621094, 48.090782165527344, 54.99020004272461, 52.894596099853516, 52.789031982421875, 44.75066375732422, 48.50497055053711, 47.94001770019531, 50.83161926269531, 43.260231018066406, 55.08700942993164, 51.20052719116211, 51.739349365234375, 45.2634391784668, 49.894317626953125, 55.338600158691406, 43.72121810913086, 51.22307205200195, 52.79521942138672, 47.23221206665039, 53.25896453857422, 48.679725646972656, 54.9190673828125, 52.11004638671875, 50.897464752197266, 58.03566360473633, 53.32758712768555, 55.23454284667969, 45.241920471191406, 47.413570404052734, 53.346275329589844, 53.6224479675293, 49.77703857421875, 53.1189079284668, 45.76658630371094, 50.85533905029297, 45.410865783691406, 51.362281799316406, 48.05531311035156, 47.43600845336914, 50.225196838378906, 57.269840240478516, 45.95295333862305, 47.59299850463867, 54.27065658569336, 48.30573272705078, 57.077056884765625, 50.89011764526367, 50.362525939941406, 51.42753601074219, 56.964717864990234, 52.62144470214844, 52.06684875488281, 51.00743103027344, 43.75032424926758, 46.45891189575195, 55.93337631225586, 53.287513732910156, 52.02425765991211, 52.54825210571289, 47.825626373291016, 51.35185623168945, 46.396820068359375, 51.56544876098633, 47.14345932006836, 49.14472198486328, 51.81365966796875, 48.692848205566406, 52.590328216552734, 51.824153900146484, 51.43783187866211, 55.297645568847656, 44.104461669921875, 43.614871978759766, 52.4649658203125, 53.76317596435547, 58.04656219482422, 53.025028228759766, 53.32086181640625, 46.46767807006836, 53.0104866027832, 52.69535446166992, 45.77653884887695, 46.414730072021484, 49.55587387084961, 51.15301513671875, 38.22085952758789, 50.28122329711914, 50.92341613769531, 49.29265594482422, 51.067298889160156, 51.82362365722656, 45.95344924926758, 51.66472244262695, 48.99477005004883, 46.21648406982422, 50.83656311035156, 55.95057678222656, 44.15953826904297, 43.3875617980957, 43.38733673095703, 57.90924835205078, 48.9936637878418, 49.762054443359375, 46.85369873046875, 46.23820877075195, 41.89255905151367, 50.68162536621094, 51.038238525390625, 42.09386444091797, 45.8274040222168, 47.02354431152344, 49.89598846435547, 50.97651672363281, 54.503170013427734, 55.45782470703125, 54.27381896972656, 55.36611557006836, 45.33433151245117, 47.78203201293945, 54.574440002441406, 51.15464401245117, 50.074886322021484, 42.084049224853516, 42.563026428222656, 52.92355728149414, 51.36630630493164, 48.81576919555664, 48.348114013671875, 43.09558868408203, 51.484859466552734, 54.1376838684082, 56.56868362426758, 50.19655227661133, 49.18321228027344, 45.61601257324219, 54.892974853515625, 56.827423095703125, 47.54732131958008, 51.16303634643555, 50.36267852783203, 47.704689025878906, 40.01020050048828, 48.53600311279297, 52.42977523803711, 44.22679138183594, 46.644474029541016, 36.86575698852539, 55.13811111450195, 52.00110626220703, 50.181915283203125, 48.91810607910156, 47.45086669921875, 49.684776306152344, 48.75873947143555, 49.00657272338867, 49.86475372314453, 50.50260543823242, 56.19591522216797, 44.2022705078125, 51.047874450683594, 47.50175857543945, 45.442073822021484, 40.52731704711914, 52.42757034301758, 46.038482666015625, 46.96125793457031, 51.839515686035156, 45.62610626220703, 50.06418228149414, 49.66047668457031, 53.16737365722656, 54.0025634765625, 46.492591857910156, 53.62871170043945, 49.57301712036133, 50.45119094848633, 49.109947204589844, 53.062156677246094, 45.11119842529297, 51.000335693359375, 54.66312026977539, 48.825321197509766, 58.07109069824219, 47.01637649536133, 50.81478500366211, 48.02956771850586, 47.48500061035156, 45.496917724609375, 47.79973602294922, 49.75105285644531, 46.74761199951172, 53.48958206176758, 56.04130554199219, 43.84441375732422, 53.299373626708984, 50.712242126464844, 50.93328094482422, 59.801998138427734, 54.77729797363281, 51.2972412109375, 50.69245529174805, 41.33713150024414, 53.08956527709961, 47.005252838134766, 50.70293045043945, 50.39052200317383, 53.841407775878906, 58.122013092041016, 53.934410095214844, 55.091793060302734, 49.06182861328125, 51.82309341430664, 53.076995849609375, 44.241485595703125, 58.16617202758789, 47.140525817871094, 44.889827728271484, 56.53617858886719, 58.4737548828125, 47.18757629394531, 47.54676055908203, 62.44765090942383, 50.450469970703125, 42.06376647949219, 47.550926208496094, 58.00847625732422, 52.636497497558594, 53.00839614868164, 54.898231506347656, 52.7031135559082, 48.66718292236328, 43.62275314331055, 51.785484313964844, 45.53586196899414, 49.775047302246094, 57.035884857177734, 52.162601470947266, 50.23796463012695, 50.26272201538086, 52.92317199707031, 53.6849365234375, 44.73274612426758, 50.74191665649414, 56.862586975097656, 53.01747512817383, 46.93949508666992, 47.281490325927734, 41.75231170654297, 46.82648468017578, 56.427860260009766, 49.72190475463867, 46.5799560546875, 47.695518493652344, 49.5570182800293, 49.573814392089844, 48.5841064453125, 46.923622131347656, 52.54627990722656, 56.154136657714844, 47.728172302246094, 47.83491134643555, 54.75741195678711, 44.14502716064453, 47.634422302246094, 48.48760986328125, 50.67112731933594, 48.900245666503906, 46.45358657836914, 51.242027282714844, 45.2491569519043, 43.029945373535156, 41.88697052001953, 52.99479293823242, 43.05257797241211, 51.788230895996094, 52.68283462524414, 53.227596282958984, 49.30126953125, 47.80720138549805, 50.34268569946289, 47.62506103515625, 44.85519027709961, 55.021759033203125, 46.42224884033203, 57.183555603027344, 52.059059143066406, 48.1817741394043, 51.75059127807617, 54.920597076416016, 44.486122131347656, 46.29290771484375, 45.80292892456055, 50.660125732421875, 51.61700439453125, 50.115875244140625, 47.087162017822266, 47.88619613647461, 54.748172760009766, 54.467769622802734, 50.525421142578125, 44.757564544677734, 52.512508392333984, 51.150638580322266, 51.010921478271484, 56.920127868652344, 53.28215789794922, 46.10205841064453, 46.14095687866211, 55.52125549316406, 50.8228645324707, 53.54823303222656, 49.57440185546875, 46.2020378112793, 51.29680633544922, 56.47566223144531, 51.59654235839844, 45.73773956298828, 49.016563415527344, 48.28635025024414, 50.989376068115234, 54.392215728759766, 58.217159271240234, 51.257965087890625, 48.654052734375, 50.68639373779297, 50.44346618652344, 48.68081283569336, 50.47441482543945, 50.38116455078125, 45.67792510986328, 52.57529830932617, 48.933189392089844, 46.44902420043945, 52.34393310546875, 48.43362808227539, 57.35301971435547, 52.283626556396484, 50.67454528808594, 47.33689498901367, 48.726436614990234, 53.74211502075195, 47.590057373046875, 53.37456512451172, 41.92814254760742, 51.12181091308594, 53.581207275390625, 51.66517639160156, 50.29096221923828, 54.345359802246094, 46.882835388183594, 46.368324279785156, 50.720855712890625, 55.110496520996094, 52.026004791259766, 49.5224723815918, 51.50028610229492, 52.06864929199219, 45.31816864013672, 39.05159378051758, 46.19755935668945, 44.32127380371094, 47.979923248291016, 45.48993682861328, 46.834938049316406, 52.03511047363281, 50.69404220581055, 50.21348190307617, 56.96041488647461, 48.48588180541992, 48.81247329711914, 55.40460205078125, 45.89860153198242, 43.829689025878906, 55.589317321777344, 51.71390914916992, 52.775787353515625, 52.92961120605469, 54.002052307128906, 48.50631332397461, 50.099063873291016, 45.95763397216797, 50.9715690612793, 51.349815368652344, 48.61445236206055, 53.10628128051758, 62.25849533081055, 45.841712951660156, 53.779823303222656, 54.24361801147461, 54.746280670166016, 47.27400588989258, 55.285614013671875, 53.12591552734375, 50.01902389526367, 49.15895462036133, 51.52075958251953, 54.28609848022461, 54.09669876098633, 59.731441497802734, 49.17412567138672, 53.07992935180664, 48.890220642089844, 53.74482727050781, 51.145259857177734, 48.230533599853516, 45.317073822021484, 47.547489166259766, 51.588748931884766, 56.50787353515625, 53.72322082519531, 45.83452224731445, 48.665550231933594, 50.73625946044922, 50.04530334472656, 50.760704040527344, 53.56521987915039, 51.255470275878906, 47.59536361694336, 42.62125015258789, 48.937232971191406, 43.54902648925781, 49.58214569091797, 51.4537353515625, 44.800498962402344, 49.53660583496094, 48.69133377075195, 57.62431716918945, 51.79507064819336, 42.3968620300293, 51.68186569213867, 51.54519271850586, 49.06026840209961, 48.34494400024414, 52.21786880493164, 52.45540237426758, 51.52812194824219, 45.7914924621582, 48.74309158325195, 43.49816131591797, 43.73244094848633, 47.4608268737793, 45.46967315673828, 48.01755905151367, 59.57109069824219, 51.610172271728516, 46.076393127441406, 40.95416259765625, 46.22866439819336, 43.8265495300293, 40.75387954711914, 52.4201774597168, 47.58981704711914, 51.265804290771484, 61.864952087402344, 50.5863037109375, 49.9703254699707, 50.896663665771484, 48.43400955200195, 46.19198989868164, 48.6500244140625, 54.07293701171875, 48.426536560058594, 44.031803131103516, 44.52577209472656, 45.401023864746094, 49.571163177490234, 51.462833404541016, 51.47335433959961, 54.32427978515625, 48.77338790893555, 52.95768737792969, 50.947391510009766, 52.497779846191406, 42.337093353271484, 54.716514587402344, 43.57621765136719, 45.27516555786133, 52.91415786743164, 48.27338790893555, 56.915809631347656, 50.50821304321289, 45.770896911621094, 50.513572692871094, 48.296024322509766, 56.12021255493164, 56.43672180175781, 52.88051223754883, 48.31000518798828, 50.277259826660156, 54.14103698730469, 46.74538040161133, 46.77307891845703, 53.354557037353516, 52.903968811035156, 49.03194808959961, 48.88693618774414, 42.15549850463867, 54.69459533691406, 49.49558639526367, 44.25491714477539, 45.45991516113281, 46.39879608154297, 56.78784942626953, 52.740230560302734, 44.63423156738281, 57.40513610839844, 55.34071731567383, 45.755035400390625, 39.35603332519531, 53.84599304199219, 47.58888626098633, 49.45486068725586, 47.900367736816406, 55.03715133666992, 55.9159049987793, 53.24047088623047, 49.39745330810547, 51.0473747253418, 52.48472595214844, 43.701454162597656, 54.97352600097656, 48.72520065307617, 51.052284240722656, 50.56000900268555, 53.63326644897461, 54.285926818847656, 58.85279846191406, 48.610084533691406, 46.41379165649414, 50.57925033569336, 47.43269348144531, 53.50041198730469, 46.83631134033203, 50.6014518737793, 53.835296630859375, 54.13731002807617, 50.905723571777344, 55.73654556274414, 55.243534088134766, 43.300025939941406, 51.54126739501953, 50.83057403564453, 48.9207763671875, 47.57090759277344, 50.30031204223633, 51.674957275390625, 51.234619140625, 55.59954833984375, 54.112457275390625, 47.786964416503906, 47.11008071899414, 53.53691864013672, 47.36019515991211, 50.81514358520508, 52.054691314697266, 54.27138900756836, 52.219482421875, 55.93452453613281, 48.32035446166992, 52.84016418457031, 47.52832794189453, 50.90743637084961, 50.657718658447266, 46.448360443115234, 39.74688720703125, 53.98836898803711, 51.61691665649414, 50.72896194458008, 47.36342239379883, 48.83415985107422, 51.9876708984375, 50.2462272644043, 54.803375244140625, 50.63280487060547, 46.87860870361328, 52.60249328613281, 47.62567901611328, 51.021995544433594, 42.91108322143555, 48.74977111816406, 52.37205123901367, 47.53287124633789], [50.022247314453125, 52.48598861694336, 50.392494201660156, 57.26289749145508, 49.08112335205078, 49.39750671386719, 49.471282958984375, 42.206085205078125, 43.976844787597656, 52.24250411987305, 48.7928581237793, 55.66462326049805, 49.79644775390625, 49.42829895019531, 54.22315979003906, 51.214046478271484, 56.52360534667969, 45.88040542602539, 51.8691520690918, 52.82958221435547, 52.33795166015625, 54.725868225097656, 54.24281311035156, 50.28029251098633, 47.91320037841797, 54.07791519165039, 50.37186813354492, 54.0247688293457, 55.61752700805664, 51.033992767333984, 46.6434440612793, 49.24519729614258, 53.43107223510742, 49.34514617919922, 55.02143096923828, 50.488128662109375, 50.87358474731445, 47.85301971435547, 49.96745681762695, 54.74509811401367, 48.996864318847656, 49.02740478515625, 47.23638153076172, 53.46492385864258, 44.247718811035156, 47.70737075805664, 53.949913024902344, 55.989261627197266, 49.40142822265625, 54.13593292236328, 42.34004211425781, 53.295326232910156, 51.42911148071289, 45.362850189208984, 43.132957458496094, 48.22592544555664, 51.28065490722656, 51.71421432495117, 50.68131637573242, 53.32011795043945, 53.03677749633789, 44.69938278198242, 48.697303771972656, 48.1812744140625, 54.330265045166016, 48.252769470214844, 54.24318313598633, 50.587520599365234, 59.60186767578125, 50.02298355102539, 50.238067626953125, 49.772891998291016, 50.80728530883789, 59.93151092529297, 48.201663970947266, 46.72710037231445, 49.42006301879883, 52.894500732421875, 46.252288818359375, 53.38423156738281, 53.250823974609375, 49.53254699707031, 49.18022155761719, 47.03339767456055, 49.08457946777344, 46.98054504394531, 55.14578628540039, 56.67725372314453, 49.03352355957031, 50.001609802246094, 42.56888198852539, 48.04283142089844, 51.5088005065918, 51.241172790527344, 50.639041900634766, 53.52399444580078, 50.50605773925781, 48.5888786315918, 56.53719711303711, 42.47040939331055, 45.50758743286133, 47.17580795288086, 47.20439910888672, 47.018123626708984, 49.20546340942383, 47.249488830566406, 49.8616943359375, 52.438568115234375, 50.18709182739258, 50.5991325378418, 46.3289680480957, 45.52619934082031, 48.888248443603516, 53.74462127685547, 51.547935485839844, 54.38084411621094, 54.00458526611328, 54.52067947387695, 46.82687759399414, 54.978179931640625, 53.06153869628906, 58.30912399291992, 52.390838623046875, 51.34184265136719, 49.530269622802734, 50.70195007324219, 43.469825744628906, 49.05264663696289, 49.89573669433594, 53.98610305786133, 47.437828063964844, 50.5748405456543, 57.310203552246094, 47.94387435913086, 53.399505615234375, 47.927730560302734, 49.65129852294922, 51.87995147705078, 53.55685043334961, 48.09288024902344, 45.991607666015625, 56.07544708251953, 50.791961669921875, 53.20104217529297, 53.67431640625, 53.05723571777344, 49.0153694152832, 49.512367248535156, 43.89011764526367, 50.76533889770508, 48.19209289550781, 49.961578369140625, 57.45500564575195, 52.775001525878906, 50.26188659667969, 44.66902160644531, 52.934844970703125, 56.07261276245117, 46.79813003540039, 51.73029708862305, 48.007713317871094, 49.453487396240234, 48.6082878112793, 43.69410705566406, 48.027713775634766, 43.65898132324219, 55.51290512084961, 54.155555725097656, 50.220272064208984, 51.14710998535156, 50.27491760253906, 50.830467224121094, 50.81962203979492, 49.10563659667969, 49.392494201660156, 44.758270263671875, 46.10786056518555, 47.664520263671875, 50.8657112121582, 48.586612701416016, 57.54353713989258, 49.305023193359375, 48.648582458496094, 51.75175094604492, 54.098182678222656, 57.33354568481445, 53.883522033691406, 47.74644470214844, 49.39193344116211, 54.18708038330078, 51.07600021362305, 53.20582580566406, 49.77659606933594, 56.3529167175293, 55.750675201416016, 54.49260711669922, 50.697322845458984, 43.771881103515625, 44.80369186401367, 51.36958694458008, 52.65528106689453, 38.69514083862305, 48.243568420410156, 54.60879898071289, 53.83995056152344, 48.16062545776367, 52.60026931762695, 50.22861862182617, 49.257667541503906, 54.493927001953125, 54.572120666503906, 48.0660514831543, 46.424625396728516, 45.00898742675781, 44.564666748046875, 55.339603424072266, 57.45930480957031, 50.96195983886719, 54.25168991088867, 47.46823501586914, 49.873016357421875, 53.725257873535156, 51.37991714477539, 50.01014709472656, 43.65497970581055, 44.47685623168945, 48.53451156616211, 47.991119384765625, 49.8641357421875, 47.66305160522461, 49.30378341674805, 50.39064407348633, 52.164085388183594, 50.409141540527344, 48.011863708496094, 49.065494537353516, 53.677127838134766, 49.34368133544922, 46.375675201416016, 46.215824127197266, 51.244632720947266, 52.31876754760742, 54.90538787841797, 48.88138961791992, 55.49414825439453, 48.77608108520508, 49.221370697021484, 47.91291427612305, 52.164608001708984, 54.200599670410156, 50.019981384277344, 48.1657600402832, 44.10828399658203, 50.68851852416992, 48.774559020996094, 49.323883056640625, 47.64996337890625, 46.69769287109375, 48.42946243286133, 51.838199615478516, 53.81859588623047, 50.04513931274414, 51.3442268371582, 49.3198356628418, 49.60068130493164, 44.77159881591797, 53.51192855834961, 52.39704132080078, 55.73772430419922, 47.7017707824707, 50.578609466552734, 46.496219635009766, 51.01583480834961, 49.60409164428711, 52.706993103027344, 49.51839065551758, 52.579933166503906, 48.1814079284668, 52.392616271972656, 48.9134521484375, 48.39225769042969, 50.428443908691406, 51.515968322753906, 44.622467041015625, 44.2112922668457, 50.82274627685547, 49.87158966064453, 42.21200942993164, 43.49065399169922, 47.11676025390625, 51.33361053466797, 50.88566589355469, 53.99932098388672, 54.63713836669922, 59.280128479003906, 44.51055908203125, 49.94485855102539, 54.22887420654297, 45.234130859375, 44.63743591308594, 50.86726760864258, 54.35075378417969, 54.96261978149414, 43.2087287902832, 45.39274215698242, 49.730567932128906, 48.052886962890625, 51.357826232910156, 50.62702178955078, 56.94364547729492, 49.86623001098633, 50.819889068603516, 50.40785217285156, 54.40657424926758, 45.801692962646484, 51.615230560302734, 47.12132263183594, 44.61865997314453, 54.70514678955078, 52.17073440551758, 55.56187438964844, 51.13106155395508, 52.66844177246094, 50.677494049072266, 52.36699295043945, 48.55636215209961, 56.80641174316406, 46.71921157836914, 56.82304000854492, 49.40565490722656, 53.56870651245117, 50.72322082519531, 51.929141998291016, 55.99612045288086, 50.75495529174805, 46.520416259765625, 51.13920211791992, 52.739707946777344, 40.625301361083984, 54.02207946777344, 54.47442626953125, 43.29692459106445, 49.60855484008789, 53.42509078979492, 45.68291473388672, 48.5648078918457, 48.88199234008789, 52.40183639526367, 54.57445526123047, 50.091800689697266, 50.46147155761719, 53.847774505615234, 52.34144592285156, 49.784202575683594, 55.434173583984375, 44.65882873535156, 50.80826187133789, 48.558380126953125, 48.23000717163086, 50.985416412353516, 50.358177185058594, 59.867610931396484, 52.737449645996094, 51.58784103393555, 56.69856262207031, 54.19749450683594, 42.87493896484375, 50.681800842285156, 53.1274528503418, 47.81663131713867, 51.359527587890625, 48.39362716674805, 51.66458511352539, 43.76972198486328, 42.50299835205078, 48.01067352294922, 51.05968475341797, 55.412559509277344, 49.070472717285156, 47.207576751708984, 53.768089294433594, 50.29121017456055, 57.26167297363281, 49.79704284667969, 42.46192932128906, 46.9952278137207, 46.37955856323242, 44.64034652709961, 47.070045471191406, 47.40511703491211, 48.997947692871094, 47.953346252441406, 51.72614288330078, 51.743595123291016, 50.808719635009766, 50.88883972167969, 42.66303253173828, 43.51396179199219, 46.624996185302734, 51.106937408447266, 44.3137092590332, 48.43440628051758, 51.078895568847656, 57.94683074951172, 51.1270751953125, 47.976295471191406, 54.409603118896484, 50.89735794067383, 48.89723205566406, 54.47932815551758, 46.4372444152832, 50.82173538208008, 50.375, 53.62898254394531, 45.50314712524414, 44.9084358215332, 48.486595153808594, 48.93443298339844, 50.182498931884766, 48.52491760253906, 51.7164306640625, 52.73838424682617, 50.884822845458984, 45.81422805786133, 54.56471252441406, 43.903682708740234, 49.164188385009766, 53.50961685180664, 47.52721405029297, 55.09877014160156, 57.56312942504883, 56.22074890136719, 51.69245147705078, 53.86005401611328, 47.92644119262695, 51.55327606201172, 51.772430419921875, 50.33399963378906, 50.57759094238281, 49.139408111572266, 42.16858673095703, 51.552574157714844, 51.20884323120117, 58.11722183227539, 52.790863037109375, 50.2324333190918, 53.437904357910156, 52.439579010009766, 49.89776611328125, 48.450599670410156, 44.12800216674805, 50.45361328125, 56.42971420288086, 42.54538345336914, 46.1854248046875, 52.973114013671875, 50.734397888183594, 51.63751220703125, 42.568904876708984, 47.621639251708984, 51.9781608581543, 52.31159591674805, 51.14324951171875, 46.45566177368164, 52.11613082885742, 45.21273422241211, 46.22384262084961, 45.32908630371094, 55.70388412475586, 48.3223876953125, 50.184017181396484, 41.206302642822266, 54.325218200683594, 51.808746337890625, 43.3679084777832, 43.56071090698242, 56.57168960571289, 46.195213317871094, 57.29615783691406, 48.08839416503906, 40.05665588378906, 54.37336730957031, 51.957942962646484, 52.67267990112305, 47.92436218261719, 49.09867858886719, 53.76858901977539, 46.66809844970703, 42.099082946777344, 53.56938552856445, 47.47100830078125, 55.03163528442383, 48.01748275756836, 48.23698425292969, 48.93550109863281, 40.53474044799805, 54.634559631347656, 47.427005767822266, 49.157108306884766, 55.78656005859375, 50.39985656738281, 53.51629638671875, 51.28097152709961, 53.02683639526367, 51.04917907714844, 54.55517578125, 49.10344314575195, 51.78605651855469, 47.114315032958984, 54.68549346923828, 41.22499084472656, 50.691017150878906, 51.391624450683594, 51.316165924072266, 49.92083740234375, 55.304222106933594, 58.40571594238281, 47.57041549682617, 42.31964874267578, 45.6032829284668, 46.70246887207031, 49.4075927734375, 49.72945022583008, 54.26185989379883, 48.2427864074707, 49.801483154296875, 50.38853454589844, 46.437278747558594, 48.67865753173828, 60.97036361694336, 49.5794677734375, 46.09872817993164, 41.23921585083008, 54.09649658203125, 48.6574821472168, 47.410186767578125, 59.72337341308594, 55.67311096191406, 43.09867858886719, 48.5543098449707, 53.935333251953125, 53.28300857543945, 45.491600036621094, 57.048377990722656, 57.34322738647461, 45.50893783569336, 42.87868118286133, 44.37724685668945, 52.603126525878906, 54.248741149902344, 50.66967010498047, 47.54814147949219, 49.555397033691406, 52.1810188293457, 45.06929397583008, 43.13265609741211, 49.577545166015625, 54.853233337402344, 50.67936325073242, 50.690528869628906, 47.215030670166016, 53.59098815917969, 43.07919692993164, 49.646278381347656, 47.53068542480469, 49.285301208496094, 46.19123840332031, 52.890499114990234, 49.58180236816406, 47.50213623046875, 51.92182922363281, 49.673622131347656, 47.79957962036133, 47.5139274597168, 52.98493194580078, 46.11223602294922, 48.977237701416016, 48.790069580078125, 51.11530303955078, 49.08616638183594, 58.48495101928711, 42.870582580566406, 52.225250244140625, 49.72401809692383, 54.28783416748047, 43.35393142700195, 54.717044830322266, 44.31834411621094, 52.610633850097656, 51.89192199707031, 49.96867752075195, 46.89598083496094, 46.57804489135742, 50.23445510864258, 46.83795166015625, 42.03853225708008, 42.61172103881836, 53.04274368286133, 53.14200210571289, 51.91489028930664, 50.04903793334961, 51.38503646850586, 51.848514556884766, 50.8795051574707, 46.70987319946289, 46.948577880859375, 45.2774772644043, 50.49475860595703, 45.92640686035156, 49.259159088134766, 55.15838623046875, 48.396934509277344, 53.79080581665039, 56.14658737182617, 42.615169525146484, 44.17998504638672, 51.81431198120117, 47.08976364135742, 52.06990432739258, 52.76350784301758, 47.883846282958984, 53.24937057495117, 56.923587799072266, 53.77792739868164, 46.458927154541016, 49.38151931762695, 47.821590423583984, 56.24536895751953, 49.20558166503906, 47.222312927246094, 50.174686431884766, 52.04987335205078, 48.49467468261719, 50.42823028564453, 50.00996780395508, 52.96632766723633, 53.911598205566406, 51.11874008178711, 48.1585693359375, 51.655826568603516, 46.46968078613281, 54.3389892578125, 51.441463470458984, 51.0297737121582, 40.35880661010742, 59.5367431640625, 55.88463592529297, 55.15389633178711, 50.58823776245117, 52.7753791809082, 61.525123596191406, 47.37605667114258, 53.10375213623047, 52.53717803955078, 50.70589065551758, 51.56585693359375, 46.24363708496094, 52.76615524291992, 44.94729232788086, 57.52866744995117, 53.40854263305664, 46.81300735473633, 41.85464096069336, 50.65375518798828, 52.27217483520508, 46.268863677978516, 56.96599578857422, 51.35621643066406, 48.42129898071289, 44.698974609375, 49.81058120727539, 49.50239944458008, 52.90425491333008, 54.503143310546875, 50.432090759277344, 52.15565490722656, 56.01225280761719, 52.78709411621094, 49.28377914428711, 52.79320526123047, 48.230464935302734, 52.12518310546875, 50.52142333984375, 45.924835205078125, 52.73211669921875, 50.65291213989258, 53.24264907836914, 45.00553894042969, 54.74470520019531, 54.91524887084961, 48.90336608886719, 49.91020202636719, 50.4901237487793, 50.20970153808594, 53.62953186035156, 47.427730560302734, 45.6764030456543, 49.29593276977539, 57.85188674926758, 48.6895637512207, 46.237884521484375, 46.550533294677734, 50.88371658325195, 54.91277313232422, 47.255027770996094, 50.889373779296875, 51.04193115234375, 52.043792724609375, 50.75581741333008, 54.37667465209961, 52.91902160644531, 47.706642150878906, 59.86898422241211, 44.71257781982422, 44.890960693359375, 54.302001953125, 50.90406036376953, 49.164241790771484, 50.957977294921875, 52.8979377746582, 48.45783996582031, 47.6066780090332, 45.94978332519531, 46.693695068359375, 46.93608474731445, 54.715396881103516, 51.24642562866211, 52.30262756347656, 47.87173843383789, 54.31605911254883, 43.87656021118164, 52.01479721069336, 52.8937873840332, 47.795074462890625, 46.004058837890625, 52.18663024902344, 47.262577056884766, 46.518898010253906, 50.00807571411133, 45.61713409423828, 49.544124603271484, 58.1963996887207, 48.34029769897461, 53.71920394897461, 45.8751335144043, 50.42132568359375, 49.002105712890625, 49.41745376586914, 53.11302185058594, 58.67933654785156, 49.57477951049805, 55.298831939697266, 51.48166275024414, 53.02598190307617, 48.2052116394043, 47.251094818115234, 49.27372360229492, 46.9853515625, 50.036495208740234, 45.849849700927734, 50.09258270263672, 50.77853775024414, 57.33560562133789, 56.96799087524414, 52.1132698059082, 48.06521224975586, 52.50718688964844, 53.34339141845703, 49.293277740478516, 46.68251419067383, 42.914894104003906, 49.92399978637695, 57.76670837402344, 49.253265380859375, 51.622745513916016, 51.54270553588867, 52.02051544189453, 53.17816162109375, 42.06531524658203, 49.484962463378906, 51.59535217285156, 49.7318000793457, 54.954833984375, 48.607757568359375, 53.78779602050781, 57.16816711425781, 49.098365783691406, 50.23103713989258, 48.95075607299805, 53.18983459472656, 41.56802749633789, 44.27793884277344, 48.959686279296875, 53.62991714477539, 52.51115036010742, 46.87263870239258, 56.579044342041016, 50.6034049987793, 47.65977096557617, 52.28953552246094, 55.3899040222168, 49.90166091918945, 44.09603500366211, 47.96873474121094, 50.5521240234375, 53.94477844238281, 46.16769027709961, 51.23517990112305, 45.44218444824219, 50.493221282958984, 44.8526496887207, 48.33391189575195, 44.08976745605469, 46.268245697021484, 43.82060623168945, 42.30567169189453, 55.095760345458984, 54.914955139160156, 52.120235443115234, 48.77206802368164, 56.08926010131836, 49.08082962036133, 51.92008972167969, 54.765445709228516, 51.116058349609375, 56.147315979003906, 49.653526306152344, 49.62065124511719, 50.767024993896484, 45.61625671386719, 45.941349029541016, 53.358177185058594, 43.33370590209961, 54.780330657958984, 51.35625457763672, 55.00777816772461, 53.32520294189453, 52.6721076965332, 57.25252151489258, 49.9735221862793, 49.864383697509766, 54.798580169677734, 47.63804626464844, 54.82566452026367, 53.044551849365234, 48.272056579589844, 45.177982330322266, 42.17601776123047, 57.784305572509766, 44.2984733581543, 45.43472671508789, 53.729766845703125, 52.15560531616211, 51.09806442260742, 50.58439636230469, 48.53717803955078, 46.54121780395508, 47.259117126464844, 52.61164093017578, 54.533843994140625, 54.955081939697266, 50.057586669921875, 41.19979476928711, 52.238502502441406, 49.60252380371094, 39.436012268066406, 55.75065994262695, 51.088829040527344, 49.98657989501953, 50.99757766723633, 50.69386672973633, 51.16946029663086, 50.68360900878906, 50.31882095336914, 52.52224349975586, 51.62897491455078, 59.92221450805664, 50.81663131713867, 48.21104431152344, 47.87010955810547, 51.82331466674805, 50.78055191040039, 51.288333892822266, 49.5003547668457, 49.58021926879883, 53.33415985107422, 55.78071212768555, 51.69967269897461, 49.88724899291992, 45.66172409057617, 45.65751266479492, 50.481937408447266, 49.298580169677734, 49.88200759887695, 45.81114196777344, 49.749847412109375, 42.42702102661133, 45.440731048583984, 47.03123092651367, 52.55720520019531, 47.73503112792969, 46.3846435546875, 51.24085235595703, 49.74750900268555, 53.83133316040039, 51.41617202758789, 50.963191986083984, 49.33472442626953, 52.73495864868164, 47.13605499267578, 43.93764877319336, 53.67537307739258, 48.13981246948242, 50.222808837890625, 52.531593322753906, 54.103599548339844, 38.387290954589844, 45.81252670288086, 54.93499755859375, 43.17792892456055, 51.90505599975586, 48.549198150634766, 46.43159866333008, 49.47545623779297, 43.213958740234375, 51.62358474731445, 54.706024169921875, 44.27012252807617, 47.305152893066406, 54.805110931396484, 51.472938537597656, 48.68828201293945, 55.74073791503906, 43.25151443481445, 48.876007080078125, 50.356414794921875, 48.703155517578125, 53.17992401123047, 43.263404846191406, 50.16776657104492, 46.8195686340332, 54.79480743408203, 54.17856979370117, 46.98368835449219, 45.55607223510742, 49.790245056152344, 46.522071838378906, 47.6849365234375, 51.694820404052734, 48.125789642333984, 57.255592346191406, 50.23808288574219, 47.870704650878906, 45.47937774658203, 47.87531661987305, 54.40818405151367, 52.16749954223633, 55.70120620727539, 55.225406646728516, 52.19472885131836, 47.166542053222656, 49.35800552368164, 50.23731231689453, 43.32378005981445, 51.286617279052734, 45.79264831542969, 49.616600036621094, 47.890804290771484, 52.29461669921875, 60.53249740600586, 45.172367095947266, 49.89125442504883, 58.1838493347168, 46.247318267822266, 49.066246032714844, 49.54533767700195, 51.22666549682617, 43.481727600097656, 51.21194839477539, 52.41386032104492, 60.216495513916016, 49.679100036621094, 47.67695236206055, 50.52894973754883, 39.50347900390625, 51.35604476928711, 56.261146545410156, 49.867584228515625, 49.86397933959961, 49.02948760986328, 50.56694412231445, 45.01705551147461, 56.22462844848633, 48.17623519897461, 48.19997024536133, 56.6868896484375, 50.12553405761719, 48.35700607299805, 56.6827278137207, 50.797386169433594], [53.8477897644043, 47.789520263671875, 42.98208236694336, 50.79085159301758, 55.839900970458984, 48.24235153198242, 51.56273651123047, 53.007667541503906, 51.16563415527344, 50.40134048461914, 53.22121810913086, 50.976356506347656, 47.979530334472656, 54.371185302734375, 53.36146545410156, 51.650177001953125, 53.038169860839844, 39.54404067993164, 51.22340774536133, 50.309898376464844, 42.159664154052734, 58.48712921142578, 51.56428909301758, 48.39110565185547, 48.64453125, 51.23198699951172, 50.098060607910156, 43.69195556640625, 48.55830001831055, 51.96574401855469, 52.63874053955078, 53.65410614013672, 45.91162872314453, 47.63965606689453, 51.680179595947266, 47.743255615234375, 51.05635070800781, 46.8590202331543, 52.563621520996094, 50.462547302246094, 52.5611686706543, 50.37529754638672, 47.080169677734375, 45.34210205078125, 50.43292999267578, 47.80476379394531, 44.28707504272461, 48.06550979614258, 53.45580291748047, 55.25763702392578, 53.35295104980469, 55.76350402832031, 48.814544677734375, 49.70869827270508, 46.487449645996094, 47.209747314453125, 56.79951095581055, 52.07693099975586, 52.943363189697266, 60.81825256347656, 58.092227935791016, 46.87163162231445, 54.288333892822266, 47.31681823730469, 54.335594177246094, 42.79930877685547, 46.583683013916016, 46.67424011230469, 58.427520751953125, 43.21595001220703, 54.836395263671875, 54.698089599609375, 50.84493637084961, 53.722015380859375, 60.76158142089844, 51.38901901245117, 49.25628662109375, 55.82714080810547, 53.52715301513672, 45.73122024536133, 51.12593078613281, 51.51502990722656, 45.7503662109375, 46.524234771728516, 47.68828582763672, 45.49739074707031, 52.4396858215332, 51.566490173339844, 41.366241455078125, 48.632083892822266, 50.96150207519531, 44.861366271972656, 49.19269943237305, 51.48942184448242, 43.42401123046875, 49.05695724487305, 40.70216751098633, 53.308345794677734, 47.284786224365234, 44.36680603027344, 42.74600601196289, 43.52549743652344, 55.385337829589844, 46.76328659057617, 47.84500503540039, 48.54866027832031, 44.02110290527344, 53.21623992919922, 39.94169998168945, 53.71981430053711, 47.00924301147461, 50.249061584472656, 51.26552963256836, 53.459110260009766, 45.019222259521484, 52.52678298950195, 49.82986068725586, 49.40110778808594, 48.8838005065918, 54.56119918823242, 51.93709945678711, 55.189125061035156, 53.74280548095703, 41.963382720947266, 49.593101501464844, 54.1080436706543, 46.33687973022461, 43.162601470947266, 40.66434860229492, 58.07904052734375, 46.230533599853516, 50.67662048339844, 45.60135269165039, 51.704612731933594, 56.72279739379883, 44.06410217285156, 49.714561462402344, 46.85388946533203, 42.312137603759766, 46.327919006347656, 50.08940505981445, 55.01593780517578, 55.19933319091797, 49.91535949707031, 43.225093841552734, 52.68734359741211, 49.965782165527344, 49.92173767089844, 45.30852127075195, 49.34949493408203, 58.0197868347168, 56.0865592956543, 55.20560836791992, 52.732513427734375, 48.37458038330078, 50.00761032104492, 50.61823272705078, 51.596614837646484, 48.57501983642578, 46.46525955200195, 54.57856369018555, 48.47943878173828, 53.44941711425781, 49.29254150390625, 52.7817497253418, 52.11620330810547, 53.04062271118164, 52.828041076660156, 46.13737487792969, 44.463260650634766, 49.825016021728516, 56.77737808227539, 46.61033630371094, 46.21915054321289, 59.32158660888672, 43.69314193725586, 47.67331314086914, 48.34001922607422, 47.238609313964844, 47.37214279174805, 54.41537094116211, 50.849937438964844, 50.751583099365234, 43.242366790771484, 53.253509521484375, 50.68056869506836, 50.626827239990234, 53.37651824951172, 51.28901672363281, 49.89370346069336, 51.71815490722656, 47.126434326171875, 46.321144104003906, 44.4921875, 53.955413818359375, 47.35231018066406, 50.73571014404297, 54.542667388916016, 57.391212463378906, 48.5557861328125, 50.69923782348633, 50.411842346191406, 44.69852828979492, 48.072757720947266, 61.41276168823242, 49.76871109008789, 57.24125671386719, 53.13985824584961, 57.19565200805664, 55.43609619140625, 50.954524993896484, 44.983943939208984, 55.86760330200195, 52.9144287109375, 52.004493713378906, 53.56569290161133, 46.7931022644043, 50.55525207519531, 50.00934982299805, 43.965660095214844, 45.129886627197266, 49.50867462158203, 40.98271942138672, 53.01393127441406, 53.15204620361328, 49.701839447021484, 51.14821243286133, 42.59709167480469, 50.69718551635742, 53.05439758300781, 52.052589416503906, 53.738487243652344, 49.72639083862305, 58.10584259033203, 51.74956512451172, 53.21538543701172, 54.389556884765625, 46.862857818603516, 45.402015686035156, 53.163692474365234, 44.880184173583984, 47.535770416259766, 53.998680114746094, 45.527801513671875, 45.98017501831055, 44.921756744384766, 45.729183197021484, 57.601661682128906, 52.207157135009766, 53.88916015625, 42.07337951660156, 40.16965866088867, 45.28350067138672, 54.22517776489258, 54.74626922607422, 41.90169906616211, 42.8326301574707, 52.60417556762695, 50.96108627319336, 51.71860122680664, 52.628997802734375, 45.81404113769531, 50.430423736572266, 48.771583557128906, 54.92702865600586, 48.65675354003906, 42.21119689941406, 55.05701446533203, 57.27021026611328, 53.869720458984375, 48.564361572265625, 46.042510986328125, 53.99913024902344, 47.74149703979492, 52.021766662597656, 53.08040237426758, 49.34410095214844, 41.736488342285156, 51.92548751831055, 45.80107879638672, 43.520389556884766, 49.68756866455078, 50.43065643310547, 49.49793243408203, 52.45022201538086, 42.089317321777344, 52.237937927246094, 56.4971809387207, 49.53586196899414, 47.82473373413086, 49.16642379760742, 54.86106491088867, 49.10977554321289, 46.6447868347168, 42.9763069152832, 55.62710189819336, 55.700374603271484, 49.92156982421875, 58.340518951416016, 48.28697967529297, 52.851287841796875, 44.56439208984375, 45.80292892456055, 49.87637710571289, 51.85149002075195, 52.425350189208984, 46.267356872558594, 47.763240814208984, 49.46873092651367, 48.7744255065918, 48.866878509521484, 53.243125915527344, 54.662109375, 42.41154479980469, 55.29347229003906, 50.66743850708008, 51.271854400634766, 51.316097259521484, 53.7437744140625, 52.785037994384766, 43.724578857421875, 53.48302459716797, 45.37852096557617, 51.52881622314453, 50.8404655456543, 43.70435333251953, 50.03054428100586, 48.93746566772461, 47.245731353759766, 58.222476959228516, 50.6282958984375, 48.47697067260742, 57.063724517822266, 49.85979461669922, 49.43473815917969, 41.84149169921875, 52.423892974853516, 54.05025100708008, 57.74760818481445, 45.872840881347656, 54.22706985473633, 55.68452072143555, 52.92498779296875, 50.00175094604492, 43.284889221191406, 50.751808166503906, 47.73310852050781, 42.913570404052734, 52.43260192871094, 49.49482345581055, 49.03044509887695, 57.99995422363281, 47.81372833251953, 41.54433059692383, 40.95735168457031, 52.10485076904297, 49.08707046508789, 46.89360427856445, 41.346553802490234, 46.589237213134766, 45.38671112060547, 49.491939544677734, 46.44561004638672, 46.377777099609375, 49.737308502197266, 46.71459197998047, 51.22724533081055, 36.04363250732422, 44.86568069458008, 54.81510543823242, 50.717105865478516, 45.47635269165039, 49.12100601196289, 47.5325927734375, 48.375152587890625, 55.674896240234375, 48.17932891845703, 50.715126037597656, 46.40914535522461, 46.84721374511719, 43.582733154296875, 47.19728088378906, 48.185035705566406, 48.602210998535156, 50.0789909362793, 50.244224548339844, 52.52348709106445, 48.99435806274414, 55.182979583740234, 52.32464599609375, 55.46649169921875, 49.66142654418945, 54.79664993286133, 50.26099395751953, 43.18189239501953, 49.7938232421875, 49.17036819458008, 43.98295593261719, 45.886566162109375, 47.425689697265625, 59.117149353027344, 51.14344787597656, 57.23550796508789, 53.67485427856445, 54.28065490722656, 53.18888473510742, 43.88764572143555, 54.9782829284668, 45.859188079833984, 51.692413330078125, 49.51155090332031, 56.476688385009766, 54.77162170410156, 57.34328079223633, 51.92048263549805, 49.981807708740234, 53.06391906738281, 59.631553649902344, 52.06144714355469, 50.14348220825195, 45.0106201171875, 46.95906066894531, 49.631690979003906, 52.695411682128906, 52.48097229003906, 49.57875442504883, 50.51408386230469, 56.157745361328125, 57.413902282714844, 54.80805969238281, 56.14314270019531, 55.000728607177734, 57.33850860595703, 44.93378448486328, 47.944820404052734, 50.949440002441406, 51.56425476074219, 55.38910675048828, 42.60674285888672, 50.60042953491211, 51.07801055908203, 47.293800354003906, 49.45293045043945, 54.040000915527344, 51.88924789428711, 49.51522445678711, 50.85332107543945, 49.528377532958984, 53.79574203491211, 49.62518310546875, 53.97972869873047, 47.800323486328125, 49.90574264526367, 45.543540954589844, 49.22298049926758, 53.42228698730469, 45.97486114501953, 48.64095687866211, 50.47755432128906, 48.62936019897461, 53.428165435791016, 45.0722541809082, 53.517425537109375, 55.37852096557617, 56.74013137817383, 49.8149528503418, 46.845951080322266, 43.733177185058594, 52.82246780395508, 53.38532638549805, 51.25294494628906, 51.469444274902344, 52.789798736572266, 48.63595962524414, 44.163700103759766, 44.33637619018555, 46.91740417480469, 46.3929443359375, 43.686790466308594, 48.348548889160156, 45.91168975830078, 52.73295974731445, 49.79568099975586, 53.28779602050781, 44.75229263305664, 45.66293716430664, 43.663978576660156, 51.2402458190918, 62.63909912109375, 50.98168182373047, 48.34773635864258, 50.952152252197266, 49.96919250488281, 53.98031234741211, 50.07925033569336, 46.54799270629883, 42.437110900878906, 51.54163360595703, 54.135799407958984, 50.696659088134766, 46.52185821533203, 48.34940719604492, 48.79462814331055, 52.43830108642578, 49.850746154785156, 54.81465148925781, 49.28791809082031, 47.494651794433594, 57.54499816894531, 46.0673828125, 46.92613983154297, 58.492767333984375, 51.230865478515625, 57.29106903076172, 40.034210205078125, 55.157073974609375, 50.096927642822266, 50.51655197143555, 52.08460235595703, 38.983646392822266, 46.47850036621094, 51.899261474609375, 63.1219367980957, 47.06214904785156, 59.60908508300781, 52.6667594909668, 54.242637634277344, 46.22185134887695, 52.563297271728516, 46.63503646850586, 45.214630126953125, 55.24909591674805, 47.26174545288086, 48.039817810058594, 52.891788482666016, 46.819339752197266, 44.995121002197266, 51.00275802612305, 51.18628692626953, 56.219547271728516, 45.54694366455078, 52.9727897644043, 52.966651916503906, 48.314186096191406, 49.40196228027344, 50.1642951965332, 45.913978576660156, 44.73637771606445, 52.920658111572266, 45.76137924194336, 46.12548065185547, 53.253875732421875, 47.86342239379883, 48.41423797607422, 44.39188766479492, 45.05059051513672, 44.43571090698242, 44.45561599731445, 55.03428649902344, 46.09576416015625, 40.627586364746094, 45.49449920654297, 52.78702163696289, 45.23750305175781, 49.7213020324707, 46.09241485595703, 42.7923469543457, 47.6513786315918, 45.718719482421875, 50.55124282836914, 52.69520950317383, 49.56245803833008, 60.4788818359375, 45.557777404785156, 55.23557662963867, 51.52279281616211, 55.46832275390625, 51.96724319458008, 40.74716567993164, 48.60272979736328, 53.971805572509766, 49.88966369628906, 44.47915267944336, 47.582916259765625, 51.113037109375, 54.734371185302734, 47.35186767578125, 50.16944122314453, 45.99957275390625, 44.905948638916016, 47.64748001098633, 55.1838264465332, 52.6455078125, 48.61233139038086, 49.6151237487793, 51.87923812866211, 49.207366943359375, 47.943111419677734, 48.476741790771484, 52.26730728149414, 48.085628509521484, 52.82707977294922, 42.15391159057617, 46.58343505859375, 47.9212760925293, 52.01410675048828, 45.912471771240234, 47.459861755371094, 50.89530944824219, 49.19399642944336, 46.89542770385742, 48.261688232421875, 45.874820709228516, 52.842098236083984, 51.35920715332031, 53.065582275390625, 58.69565200805664, 54.698856353759766, 51.6871337890625, 49.337467193603516, 58.8737907409668, 50.7786750793457, 48.96809005737305, 55.07701110839844, 51.64714813232422, 48.94473648071289, 51.77687454223633, 57.920806884765625, 50.667572021484375, 51.67412567138672, 46.66939163208008, 48.32511520385742, 53.828582763671875, 49.914695739746094, 50.789772033691406, 54.73569107055664, 49.02810287475586, 56.590797424316406, 47.77659606933594, 47.33683395385742, 47.14461898803711, 44.36065673828125, 52.57929611206055, 50.44198226928711, 55.84852600097656, 47.40612030029297, 53.34873962402344, 55.67918014526367, 49.41291427612305, 49.61428451538086, 40.28363037109375, 58.034019470214844, 49.7437744140625, 53.00864791870117, 48.155452728271484, 51.619384765625, 45.618675231933594, 53.22806930541992, 50.62883377075195, 48.73575973510742, 51.9537353515625, 43.02378845214844, 50.857688903808594, 48.44036865234375, 55.990516662597656, 53.26753234863281, 49.42367172241211, 48.9854736328125, 62.11136245727539, 46.1497917175293, 57.761417388916016, 53.83733367919922, 53.26682662963867, 42.4770622253418, 53.44424819946289, 48.094573974609375, 46.46342468261719, 52.48765563964844, 47.2366943359375, 52.37824630737305, 46.299163818359375, 52.434329986572266, 51.82334518432617, 46.65660858154297, 46.205291748046875, 47.30636215209961, 47.87939453125, 47.58998107910156, 57.41161346435547, 51.65782165527344, 48.59135818481445, 48.654788970947266, 46.724342346191406, 51.05710220336914, 49.68416976928711, 47.21648025512695, 47.154396057128906, 48.303672790527344, 45.690834045410156, 50.39269256591797, 46.27954864501953, 52.0909423828125, 48.60930633544922, 52.069515228271484, 47.61355972290039, 42.45588302612305, 44.67723846435547, 48.79894256591797, 51.51029586791992, 51.6088752746582, 52.062191009521484, 52.74855041503906, 51.47943878173828, 51.77790832519531, 49.1228141784668, 52.25767517089844, 49.59099578857422, 49.73796463012695, 50.78889083862305, 59.22954559326172, 44.16328430175781, 48.133628845214844, 48.99283981323242, 51.75782012939453, 50.12161636352539, 56.61725616455078, 43.84235382080078, 57.34413528442383, 45.075077056884766, 47.22706985473633, 52.331668853759766, 50.43955993652344, 50.01121139526367, 50.40434265136719, 51.666534423828125, 53.96320343017578, 51.75159454345703, 45.08760452270508, 47.17088317871094, 55.600135803222656, 57.8143310546875, 53.85645294189453, 54.3963508605957, 51.3468017578125, 58.401817321777344, 47.81743240356445, 48.7646598815918, 47.00113296508789, 56.37181854248047, 48.0152702331543, 52.14234924316406, 44.5844841003418, 44.248714447021484, 54.07322692871094, 52.951934814453125, 53.515010833740234, 50.946651458740234, 48.9648551940918, 48.50406265258789, 45.71052169799805, 48.614540100097656, 55.51774597167969, 49.51359558105469, 44.90728759765625, 49.18439483642578, 46.21516799926758, 45.564422607421875, 49.694515228271484, 47.081424713134766, 45.09977340698242, 56.196041107177734, 56.98728561401367, 50.68373107910156, 56.016292572021484, 48.285194396972656, 49.97087860107422, 55.32032775878906, 51.289676666259766, 53.3604850769043, 39.10036087036133, 51.23429489135742, 48.37992858886719, 49.63525390625, 43.99966812133789, 46.96761703491211, 52.44059753417969, 52.384132385253906, 53.13037109375, 53.43968200683594, 54.421871185302734, 51.5758056640625, 49.0850830078125, 47.73564147949219, 48.35247039794922, 46.15721893310547, 46.04798126220703, 49.664119720458984, 40.90576171875, 45.10697937011719, 46.80393600463867, 52.11978530883789, 54.32112503051758, 51.3372688293457, 49.84516525268555, 48.057796478271484, 54.47434997558594, 52.85853958129883, 47.401092529296875, 49.93100357055664, 51.68928909301758, 53.695621490478516, 48.730220794677734, 56.06023025512695, 54.75194549560547, 48.38865280151367, 49.30961608886719, 46.40414810180664, 46.50172805786133, 51.37550735473633, 55.830360412597656, 55.89259338378906, 50.93389129638672, 48.701568603515625, 47.62233352661133, 51.93532943725586, 48.3836555480957, 53.088138580322266, 49.76409912109375, 51.422977447509766, 48.189125061035156, 58.33686447143555, 54.81290054321289, 50.62968063354492, 46.77079772949219, 52.53241729736328, 53.29982376098633, 45.76939010620117, 50.59785079956055, 58.44590759277344, 51.0239372253418, 48.73773193359375, 52.679752349853516, 57.59345245361328, 48.560935974121094, 48.6063346862793, 49.39056396484375, 50.660865783691406, 55.114463806152344, 51.70627975463867, 41.52638244628906, 52.126277923583984, 51.991905212402344, 52.947078704833984, 51.507659912109375, 49.91941452026367, 52.45576858520508, 47.70233154296875, 43.69252395629883, 49.057579040527344, 46.00507354736328, 57.10044860839844, 49.909629821777344, 56.576454162597656, 44.963645935058594, 45.492835998535156, 52.00282669067383, 43.47367477416992, 47.716094970703125, 42.527957916259766, 51.925655364990234, 43.359527587890625, 49.67252731323242, 43.152435302734375, 52.118438720703125, 54.44087219238281, 50.65699768066406, 46.226722717285156, 51.84881591796875, 54.04888916015625, 53.19579315185547, 55.17863845825195, 57.72112274169922, 51.03917694091797, 49.09231185913086, 47.590694427490234, 50.07510757446289, 49.19624710083008, 48.00265121459961, 51.47856903076172, 58.025794982910156, 46.9453125, 48.16149139404297, 48.67210388183594, 45.9861946105957, 54.27198028564453, 49.44758987426758, 52.43380355834961, 43.627716064453125, 52.28306579589844, 49.52116775512695, 44.234676361083984, 53.464210510253906, 53.795101165771484, 50.92863845825195, 51.903255462646484, 46.24061965942383, 54.14064407348633, 49.09126663208008, 51.93547439575195, 52.421142578125, 51.96089172363281, 47.1231575012207, 51.13478088378906, 50.615840911865234, 54.75492477416992, 47.03926086425781, 53.868133544921875, 51.95223617553711, 50.27027893066406, 57.59103012084961, 53.28911590576172, 47.869327545166016, 51.25619888305664, 47.05035400390625, 45.38539505004883, 48.04062271118164, 48.089805603027344, 56.87049102783203, 53.24085998535156, 40.364749908447266, 49.41858673095703, 47.244728088378906, 49.468502044677734, 50.74198532104492, 50.19379806518555, 51.72034454345703, 46.902801513671875, 55.67639923095703, 49.495182037353516, 54.94813919067383, 49.63357925415039, 52.00615692138672, 46.190608978271484, 54.45551300048828, 51.179630279541016, 48.4141960144043, 51.54130554199219, 51.90153121948242, 43.85818099975586, 47.01365661621094, 46.03104782104492, 53.324195861816406, 50.75048828125, 47.34219741821289, 47.16895294189453, 49.84260177612305, 40.80393981933594, 49.32491683959961, 56.25639343261719, 44.38580322265625, 45.2125244140625, 45.474029541015625, 51.762481689453125, 46.98619842529297, 57.482154846191406, 49.19038772583008, 48.00820541381836, 45.59486389160156, 52.722206115722656, 54.286434173583984, 50.6832389831543, 55.217620849609375, 54.971187591552734, 51.44527053833008, 50.26914978027344, 47.82907485961914, 50.919944763183594, 48.64842224121094, 55.95732879638672, 53.07192611694336, 49.388370513916016, 48.50673294067383, 51.7218017578125, 54.26429748535156, 51.2031364440918, 54.16307830810547, 48.30501937866211, 50.99686813354492, 52.714298248291016, 50.90859603881836, 51.6745491027832, 54.201866149902344, 50.13970184326172, 54.14887237548828, 48.118499755859375, 52.15866470336914, 48.490821838378906, 48.69236755371094, 52.62269973754883, 55.33443069458008]], "dtype": "float32"}, {"__nd__": [-0.058609090745449066, 0.16296041011810303, 1.394927978515625, -0.7243450284004211, 0.9893326759338379, -1.0811567306518555, -1.8002487421035767, 0.4312465190887451, -0.20289508998394012, -0.42567312717437744, -2.3121469020843506, 0.4457888603210449, -0.38042548298835754, -1.0975983142852783, -0.03929280489683151, -1.119173288345337, -0.7006880640983582, -0.743288516998291, 1.036751627922058, -0.0502912774682045, 1.3461846113204956, -0.7511456608772278, -1.1358426809310913, 1.8109080791473389, -1.326167106628418, 1.2654954195022583, -1.0491005182266235, 0.7671260833740234, -0.6859959959983826, 1.5593405961990356, 2.025953769683838, 1.0971591472625732, 0.4107394814491272, -0.30582094192504883, 0.8485366106033325, 2.1631383895874023, -0.38581642508506775, -1.0673532485961914, 1.855400562286377, -0.15129543840885162, 0.8064190745353699, 0.2669338285923004, 0.34848177433013916, -0.1976155787706375, -0.056623492389917374, -1.589194893836975, -0.4844615161418915, -0.7823276519775391, 1.2475695610046387, -0.22885139286518097, 0.19063137471675873, -1.5898244380950928, 0.15996578335762024, 1.609471321105957, 0.2534945607185364, 0.9643060564994812, -0.22611163556575775, -0.5263950824737549, 1.046466588973999, -0.3636203408241272, -0.9595946073532104, -0.015127908438444138, -0.29786303639411926, 1.1378110647201538, 0.8100088834762573, -1.3530939817428589, 0.596145510673523, -1.2496598958969116, 0.2651059925556183, 0.6556275486946106, -0.4312736392021179, -0.4396377205848694, 0.8494310975074768, -0.21529564261436462, -1.0392838716506958, 1.7186962366104126, 0.9644219279289246, 0.5621758699417114, 0.4356314241886139, 1.1098506450653076, 0.17943821847438812, 0.8839022517204285, 0.42547059059143066, -0.5741647481918335, 1.1636534929275513, 0.2796519100666046, -0.6976014971733093, -0.3311852514743805, -2.2853822708129883, -1.5536330938339233, -1.348146915435791, 0.7956990599632263, 0.762464165687561, 1.2402788400650024, 0.5089648365974426, 1.8096579313278198, -0.37223732471466064, -0.6800718307495117, -0.8567442297935486, 0.26151347160339355, -0.5864918231964111, -0.8570237755775452, -0.4819689989089966, 0.20421355962753296, -1.241112232208252, 0.03054048679769039, 0.018350427970290184, 1.7065799236297607, 0.18260003626346588, -1.2554631233215332, 0.28380337357521057, 0.2198420614004135, -1.2326818704605103, -3.0200252532958984, 0.7750603556632996, -0.16169552505016327, -0.6993016600608826, -0.15083551406860352, 1.3017535209655762, 1.7662708759307861, -0.4445127844810486, 1.545525074005127, -0.029086990281939507, 0.02635383978486061, -0.07475262880325317, -0.04334151744842529, -0.12689724564552307, -0.6685713529586792, 0.03407662361860275, -0.9958080649375916, 0.577201247215271, -0.6321569681167603, 0.5038849115371704, -0.10716787725687027, -1.7749274969100952, 1.1907894611358643, 0.30621784925460815, -0.05482300743460655, -0.32525354623794556, 1.8049708604812622, -0.23402263224124908, 0.9681760668754578, 0.21120670437812805, -1.3210361003875732, 0.728491485118866, -1.5151286125183105, 0.4537009000778198, 0.41374462842941284, 0.2375398874282837, -0.7435579299926758, 1.397802472114563, 1.7701517343521118, -0.9155606031417847, -0.38045865297317505, 1.6291191577911377, -0.4323292672634125, 0.961184561252594, -0.9784068465232849, -0.633400559425354, 0.26013097167015076, 0.45214128494262695, -1.4457045793533325, -1.0499062538146973, 0.5112688541412354, 1.2757476568222046, 0.5147407054901123, 0.35967937111854553, 0.43580010533332825, -1.2190502882003784, 0.3962579369544983, 0.8991712927818298, 0.5954474210739136, -1.166929006576538, -0.958499014377594, 1.1200820207595825, -1.384940266609192, -1.0681498050689697, 1.583078384399414, -0.9018523693084717, -0.7317481637001038, 0.6386840343475342, 0.7848655581474304, -0.16740551590919495, -1.4323614835739136, -0.5178028345108032, 0.16304154694080353, -0.9197327494621277, -0.9756007194519043, 1.2500271797180176, 0.5410477519035339, 0.7814403176307678, 0.8961166143417358, -1.0051637887954712, 0.4836586117744446, 1.1929501295089722, 0.679314911365509, -1.1566542387008667, -1.5547057390213013, -0.803153395652771, 0.5467050075531006, -0.18544551730155945, 0.278155654668808, -1.2629302740097046, 0.8317814469337463, -0.7319740653038025, 1.1288713216781616, -0.02692096121609211, -2.983304738998413, -0.20099715888500214, -0.8784583210945129, -0.8480167388916016, -0.16243065893650055, 1.0687962770462036, 0.843956470489502, 1.3950250148773193, -0.006775322370231152, -1.8736416101455688, -0.5627657771110535, -0.9102773070335388, -0.6916677355766296, 2.660832643508911, 0.7955988645553589, -0.9716129899024963, 1.1903032064437866, -0.2986043393611908, -1.3516161441802979, 1.168671727180481, -0.479875773191452, 0.12517546117305756, 2.297947645187378, -1.0188980102539062, 1.0360150337219238, -0.31614920496940613, -1.1174938678741455, -0.25544971227645874, 0.17025402188301086, 0.18396884202957153, -1.5423492193222046, 0.8352594971656799, -0.8572018146514893, 0.3042236566543579, 0.20090848207473755, -0.5590395331382751, 0.5946524143218994, -0.7217796444892883, -0.949777364730835, -1.4248555898666382, -1.867972493171692, -0.7026321887969971, -2.198394775390625, -0.3678354322910309, -0.5476751327514648, 1.6355128288269043, -1.1710472106933594, 1.1364130973815918, 0.3579765260219574, 0.28504258394241333, -0.6431961059570312, -0.8386884927749634, -0.33704936504364014, -0.0642470046877861, 0.5602267384529114, -0.5295238494873047, 1.5474653244018555, 1.6693686246871948, -2.483076333999634, 0.12001407146453857, -0.13005711138248444, -0.9967870116233826, -2.704066038131714, -1.7473961114883423, -0.2014160305261612, 0.6785237789154053, 0.4372639060020447, 0.5011740922927856, -0.23343618214130402, 1.0114980936050415, 0.4695163667201996, 1.1200000047683716, -0.14076656103134155, -0.665596067905426, -1.5401561260223389, 1.5397344827651978, -0.5880365967750549, -1.0492141246795654, -2.1425108909606934, 0.5628193020820618, -1.3827499151229858, 0.0653817355632782, 0.2795105278491974, 1.4245837926864624, -0.7208463549613953, -1.949139952659607, 0.8661922216415405, 0.2561570405960083, -0.6391749978065491, -0.03710775822401047, 1.8843717575073242, 0.05465118587017059, -0.6617418527603149, -0.08113501965999603, 0.5789934396743774, 0.36482372879981995, 0.33901694416999817, 0.32801559567451477, -1.1705477237701416, 0.6045888066291809, 1.042079210281372, 0.6626666784286499, -0.06385974586009979, -0.07897458970546722, 0.19271844625473022, 0.06323280930519104, -1.649280309677124, -0.12423093616962433, 1.2666898965835571, -0.3213159441947937, -0.5224445462226868, 0.6217910051345825, 0.09953209012746811, -0.6863774657249451, 0.18987660109996796, 1.7299407720565796, 1.8567564487457275, 0.17497456073760986, -0.14183993637561798, 1.2208938598632812, 0.4903475344181061, 0.574055552482605, -0.03750361502170563, -1.276310682296753, -1.4215664863586426, -0.05281725898385048, -1.065730094909668, 0.8345025181770325, 0.9644421339035034, 1.5105040073394775, 0.14101266860961914, 1.9865413904190063, 1.3150159120559692, 0.42888861894607544, 0.80147784948349, -0.5164341330528259, -3.118182897567749, 1.0983068943023682, 0.5442060828208923, 1.736991286277771, 0.5411150455474854, -0.12752681970596313, -0.4405460059642792, -0.5650120377540588, 0.21583804488182068, -1.7724534273147583, -1.0814930200576782, 0.8762532472610474, -2.0076615810394287, -1.5287182331085205, -0.5604720711708069, -0.37819704413414, 0.09841744601726532, 0.8811007142066956, 0.31697705388069153, 0.623593270778656, -0.559839129447937, 0.051346153020858765, -0.6860194206237793, -0.08713014423847198, 0.6069824695587158, 0.5255100131034851, -1.87220299243927, -1.1198173761367798, 2.0182058811187744, -2.0275206565856934, 0.08992525190114975, -1.149557113647461, -0.192964568734169, -0.5637987852096558, 0.09119017422199249, -1.088201880455017, 1.069815993309021, 2.21403169631958, -2.4384334087371826, -0.8260675072669983, -1.5027443170547485, 0.14151571691036224, 1.4273366928100586, 1.4152957201004028, 1.3436318635940552, 0.24831610918045044, -0.536372184753418, 1.1526192426681519, -0.592350423336029, -0.7679307460784912, 0.08941148966550827, -1.220232367515564, -0.16935715079307556, -0.920637845993042, 0.12135239690542221, 0.992047905921936, -0.5344865918159485, 0.4972158968448639, 0.9880134463310242, -0.7551485896110535, -0.049712300300598145, -0.3396786153316498, 1.8394063711166382, -0.4816047251224518, 0.14651602506637573, -1.3879245519638062, -0.9070892930030823, 0.6201841831207275, -0.26400578022003174, 0.7687034606933594, 0.09005006402730942, 2.6467742919921875, -0.21276643872261047, 0.9800443649291992, -0.6944661140441895, -0.4127323031425476, -1.5103859901428223, 2.6688411235809326, -2.077820062637329, 0.14751334488391876, -0.41501426696777344, 0.8284316062927246, -0.5147935152053833, 1.9319367408752441, 1.176098108291626, 0.5752525925636292, 0.5455314517021179, 1.1120167970657349, 0.4912285804748535, 1.659765362739563, 0.9524226188659668, 0.030035853385925293, 0.5216901302337646, 0.2936675250530243, 1.282731056213379, -0.16396719217300415, 0.20250225067138672, -1.1694313287734985, 0.7778228521347046, 0.34858885407447815, -0.5647812485694885, 1.8266050815582275, 0.13034836947917938, -1.0519931316375732, 1.2555075883865356, -0.0755084976553917, -3.0023438930511475, 0.45173877477645874, 0.08945809304714203, 0.18226958811283112, -0.10465089976787567, -0.7131412625312805, 0.22052015364170074, -1.5977740287780762, -0.08372026681900024, 0.9634135961532593, 1.0978418588638306, -0.40297722816467285, 2.037614583969116, 0.40261754393577576, -0.3468180298805237, 0.25993403792381287, -0.5926457047462463, 0.9643853306770325, -0.8455010056495667, -0.6122591495513916, -1.0585112571716309, -0.7160094380378723, -0.18803143501281738, 1.9595186710357666, 0.33877381682395935, 1.2755241394042969, 0.1449190229177475, 1.0415313243865967, -0.9555771350860596, -1.035093069076538, 1.9607577323913574, 0.4695088565349579, -1.4889787435531616, 0.6967544555664062, -0.10883867740631104, -0.18226486444473267, 0.9165145754814148, -1.1422555446624756, 1.3184292316436768, 0.3490830063819885, -0.39783433079719543, -1.265401840209961, 1.844338297843933, -0.28609588742256165, -1.302280068397522, -0.07707741856575012, 0.5476939082145691, 0.7013795375823975, 1.5576955080032349, 0.636326253414154, 0.01624738797545433, 2.320136785507202, 2.0990424156188965, -1.0546531677246094, 1.0578608512878418, -0.5170668959617615, 1.583986759185791, -0.8697493076324463, 0.38141393661499023, 1.6703126430511475, 0.4210970401763916, 1.1836241483688354, -0.6241598725318909, -1.164780855178833, -1.099281668663025, 0.6985853314399719, -0.5676637291908264, 1.5615769624710083, -0.2562851309776306, 1.438626766204834, 0.43415161967277527, -1.008139967918396, 0.1386002153158188, 0.5796401500701904, 0.5829049944877625, -0.014744335785508156, 1.2071722745895386, 1.0491958856582642, 1.0417866706848145, -0.6953059434890747, -2.0423152446746826, 0.6714187860488892, -0.7508044838905334, -0.4588521122932434, -0.16163498163223267, 0.4102363586425781, -1.677219033241272, -0.04861164093017578, 1.4559720754623413, -1.3821014165878296, -0.21119609475135803, 1.5442267656326294, 0.9272860884666443, 0.7106500864028931, 0.24335119128227234, -1.3469922542572021, 0.20281638205051422, -0.8722490668296814, 0.21567144989967346, -0.6804844737052917, -1.0944935083389282, 0.08393443375825882, 1.1605384349822998, -0.4103006422519684, 0.27684521675109863, 0.813885509967804, 0.6065249443054199, -1.1890093088150024, -0.4749634265899658, 0.4454396367073059, 0.11049844324588776, 1.8578436374664307, 0.679482102394104, -0.8042291402816772, 0.7697632908821106, 1.2138804197311401, 0.7421479821205139, 0.5988601446151733, -1.3343217372894287, 0.026255056262016296, 0.682756245136261, -0.310396671295166, -0.2561720609664917, 0.5350619554519653, -0.5647339820861816, -0.8186501264572144, -0.07931062579154968, -0.016269981861114502, 0.9288008809089661, -0.0957370474934578, -1.0884650945663452, 0.5529963374137878, -0.7272505760192871, 0.8126926422119141, -0.508187472820282, 0.6492453813552856, 0.3831597566604614, -0.1258544623851776, -1.2019609212875366, -0.3282935619354248, 1.4864698648452759, 0.2596432864665985, 1.2317891120910645, 0.1379862129688263, 0.07829545438289642, -0.9018182754516602, 0.4542944133281708, -0.7724809646606445, -0.7655104994773865, 1.111363410949707, -0.9317512512207031, 2.370243549346924, 0.06538983434438705, -0.49641308188438416, 1.4049818515777588, -0.9127324819564819, 0.1981116086244583, 0.10118179023265839, 1.0183123350143433, 1.798588514328003, 0.4112475514411926, -1.225598692893982, -1.261889934539795, -0.42825639247894287, 0.8730645775794983, 0.1969636082649231, 0.3364540934562683, -0.2508205473423004, 1.4187977313995361, -1.0688376426696777, 1.990577220916748, 1.26265287399292, 0.7657533884048462, 0.039348162710666656, -0.23543761670589447, 0.05939410254359245, -0.48150646686553955, -0.36307594180107117, -0.7521176934242249, 0.8223510980606079, 0.15944361686706543, 0.501099705696106, -0.6747158169746399, -1.1779736280441284, -0.8672982454299927, 0.2819685637950897, -1.1254199743270874, 0.6416962146759033, -0.8955376148223877, -0.4036557972431183, 0.2831224799156189, -1.1471447944641113, -2.0295753479003906, 1.191953420639038, -0.1358472853899002, -0.6067324280738831, 1.6278566122055054, 2.2002360820770264, 0.05231134220957756, 0.341178834438324, 2.657649278640747, -0.5864152908325195, 1.229604959487915, 1.0042905807495117, -0.5409359931945801, -0.9458861947059631, -0.7368704676628113, 0.17119915783405304, 0.37926244735717773, 2.640115737915039, -0.9913631081581116, 0.2300710380077362, -0.7514102458953857, 0.9043951630592346, 0.46724733710289, -1.2744417190551758, -0.4049414098262787, 0.10765296220779419, 0.772471010684967, 0.5813251733779907, 0.7263111472129822, 0.07743563503026962, 0.6570204496383667, 0.13751058280467987, 1.0316225290298462, 0.6429873108863831, -0.8199806809425354, 0.015138978138566017, -0.7021170258522034, -0.05452507361769676, 0.632208526134491, 0.16194570064544678, -0.04425327479839325, -0.5379634499549866, 1.5462924242019653, 0.07825995236635208, -0.8941624760627747, -0.4053542912006378, 1.0155236721038818, 2.0947463512420654, 1.3427225351333618, -0.2598535716533661, 0.9273083806037903, -1.5896728038787842, 0.9028159379959106, -0.1547987461090088, 0.337981641292572, -1.156127691268921, 0.4501473307609558, -1.1998841762542725, -1.5599243640899658, 1.5622144937515259, -1.1675564050674438, 0.9930763840675354, -0.4901351034641266, -0.039744727313518524, 0.06694691628217697, 1.1796844005584717, 0.5012404322624207, -0.8121668100357056, -0.44964227080345154, 1.3879607915878296, 1.4151084423065186, -0.4072864353656769, 0.9538639187812805, -0.005431880243122578, 1.0714361667633057, -0.9404116868972778, 0.8740315437316895, -0.4989400804042816, -1.2197439670562744, 0.6629244685173035, 0.6253814697265625, -1.4380460977554321, -2.2581748962402344, -0.06952077895402908, 0.4943099915981293, -0.48744890093803406, 0.13446544110774994, -0.3825136721134186, -0.5112877488136292, 0.4277516305446625, 1.2649694681167603, 0.49866095185279846, -0.28250616788864136, -0.6538451910018921, 0.23434822261333466, -2.171501398086548, 0.1375926434993744, -0.08335935324430466, 0.006405659485608339, -0.24392864108085632, -0.17330209910869598, -1.4279048442840576, 0.6775554418563843, -0.29543396830558777, 0.038633182644844055, -0.4739074409008026, -1.4527223110198975, 0.6427724957466125, -0.19842591881752014, -0.23208148777484894, 0.40083184838294983, 1.7148016691207886, -0.01067575067281723, 0.5707886815071106, 1.8718814849853516, -0.27517208456993103, 1.247967004776001, 0.07521532475948334, 1.0412615537643433, 0.15740154683589935, 0.9957903027534485, -0.701547384262085, -0.4429342448711395, -0.573255717754364, -0.25896376371383667, 1.141019344329834, 0.47779518365859985, 0.7084728479385376, -0.16866256296634674, 0.1391945481300354, 0.46999624371528625, 0.408950537443161, -0.5055137276649475, -0.33724498748779297, -1.0063786506652832, 0.8035193681716919, 1.1075878143310547, 1.9465858936309814, -3.022599220275879, 0.8855026960372925, 0.6209810376167297, 2.6916885375976562, -0.40931326150894165, -0.7286598682403564, 0.261883020401001, -1.3990834951400757, -1.3640228509902954, 0.6135121583938599, -1.6320226192474365, -1.0834375619888306, 1.2185560464859009, 0.616369366645813, -1.2684286832809448, 1.4197355508804321, 0.09837165474891663, -0.3274315595626831, 0.16301009058952332, -0.07891504466533661, 1.179919719696045, 0.5907275676727295, -0.8738268613815308, -0.5202476382255554, 0.7822255492210388, -0.6043688654899597, -0.9650158882141113, -2.3792288303375244, 0.8469629287719727, 0.008675879798829556, -0.20740407705307007, -0.8806806802749634, -0.5542364120483398, -0.0361345037817955, -1.1340765953063965, 0.09976045042276382, 1.7121368646621704, -0.5213478803634644, 0.0158638097345829, 1.5823760032653809, 0.10097140073776245, 1.5719095468521118, 1.0225367546081543, 0.9305958151817322, -0.16413745284080505, -0.4539240300655365, 1.333112120628357, 0.14839796721935272, -0.10384184122085571, -2.1861491203308105, -0.13131318986415863, 2.5089852809906006, 0.6596968770027161, 1.8052024841308594, 0.26568523049354553, -1.3633335828781128, 0.18430054187774658, 1.817602515220642, 0.2254113405942917, -0.2435978353023529, -0.1254524439573288, -1.1273407936096191, -0.30678290128707886, 0.18408356606960297, 1.007245421409607, 0.8182203769683838, -1.0563197135925293, -1.8846975564956665, -0.30578288435935974, -0.7182192206382751, 0.8816930651664734, -0.9771134853363037, 0.6500548720359802, -0.3910331726074219, 1.180495023727417, 0.09764722734689713, -1.9364246129989624, -2.326242208480835, 0.33993229269981384, -0.9981257319450378, 0.9973939657211304, -0.020869823172688484, 1.8069126605987549, -0.01322134118527174, -0.7462223768234253, -2.008134126663208, 0.5002263188362122, -0.21392115950584412, -0.31688588857650757, 2.70031476020813, 0.7976053953170776, -0.9404081106185913, 0.6705132126808167, -1.549302577972412, -0.6488956809043884, 0.8580150604248047, -1.3166965246200562, -0.002242449903860688, 0.06336915493011475, 0.17389526963233948, -0.9803891777992249, -1.4209308624267578, -0.015031237155199051, 0.6358557939529419, -1.0020148754119873, 0.8084143996238708, 1.1043095588684082, 0.32442060112953186, 1.0261847972869873, 0.9863293766975403, 1.216808557510376, 0.27726149559020996, 0.30499300360679626, 0.3611016869544983, 1.2193549871444702, -0.49719586968421936, 1.5071804523468018, -0.9183217883110046, 1.9850945472717285, 0.6710376143455505, 2.741076946258545, 0.2953648865222931, 0.7170491814613342, -1.5355044603347778, -0.24160020053386688, 0.23035173118114471, 0.4512726664543152, 0.7002689838409424, 0.18588511645793915, -0.020422454923391342, 0.986422598361969, -1.4825314283370972, -0.4149706959724426, 0.7296224236488342, -0.04177474230527878, 1.2389990091323853, -0.6432457566261292, -0.3435799777507782, 0.6666651368141174, 1.8195815086364746, -1.2647656202316284, -1.6835395097732544, -1.1772680282592773, -1.2185548543930054, 0.29450681805610657, 0.9712257385253906, 0.01290073711425066, 0.2720201015472412, 0.6628457307815552, 0.41368475556373596, 0.32981154322624207, 0.270978718996048, 1.1760356426239014, -1.0972440242767334, -0.4595486521720886, 0.3020695149898529, 2.744908332824707, -0.8269578814506531, 0.8197419047355652, -0.7017862200737, -0.12611179053783417, -0.7437007427215576, 1.9193602800369263, 0.42006969451904297, -0.459930419921875, 1.0233185291290283, -0.6631523370742798, 0.31333446502685547, -1.2996395826339722, -0.26468712091445923, 0.38684210181236267, 0.384397953748703, -0.8066387176513672, -0.7274262309074402, -1.7075073719024658, -2.084909677505493, -0.19645905494689941, 1.1618103981018066, 0.5110016465187073, 0.6180379390716553, 0.04809848964214325, 0.012796456925570965, -0.0036893223877996206, 1.7159843444824219, -0.29239246249198914, 1.2473583221435547, -1.3711719512939453, -1.0330928564071655, -0.030937839299440384, -1.4534504413604736, 1.5877515077590942, 0.28032374382019043, 0.46985962986946106, -0.40859249234199524, 0.5461751222610474, -0.5864737033843994, 1.4033607244491577, 1.3743858337402344, -0.5971161723136902, -0.7790312767028809, -1.4193953275680542, -1.368478536605835, 1.3592506647109985, 0.8901622295379639, 1.6870664358139038, 0.6417639851570129, 1.0144795179367065, 0.8079535961151123, -0.34841135144233704, 0.34505653381347656, 1.0906347036361694, 0.29581889510154724, -0.808639407157898, -0.8305936455726624, 3.1173787117004395, -0.7110452055931091, -0.5067533254623413, 0.22049134969711304, 0.07538223266601562, 1.1806684732437134, -0.2449997514486313], "dtype": "float32"}, {"__nd__": [50.00745391845703, 50.1859130859375, 50.089969635009766], "dtype": "float32"}, {"__nd__": [0.25155019760131836, 0.2584524154663086, 0.2432824820280075], "dtype": "float32"}, {"__nd__": [[0.8252784609794617, 0.2478552609682083, 0.25034207105636597, -0.01222191657871008, -0.7047547698020935, -0.348072350025177, -0.9887223243713379, -0.27914971113204956, 0.2795493006706238, 0.38812455534935, -0.9812436103820801, -0.5982367992401123, -0.18218640983104706, 1.467629313468933, 0.3585329055786133, 0.5158430337905884, 1.3789113759994507, 0.6407270431518555, -0.3610503077507019, 1.8214436769485474, 0.02097543701529503, 1.3352537155151367, 0.5559730529785156, 0.7857832312583923, 1.4477472305297852, -0.2973117232322693, -1.2787977457046509, 0.4580390751361847, 0.08680824935436249, -0.12419930845499039, 0.14738775789737701, -1.4336656332015991, 0.11309107393026352, -0.1667010337114334, 0.4109669029712677, 0.01962805911898613, -0.252989798784256, 0.7525773644447327, 1.1807031631469727, 0.009176774881780148, -0.7648316025733948, -0.5947780013084412, 1.282982349395752, -1.5955275297164917, 0.09632227569818497, 0.3107394874095917, 0.49746307730674744, -1.9795383214950562, 0.22296489775180817, -2.180830240249634, -0.5805161595344543, -1.0273971557617188, -2.083286762237549, 1.620255947113037, -0.480985552072525, -1.7515766620635986, -1.2753428220748901, -0.31034770607948303, 0.5307537913322449, -0.14811673760414124, -1.3603079319000244, 1.409977912902832, 0.2054949551820755, 1.5031423568725586, 1.2679322957992554, 0.21525603532791138, 1.2690597772598267, -0.1754191517829895, 0.9741461277008057, -1.6886873245239258, -1.8811297416687012, -1.1591875553131104, 0.45454639196395874, 0.07814662903547287, -0.7875313758850098, -0.6560848355293274, -0.6241730451583862, 0.0701427310705185, 1.0197868347167969, 0.9252178072929382, -0.246258944272995, -0.0731838271021843, 0.5370272994041443, -1.1194283962249756, -0.037257060408592224, 0.5378785133361816, 0.4163036048412323, -0.48437991738319397, -0.8481769561767578, -0.9332056641578674, 1.3815529346466064, 0.004185743164271116, -1.2492139339447021, -2.0324342250823975, -0.0012230048887431622, 1.484907627105713, -0.2852284014225006, -0.8980448246002197, 0.7887653112411499, -0.333074152469635, -0.15907470881938934, 0.2706158459186554, -0.3367537260055542, 0.6610546708106995, 0.803743839263916, -0.10491452366113663, -0.5684873461723328, 0.2959232032299042, -2.5477612018585205, -0.43735185265541077, -0.574709951877594, 0.7080899477005005, -1.0851657390594482, 1.0531055927276611, -0.1559278517961502, 0.3917694091796875, 1.2776176929473877, 0.13758280873298645, -0.6245824694633484, 0.2689867615699768, 0.2914106845855713, 1.9139188528060913, -0.0689375251531601, 0.9147412180900574, 1.3252241611480713, 1.1022758483886719, -0.22185564041137695, -1.1573714017868042, 1.0608969926834106, 1.7141567468643188, 0.6813824772834778, 1.2061262130737305, 0.19358403980731964, 0.890209436416626, 1.3827964067459106, 0.07748039066791534, 1.2969330549240112, 0.04575445130467415, -0.830608069896698, 0.469716876745224, -0.07496832311153412, 0.5358691215515137, 0.6505110859870911, -0.08989182859659195, 0.4967024028301239, -0.3564531207084656, 0.8994153141975403, -0.8239049911499023, -1.280012845993042, 1.1010522842407227, -1.8915107250213623, -0.19339871406555176, -0.1414763629436493, 0.8939318060874939, -0.1832791119813919, -0.1838146150112152, -1.7774192094802856, 0.9990590214729309, -1.022274374961853, -0.005682750139385462, -0.777435302734375, 1.1304495334625244, -0.9250909686088562, -1.3490660190582275, 1.298081874847412, -0.6556217670440674, 0.22845375537872314, 0.4188864231109619, -1.5086030960083008, 1.0030704736709595, -1.0337074995040894, 0.061122260987758636, 0.4351845383644104, -0.08686766773462296, -0.8916633725166321, 0.929705023765564, -0.8886168599128723, -1.322123646736145, 1.171439528465271, 1.230987787246704, -1.878366470336914, -0.22098511457443237, -0.012426040135324001, -2.8408007621765137, -0.5758945941925049, 1.137115240097046, -1.0194538831710815, 1.3009381294250488, 1.973767876625061, -0.1630055457353592, -0.743348240852356, 0.05804387107491493, -1.4529953002929688, 1.2814292907714844, 0.4957069754600525, -1.427046298980713, -0.8022620677947998, -0.3241294324398041, 1.0888526439666748, 0.836169958114624, -0.6527667045593262, 0.17185263335704803, -0.44802284240722656, 0.4788268506526947, -0.27778080105781555, 2.1177985668182373, 0.46831974387168884, -0.26938575506210327, -1.0667262077331543, -1.4131758213043213, 0.7295089960098267, 0.8277548551559448, 1.5849480628967285, -1.0354958772659302, 0.7566903829574585, -1.2410223484039307, 0.6890734434127808, 0.573789119720459, 0.23780126869678497, -0.3094487488269806, -0.486259788274765, -0.37309759855270386, -1.9478912353515625, 0.67911696434021, -1.1434104442596436, 0.15981535613536835, -0.05379559099674225, 0.725807249546051, 0.07691055536270142, 0.02950834482908249, 0.30519717931747437, 1.0108662843704224, -0.3782815635204315, -0.8681812882423401, 0.0511961355805397, -0.08230289816856384, 0.7649745941162109, 0.4328940808773041, 0.19973082840442657, 1.1731125116348267, 0.34558340907096863, 0.675385057926178, -0.3522491157054901, 1.5677682161331177, 0.7143169641494751, -0.8778802156448364, -0.8327956199645996, 1.8438302278518677, 0.2975374460220337, 1.8098818063735962, 2.439117670059204, 0.2193886637687683, 0.38346973061561584, 0.8977211713790894, -0.8163151144981384, -1.7417490482330322, 1.9581966400146484, -0.7247838973999023, -0.04142562672495842, 1.5978004932403564, 0.17470817267894745, 1.0384231805801392, 0.4251341223716736, -1.183004379272461, -0.6662361025810242, 0.19472649693489075, -0.8675304055213928, 1.0816898345947266, 0.4670279026031494, -1.485260248184204, 0.5791328549385071, -0.4230351448059082, -0.22250095009803772, 0.8114640712738037, -1.786007285118103, -0.8571941256523132, 0.35326680541038513, -1.4082790613174438, 1.3620105981826782, -3.2235703468322754, 1.1367980241775513, 0.795490026473999, -0.6096446514129639, 0.11581864207983017, -0.70713210105896, 1.930295705795288, 0.01676533743739128, 0.5297750234603882, 0.3240806460380554, -1.3678534030914307, -1.9216625690460205, 0.19571027159690857, 1.455841302871704, 0.6762587428092957, -0.1477973461151123, 0.4661216139793396, -0.3326398432254791, 1.3233586549758911, 0.6402652859687805, 0.3462657034397125, 2.1564290523529053, -0.5545221567153931, -0.9038958549499512, -0.21247628331184387, -0.5039451718330383, -0.5528506636619568, 0.6496098041534424, 0.562119722366333, -1.7146941423416138, -0.6664971113204956, -0.5114247798919678, -1.850893259048462, 0.6288632154464722, 0.12625189125537872, 0.480479896068573, -0.6603986024856567, -0.6047845482826233, 0.23425008356571198, 0.062325816601514816, -0.47108694911003113, 0.4740225076675415, -0.7133165001869202, 0.9183318614959717, 0.39484554529190063, -0.6064939498901367, -1.3082178831100464, 0.27151912450790405, 1.0833890438079834, 0.01165634673088789, 1.4111902713775635, 0.7336990833282471, 1.2809360027313232, -1.5086115598678589, -0.6716793775558472, 0.09154467284679413, 0.22495202720165253, -0.1563691645860672, -0.4259496331214905, -2.1936912536621094, 1.1164088249206543, 0.751190721988678, -0.49839040637016296, 0.022532103583216667, -0.6546835899353027, 0.9935736656188965, -0.25065097212791443, 1.1508915424346924, -0.5228586196899414, -0.9375754594802856, -1.6657934188842773, 0.416054904460907, -0.648723304271698, 0.5414428114891052, 1.0652812719345093, 0.307271271944046, 1.4959256649017334, -0.4146920144557953, -0.8145109415054321, 0.27764415740966797, -0.16281543672084808, 1.5854729413986206, -1.7585898637771606, -1.1065566539764404, 1.2899729013442993, -0.803180992603302, -0.24482229351997375, -0.5156071186065674, -0.06362397968769073, 2.0720388889312744, 1.0703520774841309, -0.007531690411269665, 1.9056308269500732, -0.9686286449432373, 0.176248237490654, 1.9178740978240967, -1.2776687145233154, -0.9462248682975769, -0.955971896648407, -0.37081918120384216, 0.7791385054588318, 1.5112498998641968, 1.1446962356567383, 1.322918176651001, 0.3622460663318634, -0.541554868221283, -0.9122899174690247, 0.002183526288717985, -0.913764476776123, -1.2283377647399902, 0.542465329170227, -0.8754155039787292, -0.3201415538787842, 2.069988489151001, 0.48145902156829834, 0.5056977868080139, -0.2505156695842743, -0.9197723865509033, 0.5628900527954102, -0.4241677224636078, -0.18308478593826294, -0.6374444961547852, 0.3763900101184845, 0.22274883091449738, -2.2196829319000244, -0.9814381003379822, -0.2099340707063675, 1.8388869762420654, -0.9947884678840637, 0.3375232219696045, 1.0879923105239868, 1.0771023035049438, -0.5437743067741394, 0.8907426595687866, 0.23739676177501678, 0.05376880243420601, -0.5204508304595947, -1.803023099899292, -0.5493133664131165, -1.5938389301300049, 0.8351545333862305, 0.38948968052864075, 1.7228944301605225, -1.534836769104004, -1.6342179775238037, -0.2599266767501831, 0.13811154663562775, 0.8086139559745789, -1.226036548614502, -1.8970955610275269, -1.085283875465393, 3.300605297088623, -1.1031277179718018, -0.7220099568367004, 0.7452561259269714, 0.49810877442359924, -0.5398253798484802, -3.1994268894195557, 0.5183766484260559, -0.6529751420021057, 1.2164244651794434, -0.03439180552959442, 0.47351735830307007, -2.005758285522461, 0.007817280478775501, -0.15269735455513, -0.5296470522880554, 0.3557344377040863, 0.5215749740600586, -0.49737805128097534, 0.5819966793060303, 1.4654425382614136, 0.4993889629840851, -0.1841312050819397, 0.27192771434783936, -0.22531485557556152, 0.25572875142097473, 0.7854230403900146, -0.35654959082603455, 0.6618443727493286, 1.660173773765564, -0.22056207060813904, -0.7647355198860168, -0.4060629606246948, 0.6630030870437622, 0.3919092118740082, -0.5968066453933716, -2.6187360286712646, 0.4890161454677582, 0.43832284212112427, 0.41469696164131165, -0.7040902972221375, -2.2411065101623535, -0.38688182830810547, -0.772457480430603, 0.5731558799743652, 1.5451154708862305, 0.8699582815170288, -0.28885456919670105, 0.8150743246078491, -0.5147525668144226, 0.774875819683075, 0.10919222980737686, -0.4634404480457306, 0.4234868586063385, 0.19661743938922882, 1.438188910484314, -0.019295869395136833, -0.08849328756332397, 1.4673798084259033, 0.43405938148498535, 0.30220746994018555, 2.1054508686065674, -2.3599815368652344, -1.3833439350128174, -0.4919191598892212, 1.4510550498962402, 1.6868363618850708, 0.48839855194091797, -0.43337148427963257, -1.1848162412643433, 0.7935181856155396, 0.6275580525398254, -0.32427501678466797, 1.303299903869629, 1.9426275491714478, -0.6267013549804688, 0.5900862812995911, -1.0156712532043457, 0.5118036270141602, -1.0017659664154053, 0.3267207443714142, 2.1813621520996094, -0.3241310119628906, 1.7359906435012817, 1.6164398193359375, 1.4702837467193604, -0.14000746607780457, -1.18061363697052, 0.2884925603866577, 0.87519770860672, -0.33240559697151184, -0.6647538542747498, -1.0363606214523315, 3.231355905532837, -1.8582448959350586, 0.9991834759712219, -0.3084733784198761, 1.367713451385498, 1.4582114219665527, -0.8536527156829834, 0.1040748804807663, -0.15155692398548126, 0.736409068107605, 0.04138306900858879, 0.9767021536827087, 0.668829083442688, -0.4192309081554413, 1.2396997213363647, 0.6507455110549927, 0.6967126131057739, -0.25227710604667664, 0.8942233920097351, -0.23650552332401276, 1.4801119565963745, 0.5928309559822083, -2.1389002799987793, 0.7977235317230225, 0.7035781741142273, 0.2202211618423462, 0.987251877784729, -1.325478434562683, 0.5145424008369446, -0.5983438491821289, 0.378528356552124, -0.5617437362670898, -1.1301753520965576, -0.1791311353445053, -1.5344429016113281, 0.18851712346076965, 0.8742617964744568, 1.4413152933120728, 0.5508870482444763, -0.32710781693458557, -0.0483885183930397, 0.02827717736363411, 1.3887914419174194, -0.8073453903198242, -0.38403618335723877, -1.9343458414077759, 1.3846020698547363, -1.0681082010269165, -1.1452676057815552, -1.821010708808899, 0.3776572048664093, -0.15825296938419342, -0.2503286898136139, -1.2847799062728882, -0.44067302346229553, 1.2569974660873413, -1.4418058395385742, 0.20821894705295563, 1.4275099039077759, -1.0566538572311401, 0.7179573774337769, 0.7354019284248352, -0.7170832753181458, -0.019570140168070793, 0.728198230266571, 0.9398220181465149, -0.5282438397407532, -1.4046063423156738, 0.2257322371006012, 0.22266478836536407, 0.7346832752227783, 1.4495301246643066, 1.185120940208435, 0.9539763927459717, -0.22919951379299164, 1.3462656736373901, 1.2469595670700073, -0.2644408643245697, -0.39962276816368103, -1.2755473852157593, -0.3912849724292755, 0.10604032129049301, -0.13747794926166534, 0.692025899887085, 0.14151883125305176, -0.7692358493804932, 1.3322162628173828, -1.5929110050201416, 1.1265217065811157, 0.16440390050411224, 0.37914565205574036, -0.561843752861023, 0.23158057034015656, 1.716468334197998, -0.5230008959770203, 0.4197006821632385, 0.23364906013011932, -2.9572997093200684, -0.47725361585617065, 0.40757590532302856, -0.5801024436950684, -0.09796924144029617, -1.1183990240097046, 1.7162610292434692, 0.39009061455726624, -0.10636631399393082, -0.6637086868286133, 1.0307412147521973, 0.22040659189224243, 0.8949521780014038, 0.6844693422317505, -1.272091269493103, 0.5709240436553955, -2.308544158935547, -2.6406824588775635, -0.23886987566947937, 1.748420000076294, 0.5827145576477051, 0.5868804454803467, -0.831283450126648, -0.13493883609771729, 0.5361659526824951, -2.089913845062256, -0.7653740048408508, -0.4763548672199249, 0.9014745354652405, 1.229602575302124, -0.5839805006980896, 1.5176352262496948, 0.16222669184207916, -0.06325722485780716, 1.3406933546066284, -0.7634558081626892, -0.6397113800048828, -1.2340511083602905, -0.7788592576980591, -2.221921920776367, 2.488284111022949, 0.7014798521995544, 1.9129794836044312, -1.5365393161773682, -1.831802248954773, 1.4670580625534058, -0.2779097259044647, -0.34215447306632996, -0.48074328899383545, -1.3485223054885864, -0.734042227268219, 2.119088888168335, -1.0937628746032715, 1.46792733669281, 0.38787829875946045, 0.8205649852752686, 1.162490725517273, -0.9827112555503845, 0.8757071495056152, 0.4536117911338806, 2.746666431427002, 0.4512075185775757, 1.0482177734375, 0.2391623854637146, 0.40618932247161865, 0.31035521626472473, 0.009234639815986156, 1.0952003002166748, 0.35260236263275146, 0.5628849267959595, 0.937782883644104, -0.6196615695953369, -0.24927493929862976, 0.357412725687027, 0.5408771634101868, 1.8080708980560303, -2.0749099254608154, -0.749607503414154, -0.9511321187019348, 0.0850684717297554, -0.23895052075386047, -1.709924340248108, -0.9410361647605896, -0.9363610148429871, -1.7693899869918823, -1.0548179149627686, -0.6413883566856384, 1.0338715314865112, 0.08378145843744278, -0.5365338921546936, -0.7737843990325928, -0.24408479034900665, 1.1843680143356323, 1.2808606624603271, -0.8704200983047485, -0.37676674127578735, 0.1332108974456787, 0.2899605929851532, -1.0702589750289917, 0.5765387415885925, 0.8096770644187927, 0.8899059295654297, 1.8787288665771484, 0.9865936040878296, -0.1559108942747116, -1.419926643371582, -0.06226211413741112, -0.6617076992988586, 1.104094386100769, -0.08852441608905792, -1.2490501403808594, -0.5025092363357544, 1.4913378953933716, -0.6656333208084106, 0.1882135272026062, 1.0533355474472046, 0.34556710720062256, -1.3727366924285889, -1.4131646156311035, -0.8346819877624512, 0.44925224781036377, 0.2663756012916565, -0.7568382024765015, 0.011371023021638393, 0.23882469534873962, -1.6815943717956543, -1.2659319639205933, 0.06686481833457947, 1.1025456190109253, -0.19889360666275024, 0.18772520124912262, 1.4884920120239258, -0.8106886744499207, -0.02987673692405224, -0.2881856858730316, 1.8541910648345947, -0.5197080373764038, -0.8500446081161499, 0.984063982963562, 2.941011905670166, 0.9557345509529114, 1.3183172941207886, 1.338739037513733, 0.5710428357124329, -0.13785943388938904, 1.8556818962097168, 0.5640795230865479, -1.0602965354919434, -0.42747873067855835, 0.07436669617891312, -0.7656161785125732, 0.01915420964360237, -0.35148945450782776, -0.3571849465370178, -1.4221549034118652, 1.5407733917236328, -1.0759063959121704, -1.267290472984314, 0.6938391923904419, -0.163191556930542, -0.09963271021842957, 1.8423821926116943, -0.7013712525367737, -1.269592523574829, -0.8409980535507202, -0.6261940002441406, 0.2970099449157715, 0.5634037852287292, 1.1823716163635254, -0.40109992027282715, 0.517025887966156, -0.9443305134773254, -0.4681321680545807, 0.1562928706407547, -1.1318333148956299, 1.7320783138275146, -0.4181094765663147, -0.896774411201477, -0.9102271199226379, 0.24188244342803955, 1.717306137084961, -1.3495492935180664, 0.15143030881881714, -0.8831983208656311, 0.834039032459259, -0.5362573862075806, -2.2259268760681152, -1.1682049036026, -0.3086549937725067, -0.33190372586250305, 1.5183475017547607, 0.41749194264411926, -1.051293969154358, 0.7881765365600586, -0.14137689769268036, 2.0652332305908203, -0.28867900371551514, -0.6490789651870728, 0.29939067363739014, 1.7889533042907715, -0.39325469732284546, 1.5432764291763306, 0.07910846173763275, 0.2537718117237091, 1.3100544214248657, 1.119065523147583, 0.6159331202507019, 1.9124252796173096, -0.4499320685863495, -2.454457998275757, 0.7417799830436707, 0.231427863240242, -0.19995218515396118, 1.225999355316162, -2.2055985927581787, -0.30375638604164124, 1.310563325881958, 0.3109147846698761, 1.2712477445602417, 0.2703321874141693, -1.2198766469955444, -1.2181570529937744, 0.1788962334394455, -1.3295496702194214, 1.6056102514266968, -0.8961048722267151, -1.82364022731781, -0.2922292947769165, 0.504667341709137, -0.11795013397932053, -0.22850355505943298, -0.49848753213882446, -1.829942226409912, 0.11415518075227737, 0.767385721206665, -1.0313938856124878, 0.4766293168067932, -1.7563821077346802, 0.16852223873138428, -1.0866023302078247, 0.8074703216552734, -1.4986063241958618, -1.2568796873092651, -0.37685462832450867, -0.08961267024278641, -0.35765644907951355, 0.6653513312339783, -0.05698070302605629, -0.35984256863594055, 1.6284034252166748, 0.9158811569213867, 0.20308683812618256, 0.327973872423172, -1.8808190822601318, -1.306904673576355, 2.120358943939209, -1.5506370067596436, 0.9441961646080017, 0.39116573333740234, -0.437399685382843, 2.5407304763793945, -0.2983216941356659, 1.4734892845153809, -0.6977433562278748, 0.2806962728500366, 0.3104807436466217, 0.19660913944244385, 1.1373932361602783, 0.269223153591156, -1.3233532905578613, 0.17706887423992157, -1.7846089601516724, 1.3841500282287598, 0.964046835899353, 0.8791561126708984, -0.24478214979171753, 0.4822591245174408, -2.262791395187378, -0.6630811095237732, 0.11351140588521957, -2.3652944564819336, 0.13565871119499207, -0.4509366750717163, 0.26246944069862366, -0.633801281452179, 1.3326425552368164, -0.10827410966157913, 1.1771965026855469, 0.8529519438743591, -1.2928738594055176, -2.1866183280944824, -0.38439106941223145, 0.27305397391319275, 0.31291475892066956, -0.519102156162262, 2.1992383003234863, -0.1357472687959671, 1.495552659034729, -1.75140380859375, 0.4746882915496826, -0.16530963778495789, 1.0645830631256104, 2.5297794342041016, -0.6903075575828552, -0.5940001010894775, -0.12490882724523544, -0.31560778617858887, -0.024828974157571793, 0.040798645466566086, -0.587444007396698, 0.007899089716374874, -0.23677779734134674, 1.3452060222625732, -0.6073068380355835, 0.8240179419517517, 0.5986742973327637, 0.16685166954994202, 1.1304454803466797, -1.9383450746536255, 0.4103485345840454, -0.7540855407714844, -2.3641984462738037, -0.39639726281166077, 0.3113541603088379, -2.0502066612243652, 0.26978227496147156, -1.0393439531326294, -0.8783811330795288, 1.5180894136428833, 1.1839470863342285, 0.6623528599739075, 0.39367109537124634, -0.9402511119842529, 1.8601704835891724, 1.109928011894226, 0.7128311395645142, 1.45442533493042, 0.21132813394069672, 0.05869413912296295, 0.7973194122314453, 0.8306521773338318, 1.6861491203308105, -0.38253119587898254, 1.8551812171936035, 0.10898423939943314, 1.0866895914077759, 0.37622225284576416, -1.8155654668807983, -0.13791286945343018, -0.14933717250823975, -0.3754732310771942, 2.5752830505371094, -0.3785961866378784, 0.5970938205718994, 0.7128015756607056, -0.4201318323612213, -1.430572748184204, -0.9412431716918945, 1.5892817974090576, -0.7443103790283203, 0.3464356064796448, 0.3096601366996765, -1.1070961952209473, 0.832427442073822, 0.5256113409996033, -1.3402981758117676, -0.9707240462303162, -0.4504861533641815, 1.0859100818634033, -0.7349144816398621, -2.0300445556640625, 1.7827898263931274, -1.0944197177886963, -0.6299670934677124, -0.6534979343414307, -2.067902088165283, 1.3160443305969238, 0.9990230202674866, 1.0820132493972778, 0.2403668910264969], [3.2732276916503906, -1.4326262474060059, 0.08576612174510956, -0.32789289951324463, -2.4104762077331543, -0.8524674773216248, -0.26041722297668457, -0.1670616716146469, -0.3967569172382355, -0.07948307693004608, 0.16059094667434692, -0.2841622829437256, 1.280636191368103, -0.08225618302822113, 1.3942691087722778, 1.501094102859497, 0.18753927946090698, -0.8707332611083984, 0.4392290711402893, 1.3750724792480469, -2.310857057571411, -0.30985191464424133, 2.475829601287842, -0.2092994749546051, -0.7502995729446411, -0.7794491052627563, -0.622065007686615, -0.5144359469413757, -1.3651959896087646, 0.13037057220935822, 0.013852369040250778, -1.431694507598877, 1.274221658706665, 1.11076021194458, -0.5641658902168274, -0.28695979714393616, 0.4127286672592163, 0.015705671161413193, 0.5900428891181946, 1.1439707279205322, 1.636017918586731, 1.0924015045166016, -0.1641114503145218, -0.32468101382255554, -1.5454367399215698, 0.15364602208137512, 1.716671347618103, 0.6499969363212585, -0.6306609511375427, 0.37544941902160645, -0.39376747608184814, 0.10394161194562912, -0.6198724508285522, 1.7189702987670898, -2.2070703506469727, -1.0571423768997192, 0.652655839920044, -2.2411932945251465, -0.4433800280094147, -1.314754605293274, 0.3448668420314789, 0.009701893664896488, -0.09811045974493027, -0.00970652885735035, -0.23702266812324524, 1.98464035987854, -0.35759878158569336, -1.0788922309875488, 0.061641838401556015, -1.590362310409546, -1.123458981513977, 1.0264973640441895, 0.17820684611797333, -0.11718665063381195, 1.1401007175445557, -0.9621736407279968, -1.9316285848617554, 1.063501000404358, -0.27034300565719604, 2.513941764831543, -1.6334583759307861, 0.0718780979514122, -0.5751168131828308, 0.2411363124847412, -0.08716582506895065, -0.36485517024993896, 1.592492938041687, 1.1292383670806885, 0.40056121349334717, 0.6978684663772583, 0.36867234110832214, -1.8088046312332153, -0.8574612736701965, 0.36830082535743713, 0.21920114755630493, 1.808619499206543, -0.7493268251419067, -0.130567267537117, 0.06377211958169937, 0.2709585130214691, 1.3073303699493408, -0.8643958568572998, 1.0922484397888184, 0.43319085240364075, -1.488219976425171, -1.3377563953399658, -0.08540556579828262, 1.8580971956253052, 0.40956032276153564, 0.6514831781387329, 0.3587650954723358, -1.221747636795044, -0.6507357358932495, -1.6597981452941895, 0.8998413681983948, -0.42286476492881775, 1.3308764696121216, -0.07909727096557617, -1.501686453819275, -0.4217556416988373, 1.0586249828338623, -0.3201506435871124, 0.08488515764474869, -1.5912894010543823, -0.38501161336898804, 0.7834437489509583, -0.09614534676074982, 0.23204046487808228, 0.29744574427604675, -0.8104091286659241, 0.07524097710847855, -0.07016508281230927, -0.7050802707672119, 1.842074990272522, 1.6934869289398193, 1.407400369644165, -0.04928077384829521, -1.5695983171463013, 0.8214635252952576, -0.670816957950592, 0.8401554226875305, -0.374175101518631, -0.6954014897346497, -1.9354413747787476, 0.34189870953559875, 0.6357929706573486, 0.6959586143493652, 0.9965394139289856, -1.6866213083267212, 0.19383804500102997, 0.9462235569953918, 1.1370705366134644, -1.6307276487350464, 0.32061833143234253, 1.1263102293014526, 0.7430835962295532, -0.8926247358322144, 1.0083364248275757, 0.331466943025589, 1.539768099784851, -0.6651175022125244, 0.20702241361141205, 0.06969541311264038, 0.7518766522407532, -1.419697880744934, -0.04124514013528824, -1.3127243518829346, -0.1893422156572342, -0.562842607498169, -0.029891226440668106, 0.8500314950942993, 0.40397512912750244, 0.5295793414115906, 0.4711354374885559, -0.10630005598068237, -0.6290395259857178, 0.359324187040329, -1.782119870185852, 0.2458002120256424, -0.5170716047286987, -0.7352774143218994, 0.08773908764123917, 0.10938441008329391, -1.110041618347168, -0.013981220312416553, 0.5594531297683716, -0.7203802466392517, 0.7293131947517395, -1.4648712873458862, -0.8783168792724609, 0.7245556712150574, -0.2732691168785095, -0.5666722655296326, 0.508558988571167, 0.7557640075683594, -0.053451597690582275, 0.37059924006462097, 0.06824751198291779, -0.5704014897346497, 0.26177820563316345, -0.2726665139198303, -2.108954906463623, -0.5292848944664001, 0.5081583261489868, 0.03297705203294754, -0.038254667073488235, -0.6426332592964172, -1.5597003698349, -1.0235408544540405, -1.3948842287063599, -1.5805702209472656, 0.449943870306015, -0.5095123052597046, -1.2747927904129028, 0.4535273611545563, 0.7786860466003418, -1.5304663181304932, 0.6446101665496826, -1.300795555114746, 2.562025308609009, 0.31455302238464355, -2.0598866939544678, 1.1535439491271973, -1.0212448835372925, 0.4341508448123932, 1.725103497505188, 0.9578354358673096, -1.6462304592132568, 0.11101148277521133, -0.6522006392478943, 0.2298366129398346, -0.34214988350868225, -0.14784203469753265, 1.5492324829101562, -0.644245982170105, 1.2221477031707764, -0.9852613806724548, -0.23082248866558075, -1.5296506881713867, 1.1576136350631714, 0.07138579338788986, -1.3866899013519287, 0.10306810587644577, -0.41614899039268494, -0.47738492488861084, -0.5634847283363342, 0.6296229958534241, -0.6759562492370605, 1.0127488374710083, 0.6117656230926514, 0.9512954354286194, 1.10902738571167, 0.4446694552898407, -0.7094317674636841, 0.8943852782249451, -1.4269027709960938, -0.6764688491821289, 1.1463491916656494, -1.8082952499389648, -0.8859174847602844, -1.3372513055801392, 0.3736979067325592, -1.02860689163208, 0.19449077546596527, 0.043949492275714874, 1.979201078414917, 2.0958123207092285, -0.8327865600585938, -0.6717298030853271, -0.05900539085268974, -0.3766714334487915, -0.3415672481060028, -0.15892665088176727, 0.9098086953163147, 1.372442603111267, -1.4148451089859009, 0.5944443941116333, 0.14774231612682343, -0.9890608787536621, -0.05647042393684387, -0.5188241600990295, 0.10929341614246368, 0.10399053990840912, -1.7016546726226807, -0.5163238644599915, 0.23603498935699463, -0.6285950541496277, -3.1091034412384033, 1.3137089014053345, -0.1568969339132309, 0.6279906034469604, -0.6560253500938416, -0.8922040462493896, 0.9882558584213257, 0.10492151975631714, -0.486226350069046, 0.3620564341545105, -0.3193877935409546, 0.2488717883825302, 0.8613300323486328, 0.04190696403384209, -0.4379061758518219, 0.6884281635284424, 1.204564094543457, 0.25200197100639343, 0.03607378154993057, 0.04703017324209213, -0.6341204643249512, 0.6808624267578125, -0.39703401923179626, -0.3545634150505066, 0.4031873345375061, 1.4778506755828857, 1.1717076301574707, -1.062908411026001, 1.439521312713623, 0.8025926947593689, 0.1514747589826584, 0.9817519187927246, 0.2966782748699188, 1.922666072845459, -0.7173696756362915, 1.4971003532409668, -0.41782450675964355, -0.2691618502140045, 0.8883659839630127, -1.9553759098052979, 0.9094483256340027, 1.319114327430725, -1.680073618888855, -0.633281946182251, 1.060203194618225, -0.9616311192512512, -1.3203486204147339, 1.8076236248016357, -0.3892544209957123, 1.6459420919418335, 1.0479435920715332, 0.391729474067688, -0.7843952178955078, 1.1159218549728394, 0.5206010341644287, -0.5596349239349365, -0.7297086119651794, -0.33412590622901917, -1.1359909772872925, -0.4860326647758484, 0.4842219650745392, -0.8894041180610657, -0.35507723689079285, -0.30706682801246643, -0.13724498450756073, -0.6799649000167847, 0.4193349778652191, 1.1986234188079834, -1.2843053340911865, -0.5419999361038208, -0.1517971009016037, 1.1768007278442383, -0.13478809595108032, -0.522971510887146, 0.8836696743965149, -0.1663135439157486, -1.0703206062316895, 0.25000908970832825, 0.10676278918981552, -0.6321485638618469, -1.6834800243377686, -0.5440505146980286, 0.3968663215637207, 0.3381110429763794, -1.5034687519073486, -1.1678626537322998, 0.9026063680648804, -1.8426244258880615, 0.8732308149337769, 0.6527636647224426, 2.015495538711548, -1.6676045656204224, 0.8358821272850037, -0.6080018281936646, 0.027867209166288376, -1.0483587980270386, 0.9192641973495483, 0.5542119145393372, -1.1629598140716553, -1.4636321067810059, 0.3239675760269165, -0.47687599062919617, 0.19949114322662354, 0.18514060974121094, 0.35669150948524475, 0.0713285431265831, -0.5130950212478638, 1.0061553716659546, 0.7355831265449524, -0.035393279045820236, 0.38582512736320496, -0.008950408548116684, -0.10535839200019836, -0.4893485903739929, 1.3763456344604492, -0.5269622206687927, 1.3603503704071045, 0.6695204973220825, 0.5828642249107361, 0.7016631364822388, 0.44255688786506653, -1.2608070373535156, -1.492821455001831, -0.7636751532554626, -0.5978853702545166, -0.04890973120927811, -0.17302407324314117, 0.3164033889770508, -1.7196252346038818, 0.09780445694923401, -0.7836015820503235, -0.7513781189918518, -0.7647578120231628, -0.08814771473407745, -1.0017905235290527, 0.6183966994285583, 0.6447989344596863, -0.7979119420051575, -0.7958347797393799, -0.11332419514656067, 1.109632134437561, 0.5300443768501282, -0.7856763601303101, -0.2061791568994522, -1.3263742923736572, -0.5754131078720093, -1.1959589719772339, 0.4208774268627167, -0.36765044927597046, -0.17918722331523895, -1.7079375982284546, -0.5871677398681641, -0.522705614566803, 0.9729191660881042, -1.8086309432983398, -1.0716532468795776, -1.0886861085891724, 0.29471346735954285, -0.7221387624740601, 1.069154143333435, 1.445449948310852, -2.0357542037963867, 1.177430272102356, 0.6065773367881775, 0.7102747559547424, 0.44544118642807007, -0.3412761390209198, -0.5641161799430847, -0.6523860692977905, -0.9445432424545288, 0.22162765264511108, -0.13771748542785645, 0.12488869577646255, 0.09333427250385284, -1.726348638534546, -0.1268308460712433, -0.8091046214103699, -0.9525368213653564, -0.1212654709815979, 0.7907422780990601, -0.2882816791534424, 0.22042728960514069, 1.5082272291183472, 0.6470072865486145, 0.5331809520721436, -0.7266383171081543, 0.8471091985702515, -0.34290778636932373, 1.5095937252044678, 1.9556955099105835, -0.6744992733001709, 0.3035876750946045, 0.2052280753850937, 1.5523861646652222, -1.2408734560012817, 1.2250313758850098, 1.1440564393997192, 0.16911372542381287, 0.6299594640731812, -0.015245816670358181, 0.40197306871414185, -0.0847088173031807, -1.29597008228302, -0.47954854369163513, -0.4434421956539154, 0.6289181709289551, -0.3469647467136383, 0.5434751510620117, 1.1056432723999023, -0.30568599700927734, 0.5861474871635437, -1.2443137168884277, 0.20009282231330872, 1.0718218088150024, 1.014975905418396, 3.5673112869262695, -0.4409671127796173, 0.18436837196350098, 0.13930298388004303, -0.5929490923881531, 0.028835270553827286, -1.1736884117126465, -0.2304847538471222, 3.0203986167907715, -0.6822523474693298, 0.9831365346908569, 1.3868460655212402, 1.5015201568603516, 0.7088115811347961, 0.3626127243041992, 0.9326211810112, 1.060015082359314, -1.435847282409668, 1.5338211059570312, 0.1744007170200348, 1.0180249214172363, 1.0685348510742188, 0.4230239689350128, -1.1126775741577148, -1.6690762042999268, 0.9320728778839111, -0.6101217269897461, -1.4374793767929077, -0.4998711347579956, -0.002834590617567301, 1.563658356666565, -1.0722678899765015, 0.46185317635536194, 0.15708063542842865, -0.4446231722831726, -1.014357566833496, 1.4231094121932983, -0.2042001336812973, 0.2998610734939575, -0.2489088475704193, 1.818052053451538, 0.31657183170318604, -0.4377683401107788, 0.9046235084533691, 0.15963315963745117, -0.9867634177207947, -1.4866127967834473, -0.6034660935401917, -0.550509512424469, -1.5842849016189575, -2.1086671352386475, 0.08413279801607132, 0.003395203733816743, 0.5348941683769226, -1.2621228694915771, -0.09992027282714844, -0.0009984277421608567, -0.4402512311935425, -0.9596166610717773, -0.07679780572652817, -0.6163137555122375, -0.7154316902160645, -0.4595522880554199, -1.0566318035125732, -1.1085840463638306, -0.2633673846721649, -0.10762066394090652, 0.6040247082710266, -0.3608371615409851, -0.9232916831970215, -1.2750123739242554, 0.7855157852172852, -0.6493817567825317, -1.1817185878753662, 1.3302538394927979, 0.920102596282959, -0.6697805523872375, -0.19037850201129913, -0.9396185874938965, -0.22762344777584076, 1.2986669540405273, -0.2039039582014084, 0.9434242248535156, 1.2223377227783203, -1.909683346748352, -0.7994086742401123, 1.2624633312225342, -0.6573380827903748, -0.6673133969306946, -0.7998536825180054, 0.8196184635162354, -0.17181019484996796, 2.3786094188690186, 0.2351534515619278, -2.117511034011841, -1.295159935951233, 0.09008654206991196, 1.8315507173538208, 0.9193454384803772, -0.7152693271636963, -0.6145361065864563, -1.6496227979660034, 0.9659966230392456, -1.6817933320999146, 0.7210935354232788, 0.8792908787727356, 0.3623609244823456, 1.1138193607330322, 2.1475753784179688, -0.8094921708106995, 1.7383432388305664, -2.136404275894165, -0.4001534879207611, -0.21634535491466522, -0.1001284196972847, 1.1321702003479004, -0.23220698535442352, -0.9391445517539978, -0.9851633310317993, -0.4003034830093384, 0.5093330144882202, -1.1112287044525146, -0.7496314644813538, -0.6930408477783203, 0.4449767470359802, 0.27159377932548523, -1.166816234588623, -1.252226710319519, -0.7757284045219421, 0.265610933303833, 1.239011526107788, 0.08439398556947708, -0.733564019203186, 2.068077564239502, 0.7658389210700989, -2.429597854614258, 0.6259574294090271, 1.11574125289917, 1.0360976457595825, -0.5827099680900574, -1.6387712955474854, 0.041770175099372864, 0.6230417490005493, 0.5107347369194031, 0.30974385142326355, -0.7572271823883057, 0.06357128173112869, -0.0908389613032341, -0.551916241645813, 1.7818981409072876, -0.34286391735076904, -0.3243288993835449, 1.8203895092010498, -0.4510249197483063, -0.4770224988460541, -0.1829509437084198, -1.3457223176956177, 0.5456275343894958, -0.4376218914985657, 0.14485102891921997, 1.0282936096191406, -0.6867288947105408, -0.5724254250526428, -1.6912389993667603, 0.6811192035675049, -0.5750874280929565, 0.7118764519691467, -0.19357949495315552, -0.6427901387214661, -0.9549718499183655, 1.0946266651153564, 0.5498635768890381, 0.9548963308334351, -0.15797169506549835, -1.4800033569335938, -0.21443970501422882, 0.5532279014587402, 0.6352826952934265, 0.8350695371627808, 1.027267575263977, 0.9047111868858337, 1.6210627555847168, -1.7838870286941528, 0.6620467305183411, 0.6275163888931274, 0.8135183453559875, 0.11769786477088928, -0.7375169992446899, 0.2566777765750885, 0.7859528660774231, 0.11987647414207458, 1.0928168296813965, 0.9399109482765198, -2.0839767456054688, 2.3498482704162598, -1.0708492994308472, 2.4623608589172363, 0.7926816940307617, -0.5858049988746643, 0.3179114758968353, -0.06710135191679001, 0.6199281215667725, -1.3996895551681519, -0.3038055896759033, -2.373401641845703, -0.0018137304577976465, -0.5057306289672852, -1.1171354055404663, -1.0261698961257935, -0.18624024093151093, -0.6032435297966003, -0.17400169372558594, -0.5202615857124329, -0.581221342086792, -0.6659215092658997, 1.0639845132827759, -0.3995739817619324, -0.0237455815076828, 0.7303004860877991, 1.3135446310043335, -1.4243388175964355, -0.3537812829017639, -0.965740442276001, 0.17820216715335846, 0.16999439895153046, -1.679962158203125, -1.0156183242797852, -0.8059147000312805, 0.9065230488777161, -2.142859935760498, 0.5250865817070007, 0.02191033400595188, -0.6009505391120911, 0.15073111653327942, -0.7178676128387451, 1.9166771173477173, 0.08365476876497269, -0.4903925359249115, 1.1148492097854614, 0.07285487651824951, -0.8576450347900391, -0.2296055257320404, 0.405775785446167, 0.08062738925218582, 0.2561505436897278, 0.26827287673950195, 0.7135244607925415, -0.05450987443327904, 1.8493956327438354, 0.5918864011764526, 1.2361071109771729, 0.04546704888343811, -0.6481087803840637, -0.027397265657782555, 0.19679543375968933, -0.5326281189918518, -2.902712345123291, -0.551166832447052, 1.15830397605896, 1.5810022354125977, 1.5065174102783203, -0.3532458543777466, -0.3678855299949646, 1.7277625799179077, 0.7343124747276306, -1.1649571657180786, -1.0895551443099976, -0.902155876159668, -0.5989304184913635, -2.403777837753296, -0.7736782431602478, -0.62723308801651, 0.65675288438797, -0.31350794434547424, -1.7329713106155396, 0.3420681655406952, 0.9381769299507141, 1.7533916234970093, 0.9790100455284119, 0.887135922908783, 1.7484426498413086, -1.2241500616073608, 1.8061933517456055, 1.8824130296707153, -0.7167029976844788, 1.161973476409912, -1.3483363389968872, 2.9494147300720215, 0.8988279700279236, 0.031326454132795334, -0.8932703733444214, 0.15789411962032318, 0.7961908578872681, 0.07977962493896484, 1.1627459526062012, 0.3850899338722229, -0.48946869373321533, 1.1001945734024048, 3.189073085784912, -2.0696680545806885, 0.11438465863466263, 1.0663965940475464, -0.5741991400718689, -1.0296965837478638, -0.7000496983528137, 0.19054685533046722, -1.0445725917816162, -1.7761727571487427, -1.4074822664260864, 0.2319781631231308, -0.17965900897979736, 0.9651010036468506, 0.1207309141755104, -1.1784987449645996, -0.7218870520591736, -0.8613566756248474, 0.8150973916053772, -0.10899592936038971, -0.2095891684293747, 0.2920805811882019, 0.5511004328727722, -0.7352827191352844, -1.4503347873687744, -0.11593370884656906, -0.47743460536003113, 0.3524051904678345, -1.3727442026138306, 1.393254280090332, -0.11905152350664139, 1.2917627096176147, 0.5210446715354919, -0.11773208528757095, -0.934425413608551, -0.16670095920562744, -0.8871771097183228, 1.6970772743225098, -0.8113023042678833, 0.04613278806209564, -0.7830929160118103, -0.6320576667785645, 0.16719764471054077, 2.114938974380493, -1.1465184688568115, 0.08030100166797638, 1.0949546098709106, 0.01656132936477661, -0.47267940640449524, 2.8966751098632812, 0.37298583984375, -0.5921692252159119, -1.6615408658981323, -0.9433864951133728, -0.5785303711891174, 0.9783948659896851, 0.035062529146671295, -0.8032646179199219, 1.7297933101654053, 1.1550370454788208, -0.41155683994293213, 0.9132372736930847, 0.04969555512070656, -0.5175139307975769, -1.0658416748046875, -0.20912989974021912, 0.11002882570028305, 0.3161357641220093, 0.17875659465789795, 1.0579993724822998, 1.5154297351837158, -0.02087102085351944, 0.6771761775016785, 1.274680733680725, -0.24154174327850342, -0.7013731598854065, -2.0616202354431152, -0.24207167327404022, -2.1513290405273438, -1.0055196285247803, 0.41423743963241577, 0.8350659608840942, -1.3102165460586548, -2.1053590774536133, -1.380631446838379, 0.3243478834629059, 2.095951795578003, -0.11143551021814346, -2.0513670444488525, -1.1758959293365479, 1.1898902654647827, 0.4166967272758484, 0.638815701007843, -0.14055192470550537, -0.08746081590652466, -2.0720248222351074, 0.912452220916748, 1.6349561214447021, -0.6527641415596008, -0.7882694602012634, -0.09787412732839584, -0.378527969121933, -1.284877061843872, -1.162109375, 0.23162078857421875, 1.1910109519958496, -0.6086152791976929, -1.5126032829284668, 0.4439091384410858, 1.1310937404632568, -0.7105218768119812, -1.1273638010025024, -0.5661898255348206, -0.2694997191429138, 1.5110726356506348, -1.0620620250701904, 0.3518851101398468, 0.45665988326072693, -0.44298484921455383, 1.7807590961456299, 0.6445503234863281, -0.6059103012084961, -0.28282439708709717, 0.050780102610588074, 1.6694740056991577, -0.315757691860199, 0.7461427450180054, 0.44474464654922485, -1.026462435722351, 0.3434520959854126, -1.0671406984329224, -0.4233205318450928, -0.14164400100708008, 0.6045663952827454, -0.11205337941646576, -0.42172592878341675, -0.5652124285697937, -0.45104923844337463, 0.17213821411132812, -1.3549357652664185, -0.7203165292739868, -0.020635245367884636, -2.486882448196411, 0.6467735171318054, 0.8566967248916626, -0.725279688835144, -0.07952059060335159, -0.09864263981580734, -0.4496613144874573, 0.13944891095161438, -1.1889351606369019, 0.7590968012809753, -0.6861369609832764, 1.0561244487762451, 2.6870663166046143, 0.26473096013069153, 0.48823845386505127, 0.2995580732822418, -0.11562822014093399, 0.6959853768348694, 1.0416502952575684, -0.5440841913223267, 1.3410097360610962, -0.24565909802913666, 1.4944301843643188, -0.19581827521324158, -2.1069986820220947, -0.41552937030792236, -0.9402577877044678, -0.20668281614780426, 0.8323878049850464, -0.6934300661087036, -0.6725265979766846, 0.15019501745700836, -1.4208316802978516, -0.47103556990623474, 1.3006439208984375, 1.4739469289779663, -0.5705240964889526, -0.3819601237773895, 0.7485178709030151, 1.2367621660232544, -0.3563151955604553, -0.8675207495689392, -0.7140162587165833, 1.2010722160339355, 0.5514052510261536, -0.7877522706985474, 1.657217025756836, -0.28888264298439026, -0.7833958864212036, 1.0739519596099854, 1.0733839273452759, -0.658380925655365, -0.5101917386054993, 0.6535712480545044, -0.2902759313583374, 1.2732518911361694, 0.8794219493865967, -0.4909162223339081], [-2.785942792892456, 0.10056933015584946, -0.01538501214236021, 2.225454330444336, -1.002367377281189, -0.10403374582529068, 1.6493291854858398, -0.1668483316898346, 0.9911093711853027, 0.2756485641002655, -0.01437193900346756, 0.6910043954849243, -0.3619421422481537, -0.8401368260383606, 0.5256587862968445, -0.28461506962776184, 0.4774847626686096, 1.3869210481643677, 0.8919420838356018, -1.107468605041504, -0.8812538981437683, -1.3305654525756836, -0.5001844763755798, -0.6995905637741089, 0.5013925433158875, 1.2758833169937134, 0.46279457211494446, -2.0223660469055176, -0.23175691068172455, 0.07605810463428497, 1.3366448879241943, 0.4811779260635376, 0.8144302368164062, 0.2928929626941681, -1.4625734090805054, 0.5484941005706787, 1.0686147212982178, -1.0800366401672363, 1.7852076292037964, 2.3322341442108154, -0.8176872730255127, -0.5964502096176147, -1.0260601043701172, 0.3039773106575012, -0.8102144598960876, 1.7806837558746338, -0.5206454396247864, 0.006084630265831947, 0.7532508969306946, -0.3666476309299469, 1.661636471748352, 0.004683622159063816, 0.24243110418319702, 0.7920036911964417, -1.2625850439071655, 0.21334950625896454, -0.5892475843429565, 0.2808832824230194, -1.071326732635498, 1.4275027513504028, 0.6683521270751953, -0.32268622517585754, -0.08973655849695206, -1.6086349487304688, 0.48136091232299805, -1.1775418519973755, 0.873238742351532, -0.6808737516403198, 1.6913580894470215, -0.265873521566391, -0.5241625308990479, 1.1472299098968506, -0.12409181892871857, 2.026481866836548, 2.0486998558044434, -0.5572580099105835, 1.2261829376220703, 0.6659159064292908, 0.21805201470851898, -1.8022713661193848, -0.05654536932706833, -0.4776655435562134, 1.1926982402801514, 1.2784969806671143, -1.0840094089508057, -1.352274775505066, 0.24642325937747955, -0.38440269231796265, 0.9618211388587952, -0.50386643409729, 1.3858751058578491, 1.7485994100570679, -1.2116349935531616, -0.9311841726303101, 0.16196748614311218, -0.2306334525346756, 1.954773187637329, -0.0922708585858345, -0.18976104259490967, -1.5163140296936035, 0.5690574645996094, -0.9476891756057739, -0.6061000227928162, -1.9566380977630615, -0.2070157378911972, 0.15937048196792603, 0.1928853690624237, -1.5225390195846558, 0.1829889863729477, 0.3792528510093689, 0.7313146591186523, -1.4871175289154053, 0.23890957236289978, 0.2612496018409729, -0.0760100781917572, 0.34789663553237915, -1.0448111295700073, -0.8278177976608276, 2.286282539367676, 1.1180987358093262, 0.8371188640594482, 0.4227409362792969, -0.3331139087677002, -0.721198558807373, -1.3401923179626465, -0.16838869452476501, -1.6816905736923218, -0.00589874479919672, -0.8009464144706726, -0.38837796449661255, -0.5258849263191223, 1.8006924390792847, 1.943821668624878, -1.1254684925079346, 2.2226641178131104, 0.9667279720306396, 0.632936954498291, 1.1696771383285522, -2.4694859981536865, 0.3038582503795624, 1.8215856552124023, -0.397116482257843, 1.3514695167541504, -0.5575533509254456, -0.9451057314872742, 0.5276089906692505, -0.281904399394989, -0.5971977114677429, -0.1940467655658722, 0.19336630403995514, 0.8282409906387329, 0.0034860463347285986, 0.19435609877109528, -0.08308684080839157, -0.6756108403205872, -0.7915363311767578, -0.34897059202194214, -0.6120470762252808, -0.4492850601673126, 0.6536174416542053, 0.13391342759132385, 0.11103737354278564, -0.8472356200218201, -0.40043285489082336, -0.3318954408168793, -0.644652247428894, 0.752005934715271, 1.8611152172088623, 1.065205693244934, -0.9258942604064941, -0.3567332923412323, 0.6750712990760803, -0.25203773379325867, 0.124990314245224, -1.2066665887832642, -1.2169902324676514, 0.5921534895896912, -0.6818658113479614, 0.6822362542152405, 0.9620978832244873, -0.7281033992767334, -0.8037620782852173, 1.6192330121994019, -0.106594517827034, 0.44247353076934814, 0.42737066745758057, -0.11280351132154465, 0.12017311900854111, 0.7475338578224182, -0.13255898654460907, -1.7780025005340576, -0.37565070390701294, -0.5137608051300049, -0.42468252778053284, 1.0375975370407104, -1.616410255432129, 0.6744170188903809, -1.2442947626113892, -0.5670765042304993, 0.5068130493164062, -0.8604033589363098, 1.297218680381775, 0.9931066036224365, -0.3269566297531128, -0.34139567613601685, -0.4370616376399994, -2.5845959186553955, -0.7899416089057922, -0.7233834862709045, 0.10579467564821243, 0.6232095956802368, -3.34254789352417, -0.8290130496025085, 0.667599618434906, 0.6080718636512756, 1.4520139694213867, -0.5872042179107666, -1.276552677154541, -0.04261567443609238, 0.5261742472648621, -1.069409966468811, -0.9480657577514648, -1.2489149570465088, 0.5761538743972778, -1.2343143224716187, 0.07732219994068146, -0.0430561788380146, -0.28202009201049805, 0.3582209050655365, -0.025060798972845078, -1.3215785026550293, -0.9385651350021362, 0.32126176357269287, 1.7163664102554321, 0.7268438935279846, 0.0422331802546978, 0.22044673562049866, -0.8781000375747681, 1.420742392539978, 1.1155004501342773, -2.2221121788024902, 0.6948548555374146, -1.7874395847320557, -0.04900744557380676, 1.2307541370391846, -0.8020151853561401, 0.07729672640562057, -0.22445333003997803, -3.2846643924713135, 2.083717107772827, 0.5839570760726929, -2.6031744480133057, 1.4761407375335693, -0.6968451738357544, -0.5695924162864685, -0.5524478554725647, 0.9880194067955017, 0.49623537063598633, -0.9830886125564575, -0.09211330115795135, -1.1616291999816895, 1.1547589302062988, 0.27897292375564575, 0.9197271466255188, -1.3264442682266235, -0.010691859759390354, 1.8354469537734985, -0.5726884603500366, -0.11663331091403961, -0.5527477860450745, -0.7236055731773376, 0.11061165481805801, -0.21891574561595917, -0.8760361075401306, -0.8337336778640747, 0.5058481097221375, 1.4370992183685303, 0.36911630630493164, 0.48022300004959106, -0.6535972356796265, -0.34790775179862976, -0.8374881148338318, 0.02206178940832615, 0.013182888738811016, 1.2302244901657104, 1.3357030153274536, -0.4501102864742279, 0.7637640833854675, -0.15581750869750977, -1.1720982789993286, -0.5930302739143372, 0.3229505121707916, 0.2740674912929535, 0.6931359171867371, -0.7698983550071716, -0.22606179118156433, -1.8208484649658203, 0.9152842164039612, 3.4374983310699463, 0.3508489429950714, -2.1545486450195312, 0.02427651360630989, -0.0293339304625988, 0.8218543529510498, 0.609031617641449, 1.5338897705078125, 1.2194589376449585, 0.4631563723087311, -0.6331737041473389, -0.6024254560470581, 0.19330397248268127, -0.42363524436950684, 1.1290391683578491, 0.7487088441848755, 0.8166717290878296, -2.481024742126465, -0.9183474183082581, 1.1916863918304443, 0.1680484414100647, 0.15365883708000183, 0.8641029000282288, -0.5934807658195496, -0.6296699047088623, -0.36915087699890137, -1.213889479637146, -0.24535802006721497, -2.0327625274658203, 1.4960826635360718, 1.768652081489563, 0.729274332523346, 0.20396143198013306, -1.9716213941574097, 2.104060649871826, 0.6950942277908325, -0.7469016909599304, -0.8515223860740662, 0.215719535946846, -0.7890348434448242, 0.45944616198539734, 0.9960584044456482, -1.2871137857437134, -1.545632243156433, -0.6743455529212952, 0.9968156814575195, 0.2716684341430664, 0.1493447870016098, -0.19335272908210754, 1.206682562828064, -0.9945492744445801, 1.3325273990631104, -0.7734037041664124, -0.421720951795578, 0.4113522469997406, 1.3482378721237183, 0.20328286290168762, -1.3184165954589844, 0.08138656616210938, -1.6370784044265747, 0.8903610110282898, 0.13461121916770935, -0.08006524294614792, 0.6031178832054138, -1.1619292497634888, -1.9120562076568604, -0.35646969079971313, -0.42201247811317444, -1.9919483661651611, 0.14886035025119781, 0.973504900932312, 1.5702497959136963, 0.6665960550308228, 0.011287190951406956, -2.704575538635254, -2.480792284011841, 0.9032522439956665, 1.4930813312530518, -0.7116197347640991, 0.11985185742378235, 0.605167806148529, 0.3636355400085449, 0.8009308576583862, -0.7459663152694702, -1.7445483207702637, -0.2033787965774536, -0.025012679398059845, -2.7039244174957275, -0.12161006033420563, 2.755110740661621, 0.8847886919975281, -0.24220021069049835, 0.6795443296432495, 0.656981348991394, 0.13499975204467773, -0.26785463094711304, -1.3704311847686768, -1.2016208171844482, 0.6364750862121582, -0.6345766186714172, -0.38839712738990784, 0.7103890180587769, 0.6898196935653687, -0.5182366967201233, -1.6068466901779175, -0.5054689645767212, -0.9463163018226624, 0.6288233399391174, -1.7542431354522705, 0.09206689894199371, 0.4256588816642761, -1.945345401763916, 0.29422688484191895, 0.24584831297397614, 1.6477679014205933, 0.542008638381958, 0.8491106629371643, 0.19968603551387787, 1.5556334257125854, -0.5951604247093201, 1.8135415315628052, 1.1261776685714722, -1.6873834133148193, -0.6011731028556824, 1.4067797660827637, 1.140290379524231, -0.2254898101091385, -0.3412075340747833, 0.7605263590812683, -1.5522981882095337, 0.026583373546600342, 1.407947063446045, -1.4376176595687866, 1.7180759906768799, 1.8576717376708984, -0.6231099963188171, -1.1760344505310059, -0.025834767147898674, 0.47607776522636414, 0.8948854804039001, -0.343863308429718, -1.1520988941192627, 1.2355684041976929, -0.7279402017593384, -1.1200921535491943, -0.8507999777793884, 0.6682566404342651, -0.5586385726928711, -0.5070435404777527, 0.981611967086792, -1.0465402603149414, -2.5902819633483887, 0.33759018778800964, 0.4032415449619293, 0.4132152497768402, -0.650031328201294, -1.4146579504013062, -0.47281256318092346, -1.1700899600982666, 0.9550498723983765, 0.31329578161239624, 1.9712265729904175, 0.7899218201637268, -0.22259646654129028, 1.4386425018310547, -0.8279028534889221, 1.196254849433899, 1.1264411211013794, -0.26473698019981384, -0.7888810634613037, 0.020853696390986443, -0.45334210991859436, 0.0996149480342865, 0.5201098322868347, 0.3421051800251007, 0.13095925748348236, 0.9654352068901062, 0.3531239330768585, 1.7259016036987305, -1.620312213897705, 0.2662351727485657, -0.5921801328659058, 1.2262197732925415, 0.12452042102813721, 1.2975975275039673, 1.6699188947677612, -0.1101493090391159, 1.1577295064926147, 0.9623712301254272, -0.6914200782775879, -0.564709484577179, 0.8470078706741333, -1.9299921989440918, -0.4031526744365692, -0.5591127276420593, -0.1526847779750824, 0.016906458884477615, 2.030733108520508, -0.40436941385269165, -1.2529168128967285, 0.05146360769867897, -0.9524176120758057, 0.9264519810676575, 0.8744187951087952, -1.4202691316604614, -1.3746707439422607, 0.30312424898147583, -0.48644018173217773, -0.05058625712990761, 1.5220937728881836, -0.10497400909662247, 0.4083549380302429, -1.2929778099060059, -1.444392204284668, -1.0376129150390625, -0.6254894137382507, 0.5712074041366577, 0.9332399368286133, 1.7052420377731323, -0.895646333694458, 0.8229856491088867, -0.781237781047821, -1.6984175443649292, 1.5907405614852905, 0.5636669397354126, -0.7735167145729065, -0.22781804203987122, 0.8484704494476318, 1.9563467502593994, 1.0391325950622559, 0.8286913633346558, -0.7347940802574158, -1.615257978439331, -0.006473924964666367, -1.3265957832336426, -0.4333399534225464, 1.3843841552734375, 0.282093346118927, -1.0594866275787354, -1.7616956233978271, -1.0979303121566772, 0.6807953119277954, -2.4529407024383545, -1.1823585033416748, 0.5488222241401672, -0.6138034462928772, -0.05295569449663162, -1.1356760263442993, 0.9453004598617554, 2.344595432281494, 0.5353612899780273, 1.532835602760315, -0.3714263439178467, 0.41627037525177, 0.04970235377550125, -1.7143573760986328, -0.09267795830965042, -0.037677861750125885, -0.4816981852054596, -0.3135668635368347, -1.823951244354248, 0.6384023427963257, -0.7880344986915588, 0.23204068839550018, -0.5069634914398193, 0.9938857555389404, 0.08191949129104614, 0.7019092440605164, -1.1911687850952148, 0.6750460863113403, 0.47283580899238586, -0.1456248015165329, -0.4782790243625641, 0.3552609980106354, 0.5156674981117249, -0.030132507905364037, 0.13364839553833008, -0.405962735414505, -0.68131023645401, -1.350484013557434, 1.042153000831604, 0.4190343916416168, 0.16358882188796997, -0.3169989585876465, -0.19954873621463776, -0.6331744194030762, 0.8837574124336243, 0.24794241786003113, 0.09455978125333786, -0.21314017474651337, -0.8213616609573364, 1.378279447555542, -0.14551204442977905, 0.9328216314315796, -0.62176114320755, 0.1986643373966217, 1.1359578371047974, -1.3116897344589233, -0.8056994080543518, -0.9799292087554932, 0.3439631760120392, -1.7288790941238403, -0.4941812753677368, -1.477062702178955, -1.7083483934402466, 2.2749948501586914, -0.32272735238075256, 1.2523545026779175, 1.1838099956512451, 1.569666862487793, -0.139162078499794, -1.1326251029968262, 2.260035991668701, -1.6344363689422607, -1.8799909353256226, 0.6670931577682495, 0.9103142023086548, 0.8894047737121582, -0.29963478446006775, -1.3832966089248657, 0.2518208920955658, 1.0611765384674072, -0.08162710070610046, 1.4998586177825928, -0.07158855348825455, 0.9284087419509888, -0.7146397829055786, 0.4521181285381317, 1.512770414352417, 1.3887156248092651, 0.7150481939315796, -0.05897825211286545, -0.6227583289146423, -0.001390890683978796, 1.102766990661621, 0.8784970045089722, -1.0547735691070557, 1.3136777877807617, 1.5840703248977661, -0.6692838072776794, 0.515053391456604, -0.9876570701599121, 0.17001758515834808, -0.062388282269239426, 0.35507193207740784, -0.021324768662452698, -0.45788511633872986, 0.4718484878540039, -2.468313455581665, -1.1020554304122925, 0.013128926046192646, -0.09577500075101852, 0.5765439867973328, -0.9563531875610352, -1.4600740671157837, -0.03059164434671402, -0.5708193182945251, -2.019652843475342, 0.9232614636421204, 0.40748193860054016, -0.13205529749393463, 2.5841548442840576, -0.9323763251304626, 0.14126801490783691, 0.6972410678863525, -0.019493380561470985, -0.5074684619903564, 0.05821641534566879, 0.3699660897254944, 0.8616219162940979, -0.5769234299659729, -0.4470294117927551, -0.851676881313324, -0.5730496644973755, -0.5430657267570496, -0.6752426028251648, 1.8789315223693848, -0.601634681224823, -0.5553030371665955, 2.284723997116089, 0.839337170124054, -2.7908899784088135, -0.06614029407501221, -0.29702994227409363, -1.5700162649154663, -1.3409830331802368, 0.6956264972686768, 0.3860680162906647, 0.051935888826847076, -0.8930606245994568, 1.2549015283584595, 0.2207430899143219, 0.005342231132090092, 0.059168629348278046, -1.670541524887085, 0.3807331621646881, 1.4561846256256104, 0.34612277150154114, -0.507881224155426, 1.1356773376464844, -1.3353190422058105, -0.6403961777687073, 0.27470630407333374, 0.5331928133964539, 0.3766433894634247, -1.6821831464767456, 0.5091484189033508, 0.34131935238838196, -0.7811291813850403, 1.6300379037857056, -0.2739735543727875, -0.14292462170124054, 0.5675291419029236, 0.3372799754142761, 0.3434775173664093, -0.6091005802154541, 0.2305477261543274, -0.0070664179511368275, -1.1855071783065796, -1.5954086780548096, 2.2633488178253174, -1.1375826597213745, -0.8352458477020264, 0.6895403861999512, 0.21924564242362976, -1.7912381887435913, 0.6076593995094299, 0.744586169719696, 0.005988870281726122, -0.4493226110935211, -0.035404354333877563, -0.9513779282569885, -1.239656925201416, 0.38977909088134766, 1.8019905090332031, 1.1065620183944702, 0.42001834511756897, 0.7838947176933289, 0.5054168105125427, 0.9687461256980896, 0.9649083018302917, -1.19186270236969, -0.7187884449958801, 1.5398460626602173, -0.85878586769104, -1.8298577070236206, 1.2785967588424683, 0.09199512004852295, 0.5409001708030701, 1.2822494506835938, -0.9324820637702942, 0.20751549303531647, 0.3195757567882538, -0.24133028090000153, -0.09103785455226898, -0.28334543108940125, 0.5140523314476013, -0.7393580675125122, -0.10627766698598862, -1.1299480199813843, 0.12714357674121857, 0.9476136565208435, -0.7605385184288025, -0.2523583471775055, -1.060828447341919, 1.8155885934829712, -0.019391732290387154, 0.4182718098163605, 2.131227731704712, -0.25396236777305603, -0.09531241655349731, -0.7413858771324158, -1.2621725797653198, 1.5981435775756836, 0.2761390805244446, 0.08378727734088898, 0.4385521709918976, -0.6730919480323792, 0.2765045166015625, 0.5380395650863647, 1.0719013214111328, -0.10434254258871078, -0.4210846722126007, 0.11686112731695175, 0.7484068274497986, 1.0358035564422607, -0.3712756931781769, -0.6193749308586121, -1.4635848999023438, -0.5482596755027771, 0.20108459889888763, 0.5004163384437561, -1.1224216222763062, -1.762628436088562, -0.9395259022712708, 1.5066139698028564, 0.6433097124099731, -0.22144275903701782, 2.010270833969116, 0.21964344382286072, -0.27697181701660156, 0.249564066529274, 0.07494346052408218, -0.16062822937965393, 0.9187678098678589, -0.04942769184708595, 0.10794281214475632, 0.4221767485141754, 1.0131428241729736, -1.4308778047561646, -0.8599252104759216, -0.3672284185886383, -0.6743965148925781, -0.18707983195781708, -0.9847589135169983, -0.9671480059623718, -1.2636590003967285, 0.18556803464889526, 1.438539981842041, -0.9962128400802612, 1.4054213762283325, 1.2028369903564453, -1.9328242540359497, 0.23250187933444977, 0.3878405690193176, -2.1689293384552, 0.3181452453136444, -0.2554861605167389, 0.13337360322475433, 1.4374910593032837, 0.14067673683166504, 0.3930247724056244, -1.2400636672973633, 0.7485615611076355, 0.7086167931556702, 0.8094612956047058, 3.209430694580078, -0.08833437412977219, 0.8457263708114624, -0.15435971319675446, 0.6159102320671082, 2.2969675064086914, -0.28665605187416077, -1.0740448236465454, 1.201499342918396, -0.5396712422370911, -0.21388202905654907, 0.34531134366989136, -0.25507116317749023, 0.16606774926185608, 0.7740017771720886, 1.3758983612060547, -1.010311245918274, 2.244762659072876, 0.21196608245372772, -0.6677922606468201, -0.8562031388282776, 0.5832969546318054, 1.8201786279678345, -0.42257246375083923, -1.089469313621521, 1.4492202997207642, -1.0714870691299438, 0.5839068293571472, 0.7351388931274414, -0.016495466232299805, -1.3410420417785645, 0.756488561630249, 0.5641587972640991, -0.0029218322597444057, 1.004654049873352, -0.4106641113758087, -1.526393175125122, -2.2575302124023438, 0.23859737813472748, -0.5562986135482788, 0.6953704357147217, 0.3404626250267029, 0.15129968523979187, -0.6203209161758423, 0.063806913793087, 0.9556921720504761, -1.0382581949234009, 1.3428839445114136, -0.1775844693183899, 0.6758065819740295, -0.6917272806167603, 0.8570429682731628, -0.1043778732419014, -0.001517641358077526, 0.33716660737991333, 1.7465507984161377, 0.7136356234550476, 0.2239709049463272, -1.1539636850357056, 0.012101172469556332, -1.238045573234558, 0.3685716986656189, -0.36055371165275574, 0.031235797330737114, -0.8264997005462646, -0.01167932990938425, 0.2668876051902771, 1.6840014457702637, 0.5267770290374756, 0.7078707814216614, -0.9992595314979553, 0.8918015360832214, -0.5841625928878784, -0.9892972111701965, -0.3919677436351776, 0.23650150001049042, -0.44198405742645264, -0.5960873365402222, 0.5667828321456909, -0.871913492679596, -0.380439817905426, -0.587908923625946, -0.013651865534484386, 0.13961847126483917, -0.836245596408844, 0.9898120164871216, -2.8554301261901855, 0.33680614829063416, 1.2173455953598022, 1.5702567100524902, 0.5069186687469482, 0.5804793238639832, -0.4200607240200043, -0.40663957595825195, -1.0223637819290161, -0.568574070930481, -1.6494179964065552, -2.1037511825561523, 1.0410175323486328, 0.31300222873687744, 1.2333980798721313, -0.3030479848384857, -1.350504755973816, 0.2949436604976654, 1.1531990766525269, -2.011070966720581, -0.38047170639038086, 0.054904405027627945, 2.128750801086426, 0.22440873086452484, -1.2694746255874634, -1.209602952003479, -1.146852731704712, -0.9923461675643921, 0.5868125557899475, -0.23126612603664398, 0.056178655475378036, -1.5770328044891357, 2.105036735534668, -0.7164774537086487, -1.5560753345489502, -0.512919545173645, 0.5452418327331543, -0.4897894263267517, -0.2233441174030304, 2.0557546615600586, -1.0299097299575806, 0.6280494332313538, -2.668330669403076, 0.06683304160833359, -0.6971495747566223, 0.7145875096321106, -0.05761535465717316, -0.8951294422149658, -0.12770245969295502, -0.07688138633966446, -0.40133944153785706, -0.914107620716095, -1.3307371139526367, -0.9239300489425659, -0.5489113330841064, -1.0649186372756958, 0.4963129162788391, -1.561405897140503, -2.066798686981201, -1.492009162902832, -1.1996415853500366, -0.5139029622077942, 0.4937843084335327, -0.260924756526947, 0.7878021597862244, -0.11639469861984253, 0.07733634859323502, 1.099268913269043, 0.002393025206401944, 0.35443392395973206, -0.5562495589256287, -1.1504085063934326, -0.5187245011329651, 1.5052831172943115, 0.12850889563560486, -2.5081727504730225, -1.362428903579712, 0.3671036660671234, -0.7674247622489929, 0.6514360308647156]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [[-0.018697120249271393, -0.003043112810701132, 0.0883849486708641, 0.011311540380120277, -0.15716446936130524, 0.09977880120277405, 0.44440019130706787, -0.03659304976463318, -0.0009894459508359432, -0.016963234171271324, 0.5779100656509399, -0.07912580668926239, 0.008907601237297058, -0.3810616731643677, 0.0027616899460554123, -0.16363665461540222, -0.24039973318576813, -0.1309446096420288, -0.09848473966121674, -0.025562560185790062, 0.0017050433671101928, -0.241514652967453, -0.15594540536403656, 0.3541357219219208, -0.4740184545516968, -0.08033055812120438, 0.3202025890350342, 0.08882254362106323, -0.02379395253956318, -0.047215480357408524, 0.06235602870583534, -0.3917000889778137, 0.014914666302502155, 0.012650481425225735, 0.07804844528436661, 0.009333332069218159, 0.014326549135148525, -0.21565914154052734, 0.5576505661010742, -0.010806126520037651, -0.1364867091178894, -0.027070218697190285, 0.09066101163625717, 0.08650407195091248, -0.011731348000466824, -0.12008579820394516, -0.05635282024741173, 0.36787644028663635, 0.07070441544055939, 0.12599162757396698, -0.011835326440632343, 0.3994995057582855, -0.07068871706724167, 0.6456571221351624, -0.031188836321234703, -0.4319046437740326, 0.09475931525230408, 0.05206679552793503, 0.17449161410331726, 0.025461697950959206, 0.34286731481552124, 0.009087100625038147, -0.019432157278060913, 0.43304362893104553, 0.2793148159980774, -0.07461857050657272, 0.19643443822860718, 0.058586157858371735, 0.07645927369594574, -0.27945125102996826, 0.2058456689119339, 0.13500568270683289, 0.10215488076210022, -0.011567461304366589, 0.2210424691438675, -0.29652345180511475, -0.16305378079414368, -0.0027493005618453026, 0.12781734764575958, 0.25741270184516907, -0.019805626943707466, -0.027069687843322754, 0.06656988710165024, 0.1519198715686798, -0.023441335186362267, 0.049394816160202026, -0.08462647348642349, 0.03994861990213394, 0.49123454093933105, 0.3627375662326813, -0.46549850702285767, 0.02165304683148861, -0.2424856424331665, -0.6464763879776001, -0.019905131310224533, 0.6770575642585754, 0.02592192031443119, 0.16073061525821686, -0.17220646142959595, -0.024311093613505363, 0.01796847954392433, -0.061266086995601654, 0.04658840596675873, 0.032813623547554016, -0.24593356251716614, 0.027193447574973106, 0.007209909148514271, 0.11061806231737137, -0.09116868674755096, 0.14627772569656372, -0.04934065788984299, 0.054458729922771454, 0.34042906761169434, -0.8041765093803406, -0.03477923944592476, -0.0249111857265234, -0.22041653096675873, 0.012146571651101112, -0.20496182143688202, 0.1286018043756485, -0.004911561496555805, 0.7322866916656494, 0.007507838308811188, 0.00586357805877924, -0.034736547619104385, -0.021407853811979294, 0.010600468143820763, 0.20390185713768005, 0.006017377600073814, -0.4249802827835083, 0.11187268048524857, -0.18429513275623322, 0.03934741020202637, -0.014390671625733376, -0.6074479818344116, 0.05025585740804672, 0.11157586425542831, 0.004736724775284529, 0.08074097335338593, 0.20524367690086365, 0.003813323797658086, 0.14435456693172455, 0.04633253812789917, 0.05336416885256767, 0.09486944228410721, 0.12088952213525772, 0.13123677670955658, -0.06807563453912735, -0.0862952247262001, -0.18465466797351837, -0.6609604954719543, -0.10557406395673752, 0.030927645042538643, -0.08834007382392883, -0.05107353255152702, 0.013586355373263359, -0.42928120493888855, -0.24570830166339874, 0.16175395250320435, -0.008334288373589516, -0.08191010355949402, -0.39363861083984375, 0.25600484013557434, -0.1816370040178299, 0.4055996239185333, -0.07965832948684692, 0.015357979573309422, 0.04158485680818558, 0.44487428665161133, 0.12861409783363342, -0.21838822960853577, 0.00042845241841860116, -0.11242935061454773, 0.0418073907494545, -0.27804234623908997, -0.3271907866001129, 0.23752333223819733, -0.5145063400268555, -0.26359182596206665, -0.22581928968429565, -0.2988121807575226, -0.01624169573187828, 0.03182853013277054, 1.0205810070037842, 0.06689434498548508, 0.08474495261907578, 0.22705847024917603, -0.32287728786468506, 0.6248217821121216, -0.012298362329602242, -0.12926238775253296, 0.021792318671941757, 0.3599587380886078, 0.16608668863773346, 0.1556614339351654, -0.21616679430007935, 0.23694570362567902, 0.11339814960956573, -0.22961972653865814, 0.13723121583461761, 0.027265094220638275, -0.011143848299980164, 0.13862860202789307, 0.10390128940343857, 0.07004715502262115, 0.5921417474746704, 0.009058564901351929, 0.2160337269306183, 0.04871454834938049, 0.31359606981277466, -0.1293013095855713, -0.008130443282425404, 0.4407648742198944, -0.21187898516654968, 0.2837914228439331, 0.004031979013234377, -0.3222714364528656, -0.08850409090518951, -0.05066690221428871, 0.056506913155317307, -0.28632572293281555, -0.07341866195201874, 0.49603018164634705, 0.2239236682653427, 0.08721791207790375, -0.035685472190380096, -0.039907366037368774, -0.08268781006336212, 0.017086198553442955, -0.00985897146165371, -0.08931653946638107, 0.2806456983089447, 0.030514582991600037, 0.23373636603355408, -0.004348190501332283, -0.006741930264979601, 0.048368919640779495, -0.1584417223930359, 0.04209963232278824, -0.25774258375167847, 0.02947229892015457, 0.05477428063750267, 0.05770784988999367, 0.2463901937007904, -0.12790504097938538, 0.19695255160331726, 0.29312649369239807, -0.8543625473976135, -0.047556594014167786, -1.0087083578109741, -0.24264197051525116, -0.025096992030739784, 0.15714137256145477, -0.26521193981170654, -0.233850359916687, -0.1495024412870407, 0.14788274466991425, 0.1332712173461914, 0.024313097819685936, -0.11473935097455978, 0.011190752498805523, 0.14576676487922668, -0.0536080040037632, -0.4504755139350891, -0.2800223231315613, -0.11089739948511124, -0.007740401662886143, -0.04190962389111519, -0.13600435853004456, 1.0292478799819946, -0.25062164664268494, 0.034199945628643036, -0.05128195881843567, 0.1151069775223732, -0.21201379597187042, 0.052438635379076004, 0.08910370618104935, -0.16115877032279968, 0.3853117823600769, 0.11430006474256516, -0.20092283189296722, -0.28834909200668335, -0.2329195737838745, -0.012464726343750954, 0.1915341466665268, -1.0667407512664795, 0.012821112759411335, -0.18476291000843048, 0.005349372513592243, -0.09156971424818039, -0.6973445415496826, -0.021000823006033897, -0.71898353099823, 0.13952119648456573, -0.016823185607790947, -0.06920342147350311, 0.016345391049981117, 0.6234033703804016, -0.006775069981813431, -0.040353819727897644, -0.05390311777591705, -0.06178737431764603, -0.07993156462907791, -0.023326724767684937, -0.046043671667575836, 0.17240141332149506, 0.11387015879154205, 0.14853383600711823, -0.27001920342445374, 0.041079290211200714, 0.01121295616030693, -0.1006973385810852, 0.000247890071477741, -0.0664251297712326, -0.019472314044833183, -0.21595846116542816, 0.03567008674144745, -0.020299004390835762, -0.01989465020596981, -0.02495265193283558, -0.052387453615665436, -0.017099831253290176, 0.4100196361541748, 0.1939743459224701, -0.0003591123968362808, 0.05723297968506813, 0.08809271454811096, 0.12163272500038147, 0.004456074442714453, -0.018973927944898605, -0.24086539447307587, -0.46346649527549744, 0.023187248036265373, 0.1795690804719925, 0.0340583473443985, 0.04134796932339668, -0.05786614120006561, 0.012357722967863083, -1.092009425163269, 0.3883245289325714, 0.09927978366613388, -0.07994884252548218, 0.010215403512120247, 0.4996005594730377, 0.27480214834213257, -0.035457413643598557, 0.48680007457733154, -0.07338197529315948, 0.01782231032848358, 0.16704055666923523, -0.05272558704018593, -0.03206482157111168, -0.24648045003414154, -0.2982383668422699, 0.0645289421081543, -0.7568040490150452, 0.15797024965286255, 0.11787591874599457, -0.02826351299881935, 0.004626203794032335, 0.36179888248443604, -0.13193702697753906, -0.17833657562732697, -0.1858251839876175, -0.0011052170302718878, 0.04344543442130089, 0.014463518746197224, -0.03783317655324936, 0.24313025176525116, -0.4960649907588959, 0.010045216418802738, 0.9889642596244812, 0.5069490075111389, 0.010533195920288563, -0.5506713390350342, 0.04750720039010048, 0.14919257164001465, -0.02592523954808712, 0.11163962632417679, 0.2159450352191925, 0.848865270614624, -0.6870265603065491, -0.27744266390800476, -0.13816732168197632, -0.02649727091193199, -0.3286586403846741, 0.02577180601656437, -0.30977118015289307, -0.05786802992224693, -0.058926619589328766, -0.2407897263765335, 0.06167446821928024, -0.40910857915878296, 0.019312972202897072, -0.13121968507766724, 0.005498254206031561, 0.21611911058425903, 0.01209351047873497, -0.09808764606714249, 0.014552604407072067, -0.05545244738459587, 0.09244628995656967, -0.05830482766032219, 0.05434669926762581, 0.09997141361236572, -0.08679984509944916, -0.2122599482536316, -0.037520669400691986, -0.11697888374328613, -0.23529402911663055, 0.18347151577472687, 0.0517394058406353, 0.17866966128349304, 0.0031338075641542673, 0.037059247493743896, 0.04574307054281235, -0.42239266633987427, 0.09531521797180176, 0.16309379041194916, -0.2976568937301636, 0.27444061636924744, -0.8878951072692871, -0.0700029730796814, 0.16954684257507324, -0.05703131854534149, -0.011515328660607338, 0.3751600384712219, -0.34276100993156433, -0.26697131991386414, -0.1396634876728058, 0.9118596315383911, -0.13293562829494476, -0.2806869149208069, 0.16221274435520172, 0.01138318981975317, -0.0582016296684742, -0.2414734661579132, 0.1813865303993225, 0.026430197060108185, 0.08138682693243027, 0.02056930959224701, 0.09922900795936584, -0.14650455117225647, 0.013229829259216785, -0.04973156377673149, -0.02884763479232788, -0.09868351370096207, 0.1791255921125412, 0.024729354307055473, -0.4365454614162445, 0.1801995486021042, 0.001432405086234212, -0.001997037325054407, -0.018099915236234665, 0.04848369583487511, 0.011689308099448681, -0.3201517164707184, 0.01194190513342619, 0.1873258501291275, 0.44926717877388, 0.01838531717658043, -0.37462252378463745, -0.04282219707965851, -0.031527530401945114, 0.0321819894015789, 0.09384360164403915, -0.6270099878311157, -0.07805132865905762, -0.055422428995370865, -0.10010652989149094, 0.13374640047550201, 0.08975853770971298, -0.19829526543617249, -0.043165795505046844, 0.1981143057346344, 0.06650455296039581, 0.23977845907211304, 0.06620364636182785, -0.20419633388519287, -0.26168370246887207, 0.10022982954978943, -0.04630669951438904, -0.08024406433105469, -0.002088644541800022, -0.0094751613214612, 0.3435596525669098, 0.015083628706634045, -0.021043352782726288, 0.14948518574237823, -0.05855202674865723, -0.11287318170070648, 0.9883962869644165, 0.18557339906692505, 0.4825783669948578, 0.02291196398437023, 0.21423445641994476, 0.2900456488132477, 0.20470120012760162, -0.057047490030527115, -0.014616046100854874, 0.45538610219955444, 0.33365532755851746, 0.09342557191848755, 0.3129121959209442, -0.24806228280067444, -0.24304774403572083, -0.12764741480350494, -0.09032607078552246, 0.2245807945728302, -0.11532281339168549, 0.10630824416875839, -0.34198662638664246, 0.08660195767879486, -0.47365856170654297, 0.30677270889282227, -0.22488804161548615, -0.07239983975887299, 0.058708954602479935, 0.13337352871894836, 0.09608130156993866, 0.08725132048130035, -0.029510360211133957, -0.15940961241722107, 0.4516362249851227, 0.012782738544046879, 0.3104459345340729, -0.10294816642999649, 0.3488140106201172, -0.26083889603614807, 0.44194135069847107, 0.024410128593444824, 0.04671843349933624, -0.06385622918605804, 0.015679599717259407, 0.12164124846458435, -0.29336804151535034, 0.001757302088662982, 0.47236233949661255, -0.21884211897850037, -0.03306083381175995, -0.11956272274255753, 0.1885497123003006, -0.029228536412119865, 0.09868224710226059, -0.20094051957130432, -0.11068487912416458, -0.19336725771427155, 0.046626389026641846, -0.020769117400050163, -0.24711944162845612, -0.023644236847758293, 0.15131773054599762, 0.05146949365735054, 0.04570036008954048, -0.08949043601751328, -0.1765512228012085, 0.06100531667470932, 0.18820291757583618, 0.017507266253232956, -0.0038874323945492506, 0.6726242303848267, 0.10563303530216217, 0.051452476531267166, -0.016371598467230797, -0.029595550149679184, 0.279392272233963, -0.11151689291000366, 0.1331961303949356, -0.01251645851880312, 0.2333751618862152, 0.08610545843839645, 0.07355112582445145, -0.2445574551820755, -0.050367627292871475, 0.03790821135044098, 0.028494585305452347, -0.00954281073063612, -0.09589863568544388, -0.03453607112169266, 0.38392987847328186, 0.0024276445619761944, -0.2496824413537979, -0.22495205700397491, -0.09777012467384338, 0.12969253957271576, -0.07936935871839523, 0.004538394045084715, -0.21754369139671326, -0.06378387659788132, -0.1810259222984314, -0.0992264673113823, 0.085246741771698, 0.010079100728034973, 0.019624492153525352, -0.3279581367969513, 0.14892585575580597, -0.19727294147014618, 0.05104406177997589, 0.39497241377830505, -0.29230257868766785, -0.12818025052547455, -0.0123875942081213, 0.16559705138206482, -0.1408679485321045, -0.028664378449320793, -0.01751784048974514, 0.014301021583378315, 0.03916984796524048, -0.35470202565193176, 0.15267473459243774, 0.5141004323959351, -0.3735341429710388, -0.003460562787950039, 0.08925619721412659, -0.02114296704530716, 0.054616160690784454, -0.08932872861623764, -0.17880184948444366, -0.10691817104816437, 0.09304431080818176, -0.9257189035415649, -0.09778096526861191, 0.009992857463657856, 0.03931719437241554, 0.014517592266201973, 0.16511522233486176, -0.1404709368944168, -0.05382894352078438, -0.021285375580191612, -0.017084205523133278, 0.14346688985824585, -0.0520843043923378, -0.23539943993091583, -0.1547478288412094, -0.1028338298201561, -0.137041375041008, -0.3418678343296051, 0.5896060466766357, 0.020134273916482925, 0.16798678040504456, -0.16299830377101898, -0.3212563097476959, -0.253356009721756, 0.033899933099746704, -0.06969815492630005, -0.8424727916717529, -0.4042551517486572, 0.006077874917536974, 0.0768253356218338, 0.805377185344696, 0.09556041657924652, 0.4588735103607178, 0.0439787283539772, 0.03479037061333656, -0.3083816170692444, 0.14598789811134338, -0.02299642562866211, -0.10468390583992004, -0.5017756819725037, 0.5409905314445496, 0.15009114146232605, -0.10696251690387726, 0.4485544264316559, -0.18665872514247894, 0.5822826623916626, -0.17206400632858276, -0.013946513645350933, -0.04224473983049393, -0.06747449934482574, -0.2535879909992218, -0.017943816259503365, 0.35252872109413147, -0.035481300204992294, 0.3801265060901642, 0.056626349687576294, -0.15740954875946045, 0.027793969959020615, 0.17002300918102264, -0.015211314894258976, 0.09104542434215546, 0.09690839052200317, -0.008863359689712524, -0.14296579360961914, 0.09888412803411484, 0.008198191411793232, -0.07741871476173401, 0.006738621275871992, 0.26831674575805664, 0.1672532856464386, 0.16792629659175873, -0.04802165552973747, -0.16301767528057098, 0.10910522937774658, 0.093451127409935, -0.007040452212095261, 0.15520349144935608, 0.6001451015472412, -0.08007331937551498, 0.283210813999176, -0.04609677568078041, -0.0741509348154068, 0.4944915473461151, -0.20842450857162476, 0.12573672831058502, 0.015597398392856121, -0.008459335193037987, -0.1709042191505432, 0.1164633184671402, -0.02524126134812832, 0.050996676087379456, -0.26433852314949036, -0.07800932973623276, -0.11725812405347824, 0.3017474412918091, -0.0018471862422302365, -0.08266912400722504, -0.013530885800719261, 0.06914333254098892, 0.12129884213209152, -0.16516005992889404, 0.14240916073322296, 0.14693762362003326, -0.6537997126579285, -0.5462335348129272, -0.006007558200508356, -0.18516893684864044, 0.02898031286895275, -0.01604047417640686, -0.09119243174791336, 0.013740252703428268, -0.14281389117240906, -0.15204492211341858, 0.21146456897258759, 0.056115273386240005, -0.040853966027498245, 0.06266777217388153, -0.19052176177501678, -0.0406390018761158, 0.04737303778529167, 0.028610212728381157, -0.019835105165839195, -0.012196812778711319, 0.27775436639785767, 0.007068131119012833, -0.018247539177536964, -0.011113148182630539, 0.1558447629213333, -0.034523166716098785, 0.19020812213420868, 0.010234620422124863, -0.01858551986515522, 0.16128280758857727, -0.35098591446876526, 0.027275796979665756, -0.030371706932783127, 0.8789549469947815, 0.031183278188109398, -0.2672046422958374, 0.03428313508629799, 0.7663556933403015, 0.052332185208797455, 0.30816352367401123, -0.22895680367946625, -0.04847534000873566, 0.028911253437399864, -0.1162405014038086, 0.17949482798576355, -0.13367721438407898, -0.08406756818294525, 0.0028607365675270557, -0.0067734867334365845, 0.012448420748114586, -0.03396938741207123, 0.053925372660160065, 0.13096719980239868, -0.40129154920578003, -0.24871769547462463, -0.3615139126777649, 0.3253270983695984, 0.12134312093257904, -0.03288194164633751, 0.28140097856521606, -0.46468186378479004, 0.13665111362934113, 0.15854595601558685, -0.01530982181429863, -0.10565103590488434, -0.19339370727539062, 0.20344768464565277, 0.1552819460630417, -0.1569005697965622, -0.2679005265235901, -0.06339568644762039, -0.03729063272476196, -0.3911474943161011, 0.05935531109571457, 0.03338159620761871, -0.0327422097325325, 0.008873666636645794, 0.07860944420099258, 0.263212114572525, 0.29593536257743835, -0.006183740217238665, -0.13092495501041412, -0.13636374473571777, 0.14596109092235565, 1.3494728803634644, -0.23001936078071594, -0.0056651500053703785, 0.0379096083343029, -0.3226713240146637, -0.05443139374256134, 0.010583890601992607, -0.21627841889858246, 0.013853601180016994, 0.9062684774398804, 0.07264985144138336, -0.0015137252630665898, 0.13272111117839813, 0.04560810700058937, -0.13982559740543365, 0.4043317437171936, 0.016581736505031586, -0.021718889474868774, -0.15370631217956543, 0.3840600848197937, 0.04748910665512085, -0.0343514084815979, 0.23784096539020538, 0.0805276408791542, 0.47422894835472107, 0.04226326942443848, -0.08465494215488434, 0.09703673422336578, 0.7641262412071228, -0.01804768666625023, 0.5793635249137878, 0.01794951595366001, -0.09478552639484406, -0.006151575595140457, 0.3542923629283905, 0.08111613988876343, 0.010518040508031845, -0.33733704686164856, 0.3585342764854431, 0.24755707383155823, 0.844014048576355, 0.03156306594610214, -0.08252893388271332, -0.025447314605116844, 0.054593075066804886, -0.07071682065725327, 0.1915564388036728, 0.04249269515275955, 0.009123525582253933, 0.5021001696586609, -0.29595720767974854, -0.16648973524570465, -0.04670727625489235, -0.2833763360977173, -0.006856079678982496, -0.6468814611434937, 0.013036081567406654, 0.061924196779727936, 0.02009372040629387, -0.053334444761276245, -0.05180473253130913, -0.021271321922540665, -0.232985720038414, 0.3227362632751465, -0.20890451967716217, 0.07585851103067398, -0.12223418056964874, 0.310624897480011, -0.2754969298839569, -0.7035823464393616, -0.007572231814265251, 0.014453728683292866, 0.033831607550382614, 0.10655928403139114, -0.9234925508499146, -0.01263996958732605, 0.22471123933792114, 0.17821550369262695, 0.06546688824892044, 0.0946672111749649, 0.03356824442744255, 0.29339995980262756, 0.07995648682117462, -0.39832380414009094, 0.024040168151259422, -0.15767452120780945, 0.14450548589229584, 0.27890077233314514, -0.12133059650659561, -0.07978447526693344, -0.11320383101701736, -1.1041276454925537, -0.10659102350473404, 0.06847652792930603, -0.17038528621196747, 0.022741543129086494, 0.19743582606315613, 0.00831791665405035, -0.0238129161298275, 0.1495954543352127, -0.014473500661551952, 0.0719832181930542, -0.011061535216867924, -0.32739824056625366, 0.8298844695091248, 0.05311265215277672, 0.05073782801628113, -0.0031289272010326385, -0.18312616646289825, -0.3371497094631195, 0.013835342600941658, 0.23617087304592133, -0.8124305009841919, -0.15880954265594482, 0.09539845585823059, -0.30280303955078125, -0.7888690829277039, -0.02377823181450367, -0.12435204535722733, -0.010247685015201569, -0.051873400807380676, 0.01185599621385336, 0.00025963602820411325, -0.0467638224363327, -0.0024528237991034985, -0.05024655908346176, -0.34868675470352173, 0.08426663279533386, 0.06440318375825882, 0.42043259739875793, -0.023059941828250885, 0.21670451760292053, 0.36178141832351685, -0.013374848291277885, 0.14814718067646027, -1.1359692811965942, -0.026570133864879608, -0.018621429800987244, -0.4957708716392517, -0.04572882503271103, -0.08965902030467987, 0.29272809624671936, -0.10556234419345856, 0.1301024854183197, 0.057657159864902496, -0.07424526661634445, 0.18801289796829224, -0.7820612788200378, -0.5755059719085693, -0.013194235973060131, 0.4455200731754303, 0.009482312016189098, 0.017760900780558586, 0.016013197600841522, 0.002941217040643096, -0.005608216859400272, -0.1604490578174591, -0.1273883432149887, 0.0418529286980629, -0.3532235324382782, -0.08092375099658966, 0.010776078328490257, 0.04490809142589569, -0.04463621601462364, -0.031192971393465996, 0.31069815158843994, 0.04918816685676575, 0.0993894413113594, -0.09435615688562393, -0.12564748525619507, -0.49623769521713257, 0.15416300296783447, -0.3156237304210663, 0.2723671495914459, -0.11344308406114578, 0.09825070947408676, -0.27693071961402893, 0.3697177767753601, 0.09372996538877487, -0.3359922766685486, -0.2019968032836914, 0.03947271406650543, 0.10431739687919617, -0.19712314009666443, -0.13200898468494415, -0.3569086194038391, 0.2224094420671463, -0.48198097944259644, 0.11301849037408829, 0.27058112621307373, 0.05406950041651726, 0.018665175884962082, 0.3326440751552582, -0.018978603184223175], [-0.051012489944696426, -0.06511876732110977, 0.028986534103751183, 0.05011025443673134, -0.6164990663528442, 0.2376214861869812, 0.1204848438501358, -0.009425091557204723, 0.02759319357573986, 0.0042953225784003735, -0.09572536498308182, -0.04184117540717125, -0.12703834474086761, 0.02271088771522045, -0.021300988271832466, -0.43724697828292847, -0.04423145204782486, 0.17147190868854523, 0.11375030130147934, -0.023120276629924774, -0.8085830211639404, 0.05232807621359825, -0.7339761853218079, -0.09974072873592377, 0.25860244035720825, -0.26187896728515625, 0.16676226258277893, -0.10886698216199875, 0.2330082654953003, 0.049735162407159805, 0.010415893979370594, -0.40635058283805847, 0.12920212745666504, -0.08830491453409195, -0.13195212185382843, -0.1624942272901535, -0.043743424117565155, -0.002814324339851737, 0.2815886437892914, -0.05258360877633095, 0.3409436345100403, 0.0752861499786377, -0.012424314394593239, 0.010471886955201626, 0.029036182910203934, -0.06139097735285759, -0.22171512246131897, -0.14096848666667938, -0.2039354145526886, -0.029229937121272087, -0.010387642309069633, -0.04858936369419098, -0.028971051797270775, 0.7199466228485107, -0.13666404783725739, -0.26245737075805664, -0.04128212481737137, 0.3011791408061981, -0.12224409729242325, 0.11764465272426605, -0.09105916321277618, 0.005767568480223417, 0.007923346944153309, -0.0017824072856456041, -0.05690774321556091, -0.6930745840072632, -0.06226608157157898, 0.34625881910324097, -0.010230371728539467, -0.27091628313064575, 0.1235007643699646, -0.11772796511650085, 0.03662510961294174, -0.008381345309317112, -0.3051929771900177, -0.4243497848510742, -0.48208439350128174, 0.1491866558790207, -0.026743536815047264, 0.7151067852973938, -0.08157353848218918, 0.01565529592335224, -0.06352812796831131, -0.03315075486898422, -0.02637105993926525, -0.02366618998348713, -0.2955173850059509, -0.10713586211204529, -0.23668313026428223, -0.2816249430179596, -0.1197553351521492, -0.37072110176086426, -0.1724235862493515, 0.11497212201356888, 0.02656516246497631, 0.8397189974784851, 0.07000099122524261, 0.023467181250452995, -0.024408500641584396, 0.027149470522999763, -0.1934586614370346, 0.19390244781970978, -0.1336570531129837, 0.025516845285892487, 0.4770536720752716, -0.008220541290938854, -0.0016175340861082077, 0.814834713935852, 0.017673706635832787, -0.21360662579536438, 0.02990543842315674, -0.06473665684461594, 0.20742852985858917, 1.2890357971191406, 0.17674796283245087, 0.010315666906535625, -0.24738214910030365, -0.0044626942835748196, -0.5023161768913269, -0.20069833099842072, -0.12718310952186584, -0.14057894051074982, -0.005288895685225725, -0.014063301496207714, 0.006676503457129002, -0.01113069523125887, 0.010630309581756592, -0.04020778089761734, 0.001360900467261672, 0.20175471901893616, 0.013307059183716774, 0.009281769394874573, -0.10316126048564911, -0.04962668940424919, -0.782882571220398, 0.4345616102218628, -0.004826670046895742, 0.018283626064658165, -0.075290247797966, -0.3117433190345764, -0.04676695540547371, -0.10328873991966248, -0.04043694585561752, 0.655055820941925, 0.05797721445560455, -0.2545260190963745, 0.08154623955488205, 0.10582566261291504, -0.09664066135883331, -0.03969165310263634, 0.34289515018463135, 0.5188614726066589, 0.3743418753147125, -0.0366995669901371, 0.47247615456581116, -0.0771828144788742, -0.22713658213615417, -0.2646356523036957, -0.05131010338664055, 0.09976810216903687, -0.07641569525003433, -0.07801060378551483, -0.018420424312353134, 0.10652411729097366, -0.4668221175670624, 0.001732802833430469, -0.1309259682893753, -0.028376126661896706, 0.17563289403915405, -0.006021232344210148, 0.1957671195268631, 0.05964018777012825, -0.16223356127738953, -0.11689732223749161, -0.03134720027446747, 0.2308845818042755, -0.09530642628669739, -0.727380633354187, -0.059869974851608276, 0.09831060469150543, -0.13302749395370483, 0.017342204228043556, -0.004295962397009134, 0.4071524441242218, -0.005100766662508249, 0.012204306200146675, 0.1645595282316208, -0.18223054707050323, -0.4738335609436035, -0.12991221249103546, 0.14347170293331146, -0.06904879212379456, 0.14611729979515076, 0.05353419482707977, 0.2237994521856308, -0.01689257100224495, -0.1131354570388794, -0.020356634631752968, 0.12406589090824127, 0.033726129680871964, 0.008058327250182629, -0.1376447081565857, 0.17374980449676514, 0.1015758067369461, -0.01285939384251833, -0.010061100125312805, -0.000464170821942389, 1.2008836269378662, 0.052779778838157654, 0.3091842234134674, 0.3388005495071411, -0.01766027882695198, -0.13728401064872742, -0.2726764976978302, 0.16950683295726776, -0.01002306118607521, 0.7295828461647034, -0.09646562486886978, 0.2988490164279938, -0.45595449209213257, 0.215089812874794, -0.43002793192863464, -0.29294946789741516, -0.31558674573898315, -0.026280267164111137, -0.5965197086334229, 0.2899021804332733, 0.20550431311130524, 0.002375633455812931, -0.38557207584381104, -0.060978423804044724, -0.0935458242893219, 0.007737602572888136, -0.4494044780731201, 0.04383644834160805, 0.05364755541086197, -0.05324583873152733, 0.0915030986070633, -0.32668644189834595, -0.25272056460380554, 0.0025203388649970293, -0.07655704766511917, -0.02296101115643978, -0.0638374611735344, 0.08018449693918228, 0.138583242893219, -0.23220528662204742, 0.32777634263038635, -0.18825533986091614, -0.3547043204307556, -0.09186537563800812, -0.15588748455047607, 0.194571852684021, 0.2123800814151764, 0.2629541754722595, -0.13249824941158295, -0.04804113134741783, -0.18747515976428986, 0.3927026093006134, 0.07327388226985931, 0.015613033436238766, 0.05264658480882645, 0.13754382729530334, 0.0773099958896637, 0.018104469403624535, -1.2644590139389038, 0.05883290618658066, 0.02333373762667179, 0.16385164856910706, 0.04296109825372696, 0.16792477667331696, 0.02114354632794857, -0.030651841312646866, 0.10195694118738174, 0.1726914495229721, 0.08461487293243408, 0.1504942774772644, 0.018999991938471794, -0.2909533679485321, 0.0021310981828719378, 0.09003584831953049, -0.04548797383904457, 0.03792142868041992, 0.2645266652107239, 0.14648157358169556, -0.13322032988071442, -0.09266263246536255, 1.1203030347824097, 0.029647866263985634, -0.008815099485218525, 0.2280045598745346, 0.11961592733860016, 0.4426192045211792, 0.21353578567504883, -0.0070702433586120605, 0.08638500422239304, -0.004797873552888632, -0.16269788146018982, 0.008593777194619179, -0.14142251014709473, -0.0034582912921905518, -0.07284446805715561, 0.05676458775997162, 0.11337559670209885, 0.026226729154586792, -0.011947651393711567, 0.008595344610512257, -0.17403274774551392, 0.11435689777135849, -0.004287214018404484, 0.006018409505486488, 0.017567062750458717, 0.022197067737579346, -0.5068437457084656, 0.03843458369374275, 0.46767279505729675, -0.0641382560133934, -0.014537950046360493, 0.14997386932373047, 0.0032803048379719257, -0.35003477334976196, -0.03814238682389259, 0.6643364429473877, -0.20282848179340363, -0.016790542751550674, -0.032004423439502716, -0.6276587247848511, 0.11831536889076233, 0.1850355863571167, 0.015692275017499924, 0.20264588296413422, -0.3919100761413574, 0.009103882126510143, 0.3541254997253418, 0.3874399662017822, -0.09369662404060364, 0.6396157145500183, 0.033067334443330765, 0.21246832609176636, -0.2734600305557251, 0.1162135973572731, 0.1155514121055603, 0.07382816821336746, 0.5820167660713196, -0.09037657827138901, -0.1592281013727188, -0.21807505190372467, 0.06305393576622009, 0.021695103496313095, 0.0389038547873497, 0.04281271994113922, -0.014287471771240234, 0.3069048225879669, -0.11831733584403992, 0.26266390085220337, 0.6722673773765564, 0.21164536476135254, 0.022547952830791473, -0.11402174830436707, -0.006168641149997711, -0.12097986787557602, 0.05757812783122063, -0.03192658722400665, 0.15130747854709625, -0.007189207244664431, -0.026036251336336136, 0.022521119564771652, -0.26642513275146484, -0.07954473793506622, -0.19046613574028015, -0.10110466182231903, -0.7834413051605225, 0.6083171367645264, 0.02804717607796192, 0.5562458634376526, -0.04224598780274391, -0.09795866906642914, 0.03874316066503525, 0.4688742756843567, 0.233514204621315, -0.354434996843338, -0.019358836114406586, 0.21255120635032654, -0.35815566778182983, 0.029117584228515625, -0.4263298809528351, -0.5318556427955627, 0.11838832497596741, -0.02802222967147827, -0.025527644902467728, 0.05511462688446045, -0.05322561785578728, -0.01790396124124527, -0.015627773478627205, -0.31981316208839417, -0.03480588272213936, 0.01699536293745041, 0.019517989829182625, 0.0008928379393182695, 0.011648795567452908, -0.05655473843216896, 0.35218387842178345, 0.09997998178005219, -0.029682079330086708, -0.061710432171821594, 0.27844372391700745, -0.09473255276679993, 0.014138059690594673, 0.45236584544181824, 0.3424862027168274, -0.11896509677171707, 0.038277849555015564, -0.011627357453107834, -0.010360878892242908, 0.2211533635854721, 0.10008349269628525, 0.025430282577872276, 0.14069396257400513, 0.07850214838981628, 0.2991378903388977, -0.06453543901443481, 0.5328559279441833, 0.020973097532987595, -0.06487215310335159, -0.17844711244106293, 0.11277274787425995, -0.056848570704460144, 0.33111807703971863, 0.0807657316327095, -0.1191076934337616, -0.07093890756368637, -0.17825283110141754, -0.2505365014076233, -0.30104050040245056, 0.004685632418841124, -0.05308320000767708, -0.01741020381450653, -0.5680784583091736, 0.02269711159169674, -0.02758750319480896, -0.2848113477230072, -0.3671005964279175, -0.09959301352500916, 0.14647892117500305, 0.13393685221672058, -0.026044512167572975, -0.2967666983604431, 0.4643155336380005, 0.03846694529056549, -0.9129365086555481, 0.07740198075771332, 0.014404848217964172, 0.010842378251254559, 0.017964377999305725, 0.1077592745423317, -0.04262441396713257, 0.38764888048171997, -0.008422155864536762, -0.025589371100068092, 0.03726869821548462, -0.013810547068715096, -0.9136838316917419, -0.016152406111359596, 0.07594276964664459, -0.068269282579422, 0.023681936785578728, 0.20082375407218933, 0.06794526427984238, -0.04403511434793472, -0.4117327928543091, -0.12138219177722931, -0.01535686943680048, -0.37528088688850403, 0.07031083852052689, -0.10542817413806915, 0.06389482319355011, 0.5161119103431702, 0.17035387456417084, -0.0925358459353447, 0.10520016402006149, 0.2004925012588501, 0.47017958760261536, 0.21653878688812256, -0.037215787917375565, -0.006545150652527809, 0.14904671907424927, -0.0020229157526046038, 0.14010190963745117, 0.0016979739302769303, 0.12700030207633972, 0.1588720828294754, -0.2196180671453476, -0.04520891606807709, 0.11777675896883011, -0.010779837146401405, 0.1679740995168686, -0.06311371177434921, 0.23807460069656372, -0.2048945277929306, -0.008426901884377003, 0.6407684683799744, 0.5444462895393372, -0.9755113124847412, -0.1260787397623062, -0.027465295046567917, 0.04943547770380974, 0.13310664892196655, -0.0009860781719908118, -0.5041545033454895, -0.03285473957657814, 0.934500515460968, 0.10771805047988892, -0.29925620555877686, -0.39720940589904785, 0.26980817317962646, -0.11260402947664261, 0.13352030515670776, -0.05987207219004631, 0.4031718373298645, -0.15653620660305023, -0.396564245223999, 0.005652160849422216, 0.15147699415683746, 0.15378384292125702, -0.000623652886133641, -0.34828224778175354, -0.45452743768692017, 0.2544056475162506, 0.11003682762384415, 0.7424466609954834, -0.08757134526968002, 0.004453184548765421, -0.1749270111322403, 0.037824392318725586, 0.049393314868211746, -0.0659712627530098, -0.009032931178808212, -0.3908151388168335, -0.5003637075424194, 0.011710952036082745, 0.11292684823274612, -0.06551696360111237, 0.3386487364768982, 0.008928167633712292, 0.14101842045783997, 0.052123840898275375, -0.027706200256943703, -0.04875955730676651, 0.2565154731273651, 0.1635286509990692, -0.014253129251301289, -0.4732639789581299, 0.2228137105703354, 0.0016542922239750624, 0.006016803439706564, 0.09178426116704941, 0.3870271146297455, 0.004267455544322729, -0.0024390160106122494, -0.014912205748260021, -0.45838797092437744, -0.019769078120589256, 0.13611195981502533, -0.14325274527072906, -0.14221905171871185, -0.20310138165950775, -0.1678057760000229, 0.0854944959282875, -0.0015622563660144806, 0.10858145356178284, 0.02493419498205185, 0.060172758996486664, -0.17472809553146362, -0.11267219483852386, 0.1319391429424286, 0.028107671067118645, -0.005603831261396408, 0.22111575305461884, 0.013655979186296463, 0.05339842289686203, -0.14722882211208344, 0.05107571929693222, 0.26834943890571594, 0.025755897164344788, 0.15107601881027222, 0.12868094444274902, 0.0543038547039032, 0.2546595633029938, -0.11206738650798798, -0.25650978088378906, -0.046138275414705276, -0.25182121992111206, 0.03248147293925285, -0.005195925943553448, -0.5515012145042419, 0.037033021450042725, 0.431403785943985, 0.25070783495903015, 0.020204024389386177, -0.44506561756134033, 0.5617194175720215, -0.015371576882898808, 0.07493097335100174, -0.6016087532043457, -0.22480428218841553, -0.0833640918135643, 0.02387668751180172, 0.22934330999851227, 0.17258071899414062, 0.11799219250679016, -0.6886768341064453, 0.26478537917137146, -0.1989605724811554, -0.4918268918991089, -0.01173118595033884, -0.012301219627261162, 0.0026238474529236555, 0.4177132844924927, 0.05993124470114708, -0.48831862211227417, -0.32001760601997375, -0.08504229784011841, -0.005633148830384016, 0.061081163585186005, -0.008093792013823986, 0.0856868177652359, -0.04019469395279884, -0.06268506497144699, -0.2483140379190445, -0.049226827919483185, -0.10210289806127548, -0.050504978746175766, -0.37657129764556885, -0.020900027826428413, -0.054872866719961166, -0.6069697141647339, 0.12029462307691574, 0.5594184398651123, -0.06420066207647324, 0.07799136638641357, -0.3037858009338379, 0.29836052656173706, -0.5082052946090698, -0.004266996402293444, -0.08599372208118439, 0.20051272213459015, 0.16673734784126282, -0.018644748255610466, 0.003405429655686021, -0.06756862252950668, 0.06658016890287399, 0.5684438347816467, -0.09461415559053421, 0.04049348831176758, -0.4473845064640045, 0.08236663788557053, -0.017400594428181648, -0.02309408038854599, -0.9127774834632874, -0.15143625438213348, -0.03205635026097298, -0.025198515504598618, 0.25002938508987427, -0.08521946519613266, 0.18405750393867493, 0.18067368865013123, 0.00808036606758833, -0.1180587112903595, 0.10770148783922195, -0.030532080680131912, -0.014007426798343658, -0.16288639605045319, 0.033554233610630035, 0.13908518850803375, 0.1566983014345169, 0.02914726920425892, -0.01536476518958807, 0.03372359275817871, -0.008222987875342369, 0.09860476851463318, 0.03595716878771782, -0.01603884808719158, -0.12789854407310486, 0.6519859433174133, -0.04119646176695824, -0.15528608858585358, -0.0715503990650177, 0.21890901029109955, 0.05586995184421539, -0.2640230357646942, -0.0171480905264616, 0.1870870441198349, -0.05131863057613373, 0.2533067762851715, -0.043938782066106796, -0.1799434870481491, -0.6976667642593384, -0.12502773106098175, -0.7756848931312561, -0.3192014992237091, -0.23280955851078033, -0.09264333546161652, -0.01982431672513485, -0.08660963177680969, 0.016708971932530403, -0.00786619819700718, -0.7264488339424133, -0.004413928836584091, 0.10372812300920486, 0.12247312813997269, -0.37347835302352905, -0.06639793515205383, 0.048682887107133865, -0.03710880130529404, 0.006275443825870752, -0.16819798946380615, 0.1592237949371338, 0.24008506536483765, 0.04882313311100006, 0.0021455478854477406, 0.12582163512706757, 0.21416312456130981, 0.5334846377372742, 0.2095719426870346, 0.020117027685046196, 0.014955408871173859, -0.024511266499757767, -0.06291443109512329, 0.1018984317779541, 0.0992283970117569, 0.1071433424949646, -0.7047141790390015, 0.06233872100710869, -2.7528233204066055e-06, 0.10558518767356873, 0.004756313282996416, 0.40520960092544556, 0.07149118185043335, -0.003213774412870407, 0.0037459770683199167, -0.07106524705886841, -0.0158064141869545, 0.3173657953739166, -0.04666419327259064, -0.026776153594255447, -0.001168227638117969, -0.03141786530613899, -0.10133417695760727, 0.11290249228477478, -0.01040428876876831, -0.11175291985273361, 0.05271296203136444, 0.5444212555885315, -0.005639669019728899, -0.09457066655158997, -0.010918047279119492, -0.014409027062356472, -0.16909611225128174, -0.05787762999534607, -0.14408664405345917, 0.04559437930583954, 0.40443480014801025, -0.2845296263694763, 0.029565434902906418, 0.05023225396871567, -0.11440904438495636, 0.21173922717571259, -0.14980316162109375, -0.1999443918466568, 0.042435601353645325, -0.013315171003341675, -0.29328808188438416, -0.09373241662979126, 0.08156344294548035, -0.060850318521261215, 0.07804597169160843, -0.36403611302375793, 0.09219890087842941, 0.4813830852508545, -1.3704464435577393, 0.22048722207546234, 0.14134414494037628, 1.208208680152893, 0.12999258935451508, -0.3466992676258087, 0.11626443266868591, 0.2589825391769409, -0.41135090589523315, -0.2137712836265564, -1.249800682067871, -0.24162469804286957, 0.016244353726506233, -0.14228612184524536, -0.05809743329882622, 0.28733524680137634, 0.004879384767264128, -0.10874241590499878, 0.014003165066242218, 0.011764055117964745, 0.33099469542503357, 0.4781637489795685, 0.46615251898765564, -0.008754345588386059, 0.21695229411125183, 0.08753915131092072, 0.2500540316104889, 0.4342821538448334, 0.03863094747066498, 0.002453441033139825, 0.09313909709453583, 0.3259598910808563, -0.03236493468284607, 0.008312095887959003, -0.2792030870914459, 0.010113012045621872, -0.5124329924583435, 0.08894166350364685, -0.011613807640969753, 0.32906660437583923, -0.0025755621027201414, -0.09482713043689728, 0.07703928649425507, 0.12853698432445526, 0.023313000798225403, 0.16723228991031647, -0.04970206320285797, -0.01924092136323452, -0.010342702269554138, 0.7731785178184509, -0.042725514620542526, -0.07308226078748703, 0.21428003907203674, 0.25076013803482056, -0.015983501449227333, 0.3260069191455841, -0.016148854047060013, -0.4226846396923065, 0.09383536875247955, 0.039818283170461655, -0.002860291860997677, 0.22694891691207886, 0.04219108074903488, 0.00976522732526064, 0.5426107048988342, -0.24799491465091705, -0.02097426913678646, -0.5282025933265686, 0.007927079685032368, 0.07575835287570953, 0.6664323806762695, -0.08938702195882797, -0.10596030205488205, 0.16359010338783264, -0.29072219133377075, -0.016795432195067406, -0.48907268047332764, -0.017778804525732994, -0.06824624538421631, -0.4511822760105133, 0.2901800572872162, -0.005916883237659931, 0.42500388622283936, 0.010393254458904266, 0.09536560624837875, 0.5523192882537842, -0.014076427556574345, -0.015301643870770931, -0.028772221878170967, 0.12337248027324677, 0.21534253656864166, -0.3706701099872589, -0.006607201416045427, -0.273485392332077, -0.21560892462730408, -0.05839274823665619, 0.23506462574005127, -0.01369455736130476, -0.006475311703979969, -0.09565706551074982, 0.25627753138542175, -0.15600503981113434, -0.005705748684704304, -0.21847091615200043, 0.5445102453231812, -0.2892943322658539, 0.08663943409919739, 0.16648069024085999, -0.03326606750488281, -0.5241801142692566, -0.36530688405036926, 0.08976919203996658, 0.03079093247652054, 0.059172190725803375, -0.04553436487913132, 0.015532947145402431, -0.8081854581832886, -0.2076689749956131, 0.8436159491539001, -0.11057429015636444, -0.5633169412612915, -0.005792848765850067, -0.06663552671670914, 0.5068222880363464, 0.07150736451148987, 0.0071804700419306755, 0.13558486104011536, -0.11286090314388275, -0.07316526025533676, -0.00746163772419095, 0.2908577024936676, 0.2790873348712921, 0.11451292783021927, -0.10563957691192627, 0.001206327578984201, 0.47903701663017273, 0.1695869266986847, -0.01686050370335579, 0.08297527581453323, -0.21643425524234772, -0.5742237567901611, -0.28444376587867737, 0.18493078649044037, 0.09252288937568665, 0.003177775302901864, 0.42688897252082825, -0.004660516045987606, 0.044658832252025604, 0.0825800821185112, -0.10748407989740372, 0.021343179047107697, -0.07814022153615952, -0.1282852292060852, 0.030963333323597908, -0.06403140723705292, -0.008620547130703926, -0.3010692894458771, 0.1211649477481842, -0.10128472000360489, -0.02346448227763176, 0.04253406077623367, 0.14137612283229828, -0.018155261874198914, -0.2770770490169525, -0.07418198138475418, 0.23121927678585052, 0.12319274991750717, -0.003112121019512415, 0.03488031402230263, 0.027056341990828514, 0.01508956216275692, -0.12938325107097626, -0.1599789410829544, 0.13049142062664032, -0.4613325297832489, -1.4464375972747803, -0.020835071802139282, 0.14225754141807556, 0.030411425977945328, -0.026973946020007133, 0.004267754498869181, 0.005896509159356356, -8.94071035872912e-06, 0.5930150747299194, 0.026239929720759392, 0.478628933429718, 0.07371385395526886, 0.5617006421089172, 0.00478943157941103, 0.3486858010292053, -0.10053305327892303, 0.06546927988529205, -0.08546028286218643, 0.058493804186582565, 0.02490270882844925, 0.2152322381734848, -0.171627938747406, 0.45893797278404236, -0.22000764310359955, 0.11182265728712082, 0.13543854653835297, -0.28003019094467163, 0.43351230025291443, -0.08021790534257889, -0.3803815245628357, -0.10556147992610931, 0.3116707503795624, 0.10523078590631485, 0.06971483677625656, 0.1465761810541153, -0.08151054382324219, -0.06206585839390755, -0.2190767526626587, -0.24028484523296356, -0.5293747782707214, 0.09480520337820053, -0.09609037637710571, -0.018112976104021072, 0.025639500468969345, 0.2578671872615814, 0.028600675985217094], [0.033549319952726364, -0.008848017081618309, -0.02334185689687729, -0.4017058312892914, -0.24524106085300446, 0.015026377514004707, -0.7310416102409363, -0.02460276521742344, -0.0580451563000679, -0.038509342819452286, 0.0012211593566462398, 0.06560994684696198, 0.020871836692094803, 0.21873998641967773, -0.011733831837773323, 0.06890320032835007, -0.08845888823270798, -0.27269646525382996, 0.21590909361839294, 0.0034858211874961853, -0.30763840675354004, 0.2420749068260193, 0.12953141331672668, -0.3203868567943573, -0.17366096377372742, 0.38375961780548096, -0.12841498851776123, -0.39477109909057617, 0.026688221842050552, 0.020609935745596886, 0.6513007283210754, 0.12204864621162415, 0.06648270785808563, -0.03479139506816864, -0.3104824721813202, 0.27576103806495667, -0.10954605787992477, 0.2665931284427643, 0.79823237657547, -0.09573999047279358, -0.1680089682340622, -0.048725806176662445, -0.10060391575098038, -0.030140137299895287, 0.0012323397677391768, -0.701272189617157, 0.044677820056676865, -0.013689753599464893, 0.2220153510570526, 0.01578916423022747, 0.07034388184547424, -0.00587946642190218, -0.0022734827362000942, 0.2993888854980469, -0.09213133156299591, 0.0365789569914341, 0.029485030099749565, -0.04409191012382507, -0.2799144685268402, -0.12479079514741898, -0.1575361043214798, -0.012656723149120808, 0.0008128858753480017, -0.45864027738571167, 0.08921954780817032, 0.3693062663078308, 0.11248643696308136, 0.19293813407421112, 0.10794629156589508, -0.060271188616752625, 0.04990854114294052, -0.12794265151023865, -0.035119567066431046, -0.11245489120483398, -0.5165650844573975, -0.24188314378261566, 0.27647310495376587, 0.08707769960165024, 0.01658271998167038, -0.5017244815826416, -0.011635278351604939, -0.11145524680614471, 0.10837852209806442, -0.19281212985515594, -0.3198257088661194, -0.1073564812541008, -0.04954393208026886, 0.022289473563432693, -0.5546635985374451, 0.1785385012626648, -0.46388766169548035, 0.32243868708610535, -0.23604360222816467, -0.2897409498691559, 0.0024202875792980194, -0.11297985911369324, -0.19764943420886993, 0.008498841896653175, 0.026161987334489822, -0.11306870728731155, -0.09957526624202728, 0.18006882071495056, 0.06658428907394409, -0.11117231845855713, 0.04973238706588745, -0.010816358961164951, -0.016117295250296593, -0.6389978528022766, -0.013334603048861027, -0.1221509575843811, 0.03680014982819557, -0.08966751396656036, -0.08065991848707199, -0.19854667782783508, -0.03021335043013096, -0.021312087774276733, 0.16715963184833527, 0.01931406930088997, 0.7124195694923401, 0.47506025433540344, -0.0988028347492218, 0.15425077080726624, -0.0039324224926531315, -0.02386472187936306, 0.013520603068172932, -0.0041125561110675335, 0.03748467564582825, -0.01696285977959633, -0.027309102937579155, 0.09256739169359207, -0.08839550614356995, -0.2865942120552063, 0.22304508090019226, 0.02081269770860672, -0.9627791047096252, 0.26312825083732605, 0.03643358126282692, -0.029464371502399445, 0.17654933035373688, 0.11898744106292725, -0.11401590704917908, -0.09842680394649506, 0.06475416570901871, 0.16869142651557922, -0.18535393476486206, -0.2019290179014206, -0.04155828803777695, -0.07060283422470093, -0.026776745915412903, -0.04609890282154083, 0.28006511926651, -0.0022115062456578016, -0.047972191125154495, 0.00028998535708524287, -0.27996087074279785, 0.07285591959953308, -0.09132809937000275, 0.1370357722043991, 0.05726120248436928, 0.027073264122009277, 0.009359454736113548, -0.05113004893064499, 0.20979191362857819, -0.060989540070295334, -0.11035571992397308, -0.08880618214607239, 0.05874161049723625, 0.190024733543396, -0.330563485622406, -0.10575088858604431, -0.08863350003957748, 0.09483890235424042, 0.05742023512721062, -0.043707702308893204, -0.3289681673049927, 0.3927036225795746, -0.16684116423130035, -0.27484041452407837, -0.16312681138515472, -0.1845681369304657, -0.118683360517025, -0.16294386982917786, -0.07552465051412582, 0.019310226663947105, -0.06256705522537231, 0.007295222021639347, 0.015524746850132942, -0.03521505370736122, 0.2183447778224945, -0.02797003835439682, -0.34653306007385254, -0.09545958787202835, 0.11118485033512115, -0.06643100827932358, 0.29507967829704285, -0.28045254945755005, -0.19937258958816528, 0.46522271633148193, 0.10852453112602234, 0.05541539564728737, 0.029181677848100662, 0.07783135026693344, -0.3213644027709961, -0.07868596166372299, 0.06293778866529465, -0.1306913197040558, 0.014484344981610775, 0.5663757920265198, 0.03287944942712784, -0.02703768014907837, -0.13792817294597626, 0.1161661222577095, -0.21951298415660858, 0.12987101078033447, 0.19816967844963074, -0.008877764455974102, 0.25373122096061707, 0.16497988998889923, -0.0009570536203682423, -0.10557904094457626, -0.7080247402191162, -0.1944480836391449, 0.2748950123786926, 0.15975156426429749, 0.0827278345823288, -0.03615802153944969, -0.02138395793735981, 0.01438047643750906, 0.0012706692796200514, -0.02105678990483284, 0.3194449245929718, -0.24285440146923065, -0.03541497886180878, -0.4681154489517212, -0.05365192890167236, -0.005120248068124056, 0.0042878324165940285, 0.3156324625015259, 0.27324041724205017, -0.2395552694797516, -0.18049746751785278, 0.020848657935857773, 0.2370915412902832, -0.022411661222577095, -0.23094038665294647, 0.16932883858680725, -0.041894737631082535, 0.09995473921298981, 0.553496241569519, -1.1205650568008423, -0.0713769719004631, 0.3256334066390991, 0.5717536211013794, 0.19276858866214752, -0.16266123950481415, -0.06742098927497864, 0.05022996664047241, -0.08519163727760315, 0.19123956561088562, -0.000962086662184447, 0.010642207227647305, 0.14237885177135468, -0.04586990177631378, 0.3344953954219818, -0.5436939597129822, -0.005422795657068491, 0.034621790051460266, 0.013275498524308205, 0.025872303172945976, 0.35747605562210083, 0.2956293821334839, -0.020175999030470848, -0.04214490205049515, -0.10607946664094925, -0.10983643680810928, -0.0357455350458622, 0.34251514077186584, 0.022672075778245926, 0.12256167829036713, 0.0073617491871118546, 0.038807038217782974, 0.3030526638031006, -0.0016671371413394809, -0.012842708267271519, -0.3217322826385498, -0.7153180241584778, -0.06957515329122543, -0.2601906359195709, -0.013393543660640717, -0.09249911457300186, -0.21685148775577545, -0.06169576197862625, -0.14134405553340912, 0.1319703757762909, -0.06610605120658875, 0.030934704467654228, 0.012300653383135796, 0.4091069996356964, 0.04446927085518837, -0.06877142935991287, 0.03525819629430771, -0.012961504980921745, -0.017622871324419975, 0.05724329873919487, 0.040231890976428986, -0.44454970955848694, 0.1648564636707306, 0.10455520451068878, -0.11306610703468323, -0.0023929968010634184, -0.015364578925073147, -0.0267011858522892, 0.012089595198631287, -0.3191607594490051, -0.02926718071103096, -0.7742320895195007, 0.06278131902217865, -0.16042301058769226, 0.01913224346935749, -0.0036219353787600994, -0.1615954488515854, -0.033990319818258286, -0.28049153089523315, -0.1754753440618515, -0.061153843998909, -0.008860047906637192, -0.6141465306282043, 0.16689914464950562, 0.2335727959871292, -0.008018257096409798, -0.07304482161998749, 0.6697906255722046, -0.029674550518393517, -0.19077840447425842, -0.16266211867332458, -0.21916912496089935, 0.07153267413377762, -0.03302018716931343, 0.2201595902442932, 0.30371686816215515, -0.1400560736656189, -0.30553069710731506, 0.07753553241491318, -0.7665864825248718, 0.054801639169454575, 0.010194448754191399, -0.09460406005382538, 0.14065618813037872, 0.023125743493437767, -0.1537766456604004, 0.09483939409255981, -0.023753440007567406, -0.19018642604351044, -0.37443411350250244, 0.0229883324354887, 0.6358622312545776, -0.04167701676487923, 0.20940044522285461, -0.10183996707201004, -0.010931778699159622, -0.03263944759964943, 0.03554604947566986, -0.19058822095394135, 0.2460329383611679, -0.015146473422646523, 0.05641542375087738, 0.033168356865644455, -0.0037674724590033293, 0.10841023921966553, -0.7203189730644226, -0.19121839106082916, -0.009836449287831783, 1.3226884603500366, -0.06739050149917603, -0.26480117440223694, -0.07425792515277863, 0.08520090579986572, -0.006959595717489719, -0.17456521093845367, 0.08077134191989899, 0.4139491021633148, 0.42904192209243774, 0.3381976783275604, 0.06241193413734436, -0.01117906253784895, -0.949063241481781, -0.04950262978672981, 0.8890859484672546, 0.04874471202492714, 0.023755813017487526, 0.1861579269170761, -0.10545360296964645, -0.030352188274264336, -0.01594425179064274, 0.3889266848564148, 0.03887728229165077, -0.15387147665023804, -0.035754892975091934, -0.10866619646549225, -0.10560790449380875, 0.08306258171796799, -0.1337142437696457, 0.29275140166282654, -0.00025115697644650936, 0.07250324636697769, 0.274496853351593, 0.18841250240802765, -0.001649576472118497, -0.15868449211120605, 0.42075273394584656, 0.0334511324763298, -0.01907416619360447, 0.3029940128326416, 0.009543079882860184, 0.5384608507156372, -0.02076098695397377, 0.36386990547180176, 0.10073807090520859, -0.19023694097995758, -0.42406150698661804, -1.111478567123413, 0.29014313220977783, 0.03967597335577011, -0.12257145345211029, -0.0531228743493557, 0.03186500072479248, 0.3476126790046692, -0.4477843642234802, 0.0014669293304905295, 0.1817418336868286, -0.39257529377937317, 0.20041602849960327, 0.7477763891220093, -0.160354346036911, -0.021257895976305008, -0.012639782391488552, 0.02532806620001793, 0.274783730506897, -0.0048166257329285145, -0.06650304794311523, -0.36074140667915344, -0.15112876892089844, -0.10599629580974579, 0.11093819886445999, 0.28863298892974854, -0.028653018176555634, 0.12030185014009476, 0.28890320658683777, 0.012993382290005684, 1.8811708688735962, 0.031072041019797325, -0.0040472690016031265, 0.00781480222940445, 0.001245100051164627, 0.23417668044567108, -0.03200783208012581, 0.4399958848953247, -0.03135114163160324, 0.06355107575654984, 0.5145746469497681, -0.0840773954987526, -0.12616729736328125, 0.13437728583812714, 0.06536287069320679, 0.0726538673043251, -0.17301885783672333, -0.07598463445901871, 0.14497438073158264, -0.010407739318907261, 0.11006072163581848, -0.02637937106192112, -0.032581448554992676, 0.15574975311756134, -0.0011111270869150758, 0.2827651798725128, -0.004181938245892525, 0.423525869846344, 0.3623117506504059, -0.08438931405544281, -0.29470106959342957, 0.12516309320926666, -0.05250653252005577, 0.2093239575624466, -0.051006946712732315, -0.011290254071354866, 0.24296794831752777, -0.2848051190376282, -0.23081465065479279, -0.04446681588888168, -0.09130404144525528, 0.5819268822669983, -0.19025057554244995, 0.028476789593696594, 0.04234536364674568, -0.010634656995534897, 0.2563835382461548, -0.08771886676549911, -0.4835151731967926, 0.0021093597169965506, -0.013403435237705708, 0.5087056756019592, 0.4343113601207733, 0.35268068313598633, -0.3615078330039978, -0.04869987443089485, -0.1925639510154724, -0.00048389448784291744, 0.12807777523994446, -0.04476625472307205, 0.02710556611418724, -0.3861038386821747, 0.21825966238975525, 0.2849777042865753, 0.16488993167877197, 0.07571686059236526, -0.1336176097393036, 0.6375306248664856, 0.04600643366575241, 0.2799261510372162, -0.10503244400024414, 0.40228205919265747, 0.04532160609960556, 0.08350867033004761, -0.12332770228385925, 0.0009773517958819866, 0.24170935153961182, 0.49361932277679443, 0.24880780279636383, -0.1477643996477127, 0.35098499059677124, -0.2795095443725586, -0.003451018128544092, 0.13467338681221008, 0.004480275325477123, 0.13094080984592438, -0.12900662422180176, -0.0033776243217289448, -0.6333168745040894, 0.36006876826286316, -0.03854601830244064, -0.936829686164856, -0.2738674283027649, 0.0877421572804451, -0.04859732836484909, 0.006291267462074757, -0.06626028567552567, -0.21549275517463684, 0.10682680457830429, -0.09582261741161346, -0.4232148230075836, -0.022249137982726097, 0.11070200800895691, -0.01771511323750019, -0.12761296331882477, -0.034921348094940186, -0.021406136453151703, 0.12281593680381775, 0.019732072949409485, -0.2025270313024521, 0.002464377786964178, -0.37688595056533813, 0.022999297827482224, 0.09184935688972473, 0.17048390209674835, 0.013480935245752335, 0.1120297908782959, -0.19187337160110474, -0.23211851716041565, -0.01209172885864973, -0.03398735448718071, 0.028675338253378868, -0.033026471734046936, 0.06824153661727905, -0.011148967780172825, -0.031266238540410995, -0.0008974935044534504, -0.0016957323532551527, -0.313398540019989, -0.044850945472717285, -0.12290321290493011, 0.015970434993505478, 0.04555970057845116, -0.055928342044353485, 0.06521903723478317, 0.13040849566459656, 0.017912758514285088, -0.01621156558394432, 0.052107132971286774, 0.050797443836927414, 0.4824245274066925, -0.022182809188961983, 0.27483588457107544, -0.028368394821882248, -0.008146329782903194, -0.26005318760871887, -0.15330900251865387, 0.14013971388339996, 0.16983114182949066, 0.08091957867145538, 0.38398823142051697, -0.29747363924980164, -0.03079388104379177, 0.18728353083133698, 0.7634490728378296, 0.05897193029522896, 0.052169524133205414, 0.014241485856473446, 0.37566784024238586, -0.07031286507844925, -0.1246093213558197, -0.6876856088638306, 0.4894489645957947, 0.18093085289001465, 0.1344115138053894, 0.03470989316701889, 0.06576655060052872, 0.017439652234315872, -0.4827091097831726, -0.07403066754341125, 0.5027654767036438, -0.025722596794366837, 0.26986661553382874, -0.012224621139466763, -0.05799998715519905, -0.018919993191957474, -0.06452706456184387, -0.14207418262958527, -0.25579914450645447, 0.1333843320608139, -0.01085186842828989, -0.0899863913655281, -0.012017947621643543, -0.32222670316696167, -0.1958600878715515, -0.08189180493354797, -0.3648763597011566, 0.23582133650779724, 0.1426575630903244, -0.06342881172895432, -0.08136142045259476, -0.06099272891879082, 0.014199774712324142, 0.09539534896612167, -0.009214051999151707, 0.06361274421215057, 0.17360901832580566, -1.3279590606689453, -0.018185898661613464, -0.009960513561964035, -0.07275308668613434, -0.10334005206823349, -0.28765636682510376, -0.3674209713935852, -0.0030709728598594666, 0.11892274022102356, 0.35343417525291443, 0.02323170006275177, 0.030741913244128227, -0.09453169256448746, -0.635044515132904, -0.06044389307498932, -0.04389941319823265, 0.14394766092300415, -0.01433541253209114, 0.15352191030979156, -0.012547403573989868, -0.0013489831471815705, 0.1504032462835312, -0.07868076115846634, -0.09362751990556717, -0.017915815114974976, -0.09778288006782532, -0.024980654940009117, -0.1881455034017563, 0.2872987389564514, 0.1075185239315033, -0.016338342800736427, -0.39792969822883606, -0.02457674965262413, -0.43704351782798767, -0.017079567536711693, -0.004530449863523245, 0.1970791071653366, -0.5185396075248718, -0.0013327209744602442, -0.09734929352998734, -0.017858004197478294, -0.23369328677654266, 0.6372608542442322, 0.06352609395980835, -0.012291254475712776, 0.0014645274495705962, 0.6320582032203674, 0.07438145577907562, -0.06559167057275772, 0.014994777739048004, 0.12931609153747559, 0.11210155487060547, 0.3746514618396759, 0.23305879533290863, 0.08990941941738129, -0.15955713391304016, 0.07906252890825272, 0.19245629012584686, -0.01795164868235588, -0.013140245340764523, -0.24043837189674377, 0.18704643845558167, 0.0453890897333622, 0.006998684257268906, 0.18349815905094147, 0.10873293876647949, -0.042811859399080276, -0.14979690313339233, -0.011673873290419579, -0.009764458984136581, 0.2603726089000702, -0.3499350845813751, -0.28427034616470337, 0.3373118042945862, -0.15152877569198608, 0.09245288372039795, -0.08821551501750946, 0.975588321685791, -0.020548511296510696, 0.08641239255666733, -0.01788513734936714, -0.017028819769620895, -0.012524896301329136, 0.10488585382699966, -0.1368454098701477, 0.11003104597330093, 0.20821689069271088, -0.08601288497447968, -0.07538439333438873, 0.038644712418317795, -0.27548471093177795, 0.016621878370642662, -0.03308352082967758, -0.00610495125874877, 0.04084234684705734, -0.07108671218156815, 0.2927575707435608, -0.3105529248714447, -0.09306494891643524, -0.011939799413084984, -0.07412520051002502, -0.46687695384025574, -0.14921627938747406, -0.022604433819651604, -0.026092994958162308, -0.03989239037036896, -0.05470743030309677, -0.0051904539577662945, 0.06422359496355057, -0.34324079751968384, -0.0022492394782602787, -0.35460472106933594, -0.009722963906824589, 0.22492921352386475, -0.04105137288570404, -0.06547407060861588, 0.17011648416519165, -0.21164852380752563, -0.008597072213888168, -0.0409177727997303, 0.5763259530067444, -0.04026102274656296, -0.030041616410017014, 0.014628591947257519, -0.04633409157395363, 0.18001225590705872, 0.017820192500948906, -0.01409439742565155, -0.04827151075005531, 0.15435920655727386, 0.049496494233608246, 0.13599157333374023, 0.5009105801582336, 0.05433918908238411, -0.09976082295179367, 0.005468663759529591, 0.4792814254760742, -0.12014599144458771, 0.05207731947302818, -0.04718273505568504, 0.49038058519363403, 0.17497305572032928, 0.023390304297208786, -0.20422960817813873, 0.2871772348880768, -0.533947765827179, -0.15377803146839142, -0.47713688015937805, 0.20756730437278748, -0.02004949375987053, -0.17090894281864166, -0.011693124659359455, -0.010467025451362133, 0.05771956220269203, 0.002696278505027294, 0.028493627905845642, -0.12522028386592865, -0.019981181249022484, -0.028411252424120903, -0.10460039228200912, -0.5936939716339111, -0.30809634923934937, -0.012295662425458431, 0.009982090443372726, 0.13815073668956757, 0.013424242846667767, 0.004915199708193541, 0.2616569399833679, -0.04284536466002464, 0.06613136827945709, -0.19681525230407715, -0.018095793202519417, 0.5321447253227234, 0.02555273286998272, -0.7430729269981384, 0.04846041649580002, 0.07597383856773376, 0.0735902339220047, -0.04341026768088341, -0.09504188597202301, -0.002194305881857872, -0.046979326754808426, -0.08365941792726517, -0.024951422587037086, -0.7581637501716614, 0.11502555012702942, 0.30149373412132263, 0.03836560994386673, -1.0721089839935303, -0.010737357661128044, 0.3589162528514862, -0.018212322145700455, -0.037619464099407196, -0.07938329875469208, 0.06682634353637695, 0.07270278036594391, 0.05175234004855156, -0.14423081278800964, -0.0545121394097805, -0.09981434047222137, 0.10727550834417343, -0.017135603353381157, -0.14377020299434662, 0.2754095792770386, 0.2320987433195114, 0.3467876613140106, -0.02732918970286846, -0.2005329132080078, -0.030833210796117783, -0.28249484300613403, -1.043031930923462, -0.05228622630238533, 0.2531110942363739, 0.3368540108203888, 0.0028422032482922077, 0.2461755871772766, -0.0055387248285114765, -0.012947467155754566, 0.639797568321228, 0.0838591456413269, -0.04694095253944397, -0.012690606527030468, 0.6413766145706177, -0.08797422796487808, 0.3315097987651825, -0.3790224492549896, -0.10786516964435577, 0.07974445074796677, 0.13962949812412262, -0.11874253302812576, -0.014635924249887466, -0.017935287207365036, -0.0032537297811359167, -0.2348346710205078, 0.3542019724845886, -0.006826310884207487, -0.036733221262693405, -0.17614591121673584, -0.14909785985946655, 0.2199302613735199, -0.019526584073901176, -0.012979681603610516, 0.07212625443935394, 0.5154476761817932, 0.0343734547495842, 0.004192361608147621, -0.11324013024568558, -0.011228051967918873, 0.14404496550559998, 0.12413227558135986, 0.07282301783561707, -0.002325943438336253, -0.14282207190990448, -0.018719661980867386, 0.002434186637401581, 0.28717052936553955, -0.20301568508148193, -0.05099034309387207, -0.06431123614311218, 0.08336987346410751, -0.10537203401327133, -0.05614247918128967, -0.006329303607344627, 0.0490126870572567, 0.15116330981254578, 0.046610280871391296, 0.0914493128657341, -0.0008663552580401301, -0.11985159665346146, 0.07834209501743317, -0.005010741762816906, 0.014386096969246864, -0.38029035925865173, -0.30661916732788086, 1.1627262830734253, -0.10921159386634827, -0.3699094355106354, 0.09885847568511963, 0.1042974665760994, -0.010737201198935509, -0.040303636342287064, -0.06842504441738129, -0.10973452776670456, -0.06661935150623322, -0.11978086829185486, -0.615336000919342, -0.2888786196708679, -0.04458266496658325, 0.08044853806495667, -0.21088513731956482, 0.25789040327072144, 0.05465655401349068, -0.20784811675548553, 0.05673682317137718, 0.058030866086483, 0.017438380047678947, 0.20295587182044983, -0.030615821480751038, -0.3251504600048065, 0.18300098180770874, -0.09613333642482758, 0.30544546246528625, -0.054944586008787155, -0.03545306622982025, -0.009514846839010715, 0.30272871255874634, -0.38210809230804443, 0.2843025326728821, 0.7757582068443298, 0.013937191106379032, 0.13359598815441132, -0.07203651964664459, -0.03710756450891495, 0.007478030864149332, -0.01887473277747631, -0.015944652259349823, -1.122412085533142, -0.01847245916724205, -0.2137356549501419, -0.24966852366924286, 0.0018859715200960636, -0.00851078238338232, 0.037743691354990005, -0.03538908809423447, -0.03702405467629433, -0.10915851593017578, 0.12734070420265198, -0.13158278167247772, 0.06820915639400482, -0.37636879086494446, 0.15655586123466492, 0.21493136882781982, 0.3878544867038727, 0.5081844329833984, 0.38831567764282227, -0.18198497593402863, 0.09842266887426376, -0.11280860751867294, 0.11391755193471909, -0.0345543809235096, 0.0029329543467611074, -0.10248538851737976, -0.007219585124403238, 0.08463698625564575, -0.04859556630253792, 0.2205324023962021, 0.09456691145896912, 1.135769009590149, -0.034703515470027924, 0.3011867105960846, -0.08514704555273056, -0.005110094323754311, -0.22795316576957703, -0.04336799681186676]], "dtype": "float32"}, {"__nd__": [-3.3528666496276855, -1.2386730909347534, -0.03184697777032852, -0.22541725635528564, -1.5181697607040405, 0.18269620835781097, 1.1907075643539429, 0.4470689594745636, 1.1061440706253052, 0.6163389682769775, -0.336239755153656, 0.4901595413684845, 0.23293347656726837, 1.5011107921600342, 1.9457203149795532, -0.6091729402542114, 0.5309506058692932, -3.340548276901245, 0.6663204431533813, -0.017520980909466743, 0.399815171957016, -2.340778350830078, 2.3802080154418945, -0.18347209692001343, 0.858416736125946, -0.6772615313529968, 2.0875351428985596, 2.5182788372039795, -1.9159493446350098, 0.08515622466802597, 0.6246330738067627, 0.7381555438041687, 0.236308753490448, -0.3646045923233032, -1.7049974203109741, -0.3434186279773712, 0.6028709411621094, -0.18815919756889343, 1.3058542013168335, 1.54911470413208, -1.8935571908950806, -0.796227216720581, -1.705045223236084, -1.060162901878357, 2.197284698486328, -1.0786422491073608, 2.4259276390075684, 4.936219215393066, 0.6918802261352539, 0.48772507905960083, 1.5566946268081665, 1.3132641315460205, -1.8175740242004395, -4.010911464691162, 5.290969371795654, 1.8705800771713257, -2.6338701248168945, -0.9263343214988708, 0.4971950650215149, 2.5653889179229736, 0.4012293517589569, 1.428945541381836, -0.17972321808338165, 0.981575608253479, 1.9637526273727417, 1.0107241868972778, -0.8811661005020142, 0.45788711309432983, 4.176046848297119, 1.134802222251892, -0.32848888635635376, 0.8960219621658325, 0.052094247192144394, 1.427298903465271, 4.025830268859863, 1.542436957359314, 0.889166533946991, 1.5833017826080322, 1.4474902153015137, 3.651754379272461, -1.0670976638793945, -0.09394939243793488, -0.8831499218940735, -0.11530351638793945, 0.7059419751167297, 2.144440174102783, 1.6802057027816772, 1.914503574371338, -2.1532435417175293, 0.5653846263885498, -0.5097260475158691, -1.2167669534683228, 0.6233768463134766, 2.3625552654266357, -0.23473073542118073, 1.309592366218567, -4.424851894378662, -0.25620898604393005, -0.13574476540088654, 1.7319523096084595, -2.4816832542419434, 2.0439178943634033, -1.676145076751709, 0.9746576547622681, 0.5699012875556946, 0.7549960613250732, -0.5506245493888855, -0.5460492372512817, -4.890885829925537, 0.2522476613521576, -0.36026906967163086, 2.0587375164031982, 0.26933014392852783, -1.9663581848144531, 0.5102147459983826, -0.6445649862289429, 1.4390803575515747, 0.1978219598531723, 0.8382559418678284, 0.8072547912597656, 1.7123889923095703, -2.490351438522339, -0.265389084815979, 0.6666845083236694, -1.1876988410949707, -1.1999610662460327, 1.7062023878097534, -0.561107337474823, 1.2448855638504028, -1.4602113962173462, 0.9342052936553955, 0.607591986656189, -3.2520737648010254, -1.097893476486206, 5.669551372528076, -2.09635591506958, 0.7597948312759399, -1.6022050380706787, 4.7966837882995605, -0.35051119327545166, -0.8853877782821655, -0.6187677383422852, 1.9826363325119019, -1.6244213581085205, 1.8909790515899658, 1.3303678035736084, 1.5585256814956665, -1.0535134077072144, 4.3361334800720215, 1.5163770914077759, 1.0521103143692017, 0.2925078868865967, -2.7616825103759766, -0.2982935905456543, 0.011030921712517738, -0.896827757358551, -0.22211791574954987, 1.029686689376831, 0.26709625124931335, 0.043456945568323135, 0.3507440388202667, 1.140007495880127, -1.299939751625061, 0.07771365344524384, -0.929693877696991, -0.32545915246009827, -1.4303532838821411, 0.7790865302085876, 1.5232365131378174, 3.2267439365386963, -0.9089052081108093, 1.1055790185928345, 0.6969685554504395, -0.36642372608184814, -0.5426825284957886, 2.26106858253479, -0.3783031702041626, 0.591720700263977, -0.5754402875900269, -0.7101362943649292, -2.05053448677063, -0.580263078212738, 0.1901855766773224, 1.2338459491729736, 0.864211916923523, 4.173492908477783, 0.303740918636322, -1.116762399673462, 0.5876106023788452, -0.9809025526046753, -1.3058832883834839, 0.08027336001396179, 1.8091623783111572, 2.0394446849823, 2.1864356994628906, -1.675309419631958, 0.1685454398393631, -1.0243871212005615, -1.3607921600341797, 1.1094468832015991, 0.05443785339593887, 6.000568866729736, -0.774278461933136, 0.7404428720474243, -1.2410389184951782, -2.111548900604248, -4.584357738494873, -0.8184050917625427, -0.25226712226867676, -1.146098017692566, -0.36322885751724243, 5.337036609649658, 0.6914400458335876, 1.8204751014709473, 0.48959603905677795, 2.442988157272339, -2.473255157470703, -0.5130115151405334, -1.365167498588562, -2.5573911666870117, -0.09113617986440659, -1.6771163940429688, 0.6360028982162476, 1.3572981357574463, -1.4356329441070557, -2.364978313446045, -0.30196741223335266, 1.5158791542053223, 0.10988309979438782, 0.3359358608722687, -1.0399572849273682, 0.22107918560504913, -0.005468116607517004, 4.397480010986328, 0.636116623878479, -0.2767030894756317, -0.10223784297704697, 0.9365320801734924, -0.1706976592540741, -1.1412118673324585, 2.816566228866577, -0.2969040870666504, -1.6970410346984863, 1.1996911764144897, -1.9951364994049072, 2.3564600944519043, 0.35825735330581665, 1.1916764974594116, -1.1435859203338623, 0.9113212823867798, -5.154904842376709, 5.727583408355713, -2.55629825592041, -1.1115108728408813, -0.6993421316146851, 0.9241241216659546, -0.7245675325393677, -1.4302504062652588, 0.5739270448684692, 1.7238043546676636, -1.8316280841827393, -1.5674558877944946, -0.3088383972644806, -0.924593985080719, -1.3583106994628906, -2.6585638523101807, -2.7218997478485107, -2.0420494079589844, -2.000410556793213, -2.251491069793701, 0.2396026998758316, -0.08405252546072006, 0.05640995129942894, 1.7771649360656738, -0.8215882182121277, 0.7224547266960144, -0.018743764609098434, -0.9854300022125244, -0.5806968212127686, 1.6105355024337769, -0.4862210154533386, 1.0981284379959106, 0.06299684196710587, 2.4533674716949463, 1.4389039278030396, -7.144890308380127, -0.17528435587882996, 7.421546459197998, -2.3471670150756836, 0.676902711391449, 2.2034831047058105, 0.42192718386650085, -1.9667778015136719, -0.06019848957657814, 1.7075152397155762, 0.48225900530815125, -2.7568471431732178, -1.1660642623901367, 5.60717248916626, -1.0195211172103882, -3.75506591796875, -1.1696306467056274, 0.9315707683563232, -2.065159559249878, 0.2743072211742401, 0.6113553047180176, -0.5798839330673218, -0.5673759579658508, -1.4572386741638184, -1.8989752531051636, 0.07992967963218689, 1.8852548599243164, 0.6722607016563416, -0.2973197400569916, 1.9280833005905151, 0.6654052138328552, -0.09224273264408112, 0.26204854249954224, 1.1321924924850464, 0.879891574382782, 2.2935802936553955, -1.4086718559265137, 2.1636500358581543, -0.0013786409981548786, -1.4529483318328857, -0.6955915093421936, -3.29573917388916, -2.577359914779663, 1.0380736589431763, 0.7369521260261536, -1.0496692657470703, -0.0020480549428611994, 3.207080125808716, -1.7975484132766724, 0.46497708559036255, 1.7769224643707275, 0.5549704432487488, -0.8646950125694275, -0.1937600076198578, -0.45526251196861267, 0.799290657043457, -3.6899490356445312, -0.3649022579193115, 0.2783008813858032, -0.33008262515068054, 0.5963950753211975, -1.5211162567138672, -1.5843898057937622, -0.3945331275463104, 2.6104934215545654, 0.2649950087070465, -0.9117636680603027, -0.982341468334198, -3.8674473762512207, 1.006263256072998, 0.5923559665679932, 0.06403051316738129, 1.3816007375717163, -2.61098313331604, -0.20558613538742065, 0.8994939923286438, 1.4934701919555664, 1.6608870029449463, 0.5363618731498718, 0.10146725177764893, 0.5062346458435059, 0.6666926741600037, -0.5647971630096436, -7.284213542938232, 1.9231480360031128, 0.20177702605724335, 3.352027416229248, -0.509817361831665, 0.08568087965250015, 3.3099958896636963, 3.3743460178375244, -0.3608991205692291, 3.323220729827881, -0.2479558140039444, -0.7749354839324951, -1.419930338859558, 1.5517570972442627, -1.7632468938827515, -0.16060170531272888, -0.6389110684394836, 1.2054684162139893, 1.3715239763259888, -0.8629277348518372, 0.017253799363970757, 0.17479504644870758, 0.1912674903869629, -0.5323824882507324, -1.933464765548706, -0.037698134779930115, 3.272411823272705, 0.39606747031211853, -0.028257926926016808, -0.11388252675533295, 0.27091315388679504, -0.2865041196346283, 1.2263787984848022, -0.8992881774902344, -3.26111102104187, -1.70768404006958, -1.7585049867630005, 0.032079990953207016, 4.394952297210693, 0.5508700013160706, -0.09465976059436798, -1.6243139505386353, 1.690970778465271, -0.22551266849040985, 2.063190221786499, 0.6906108856201172, -0.015291442163288593, 1.7601770162582397, -1.5263631343841553, -0.9384115934371948, 1.6389367580413818, 1.3897546529769897, 2.3331310749053955, 1.002557635307312, 1.98444664478302, 0.6118946671485901, -0.8989924192428589, 1.3632322549819946, -1.2640364170074463, -2.9048659801483154, -0.8862048387527466, 0.14566417038440704, -6.4489970207214355, 0.015212886966764927, 2.0835976600646973, -1.5325126647949219, 0.5202836990356445, -0.5138884782791138, 2.3308403491973877, 1.512296199798584, 0.7846645712852478, 1.501216173171997, -1.7371759414672852, -0.03763885423541069, -4.160354137420654, -3.0427279472351074, 0.2901574373245239, 0.7043548822402954, 0.5723311305046082, 1.1458723545074463, -1.2441926002502441, -0.2689111828804016, 0.516740620136261, -0.7082193493843079, 0.6610272526741028, 1.0784778594970703, 0.8048552870750427, -0.9788370728492737, 0.529471755027771, -0.27305513620376587, 1.5097503662109375, -2.476454734802246, 0.8187081813812256, -1.4968161582946777, 1.3420391082763672, 0.8939201831817627, 1.5472080707550049, 0.027193790301680565, -1.5362614393234253, 2.437697410583496, 0.6164406538009644, -0.8750268816947937, -0.1492428183555603, 2.464604139328003, -0.2073206752538681, -0.8425028324127197, -0.31476473808288574, -2.2952818870544434, 2.451639413833618, 2.311560869216919, 0.41880035400390625, 0.6071523427963257, -5.006810665130615, -1.3415453433990479, 0.5687206387519836, 2.226864814758301, -0.020830966532230377, -0.4894910454750061, -1.5274397134780884, -0.5907983183860779, 0.4057117700576782, -1.5940687656402588, 0.668556809425354, 0.6616387963294983, -2.565638780593872, -2.76719331741333, -0.5490818023681641, -3.2983291149139404, -1.1006208658218384, -0.4914930760860443, 0.08849333226680756, 1.4100457429885864, -1.4620918035507202, 0.4320773780345917, 1.3637521266937256, -4.973317623138428, 0.1572759449481964, -0.5462211966514587, 0.06965623795986176, -1.2199000120162964, 0.9751790165901184, 0.3539232313632965, -5.863742351531982, -3.5974507331848145, 0.32746177911758423, -0.3286123275756836, 0.9163029193878174, -0.07476866990327835, 1.0063765048980713, 1.2426811456680298, -1.1822715997695923, 3.5882978439331055, 0.13185523450374603, 1.1924800872802734, 2.649606943130493, -4.900757789611816, -1.0551698207855225, 0.9015302658081055, 2.5031020641326904, -3.3190441131591797, -0.3582247495651245, -3.365090847015381, 2.0195512771606445, -0.1765267550945282, -1.693218469619751, -0.8231931328773499, 2.077765703201294, -1.1233718395233154, 0.4519209563732147, -0.40555325150489807, -2.7117726802825928, 1.1131423711776733, 3.504486322402954, -2.7030394077301025, -1.9951601028442383, 1.3325830698013306, -0.9786432385444641, -0.2212296426296234, -2.648839235305786, -1.31081223487854, -0.33125725388526917, -0.6184135675430298, 0.2263580709695816, 1.2937965393066406, 0.9776735305786133, 1.207478642463684, -0.8397232890129089, -0.22947709262371063, 0.8089931607246399, 0.17217370867729187, -2.3032963275909424, -2.876368999481201, 2.01741099357605, 0.008668185211718082, 1.2750275135040283, -1.0325863361358643, 0.21454177796840668, 1.3557149171829224, 2.8544342517852783, -0.6014315485954285, 0.04402706399559975, -1.3257622718811035, -0.37832993268966675, 0.4363992512226105, 2.548166513442993, -0.5227946639060974, -0.3221767246723175, 0.7129747271537781, 0.5691397190093994, -1.0640439987182617, -3.3219308853149414, 1.5582988262176514, -2.357449531555176, 1.3148478269577026, 2.0118355751037598, -0.1396213322877884, 1.5590487718582153, -1.0878645181655884, -2.299664258956909, 1.14461350440979, 2.355553388595581, -2.558964967727661, 1.3658344745635986, 2.041729688644409, -1.175156593322754, 0.008434869349002838, -2.516669273376465, -0.15548047423362732, 3.143204689025879, -0.5017221570014954, 1.5104888677597046, -0.46816059947013855, -0.3279436230659485, -0.9045577049255371, 2.775202989578247, -2.0417823791503906, -0.7648550868034363, 2.1493759155273438, -2.5022153854370117, -0.9433067440986633, 0.20460295677185059, 1.1470277309417725, -1.414241075515747, -0.6452017426490784, 3.6836206912994385, -2.777371883392334, 0.9332119822502136, 1.550247073173523, 1.3900541067123413, -2.6266937255859375, 0.057100895792245865, -1.3295084238052368, -1.8810970783233643, 0.29478174448013306, 0.9777795672416687, 0.03887627646327019, 0.356866717338562, -2.2647829055786133, 2.0442912578582764, 3.5699005126953125, 0.42141565680503845, 0.6332805156707764, 1.3163701295852661, -0.19113050401210785, 2.298166513442993, -0.5257977843284607, 1.4913966655731201, 3.908508777618408, -4.5882887840271, 0.22889116406440735, -0.4700002670288086, 6.556785583496094, -1.0520281791687012, -1.7112312316894531, 0.1969369351863861, -0.2643020749092102, -1.8693166971206665, -0.6514203548431396, -2.442298412322998, -2.7938108444213867, -0.2994803190231323, -2.0245418548583984, -3.25413179397583, -4.849415302276611, -0.14507333934307098, -0.3306524455547333, 1.2401152849197388, -0.9565991759300232, -0.559356153011322, -0.7160286903381348, 1.0842070579528809, 3.132267475128174, -0.3275321424007416, 0.8405163884162903, -0.6354812979698181, 1.1106048822402954, 0.21897508203983307, -1.2893022298812866, 1.3559497594833374, -0.9580652117729187, -1.9773998260498047, 1.8657087087631226, -1.1002596616744995, -0.5938377380371094, 0.46863189339637756, 1.3385634422302246, 1.2931413650512695, 0.7356868982315063, 0.058736421167850494, 1.7519911527633667, -1.1901943683624268, -0.5835902690887451, -4.411533355712891, 0.07612491399049759, -0.9843766689300537, -0.6252343058586121, -1.9444767236709595, -0.4589894413948059, 0.4706880748271942, -1.8569416999816895, 1.7550551891326904, -1.9670851230621338, 0.617681086063385, 1.0074336528778076, 1.2655975818634033, 0.3368362784385681, 1.1308965682983398, 0.9224303364753723, -1.2276095151901245, -0.31043514609336853, 6.877756595611572, -0.4639744460582733, 0.0416870191693306, 1.5029689073562622, -1.8464871644973755, -0.5360127091407776, 1.5661042928695679, -1.1516704559326172, -0.2891758382320404, -1.948776125907898, -0.24744457006454468, -0.6953326463699341, -0.5795996785163879, 0.23866011202335358, -1.3612765073776245, -0.9449026584625244, 1.1221870183944702, -1.0713425874710083, 0.16977271437644958, -0.10654294490814209, 1.7185128927230835, -2.182753324508667, 1.1109737157821655, -0.9791713356971741, 4.76779317855835, 0.40521135926246643, 0.9689962267875671, 2.7720437049865723, -0.04886600747704506, -1.847753882408142, 1.6340388059616089, -0.18799883127212524, -0.9586473703384399, -1.1427178382873535, 2.7594830989837646, -0.19498459994792938, 0.6084615588188171, 0.5540446043014526, 0.597080647945404, -3.334550619125366, -2.2477898597717285, -2.7424087524414062, -1.3943510055541992, 1.471137285232544, -0.6198802590370178, -0.767932653427124, 2.0536975860595703, -0.2434825748205185, -0.3717772662639618, -1.089662790298462, -0.17310397326946259, -0.17065078020095825, -0.3005806505680084, 1.9802299737930298, 0.8642796277999878, -0.296409547328949, 0.5246881246566772, -0.28414466977119446, 0.28058332204818726, 1.0237947702407837, 0.9875238537788391, -2.1803736686706543, 1.054557204246521, -2.7703654766082764, 3.3078482151031494, -2.3950252532958984, -0.2364700585603714, -1.2089390754699707, -1.2903858423233032, -0.09284812211990356, 0.7124503254890442, 1.730276107788086, -1.7322397232055664, 2.854121446609497, -1.4330787658691406, 0.13786257803440094, -0.6860470771789551, -1.9079172611236572, 2.495281219482422, 1.6361216306686401, -2.1086061000823975, 0.04455135390162468, 0.3526201546192169, -1.6230446100234985, 1.7144923210144043, -1.2546815872192383, 1.9198533296585083, 1.9475408792495728, -0.7291008234024048, -0.5613411664962769, 2.1995487213134766, 3.231786012649536, -3.211169719696045, -0.942873477935791, 0.620275616645813, -0.38746869564056396, -0.9689934849739075, 1.8899024724960327, 1.8708441257476807, -0.46987566351890564, 1.5826122760772705, 0.45731276273727417, 4.906146049499512, 0.4595996141433716, 0.21680298447608948, -3.329936981201172, -0.9816965460777283, -1.0589020252227783, -0.9625532627105713, -0.40487009286880493, 1.52616286277771, -0.7246312499046326, 2.539558172225952, -0.04793565347790718, -0.9228702187538147, -1.7740743160247803, 0.17224818468093872, 4.489383697509766, -2.912109375, -0.047065239399671555, 1.0784293413162231, 1.2168662548065186, -3.418013572692871, 0.4060230553150177, 0.08064179867506027, 0.13238811492919922, -1.0164964199066162, 0.3725692629814148, 1.1701390743255615, 1.6221013069152832, 0.31174346804618835, -0.9987211227416992, -2.2213778495788574, 1.921394944190979, -0.8675982356071472, 0.3037797212600708, 0.9319932460784912, -0.09241916984319687, -4.043806552886963, 0.9877039194107056, -2.29498553276062, 0.49045804142951965, -0.28442826867103577, 0.16534367203712463, 2.062007188796997, 3.1770646572113037, 2.3065128326416016, -0.5014322400093079, 2.038621187210083, 0.04562501609325409, -0.4286910593509674, -7.2460503578186035, -0.8629550933837891, 0.21911638975143433, -1.8253544569015503, -0.40905511379241943, -0.8378881812095642, 0.246843159198761, -1.8707557916641235, 4.037720680236816, 1.2501461505889893, 1.0980764627456665, -2.738922119140625, -3.661155939102173, 0.5379385948181152, 0.11868894100189209, 1.864610195159912, 1.273048758506775, -1.78433358669281, 0.20809729397296906, -2.0758724212646484, -0.9517799615859985, 2.9881739616394043, 0.8297197222709656, -0.26783815026283264, -0.21317638456821442, 0.29786866903305054, -1.0538585186004639, -3.8417296409606934, -0.6273002028465271, 1.5595483779907227, 1.4978876113891602, -4.929211139678955, 3.0327978134155273, -2.1217234134674072, 0.2855862081050873, 0.7394822835922241, 0.3753030002117157, 3.266916036605835, -0.3959072530269623, 0.4720472991466522, 4.4340386390686035, -1.8276554346084595, 3.3701698780059814, 1.8911645412445068, -1.5606563091278076, -2.2072651386260986, -0.0030587692745029926, -2.2457008361816406, -5.986498832702637, 0.007723683025687933, -0.6024574041366577, -0.6193175315856934, 1.6958674192428589, -0.5688802003860474, 0.700591504573822, -0.6806482672691345, -1.025485634803772, -0.7052807807922363, 1.797179937362671, -0.5834460854530334, 0.2944430708885193, -0.8948226571083069, -0.8665581941604614, 0.3764752447605133, -0.20231854915618896, 1.5096312761306763, 1.9539759159088135, -1.0681016445159912, -2.6170551776885986, 1.423082947731018, -1.8499826192855835, -2.217939853668213, 0.8061774969100952, -2.7995219230651855, -2.4550912380218506, -4.385345935821533, -0.2731211185455322, 1.9217016696929932, -1.3747522830963135, -0.0442793034017086, 1.8369178771972656, -0.0897461473941803, 1.3204879760742188, 1.6385960578918457, -1.5277636051177979, -0.05735139176249504, 0.017823103815317154, 1.3677343130111694, -1.7413718700408936, -2.896261215209961, -2.502903699874878, 0.4707542955875397, -0.32759779691696167, -4.94721794128418, 0.03814626485109329, -5.923458099365234, 0.47247594594955444, 0.609917163848877, -0.49992942810058594, -2.048255443572998, 1.1391140222549438, -2.756227731704712, -1.1718000173568726, -0.15653423964977264, 1.1267362833023071, -0.24806375801563263, 1.34705650806427, 0.9339554309844971, 0.16159942746162415, -0.4630132019519806, -2.326385736465454, 0.18215960264205933, -1.6223289966583252, -1.0960909128189087, 1.163395643234253, -0.7949684858322144, 1.594590663909912, 0.7276967167854309, 2.2394888401031494, -0.49370208382606506, -0.7637678384780884, -0.8864689469337463, -0.5642795562744141, -3.1654109954833984, 0.18291054666042328, 0.7838619947433472, 0.0373295322060585, 1.0571894645690918, -2.677037477493286, -4.09260892868042, -1.4708342552185059, 2.2019097805023193, -0.24129372835159302, 3.2845499515533447, 0.49169984459877014, 2.397475242614746, 0.004636857658624649, 1.4778456687927246, 0.44030264019966125, 0.4045576751232147, 0.11278815567493439, -2.7406580448150635, -2.3050646781921387, 2.5303521156311035, 1.4171422719955444, 0.5917651653289795, -0.6919258832931519, -1.81468665599823, -1.0427285432815552, 1.647387146949768, 0.6039509773254395], "dtype": "float32"}, {"__nd__": [1.312563419342041, -1.084201693534851, 0.32072317600250244, 1.8853394985198975, -4.117598533630371, -1.3045735359191895, 0.4001896381378174, -0.6130596995353699, 0.8739017248153687, 0.5842900276184082, -0.8350245952606201, -0.19139468669891357, 0.7365076541900635, 0.5452362895011902, 2.278460741043091, 1.732322096824646, 2.0439352989196777, 1.1569148302078247, 0.9701208472251892, 2.089047431945801, -3.171135425567627, -0.3051636517047882, 2.531618118286133, -0.12310680747032166, 1.1988401412963867, 0.19912248849868774, -1.438068151473999, -2.078763008117676, -1.5101447105407715, 0.0822293683886528, 1.49788498878479, -2.3841822147369385, 2.2017428874969482, 1.2369521856307983, -1.6157723665237427, 0.2811623513698578, 1.2283536195755005, -0.31175360083580017, 3.5559537410736084, 3.485381603240967, 0.053499042987823486, -0.09882670640945435, 0.09281079471111298, -1.6162312030792236, -2.259328842163086, 2.2450692653656006, 1.6934889554977417, -1.3234567642211914, 0.34555482864379883, -2.1720285415649414, 0.6873528361320496, -0.9187719225883484, -2.460728168487549, 4.131229877471924, -3.950640916824341, -2.595369577407837, -1.2119345664978027, -2.270657777786255, -0.9839529991149902, -0.03536859154701233, -0.3470889627933502, 1.0969935655593872, 0.017647936940193176, -0.11519911885261536, 1.5122705698013306, 1.0223546028137207, 1.7846996784210205, -1.935185194015503, 2.7271461486816406, -3.5449230670928955, -3.5287513732910156, 1.0145397186279297, 0.5086613893508911, 1.987441897392273, 2.4012691974639893, -2.1755166053771973, -1.3296186923980713, 1.7995595932006836, 0.967495858669281, 1.6368882656097412, -1.9362627267837524, -0.4789712727069855, 1.1546087265014648, 0.4002048969268799, -1.2084323167800903, -1.1792514324188232, 2.2552196979522705, 0.26045575737953186, 0.5142053961753845, -0.7392036318778992, 3.1361002922058105, -0.05601947754621506, -3.318310260772705, -2.5953176021575928, 0.37994563579559326, 3.0628936290740967, 0.9202179908752441, -1.1208829879760742, 0.6627764105796814, -1.5784296989440918, 1.7173131704330444, -1.5414692163467407, 0.149394690990448, -0.8623925447463989, -0.8914918899536133, -1.2833003997802734, -0.4610075354576111, 0.631481409072876, -1.955211877822876, 0.5933841466903687, 0.5153697729110718, -2.0007753372192383, -1.4969918727874756, -0.3454429507255554, 0.6679034233093262, 0.3168012797832489, 1.563683032989502, -0.7693322896957397, 0.1600136160850525, 0.9653298854827881, 2.187154531478882, 2.0165090560913086, -0.317166268825531, -1.3977468013763428, -0.39997977018356323, 1.7173309326171875, -1.9996916055679321, -0.9312297105789185, 0.5573962926864624, 0.5153696537017822, 0.23073852062225342, 2.9366536140441895, 1.4323254823684692, 1.6068159341812134, 5.298947334289551, 2.451608657836914, 1.880589246749878, -0.354166716337204, -2.478630542755127, 0.10275816917419434, 2.5867726802825928, -0.23542246222496033, 1.3065791130065918, -2.5828864574432373, -0.10650461912155151, 0.8069488406181335, 1.3134695291519165, -0.42456328868865967, -3.1606810092926025, 1.4882566928863525, -0.11704617738723755, 0.9471578598022461, -1.577847957611084, 1.1314632892608643, 0.26742029190063477, -0.2322673499584198, -3.019014596939087, 1.395348310470581, -1.1400924921035767, 2.1877028942108154, -1.308639407157898, 1.4485093355178833, -1.7026312351226807, -0.9976222515106201, -0.45351144671440125, -1.3415191173553467, -0.33226466178894043, 2.0906593799591064, -1.0062400102615356, 0.047284986823797226, -0.5404093265533447, 1.1401686668395996, 0.7127261161804199, 0.5092580914497375, -2.204629898071289, -0.9163247346878052, 0.06286081671714783, -3.786109447479248, 2.0994760990142822, 1.6760140657424927, -3.341747283935547, -0.9370080828666687, 1.7161914110183716, -4.057436943054199, -0.14740228652954102, 2.123939037322998, -1.8526376485824585, 2.1504244804382324, 1.2564303874969482, -1.1738814115524292, -1.796795129776001, -0.5908759236335754, -2.533428430557251, 1.365305781364441, 2.2890684604644775, -3.0969080924987793, 0.24275419116020203, -1.5001766681671143, -0.04862534999847412, 1.6047612428665161, -1.7858365774154663, -0.6398835778236389, 0.015798866748809814, 0.6600285768508911, -0.5861994028091431, 1.6424822807312012, -2.7589094638824463, -2.619027614593506, -2.813650608062744, -2.7022652626037598, -0.22785162925720215, -2.0648491382598877, 0.24642270803451538, -1.6426889896392822, 1.8182896375656128, 0.9896776676177979, -1.428597092628479, -0.058153390884399414, -1.1056100130081177, 2.7787508964538574, -1.2411167621612549, -3.3810501098632812, -2.043262243270874, 0.2340259552001953, -1.9435739517211914, 1.9622410535812378, 0.860983669757843, -1.2024433612823486, 0.5461429357528687, -0.647753119468689, -0.786544680595398, -0.2698487341403961, -0.2048618346452713, 2.3974175453186035, 0.13379403948783875, 1.1820780038833618, 0.00015994906425476074, -0.6760284304618835, 0.09082253277301788, 3.4462265968322754, -1.8051429986953735, -0.01644998788833618, -2.036620616912842, 1.102611780166626, 1.4676861763000488, -2.243380069732666, -0.12587589025497437, 0.9434206485748291, -1.9743781089782715, 4.505364418029785, 3.974370241165161, -1.2747583389282227, 2.3042798042297363, -0.5085557699203491, -0.49152225255966187, -3.721099615097046, 2.269747257232666, 0.9178006649017334, -2.8328094482421875, 0.6197696924209595, -2.324172258377075, 2.566879987716675, -0.32449984550476074, -0.06878645718097687, -1.9487308263778687, 2.163235664367676, 3.0637288093566895, -0.3237851858139038, -0.32133519649505615, -2.097013473510742, -0.5211441516876221, -0.6539907455444336, -0.6003433465957642, 0.8452366590499878, -1.2472983598709106, -1.7661911249160767, 2.384810447692871, -0.8914204239845276, 0.8531727194786072, -3.933638095855713, 0.27006611227989197, 0.06729532778263092, -0.48359233140945435, -1.5726531744003296, 0.006768524646759033, 3.5020337104797363, -1.0619399547576904, -1.8155643939971924, 1.4819719791412354, -2.6968486309051514, -1.886702299118042, -0.13736456632614136, 0.8377047777175903, 2.3576505184173584, -0.8127741813659668, -0.24616652727127075, -1.7914319038391113, 1.919255018234253, 4.326635360717773, 1.5584447383880615, 0.04378737136721611, -0.9681518077850342, -0.2448016256093979, 1.8139421939849854, 0.3570884168148041, 1.0171128511428833, 1.916098952293396, 0.3911556303501129, -1.6670054197311401, -1.6659566164016724, -0.6726841926574707, -1.8713412284851074, 3.235753059387207, 2.046668291091919, 0.2342432141304016, -1.7019020318984985, -0.7205392718315125, 1.57741117477417, 1.2121261358261108, -0.020749837160110474, 3.260791540145874, -2.024167060852051, 1.7857623100280762, -0.3921298384666443, -2.089545249938965, -0.6652098894119263, -3.7166192531585693, 3.488919973373413, 3.0994226932525635, 0.46039098501205444, 0.30437856912612915, 0.36951780319213867, -0.36618202924728394, -1.2969337701797485, 1.1522666215896606, -1.015824794769287, 1.7052924633026123, -0.1670408844947815, -1.3425155878067017, 1.3280720710754395, 0.579998791217804, -1.5234216451644897, -1.2114484310150146, -0.38757652044296265, 0.9311162233352661, -1.2372971773147583, 0.47150614857673645, 1.1680458784103394, -2.821528911590576, -0.6883432865142822, -0.6644155979156494, -1.2076892852783203, 0.27283015847206116, 2.8328540325164795, 1.7091774940490723, -1.1067962646484375, -0.8753054141998291, -2.603386402130127, 2.344805955886841, -0.16299231350421906, 0.9824361801147461, -0.27180230617523193, -2.4347994327545166, -1.6924039125442505, -0.9096416234970093, -0.5600720047950745, -3.1397039890289307, -1.5982437133789062, 2.501493215560913, 3.037468194961548, 0.997175395488739, 0.4134492576122284, -4.841066837310791, -1.40193772315979, 0.9785019159317017, 1.0886434316635132, -1.0050809383392334, 1.1793755292892456, -1.433255910873413, 1.9786561727523804, 1.7041789293289185, 0.42659711837768555, -1.4699889421463013, 1.0781314373016357, -0.012355633080005646, -4.779174327850342, -1.5830585956573486, 2.165313720703125, -0.8204250335693359, 0.4997562766075134, -0.010730564594268799, 0.6935312747955322, 2.2763168811798096, -0.29949063062667847, 0.14142197370529175, -0.7165533304214478, -0.31869056820869446, 0.3141385614871979, -0.8215152621269226, 0.42194584012031555, -0.4369733929634094, 1.2344989776611328, -1.911060094833374, -1.3648015260696411, -1.258233904838562, 1.0017534494400024, 0.7863069772720337, -0.4601646661758423, -0.497624933719635, -2.3501744270324707, 0.6076540350914001, -0.8958113789558411, 2.489600896835327, 0.6063812971115112, 1.219282865524292, -2.0403900146484375, -0.14958521723747253, -1.9280753135681152, -0.5316755175590515, 1.1965744495391846, -1.3860414028167725, 0.11993080377578735, 0.4903396964073181, 0.15087133646011353, -1.2833284139633179, -0.9989307522773743, 1.4558161497116089, -1.6687026023864746, -1.3404678106307983, -0.4630131721496582, 1.6568084955215454, -0.7114260196685791, 0.5602486729621887, -1.0738128423690796, -0.25704824924468994, -0.9333105683326721, -2.902536392211914, -0.29467546939849854, -1.5840061902999878, -0.4583800435066223, 2.174095869064331, -2.063053846359253, -4.197503566741943, -1.931668758392334, 0.810272753238678, -1.8104243278503418, 0.9178450107574463, 2.948637008666992, -3.5796725749969482, -0.8308550119400024, 2.4096100330352783, 1.6129052639007568, 0.674525260925293, -0.7193797826766968, -2.2040889263153076, -0.8694698810577393, -1.3292101621627808, 0.8201279640197754, 0.8374226689338684, 3.756289005279541, 0.6626940369606018, -2.7136807441711426, 0.905748724937439, -0.9740043878555298, 0.635627269744873, 0.4083690047264099, -2.092730760574341, -0.5881465673446655, 0.6796038150787354, 1.4695820808410645, 0.04253193736076355, -1.1878156661987305, -0.7714149951934814, 0.2056109756231308, 1.195683240890503, 3.4078330993652344, 4.551555633544922, -2.5836660861968994, 1.384897232055664, -0.9017046093940735, 3.5534818172454834, -1.0071607828140259, 2.0591883659362793, 3.237462282180786, 0.2555818557739258, 3.2258777618408203, 0.9278295636177063, -0.37794029712677, 0.8179615139961243, -0.014902830123901367, -2.107333183288574, 1.2588560581207275, -2.2901761531829834, -1.882993459701538, 0.06846244633197784, 4.58743143081665, 0.9767809510231018, -0.17837077379226685, -1.6262215375900269, -1.9371410608291626, 2.7917919158935547, 2.5169527530670166, 1.8227671384811401, -0.5123379230499268, 2.4301202297210693, -0.9738385677337646, -0.05344906821846962, 0.5352578163146973, -0.7668588161468506, -0.8238958120346069, 2.0541415214538574, 0.05471760034561157, -0.3786073923110962, 2.497347354888916, 3.6891674995422363, 3.112335205078125, 1.9278472661972046, -1.1436388492584229, 2.1714932918548584, -1.341887354850769, -0.4970020353794098, 1.100387454032898, 0.5453312397003174, 3.526374101638794, -1.663038969039917, 0.7349763512611389, -0.021202832460403442, 3.338918924331665, 1.6767810583114624, -3.025926113128662, -2.011054277420044, -0.1608654409646988, 0.9734716415405273, -1.4642248153686523, 2.822939395904541, 1.10800302028656, -1.9233406782150269, -1.5363534688949585, 0.9759246110916138, 1.1733077764511108, -2.4053566455841064, -0.5370439291000366, 2.130368709564209, 1.1828804016113281, 0.10210692137479782, -2.36995267868042, 1.902657151222229, 2.0614101886749268, -0.7310303449630737, 1.916621446609497, -2.2474143505096436, -0.6534721255302429, -2.657308578491211, -1.251696228981018, -0.6510264873504639, -0.6329590678215027, -1.9229521751403809, -1.947930097579956, -1.6364325284957886, 1.0724129676818848, -0.3063358664512634, 0.7061299085617065, -1.4503850936889648, 0.23006555438041687, -0.3493556082248688, 1.0340688228607178, -3.10709810256958, 0.027642518281936646, -1.5691306591033936, 1.8430019617080688, -1.907224416732788, -1.7132983207702637, -2.580355644226074, 1.133040428161621, -0.6739863157272339, -1.8380100727081299, -0.6358363032341003, -0.8710544109344482, 1.6293699741363525, -1.213149905204773, -0.5678108334541321, 0.8828874826431274, 0.04246436059474945, -0.1191210001707077, 2.5625834465026855, 0.7531968355178833, -1.8346936702728271, -0.2843506336212158, 1.3809237480163574, 0.19269752502441406, -2.2174317836761475, 0.3587001860141754, 0.4205220937728882, 0.7615374326705933, 4.964097499847412, 0.10858465731143951, -1.9692339897155762, -2.504288673400879, 1.7803153991699219, 1.3496311902999878, 0.16072329878807068, -2.5919547080993652, -3.5984318256378174, 0.23408707976341248, 0.7493095993995667, -0.5669167637825012, 2.5969295501708984, 2.5904765129089355, -0.5460370182991028, 1.3134105205535889, 2.8147003650665283, -1.3174068927764893, 0.02275620400905609, -1.0901654958724976, -0.05168303847312927, 0.9046399593353271, 1.3167051076889038, -0.7741273045539856, 0.439314603805542, 0.35568106174468994, -4.02409029006958, 0.6223015189170837, 0.8453203439712524, -0.7629224061965942, -1.562240481376648, -1.3593217134475708, 3.6740081310272217, 2.0504000186920166, -0.5581343770027161, -1.9749135971069336, -0.3677455186843872, 0.4846266210079193, 3.2367305755615234, 1.6473603248596191, -3.0604288578033447, 3.952679395675659, 0.041365087032318115, -5.739563941955566, 0.9021409749984741, 1.8765041828155518, 1.7888298034667969, -0.05821780487895012, -2.114982843399048, -0.11449342966079712, 0.7013225555419922, -1.107330560684204, -2.923943519592285, -2.3356375694274902, 0.9781747460365295, 1.042988657951355, -0.5593527555465698, 2.3431801795959473, -1.6407113075256348, -0.4181777834892273, 2.590263605117798, -3.23413348197937, -0.19347241520881653, -1.0095200538635254, -2.25663685798645, 0.9078604578971863, 1.118285894393921, 0.9875988960266113, 3.6385140419006348, -2.2427616119384766, -2.911696195602417, -0.1659645140171051, 0.7731755971908569, -0.0556199848651886, -0.3457902669906616, -1.989131212234497, -2.2285091876983643, 0.591067373752594, -0.5422019362449646, 1.3425483703613281, 3.2217061519622803, 0.060958608984947205, -0.8728156685829163, 1.0875730514526367, 2.2682721614837646, -1.7019954919815063, 3.5155956745147705, 1.1814451217651367, 0.38291269540786743, 0.5192421078681946, -0.6820712089538574, 1.3584699630737305, 0.6886869072914124, 1.0156580209732056, 1.7252017259597778, 0.04611101746559143, 1.1998028755187988, 0.22545993328094482, -1.7999399900436401, 1.8309626579284668, 2.9369726181030273, 0.07021692395210266, -0.2329428791999817, -0.6847794651985168, 0.17590969800949097, 0.23735398054122925, -0.5500491857528687, -0.8588200807571411, -0.6314941048622131, -1.9986159801483154, -2.659931182861328, -1.0173041820526123, -3.795919179916382, 2.662095785140991, -0.6959227323532104, -1.7965939044952393, -1.2324252128601074, -0.09304505586624146, 0.9246020317077637, 0.4977583885192871, -1.160133957862854, -0.9650545120239258, -1.7182178497314453, -0.2414635717868805, 0.7935158610343933, -0.584789514541626, 0.7047317028045654, 2.892991065979004, 0.673635721206665, -1.1584258079528809, -0.5139919519424438, -0.4971383213996887, 0.11372115463018417, -2.790992498397827, 0.05307170748710632, -1.8458170890808105, -1.582184076309204, -2.2555899620056152, 3.8184149265289307, 0.4628390371799469, 0.0072813332080841064, 1.9879614114761353, 0.13311630487442017, 1.5126864910125732, -0.3646015524864197, -2.516937255859375, 0.8453130125999451, 1.8790764808654785, -2.473268985748291, -2.0480921268463135, 1.9231972694396973, -1.5089718103408813, -0.4688812494277954, 1.6173871755599976, 0.8835880160331726, -0.045887988060712814, 2.356696605682373, 1.839048147201538, 0.334380567073822, -0.2677551209926605, -0.422242134809494, 1.0874357223510742, -0.4291902780532837, -2.512620687484741, -1.7915047407150269, 3.337458610534668, 1.3535000085830688, 2.646961212158203, 1.7844280004501343, 2.0333855152130127, -0.5251367092132568, 4.001716136932373, 3.429619789123535, -2.4792160987854004, -1.6123462915420532, -1.5691750049591064, -2.6267192363739014, -0.7864800691604614, -0.8490285873413086, -0.9006307721138, -0.3268498480319977, 0.554173469543457, -2.5323731899261475, -0.38718274235725403, 2.7039175033569336, 1.4858574867248535, 0.4582926630973816, 2.846379280090332, 1.7954782247543335, -1.4579390287399292, 0.5939196348190308, 0.6368440985679626, -1.8832778930664062, 1.1771175861358643, 0.03511987626552582, 3.0487310886383057, 0.29343223571777344, -2.6756324768066406, -2.3009283542633057, 1.820801019668579, 0.30766725540161133, 1.5904152393341064, 2.7549073696136475, -0.29204103350639343, -1.6766676902770996, 1.5916410684585571, 4.981322765350342, -3.579845666885376, 1.1845828294754028, 0.13377058506011963, 0.36778271198272705, -1.1437772512435913, -1.9128336906433105, -2.408535957336426, -2.2131528854370117, -2.4753048419952393, -0.5635312795639038, 0.462390273809433, -2.215711832046509, 0.7861295342445374, -1.2843049764633179, 1.0723025798797607, 0.4279739260673523, -2.506648540496826, 2.519909381866455, 2.8827943801879883, -2.535668134689331, 2.067858934402466, 1.0180494785308838, -2.650440216064453, 0.17786487936973572, 0.7476456761360168, 0.2718721032142639, 3.7023215293884277, -1.6819995641708374, -0.668178915977478, -0.6173352003097534, 2.271752119064331, 1.0297093391418457, 1.9177285432815552, 0.06940668821334839, -0.5587916970252991, 1.2691125869750977, 1.8536323308944702, 1.0758557319641113, 2.6134324073791504, -2.289625644683838, -2.924259662628174, 1.5475932359695435, 0.2457180619239807, 0.24520975351333618, -0.47049254179000854, -0.9837567806243896, -0.10960021615028381, 0.8059897422790527, 4.154623508453369, -0.8658289909362793, 1.1541059017181396, -3.2795169353485107, -1.497023582458496, -0.66734778881073, 0.5302979350090027, 2.3318705558776855, -2.9822192192077637, 0.8088462352752686, 1.5176550149917603, -0.6755735874176025, -0.0014622211456298828, -0.47204524278640747, -0.910863995552063, -2.4964964389801025, 0.18970221281051636, 1.3395389318466187, 0.25623321533203125, 0.8235681056976318, 2.2757387161254883, 0.9049177169799805, -2.0753142833709717, 1.2437474727630615, -1.1624369621276855, -0.8530759811401367, 1.7594484090805054, -3.4609575271606445, 0.0818035751581192, -1.6963564157485962, -0.4872271418571472, 1.9167096614837646, 1.8796281814575195, -0.014311730861663818, -2.127295970916748, -1.7916624546051025, 1.4918715953826904, 2.188183069229126, 1.024440050125122, -1.4449772834777832, -0.7526984214782715, 2.080594778060913, -1.1439412832260132, 0.8690020442008972, 0.8355960845947266, -0.4463502764701843, -1.948235273361206, 1.034157633781433, -0.5965994596481323, -2.1423449516296387, -0.6864373683929443, -2.1962809562683105, 1.4411321878433228, -1.2090367078781128, -0.19176915287971497, -1.4014400243759155, 3.4154551029205322, -1.3010519742965698, -1.3247039318084717, 0.9048933386802673, 0.07472138106822968, -3.3391242027282715, -2.107842206954956, 0.2736469805240631, -0.8284984827041626, 0.6115306615829468, 0.5492673516273499, 0.20248597860336304, 2.0918309688568115, -3.0306341648101807, 3.2452592849731445, -2.3761894702911377, 0.7954789400100708, 3.4643006324768066, 0.9307292699813843, 1.5823925733566284, 0.13981279730796814, 0.010474234819412231, 0.013276096433401108, -2.0080275535583496, -0.8125659823417664, -2.7086596488952637, -2.7638494968414307, 2.244579553604126, 0.3102617859840393, 1.9453626871109009, -0.1260996162891388, -1.7488654851913452, 0.9743399024009705, -0.6130077838897705, -2.955658197402954, -1.854873776435852, -2.3299293518066406, -0.7545288801193237, 1.1825363636016846, -2.462984561920166, -1.665100336074829, -2.2657172679901123, -1.969369888305664, 1.6552406549453735, 1.0921298265457153, -0.47040364146232605, -0.42426490783691406, 0.47864866256713867, 2.199817419052124, 2.2409191131591797, 0.46464255452156067, 2.487905502319336, 0.02109678089618683, -0.28027820587158203, 3.5490593910217285, 0.8423927426338196, 1.7701144218444824, -1.7098520994186401, 1.6763551235198975, 0.9062648415565491, 1.6054588556289673, -1.7883918285369873, -3.1262242794036865, -1.2058731317520142, -0.43290138244628906, 0.05557513236999512, 0.9677453637123108, -2.38185977935791, -0.17664121091365814, -1.2569414377212524, -1.9560860395431519, 0.3663840889930725, -1.0287021398544312, -1.0480409860610962, -2.6182796955108643, -0.10468810796737671, 1.0325193405151367, -0.9696270823478699, -0.2960180640220642, 0.5993972420692444, -0.25562065839767456, -0.3419824540615082, -0.1389695107936859, 2.7455201148986816, -0.6693632006645203, -3.36968994140625, 1.7063332796096802, -0.5397602915763855, 0.21693509817123413, -1.0351808071136475, -3.9225034713745117, -0.3366605043411255, 2.639378547668457, 1.1940104961395264, 0.4008867144584656], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.053115725517273, 1.7764912843704224, -2.5532917976379395, -0.13796506822109222, 1.0137194395065308, 1.3521418571472168, 0.6537883877754211, 1.4971178770065308, 0.28995758295059204, 0.551267147064209, 0.17873768508434296, -1.0738587379455566, -0.8466289639472961, 0.37958425283432007, -0.5801951885223389, 1.2715513706207275, 1.2923866510391235, 1.7987862825393677, -0.026073837652802467, 1.3837097883224487, -0.9058431386947632, -0.8163147568702698, 0.08130305260419846, 0.28143084049224854, -1.5989882946014404, -1.7307242155075073, 0.35507839918136597, -0.8621881008148193, 1.2083570957183838, 0.3945462107658386, 0.3071846067905426, 0.20844262838363647, 0.8954750895500183, -0.1530461460351944, 1.085631012916565, -0.5259143114089966, -0.22499771416187286, -0.6805052161216736, 0.5976618528366089, 0.03588424623012543, -0.8622670769691467, 2.001879930496216, -0.3637140393257141, -0.11629977077245712, 0.3941647708415985, 1.7476508617401123, -0.14838363230228424, 0.7031832337379456, 0.0987953394651413, -1.4637936353683472, 0.7919772863388062, 1.0206393003463745, -0.23998627066612244, -0.01543633732944727, -1.5013161897659302, -1.5229519605636597, -0.3915199339389801, 0.6259333491325378, -0.46957460045814514, 0.2920228838920593, 0.9677673578262329, 0.09576239436864853, 0.5744683742523193, 1.4478734731674194, -1.2460280656814575, 0.7440750598907471, -1.8846845626831055, 0.4316425919532776, 0.20299464464187622, -0.5433347821235657, -1.0773130655288696, -0.06183105707168579, -0.410238653421402, 0.09762159734964371, -0.9078017473220825, 0.5455273389816284, -1.0358502864837646, -0.3042352795600891, -0.391988068819046, 1.2678874731063843, -1.4956938028335571, -0.41402381658554077, 1.1824606657028198, -0.6583412289619446, -1.2394710779190063, 0.2828414738178253, -0.29003041982650757, 0.8347549438476562, -0.9203901290893555, 0.19091922044754028, 0.6661273241043091, 1.0357218980789185, -1.1061452627182007, 1.4237887859344482, -0.9006966352462769, -1.5125505924224854, -0.9545692205429077, 0.15484929084777832, -0.321784108877182, 0.4404556155204773, 0.4956032633781433, 0.3181445002555847, -0.25452789664268494, 1.8140779733657837, -2.216033458709717, -1.734349012374878, -0.5068859457969666, -1.2552173137664795, -2.5559253692626953, -0.7950685024261475, -0.7492610812187195, -2.448549270629883, 0.8586929440498352, 1.29911208152771, 1.2343010902404785, -0.5083407163619995, 0.14761216938495636, -1.1208102703094482, 0.20643730461597443, -0.6741753816604614, 0.4098832309246063, -0.3429722785949707, 1.0337574481964111, 0.8266077637672424, 1.23894464969635, -1.6052957773208618, -1.8196072578430176, 0.4047423303127289, -0.5771601796150208], [0.3978289067745209, 0.29888084530830383, 0.18021494150161743, 0.9192535281181335, 0.556616485118866, 0.4724460244178772, -0.8938198685646057, 1.7246652841567993, 0.12661360204219818, 1.1070493459701538, 0.32706713676452637, 1.2693654298782349, 0.15353532135486603, 0.2176106721162796, 0.13781337440013885, -0.7086793184280396, -0.5720164775848389, -0.14694882929325104, -0.11698143184185028, -0.9826943278312683, 2.356566905975342, -0.9813703894615173, -0.30599862337112427, 2.4452872276306152, -0.6596879363059998, 0.7659382820129395, 0.6608838438987732, -0.2080206423997879, -1.4289289712905884, -1.553024411201477, -0.00791721697896719, 0.4208160936832428, -2.1561195850372314, 0.15284812450408936, -0.654123067855835, 0.8952039480209351, 0.17636170983314514, -0.8014308214187622, -0.42353519797325134, 0.048485130071640015, 1.3498164415359497, 3.0509796142578125, 0.36641550064086914, 0.1300293207168579, 0.7127132415771484, 0.045923635363578796, 0.7104279398918152, 0.4690491259098053, 0.08745628595352173, -0.7064696550369263, 0.24971434473991394, 0.994346022605896, 0.16790692508220673, 1.608073353767395, 1.0373547077178955, -0.03474288433790207, 1.8357981443405151, -1.0261919498443604, 1.2709004878997803, 1.9574341773986816, -0.15091410279273987, 0.0018051289953291416, 1.0439820289611816, 1.5392496585845947, 0.5528356432914734, 0.3550683259963989, 1.033794641494751, 0.3781958520412445, -1.0895569324493408, -0.0690082460641861, 0.1042112335562706, -1.3806716203689575, 0.518618643283844, -0.6232043504714966, 0.861382782459259, -1.3710020780563354, -0.7937610149383545, -1.3396885395050049, 1.4022269248962402, 1.3917750120162964, 1.5862153768539429, 1.5932339429855347, 1.0219627618789673, 1.0119965076446533, -1.5809804201126099, 0.4466247856616974, -0.3821673095226288, -0.20327728986740112, -1.1104215383529663, 1.431710958480835, 0.27714720368385315, -1.3678317070007324, 0.6259223818778992, 0.03098929673433304, -0.9454757571220398, -0.21463562548160553, -0.700574517250061, -2.2325522899627686, -1.4735822677612305, 0.5461509823799133, 0.3729606568813324, 0.2895790934562683, -1.166759967803955, -0.5945008993148804, 0.06568827480077744, 0.6906121373176575, -0.8430119752883911, -1.059950590133667, 1.1398041248321533, 0.06655634194612503, -0.3809283673763275, -0.7401725649833679, -0.6576257348060608, 1.5846054553985596, 0.9693785905838013, -0.4177705943584442, -1.2538760900497437, 1.6734874248504639, 1.5702478885650635, 1.2041453123092651, 0.8415461182594299, -1.887092113494873, -1.2911195755004883, 0.5649392604827881, -0.28491517901420593, 0.2554653584957123, -0.5254474878311157, -0.6570103168487549, 0.5533406734466553], [-0.4006863832473755, -0.002750141080468893, 0.25537389516830444, 0.2316483110189438, 1.158631682395935, 1.36827552318573, -1.3684428930282593, -0.2694779336452484, -0.6488304138183594, 0.7532873153686523, -1.0702744722366333, 0.7732221484184265, 0.6983717679977417, 0.5756646990776062, -0.9760543704032898, -0.21768690645694733, -1.0833427906036377, 0.39390647411346436, 0.9520752429962158, -1.5239372253417969, -0.8299877047538757, -0.16650493443012238, -0.5342420935630798, 1.4900349378585815, -0.14836764335632324, 0.2496156245470047, -1.1146836280822754, 1.5401816368103027, 0.16285106539726257, 0.7711102366447449, 0.08412633091211319, -2.1234958171844482, -1.600638747215271, -1.2906688451766968, 0.4687395989894867, -0.5121772885322571, 0.42633765935897827, 0.284776896238327, 0.2474629282951355, 0.9171476364135742, 0.18053820729255676, -1.6436583995819092, -0.6808697581291199, 1.6592670679092407, -1.0736650228500366, 1.3619978427886963, 1.652419090270996, -0.595105767250061, 0.34424883127212524, -1.4023863077163696, 0.7722803354263306, -0.7778273224830627, -1.3104488849639893, -0.39303863048553467, -0.4573878347873688, -0.5853225588798523, 0.19626294076442719, -0.16898725926876068, -0.6634804010391235, 0.9487982392311096, 2.421513795852661, 0.06504317373037338, -0.704403281211853, -0.992775022983551, 0.5899581909179688, -0.25655415654182434, 0.015056869015097618, -1.9947081804275513, -1.1320641040802002, -1.162248134613037, 0.943668007850647, -0.2703447639942169, -2.4855313301086426, 0.4818969666957855, 1.1324982643127441, 1.3041704893112183, -2.8018319606781006, 0.15273961424827576, 0.8111956715583801, 0.6056417226791382, -0.23600471019744873, -1.8013386726379395, -0.4336090087890625, -0.32346421480178833, 0.5738714337348938, 0.6397247910499573, -0.24265602231025696, 0.8471004366874695, -1.05661940574646, -0.6826376914978027, -0.031049536541104317, 0.573591411113739, -0.32215219736099243, 1.2306468486785889, 0.41090700030326843, -0.9491457939147949, 0.08558788895606995, -0.8875037431716919, -0.9625638723373413, -1.42739737033844, 1.397571086883545, 0.9094139933586121, -1.600577473640442, 1.1129155158996582, 0.9609569907188416, -0.3633536696434021, 0.5071889162063599, 2.244964361190796, 0.2862309515476227, 0.4218981862068176, 1.1223537921905518, -1.0346606969833374, -1.2659227848052979, -1.4094098806381226, -0.07717163860797882, 0.8727633953094482, -0.8587529063224792, -1.3691868782043457, 0.21724383533000946, -0.34941476583480835, 0.6297116875648499, -0.09816327691078186, -0.1489013135433197, -0.0463363453745842, 0.07550323754549026, -0.9222617149353027, -1.0816967487335205, 1.631859540939331, 3.1339569091796875], [1.1654694080352783, -1.5833669900894165, 0.07792814075946808, 1.2387341260910034, -1.5705281496047974, -0.43609005212783813, -0.35702264308929443, -0.8635562658309937, -0.7844062447547913, 1.0680334568023682, -0.39739489555358887, -0.17256353795528412, 1.7736471891403198, -0.7939521074295044, -1.8595759868621826, 0.7050238251686096, 2.070965528488159, 0.2678645849227905, 0.36346182227134705, -1.2320520877838135, -0.8820333480834961, 1.5223171710968018, 1.316070795059204, 0.6436273455619812, -0.5896125435829163, -1.1411457061767578, -0.9983368515968323, -1.128322720527649, 0.655798614025116, -0.7499189972877502, -0.4616090953350067, 0.7248115539550781, 0.7008728981018066, 0.07221673429012299, 0.4980469346046448, 0.020505748689174652, -1.9255032539367676, -0.06928952038288116, -0.4177575409412384, -0.351639986038208, 0.245271697640419, -0.5644857287406921, 1.2273246049880981, -0.16950161755084991, 0.1682470142841339, 0.3320276737213135, -0.1713148057460785, 0.6501687169075012, 0.7594847679138184, 0.6337487101554871, -0.7116878628730774, -0.9120219349861145, -0.6162862777709961, -0.006799676455557346, -1.8901993036270142, -0.8647224307060242, -1.1730259656906128, -0.6971823573112488, -0.9477648735046387, 0.7924922108650208, 0.42436692118644714, 0.9355729818344116, -1.0139329433441162, -0.8045815229415894, -0.3223399221897125, 1.6381497383117676, 1.6612129211425781, -0.6388212442398071, 0.6133551597595215, 2.135479688644409, -1.0440572500228882, -0.20296190679073334, 1.2899534702301025, 0.7145993113517761, -1.0057107210159302, 1.513991355895996, 0.9775364398956299, 0.6870700716972351, -1.249013900756836, -0.17100317776203156, 1.3151508569717407, 0.4874369502067566, 0.7959398031234741, -0.2713276743888855, 1.5103710889816284, -0.24463891983032227, 0.9212328791618347, 0.3213050067424774, -0.9683389663696289, 1.484122633934021, -1.2643744945526123, -0.7401135563850403, 0.13887980580329895, -1.4102716445922852, -1.8120152950286865, -0.12345033884048462, -0.4340866208076477, -0.28277426958084106, 0.49431800842285156, 2.253410577774048, -0.42737308144569397, 2.4396355152130127, 0.2298867404460907, -0.10959560424089432, -2.719900131225586, -1.5501340627670288, 0.24153867363929749, 0.5344241857528687, -0.8595501780509949, 0.2748866081237793, 0.8693237900733948, -0.5975802540779114, -1.6571013927459717, -1.018128514289856, -0.3396628201007843, 0.2474014014005661, 0.7384887337684631, -0.9669080972671509, -0.004420398268848658, -0.2993156611919403, -0.0048146857880055904, 1.0628777742385864, 1.2589597702026367, -0.5159816741943359, -0.647063136100769, -0.7299153208732605, -0.5654348731040955, -0.4520675539970398, -0.2069132775068283], [-0.5414893627166748, -1.1098568439483643, -1.2063615322113037, 0.02174418978393078, 2.4084525108337402, 0.5240306258201599, 0.6159293055534363, 0.3172799050807953, -0.19416490197181702, 1.970199704170227, -0.34684139490127563, -1.0080866813659668, -1.2690603733062744, 0.1529688686132431, -1.0308995246887207, -0.9069036245346069, -0.9033064246177673, -1.3497816324234009, 0.13464882969856262, -1.713575005531311, 2.2509138584136963, -2.7968289852142334, -1.4270254373550415, 0.03275440260767937, 0.38978269696235657, -0.36991211771965027, 0.26125213503837585, 0.15438444912433624, 0.10897232592105865, 0.5711301565170288, 0.23191837966442108, -0.10869165509939194, -0.14777351915836334, 0.8798985481262207, -0.18724963068962097, 0.5695562362670898, 1.582106113433838, 0.3888968229293823, 1.2129242420196533, -0.4254191219806671, 0.2800106704235077, 0.518549382686615, 0.23966901004314423, 0.48911815881729126, 1.4156498908996582, -0.307263046503067, -0.662899374961853, 0.3342130780220032, -1.4527482986450195, -0.010373621247708797, 1.1942331790924072, -0.010675483383238316, 0.5472183227539062, 0.3842201828956604, 1.2026811838150024, 0.5062756538391113, 0.20311234891414642, -3.330916404724121, -0.5255213975906372, 0.70558100938797, -0.151149719953537, -0.7597670555114746, 0.8911117911338806, -0.9005152583122253, -0.36247649788856506, 0.6591323614120483, -0.543191134929657, 0.1413487195968628, -0.05186690017580986, -1.2371439933776855, -0.5678940415382385, -0.4102286398410797, -0.6957278251647949, 1.0421006679534912, -0.07706622034311295, -0.6964074373245239, -1.4492733478546143, -0.3845376670360565, 1.2465777397155762, -0.3709334135055542, 0.6879695057868958, 0.3841325044631958, -0.5873270630836487, -1.0206414461135864, 0.4697866141796112, -1.7524758577346802, 0.5019621849060059, -1.254660725593567, 0.7902444005012512, -0.45551109313964844, -0.14976520836353302, 0.9909161329269409, -1.0586220026016235, -0.005987464915961027, -0.37237364053726196, 0.4554051160812378, 0.6663734316825867, 0.2359646111726761, 0.15682300925254822, -0.45332637429237366, 0.5567842125892639, -0.2823081612586975, 0.08340262621641159, 0.13975845277309418, -1.4024888277053833, -1.1724172830581665, 0.17775461077690125, -0.05963585898280144, 1.2111958265304565, -2.466388463973999, 0.17145569622516632, 1.049913763999939, 0.05528709664940834, 0.7440495491027832, -0.01333412155508995, -0.08906084299087524, 0.8680394887924194, -1.3216484785079956, -2.2654714584350586, 0.4440525770187378, 0.09128867834806442, 0.08276887983083725, 0.255220890045166, 0.5934258699417114, -0.4916696846485138, 0.4404113292694092, 0.28402259945869446, -1.1699687242507935, -0.17731508612632751], [0.24417245388031006, -0.8118196129798889, 1.011443853378296, -0.3235475718975067, 0.7760036587715149, 1.424663782119751, -1.4910824298858643, -1.3391038179397583, 0.565864086151123, -0.2063177078962326, 0.4963715672492981, 1.704626202583313, -0.49641141295433044, 0.8446699380874634, -0.2679114043712616, -0.6522795557975769, 0.46042338013648987, 0.2474452257156372, 1.0642869472503662, 0.30352360010147095, 0.8629441261291504, 0.5154531002044678, -1.701560616493225, -0.3643140196800232, -0.32399246096611023, 0.8859382271766663, -1.0463389158248901, 0.18684017658233643, -0.41130778193473816, -1.8106598854064941, -0.15061433613300323, 0.6362862586975098, -0.36367934942245483, 0.9256194233894348, 0.3869490325450897, 0.3135834336280823, -0.5063206553459167, -0.15966038405895233, 0.4679069519042969, 0.025725187733769417, -1.2233233451843262, -0.17048896849155426, 0.3999524712562561, 1.123455286026001, 0.8411155939102173, -0.974478542804718, 0.6975511312484741, -1.5987610816955566, -0.6440047025680542, -0.808189332485199, 1.5204663276672363, 0.6865401864051819, 0.2996228337287903, -0.6151177287101746, 1.2398146390914917, -0.24120475351810455, 0.475904643535614, 1.1070510149002075, -0.29269394278526306, 0.5463207960128784, -1.9675624370574951, 0.5860067009925842, -1.683449625968933, 0.4205244779586792, 0.3003576397895813, 0.4886377453804016, 2.3220133781433105, -2.4713408946990967, 0.47103631496429443, -0.381017804145813, 0.7693902254104614, 0.32892364263534546, 0.12160792946815491, -0.48470211029052734, -0.6917915344238281, 0.37192922830581665, -1.4878300428390503, 1.0914934873580933, -0.5540403723716736, 0.48025792837142944, -1.1611723899841309, 0.4810890555381775, -0.20879074931144714, 0.7950872778892517, 1.2482962608337402, -0.5161319375038147, 0.13053235411643982, 0.268352746963501, 1.480513572692871, -1.2684950828552246, 0.6222208142280579, 0.008464104495942593, -1.1714502573013306, -0.30079272389411926, -0.637732207775116, 1.1352877616882324, -0.909988522529602, 1.0509458780288696, -0.42565950751304626, 0.7130054831504822, -1.6719897985458374, -0.7254576683044434, 0.19709257781505585, -0.8780776858329773, -1.5183849334716797, 0.056950848549604416, 0.8187955617904663, 0.04755261912941933, -0.1742197424173355, 0.4426378309726715, -0.38791602849960327, 0.9598175287246704, 1.140203833580017, -1.2348030805587769, 1.0836517810821533, 0.7204285264015198, 0.328364759683609, 0.730134129524231, 1.4700920581817627, 1.966498613357544, 1.6657438278198242, -0.19499263167381287, 1.386717438697815, -1.6184338331222534, 0.9510974884033203, 0.7594246864318848, 1.0354355573654175, -0.8381714820861816, 1.4984371662139893], [1.3358908891677856, 0.26876112818717957, -1.3583444356918335, 0.38818779587745667, -0.7873560190200806, -1.5037659406661987, -0.6586645245552063, 2.3984036445617676, -1.4218206405639648, -0.29630160331726074, 0.22874340415000916, 0.9761285781860352, -0.9679147601127625, -0.9552618861198425, 0.9066769480705261, 0.28697824478149414, 1.2440530061721802, -1.1496223211288452, -0.10223399847745895, 0.9294549822807312, 0.12448028475046158, -1.1227900981903076, 0.1202663853764534, -1.8929173946380615, -0.7715372443199158, 0.7639294266700745, 0.8818139433860779, 0.3931408226490021, 0.6788390278816223, -1.5156569480895996, -0.6639026403427124, 1.0279821157455444, 2.2717785835266113, 0.4368479251861572, -0.9305210113525391, -0.9720276594161987, 1.1869226694107056, 1.0721395015716553, -1.2194865942001343, 0.12531842291355133, 1.3691394329071045, 1.3912029266357422, -0.531356930732727, -0.18778826296329498, 0.4538325369358063, -1.103420615196228, -1.6351295709609985, -0.6922694444656372, 1.194830060005188, 1.072953462600708, 0.46120762825012207, -1.1852370500564575, -1.9787217378616333, -1.0636752843856812, 0.002857211511582136, 0.9872521758079529, -0.9279391765594482, 0.021518701687455177, -1.8465906381607056, -2.168660879135132, -0.07461746782064438, -1.2613791227340698, -1.1164393424987793, -2.3537614345550537, 0.7163331508636475, -1.894030213356018, -0.08429120481014252, -1.6114270687103271, -0.5052608251571655, 1.2104060649871826, 0.1114782989025116, -0.012641717679798603, 0.7612113952636719, 0.40015843510627747, 1.2579433917999268, -1.5863035917282104, 1.230648398399353, -0.056259363889694214, -1.1585197448730469, 0.48044610023498535, 0.42010125517845154, -0.12245702743530273, 0.7443864345550537, -1.4195377826690674, -1.7637375593185425, -0.6997578144073486, 0.9869081377983093, 0.43078312277793884, -0.09235268086194992, -0.7237256765365601, -0.09860637038946152, 1.0770570039749146, -0.6394053101539612, -0.6668735146522522, 0.35794797539711, 0.5762561559677124, 0.07261770963668823, 1.3719090223312378, 0.14468437433242798, -1.022006630897522, -2.248032808303833, 1.1292943954467773, -0.2037913054227829, 0.2919648587703705, 0.42350542545318604, 1.3222570419311523, 1.0219334363937378, 0.978955090045929, 0.27863141894340515, -0.22473406791687012, 0.2740042805671692, -0.8067792654037476, 0.37883684039115906, 1.0828473567962646, 0.7367536425590515, 0.16932864487171173, -0.8541640043258667, -0.8614801168441772, -0.7042945027351379, -0.3851498067378998, 0.46114492416381836, 0.5073408484458923, 0.5908446311950684, 1.0259931087493896, -0.9561517834663391, -0.1949096918106079, 0.5436128973960876, 0.8082082271575928, -0.6571894884109497], [-0.6810776591300964, 1.1555867195129395, 0.6977918744087219, 0.4627685546875, -0.2669364809989929, -0.935203492641449, -0.012505178339779377, 0.3257319927215576, 0.5133699774742126, 0.8759117126464844, 0.12845008075237274, 1.2625778913497925, -0.31729820370674133, 0.35374587774276733, 0.08300900459289551, -1.7643579244613647, -1.2842836380004883, -0.5118417143821716, 0.16003383696079254, 0.3381681740283966, 1.2812981605529785, -0.006536929868161678, -0.3182213306427002, 0.3255298137664795, -1.2052040100097656, -0.18832305073738098, -1.0033924579620361, 0.057175688445568085, -0.6999725699424744, -0.4237997829914093, 1.8882497549057007, 1.0905708074569702, 2.469655990600586, 0.17811331152915955, -0.188689723610878, -0.7956092357635498, -0.04698201268911362, -1.7590367794036865, 0.07384411990642548, -0.5748552083969116, -1.3497151136398315, -1.8121613264083862, -0.9193807244300842, -0.4275350868701935, -0.05947494879364967, -1.64388108253479, -0.07476512342691422, 0.28387925028800964, 2.1770005226135254, 0.33877798914909363, 0.8937822580337524, -0.730715274810791, -0.019404377788305283, -0.7404152750968933, -0.8446981310844421, 0.47775429487228394, -0.24072566628456116, 1.4348188638687134, -0.0411723256111145, 0.41117051243782043, -1.2196792364120483, -1.0053080320358276, 0.6512614488601685, -0.6899899840354919, 1.2565813064575195, -0.8239697217941284, -0.8898882865905762, 1.910224437713623, -1.6999537944793701, -0.5155699253082275, 0.5123782753944397, 0.5893102884292603, -1.1347285509109497, 2.133704900741577, -0.4575192928314209, -1.913766860961914, 0.9003212451934814, -0.48210787773132324, 0.05883335694670677, 0.76070636510849, -0.9017879366874695, -0.7199591994285583, -0.10530928522348404, 0.8340950608253479, -1.3156057596206665, 1.4358388185501099, 0.3610948622226715, -1.1238980293273926, 0.0377153642475605, 0.5611842274665833, 1.221332311630249, 0.8569538593292236, 1.1181904077529907, -0.14424821734428406, 0.7275230884552002, -2.6534476280212402, 1.0056447982788086, -0.04160373657941818, 1.5882720947265625, -0.5909373760223389, -0.4746071398258209, 0.0444914847612381, 0.8144798278808594, 0.3324418365955353, 1.0322227478027344, 0.055682744830846786, 1.0466878414154053, 2.801940441131592, 1.2472538948059082, 0.6279377937316895, -1.4446167945861816, 0.6415212750434875, 1.0955394506454468, -0.16186515986919403, 1.0557392835617065, -2.166597604751587, -0.6614884734153748, -0.9129278659820557, 0.5495222806930542, -0.7919466495513916, -0.942645788192749, 1.4859510660171509, -1.4311360120773315, 0.5201629400253296, 1.111341118812561, -0.25587236881256104, -0.8915174603462219, -0.17545627057552338, -0.564756453037262], [-0.781004786491394, 0.23601177334785461, -0.030951356515288353, 3.12978196144104, 1.3094782829284668, 0.20212064683437347, 0.5816898345947266, 1.6998707056045532, 1.1947802305221558, 0.07647979259490967, -0.7568893432617188, 1.795886754989624, 0.017559301108121872, 0.26916876435279846, 1.3895355463027954, 0.2169291377067566, 1.324846625328064, -0.26009559631347656, 0.21682462096214294, -0.16549158096313477, 2.1255083084106445, -1.426814079284668, -0.16830793023109436, -0.863045871257782, -0.7166242003440857, -1.2753571271896362, -0.8090828061103821, 1.2186641693115234, 1.452551245689392, -0.6537447571754456, -1.2274945974349976, -0.48667848110198975, 1.6302982568740845, -0.08179822564125061, -2.144144058227539, 0.836155891418457, -0.13665980100631714, 0.6002059578895569, 0.39171287417411804, -1.3143384456634521, 1.0052317380905151, 0.6517049670219421, -0.6921901702880859, -2.2925961017608643, -1.1858088970184326, -1.1594005823135376, 0.665918231010437, -0.6130726337432861, -0.46756985783576965, 0.008801755495369434, -0.4682404398918152, 0.3647231459617615, 0.2897820472717285, 0.9168935418128967, -1.2583324909210205, 0.32904526591300964, -0.29602566361427307, 2.088764190673828, 1.445675253868103, -0.42320311069488525, 0.935989499092102, -1.7749720811843872, -1.0734031200408936, -0.2833672761917114, 0.25009357929229736, -1.120275616645813, 0.018291369080543518, 1.2323355674743652, -1.6252540349960327, -1.3708462715148926, -1.934088945388794, 1.1241356134414673, -1.0970332622528076, -0.41185644268989563, -0.914253294467926, -0.11974187195301056, -0.36361655592918396, 0.1533760279417038, 0.13878530263900757, 1.5657347440719604, -0.9759353399276733, 0.14151504635810852, -0.11495938897132874, -0.9491914510726929, 0.7995018362998962, 1.7809921503067017, -1.1497052907943726, 2.3961565494537354, -0.19718724489212036, -0.13453000783920288, -2.6565663814544678, 0.010621080175042152, -0.17861679196357727, -0.3124305009841919, 0.6121293306350708, -0.22936029732227325, 0.31329914927482605, -0.7099069356918335, -1.1891077756881714, 0.05811619758605957, 0.5298157930374146, 0.6395646929740906, 0.052888739854097366, -0.7934590578079224, -0.008509693667292595, -0.30918553471565247, -1.308276891708374, 0.2794716954231262, -0.23103153705596924, -1.1538255214691162, -0.3506150543689728, 0.5476126074790955, 1.4532030820846558, -1.8724522590637207, -0.4146900177001953, 0.5015290379524231, 1.1106642484664917, 0.3363903760910034, -1.070554256439209, -0.5431464314460754, -0.0859975665807724, -0.8401591181755066, -1.112012267112732, -1.3379122018814087, 2.510154962539673, -0.02247120812535286, -0.2904917299747467, 0.6455308198928833, 0.010824425145983696], [-0.19010086357593536, 0.18665464222431183, -0.7990447282791138, -0.43834441900253296, 0.8836231827735901, 0.5880078673362732, -1.1156197786331177, 1.475097417831421, -0.41958656907081604, -1.613417387008667, 0.694321870803833, 0.634386420249939, 0.30448979139328003, -0.9891180992126465, 1.1895332336425781, 0.5138092041015625, 0.3494374454021454, 0.22866801917552948, 0.3709028661251068, 2.2266530990600586, -0.4959789216518402, 0.3082331717014313, 1.1446834802627563, 0.8039283752441406, -0.7510312795639038, -0.0860595852136612, -0.34908047318458557, -1.4209109544754028, -0.1631660759449005, 0.9902504682540894, 0.3163989186286926, 1.1417196989059448, 0.4738064706325531, -0.6775201559066772, 0.5678042769432068, -1.703792691230774, 1.2794784307479858, -0.3580833077430725, 1.676737904548645, -1.3553537130355835, -0.9564211964607239, -0.430580198764801, -0.2591986060142517, -1.5880521535873413, -1.277146816253662, 0.47548016905784607, -0.8243340253829956, 0.8623448014259338, 1.631091594696045, -1.489837884902954, -0.40216246247291565, 1.7312626838684082, 1.0769363641738892, -1.3606146574020386, 0.37083956599235535, 1.6638284921646118, -2.159972667694092, 0.3640095889568329, 1.0840883255004883, -0.4927200973033905, -0.6392617225646973, 0.7590381503105164, -1.5954642295837402, 0.5694605708122253, 1.2425590753555298, 0.10230327397584915, -1.5625170469284058, 1.60013747215271, -0.020088577643036842, 0.5741780400276184, -0.8748985528945923, -0.08187197893857956, 1.1121667623519897, -1.5197676420211792, -1.3320225477218628, -2.093881130218506, 0.44056981801986694, 0.4414888024330139, -0.8694364428520203, 0.32989615201950073, 0.014437459409236908, 1.9434773921966553, -1.2140159606933594, 0.2392989993095398, -0.45752808451652527, 1.8314324617385864, 0.44110772013664246, -1.216626524925232, 0.4251086115837097, -0.175266295671463, -0.9761208295822144, 0.3857848048210144, 0.11490046232938766, -1.2078213691711426, -0.485670804977417, 1.1292731761932373, 0.976953387260437, -0.16064292192459106, -0.38204488158226013, 0.25025561451911926, -0.14455831050872803, -1.24394690990448, 0.6068224310874939, -0.8393580317497253, 1.4386637210845947, 0.5618407130241394, 0.6995368003845215, -0.34382256865501404, 0.14577457308769226, -0.6129347085952759, -0.42109155654907227, -1.0600837469100952, 1.4091893434524536, -1.149919033050537, 1.6850876808166504, 0.7327732443809509, -0.9257792234420776, 1.3085623979568481, 0.4117244780063629, -0.8058820962905884, 0.24753502011299133, -0.3361808657646179, -0.7653293013572693, -0.0618341825902462, 0.2721654176712036, -0.32860127091407776, -0.03840440511703491, 0.7917439937591553, -0.11992919445037842], [-0.378557950258255, -1.294352412223816, -0.06459268927574158, -0.4687389135360718, -0.22050458192825317, 0.8020992279052734, 0.8703820705413818, 0.040912579745054245, -0.5724228620529175, 0.14720268547534943, 0.32861343026161194, 0.25478291511535645, -1.4172006845474243, 0.32326337695121765, -0.9996967315673828, 0.320315420627594, 0.41743022203445435, 0.16318683326244354, 1.207855463027954, 0.01479051262140274, 0.32615166902542114, -0.8809711337089539, -0.10002771764993668, -0.21992841362953186, 0.06490117311477661, -0.1686016172170639, -2.3560774326324463, 0.805406391620636, 0.6943036317825317, -2.079439163208008, 0.2602466940879822, 1.0137591361999512, 0.7354781031608582, 0.7151109576225281, -0.17722101509571075, -0.5046268701553345, 1.9121613502502441, 0.4398476481437683, 1.6501613855361938, -0.052299149334430695, 0.9439425468444824, 0.8412574529647827, 0.818879246711731, -1.188692569732666, 2.3924479484558105, 0.33551454544067383, -0.29756319522857666, 0.32479655742645264, -1.7309755086898804, 1.4188578128814697, -0.346971333026886, 0.32055410742759705, 0.22560487687587738, -1.7081478834152222, -0.2798407971858978, 1.3001861572265625, -0.730276346206665, 0.30337750911712646, 1.065657615661621, 1.6404671669006348, -0.7691839933395386, 1.1016300916671753, 0.7452482581138611, 0.6107906699180603, 1.0389691591262817, 0.8201932311058044, -0.9001283645629883, 0.5619605183601379, -0.5230181813240051, 0.4114916920661926, -0.33134451508522034, 1.5659278631210327, 1.0170353651046753, 0.5815622806549072, -0.5807849764823914, 0.51385098695755, -1.0035947561264038, -0.7715128660202026, -1.5717649459838867, 0.9636313915252686, -0.4024624526500702, 1.0030198097229004, 1.2182425260543823, 1.1063387393951416, -0.3077954649925232, 1.5379661321640015, 1.265000343322754, -1.1544547080993652, 0.4682372808456421, -0.6951484084129333, -0.022936604917049408, 1.3036699295043945, 0.42246028780937195, 1.683321475982666, 0.7273616194725037, 0.8500034213066101, 1.7642631530761719, 1.3434134721755981, 0.911620557308197, 1.4728586673736572, 1.330311894416809, -1.243754267692566, -0.9052606225013733, 0.21332314610481262, 1.2077810764312744, -0.6194227337837219, -1.5395796298980713, 0.05504227802157402, -0.4783228635787964, 1.575105905532837, -2.2478721141815186, 0.9079110026359558, 0.7472369074821472, -2.5606913566589355, -0.9227759838104248, -0.16732941567897797, 0.44995999336242676, 0.09719620645046234, 0.6899613738059998, 0.27491047978401184, -0.7153596878051758, 1.7368595600128174, 0.3642338514328003, -0.7737829685211182, 0.563937783241272, -0.06117634102702141, -0.3448496162891388, -0.5581191778182983, -1.4117670059204102], [-0.8076779246330261, -1.1939810514450073, -1.897962212562561, 0.5509454607963562, 0.3628039062023163, -0.15085627138614655, -1.0407567024230957, -0.7167680263519287, 1.6299352645874023, -0.7959619760513306, 0.9119446873664856, 0.4162493348121643, 0.5679078698158264, -0.3996126651763916, -0.15792572498321533, 0.003336943220347166, -1.018898606300354, 1.43120276927948, -0.7506768107414246, 0.23978865146636963, 1.8231441974639893, 1.5403310060501099, -0.5311506390571594, 0.5738212466239929, 1.0874178409576416, -1.8910123109817505, 2.110719680786133, 0.6049327254295349, -0.21609407663345337, -0.015538937412202358, -2.1641530990600586, 0.7712533473968506, 1.6535958051681519, -0.11123259365558624, 1.6781489849090576, 0.7513460516929626, -1.2658034563064575, 0.6297914981842041, -1.9534791707992554, -1.8066571950912476, 0.7138416171073914, 0.6305066347122192, -0.024764394387602806, -0.0402241013944149, -1.4663721323013306, 1.0780456066131592, 0.04856136068701744, -0.13359074294567108, 3.669452428817749, 0.352760910987854, 0.05274422839283943, 2.0926833152770996, 0.386934369802475, -0.3843454420566559, 1.639045000076294, -0.034254033118486404, -0.6870525479316711, -0.23723991215229034, 0.1367483139038086, 1.0486431121826172, 1.4102795124053955, -1.0812240839004517, 1.394909381866455, -0.501691460609436, -0.329020619392395, -2.489926338195801, 1.7764335870742798, 0.7214294075965881, -0.08954618871212006, -0.04613678902387619, 1.1112926006317139, 0.15243792533874512, -2.8551864624023438, 0.4045675992965698, 0.10427498072385788, 0.2573631703853607, 0.1795583963394165, 0.9954039454460144, -0.6039557456970215, 2.4490466117858887, 0.26448360085487366, -0.6055952310562134, 0.22633281350135803, -1.5984420776367188, -0.03601393103599548, -0.6258882880210876, -0.8693615794181824, 0.9446389079093933, 0.41372936964035034, -0.8563627600669861, -0.5115823149681091, 0.4500369131565094, 2.420621395111084, -0.008905640803277493, -0.043289948254823685, 0.5969404578208923, -1.1512302160263062, 0.009762950241565704, 0.8374390006065369, 0.5077797770500183, 0.4556768536567688, -1.1090019941329956, 0.1494082510471344, 1.5586838722229004, 2.8172097206115723, 1.1182386875152588, -0.4151712656021118, -0.32243838906288147, -0.04657766595482826, 2.2881243228912354, -1.2225061655044556, -0.09062597155570984, -1.544969081878662, -0.28512293100357056, -0.16899797320365906, 1.0452277660369873, 0.9398621916770935, 0.3478604853153229, -0.1491071879863739, -0.2873014807701111, 1.7840752601623535, 0.011387032456696033, -0.9912266731262207, -0.6326099634170532, 0.15697598457336426, -0.6765087842941284, -1.4683529138565063, 1.0616841316223145, -0.5892715454101562], [-0.02207467332482338, 0.20102348923683167, -0.1616804599761963, -0.06647615134716034, -2.4991862773895264, 0.13932743668556213, -0.1405157595872879, -0.32424259185791016, 0.018885863944888115, 2.3227319717407227, 1.7679246664047241, 0.8382335901260376, 1.664363145828247, -0.3326779901981354, 0.03519587218761444, 0.5706882476806641, 1.0013527870178223, 0.931293785572052, 1.9553847312927246, -0.031207812950015068, 1.4647469520568848, -0.5321940779685974, 0.6592119932174683, -0.878399670124054, 1.3640141487121582, -0.4347444772720337, 1.4308451414108276, -1.821637749671936, 1.9322527647018433, 0.07948070019483566, 0.5502247214317322, 0.3919917345046997, -1.5939762592315674, -0.5102814435958862, -0.5299928784370422, -0.005466143600642681, -1.492484211921692, -0.5690298080444336, 0.15728746354579926, 1.21574866771698, 0.806494951248169, -1.1163264513015747, 0.9458615183830261, 1.6814056634902954, 0.857367992401123, -0.2690417170524597, -0.8248648643493652, -0.34697049856185913, 1.7242757081985474, -0.9279528856277466, -2.4194793701171875, 0.9917749762535095, -0.5754764676094055, 0.010429725050926208, -0.3220331370830536, 1.1960948705673218, -0.3138681948184967, -0.6567397713661194, -0.7225727438926697, 1.8534576892852783, 1.5970205068588257, -0.2778395712375641, -0.4083525240421295, -0.011046482250094414, 0.6683026552200317, -0.7485581636428833, 1.651876449584961, 0.3821668326854706, -2.5454466342926025, -0.9657443165779114, -0.20349299907684326, 2.5273561477661133, 1.7099511623382568, 0.3123376965522766, -1.0619319677352905, 0.012717599049210548, 0.11672402173280716, -0.37756049633026123, 0.1990797519683838, 0.7746176719665527, -0.433651328086853, -1.1050851345062256, 0.18594303727149963, -1.5397838354110718, -0.4531818628311157, 1.191870093345642, 0.17804329097270966, -1.2440470457077026, -0.0859241783618927, -0.323657751083374, 0.0636184960603714, 0.962037980556488, -0.5909261703491211, -1.321194052696228, 1.5092127323150635, -0.9343401789665222, 0.5398938655853271, 0.3421713709831238, -0.39345088601112366, 0.5084875226020813, 0.2783619463443756, 0.13496342301368713, 0.7137356400489807, -0.7643887400627136, 0.8992970585823059, -0.8980752229690552, 0.9496381282806396, 1.3134031295776367, -0.8491531014442444, 1.9544042348861694, -0.6800726652145386, -0.1002228707075119, -0.8754423260688782, -0.34104523062705994, 0.02055792137980461, -0.6677989363670349, 0.9256113767623901, -0.8508288860321045, 0.26300105452537537, -1.1409105062484741, -1.0019426345825195, 0.035798296332359314, 0.22831594944000244, -0.44901296496391296, 0.24885015189647675, 0.3856405019760132, -0.750982940196991, -0.03777866065502167, -0.9376604557037354], [-1.3876490592956543, -0.6766886115074158, -0.29553356766700745, 1.1181421279907227, 0.4539259076118469, 0.056715548038482666, -2.2260327339172363, -0.9782673716545105, 0.44575127959251404, 0.5411408543586731, -0.0437752828001976, -1.240713357925415, -1.9938713312149048, -1.7112352848052979, 0.13042163848876953, 0.3669484555721283, 1.7728031873703003, -0.6860250234603882, 0.31447452306747437, -0.7926022410392761, -0.2125956416130066, 0.0902768149971962, -0.533419668674469, 1.4189165830612183, -1.0891096591949463, 0.8001328706741333, -0.2083030492067337, -0.48843857645988464, 1.0469019412994385, -0.5809113383293152, 0.3130585253238678, 0.014320558868348598, 0.729598343372345, -1.1695497035980225, 0.3034633994102478, 0.505075991153717, -0.6264676451683044, -0.9858750700950623, -1.00603187084198, -0.012379027903079987, -0.7526535987854004, 0.5597797632217407, -0.09980011731386185, -0.2373424917459488, -0.020668858662247658, 0.08994413912296295, 0.13165126740932465, 0.6020428538322449, 1.174375295639038, 0.4552365243434906, 0.5402231812477112, 1.5493414402008057, 1.5585445165634155, 0.12316928058862686, -1.0120285749435425, 1.230969786643982, -0.7424295544624329, -0.696306586265564, -0.7462608814239502, -0.29446151852607727, -2.276244878768921, -0.4572458267211914, 1.1792857646942139, 2.6208910942077637, 0.5391353368759155, 0.8768848776817322, 0.9694157242774963, -0.49345776438713074, -0.8622544407844543, 0.46727344393730164, 1.2277569770812988, 0.23879915475845337, 0.45445796847343445, 0.08548691123723984, -0.021056214347481728, -0.402068167924881, 1.2033315896987915, 0.968112587928772, -0.0013365935301408172, 0.40697431564331055, 0.7513900995254517, 1.251464605331421, 2.084843873977661, 1.8908519744873047, 0.16530440747737885, 0.4058784246444702, -0.10618538409471512, 0.9447513818740845, 1.262300729751587, -1.4496774673461914, 0.2826324701309204, 1.60035240650177, -1.2605453729629517, 1.6886570453643799, 1.2878575325012207, -0.34872716665267944, -0.8445107340812683, 1.8808796405792236, -0.221232607960701, -1.1113379001617432, 0.21268554031848907, -1.2464568614959717, -0.8771566152572632, 1.8787096738815308, -0.3198715150356293, -0.04607617110013962, -1.5581611394882202, -0.7883463501930237, 1.104463815689087, 0.4750637114048004, 1.1164833307266235, 1.0361287593841553, 0.28422293066978455, -0.36383056640625, -2.1247739791870117, 1.1420001983642578, 0.1545884907245636, 2.6226654052734375, -0.09006505459547043, -0.0731600970029831, -0.43371182680130005, -1.176361083984375, 0.06325434893369675, 0.36409491300582886, 0.25718995928764343, -0.25415897369384766, -0.7438240647315979, 1.1780551671981812, 1.027952790260315], [-0.599527895450592, -1.2342503070831299, -0.6160076856613159, -0.6271891593933105, 1.0809812545776367, -0.8035887479782104, 0.43243607878685, -0.9495375156402588, -0.039725031703710556, 2.708012819290161, -0.5832083821296692, 0.5030795335769653, -0.37884974479675293, 1.234366774559021, 0.44362279772758484, 0.1136258989572525, 1.5481984615325928, 0.9377270340919495, 0.7578338980674744, -1.073061227798462, -0.7949923872947693, 1.5373121500015259, -1.0829328298568726, 1.7018065452575684, 1.6819370985031128, -0.14890903234481812, -0.8005144596099854, -1.1792784929275513, 0.8401976227760315, 0.5751713514328003, -0.7591441869735718, -1.0343009233474731, 0.48247677087783813, -0.640798807144165, -0.03220880776643753, -1.5299360752105713, 0.402008593082428, 0.12186194956302643, -0.4977613389492035, -1.1977771520614624, -0.09964291006326675, 0.33317652344703674, 0.3829517960548401, 0.09092021733522415, -0.779270350933075, 0.6315711736679077, 1.1209423542022705, -0.6255283951759338, -1.286483883857727, 1.4252437353134155, 0.16330048441886902, -1.9399609565734863, 0.7842692732810974, 0.12387283146381378, -0.7447664737701416, 0.027807021513581276, -1.1486802101135254, -0.9307671785354614, -1.722091794013977, 1.0621122121810913, -0.6326982378959656, 0.08502670377492905, 0.2169565111398697, -2.732394218444824, 0.014081151224672794, -1.2954497337341309, 1.0639091730117798, -1.380427598953247, 0.6587836742401123, 1.471825122833252, 0.08669890463352203, 0.8584582805633545, 1.194773554801941, 0.02792491763830185, -0.38900026679039, -1.5889815092086792, 1.6951990127563477, 0.8665593266487122, 0.506310224533081, -1.8434938192367554, 0.975393533706665, -0.5094924569129944, 0.042766645550727844, 0.22528059780597687, 1.0057882070541382, -0.4199599325656891, -0.7902737259864807, 0.47320881485939026, 0.1597360521554947, -0.699326753616333, 2.0074105262756348, 1.0434210300445557, -1.6673433780670166, -2.3010034561157227, -1.9957977533340454, -0.5268765687942505, 0.352260023355484, 1.6282919645309448, 0.08191370964050293, -0.7347097396850586, 0.05440321937203407, -0.041692931205034256, -1.0020501613616943, 0.06976798176765442, -1.9326742887496948, -0.43648406863212585, 0.3837496042251587, -0.2843954265117645, 0.4668576121330261, 1.0231965780258179, 0.5493147969245911, 0.6228705644607544, 0.6184104681015015, -0.21842464804649353, 0.9824907779693604, 0.25517991185188293, -0.964322030544281, 0.9549018144607544, -0.0004831248661503196, -3.0660784244537354, 1.7354286909103394, 0.08594142645597458, -0.7083824276924133, 0.8257585763931274, 0.45420125126838684, 0.33050107955932617, 1.8281474113464355, 1.233863115310669, -0.13985268771648407], [-0.5444862246513367, -1.1087369918823242, 0.570365846157074, 0.2608412504196167, 1.2310672998428345, 0.3583904206752777, 0.8770138025283813, 1.5032603740692139, -0.3114337921142578, -0.7543888092041016, 0.1867266595363617, -1.077838659286499, 0.013387348502874374, -0.07032518833875656, 1.6018507480621338, -0.14568272233009338, 1.377049446105957, -0.19923368096351624, 0.1428505778312683, -0.28498879075050354, 0.10054522007703781, -0.41120389103889465, -0.9107153415679932, -1.077695608139038, 1.0522812604904175, 0.3327541947364807, 1.7538800239562988, 1.6509888172149658, -0.8366454839706421, 2.0371389389038086, -0.28609010577201843, -1.4162380695343018, -0.9979317784309387, -0.3910221457481384, -2.2535486221313477, 0.6009042263031006, -0.21070602536201477, 0.7628827691078186, 0.47557440400123596, 0.40400230884552, 0.853766143321991, 0.6308665871620178, -1.2478245496749878, 1.6848945617675781, -0.3208509385585785, 0.17833884060382843, 0.8454301357269287, -0.5733767747879028, -0.3965543210506439, -1.6642366647720337, 0.7736933827400208, 0.7775303721427917, -0.299230694770813, -0.6773855090141296, -2.514958620071411, 0.5378599762916565, -0.8507954478263855, -0.42537787556648254, -0.2967323064804077, -0.3130812644958496, 0.18705753982067108, 0.6300008296966553, 0.19919556379318237, -1.0451347827911377, -0.020509161055088043, -0.08655871450901031, -0.9038636684417725, -0.22128744423389435, 0.9081897139549255, 0.7853316068649292, 0.5633329749107361, 0.6661117672920227, -1.7912592887878418, 0.2651492953300476, 1.5451120138168335, -0.217426136136055, -0.2132813036441803, -0.6274972558021545, 0.3971070349216461, 0.07662956416606903, -0.8529272675514221, 0.20238222181797028, 0.1325932890176773, 0.4724142253398895, 0.5175940990447998, -0.6043939590454102, 0.524985134601593, 0.08333409577608109, 0.031214985996484756, -0.2661878764629364, 0.1888849139213562, -0.49033042788505554, -0.9569790363311768, 1.1740267276763916, -1.1024752855300903, 0.33115118741989136, 0.8058168888092041, 0.2902373671531677, -0.23658229410648346, -0.09747184067964554, -1.4204792976379395, 0.3619725704193115, -0.8461920022964478, -0.08307235687971115, -0.8300307393074036, 0.8621885776519775, -0.3533377945423126, -0.013483098708093166, 1.395234227180481, -0.8890802264213562, -0.21504710614681244, -2.094970226287842, -0.3042507469654083, -0.42280420660972595, -0.6085910201072693, 0.6500580310821533, 1.455389142036438, 0.7865899801254272, 0.7212127447128296, 1.0137466192245483, -0.5396412014961243, -0.7289538383483887, 0.32752540707588196, 1.6877877712249756, -0.5063605904579163, -1.3823210000991821, 0.09076593816280365, 1.0088201761245728, 0.7031950354576111], [0.49174609780311584, -0.049010030925273895, 0.7770435810089111, -0.3234564960002899, 1.2346559762954712, 1.3415954113006592, 0.8487607836723328, 1.9967336654663086, 2.4077866077423096, -0.6812154650688171, -0.09861690551042557, -0.8367488384246826, -1.5391485691070557, -0.9994667768478394, -0.09714461117982864, 0.5614930391311646, 0.46621599793434143, -2.4200003147125244, -0.015830831602215767, -1.083845615386963, 0.6593484282493591, -0.7751826643943787, 0.9774006605148315, 1.7005267143249512, 1.8602111339569092, -0.15513384342193604, 0.260910302400589, -0.28917360305786133, 0.002173990709707141, 1.7165462970733643, -1.713084101676941, -0.5573107600212097, 0.09434834122657776, -0.4529511630535126, -1.086175560951233, 1.0495578050613403, -0.06758122146129608, -0.42223218083381653, -1.320471167564392, -0.1297682374715805, 0.9048753380775452, -0.16614428162574768, 1.1050201654434204, -0.3137668967247009, -0.8260130286216736, 0.04329283908009529, -0.898741602897644, -1.4730079174041748, 0.5234805345535278, 0.9651668667793274, 1.6584739685058594, -1.8449057340621948, 0.9482500553131104, 0.6545873880386353, 0.16653674840927124, 0.49001583456993103, 0.6925751566886902, 0.5347399115562439, 0.5861974358558655, -0.25116679072380066, 1.7307138442993164, 0.01642986387014389, 1.1019518375396729, -0.3379424810409546, -0.01493168342858553, -0.08982212841510773, 0.2541075646877289, 1.377838134765625, -1.4568641185760498, 1.2326050996780396, -0.6656433939933777, 2.084641218185425, -0.9200748205184937, 0.18127202987670898, 0.6569408178329468, -1.55116868019104, 0.018506959080696106, 0.09272684156894684, 0.4064425528049469, -0.160739466547966, -0.9993754625320435, -1.8157023191452026, -0.5328195691108704, -0.3720066249370575, -0.3506377637386322, 0.8372249603271484, 0.12250470370054245, -1.670135498046875, -0.8287774324417114, 1.137454867362976, 0.6644172668457031, 0.17052625119686127, -0.3093158006668091, 1.0887192487716675, -0.740144670009613, -0.6393713355064392, 1.1116560697555542, -1.788040041923523, 1.1337958574295044, -0.4279007911682129, 0.4255218803882599, -0.3938894271850586, -1.8064815998077393, 0.399572491645813, -0.6515771150588989, -0.05830460041761398, 0.4304400086402893, 0.20775733888149261, 0.1667565256357193, 0.667146623134613, -1.351588487625122, 2.097111463546753, -1.4840242862701416, -0.1670117825269699, 0.031077882274985313, -0.46986934542655945, 0.7826470136642456, -0.45550522208213806, 0.08494602888822556, 2.6720945835113525, -0.4000047445297241, 2.3715035915374756, -1.348454475402832, 0.6242257952690125, 0.25933998823165894, 0.707385241985321, -2.2703182697296143, -2.0064947605133057, 0.6915615797042847], [0.8800045251846313, 0.7162166833877563, -0.4391005039215088, 0.5487795472145081, 0.43563732504844666, -0.5491616129875183, 1.919872522354126, -0.00032434906461276114, -0.4695151746273041, 0.4183412492275238, -0.35233211517333984, 0.30032283067703247, -0.8742133975028992, -0.8191576600074768, -0.49221599102020264, 1.0834145545959473, -0.21076446771621704, 1.6385637521743774, 1.4967293739318848, -0.5037083625793457, -0.42488497495651245, 1.2180496454238892, -0.2549029290676117, -0.30171120166778564, 1.821458339691162, 1.3213258981704712, 0.6479431986808777, -1.7392300367355347, 0.21691802144050598, -1.634506344795227, 0.2887404263019562, 0.4238879382610321, 1.8846768140792847, -1.241048812866211, 0.24225294589996338, 0.41023921966552734, 0.5130223035812378, 0.626878559589386, 1.2477844953536987, 0.6180711984634399, -0.28965404629707336, -0.13585925102233887, 1.2116777896881104, -0.21065767109394073, -1.8686232566833496, -1.4722355604171753, -1.1146342754364014, -1.1786876916885376, 0.356062650680542, 0.2759130299091339, -1.661330223083496, -0.9336567521095276, 0.5167797207832336, 0.5057218670845032, -0.03286616876721382, 2.239790678024292, -0.7739237546920776, 1.2027331590652466, 0.3283206820487976, -2.0239779949188232, -1.0456703901290894, 0.13546480238437653, -0.7525551319122314, -0.35035979747772217, -0.12367137521505356, 0.34345492720603943, -0.29454147815704346, 0.6660486459732056, -0.19884058833122253, -1.009178876876831, -0.4146901071071625, -1.2347060441970825, 0.7957962155342102, -1.4912142753601074, -0.5723325610160828, -0.4268089830875397, 0.5205849409103394, 1.4249812364578247, -1.1816209554672241, -0.3602447509765625, -0.5160291194915771, -0.7679498195648193, -1.7065409421920776, 0.0699097067117691, -1.917862057685852, 1.4530715942382812, 0.8312211036682129, 1.0619817972183228, 1.8629705905914307, -0.12079829722642899, -0.762529730796814, -1.9147412776947021, 1.1343944072723389, 0.787158191204071, -0.5209838151931763, -1.1111725568771362, 0.30048492550849915, 0.2542317807674408, -1.195390224456787, 1.4576613903045654, -0.5778392553329468, 0.568040668964386, -1.8905266523361206, -1.1276369094848633, 0.08875586092472076, -1.1010946035385132, 1.7238500118255615, -2.0404491424560547, -0.5840736627578735, -1.9244705438613892, -0.46767693758010864, 1.204102635383606, 0.1988835483789444, -0.2727375626564026, 1.2012814283370972, -1.6561877727508545, -1.3464322090148926, -1.6909348964691162, 1.0329740047454834, -1.7447850704193115, 1.3032145500183105, -0.8057382702827454, -0.2886289358139038, -0.44818368554115295, 0.794296145439148, -0.5666796565055847, -0.3662557601928711, -0.360377699136734, -0.1917087286710739], [-0.9802372455596924, -0.041272737085819244, -1.1197788715362549, 0.5011885166168213, -0.5564223527908325, 1.3714535236358643, -1.469316840171814, 0.1052069291472435, -0.17192314565181732, 1.7377996444702148, 0.4717894494533539, -0.2514336109161377, -0.24114155769348145, -0.7756066918373108, 2.019641637802124, 2.1111929416656494, -0.14483901858329773, -0.8362942934036255, 0.3815666437149048, 0.6736478805541992, -1.0197395086288452, -0.4930746853351593, -0.6245810389518738, -0.37381091713905334, 0.4082507789134979, -1.156516194343567, -1.9344582557678223, 2.000594139099121, -0.4570998251438141, 0.39182835817337036, -0.602501630783081, 0.0321047268807888, 0.19940733909606934, -0.25421416759490967, 2.461193561553955, 0.5463696122169495, -0.4257553517818451, -0.018303127959370613, -0.10656332224607468, -0.3405010998249054, 0.9991199374198914, 0.9842782616615295, -0.16202853620052338, 0.1363089680671692, -1.3802944421768188, 0.3901398479938507, -0.24612028896808624, -0.9726874828338623, -0.47533488273620605, 1.3551169633865356, -1.0910536050796509, -1.016846776008606, 0.27631860971450806, 0.34194129705429077, -0.2551145851612091, -1.1886680126190186, -0.7390343546867371, 1.889793872833252, -1.5242053270339966, 0.09363221377134323, 0.8008700013160706, 0.6158519387245178, 2.2204933166503906, 0.3060794770717621, -0.5211189985275269, -0.8112620115280151, -0.3621951937675476, 1.3080341815948486, -1.2139594554901123, 0.17393513023853302, -0.9175522327423096, -0.1936272829771042, 1.9367077350616455, 0.16325253248214722, 0.36571651697158813, -0.5954098701477051, -1.6549508571624756, -0.9547434449195862, -0.5510498285293579, -0.12529003620147705, -2.078146457672119, -0.22789478302001953, 1.3841413259506226, -0.5712010264396667, 0.6989284753799438, 0.16103795170783997, -0.11692284047603607, -0.33527228236198425, 0.6750202775001526, 1.6486066579818726, -1.8783437013626099, 0.9962577819824219, -0.4499322772026062, 0.31253546476364136, 1.0477606058120728, 0.1380656659603119, 1.927510142326355, 1.056329369544983, -0.33794501423835754, 0.8510277271270752, -1.520365595817566, 0.223826602101326, -0.5904584527015686, -0.8707898259162903, 1.0219483375549316, 0.21771720051765442, -0.5061797499656677, 0.34664615988731384, -0.10265473276376724, -0.15373732149600983, -1.622632384300232, -0.5325694680213928, 0.5274972319602966, 0.38177070021629333, 0.06147228181362152, -1.1792473793029785, -0.7709571123123169, -2.4615800380706787, -1.0615229606628418, 1.2579439878463745, 1.728068232536316, 0.0780329704284668, -0.018412165343761444, -1.458206295967102, 0.22092711925506592, -0.5564751029014587, 2.1669185161590576, 0.515734851360321, -0.0918591246008873], [-1.563861608505249, 1.495393991470337, 0.09101404994726181, 2.1288444995880127, 0.6906834840774536, -0.1310042440891266, 0.7752655148506165, 0.8430606126785278, -0.5347410440444946, 1.4196686744689941, 2.0081684589385986, -0.6506512761116028, -2.3557257652282715, 0.7959073781967163, 0.1674852967262268, 0.061347492039203644, -0.8932486176490784, 1.6052732467651367, -2.1013050079345703, 1.3137673139572144, -0.46130919456481934, -2.1218760013580322, -0.8169575929641724, -0.020301692187786102, 2.0610735416412354, 1.2053632736206055, -0.15118201076984406, 0.04869851469993591, 1.372254490852356, 0.8412649631500244, 2.2972002029418945, -0.674870491027832, 0.45088455080986023, -0.5436086654663086, 0.1601867526769638, -1.4988503456115723, 0.036795858293771744, -1.4628453254699707, 0.3694289028644562, 1.5107907056808472, 0.2687346637248993, 0.17862524092197418, -0.14459551870822906, -1.3811910152435303, -0.7164093255996704, -0.43335428833961487, -1.0589871406555176, 1.0983706712722778, -1.8928616046905518, -1.0933756828308105, -0.9462209343910217, -0.33402547240257263, -1.1718027591705322, -1.4688470363616943, -1.317297101020813, -0.7429214715957642, -1.4822328090667725, 1.0537409782409668, -0.04808179289102554, 1.2225860357284546, -0.6239447593688965, 0.570334792137146, -0.09944514185190201, 0.16496413946151733, 1.5699793100357056, 0.7109699249267578, 0.23582927882671356, -0.17268294095993042, -1.8204898834228516, -0.25403645634651184, -0.6886768937110901, 1.8073029518127441, -0.553601861000061, 0.12847867608070374, -0.15285009145736694, -0.6626530289649963, 1.0083727836608887, -1.1219316720962524, 0.5660475492477417, -1.9433139562606812, -0.6101096868515015, 1.6785695552825928, -0.2211441844701767, 0.21589738130569458, 0.271149218082428, 0.0286391768604517, -2.6769917011260986, -1.5904300212860107, 0.878007709980011, 0.1152067631483078, -1.4611774682998657, -1.4057828187942505, 1.206874966621399, -0.5435417890548706, -0.282929390668869, -0.5047668814659119, -0.876478910446167, 0.244895800948143, 0.44913697242736816, -0.3043368458747864, -0.12481492757797241, -0.8880175948143005, 1.2632039785385132, -1.5344189405441284, 0.22167478501796722, 0.6377472281455994, 1.0842390060424805, -0.018856290727853775, 0.45132070779800415, -0.2553410530090332, -0.8868060111999512, 0.06265944242477417, 0.9765191078186035, 0.6428951025009155, -1.2719883918762207, 0.006598939653486013, 1.2464561462402344, 0.33467549085617065, 1.4299421310424805, 0.043102312833070755, -0.18768860399723053, -3.370133638381958, 1.2786056995391846, -1.7094963788986206, -0.4977390468120575, -1.0145713090896606, -0.7862887978553772, 0.7882120609283447, -1.6614556312561035], [0.2768338620662689, -1.28827702999115, -0.07273998856544495, 0.3783629834651947, -0.24906909465789795, -0.34851402044296265, -0.1986112743616104, -0.22784847021102905, 0.5674247145652771, 0.35082143545150757, -0.4564245045185089, 2.2033889293670654, -0.7315242290496826, 1.7768008708953857, -0.22081324458122253, 1.2851892709732056, 1.032975673675537, -0.35795703530311584, 0.7830169200897217, 2.027700424194336, -1.6016870737075806, -1.3236498832702637, -0.8390014171600342, 1.8029696941375732, 0.49645504355430603, -0.7004523277282715, -0.19771859049797058, 0.26636525988578796, -0.8437486290931702, -0.2678534686565399, 0.10900364816188812, 0.49232372641563416, 0.9651598334312439, -0.4794887602329254, -0.19522236287593842, -1.2675235271453857, -0.1175745502114296, 0.11540621519088745, -0.06132514029741287, 0.4513113498687744, -1.070207118988037, 1.1201826333999634, 0.1658138781785965, 1.1973832845687866, -0.948409914970398, -0.19147612154483795, 0.11389109492301941, -0.6434137225151062, 1.0858477354049683, -0.002018057508394122, -1.3862680196762085, 0.9660335183143616, -0.6246188879013062, 0.292257159948349, 0.8159233331680298, -1.0273146629333496, 1.364378571510315, 0.7884781956672668, -1.5330575704574585, 1.1707109212875366, -0.834356963634491, -0.756402313709259, 1.6193252801895142, -1.394229769706726, -2.574143886566162, 0.16555047035217285, -0.22486989200115204, -0.9259911179542542, 0.9805583953857422, -1.523705005645752, -0.9774053692817688, 0.7634568214416504, -0.7090926766395569, -0.9702633619308472, -0.3905709981918335, -1.2536758184432983, -1.305534839630127, -0.14431223273277283, -0.5343497395515442, -0.1910923272371292, -0.8801730871200562, 0.16412353515625, 0.3716287910938263, 0.719578742980957, -0.6482669115066528, -0.7074137330055237, -1.226779818534851, 0.2250070869922638, 0.8244946599006653, 0.8518427610397339, 0.7063177227973938, 1.108468770980835, 1.6698336601257324, 0.18308372795581818, 0.59214186668396, -0.7830852270126343, 0.40906980633735657, 0.8687995672225952, -0.5991575717926025, 1.9720345735549927, 0.5585143566131592, 0.8201404213905334, 0.3109128475189209, 1.985021710395813, 0.2704832851886749, 0.9420554637908936, 0.31152647733688354, -0.35367101430892944, -0.0656011775135994, -0.47194811701774597, -0.26262420415878296, 0.8970198631286621, -0.6963518857955933, -0.23698307573795319, 0.8377557992935181, 1.2324193716049194, 0.1304905116558075, -0.24315527081489563, -1.2064695358276367, 0.6899863481521606, -0.7136940360069275, -0.5037974715232849, 0.7555248141288757, -0.4429056942462921, 0.5560126304626465, 0.5422175526618958, -0.6368101835250854, -0.10124419629573822, 1.2605576515197754], [0.38788577914237976, 0.25697097182273865, -1.2730209827423096, -0.07194429636001587, 1.2581969499588013, -0.2369992733001709, 0.41875508427619934, 0.9762288928031921, 2.370945692062378, -1.6120505332946777, 1.0631515979766846, 1.1551449298858643, 0.10620122402906418, 1.9813672304153442, -1.1884150505065918, -1.4039037227630615, -0.10005427896976471, 0.72105473279953, -1.0057862997055054, 0.016240699216723442, -0.48947206139564514, 1.9849827289581299, -0.5241298675537109, 1.3775871992111206, -2.0189809799194336, -0.05554557591676712, 0.03701860085129738, -1.2719154357910156, -0.28887778520584106, 0.34866106510162354, -0.2620466649532318, -0.49298644065856934, -0.08474335074424744, 0.43952426314353943, 0.5230534672737122, -0.5819536447525024, 0.9295130968093872, -0.10053921490907669, 2.4540605545043945, -0.021784977987408638, 0.22694864869117737, -1.0363504886627197, 1.5043869018554688, -0.20113001763820648, 0.3571065664291382, 0.949825644493103, -0.3539852499961853, 0.24864299595355988, -0.3954518139362335, -0.7757024765014648, -1.2587718963623047, -1.9908123016357422, -1.0486527681350708, -1.9177871942520142, 0.7303601503372192, -1.1396435499191284, 2.6571030616760254, -0.2259160727262497, -0.7303973436355591, 1.4451613426208496, 0.2148122638463974, -0.22577309608459473, -0.24953626096248627, 1.4156718254089355, -0.7746990323066711, -1.1835341453552246, -0.1769597977399826, 0.3291323482990265, -2.123547077178955, -1.3525723218917847, -0.9142471551895142, -0.5161382555961609, -0.10813255608081818, -1.9442353248596191, -0.48165833950042725, -1.932750940322876, 0.6988195776939392, -0.3732512593269348, 0.7102636098861694, 1.3250758647918701, 0.20278896391391754, 0.06276010721921921, -0.7073216438293457, -0.582403302192688, 0.5517502427101135, 0.8998998999595642, 1.798844814300537, -0.7425676584243774, -0.975374698638916, 1.9285402297973633, 0.060471825301647186, -0.9850966930389404, 1.0243605375289917, -0.17259612679481506, -0.4591682255268097, -1.250525951385498, 0.865337073802948, 0.896608293056488, 1.6602977514266968, 0.17142106592655182, -0.33358415961265564, 0.18970689177513123, 0.26700854301452637, -0.2972638010978699, -1.3055901527404785, -1.0352529287338257, -0.08884114772081375, -0.37535348534584045, 1.708493709564209, 0.18797163665294647, -0.14208365976810455, -0.5870552659034729, 0.6652262806892395, -0.11236711591482162, 1.0283054113388062, 0.17294330894947052, 1.172786831855774, -0.05225992947816849, 0.7532159090042114, 1.3254108428955078, -0.3457348346710205, 1.5195088386535645, 0.03353935107588768, 2.2391839027404785, 1.5857810974121094, 1.206868052482605, 1.08186674118042, 1.882498860359192, -0.35227152705192566], [1.0902862548828125, 0.08324446529150009, 0.483358770608902, 0.6988820433616638, 1.2501872777938843, -0.5313143730163574, -0.86055588722229, -1.3037354946136475, 0.7239990234375, -0.33735358715057373, 0.7800387740135193, 1.1827212572097778, 1.3168779611587524, -1.8137363195419312, 0.2995480000972748, 0.25755175948143005, -0.8812285661697388, -1.4743562936782837, 0.48734909296035767, 2.3232619762420654, -1.154697060585022, -0.10659538954496384, 0.5565210580825806, -2.4317612648010254, 1.7804522514343262, -0.7876074910163879, 1.1353323459625244, 2.8790640830993652, 0.3255450427532196, -0.9858913421630859, 0.045655593276023865, -0.4142349362373352, 0.7170513868331909, -0.9084858894348145, 1.1641769409179688, -0.6308771967887878, 3.297049045562744, -1.3866647481918335, 0.9288282990455627, -1.367081880569458, 1.2776907682418823, 0.06458083540201187, -1.931300163269043, -0.9387201070785522, -0.636250913143158, -1.3307783603668213, 0.04471113532781601, -1.077725887298584, -0.07405751198530197, 0.5164960622787476, 0.03103223443031311, -0.371587872505188, -0.3774874806404114, 0.5103338956832886, -0.5032612681388855, -0.06671582162380219, -2.2964651584625244, -0.953279435634613, 0.37940189242362976, -0.17462731897830963, 0.094627246260643, 0.09762752056121826, 1.4192060232162476, -0.6381536722183228, -1.7466950416564941, 1.199786901473999, 0.17090719938278198, -0.8313361406326294, -1.3388603925704956, -2.0449109077453613, -0.9258684515953064, -0.2480202317237854, 1.249472975730896, -0.46571865677833557, -0.3248603045940399, -0.8261905908584595, -1.4045153856277466, 0.27511918544769287, 0.11807303875684738, -0.29390382766723633, 0.748828649520874, -1.5928603410720825, -0.10979261249303818, -0.4522913992404938, 0.5622940063476562, -0.9261947870254517, -0.5373826026916504, -0.6107172966003418, 0.08448956161737442, -0.4769534170627594, -0.552126944065094, -1.2156164646148682, -1.0218759775161743, 0.3525184988975525, -0.9322224259376526, 0.8638546466827393, 1.7320483922958374, 1.3255101442337036, 0.4014439582824707, 0.33329230546951294, 0.5588715076446533, -0.08777998387813568, -0.27990052103996277, -0.2207765281200409, -0.9586986303329468, -0.6348710656166077, 1.608700156211853, -1.4134435653686523, -1.3099560737609863, -2.0113120079040527, 0.5908345580101013, 1.7790930271148682, 1.043725848197937, -0.3892809748649597, 0.7341077923774719, 0.35735392570495605, 0.6047242879867554, -1.0512694120407104, 0.8282725214958191, -0.8141987919807434, 0.29069435596466064, 0.6230687499046326, 0.8179329037666321, 1.7608140707015991, 1.4611084461212158, -0.20040982961654663, 2.239717960357666, -1.6589038372039795, -0.654655396938324], [0.19223211705684662, 0.5787293314933777, 0.006907264236360788, -0.04949382320046425, 1.5687330961227417, -1.74116849899292, -0.242389515042305, -0.7750416994094849, -0.09747552126646042, -0.5686659812927246, 0.45828333497047424, 1.1464225053787231, 1.63169264793396, -2.211745023727417, -0.5385403633117676, 1.1945058107376099, 0.04438477382063866, -0.3071184754371643, -0.10667265206575394, 0.9478039741516113, -0.12707297503948212, 0.2025008648633957, -0.23023931682109833, 0.06926090270280838, 0.3059774339199066, 1.9955624341964722, 0.7036504149436951, 0.23307667672634125, 1.1917074918746948, -1.5256694555282593, -0.3215099275112152, 0.14860336482524872, -1.116136908531189, -0.16235853731632233, -0.2728215456008911, -0.4582945704460144, -0.5791647434234619, 0.068404920399189, 1.0066394805908203, -0.39921703934669495, 0.35509446263313293, 0.5145436525344849, -0.5142641663551331, -0.1437956690788269, 1.0240795612335205, -1.4205577373504639, 0.11169066280126572, 0.05051083490252495, -0.5367143154144287, -2.1671712398529053, 0.723008394241333, 2.41970157623291, -0.18514464795589447, 0.8719507455825806, 0.29221397638320923, 0.2715490460395813, -0.22182369232177734, 0.37669676542282104, -0.962404191493988, -0.05133551359176636, 0.08767084032297134, 0.10677001625299454, -0.813228964805603, 0.802558422088623, -1.5136164426803589, -1.2827293872833252, 0.16621063649654388, -1.7722145318984985, 2.160320520401001, -1.3086761236190796, -0.5052151679992676, 1.3190326690673828, -1.3065615892410278, 1.5415034294128418, 1.3526607751846313, 1.1479740142822266, 0.032842811197042465, 1.480133295059204, -0.07121367752552032, 0.955133318901062, -0.8979144096374512, 0.9896824359893799, 0.8208983540534973, 0.6875496506690979, 0.9469583034515381, -0.966835618019104, 1.030153751373291, -0.1339181810617447, 0.14132627844810486, 0.5753971338272095, -0.5533998608589172, 0.8999382853507996, -0.1609657257795334, 0.7611567974090576, 0.0663747563958168, -0.8015317320823669, 0.9621618986129761, 0.007485271897166967, 0.5497468709945679, -0.44088080525398254, -1.533830165863037, 0.8628532290458679, -2.0456488132476807, -0.7187368869781494, -1.8244072198867798, 0.3379386067390442, -2.0698416233062744, 0.6963571310043335, 2.4558160305023193, -0.9397079944610596, 1.0603611469268799, 1.9305551052093506, -1.2484829425811768, 0.06563562154769897, -0.6271955370903015, 0.04872806370258331, -0.29656699299812317, 1.958443284034729, 0.2651039958000183, 2.5341625213623047, 2.3244378566741943, 1.3475080728530884, -0.2734006941318512, -0.3572967052459717, -0.5923944115638733, 0.8524808287620544, 0.20688515901565552, 0.281845360994339, 0.3389001488685608], [2.1001758575439453, 0.003438264597207308, -0.16570161283016205, 0.5121411681175232, 0.5087964534759521, 1.426455020904541, 1.5324100255966187, -0.6943824291229248, 0.6431236267089844, -0.26757752895355225, 0.674913227558136, -0.20122486352920532, -0.7374213337898254, -0.12065818160772324, 0.5343087911605835, -0.9556262493133545, 0.5144143104553223, 0.06553841382265091, -0.20641480386257172, 0.5433259606361389, 0.8254660367965698, -0.1093980222940445, -3.0474255084991455, -0.7639326453208923, 1.198258876800537, -1.2560092210769653, 0.3714872896671295, -1.7325125932693481, 2.1928343772888184, -0.8313685655593872, 0.7199209928512573, -0.5020292401313782, 0.2217177152633667, 1.2619448900222778, 0.03339386358857155, -0.9599210023880005, 1.5397101640701294, -0.10799399763345718, 1.497847080230713, -0.6940422058105469, -0.2749098539352417, 0.777789294719696, 0.9853416681289673, 0.7521869540214539, -0.726560652256012, 0.5834115743637085, -1.4175130128860474, 0.9483844637870789, 0.2403315305709839, 0.9074494242668152, -0.3627808690071106, -0.05972239747643471, 0.6901087164878845, -1.2665190696716309, -0.40336114168167114, 1.0201700925827026, 0.7464519739151001, -0.24818535149097443, -0.5202367901802063, -1.0445215702056885, -0.1798485368490219, 0.8010074496269226, 0.27071675658226013, -1.0459744930267334, -0.7510895133018494, 0.7434871196746826, -2.2552688121795654, 1.6674696207046509, -2.880190134048462, 0.04598414525389671, -0.66771399974823, -0.7340796589851379, -0.7050997018814087, 0.27851802110671997, -0.17610591650009155, -0.32257166504859924, 1.6364076137542725, 0.3855096399784088, -0.08022498339414597, -1.5044862031936646, -0.1994561403989792, 0.0020041947718709707, 0.5653350949287415, 0.043102093040943146, -0.045338064432144165, -0.5036266446113586, 0.8671906590461731, -1.6922842264175415, 0.11844733357429504, 1.3036158084869385, -2.7680234909057617, 1.3542295694351196, 0.4819377362728119, 0.34825044870376587, -0.850653350353241, -0.7909123301506042, -0.9589698910713196, 3.472222089767456, -0.9154431223869324, 2.284057855606079, -0.1974458545446396, -0.9781894683837891, 0.6718553304672241, -0.24345673620700836, -0.8701848983764648, -0.4292011260986328, 0.7333071231842041, -0.30042943358421326, -0.6133330464363098, 0.14931102097034454, 0.47944578528404236, -1.3310853242874146, 1.337803840637207, -1.0736792087554932, -0.0922059640288353, 1.061418056488037, 0.5648137927055359, -0.20612306892871857, -0.3050858676433563, 0.13498057425022125, 1.2966846227645874, 0.0883878841996193, 1.3050596714019775, -0.008422726765275002, 2.2383368015289307, 0.35563692450523376, 1.4436814785003662, -1.1073678731918335, 0.07915308326482773], [0.8109128475189209, -1.7469323873519897, -0.5737064480781555, -1.0100510120391846, 0.6443972587585449, -0.789484441280365, 0.20208629965782166, -0.09673936665058136, -0.06431077420711517, -1.6500072479248047, 0.6600853204727173, -1.9038927555084229, 1.1371395587921143, 0.10236755013465881, -0.8833778500556946, -0.20349393784999847, -1.775264024734497, -0.5832944512367249, 0.025296000763773918, 2.2850496768951416, -1.432558536529541, -0.15330497920513153, -0.961442768573761, 0.4769793450832367, 1.426732063293457, -1.063183069229126, 0.1855451911687851, 1.1140365600585938, -0.4011039137840271, -0.2248348593711853, -0.3591751754283905, 0.05483730509877205, 0.2605670690536499, -0.718945324420929, -0.24869467318058014, -1.0675734281539917, 0.2079891711473465, 0.1639561802148819, 1.0854426622390747, -0.1363147348165512, -0.1939164400100708, -1.4425809383392334, 0.7861199378967285, 0.17016497254371643, -0.46724554896354675, 0.3882421553134918, 1.171754002571106, 0.6560951471328735, 0.5692968368530273, 0.7822685241699219, -0.9353238344192505, -0.12985892593860626, 0.41731101274490356, -1.4019827842712402, -0.29549458622932434, 1.3300186395645142, -1.0842862129211426, -0.3901962637901306, -0.07687680423259735, 0.9714520573616028, -1.1786288022994995, 0.7372232675552368, 1.4869041442871094, 0.20697607100009918, -1.0004206895828247, 1.6486313343048096, 0.6454208493232727, 0.5945714712142944, 1.4904117584228516, 0.6010426878929138, -0.20138996839523315, -1.5327805280685425, 1.7664506435394287, 0.03531878441572189, -0.16137467324733734, 0.6384869813919067, 1.004434585571289, -0.9559895396232605, 0.013411082327365875, -0.6195273995399475, -1.051143765449524, -0.5634266138076782, -0.5290825366973877, -0.17416779696941376, -1.3922350406646729, -0.8510614037513733, 0.8220251202583313, -1.4410006999969482, -0.17546041309833527, -1.3465406894683838, -0.6320555210113525, 0.06044068560004234, 0.6890616416931152, 0.8289103507995605, -0.9831404089927673, 1.259721040725708, -1.3265854120254517, 1.1646045446395874, -1.3962258100509644, -0.9550991058349609, 0.5728387832641602, -0.27150091528892517, 0.8420060873031616, 0.25783771276474, 0.16398806869983673, 1.089277744293213, -0.320321261882782, -0.3975033462047577, 0.37690240144729614, 0.1442468911409378, 0.3822108507156372, 0.5951980948448181, 1.0348519086837769, -0.634972095489502, -1.6423897743225098, 0.12488745152950287, 0.7077903747558594, 0.3131999671459198, -1.1567928791046143, -1.6782314777374268, -1.6569533348083496, 0.3478665351867676, -0.6620119214057922, 1.0713753700256348, 1.9883676767349243, 0.8264356851577759, 0.3776049017906189, 0.20110349357128143, 0.5232517123222351], [2.0638680458068848, -0.27896592020988464, -1.1792038679122925, 0.609287679195404, 1.3511395454406738, -1.0583949089050293, -1.2624810934066772, -1.72567617893219, -0.11263401061296463, -1.3592593669891357, 1.5635380744934082, -1.069260597229004, 1.2912687063217163, 0.3351663649082184, -2.011031150817871, 0.20878103375434875, 1.6844426393508911, 0.8216366171836853, -0.41236820816993713, -0.12855945527553558, 0.13205718994140625, 0.1774825006723404, -0.0799497589468956, -0.8492032289505005, -1.0451107025146484, -1.9530644416809082, 0.043373268097639084, -0.14874334633350372, -1.0602821111679077, 1.2131448984146118, -0.6693601608276367, 1.3906584978103638, -0.33456525206565857, -0.5825166702270508, 0.45710083842277527, -1.0820866823196411, 1.3018871545791626, 1.6986392736434937, -0.16414479911327362, 0.4311096966266632, -1.0112738609313965, 0.46928995847702026, 1.1998474597930908, -0.44134166836738586, 0.40043607354164124, 0.32145974040031433, -0.23080144822597504, 1.6948086023330688, 2.378533363342285, -0.9595473408699036, 0.6816160678863525, -2.049564838409424, 0.042460836470127106, 0.2991481423377991, -0.21227547526359558, 0.1852637082338333, 0.3943139314651489, 1.5222550630569458, -0.030319930985569954, -0.7938611507415771, 0.4085700511932373, 2.1152408123016357, -0.4008237421512604, 1.5488896369934082, 0.4897771179676056, -1.2440372705459595, -0.014674708247184753, -0.4596136510372162, 0.7174389362335205, -1.4214798212051392, 0.33411166071891785, 0.12341638654470444, -2.301590919494629, -0.13989375531673431, -0.6466273665428162, 1.1539210081100464, 0.65511554479599, 0.928654670715332, 0.15399304032325745, -0.9803361296653748, 1.2463717460632324, 0.8128460049629211, 0.03402429819107056, 0.1153511106967926, 0.9880802631378174, 0.8945809602737427, 0.011539069935679436, -0.26297467947006226, 1.182291030883789, 0.15266595780849457, -0.42119163274765015, 0.09162689000368118, -0.3125666081905365, 0.36653855443000793, -0.3486858904361725, 0.05244891718029976, 0.5977470874786377, -0.32014545798301697, -0.5007175207138062, 0.29491469264030457, -0.8495615124702454, -0.22409936785697937, -0.2289804369211197, 0.24680788815021515, 0.27147915959358215, -0.10895171761512756, 1.5549728870391846, -0.8746418356895447, -0.1672239601612091, 1.4229602813720703, -0.3244260549545288, -0.047210872173309326, -0.8682376146316528, 0.28725704550743103, 1.4803436994552612, -0.14116667211055756, -0.8013339042663574, -0.1279768943786621, -1.4804147481918335, -0.7396244406700134, 2.412994384765625, -1.33128821849823, -1.3919869661331177, -0.2229013592004776, -1.0069586038589478, -0.19099180400371552, 1.9984480142593384, 1.059340000152588, -0.9481912851333618], [1.108962893486023, 1.0214512348175049, 1.2359191179275513, -0.4560266137123108, 0.3413582444190979, -0.37315869331359863, 0.5021788477897644, 0.2897445857524872, 1.3204528093338013, 0.6330519318580627, -0.20619584619998932, -2.2851152420043945, 0.44324710965156555, 0.6604573130607605, 0.9991644620895386, 0.17000417411327362, 0.26526573300361633, 0.024629054591059685, -0.16042447090148926, -1.640318512916565, -0.6158807873725891, -0.3103751838207245, -1.6322112083435059, -0.14907456934452057, -1.0135878324508667, 0.6399896740913391, 0.28666791319847107, -1.6667096614837646, -2.0971384048461914, 0.6225834488868713, 1.128770112991333, 0.8171005845069885, -1.8046787977218628, 0.868608832359314, -0.21024638414382935, 1.0552881956100464, 0.19233697652816772, -0.3878668248653412, -0.09656434506177902, 0.7839304804801941, 0.4693603217601776, -1.4722329378128052, -0.20682676136493683, 0.6064988970756531, 0.27411648631095886, -0.028190309181809425, 0.04341443255543709, -1.416563630104065, -1.264291524887085, -0.06932508945465088, -0.1342364102602005, 0.21497036516666412, -0.8422508835792542, -0.7675060629844666, -0.9331611394882202, -0.08590205758810043, 0.08358407765626907, 0.7434544563293457, 0.764553427696228, 0.6620254516601562, 0.6982331275939941, 1.4853078126907349, 1.3405232429504395, -0.07709788531064987, -1.4247936010360718, -0.37939968705177307, 0.17672303318977356, 0.11582937091588974, -0.6228737235069275, 1.0022451877593994, -1.5434867143630981, 0.048644907772541046, -0.12840168178081512, 1.341982364654541, 0.49799874424934387, -0.6355664730072021, -1.4485383033752441, -0.8022404313087463, 0.8356043696403503, -0.3275825083255768, 0.5569321513175964, 0.4669986963272095, -0.0814405307173729, 1.187933087348938, -0.6122938990592957, 0.2585231065750122, 0.9727864265441895, 0.17014369368553162, 0.23077304661273956, -0.6430297493934631, -0.9609073400497437, 0.17699483036994934, 0.2071707844734192, -0.6358237862586975, -0.43065252900123596, -0.1715850830078125, 0.3856348693370819, 1.4210500717163086, 0.36356011033058167, 0.01005481742322445, 2.059805154800415, -1.6869404315948486, 0.4740479290485382, -0.2904472053050995, 0.3457482159137726, 1.8814622163772583, -1.0600765943527222, -0.9190385937690735, 0.20046652853488922, 1.4351205825805664, -0.13853393495082855, 0.2141474187374115, 0.20578798651695251, -0.921176552772522, 0.2847844660282135, -0.3580044209957123, -0.03800664469599724, 0.2551921308040619, -1.5488331317901611, -0.08962786942720413, -0.3009892404079437, -0.22009214758872986, -0.5275431871414185, 0.5273258090019226, 2.009509801864624, -0.0760292187333107, 1.4464958906173706, -0.1905025839805603, 0.2419547438621521], [1.0380185842514038, -0.30699723958969116, -0.11355747282505035, -1.083021879196167, 1.6694694757461548, -1.515976071357727, 0.26037734746932983, -0.17471620440483093, 0.1698862463235855, -0.9567088484764099, 0.12647132575511932, 0.5348901152610779, -0.6611786484718323, -0.7533102035522461, 0.7379751205444336, 0.5762167572975159, 0.4365670680999756, 0.9902710914611816, 0.13518841564655304, -0.38328224420547485, -0.43335089087486267, 0.8179309368133545, -0.988742470741272, -0.10929125547409058, -1.4366858005523682, -0.376861572265625, 1.3289320468902588, 0.5152209997177124, -0.27876079082489014, -1.186575174331665, -1.145338535308838, -1.0641504526138306, 0.09390967339277267, -0.848019003868103, 0.6549689769744873, -1.142007827758789, 2.1067352294921875, 0.22393281757831573, 0.6858968138694763, 0.4644758105278015, -0.6017582416534424, -0.6906468868255615, -0.8644346594810486, -0.48269373178482056, 2.1885299682617188, 0.1424420177936554, 0.6218236684799194, -0.8665800094604492, -0.37846142053604126, -0.42201822996139526, -2.0478925704956055, -2.6727590560913086, -0.6363525390625, -0.8775485157966614, -1.5014395713806152, 1.1108062267303467, 1.3760254383087158, -1.479148507118225, 1.1663694381713867, 0.27148866653442383, -1.1327402591705322, -1.5906916856765747, -0.17530308663845062, -1.2868139743804932, 0.17867690324783325, 0.9572520852088928, 0.4236800968647003, 2.7673983573913574, 0.8275542855262756, 1.0570827722549438, 0.7279934883117676, 0.36411377787590027, 0.15010662376880646, 0.3988087475299835, -0.8913453817367554, 1.9094457626342773, -0.7765841484069824, 1.7247633934020996, 0.373653769493103, 1.0280855894088745, -1.0937758684158325, -0.9896049499511719, 1.593564748764038, 0.4756768047809601, -0.8379026651382446, -1.0633540153503418, -0.7935344576835632, -0.6368519067764282, 1.0084033012390137, 1.8234246969223022, 0.752849817276001, 1.0395742654800415, 0.7414272427558899, -0.7321352958679199, 0.24865782260894775, 0.6001213192939758, 1.0015482902526855, -0.4058472216129303, 0.704168438911438, 1.8625061511993408, 2.2484731674194336, 1.0206377506256104, -1.2264258861541748, -0.3123636543750763, 0.8135709762573242, -0.717998743057251, -0.3683496415615082, 0.8345053195953369, -1.4992215633392334, 1.578954815864563, 0.14456790685653687, -0.10248009115457535, -0.22102059423923492, -0.6468300223350525, 1.6879462003707886, 0.8633266091346741, -0.07848414778709412, 1.1492747068405151, 0.45752954483032227, -0.1697009801864624, -0.5772374868392944, 1.2092647552490234, -0.6153567433357239, -0.4409056007862091, -1.1554042100906372, -0.13710364699363708, -1.714400053024292, -0.8430438041687012, 0.29200685024261475], [0.641018271446228, 2.431713342666626, -0.18543025851249695, -0.5710818767547607, -0.6433430910110474, 0.9661287665367126, 0.08742063492536545, -0.8854589462280273, 0.4161193072795868, 0.04479387030005455, 0.09962879121303558, -0.32437145709991455, 1.2600489854812622, -0.06274889409542084, 0.5738426446914673, -0.05718039721250534, 1.7614243030548096, -0.4226651191711426, 0.7485467195510864, -1.6374115943908691, 1.50083589553833, -1.31373929977417, -0.7711995244026184, -0.19557146728038788, 0.6224027872085571, -0.5524295568466187, 0.8830882906913757, 0.5440758466720581, 0.957575261592865, -1.3308403491973877, -0.6027680039405823, -0.7464762926101685, 0.15631738305091858, 0.01755361817777157, 0.4417427182197571, 0.40194129943847656, 1.156772255897522, -0.5290924906730652, -1.0764248371124268, 0.4886947572231293, 0.629403829574585, 0.4644918143749237, -0.516446590423584, -2.6936023235321045, 1.2429519891738892, 0.3116191029548645, 2.1531572341918945, -0.4351440668106079, -0.3800508677959442, -2.436652898788452, -1.116565227508545, -0.34571561217308044, -0.1046018898487091, 1.7808148860931396, 2.088076591491699, -0.6937441825866699, 0.7496273517608643, 1.2509229183197021, 0.6356128454208374, 0.9788265228271484, 2.5656957626342773, 0.40482550859451294, 0.4644070267677307, -0.08822279423475266, 0.29527950286865234, 0.1700046807527542, -0.2616005837917328, -0.12258376181125641, -0.5114619135856628, -0.45903825759887695, -1.001535415649414, 0.31995972990989685, 0.1690613478422165, -1.7597993612289429, 1.127177119255066, 1.319222092628479, 0.9037203788757324, 0.5883115530014038, 0.47691118717193604, 0.35152560472488403, -0.5065749287605286, -0.20998898148536682, -0.8463031649589539, -0.5935871005058289, -1.5047738552093506, 0.1367964744567871, 0.27491843700408936, 1.9480408430099487, 0.515733003616333, 0.5603708028793335, -0.6344666481018066, -0.006257951259613037, -1.0440229177474976, 0.41671884059906006, 3.276104688644409, -0.2304644137620926, -2.2160537242889404, 1.5685620307922363, 1.126395344734192, -0.8810399770736694, 0.4028748869895935, -0.5421399474143982, 0.8125495314598083, -0.9439566731452942, 0.22226311266422272, 1.3295716047286987, 0.36633825302124023, 1.0463910102844238, 0.7559252977371216, -0.07883711904287338, 0.16315554082393646, -0.14472433924674988, 0.6406306624412537, 0.4290497601032257, 1.396089792251587, 0.4221903085708618, 0.21935079991817474, -0.27613428235054016, 0.07135547697544098, 0.4041513204574585, 1.3977785110473633, 0.9418787360191345, 2.011932849884033, -0.387999951839447, -0.32275524735450745, -0.5649946928024292, 0.8276852369308472, -0.3613249361515045, 1.2480294704437256], [0.566478967666626, -2.0209896564483643, 2.197901964187622, 0.06161169707775116, -0.42535167932510376, 0.7824591398239136, -0.8009304404258728, -0.013906629756093025, -0.5935245752334595, -1.1398648023605347, -1.3088690042495728, 0.6513956785202026, 0.02350516803562641, 0.038327641785144806, -0.3002147376537323, -0.9044455289840698, 0.21640096604824066, -0.10474758595228195, 1.4373152256011963, -0.5455637574195862, 0.2861107885837555, -0.8597714900970459, -0.07210629433393478, -0.7948259711265564, 2.6999411582946777, 1.2664158344268799, 0.13500605523586273, 1.2111762762069702, 1.7285785675048828, 0.9467645883560181, -2.0200092792510986, 1.11465322971344, 0.09432262182235718, -0.1956775188446045, 0.9100649952888489, 0.19733282923698425, -1.596293568611145, -0.8884481191635132, -0.288833886384964, 0.7620530724525452, 0.41601574420928955, -2.256105661392212, -0.3788055181503296, -1.5739965438842773, -2.069227933883667, -1.4750638008117676, -1.2183414697647095, -0.7029575705528259, -0.18577760457992554, 1.5950615406036377, 0.6101888418197632, 0.7550151944160461, 0.15826694667339325, 1.3253134489059448, 3.058734178543091, 1.0101348161697388, -0.39330610632896423, -0.7763051390647888, -0.14902299642562866, 0.9901934266090393, -0.5374755263328552, -1.2311253547668457, 0.49188241362571716, 2.366550922393799, -0.01888693869113922, 0.5898265242576599, 0.445093035697937, -0.13613396883010864, 0.5005343556404114, 1.5774654150009155, -0.47937270998954773, 0.828889012336731, 0.9810497760772705, 1.0366193056106567, 0.17062921822071075, -2.5245325565338135, 0.8421059250831604, 0.5854461789131165, 0.1219150498509407, -0.4529254734516144, -0.7455241084098816, 0.6001304984092712, 1.0000940561294556, -0.5339512228965759, -0.4792599081993103, 0.04115104675292969, -0.6084871888160706, 0.04366280138492584, 1.131401538848877, 0.20225736498832703, -0.1444064974784851, 0.6990165114402771, -1.9152004718780518, -1.2208397388458252, 0.5771746635437012, 0.8501450419425964, -0.8933519124984741, 1.9222760200500488, 1.8532241582870483, 0.16929641366004944, 1.2919384241104126, 0.957933247089386, -1.160425066947937, 0.6048306226730347, -0.21349430084228516, -0.05628158152103424, -0.05557243525981903, -1.9287781715393066, 0.09074446558952332, 1.9926183223724365, -0.2846316695213318, 1.031046748161316, -1.6839420795440674, 0.9482885599136353, -0.7763742208480835, 0.5319405794143677, 0.9454894065856934, -1.5249203443527222, -0.30577605962753296, 0.8649083375930786, 0.39116695523262024, -2.097860336303711, 0.1385471671819687, -1.4499635696411133, -0.06039407104253769, -1.7195751667022705, -2.3363037109375, 0.11198689788579941, -1.1649198532104492], [0.8026769161224365, 0.4828742742538452, -0.008776877075433731, 0.9373047351837158, -0.31547731161117554, 0.6400786638259888, -0.07926733791828156, -2.6395843029022217, -0.019810786470770836, -1.5623942613601685, -0.5455800890922546, 0.875748336315155, -1.1299611330032349, 0.7548880577087402, 0.8325334191322327, 1.9361646175384521, 0.7053979635238647, -1.3212461471557617, -0.1760462522506714, -1.5280272960662842, -0.0589119978249073, -0.31409257650375366, 1.2770164012908936, -0.22855563461780548, -0.950487494468689, -0.9076934456825256, -0.4193549156188965, -1.3818238973617554, -0.3190860152244568, -0.28065285086631775, -0.5264239311218262, -1.1744834184646606, -0.3301851451396942, -0.003671590704470873, 0.772220253944397, -0.2387235462665558, 0.8136644959449768, -1.6984481811523438, 0.8594071865081787, -2.170902729034424, -1.1826995611190796, 0.00472857104614377, -0.2708630859851837, 0.8115869760513306, 0.6234886646270752, 0.5652230381965637, -0.1322021186351776, -0.7282127141952515, 0.667151689529419, -0.06969285756349564, 0.194909006357193, 0.18954572081565857, 0.03529084473848343, 0.42386624217033386, 0.3646063506603241, -0.3758378028869629, -1.2995389699935913, 0.2672591805458069, -0.48917311429977417, -0.3959287405014038, 0.23592354357242584, -0.8188461065292358, -0.23273667693138123, -0.09041784703731537, 0.8652309775352478, 2.1718356609344482, -0.06638281792402267, 0.3111726641654968, -0.31233495473861694, 1.2711836099624634, 0.27318882942199707, 1.0476189851760864, 1.5618436336517334, 2.2254209518432617, -0.01764259859919548, -1.1232147216796875, 0.042618583887815475, -1.6429119110107422, 3.3060994148254395, 0.0900583565235138, -0.33999282121658325, 1.5076791048049927, -0.32209497690200806, 1.2881853580474854, -1.278727412223816, -1.6581811904907227, 0.4208323359489441, -1.4313236474990845, -1.1966475248336792, 1.0468965768814087, -0.6070457696914673, -0.09052738547325134, -0.4519417881965637, 1.07375967502594, -0.017896737903356552, -0.24735869467258453, -0.5580902099609375, 0.3525758385658264, 0.7389667630195618, -1.3962435722351074, 0.9030511379241943, -0.1946399211883545, 0.29373040795326233, 0.7051297426223755, -0.4963078200817108, -1.961822509765625, -1.6516718864440918, 0.9849512577056885, 1.2793736457824707, 3.0450239181518555, -0.2347392588853836, 0.8051751852035522, -0.17690370976924896, -0.9148199558258057, 0.7960973978042603, 0.3368380069732666, 0.4652152359485626, 0.14437001943588257, 0.7144559621810913, -0.05577138438820839, 0.957530677318573, -0.25398725271224976, -0.6071776747703552, 1.692386269569397, 1.8181499242782593, 0.9360502362251282, -0.9378830194473267, -0.3336239159107208, -2.5077996253967285], [-1.5516433715820312, 2.2339446544647217, -1.9102295637130737, 0.3937511146068573, 0.7912075519561768, -0.7194796800613403, 0.7919127941131592, -0.6287193298339844, 1.0067373514175415, 0.276155024766922, 0.07308007776737213, 0.021135320886969566, 1.2916324138641357, -1.2075444459915161, -0.5303464531898499, -1.5165351629257202, -0.24997206032276154, 0.15592417120933533, -0.3508981466293335, 0.1545874923467636, -0.40615302324295044, 1.8043646812438965, 0.7846450209617615, -0.10554630309343338, 0.28902512788772583, -0.33884379267692566, 1.1660526990890503, -1.522972583770752, -1.548854112625122, -1.5283852815628052, 1.0036191940307617, 2.655311107635498, 1.337342619895935, -0.3910333812236786, 0.11079167574644089, 0.5383013486862183, -0.24550245702266693, -0.33967825770378113, 0.7629555463790894, -0.848373532295227, -3.0702598094940186, 1.9346007108688354, 1.1431373357772827, -0.527516782283783, 1.8539929389953613, -0.8817811608314514, 0.1427917331457138, -0.17491762340068817, 0.23969733715057373, -1.2133246660232544, 0.4144420921802521, 0.8489058017730713, -0.29270684719085693, 0.2305438071489334, 0.5579003691673279, 0.7619463205337524, 0.20453961193561554, 0.5022400617599487, -0.05223334953188896, -0.5851078629493713, -2.7491567134857178, 1.6337194442749023, -0.07879696786403656, 0.5106501579284668, 0.8328275680541992, -0.5594188570976257, 0.679962158203125, -0.33483442664146423, -0.044089268893003464, -1.8407950401306152, 0.6207610964775085, 0.33536434173583984, -1.384232521057129, 0.8106724619865417, 0.8535672426223755, -0.39128023386001587, -1.3976976871490479, 0.028495317324995995, -1.2681665420532227, 0.28647977113723755, 0.5715357065200806, -0.9600921869277954, 0.9698431491851807, -1.2434200048446655, 1.3243389129638672, 0.740329921245575, -0.23723745346069336, 0.8831305503845215, 0.7834170460700989, 2.0512876510620117, 0.3718193471431732, -0.36172717809677124, 1.4451462030410767, -0.5589752197265625, -0.8999493718147278, -0.560206413269043, -0.6760034561157227, -0.8558144569396973, -2.3450520038604736, 1.5556238889694214, -0.3813883364200592, 0.2836783230304718, 0.6631749272346497, 0.5162972211837769, -1.2317078113555908, -0.8855681419372559, 1.00798761844635, -0.3876875936985016, -0.10975117236375809, 0.569980263710022, 0.4595378041267395, -0.23756279051303864, 0.9445824027061462, -0.853858470916748, -1.5038092136383057, -0.6223612427711487, 0.7889233827590942, -0.0856100469827652, -0.7162245512008667, -0.33351051807403564, -0.4261458218097687, 1.1613775491714478, -0.16900335252285004, 1.6066709756851196, -1.436734676361084, -0.5720832943916321, 0.3426758646965027, 0.04332411289215088, -1.8890957832336426], [1.449522614479065, 0.16842423379421234, -0.6545245051383972, -1.3363450765609741, -0.05042096599936485, 0.37570029497146606, 0.15006697177886963, -0.04745179042220116, -0.5666244626045227, 1.0966085195541382, 0.47612541913986206, 0.3272716999053955, -0.04381486773490906, 0.3757920563220978, 0.710624635219574, 1.142052173614502, -0.3047958314418793, -0.5331560373306274, -1.0212055444717407, -1.132907748222351, 0.9797585010528564, 1.902970314025879, 0.023948950693011284, -0.4843870997428894, 0.9804410934448242, 2.178054094314575, -0.7123987078666687, 0.3667134642601013, -0.6734734773635864, 0.1855345070362091, -1.031070351600647, -1.0772072076797485, -0.9706543684005737, 0.6784265041351318, -0.014009570702910423, 0.0146158616989851, 0.05700526013970375, -0.2059589922428131, 0.07833084464073181, -0.20882222056388855, 1.309248447418213, 1.3445314168930054, 0.5164667367935181, 1.1965364217758179, 0.3567587733268738, -0.2564677894115448, 0.017135443165898323, 0.31088200211524963, -0.7255035638809204, -1.4801737070083618, 0.5129438638687134, 0.8173176050186157, -1.2895197868347168, 1.8602533340454102, -0.6322759985923767, 0.16288210451602936, -0.41002118587493896, -0.8814889788627625, 0.3447593152523041, -0.7869101166725159, 0.5136566162109375, 0.24548126757144928, -0.20700114965438843, 1.0818332433700562, 0.6557433605194092, -0.01997835747897625, 0.042488835752010345, 0.07453466206789017, 0.5503897070884705, -0.9069543480873108, -1.2710676193237305, 0.24573633074760437, -0.1963706761598587, 0.6300494074821472, -0.17182882130146027, -0.14610545337200165, 0.34012776613235474, 0.6517494320869446, 0.6231544613838196, 0.5178658366203308, 0.4554789066314697, -1.3985368013381958, 1.4455134868621826, 0.14763908088207245, -0.11229677498340607, -1.3878812789916992, 0.049756355583667755, 0.2944406569004059, 0.39592012763023376, 0.5231173634529114, -1.5742942094802856, -1.4351693391799927, 2.5625789165496826, -1.4253581762313843, 0.2127869576215744, 0.372359037399292, -0.17850635945796967, -1.9610984325408936, -0.6912216544151306, -0.10139422118663788, -2.6501622200012207, 0.5696835517883301, 0.11493132263422012, -0.602446436882019, -1.0779683589935303, -1.084587574005127, 1.2585185766220093, 0.9251490235328674, 0.6232056617736816, 0.692660927772522, -1.3944661617279053, 0.22176043689250946, -0.10952475666999817, 0.00926223210990429, -0.8810520172119141, -0.07507559657096863, -0.8803600668907166, 0.7511554956436157, -0.17873822152614594, 1.084082007408142, 0.18487536907196045, -0.2918175160884857, 0.8003389239311218, 0.9605349898338318, -0.11505839973688126, 0.36241307854652405, -0.3993242383003235, 0.15760019421577454, -0.20851901173591614], [0.45621275901794434, -0.6728506684303284, -1.4998034238815308, 0.06503751128911972, -1.037286400794983, -0.8497605323791504, -1.6251716613769531, 1.4129705429077148, -0.07396857440471649, -0.16217344999313354, -0.7524024248123169, -0.053580086678266525, -0.5657508969306946, -0.2968378961086273, 1.1066761016845703, 1.7247189283370972, -2.5260119438171387, -1.1736388206481934, -0.4590071141719818, -1.071401834487915, -0.4642210304737091, -0.06407786905765533, 0.04989018663764, -1.0000253915786743, -1.5904855728149414, -2.525423526763916, 0.373058021068573, 1.1163510084152222, 1.3112914562225342, -0.3371613919734955, -1.4709160327911377, -1.7323851585388184, 0.9598344564437866, -0.07709436118602753, 2.381255865097046, 0.7777596712112427, -0.3121200203895569, 1.1326693296432495, -0.25782233476638794, 0.0432865209877491, -0.6644918322563171, 0.5382633805274963, -0.8495004177093506, -0.3285229504108429, -0.5821883082389832, 0.5775085687637329, 0.729031503200531, -0.2250068187713623, 0.37495726346969604, -1.3876897096633911, -0.6334213614463806, -0.9328606724739075, -2.4649980068206787, -1.4273536205291748, -0.4382697343826294, -0.16733582317829132, 0.2517606019973755, 1.665194034576416, 0.048093393445014954, 1.1444793939590454, -0.1143779382109642, -0.38713690638542175, -0.08009578287601471, -1.5892720222473145, -0.8636420965194702, -0.6206085085868835, 0.7851098775863647, 0.2598589062690735, 0.21877717971801758, -0.7491254210472107, 0.021496813744306564, -1.0656075477600098, 2.9434974193573, -0.06534044444561005, -1.875351071357727, 0.695054292678833, -0.6351702809333801, 0.9548496007919312, -0.16695360839366913, -1.2305833101272583, 0.6195279955863953, -0.44104838371276855, 2.5811893939971924, 1.361596941947937, -1.3909326791763306, 1.000866413116455, -1.9668571949005127, 0.08943869173526764, -0.08044074475765228, 0.7345478534698486, -0.5424973964691162, -0.6324082016944885, -0.749057948589325, 0.7589138746261597, 0.3884921669960022, 0.5570796728134155, 0.3794730007648468, -0.21663132309913635, 1.5348734855651855, -0.7263414859771729, 1.4955146312713623, 1.1462604999542236, -1.0346403121948242, 0.5793027281761169, 0.6450657844543457, 0.713499128818512, -1.0691298246383667, 0.9028148055076599, 0.5103397369384766, -1.530627727508545, -0.40494218468666077, 0.8489989042282104, -0.43051326274871826, 0.4384905695915222, -0.42171841859817505, -0.9310100674629211, -0.09401184320449829, 0.4555838704109192, -0.37229952216148376, -1.2370517253875732, -1.7336028814315796, 0.6721821427345276, 0.34992727637290955, 1.3960572481155396, 0.06586173921823502, 1.373519778251648, -0.5862443447113037, 0.9246897101402283, -1.604058861732483], [-2.3414974212646484, 0.3178243935108185, -0.8042358160018921, -0.11128618568181992, -0.21571506559848785, -1.191056728363037, -0.3025336265563965, -1.3785289525985718, -0.2423115074634552, -1.0240854024887085, 0.010731256566941738, -0.36354196071624756, 1.4953532218933105, -0.20060765743255615, 1.7355238199234009, -0.6013982892036438, 0.8513478636741638, 0.7362316250801086, -0.26989442110061646, 0.6525896787643433, -0.620037853717804, -0.19293005764484406, 0.7489644289016724, -0.0010080176871269941, -2.161668300628662, -0.9235721230506897, 1.5146468877792358, 0.5378808975219727, 0.166696235537529, -0.4100521504878998, -0.3399519920349121, -1.1737642288208008, 0.741601824760437, 0.2683365046977997, 1.4146467447280884, 1.110140085220337, -0.1185598373413086, 0.06436984986066818, -0.14146970212459564, 0.4534303545951843, -0.446659654378891, -2.4173901081085205, -0.7332437038421631, -0.31259605288505554, 1.2886091470718384, -0.5850085616111755, -0.8500651121139526, -0.6322757601737976, -0.42757055163383484, 0.7759252786636353, -1.7811264991760254, 1.9888616800308228, 0.5049442052841187, 1.2036399841308594, -0.06756293773651123, 0.386732280254364, -1.9223532676696777, 0.23388825356960297, 0.09899842739105225, 1.1454037427902222, 0.09440606832504272, 0.3040845990180969, 0.47559887170791626, -1.1841384172439575, 0.5721312165260315, 2.488922357559204, -0.058212634176015854, 1.3720905780792236, -0.0698176845908165, 1.8843592405319214, 1.8607372045516968, -0.08066968619823456, -0.4860813617706299, 0.4683266878128052, 0.13677240908145905, -0.4884836971759796, -0.7249135375022888, -1.6489148139953613, -0.6419762969017029, -0.3832315504550934, 1.6370912790298462, 0.9527953863143921, -0.7018930315971375, -0.1522531658411026, -0.7794746160507202, 1.0270277261734009, 2.0897300243377686, -0.03370761126279831, 0.25115838646888733, 0.9390420317649841, -0.8147254586219788, 1.3746223449707031, 0.4300142228603363, -1.6207293272018433, -0.9282040596008301, -0.38990551233291626, 0.15236537158489227, 0.2532983720302582, 0.1556629091501236, 0.3027004301548004, 0.37864816188812256, 0.37558141350746155, 0.6404697299003601, 1.4364687204360962, 0.6408646702766418, 0.5442326068878174, -1.1759142875671387, -0.2327936738729477, -0.01810842752456665, 0.7997627854347229, 0.0901142880320549, -0.7966607213020325, 0.7495880722999573, 0.5921914577484131, 0.3032565116882324, -1.495566487312317, -0.19452925026416779, -0.3010806739330292, -0.2914502024650574, 1.6404417753219604, -1.0497276782989502, -2.492027759552002, 0.4607222080230713, -0.4640134274959564, 0.3163404166698456, 1.603629469871521, 0.7376576662063599, -2.0859785079956055, -0.5121464729309082], [-0.034085992723703384, -0.13613145053386688, -0.9654237627983093, -1.3458305597305298, 1.5446797609329224, -0.16472475230693817, -0.41648197174072266, 1.2667906284332275, -1.6379899978637695, -1.6336761713027954, 0.9987990856170654, 0.1525488942861557, -0.023265408352017403, 0.917993426322937, -0.9255216717720032, 0.5439004898071289, 0.6669827103614807, 0.29792237281799316, 0.969037652015686, -0.14101257920265198, -2.166276693344116, 1.6318747997283936, 0.4548494219779968, -0.44139760732650757, -1.3701553344726562, -1.3436601161956787, -1.2929333448410034, -1.23591947555542, 0.7295567989349365, 0.34285786747932434, 0.5098312497138977, 0.6368385553359985, -0.15509742498397827, 0.510658860206604, -0.10970357805490494, 0.32456618547439575, 0.9007563591003418, -0.5029202103614807, -0.556553840637207, 0.534509539604187, -0.18681246042251587, 0.7842893600463867, -0.17909525334835052, -0.24187304079532623, -1.5374572277069092, -0.3177213668823242, -0.5905167460441589, 0.2928236424922943, -1.0941129922866821, -0.5477216243743896, 0.007822380401194096, 2.043074131011963, 0.4071780741214752, 0.44374755024909973, -0.21020948886871338, 0.03366423770785332, -0.30423370003700256, 0.23044058680534363, 1.8402459621429443, -0.6121876239776611, -0.9663277268409729, -1.307329773902893, 0.7135908007621765, -0.2643146514892578, -0.09021571278572083, 1.6216691732406616, 2.510500192642212, 0.5809953212738037, -0.33486565947532654, 1.4721195697784424, 0.19229809939861298, 1.3224481344223022, 0.5460497140884399, -0.30688703060150146, -0.8954018354415894, 0.27306559681892395, -0.02114095538854599, 1.559065580368042, -0.4082563519477844, -0.07720528542995453, -1.3317049741744995, -0.7485318183898926, 0.4185612201690674, -0.5035837292671204, 0.6022356748580933, -0.8932846188545227, -0.17209279537200928, 0.621490478515625, 1.3845930099487305, -0.4549228549003601, -2.4435348510742188, 1.3940553665161133, 0.7789029479026794, -0.27070537209510803, 1.9497671127319336, 0.32907387614250183, 0.4292006492614746, -0.32601118087768555, -0.31855350732803345, -0.5152431726455688, 0.024530889466404915, 1.2855892181396484, 0.904854953289032, 0.27881449460983276, 1.06513512134552, 0.43092671036720276, -0.6819407343864441, -0.7136708498001099, -2.7292346954345703, 0.35883161425590515, 0.8945841789245605, 0.43141016364097595, 1.1763222217559814, 1.1337194442749023, 1.2070770263671875, -0.8518136143684387, 0.13419148325920105, 0.9749716520309448, -1.1122275590896606, -0.7657662630081177, -0.08432255685329437, -0.17784501612186432, 0.5816754698753357, 1.341762900352478, 0.747486412525177, -0.6052663922309875, 1.1783382892608643, -0.028134144842624664, 0.8350731730461121], [0.19108153879642487, 1.0722010135650635, -0.9562996625900269, 1.069440484046936, 2.523109197616577, 0.28666627407073975, -0.8960488438606262, -0.8552054762840271, 1.047379493713379, 0.11471651494503021, 0.5744025707244873, 0.2269134223461151, 0.2290668934583664, 2.525566577911377, -2.6966710090637207, 0.5366421341896057, 1.0677132606506348, 0.1434820145368576, -1.5360593795776367, 0.18234024941921234, -0.5657671689987183, -0.13834819197654724, 0.8508901596069336, -0.778653085231781, -0.7821919918060303, 1.4803942441940308, -0.5965862274169922, -0.3175797462463379, 0.942477822303772, -0.7528725266456604, 0.47899946570396423, -0.7644632458686829, -0.8385291695594788, -1.080076813697815, -0.649908721446991, 2.1355481147766113, 0.3318909704685211, -1.5913280248641968, -1.1113064289093018, -0.5248059630393982, 0.47625410556793213, -1.3371251821517944, 0.9162339568138123, -1.371700644493103, -1.847050666809082, -0.47397592663764954, -0.7642427682876587, -1.9370970726013184, -0.0662403330206871, -0.3588380813598633, 0.04788913205265999, -0.4700193405151367, 0.09763266146183014, 1.423922061920166, -1.1136200428009033, -0.01118630263954401, -0.5291798114776611, 0.08552853018045425, 0.3371169865131378, -0.894217312335968, 0.881969690322876, 0.23434396088123322, -0.6863037943840027, 0.47285524010658264, -1.2408597469329834, 2.7274069786071777, -0.9193611741065979, 1.2132433652877808, 0.15591812133789062, 0.5542590618133545, 0.5954023599624634, -0.7708650827407837, -1.1687911748886108, 0.7709318995475769, -0.38627102971076965, -0.7245025038719177, 1.4467614889144897, 1.6666902303695679, -0.8127321600914001, 1.0597950220108032, 0.5289039015769958, -0.4363931119441986, 0.4185066223144531, -2.596045970916748, 1.416957974433899, 1.222133755683899, 0.9465311765670776, -0.887241005897522, 0.5304895639419556, -0.15908533334732056, -1.0968172550201416, -1.0941886901855469, -0.48207417130470276, -0.2820914387702942, -1.4587948322296143, 0.23350679874420166, 0.23638133704662323, 1.3374247550964355, 0.22207802534103394, 0.9912443161010742, 1.2394355535507202, -1.5651297569274902, 1.0727084875106812, -0.9977736473083496, 0.748334527015686, 0.11492107808589935, -0.9231402277946472, 1.1127206087112427, -0.8655876517295837, -0.6676025390625, -0.7185642719268799, -2.0570108890533447, 1.2235665321350098, 1.177120327949524, 0.9500948786735535, 1.302142858505249, 0.1995377391576767, -0.0703236311674118, -0.3620489239692688, -2.3928415775299072, -0.471644788980484, -1.36749267578125, 0.27826356887817383, -0.34365686774253845, -0.6896663308143616, 2.0556387901306152, 1.4037052392959595, -1.4311405420303345, -0.7801169753074646], [-1.1276166439056396, 0.37849631905555725, 1.404077172279358, 0.199984610080719, 1.428297758102417, -0.9600772261619568, -0.6045684218406677, -0.3114866316318512, 1.6842081546783447, 0.8766084313392639, -1.5829392671585083, -1.8379833698272705, 0.14354164898395538, 2.5281567573547363, 0.4872848093509674, -1.3720816373825073, 0.6900500655174255, -0.5304628014564514, 0.2282615303993225, -0.17536352574825287, -0.704908013343811, -1.8050163984298706, 0.9333113431930542, 0.8861243724822998, 0.47375452518463135, -0.8122588396072388, 0.02630838192999363, 0.35514122247695923, 0.793310284614563, -0.2562713921070099, -0.43167608976364136, 0.46941113471984863, 0.4659854769706726, -1.027803659439087, 1.4261343479156494, -1.0484458208084106, 1.1680973768234253, -0.10461879521608353, 1.3379404544830322, -1.211647868156433, 0.3225407004356384, -0.72404545545578, -1.780713677406311, 0.004044204950332642, -2.195261001586914, 1.1937131881713867, 0.8441624641418457, 0.04979388043284416, -0.6749763488769531, -0.434933066368103, 0.45120343565940857, -1.1096659898757935, -1.8224607706069946, 0.6518586874008179, 0.5712243914604187, -1.8422540426254272, -0.026415279135107994, 0.7701865434646606, 0.49843984842300415, -0.7648103833198547, 1.0572576522827148, 1.7817885875701904, -0.24257595837116241, 0.2695053815841675, 0.17351646721363068, -0.04560994356870651, 0.5797958374023438, -0.36305636167526245, -0.6674108505249023, 1.2737767696380615, -1.1884862184524536, 0.26695647835731506, -0.2849574685096741, 0.032123174518346786, 1.7536391019821167, -0.40163514018058777, 0.8106917142868042, -0.18085910379886627, -1.0852887630462646, 0.3628406524658203, -0.9850319027900696, -1.241877555847168, -0.3765660226345062, -0.5654336810112, -0.7326958179473877, 0.453966349363327, -0.7706189751625061, 1.4873079061508179, -0.5993706583976746, 1.320911169052124, 0.22457918524742126, -0.7080249786376953, -0.43538567423820496, 0.24453973770141602, 0.09640033543109894, 0.3080190420150757, 0.8482436537742615, -1.372412085533142, 0.46271756291389465, -0.02721773460507393, 0.29487350583076477, 0.5781142115592957, 0.25401151180267334, -0.7319478988647461, -1.304112195968628, 0.13669995963573456, 1.1981714963912964, -1.4333159923553467, -0.42633089423179626, -1.0035648345947266, -1.2278425693511963, 0.6261162757873535, 1.204463243484497, -0.23464658856391907, 0.48835423588752747, 0.29312482476234436, -0.15721972286701202, 1.6718108654022217, 1.9840270280838013, 0.1997089385986328, -1.402496337890625, -0.6800618767738342, -0.11016378551721573, -0.49447256326675415, 0.5096887350082397, -0.05891703441739082, -0.6223556995391846, -1.085404396057129, 0.8641399145126343], [-0.39039334654808044, -0.3452511727809906, 0.010247800499200821, -0.40395399928092957, 1.4366084337234497, -0.4561362862586975, 2.27231764793396, 0.19920338690280914, -0.3403145670890808, -0.21327655017375946, 0.9967445731163025, 0.16593237221240997, -0.5066375732421875, -1.0967965126037598, -1.446084976196289, 0.7050713300704956, 0.6116718053817749, -3.204292058944702, 0.7087999582290649, -0.4652155637741089, 0.34195414185523987, 0.8231741786003113, 0.8584583401679993, -0.7230868935585022, -0.9535940885543823, -0.4099027216434479, -0.8020817041397095, -1.0497092008590698, -0.8078845739364624, -0.4879570007324219, -1.4327125549316406, 1.0401501655578613, 0.11658415198326111, 0.8974524140357971, -1.435395359992981, -0.2973279058933258, 0.014683464542031288, -0.9042988419532776, 2.8142244815826416, 0.7206465601921082, 1.5785911083221436, 0.5461752414703369, 1.3446991443634033, 1.0277329683303833, 0.6910945773124695, 1.3505570888519287, 1.833315134048462, -0.3968615233898163, 0.18989329040050507, -0.20961833000183105, -0.8406546115875244, -0.621143102645874, -0.022349920123815536, 0.9904829263687134, 1.0327119827270508, 1.6076419353485107, -0.9762768745422363, -0.5261813998222351, 0.05277794227004051, 0.11936813592910767, 2.9119231700897217, 0.3247216045856476, 0.16933001577854156, -0.012340126559138298, -0.8896982669830322, 0.42448335886001587, -0.7005100250244141, 1.7151111364364624, -1.682310700416565, 0.8209774494171143, -0.9028005599975586, 0.8181421160697937, -1.0369821786880493, 0.648577868938446, 0.6784617900848389, 0.021639278158545494, -0.7151385545730591, 0.37750282883644104, -0.9819225668907166, 0.08393456786870956, -0.5599335432052612, 1.1651993989944458, 0.10661096125841141, -0.08983112126588821, -0.535686194896698, -0.12841887772083282, 0.8538742661476135, 1.04806387424469, 0.04962940886616707, 0.46420538425445557, -0.20912609994411469, -1.1780322790145874, 0.5885878801345825, -0.16594412922859192, -0.2397843599319458, 1.1015092134475708, -0.17667576670646667, 0.8762695789337158, -1.3508212566375732, 1.5962319374084473, 1.4090675115585327, 0.44344362616539, 1.4769765138626099, -0.46855485439300537, 1.043060541152954, -0.8710699081420898, -0.7662305235862732, -0.4452545940876007, -0.16135631501674652, -1.3204371929168701, 0.6925656199455261, -0.030187930911779404, -0.3426457345485687, 0.9066972136497498, -0.772139847278595, -1.2082526683807373, -0.441774845123291, 0.6270871162414551, -0.3107500970363617, -0.21218596398830414, 1.4402544498443604, -1.6416057348251343, 1.781725287437439, 0.4752655625343323, 0.26487040519714355, -0.5863445401191711, -0.3031580448150635, -1.392621636390686, -0.23559421300888062], [-1.4383655786514282, -0.234538272023201, 0.9396365284919739, -1.1574598550796509, 0.47625312209129333, 0.19389455020427704, 0.33882978558540344, -0.21988755464553833, -1.134265661239624, -1.9594213962554932, -1.7244563102722168, -1.6514698266983032, 0.9527612924575806, -0.3836883306503296, -0.7847044467926025, 0.5284463763237, -0.02969534322619438, 0.966913104057312, 0.07162991911172867, -1.6964432001113892, -0.0724480003118515, -0.7946916818618774, 0.745873212814331, -0.05607711896300316, -0.7530508637428284, 0.5679979920387268, -0.24920198321342468, 0.5564383864402771, 0.013257751241326332, -1.335055947303772, 0.4854486882686615, 2.442070484161377, -1.5804142951965332, -0.778079092502594, -0.41276729106903076, 0.19968052208423615, 2.3968393802642822, -0.5058404803276062, -1.1211762428283691, -1.775544285774231, -1.557727336883545, 0.5568225979804993, -0.6053005456924438, -1.8208012580871582, -1.0428194999694824, -1.1984784603118896, -0.14709420502185822, 0.5897514224052429, 0.040455158799886703, -1.4708752632141113, -0.1330866813659668, 2.2082126140594482, -1.9076250791549683, 1.063339352607727, 1.7944607734680176, -1.2167954444885254, -0.8860523104667664, 1.1985082626342773, 0.5232975482940674, -0.44601014256477356, 0.8345984220504761, -0.23672132194042206, -0.2520560026168823, 0.23583409190177917, -0.30594193935394287, -0.8336259126663208, -0.9849643707275391, 0.020484646782279015, -0.4353785514831543, -0.3224315047264099, -0.6672434210777283, 0.7739786505699158, 1.8076972961425781, 1.3301587104797363, 0.07055319845676422, -0.6962133646011353, 0.39347314834594727, 1.019025444984436, 0.18997199833393097, -1.065674901008606, -0.6029911637306213, 0.9167760610580444, -0.395389586687088, -2.756700038909912, -0.5180688500404358, -1.6583199501037598, 0.9826561212539673, -0.010426145978271961, 0.290505588054657, 0.15271815657615662, 0.5075893998146057, -0.45177701115608215, 0.6991907954216003, 1.6599897146224976, -0.48831796646118164, -0.5284322500228882, -1.0639454126358032, -0.028645800426602364, -0.9979474544525146, 1.5264298915863037, 0.6074068546295166, 0.1238347589969635, -0.6719650030136108, 1.2378815412521362, -1.605826735496521, 1.1041816473007202, 0.5037644505500793, -0.7814710140228271, -0.5019434690475464, -1.0435373783111572, 0.7458170652389526, -0.6329835057258606, -0.5702340602874756, -1.6835054159164429, 2.076922655105591, -0.18325412273406982, -1.1849501132965088, 0.5027806162834167, 0.02197120152413845, -1.4003039598464966, 0.9295525550842285, -0.3998274803161621, 0.9289913773536682, -0.29268574714660645, -1.1999971866607666, -1.5341250896453857, 0.13634413480758667, -1.1335302591323853, 0.4654413163661957], [1.5076484680175781, -0.5064120888710022, -0.03665253147482872, 1.6256964206695557, 0.08073391765356064, 0.8774537444114685, 0.04331201687455177, 1.0070116519927979, -1.1624042987823486, -0.925839900970459, 0.5448285341262817, -1.0537632703781128, -1.789209008216858, -0.5407330989837646, 1.1219319105148315, 0.35213717818260193, -0.28621017932891846, 0.5364506244659424, 0.5839759111404419, 0.5992943048477173, -0.08983343094587326, 0.020870391279459, 0.16443197429180145, -0.6921647191047668, 0.7090577483177185, 1.0796921253204346, 0.23681411147117615, 0.0646040141582489, -0.5999570488929749, -0.9556078314781189, -0.3321513533592224, -1.0549741983413696, 0.13644511997699738, 0.7469517588615417, -1.2027925252914429, 0.9332366585731506, -0.09058436006307602, -0.2513681948184967, -0.14588609337806702, -0.7232341766357422, 0.3830891251564026, 1.9687554836273193, -0.4329865574836731, -0.9216610193252563, 1.1492822170257568, 0.7887910604476929, -0.48973870277404785, -0.06433373689651489, -0.5591898560523987, -0.40536120533943176, 1.169451117515564, -1.0766785144805908, -0.1631164699792862, 2.4292778968811035, 1.62875235080719, -1.2219939231872559, -1.1142686605453491, -1.9674267768859863, 0.7217718362808228, 1.5226279497146606, -0.2869842052459717, 0.10754958540201187, -0.2210642397403717, 0.9213010668754578, 1.3575668334960938, -0.36567360162734985, 0.22734926640987396, 1.908542513847351, -1.328208327293396, 0.9764220118522644, 0.9879270792007446, 0.15749889612197876, 0.48525527119636536, -0.6497891545295715, -0.18487495183944702, 0.6269769668579102, -0.24679994583129883, 0.1299256533384323, 1.0660083293914795, 0.6810342073440552, 1.0732797384262085, 0.37934938073158264, 0.9546821117401123, 1.318350911140442, -0.01621854119002819, -1.0065231323242188, 0.9499515891075134, -0.7474905252456665, 0.13023804128170013, 0.34679242968559265, 1.7835783958435059, -0.11510070413351059, 1.5033092498779297, 1.137699842453003, 0.8947808146476746, -0.24617882072925568, -0.0022919790353626013, 1.7882320880889893, 0.9352039694786072, -1.2390861511230469, -1.8919997215270996, -1.5119861364364624, 0.37170225381851196, 0.4758858382701874, 0.23737512528896332, -0.9593719840049744, -0.5628800988197327, -0.6441649794578552, -0.33965790271759033, 1.6898361444473267, -0.5673736929893494, -0.2257298082113266, -0.04238768294453621, 1.4963687658309937, -0.28844401240348816, -1.0120153427124023, -2.174548625946045, 2.005784034729004, -1.2877854108810425, 0.5550990700721741, 0.06257197260856628, -1.5826561450958252, 0.3539443016052246, -0.25166359543800354, 0.2427731454372406, -1.0816609859466553, 1.0101441144943237, 1.9193453788757324, -2.412994146347046], [-0.07425171136856079, 0.6248031854629517, -0.12267295271158218, 0.5522066950798035, -0.6935552358627319, -0.5138996839523315, -2.087815523147583, -0.5591131448745728, -0.35373973846435547, 0.15396824479103088, 0.008032209239900112, 0.4595632553100586, 0.7824262976646423, 1.1620075702667236, 0.8145173788070679, -0.3738742470741272, -1.4545135498046875, -1.8385214805603027, 1.2467477321624756, -0.7635887265205383, 1.2418038845062256, 0.3296020030975342, -0.10214363038539886, 1.3389782905578613, 0.515543520450592, 0.7313615679740906, 0.7509662508964539, 0.06942079216241837, 0.7366068959236145, -1.8979617357254028, 0.6890767216682434, 0.16413943469524384, -0.5200825929641724, 0.5680971741676331, 0.9270528554916382, 0.8993439078330994, 0.15689677000045776, 0.2524382174015045, -0.08044710010290146, -1.5577120780944824, 0.13513024151325226, 0.027914676815271378, 1.3042593002319336, -1.0610584020614624, 2.515453577041626, -0.9113220572471619, -0.37791723012924194, -1.7988686561584473, 1.03005850315094, -0.6315466165542603, 0.33151891827583313, 0.028232693672180176, 0.6069990396499634, -0.885381281375885, -0.06935171037912369, -0.7212300896644592, -0.20397815108299255, -1.836161732673645, 0.7735525369644165, -0.08417300879955292, 0.0450642928481102, -1.5073554515838623, 0.16948473453521729, -0.38788506388664246, 0.7596845626831055, 0.3734694719314575, 1.3995122909545898, -0.3677671551704407, -0.9217055439949036, 1.9124784469604492, -0.09100951999425888, -0.01927228644490242, 1.2918843030929565, 1.284528374671936, -0.282661497592926, -1.5864661931991577, 1.3963115215301514, -0.5870968103408813, -0.9792563319206238, 0.07704603672027588, 0.913249135017395, 0.5740957856178284, 0.07164883613586426, 2.585669755935669, 1.663401484489441, 0.5247824192047119, 0.039530377835035324, -1.5703057050704956, 0.8728836178779602, -0.7029204368591309, 1.415280818939209, 2.5250136852264404, -2.5741374492645264, 0.461270272731781, 0.5634877681732178, -0.7696706652641296, 1.4359346628189087, -0.7096710801124573, -0.6572487354278564, -1.0948396921157837, -0.4380410313606262, -2.5407164096832275, 1.3477526903152466, -0.48750853538513184, 0.004233963321894407, 0.8528311252593994, 0.21323630213737488, 0.5259265899658203, 1.4972760677337646, -1.5930819511413574, 1.2442569732666016, -1.3555670976638794, -1.224032998085022, 0.20476174354553223, 0.9528970122337341, -1.6703760623931885, 0.8010985851287842, 0.13455121219158173, 0.5165167450904846, -0.0064842491410672665, -0.038572028279304504, 1.0294619798660278, -0.01724317856132984, 0.4044431447982788, 1.8157234191894531, 0.4913894236087799, 0.4827728271484375, -0.25855040550231934, -0.22830767929553986], [2.023625373840332, 1.1287732124328613, 0.9504954218864441, -0.3213304579257965, -0.40404999256134033, 0.2523255944252014, 1.6629161834716797, -1.2190768718719482, 0.008495720103383064, -0.5485209226608276, -0.38206928968429565, -0.7575385570526123, 0.12848153710365295, 0.8104844093322754, 0.3528554141521454, -0.6116047501564026, 0.010596588253974915, 0.9525073766708374, -1.1474695205688477, -0.6414288878440857, -0.27942803502082825, 1.2926515340805054, -1.8245570659637451, 1.3844197988510132, -1.7409049272537231, 1.2382475137710571, -1.2437175512313843, -0.8680654168128967, 0.7324641942977905, 0.6785104274749756, -0.14077574014663696, -0.9502944946289062, 1.6194956302642822, 0.6430041193962097, 0.47806626558303833, 0.16218915581703186, 0.11915940791368484, 0.09019943326711655, 1.2396047115325928, -2.0719470977783203, 0.2529658079147339, 0.9953498840332031, -2.3657002449035645, 0.1708335429430008, -0.6593248248100281, 2.9596619606018066, 0.7388351559638977, -0.8331257104873657, -0.07334323227405548, 0.37688207626342773, -1.2427610158920288, -0.9495494365692139, 0.21551784873008728, 1.8609111309051514, -1.0425478219985962, 3.0695085525512695, -1.0026835203170776, -0.6013523936271667, -0.37088966369628906, 0.46884027123451233, 0.08222027122974396, 0.6223254799842834, -1.2184497117996216, -1.1524200439453125, -0.10504722595214844, -0.220860093832016, -1.558840274810791, 0.2011890411376953, -0.6635972857475281, 0.6281622648239136, -0.08611427992582321, -1.2038354873657227, 1.5349229574203491, 0.8916401863098145, 1.3015879392623901, -1.500864863395691, 1.2872053384780884, -2.079409599304199, 0.2950136661529541, -0.7412868738174438, -0.8972824215888977, 0.5714971423149109, 1.2930837869644165, -0.44030630588531494, 0.9819650053977966, -0.79868084192276, 0.6664026975631714, -1.695385217666626, 0.11695791035890579, -1.263729453086853, -0.45377424359321594, 1.0687284469604492, 1.2894651889801025, -0.3883975148200989, 1.020982265472412, -0.635219931602478, 0.7060661911964417, 0.6914399266242981, -0.17875447869300842, 1.130799651145935, 0.8332325220108032, 1.3426498174667358, 1.3000462055206299, 0.6108912229537964, 0.11297125369310379, -0.44527000188827515, -1.9953763484954834, -0.10163626819849014, 0.29804903268814087, -1.0851560831069946, -0.5853840112686157, -2.2035298347473145, -0.10238847881555557, -1.7494374513626099, -0.009196986444294453, 0.1152564287185669, 0.061891380697488785, -0.8486163020133972, 0.5282934308052063, -0.7219744920730591, -1.1937624216079712, -0.13192258775234222, 1.1595702171325684, 1.2644405364990234, -0.25478798151016235, -0.5424361824989319, -0.17728540301322937, 1.9382137060165405, -0.2606799602508545], [1.0210366249084473, -0.09694886952638626, -1.3241302967071533, 1.0898525714874268, -0.5080885887145996, 2.149488925933838, -1.8094381093978882, 1.5798481702804565, -0.19761179387569427, -1.3092347383499146, -1.0434025526046753, -0.19610978662967682, 1.5703102350234985, -0.04772142693400383, 0.41535964608192444, -0.3595741391181946, 0.7264901399612427, 0.338672935962677, -0.9819627404212952, -1.1896321773529053, 1.197836995124817, -1.0817683935165405, 1.252665638923645, -0.1494978368282318, -0.34118515253067017, 1.198257565498352, 1.3160877227783203, 1.474272608757019, 0.5700857043266296, 1.2502113580703735, 1.2935709953308105, 1.9106295108795166, 0.7888050079345703, -0.06900718063116074, 0.430839478969574, 1.294560432434082, -1.4343153238296509, -0.8260708451271057, 0.660294234752655, -0.2053278237581253, -1.3961951732635498, 2.732131004333496, -0.1623314470052719, -0.19982247054576874, 0.6245294213294983, 0.5949131846427917, 0.1686534881591797, 0.17116773128509521, -0.780353844165802, 0.30812257528305054, -1.4986181259155273, -2.006943464279175, -0.8294796943664551, 0.9483111500740051, -0.2435598373413086, -0.1710808128118515, 1.0431591272354126, -1.6790134906768799, 0.783602774143219, -0.5846010446548462, -0.11491439491510391, 0.17470170557498932, -1.9374264478683472, 0.5339526534080505, 0.9968417882919312, -0.26737794280052185, 0.5266982316970825, -0.5002529621124268, 2.0284311771392822, -0.5550891160964966, -1.5881768465042114, -0.4327341616153717, 1.7024072408676147, -1.6150848865509033, 2.256800651550293, 0.6401876211166382, 0.595108687877655, 0.10209985077381134, -1.3870134353637695, 0.1218743547797203, -0.6325118541717529, 1.4311617612838745, 0.009027483873069286, -0.14802691340446472, -0.8049952983856201, -0.48073723912239075, 1.0851736068725586, -0.5514785647392273, -0.05009256675839424, -1.15656316280365, 1.2956606149673462, 0.8662887215614319, -1.464874267578125, 1.680605411529541, 0.47538134455680847, -1.337950587272644, 0.14304035902023315, 1.3425347805023193, 0.44870540499687195, 1.0467875003814697, -0.14101208746433258, -0.5412552952766418, 1.6699236631393433, 0.891418993473053, 1.1636027097702026, 0.6235331296920776, -0.8743899464607239, -0.98067307472229, -0.2925677001476288, -1.1168968677520752, 0.33940690755844116, -1.7825599908828735, -1.662623405456543, 0.3363645374774933, 0.41086089611053467, 0.7472764253616333, 0.463191419839859, -1.3459746837615967, 0.8467383980751038, 1.5774363279342651, -0.11062590777873993, -1.5863579511642456, 1.727102279663086, 0.8199213743209839, -1.6470129489898682, 0.38373786211013794, 1.5486139059066772, -0.6260678768157959, 0.5373551249504089], [-1.7217926979064941, -0.35269156098365784, 0.2641475796699524, 0.7938088774681091, 1.0088002681732178, 1.3774312734603882, -0.2292451709508896, 0.8463097810745239, -0.8885288238525391, 0.4482590854167938, 2.5148279666900635, 1.2003734111785889, -0.5356765985488892, 0.8287320137023926, 1.0758650302886963, 1.1870678663253784, 1.160625696182251, 0.39978766441345215, -0.950295090675354, 0.539972722530365, -0.6698821187019348, -0.23425905406475067, -1.0568479299545288, -0.23291786015033722, 2.582787036895752, -0.019962245598435402, 0.42368799448013306, -0.4755063056945801, 0.3739500939846039, 1.3606188297271729, 1.0980089902877808, -0.14000655710697174, -0.7331573367118835, -0.7279837727546692, -0.8614044785499573, -0.6333882808685303, 1.1375367641448975, -1.3501569032669067, 0.07771583646535873, -0.09953085333108902, 0.4638332724571228, 0.29471927881240845, 0.48777592182159424, -0.8262292146682739, 0.8513886332511902, -2.731004238128662, 1.2449584007263184, 0.1535685658454895, -0.45782530307769775, 0.6661826968193054, 0.32415831089019775, 0.15378904342651367, -1.367722749710083, -1.4201828241348267, -0.21838310360908508, 0.3464730978012085, -0.9212077856063843, 1.1156376600265503, 1.301342248916626, 0.4544356167316437, 1.1230716705322266, 0.01824154518544674, -0.9373900294303894, -0.7939706444740295, -0.006686138920485973, 0.8510489463806152, 0.28335028886795044, 1.5880396366119385, -1.3658726215362549, -0.3792663812637329, -0.6512212753295898, -0.12903542816638947, -0.6278233528137207, 0.945219099521637, -0.9735230207443237, 1.8895771503448486, -0.3999005854129791, 0.9306434392929077, 1.3744088411331177, -1.6734399795532227, 0.5965462327003479, -0.1842932105064392, -0.18127232789993286, 0.03969555348157883, 0.5351402759552002, 0.20353294909000397, -1.202046513557434, 0.4718325734138489, 0.9106723666191101, 0.3542822003364563, -2.036234140396118, -1.1318856477737427, 0.1829068958759308, 0.12786905467510223, 0.4671914875507355, -0.16582465171813965, -0.3169746696949005, -0.25294381380081177, -0.27751514315605164, -0.32252851128578186, -1.2105004787445068, 0.35594284534454346, -0.05742014944553375, 0.9864324331283569, -0.7911025881767273, -0.4552868902683258, 0.03420182690024376, 0.2615335285663605, 0.7134597897529602, -0.6733894944190979, 1.6235729455947876, 0.32927948236465454, -0.9600904583930969, -0.3345820903778076, 0.0800459235906601, 0.5372108221054077, 0.9540427923202515, -0.18903914093971252, -0.23470492660999298, -2.460451126098633, -1.4743238687515259, -2.286038875579834, -1.3672716617584229, 0.9923284649848938, -0.23819054663181305, -1.8466516733169556, -1.2558605670928955, 1.1892451047897339, -1.2182656526565552], [-2.3865952491760254, -1.5281928777694702, -0.30405110120773315, -0.09336363524198532, 1.0102107524871826, 0.20133161544799805, 0.23820984363555908, -1.1146668195724487, -0.11364538967609406, -1.1201128959655762, -1.2427382469177246, -1.193400502204895, 0.5786491632461548, 0.4139586389064789, 0.22437992691993713, -0.8011009097099304, 2.2801430225372314, -0.5465874671936035, 0.3320136070251465, 0.17737919092178345, 1.279810905456543, 0.201491117477417, 0.5032233595848083, 0.15513251721858978, -0.6533202528953552, -0.8283049464225769, -0.7101232409477234, -0.02668105624616146, 0.32756292819976807, -1.6482501029968262, 0.15369045734405518, -0.20498822629451752, -1.4310121536254883, 0.1089056208729744, -0.07290475815534592, -0.8081399202346802, -0.4416898787021637, 0.8858351707458496, 0.4885101020336151, 0.15674203634262085, -1.4052622318267822, -0.016857219859957695, 0.5611541271209717, 0.32716071605682373, 2.267592191696167, -0.3115912973880768, 1.7034573554992676, 1.3555256128311157, -0.6566774249076843, -1.416486144065857, 1.8148385286331177, 0.2349185049533844, -0.5137593150138855, 1.7250043153762817, -0.3790135979652405, -0.5792677998542786, 0.20893405377864838, 1.2940045595169067, -1.4123317003250122, -1.3513754606246948, -0.47909852862358093, -0.03563309833407402, -3.079108476638794, 0.5699110627174377, -1.8927607536315918, -1.34308660030365, 1.507485270500183, -1.3770533800125122, 0.13995899260044098, -0.07511723786592484, 0.6347888708114624, -0.7287178039550781, 0.2478320449590683, -0.39548733830451965, 0.25298118591308594, -0.6808925271034241, 0.16472695767879486, -0.11011616885662079, -0.02292989008128643, -0.7030099630355835, -0.9400321245193481, 0.7522429823875427, -0.24870029091835022, 0.1594173163175583, 1.0887068510055542, -0.5367884039878845, 0.6048467755317688, 0.03993862494826317, 0.16258737444877625, 1.7576217651367188, -0.05938231572508812, 0.4507503807544708, 0.28927165269851685, 0.7992228865623474, -0.7766356468200684, 1.1726669073104858, -2.908247232437134, -1.9309208393096924, -1.9718704223632812, 0.18603231012821198, 0.48652783036231995, -0.7435137033462524, 1.3025476932525635, 0.4460150897502899, -1.621446967124939, -1.3813098669052124, -0.6593946218490601, -0.40094518661499023, 0.30865582823753357, -1.1282553672790527, 0.7040590047836304, -0.20471906661987305, -1.025826096534729, 1.2809090614318848, 1.696738839149475, -0.7913632988929749, -0.23559395968914032, 0.1950809508562088, 0.07526300847530365, -0.399596244096756, -0.501977264881134, 0.2722618281841278, 1.2406359910964966, -1.6996490955352783, -0.8831876516342163, 1.3079922199249268, -0.002754588145762682, -1.2234611511230469, -0.6323530077934265], [0.39639756083488464, 0.2808319628238678, 0.8025604486465454, -0.13772372901439667, -0.6223822236061096, -1.7346816062927246, -0.23388442397117615, -0.3094048798084259, 1.6924761533737183, 1.099288821220398, -0.29202547669410706, -1.2322918176651, -1.6597681045532227, -2.5812129974365234, 0.09873419255018234, 1.0825341939926147, -0.12797869741916656, 0.5760104656219482, -0.7347585558891296, 1.5413365364074707, 2.900867223739624, 1.0198084115982056, 0.5336171388626099, 0.648688554763794, -0.2522176206111908, -0.3690207302570343, -0.36759787797927856, -0.19075629115104675, 0.04568437486886978, 0.37494274973869324, -0.07843216508626938, 0.7756830453872681, -1.225924015045166, -0.29202815890312195, -0.15126442909240723, 0.2792605459690094, 0.05299634858965874, 0.4410899579524994, -0.4631665349006653, -0.9256609678268433, -0.2636289596557617, 0.25091806054115295, 0.6135880351066589, 1.8305225372314453, 0.09402208775281906, -0.6963977813720703, 0.3966754674911499, -0.35713711380958557, 0.6692546606063843, 1.5470770597457886, 0.1739947646856308, 2.0205330848693848, -0.20423291623592377, -0.600510835647583, -2.3070366382598877, -0.0333220511674881, -1.124167799949646, -0.3891768157482147, -0.07029165327548981, -1.7246733903884888, -2.0060904026031494, 0.943142294883728, -0.4374552369117737, 1.2715225219726562, -1.5204585790634155, -1.4625221490859985, 0.7939441800117493, 1.5661675930023193, -1.7206623554229736, 0.370717316865921, 1.9580636024475098, 1.1676907539367676, 0.8617788553237915, -0.145808145403862, -0.8116961121559143, 0.695666491985321, 0.30110472440719604, -1.312834620475769, -1.7977564334869385, -1.3465451002120972, 0.2525506615638733, 0.35388144850730896, -0.9282721281051636, 1.2973356246948242, 0.6192271709442139, 0.5530564785003662, 0.01523431297391653, -0.6733594536781311, -1.6926857233047485, -1.1980000734329224, -0.4169105887413025, 0.20404992997646332, -0.39836615324020386, -0.10750015825033188, -0.025327177718281746, 0.4683438837528229, 0.5677609443664551, -0.38660186529159546, -0.013877090066671371, -0.22651778161525726, -1.338182806968689, 1.814509391784668, -0.3571455776691437, -0.7544605731964111, -1.4370099306106567, 0.22164376080036163, -0.7190577983856201, 0.2018260359764099, -0.29135093092918396, 0.4902902841567993, -0.3163573741912842, -1.154497742652893, -0.12503516674041748, 0.8358052968978882, 0.21506954729557037, -0.038539819419384, 1.0261098146438599, 0.3241797387599945, 0.7366626858711243, -0.5805420875549316, -0.24986517429351807, -0.40451061725616455, 1.3431590795516968, 1.2049542665481567, 2.3931453227996826, 1.052734375, -2.11016583442688, -0.35664448142051697, 0.5735622644424438], [2.6805408000946045, -1.0828860998153687, 0.9085518717765808, 0.37177738547325134, -2.2620859146118164, -0.4565020799636841, 0.8577795028686523, 0.13974790275096893, -1.302291750907898, -0.3273810148239136, -0.2503725290298462, 1.5328433513641357, 1.130527377128601, 2.7389111518859863, 0.4551411271095276, 0.9120491743087769, -2.074742555618286, -1.194266438484192, -0.47140321135520935, -0.25482386350631714, -1.6001490354537964, -0.24328923225402832, -0.6412392854690552, 0.9958197474479675, 0.4774041175842285, -0.5823957920074463, 0.03562336787581444, 1.473889708518982, -1.6002779006958008, 1.6811456680297852, -1.8309788703918457, -0.0925496369600296, -0.9463788866996765, -0.05849919468164444, -0.07759766280651093, -1.3099380731582642, 0.2444406896829605, -0.4748769998550415, -0.44093573093414307, -0.005892532877624035, 0.026996798813343048, -0.040610987693071365, -0.07743275165557861, -0.00610736757516861, -0.5343579649925232, -1.1474663019180298, 1.0248748064041138, -1.1753923892974854, 0.18865621089935303, 1.6130433082580566, -0.37643805146217346, -0.20293673872947693, -0.02065340057015419, 0.5611531734466553, 0.32134750485420227, 3.049879789352417, -1.8152999877929688, -0.5980192422866821, -0.053971026092767715, 0.6172769665718079, 0.6208365559577942, 0.09622886776924133, 1.397546648979187, -0.7298697233200073, 0.21117407083511353, 0.5489959716796875, -1.6993257999420166, 0.17329168319702148, -0.6527122259140015, 1.0900779962539673, 1.0411750078201294, -2.4944653511047363, 0.08646582812070847, 1.2421562671661377, -0.8164978623390198, -0.13649694621562958, -0.19122964143753052, -0.40404996275901794, -1.4738469123840332, -1.3818459510803223, -1.0717458724975586, 1.5361511707305908, -0.2045816332101822, -1.0183651447296143, -0.7086054682731628, -0.1874450147151947, 1.213868498802185, -0.37714725732803345, 0.868194580078125, 0.8374942541122437, 0.35336002707481384, -0.5737698078155518, 0.9555114507675171, -1.5975409746170044, 0.11603087931871414, -1.7990069389343262, -1.9545857906341553, 0.23254333436489105, 0.2800348997116089, 0.7068536877632141, -0.07653088867664337, 1.9435601234436035, 0.8218213319778442, -1.1293524503707886, 0.5759851336479187, -1.8131216764450073, 0.16256046295166016, 0.32231926918029785, -0.631483256816864, 0.48838865756988525, 0.7779349088668823, -0.3860090672969818, 0.9442064762115479, -0.2464020997285843, 0.3472314476966858, -1.2579545974731445, 0.7122609615325928, -0.7997972965240479, -0.01723339967429638, 0.8396509885787964, 0.5492438673973083, 0.696499228477478, -1.0063670873641968, 1.2456661462783813, -0.686161994934082, -0.5541301965713501, 1.0225155353546143, -0.2259112298488617, -0.04458273947238922], [-0.3116167485713959, -1.4020273685455322, 0.22121596336364746, 1.0140482187271118, -0.23871177434921265, 1.0938173532485962, -0.5819692015647888, -0.1400458663702011, -0.21436211466789246, -0.21366974711418152, -0.11939262598752975, -1.3310959339141846, -0.47367674112319946, -0.01957208476960659, -1.0060359239578247, -0.688759982585907, -0.1459392011165619, 0.07871163636445999, 2.5157718658447266, -0.8953161239624023, 0.912296712398529, -0.1154838502407074, 0.794211745262146, 0.8282248973846436, -0.47092920541763306, 0.20570972561836243, -0.6239245533943176, -0.26536253094673157, -2.086679458618164, -0.2535620331764221, -0.10640689730644226, -0.41567766666412354, 1.7910341024398804, -0.6783578395843506, -0.18071512877941132, -0.4263913333415985, -1.5680838823318481, 0.5786283016204834, -0.9808509349822998, -0.6695099472999573, -0.26777687668800354, 0.37244337797164917, 1.4266499280929565, 1.9210412502288818, 0.8653546571731567, 1.0973252058029175, -1.2011897563934326, -0.625475287437439, -0.04166017845273018, 0.6538611650466919, 0.6365131735801697, 0.5518949031829834, 1.5948004722595215, -0.060353197157382965, -0.0444914773106575, 1.8929330110549927, 1.3464738130569458, -0.1655263900756836, -0.9369336366653442, 0.288269579410553, -0.4661663770675659, 0.9483988881111145, 0.3482738137245178, 2.5337166786193848, 0.3109531104564667, 0.9531207084655762, 0.854542076587677, 0.013595027849078178, -0.20432107150554657, -0.6521093845367432, -0.6640673875808716, -0.37775957584381104, 0.09676383435726166, 0.3169108033180237, 0.4658055007457733, -1.1773524284362793, 1.1804169416427612, 1.1546744108200073, -0.040454842150211334, 1.7423920631408691, 0.6778358221054077, -1.186145305633545, 2.1305017471313477, -1.1846582889556885, 0.6519148349761963, -0.2020496279001236, 0.08488448709249496, -1.2244054079055786, -1.2545289993286133, -1.3906208276748657, -0.22281138598918915, -1.3516966104507446, -0.1397325098514557, -0.13222090899944305, 0.892008364200592, 1.8342816829681396, -0.9376060962677002, 0.06264781206846237, -1.6696574687957764, 0.2586234509944916, 0.8442199230194092, 0.9026780128479004, 0.04458007588982582, 0.2565670907497406, -0.3833146095275879, -1.266432285308838, 1.0449570417404175, 2.2113733291625977, -0.2403264343738556, -0.9458144903182983, -0.6589712500572205, -0.39668431878089905, 1.958140254020691, 0.5005484819412231, 0.47520437836647034, 0.7718027830123901, -0.6760972142219543, -0.30890971422195435, -1.6954374313354492, 0.7917541861534119, -1.0675303936004639, -0.26217028498649597, -1.0400336980819702, -0.0001572299952385947, 0.8769636750221252, 0.8683261871337891, 0.07199672609567642, 0.7294133305549622, -1.3878707885742188], [0.10751219093799591, -0.5027171969413757, -0.014204532839357853, -0.8049741387367249, -1.4550209045410156, -0.24756689369678497, -0.44921937584877014, -1.1346664428710938, -2.3711509704589844, 1.618134617805481, 0.11697064340114594, 1.0024508237838745, 2.6686387062072754, -1.2428903579711914, 1.141586422920227, -0.45455098152160645, 0.9893040657043457, 0.027794713154435158, 0.7375674843788147, 2.094532012939453, 1.7948201894760132, 0.29260188341140747, -0.4058862030506134, 0.49262750148773193, -0.8969665765762329, 0.8377947807312012, 2.221126079559326, -0.5664206743240356, -0.3523169159889221, 0.5053797364234924, -0.5062692165374756, -0.4171290695667267, 0.440906822681427, 1.3686504364013672, 0.7968143820762634, 0.13690461218357086, -0.5758339166641235, 0.8499322533607483, 0.7134886980056763, 1.667600393295288, 1.011833667755127, 0.06781832873821259, 1.766798496246338, 1.2407735586166382, -0.5846113562583923, 1.0599173307418823, 1.1800832748413086, 0.932349443435669, -0.9177740812301636, 0.817674458026886, 0.45276033878326416, -0.49284395575523376, -0.45103341341018677, -1.0790071487426758, 0.6251714825630188, -0.9778844714164734, 0.3161906599998474, 0.38141369819641113, -1.2179148197174072, -1.224931001663208, -0.3774356544017792, -0.5047041177749634, -1.8490166664123535, 0.6154400110244751, 0.8182476758956909, 0.8107960820198059, 0.602529764175415, -1.9447064399719238, -3.099494218826294, -0.036001674830913544, -1.0489367246627808, 0.39836251735687256, -0.7979001998901367, 0.21972627937793732, 0.9959931969642639, -0.6659238338470459, -0.40119606256484985, 0.8775753378868103, -1.4083536863327026, 0.2929617464542389, -1.3882737159729004, -1.8191916942596436, 0.6260702013969421, -1.6629942655563354, 2.8028576374053955, 0.040683578699827194, 0.9517266750335693, 0.5497757196426392, -0.7598434686660767, -0.014928554184734821, -1.2797298431396484, -1.2901411056518555, -0.5420331358909607, -0.30816859006881714, 2.0328075885772705, 0.1583031862974167, 0.5796157717704773, 0.42030593752861023, -0.428275465965271, 0.7770634293556213, -0.3410487771034241, 0.36571988463401794, -0.6914463043212891, 0.7105944752693176, -2.2096548080444336, 0.5252910256385803, 1.3229906558990479, 0.5025961995124817, -0.8149785995483398, 1.237156867980957, -0.809616208076477, 0.4442882239818573, -0.007788228336721659, 0.6205053329467773, 0.5356705188751221, 0.6261343359947205, 1.1482969522476196, -0.47495758533477783, 1.8088322877883911, 0.14656607806682587, 1.1501708030700684, 0.6063080430030823, -0.006842467002570629, 0.14224053919315338, 0.9932652115821838, 0.1652945578098297, 1.2919483184814453, -0.19959089159965515, -1.444713830947876], [-0.17060159146785736, 0.7956641912460327, -0.7970879077911377, -0.7946209907531738, -0.3134273588657379, -1.056143045425415, -0.5312632918357849, 0.699876606464386, 0.8378916382789612, -0.3527239263057709, -0.5089303255081177, 0.129592627286911, 0.7427326440811157, 1.1970144510269165, 0.6956035494804382, 1.5333389043807983, 0.6295729279518127, -0.11631602793931961, 0.7681041955947876, 0.02131246216595173, -1.6968612670898438, -1.523764729499817, 2.499063730239868, 0.23177815973758698, -0.492217481136322, -2.1958723068237305, 1.4512274265289307, 0.9822646379470825, -0.005330560728907585, -1.5069156885147095, 0.2495870590209961, 0.5860562920570374, -0.6248796582221985, 0.4171220064163208, 0.7933129668235779, 0.30291977524757385, 0.10205807536840439, -0.06901289522647858, 0.774794340133667, -2.13102650642395, -2.009464740753174, -2.3086206912994385, -1.1475367546081543, -0.736056923866272, -2.3743631839752197, -1.2180618047714233, -0.5514796376228333, 0.9122444987297058, -1.0546425580978394, 0.16827794909477234, -0.433673232793808, 0.07877833396196365, -1.0872619152069092, -0.7904099225997925, 0.032573964446783066, 0.25240305066108704, 0.3755755126476288, 1.2465523481369019, 2.0032894611358643, 1.268570899963379, 0.6908366680145264, -0.9728447794914246, -0.530902087688446, 1.227947473526001, 0.46454286575317383, -0.6319096088409424, 1.6148120164871216, 1.6365889310836792, 0.6948015689849854, 1.1513396501541138, 0.9646235704421997, -0.2163521647453308, 1.0896497964859009, 0.28403306007385254, 1.1078805923461914, 1.2648193836212158, -1.3036882877349854, -0.0163815226405859, -0.36279651522636414, -0.2989669442176819, -0.8770635724067688, -0.21281862258911133, -0.6080819964408875, -2.3767244815826416, 1.2459760904312134, -0.3831373155117035, -1.1027231216430664, -0.005614239722490311, -1.7379546165466309, -0.5973263382911682, -1.9429149627685547, 0.5280351042747498, -0.5568037629127502, 1.5447040796279907, 0.5948300957679749, 0.9169999957084656, 1.2504453659057617, -0.7391137480735779, -0.6530927419662476, -1.9990473985671997, -2.8431735038757324, -0.6952422261238098, -0.5548998713493347, 0.24114002287387848, -0.49737292528152466, -1.9718798398971558, 0.87940913438797, -0.2550860643386841, 0.05887775495648384, 0.18757620453834534, -0.5627509355545044, -1.7085926532745361, -1.8079997301101685, 0.7168927192687988, -1.8786598443984985, 0.2146599143743515, 2.4589343070983887, -2.0290091037750244, 1.1506630182266235, -0.7225774526596069, -0.2882043421268463, 0.0563771091401577, 1.4559873342514038, -0.45311692357063293, -0.7291274070739746, 1.0478190183639526, -0.15374167263507843, -1.7833036184310913, -0.7588233947753906], [0.9148740172386169, -0.7891706824302673, 0.9117428064346313, -1.3220916986465454, 1.1008964776992798, -1.8318055868148804, -0.13527509570121765, 0.5704137682914734, -1.697015643119812, -0.578765869140625, 3.2573025226593018, 0.4626418650150299, -0.16636192798614502, 0.17364710569381714, -1.4581753015518188, 1.2947332859039307, -1.2074824571609497, 0.7328172326087952, -0.14435407519340515, -0.2641908526420593, -0.9298271536827087, 0.35253703594207764, -0.6952211260795593, -0.6306249499320984, 0.7302138805389404, -2.6161534786224365, 1.5063526630401611, -1.41900634765625, -0.05055941641330719, 0.8755166530609131, -0.9637709259986877, 1.7629410028457642, 0.12891362607479095, -0.7604292035102844, 0.2538726329803467, -0.48497581481933594, -2.1052961349487305, 0.6199886202812195, -0.41925475001335144, -0.5803526639938354, 1.033806324005127, -2.397871255874634, -1.655700922012329, -0.19339478015899658, -1.1184965372085571, -1.1552660465240479, 0.22051094472408295, 0.5757213830947876, -0.5726429224014282, 1.7415863275527954, 0.8091288208961487, -0.15385282039642334, -0.02375911921262741, 0.7309509515762329, -1.000002384185791, -0.39187583327293396, -0.39076313376426697, -0.5205514430999756, -0.7513901591300964, 0.18021608889102936, 0.0038332168478518724, 0.918547511100769, 1.612642526626587, -0.5414583086967468, 0.02434300258755684, 0.8521891236305237, -0.6643487215042114, 0.5830793976783752, -1.1560122966766357, 0.36299440264701843, -0.7562381625175476, 1.0817238092422485, 0.4418986737728119, 0.8802552819252014, 0.08737051486968994, -0.3597034811973572, 1.6538300514221191, -0.044193316251039505, -0.41175827383995056, 0.08270582556724548, -0.8329707384109497, -0.20685985684394836, 0.008210606873035431, -0.28118860721588135, 0.8642650842666626, -0.17263607680797577, 0.5988568663597107, -0.1020258218050003, -0.39015233516693115, -1.0625964403152466, 0.30021923780441284, -0.156975656747818, 0.0019240794936195016, -0.8963117599487305, -0.4226319491863251, -0.8318935036659241, -1.345583438873291, -2.252751588821411, 0.27932578325271606, 0.9207050800323486, 1.1703251600265503, 1.4304773807525635, -0.3332391381263733, -0.39367759227752686, 0.3941333293914795, 0.2552422285079956, -2.078435182571411, -0.289246529340744, 0.8141608238220215, -0.15467169880867004, 1.4757821559906006, -0.9575480818748474, 1.2841726541519165, 0.20928989350795746, -0.18470220267772675, 1.508289098739624, 0.7725274562835693, 0.8515479564666748, -0.16805796325206757, -1.0782597064971924, 0.8101758360862732, -1.756856918334961, -0.2056441307067871, 1.3062304258346558, -0.5136104226112366, 1.0557615756988525, 2.6896727085113525, -0.04697207361459732, -1.7181766033172607], [-1.9351321458816528, -0.23203764855861664, 0.10090489685535431, 1.5985417366027832, 2.3329758644104004, 1.079380750656128, -0.04095545411109924, -0.18514472246170044, 1.1046909093856812, -0.8490802645683289, 0.9701775908470154, -0.326710969209671, -0.08831773698329926, 0.08114615827798843, 1.3079866170883179, -0.9446919560432434, -0.4749656319618225, 0.33753424882888794, 0.00816721748560667, -0.07875154912471771, 0.040069058537483215, 0.30733802914619446, 0.7537570595741272, -0.8428937196731567, -2.5724728107452393, 1.130571722984314, -0.051434267312288284, -0.620050311088562, -1.2566426992416382, -0.1589421182870865, -0.5864980816841125, 0.5220163464546204, 1.32134211063385, -0.6253741979598999, 0.4747294485569, -0.9955009818077087, -1.9225513935089111, -1.3105525970458984, -1.1814799308776855, 0.8669161796569824, 0.13658615946769714, -0.317060261964798, -1.5695229768753052, -1.9204082489013672, -1.5359686613082886, -0.2635193169116974, -0.5212906002998352, -1.4162088632583618, 0.5747796297073364, 0.9604085087776184, 0.14167733490467072, -0.4249941110610962, -0.5983807444572449, 0.11925706267356873, -0.7672942280769348, 0.8702788352966309, 0.4986012578010559, 0.25052282214164734, 0.10297483950853348, -0.4558924734592438, 0.5248005390167236, 0.1396844983100891, -0.25638046860694885, -0.8777816295623779, 0.3975999057292938, -0.23027117550373077, 0.24519701302051544, -0.23919521272182465, 1.4395860433578491, -0.9839502573013306, -1.17287278175354, -0.04467838630080223, -0.32197195291519165, 1.8863900899887085, 0.7394708395004272, 0.9132821559906006, 1.1944031715393066, -1.1250981092453003, -0.2991529703140259, -0.07032693922519684, 0.3476420044898987, 1.6649136543273926, 0.5919890403747559, -1.3926074504852295, 2.2304892539978027, 0.7074306607246399, -1.4027992486953735, 0.9404647350311279, 0.531106173992157, -0.4783329665660858, 0.08065850287675858, 0.41310784220695496, -2.061096668243408, 0.7604638934135437, 0.5530677437782288, 0.1436004489660263, -1.1370513439178467, 0.411522775888443, 0.638054609298706, 1.0355662107467651, -1.9184832572937012, 0.25451236963272095, -1.00383722782135, 0.47712191939353943, -1.591415524482727, -0.6987689137458801, -2.64085054397583, -0.9090670347213745, 0.9408869743347168, -0.8923376202583313, -1.2879462242126465, -0.9089721441268921, -0.6671612858772278, -1.0259642601013184, -0.31906187534332275, -0.31337979435920715, -0.09427618235349655, 2.0195305347442627, -1.5916697978973389, 1.1672298908233643, 1.1233983039855957, -2.3378543853759766, 1.4291620254516602, 1.7843493223190308, 0.6434676051139832, -1.4762489795684814, -0.839393138885498, 0.03610146418213844, 1.2500625848770142], [0.026992425322532654, -0.9702140688896179, 0.9457728862762451, 1.1133427619934082, 0.6347265243530273, 0.532999575138092, 0.07262077182531357, -0.49361491203308105, -0.9134784936904907, 0.08849591016769409, 1.6434625387191772, 2.870596170425415, -1.0128573179244995, -1.0083764791488647, -0.4120955169200897, -0.8541894555091858, -1.6111863851547241, 1.0797926187515259, -0.4374149441719055, -1.788880705833435, -0.3259948194026947, 1.011179804801941, 0.3478853404521942, -0.40954291820526123, -0.23326823115348816, 0.6979251503944397, 1.4613137245178223, 0.6278119683265686, 0.48786458373069763, 0.38784265518188477, -0.9747788310050964, 0.5406160950660706, -0.18557198345661163, 0.33890536427497864, -0.6209598779678345, 0.08569096028804779, -0.18395867943763733, -2.0770785808563232, -0.44930651783943176, 0.07957075536251068, -0.9491544961929321, -2.363992691040039, -0.22393898665905, 0.7117805480957031, -0.17255286872386932, 0.8694179654121399, -1.4529801607131958, 0.28231507539749146, -0.03413207456469536, -2.9065611362457275, 0.026859311386942863, 0.8699674010276794, -0.6000837683677673, -0.7516281008720398, -1.5248345136642456, 0.9071810245513916, 0.2690786123275757, -0.8193839192390442, 0.6146824955940247, -0.004139048047363758, 1.6491727828979492, 1.635040044784546, 0.4946698546409607, -0.7797179222106934, 1.3505178689956665, -0.33084654808044434, 0.04175329580903053, -0.10707530379295349, -0.11660371720790863, -0.001757543534040451, -0.7418292164802551, -0.4402891993522644, -0.5302639603614807, -1.6707892417907715, -2.206157684326172, -0.28767529129981995, -0.09855640679597855, -0.15212757885456085, 0.13630682229995728, 0.2889065146446228, -0.1822957843542099, -0.45619869232177734, -0.51914381980896, -0.21456044912338257, -0.9446816444396973, -0.9581956267356873, -0.8204489946365356, -0.3470233976840973, -0.34133246541023254, 0.427554726600647, -0.3986442983150482, -0.7129614949226379, -0.5400170683860779, 1.2136543989181519, 0.5875848531723022, 0.9546855688095093, 0.11832187324762344, 0.609622061252594, 0.13452818989753723, -0.3888862431049347, -0.9108496308326721, -0.5685522556304932, -1.3482210636138916, -0.3144364356994629, -0.8442810773849487, 0.8441507816314697, 1.4705241918563843, 0.6458930373191833, -0.316475510597229, 0.45174652338027954, -1.5692862272262573, -0.018696242943406105, -1.9850609302520752, -1.5227816104888916, 0.18291853368282318, 0.4649271070957184, 0.9967927932739258, -0.7458586692810059, 0.9103814959526062, -1.0704981088638306, -0.5030453205108643, -0.00236737378872931, -0.05823429673910141, 1.3542696237564087, -0.5308659076690674, -0.22656767070293427, -0.41860857605934143, 0.014078398235142231, -0.978794276714325], [-0.8136262893676758, -1.760608434677124, 0.9230326414108276, -0.194135382771492, 1.8543726205825806, -0.3964970111846924, 1.84109628200531, 0.15269237756729126, -1.0159021615982056, -0.06854075938463211, 0.8019546270370483, 0.5785476565361023, 1.1736212968826294, -0.24868300557136536, 1.6750530004501343, 0.7023651003837585, -0.11858905106782913, -0.2621724605560303, -0.11667048931121826, -0.6839942932128906, 1.2622573375701904, 0.25650519132614136, -0.7022521495819092, 0.2037121057510376, -0.6965622305870056, 0.04398457705974579, 0.33307310938835144, 1.1777498722076416, 0.17561893165111542, -0.07644902914762497, -0.519553005695343, -0.24342545866966248, -0.6554310321807861, -0.7232974171638489, -0.8617635369300842, 0.46270376443862915, 2.1179471015930176, -1.0667117834091187, 0.5430213212966919, -0.06546791642904282, -0.08879867941141129, 0.6889249086380005, 2.258258104324341, 2.108100414276123, -1.2775626182556152, 1.1752508878707886, -0.26858800649642944, 0.4897449314594269, -0.3580993115901947, 1.056296706199646, -1.3545053005218506, -0.07180235534906387, -0.1941123604774475, -0.6302313208580017, -0.4365255832672119, 1.0856099128723145, -1.1897597312927246, 1.0277812480926514, 0.6330596804618835, 0.665145754814148, -0.10178075730800629, -0.5399975180625916, -0.5842097401618958, 0.3221272826194763, 1.2503491640090942, 0.022532129660248756, 1.0072263479232788, -0.6506731510162354, -0.5005955696105957, 0.02094700001180172, -0.13162292540073395, 1.238916039466858, 0.20763513445854187, -0.2274487316608429, 0.4629117548465729, 1.4058853387832642, 0.9619524478912354, 2.018378973007202, 0.7115762233734131, -1.8607988357543945, -0.4372311532497406, -0.07537852972745895, -0.043419186025857925, 0.9867547750473022, 0.5709963440895081, 0.12456543743610382, 1.3394256830215454, -0.7616949081420898, 0.20393522083759308, -1.5983284711837769, 1.5384327173233032, -1.628637433052063, 0.790250837802887, 0.14234767854213715, 0.17813700437545776, 0.26763445138931274, -0.2486717402935028, 2.560783863067627, 0.056146953254938126, -0.44647035002708435, 2.3085029125213623, 0.8312559127807617, -1.0765243768692017, 0.5108767747879028, 0.016105730086565018, 0.8846461772918701, -0.10490404814481735, -0.03300897777080536, 0.04350590705871582, -0.713505744934082, 1.7202537059783936, 0.4363335967063904, 0.3179163634777069, 0.23416756093502045, 0.23388616740703583, -0.37043264508247375, 0.9752125144004822, 0.0707060769200325, -1.0877293348312378, 0.020539984107017517, 0.334441602230072, 0.8804355263710022, -0.5239914655685425, -1.1610016822814941, 0.648776113986969, 0.48138657212257385, 1.0220578908920288, 0.6282549500465393, 0.5678902268409729], [-0.5035156011581421, 0.2601912319660187, -0.38289788365364075, 0.7210938930511475, -0.5116230249404907, -0.12795735895633698, -0.41738617420196533, -0.567371129989624, -1.0302369594573975, 2.732481002807617, -0.5835486650466919, -0.7413817644119263, 0.0828709676861763, -0.3650175631046295, -0.954032838344574, 0.9619420766830444, -0.5517721772193909, -0.4006213843822479, 1.1668001413345337, 0.15678982436656952, 0.8477628231048584, 1.0882072448730469, -0.779703676700592, 0.7811871767044067, 0.821588397026062, 0.42472001910209656, 0.5811221599578857, -0.8341260552406311, -0.7919529676437378, -0.4296938478946686, -0.1533612310886383, -0.7512840032577515, 2.2445321083068848, 0.5261974930763245, 0.5739424824714661, 2.017456293106079, 0.5121267437934875, 0.37889206409454346, -0.2617301344871521, -1.7241419553756714, 1.2874488830566406, 0.5318170189857483, -0.48043084144592285, 0.398995041847229, -0.39706316590309143, -1.043758749961853, -0.028799740597605705, -0.8785826563835144, -1.5970276594161987, -1.2327914237976074, -1.443556547164917, 0.3228267729282379, 0.06741727888584137, -0.4308664798736572, -0.963505744934082, -0.3278063237667084, 1.2768646478652954, -0.4710010290145874, -0.5492995977401733, -0.7513494491577148, -0.5296176075935364, -1.2976397275924683, 2.270516872406006, 0.7447002530097961, 0.1381242424249649, -0.20105846226215363, -0.38964492082595825, -1.2416770458221436, -0.9158055782318115, -1.5095359086990356, 1.747780680656433, -0.45351871848106384, 0.46604278683662415, -0.8567407727241516, -0.2213793694972992, 0.7853860259056091, -1.2508374452590942, -1.3876619338989258, -0.3333885073661804, -0.26171210408210754, -0.8985453844070435, -0.7793589234352112, -1.7617743015289307, 0.09709736704826355, -0.5721663236618042, 1.2727090120315552, 0.6729307770729065, 0.03545672073960304, -0.19558528065681458, 0.34568944573402405, 1.592157006263733, 1.3370361328125, -1.4879028797149658, -0.42560669779777527, 0.8021186590194702, -0.8137916922569275, 2.248119592666626, -2.0260841846466064, 0.8920751810073853, 1.2430578470230103, -0.915705144405365, -0.730807900428772, -0.6305858492851257, 1.1624456644058228, 0.27946507930755615, 0.6527602672576904, -0.22197838127613068, 1.5521944761276245, 1.5651586055755615, 0.5719152688980103, -0.6157864332199097, 1.0294398069381714, -0.9250834584236145, 0.6046220660209656, -0.7088370323181152, 0.2663634121417999, 0.2976701855659485, -1.085281252861023, -1.9945625066757202, 0.6690977215766907, -0.22119106352329254, -0.026818830519914627, -0.8251935839653015, -0.06274566799402237, -0.48597100377082825, 0.41506293416023254, -1.1699302196502686, 0.21039345860481262, 0.11799369007349014], [1.9416368007659912, 0.34306710958480835, -1.2467252016067505, -1.7102015018463135, -0.10793334245681763, 0.002329838927835226, 0.26030799746513367, -1.6095508337020874, 0.234112948179245, 0.596012532711029, 1.373480200767517, -0.9495449662208557, -0.8074210286140442, -1.0490702390670776, -1.6734802722930908, -1.1030954122543335, 0.26930269598960876, 0.39689216017723083, -0.7480024695396423, 0.10496405512094498, 0.47224387526512146, -1.955251693725586, 0.703565776348114, 0.523194432258606, 1.632763147354126, -0.36387157440185547, -1.3887604475021362, -0.5935229659080505, -0.6374448537826538, -0.17809510231018066, -0.33899906277656555, 0.7746171951293945, 1.3826427459716797, -1.1139671802520752, -0.8448835611343384, -0.37151989340782166, -0.24445423483848572, -0.48676151037216187, 0.05342193320393562, 1.6474448442459106, 1.0242185592651367, -0.35105499625205994, -0.1009000837802887, 0.08552276343107224, -1.4080352783203125, -1.2246196269989014, -0.9525720477104187, 0.3829038143157959, -1.9717159271240234, 0.5112364292144775, -2.2630093097686768, 0.7705456614494324, 0.9892060160636902, 0.15436811745166779, 0.08186672627925873, 0.6102132201194763, -0.9363414645195007, 0.10389005392789841, 0.4077492952346802, -1.3023117780685425, 0.4234953820705414, 1.2797410488128662, -0.779697597026825, -0.020570574328303337, -0.16760805249214172, 0.10257948189973831, 0.8157662749290466, -0.81275874376297, -0.8418826460838318, -0.492328941822052, -0.5497108101844788, -0.18219971656799316, -1.340492844581604, 0.14804163575172424, -0.6662117838859558, -1.0094897747039795, -0.5723547339439392, -0.8432042598724365, -1.4065333604812622, 0.025737686082720757, -0.8603262305259705, 0.0005241127219051123, -1.8917243480682373, 1.468313455581665, 1.4977617263793945, 1.0766687393188477, -0.17924946546554565, -0.5268059968948364, -1.7120485305786133, 0.2149677574634552, -1.408229947090149, -0.2107544094324112, -1.4210166931152344, 1.4537793397903442, -1.7398167848587036, -0.5750644207000732, 0.6259607672691345, 0.5586163401603699, -0.28230392932891846, -1.5443488359451294, 1.1967294216156006, -0.2510501444339752, 0.09270075708627701, 2.140177011489868, -0.7784269452095032, 1.137518048286438, -0.9127241969108582, -0.27840298414230347, 0.6721411347389221, -0.9310702085494995, 1.4227566719055176, 1.2801847457885742, 0.2884445786476135, 0.3858582377433777, -0.9486112594604492, -0.9106515645980835, -0.6339495182037354, 1.0232129096984863, -0.20707815885543823, -1.4066493511199951, 1.1815450191497803, -1.1324325799942017, -0.4746842384338379, -1.846153974533081, -0.6782811284065247, -2.2237601280212402, 0.5139249563217163, 1.0411616563796997, 1.250095248222351], [0.6165018677711487, 0.2193685919046402, -0.3496083915233612, 0.5990480780601501, 1.905666708946228, 1.2165042161941528, 0.08976922184228897, -0.2144380658864975, -1.5024334192276, -1.2395659685134888, -0.9791186451911926, 1.071802020072937, 1.829628825187683, 2.162367582321167, -0.055288370698690414, 1.056431531906128, -0.22702284157276154, -0.5900145173072815, 0.369754821062088, 0.048989735543727875, 0.46239861845970154, -0.024861931800842285, 0.22854149341583252, -0.4854888916015625, -0.06190476194024086, -0.49269911646842957, -0.10489965975284576, 0.37847307324409485, -0.33806851506233215, 1.529988408088684, -1.8330336809158325, 1.1764813661575317, -0.35610827803611755, 0.7859622836112976, 0.06001923978328705, -0.6581465005874634, -0.039483219385147095, -1.621324896812439, 0.23519591987133026, 1.4851025342941284, -1.4656401872634888, 0.9290000200271606, 0.36942246556282043, 0.7444432973861694, 0.1244279146194458, 1.3810155391693115, -0.37015828490257263, -0.22746476531028748, -0.8484293222427368, 1.6416484117507935, -0.9938154816627502, -1.0406053066253662, -0.4030553996562958, 0.46288180351257324, -0.9727113842964172, -0.6411108374595642, -0.5090878009796143, -0.3537716269493103, -1.3571662902832031, -0.7647987604141235, -1.1753604412078857, -0.5030454397201538, 0.08258042484521866, 1.359846591949463, -1.7628093957901, -0.646583616733551, -0.08932182937860489, -0.2686617970466614, -0.4274901747703552, -2.197979688644409, -0.6066145300865173, 0.2738041281700134, -2.162069797515869, -0.7527307271957397, 0.946913480758667, 0.10281486064195633, -0.4152379035949707, -0.125796839594841, -0.6076302528381348, 0.8417443037033081, -0.5241809487342834, -1.1746516227722168, 0.05660926178097725, -0.03943023085594177, -1.5397450923919678, 1.3705143928527832, -0.40315911173820496, -0.6914716362953186, 1.4884651899337769, -0.014789319597184658, -1.2768803834915161, 0.09146124124526978, -1.064251184463501, -0.5582295060157776, 1.402721643447876, -0.7100245952606201, 0.9872732758522034, 0.6751184463500977, -0.004445141181349754, -0.07099664956331253, -0.3569231927394867, 0.9583266973495483, 1.3125135898590088, -0.9347177147865295, 0.19621917605400085, 0.5095171332359314, 0.7049179077148438, 0.183909073472023, -0.069198839366436, 2.103776693344116, -0.0067765722051262856, -0.995849609375, -0.3757117986679077, -0.6280075311660767, -1.8832147121429443, -1.3433243036270142, 0.7532721161842346, -0.3933638632297516, 0.4450959861278534, 1.9480973482131958, 0.9688999056816101, -0.15530000627040863, -0.4419199228286743, -0.03976590558886528, -0.16414238512516022, -0.1632099598646164, 2.2630536556243896, -0.4226818084716797, -0.10960511863231659], [0.8645660877227783, -0.5200619697570801, -0.19374579191207886, 0.5879859328269958, 0.1587204486131668, 1.1027084589004517, -0.013473357073962688, -0.7464241981506348, 0.5057756304740906, -0.6154431700706482, -0.12431942671537399, 0.00638327794149518, 0.5349258184432983, -0.11708912253379822, 0.7708047032356262, -1.2903398275375366, 0.537024199962616, -0.4792318642139435, 2.4472644329071045, -1.0754649639129639, -0.4436701834201813, 0.7258234620094299, -0.00100231496617198, -0.5335372090339661, 0.22218956053256989, -0.9179014563560486, 1.543432354927063, -0.4802184998989105, -1.1548882722854614, -1.49604070186615, 1.1031830310821533, 0.08423363417387009, -0.7883493900299072, 1.2905328273773193, -1.9817981719970703, -0.25861337780952454, -1.1901007890701294, -0.18795280158519745, -1.0160431861877441, -0.2401667833328247, 1.595572829246521, -0.30821552872657776, 2.190703868865967, -1.24837064743042, 0.5381188988685608, 1.9796348810195923, 0.2879401445388794, 0.3731895685195923, -0.1271320879459381, -1.4785267114639282, -0.9634402394294739, -0.4167501628398895, -1.0476665496826172, -0.9654572010040283, -0.21140477061271667, -1.1587764024734497, -0.13414299488067627, -0.07865769416093826, 0.05498296767473221, -0.2589181065559387, -1.651911735534668, -0.979465126991272, -0.7922228574752808, 0.04551929235458374, -0.2229417860507965, -0.16121476888656616, 0.7300211191177368, 0.1389630138874054, 0.793704092502594, 0.31618309020996094, 1.1262840032577515, 0.45208773016929626, -1.163745641708374, 0.14514270424842834, 0.4186958372592926, -0.43854019045829773, -1.2309476137161255, 0.8019423484802246, -0.2516437768936157, 0.24122661352157593, 1.135040283203125, -1.064143419265747, 1.3806772232055664, 0.2587375044822693, -1.2484490871429443, 1.3061779737472534, 1.5976818799972534, 1.2236906290054321, 0.683619499206543, 2.474005937576294, -0.5844851136207581, -0.24747250974178314, -0.06648104637861252, 0.8363243341445923, -0.21853706240653992, 0.17390771210193634, 0.5508434176445007, 0.6711599230766296, -0.5969929099082947, -0.472970187664032, -0.44405534863471985, -2.539095163345337, -0.5823968052864075, 1.8040153980255127, 0.041483715176582336, -0.28370726108551025, -0.013902561739087105, 0.8030951023101807, -1.2295022010803223, 0.6395123600959778, -0.04474376514554024, 1.4605145454406738, -1.4333964586257935, 1.429884910583496, 0.35814544558525085, 0.12020192295312881, 0.7961315512657166, -1.0167709589004517, -0.9960488677024841, -1.2017271518707275, 0.4659935534000397, -0.520163893699646, 1.3327288627624512, -0.5273300409317017, -0.037366580218076706, 1.947403073310852, -0.8278114795684814, 1.9298757314682007, -0.7004263401031494], [-1.2127197980880737, -0.9774988889694214, -0.36805206537246704, -0.04233664646744728, 0.1042519062757492, 0.9786937236785889, -0.1281539797782898, 2.557910442352295, 0.6705026626586914, -0.15547685325145721, -0.7455793023109436, 1.75514554977417, -2.5714175701141357, 1.0395886898040771, 0.5250852108001709, -0.22196292877197266, -1.1237411499023438, 0.05038962885737419, -0.602631688117981, 0.5820760130882263, 0.7100008130073547, 0.7050378918647766, 1.0200116634368896, -0.1577063351869583, 1.0457050800323486, 1.0968300104141235, -0.21182681620121002, 0.7264125347137451, 0.47618985176086426, 0.42986050248146057, -0.33702221512794495, -0.5699371695518494, -0.8921119570732117, -0.34688758850097656, 1.423702359199524, -1.3112037181854248, -0.4102144241333008, 0.48454850912094116, -0.2819249629974365, -0.31418102979660034, 0.9219885468482971, 0.8807889819145203, -1.9841715097427368, -1.408424973487854, -1.0914521217346191, 1.2375794649124146, 0.4949533939361572, -0.3362753689289093, -0.04295456409454346, 1.4101378917694092, -0.36010900139808655, 0.049507930874824524, -0.29531729221343994, -0.49967098236083984, -0.2816096842288971, -2.456346035003662, -0.5183104276657104, -1.8692387342453003, 1.4583196640014648, -1.9507403373718262, -1.0995391607284546, -0.3756219446659088, 1.0733187198638916, -1.6368207931518555, 0.6403891444206238, 1.5115758180618286, -0.9067089557647705, 1.2838960886001587, -0.5555018782615662, -1.6291649341583252, 0.6651051640510559, -0.4297366738319397, -1.2854939699172974, -0.2599807679653168, 0.3231532573699951, -0.48966675996780396, -0.22488610446453094, 0.14745722711086273, -0.2256820797920227, -0.07670184969902039, -0.30404767394065857, -0.8618712425231934, 1.3047981262207031, -0.16281720995903015, 0.9762217998504639, -0.7577988505363464, 0.1482318937778473, 1.2131189107894897, 0.05344114825129509, 0.23598545789718628, 1.350813388824463, -0.8047713041305542, -0.6410066485404968, 1.7667590379714966, 0.8568207621574402, -0.8481960892677307, 1.160313606262207, -0.5339387655258179, 0.5799976587295532, -1.4144564867019653, 0.6546201705932617, -1.9812798500061035, -1.9045687913894653, -0.2106804996728897, 0.06589532643556595, 0.33939722180366516, 1.2722142934799194, 0.200303316116333, -0.04929256811738014, 0.6744298934936523, 1.1101948022842407, 0.21088503301143646, -1.7877678871154785, 0.057813216000795364, -2.3933351039886475, 0.5802168250083923, 0.11569555103778839, 0.3754129111766815, 1.2360395193099976, -0.4886173605918884, -2.6058545112609863, 0.9958709478378296, -1.0732958316802979, 1.6878197193145752, -0.11396130919456482, 2.693143844604492, -0.48366332054138184, 1.0537806749343872, 1.3008776903152466], [0.5441378951072693, -0.21508075296878815, 1.0281709432601929, -0.17439250648021698, 0.678726315498352, -1.7749274969100952, -0.6091707944869995, -0.7999399304389954, 1.3647562265396118, -0.284639835357666, 1.1060919761657715, 0.007420608773827553, -0.663765549659729, -1.4881511926651, 0.9472479820251465, 0.5750863552093506, 1.2766305208206177, -0.9720101356506348, -0.21263311803340912, -0.2584153711795807, 0.5482627749443054, 0.45328137278556824, -0.9430223107337952, 0.5128982067108154, 0.46986883878707886, -0.9553464651107788, -1.0555967092514038, 0.14894239604473114, 0.1807785928249359, 1.3808001279830933, 0.4270781874656677, -0.3900738060474396, -0.10501692444086075, 0.9181268811225891, 1.1474624872207642, -0.6824708580970764, -0.05392272025346756, -0.4244728982448578, 0.7578269839286804, 0.7435468435287476, -0.03184831142425537, 0.4048696458339691, -0.15758202970027924, -1.652969479560852, 0.7823771834373474, -0.9091247916221619, 1.7223224639892578, -1.1380513906478882, -0.0731976255774498, -0.4782983958721161, -0.8263330459594727, -0.3114270269870758, -0.05322356894612312, -0.41412636637687683, 0.48990270495414734, -0.19963160157203674, -0.9115196466445923, 1.3195605278015137, 0.002401967067271471, 0.43779733777046204, 0.8333842754364014, 1.1426019668579102, 0.10731752961874008, 0.674477219581604, -1.3882442712783813, -0.6908186078071594, 2.389610528945923, 1.5389608144760132, -0.13531681895256042, -0.9368066787719727, -0.30827343463897705, 0.30632975697517395, 0.7187203168869019, -1.294596552848816, 0.19724717736244202, -2.2810544967651367, 0.08095213770866394, 0.9383121132850647, 0.7073320150375366, 1.2012661695480347, 0.3878878355026245, 0.06953766942024231, 0.23396751284599304, 0.13344547152519226, 2.9668824672698975, 2.086049795150757, 0.28513649106025696, -0.050427623093128204, -0.8033025860786438, 1.4217890501022339, 1.7239909172058105, 0.8590359687805176, -1.9227449893951416, 0.9053619503974915, -0.1627349555492401, 0.6215934753417969, 0.09374091029167175, -0.5952104926109314, -0.8643226027488708, 0.23783768713474274, -0.8586691617965698, -0.6734485626220703, -0.24822385609149933, 0.18390516936779022, 0.03566991537809372, 1.6964678764343262, -0.5968769788742065, 0.06828314810991287, -1.5760247707366943, 0.4854526221752167, 0.8227246403694153, -1.1533697843551636, 0.7086966633796692, -0.05732441321015358, 0.4334297478199005, -0.7785418033599854, -0.06836263835430145, 2.438228130340576, -0.6387326717376709, 1.3499103784561157, -0.3548599183559418, 0.9836530685424805, 0.07812835276126862, -0.3137473165988922, -0.2570606768131256, -0.9629976153373718, 0.7936834096908569, 0.4027565121650696, -0.2633540630340576], [-0.19698978960514069, -0.05852388218045235, 1.2423789501190186, -1.668371558189392, 0.7227180004119873, -0.0319979153573513, 0.4997937083244324, -1.5689172744750977, -0.7794817686080933, -0.4047155976295471, 1.4135386943817139, -1.4943273067474365, 0.9836204648017883, -1.178362250328064, 1.0109339952468872, -1.1746879816055298, 1.836996078491211, -0.08342716842889786, 0.39471715688705444, 0.05947462096810341, 0.18550704419612885, 1.1707943677902222, -0.59617018699646, 0.5913030505180359, -0.10781672596931458, 0.034438133239746094, -0.2916046679019928, 0.18477395176887512, -1.2105339765548706, -0.03435995802283287, 1.3885198831558228, -0.29976558685302734, -0.6822545528411865, -0.9346808195114136, -0.6195757985115051, -0.9533127546310425, 0.5122057199478149, -0.0912737026810646, 0.045274268835783005, 1.2734051942825317, 0.5104602575302124, -0.11952193081378937, -0.2003876268863678, 0.6788090467453003, 0.3438272178173065, -0.18914003670215607, 0.04723401367664337, 1.7827779054641724, 0.550778865814209, 0.871677041053772, -0.8193693161010742, 1.7213194370269775, 1.6894166469573975, -0.8968902826309204, -1.7500256299972534, -1.023242473602295, -0.5741918087005615, -1.5478965044021606, -0.5025352835655212, 0.04360128194093704, 0.8692582249641418, 0.5578067302703857, 0.4623230993747711, -2.3108983039855957, 0.6069641709327698, -0.9140389561653137, -0.07224411517381668, -2.100679874420166, 0.21840451657772064, 0.7473002076148987, 0.9820863008499146, -0.7784124612808228, 2.1329751014709473, -1.5370455980300903, 0.014943068847060204, -1.0627188682556152, 1.7654681205749512, -1.0910829305648804, -0.769934892654419, 0.39779606461524963, -0.34270086884498596, 0.10348952561616898, -0.8239849209785461, -0.17152588069438934, -0.05946999043226242, -0.9470975995063782, 1.0702794790267944, 1.4814074039459229, -0.5090373754501343, 0.8741320371627808, 2.3447265625, -1.1236664056777954, 0.6954835653305054, 0.7351676821708679, -0.8215717077255249, 0.30726158618927, -0.6141051054000854, -1.3087023496627808, -0.40254878997802734, 0.4761643409729004, 0.6061733365058899, -1.6951767206192017, 1.1754436492919922, 0.8321872353553772, 2.255998373031616, 1.5732213258743286, -0.5863610506057739, 0.5563568472862244, -1.0828713178634644, 0.09550528973340988, -0.9671642184257507, 1.0470715761184692, 0.44625553488731384, -1.4722613096237183, -0.8084516525268555, 0.10416249185800552, 0.23703503608703613, 1.5766438245773315, 0.17754919826984406, 0.026934336870908737, -0.31565940380096436, 0.7714290618896484, -0.21122819185256958, -1.4654715061187744, 0.022957244887948036, -0.5269246697425842, -0.553257405757904, -2.293083667755127, 0.23571963608264923], [-0.5396842360496521, -0.8953465223312378, 0.6021620035171509, -0.03277093544602394, 1.19045889377594, 0.1670040637254715, -1.4098153114318848, -0.3495297133922577, -1.371262788772583, 1.4995161294937134, 1.6702584028244019, 0.267143577337265, 2.03726863861084, 1.493728518486023, 1.1632148027420044, 0.14019344747066498, -0.07947803288698196, -0.3736848533153534, -0.3080626428127289, -0.3064972758293152, 1.0269262790679932, 0.08419498801231384, -2.6309049129486084, -0.024147609248757362, 0.4140632152557373, 0.18863338232040405, -0.3812679946422577, -0.6076884269714355, -1.5081762075424194, -2.3285129070281982, -0.34287694096565247, -0.1445939540863037, -0.5684019923210144, 1.2379692792892456, -1.579161524772644, -0.5009315609931946, 0.8237884640693665, -1.7610929012298584, -0.21631217002868652, 0.264815092086792, 1.2900832891464233, 1.5056649446487427, 3.0146872997283936, -1.0800408124923706, -0.8281448483467102, 1.269610047340393, 2.6732730865478516, 0.7813975214958191, -0.17190805077552795, 1.3133164644241333, 0.06335830688476562, -0.886238694190979, 2.1473426818847656, -0.0019894628785550594, -2.0172858238220215, -0.6103277206420898, 0.900646984577179, 1.7321680784225464, 2.061248540878296, -0.9446251392364502, -0.41759276390075684, -0.8128398060798645, 0.24506501853466034, -0.019881324842572212, -0.8809642791748047, -1.627946376800537, 0.3135012686252594, -1.2373051643371582, 0.09979718923568726, -0.15895502269268036, 1.2787574529647827, -0.743605375289917, -0.45306500792503357, -1.5685527324676514, 0.32170096039772034, 1.075576663017273, -1.3316057920455933, 0.6104598045349121, 0.09936880320310593, 0.938683807849884, -1.4647350311279297, -0.3409891724586487, -0.6714977622032166, 0.48877018690109253, 0.9252742528915405, 0.3046037256717682, 1.1958550214767456, -0.22730770707130432, 0.05434873327612877, 0.15438207983970642, -0.9014818072319031, 0.20649519562721252, 1.4646763801574707, -1.3572410345077515, -1.318147897720337, -1.789089560508728, -0.577277421951294, -0.9722539782524109, -0.6527486443519592, 1.3589133024215698, 1.8767038583755493, 0.7037734985351562, 1.4704035520553589, -0.4048263430595398, -0.8473812341690063, -0.8586699366569519, -1.6182045936584473, -1.242944359779358, 1.4077714681625366, 0.3537696301937103, 2.4917826652526855, -0.11824546009302139, -0.32742545008659363, 0.12105013430118561, 1.0345739126205444, -0.6314639449119568, -0.018719103187322617, -0.5533950924873352, -0.5243360996246338, -1.7369742393493652, 0.797809898853302, -0.5019837021827698, -1.1771342754364014, 1.411350965499878, 1.0881588459014893, -0.3056952655315399, -1.5128222703933716, -0.1966540813446045, 0.9482886791229248], [0.39630597829818726, -0.8717873692512512, -1.2263163328170776, -1.7549409866333008, 0.5077613592147827, 0.23556314408779144, 0.18485495448112488, -0.648983359336853, 2.0299875736236572, 1.4171894788742065, -0.14715576171875, -0.4619436264038086, 0.3037477433681488, 0.2951933741569519, 0.5398663282394409, -0.12497076392173767, 1.2349809408187866, 1.9232531785964966, -1.6441277265548706, 0.47616899013519287, 2.560292959213257, -1.3127009868621826, 0.23169267177581787, -1.5675712823867798, 1.8839703798294067, -0.3429998457431793, -0.43122249841690063, -0.027186814695596695, 0.7791767120361328, 0.31054943799972534, -0.6931514143943787, -1.1300535202026367, -0.10593057423830032, -0.10493334382772446, -0.31384944915771484, 1.0668946504592896, -0.21968834102153778, -1.6106036901474, 0.37239477038383484, -0.4884360134601593, 0.5375394821166992, 0.5902449488639832, -1.378731369972229, 1.090999722480774, 0.5824456810951233, 0.1932799071073532, -0.2344914972782135, 0.3382140100002289, 0.3222923278808594, -0.5760489106178284, -0.6429193019866943, -0.939043402671814, 1.6496915817260742, -0.1585785150527954, -1.0357245206832886, -0.9520306587219238, 2.2798783779144287, -0.4734860360622406, 0.2028554081916809, -0.5998223423957825, 1.692074179649353, 1.0056368112564087, 0.03574572131037712, -1.379316806793213, 0.06388462334871292, -1.9131108522415161, -1.3811378479003906, 0.7195694446563721, -0.6709823608398438, 0.7765312194824219, -1.0262022018432617, -0.4811583161354065, 0.7461099028587341, 0.6933651566505432, -1.2869895696640015, 0.8389032483100891, -0.4410926401615143, 0.7727627754211426, 0.8421264886856079, 0.1541012078523636, 0.02281954698264599, -0.156112939119339, -0.05605736002326012, 0.7743561267852783, -0.19075456261634827, -1.5783849954605103, -1.2503787279129028, 0.10284917801618576, -0.9333890080451965, 1.4482940435409546, -1.4811609983444214, -1.474949598312378, 0.12029273062944412, 0.6104772090911865, 0.523719310760498, 0.1797395646572113, 0.5516124367713928, 0.46006157994270325, -0.8468272089958191, 0.5657736659049988, -2.5615127086639404, 1.6194285154342651, -1.8970006704330444, 0.21319183707237244, 0.47087201476097107, 1.3220473527908325, -1.4535990953445435, 0.46264132857322693, 1.2139153480529785, -3.032076597213745, 0.28317269682884216, -0.42203226685523987, 0.7071710824966431, 0.46744197607040405, 0.1709103286266327, -0.6548001170158386, -0.6054624319076538, -0.6729172468185425, 0.2279961258172989, 0.9481850862503052, -1.401318907737732, -1.3838082551956177, 0.8698052167892456, -0.29700368642807007, -0.7258996963500977, -0.7014208436012268, -0.6144695281982422, -0.2934603691101074, 0.47144731879234314], [-1.0168917179107666, -2.18168568611145, -0.16014792025089264, -0.8876746296882629, -0.41591110825538635, 0.046232614666223526, -0.17973913252353668, 0.45585060119628906, -0.138654425740242, -0.19150389730930328, -0.7186506390571594, 0.8059828877449036, -3.2120604515075684, 0.46613308787345886, -0.6538759469985962, 1.2092140913009644, -1.2976651191711426, 0.4244083762168884, -3.5329055786132812, 0.029231803491711617, 0.3081178665161133, -0.25907179713249207, -0.9468479752540588, 0.18046046793460846, -0.4985421895980835, 0.434826135635376, -1.0361852645874023, -2.578376531600952, -0.33063608407974243, 0.4517654478549957, 0.5614749789237976, 1.2762433290481567, 0.3730598986148834, 0.2481934130191803, -0.2973317503929138, 1.3463932275772095, -0.36404043436050415, -0.5876684188842773, -1.1438664197921753, -1.9165995121002197, 0.20874841511249542, -0.14072148501873016, -0.7913004159927368, -1.4379347562789917, 0.18316882848739624, -0.047949064522981644, 1.2525715827941895, -0.9013288617134094, -1.5942587852478027, 2.647265911102295, -0.6883627772331238, -0.2464972883462906, 1.0528182983398438, -0.7672361731529236, -0.9218154549598694, 1.015118956565857, 0.20597998797893524, -1.1106809377670288, -1.3985302448272705, 0.8554829955101013, -1.0654585361480713, 1.342061996459961, 0.28686997294425964, -0.3334589898586273, -1.5485836267471313, -0.9036532044410706, 0.5075182318687439, -0.846261203289032, 0.8330219984054565, -0.5410585403442383, -0.12743529677391052, 0.4622156023979187, -1.6151835918426514, -0.1565394103527069, -2.5183701515197754, -0.3671177625656128, 0.12122762203216553, -0.22957485914230347, -0.6884176731109619, 0.9332243800163269, -1.1784266233444214, 0.3433060944080353, -1.390796184539795, 1.2446426153182983, 0.0893126055598259, 0.4590040445327759, -0.2349523901939392, -0.1339225023984909, -1.0954450368881226, 1.1521093845367432, 0.7519714832305908, -1.1348508596420288, -0.7819005846977234, 0.19619154930114746, 0.18157216906547546, 0.593887984752655, 0.8290423154830933, 0.9201582670211792, 0.26541581749916077, -0.6768495440483093, -1.1843369007110596, 0.5127286911010742, -0.5667500495910645, -0.42698583006858826, 1.439797282218933, -1.3457847833633423, -0.9849979281425476, 0.8974578380584717, -0.2886820137500763, 0.6926189064979553, 0.4883253872394562, -1.6633509397506714, 1.2340468168258667, -0.4747541844844818, -0.5051763653755188, -1.5006489753723145, 1.4251809120178223, 0.18469902873039246, -1.3117457628250122, 0.15193407237529755, -2.354750871658325, -0.040434714406728745, -0.18267594277858734, -0.49188312888145447, 0.5727359652519226, -0.17416119575500488, -0.3430468738079071, 0.5021470189094543, 0.6825465559959412], [-1.1030467748641968, 0.14518791437149048, 0.33006829023361206, 0.17455652356147766, -1.4462031126022339, 0.7933453917503357, 0.4563552141189575, 1.0849844217300415, 1.108099341392517, 0.7162279486656189, -1.6848520040512085, -0.17902109026908875, 0.47671958804130554, 0.6298306584358215, -0.09366805851459503, 1.6992803812026978, -1.2758920192718506, 2.329421043395996, -1.1901096105575562, -0.6450430154800415, -1.9403669834136963, -1.7028743028640747, -0.2130471169948578, 0.4618799090385437, 1.108263611793518, -0.0399455800652504, -0.14392656087875366, 1.838644027709961, 0.3068617284297943, -0.4448411762714386, -0.8491224050521851, -0.6366904377937317, 0.12156848609447479, -1.1281304359436035, -0.34865042567253113, -1.4516910314559937, 1.0260034799575806, -1.4424046277999878, 1.410941481590271, 1.1564205884933472, 0.31787481904029846, -0.5311286449432373, -0.41937342286109924, 0.24013806879520416, -0.6979671716690063, 0.1254204362630844, -2.1965551376342773, 0.40098336338996887, 0.1816418617963791, 1.2577074766159058, 0.754289448261261, -0.5192995071411133, 1.3479375839233398, -0.5619919896125793, -1.594895601272583, 2.89021372795105, -0.48298338055610657, -0.37054672837257385, -1.114868402481079, -0.37214261293411255, -1.241943120956421, 0.2032836675643921, 0.9190067648887634, 0.0019032768905162811, -0.09136587381362915, 1.7246320247650146, -0.4396267235279083, 1.9757519960403442, 0.6093299984931946, 1.2387330532073975, -0.1666715294122696, 0.22014327347278595, 0.7674674987792969, -0.4000277817249298, -0.3970649540424347, -0.5299543738365173, -0.8898319602012634, -0.5722848176956177, 1.0745983123779297, 0.2887559235095978, 1.4645261764526367, -0.7845433354377747, 0.3666745722293854, 1.6060012578964233, -0.4890414774417877, -0.016897233203053474, 1.2192720174789429, -0.728780210018158, 0.9874179363250732, 2.116755247116089, -0.8599410057067871, -1.2676371335983276, 0.8892537951469421, -0.5047603249549866, 0.13931193947792053, -0.5022419095039368, 1.2830902338027954, -0.7221985459327698, -0.4576721489429474, 1.9928417205810547, -1.2010326385498047, 0.05745157599449158, 1.120508074760437, 0.2681567072868347, -0.08823855966329575, 0.2839691936969757, -0.5681674480438232, 1.1734118461608887, 0.9228999018669128, 0.7197421193122864, -1.8877445459365845, 1.5036596059799194, 0.7863793969154358, 0.3078056871891022, 2.843681573867798, 1.657119631767273, -0.9443982243537903, 0.5587822198867798, -0.4035976827144623, -0.44298055768013, -1.01398766040802, -1.7878106832504272, -1.8208470344543457, 0.3732782304286957, -2.4975576400756836, 0.8946224451065063, 0.9299934506416321, 1.2416813373565674, -0.4040374159812927], [-0.8636058568954468, 0.8602102398872375, -1.0002737045288086, -0.9583437442779541, 1.5850028991699219, -1.2203205823898315, 0.8169115781784058, -0.7289317846298218, -0.578042209148407, 0.2350294589996338, 0.18717887997627258, -0.36413106322288513, 0.8204175233840942, -0.7736057043075562, 1.770540714263916, 0.3902643620967865, -0.12005188316106796, 0.031247390434145927, 0.32984238862991333, 0.326820969581604, 0.826367974281311, -0.9454329609870911, -0.574682891368866, 0.4750443398952484, 0.06583351641893387, -1.4423365592956543, 0.6870965361595154, -1.9521214962005615, -0.004069473594427109, 0.47413673996925354, 0.48353248834609985, -0.9984140396118164, -0.3192717730998993, 0.32249021530151367, 0.41450005769729614, 1.3156124353408813, 0.6520816087722778, -0.5048967003822327, 0.2205881029367447, 0.9463310241699219, 0.12752678990364075, 1.8081680536270142, -0.9997882843017578, 0.5018510222434998, -0.7904995679855347, 0.18335175514221191, -0.7728550434112549, -0.23788350820541382, 0.7530721426010132, 0.5018009543418884, -0.11204037070274353, -0.07648302614688873, 1.328768014907837, -1.7042564153671265, -1.5079706907272339, -0.27889156341552734, 1.1567565202713013, 0.19042977690696716, -0.4015491008758545, 1.0186896324157715, 0.960073709487915, -0.3329384922981262, -1.105128288269043, -0.9446447491645813, 0.058617930859327316, -0.5145163536071777, -1.0278301239013672, -1.0201250314712524, 0.5222353935241699, -0.5859609842300415, -0.126570463180542, -1.62028968334198, 0.586645245552063, -1.1002204418182373, -2.2864956855773926, -0.4089581072330475, 0.1673167198896408, -0.5649721622467041, -0.54936683177948, -1.8607505559921265, 0.8470854163169861, 1.5476340055465698, -1.1002894639968872, -2.0919151306152344, -0.794367790222168, 0.001661938615143299, -3.701857328414917, 0.662274956703186, -0.39720603823661804, 1.0215356349945068, 0.6674043536186218, 0.5442067384719849, 1.0389777421951294, 1.7339195013046265, 0.08519299328327179, 1.2012224197387695, 0.2448899894952774, -0.09422222524881363, 0.6918367147445679, 0.48230013251304626, 0.24733087420463562, 0.5795191526412964, 0.7410804629325867, 0.21350456774234772, -1.297244906425476, -0.37965652346611023, -0.8933722972869873, 0.6146858334541321, 1.5109950304031372, 0.9780614972114563, -0.29129868745803833, -0.992983877658844, 0.8955778479576111, 1.0655204057693481, -0.3514465391635895, -0.9018325805664062, -0.008728517219424248, 1.6645950078964233, -0.13758356869220734, -0.09602285921573639, -0.31716978549957275, 0.2338031828403473, 0.319696307182312, -1.2945221662521362, -1.4128797054290771, -2.139305353164673, -1.5238193273544312, -1.4026696681976318, 0.6210379004478455], [0.17007939517498016, 0.056628964841365814, -0.4310902953147888, 0.4703356921672821, -0.6732321381568909, 0.977580726146698, 1.3579082489013672, 0.1416047215461731, -2.288356304168701, 0.5968613624572754, -1.4391928911209106, 0.6598096489906311, 0.067414790391922, -1.857473611831665, 0.5004876255989075, 0.8223716616630554, 1.1709098815917969, -0.058870404958724976, -0.5059347152709961, -0.6842558979988098, -0.5813978314399719, 0.4191729724407196, 0.07530847191810608, -0.8392813205718994, 1.0020793676376343, 1.1626288890838623, -0.6359977722167969, -0.020314523950219154, 0.19683514535427094, 1.3660788536071777, -0.6808156967163086, 0.15283270180225372, 0.7835349440574646, -1.5911991596221924, -0.8057156205177307, 1.4365758895874023, 2.2554001808166504, 1.8915785551071167, -0.45448094606399536, -0.7033060789108276, 0.17048467695713043, 0.08491545915603638, 0.06294532865285873, 1.0186704397201538, -1.7312333583831787, 0.6555965542793274, 1.651638388633728, -0.5230268239974976, 0.9419234395027161, -1.750502347946167, -0.027367377653717995, -0.3069343864917755, 0.10368477553129196, -1.2893742322921753, 0.1479087918996811, -1.4459484815597534, -0.15524601936340332, 0.7410677671432495, 1.5647292137145996, -1.685581088066101, 0.507997453212738, -0.9399223327636719, -0.8926008939743042, 1.295592188835144, 0.026265012100338936, 0.4074995219707489, 0.5962860584259033, -0.6361473202705383, -0.04982184246182442, 0.8585861921310425, 0.42629972100257874, -1.8833249807357788, -1.3230464458465576, -1.3230900764465332, -1.2499878406524658, -0.5675183534622192, -1.4844387769699097, -1.2702796459197998, 0.3535539209842682, 0.23478904366493225, 0.7879721522331238, -0.6987035870552063, 0.47824761271476746, -0.02508387342095375, 0.631985068321228, 1.0683033466339111, -0.784898042678833, -1.5891715288162231, -0.31324535608291626, -0.13707122206687927, -1.0313611030578613, -0.1417054533958435, 1.7236930131912231, -0.7619397044181824, 0.14826460182666779, -1.117506980895996, -0.8946460485458374, 1.6541097164154053, 0.1851869523525238, 0.6759294271469116, 0.12949107587337494, -0.20516282320022583, 0.5753231048583984, 0.19604219496250153, -0.3115193545818329, -0.8017395734786987, 0.13547565042972565, -1.7931357622146606, 1.7528620958328247, 1.2376750707626343, -0.22424910962581635, 1.3567349910736084, 2.0830774307250977, -0.5893968939781189, 0.22951102256774902, 0.949681282043457, -0.11703459173440933, 0.3119846284389496, -1.2487239837646484, 0.1456815004348755, -1.3771928548812866, -0.4481952488422394, -0.08812429755926132, 1.1513479948043823, 1.4109077453613281, 1.834323763847351, -0.8269503116607666, 1.5915017127990723, 0.5559693574905396], [-1.8921008110046387, -1.5514733791351318, -0.17781707644462585, 1.428848147392273, 1.7595508098602295, -0.9547738432884216, 0.11367850005626678, -1.0384982824325562, 1.6030017137527466, 1.6707388162612915, 0.7008680701255798, -2.0218584537506104, 0.35872235894203186, 0.2432638555765152, -0.6851086616516113, 1.4754834175109863, -1.0338045358657837, -1.6310473680496216, 0.3117053210735321, -0.022886626422405243, 1.7478865385055542, 1.4264154434204102, -0.6335556507110596, -0.9352868795394897, 0.7563415169715881, 1.8232886791229248, 0.2771672308444977, -0.26261380314826965, -2.0918357372283936, 0.9626535177230835, -1.1671206951141357, -0.44087642431259155, -0.11796563118696213, 0.8097873330116272, 1.9985694885253906, -0.5209887623786926, -0.10801923274993896, -0.8337804675102234, 0.9649543762207031, 1.213105320930481, 2.180870294570923, 0.21176159381866455, -1.1042039394378662, -0.4495002329349518, 1.6030725240707397, -1.2660353183746338, 1.5237797498703003, -0.6361926198005676, 2.069756031036377, -0.042886361479759216, -0.13171027600765228, 1.3265095949172974, -0.30895283818244934, -1.06465744972229, 0.4421265423297882, 0.2662530243396759, -0.2872384488582611, -1.325790524482727, 0.18970084190368652, 1.2041041851043701, -0.8669927716255188, 0.10897033661603928, 0.12566889822483063, 1.925910472869873, 1.0334446430206299, -2.2509119510650635, 0.5152437686920166, -0.34893447160720825, 0.27707818150520325, 0.5131539702415466, -0.623589277267456, 0.4072858393192291, 0.28881481289863586, -0.9497038722038269, -1.9239298105239868, -1.278983235359192, -0.35644856095314026, -1.0872406959533691, -1.4292099475860596, 0.9385960698127747, -1.0601726770401, 0.22604988515377045, 0.03810055926442146, -1.4316569566726685, -0.7149581909179688, 0.9340617060661316, 0.2968689799308777, -2.098048448562622, 0.2764800786972046, 1.9581657648086548, 1.634382724761963, 0.7353342175483704, -1.021284580230713, -0.7412917613983154, 0.80943763256073, -0.42839282751083374, -0.5906286239624023, -2.10837721824646, 0.5992845892906189, 1.556436538696289, 0.0498511977493763, -0.5138407945632935, 0.8674787878990173, 1.3751108646392822, 1.9526019096374512, 0.35432982444763184, -2.147181749343872, -0.31352898478507996, -1.5586739778518677, 0.5722495317459106, 1.6812196969985962, 0.18177537620067596, -0.09041076898574829, -1.240830898284912, -0.761287271976471, 1.3281768560409546, 0.4669717252254486, -1.4325939416885376, -0.6332912445068359, 1.6549018621444702, -1.350028395652771, 0.28167545795440674, 0.9640606641769409, -1.5471367835998535, 0.615655779838562, 0.06277382373809814, -0.5634049773216248, -0.1259927749633789, 0.11294898390769958]], "dtype": "float32"}, {"__nd__": [-0.14258484542369843, 0.9580066204071045, -0.8787806034088135, 0.41542956233024597, 0.40497681498527527, 2.4926600456237793, 0.956341028213501, -1.3067578077316284, 0.2351149618625641, 0.1516866385936737, 0.1244405135512352, -0.2973306477069855, 0.6253012418746948, -0.09812262654304504, 0.5465527176856995, 0.10532060265541077, -1.4982372522354126, 1.096398949623108, 1.2594066858291626, -0.0928901731967926, 0.4203416407108307, -0.37919533252716064, 0.6743683815002441, 1.5738595724105835, 0.7043822407722473, -0.7849615812301636, -1.9847378730773926, -0.7068747878074646, 0.5941028594970703, 1.3857280015945435, 0.09585878252983093, -2.3020036220550537, 0.6203649044036865, -1.2941969633102417, -0.7566472291946411, 1.9585058689117432, -0.6058035492897034, -0.2931734621524811, 1.814399242401123, 1.1305851936340332, -0.9998643398284912, 0.6491744518280029, -1.6675746440887451, -1.1701455116271973, 0.9668128490447998, -0.8063733577728271, -0.8090540170669556, 1.167931079864502, -0.03243973106145859, 0.7808309197425842, -1.6782108545303345, -1.533843755722046, -0.9471075534820557, 0.784278929233551, 0.2853392958641052, 0.4212246537208557, 1.1088944673538208, 0.4495207369327545, 0.9528346657752991, 0.1107967346906662, -0.7845432758331299, 0.8740447759628296, -0.6221426129341125, -0.26969027519226074, 0.16735099256038666, 0.22587299346923828, 0.0528043732047081, 0.6761682033538818, -0.9475772380828857, 1.3597495555877686, 1.4242945909500122, -0.694370687007904, -1.2512872219085693, -1.3872432708740234, -2.0442440509796143, 0.5727105736732483, 2.5045812129974365, -0.7806490659713745, -0.048659469932317734, -0.1779758781194687, 2.208285093307495, -0.9900867342948914, -1.9792956113815308, -0.4594719111919403, 0.2359924167394638, 0.7145942449569702, 0.8378278613090515, 0.03372553735971451, 0.14888033270835876, 1.2754713296890259, -1.732996940612793, -1.6096599102020264, 0.8912532925605774, -2.5665202140808105, -0.16647757589817047, 0.4651629328727722, 2.4521007537841797, -0.4111967980861664, -0.7193526029586792, -1.2548096179962158, -0.9309219717979431, -1.6103137731552124, 1.6174001693725586, -0.9067160487174988, -2.0606284141540527, 1.1538074016571045, -1.3410203456878662, -0.044339604675769806, -0.4463757276535034, -0.1879572868347168, -1.7530804872512817, -1.2569464445114136, 0.3745861053466797, -0.9411023259162903, -1.6497517824172974, 1.998763918876648, 0.17679300904273987, 0.8775332570075989, -0.7862519025802612, -0.2294609546661377, -1.1338638067245483, 0.015374638140201569, -1.5633866786956787, 0.24836061894893646, 0.7592755556106567, 0.16703741252422333, -0.35540440678596497, 0.8786084055900574, -0.8900238275527954], "dtype": "float32"}, {"__nd__": [-0.04773571342229843, 0.14182138442993164, -0.06270420551300049, -0.04078532010316849, -0.08259879797697067, 0.09049617499113083, -0.07203657925128937, 0.011860193684697151, -0.03840802609920502, 0.020262815058231354, 0.16156454384326935, 0.12677960097789764, 0.06769189238548279, 0.11310553550720215, -0.013309882953763008, 0.01377036515623331, 0.049128662794828415, -0.09825418144464493, -0.011044085025787354, -0.10157432407140732, 0.04491804912686348, 0.08793607354164124, -0.04923728480935097, 0.11453760415315628, 0.0598534494638443, -0.04365915060043335, 0.0497782863676548, 0.02472110465168953, 0.04474681615829468, 0.19724592566490173, 0.015364701859652996, 0.01585059054195881, -0.009094653651118279, 0.0244226586073637, -0.10658369213342667, 0.02618090622127056, 0.08435797691345215, -0.041673868894577026, -0.020733291283249855, 0.07498803734779358, -0.1686466783285141, 0.10568895936012268, 0.08186767250299454, 0.007991333492100239, 0.10881686955690384, -0.010926073417067528, -0.13671302795410156, -0.0130316698923707, -0.03783996030688286, 0.06020442023873329, 0.10402298718690872, -0.13099247217178345, -0.045078158378601074, -0.09326612949371338, -0.142504021525383, 0.2089620679616928, -0.05457250401377678, -0.1901678442955017, -0.03263816982507706, 0.016142431646585464, -0.015059753321111202, 0.11057598888874054, -0.011235297657549381, 0.007465754169970751, -0.060448840260505676, -0.2299678772687912, 0.08277981728315353, -0.08792229741811752, 0.017617594450712204, 0.015945566818118095], "dtype": "float32"}, {"__nd__": [0.9857756495475769, 0.9992119073867798, 0.9745052456855774, 1.0091718435287476, 1.0810811519622803, 1.0708134174346924, 0.9964854121208191, 0.9933961629867554, 0.958199679851532, 1.039149284362793, 1.0247588157653809, 0.9028687477111816, 1.004437804222107, 1.0130316019058228, 0.9637994170188904, 1.1476519107818604, 0.9632821679115295, 0.9762884378433228, 1.0051114559173584, 0.9121800065040588, 1.107108473777771, 0.9606073498725891, 0.929424524307251, 0.9890903234481812, 0.9612724184989929, 1.0927374362945557, 1.0213879346847534, 1.1543914079666138, 0.9842782020568848, 1.0183651447296143, 0.908743143081665, 0.9747676849365234, 0.9726569056510925, 1.1692728996276855, 0.9688133001327515, 1.0157352685928345, 1.0644820928573608, 0.9349851608276367, 1.0670132637023926, 1.031813621520996, 0.9895656108856201, 1.0185855627059937, 0.9810726642608643, 0.9509419798851013, 0.9401872754096985, 1.0247373580932617, 0.997437596321106, 1.000860571861267, 0.9720928072929382, 1.0476423501968384, 0.9568197131156921, 0.9056574702262878, 0.9829060435295105, 0.967621922492981, 1.0971877574920654, 1.1220413446426392, 1.050896167755127, 1.0155056715011597, 1.0624383687973022, 1.0411399602890015, 0.9528188109397888, 1.0866888761520386, 1.0098800659179688, 0.9034554362297058, 0.9976527094841003, 1.0024936199188232, 0.9326722621917725, 1.0297731161117554, 1.0115771293640137, 0.8974490165710449], "dtype": "float32"}, {"__nd__": [[-0.39389243721961975, 1.8263683319091797, -0.40772485733032227, 2.3728885650634766, -0.38409560918807983, -1.1794058084487915, -0.9853062033653259, -1.3681772947311401, 0.14505884051322937, -1.7466832399368286, -0.18578997254371643, 0.7035605311393738, -0.7936165928840637, 0.42995938658714294, 0.3755035698413849, 0.9987451434135437, -0.18119430541992188, 1.2238373756408691, -0.2222585678100586, -1.4544626474380493, -1.0472538471221924, 0.1341286152601242, 1.381457805633545, -0.9527710676193237, -1.042468786239624, -0.7412058115005493, -0.22296495735645294, 1.9843026399612427, -0.7784408330917358, -0.5423910617828369, 0.5415475964546204, -0.637939453125, -0.45762014389038086, -0.029156651347875595, -0.7615862488746643, -0.24225667119026184, -1.4405068159103394, -0.10165543109178543, -0.5751183032989502, 0.40264177322387695, 0.6608120799064636, 0.425546258687973, -0.15873786807060242, -1.3085856437683105, -0.7685216665267944, 0.8469318151473999, -0.3642186224460602, -0.2543630003929138, 2.489971399307251, 0.3847293555736542, 0.06472823023796082, -0.6358699202537537, 1.297534704208374, -0.4036614000797272, 0.1315770000219345, -1.0659395456314087, 0.2090947926044464, 1.2495315074920654, 0.3009070158004761, 2.107938051223755, 1.4193902015686035, -1.1187888383865356, 2.560001850128174, 0.4451787769794464, 0.39596521854400635, -0.7491695880889893, 0.18155322968959808, 1.522576928138733, 0.171232208609581, -1.0235487222671509, -0.8711239099502563, 1.9493329524993896, -0.3589337468147278, -2.2184529304504395, -1.6054518222808838, -0.9427865743637085, 1.4195986986160278, -0.35546988248825073, 0.4231114983558655, -1.4742767810821533, -0.19923363626003265, 0.8230772018432617, -0.3347190320491791, 1.1584663391113281, -0.37758558988571167, -0.3925854563713074, -0.19824737310409546, 1.8611561059951782, -1.2050553560256958, -0.05109886825084686, 0.8019075393676758, -0.03421977534890175, 0.6941776871681213, 0.5286411643028259, 0.9894707202911377, 2.248427391052246, -0.5681909918785095, -0.2413230538368225, -1.382749080657959, 1.0619703531265259, -1.6996150016784668, -3.0681354999542236, -1.5108941793441772, 0.8925504088401794, -0.5897068977355957, 1.9815864562988281, -0.733925461769104, -0.918541431427002, 1.6438512802124023, 1.295781135559082, -1.114733099937439, -0.19818152487277985, -1.18130362033844, -7.141118840081617e-05, 0.39606091380119324, -0.33981844782829285, -0.5230399370193481, -1.7528959512710571, 0.06152726709842682, -0.17109906673431396, -0.5911858081817627, -0.7539303302764893, 1.7077889442443848, -0.828081488609314, -1.4948372840881348, 0.1822492927312851, 0.6598117351531982, -0.8869144320487976, 2.0531742572784424], [0.8171267509460449, 0.586224377155304, 0.12457747757434845, 1.0191118717193604, -0.4694661796092987, 1.4583061933517456, -1.1240174770355225, 1.6770097017288208, 0.8977617025375366, 0.03460811451077461, 1.6885199546813965, -0.6728309392929077, 0.6602584719657898, -0.5402107834815979, 0.05962275341153145, 2.2464044094085693, 1.161211609840393, 0.21426226198673248, 0.07049769908189774, 0.6650854349136353, 0.8232476711273193, -0.9347208738327026, 1.8711650371551514, -0.30129194259643555, 0.6082945466041565, 0.6045770049095154, -0.8389634490013123, 0.5762860774993896, 0.5433921217918396, 0.07805309444665909, 1.2836791276931763, 0.8309276700019836, 0.7638668417930603, 0.35617518424987793, -2.7296111583709717, 0.6859625577926636, 2.0989573001861572, -0.8303853869438171, 0.1653335690498352, -0.7370832562446594, -0.1712263822555542, -0.29288169741630554, 1.3564019203186035, 0.2852855920791626, -1.1428598165512085, -0.5404712557792664, 3.521409749984741, 0.636076033115387, -0.7503697276115417, -1.0891376733779907, 0.8251878023147583, 0.3541752099990845, 0.9443191289901733, -1.0666112899780273, -0.6289893984794617, 1.4522618055343628, -0.21703636646270752, -0.3525213599205017, 0.4094206988811493, 2.328555107116699, -1.0239773988723755, 0.9967338442802429, 0.3165439963340759, 1.4832931756973267, -0.5268423557281494, -0.4846046566963196, 1.4697750806808472, -0.5252320170402527, 2.43922758102417, -0.11140718311071396, 1.2587604522705078, 1.322497010231018, 0.25317442417144775, 1.374433159828186, -1.2748042345046997, -0.5610141754150391, -1.4143460988998413, 0.31723353266716003, -0.6851138472557068, -0.6198127269744873, -0.8595035076141357, 0.4653061628341675, -0.9468024373054504, 1.0058157444000244, 0.20137208700180054, -0.8228685855865479, -0.15899048745632172, 0.5591530799865723, -0.13013111054897308, 0.5703333020210266, 0.7727594375610352, 3.146939992904663, 1.420677661895752, 0.9637427926063538, 0.05527891963720322, 0.6335064768791199, -0.8400439620018005, -0.2765578627586365, -0.279432088136673, -0.043898992240428925, -0.6581145524978638, -0.5223759412765503, -1.1817115545272827, 2.260615110397339, -0.8003618717193604, 1.2036004066467285, -0.6799933910369873, -0.945200502872467, -0.5105630159378052, -1.1497606039047241, 0.16606488823890686, -0.11768617480993271, -1.1830452680587769, 0.7441332340240479, 0.4171035885810852, 1.453531265258789, -1.0524133443832397, 0.035276368260383606, 1.577080488204956, 0.08911747485399246, 0.11870715022087097, -1.1888396739959717, 0.006016930565237999, 1.0729691982269287, 0.16801264882087708, 1.0527856349945068, -0.3252769112586975, 0.026630425825715065, -0.9518246054649353], [-0.5812048316001892, -0.3873344659805298, -0.6636120080947876, 1.4965441226959229, -0.6771820187568665, 1.1940315961837769, -0.9686247110366821, -0.9087090492248535, -0.14867466688156128, 0.06669872254133224, 0.9937275648117065, 1.259731650352478, -2.590646743774414, 0.9388537406921387, 0.6355647444725037, 0.31125548481941223, -0.5776633024215698, -0.2276707887649536, 0.30970707535743713, -0.25616374611854553, -0.6140307784080505, 0.5551480054855347, -0.3502829968929291, 1.3218753337860107, -0.27843767404556274, -1.2743372917175293, -1.4799673557281494, -2.121610641479492, -1.6217445135116577, 0.9725148677825928, -1.2729673385620117, 0.38537701964378357, -1.327337384223938, 0.03457648679614067, -1.0370256900787354, -0.8633797764778137, 1.4580192565917969, -0.049324218183755875, 0.48182564973831177, 0.8401114344596863, 0.06460095942020416, 2.3714609146118164, -0.3849460780620575, 0.9700227975845337, -1.8376085758209229, -0.6760022640228271, -0.2785107493400574, -0.05654262378811836, -0.7836199402809143, -0.672713577747345, -0.6603221893310547, -0.029149120673537254, 1.1482197046279907, 0.2803095579147339, -0.021283011883497238, -0.08454806357622147, -0.6506998538970947, -1.0044751167297363, 0.754859983921051, -1.6777759790420532, 0.2587926387786865, 0.5058932900428772, -1.690649151802063, 0.11250335723161697, -0.7804498672485352, -0.24520070850849152, 1.1030570268630981, -0.653812050819397, -1.6154253482818604, -0.14420738816261292, -1.6023939847946167, -0.6038587689399719, 0.9859322309494019, -1.3443894386291504, 1.2410322427749634, 0.4566424489021301, -0.04220180958509445, -1.7890413999557495, 1.0775811672210693, 0.7090474963188171, -0.9190525412559509, 0.30046167969703674, -0.6770568490028381, 1.0538901090621948, -0.3344059884548187, 1.4721754789352417, -0.3716394305229187, -0.2775784134864807, -1.4791545867919922, -0.7194026708602905, -0.2607789635658264, -0.6101240515708923, 0.14851507544517517, 2.653132438659668, -0.7988488078117371, -0.9646608233451843, 0.6479467153549194, -0.655053436756134, -0.48793619871139526, 0.6932990550994873, -0.2417302280664444, -1.5431610345840454, -0.4929245412349701, 0.2572683095932007, -1.149085283279419, -0.9849947690963745, -1.2199900150299072, -1.8682352304458618, -0.4161999225616455, -1.5837451219558716, 1.4961214065551758, 1.8510814905166626, -0.7789693474769592, 1.5408191680908203, -1.8111474514007568, 0.5592787265777588, 0.2659735083580017, 1.283080816268921, 0.7679762840270996, -0.5575922727584839, -4.362992763519287, -0.3631739020347595, 0.1044854000210762, -0.04101278632879257, 0.8244226574897766, 0.39616405963897705, -1.6827164888381958, 0.04883834347128868, -0.0570441372692585], [1.1079634428024292, -0.1835707277059555, -0.28296157717704773, 0.11228859424591064, 0.5744744539260864, 0.4801254868507385, -1.8866698741912842, 1.2315996885299683, -1.2074558734893799, -1.469393253326416, 0.4952714741230011, 0.11738507449626923, 0.337017297744751, -1.043473720550537, 0.30149996280670166, 0.2940911650657654, -0.837408721446991, -1.4512097835540771, 0.8406237959861755, 0.2270977646112442, -0.749320924282074, 0.6899604201316833, 0.13157892227172852, -1.7395638227462769, -2.497556447982788, 1.0689300298690796, -0.6613015532493591, -0.16004963219165802, 1.1779111623764038, -1.2520015239715576, 0.46509039402008057, -0.25561296939849854, 0.7726796865463257, 0.18772391974925995, -0.3409196436405182, -0.9299320578575134, -0.7005744576454163, 0.09786130487918854, 0.032095931470394135, 0.9719363451004028, -1.107343077659607, 0.5157487392425537, -0.6852474808692932, -0.963577151298523, 1.299980640411377, -0.06770187616348267, 2.21816086769104, 0.9872967600822449, -1.1372379064559937, 0.5066825151443481, -0.8146907687187195, 0.06390894949436188, -0.4718826115131378, 1.1179149150848389, 1.1265758275985718, -1.5926320552825928, 1.4658665657043457, 0.6582448482513428, -0.414893239736557, -0.7888786196708679, -0.4594730734825134, -0.13589221239089966, -1.5183875560760498, 0.128922238945961, -0.7624881267547607, -0.7272624373435974, -1.2221901416778564, 0.09748770296573639, -1.812668800354004, 2.160043716430664, -0.5294215679168701, -1.0908759832382202, -0.27794915437698364, 0.9947575330734253, 0.019572263583540916, 0.5008907914161682, 0.8705167770385742, 0.8815410137176514, 1.3510704040527344, 1.345631718635559, 0.5378206968307495, 0.5599446296691895, 2.0161352157592773, -1.6107420921325684, 1.8855993747711182, 1.1348176002502441, -1.7897980213165283, 0.45700496435165405, -0.17619314789772034, -0.5559309124946594, -2.6599977016448975, 0.46653690934181213, 0.573370635509491, -0.25038230419158936, -0.514999508857727, -0.19584839046001434, 0.11539407074451447, -0.39150428771972656, 0.16780012845993042, -0.5067068934440613, -1.2133609056472778, 1.110025405883789, 0.8186194896697998, -1.408511757850647, 0.39494645595550537, 1.1795337200164795, 1.5870915651321411, 0.16041193902492523, -0.09351366013288498, 0.5809840559959412, 0.02264646627008915, 1.391599178314209, -0.6577702760696411, -0.8914299607276917, 0.17839612066745758, 1.3142133951187134, 1.383744716644287, -0.40481820702552795, 0.5229068994522095, 0.1766277253627777, 0.25594696402549744, 0.030904529616236687, 0.5614782571792603, -2.0903806686401367, -1.4354444742202759, 0.22971220314502716, -1.2386951446533203, 0.18558500707149506, 0.8413995504379272], [1.0687319040298462, -0.09691276401281357, 1.3302946090698242, 0.2444610446691513, 1.204057216644287, -0.9412125945091248, 0.47728580236434937, 0.5638589859008789, 0.3364031910896301, -0.9090225696563721, 1.556006908416748, -0.4512207508087158, 0.28659766912460327, 0.10785666108131409, -2.752809762954712, 0.9608855247497559, 0.19493038952350616, -0.8203439712524414, -0.4458635449409485, -1.8346914052963257, -0.5138840675354004, -0.044824279844760895, -0.1993292272090912, 0.20064643025398254, 0.04251035302877426, -1.6255444288253784, -0.023752640932798386, 1.2773864269256592, -0.6688277125358582, 0.5937063097953796, 0.0908501148223877, -2.081486940383911, -0.5260750651359558, 1.1944669485092163, 0.8638748526573181, 0.7507883310317993, 0.4636082947254181, -1.2490986585617065, -0.9391221404075623, 0.3177773952484131, 0.3473643362522125, 0.8315920829772949, 0.5032085180282593, 0.7487449049949646, -1.2792325019836426, -0.48189371824264526, -0.8446643352508545, -0.41576462984085083, 1.8333672285079956, -0.21387715637683868, -0.980539083480835, 0.31266340613365173, 0.4792233407497406, -0.3051556944847107, 0.057376597076654434, 0.44388771057128906, 0.024439940229058266, 0.9351912140846252, 0.4376683533191681, -0.07354486733675003, -0.26499223709106445, -0.801095187664032, 1.7748628854751587, -0.45565131306648254, -0.8733352422714233, -1.7056058645248413, -0.08185978978872299, -0.478413850069046, 0.2086440771818161, 0.37871408462524414, 0.4446396827697754, 1.357846736907959, 0.3573029339313507, 0.058007512241601944, 1.1762540340423584, -1.9279894828796387, 0.8463301658630371, 0.6787458062171936, -0.8599372506141663, -0.1641983538866043, 0.12542985379695892, -1.2048887014389038, -1.1661229133605957, -2.00341534614563, -0.6824735999107361, -0.3841838538646698, -1.2208669185638428, -2.2241387367248535, 0.023320872336626053, -0.2354496717453003, 0.4145945906639099, 0.24480466544628143, 0.7403075695037842, -0.4999033212661743, -0.6223763227462769, 0.20717906951904297, -0.2088545858860016, -0.15926948189735413, 0.0353684239089489, -1.7687216997146606, 2.0698869228363037, 0.6795728206634521, -1.547593116760254, 0.19661661982536316, -0.9345831871032715, -0.5463070869445801, 0.14406464993953705, -0.2864980101585388, 0.014146161265671253, 0.6921038627624512, -0.3548656404018402, 1.6104400157928467, -1.1309925317764282, 1.1405969858169556, -0.24465247988700867, -0.37213316559791565, 0.1613726168870926, 0.227107971906662, -0.6098830103874207, 1.5524555444717407, -0.4121915102005005, 0.49164143204689026, 0.1152273640036583, -0.6761751174926758, -1.5489661693572998, 1.1673296689987183, 0.4768845736980438, -1.5653554201126099, -1.0476349592208862], [0.1128825694322586, 0.24841707944869995, -1.2720685005187988, -1.606789469718933, -0.12923745810985565, -1.0195192098617554, 0.8718900680541992, 0.13205966353416443, -2.027649402618408, 0.22734510898590088, 0.9789473414421082, -0.157072514295578, -0.43137452006340027, 0.906132698059082, -0.09145576506853104, 0.6627222299575806, 1.5606766939163208, -1.2738946676254272, 2.0368926525115967, 0.2820654511451721, -1.6747857332229614, -0.21134187281131744, -0.11262575536966324, -0.7596502900123596, -2.853858470916748, -1.1410889625549316, -2.820417881011963, -0.9546950459480286, 0.3632861375808716, 1.2798436880111694, 2.4458096027374268, 1.3235639333724976, 0.07411955296993256, -0.8310706615447998, -0.02676660567522049, 0.2747531235218048, -0.3752445876598358, -0.587324321269989, -0.6642879843711853, -1.0642285346984863, -0.6403458118438721, -0.280994713306427, 0.6219596862792969, -0.06364154815673828, -1.162372350692749, 0.9210211634635925, -1.5761511325836182, -0.963558554649353, 0.43269816040992737, -0.3191863000392914, -1.684388279914856, -0.0946560949087143, -0.13085606694221497, -0.8055768609046936, -0.1445802003145218, 0.024619242176413536, -0.44232702255249023, 1.0787761211395264, -0.9522220492362976, 0.5430872440338135, -1.7228388786315918, -0.12433141469955444, 0.4238101541996002, -0.44560036063194275, -0.739976167678833, -0.8584296107292175, -1.1380441188812256, 0.5494868159294128, -0.5858680605888367, 0.8006741404533386, -0.2847036123275757, 0.5559213757514954, -0.20648039877414703, 1.9542882442474365, 0.6846609711647034, -0.04970628395676613, 0.4576464593410492, 1.4032689332962036, 0.10051315277814865, 0.14352166652679443, -0.05348698049783707, 0.47608283162117004, 0.08900322765111923, 1.2046265602111816, -0.06223294511437416, 0.5429460406303406, 0.6116216778755188, -0.006248379126191139, 0.7235373854637146, 0.6998865604400635, 2.6821131706237793, -0.39736053347587585, 0.17537322640419006, -0.2993401885032654, 0.7549278736114502, 1.844126582145691, -0.2692514657974243, -1.5402952432632446, -0.729336142539978, 1.409195065498352, -0.07678483426570892, 2.365964889526367, -0.7109500765800476, -1.2328722476959229, 1.9583098888397217, -0.07526760548353195, -2.2972116470336914, -0.1238187775015831, 0.18067961931228638, -0.11646532267332077, -0.7738118767738342, -0.026673097163438797, -2.1496033668518066, 1.0246225595474243, 0.48129802942276, -0.29215508699417114, 0.5547923445701599, 0.27576160430908203, -0.44476184248924255, 1.300101637840271, 1.1755390167236328, -0.7808879017829895, 1.5291508436203003, -0.004140476696193218, -0.45740166306495667, -1.717156171798706, 0.3101910948753357, 0.5122691988945007, -0.9936597347259521], [1.0220063924789429, 0.8782705664634705, -0.3386182487010956, 0.3572021424770355, -0.7625911235809326, 2.7960824966430664, 0.6650984883308411, 0.3688712418079376, 1.4724311828613281, -1.6917110681533813, -0.21370330452919006, 0.28347769379615784, -0.7650986909866333, -1.3392618894577026, 0.19242151081562042, 0.051834192126989365, -0.13420137763023376, 0.9431683421134949, -0.026140648871660233, 1.1674935817718506, 0.7131784558296204, -0.019521351903676987, 1.1475632190704346, -0.152398943901062, 1.2568714618682861, 1.746662974357605, 0.4582448899745941, -0.13975539803504944, 1.3102402687072754, 0.8494254350662231, -0.6631882786750793, -0.29612964391708374, -1.4956836700439453, 0.6916843056678772, 0.204342320561409, -0.27973347902297974, -2.0465774536132812, -0.7059399485588074, 0.5244595408439636, 1.2947646379470825, -0.2469167411327362, 0.6242370009422302, 0.4301939606666565, -0.0796843022108078, -0.17658156156539917, 0.0939914658665657, 2.2077717781066895, 2.4831340312957764, 0.060684651136398315, -0.5187511444091797, 0.03577995300292969, -1.707701563835144, 0.4955652356147766, -0.3038892149925232, -0.02348792739212513, -1.2474945783615112, -0.4423815608024597, 0.49146345257759094, 1.0383905172348022, -0.036367084830999374, 0.21530504524707794, -0.3984605371952057, 0.911580502986908, -0.9264615774154663, 0.39402252435684204, -0.2570413649082184, -0.5476100444793701, 0.09242504090070724, 0.2552827000617981, -2.480820417404175, -0.4543454647064209, -1.0800409317016602, -0.7388522028923035, 0.21756522357463837, -0.11453951150178909, -1.4286634922027588, 0.8703808784484863, 0.6167792081832886, -0.8857625126838684, 0.6443515419960022, -0.015200786292552948, 1.892214059829712, 2.2618496417999268, 0.09446476399898529, -0.1190088540315628, -0.9869707226753235, 3.5382297039031982, -0.5514591932296753, -0.24647629261016846, 0.13643455505371094, 0.8540342450141907, 0.7037221193313599, 1.8295484781265259, -0.47157108783721924, -0.7020696401596069, 0.31481364369392395, 1.2878752946853638, -0.46280527114868164, -1.356824517250061, 0.597902774810791, -0.660811185836792, -0.005334869492799044, -0.06961972266435623, 0.7793782949447632, 0.08685065805912018, -1.6356397867202759, -1.3517802953720093, -0.46836963295936584, -1.137019395828247, 0.9734624028205872, 0.21594537794589996, -0.6268072128295898, -0.9003511071205139, -0.17922495305538177, 0.7368571162223816, -1.1252672672271729, 1.4202208518981934, -0.15227411687374115, -0.5143110156059265, 0.347164124250412, -0.19377750158309937, -0.5696770548820496, -0.5958111882209778, 2.147831678390503, 0.4940466284751892, 0.4778907299041748, -0.2661048471927643, -0.49938273429870605, -0.06236335635185242], [0.5128711462020874, -0.5304256677627563, -0.2836485803127289, 1.2583270072937012, -0.8985427021980286, 0.35872817039489746, -1.8314167261123657, -0.9637308716773987, -0.20778684318065643, -0.6214389801025391, 1.1814587116241455, 0.44717657566070557, -0.33394402265548706, 1.054159164428711, -0.6878491044044495, -0.7539179921150208, 0.8327152729034424, -1.8841334581375122, 0.9622239470481873, 0.26020488142967224, -0.7235737442970276, -1.1029605865478516, -0.11122214049100876, 0.04060359671711922, -0.5939971804618835, 0.6510939598083496, 0.26551878452301025, 0.9702306985855103, -1.1916546821594238, -0.5633487105369568, -0.585403561592102, -0.6022021770477295, -0.32763004302978516, -0.2777928113937378, 0.6864416003227234, 1.2039343118667603, 0.808070719242096, -1.7287510633468628, 0.4602040648460388, 0.5046814680099487, -0.2842564880847931, -1.5196096897125244, 0.9553273916244507, -0.25133949518203735, 0.8936567306518555, -0.13204960525035858, -0.06680285185575485, 0.8778870701789856, -1.7221975326538086, -0.9063955545425415, -0.7376438975334167, 0.4528932571411133, 2.008566379547119, 0.013909012079238892, -0.7228949666023254, -0.28557300567626953, 0.7945047616958618, -0.5193750858306885, -0.6297734975814819, -0.14289239048957825, -1.067397117614746, 2.4505765438079834, 0.5449285507202148, -0.26613473892211914, -1.0944887399673462, 0.6213437914848328, -1.2817820310592651, 1.2796584367752075, -1.7194983959197998, 0.569614827632904, -0.1511690765619278, 0.6395221948623657, -1.5040949583053589, 0.17726314067840576, -0.6030696630477905, -0.02414390631020069, -0.34927698969841003, -0.2936497628688812, 1.6082234382629395, 0.8468693494796753, -1.0102680921554565, 1.3277958631515503, -1.2809439897537231, -1.220630168914795, 0.24454551935195923, 0.5798736214637756, -0.13921323418617249, 1.150783658027649, 1.6925832033157349, 0.015576742589473724, 0.6614177823066711, 0.022026441991329193, -1.2968571186065674, -0.3980027139186859, -0.24651406705379486, -1.7938882112503052, 0.8817366361618042, -0.8106715679168701, -0.2473941147327423, 0.9257858395576477, -0.16673161089420319, -0.06413493305444717, 1.6901214122772217, 0.7841234803199768, 0.34500083327293396, 0.8030235171318054, 0.7262529730796814, -0.5734902024269104, -0.7700070738792419, 0.9002650380134583, -0.1444038301706314, -0.6307995319366455, 0.2643323540687561, 0.5860395431518555, -2.587524890899658, 0.009507277980446815, -1.5014526844024658, -0.5547770857810974, -0.5348071455955505, -0.0566534548997879, 0.17688289284706116, 0.2335679978132248, -0.2965105473995209, -0.24244649708271027, -0.8699790835380554, 2.1912379264831543, 1.3739447593688965, 0.338841050863266, 1.3983792066574097], [-0.5190134048461914, 1.5575687885284424, -1.4447846412658691, -0.5089920163154602, 1.2122801542282104, 0.07551887631416321, 1.3898169994354248, -0.5249113440513611, 0.736357569694519, -0.4018503427505493, 0.545708179473877, -0.20575661957263947, -1.2165452241897583, 2.049654006958008, -0.8832146525382996, 1.9863767623901367, -0.012802368961274624, -0.9590196013450623, 0.07765409350395203, -1.8184791803359985, 0.09629885852336884, 0.5005091428756714, -3.189115047454834, 0.5614320635795593, -0.07392136007547379, -0.44312503933906555, 2.1029670238494873, -0.677544355392456, -0.12344899773597717, -0.8739868402481079, -2.8235204219818115, 0.4829670786857605, -0.0007338634459301829, -0.6727702021598816, 0.7566688656806946, 0.41671526432037354, -1.3687052726745605, 1.4743825197219849, 1.2569438219070435, -0.7425704598426819, -0.051728714257478714, -1.0597240924835205, -0.1782159060239792, 0.8039413690567017, 1.6623538732528687, 1.6954383850097656, -1.1982659101486206, 0.4299539029598236, -1.424818754196167, -1.0803375244140625, 0.5965569019317627, 0.44851335883140564, -0.47013044357299805, -0.1371532529592514, -0.07733885943889618, 0.967918872833252, -0.2564718723297119, 1.4004185199737549, 0.09188748151063919, -0.6597416400909424, -1.1728910207748413, -0.9773085117340088, 1.2357512712478638, 0.30178600549697876, 1.5649173259735107, 0.017202777788043022, 2.2685136795043945, -0.06450483202934265, 2.618462562561035, -2.1488280296325684, 0.815944492816925, -2.1006765365600586, -0.4605969786643982, 0.02678476832807064, 1.9444730281829834, 0.021733304485678673, -0.7361739873886108, 0.7994698882102966, 0.38390982151031494, -0.3074697256088257, 1.6667569875717163, 0.7460689544677734, 0.7092337608337402, 1.4736199378967285, -0.08641226589679718, 0.7098323106765747, 0.03924954682588577, -0.265134334564209, -0.30867618322372437, 2.2670109272003174, 1.597644567489624, 1.246825933456421, 0.3156353235244751, -1.32261061668396, -0.7535480856895447, -0.49363645911216736, 0.6966980695724487, -0.6259382963180542, -1.1394176483154297, 0.2554747760295868, -1.1105812788009644, -0.16275723278522491, -0.13261087238788605, 0.00685901241376996, 0.4879169166088104, -0.6255735158920288, -0.9747280478477478, -0.8390831351280212, -0.9200764298439026, 0.9845342636108398, 0.43221303820610046, 0.39511731266975403, 0.022292090579867363, -1.3112645149230957, -1.0887272357940674, -0.5157000422477722, 0.4486422836780548, -0.5680012702941895, -1.4567633867263794, -0.03515821322798729, -0.14852972328662872, -0.46048635244369507, 0.07209203392267227, -0.3372836410999298, -1.213294267654419, 1.4045226573944092, 0.3863855302333832, 0.25205305218696594, 1.3295114040374756], [-0.4278832674026489, 1.4650448560714722, 2.0620224475860596, -0.2165892869234085, -0.22255279123783112, -1.2368048429489136, 0.758600652217865, -0.45385047793388367, 0.805920422077179, -0.9070391058921814, -0.20730221271514893, 0.2864883244037628, -0.020208338275551796, -0.8048983216285706, -0.7934588193893433, 1.489546298980713, 0.3523622453212738, -0.3260509669780731, 0.4903595447540283, -1.8025290966033936, -1.0624579191207886, -0.8625885844230652, 1.5961321592330933, 1.0214227437973022, -1.1682140827178955, -0.276632696390152, -1.0492117404937744, 0.8461725115776062, -1.6217353343963623, -0.1569390743970871, 0.006044617388397455, 1.0494705438613892, 1.164609670639038, 0.24416469037532806, 1.4668902158737183, -1.217433214187622, 0.8965014815330505, 0.30371224880218506, 0.275040864944458, 0.4247014820575714, -0.21933133900165558, -2.6742613315582275, 0.43924105167388916, 0.8801062703132629, 1.805446982383728, 0.07725457847118378, 1.513479471206665, 0.713121771812439, 0.6519100069999695, 0.3344869315624237, 0.9106405377388, -0.0933767780661583, 0.29217615723609924, 0.17392784357070923, -1.3491483926773071, -1.7655622959136963, -1.4631832838058472, -0.4704052805900574, 1.3489950895309448, -0.3190380036830902, 0.5602086186408997, 1.084515929222107, 0.001536484807729721, 0.8399422764778137, -0.9260067343711853, -0.08394869416952133, -0.09677472710609436, 2.156803846359253, 1.4451862573623657, 0.061998460441827774, -3.373701572418213, -0.3113371729850769, 0.8668463230133057, -0.9299823045730591, -1.0451215505599976, 0.5332642793655396, 0.16948005557060242, 0.9997644424438477, 1.3490982055664062, 1.0973188877105713, -1.0241187810897827, 0.2900392711162567, -0.601570188999176, -1.4137710332870483, -1.4563895463943481, 1.8982125520706177, 0.3432672321796417, 0.7261376976966858, -0.7765302658081055, -0.2352677434682846, -0.34201177954673767, -0.684781551361084, 0.28824302554130554, -1.8633548021316528, 0.10493875294923782, -1.0171985626220703, 0.6800845861434937, 0.180269256234169, -0.19261039793491364, -0.45383578538894653, -0.31479355692863464, 1.078532099723816, 0.09096274524927139, 0.3714948892593384, -0.4196989834308624, 0.31504085659980774, -1.5350854396820068, -0.6372556686401367, -0.4324799180030823, 1.5984606742858887, 1.8014334440231323, 0.09750388562679291, 0.9503852128982544, -2.061583995819092, -0.6810109615325928, 0.05196690559387207, 0.7207369208335876, 0.24501945078372955, 0.6691953539848328, -0.7558767795562744, -1.8571103811264038, 1.9375073909759521, -0.8146063685417175, -1.0703046321868896, 0.08859553188085556, -0.2653409242630005, 0.07525477558374405, -1.4451490640640259, -1.3188564777374268], [1.3426696062088013, -1.8284765481948853, -0.46459728479385376, -1.2492314577102661, 0.90449458360672, -0.25170454382896423, -0.24541804194450378, -1.2983949184417725, 2.008924722671509, 0.6035707592964172, -0.7379353642463684, 1.0530587434768677, -0.3666718900203705, 0.4547485411167145, 0.49815189838409424, -0.6258546113967896, 0.06876898556947708, -0.93975430727005, -0.35470467805862427, -1.5940483808517456, -0.7115353345870972, 0.3627297282218933, -2.193159341812134, -0.3196093440055847, -1.2224369049072266, -0.7724258303642273, 0.613278329372406, -0.8342418670654297, -1.4949009418487549, -0.726975679397583, 0.26583901047706604, -0.8476349115371704, -0.4460965394973755, 0.4617055356502533, 0.20019374787807465, -0.4552338719367981, -1.097525954246521, -1.1245514154434204, 0.32814842462539673, -0.4240027666091919, 0.34858018159866333, -1.1666629314422607, -1.4304695129394531, 0.6783096194267273, -1.997343897819519, -1.4025585651397705, 0.517438530921936, 1.0417578220367432, -0.6572598814964294, 2.462883710861206, -1.0338151454925537, -1.9126858711242676, -0.06093940883874893, 1.0625791549682617, 1.6375001668930054, -2.396458148956299, 0.42268723249435425, -0.39370694756507874, -0.5758159160614014, 1.6393027305603027, -0.4395674765110016, 0.3425220251083374, -0.5578734874725342, -0.2696503698825836, 1.0284000635147095, 1.7194819450378418, -1.786936640739441, -1.7743282318115234, 0.06967728585004807, -0.3893292248249054, -1.3054925203323364, 0.9646601676940918, -0.6857978701591492, 0.04773559048771858, 1.7383906841278076, 0.3611504137516022, -1.442226529121399, 0.03498619794845581, 0.17606842517852783, -1.2568832635879517, 0.0013631207402795553, -0.5802779793739319, -0.77391117811203, 1.9651674032211304, 0.7619802355766296, 0.3548835217952728, -0.18205328285694122, 0.24145740270614624, -0.4546077251434326, -0.1916850060224533, 0.4917713403701782, 2.1367554664611816, 0.6602137684822083, -0.39452269673347473, 0.3092745840549469, -2.135108709335327, 0.3172270357608795, -0.9452672004699707, 2.295182704925537, -0.41794660687446594, -0.21126702427864075, -0.8718629479408264, -0.3312951624393463, -1.5733489990234375, -0.8783329725265503, 0.11541067063808441, -0.004614486359059811, 1.1771364212036133, -0.932172417640686, 1.8153234720230103, -1.1979880332946777, -1.2179299592971802, -0.206461563706398, 1.1434842348098755, 0.8665139675140381, 1.7083747386932373, -0.8844826817512512, -0.08727606385946274, 1.064684510231018, 1.3524914979934692, 0.6605779528617859, -0.0513174869120121, 0.7776636481285095, -1.6090495586395264, -0.6048324704170227, -0.5212796926498413, -0.44540858268737793, -1.0748640298843384, -0.6457002758979797], [0.07861829549074173, -1.157015323638916, 0.018931778147816658, 0.30470624566078186, 0.01741570420563221, 0.17089998722076416, -0.05376432463526726, -2.001319169998169, -0.48354387283325195, -1.6799688339233398, 1.5346591472625732, -0.06496814638376236, -1.9213414192199707, -0.7783082127571106, -0.6795158386230469, 1.3807053565979004, 0.6260823011398315, -0.7366877794265747, 1.1247384548187256, 0.06571521610021591, -1.185351014137268, 0.038534387946128845, 2.0347702503204346, 0.03802060708403587, -1.398834228515625, -0.2223893105983734, -0.28573077917099, 1.535848617553711, 0.08957428485155106, -1.1331701278686523, -0.9930000305175781, -0.29725974798202515, -1.6693094968795776, -0.026474500074982643, 0.6743000745773315, -1.2364768981933594, 0.3373050391674042, 1.0876352787017822, -0.8123277425765991, 0.37960106134414673, -0.5358626246452332, 0.21170854568481445, -0.12206383049488068, 0.3946697413921356, -0.8543912768363953, 1.1173149347305298, 1.372982382774353, 0.1946968138217926, 1.5026112794876099, -0.6639245748519897, 0.9278584122657776, 0.2823053300380707, 0.2503568232059479, -1.7352206707000732, -0.918555736541748, 0.4346398711204529, 0.6173672676086426, -0.14173956215381622, -0.9165145754814148, 0.10842051357030869, -0.710429847240448, 0.10702501237392426, 0.6901639699935913, 0.19577273726463318, 1.0033783912658691, -0.03987345099449158, -0.8651083111763, 2.893885374069214, -0.41705620288848877, -1.4014718532562256, -2.3077666759490967, -0.1184927299618721, 0.16343048214912415, 0.970940351486206, -0.0027975437697023153, 0.40342992544174194, -0.46987515687942505, 1.3089298009872437, -0.18184597790241241, 0.09787799417972565, 1.2648530006408691, 0.06153649836778641, -0.700920581817627, -2.412829875946045, -0.4198238253593445, 0.45500028133392334, 1.2731729745864868, -0.053995102643966675, -1.3741307258605957, -0.9665859341621399, 0.8599739074707031, -0.4529068171977997, 0.31910231709480286, -0.2927541732788086, 0.39954301714897156, 0.4132850468158722, -0.023878730833530426, -2.141767740249634, -0.9084901809692383, -0.37159642577171326, 0.2560046911239624, -0.41634178161621094, -0.8104292750358582, -0.08867023140192032, -0.9349408149719238, 0.2932000756263733, -0.01344225462526083, 0.4950203001499176, 0.2626170814037323, 0.5522345900535583, 0.6595957279205322, -0.021510189399123192, -0.6237505674362183, 0.14727964997291565, -0.5296772122383118, -0.7870316505432129, 0.6352020502090454, -0.7736603617668152, -0.0843801274895668, -2.6248860359191895, -2.5430214405059814, 2.7490322589874268, -0.7404048442840576, -0.14249593019485474, -0.06317569315433502, 0.7115715146064758, 0.11818812787532806, 0.8321993350982666, -1.0992599725723267], [0.32359591126441956, -2.101727247238159, 0.011672855354845524, 0.8680099248886108, -0.2865886986255646, 0.46691882610321045, 1.7083773612976074, -0.2764561176300049, -0.44667550921440125, -0.7096774578094482, -1.4487392902374268, 1.5294129848480225, -0.6987227201461792, 0.13560689985752106, 1.1088221073150635, 1.4177770614624023, -0.23891597986221313, 0.6936715841293335, 0.598263680934906, -1.9054925441741943, 1.1885688304901123, -1.9168760776519775, 0.5561524629592896, -1.7255908250808716, 0.3986436724662781, 0.33733034133911133, -1.6133403778076172, -0.14711813628673553, 1.1800453662872314, -1.2323825359344482, 0.98813796043396, -0.2897503077983856, 0.6960372924804688, 0.644863486289978, -0.8421331644058228, 1.093181848526001, 0.5037265419960022, -0.007647581398487091, -0.2536851167678833, 0.10145152360200882, -0.4899294972419739, -0.27999216318130493, 0.5110526084899902, -0.5426021814346313, 2.3738656044006348, -0.6756870746612549, -0.2957574427127838, 0.5110520720481873, 0.48212870955467224, 0.04346199333667755, 0.9397702813148499, -0.47315266728401184, 0.0524645559489727, 0.7933613061904907, -0.9185190796852112, -0.08163846284151077, 0.31483620405197144, -2.3548061847686768, -0.1375594288110733, -0.7120072245597839, -0.8910618424415588, 0.4596579372882843, -1.062893033027649, 1.2294514179229736, -2.0626890659332275, 0.7874313592910767, -1.6250633001327515, 0.9958149194717407, 0.01797270029783249, -1.6165729761123657, -0.4141593277454376, 0.5911677479743958, -0.11593520641326904, 1.2392562627792358, 0.24925892055034637, -0.21020008623600006, 1.6169281005859375, 0.011774122714996338, 0.705725371837616, 1.171729564666748, -0.9765811562538147, 1.7815746068954468, 1.154487133026123, 0.6884408593177795, -0.6939284205436707, 0.4625382125377655, -1.2684365510940552, -0.21513257920742035, 1.5659408569335938, 0.43465906381607056, 3.189293384552002, -1.6723392009735107, -1.3541827201843262, -0.5710507035255432, 1.0548359155654907, -0.26690030097961426, -0.06393498182296753, -0.8848508596420288, -2.4117815494537354, 0.0751209631562233, -1.9175374507904053, -1.7776302099227905, 0.36826595664024353, -0.6131161451339722, -0.6409550309181213, -0.3400033712387085, 1.1668570041656494, -0.06335465610027313, -3.106199264526367, 1.162728190422058, -0.6821791529655457, 1.03563392162323, -1.3237617015838623, 0.4988478720188141, 1.7671817541122437, 0.3898394703865051, 1.663866639137268, -0.21387939155101776, 0.7769042253494263, 0.6632730960845947, -2.5417962074279785, 0.07405368983745575, 0.09604061394929886, 0.015516994521021843, -1.2591782808303833, 0.6427772641181946, -0.5191108584403992, 1.5002684593200684, 0.4052464962005615], [-0.6193512678146362, -1.1985732316970825, 1.4115080833435059, 1.6407842636108398, 0.28412920236587524, 0.2691976726055145, 0.09760444611310959, -0.15333834290504456, 0.2894483506679535, 1.8806300163269043, 0.4940001368522644, -0.8522541522979736, 1.1769105195999146, 0.15106777846813202, -0.722399890422821, 1.9839727878570557, 1.020056128501892, 1.6683255434036255, 0.016582995653152466, -0.22586241364479065, 1.1501561403274536, 0.7183226943016052, 0.7247190475463867, 0.22808444499969482, 0.3464716374874115, -0.4803331792354584, 1.3067184686660767, -0.0693681612610817, 2.059807538986206, 0.48887521028518677, -0.05892563238739967, 0.30220696330070496, 0.9316873550415039, 0.48295196890830994, 0.7003428339958191, -0.20427024364471436, -0.4948157072067261, -0.2890056371688843, -0.6281678080558777, -1.6734578609466553, 0.6801989674568176, 1.133377194404602, 1.3633639812469482, -0.5260208249092102, 0.7090278267860413, -0.10204929858446121, -0.08792196214199066, -1.6724518537521362, 0.22233480215072632, 0.33817076683044434, 1.2847596406936646, -0.8364982604980469, 0.735995352268219, -0.030611231923103333, -1.821494460105896, -1.5025920867919922, -1.1044607162475586, 0.33662381768226624, 1.0196369886398315, -0.11472361534833908, -0.08389443159103394, -0.41737478971481323, 1.0877199172973633, 0.12165436148643494, 1.0265201330184937, 1.5838844776153564, 1.439220666885376, 0.10576821118593216, 1.1770429611206055, 0.11008574813604355, 2.2039995193481445, -0.5502030253410339, 1.2656033039093018, 0.24079529941082, -0.23272502422332764, -1.2849414348602295, -3.106797456741333, 1.3561068773269653, -0.849957287311554, 1.3906654119491577, -0.1623600572347641, 1.313255786895752, 0.5213161110877991, -0.3307102620601654, 1.0712674856185913, 1.3976880311965942, -0.9929388761520386, -0.12491142004728317, -0.8835529088973999, -0.7211943864822388, -0.024790961295366287, 0.8050099015235901, -0.5967006683349609, 1.7858335971832275, 1.4169533252716064, 1.0769068002700806, -0.05331343412399292, 0.41649675369262695, 0.3230583369731903, -0.18317456543445587, -0.8269246816635132, -1.0030804872512817, -1.364001989364624, 0.8515992164611816, 0.4140433371067047, -0.7771599888801575, -0.2578718960285187, -0.6942084431648254, -0.7341629862785339, 1.539870023727417, -0.5694448351860046, -1.1844581365585327, 0.1569443941116333, -0.801423192024231, 0.14101475477218628, 0.04522883892059326, -1.608241319656372, 1.0437840223312378, -0.09914965182542801, -0.5021952986717224, 0.9828150272369385, -1.670555591583252, -0.8940420150756836, 1.1218581199645996, 0.8209176063537598, 0.4555952847003937, -1.2277570962905884, -0.8130027651786804, -0.2825072407722473], [-0.029552718624472618, -0.07344488054513931, -1.3486169576644897, 1.0910152196884155, -2.0715723037719727, -1.1374096870422363, -1.123591661453247, -0.7712036967277527, 1.4508249759674072, 0.37413105368614197, -0.6567620635032654, 0.8336186408996582, 0.6122493147850037, 0.5195813179016113, -0.38519951701164246, -0.2981790006160736, -0.035085875540971756, 0.3197571337223053, -0.15656031668186188, -0.1194109320640564, 0.6344497203826904, -0.8232423067092896, -1.2614073753356934, -1.1845377683639526, -0.13358083367347717, -0.5140485167503357, -0.09402734041213989, -0.2925654947757721, 1.0117343664169312, -0.08615747839212418, -0.4154111444950104, 0.798474907875061, 0.9242191910743713, 1.8860933780670166, 0.30221232771873474, -0.15037837624549866, -0.3505844175815582, 0.26753726601600647, 2.067579984664917, 1.2960256338119507, -1.3067117929458618, -1.037514090538025, 0.19763371348381042, -0.8247784376144409, 0.8021690845489502, -0.26609307527542114, 0.9629409909248352, 0.2983607053756714, 1.3705642223358154, 0.8104856610298157, 0.018067214637994766, 0.5058521032333374, 0.37330055236816406, 1.689290165901184, 1.8339998722076416, 0.5752074718475342, -0.6839051842689514, -0.6787333488464355, -1.1880302429199219, -1.8694829940795898, -1.9030578136444092, -2.595773696899414, 0.7092229723930359, 1.0686917304992676, -0.286171019077301, -0.3910068869590759, -0.9781267046928406, -0.6946601271629333, 0.32018277049064636, -1.3819701671600342, -0.8778509497642517, 1.1590912342071533, 0.08706918358802795, 2.167137384414673, -0.5046781301498413, 0.5540976524353027, 0.6550576686859131, 0.6509917974472046, -1.1601567268371582, 0.4097239673137665, 1.7889484167099, 1.625344157218933, 2.558166027069092, -0.2526136040687561, -0.8668947815895081, -0.9497296214103699, -0.566190242767334, 0.03839534893631935, 0.9340057969093323, -0.6171934008598328, -1.951664924621582, -0.18256954848766327, -0.6930527687072754, 1.0098381042480469, -0.4523114264011383, -0.2749904990196228, 0.17083632946014404, 0.20679713785648346, 0.8974158763885498, 0.8449339866638184, 0.15937118232250214, 0.3448580503463745, -1.1352298259735107, 0.6384207606315613, -0.9424852728843689, -0.8870552182197571, 0.8695204257965088, 1.3881421089172363, -0.6731442213058472, -0.01301636639982462, -0.13292641937732697, 0.24179190397262573, 0.1730843186378479, 0.5123880505561829, -0.8465268611907959, -1.2059853076934814, 0.32172277569770813, 0.5047823786735535, 1.2994664907455444, -0.14968837797641754, 1.4751390218734741, 1.5530427694320679, 1.0066211223602295, -0.3753820061683655, -0.6738876700401306, 0.9280381798744202, -0.07375560700893402, 0.9052051901817322, 1.4914876222610474], [-0.14223501086235046, 0.2315809428691864, -1.634572148323059, 1.3136931657791138, 0.2141784429550171, 0.17628608644008636, 0.10063612461090088, -1.0682793855667114, 0.984379768371582, 0.05306960269808769, -0.058895692229270935, -0.11462454497814178, -0.7948101162910461, -0.2768782675266266, -0.44025805592536926, 2.0137646198272705, -0.6234530806541443, 0.06282270699739456, 0.7979312539100647, -0.8038416504859924, -0.36722332239151, 2.7948110103607178, -0.9434340000152588, -0.3298170566558838, 0.44237732887268066, -0.3198481798171997, 0.884748637676239, -2.989833354949951, -0.11969407647848129, 0.6598420143127441, -0.8414410352706909, -0.39885494112968445, -0.6963747143745422, -0.29709580540657043, -0.7929984331130981, 0.65836101770401, -1.1117726564407349, 1.4747281074523926, 0.8686476349830627, -0.18136370182037354, 0.1620311141014099, 1.2807525396347046, 0.03250853717327118, 0.913867712020874, 0.8047198057174683, -0.11938977241516113, -0.29517000913619995, -0.4374140202999115, -0.15871858596801758, -0.9938743710517883, -0.09542756527662277, -0.8327997326850891, -0.4763736128807068, 2.1407127380371094, 1.1502759456634521, 0.31327906250953674, -0.10726188868284225, -0.3894657790660858, 1.6287871599197388, -1.3404555320739746, 0.24525286257266998, -1.039091944694519, -0.645940363407135, -0.8179232478141785, 1.1450862884521484, 0.4559989273548126, 1.097737431526184, 0.20171217620372772, 0.8380735516548157, -0.8740242123603821, -0.6910623908042908, -0.6805112361907959, 0.4829771816730499, 1.8822258710861206, -0.5371406078338623, -0.6424517035484314, 1.7843399047851562, -0.4543575048446655, 1.984296202659607, 0.06701046228408813, 0.25465962290763855, -0.5717743039131165, 0.9219650626182556, 0.9636223316192627, -1.1361249685287476, -1.5594924688339233, -0.02079811319708824, -1.3380697965621948, 1.5589789152145386, 0.06076626852154732, 2.828994035720825, -0.19194532930850983, 0.3618658781051636, -1.2751497030258179, 0.26763299107551575, 0.6065055727958679, -1.2093905210494995, -0.3678761422634125, 0.22680023312568665, -0.8564161062240601, -0.2624194920063019, 0.42387551069259644, -1.0760886669158936, -0.9424559473991394, -1.0885859727859497, -0.014277436770498753, -0.0807664692401886, 0.497192919254303, -0.5458793044090271, 0.573347270488739, 0.9880785346031189, 1.1223926544189453, -0.6602783203125, 0.6255049109458923, 0.7699289321899414, -1.0436851978302002, -2.039207696914673, -0.3373566269874573, -0.006760719232261181, -0.09797553718090057, 0.08149968832731247, -0.16507457196712494, -0.5206825733184814, 0.2939532399177551, 0.7169196009635925, -1.4789118766784668, -0.5064586997032166, 0.5239202380180359, 0.8397953510284424], [-0.49020877480506897, -0.9190903902053833, -0.21379511058330536, -0.028635410591959953, -0.3870796263217926, -1.5955055952072144, 1.026159644126892, 0.6703917980194092, 0.2580696940422058, -0.5784117579460144, 0.6866096258163452, 1.7872425317764282, -0.8164546489715576, -1.2678862810134888, -0.7895627617835999, 0.09432132542133331, -1.491738200187683, 0.9379398226737976, 0.8643438220024109, -0.8177959322929382, 0.7218550443649292, -1.522447943687439, -0.13300977647304535, -1.494937777519226, 0.807425320148468, 1.8161383867263794, -0.004578231368213892, 0.8996754884719849, -0.18475675582885742, 0.22224974632263184, 1.275686264038086, -0.41631314158439636, -1.4279354810714722, 0.5708869099617004, 1.39713716506958, 0.45429158210754395, 1.6105388402938843, 0.9017730355262756, 0.5852752327919006, -0.5300121903419495, -0.9057636857032776, 0.14721161127090454, 0.39862534403800964, 0.7133249044418335, -0.08455367386341095, -0.487883985042572, -0.5612192749977112, 0.01359137985855341, 0.5022639632225037, 1.720892310142517, 0.10480311512947083, 1.4124711751937866, 1.5778553485870361, -0.8320253491401672, 1.183004379272461, -0.12605975568294525, -0.16835351288318634, 1.062376856803894, -0.3108351230621338, -0.6257498860359192, -1.146998405456543, -1.183193564414978, -0.7449539303779602, 1.6810141801834106, 0.7828101515769958, 0.10104858130216599, -1.2632070779800415, 0.6461454629898071, 0.11633115261793137, 0.6408358812332153, -1.1238682270050049, 1.3031171560287476, -0.1298951953649521, 0.5771770477294922, -0.2130497395992279, 0.20207233726978302, -1.8077287673950195, -1.1094897985458374, 0.8511022329330444, -0.8565994501113892, 1.0918338298797607, -1.849412202835083, 0.1309470236301422, 1.4475243091583252, 0.3794366121292114, -1.0954034328460693, 1.4796695709228516, -1.3921961784362793, -0.5935235619544983, 0.5360193252563477, 1.0328010320663452, 0.32035112380981445, 0.559403657913208, -0.685994029045105, -0.8599913120269775, -0.4797494411468506, -2.037524461746216, 1.1092411279678345, -0.0020926513243466616, -1.2715798616409302, 0.4481571912765503, -0.3374834358692169, -0.522818922996521, -1.6672049760818481, 1.386620283126831, 0.3011070489883423, 0.8394401669502258, -0.14520779252052307, -1.3960142135620117, 0.5435963273048401, 1.7719653844833374, -0.8254005908966064, 0.6087092161178589, 0.6175032258033752, -1.2772103548049927, -1.0498028993606567, -0.6425913572311401, 1.6356436014175415, -1.3147344589233398, -0.1625216007232666, -0.376020610332489, -0.8586229085922241, -1.5383201837539673, 0.010581494309008121, -0.6757755279541016, 1.3270971775054932, -1.0593963861465454, -1.333339810371399, 0.7189195156097412], [0.8869202733039856, -0.277181476354599, -1.7308976650238037, -1.0331116914749146, 2.002930164337158, 1.8607838153839111, -0.7461351156234741, -1.3503587245941162, 1.1332452297210693, 0.033383775502443314, -0.7544086575508118, -0.35582268238067627, 0.6709980964660645, 1.739038109779358, -2.782045602798462, 0.48374128341674805, -0.5717282891273499, 0.9224352240562439, 2.115377902984619, -0.42561039328575134, 0.8722408413887024, -0.8586425185203552, -1.8995767831802368, 0.666660487651825, -0.37474608421325684, -0.2201400101184845, -0.7978173494338989, -1.9523545503616333, 1.4882543087005615, -0.09277606755495071, -1.8086869716644287, -1.132389783859253, -0.7487752437591553, -0.9083569049835205, -0.1398981362581253, -0.802501380443573, -2.1054625511169434, -0.20789413154125214, -1.1692763566970825, 1.7072685956954956, -0.9433761239051819, 1.5356632471084595, -1.3826898336410522, 1.3729482889175415, 0.6712296605110168, 0.3448399603366852, -0.15317274630069733, 0.07152340561151505, 0.6919167637825012, -1.0888104438781738, 0.16374258697032928, -0.04498972371220589, -0.3682697117328644, -0.06832819432020187, -0.6814391613006592, 1.128455638885498, 0.84990394115448, 0.25665712356567383, -0.8661645650863647, 0.4364303946495056, 1.7252612113952637, 1.1375967264175415, 1.4195284843444824, -0.13713839650154114, -3.093233108520508, -0.482565701007843, 1.1232366561889648, 0.223020538687706, 1.3150626420974731, -0.8754442930221558, -1.3568867444992065, -0.7440537810325623, 1.1694011688232422, -1.0429072380065918, 0.23588097095489502, 0.76619952917099, -1.9390320777893066, -0.2667056918144226, 0.22002103924751282, 1.3180361986160278, 1.0502254962921143, -0.03850368782877922, -1.0940759181976318, -1.7835884094238281, 1.3504765033721924, -1.327301025390625, 0.9067131876945496, 0.5330442786216736, -0.5645675659179688, -0.004048475995659828, -0.6213103532791138, 1.7204803228378296, 1.062671422958374, -0.8680039048194885, -0.18885667622089386, -1.570313811302185, -1.887027621269226, 0.36011749505996704, -0.5561224818229675, 1.3050241470336914, 0.8571038246154785, 0.4985843896865845, 1.7990303039550781, -0.5416749119758606, 0.670752763748169, -0.9051927924156189, 1.0520846843719482, -0.6086652278900146, -0.6923292875289917, 0.40598222613334656, 2.1111207008361816, -0.12262094765901566, 0.901341438293457, -0.343294620513916, 0.1330665946006775, 0.02700452134013176, -0.1695275902748108, 0.3866257071495056, 0.2806175649166107, 0.5779882073402405, -0.3948213756084442, 0.5971699357032776, 1.4555284976959229, 2.2363977432250977, -0.33266258239746094, 0.9085578918457031, -0.8849363923072815, -0.9155762195587158, -1.3068913221359253], [-0.5585949420928955, 0.16165496408939362, 1.7473880052566528, -0.25517475605010986, 0.3857206702232361, -1.674027681350708, 0.255173921585083, 0.30694159865379333, 0.6230117678642273, 1.2485089302062988, 0.43743181228637695, 1.429355502128601, 1.2479324340820312, -1.3475769758224487, -2.0385520458221436, 0.11094875633716583, 2.901238441467285, 1.1525218486785889, 0.8586364984512329, 1.1384224891662598, 0.23536956310272217, 1.1412580013275146, -0.06567392498254776, 0.6111973524093628, 1.0962719917297363, 0.19237416982650757, 0.0446382537484169, -1.755873203277588, 1.412158489227295, -0.49302762746810913, -0.8742428421974182, 1.2132797241210938, 1.4839757680892944, 0.7005364298820496, 0.5688316226005554, 1.568343162536621, -0.8543238043785095, -2.204470634460449, -1.5717626810073853, 0.8238117098808289, -0.4541822075843811, -0.4129805862903595, 0.29687583446502686, -0.00576609605923295, 0.3249197006225586, 1.2674481868743896, -0.4297432005405426, -1.3384021520614624, -0.4475020468235016, 0.6977856159210205, 1.1425459384918213, 0.3555487394332886, -0.07970290631055832, -1.2621231079101562, -2.0358006954193115, 0.48689550161361694, 1.4882175922393799, -0.08566483110189438, 0.622845470905304, 0.1981695145368576, 0.6924108862876892, -0.56071537733078, 0.7912574410438538, -0.9425341486930847, -0.7878658175468445, -0.6464067697525024, -0.6282597184181213, -0.5469419956207275, 0.10087795555591583, 1.2727633714675903, 0.02828441746532917, 0.2068856656551361, -0.9173610210418701, -1.8391722440719604, 1.0956957340240479, -1.203385353088379, 0.43399760127067566, 0.22929905354976654, 0.11173848062753677, -1.0702260732650757, 1.1728073358535767, 0.4154197871685028, -0.5518020391464233, 0.05759536102414131, -0.6056470274925232, 0.26946696639060974, -0.29973462224006653, 1.9116504192352295, 0.7498771548271179, 1.6155306100845337, -0.6267033815383911, -0.5605225563049316, 1.8343560695648193, -0.5058154463768005, -0.5239658951759338, 1.4587972164154053, -0.21176117658615112, 0.35417962074279785, -1.2296385765075684, 1.8613011837005615, -0.4587098956108093, -1.8803887367248535, -1.2356700897216797, -0.5752301216125488, 1.1173828840255737, -0.18236768245697021, 0.7100113034248352, 0.22250214219093323, -2.0132744312286377, 2.180032253265381, 0.3213319480419159, 0.01279896404594183, 1.4209601879119873, -1.9665379524230957, -0.4981323480606079, 0.7597070336341858, 0.7148154377937317, 0.7575903534889221, -2.097186326980591, -1.0086464881896973, 0.24031369388103485, 1.3135933876037598, 0.46399208903312683, 0.3273663818836212, -0.126552551984787, -1.266611933708191, 1.6309548616409302, -2.8334124088287354, 0.11349902302026749], [-0.32201725244522095, 0.018229935318231583, -1.650744080543518, -0.5427161455154419, -1.7096083164215088, 1.515794038772583, -0.32980260252952576, -0.20755831897258759, -1.9543864727020264, 0.7496715784072876, 0.02990378998219967, 0.16418567299842834, -0.870314359664917, 1.4750667810440063, 0.7188122272491455, -0.7739178538322449, -0.5704929232597351, -0.42074283957481384, 1.3745158910751343, 0.7735081911087036, -0.897656261920929, -1.2223831415176392, 0.17834702134132385, -0.5288140177726746, -1.6959582567214966, 0.48495063185691833, 1.7379164695739746, 0.04823299124836922, 0.2584281861782074, 0.9379885792732239, -0.5873027443885803, 0.7939202189445496, -1.3660467863082886, -0.17569050192832947, 0.8343648314476013, 0.3017488420009613, -0.31716856360435486, 0.2793510854244232, 0.7889112234115601, -2.083704948425293, -1.6045886278152466, 0.13825072348117828, -0.08847006410360336, 0.7644356489181519, 0.2592322528362274, 0.7832649350166321, 0.9685038328170776, -0.0778774544596672, 0.44586119055747986, -0.5631528496742249, -0.26033028960227966, 0.583227813243866, 1.1938397884368896, 0.15045687556266785, 0.2542734444141388, -0.778634786605835, -0.2942947447299957, -1.3453353643417358, 1.2776525020599365, 0.9319413304328918, 2.990628957748413, 0.4895690977573395, 1.2976408004760742, 1.0004950761795044, -0.3256690502166748, -0.11948103457689285, -0.08063028752803802, -0.027960598468780518, 1.5242851972579956, 0.24839811027050018, -0.673907458782196, 0.8798869252204895, 1.0603628158569336, -1.1383895874023438, -0.8673028945922852, 0.3712676465511322, 0.44196513295173645, -0.992285430431366, 0.8851332068443298, -0.690758228302002, 2.0253450870513916, 1.6277306079864502, -0.13731873035430908, 0.12441276013851166, -0.3553767800331116, 1.0689802169799805, -2.158376693725586, 1.8374131917953491, -1.4823485612869263, -0.26968976855278015, 1.3374189138412476, -0.2536650598049164, -0.4575998783111572, -1.7485042810440063, -0.5467215776443481, -1.2777888774871826, -0.08987696468830109, -0.8066468834877014, -0.07884813100099564, -1.0513707399368286, 0.015492938458919525, 0.28366488218307495, -0.8755088448524475, 0.5143834352493286, -0.6694098711013794, -0.3209245800971985, 0.6472450494766235, 0.37397634983062744, 0.5903846621513367, -2.2078351974487305, 2.2170989513397217, -0.013588139787316322, 0.6543778777122498, 0.9711205959320068, -0.8928059935569763, -0.5481693744659424, 0.7468022108078003, 0.6188520789146423, -1.3785618543624878, 0.5460857152938843, 1.742151141166687, -0.9633195996284485, -1.2267879247665405, 0.5901917815208435, 0.13590890169143677, -1.0042325258255005, 0.05151379108428955, 0.7333409786224365, 0.5767114162445068], [0.9360010027885437, -0.30782362818717957, -0.2141813337802887, 0.04718320444226265, -0.0751969963312149, 1.1360174417495728, -0.22385671734809875, 0.7829889059066772, 1.6262881755828857, -1.2935067415237427, -1.880371332168579, -0.022970957681536674, 0.8975345492362976, 0.3807264566421509, -0.0592103935778141, 0.4079146385192871, -1.4968080520629883, -0.31826043128967285, 0.7880709767341614, 0.5680928826332092, 1.6057755947113037, -0.6685810685157776, 0.20109353959560394, -0.9038164615631104, -0.37001416087150574, -2.3439993858337402, 0.7748448252677917, -0.016366751864552498, 1.4279249906539917, 0.2400100827217102, -0.6651012301445007, -1.0327792167663574, -1.708309292793274, 0.19907750189304352, 0.9061537981033325, 0.25433140993118286, -0.27501752972602844, -0.631117582321167, -1.9381616115570068, -0.3893648087978363, -0.631321132183075, -1.7831337451934814, 0.5460245609283447, 1.023128867149353, 0.4234585165977478, -0.8359004855155945, -0.5234180688858032, -1.5271151065826416, 0.34061717987060547, -1.677788257598877, -0.8004541397094727, -0.6208335161209106, -0.05919504165649414, -1.410104751586914, 0.2191368192434311, 1.594199299812317, -0.4092157185077667, -0.03312089294195175, -1.2305678129196167, -0.02435920014977455, 0.2620735466480255, -0.8313979506492615, -0.5520691275596619, -0.4916173219680786, 0.00230684713460505, -0.4657530188560486, 0.6709553599357605, 0.01883213222026825, -0.8787549734115601, -1.175258755683899, -2.1667838096618652, -0.5521913766860962, 0.8899834156036377, -1.8562449216842651, 0.7490293979644775, -1.7097450494766235, -0.7866619825363159, 1.0501028299331665, -0.0003812916693277657, -0.845818281173706, 1.748244047164917, 0.7575792670249939, 0.9150103330612183, -0.607863187789917, 1.1382495164871216, -0.5572682619094849, -0.22227011620998383, -0.2462274134159088, -1.5675623416900635, -1.2429481744766235, 1.7216367721557617, -1.6028811931610107, 1.0745477676391602, -1.645433783531189, -0.8923491835594177, -0.1882087141275406, 0.07077126950025558, -1.0384507179260254, 0.4139106273651123, 0.1902739256620407, -1.4580644369125366, 0.45321759581565857, -2.0815999507904053, -0.10648346692323685, -0.20859453082084656, -0.868880033493042, -0.7647240161895752, -1.072723150253296, 0.39534327387809753, 0.41959819197654724, -0.7590426206588745, -0.3251941502094269, -0.8165119886398315, 1.4915697574615479, -0.9595891237258911, 1.1114808320999146, -1.6746636629104614, -0.04053058847784996, -1.208691954612732, -0.5914341807365417, -1.4203726053237915, 1.1686381101608276, -1.4321067333221436, -0.34113648533821106, 0.7529226541519165, 0.21605558693408966, 0.20517094433307648, 1.7514793872833252, 0.6564998030662537], [-1.5902155637741089, 0.744907021522522, -0.1833735853433609, 0.6417600512504578, -0.5413233041763306, 0.4339025616645813, -2.3951268196105957, -0.49829789996147156, 0.4018794000148773, -0.452192485332489, -0.4391518235206604, -0.15701612830162048, -0.6736728549003601, -1.0189892053604126, 1.5110182762145996, 0.7425318360328674, -1.0528804063796997, 1.0924608707427979, 0.8694482445716858, -1.8072749376296997, 0.9347418546676636, -0.1994554102420807, 0.5364553332328796, -0.2589119076728821, 0.11917169392108917, -1.0359641313552856, -0.3705950975418091, 0.877076268196106, -0.9480239748954773, -0.29200735688209534, 0.11137179285287857, 1.24283766746521, 0.08534356206655502, 1.2814940214157104, -0.6103447675704956, 0.8809207677841187, -2.1096765995025635, -1.9728918075561523, 0.4716376066207886, 0.31419798731803894, -0.20321710407733917, -1.393778920173645, -0.7950303554534912, 0.490571528673172, -2.392561674118042, 1.5329073667526245, -0.10273599624633789, -0.1945880502462387, -0.11930175125598907, -1.4231033325195312, 1.0074058771133423, -0.6039883494377136, -0.5207675695419312, -0.050284285098314285, -0.6297666430473328, 0.16638270020484924, 0.01515245158225298, 0.8933272361755371, -0.6633383631706238, -0.08549266308546066, -1.7069858312606812, 0.9957143664360046, -4.746011734008789, -1.9474693536758423, -0.5788792371749878, -0.244010329246521, -0.9398673176765442, -1.0389742851257324, -0.18077096343040466, -1.2970317602157593, -0.2617383599281311, -1.3782293796539307, 0.06929177045822144, -1.8860598802566528, -2.235781192779541, -1.977358341217041, -0.0002731652057264, 0.23123031854629517, -0.7958837151527405, -0.7093236446380615, 0.5181522965431213, 0.1414751261472702, 0.3409115672111511, -0.742169976234436, 1.0253524780273438, -0.20374637842178345, 1.5238184928894043, 0.6745664477348328, -2.8437752723693848, 0.25305259227752686, -0.039271220564842224, -0.9627583026885986, -1.4220143556594849, -0.09071878343820572, 2.538381814956665, 0.08310206979513168, -3.5577712059020996, -0.1223272904753685, 0.7885255217552185, 1.4013057947158813, 1.2205250263214111, -0.47562435269355774, -1.0991065502166748, -1.32430100440979, 1.2278931140899658, 1.231947660446167, -0.5704468488693237, 1.417714238166809, -0.9378984570503235, -0.7918304204940796, -1.321514368057251, 1.474554181098938, -0.6829487085342407, 0.02938469685614109, 0.8998270630836487, -0.12694737315177917, 1.0347511768341064, 0.7343474626541138, 0.7022220492362976, -0.4211897850036621, 0.009798138402402401, -1.1573694944381714, 1.1044903993606567, 0.13025473058223724, 0.20235136151313782, 0.48658356070518494, 1.3841749429702759, 0.40760764479637146, -0.7019076943397522], [-0.09148268401622772, -2.0287561416625977, -0.020542241632938385, -1.2386434078216553, -1.1008256673812866, -1.566483736038208, -0.7005301713943481, 1.6024079322814941, 3.1069753170013428, 0.5195130705833435, -0.6653209328651428, -2.158705234527588, -0.7750634551048279, -1.6363898515701294, -1.0733877420425415, -0.7075581550598145, 0.05335447937250137, -0.7983070611953735, 0.6641597747802734, 0.31715911626815796, -0.8692222237586975, 0.8195886015892029, -0.5634963512420654, 1.6532588005065918, 1.4179494380950928, -0.03478313609957695, -1.1942825317382812, -0.4749320447444916, -1.410878300666809, -0.04762725904583931, 1.2555619478225708, -0.15484915673732758, -0.8428948521614075, -0.2198096513748169, 1.014981985092163, 1.211411952972412, 0.11918092519044876, -0.3619844317436218, -0.01935419626533985, 0.5115406513214111, -0.09523119777441025, 0.14293673634529114, 1.3978012800216675, 0.3118217885494232, -0.6316347718238831, -1.191957712173462, 0.3383651077747345, 1.1593934297561646, -0.537943959236145, 0.8840667605400085, 0.34174618124961853, -0.8316394090652466, 0.9431957602500916, -0.6949065923690796, 0.6838201284408569, -0.1099129468202591, -0.002618467668071389, 0.4838173985481262, -1.7670507431030273, -1.8750300407409668, 2.2093141078948975, 0.4265727698802948, 1.0878607034683228, 0.5475813746452332, 0.9153650999069214, 0.7074702382087708, 0.35018229484558105, -0.27439117431640625, 0.15696291625499725, -0.6606072783470154, 0.364600270986557, 1.907507061958313, 1.7289934158325195, 1.032965898513794, -0.0030392277985811234, 0.4264852702617645, 0.1881462186574936, -1.6157199144363403, -0.21120165288448334, -1.5044161081314087, 1.4455957412719727, -0.5514383912086487, 0.49789002537727356, -0.1972532719373703, -1.5843112468719482, 0.013258450664579868, -1.029557704925537, 0.8935410976409912, -0.5493823885917664, 0.024994883686304092, 2.499004364013672, 0.5867785215377808, 0.389055073261261, 1.111525535583496, -1.432490587234497, 0.5915749669075012, -2.4295759201049805, 0.19041217863559723, -0.7868223190307617, 0.1417296975851059, 0.6340967416763306, -0.9095526337623596, -0.9091195464134216, -0.7907012701034546, 0.2471538782119751, 1.9681477546691895, -0.4238751530647278, 1.123934030532837, 0.17738871276378632, 0.6443889737129211, 1.7333064079284668, 0.8696199655532837, 0.3885088562965393, -1.5445313453674316, -0.3958149552345276, -1.1826627254486084, 0.6248638033866882, -0.8819575309753418, -1.7492856979370117, -0.5950249433517456, 0.63029545545578, -0.11216679215431213, -0.8332377672195435, -1.99513840675354, -1.553568959236145, -1.4259140491485596, -0.35350558161735535, -1.0344419479370117, -0.2262108474969864], [1.369384527206421, -0.3752979040145874, -1.3882431983947754, -0.28894975781440735, 1.2586560249328613, -0.17227497696876526, -0.949106752872467, 0.25973987579345703, 1.6626893281936646, -1.9181863069534302, 0.9255250692367554, 0.6781947016716003, 0.4074666202068329, 0.8156191110610962, 2.2787177562713623, -2.0766899585723877, -0.4573807120323181, -0.9866809248924255, 0.28129488229751587, -0.25957050919532776, -0.9304327964782715, 0.6792735457420349, 0.6665518879890442, 1.0994224548339844, 0.06833722442388535, -1.275294303894043, 0.4135844111442566, 0.40294045209884644, -1.0945624113082886, -2.3079094886779785, -1.7956981658935547, -1.3293906450271606, -0.3985336720943451, -1.0418789386749268, 0.7079377174377441, 1.0613142251968384, -0.18510568141937256, -0.627652645111084, -1.0137289762496948, -0.20155467092990875, -1.0524122714996338, -0.279382586479187, 0.4731846749782562, 1.4115780591964722, 1.654790997505188, -0.7729246020317078, 1.6269060373306274, 0.1941378265619278, 0.8752893209457397, 0.6348835825920105, -0.11708174645900726, 0.9912768602371216, 0.6836665272712708, 0.4236879348754883, 0.044356975704431534, 1.2411648035049438, -0.480267733335495, -2.187490701675415, -0.1253167986869812, -0.5760602951049805, 0.15862728655338287, 2.521630048751831, 0.4098225235939026, -0.12863047420978546, -1.5132333040237427, 0.11143476516008377, -0.29676225781440735, -1.053199052810669, -1.291106104850769, -0.8467250466346741, 0.2722408175468445, 0.6937357187271118, -0.669387698173523, 0.47507503628730774, -1.3743706941604614, -1.0044318437576294, 0.061138223856687546, -0.5617479085922241, 0.7131822109222412, 0.7294089794158936, 0.7026939392089844, -0.09255529195070267, 1.2543745040893555, -1.3548225164413452, 0.24881257116794586, 2.1865427494049072, 0.04038462042808533, 0.26805517077445984, 0.7779330015182495, -0.0648714154958725, -0.5319445133209229, 0.708167552947998, -0.024126803502440453, -0.8798118829727173, 1.2204581499099731, 1.07763671875, 1.4186718463897705, -0.11362956464290619, -0.3610887825489044, 0.0676182433962822, -0.801476776599884, 1.1814651489257812, -0.23039157688617706, 0.44896817207336426, -0.18099771440029144, 0.2743709087371826, -0.03883178159594536, 1.6372370719909668, -1.1780927181243896, -0.3105142414569855, 0.25627338886260986, 1.4385125637054443, 0.15398861467838287, -0.016198283061385155, -1.0919716358184814, 0.09181740134954453, -0.2091096192598343, 0.7456277012825012, -1.5047203302383423, -0.7468227744102478, 2.164994955062866, -1.6782753467559814, 0.4286406338214874, 0.14639459550380707, 0.6153419613838196, 0.7582148909568787, -1.3105871677398682, 0.8192344903945923, 0.09675127267837524], [0.08118196576833725, -0.46155810356140137, 1.9052958488464355, 1.7549850940704346, -0.5989377498626709, 0.41708049178123474, -0.2980448603630066, 0.7850479483604431, 0.1921212077140808, 1.0370047092437744, -2.3323190212249756, 0.9241865277290344, -0.3939635157585144, -1.4298949241638184, -1.1106513738632202, 0.04772777482867241, 1.2766984701156616, -0.9629577994346619, 2.067368507385254, 0.5027000904083252, -0.47783488035202026, -1.3372467756271362, 0.34074243903160095, -1.8000454902648926, -0.7475487589836121, 0.91445392370224, 0.8030543923377991, 0.5984316468238831, 1.9264034032821655, -0.7323150634765625, -1.720407485961914, -2.128352165222168, -0.6846678853034973, 0.660758912563324, -1.5685210227966309, 0.5012421607971191, -0.7278808355331421, 0.28933343291282654, -0.9350853562355042, 0.8945680856704712, 0.2999834716320038, -0.8976749777793884, 1.8123958110809326, -0.29652246832847595, 0.5767846703529358, 1.5462749004364014, -0.35612571239471436, -0.12790325284004211, -0.31542715430259705, 0.9462136030197144, 0.6607563495635986, 0.19255666434764862, 0.8129971027374268, 0.8113318085670471, -0.6041830778121948, 0.0371735543012619, 0.7981622815132141, -1.1819161176681519, -1.218666434288025, -0.3543778359889984, 0.3089735507965088, -0.303713321685791, 1.6535018682479858, 0.47695815563201904, 0.1824682503938675, -1.592239260673523, 0.9205474853515625, -1.1406383514404297, -0.4365406334400177, -0.8228392601013184, -0.6983437538146973, 0.47352609038352966, -1.1372389793395996, 0.9368027448654175, 1.2242786884307861, 0.19160380959510803, 0.47629818320274353, 1.1803841590881348, 0.35490918159484863, -0.3223775029182434, -0.0010481717763468623, 0.18482930958271027, 0.5411760807037354, 0.3466593623161316, 0.121376171708107, -0.1487518846988678, 0.8352662324905396, 0.5613211989402771, -0.03247944638133049, 0.01701841503381729, -0.23050615191459656, 1.892673373222351, 1.7315884828567505, -0.3227737247943878, -1.3469898700714111, -1.8781930208206177, 1.67721426486969, 0.3234681487083435, -1.205247402191162, -0.10366813093423843, -0.6510275602340698, 0.3821181356906891, -0.9575223922729492, -0.49150702357292175, 1.4464699029922485, 0.7099555730819702, 2.0984277725219727, 1.4065656661987305, -0.611722469329834, 0.35572344064712524, -2.276165723800659, 0.308517724275589, -0.47116994857788086, 0.6118491291999817, -1.9454247951507568, -1.5119980573654175, 0.9248149394989014, -0.30187392234802246, 0.814039945602417, -0.39083224534988403, -0.2473428100347519, 0.04133393242955208, -0.7645408511161804, 1.026441216468811, -0.7332592606544495, -0.6054869294166565, -0.9243348240852356, 0.13194260001182556, 1.1384354829788208], [0.5347214937210083, -0.6978234648704529, 0.206736221909523, 0.45775431394577026, -1.8497387170791626, -1.2081377506256104, -0.4076346158981323, 0.8128394484519958, 0.3302845060825348, 0.220635786652565, -0.3246200382709503, 1.820671558380127, 1.1360280513763428, -0.13491159677505493, -0.9910096526145935, 0.23410747945308685, -1.0798920392990112, -1.559583306312561, -0.6230787634849548, 0.5852922201156616, -0.064033642411232, 1.7499325275421143, 1.3500288724899292, 0.6934625506401062, 1.1849181652069092, 0.15618273615837097, -0.07019328325986862, 0.7971758842468262, -0.26273781061172485, -0.004525236319750547, -0.24800078570842743, -1.0602048635482788, -0.17299266159534454, 0.5093227624893188, -0.09919100999832153, -1.07733952999115, -1.5116361379623413, -0.7482486963272095, -0.20561091601848602, -0.6291259527206421, -0.08162064850330353, 0.7574113011360168, -0.17942458391189575, -1.0080294609069824, 0.29048165678977966, 0.4653204381465912, -1.1367063522338867, 0.5583462715148926, 0.37127941846847534, 1.0211713314056396, -1.1828094720840454, 1.09430992603302, -0.8976486921310425, -2.0192363262176514, -1.0410003662109375, 0.0004479947965592146, -1.6466339826583862, 0.14258301258087158, -1.9651578664779663, -1.178897738456726, 0.6346826553344727, 0.7539008855819702, -0.975961446762085, 1.403996467590332, 0.6322638988494873, -0.15371638536453247, -1.8018362522125244, -1.3057091236114502, -2.9679579734802246, -1.050583839416504, 1.0807126760482788, -0.08100774139165878, -0.696922779083252, 1.994245171546936, 0.9584899544715881, -1.1945897340774536, 0.29497960209846497, -0.4566057324409485, -0.30819904804229736, -1.2241430282592773, 0.7555445432662964, 0.20549972355365753, 1.8969900608062744, 0.16346469521522522, -0.4723733961582184, 0.014591407030820847, -1.09821355342865, 1.0286513566970825, -1.7039746046066284, 0.8900555968284607, 1.0304211378097534, 2.6383962631225586, 0.2851526439189911, 0.5116601586341858, -0.16512535512447357, 0.4757966101169586, 0.5189806222915649, -0.37845489382743835, 1.8384593725204468, 0.43345803022384644, -1.3611547946929932, -1.210114598274231, 0.8867613673210144, 1.1221238374710083, 2.173701524734497, 0.8225803971290588, 0.6368649005889893, -0.9616546630859375, -0.7767834663391113, -1.148579716682434, 2.781041145324707, -1.994958758354187, 1.5495102405548096, -1.117897391319275, -0.4624751806259155, -0.33939602971076965, -1.8395217657089233, -0.8468108773231506, -0.6872459650039673, 0.8063072562217712, -0.5510997772216797, -0.3138926029205322, 0.6561884880065918, 0.35822606086730957, 0.308222234249115, -0.5062973499298096, 1.254744291305542, 1.197116732597351, 0.2044067531824112], [-0.8526766300201416, -0.26982730627059937, -1.3195093870162964, 0.006340762600302696, -1.3103668689727783, -0.6216317415237427, 0.8036893606185913, -1.3577170372009277, 0.7566272020339966, 0.3804813027381897, 0.2058061808347702, 0.07338722050189972, -1.177133321762085, 0.7928304672241211, -1.004022479057312, 0.6247584223747253, -1.0442116260528564, 0.3505164384841919, 0.8322606086730957, -0.8784787058830261, -0.016192320734262466, -0.8684108853340149, -0.5610047578811646, 0.7966135144233704, -0.5729200839996338, -0.7359460592269897, -1.5816587209701538, 1.6960369348526, 0.6589868068695068, 0.733973503112793, -1.2101343870162964, -1.420416235923767, -0.6872311234474182, 0.395243376493454, -0.46030592918395996, -0.2882322371006012, 0.1444682627916336, -2.900438070297241, 0.9002652764320374, -0.5049330592155457, -3.2874581813812256, -0.8511899709701538, -0.052712731063365936, 0.7025068402290344, -1.495246410369873, -0.40211305022239685, -0.9263847470283508, -0.017761649563908577, 1.126029133796692, 1.125024437904358, 0.9810953140258789, -0.1811227649450302, -1.4315499067306519, -0.616723895072937, -0.4578569233417511, 0.7692515254020691, 1.1489872932434082, 1.01284658908844, 1.2600014209747314, -0.9321973323822021, -0.5843592882156372, 0.48377764225006104, 0.3920289874076843, -0.14075256884098053, 1.0677648782730103, 0.14229793846607208, -0.9366233944892883, -0.9061379432678223, -0.3201964199542999, -0.31845495104789734, 0.5240668654441833, 1.0511740446090698, 0.7270562648773193, -1.2985281944274902, -0.09275862574577332, -0.24172644317150116, -0.7828463315963745, -0.44104617834091187, 1.5013659000396729, -1.3065767288208008, -0.3957136869430542, -0.44119396805763245, 0.7530215978622437, -0.28128230571746826, -0.3194635212421417, 0.6949463486671448, -0.5074993968009949, -0.16677793860435486, 1.4338303804397583, 0.4242628812789917, 1.2923556566238403, 0.6075676083564758, 0.35592591762542725, -1.6470229625701904, -0.3807717561721802, -0.4265471398830414, 2.630532741546631, -1.087286353111267, -0.2640736699104309, 0.4134658873081207, -0.5257229208946228, 0.2490098476409912, -0.6325307488441467, -0.7129005193710327, -1.0301384925842285, 0.2389947474002838, 0.2368306666612625, -0.014400692656636238, 1.7695674896240234, 2.416822910308838, 2.000498056411743, 0.5073423385620117, -0.16687893867492676, 0.031846288591623306, 1.1905601024627686, 0.5804070234298706, 1.7666538953781128, -0.1355392336845398, 0.341423362493515, 1.3654826879501343, 1.052139401435852, -0.2685871422290802, -0.1874057501554489, -0.919822096824646, -0.6729228496551514, -0.7370606064796448, 1.030527949333191, 0.6107245683670044, -0.11640052497386932], [-1.4284167289733887, -0.9758089184761047, -0.46575623750686646, -0.5201038122177124, -1.0374187231063843, 2.1818008422851562, -2.177438735961914, -1.8295114040374756, 2.323775053024292, 0.5788772702217102, 0.5857277512550354, -1.0152074098587036, -1.804762601852417, 0.25356078147888184, -0.48046571016311646, -0.1719866842031479, 0.11153657734394073, 0.16700932383537292, 0.14293700456619263, 2.42826509475708, -2.0731770992279053, 1.6133772134780884, -0.1244240403175354, -0.7543472051620483, -0.34738877415657043, 0.5387533903121948, 0.5278387665748596, -2.09320330619812, -0.7249677777290344, 0.4023939371109009, -0.17632707953453064, 0.23552170395851135, 0.27026498317718506, 1.6890405416488647, 0.25412049889564514, 2.104252576828003, 0.5485073328018188, 0.9769002199172974, 0.38036587834358215, -1.4123588800430298, 0.7467535138130188, -0.021349318325519562, -0.9028244018554688, 0.8930718302726746, -0.3213270902633667, 0.6159989833831787, 1.3895010948181152, 1.255600929260254, 0.519030749797821, 1.2125123739242554, 1.48634672164917, -0.654408872127533, 0.9717164635658264, -0.15934769809246063, -2.0341603755950928, 1.194528341293335, -1.0308353900909424, 1.0582268238067627, -0.1976201981306076, 0.773623526096344, -1.202967643737793, -0.18954509496688843, -0.1124810203909874, 0.9989652037620544, -0.5830614566802979, -1.4445931911468506, 0.9823630452156067, -0.31698179244995117, 0.836569607257843, -1.2856942415237427, -0.010906828567385674, -0.6074753999710083, 0.18566147983074188, 0.8506292104721069, -0.7906363606452942, 0.9404649138450623, 0.15082859992980957, -1.5697215795516968, 0.6773877739906311, 1.380579948425293, 0.16873736679553986, -0.9048922061920166, -0.36963191628456116, -0.6312583088874817, -0.949777364730835, 0.46065089106559753, 1.7203986644744873, 0.30048301815986633, 1.2721292972564697, 0.2849251925945282, 1.4937752485275269, 1.4312341213226318, -0.20770308375358582, 1.1583771705627441, 0.022291377186775208, 1.6952028274536133, 0.06566804647445679, 0.7476333975791931, 0.539190948009491, -0.763483464717865, -1.0085575580596924, 0.1419779360294342, -0.329403817653656, -1.037879467010498, -0.6056878566741943, 0.5892004370689392, -1.1610840559005737, -0.7816799879074097, -2.1338517665863037, 1.2302151918411255, -0.24259623885154724, 0.05776981636881828, -0.9377678036689758, -0.2913626432418823, -0.6467480063438416, -1.3596922159194946, -1.6538794040679932, 1.2375962734222412, -0.8888466954231262, -3.1202685832977295, -0.5896636247634888, -0.3002411425113678, -0.31550484895706177, -0.01934489980340004, -0.40711766481399536, 0.3503150939941406, -1.2116230726242065, -1.8423936367034912, -1.335491418838501], [0.6134812235832214, 0.5632944107055664, 0.11150013655424118, 0.9452760815620422, -1.773239254951477, 0.5406899452209473, 0.35688087344169617, -1.2149090766906738, 0.15079618990421295, -0.3638303577899933, -0.2006557285785675, -1.647519826889038, 0.4545753002166748, -0.809396505355835, 0.09332004189491272, 0.8773238062858582, -1.2968103885650635, -0.765189528465271, -1.8204848766326904, 0.6506235599517822, 1.5025348663330078, -0.34615910053253174, -0.2826978862285614, 0.509745180606842, -0.2259276807308197, 0.14018532633781433, -0.5821225643157959, -0.1495184451341629, 0.8997585773468018, -0.6483113765716553, 0.3640143573284149, -0.4384881854057312, -0.17481905221939087, -0.3182697296142578, 0.7539570331573486, -0.8683170080184937, 0.055203113704919815, -1.2087116241455078, -0.34453752636909485, -0.5434778928756714, -0.7951616048812866, -0.07032688707113266, -1.0144731998443604, 0.6104469895362854, -0.015564561821520329, 0.24361135065555573, 1.1883729696273804, 0.21565434336662292, 0.9587889909744263, -1.0498911142349243, 0.9769195914268494, 2.0804331302642822, -0.9818565249443054, -1.2712510824203491, 0.6665719151496887, 2.137817859649658, -0.6325357556343079, 0.43880853056907654, 0.5928587317466736, -0.6736212372779846, 0.5602851510047913, -0.3443010151386261, -0.8284319043159485, -0.7749521732330322, 0.4277386963367462, -0.2742401361465454, 1.2340178489685059, 0.8990662097930908, -0.8792478442192078, -0.8660639524459839, 0.6471024751663208, -0.91194087266922, 0.5894433856010437, 0.441751629114151, -1.4488872289657593, -0.6104300022125244, 2.053386926651001, 1.7514041662216187, -0.5472201108932495, 0.3199746608734131, 0.23345111310482025, -0.791755735874176, -0.4515575170516968, 0.1823163479566574, -0.45763930678367615, 1.4016612768173218, 0.592022180557251, 0.41473984718322754, 0.01876433938741684, -1.2053585052490234, -0.24825914204120636, -0.6652984023094177, -0.3018973469734192, -0.9512782692909241, -0.031085466966032982, 1.0280190706253052, 1.4251917600631714, 3.221189022064209, 0.3664699196815491, 0.9902549386024475, -0.8319249153137207, 0.5854055881500244, -0.7826747894287109, -0.39876589179039, -0.045158155262470245, 0.4270865023136139, -0.04583200812339783, -0.6090892553329468, -1.5621272325515747, 2.624570608139038, -1.8225016593933105, 1.4093921184539795, 0.9691932797431946, 1.3148623704910278, 2.0340816974639893, 1.665583848953247, 0.8016803860664368, -1.1604020595550537, 0.5856171250343323, -1.7804956436157227, 0.976211667060852, 1.4911609888076782, -1.2820854187011719, -0.9028593301773071, 1.454833745956421, 0.645160973072052, 0.6968733072280884, -0.018385285511612892, -0.9981657862663269], [-0.3349044620990753, 2.9009838104248047, 0.5135571360588074, 0.6237580180168152, 2.0945024490356445, -1.1065540313720703, -0.021564355120062828, 0.6997478008270264, -0.05535624921321869, 1.0983332395553589, -0.8474928736686707, -1.6710145473480225, -0.5530638694763184, 1.8313485383987427, 0.8242653012275696, -0.7469592094421387, 1.0004918575286865, 0.25020912289619446, -0.6671592593193054, 0.4183451533317566, -0.5476207137107849, 0.25476959347724915, -0.11606978625059128, 0.7390168309211731, -0.15310147404670715, -0.8663741946220398, 1.5976799726486206, -0.010233545675873756, -0.3868582248687744, -0.318812757730484, -0.5008837580680847, 1.3246877193450928, -0.7323782444000244, 0.6327158212661743, -0.14356110990047455, -0.003320040414109826, -0.1488981992006302, -0.312833696603775, -0.6511054039001465, -0.34497374296188354, 0.639543890953064, 0.8408633470535278, -1.9813227653503418, -1.1155574321746826, -0.7393603324890137, 0.31663671135902405, 0.15948262810707092, -0.4116511642932892, -0.11183977872133255, -2.0108585357666016, 1.6777149438858032, -0.6603504419326782, -1.0318279266357422, -0.5589740872383118, -0.7213684916496277, 0.10837380588054657, 0.3388127088546753, 0.06468956917524338, 0.6059659123420715, 0.8485094308853149, 1.8899537324905396, -2.188173532485962, 1.3620636463165283, -0.8321189284324646, -0.5732945203781128, 0.05105409398674965, -1.6892889738082886, 1.0479525327682495, -2.1661572456359863, -0.24478411674499512, 0.050626810640096664, 0.35542216897010803, -1.3892086744308472, 0.7092402577400208, -0.8762076497077942, -0.4943807125091553, 1.0520552396774292, -0.2034800499677658, -1.3219560384750366, 0.4941472113132477, -0.17449311912059784, 0.6713517904281616, -1.0784035921096802, -1.2189048528671265, -0.9646316766738892, 0.20133081078529358, -0.15962935984134674, -1.0766665935516357, -0.36798566579818726, 0.8540439009666443, -0.6925670504570007, -0.2680482864379883, -1.1935147047042847, -0.49809738993644714, 0.387319415807724, 0.3121444284915924, -0.18035100400447845, 2.2332100868225098, -0.23803387582302094, -0.4043918251991272, -1.816868782043457, 0.17370787262916565, -1.4615802764892578, 0.9020912647247314, -0.7170402407646179, -2.3811583518981934, 0.11070568859577179, -0.6090039610862732, 1.1640167236328125, 0.24164456129074097, 0.29082825779914856, -0.3569985032081604, -0.22865788638591766, -1.0895235538482666, -0.9073629379272461, -0.9510307908058167, -1.0222346782684326, 0.8903239369392395, -0.012759567238390446, -1.4054615497589111, 0.8736581206321716, -0.8322597742080688, -0.10415245592594147, 0.8586589097976685, 0.4341646432876587, 0.5170089602470398, 0.22209033370018005, -1.5612704753875732, 0.32271042466163635], [-0.5488830804824829, -0.4087297320365906, -1.1435977220535278, 0.06316406279802322, -0.18033255636692047, -0.08133704215288162, -0.799426794052124, -0.2348950207233429, -2.40887451171875, -0.2997354567050934, -0.30902934074401855, 1.0899808406829834, -0.23157480359077454, -1.1217041015625, 1.2486262321472168, -1.086475133895874, 0.3380078375339508, -0.6973842978477478, -1.1433662176132202, 0.5876495838165283, -0.4024973511695862, -1.4003106355667114, 2.060157060623169, -0.2553853690624237, -0.07340412586927414, -0.7833037376403809, 0.3324238359928131, -1.1165006160736084, 0.6020615696907043, -0.3767867684364319, -0.5438567996025085, -1.1024930477142334, -0.5762654542922974, -0.5448136329650879, 0.3005902171134949, -0.9111839532852173, -0.6674185395240784, 0.020000487565994263, 0.6004096865653992, 0.9282549023628235, -0.854292094707489, -2.008646249771118, 0.14151912927627563, -1.4604554176330566, 1.324696660041809, -0.5387420058250427, -0.46616217494010925, 1.6840132474899292, -1.4788670539855957, 1.3106858730316162, -0.39339005947113037, -0.5213680267333984, 1.8789379596710205, -0.18047581613063812, -0.7153612375259399, -0.22245720028877258, 1.083079218864441, 0.7837745547294617, 0.9557026028633118, 0.6622301936149597, 0.29417359828948975, -1.5649738311767578, -1.3757250308990479, -0.27260690927505493, -2.189173936843872, 1.9889508485794067, 0.15665028989315033, -1.2218689918518066, -0.14111702144145966, 0.7132260203361511, 0.7792854905128479, -0.3595599830150604, 1.0574772357940674, 1.2399173974990845, 0.6941852569580078, 3.3178110122680664, -1.053162693977356, 0.4618150293827057, 0.899178683757782, -1.0752296447753906, -0.20542356371879578, 0.037470705807209015, 0.08350138366222382, -1.6637935638427734, 0.6898669004440308, -0.3992357552051544, 1.3231091499328613, -0.5950470566749573, -0.7575840950012207, 0.41043028235435486, -0.4405500292778015, 1.3964228630065918, -0.47330954670906067, 0.3367815911769867, -0.8670418858528137, -0.05101872980594635, 1.185255527496338, -1.4002821445465088, 0.7289454936981201, 0.039576563984155655, 0.446109414100647, -0.9666749835014343, -0.2075212299823761, -1.7362475395202637, 0.6798727512359619, -1.4846924543380737, -0.7695034742355347, -0.1360415518283844, -0.7475860118865967, 0.30748969316482544, -0.07647451013326645, -0.08856136351823807, -0.7638409733772278, 0.15887799859046936, -0.20599499344825745, -1.618586540222168, 1.9567112922668457, 0.2642251253128052, -1.1319327354431152, 0.4112526774406433, -0.028678301721811295, -0.6586377024650574, -1.3357996940612793, 1.9437248706817627, 0.41999828815460205, 0.26191338896751404, -0.1407393217086792, -0.5267382264137268, -1.4907337427139282], [1.060050129890442, 1.6180535554885864, 1.9073714017868042, 0.2917536497116089, 0.6631481051445007, -0.2249251902103424, -0.6181172132492065, 1.0930452346801758, -0.2426937073469162, 0.7634090185165405, 1.2407610416412354, 0.34023019671440125, 0.6117701530456543, -0.3840010464191437, -0.9590920209884644, -0.5962849855422974, 1.6936404705047607, 1.3404313325881958, 0.49528488516807556, -0.386538565158844, 0.15706010162830353, 0.664060115814209, 0.9025636911392212, -1.6850128173828125, 0.7146389484405518, -0.016343265771865845, -0.9260430932044983, -1.5758016109466553, -0.3811667263507843, 0.9228852391242981, 2.698183059692383, 1.2245160341262817, 1.164323329925537, -0.724543035030365, -0.2555983066558838, 0.8250031471252441, 0.16847620904445648, -0.7450722455978394, 1.6621513366699219, 0.5327481627464294, 1.1166350841522217, -0.8055083155632019, 0.15470603108406067, 1.5218796730041504, 0.8235185146331787, -1.3282997608184814, 0.09936996549367905, 0.8174191117286682, -0.5211326479911804, -1.5137561559677124, -0.5097774267196655, -0.32173898816108704, 0.10417889803647995, -0.16783596575260162, -0.17100822925567627, 0.8093565702438354, -1.669911503791809, -0.9529470205307007, 1.2361478805541992, 1.5565191507339478, -1.764561414718628, 1.0317434072494507, 0.5542884469032288, -0.1130036935210228, -1.561561107635498, -1.3018946647644043, 0.6842159628868103, 0.9333295822143555, -1.2154314517974854, -0.41754773259162903, 1.6815851926803589, 0.8318639397621155, -0.4643928110599518, 0.9365459084510803, 0.4385960102081299, -0.3448432385921478, 0.08078949898481369, -0.9233382940292358, -2.220647096633911, -1.622869610786438, -0.34919774532318115, -0.06749515235424042, 2.90669846534729, -0.8606377840042114, -0.4695107042789459, -0.4016718566417694, 1.2386906147003174, 2.2497503757476807, -0.3072964549064636, 1.4126992225646973, -0.6834753155708313, 0.5901927351951599, -0.8519662022590637, 0.8576853275299072, 1.474696397781372, -1.8233087062835693, -0.05894457921385765, 1.1960968971252441, 0.6664647459983826, -1.3509936332702637, -2.1495044231414795, 0.854329526424408, 0.2856251001358032, -0.43320053815841675, -1.3503926992416382, -0.6222190856933594, -1.6898835897445679, -1.2232792377471924, -0.03861411660909653, -1.5458786487579346, -1.0339558124542236, -0.9374569654464722, 0.27848687767982483, 0.3759959638118744, 0.28033214807510376, 0.25128158926963806, -1.3864123821258545, -0.15279801189899445, 0.7766644954681396, -0.44356095790863037, 0.7394810318946838, -0.10374028235673904, 1.0131031274795532, 2.4876012802124023, -0.9785395860671997, -1.0087255239486694, 0.1161813735961914, 0.6430444717407227, -0.11821286380290985], [-0.6256091594696045, 0.5218848586082458, 1.5116219520568848, 0.12676678597927094, 0.0913519486784935, 2.479614019393921, 1.2891672849655151, -0.47057974338531494, 0.03823902830481529, 1.2826838493347168, 0.6260842084884644, -0.7177966237068176, -0.6946607232093811, 0.3121539354324341, 0.4057409465312958, -0.9731329083442688, 1.5018703937530518, 1.567484736442566, 0.168680340051651, 1.009533405303955, -0.5176720023155212, 0.100494883954525, 0.5378463864326477, -0.37553104758262634, -0.008291898295283318, 0.963193416595459, -0.01910998299717903, -0.02955828234553337, 1.5537830591201782, 0.8139394521713257, -1.991240382194519, 1.3509390354156494, -1.6823160648345947, -1.798416018486023, -1.0932400226593018, -0.4147285223007202, -0.6989035606384277, -0.4801540970802307, -1.5767652988433838, -0.6485739350318909, -0.5361467003822327, -0.2982749938964844, 0.32497864961624146, -1.7513761520385742, -1.891711950302124, -1.1923882961273193, 1.3219044208526611, -1.9554885625839233, 2.0341904163360596, 0.8544784784317017, -0.4995441138744354, -0.519642174243927, 0.8789669871330261, 0.5277642011642456, -0.8947518467903137, -0.4010493755340576, 2.5720415115356445, -1.4250560998916626, 1.072487711906433, -0.08762602508068085, 0.5483588576316833, 0.12469788640737534, 1.8183643817901611, -0.8061032295227051, 0.87159264087677, -1.716021180152893, 0.03958909586071968, 1.1597050428390503, -0.8728192448616028, 0.7602300643920898, -1.269747257232666, 0.5879287123680115, 2.3942689895629883, 2.2092196941375732, 0.3946062922477722, -0.7131265997886658, -0.3201068937778473, 0.7791697978973389, -0.6071459650993347, 0.47038906812667847, 1.2370103597640991, 0.49480175971984863, -0.9609445929527283, 0.36211901903152466, 0.4328708350658417, -1.9944965839385986, 0.13203975558280945, 1.9204179048538208, 0.05956876277923584, -0.7697862386703491, 0.4748706519603729, 0.9015558362007141, 0.6062793731689453, -0.5668370723724365, 1.0537214279174805, 1.9956855773925781, -1.1600786447525024, -0.2117737978696823, 0.35472381114959717, -0.45334193110466003, 0.6143956780433655, 0.740040123462677, -0.2932308316230774, 1.04500412940979, -1.4008902311325073, -0.11192262917757034, -0.24260209500789642, -0.17486461997032166, 0.13252893090248108, 1.5806500911712646, 0.1309853196144104, 0.217420756816864, -1.1336907148361206, -0.23105603456497192, 1.412958025932312, -0.9274459481239319, 0.18343877792358398, 1.1681066751480103, -1.0434961318969727, 0.5252644419670105, -0.029580585658550262, 0.14084148406982422, -0.6573012471199036, 2.733877658843994, -0.28482604026794434, 0.7982611060142517, -0.7279486060142517, -2.040550470352173, 0.14555992186069489], [-0.48554790019989014, 0.4305431544780731, -1.25982558074611e-06, 1.0705997943878174, 1.1870030164718628, 0.43288201093673706, 1.993259310722351, 0.26845014095306396, -0.39106664061546326, -0.6696602702140808, -0.11304716020822525, 1.1155362129211426, 0.7301204800605774, -1.1180139780044556, -0.8324161171913147, -1.1238347291946411, -0.11070802807807922, -0.2917635440826416, 2.0019192695617676, 1.2079027891159058, 0.7971482872962952, -1.061022162437439, 1.3551905155181885, -1.3098236322402954, -0.1429608017206192, 1.1186254024505615, 0.3244991600513458, -0.40822598338127136, 1.428321361541748, -2.2544167041778564, -0.08189081400632858, 0.5709099173545837, 0.38691607117652893, -0.08854391425848007, 0.5904014706611633, 2.2363526821136475, 0.19876600801944733, -0.6520734429359436, -0.6016778945922852, 1.0005742311477661, 0.7340928316116333, -0.41384199261665344, 3.132143974304199, -1.4216963052749634, 0.2107851505279541, 1.214112401008606, 0.2281818687915802, -1.6525956392288208, -1.642420768737793, 1.6094733476638794, -0.6445738673210144, -2.4421448707580566, 0.7455025315284729, 0.6788351535797119, -0.1381179541349411, 0.4189113974571228, 1.5026906728744507, -0.5206174850463867, 0.23305976390838623, -0.3682669997215271, 0.4988737106323242, -1.1989083290100098, 1.6796261072158813, -0.002821570262312889, 0.7833744287490845, -0.3065299093723297, 2.5358622074127197, -2.1421127319335938, 0.10542415082454681, -1.7133007049560547, 0.5731636881828308, -0.13370375335216522, 0.5250406265258789, -0.16482247412204742, 0.27503490447998047, -1.3540773391723633, 0.7658283710479736, 0.3144904673099518, -0.5586770176887512, -0.3287167251110077, 0.003906433470547199, 1.062448263168335, 1.940381646156311, -0.7310570478439331, -1.9064143896102905, -0.3356422185897827, 0.7878365516662598, 0.4080171287059784, 0.3654536008834839, 0.6361704468727112, -0.0579875111579895, -0.5467633605003357, 1.4140241146087646, -0.735095202922821, -0.10592305660247803, 1.7409629821777344, -1.4188878536224365, -0.18293754756450653, 0.6378718018531799, 1.3924084901809692, 1.8652770519256592, -0.31219983100891113, 1.1596460342407227, 1.315967321395874, 0.3488483428955078, 1.6339832544326782, 1.0359225273132324, -1.5663225650787354, -1.5977858304977417, -0.8966525197029114, 1.0544719696044922, 0.13256527483463287, -1.2297102212905884, -1.298642873764038, -0.7898833751678467, -0.3014315664768219, -0.41589781641960144, -0.776452898979187, 1.1405621767044067, 0.825523316860199, -0.876714825630188, -0.6535218358039856, -1.222245454788208, -0.033866140991449356, -0.2602055072784424, -1.1640619039535522, 1.2037286758422852, -0.49113622307777405, 0.5103242993354797], [0.6268130540847778, 0.8517841696739197, -1.3932684659957886, -0.391907662153244, -0.46845415234565735, -0.34180110692977905, -0.6983304619789124, -0.39264193177223206, -1.0706459283828735, -0.9231951832771301, -0.46642643213272095, 0.5985053777694702, 0.027356186881661415, 0.48631399869918823, 1.497031331062317, -0.8052619695663452, 1.2703065872192383, 0.6057647466659546, 0.8430390954017639, -0.18893219530582428, -1.3097437620162964, -0.3111250400543213, -1.5654752254486084, 0.22152023017406464, -0.458175927400589, -0.830466091632843, -0.6533895134925842, -0.5840305685997009, -0.33989784121513367, -2.5195350646972656, -1.1670948266983032, -1.4792784452438354, -0.8881264925003052, 0.08803731203079224, 0.8872222900390625, 1.025821328163147, -1.3949447870254517, -0.4360148310661316, -0.01914433389902115, -1.1649752855300903, -0.3459642827510834, -0.23165585100650787, 0.357103168964386, 0.17914581298828125, 0.5215580463409424, -2.1070473194122314, -0.34939828515052795, -0.01377821620553732, -0.4856780469417572, -0.45610707998275757, -0.26680609583854675, 1.517622470855713, -1.339440941810608, 0.8505162596702576, -0.5079988241195679, -0.6702447533607483, 0.3979428708553314, 0.264768123626709, 0.964055597782135, -2.6609046459198, 2.318481206893921, -0.24136947095394135, 1.5972814559936523, 0.2074892371892929, -1.284874439239502, 1.3791680335998535, -0.3465738296508789, -0.5856103897094727, 0.8243125081062317, -0.186345174908638, -1.0889447927474976, 0.31129446625709534, -0.6682309508323669, 0.4727558493614197, 0.9738289713859558, 0.5320331454277039, 0.1467200368642807, -0.7964988946914673, -0.3737751543521881, 0.1572352796792984, -0.3850531578063965, 1.5469111204147339, 0.2444208562374115, -0.4051761031150818, -0.9686038494110107, -0.16874442994594574, 0.7105840444564819, -0.7980184555053711, -1.4745616912841797, 0.0035161315463483334, -0.8764078617095947, 0.7004897594451904, -0.5880934000015259, 0.12325334548950195, -1.8488634824752808, -0.812240719795227, 0.395264208316803, 0.3847367763519287, 0.6536381840705872, 1.0202128887176514, 0.25873780250549316, 1.4906795024871826, -0.3056114614009857, 0.28779396414756775, 0.6568365097045898, -1.4278545379638672, -1.0788600444793701, 0.12145035713911057, -1.1302107572555542, -0.19621224701404572, 0.2570545971393585, -0.6780363917350769, 2.587099552154541, -0.5758596658706665, -1.0075422525405884, 0.9627355337142944, -0.5891108512878418, -0.8919402956962585, 0.059980250895023346, 1.045725703239441, 0.07189405709505081, -0.7628113031387329, 0.019557125866413116, 0.5152974128723145, 0.8802873492240906, -0.42652419209480286, -0.5254558324813843, -1.780580759048462, -0.052418582141399384], [-1.3169654607772827, -0.060591746121644974, 0.05829666182398796, 0.22818028926849365, 1.105889081954956, -0.19198940694332123, 0.76643306016922, -0.4626396894454956, 0.24825729429721832, -0.8612842559814453, -0.09410911053419113, -1.2791271209716797, 1.0798957347869873, 0.5883100032806396, -0.2349015325307846, 0.7508483529090881, 0.7863715887069702, -1.000054121017456, 0.7315406799316406, -0.9397244453430176, 1.9181711673736572, 1.2032537460327148, 1.5296562910079956, 0.668533205986023, 0.2762066721916199, -0.41355106234550476, 0.40044966340065, 0.34602591395378113, 1.0000593662261963, -0.5721530914306641, 0.4796174168586731, 0.11428973078727722, 0.42441850900650024, 1.2229996919631958, -0.39849135279655457, 0.44255322217941284, 1.1871625185012817, 1.2723119258880615, 0.6892138719558716, 0.2959814667701721, 0.9057656526565552, -0.5144907236099243, -0.6138185858726501, -1.53891122341156, 0.8664478063583374, 0.3480928838253021, 0.6675153374671936, 0.1459316611289978, 0.3866719901561737, 2.202573299407959, 0.31908339262008667, 0.7073583006858826, 0.36856788396835327, -0.8886384963989258, 0.7921645641326904, 1.455060601234436, 0.12776827812194824, 0.21923673152923584, 1.0696097612380981, -2.1260197162628174, 0.03238878399133682, 0.6064867973327637, 1.289989709854126, -0.272198349237442, 0.470244437456131, 0.2623058557510376, -1.2089310884475708, 0.28292107582092285, -2.344456672668457, -2.2591958045959473, 0.7747371196746826, 0.9668290019035339, 1.0006142854690552, -1.0634180307388306, -0.07145242393016815, 1.441178321838379, -0.2834879755973816, -1.4933278560638428, -0.4357260763645172, 1.8323373794555664, -0.025407372042536736, 0.5513644218444824, 0.42553412914276123, -0.935340404510498, 0.3785231411457062, -0.5476269721984863, 1.261765956878662, 1.0023424625396729, 0.8914353847503662, -1.586204171180725, -0.9800666570663452, 0.2645035982131958, 2.2704341411590576, 0.20084550976753235, 0.241065114736557, -1.8492698669433594, 0.266624391078949, -0.7031691074371338, -1.6609125137329102, 1.142080307006836, -0.21536916494369507, 1.2682899236679077, -0.9428983330726624, -2.520718574523926, -0.1561916470527649, -1.7453911304473877, -0.7532939910888672, 1.2391571998596191, -1.4288337230682373, 0.41406092047691345, -1.981203317642212, 1.0748339891433716, -0.10976778715848923, -0.570168137550354, -0.10059510916471481, -0.06899924576282501, 0.6976136565208435, -0.3930727243423462, -0.9174569249153137, 0.7775211930274963, -0.14841865003108978, 1.5332555770874023, 0.500603199005127, 1.8140214681625366, -1.562173843383789, 0.2891702950000763, -0.4922352135181427, -0.29773712158203125, 0.8551750779151917], [1.8213131427764893, -0.1320144087076187, -1.663560390472412, -0.14991207420825958, -0.294817715883255, 0.5736647844314575, -0.9373873472213745, -1.5041731595993042, -0.008305254392325878, -1.3896753787994385, -1.019554853439331, 0.656531572341919, 0.7537802457809448, 0.011676538735628128, -0.4513228237628937, 0.3517219126224518, -1.2054147720336914, -0.2722066640853882, 2.1189486980438232, 1.3463075160980225, 0.106791652739048, -0.09976404160261154, -2.188765525817871, -0.531117856502533, 2.388967514038086, -0.9244477152824402, -0.26352378726005554, 1.9033557176589966, -0.1212579756975174, -0.3443867266178131, -0.13083012402057648, 0.7226008176803589, -0.8603496551513672, -1.045828938484192, -0.5542502999305725, -1.0905139446258545, 0.39557671546936035, 0.1728041172027588, -0.6081897616386414, 0.3930979073047638, -0.5886135101318359, 2.0683517456054688, 0.8170151710510254, 1.0311280488967896, -0.14426405727863312, -0.09038730710744858, 0.060161348432302475, -0.6870183348655701, 0.296607106924057, -0.33382198214530945, -0.39766645431518555, 0.20504115521907806, -0.18815621733665466, 0.08866868913173676, 1.6611933708190918, 0.6528490781784058, 0.3563821017742157, 0.6291500926017761, -1.3049371242523193, -0.2770426571369171, 0.806343674659729, -0.12062306702136993, 2.2206382751464844, 0.7354170680046082, 0.06519225984811783, -1.199566125869751, 1.2555756568908691, -1.642386794090271, -1.7847031354904175, -0.010212991386651993, -0.9809674620628357, 0.49031686782836914, -0.29059144854545593, 1.0677690505981445, 0.4355211555957794, -0.9932432770729065, 0.4718731939792633, 0.4728292226791382, 0.2446538507938385, -0.8856410384178162, -0.1610788106918335, 1.2135111093521118, 1.529396414756775, 1.3269309997558594, 0.4717153310775757, 0.5407911539077759, 0.2476043403148651, -1.6883199214935303, -0.39664673805236816, 0.7969332933425903, -3.4684159755706787, 0.07880407571792603, 1.3309069871902466, -0.33190232515335083, 0.12782521545886993, -0.7780399918556213, -1.4217922687530518, 0.16184990108013153, 0.4905208945274353, -0.003517904318869114, 0.9088786244392395, -0.423177033662796, 1.1554100513458252, -0.629176676273346, 0.5147320628166199, -0.1811530590057373, 2.911194086074829, -2.342557668685913, -0.5676224231719971, -0.4030027985572815, -0.28698691725730896, 0.5817384123802185, 2.133561849594116, 1.5023906230926514, -0.580906093120575, 1.0125094652175903, -1.277434229850769, -0.31001099944114685, 0.6109191179275513, 1.2180893421173096, 0.12225915491580963, -0.6814168691635132, 0.510526716709137, 0.2733684778213501, -0.6710827350616455, -0.8738687634468079, 0.12883728742599487, 0.6657261848449707, -1.158948302268982], [1.2514479160308838, -0.6712464690208435, 0.4911181628704071, -1.3921808004379272, -0.6756889224052429, -0.1298123598098755, -0.3520149886608124, -0.8890870809555054, -0.9820334911346436, 0.016395853832364082, -0.8606854677200317, -0.05717745050787926, -0.036439377814531326, -0.8610391020774841, -0.43444764614105225, -0.48615744709968567, 0.6583497524261475, -0.3157722055912018, -0.33648374676704407, 0.7368545532226562, 0.29802531003952026, -0.1957671344280243, -1.0052399635314941, -0.3971386253833771, -0.8005784153938293, 0.32331910729408264, 0.9156454205513, -0.21348056197166443, -1.8672723770141602, 0.9646051526069641, 0.927856981754303, 0.6946033835411072, -0.6820339560508728, 0.6054558753967285, -0.9925389289855957, 0.8554376363754272, -1.6089646816253662, -0.07472410053014755, 1.9580752849578857, 0.05982128158211708, 1.2965757846832275, 0.5705598592758179, 0.0754980817437172, -0.287664532661438, 1.8451809883117676, 1.3986902236938477, -0.017462126910686493, -1.89582097530365, 0.6236304640769958, -0.4022887051105499, -0.5440446138381958, -0.4256766438484192, 0.4635697603225708, 0.4650328457355499, 0.6154413819313049, -0.2305309921503067, -0.5011919140815735, -0.23917315900325775, 0.1254982203245163, -1.0453145503997803, 0.3959207236766815, -0.4742797017097473, -0.04402611777186394, 0.6101734042167664, 0.21950240433216095, 0.4179835915565491, 0.15366120636463165, 0.9592999815940857, 0.7617444396018982, -0.031164079904556274, -1.8672832250595093, -0.19377334415912628, -1.0994352102279663, -1.3738454580307007, 0.5577887892723083, 0.792447030544281, -1.6207880973815918, 0.11281125992536545, 0.6040810346603394, 0.11110733449459076, -0.3391059935092926, -0.629783570766449, 1.0229233503341675, 1.1679857969284058, -0.8734164834022522, 0.5461387038230896, -0.40066778659820557, 1.2218334674835205, 0.6777735352516174, 0.5655867457389832, -1.369386911392212, -0.8625293970108032, 0.5583257079124451, -0.9284646511077881, 1.155274510383606, -1.2651152610778809, -0.8385226130485535, 0.3929145038127899, 0.23306594789028168, -2.35422945022583, -0.3633473515510559, -0.32570910453796387, 0.48083266615867615, -0.269919753074646, -0.2839204668998718, 0.08932382613420486, -1.2392534017562866, 0.4260208010673523, -1.2902846336364746, -1.4865505695343018, 2.344573974609375, -0.0013331595109775662, 1.3698152303695679, 0.04930853843688965, -0.87965327501297, 1.2469547986984253, -0.6031021475791931, -0.381494402885437, 0.5191599726676941, -0.10420461744070053, 0.9705002903938293, 0.05754639580845833, 1.1304612159729004, -1.4174659252166748, 0.9275802373886108, -0.3611488938331604, -0.7525802254676819, -0.05354828014969826, -0.3875563442707062], [-1.4583985805511475, -0.9527625441551208, 1.308070182800293, -0.3759738802909851, -1.644530177116394, 0.616530179977417, -1.6447774171829224, -1.8958419561386108, 0.4767919182777405, -0.46231842041015625, -0.7620449066162109, -1.0948046445846558, 0.8088518381118774, 0.6315622925758362, -1.220646858215332, -2.2416131496429443, -1.0626851320266724, 1.3119614124298096, 0.4418678879737854, -2.553593873977661, 1.4746346473693848, 0.3143728971481323, -0.41461989283561707, 0.5290470719337463, 0.3451477885246277, 0.04954686015844345, 0.03669393062591553, 0.010973813943564892, -0.05270564556121826, 0.5413345098495483, -0.5006395578384399, 0.29461345076560974, 0.43495383858680725, 1.1450501680374146, 0.8687800168991089, 1.1713135242462158, -0.7573760151863098, -0.496194988489151, 1.5446555614471436, 0.9420267939567566, -1.2702709436416626, -0.5446178913116455, -0.9399297833442688, 0.36071106791496277, 1.0956381559371948, -0.6583444476127625, 1.5598533153533936, 0.8429496884346008, -1.210152268409729, 0.6773008704185486, -1.1650680303573608, -1.0690836906433105, -2.172149419784546, -1.7067623138427734, 0.611550509929657, 0.18665236234664917, -0.7710816264152527, 1.7965542078018188, 1.198452115058899, 0.28939205408096313, -0.3087436854839325, 0.9014959931373596, -0.11002463102340698, -0.9641096591949463, 0.5538904666900635, -1.036185383796692, 1.151572823524475, 0.274526983499527, -0.47729256749153137, -0.7298040390014648, -0.9997978806495667, 1.4866012334823608, 1.572800874710083, 1.2783724069595337, -0.649101197719574, -2.0613694190979004, 1.2240737676620483, -1.0034434795379639, -0.6011598110198975, -0.9066869020462036, 1.5351585149765015, -0.9927440285682678, -0.47861963510513306, -0.48768913745880127, -0.810067892074585, -0.9198392629623413, 2.196671485900879, -1.073522925376892, 1.0036473274230957, 1.2846148014068604, 0.1906871348619461, -0.4949829876422882, -0.7643799781799316, 0.3694327473640442, 0.7694745659828186, 0.2707742750644684, 0.2995242774486542, 1.0299447774887085, 0.6258057355880737, 1.4312244653701782, -0.6778703927993774, -1.4695756435394287, 0.5325683951377869, 1.117546796798706, 0.01910988800227642, -1.2193716764450073, -0.9602344036102295, 0.5524469614028931, -0.20957636833190918, 0.6841263175010681, 1.4433128833770752, -0.4121130704879761, 0.3301837146282196, 0.8639224171638489, 0.5570410490036011, 0.8397361040115356, 0.9466630816459656, -1.0507560968399048, 0.9165841937065125, -0.5533218383789062, 0.722114086151123, 1.2370636463165283, 2.5623857975006104, 0.19186577200889587, -2.4972169399261475, -0.34350594878196716, 1.1250768899917603, -0.7888293862342834, 0.15998008847236633], [-0.5012767314910889, 1.3553775548934937, -1.496008276939392, 1.207894206047058, 0.2175656110048294, -0.32509148120880127, 0.033530138432979584, -2.208315134048462, 0.056975334882736206, -0.4315069615840912, 0.15282349288463593, 0.18805904686450958, -2.5944018363952637, -0.48468196392059326, -0.49486368894577026, 0.6016770601272583, -0.6747575402259827, -1.1222577095031738, -0.8662815093994141, 1.604793667793274, 3.0689444541931152, 0.5543420910835266, -0.03375617787241936, -0.5253035426139832, -1.5454740524291992, 0.4749712347984314, 2.0461552143096924, -0.13997957110404968, 0.3327348232269287, -0.7292920351028442, -0.17191992700099945, 0.10825779289007187, -0.45880332589149475, -0.2570851445198059, 0.28115081787109375, -0.27948856353759766, -0.16772441565990448, -0.6168546676635742, 1.574949860572815, 0.07844450324773788, -0.05984616279602051, -0.3693104088306427, -0.8035575747489929, -0.6633792519569397, 1.292738676071167, -1.848577618598938, 0.30381062626838684, 0.24496202170848846, 0.45302456617355347, 1.752074122428894, 0.49959108233451843, 1.1709659099578857, 1.2488936185836792, -1.2908170223236084, -1.2258621454238892, 2.6089727878570557, -0.32192811369895935, -1.179596185684204, -0.3816748559474945, -0.11360154300928116, -1.2413331270217896, 0.5815128684043884, 0.43105748295783997, 0.22944195568561554, -0.036559637635946274, 1.2779711484909058, 1.390980839729309, 0.31357815861701965, 0.009507237002253532, -0.22208340466022491, -0.31808239221572876, -1.6890617609024048, 0.5078102350234985, 1.687891960144043, 1.7485042810440063, -0.7901867628097534, 0.5282261967658997, 1.3071554899215698, 0.09641917794942856, -0.4854945242404938, -1.2680447101593018, 1.0847238302230835, 1.6243691444396973, 1.8254704475402832, -1.100919246673584, -0.037584442645311356, 0.5059886574745178, 0.9215772151947021, 0.6366853713989258, 0.3856028914451599, -3.0166587829589844, -1.3676570653915405, 0.8761274218559265, -0.6188519597053528, -0.8015096187591553, 2.746553421020508, -0.14587870240211487, -0.31833094358444214, -2.3300364017486572, 0.519423246383667, -1.1389700174331665, 0.7106499075889587, 1.1635323762893677, -0.6431039571762085, -0.3271467089653015, -1.6066421270370483, 0.021991189569234848, -0.6248971819877625, -0.2931598722934723, 0.17914573848247528, 0.6237112283706665, 1.0703063011169434, 0.6169822216033936, 0.5348482728004456, -0.730658233165741, -2.131319761276245, 1.9542921781539917, -0.10431606322526932, -2.0878477096557617, -0.7285425662994385, -0.5159679651260376, -0.5599681735038757, -0.5195447206497192, -0.25949549674987793, 0.04635181277990341, -0.5250705480575562, 0.07933270186185837, -0.42053931951522827, 0.7329440712928772], [-0.10935194045305252, 0.4328590929508209, 1.6768580675125122, 0.14947940409183502, 0.4573521912097931, -0.7487205862998962, -1.0487415790557861, -0.2132924348115921, -1.2253844738006592, 0.9470852017402649, 2.932845115661621, 0.23053443431854248, -0.05478815734386444, -0.3158564865589142, 0.17238637804985046, 0.25810807943344116, -2.195359468460083, -0.8261716365814209, -1.8940452337265015, 0.2426481693983078, 0.024946074932813644, 1.4354465007781982, -0.34188389778137207, -0.6981140375137329, 0.17201557755470276, 0.39697617292404175, -0.17116452753543854, -1.3656866550445557, -0.048475831747055054, 0.5890359282493591, -0.481125146150589, -0.439290314912796, -0.9410610198974609, 1.5309858322143555, -0.12185104936361313, -0.47590795159339905, 0.7402507662773132, -0.25333160161972046, 0.48103705048561096, -1.3004117012023926, 2.802382469177246, -1.5514322519302368, 1.2317979335784912, 0.0816451758146286, -0.9197684526443481, -1.3146946430206299, 0.9322423338890076, -0.1494053304195404, -0.5442311763763428, 0.404289573431015, 1.1721669435501099, 1.3928545713424683, 0.36447590589523315, -1.7728170156478882, -2.3734443187713623, -1.320502758026123, 0.7288708686828613, 0.1393563598394394, -0.012020769529044628, 0.9009351134300232, -0.3301241993904114, 0.55591881275177, -1.3935097455978394, -0.6588230729103088, 0.8557928204536438, -0.9837691783905029, 0.5341395139694214, 0.5395426750183105, -0.05695187300443649, -0.6571337580680847, -0.11531076580286026, 0.5286185145378113, 0.8441378474235535, -0.31307724118232727, 1.0927984714508057, -2.4963014125823975, -0.1403731405735016, 0.6316877007484436, -0.010022780857980251, 0.03004825860261917, 1.005253553390503, -0.9572246074676514, -0.5266743302345276, 0.19345085322856903, -1.5676027536392212, -1.0816650390625, 0.0909942090511322, -2.7039568424224854, -0.4902173578739166, 0.28590086102485657, -1.032287359237671, 2.3715240955352783, 1.0133110284805298, -0.0579683892428875, 0.5295757055282593, -0.23920252919197083, 0.5080142021179199, -0.8427976965904236, 0.5294132828712463, -1.1550706624984741, -0.6142306923866272, 1.9439812898635864, -2.0837252140045166, 0.14717485010623932, -1.2288042306900024, -0.1902114301919937, -0.9680595993995667, 0.379474401473999, 1.1151124238967896, -0.34944137930870056, 1.095554232597351, -0.6144769191741943, 1.2845004796981812, -0.2457212209701538, -2.072384834289551, 1.622107744216919, 0.3807026445865631, -1.5146781206130981, -0.8987356424331665, 0.5587998032569885, -0.2907089293003082, 1.3087823390960693, 0.267711877822876, 1.263822317123413, 0.8740580677986145, 0.1105240061879158, -0.6748318672180176, -1.4855328798294067, 1.3202576637268066], [-1.2514842748641968, 0.7127161622047424, -1.0957496166229248, 0.41944143176078796, 0.8485651612281799, 0.8497704863548279, -1.7298099994659424, 0.17748813331127167, 0.775752067565918, 0.5109902024269104, -0.936215341091156, 0.8327103853225708, 0.3007858395576477, -0.42547592520713806, -1.1001784801483154, -0.1781178116798401, -0.27470964193344116, -2.283149242401123, 1.6358619928359985, 0.3988545835018158, 1.2092130184173584, 0.1111830398440361, -2.268784761428833, -0.5181336998939514, -0.2136976569890976, 0.3737119734287262, -1.0202646255493164, 0.5683221220970154, -0.10973897576332092, 0.27135443687438965, -1.412009358406067, 2.2264554500579834, -1.208824872970581, -0.25494009256362915, -0.5516074895858765, 0.2526150345802307, 0.5634688138961792, -1.1780234575271606, 2.1215591430664062, -0.20701201260089874, -1.2637739181518555, -0.5040084719657898, -0.8706541657447815, -1.7423219680786133, 0.7405446767807007, 0.5216850638389587, 0.7913642525672913, -1.0439401865005493, -0.3585374653339386, 0.10555461049079895, 0.24891500174999237, 1.2018946409225464, 0.4219406545162201, 0.07959980517625809, 2.4431490898132324, -1.5609642267227173, 1.2276443243026733, 1.6964116096496582, -1.5947178602218628, -0.038379013538360596, 0.7237530946731567, 1.449329137802124, -0.006371885538101196, 1.0548135042190552, 0.8359058499336243, -0.4387170076370239, 0.3650205135345459, 0.7866199016571045, 1.2179090976715088, -0.47971683740615845, 0.47695392370224, 1.1444816589355469, -0.9852789044380188, 0.6811329126358032, 0.10138237476348877, -2.4825220108032227, 2.0225110054016113, 1.864493727684021, -0.8004446625709534, -1.7142901420593262, -0.08021586388349533, 0.349139541387558, 0.5465262532234192, 2.288874387741089, -0.5787705779075623, 0.658573567867279, 2.0071237087249756, -1.5177102088928223, 0.6482402682304382, -0.1680883765220642, -0.03348661586642265, -1.2578630447387695, -0.11944888532161713, -0.4481513798236847, 1.8691062927246094, 0.5939283967018127, 2.2297370433807373, -0.1069834977388382, 0.9039035439491272, 1.0885528326034546, 1.0315978527069092, -0.42968589067459106, -0.09531553089618683, 0.9306899309158325, 0.15883320569992065, 0.7947885990142822, -0.014197702519595623, -2.464712142944336, -0.7872762084007263, -0.3170276880264282, 2.1952016353607178, 0.13141940534114838, 1.355729103088379, -1.2045822143554688, -0.31240910291671753, 0.45002487301826477, -0.5010102391242981, 1.3670334815979004, -0.31469467282295227, -0.4633021056652069, -0.8992663621902466, -0.12651553750038147, -0.20895817875862122, -0.3877246677875519, 1.0254592895507812, 0.49753984808921814, 2.7988367080688477, 0.8030894994735718, -0.11426106840372086], [0.8793519139289856, -1.8616267442703247, 2.687607765197754, 0.5819464325904846, 0.16597086191177368, -0.39957883954048157, 0.7952825427055359, 0.9625229239463806, 0.19528885185718536, -0.756091296672821, 2.1963863372802734, 0.73924320936203, 0.8392349481582642, -0.5579386353492737, -0.08639830350875854, 0.793999969959259, -0.15520671010017395, 0.4224664866924286, -2.045006275177002, 0.25035884976387024, 0.3557537794113159, 0.4693344235420227, -0.21484126150608063, -0.4218001961708069, 0.645194947719574, 0.07607017457485199, 2.6508424282073975, 0.013688051141798496, -0.10530859231948853, 0.7590802311897278, -2.3666210174560547, 0.07568351924419403, 0.04477114602923393, -0.2301235944032669, -0.797510027885437, -0.6403225064277649, 0.41903889179229736, -0.5614697933197021, -0.5494069457054138, 0.3636942505836487, 0.8183582425117493, 0.20835521817207336, -0.8330745100975037, -0.4124585688114166, -1.9652886390686035, -0.49364641308784485, 0.791175901889801, 0.3028036952018738, 0.8003108501434326, 0.9572184085845947, 0.6394370794296265, 1.27743661403656, 0.6577181220054626, 0.4299139976501465, -0.5831021666526794, -0.49272260069847107, -0.6021991968154907, -0.2976122498512268, -0.5606722235679626, -0.683035135269165, 0.8032811880111694, 0.3078141212463379, 1.3901808261871338, -0.9910851120948792, -1.6350551843643188, -1.307976245880127, 1.8877540826797485, 0.05000769719481468, -0.6721419095993042, 0.24779506027698517, 1.3239444494247437, 0.4374132454395294, 0.24604609608650208, 0.06998175382614136, -0.14178019762039185, 0.8680356740951538, -2.4067347049713135, -0.5761356949806213, 1.3560116291046143, 0.06363669782876968, 0.7430998682975769, -0.5238189697265625, 0.2722665071487427, -0.9850783348083496, -1.870884895324707, 1.9260095357894897, -1.115470051765442, 0.8438019752502441, -0.0210132896900177, -1.7977977991104126, 0.2882831394672394, -1.0393742322921753, -1.8520283699035645, -0.26051151752471924, 0.44010061025619507, -0.5241619348526001, 0.2761022746562958, 2.2099215984344482, -0.8210813999176025, -0.38593190908432007, -0.4629201889038086, -0.31145086884498596, -0.5811033844947815, 0.9392693638801575, -0.2896195352077484, -0.3007359802722931, 1.0128390789031982, -1.392727255821228, -1.0325769186019897, 0.9151849150657654, 0.35632872581481934, 0.7214176654815674, -1.1641494035720825, -0.6553653478622437, -0.934756338596344, 1.0145621299743652, 1.1265501976013184, 1.723645567893982, 0.4587118327617645, -1.630897045135498, -0.5764700770378113, 0.8426148891448975, 1.025578260421753, 0.3964666724205017, 1.985525369644165, 0.06390347331762314, 0.07820917665958405, 0.8901675343513489, -1.6859973669052124], [0.5506643652915955, 2.0907256603240967, -0.683735728263855, 1.0521249771118164, -0.2097684144973755, -0.09455183893442154, -0.3922956883907318, 1.8206148147583008, -0.744890570640564, -1.9738177061080933, -0.92885822057724, -1.6535990238189697, 0.21204738318920135, 1.5756092071533203, 1.2555712461471558, 0.1530207246541977, 0.8409103751182556, -0.44860976934432983, 0.38453030586242676, -1.5563328266143799, 0.2079835832118988, 1.2823200225830078, -0.9257874488830566, -0.8120843768119812, 0.08201105147600174, -0.6680809855461121, -0.5024721026420593, -0.9413763880729675, 0.2726016044616699, 0.12494435161352158, 0.7386857271194458, -0.44206371903419495, -0.6095077991485596, 1.5345853567123413, 0.9043458700180054, -1.322730302810669, -0.6053644418716431, -0.6114186644554138, 0.2854902148246765, -0.6414704322814941, -0.1349482238292694, 0.5863096117973328, -0.636078417301178, -1.59930419921875, 0.44475057721138, -0.7327466011047363, -0.05764985829591751, -2.0912575721740723, 0.6889386773109436, 0.8080621361732483, 0.9186721444129944, -2.2672481536865234, -0.9324973225593567, 1.3447993993759155, 0.50498366355896, -0.019601577892899513, 0.5806992650032043, 0.23394428193569183, 0.3291739225387573, 1.267154335975647, 2.072801351547241, -0.17114944756031036, 0.4733193814754486, 0.2945428192615509, 0.10573636740446091, -0.8458868861198425, -0.578188955783844, 0.7895612120628357, 2.003584623336792, -2.587094306945801, 0.03539145737886429, 0.5173075795173645, -1.3200695514678955, 0.0601794607937336, -0.013200410641729832, 0.5348467230796814, -1.9326190948486328, 1.6358582973480225, 1.1627777814865112, -0.7409375309944153, 0.29241618514060974, -1.0338573455810547, 0.7718138098716736, -0.8963079452514648, -1.9016481637954712, 1.2791337966918945, 0.22615811228752136, 0.585701048374176, -1.6186695098876953, 1.0398725271224976, 1.0932519435882568, 0.49275267124176025, 0.4074329137802124, 1.801113247871399, -0.14322641491889954, -0.20804426074028015, -0.234683558344841, -0.2783644199371338, -0.9626995325088501, 0.7224034667015076, 0.39176785945892334, 2.236757516860962, -0.2605535387992859, 0.2510535418987274, 0.7391811609268188, -0.03807263821363449, -1.0917800664901733, 1.646255373954773, 0.7053466439247131, 1.1112749576568604, 0.47024235129356384, 0.5913696885108948, -0.01575140282511711, 0.20833036303520203, 0.47329914569854736, -1.279438853263855, -0.49657440185546875, 0.3248060643672943, 0.55306476354599, 1.1239299774169922, -0.5135720372200012, -0.5162045955657959, 1.1126124858856201, -0.3869798183441162, 1.5894999504089355, -1.795020580291748, -1.4696094989776611, -0.9325204491615295, -0.10504758358001709], [-0.06537177413702011, 0.8459305763244629, 0.10788504779338837, -0.6859611868858337, 0.7377563118934631, 0.4286406636238098, 2.4566385746002197, -0.4106241762638092, 0.6717833280563354, -0.16209252178668976, 0.17742903530597687, -1.203260898590088, 0.5849770307540894, 0.5101861357688904, -0.2579911947250366, -0.35613831877708435, 0.08965017646551132, -1.431613802909851, -0.09260421991348267, 0.5042412281036377, 0.5308576822280884, -0.5107458233833313, 0.6289519667625427, -0.6621919274330139, 0.9516260623931885, -0.7257174253463745, -1.2207870483398438, 1.8141767978668213, 0.4428096115589142, -0.11610079556703568, 2.4504988193511963, 0.7353439331054688, 0.956750214099884, 0.13668279349803925, 0.6654440760612488, 0.8621967434883118, 0.47033601999282837, -0.1645517647266388, -0.5068051815032959, 0.3831782042980194, -0.8873348236083984, 1.1983057260513306, -1.4233342409133911, -0.9498671293258667, -1.1791305541992188, -0.3892972469329834, -0.09360144287347794, -1.5064568519592285, 1.7133115530014038, -2.1474609375, -0.09404826164245605, 0.2271512895822525, 1.357502818107605, -0.676952064037323, 0.9713606834411621, 0.46957576274871826, -0.38448578119277954, 0.9399192929267883, 0.2711763083934784, -0.37439823150634766, -0.24645431339740753, -0.08514773845672607, 0.05395129695534706, -1.753358006477356, -0.2642216086387634, -0.061162713915109634, 0.19978658854961395, -0.5900951623916626, 1.321320652961731, 0.12047887593507767, -0.027389775961637497, 0.4468645453453064, 1.3261655569076538, -0.5030148029327393, -0.20753513276576996, -0.2518523931503296, -0.33449220657348633, -0.3960152864456177, 1.736653447151184, 0.9339923858642578, -0.5049809813499451, -0.8082293272018433, 1.4572886228561401, 0.877695620059967, 0.6402098536491394, -0.19963495433330536, -0.0959676057100296, 0.8542822003364563, -0.31940755248069763, 0.17483821511268616, 0.11614912003278732, 1.1700446605682373, 0.41774851083755493, -1.1758886575698853, -0.8040462732315063, -2.1162071228027344, -0.34189555048942566, -0.7145968079566956, 0.5252538323402405, 1.0616261959075928, -0.365649551153183, 0.11315644532442093, 0.2534921169281006, 1.0413897037506104, -0.627420961856842, 0.037523411214351654, 0.4529820680618286, 1.2703907489776611, -0.6496633291244507, -0.6099832653999329, 0.6760296821594238, 1.5720375776290894, -1.1719591617584229, -2.4555068016052246, 0.8785699009895325, -0.854732871055603, -0.7388203740119934, -0.5077917575836182, 1.4904133081436157, 0.25666114687919617, -0.29157593846321106, -0.8384729623794556, 0.013141840696334839, 1.6675411462783813, -0.3357026278972626, 0.9733554720878601, 0.6967359781265259, 0.8377712965011597, -0.7866612076759338], [2.548194408416748, 0.7173922061920166, 0.1615586131811142, 1.5368585586547852, 0.3700278401374817, 1.446234107017517, -1.4985321760177612, 0.9024546146392822, -1.3050323724746704, 2.3286631107330322, 1.245169997215271, 0.5854418873786926, -0.6622718572616577, 0.2839885652065277, 0.45963147282600403, -0.820855975151062, -0.8667860627174377, -0.6236968636512756, -0.3479789197444916, 1.4868674278259277, -0.785257875919342, 0.2636357247829437, -0.33264386653900146, 0.9576336741447449, 0.7911127209663391, 1.77241849899292, 0.9190256595611572, -0.1385948210954666, -0.3372282385826111, -0.44216740131378174, 2.2518198490142822, 0.5368966460227966, -0.6230664253234863, 0.40452030301094055, -0.5039746165275574, -1.49724280834198, -0.5217036604881287, 0.1385970413684845, 0.5556879043579102, -0.7737021446228027, 0.19538357853889465, -0.7099968791007996, -0.6083168983459473, -0.17084908485412598, -0.43508243560791016, 0.7807776927947998, -1.302890419960022, 1.4237364530563354, -1.0313907861709595, 0.41295021772384644, -0.29512956738471985, 0.7253645658493042, 1.3754973411560059, -0.26729175448417664, -0.5826489329338074, -0.3360480070114136, -1.3730632066726685, -0.9470732808113098, 1.5851892232894897, 0.16473521292209625, 1.7710680961608887, 0.29993388056755066, 0.10250495374202728, 0.09260354936122894, 0.7094265818595886, 0.7769955992698669, -0.3977358639240265, -0.5174518823623657, 1.097596287727356, -0.8616154789924622, 0.32544636726379395, -1.2474364042282104, -0.03059445321559906, 0.5875729322433472, -0.7746588587760925, -0.9232223629951477, 0.19412051141262054, -0.8983118534088135, 0.3819926381111145, -0.15686796605587006, 0.3532133996486664, -0.34852561354637146, 0.6532353758811951, -0.2339029461145401, -2.198824644088745, -2.3281099796295166, 0.9921766519546509, -0.8875282406806946, -0.8810117244720459, 0.1760011911392212, 0.5248314142227173, 2.0067951679229736, 0.05156492814421654, -0.6650556921958923, -1.990089774131775, 0.800360918045044, -1.1094824075698853, 0.26925864815711975, 0.6160525679588318, -0.6226573586463928, -1.0078281164169312, 0.2782050371170044, -1.3934298753738403, -1.8376741409301758, -0.5907461047172546, 0.6992283463478088, 0.3010951578617096, -1.0327435731887817, -0.6037188172340393, 1.334892749786377, 0.45243749022483826, 1.1363219022750854, 0.8540543913841248, 0.05684903264045715, -0.7104859352111816, 0.05092351883649826, -0.40586182475090027, -2.0417633056640625, 0.47011205554008484, -1.4181029796600342, -1.1694116592407227, -0.9442707300186157, -0.24772414565086365, -0.3125712275505066, 0.21616871654987335, 0.7691328525543213, 1.3739662170410156, 0.1146479994058609, -1.2953544855117798], [0.788188099861145, 0.5459080338478088, -1.1083353757858276, -0.5416309833526611, 1.0560014247894287, -0.5300238728523254, -0.0013005953514948487, -0.5126298666000366, 1.3061885833740234, 1.786430835723877, 1.0354294776916504, -0.5975505709648132, -0.44883468747138977, -0.2719391882419586, 1.2114225625991821, -0.017648013308644295, 0.6839051246643066, -1.5108165740966797, 0.5698172450065613, 2.012972116470337, -0.07048075646162033, -1.4554979801177979, 0.0293294545263052, 0.796700656414032, 1.1451547145843506, -1.921809196472168, -0.001095118816010654, -0.17789536714553833, 0.03846963495016098, 1.4371193647384644, -0.0493900403380394, -0.5742712616920471, -2.1466610431671143, -1.1380308866500854, 0.16813018918037415, 3.291794776916504, 2.102722644805908, 0.11505408585071564, 0.45159438252449036, -0.920111894607544, 0.35074207186698914, 0.3671407103538513, 0.283305823802948, 0.17020796239376068, -0.3146710991859436, 0.2967226505279541, -0.7007304430007935, 1.0106011629104614, 0.5206649303436279, -1.41927170753479, 1.2141737937927246, -1.0197833776474, -0.24243590235710144, -1.497356653213501, -1.187114953994751, 0.4415876269340515, -0.8014315962791443, 0.608670711517334, 0.5544390678405762, -0.8826294541358948, -0.8358646631240845, -0.1695079356431961, 1.5459575653076172, -0.21065445244312286, -0.25244975090026855, 0.7370759844779968, -0.06742784380912781, -0.3437720239162445, -0.0012618104228749871, 0.10859071463346481, 0.39072009921073914, 0.46260353922843933, -0.18956159055233002, 0.8664896488189697, 0.13878987729549408, 0.4708748161792755, -1.4129183292388916, 1.781793475151062, -0.14567674696445465, 0.7218009829521179, 1.217634677886963, 0.07020926475524902, -0.471495121717453, -1.2932333946228027, -0.9272387027740479, 1.1032118797302246, 2.0564651489257812, 1.1696933507919312, -0.855021059513092, 0.6857722997665405, 0.5664811134338379, -0.010717385448515415, -0.22470307350158691, -1.6544899940490723, -0.9532656073570251, 0.7952346801757812, -2.285994529724121, 1.0624451637268066, 0.6542730331420898, 0.4144381284713745, 0.379850298166275, 0.5842298269271851, 0.9057374000549316, 2.7578020095825195, 0.775280237197876, -0.21511395275592804, -2.340226173400879, 0.5514473915100098, -0.9325768351554871, 0.37096306681632996, 1.443949580192566, 0.32641172409057617, -0.502333402633667, 1.1706393957138062, 0.6170947551727295, 0.44087108969688416, -1.1678467988967896, -1.4242982864379883, -0.7771999835968018, -0.5751651525497437, 0.8474993705749512, -0.3450672924518585, 0.3849364221096039, 0.9188101291656494, 1.6838963031768799, -0.11710124462842941, 1.13383948802948, -0.9674747586250305, 0.4497145116329193], [0.05941532179713249, 1.0378837585449219, 1.6231060028076172, 0.514590322971344, 1.2492510080337524, 2.1205475330352783, 1.1304113864898682, 0.08533868938684464, 0.9884830713272095, 0.08237434923648834, -0.20211829245090485, 0.5077528357505798, -1.113000512123108, 1.1281834840774536, 0.378988116979599, -0.33878687024116516, 1.3928332328796387, -0.880050539970398, -0.9243040084838867, 1.0230700969696045, 1.015817642211914, -0.6968125104904175, -0.293331503868103, 0.18012432754039764, 1.4312193393707275, -1.0998187065124512, -0.8710850477218628, -1.1572363376617432, -0.843463122844696, 0.7746815085411072, -0.23227284848690033, 0.05571151524782181, -1.4685523509979248, 0.16714046895503998, -0.29009950160980225, -0.13643532991409302, 1.937374234199524, -0.5171079635620117, 1.292765498161316, 0.6118757724761963, 0.5593032836914062, 0.09505504369735718, -1.1278736591339111, 0.6874330043792725, -1.1755346059799194, -1.1478931903839111, 0.5331676006317139, 0.6601845622062683, -2.4210569858551025, 0.18473058938980103, -0.301724910736084, 1.7573515176773071, 1.2470556497573853, 0.5439941883087158, 1.068014144897461, 0.16441430151462555, -0.6262981295585632, -0.3277362287044525, -0.9697378873825073, -0.8062566518783569, -0.25275367498397827, -0.7965980768203735, -1.8388538360595703, -0.2810099720954895, -1.3107707500457764, -0.43594038486480713, 0.28756171464920044, 1.4845499992370605, 1.3003206253051758, 1.660311222076416, -0.08649531751871109, -1.6802648305892944, -1.8119908571243286, -0.3987298607826233, 0.2999587953090668, -0.1149144172668457, -1.2398141622543335, 2.2199294567108154, -0.03417364880442619, -0.29328060150146484, 0.6265245676040649, -1.085219144821167, -1.3810280561447144, -0.008857292123138905, -2.1205971240997314, 0.34922683238983154, 0.4524400532245636, -0.4365732669830322, 0.031181151047348976, -0.45799893140792847, -0.22353997826576233, 0.43288546800613403, -0.3398975133895874, -0.7664599418640137, 2.3116726875305176, 0.12855879962444305, -1.5718703269958496, -1.0327378511428833, 0.8283921480178833, -0.9246243834495544, 0.324241578578949, 0.3708149790763855, -1.0997469425201416, 0.33972832560539246, 1.4295543432235718, 0.21860545873641968, 0.401476114988327, -0.7879072427749634, 0.9591422080993652, -1.1150530576705933, 0.18197204172611237, -0.024946287274360657, 1.1302132606506348, 0.7325697541236877, -2.372576951980591, -0.2533685564994812, -0.18809817731380463, 0.7446415424346924, -0.8321440815925598, 0.16565139591693878, -0.3624424338340759, -0.2297237366437912, -2.4409408569335938, 1.498530626296997, 0.43491145968437195, 1.5315457582473755, -0.36776524782180786, -0.018988419324159622, -0.12833666801452637], [0.12421535700559616, -0.4219372868537903, -0.22547395527362823, -0.5032430291175842, -0.26493141055107117, -0.7233165502548218, 0.5956045389175415, -1.5210915803909302, 0.6627705097198486, 0.5956050157546997, 1.6401011943817139, 0.6688281893730164, 0.46956080198287964, 0.1105625182390213, -0.08389812707901001, -0.829861044883728, -0.5651030540466309, -0.8469688892364502, -0.5599871873855591, 0.19083841145038605, -0.9182567000389099, 0.40554651618003845, 0.9759336709976196, 0.46433842182159424, 1.4647811651229858, 1.4576921463012695, 1.74433434009552, -0.3848804831504822, 0.12209823727607727, 2.0699963569641113, 0.04798536375164986, -1.2956163883209229, 0.0703832134604454, -1.5103585720062256, 0.5671143531799316, -1.7865713834762573, 0.2784596383571625, 0.19149889051914215, -0.919842004776001, 0.3053267002105713, 0.3154467046260834, -1.7427762746810913, 0.3299962878227234, -1.1720184087753296, -0.9755233526229858, -2.1159331798553467, -1.4351229667663574, -0.8974385857582092, 0.9013653993606567, -0.4694775640964508, -2.253441572189331, -1.2978631258010864, 0.5879781246185303, 0.15538394451141357, 0.10469391196966171, -0.8734351992607117, 0.07979054003953934, -0.7114453911781311, 0.5518920421600342, 0.5402587652206421, 1.0077121257781982, -1.828355073928833, -0.019583478569984436, 1.3057397603988647, -0.2538147270679474, -0.8705389499664307, -0.017806757241487503, -0.576555073261261, 0.5842369198799133, -1.4780524969100952, -0.929137647151947, 0.5138307213783264, 0.42231443524360657, -0.00936146080493927, 1.311103343963623, 0.20957286655902863, -0.5371072888374329, -1.3705283403396606, -0.8662174344062805, 0.6774642467498779, 1.8351808786392212, 0.9759786128997803, 1.5914400815963745, -0.7258162498474121, -0.6679115891456604, 0.3623221218585968, 0.7184029817581177, 2.3740363121032715, 0.7545816898345947, -0.45667940378189087, 0.8274665474891663, -0.07577838003635406, 0.37624287605285645, 0.4022366404533386, 1.0793647766113281, -0.4123266637325287, 1.1571745872497559, -1.0378330945968628, -1.7652584314346313, -1.0809762477874756, -0.05930187553167343, 0.32080093026161194, 0.7280560731887817, 1.479442834854126, 1.8881525993347168, 1.5852854251861572, 0.24181361496448517, -0.07944291085004807, 1.6977081298828125, 0.9193282723426819, 1.013405442237854, -0.3890111446380615, 0.4480157494544983, 0.8420636057853699, 1.258025050163269, 0.13967758417129517, -0.4044366776943207, 1.9668548107147217, 0.2300546020269394, -0.7276917099952698, -0.48490121960639954, -0.13895177841186523, 0.9427364468574524, -0.3329774737358093, -0.006321680266410112, -0.3059930205345154, -1.3543058633804321, -1.657834529876709, 0.26923370361328125], [-1.1314042806625366, -0.2408684343099594, 0.34855926036834717, 0.7575733661651611, 1.8855319023132324, -0.3691120147705078, -0.714515209197998, -0.21523237228393555, -0.8496484756469727, -0.08148052543401718, 1.0587939023971558, -0.27823513746261597, 0.3071710169315338, 1.272342562675476, 2.786135673522949, -0.5427166223526001, 1.292840838432312, 1.2785547971725464, 0.08912072330713272, 1.6233762502670288, 0.003906449303030968, -0.0519142672419548, -0.9603598117828369, -0.1966678947210312, -0.9567320346832275, -0.36782899498939514, 1.6246991157531738, -1.0849967002868652, -0.8513795137405396, 0.8610867857933044, -1.0969271659851074, -1.2236311435699463, -0.37146785855293274, -1.0958083868026733, 0.25272658467292786, 0.08565831929445267, 0.5842040777206421, -0.20287781953811646, 0.4035450518131256, -1.5455763339996338, -0.7896653413772583, -0.5337563157081604, -0.7271918058395386, 1.4180951118469238, 0.7204491496086121, -0.03481457754969597, 0.0658935159444809, 1.1762014627456665, -0.32760027050971985, 0.8340551257133484, -2.4954700469970703, -1.3616440296173096, 0.8188445568084717, -0.05704531446099281, -0.9798572659492493, 0.6281762719154358, 0.05291019380092621, 1.1181057691574097, 0.8857517838478088, -1.0933283567428589, -0.024849997833371162, -1.3724085092544556, -0.7915402054786682, -0.29922810196876526, 1.6889331340789795, -0.14954924583435059, -1.6884183883666992, 0.429962694644928, 0.24698881804943085, 0.9678397178649902, 0.963168203830719, -0.6681816577911377, 1.1654553413391113, -1.7588034868240356, 1.0808249711990356, 1.3960933685302734, 0.6735000014305115, 0.631450355052948, 0.689287543296814, 1.0299458503723145, 0.5931531190872192, 0.5743587017059326, 0.2420111447572708, 1.0766208171844482, -0.7763449549674988, -0.40496399998664856, 1.4733021259307861, -0.8260095119476318, 0.4096778929233551, -2.118027448654175, 0.939873456954956, 0.46213072538375854, 0.1618746668100357, -0.8641342520713806, 0.5182427167892456, 0.5122875571250916, 0.6020142436027527, -2.4348180294036865, -0.48303255438804626, -0.3860146999359131, 1.1035593748092651, 0.7555283904075623, 0.4701439440250397, 0.42439204454421997, 0.7620102763175964, 0.14071835577487946, -0.08415909856557846, -1.541885495185852, 1.2651350498199463, 0.07993990182876587, -1.2247529029846191, -0.20404711365699768, 0.05879507213830948, -0.6036805510520935, 0.016087805852293968, 1.4696941375732422, -1.750111699104309, 2.6069178581237793, 0.18596260249614716, 0.5052416324615479, 1.0150457620620728, 0.07515756785869598, -0.2645142674446106, 3.0120253562927246, -0.7371482849121094, -0.030167507007718086, -0.623072624206543, 1.4995516538619995, -0.1937437802553177], [0.06328374147415161, -1.2644299268722534, 0.38775235414505005, -0.5001983642578125, -0.3817764222621918, 0.4110199213027954, -0.3834051191806793, -1.3173575401306152, 0.9758888483047485, 0.9215865731239319, 0.8017796874046326, -0.2668613791465759, -0.6664171814918518, 0.04067208990454674, -1.1910234689712524, -1.1532646417617798, -0.5270959138870239, -2.2829627990722656, -2.3178536891937256, 1.908492922782898, 0.9238824248313904, -0.1203308254480362, 0.5547834634780884, 0.45019984245300293, 0.9895591735839844, 0.13586658239364624, 0.3368081748485565, -1.7174588441848755, 0.5934733152389526, 0.6769015789031982, -0.9337262511253357, -0.5141944289207458, 0.20721860229969025, 0.14801914989948273, 0.3428322672843933, 0.5729804635047913, -1.72321355342865, -0.4639233946800232, 0.750978946685791, -0.41283485293388367, -0.7182372212409973, 0.24253660440444946, 1.5137279033660889, -1.2591397762298584, -1.41024649143219, 0.49419111013412476, -0.4186817705631256, 0.5042163133621216, -1.2045763731002808, -0.014894825406372547, -0.49125948548316956, -3.2074999809265137, -0.16372638940811157, -0.6032751798629761, -0.8750303983688354, -0.1962585598230362, 1.4584628343582153, -1.4598584175109863, 0.4491501748561859, -1.425539255142212, 0.12803207337856293, 2.0083539485931396, -0.03881102800369263, -1.6665135622024536, 0.04063257575035095, -0.16047827899456024, -0.6534297466278076, 0.023998629301786423, 0.8590022325515747, 0.054629966616630554, 0.7159299254417419, 1.1865692138671875, 1.4916359186172485, 0.5358548164367676, -0.34916815161705017, -1.048459768295288, 0.8595791459083557, 1.2118160724639893, 0.9446073770523071, -0.13045726716518402, 0.42198866605758667, -0.09123174101114273, 0.02469504438340664, 1.0221489667892456, 1.3132096529006958, 0.8708921670913696, 1.2364531755447388, -0.8685156106948853, -0.47853633761405945, -1.0899165868759155, 0.8398074507713318, -0.42063578963279724, -1.1457704305648804, 1.0121228694915771, 0.5106816291809082, -1.390519380569458, 0.641152560710907, -1.2724319696426392, 0.3257920742034912, -0.4105874300003052, -1.2580317258834839, -0.8985336422920227, 0.9312319755554199, 0.6210086345672607, 0.7392844557762146, -3.3448286056518555, -0.39387649297714233, 0.10588396340608597, 0.17189835011959076, 0.4129405915737152, -0.34628790616989136, -2.097381114959717, 0.6504279375076294, 0.7914876341819763, 0.1869826763868332, 1.4511257410049438, 2.267009973526001, 0.17825791239738464, -1.546615719795227, -1.9912351369857788, 0.5404757261276245, 0.6747230291366577, 0.6098837852478027, -0.165532648563385, -0.2356119602918625, -1.3675774335861206, 0.2333565354347229, -0.07806222140789032, 1.2884862422943115], [0.38972508907318115, -0.5119503736495972, -0.6478821039199829, -0.17993482947349548, 1.3215389251708984, 0.539716362953186, -0.1792972981929779, 0.3708105981349945, -0.5702524781227112, -0.7705652117729187, 0.1342020481824875, 0.7868127822875977, -0.18552090227603912, -1.113769769668579, 0.4372115731239319, -2.420258045196533, -0.46168890595436096, 0.1888040006160736, 1.155635118484497, 0.5733336210250854, -1.7337782382965088, -0.8269321918487549, 1.0288066864013672, 0.2438446581363678, -1.0379812717437744, -0.5114757418632507, 0.2006354182958603, -1.18763267993927, 1.1935032606124878, 0.9650792479515076, -0.9757289886474609, 0.6340978145599365, 0.6121850609779358, 0.22021932899951935, -0.4072520136833191, -0.2909688651561737, 0.9486308693885803, -2.176068067550659, 0.14107646048069, 0.6878033876419067, -0.7958976626396179, 1.5363476276397705, -0.07239970564842224, 1.9828299283981323, 0.37730616331100464, -0.9018176794052124, 1.4391610622406006, -2.040351152420044, 1.1605912446975708, -0.4470900297164917, 0.9598672389984131, -0.23946243524551392, -1.8764561414718628, -0.12212822586297989, -0.5765511393547058, 1.3255975246429443, 0.06700964272022247, 1.4910742044448853, -0.47770676016807556, -1.0677565336227417, -1.2524042129516602, 0.3975575268268585, -1.1092699766159058, 1.1661821603775024, -1.144896388053894, 2.2159297466278076, 2.748046636581421, 0.6061730980873108, 0.13764095306396484, -0.5701435804367065, -0.7451171875, -0.6390940546989441, 1.7881299257278442, 1.4434959888458252, 0.9446141123771667, 0.3440180718898773, -2.5843441486358643, 0.09510505944490433, -0.6102926135063171, -0.652923583984375, 1.4616397619247437, -0.15321707725524902, 0.4610739052295685, 0.5330797433853149, 1.032471776008606, -0.1970524936914444, 0.6661948561668396, 0.5458182692527771, 0.2978566884994507, 0.05598367005586624, -0.004797813016921282, -1.0456044673919678, 1.7869682312011719, 0.8608827590942383, 1.5051653385162354, 1.178603172302246, 0.44168925285339355, -0.8586920499801636, -0.05291037634015083, 0.6476197242736816, -0.024678895249962807, 0.3506122827529907, -0.8024160861968994, -0.9347237944602966, -0.9380594491958618, 0.8010203838348389, -0.5214236974716187, -1.9430556297302246, -0.5235269665718079, -0.07543055713176727, -0.5855911374092102, -0.1803467869758606, -1.6449426412582397, 0.7085825800895691, -1.6359446048736572, 0.5800045132637024, 0.080858014523983, -0.12778034806251526, 0.5527293086051941, 0.29955604672431946, -0.14494971930980682, -1.6207854747772217, 0.5646802186965942, -0.18374253809452057, 0.3184536099433899, -0.48469892144203186, -0.3615809679031372, -1.7624843120574951, -1.4905085563659668], [-1.2977262735366821, -0.6719294190406799, -0.055452216416597366, 0.8357459306716919, -0.5053598880767822, -0.23868560791015625, -0.10778964310884476, -0.8071640133857727, 0.3288101851940155, 0.9947050213813782, 0.02437889389693737, -1.8874691724777222, 0.7348765730857849, 0.5708631277084351, 1.5306323766708374, -1.4044824838638306, 0.37881526350975037, 0.10797277092933655, -0.1576598584651947, 2.206821918487549, -0.5388413071632385, -1.7447274923324585, 0.8397330045700073, 0.7513266801834106, 1.5723023414611816, -0.0050148493610322475, -0.6783159375190735, -0.2891635298728943, -1.7630512714385986, 0.8968991041183472, 0.3602389693260193, 1.091833233833313, -0.643952488899231, 0.18050646781921387, 1.3673944473266602, -1.514705777168274, -1.159203052520752, -1.2427451610565186, -0.8464420437812805, -0.09736248850822449, -1.473949909210205, -0.7641968131065369, -0.8586405515670776, 1.5641263723373413, 1.2727532386779785, 0.9416506886482239, 0.7632545232772827, -0.4400685429573059, -1.1623421907424927, 0.5006058216094971, 0.19981269538402557, 1.2399799823760986, -1.8064589500427246, 3.093261957168579, 1.458927869796753, 2.072565793991089, -0.5645559430122375, -0.8759967684745789, 0.27946069836616516, -1.010304570198059, 0.7175179719924927, 0.5140931606292725, -0.7626770734786987, 0.08123470097780228, -1.352632999420166, -0.27706021070480347, -1.2670328617095947, -0.49496346712112427, 0.9583621025085449, 0.5440614223480225, -1.7453614473342896, 0.11001940816640854, -0.42558690905570984, 0.6714979410171509, -1.02749764919281, 0.24544288218021393, 0.945878267288208, 1.6791729927062988, 0.07971523702144623, -0.3180181384086609, 0.6497758030891418, -1.522351622581482, 1.7203775644302368, 0.11329773813486099, -0.032589834183454514, 0.41610315442085266, -0.7310569882392883, -0.5868667960166931, 0.9284271597862244, -0.7564938068389893, -0.645197868347168, -0.6720547676086426, 0.8711211681365967, 0.4091256856918335, -0.17195425927639008, 1.2544232606887817, -0.8582313656806946, 0.32605230808258057, 0.5439121127128601, -1.1169447898864746, 0.28775709867477417, 1.1689026355743408, 1.1776189804077148, 1.1843923330307007, -2.303018093109131, 1.028407335281372, -0.6269705891609192, -1.310986042022705, 1.2603695392608643, 1.2388074398040771, -0.5215536952018738, 0.03487787023186684, -1.8618830442428589, -1.4749451875686646, 0.4116092622280121, -0.35333022475242615, 0.0050355419516563416, -0.3989424705505371, 0.4359208941459656, 0.314301460981369, -0.5603432655334473, -0.7541015148162842, 0.8553913831710815, 1.371476173400879, 1.0394115447998047, 0.18618950247764587, -0.226875439286232, -0.8151291608810425, -0.9684930443763733], [0.35170963406562805, -1.7277625799179077, -0.3795754909515381, 0.9874847531318665, 0.17130228877067566, 1.0342047214508057, -0.5553209781646729, -1.1219749450683594, 1.2522047758102417, -0.9925370812416077, 1.5748708248138428, -0.3498000502586365, 0.11312568187713623, -1.0488111972808838, -0.518173098564148, 2.431063413619995, 0.1143697127699852, -1.4649672508239746, -0.12821856141090393, -0.35603955388069153, 0.22722771763801575, -1.078903317451477, -0.37240663170814514, 1.1646780967712402, 0.25407934188842773, 0.2926708459854126, -0.4037700593471527, 1.8415398597717285, -0.10156812518835068, 0.9630312919616699, -1.2720474004745483, 1.2675527334213257, 0.43918749690055847, -1.0991781949996948, -1.077519416809082, -1.1228233575820923, -1.0063008069992065, 1.1944588422775269, 1.718674898147583, -0.7221848368644714, -0.5378179550170898, -0.596176266670227, -1.8187406063079834, 1.114651083946228, -1.218385934829712, 0.4302710294723511, 0.2582179307937622, -0.48354655504226685, 0.29607445001602173, -0.4262906312942505, -0.6904228329658508, -0.5117077827453613, 0.6444624066352844, -1.979903221130371, 0.8973134756088257, 0.5230545997619629, 1.6366411447525024, 0.892797589302063, 0.3259250223636627, 0.3369770050048828, 0.4445870816707611, -0.7815781831741333, -2.0964200496673584, 0.3177010118961334, -1.7090356349945068, -0.3066035807132721, 1.0195332765579224, -0.3889695107936859, 0.29303693771362305, -1.7385772466659546, 1.0186187028884888, 1.2159552574157715, -1.6139572858810425, 1.0865710973739624, -0.1643552929162979, 0.3876132369041443, -0.16425320506095886, 0.7652239203453064, 0.8390083312988281, 0.9182445406913757, 0.30373337864875793, -0.5029825568199158, 0.27796176075935364, 0.8244404196739197, 0.07554148882627487, -1.7214481830596924, -1.3154327869415283, 0.5032161474227905, -0.023311743512749672, 2.1061747074127197, 1.2587295770645142, 0.1267216056585312, 0.7237739562988281, -0.9618737697601318, -0.08091982454061508, 2.224384069442749, -1.8724285364151, 0.1800728589296341, -1.3764389753341675, 0.267798513174057, -1.8601661920547485, 1.476908564567566, -0.8339495062828064, 0.8204427361488342, 0.7731477618217468, -0.3909129798412323, -0.7776834964752197, 0.2242133915424347, 1.3521896600723267, 0.19295459985733032, 0.8464213013648987, -0.030556077137589455, -0.6225385665893555, -0.37478968501091003, -0.0671415776014328, 0.4281192421913147, -0.8811566233634949, -0.5084163546562195, 1.3258262872695923, -0.04393897205591202, 0.6954057216644287, 0.9988805651664734, 0.7004633545875549, 1.069669485092163, 0.07495398819446564, -0.32519519329071045, 0.013908357359468937, 0.6600617170333862, -0.46443280577659607], [-2.8979482650756836, -0.7625170350074768, -0.4089032709598541, 0.2310396283864975, -0.6510902643203735, 1.8635940551757812, 0.4573611617088318, 1.0360372066497803, 0.745226263999939, 1.2651536464691162, -0.31537631154060364, 1.0429738759994507, 1.1336067914962769, -0.762676477432251, -1.8260585069656372, -1.5052180290222168, 0.17714643478393555, 0.47532451152801514, 0.6548147201538086, 0.22761276364326477, -0.21562184393405914, -0.07663067430257797, -0.4369356036186218, 0.2354632019996643, -0.3773016631603241, 0.1613476276397705, -0.4758504629135132, 0.6760539412498474, -0.7073242664337158, -0.5633150339126587, -0.18935255706310272, 0.539114236831665, 0.43588918447494507, -0.9451850056648254, 0.14785067737102509, -1.181156873703003, -1.6418043375015259, 0.6231313943862915, 0.6395411491394043, 0.32121551036834717, -0.5807905793190002, 0.1289130300283432, -0.9504009485244751, 1.1553795337677002, -1.3721650838851929, 1.891592264175415, 0.11890579015016556, -0.04116995260119438, -1.4875553846359253, -0.8372671008110046, -0.4451921284198761, -0.23824386298656464, 0.6087389588356018, -0.1628640741109848, -1.1184725761413574, 0.22985808551311493, -0.5501817464828491, -0.99543297290802, -0.9647710919380188, -1.9092841148376465, 0.7222491502761841, -1.0513412952423096, -1.2269420623779297, -0.1232014149427414, 1.734488844871521, 0.09326065331697464, 0.9297672510147095, 0.48111721873283386, 0.5633335113525391, -1.3582347631454468, -1.6784905195236206, -0.3681058883666992, -0.31393155455589294, 0.06424810737371445, 0.6615603566169739, -0.867345929145813, -1.4908071756362915, -1.2167011499404907, -0.14937807619571686, 1.221549153327942, -0.6813256740570068, 0.5558916926383972, 0.5415294766426086, -1.5459188222885132, -0.4156174063682556, -0.9983047842979431, 0.7041237950325012, -0.24180428683757782, -0.030214067548513412, -0.008011489175260067, 0.8966021537780762, -0.1632920205593109, -0.24579250812530518, 0.3151656985282898, 0.9856916666030884, -0.2140485793352127, 2.0001752376556396, -0.5816835165023804, 0.9170835614204407, 0.13145749270915985, -0.2816610038280487, -0.7979561686515808, 0.7415804862976074, -0.2525286376476288, 1.4468886852264404, -0.8259699940681458, -0.16437748074531555, -0.9655527472496033, -0.11571567505598068, 1.031557321548462, 0.07401877641677856, -0.29565081000328064, 0.6617980003356934, 0.6302395462989807, 0.16913914680480957, 0.5955392718315125, -0.8171181678771973, -0.4644971489906311, -0.3985738754272461, 1.852534532546997, 0.23073288798332214, -0.6999882459640503, -0.9208265542984009, 0.4615938067436218, -1.429428219795227, 0.08824633806943893, -0.1237759217619896, 0.41079917550086975, -0.5661219954490662], [0.037878382951021194, 0.27649420499801636, 0.01620953530073166, -0.45044073462486267, 0.9765149354934692, 2.110938787460327, 0.1996944695711136, -0.019606739282608032, -0.7937111854553223, -0.6610435843467712, -0.3109639286994934, -1.3612130880355835, 0.7268345952033997, 0.5401692986488342, 1.5432082414627075, 1.1144931316375732, 0.8265038132667542, -2.0176167488098145, 0.27767977118492126, -1.2850267887115479, -0.43819284439086914, -0.5751997828483582, -1.121488094329834, -1.5238715410232544, 0.706986665725708, 0.25442183017730713, -1.427515983581543, -0.4182780683040619, -1.1347957849502563, 0.8483946919441223, 2.0450644493103027, -0.48584210872650146, -1.736256718635559, -0.7137405276298523, 0.1576928347349167, -0.11044980585575104, -0.8658841848373413, 0.7865619659423828, 2.0994811058044434, -1.1911649703979492, 0.08194712549448013, -0.4940902292728424, -1.5274713039398193, 0.8009833693504333, -0.6941891312599182, 0.613281786441803, -0.016244718804955482, 0.2313799113035202, 1.1649644374847412, -0.9327512383460999, 0.28084489703178406, -0.13827167451381683, 0.1751437485218048, 0.5346766114234924, 0.22932717204093933, -0.17794528603553772, -0.24759523570537567, 0.3181731104850769, 0.3630737066268921, 0.4697268009185791, -0.9630874991416931, -1.2677358388900757, -0.26871880888938904, 0.1542758345603943, 0.28284937143325806, -1.032049298286438, 0.9206557273864746, -0.42869940400123596, -1.1586837768554688, 0.7646875381469727, -0.013527065515518188, -0.037666209042072296, -0.1322668045759201, 2.5012431144714355, -0.498884916305542, -0.009658033028244972, -0.43268242478370667, 0.2511962056159973, -1.9665627479553223, -0.9178867340087891, 2.5475754737854004, -1.5259901285171509, 0.2153693437576294, -0.2829577922821045, -1.2897334098815918, -0.27253636717796326, 0.6429709792137146, 1.059954047203064, -0.28883808851242065, -0.5742650628089905, 0.2366064041852951, -0.14928461611270905, 1.062986969947815, 1.503690481185913, -0.38960275053977966, -1.0936834812164307, -0.062045205384492874, -0.2460683286190033, 0.5290334820747375, -0.31791746616363525, -0.2714245319366455, 0.08283486217260361, -0.9849733114242554, -0.4610976576805115, 0.08596502989530563, 1.2558428049087524, 1.409072756767273, -0.6202481985092163, 0.2506675720214844, -1.6673351526260376, -0.38792183995246887, 0.4673773944377899, -0.3887958526611328, -1.238338589668274, -0.7308349609375, -0.7508895993232727, -1.8042738437652588, -0.027622662484645844, 0.5996075868606567, -0.3456791341304779, -0.9354716539382935, -0.8529316782951355, -0.02368500642478466, -0.4495289623737335, 0.6401463150978088, -0.6438120603561401, -1.1850433349609375, 0.6793282628059387, -0.34911859035491943], [-0.6818670034408569, -1.8185551166534424, 0.6278709173202515, 0.4367210268974304, 1.0006414651870728, -0.21798256039619446, 1.2214607000350952, 3.3527703285217285, -0.5614649653434753, 1.0280488729476929, -0.832379937171936, -1.0838786363601685, -1.5457160472869873, -1.1564748287200928, -0.37734588980674744, 1.137880563735962, -3.3489561080932617, 0.058997318148612976, -0.48464202880859375, 0.08996422588825226, -0.6188990473747253, -1.0575180053710938, 0.5526344180107117, 1.1219547986984253, 0.7539878487586975, 0.03922983258962631, 1.011496663093567, 1.6547898054122925, -1.2832380533218384, -0.1323966532945633, 1.5102674961090088, 0.5762816667556763, 1.0001567602157593, 1.4272117614746094, -0.3568473756313324, 1.3697787523269653, 0.30748096108436584, -0.4239858388900757, -1.1784930229187012, -0.4122367203235626, 1.8336795568466187, -0.5102408528327942, 0.021257221698760986, -0.7340900301933289, -0.7036764025688171, 0.2867918014526367, -0.43906810879707336, 0.20751313865184784, 0.7931875586509705, 0.04194262623786926, 0.09022748470306396, 0.037526488304138184, 1.3271838426589966, 1.323676347732544, -0.6653032302856445, 1.804845929145813, 1.0534745454788208, 0.3286619484424591, 1.1383616924285889, -0.827422022819519, -0.4735301434993744, -0.5750045776367188, -1.5198336839675903, -0.510942816734314, 0.716106116771698, -2.1098153591156006, -0.8451792001724243, 1.064671277999878, -0.32758641242980957, 0.37220054864883423, 0.13044226169586182, -0.38407766819000244, -0.3237139880657196, -0.6549729704856873, -2.2720181941986084, 1.4653204679489136, -0.7591615915298462, 0.08792741596698761, -0.032112233340740204, -0.6382160782814026, 1.1502017974853516, -0.1970127522945404, 1.0943089723587036, 0.2758428752422333, 0.14116887748241425, 0.12183047086000443, 0.027069414034485817, 1.8700037002563477, 2.4582157135009766, -1.260494589805603, -0.8630645275115967, -0.8639318346977234, 2.339216470718384, -0.32628846168518066, -0.10219936817884445, 0.6811707019805908, -0.10093320906162262, -1.4684065580368042, 1.644152045249939, -0.8412363529205322, -1.8796213865280151, -0.09299439936876297, 0.8701415657997131, -1.2405762672424316, -0.7739265561103821, 1.6021727323532104, -0.6283969879150391, -0.961922287940979, -1.5229171514511108, 0.8886004090309143, 0.7378581166267395, 0.30403608083724976, -0.23187163472175598, -1.262382984161377, 1.2123751640319824, -0.22810709476470947, -0.08451660722494125, -1.684293508529663, -1.030117154121399, 1.49144446849823, -0.0073539456352591515, -0.15303586423397064, 0.4024830162525177, 1.1456083059310913, -0.0826946571469307, -0.87724769115448, -0.271702378988266, 0.4493195414543152, 0.2512935400009155], [1.0704039335250854, -2.3833816051483154, -0.43338367342948914, 1.612008810043335, 1.598035454750061, -0.6293371915817261, -0.45532849431037903, 0.03668105974793434, 0.4524911344051361, 1.5077437162399292, 0.38896384835243225, -0.1688685268163681, 0.06853169202804565, -0.12954789400100708, -0.05628480762243271, -0.2508696913719177, -0.9966040253639221, 0.850771963596344, -0.5780560970306396, -0.5637876987457275, 0.9374828338623047, -0.1459750235080719, -0.6246578097343445, 1.0404846668243408, 0.06183908134698868, 0.6365785598754883, -0.0893387645483017, -1.0119688510894775, 0.49849942326545715, -0.27119749784469604, 1.594312310218811, -0.6022300720214844, 2.9444189071655273, -1.1100255250930786, -0.19154787063598633, 1.1199756860733032, 0.08624833822250366, -1.5293456315994263, -0.9938423037528992, -0.6471593976020813, 1.1888679265975952, 1.0909029245376587, -0.4922412037849426, -0.836816668510437, 0.6932899951934814, -0.9472537040710449, 1.1603108644485474, -1.1934369802474976, 1.2000813484191895, 1.0627824068069458, -0.797156572341919, 0.010654055513441563, -1.2845962047576904, 1.1770085096359253, 0.3926628828048706, 1.0119314193725586, 1.4581185579299927, 0.280617356300354, 0.6825913190841675, 0.5764803290367126, -0.11870977282524109, 1.0813144445419312, -0.24619032442569733, 1.2504100799560547, -0.3119141757488251, -1.3909107446670532, -1.8009815216064453, -1.064211368560791, -0.20911478996276855, 0.9580978155136108, 0.05343896523118019, -1.1021690368652344, -0.43052107095718384, 1.0854473114013672, -1.660925269126892, 0.2952997088432312, 0.09194426238536835, 0.3152611553668976, 0.9724256992340088, -0.4285827875137329, -1.2440961599349976, -1.3492417335510254, 0.0900760367512703, 2.1022071838378906, 0.9976314306259155, 0.042877860367298126, -0.031339142471551895, -0.03493431583046913, 1.1893303394317627, 0.5002689361572266, -0.20023953914642334, -0.031733397394418716, -0.22705844044685364, 2.59016489982605, 0.1440095752477646, -0.7458493113517761, -0.4099790155887604, -2.4251890182495117, 0.33200228214263916, -0.32375842332839966, -0.9458948373794556, -1.1193612813949585, 0.2657466530799866, -0.2978927791118622, -1.3098716735839844, -0.43135279417037964, -1.044360876083374, 1.098998785018921, 1.7362838983535767, 0.5311829447746277, 0.15867717564105988, -3.2600035667419434, 0.5798816084861755, -0.20825786888599396, -1.0585352182388306, -0.6618510484695435, -1.0994805097579956, 1.2780895233154297, 0.2637629806995392, 1.3747529983520508, -0.9836982488632202, 0.9795376658439636, -0.6548742055892944, 0.2846093773841858, -1.0172595977783203, 1.4745033979415894, -0.3818550109863281, -0.09191573411226273, 0.3052082657814026], [1.3367384672164917, -1.4481672048568726, 0.5240283608436584, -0.6379608511924744, -1.5302308797836304, -2.1237716674804688, -0.15316835045814514, 0.3197658360004425, 0.8075013756752014, 1.7819180488586426, 0.7811321020126343, 0.9206183552742004, -0.775086522102356, -0.8243502974510193, 1.3013057708740234, -1.132338523864746, -0.029959963634610176, -1.201969027519226, -1.19158136844635, -0.9037455320358276, 0.6907702684402466, -1.0065478086471558, -0.9825971126556396, 1.5317233800888062, 0.861747682094574, 0.6777079701423645, -0.6960678100585938, -2.048332929611206, 2.9267220497131348, 0.874565064907074, -0.49989813566207886, -0.33659425377845764, 1.0771232843399048, 1.2140065431594849, 0.23982611298561096, 0.6610738635063171, -0.8186455368995667, -0.8710946440696716, -1.5799362659454346, 1.444620132446289, -0.21925677359104156, -0.8827975988388062, -1.0299127101898193, -0.3555087149143219, -0.2855667471885681, -0.03140944242477417, -0.3998681306838989, 1.21696138381958, 0.38552576303482056, 1.8225905895233154, -1.4557483196258545, 2.154176950454712, 0.8500264286994934, -0.025796936824917793, -0.06329874694347382, -1.7667988538742065, -0.6610345840454102, -0.32537299394607544, -0.5983248353004456, 0.6949640512466431, 0.7931592464447021, 0.8863007426261902, 0.8028349280357361, 1.5741780996322632, -1.4231154918670654, 1.022934913635254, -0.359797865152359, 0.4942817986011505, 1.4098927974700928, -0.9228878021240234, -0.5520387291908264, -1.4989874362945557, -0.8774945139884949, -0.5122223496437073, 1.7955002784729004, 0.3502134382724762, 2.892852306365967, 0.6761249899864197, -1.24118971824646, 1.4578900337219238, -1.7757580280303955, 0.057486582547426224, -0.3518575429916382, -0.4384934902191162, 1.4789634943008423, 2.7381210327148438, 0.3843878209590912, -0.9379169940948486, -0.4593670666217804, -2.1791741847991943, -0.10842769593000412, -1.1589375734329224, -1.3583701848983765, 0.7594302892684937, -1.364752173423767, -0.5886062979698181, -0.3820461928844452, -0.808871865272522, 0.8192647695541382, -0.8943516612052917, -1.2029951810836792, -0.4076399505138397, 0.42105579376220703, 0.36453357338905334, -0.10613658279180527, 0.24905243515968323, 0.044127583503723145, -0.3393898010253906, -1.4386658668518066, 0.6280757188796997, 1.956461787223816, 1.2764204740524292, 0.500981867313385, 1.58562433719635, -1.0669611692428589, -0.3076517581939697, 0.7838883996009827, -0.472881555557251, -0.4420939087867737, 0.903606653213501, -0.5810065865516663, -0.9895719885826111, 0.14317341148853302, 0.45389559864997864, 0.40022844076156616, 0.1208040788769722, 0.11574748903512955, -1.1730786561965942, -1.2288919687271118], [0.6389551162719727, -0.5407201051712036, 2.1258952617645264, 0.321981281042099, -0.10208221524953842, -0.028713583946228027, 3.724555015563965, -0.7194525599479675, -0.36532601714134216, 0.7054011821746826, -0.12194298207759857, 1.2835582494735718, -0.45329126715660095, -0.08087507635354996, -0.4795072376728058, 0.5121810436248779, -1.1520955562591553, -0.054392557591199875, -0.16847273707389832, -0.36571475863456726, 0.8094913959503174, 0.9387965202331543, 0.5637288689613342, 0.43690741062164307, 0.3089488446712494, -0.7122067213058472, -0.0930401086807251, -0.4301196336746216, 0.05917108803987503, -0.3427862226963043, 1.0116147994995117, 0.9004595875740051, -1.058651328086853, -1.7330981492996216, -0.49691906571388245, 0.01719658449292183, 0.25061655044555664, -0.5200885534286499, -1.3929879665374756, 0.5706484317779541, 1.00312077999115, -0.11437974870204926, -1.1140443086624146, 0.42925623059272766, 1.0077948570251465, -0.22143034636974335, 0.6269290447235107, -0.1432328224182129, -0.2707529664039612, -0.5336691737174988, -0.8096877932548523, 0.01669037714600563, 0.6325089335441589, 0.6423555016517639, -1.1583231687545776, -0.13149894773960114, 0.5373458862304688, -0.22191983461380005, 0.4686514735221863, -0.5989519953727722, -0.8687292337417603, -0.23979176580905914, 2.1721248626708984, -0.2445324957370758, -1.6890395879745483, 0.8587695360183716, -0.18805503845214844, 2.0377190113067627, 1.6120121479034424, 0.6811508536338806, -1.301395297050476, -1.8905832767486572, 0.6995092034339905, -0.6292856335639954, -0.357295423746109, -0.6588494777679443, -0.2168581783771515, -0.007641097065061331, 1.349096417427063, -1.3947466611862183, -0.08151616156101227, 1.414534091949463, 0.097455233335495, 1.5711110830307007, -0.8779832124710083, -0.5611281991004944, -1.7553496360778809, 0.054444342851638794, 0.804909884929657, 0.43550658226013184, -1.3792966604232788, -1.0782780647277832, 0.916370689868927, 0.2841261625289917, -1.3718093633651733, -0.6551002860069275, 0.839461088180542, -1.0569051504135132, 1.5996856689453125, 1.7121741771697998, 0.4539261758327484, -1.2171602249145508, -0.3450358510017395, -0.5817918181419373, 0.6201335787773132, 1.4573493003845215, -0.9712889790534973, 1.1205419301986694, 0.6911949515342712, 0.7553138136863708, -0.4253707230091095, -0.7677220702171326, -0.21650581061840057, 0.3277328908443451, 0.0474814772605896, -1.2787705659866333, 0.4650861918926239, -0.8600353002548218, 1.1630730628967285, 0.2017510086297989, 1.9849205017089844, -0.09973686933517456, 1.204464077949524, 0.1259540617465973, -1.4812883138656616, -1.0673062801361084, 1.1348590850830078, -1.8927098512649536, -1.2964881658554077], [0.7872218489646912, 0.42880430817604065, 1.0048426389694214, -0.10963857918977737, 0.83106529712677, 0.7798346877098083, -0.6900555491447449, 0.0639767274260521, -0.3348988890647888, -0.21930921077728271, 1.8270756006240845, 1.9225850105285645, -0.3278205692768097, -0.9755867123603821, -0.3386334478855133, 1.3960493803024292, -0.7179803252220154, 0.9916967749595642, 1.7045472860336304, 0.7490577101707458, 0.4531419575214386, 2.184549331665039, 0.0984291061758995, 1.929267406463623, -0.5512275695800781, 0.7583464980125427, 0.7005266547203064, 0.516768753528595, -0.46680641174316406, 1.0294357538223267, -2.207516670227051, -1.9140905141830444, -0.848310649394989, 0.8516212105751038, -0.40331438183784485, 0.6443816423416138, -2.068941831588745, -0.5696902275085449, -0.44739341735839844, -0.6533661484718323, 0.17288322746753693, -0.8983778357505798, -0.9186131954193115, -0.4160560965538025, 0.953571617603302, -1.3461965322494507, -1.9352236986160278, -0.9792032837867737, 0.6075149774551392, -0.8073329329490662, 0.41762059926986694, 0.040814585983753204, -0.060278479009866714, -0.405386745929718, 0.5287624001502991, 1.718636393547058, 0.4916442036628723, -0.9728103280067444, -0.5456492304801941, -0.8856167793273926, 2.0506703853607178, -1.4378582239151, 1.7945053577423096, 0.4173465669155121, -0.032671067863702774, 0.1339576095342636, -0.5636804103851318, 0.48866721987724304, -0.7111214995384216, 0.5624257922172546, -2.3473987579345703, 0.9156395792961121, 0.6087907552719116, 0.13254046440124512, 0.06049570068717003, 0.43082737922668457, -0.46916431188583374, -1.1833398342132568, 0.33210769295692444, 0.2259884625673294, -0.4859413206577301, 0.30530428886413574, 1.6615878343582153, 0.8053680062294006, 0.6698216199874878, 0.12007459253072739, 0.8409258127212524, 0.22039619088172913, -0.16982752084732056, 1.426634669303894, 0.7032566070556641, -1.216223955154419, -0.59426349401474, -0.4799739420413971, 1.4037196636199951, -0.7455029487609863, 0.8677802681922913, -0.8334867358207703, 1.2139965295791626, 0.45525822043418884, -0.1584487110376358, -1.5171810388565063, 0.7918919324874878, 0.08123008906841278, 0.7568356394767761, -1.139499306678772, 0.30117928981781006, 0.31749147176742554, -0.9899025559425354, 0.12101887911558151, -0.8223353028297424, -1.0408326387405396, 0.9180274605751038, 0.27613911032676697, 0.7961682677268982, -1.2004282474517822, -1.9883418083190918, -0.048489753156900406, 0.7238065600395203, -1.2285743951797485, 1.1353414058685303, 0.022366663441061974, -1.102148175239563, -0.3588598072528839, -0.16803310811519623, -1.3407062292099, 0.5794724822044373, 0.11221153289079666, -0.028963500633835793], [-0.3966476321220398, 0.1953122615814209, -0.3210360109806061, -0.16258743405342102, -1.1227047443389893, -0.8261531591415405, -0.3125125467777252, 0.30068454146385193, 0.5116131901741028, -0.5207260251045227, 0.9599693417549133, 1.0288954973220825, -0.4750438928604126, -2.1314620971679688, -1.2186126708984375, -0.7972638010978699, 0.5941770076751709, -1.2068628072738647, 0.31634244322776794, -0.4451068639755249, -0.967467188835144, -0.090817891061306, -0.7334079146385193, 0.013241657055914402, 0.47519707679748535, 0.11929017305374146, -0.5381099581718445, 1.8522130250930786, -0.03901515528559685, 0.16212423145771027, -0.8847532868385315, 0.4782160818576813, -0.2738078534603119, -0.9345929026603699, 1.0265817642211914, 0.1161464974284172, -1.2459385395050049, -0.7925612330436707, 1.391732931137085, 1.2610561847686768, -0.06281672418117523, 0.12616266310214996, -0.6029677391052246, -0.6062888503074646, 0.5781386494636536, -0.4404594302177429, -1.7727775573730469, 0.3217211067676544, 0.6110743880271912, 2.242475986480713, -0.3863368034362793, -1.004770278930664, 0.9156138300895691, -0.21847227215766907, 1.47825288772583, 0.15546420216560364, 0.15749500691890717, 0.992406964302063, 1.6781564950942993, -0.6380584836006165, 0.9901064038276672, -1.7700390815734863, 0.1519494354724884, -1.8821607828140259, 2.261895179748535, -0.24629716575145721, -2.6735358238220215, -0.41667744517326355, -0.576684832572937, -0.3375202417373657, -0.12709346413612366, -1.6678186655044556, 0.3171112835407257, -1.9885807037353516, 0.6503332853317261, -0.6783531904220581, -1.1503499746322632, 0.14403386414051056, -0.20567142963409424, 0.013740171678364277, -1.03811514377594, 0.8188679814338684, -2.0318799018859863, -1.943809151649475, 0.959340512752533, -0.34481173753738403, -0.6052516102790833, 0.06478247046470642, 0.7028854489326477, 0.6091349124908447, -0.2635561525821686, 0.9802230000495911, -1.4259666204452515, 0.8401457667350769, 0.35521551966667175, 2.34047269821167, 1.7129682302474976, 0.4056183993816376, -1.446128010749817, -0.3021908402442932, 0.04799770936369896, -0.8178383111953735, -1.3339791297912598, 0.9610195159912109, 1.4711849689483643, 1.2671890258789062, -1.4477428197860718, 0.6067322492599487, 1.6321028470993042, -1.316584825515747, -0.7843410968780518, -1.9946073293685913, 1.2462387084960938, 0.21977025270462036, -0.755440354347229, 1.1442925930023193, 0.6330652832984924, -0.8374453783035278, -2.1149349212646484, 0.7268694639205933, 0.6222774386405945, -2.1076197624206543, -2.522597312927246, -2.262146472930908, -0.6333662867546082, 0.8887059092521667, 0.36551499366760254, 0.029538648203015327, 0.5521324872970581], [1.3553272485733032, -0.6517120599746704, 0.2795283794403076, -1.3811790943145752, 0.4217759072780609, -0.06803905963897705, 0.45196157693862915, -1.2745336294174194, 1.3036842346191406, 0.027919825166463852, -0.9356111288070679, 1.6561636924743652, -0.8215784430503845, 1.4556610584259033, 0.8948807120323181, 1.0256098508834839, 1.0602072477340698, 0.0670783668756485, -0.17319534718990326, -0.21142970025539398, 2.176499366760254, 0.49460217356681824, 1.4356532096862793, 0.22750256955623627, -0.3360368013381958, -0.9607751369476318, 0.44155362248420715, 0.3376074731349945, -0.04631422087550163, 0.8050847053527832, 0.7526201009750366, 1.0142806768417358, -0.09197331964969635, -0.7878813147544861, 0.17621377110481262, -2.3760995864868164, -1.0400017499923706, -0.43131908774375916, -0.3606218993663788, -0.28070709109306335, -0.18686062097549438, 0.6324946284294128, 1.0883800983428955, -0.8758036494255066, 1.6670384407043457, 0.4497038424015045, -0.5471193790435791, 0.44957563281059265, 0.048518434166908264, -1.29706871509552, 1.9306657314300537, 1.7432221174240112, 0.5588067173957825, -1.244254469871521, 1.2438000440597534, 0.1941099464893341, -2.1418473720550537, -1.379888892173767, -0.17110604047775269, -0.23845455050468445, -0.3946596086025238, 1.320534586906433, 1.3063735961914062, -1.4742765426635742, -0.7335941195487976, -0.9572962522506714, 2.044187545776367, 0.22441840171813965, 1.139543890953064, 0.9314520359039307, -0.21940337121486664, -0.5419881343841553, -1.9197330474853516, 0.16399461030960083, -0.49569931626319885, 0.15245144069194794, 1.0993399620056152, -0.22953802347183228, 0.6142237186431885, 0.022865841165184975, -0.7285438776016235, -0.978342592716217, -0.6968873739242554, -0.4906795918941498, 0.7499879002571106, -0.02381882071495056, -1.4828792810440063, -0.15463787317276, 0.8761152625083923, 0.750747799873352, -0.8309161067008972, -2.053048610687256, -0.6771849989891052, -1.4268536567687988, -0.5086204409599304, -0.13060522079467773, -0.6205140948295593, -0.1372579038143158, -0.23421400785446167, -0.35111990571022034, 0.3366725742816925, 0.29544925689697266, 0.08436977863311768, -0.9301918745040894, -0.9459748268127441, -0.4331454336643219, 0.2949850261211395, 0.43208861351013184, -0.22156482934951782, 0.19758637249469757, -0.8821029663085938, -0.6014763712882996, 1.2079708576202393, -1.546000599861145, -0.39802086353302, -0.46942731738090515, 0.27978086471557617, 0.6953622698783875, -1.6725291013717651, -1.2405844926834106, 0.07302621006965637, -0.4353489577770233, -1.039101004600525, -0.33938121795654297, -0.9980378746986389, 1.3279123306274414, 0.8784265518188477, 0.27813538908958435, -0.8670129179954529], [1.79213285446167, -0.6314646601676941, -0.3730252981185913, 2.416311740875244, 0.5601198673248291, -0.3930217921733856, -0.2502790689468384, -1.298903226852417, -0.6793396472930908, 0.3459067940711975, 1.5147444009780884, -1.7254186868667603, 1.439365029335022, -1.8326845169067383, 0.21094655990600586, -0.5026144981384277, 1.0746558904647827, -0.2071661800146103, 2.9495179653167725, -0.06880749017000198, 0.6999249458312988, -2.12764048576355, 0.7954736948013306, -1.6111632585525513, -1.0176664590835571, -0.09193996340036392, -1.5849647521972656, -1.0045373439788818, -0.8610387444496155, 1.78110671043396, -0.0014535416848957539, -1.1017953157424927, -0.06413723528385162, -1.2736015319824219, 0.31320151686668396, -0.2595132291316986, 2.3149640560150146, 0.6713294386863708, -2.99613094329834, -0.2931033670902252, 0.7010681629180908, 1.339167833328247, 0.8415288329124451, 1.5527759790420532, -0.49474042654037476, -0.25809013843536377, -1.8054739236831665, -0.25256088376045227, 0.8627254962921143, -2.15445876121521, 0.6868023872375488, 0.012475905939936638, -0.36601677536964417, 0.43505972623825073, 1.3623123168945312, -0.14720137417316437, -1.7931140661239624, 1.2057827711105347, -1.9978282451629639, -0.2592867910861969, 0.19021464884281158, 1.1849360466003418, 0.16081932187080383, 0.15464091300964355, -0.02498514950275421, 1.0076874494552612, -0.06074891611933708, 0.8646313548088074, 0.9496622085571289, -1.4043956995010376, -1.9703999757766724, -0.7372519969940186, -0.4237194359302521, 0.01486441120505333, -1.786026120185852, 0.535261332988739, 0.9408800005912781, 0.3158131539821625, 0.3004918694496155, -0.4177882671356201, 2.3582305908203125, 1.0973061323165894, -0.7124248147010803, -1.126753568649292, -0.6408408880233765, -2.1234939098358154, 2.001765012741089, -0.8676174283027649, 1.2096105813980103, 0.46604329347610474, 1.7986451387405396, -0.1111898347735405, 0.37770599126815796, 1.505811095237732, 1.3228330612182617, -0.227430060505867, 1.682682991027832, -2.8821964263916016, 0.9480640292167664, -2.886354684829712, 0.49631717801094055, 0.11775245517492294, -0.7500669956207275, -0.21558703482151031, 1.0646469593048096, 0.3228226602077484, -0.6410813331604004, -2.631279230117798, -1.1098971366882324, -0.7598679065704346, -0.1066487655043602, -0.6342374086380005, -0.539818525314331, -0.6764746904373169, 0.400092750787735, 0.007895463146269321, 0.8442891836166382, -1.3661949634552002, -0.5801005959510803, 0.8859793543815613, 0.37262094020843506, -0.48970744013786316, -0.5319019556045532, -0.9750853180885315, 0.011913610622286797, 1.008249282836914, -2.187791109085083, -0.7381348609924316, -0.27095314860343933], [0.338657945394516, -0.24092070758342743, -0.31687888503074646, -1.2083128690719604, 0.4681099057197571, 1.3182927370071411, 1.188826084136963, 0.2522420883178711, 0.6105054616928101, 1.0237412452697754, 0.41661614179611206, -1.5936775207519531, 1.0320258140563965, 0.04515761137008667, -1.2810754776000977, 0.1691460907459259, -0.2536930739879608, 0.7569397687911987, 0.22620579600334167, -0.23082135617733002, -0.04327218607068062, -1.1482948064804077, -0.9297388195991516, -0.20655527710914612, 0.8017788529396057, 0.28850218653678894, 0.8529184460639954, -0.6429224014282227, -0.9704451560974121, -2.097580671310425, 0.778965413570404, 0.4663621485233307, 0.3955882489681244, -0.21310144662857056, 0.7870323657989502, -1.262567400932312, -0.9414032697677612, 1.0483003854751587, 0.5973736047744751, 0.796071469783783, 2.6976797580718994, 0.1555134803056717, 0.4353243410587311, -1.2294416427612305, -0.38567110896110535, -1.4884779453277588, 0.16063348948955536, 0.9801123738288879, 0.214058518409729, 0.5017400979995728, 0.3773379921913147, -0.9674713611602783, -0.05622914060950279, -1.4460031986236572, -0.04332391545176506, -0.5836530923843384, 1.2562177181243896, -0.39354208111763, -0.5472745895385742, -0.40317484736442566, 1.9276193380355835, -2.2861969470977783, 2.4239015579223633, 0.8955579400062561, -0.14673039317131042, 1.4351410865783691, -0.5682393312454224, 0.4384552538394928, -0.3295919895172119, -0.6917557120323181, 0.3909398317337036, 1.319573998451233, 0.31148451566696167, 0.5659220814704895, -0.617527961730957, 0.36945635080337524, -0.39619556069374084, 1.5452253818511963, 1.100428819656372, 0.24846839904785156, 1.0580718517303467, -0.7749313712120056, 1.076473355293274, 0.49641749262809753, -0.7920615673065186, -0.43893373012542725, 0.7687953114509583, 0.9810654520988464, -0.21230636537075043, 0.2671792507171631, 2.341798782348633, -0.8527941703796387, -0.0984780490398407, -1.0303676128387451, -1.4204167127609253, -0.20650456845760345, -0.29920801520347595, -1.3943352699279785, 1.0470167398452759, 1.9227091073989868, 1.3971471786499023, -1.1019582748413086, -1.0939944982528687, -0.6164469718933105, -1.0974483489990234, 1.1350221633911133, 1.3785276412963867, 1.19796884059906, -0.0002648097288329154, 1.410256028175354, -0.3333061933517456, -0.19271604716777802, 0.38347965478897095, -0.24300670623779297, -0.8441690802574158, -0.20807228982448578, -0.16196684539318085, 0.9768916368484497, -1.8245285749435425, 1.1963162422180176, 1.2838174104690552, -1.4667943716049194, 1.1390700340270996, -0.7864686846733093, -0.9168909192085266, -0.649787425994873, 1.8428244590759277, -0.3165608048439026, 0.5772594213485718], [1.2154403924942017, -0.17693206667900085, -0.4837951064109802, -1.177075743675232, -0.4434265196323395, -0.2414080798625946, 0.13486436009407043, 0.7965395450592041, 0.2780134081840515, -0.36693429946899414, -0.2585248053073883, -0.1613602489233017, -2.149813652038574, 0.21493694186210632, -1.900235891342163, -2.373314380645752, 1.6968168020248413, 0.7664710879325867, -0.9958914518356323, 1.007691502571106, -1.3271437883377075, 0.17668285965919495, 0.3294120728969574, 0.1145409569144249, -1.5300570726394653, -1.015600562095642, -0.8215070962905884, 0.44245511293411255, 0.39625173807144165, 1.4589120149612427, -2.168976068496704, -0.9489340782165527, -0.5558604598045349, 0.7937920689582825, -0.048891548067331314, -1.4745818376541138, -0.24430538713932037, 0.6566022634506226, -0.7940982580184937, -0.3713676333427429, 0.8006906509399414, 0.19206653535366058, 0.8431559801101685, -2.186861276626587, -1.217775821685791, -0.02849649451673031, 0.8182927966117859, -1.1231577396392822, -0.8921290040016174, -0.15247842669487, 2.1693503856658936, 1.479349136352539, -0.663166880607605, 0.546349287033081, -1.7179114818572998, -0.7949936985969543, -0.6119343638420105, -1.6519463062286377, 1.4335289001464844, -0.36230430006980896, -0.3862346410751343, 0.3347764313220978, -1.0876448154449463, 1.2871458530426025, -0.043669845908880234, -0.43916136026382446, 0.3636409044265747, -0.1712447702884674, 0.2611091136932373, -1.8939976692199707, -1.5769391059875488, -0.06279941648244858, 0.9182826280593872, 0.5723872780799866, -1.0470877885818481, -0.2621269226074219, -1.292576789855957, -0.019574683159589767, 0.36539000272750854, -0.44342923164367676, -0.4945294260978699, -0.17691804468631744, 2.256791591644287, 1.2486625909805298, 0.29838502407073975, 1.0796769857406616, -1.2128788232803345, 1.149707317352295, -1.0247693061828613, 1.9619555473327637, 0.2475253939628601, 1.2251962423324585, -1.4781653881072998, 0.097141794860363, 1.1903190612792969, -0.594573974609375, 0.2518141567707062, 0.7616503834724426, 0.4863574802875519, -0.30507248640060425, 0.4254451096057892, -0.3055655062198639, 0.8260540962219238, -1.5097469091415405, 0.30430054664611816, -0.027430551126599312, 0.6782724857330322, 0.054807230830192566, 0.9184992909431458, -1.2594114542007446, 2.018996238708496, -2.6771240234375, 1.6231268644332886, 0.9086476564407349, 0.37265533208847046, 1.226243257522583, 0.2819216549396515, 1.291458010673523, 0.18017424643039703, 1.6815546751022339, -0.38017696142196655, -0.4679948091506958, -0.26932400465011597, 1.7945483922958374, -1.3072333335876465, -0.46363934874534607, 0.5610357522964478, 0.6237586736679077, 1.7550804615020752], [1.0345754623413086, 0.677331805229187, 0.26761293411254883, 0.46463721990585327, -0.12360039353370667, 0.9937542080879211, -0.33761659264564514, 0.01910439133644104, 0.06033848598599434, -0.920569121837616, -1.4031274318695068, 0.9249445796012878, 0.2805253565311432, -0.6341529488563538, -2.0447332859039307, -1.6605671644210815, -0.9517078995704651, -0.09529506415128708, 0.24218596518039703, 0.3515984117984772, -0.2624567747116089, 0.7701313495635986, 1.0844883918762207, -0.2117958515882492, -1.2047196626663208, 0.5249745845794678, 0.8812763094902039, -0.6255518794059753, 0.8910657167434692, -1.3030714988708496, -2.538978099822998, -0.7677347660064697, 1.3219040632247925, 1.7170075178146362, -0.391182541847229, -0.4531656503677368, 0.2594825029373169, -0.8686086535453796, 0.9387578368186951, -0.6278567910194397, -0.1799921989440918, -1.2237827777862549, -1.7596662044525146, 0.12372814118862152, -0.48511233925819397, 0.9980692863464355, 0.5636810064315796, 1.8345293998718262, -1.4079822301864624, -0.9677069783210754, 0.36205852031707764, -0.41016092896461487, -0.069008968770504, -0.6008895635604858, -1.7196587324142456, -1.7413181066513062, -0.587697446346283, -0.732063353061676, -0.01925770752131939, -1.260453462600708, 0.4177408814430237, -0.37831515073776245, 0.9697006940841675, -2.0198051929473877, 0.5441317558288574, -0.28824976086616516, 0.7472348213195801, 0.8692857623100281, -1.1923249959945679, 0.6630513072013855, 1.3908897638320923, 0.8432503938674927, -0.5349255204200745, -0.01655915006995201, -1.9794456958770752, -0.8646589517593384, 1.2800554037094116, 1.5607142448425293, 1.2438788414001465, 0.15163056552410126, 1.2264047861099243, 1.1349108219146729, -0.8795464038848877, -1.140721321105957, -0.8372859954833984, 0.8745138645172119, -1.2780005931854248, 0.9996966123580933, -0.48784294724464417, 0.0015682759694755077, 0.0943196713924408, -0.08173193782567978, -1.2909597158432007, 0.9783825874328613, -0.3306053876876831, -0.6476478576660156, 0.6041268110275269, 0.2935037910938263, 0.3373487591743469, -0.1747741997241974, 1.389363169670105, -0.5121285319328308, -1.6784433126449585, -0.6057080030441284, 0.24027906358242035, 0.07633935660123825, -0.8141867518424988, 0.9719176888465881, -0.8891218304634094, 1.888517141342163, 0.8366221785545349, 0.9505264163017273, -0.47120362520217896, 1.2300693988800049, -1.4529484510421753, -0.35330042243003845, -0.38615840673446655, -0.06849514693021774, -1.197192668914795, 0.1044117882847786, 0.43687817454338074, 0.9324919581413269, 0.6445426344871521, -2.00758957862854, -0.08929666131734848, -0.4675292372703552, -0.05140328034758568, 1.5177440643310547, -0.854790985584259], [-0.0399344339966774, -0.9091798663139343, 1.9198856353759766, 0.7232599854469299, 2.233690023422241, -0.5856001377105713, -1.4090230464935303, 0.41323286294937134, 0.033623386174440384, 2.4001011848449707, 0.11747647076845169, -0.6169934868812561, 0.3539385497570038, -1.2763887643814087, 0.4733212888240814, 0.42953041195869446, -0.7299238443374634, -0.4426394999027252, 1.0761799812316895, 0.5948842763900757, 0.32826879620552063, -1.3806692361831665, 0.27691981196403503, 0.007371054962277412, 2.111039161682129, -0.48986443877220154, 0.9076681733131409, -0.557765007019043, -0.42594286799430847, -1.3673815727233887, 0.050818491727113724, 1.4317134618759155, -0.004592156503349543, -0.2381986677646637, 0.9663160443305969, 0.688991904258728, 1.0037378072738647, 0.12865617871284485, -0.4997621476650238, 0.4004715383052826, -1.180141568183899, -1.5590722560882568, 0.9087304472923279, 1.1591591835021973, 1.0318326950073242, 0.5705614686012268, 0.1682741343975067, 0.5756921172142029, -1.2696627378463745, 0.6064404845237732, -0.40899452567100525, 1.4790886640548706, 0.03234199061989784, -0.22591909766197205, 1.1766257286071777, -0.6076871752738953, -0.10269507020711899, 2.4746322631835938, -0.8381913304328918, -0.71815425157547, -1.1929892301559448, 0.32774171233177185, 1.2939091920852661, 0.4336469769477844, 1.1377062797546387, -1.1283835172653198, 0.22498227655887604, 0.6507342457771301, 2.116670846939087, 0.40770623087882996, 0.874921441078186, 0.4644540250301361, -0.222025528550148, -0.05951632931828499, -1.0371134281158447, -0.8906630873680115, -1.910428762435913, 0.6879702806472778, -0.08049651980400085, 1.7811304330825806, 0.9936185479164124, -1.2679059505462646, -0.0733705535531044, 0.029835553839802742, -1.0818578004837036, -0.06191186234354973, 0.8010082244873047, -0.13158394396305084, -0.8939383625984192, 0.6399759650230408, 0.8006298542022705, -1.9996272325515747, 0.5642868876457214, -0.5531241297721863, -1.3236029148101807, -0.6838607788085938, 0.30687281489372253, 1.098044991493225, -2.193127155303955, 0.44388431310653687, -0.49466994404792786, 0.821971595287323, -0.5093398690223694, -0.9348275661468506, 0.5379077792167664, 0.2414638251066208, -0.17711244523525238, -0.6174156069755554, -1.3095166683197021, -0.09426360577344894, 0.46570247411727905, -0.5556866526603699, -1.0015654563903809, 2.089766025543213, 1.131666898727417, -0.2212279587984085, -0.6523651480674744, -0.37318289279937744, 0.46053215861320496, -0.06681566685438156, 2.330554723739624, 0.6858856081962585, 1.4671720266342163, 0.7387725710868835, 0.17203259468078613, -0.22234061360359192, -0.9026100635528564, 0.05201437696814537, -0.34822285175323486], [0.4594618082046509, 0.06601758301258087, -1.00944983959198, -0.09244368970394135, -0.5582781434059143, -0.29730793833732605, 1.080458164215088, -1.7607728242874146, 0.14375153183937073, -1.3919262886047363, 0.09222175180912018, -0.5548959970474243, 1.1177703142166138, -1.9462294578552246, -0.18560416996479034, 0.46499672532081604, 0.8946002721786499, 0.16792024672031403, 1.015280842781067, -0.087750643491745, 0.15545456111431122, -0.125491663813591, 0.21436446905136108, 0.4256376624107361, 1.286975383758545, -0.7231495380401611, -0.6153779029846191, -0.12063724547624588, 1.078284740447998, -0.4956754148006439, -0.9355958700180054, -1.2412556409835815, 0.8575485348701477, 0.48025673627853394, -2.0785539150238037, 1.1017364263534546, 0.5452978014945984, -1.4654079675674438, 1.249290943145752, -0.1430669128894806, -0.0011480296961963177, -0.13418181240558624, -0.995697021484375, 0.27405935525894165, 0.8234025239944458, -1.028855800628662, -1.2064577341079712, -0.8929857015609741, 0.16642315685749054, 0.09276904910802841, -0.55668705701828, 0.4900675117969513, -0.6748969554901123, 1.0069797039031982, -0.4287067651748657, -1.3332794904708862, -0.920112669467926, -0.6182594299316406, 0.7043618559837341, 0.0212427768856287, -0.35409271717071533, 1.1253036260604858, 1.0537358522415161, 1.3254929780960083, -0.12445659190416336, 1.3067082166671753, -0.4877532124519348, 0.46900659799575806, -0.07692215591669083, -0.42861756682395935, 1.2663278579711914, -0.29858219623565674, -0.16858291625976562, 0.35877060890197754, 0.5790838599205017, -1.318565845489502, 0.07946861535310745, 0.6396734714508057, 0.11778848618268967, 0.6881294250488281, -1.0010483264923096, 0.40675169229507446, 0.21168093383312225, -0.1904350221157074, 0.516417920589447, -0.04525340348482132, -1.7205427885055542, 0.42204269766807556, -0.13738973438739777, -0.699295163154602, -0.4640844762325287, 0.5609484314918518, -0.4806319773197174, 1.0555974245071411, 1.0484645366668701, -0.43239548802375793, 0.18671222031116486, 1.351432204246521, 1.5231274366378784, -0.3998816907405853, 1.6020475625991821, -0.29607969522476196, 0.6944290399551392, 0.5341346859931946, -0.4540340006351471, -0.810936689376831, -1.0220385789871216, -0.2630138695240021, 0.021745026111602783, 0.444013386964798, -0.8965032696723938, -1.5929532051086426, -0.3923737108707428, 0.9790509343147278, 0.8047808408737183, 0.14924877882003784, 0.5892572402954102, 0.7896798253059387, -0.12486274540424347, -0.5693289041519165, 1.8490909337997437, -0.17833606898784637, -0.3967345356941223, -1.6963677406311035, 0.2022196352481842, 1.7828102111816406, 0.6235856413841248, -1.923694133758545, -0.6321226954460144], [0.06453399360179901, -0.5681372284889221, -0.44089803099632263, -0.6475204229354858, -0.9077690839767456, -2.013944387435913, 0.08504950255155563, 0.20298737287521362, 0.3901100754737854, -0.6620591878890991, -0.1278836578130722, -0.8100597858428955, 1.8254048824310303, -1.0328854322433472, 0.04407576844096184, 1.5909258127212524, 1.0553666353225708, 0.765303373336792, 0.7282819151878357, 3.114987373352051, 0.48655492067337036, 0.06016198545694351, -0.6924170255661011, -0.14203646779060364, -0.11460741609334946, 0.27612605690956116, 1.2841782569885254, 1.8031502962112427, -0.008951679803431034, 0.9293128252029419, -2.8130528926849365, -0.7167500853538513, 2.378115177154541, 0.03750399500131607, 0.46929818391799927, -1.3832696676254272, 0.39187735319137573, -1.9438213109970093, -1.1489307880401611, 2.2826497554779053, -1.5828983783721924, -0.517099142074585, -0.14070479571819305, -1.3381494283676147, 1.0007048845291138, -0.24493606388568878, 0.21760527789592743, -0.6012787818908691, 0.18553102016448975, 1.0754847526550293, 1.494412899017334, 0.37253615260124207, 1.211146354675293, -0.3508847653865814, -0.7514660358428955, -0.30069980025291443, -0.12430877238512039, -0.8765279650688171, 0.15153661370277405, 0.6273914575576782, 0.09651913493871689, -0.9898176193237305, 0.2908940315246582, 0.38390710949897766, 0.9434973001480103, 0.8115478157997131, -1.2175251245498657, -0.3723010718822479, -0.23251734673976898, 1.1662157773971558, 1.5552289485931396, 0.3856923580169678, 0.16595368087291718, 0.6242990493774414, -1.5990837812423706, 0.2659958600997925, 0.2806364893913269, -1.3579145669937134, 1.1911308765411377, 0.5026909708976746, 0.6462868452072144, -1.0563071966171265, 0.8269928693771362, 0.7666919231414795, 0.7992680668830872, -0.6586353778839111, -0.481721431016922, -2.4415173530578613, -0.9918084740638733, -0.5631163120269775, -1.7749805450439453, -0.2928251624107361, -0.20384490489959717, -0.5418012142181396, 0.20249618589878082, -0.7015011310577393, -1.7936031818389893, 1.5299988985061646, -0.34979912638664246, -0.5425182580947876, 0.5815361142158508, -0.07756385952234268, 1.4142781496047974, 0.2520218789577484, 1.0170526504516602, 2.2093451023101807, 0.4121372401714325, 0.5981785655021667, 0.16792958974838257, 0.377790242433548, -1.967346429824829, -0.7148484587669373, -0.3938859701156616, 0.1861635446548462, -1.6299197673797607, -0.2603856027126312, 0.455302357673645, -1.1502238512039185, 0.47489142417907715, -0.17484143376350403, 0.8068532347679138, 0.24055470526218414, -1.2147626876831055, 0.9998303651809692, 0.16421936452388763, -0.9707558751106262, 0.6422227621078491, -1.34787917137146, -0.317816823720932]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [[0.15947221219539642, 1.889844536781311, 0.15343689918518066, 0.9754939079284668, -0.052548736333847046, -2.7687363624572754, -0.8584245443344116, 1.9039645195007324, 0.0734248235821724, -0.19935745000839233, 0.007642148062586784, -0.28132495284080505, -0.5451547503471375, 0.0057677398435771465, 0.16879874467849731, 0.22620555758476257, 0.39187946915626526, 1.4896656274795532, -0.2627566158771515, 0.26514729857444763, -0.49489626288414, -0.10354732722043991, 0.9405831694602966, -1.4391120672225952, -0.8432093858718872, 0.4430842995643616, 0.4815238416194916, -1.4399768114089966, -0.338706910610199, -0.6922982931137085, 0.09242967516183853, 1.4805859327316284, -0.1890280693769455, 0.03967474400997162, 0.6749030351638794, -0.4966529607772827, 0.8566654920578003, -0.012587807141244411, -0.9629196524620056, 0.46714189648628235, -0.7086058855056763, 0.4563749134540558, 0.24566815793514252, 1.5150281190872192, -0.6838627457618713, -0.5105994343757629, 0.2933509051799774, -0.21822993457317352, -0.05592788755893707, 0.18816916644573212, -0.024977926164865494, 1.0628232955932617, -1.2162730693817139, -0.29800793528556824, -0.07411788403987885, -0.5555642247200012, 0.21094922721385956, 0.6218143701553345, 0.2584427297115326, 0.2702089846134186, -1.0008165836334229, -0.9405203461647034, -1.5062540769577026, 0.019017616286873817, -0.024294812232255936, -0.08874186873435974, -0.13398003578186035, 1.0666166543960571, -0.12747079133987427, -1.4023815393447876, -1.2984899282455444, -1.3241432905197144, 0.423546701669693, 3.057356119155884, 3.17413330078125, -0.4709247946739197, 3.4330177307128906, 0.26328790187835693, -0.03795168548822403, 0.38085806369781494, -0.5443606376647949, -0.8228390216827393, 0.7680907845497131, -0.5648103952407837, -0.17690475285053253, -0.23734396696090698, -0.17279978096485138, 0.14758461713790894, -0.23903627693653107, -0.03278857097029686, -1.2984352111816406, 0.15694110095500946, 0.5320568084716797, -1.202128291130066, -0.22290143370628357, 0.9189335107803345, -1.4385024309158325, 0.12624001502990723, 0.9687946438789368, -1.2611303329467773, 1.616835117340088, 4.912562847137451, -2.415034055709839, -0.6295515894889832, 1.0265319347381592, 2.1230838298797607, 0.9428718090057373, -0.05024325102567673, -0.9233269095420837, -0.29170677065849304, 1.8786594867706299, 0.05461861565709114, -0.34847965836524963, 0.12490186095237732, -0.524733304977417, -0.6970154643058777, -0.06334403157234192, -1.5954108238220215, -0.014920766465365887, -0.002731108572334051, 0.7106986045837402, -0.02495262399315834, -2.529479503631592, -0.11771491914987564, -0.9990829229354858, -0.08987981081008911, -0.36911147832870483, -0.7186899781227112, -1.834639549255371], [-0.008618353866040707, 0.6776396632194519, 0.01749001443386078, 0.48511698842048645, -0.09609578549861908, 3.7334542274475098, -0.8530460000038147, -2.1982550621032715, 0.34249061346054077, 0.05086105316877365, 0.32395878434181213, 0.2312794029712677, 0.5417543053627014, 0.17656628787517548, 0.1631590574979782, 0.44122403860092163, -1.54556405544281, 0.3902973532676697, 0.24165335297584534, 0.16711214184761047, 0.2818310558795929, 0.5828897356987, 1.4303710460662842, -0.5455379486083984, 0.6286565065383911, -0.3986716568470001, 1.7485454082489014, -0.24612073600292206, 0.5905454158782959, 0.38692474365234375, 0.2663303017616272, -1.8055068254470825, 0.8052314519882202, -0.33131805062294006, 2.2637593746185303, 1.4065930843353271, -1.1433358192443848, 0.4562053382396698, 0.4795612394809723, -0.6942486763000488, 0.19539812207221985, -0.31480875611305237, -2.149564743041992, -0.20228230953216553, -1.0238699913024902, 0.5741327404975891, -2.766374349594116, 0.8438626527786255, 0.159335657954216, -0.6451380252838135, -1.2629613876342773, -0.48732393980026245, -0.765707790851593, -0.8341754674911499, -0.1276075541973114, 0.7569738030433655, -0.2587614953517914, 0.07431642711162567, 0.4210520088672638, 0.22884723544120789, 0.9586347341537476, 1.0130263566970825, -0.1456349939107895, -0.39199399948120117, 0.0061106919310987, 0.0021761043462902308, 0.12958793342113495, -0.2453441619873047, -2.071321964263916, -0.0026347024831920862, 1.9249767065048218, -0.6538416147232056, -0.21933533251285553, -1.7078471183776855, 2.6711127758026123, -0.058155547827482224, -3.327275276184082, 0.012691542506217957, 0.053045570850372314, 0.13087576627731323, -1.8929303884506226, -0.4573432207107544, 1.9256010055541992, -0.4078300893306732, 0.32878562808036804, -0.48403242230415344, 0.04308773949742317, 0.17934730648994446, 0.22068370878696442, 0.7440199255943298, -1.2197532653808594, -4.798899173736572, 1.3529863357543945, -2.331552028656006, 0.21638458967208862, 0.4559518098831177, -1.8541418313980103, 0.45206040143966675, 0.4727359712123871, 0.14983637630939484, 0.7221506237983704, 0.9578172564506531, -1.6648120880126953, -1.85331392288208, 1.7848702669143677, 1.4697571992874146, 1.1277624368667603, 0.2774939239025116, 0.27046701312065125, 0.35278111696243286, -0.114814892411232, 0.355387806892395, -0.24246159195899963, -0.6960099339485168, -0.6298885941505432, 3.082287311553955, 0.06670774519443512, 0.026882365345954895, -1.234002709388733, 0.01666940748691559, -0.06559686362743378, 0.28987693786621094, 0.24648655951023102, 0.3594208359718323, 0.29512977600097656, 0.29599809646606445, 0.3042658865451813, 0.22366680204868317, 0.9406434893608093], [0.04296620562672615, -0.44668635725975037, 0.4525517523288727, 0.49292775988578796, -0.4903327524662018, 2.6524415016174316, -0.8255150318145752, 1.103805422782898, -0.04237020015716553, -0.16505616903305054, 0.16228057444095612, -0.5422919392585754, -1.7470248937606812, -0.24358327686786652, 0.3690909147262573, -0.027591485530138016, 0.8867377042770386, -0.37546536326408386, 0.181566059589386, 0.11886923015117645, -0.23830176889896393, -0.27076178789138794, -0.25211936235427856, 1.7649444341659546, -0.2589007019996643, 0.8597376346588135, 2.909510612487793, 1.1930582523345947, -1.0436749458312988, 1.1362498998641968, -0.21431507170200348, -0.6975972056388855, -0.6976453065872192, 0.024354219436645508, 0.6235543489456177, -1.6723682880401611, -0.9968194365501404, -0.10515184700489044, 0.7371261715888977, 0.7312155961990356, -0.16980277001857758, 1.6101497411727905, 0.6210619807243347, -1.3887001276016235, -1.6891528367996216, 0.2839658260345459, -0.06217138096690178, -0.07904396951198578, -0.10153828561306, -0.4306465685367584, 0.90273517370224, 0.050592921674251556, -0.9894497990608215, 0.17553530633449554, -0.03697239235043526, -0.050557974725961685, -0.8118873834609985, -0.5053448677062988, 0.6943545341491699, -0.37930038571357727, -0.5710040330886841, 0.3377669155597687, 1.0233067274093628, 0.0029967189766466618, -0.2827867865562439, -0.108891062438488, -0.030432380735874176, -0.27919483184814453, 1.5408300161361694, -0.13838450610637665, -2.421632766723633, 0.355331689119339, -0.9922875761985779, 1.6747806072235107, -2.6922693252563477, 0.014481009915471077, 0.1445263773202896, 1.2574530839920044, -0.23289434611797333, -0.2803472876548767, -2.0351483821868896, -0.16125522553920746, 1.272050380706787, -0.518855631351471, -0.23050133883953094, 0.8637662529945374, -0.36000189185142517, -0.19518589973449707, -0.17445194721221924, -0.8984726071357727, 0.35869237780570984, 0.8034912347793579, 0.08186592906713486, -6.867354869842529, -0.004631567280739546, -0.40990445017814636, 1.4527491331100464, 0.2825438678264618, 0.37102389335632324, -0.763569712638855, -0.032179150730371475, 2.22815203666687, -0.6721426248550415, -0.44497132301330566, 2.1078779697418213, -1.1497477293014526, 1.4486489295959473, -0.2714425027370453, 0.06162857636809349, 0.15454977750778198, -2.774723768234253, -2.2298500537872314, -0.2193303108215332, -1.3310275077819824, 2.8355324268341064, 0.9002538919448853, 0.06246219947934151, 1.1745326519012451, -0.6996474862098694, 0.08079922944307327, 4.660684585571289, -0.07918132096529007, -0.2268974334001541, -0.08982472866773605, 0.5156288743019104, 0.0886724665760994, 0.6259284019470215, -0.23749804496765137, -0.4083276689052582], [-0.22873839735984802, -0.24773649871349335, 0.18125496804714203, -0.022209443151950836, 0.16452553868293762, 1.1379016637802124, -1.8906859159469604, -1.694176197052002, -0.3564799129962921, -0.2942759692668915, -0.007654168643057346, -0.10499580204486847, 0.1435699313879013, 0.0333390086889267, 0.09593991190195084, -0.03821197524666786, 1.1971468925476074, -1.6753194332122803, 0.998809278011322, -0.09142865240573883, -0.3878785967826843, -0.3332165777683258, 0.02028796635568142, -2.8324313163757324, -1.845287561416626, -0.9168733954429626, 1.2544891834259033, 0.04406857118010521, 0.6367319226264954, -1.82081937789917, -0.02488159015774727, 0.5243558287620544, 0.41426846385002136, -0.31486886739730835, 0.19078031182289124, -1.9076886177062988, 0.3579232096672058, -0.09869126230478287, -0.01108937244862318, 1.039099931716919, 1.04771888256073, 0.2679726481437683, 1.0838927030563354, 1.0680944919586182, 1.1987098455429077, -0.014505244791507721, -1.8808449506759644, 1.0941816568374634, -0.03222102299332619, 0.32976749539375305, 1.3098028898239136, -0.1689547300338745, 0.3810957074165344, 0.8150824308395386, 0.2540370225906372, -0.7470217943191528, 1.5702804327011108, 0.22865362465381622, -0.4689922034740448, -0.15764650702476501, 0.29421573877334595, -0.18925543129444122, 0.883253276348114, -0.10508011281490326, -0.19859881699085236, -0.23492272198200226, -0.13426804542541504, -0.003412124700844288, 1.6638959646224976, 2.8950822353363037, -0.8310433030128479, 0.6946356892585754, 0.28171655535697937, -1.4620941877365112, -0.11043917387723923, 0.22030647099018097, 2.130901575088501, -0.7639621496200562, -0.13649123907089233, -0.3114595115184784, 1.1292955875396729, -0.6290237307548523, -4.096566677093506, 0.6770710945129395, 0.3798771798610687, 0.7485734224319458, -1.5826913118362427, -0.0540485605597496, -0.09652107954025269, -0.7847778797149658, 4.581896781921387, -0.8278233408927917, 0.44603967666625977, 0.5783035159111023, 0.016181580722332, -0.1616934984922409, 0.21568915247917175, 0.09265021234750748, -0.19135737419128418, 0.5727180242538452, 1.0700427293777466, -1.872753620147705, 1.2665448188781738, 1.219081997871399, -0.8919569849967957, 1.3031871318817139, -2.217473030090332, -0.07670677453279495, -0.027886075899004936, -0.1798204779624939, -0.10947829484939575, -1.8351292610168457, -0.3189385235309601, 0.7765552997589111, -0.3668399751186371, 2.5812666416168213, 0.17742180824279785, -0.4285479485988617, -0.4846227765083313, -0.1107182502746582, -0.36258623003959656, -0.06886662542819977, -0.9551370143890381, -0.5938223004341125, -1.1698318719863892, -0.03124385140836239, 0.3743661046028137, 0.09468188136816025, -0.8255195021629333], [-0.13912749290466309, -0.16403470933437347, -1.3426463603973389, 0.22387410700321198, 1.016129493713379, -2.3433687686920166, 0.7008729577064514, -0.636064887046814, 0.16567575931549072, 0.2710723280906677, 0.26551762223243713, 0.09736157953739166, 0.10506957024335861, 0.12325442582368851, -1.6778082847595215, 0.07762179523706436, -0.34695160388946533, -1.0737007856369019, -0.4752344489097595, 0.02574964612722397, 0.23071138560771942, -0.3102712035179138, -0.25880372524261475, 0.45720675587654114, 0.20426484942436218, 1.4320123195648193, 0.20266954600811005, -0.8412466049194336, -0.30178534984588623, 1.0898009538650513, 0.15651166439056396, 5.273693084716797, -0.2653637230396271, -1.4223380088806152, -0.6253915429115295, 1.789778470993042, 0.055551107972860336, 0.5676490068435669, -1.5409109592437744, 0.43224966526031494, -0.22082717716693878, 0.7757394313812256, -0.7588618397712708, -0.759682297706604, -1.0040253400802612, 0.4824976325035095, 0.7453306913375854, -0.36179137229919434, -0.18181996047496796, -0.07150407135486603, 2.077227830886841, -0.4094703495502472, -0.2940540313720703, -0.08771239221096039, 0.3172794580459595, 0.3923284113407135, 0.17187075316905975, 0.04193458706140518, 0.47895970940589905, 0.21268853545188904, 0.31168046593666077, -0.7656386494636536, -0.9431089162826538, 0.10206707566976547, -0.10427217185497284, -0.202285036444664, 0.020672695711255074, -0.21684657037258148, -0.11121570318937302, 0.4730519950389862, 0.7061119079589844, -0.9730653166770935, -0.4819531738758087, 0.1873617172241211, -2.5009512901306152, -1.1924290657043457, 2.1745924949645996, -0.5225577354431152, 0.351712167263031, 0.08399607986211777, 0.5181745290756226, 1.460675597190857, 2.5136585235595703, 0.9454991221427917, 0.010342730209231377, -0.46139901876449585, -0.9162981510162354, -0.16750121116638184, 0.2385498732328415, -0.2855405807495117, -0.6896038055419922, -0.1596854329109192, 0.6576829552650452, 1.496767520904541, 0.16418969631195068, 0.2863871455192566, -0.3383175730705261, 0.21853353083133698, 0.1077958270907402, 2.4388225078582764, -1.8850133419036865, -1.1167304515838623, -2.582261323928833, -0.06010955944657326, 1.9723478555679321, -0.7549318671226501, -0.07026904076337814, 0.115033820271492, 0.29409119486808777, -0.41737574338912964, 0.8101487755775452, -1.9127801656723022, -0.33865219354629517, -0.9329122304916382, 0.5449150204658508, -0.7074363827705383, 0.27785882353782654, 0.11852267384529114, 0.2732188105583191, -0.20469336211681366, 0.6302710771560669, 0.13184066116809845, -0.043994251638650894, 0.022329235449433327, -1.2380125522613525, 0.39064353704452515, -0.027947982773184776, -1.559954047203064, 1.090837836265564], [0.1293957531452179, 0.318526029586792, 1.4039251804351807, -0.6127368807792664, 0.13235895335674286, -2.481921911239624, 0.9032059907913208, -0.16251736879348755, -0.338591605424881, 0.1481744796037674, 0.29688701033592224, 0.31135138869285583, -0.20037823915481567, 0.09858845919370651, 0.05288435146212578, 0.1509602665901184, -2.340228319168091, -1.348713755607605, 2.9579787254333496, 0.12323607504367828, -0.5586014986038208, 0.25375333428382874, -0.08752577751874924, -1.1814087629318237, -2.0505523681640625, 1.1561775207519531, 6.039456844329834, 0.864766001701355, 0.3262593150138855, 1.884338617324829, 0.3666771352291107, -3.085177421569824, 0.1481245756149292, 1.3518887758255005, 0.1795322597026825, 0.7282934188842773, 0.3311057388782501, 0.29929304122924805, -1.126429796218872, -1.158930778503418, 0.7169647216796875, -0.08127059042453766, -0.9517427682876587, 0.29543718695640564, -1.0098588466644287, -0.7443668246269226, 1.5477334260940552, -1.2031830549240112, 0.06183865666389465, -0.20290598273277283, 3.273808002471924, 0.33684539794921875, 0.2836972177028656, -0.5973985195159912, 0.1806574910879135, 0.11961163580417633, -0.3603953421115875, 0.733677327632904, -0.8670970797538757, 0.23479630053043365, 1.42026686668396, 0.057113517075777054, -0.2871153950691223, 0.2891663908958435, 0.018438925966620445, -0.041794199496507645, 0.2454850971698761, 0.331197589635849, 0.758917510509491, 1.26333749294281, -0.24633285403251648, -0.26006296277046204, 0.41366615891456604, -2.8136720657348633, -1.4256097078323364, 0.12618228793144226, 1.2379711866378784, -0.9598504304885864, 0.07869821041822433, 0.13782191276550293, -0.09023012220859528, -0.33950275182724, -0.07758545130491257, -0.40278443694114685, 0.20977294445037842, 0.5023736953735352, 0.6864258646965027, 0.14830443263053894, 0.35908743739128113, 0.9837181568145752, -4.80091667175293, 0.8130258917808533, 0.20281143486499786, 0.9264926910400391, -0.05721636861562729, 1.1351876258850098, -0.651007354259491, 0.8882138133049011, 0.6558212041854858, -1.710033655166626, 0.07266895473003387, -4.009269714355469, -1.0883854627609253, 1.2555088996887207, -4.3129143714904785, 0.038931526243686676, 3.490520477294922, 0.13706625998020172, 0.027406198903918266, 0.18565990030765533, 1.54960036277771, 0.23874177038669586, -0.6452219486236572, -1.0020930767059326, -0.6376821994781494, -0.44126221537590027, 0.2582727372646332, 0.44392669200897217, 0.617378294467926, -0.037537362426519394, -1.1690006256103516, 0.09928072243928909, -2.3235726356506348, -0.0007681415299884975, -0.16973090171813965, -0.12003950029611588, 0.09073039144277573, 0.5435745120048523, 1.1921535730361938], [0.004454502370208502, 0.8098105788230896, -0.003937957808375359, 0.13920217752456665, -0.5128140449523926, 6.620428562164307, 0.45025834441185, -0.15315134823322296, 0.03390559181571007, -0.3787349462509155, -0.06180700659751892, 0.005568794906139374, -0.7119695544242859, -0.10216987878084183, 0.18275469541549683, -0.020137755200266838, 0.33468037843704224, 0.7648624777793884, -0.12340980768203735, -0.026306157931685448, 0.2459985464811325, -0.2537326216697693, 0.7177299857139587, -0.6287863850593567, 0.6797815561294556, -1.3121377229690552, -0.8324975371360779, 0.09060125797986984, 0.8155724406242371, 0.8461900949478149, -0.24779126048088074, 0.7775180339813232, -0.6185840368270874, -0.8925684690475464, -0.3830580711364746, -0.7818551659584045, 1.3602430820465088, 0.31183525919914246, 0.6709704399108887, 1.4061105251312256, 0.4012353718280792, 0.5627204775810242, -0.8771569728851318, -0.01198290940374136, -0.16782259941101074, -0.3333987593650818, -2.1266348361968994, 2.7007551193237305, 0.12413614243268967, -0.2978975474834442, -0.056304387748241425, 2.338599443435669, -0.8718142509460449, -0.48872891068458557, -0.07972459495067596, -0.4322110414505005, -0.717385470867157, 0.15021823346614838, 0.6039021015167236, -0.43985873460769653, -0.25431233644485474, -0.6313114762306213, -0.8251878023147583, -0.2177889198064804, 0.11186322569847107, -0.4478154182434082, -0.11642066389322281, -0.28047075867652893, -0.398987740278244, -3.232738494873047, -0.6997461318969727, 0.6716774702072144, 0.9749197959899902, -0.30742380023002625, 0.3699660003185272, -1.1538835763931274, 2.3043599128723145, -0.5627186894416809, -0.22412912547588348, -0.10753091424703598, -0.03678649291396141, -1.9608540534973145, -4.410293102264404, -0.3539382517337799, -0.3961791396141052, -0.8932374715805054, 3.0453689098358154, -0.020085062831640244, -0.12551942467689514, -0.02103019878268242, -1.564834713935852, -1.022350788116455, 1.444510817527771, 1.0211044549942017, 0.10274726897478104, 0.16867709159851074, 3.085509777069092, 0.3453177511692047, 0.9232539534568787, -0.9918883442878723, 0.1638973355293274, 0.12370933592319489, -0.21977826952934265, -0.7289357781410217, -0.18110600113868713, -1.7331933975219727, 1.903605580329895, 0.11072573065757751, 0.4787817895412445, -0.2933962941169739, -0.4049844741821289, 0.5767812728881836, -0.3463045656681061, 0.2754649817943573, -1.161792278289795, -2.2864723205566406, 0.03397325426340103, -0.35060766339302063, 0.21176621317863464, -0.21725143492221832, 0.22246548533439636, 0.0025105993263423443, 0.9533989429473877, 0.6294520497322083, 0.1405305564403534, -0.026542136445641518, 0.11154083907604218, -0.3757152259349823, -0.12801089882850647], [-0.08997240662574768, -0.6344432830810547, 0.14596717059612274, 0.43201521039009094, -0.4041401147842407, 0.8864955306053162, -1.7981064319610596, 1.1721477508544922, -0.13890455663204193, -0.20618589222431183, 0.07921657711267471, -0.2682719826698303, -0.2470110058784485, -0.18336544930934906, -0.43751803040504456, -0.029959067702293396, -1.2198050022125244, -2.0797972679138184, 1.1350631713867188, -0.10366988182067871, -0.4394751787185669, 0.3568967282772064, -0.11402730643749237, -0.015403788536787033, -0.4009130597114563, -0.5551704168319702, -0.5211217999458313, -0.7437775731086731, -0.7194623351097107, -0.8085541725158691, -0.23019880056381226, 1.2514451742172241, -0.4119167923927307, 0.28727343678474426, -0.5634034872055054, 2.332017660140991, -0.5424047112464905, 0.552071213722229, 0.7659853100776672, 0.5429937243461609, 0.30590343475341797, -0.9281340837478638, -1.5853136777877808, 0.2593286633491516, 0.8029516935348511, 0.14733007550239563, -0.0007161466055549681, 0.942201554775238, -0.13661299645900726, -0.782764196395874, 1.1161097288131714, -0.7043721675872803, -1.9475575685501099, -0.0028624117840081453, -0.21222972869873047, -0.20769134163856506, 0.830946683883667, -0.37865209579467773, -0.6525679230690002, -0.09985090047121048, 0.8474992513656616, 2.1302692890167236, -0.4355904757976532, 0.05451732128858566, -0.31777819991111755, 0.13082876801490784, -0.07179471105337143, 0.6837531328201294, 1.6635786294937134, 0.7419694066047668, -0.3042001724243164, -0.5361503958702087, 1.8800411224365234, -0.43374714255332947, 1.193681001663208, 0.044320520013570786, -0.9830527305603027, 0.19822801649570465, -0.1403149962425232, -0.2552260458469391, -2.220057249069214, -1.3209013938903809, 2.466085195541382, 0.4471544027328491, 0.07880627363920212, 0.26485148072242737, -0.19692756235599518, 0.048307228833436966, 0.18904320895671844, -0.07356094568967819, -1.2723338603973389, -0.146605983376503, -1.2755560874938965, 0.964579164981842, -0.0627017393708229, -0.7256483435630798, 2.027350664138794, 0.27470988035202026, 0.02067999728024006, -1.1768535375595093, 0.12423486262559891, 0.04089593142271042, 2.6067628860473633, -0.7855910062789917, -0.8283272385597229, 0.8580319881439209, -1.0904765129089355, -0.20506909489631653, 0.20618903636932373, -0.2654736340045929, 0.28084585070610046, 0.6894347667694092, -0.027614252641797066, -0.5969607830047607, 4.117042064666748, 0.09239569306373596, -0.2822190225124359, -0.4867693781852722, 0.32513201236724854, 0.0023663584142923355, -0.2005682736635208, -0.14628367125988007, 0.4890410304069519, -0.15060532093048096, -0.7831336259841919, 0.32027074694633484, -0.4895396828651428, 0.2474939376115799, -1.2608129978179932], [0.22977186739444733, 1.4215384721755981, 1.2521932125091553, -0.6863533854484558, 0.28578633069992065, 0.17769314348697662, 1.208530306816101, 0.4084748923778534, 0.00010119308717548847, -0.04044374078512192, 0.21996887028217316, -0.20594137907028198, -0.7012642621994019, -0.20640884339809418, -0.6603373885154724, 0.1953471153974533, -0.16878338158130646, -0.934248149394989, 0.0886128768324852, 0.21958130598068237, -0.2799374461174011, 0.08312096446752548, -2.0025548934936523, 1.0190222263336182, 0.09839065372943878, 0.5733880996704102, -3.83589506149292, 0.28920406103134155, -0.27842217683792114, -1.022532343864441, -0.027118520811200142, -0.9548193216323853, -0.2377893328666687, 0.8782729506492615, -0.16575247049331665, 0.6751613616943359, 0.8474928736686707, -0.48227444291114807, 2.1514341831207275, -0.5579493045806885, -0.08508515357971191, -0.7357443571090698, 0.42903390526771545, -0.49416568875312805, 1.7653816938400269, -1.08897066116333, 0.8500477075576782, 0.6124341487884521, 0.15164822340011597, -0.7792156338691711, -0.851830244064331, -0.6885921955108643, 0.40956664085388184, -0.22319960594177246, 0.21614883840084076, 0.3671320080757141, -0.19334009289741516, 0.2905169129371643, -0.12312282621860504, 0.030027637258172035, 0.7584525346755981, -0.49631887674331665, -0.5297694802284241, -0.0008945257868617773, 0.24037952721118927, 0.21833252906799316, 0.1423051953315735, -0.2137545347213745, -2.079895496368408, -2.5439510345458984, 1.461898922920227, 1.2434935569763184, 0.763038158416748, 0.06260111927986145, -3.628065586090088, 0.062132541090250015, -1.676457405090332, -0.5926892161369324, -0.010174118913710117, -0.17430950701236725, 3.7177131175994873, -0.7005189061164856, -1.295684576034546, -0.4622870087623596, -0.12038043886423111, 0.22392474114894867, 0.25095483660697937, -0.3717636466026306, 0.018896574154496193, 2.823277473449707, -2.185934066772461, -1.8942886590957642, 0.3294321298599243, 3.3344831466674805, 0.05015307664871216, -0.15180517733097076, 1.616013765335083, 0.3938044011592865, 1.0113011598587036, -0.2861911356449127, 0.9341220855712891, 0.1765747219324112, -0.18367929756641388, 0.15494900941848755, -0.9314600229263306, -0.610285758972168, 1.4979925155639648, 0.020258333534002304, 0.4620249569416046, 0.04280686005949974, -0.6378899216651917, -0.5353323817253113, -0.20025232434272766, 1.5206547975540161, 1.8197214603424072, -1.0395550727844238, -0.07596772909164429, -0.502348780632019, 1.3039449453353882, 0.12750814855098724, 0.20603398978710175, 0.161797896027565, 0.10525500029325485, 0.17010396718978882, -1.2646445035934448, 0.2590239942073822, -0.05332016199827194, 0.13665863871574402, -1.1050827503204346], [0.07814154773950577, 1.4356764554977417, -1.8075933456420898, -0.054024018347263336, -0.18588992953300476, -3.266408920288086, 0.8608366847038269, 0.46512916684150696, 0.23450937867164612, 0.013577654957771301, -0.10017918795347214, -0.15591759979724884, -0.047664135694503784, 0.1764209270477295, -0.5733556747436523, 0.10763382911682129, -0.5876013040542603, -0.39845603704452515, 0.6005892753601074, -0.05204777047038078, -0.41886088252067566, 0.3049888014793396, 1.0002782344818115, 1.5862187147140503, -0.7844545841217041, 0.2300248146057129, 2.1945178508758545, -0.48418453335762024, -0.9891371726989746, -0.32884496450424194, -0.035117536783218384, -2.6284122467041016, 0.6993653178215027, -0.2650628387928009, -1.214138388633728, -2.3121416568756104, -0.6960380673408508, -0.06104659661650658, 0.34731927514076233, 0.6297978162765503, 0.3189796805381775, -1.7653251886367798, -0.7395151257514954, -0.916147768497467, 1.9369124174118042, -0.1163039430975914, -1.1944917440414429, 0.7753708958625793, -0.18867941200733185, 0.41564005613327026, -1.5522053241729736, -0.027851291000843048, -0.39904841780662537, 0.2731109857559204, -0.4411797821521759, -0.9427789449691772, -1.4750325679779053, -0.2601984739303589, 1.2234853506088257, 0.008161399513483047, -0.3972206115722656, 0.905205488204956, 0.15376771986484528, -0.2963249385356903, -0.2890288829803467, -0.034093182533979416, 0.1461683213710785, 1.3518363237380981, -1.4252327680587769, 0.02620033174753189, -4.910292625427246, 0.22860728204250336, -1.2421406507492065, 1.4878380298614502, 2.348639488220215, 0.521781325340271, 0.3930036127567291, -0.8592023253440857, 0.014209819957613945, -0.24000640213489532, -2.3557190895080566, -0.4962333142757416, 1.354056715965271, 0.646980881690979, -0.3157649636268616, 1.2228920459747314, 0.25071364641189575, 0.1424652338027954, -0.16668663918972015, -0.29855966567993164, 0.7089646458625793, 1.1027857065200806, 0.2513107359409332, 5.085702896118164, 0.026037419214844704, -0.6083941459655762, 1.6313914060592651, -0.06521965563297272, 0.17784607410430908, 0.5626423954963684, 0.3147268295288086, -1.6850295066833496, 0.08822810649871826, -0.2705979347229004, 0.751169741153717, 0.317553848028183, 2.0652806758880615, 0.05942068621516228, 0.18188606202602386, -0.25533372163772583, -3.24393630027771, -0.025934584438800812, 0.22534289956092834, 2.126488447189331, 0.9953998327255249, 0.03073258139193058, 0.22044941782951355, 0.08886048197746277, -0.591971755027771, 0.25632867217063904, 2.1592886447906494, 0.06025215610861778, 1.3954570293426514, -0.2742641866207123, 0.03858930617570877, -0.017514711245894432, -0.028163226321339607, -1.4025074243545532, 1.2275197505950928], [-0.08579356968402863, -1.5744425058364868, 0.49098798632621765, -0.4105713963508606, 0.4667351245880127, -0.6746665239334106, -0.28045183420181274, 1.798596978187561, 0.6177477240562439, 0.14092808961868286, -0.06883035600185394, -0.28669992089271545, 0.0004494538006838411, -0.019809652119874954, 0.46416109800338745, -0.04127630591392517, -0.09100133180618286, -1.0106719732284546, -0.5383350253105164, 0.2147829681634903, -0.28092440962791443, 0.029912598431110382, -1.4387500286102295, -0.4241770803928375, -0.82536780834198, 0.7064558863639832, -0.8989189863204956, 0.5721881985664368, -0.9288577437400818, -0.7172157168388367, 0.05961514636874199, 1.9423744678497314, -0.3072943091392517, -0.6335805654525757, -0.06906955689191818, -0.7880876660346985, 0.5160160064697266, 0.34973493218421936, 0.47633811831474304, -0.4201192259788513, -0.4059537351131439, -0.8125512003898621, 2.410736322402954, -0.6054702997207642, -2.2020089626312256, 1.1834293603897095, -0.32835885882377625, 1.2725571393966675, 0.2950231730937958, 1.8647547960281372, 1.8845040798187256, 3.032639980316162, 0.09681602567434311, 1.1244176626205444, 0.5773201584815979, -1.1261106729507446, 0.6330455541610718, -0.1530514508485794, -0.6256832480430603, 0.0534978024661541, 0.5108062624931335, 0.23902028799057007, 0.3307942748069763, 0.06583071500062943, 0.11613542586565018, 0.36410394310951233, 0.07647506892681122, -1.2322630882263184, 0.060119304805994034, -0.5272006392478943, -1.8007347583770752, -0.8100726008415222, 0.8217869997024536, -0.07303424179553986, -3.506948471069336, 0.21493187546730042, -3.5159835815429688, 0.1297004222869873, 0.24523110687732697, 0.17807190120220184, 0.11635264754295349, 0.5328477025032043, 1.4879169464111328, -0.9936332702636719, 0.2861471176147461, 0.139585480093956, -0.24374134838581085, 0.21212613582611084, -0.060891374945640564, -0.1020449846982956, -0.805752158164978, -3.616696357727051, 0.6169626712799072, 0.8998333811759949, -0.07548494637012482, -1.0552479028701782, 0.6496027112007141, 0.30144062638282776, -1.7368247509002686, 0.42497363686561584, 0.10624571889638901, 1.6532633304595947, -0.37531647086143494, 1.501050353050232, 1.7741808891296387, 0.2758409082889557, 0.2564779818058014, 0.004714549984782934, 0.5488001108169556, -0.47441238164901733, 2.4875571727752686, 1.5243207216262817, -0.10436760634183884, -0.7297371029853821, -1.2890311479568481, 3.584146499633789, -0.14957545697689056, -0.025357140228152275, -0.8760573267936707, -0.28629153966903687, -0.6166201829910278, -0.14503927528858185, -1.2249056100845337, -0.25155696272850037, -0.4736570715904236, -0.017039859667420387, 0.26855289936065674, -0.8357629179954529, 0.8236697912216187], [0.04203447327017784, -0.9473374485969543, 0.04072578251361847, 0.16196900606155396, 0.054667893797159195, 0.4346100389957428, 0.0065003372728824615, 2.413074016571045, -0.05851948261260986, -0.17796014249324799, 0.2189154475927353, 0.06556469202041626, -1.0370975732803345, 0.1197630763053894, -0.2853023111820221, 0.18077625334262848, -0.7940576076507568, -0.6844696402549744, 1.3308840990066528, 0.04319382086396217, -0.4063667058944702, 0.03122859261929989, 1.290147304534912, 0.1016317829489708, -0.8436959981918335, 0.21333496272563934, 0.5545600652694702, -0.9326993823051453, 0.09825419634580612, -1.3681974411010742, -0.029318563640117645, 0.664781391620636, -0.8909454345703125, 0.08079660683870316, -0.41668280959129333, -2.1394095420837402, -0.13082829117774963, -0.24047338962554932, -1.2747968435287476, 0.4429313540458679, 0.5308917164802551, 0.1715046912431717, 0.2333560287952423, -0.367336243391037, -0.6914774179458618, -0.7675162553787231, -0.9535868167877197, 0.25524061918258667, -0.006601423490792513, -0.4197256565093994, -1.3565735816955566, -0.34835126996040344, -0.1658630520105362, -1.177950382232666, -0.19254572689533234, 0.2149060070514679, 0.6698576807975769, -0.0072496915236115456, -0.7394188046455383, 0.056886762380599976, 0.5480753183364868, 0.13751500844955444, -0.34277379512786865, 0.0034778809640556574, 0.2021852284669876, 0.04956502094864845, 0.0023992692586034536, 1.813810110092163, 0.406597375869751, -1.6709054708480835, -2.9218404293060303, 0.12327904999256134, -0.1257355958223343, -1.1679389476776123, 0.05431494861841202, 0.25725385546684265, -1.0136289596557617, -0.8763483762741089, 0.05947345495223999, 0.02570025809109211, 2.570477247238159, -0.0035188114270567894, 1.3013255596160889, 1.0557053089141846, -0.0398382768034935, 0.34511590003967285, 1.0154485702514648, 0.04473936930298805, -0.1365775614976883, -1.0607882738113403, -1.2943942546844482, 0.7062274217605591, 0.29829856753349304, 0.7279033660888672, -0.010416645556688309, 0.22110068798065186, 0.0004213305364828557, 0.8446086049079895, 0.6367837190628052, 0.4688146114349365, -0.1671779453754425, 0.6584681272506714, -1.1344670057296753, 0.11695040762424469, 1.779651403427124, 0.35124894976615906, 0.06713797897100449, 0.0307400394231081, -0.056191135197877884, -0.05175640806555748, -0.9904873371124268, 0.07420454919338226, -0.15636976063251495, -0.07470812648534775, 0.8390108346939087, -1.3742420673370361, 0.14779074490070343, -0.5646198391914368, 0.10988613218069077, 0.5942472219467163, 2.646986246109009, 0.08761761337518692, 1.0978658199310303, 0.019625913351774216, 0.005669383332133293, 0.1590375155210495, 0.016406994313001633, 0.7061551213264465, 0.9347735047340393], [-0.08688725531101227, -2.095379590988159, -0.030550803989171982, 0.3281095027923584, 0.20299117267131805, 1.1050283908843994, 1.6177176237106323, 0.3662495017051697, -0.15198342502117157, -0.48954570293426514, -0.48184606432914734, -0.6223695278167725, -0.724560022354126, -0.008753754198551178, 0.559850811958313, 0.023268133401870728, 0.17021773755550385, 0.5847771763801575, 0.4287847578525543, 0.13857214152812958, 0.24513249099254608, 0.7637122869491577, 0.23713122308254242, -2.643944263458252, 0.03999806568026543, -0.2465171068906784, 2.9645066261291504, 0.32552483677864075, 0.37952613830566406, -1.7706319093704224, -0.028598710894584656, 0.5692287087440491, 0.6216862201690674, -0.8078535199165344, 0.6733216047286987, 2.1075472831726074, -0.13329508900642395, 0.04122311621904373, -0.5289461612701416, -0.10528285056352615, 0.3310408294200897, -0.06403592973947525, -1.0372569561004639, 0.34955474734306335, 2.1368770599365234, 0.5426343083381653, 0.31650879979133606, 0.6062116026878357, -0.31012821197509766, 0.12523669004440308, -1.2761510610580444, 0.5410317182540894, -0.010001808404922485, 0.5797098875045776, -0.2601887881755829, -0.2521750330924988, 0.352546364068985, -1.0115097761154175, -0.07036063075065613, -0.3924367129802704, 0.4262600243091583, 0.4001838266849518, 0.6798182129859924, -0.3751883804798126, -0.48763135075569153, 0.24371884763240814, -0.37008488178253174, 0.5770171880722046, 0.30918702483177185, -2.111245632171631, -0.6066715717315674, -0.8234779238700867, -0.14662422239780426, -1.8159370422363281, -0.4011804759502411, -0.16651861369609833, 4.006978988647461, 0.0019039568724110723, -0.10718560963869095, -0.3658269941806793, -2.1468496322631836, -1.6548410654067993, -2.365995168685913, -0.13762833178043365, -0.1423589140176773, 0.11497385799884796, -1.1370824575424194, 0.12981420755386353, 0.20291192829608917, 0.5601555705070496, -5.6045637130737305, 2.520235300064087, -1.1701215505599976, 1.6204344034194946, -0.43953970074653625, -0.03262491896748543, -0.27970966696739197, 0.2719675600528717, 1.7560667991638184, -0.21235403418540955, 1.7087808847427368, 2.8118736743927, 0.4507655203342438, 0.6257607936859131, 1.152144193649292, -0.30723321437835693, -1.7535287141799927, -0.23186640441417694, 1.472379207611084, -0.5473867654800415, 1.2563362121582031, -1.3366950750350952, -0.4145474135875702, -0.4657225012779236, -2.9750895500183105, 0.8359846472740173, 0.11715272814035416, -0.10858149826526642, -0.6955388188362122, -0.030762314796447754, 2.99674654006958, -0.047812316566705704, -0.22775815427303314, 0.025394534692168236, -1.0402344465255737, 0.008030025288462639, 0.2507356107234955, 1.285738468170166, -0.26971879601478577], [0.10843776166439056, -1.0679454803466797, -1.1204155683517456, 0.9783475399017334, 0.3331329822540283, 0.8537119626998901, 0.023582400754094124, 0.2658885419368744, 0.2846311628818512, 0.5149093866348267, 0.22544187307357788, 0.2914457321166992, 0.6994463801383972, -0.030758531764149666, -0.218117356300354, 0.41891175508499146, -1.1901260614395142, 1.94724440574646, 0.22276228666305542, 0.10407591611146927, 0.6348124146461487, -0.09838401526212692, 0.60572749376297, 0.6837568879127502, 0.29823729395866394, 0.6356704235076904, -2.481776714324951, 0.16513080894947052, 1.5198756456375122, 0.7918117046356201, 0.19573131203651428, -0.5353468656539917, 0.8316641449928284, -0.5908024907112122, -0.3363941013813019, -0.18322207033634186, 0.40431493520736694, 0.14791792631149292, -1.0946776866912842, -1.7501064538955688, -0.6018611788749695, 0.9732729196548462, -2.1459827423095703, 0.7659405469894409, 0.8600766658782959, 0.26087671518325806, 0.25405004620552063, -1.7463033199310303, 0.2865615785121918, 0.48420363664627075, -1.958367943763733, 1.6338777542114258, -0.37106335163116455, 0.15675878524780273, -0.467238187789917, -0.34123659133911133, -1.1524852514266968, 0.246445432305336, 1.072000503540039, 0.1233932375907898, -0.009688097983598709, -0.2507542073726654, -0.3911406993865967, 0.4158347249031067, 0.3997381329536438, 0.6243678331375122, 0.34886452555656433, 0.1873677521944046, -1.0545250177383423, 0.3696396052837372, 3.4796483516693115, 0.58050936460495, -1.3876457214355469, -0.16135849058628082, 0.6475512385368347, -0.620765209197998, -7.585653305053711, -0.8007022142410278, 0.20961801707744598, -0.03920016810297966, -0.11472556740045547, -1.0150419473648071, -0.6537288427352905, 0.5246743559837341, 0.4417057931423187, 1.2232080698013306, -0.6862820982933044, 0.2649630606174469, 0.17004407942295074, -0.9195285439491272, 0.24171145260334015, -0.9731053709983826, -0.5061274170875549, -4.2940592765808105, 0.0670790746808052, 0.6379126310348511, -0.05518092215061188, 0.196178138256073, -0.09129436314105988, 0.2814680337905884, 0.9705181121826172, 1.67044997215271, -2.1611366271972656, -0.41278186440467834, -0.7307643890380859, -0.7454580068588257, 0.35099977254867554, 0.11446079611778259, 0.618350088596344, -0.07436579465866089, 1.2989484071731567, 1.7872368097305298, 0.25791487097740173, 0.892876923084259, -0.2957836389541626, 0.3819742202758789, -0.1035805493593216, 1.3771531581878662, 0.23717300593852997, 0.27675020694732666, -1.0075714588165283, 0.015627987682819366, 1.5905989408493042, 0.4891856908798218, 0.826884388923645, 0.21768730878829956, 0.5300923585891724, -0.42935681343078613, 0.5328724980354309], [0.24138368666172028, 0.2433992624282837, 1.381476640701294, 0.6773750185966492, -0.7668805122375488, -2.4714627265930176, -0.9184502363204956, 1.2493627071380615, 0.5009160041809082, -0.09302433580160141, 0.15665343403816223, -0.12992383539676666, 0.5806137323379517, -0.025305094197392464, -0.08702504634857178, 0.12403404712677002, 0.0379616878926754, 0.3962550461292267, -0.11072838306427002, 0.303138792514801, 0.517108678817749, 0.28943389654159546, -0.5262616872787476, -1.8273918628692627, -0.11895711719989777, 0.5737653374671936, 0.4405839443206787, 0.5041339993476868, 0.6490337252616882, -0.014497444033622742, 0.2175247222185135, -1.4836161136627197, 0.6639597415924072, -2.1104846000671387, -0.04911145567893982, 0.061779480427503586, 0.32542669773101807, 0.07774735242128372, 3.841087579727173, 1.7191907167434692, 1.438366174697876, -0.5204026699066162, -0.1946900188922882, 1.0871180295944214, 1.0057190656661987, 0.30080845952033997, -0.7138323187828064, 0.5761994123458862, 0.27444231510162354, 0.6115539073944092, 0.11929581314325333, -0.35444289445877075, -0.2645273506641388, 1.430022120475769, 0.758598268032074, 0.39781254529953003, -0.42967361211776733, -0.018176492303609848, -0.723013162612915, -0.15574951469898224, 1.6801666021347046, -2.0290589332580566, -0.28299227356910706, 0.2078351527452469, 0.11973263323307037, 0.2332165688276291, -0.00610404321923852, -0.12447352707386017, -0.20157697796821594, -1.8149189949035645, -1.0476198196411133, -0.7081104516983032, -0.07656337320804596, -2.733236789703369, 1.2071517705917358, 0.6583582162857056, 1.5514376163482666, -0.42314931750297546, 0.16299673914909363, 0.3118559420108795, 3.8614773750305176, -1.324135422706604, -4.717519283294678, 0.25317054986953735, -0.14673300087451935, -0.4376847445964813, -0.1976695954799652, 0.11368923634290695, 0.28295496106147766, -0.5097745060920715, 3.193629741668701, 0.32929712533950806, -0.2336941659450531, -2.062548875808716, 0.4724412262439728, 0.10558007657527924, 0.5302644968032837, -0.1039823368191719, -0.4641955494880676, -0.7687908411026001, 0.018204577267169952, -0.3628435730934143, -1.4854706525802612, -0.3985038101673126, 2.2643253803253174, -0.768097996711731, -1.000977635383606, 0.14131519198417664, 0.4027784466743469, 0.05077304691076279, 0.3281762897968292, -0.19789887964725494, 0.15802565217018127, -0.27179649472236633, 1.3991566896438599, -2.185396432876587, 0.334608793258667, 0.4832933247089386, -0.8171361088752747, 0.5575670003890991, -1.6465559005737305, 0.18053753674030304, -1.2667732238769531, -0.018455542623996735, -0.3784898519515991, 0.27845972776412964, -0.02002807892858982, 0.790420413017273, -1.0955947637557983], [0.03466043248772621, 0.30667778849601746, 1.5795338153839111, 0.5796540975570679, -0.01707625389099121, 0.45059654116630554, 0.019357312470674515, 1.4658581018447876, 0.2601991593837738, 0.035756565630435944, -0.04974145069718361, 0.08895023912191391, -0.5992117524147034, 0.008380908519029617, -0.41950443387031555, 0.2260407954454422, 0.9448545575141907, 0.0655437707901001, 1.115131139755249, 0.07837133854627609, -0.21226786077022552, -1.2144811153411865, -0.6923733353614807, -0.5459033846855164, 0.2538802921772003, 0.2362813502550125, -2.169581890106201, 2.278596878051758, -0.04916200786828995, 0.8746357560157776, -0.09981232136487961, 1.1279674768447876, -0.4517171084880829, 0.44159436225891113, 0.8232119083404541, 1.4085971117019653, 0.7602837681770325, -0.5790560245513916, 1.7466295957565308, -0.2923171818256378, -0.27534961700439453, 0.8808444142341614, -0.00012313933984842151, -1.3765894174575806, 0.8881513476371765, 0.06976199150085449, 0.18525046110153198, -0.5728320479393005, 0.006629496812820435, -0.7985206246376038, 0.10014688223600388, 1.3820695877075195, 0.5114982724189758, 1.9477776288986206, 0.5301247835159302, 0.08480040729045868, -0.1030358225107193, -0.19812433421611786, 1.774639368057251, -0.1757444590330124, -0.2621755599975586, -1.1155997514724731, 0.41897425055503845, 0.30063480138778687, 0.19353611767292023, 0.0965774804353714, 0.10381822288036346, 0.14461468160152435, -1.004738688468933, -1.4484174251556396, -1.1980879306793213, 0.4664057195186615, -0.5923059582710266, -3.043623924255371, 1.120914101600647, -0.43445926904678345, 5.116391658782959, 0.42443493008613586, -0.16730865836143494, -0.04708018898963928, 0.6790167689323425, 0.6072338223457336, -2.1317102909088135, -0.5700578689575195, -0.3728885352611542, -1.2632439136505127, -0.08571464568376541, -0.08566658943891525, 0.2362525910139084, 0.0802713930606842, -5.668007850646973, 0.3620673418045044, 0.41125744581222534, 3.643409252166748, 0.00047929794527590275, 0.27203768491744995, -3.4893789291381836, 0.12481197714805603, -0.1980511099100113, 1.212470293045044, 0.35490110516548157, -0.83732008934021, -1.9643126726150513, 0.9588348269462585, 2.606350898742676, -0.10893246531486511, 0.12190642952919006, -0.052196845412254333, 0.15118935704231262, -0.08744806796312332, -2.0003061294555664, -1.4959276914596558, -0.2897840142250061, -0.6729685664176941, -1.4417308568954468, -2.468827486038208, -0.5465414524078369, -0.42432934045791626, -0.07376278936862946, -0.07515206933021545, -0.09501788020133972, 0.021771633997559547, 0.8827400207519531, -0.06576112657785416, 0.6333491206169128, -0.21172073483467102, 0.1721625030040741, 0.4276900589466095, -0.9363628029823303], [0.18637098371982574, -0.7439783215522766, 0.3078587055206299, 0.08518824726343155, -0.011553606949746609, -3.6886377334594727, 1.0741734504699707, -0.6834902763366699, 0.23012293875217438, 0.002303984249010682, 0.1851285994052887, -0.42934277653694153, -0.428532212972641, 0.19791774451732635, -0.31282830238342285, 0.13052625954151154, 2.271254062652588, 1.0296525955200195, 1.1536887884140015, 0.148935467004776, 0.41592973470687866, 0.6403462886810303, 0.04597794637084007, -2.114185333251953, 0.7044861912727356, -1.2719826698303223, 0.12145296484231949, -0.5150177478790283, -0.00014167733024805784, 0.449355810880661, 0.176269993185997, 1.0133891105651855, -0.7451903223991394, -0.618621289730072, -0.9426285624504089, 0.9914278388023376, -0.8361693024635315, -0.16073453426361084, 1.0921908617019653, -0.4752536714076996, 1.0027774572372437, 0.1930258572101593, -0.504442036151886, -0.7071321606636047, 0.004096216522157192, 0.4856933057308197, 0.518229067325592, 0.08036017417907715, 0.1042177751660347, 1.4264332056045532, -0.018332593142986298, -2.032107353210449, -1.3079463243484497, -0.5050657391548157, 0.4352708160877228, 0.06784377247095108, -0.05527369678020477, 0.5802475810050964, -0.16366392374038696, 0.031847886741161346, 1.0199041366577148, -0.8902077674865723, 0.5822535157203674, -0.3404575288295746, 0.23131650686264038, 0.12505148351192474, 0.04825947433710098, 0.564242959022522, -0.04067254438996315, 0.9787737727165222, -1.4546968936920166, -0.7088243961334229, 0.23682653903961182, -0.6607732176780701, 0.543113112449646, 0.17440180480480194, -4.255318641662598, 0.9424000382423401, 0.0768042504787445, 0.24797354638576508, 2.4006307125091553, 1.8195006847381592, -0.15876999497413635, -0.5453620553016663, 0.18215706944465637, -0.6254960298538208, 1.3030869960784912, 0.01442556269466877, -0.002353188581764698, 0.7953528761863708, -1.6003376245498657, -0.3865054249763489, 0.5772998332977295, 1.8314114809036255, 0.22311341762542725, -0.1269983947277069, -4.676694869995117, -0.3829534947872162, 0.13812829554080963, 1.6307823657989502, -0.284658819437027, 0.6182124018669128, -0.7586498260498047, 1.57268488407135, -2.6647610664367676, 0.4385942220687866, -0.9670158624649048, 0.11744195222854614, 0.7103798985481262, 0.02543802000582218, -2.9239330291748047, 1.1625306606292725, 0.28440845012664795, -0.45885077118873596, 2.136099100112915, -1.9286373853683472, 0.01759783737361431, 1.4756494760513306, 1.1036230325698853, 0.21485839784145355, 0.5052459239959717, 0.15796178579330444, 2.385174036026001, 0.12521183490753174, -0.38160303235054016, 0.33850017189979553, 0.4058564603328705, -1.0780549049377441, -0.49183136224746704], [-0.0920407623052597, -0.23932170867919922, 1.4238284826278687, -0.41083958745002747, 0.7921327948570251, 4.459409713745117, -0.5922157168388367, 1.6923705339431763, 0.19680628180503845, 0.003955497406423092, -0.14674517512321472, 0.09401477873325348, 0.31789711117744446, -0.2544578015804291, -1.549392819404602, 0.09544197469949722, 0.7911199927330017, 1.0720516443252563, 2.675668716430664, -0.027121741324663162, 0.29775989055633545, 0.37302684783935547, -1.2988920211791992, 0.9728126525878906, -0.1601901799440384, 0.23110835254192352, 1.5610425472259521, 1.1948894262313843, 0.8480820655822754, -0.2706191837787628, -0.17935417592525482, 2.544356107711792, -0.3515460789203644, 1.030234456062317, 0.08999328315258026, -1.535993218421936, 1.2509113550186157, 0.07315557450056076, -2.0139877796173096, 1.897477626800537, 0.8701902627944946, 0.9333798885345459, 2.305772542953491, -1.6136016845703125, 0.4720247685909271, -0.40518951416015625, 0.012378163635730743, -0.03155149891972542, -0.02727384679019451, -0.8410050868988037, -0.41526710987091064, -0.028532229363918304, 0.3464430570602417, -0.04717158526182175, -0.22249546647071838, 0.5909460783004761, 0.8354173302650452, 0.16671571135520935, -0.8130502700805664, -0.12523894011974335, -1.425214409828186, 0.9498866200447083, -0.9453975558280945, -0.01884581334888935, -0.5444206595420837, -0.11265961080789566, 0.006870333105325699, 0.1636253148317337, -1.2608925104141235, -1.2633641958236694, -1.9462512731552124, 0.38736072182655334, -1.4030503034591675, 1.27741539478302, -0.5413015484809875, 0.36808496713638306, -4.735119342803955, 0.2729479670524597, -0.12376334518194199, -0.284664124250412, 2.1976213455200195, -0.047051917761564255, 1.9639899730682373, 0.7746275663375854, 0.14614754915237427, -0.8543384075164795, 0.7696539163589478, 0.06174839287996292, 0.01836961880326271, -0.043879009783267975, 0.967308521270752, -2.868500232696533, 0.9739330410957336, 2.1998300552368164, -0.03623722121119499, -0.8214970231056213, -4.526726245880127, -0.15707731246948242, 0.2762846350669861, -1.5267523527145386, -0.8499009609222412, -0.7743187546730042, 2.677678346633911, 0.36997827887535095, -1.3735250234603882, -1.1273112297058105, -1.2867501974105835, -0.14725571870803833, 0.23034994304180145, -0.23995962738990784, -3.676398754119873, 0.20464816689491272, 0.3132292628288269, 0.2659103572368622, -0.16034476459026337, -0.09393148869276047, -0.15414203703403473, 0.1821654587984085, -0.1732434630393982, -0.2823280990123749, 0.4981914460659027, -0.07795928418636322, -2.272217035293579, 0.4804416298866272, -0.22118842601776123, 0.0780244916677475, 0.2509830892086029, -0.841015636920929, 1.0917680263519287], [-0.05466700345277786, 0.13517434895038605, -1.6951180696487427, -0.061035145074129105, 0.07384780794382095, -4.042717933654785, 0.051056332886219025, -0.4058111011981964, 0.11084897816181183, 0.38631415367126465, 0.09664953500032425, -0.47321614623069763, 0.739521861076355, 0.02307840622961521, -0.889617383480072, 0.2531372010707855, -4.402045726776123, 1.1528739929199219, 1.117860198020935, -0.039792969822883606, -0.04008593037724495, -0.5104232430458069, -0.13596577942371368, 0.9059142470359802, 0.8103482723236084, -0.30794617533683777, -0.33986303210258484, 1.4754632711410522, 0.7721810936927795, -0.6544856429100037, -0.17299693822860718, -2.8188068866729736, 0.9341102838516235, -0.9576572775840759, -0.14863035082817078, 3.1383204460144043, 0.45293664932250977, 0.6319065690040588, -2.8948092460632324, 0.8792611956596375, 0.5625356435775757, -0.16517867147922516, -0.5327684283256531, 0.007903057150542736, 0.1323486715555191, -0.9952572584152222, 0.30405643582344055, -1.7049542665481567, -0.05870215594768524, 0.6970431804656982, -2.075437545776367, -0.6873190999031067, 0.09402850270271301, -0.9687768816947937, -0.6303636431694031, 0.04606104642152786, 1.5533357858657837, 0.175751194357872, 0.39560002088546753, 0.017997968941926956, -0.46402889490127563, -0.43313124775886536, -0.24010108411312103, 0.27726781368255615, -0.2113875299692154, -0.2609153389930725, -0.09287269413471222, -0.2280377894639969, -0.2592349648475647, 1.7451846599578857, -0.0866035670042038, -0.18340936303138733, 1.373913288116455, 2.568819999694824, -2.2222867012023926, -0.7806162238121033, 0.8757291436195374, -0.31153666973114014, -0.08796986192464828, 0.16074243187904358, 2.334834575653076, -0.4565927982330322, 1.2506988048553467, -0.111555315554142, -0.07408713549375534, 0.19767381250858307, -0.2820972502231598, 0.00854825135320425, 0.1788754016160965, 2.2562079429626465, 0.8476386070251465, 1.0126080513000488, 1.57302987575531, 1.3273829221725464, 0.19968603551387787, 0.6833820343017578, -0.3028690814971924, -0.03332783654332161, 0.8324871063232422, -2.2594404220581055, 0.22877074778079987, 3.055262565612793, -2.096088171005249, 0.4028308093547821, -2.205408811569214, -0.20046693086624146, -1.0340523719787598, 0.016795627772808075, 0.8753186464309692, -0.4460139572620392, -0.7790773510932922, -0.09642639756202698, 0.5837072730064392, 1.8911583423614502, 0.8180118203163147, 1.3673031330108643, 0.0177607499063015, 0.3532625436782837, 1.512734293937683, 0.3702112138271332, -0.07909571379423141, 0.014330602250993252, -0.7468093633651733, -0.11115207523107529, -0.08516308665275574, -0.2958180010318756, -0.33443936705589294, -2.4549012184143066, -0.12817145884037018], [0.09889893233776093, 0.03555884584784508, 1.360038161277771, -0.19377385079860687, -0.60208660364151, 3.486048698425293, -0.259275883436203, 0.27500972151756287, -0.37471258640289307, 0.1242823377251625, 0.01675582118332386, 0.0013254464138299227, -0.42972221970558167, -0.10384974628686905, 0.39422404766082764, -0.03719718009233475, 0.8284916281700134, -0.40250515937805176, 1.6426361799240112, -0.043693434447050095, -0.3006441593170166, 0.48665156960487366, 0.15759755671024323, -0.7210362553596497, -1.076978087425232, -0.3240646421909332, -3.1066389083862305, 0.006208246573805809, 0.16118276119232178, 1.2132713794708252, -0.04152500256896019, -1.620954990386963, -0.7406311631202698, 0.2519572973251343, -0.5399311184883118, 0.5952991843223572, 0.21272215247154236, -0.018924178555607796, 1.3390834331512451, -2.129479169845581, 1.4980946779251099, 0.11758795380592346, 0.17424535751342773, -0.7611619830131531, 0.2752782106399536, -0.5329364538192749, -0.6639124751091003, -0.058487456291913986, 0.04784379154443741, -0.3498438894748688, 0.4479895234107971, -0.7740321755409241, -0.9791721701622009, 0.1634925752878189, 0.12018553912639618, -0.25219354033470154, -0.24166369438171387, -0.5266206860542297, 1.1489717960357666, 0.11715035885572433, -2.094698429107666, 0.4212602972984314, -0.6972997784614563, -0.21024005115032196, -0.030998118221759796, 0.004597396124154329, 0.031138021498918533, 0.022768579423427582, -1.2573800086975098, 0.3491057753562927, -0.829228937625885, -0.5414962768554688, -1.1656275987625122, 1.4768695831298828, 1.6570477485656738, 0.2399582415819168, 1.0353065729141235, 0.7582144141197205, -0.008301659487187862, 0.1737956702709198, 4.125121116638184, -1.4526747465133667, 0.28853195905685425, -0.01687867008149624, -0.04191100224852562, 0.7343564629554749, -1.578918695449829, 0.11386683583259583, -0.17413786053657532, -0.2772772014141083, -2.0584373474121094, 0.42754074931144714, -0.34886685013771057, 4.138019561767578, 0.12438545376062393, -0.49810874462127686, -0.15241730213165283, 0.33747249841690063, 0.08415333181619644, 1.245035171508789, 0.02627319097518921, -0.367917537689209, -1.269225001335144, -0.3687839210033417, 1.2934603691101074, -0.3076569437980652, -0.7670907378196716, 0.023008612915873528, -0.2080017328262329, 0.4195598065853119, -3.496676445007324, 0.05271775275468826, 0.24956387281417847, -0.8036162257194519, 1.3970067501068115, -0.9616169333457947, 0.14310511946678162, 0.5291851758956909, 1.0091371536254883, -0.0769239068031311, -1.7616881132125854, 0.06558144837617874, 1.7717876434326172, 0.1925039440393448, 0.13811689615249634, -0.10270976275205612, 0.030813004821538925, 0.6160063147544861, -0.41000092029571533], [-0.18224196135997772, -0.3011515140533447, 0.18725131452083588, -0.016667483374476433, -0.048102304339408875, 3.124418258666992, -0.2533295154571533, -1.1479663848876953, 0.37772491574287415, -0.25453880429267883, -0.2655194103717804, -0.10056010633707047, 0.6253941059112549, -0.13317644596099854, -0.051295239478349686, -0.025464197620749474, 2.419384241104126, -0.3965406119823456, 1.0449728965759277, -0.15982910990715027, 0.7845800518989563, 0.30736076831817627, 0.15829624235630035, -1.6676572561264038, -0.33142900466918945, 2.0398881435394287, -1.7189322710037231, -0.02127922885119915, 0.9475395679473877, 0.35454222559928894, -0.09865777939558029, 2.589383840560913, -1.2340819835662842, -0.29082223773002625, -0.775521993637085, 0.5759996771812439, 0.1650383025407791, 0.1765269637107849, -3.9148197174072266, -0.5285167098045349, 0.7158429622650146, -1.3482707738876343, -1.0383074283599854, -1.3951125144958496, 0.46559756994247437, 0.729655385017395, 0.4405710995197296, -1.9739155769348145, -0.07764168828725815, -1.4742183685302734, 1.5162897109985352, 0.993427574634552, 0.0620352178812027, -1.2594455480575562, 0.01413364615291357, 0.7587985396385193, -0.5784346461296082, -0.07052626460790634, -1.2634280920028687, -0.07164016366004944, -0.21964789927005768, -0.7995094656944275, 0.2944553792476654, 0.17616644501686096, 0.0749054104089737, -0.14670339226722717, 0.023911772295832634, 0.02558237873017788, 0.8604917526245117, -1.7348933219909668, -3.403240203857422, 0.3714069128036499, -1.2297086715698242, 2.864051342010498, -1.704182505607605, -1.0600594282150269, -2.155299186706543, -0.9259565472602844, -0.0034633795730769634, 0.15005546808242798, 4.283860206604004, -0.8605878353118896, -2.043168306350708, 0.257801353931427, 0.29825958609580994, -0.43774282932281494, -0.18318918347358704, -0.04170050472021103, -0.31379571557044983, -1.8116134405136108, -3.354062080383301, 2.790170192718506, 0.9725425243377686, 4.644457817077637, 0.11792877316474915, -0.0909019485116005, 0.15258382260799408, 0.4156307578086853, -0.33064591884613037, -0.3636096715927124, 1.4574735164642334, -0.8632459044456482, -3.76318097114563, 0.007116094697266817, 0.4416298568248749, -1.169811487197876, 1.0995376110076904, 0.04226909205317497, -0.21677297353744507, -0.0931827500462532, 1.4593186378479004, 0.394340842962265, -0.3359057307243347, -1.5689194202423096, 1.6967190504074097, 2.3885278701782227, -0.35667476058006287, -0.05398997291922569, 1.07433021068573, 0.09996820986270905, 1.7863789796829224, 0.015240204520523548, 2.425962209701538, -0.10077857971191406, 0.5877490043640137, -0.0046760039404034615, -0.08029669523239136, 1.6836525201797485, -0.7189701795578003], [0.23755095899105072, 0.699811577796936, 0.10546004772186279, 0.25672104954719543, -0.15464791655540466, 1.0327180624008179, -2.1793007850646973, 0.6697161793708801, 0.1929861605167389, -0.1293274313211441, -0.00466806348413229, 0.09650024026632309, -0.3966277837753296, 0.18206451833248138, 0.7475004196166992, 0.02034248225390911, 1.5147722959518433, 1.184188723564148, 1.0136319398880005, 0.16554991900920868, 0.3606848418712616, 0.15882089734077454, 0.3293277323246002, -0.33053356409072876, 0.00027303872047923505, 0.7824575901031494, 0.711708664894104, -0.6448515057563782, -0.5494421720504761, -0.37059199810028076, 0.002966060535982251, -2.7652082443237305, 0.05094347894191742, -1.5712827444076538, 0.4689875841140747, 1.6367300748825073, 1.2699730396270752, 0.5550428032875061, 0.9277063608169556, 0.3439376950263977, 0.2082339972257614, -0.9086567163467407, 1.3397258520126343, -0.556182861328125, -2.203575849533081, -1.1442885398864746, 0.06873149424791336, -0.2043619006872177, -0.00912019144743681, -1.0960828065872192, -1.6727852821350098, 0.8107383847236633, 0.43378815054893494, -0.11403726041316986, -0.13863220810890198, 0.023533865809440613, 0.13026268780231476, 0.37996360659599304, -0.633922278881073, 0.054616618901491165, 1.2989933490753174, 0.8302348852157593, 2.8296124935150146, 0.5670132637023926, -0.12167108058929443, -0.09855949133634567, -0.051424551755189896, -0.6575486063957214, 0.07983499020338058, -1.7468098402023315, -0.39252251386642456, 0.9014458656311035, -0.08417625725269318, 2.4361021518707275, 4.374015808105469, -1.1646199226379395, 0.032017190009355545, -0.18531347811222076, 0.07035208493471146, 0.1799907386302948, 1.111197590827942, -0.12833499908447266, -0.673992931842804, 0.30695968866348267, 0.2590009570121765, -0.09882296621799469, 1.3048310279846191, -0.00542095210403204, -0.44366171956062317, 0.3938670754432678, 0.07150059938430786, 1.4513040781021118, -1.1712355613708496, 0.22009076178073883, -0.42142486572265625, -0.01126832701265812, -8.340751647949219, 0.0892198234796524, -0.47221967577934265, -1.6783658266067505, -1.1017197370529175, 0.7472331523895264, -1.6929521560668945, 1.144709587097168, -2.481250762939453, 1.3259884119033813, 0.7347607016563416, -0.07238657027482986, 0.476831316947937, 0.15439501404762268, 2.2231593132019043, -1.801230788230896, -0.2144683301448822, -0.027628611773252487, -1.3796350955963135, -0.2329399436712265, 0.22811776399612427, 0.620465099811554, -0.49543699622154236, 0.15157464146614075, -0.021442167460918427, 0.0497148223221302, -1.6537200212478638, 0.1278160661458969, 0.21715249121189117, 0.1318807601928711, -0.42395710945129395, 0.4259251654148102, 0.5890634655952454], [0.2555951178073883, -1.6202151775360107, 0.2257203310728073, -0.2570492923259735, -0.16177693009376526, -3.477919340133667, -0.49017995595932007, -1.8389062881469727, 0.9015740156173706, 0.2354917675256729, 0.1488712728023529, 0.8452808856964111, -0.1940809190273285, 0.22749033570289612, -0.3467741012573242, 0.1268344670534134, 0.05701189488172531, -0.7159273028373718, 0.9865838885307312, 0.28623655438423157, -0.22383439540863037, -0.11347195506095886, -0.1400805413722992, 2.4614548683166504, 1.2110258340835571, 0.16201069951057434, 2.449084520339966, 0.6572631597518921, -0.5790861248970032, 0.06401287764310837, 0.2959027886390686, 0.4891819655895233, -0.2637607753276825, 0.3941570818424225, -0.4661070704460144, 2.350663900375366, 0.3019145131111145, 0.20118702948093414, 0.2016480714082718, 0.6411903500556946, 0.3426316976547241, 0.2713591754436493, -2.0948638916015625, -0.2110884040594101, -0.4223288595676422, 0.9990604519844055, -0.07044857740402222, 1.3786547183990479, 0.19344879686832428, 0.8524154424667358, -0.34983715415000916, 1.3458809852600098, -0.6702953577041626, -0.2960599958896637, 0.33416131138801575, 0.13461677730083466, 0.04809337854385376, 0.32934558391571045, -1.3618508577346802, -0.021577350795269012, -1.424147367477417, 0.5335272550582886, -0.36685407161712646, 0.007882010191679, 0.2244456708431244, 0.3982219696044922, 0.20835234224796295, -0.0382939837872982, -0.03296521678566933, -0.7697617411613464, 0.611416757106781, -1.0637060403823853, -1.7582515478134155, -1.1768949031829834, 0.1687353551387787, 0.3614535331726074, 0.5395073890686035, 1.3693877458572388, 0.1977120041847229, 0.41357532143592834, 3.191035747528076, 0.598261296749115, -0.7407248616218567, 0.23994700610637665, -0.13406525552272797, 0.13755545020103455, -0.6508442163467407, 0.17470625042915344, 0.11023000627756119, 0.18393830955028534, -3.875089645385742, -0.7655704617500305, 0.44558098912239075, -2.4499199390411377, 0.35005366802215576, 0.48634734749794006, -5.257123947143555, 0.1840830147266388, 0.7303386926651001, 0.03511226177215576, -0.3353959321975708, 1.5377446413040161, -1.2011164426803589, 0.835225522518158, -0.34644827246665955, 2.255919933319092, 0.8012736439704895, 0.054711390286684036, 0.03332298621535301, -0.04555463045835495, -2.609116315841675, -0.7332627177238464, 0.3760819137096405, 1.5102721452713013, 0.8301187753677368, -1.9952592849731445, 0.31852197647094727, -0.5976914763450623, 1.5068763494491577, 0.2620203495025635, -0.466249942779541, 0.21528783440589905, 1.4387129545211792, -0.17892324924468994, -0.8317685127258301, -0.0513300821185112, 0.42563584446907043, -0.7576612234115601, 0.33132204413414], [-0.26892146468162537, -0.4240858256816864, 1.1273400783538818, -0.19910980761051178, 0.45446503162384033, -0.5371923446655273, -0.9818056225776672, -0.429851233959198, 0.30536746978759766, -0.3780132234096527, 0.04316234216094017, -0.2571554183959961, 0.2035028636455536, -0.2005336582660675, 1.142198920249939, -0.2731277346611023, 0.5991875529289246, -1.155259370803833, 0.26893454790115356, -0.037624459713697433, -0.4686848521232605, -0.3303707242012024, 0.3607896566390991, 1.6333295106887817, -0.026031214743852615, 0.9485276937484741, -0.8780032992362976, -0.35674482583999634, -0.7000370025634766, -3.2716126441955566, -0.25579389929771423, 2.9502503871917725, -0.34514304995536804, 1.2511650323867798, -0.6144305467605591, 1.967781662940979, 0.020490940660238266, 0.10385777056217194, -1.8796002864837646, -0.31240034103393555, 0.9680792093276978, -0.24907667934894562, -0.8696551322937012, -1.7159006595611572, 1.5223922729492188, 0.5100905299186707, -1.3792201280593872, 0.14578086137771606, -0.1177028939127922, 0.36979740858078003, 0.1286092847585678, -1.537442684173584, -0.7233946323394775, 0.26575326919555664, -0.06138383969664574, 0.44281119108200073, -0.6104044914245605, -1.0448955297470093, -0.21579284965991974, -0.14354531466960907, -0.20087282359600067, 2.1025540828704834, -0.34704750776290894, -0.029914801940321922, -0.35861828923225403, -0.07886700332164764, -0.09179087728261948, -0.817413330078125, 1.1715911626815796, -1.2430278062820435, 0.29449957609176636, -0.5308891534805298, 0.7242439985275269, -0.702071487903595, 2.7251007556915283, -0.6266508102416992, 0.07263503223657608, 0.3823634684085846, -0.1151173785328865, -0.18973493576049805, 1.4383528232574463, 0.0299590565264225, -2.5195705890655518, 0.5493043661117554, -0.0034117083996534348, 1.4476712942123413, -0.026445433497428894, -0.07304007560014725, 0.03779209405183792, -0.15037252008914948, 0.8218661546707153, -1.189853310585022, -0.10376305878162384, 2.1684086322784424, -0.27914661169052124, 0.4011685848236084, 3.3795738220214844, -0.03308646008372307, 0.1878964900970459, -0.17172393202781677, 0.6294492483139038, -1.9448572397232056, -0.486797034740448, -0.49571502208709717, 0.25486767292022705, 0.24008186161518097, -0.06717957556247711, -0.13804270327091217, 0.4872529208660126, -0.03953234478831291, -0.5037054419517517, -1.8312476873397827, -0.046060509979724884, -0.06312040239572525, 1.6904945373535156, 0.10300026088953018, -0.12163127213716507, 0.6048629283905029, 1.0957655906677246, 0.1381005346775055, -2.4634010791778564, -0.0794161930680275, -0.7474464178085327, -0.05025487765669823, 0.3714427649974823, 0.061988309025764465, 0.3851865231990814, 0.6378345489501953, -0.15818889439105988], [0.315788596868515, -0.3123828172683716, -1.514108419418335, 0.8654858469963074, -0.06885570287704468, 1.2574511766433716, -0.005093273241072893, -0.94477379322052, 0.22145353257656097, 0.23618370294570923, -0.09771427512168884, -0.17239128053188324, -0.19983890652656555, 0.23485954105854034, -0.4166072607040405, 0.019502202048897743, -1.6738386154174805, -0.8958839178085327, 2.5940499305725098, 0.12294737994670868, 0.003588952822610736, 0.5885785818099976, 0.021814515814185143, -2.6890451908111572, -0.2714110016822815, -0.7060354351997375, -1.381850242614746, -0.4713468551635742, 1.4365423917770386, -0.9481222033500671, 0.027350185438990593, 4.770748615264893, -0.27332207560539246, -0.5807700157165527, 1.2565839290618896, 0.957898736000061, 0.6935223340988159, 0.019742395728826523, -1.365543246269226, 1.0136160850524902, -0.20410016179084778, -0.3683866560459137, -2.69225811958313, 0.5227134823799133, 0.5741230249404907, -1.0266555547714233, 0.24444058537483215, 0.06562723964452744, 0.14671063423156738, 0.9152610301971436, -0.990696132183075, -0.17769891023635864, -0.5573423504829407, 0.5945704579353333, -0.09461939334869385, 0.23161135613918304, 1.039391279220581, -0.423762708902359, -1.0570589303970337, -0.03215658292174339, -0.1390753984451294, -0.061013396829366684, -0.8488950133323669, -0.11821114271879196, 0.06492360681295395, -0.15742814540863037, -0.0714024230837822, -0.4586930274963379, 0.21565201878547668, -0.9585088491439819, -0.9120375514030457, -0.2787608504295349, 1.4081720113754272, -1.10846745967865, -2.3114774227142334, 0.18484023213386536, 1.426255464553833, -0.7340706586837769, 0.08751881122589111, 0.013740407302975655, 0.08971148729324341, -0.06338804960250854, -0.8595806956291199, -0.03639030456542969, 0.1352190226316452, -0.04132459685206413, 0.8736351132392883, -0.04240471124649048, 0.11977197974920273, 0.26638802886009216, 0.2134733498096466, -2.6778805255889893, 1.645078420639038, 0.9442249536514282, 0.24095560610294342, -0.8083299994468689, 3.967751979827881, 0.3392549753189087, 0.8521972298622131, 0.4707508385181427, 0.6747267246246338, -0.5791352391242981, -1.3077514171600342, 0.5158385634422302, -2.841803550720215, 0.8558891415596008, -2.5178062915802, 0.021667679771780968, 0.31212925910949707, 0.06330249458551407, 3.997074604034424, -0.39646676182746887, 0.07935700565576553, -0.5509054660797119, 3.1880686283111572, -2.6843132972717285, 0.3271993398666382, -0.16339002549648285, -0.5341100692749023, 0.21231727302074432, 0.5144057273864746, 0.1219596341252327, 1.3946526050567627, 0.3565113842487335, -0.19415134191513062, 0.0514344684779644, 0.575624406337738, 0.11060136556625366, -0.8535894751548767], [-0.05703843757510185, -0.32734188437461853, 0.031775377690792084, 0.5023948550224304, -0.7677595615386963, -3.0286591053009033, -0.31000417470932007, -1.0006704330444336, 0.2400960773229599, 0.4254636764526367, 0.00435771606862545, -0.16524150967597961, 0.7544424533843994, 0.14514517784118652, -0.3159397840499878, 0.20269131660461426, 2.1753265857696533, -1.6367875337600708, -0.715445339679718, -0.25034889578819275, 0.32743921875953674, -0.5567505955696106, 1.2822788953781128, 1.2681080102920532, 0.8475714921951294, 0.16845697164535522, 0.270658403635025, -0.6341513991355896, 0.03429730609059334, 0.17204071581363678, 0.1727103739976883, 2.8046133518218994, -0.009902110323309898, -0.4685928523540497, 0.2644215226173401, -2.0025787353515625, 1.1157954931259155, 0.36131471395492554, -0.4218336343765259, -0.6113932728767395, 0.26351499557495117, 0.8956184983253479, 0.356879860162735, 1.4096184968948364, 0.5214982032775879, -0.3214631676673889, 0.9780488014221191, 0.7616727352142334, 0.048716362565755844, 0.9018035531044006, 2.452671527862549, -1.6692620515823364, 1.0132850408554077, -1.3781616687774658, -0.13528016209602356, -0.050008807331323624, -1.6897432804107666, 0.273296982049942, -1.8890273571014404, -0.14011134207248688, -0.22467856109142303, 0.7571853399276733, 0.590164065361023, -0.2984931170940399, 0.4087982773780823, -0.13510292768478394, -0.05330769717693329, -0.90660160779953, 2.9993324279785156, -1.5038094520568848, 1.8574379682540894, 0.4330849051475525, 0.8383989334106445, -2.882502317428589, -1.9715538024902344, -0.6959185600280762, 0.8050746917724609, 0.6761338114738464, 0.16017496585845947, 0.47512346506118774, 2.123836040496826, 0.006453424226492643, -3.8791818618774414, 0.0893544852733612, 0.22909508645534515, 0.2825615108013153, -0.980804979801178, 0.3960043489933014, -0.10559704899787903, 1.5846962928771973, -1.7124223709106445, -4.50391149520874, 0.32194197177886963, -1.411344289779663, 0.320668488740921, 0.2019912153482437, 1.7318459749221802, 0.14420998096466064, -1.093648076057434, -0.30785027146339417, 1.4459948539733887, 2.315147638320923, 1.5889501571655273, -1.0040321350097656, -4.772980213165283, 1.0223746299743652, -0.7402881383895874, 0.2509293854236603, 0.46911969780921936, 0.36041316390037537, -5.238075256347656, 2.798161506652832, 0.6275299787521362, 1.388947606086731, 1.2214951515197754, -0.6139217019081116, -0.31390368938446045, -0.7124049067497253, 0.906676173210144, 0.19087737798690796, 1.0727370977401733, 0.08923178166151047, -0.8777023553848267, 0.08511615544557571, 0.10850565135478973, -0.06842540949583054, -0.39717376232147217, 1.2654685974121094, -0.13013604283332825], [-0.04858216643333435, -0.392019659280777, 1.0735669136047363, -0.142276793718338, -0.7011606097221375, -1.6957589387893677, 0.6758382320404053, 1.7118029594421387, 0.05052631348371506, -0.04840414598584175, -0.13704261183738708, -0.13518068194389343, -0.9098042845726013, -0.21918785572052002, -0.6553863286972046, -0.07010716944932938, 1.4324264526367188, 0.24350054562091827, 0.9451268911361694, -0.04752121493220329, -0.14280027151107788, 0.1996239721775055, -0.5182127952575684, 1.1634745597839355, -0.5255411267280579, 0.4940388798713684, 3.07216477394104, -1.3550102710723877, 0.2868141531944275, 0.8823353052139282, -0.2390180379152298, 3.179840087890625, -0.5623846650123596, -0.644657552242279, 0.21367909014225006, -0.6892267465591431, -0.24759264290332794, 0.7027357816696167, 1.5381879806518555, -0.7246422171592712, 3.2433128356933594, -0.7066813111305237, -0.06646916270256042, -0.9645088315010071, -1.6175189018249512, 0.1917208731174469, 0.6366105079650879, -0.1868973672389984, -0.21608318388462067, 0.7822520732879639, -1.8280490636825562, 0.18959371745586395, 1.2506937980651855, -0.6330696940422058, -0.2627077102661133, 0.19409292936325073, 1.1604965925216675, 0.3026212155818939, 1.0935051441192627, -0.22364948689937592, 0.3271285891532898, 0.25814419984817505, -0.3747805655002594, -0.1241496205329895, 0.03982968255877495, -0.07672541588544846, -0.18356026709079742, -0.7503488659858704, 0.16286635398864746, -0.5484439134597778, 0.6226809024810791, -0.8811988830566406, -1.018560528755188, 1.7092502117156982, 0.07270751148462296, -0.296774297952652, -2.1484804153442383, 0.20059649646282196, -0.21088571846485138, 0.12291978299617767, -1.0496790409088135, 0.2973061501979828, -1.656305193901062, -0.0035236510448157787, -0.22920863330364227, 0.3568073809146881, -0.5678367614746094, -0.13404513895511627, 0.06211864575743675, 0.41646671295166016, -2.4128267765045166, -1.133970022201538, 0.19665226340293884, 4.17719841003418, -0.06191666051745415, -0.3369848132133484, 6.443543910980225, 0.32944321632385254, 0.07026758790016174, -0.6688777208328247, 0.3827826976776123, -0.5386031866073608, -1.1738860607147217, 0.5221824049949646, 2.029621124267578, 0.15040849149227142, -0.4874247610569, -0.11596030741930008, -0.9369158744812012, -0.6244900226593018, -3.7091681957244873, -0.7837642431259155, -0.18058235943317413, -0.16942556202411652, -2.1677486896514893, 1.0542809963226318, 0.2009989470243454, -0.25236374139785767, -0.37922367453575134, -0.4392187297344208, -1.3979305028915405, -0.11210467666387558, 0.1925269514322281, -0.36239948868751526, -0.6359458565711975, -0.25542551279067993, -0.5455988049507141, 0.39449742436408997, -0.009392092935740948], [0.2130567729473114, -1.10135817527771, 0.4506310820579529, -0.27395328879356384, -0.50826495885849, 6.253746509552002, -2.4268908500671387, 2.736485004425049, 0.6089816093444824, 0.07855507731437683, 0.06000778451561928, 0.3210418224334717, -1.325863242149353, -0.05148773640394211, -0.3253750205039978, -0.04445058852434158, -0.21629828214645386, 0.18760967254638672, 0.18374712765216827, -0.2867833077907562, -1.0307661294937134, -0.7305376529693604, -0.123245969414711, -1.3945794105529785, -0.30788087844848633, -0.5109916925430298, -1.2327215671539307, 1.6816370487213135, -0.5243198871612549, 0.6208723187446594, -0.04153963550925255, -0.6483977437019348, 0.16689279675483704, -2.5458812713623047, -0.2461060732603073, 4.735323905944824, -0.4070950448513031, -0.3550395369529724, 0.7727257609367371, -1.8658946752548218, -0.8849968314170837, -0.04213051497936249, 1.7138327360153198, -1.2292182445526123, -0.3820028603076935, -0.5972682237625122, -1.3214854001998901, 1.6668199300765991, -0.04523996263742447, 1.0690220594406128, -2.9035379886627197, 1.1352636814117432, -1.0875468254089355, -0.16928721964359283, -0.695319414138794, 0.5569050908088684, -1.3432471752166748, 0.5265026092529297, -0.23997336626052856, 0.07618466764688492, 1.0667848587036133, -0.2127140611410141, 0.05908999219536781, -0.33493608236312866, -0.13869725167751312, -0.4010787308216095, 0.03635217994451523, -0.2710498869419098, -0.9398938417434692, -2.0403597354888916, -0.04417666420340538, 0.4632057249546051, -0.2921944558620453, -1.3839071989059448, 1.8427666425704956, 0.5969603657722473, 0.4099918305873871, 1.3895189762115479, -0.06054035574197769, -0.3079707622528076, 0.407219797372818, 1.0111727714538574, 0.82062828540802, 0.31289181113243103, -0.2835201025009155, 0.35660022497177124, 1.6416637897491455, -0.01184147596359253, 0.1951916366815567, 0.3946988880634308, -3.013706684112549, -2.683016300201416, -0.2371779829263687, -3.456815481185913, -0.0287719015032053, 0.8862109780311584, 0.16268546879291534, -0.37645453214645386, -0.47098681330680847, 1.082144856452942, 1.0632860660552979, -0.29039743542671204, -0.6380946636199951, 1.0620871782302856, 1.4175301790237427, 0.7639428377151489, 1.7719495296478271, 0.014751995913684368, 1.076065182685852, -0.28847169876098633, 0.4669249355792999, -0.10729507356882095, -0.4289923906326294, 0.29127463698387146, 1.2083461284637451, -3.1616671085357666, -0.3614059090614319, 1.2303000688552856, 0.7805027961730957, 0.8025710582733154, 0.7475407123565674, -0.02948450855910778, 0.5447697639465332, -0.028522759675979614, -0.3774770498275757, 0.04362162575125694, 0.4755733907222748, -1.8927714824676514, 1.3487050533294678], [-0.15059241652488708, 0.39984631538391113, -0.21814373135566711, 0.2166626900434494, -0.7399572730064392, 1.1352006196975708, 0.23280945420265198, 1.4378939867019653, -0.07272271811962128, -0.21790489554405212, -0.13435356318950653, 0.39266881346702576, 0.13087400794029236, -0.07530885934829712, -0.029196999967098236, 0.003512643976137042, 1.8180111646652222, -0.8926295638084412, -2.366028308868408, -0.19458524882793427, 0.4840623438358307, 0.053770676255226135, -0.35282090306282043, 0.6681661009788513, -0.3440650999546051, -0.24308998882770538, 1.0871531963348389, 0.013564222492277622, 0.39623838663101196, -1.0592621564865112, -0.13860931992530823, 0.82461017370224, -0.21814049780368805, 0.2472427636384964, -0.6450333595275879, -1.8466566801071167, -0.044321559369564056, 0.2438560128211975, -0.6972869634628296, -0.6977735757827759, 0.6366022229194641, -0.19530414044857025, 1.5061125755310059, -0.8431187868118286, -0.022153301164507866, -0.30233636498451233, -1.0315114259719849, 0.08880244940519333, -0.16547326743602753, -0.9439220428466797, -1.8314913511276245, -3.389720916748047, 0.767633318901062, -1.140992522239685, -0.0034354315139353275, 0.8254653811454773, -0.7380916476249695, 0.0046153622679412365, 0.49789658188819885, -0.17603419721126556, -0.6049860119819641, -0.49128127098083496, 0.38253238797187805, 0.025728648528456688, -0.036725979298353195, -0.12947624921798706, -0.03087523952126503, 0.6197763085365295, 0.7451070547103882, -1.2226630449295044, 0.8272790312767029, 0.5252982378005981, -0.8345694541931152, -0.6994302272796631, 2.7549755573272705, -0.3653090000152588, 4.907382488250732, -1.3761165142059326, -0.07128861546516418, -0.12104040384292603, 0.3370283246040344, 0.6063639521598816, 0.842816948890686, -0.17488157749176025, -0.26398399472236633, 0.8169897794723511, 0.33273595571517944, -0.13392776250839233, -0.0632159635424614, -1.5387083292007446, 0.34480777382850647, 0.9896503686904907, -0.34406474232673645, 2.250662088394165, -0.09861303120851517, 0.38443079590797424, 3.373464345932007, -1.4399381875991821, -0.3405553102493286, -1.2465832233428955, 0.7579171657562256, -0.9932239055633545, -1.4229788780212402, 0.22430700063705444, 0.015947047621011734, 0.3333020508289337, -0.07386171817779541, -0.048022039234638214, 0.495798796415329, -0.523174524307251, 3.035883665084839, -1.8648285865783691, 0.2303002029657364, -1.3661564588546753, -3.3351805210113525, 3.203596830368042, 0.01954544521868229, -1.061249852180481, -0.5465344190597534, 0.27764198184013367, -1.2342225313186646, -0.03342209756374359, 1.826255202293396, -0.358670711517334, 0.9137983918190002, -0.016797998920083046, -0.4450013041496277, -0.17383743822574615, 0.7728708982467651], [0.11199509352445602, 2.940047025680542, -0.41767799854278564, 0.2957907021045685, 0.8938313126564026, -2.737114429473877, 0.027994882315397263, -0.9074497818946838, 0.04427417740225792, 0.21755222976207733, -0.05808589980006218, 0.5442763566970825, -0.27274924516677856, -0.13789822161197662, 0.5204002261161804, -0.03487177938222885, -1.4340556859970093, 0.31512370705604553, -0.7894988656044006, -0.03534747287631035, -0.1487317830324173, -0.08575398474931717, -0.05300134792923927, 1.2261204719543457, -0.04693986847996712, 0.7249484658241272, -3.159562349319458, 0.0682162418961525, -0.16247206926345825, -0.437717467546463, -0.017814088612794876, -3.078087568283081, -0.41190099716186523, -0.7867143154144287, 0.16881340742111206, 0.05053853988647461, 0.1686124950647354, 0.12639303505420685, -1.184273600578308, -0.3377729654312134, -0.5881364941596985, 0.614675760269165, 3.3980045318603516, 1.3061493635177612, -0.6489611268043518, -0.2052006870508194, -0.02878447063267231, -0.4541770815849304, 0.0405573844909668, -1.6154953241348267, -2.849522113800049, 1.0692299604415894, 1.0392117500305176, -0.3534920811653137, -0.10868896543979645, 0.07520848512649536, 0.44879236817359924, 0.10880999267101288, 0.651214599609375, 0.16787266731262207, -1.396658182144165, -1.8904509544372559, -0.804178774356842, 0.2729732394218445, -0.043311648070812225, 0.06288359314203262, -0.050902824848890305, 0.7651511430740356, 2.1237494945526123, -0.30414536595344543, 0.09416306018829346, -0.19629481434822083, 1.8213387727737427, -1.000908374786377, 1.9000622034072876, -0.20736747980117798, 2.7535338401794434, 0.22376149892807007, 0.12461324036121368, -0.03370975703001022, -0.358828604221344, -0.6356268525123596, 2.1984400749206543, 0.6016581058502197, -0.22415737807750702, 0.19679014384746552, -0.08233487606048584, 0.06031317636370659, 0.004321901127696037, 1.170586347579956, 1.2525177001953125, 0.48595547676086426, -1.0636308193206787, 1.3593984842300415, 0.06609854847192764, 0.1886100023984909, -0.46115362644195557, -0.847711980342865, 0.2503393292427063, 0.5406127572059631, 1.7796096801757812, -0.25220587849617004, -2.3395564556121826, -0.8107364177703857, 1.5571855306625366, -2.71661639213562, -0.09494888782501221, 0.1013866737484932, -0.46278277039527893, -0.0015719023067504168, -0.4682464301586151, 0.49993985891342163, -0.023869328200817108, 1.1020478010177612, 1.60737943649292, -1.8781101703643799, -0.1316220462322235, 0.8351973295211792, 0.0587959885597229, 0.38563892245292664, -0.9257935285568237, 0.058144330978393555, 0.2647705674171448, 0.2538299262523651, 0.37405455112457275, 0.12000811100006104, -0.01217049453407526, -1.359588861465454, -0.21337325870990753], [0.03359554708003998, -0.5503383278846741, 0.9747138619422913, -0.04430936276912689, -0.16406482458114624, -0.20866511762142181, -0.8152415752410889, 0.20620287954807281, -0.62257981300354, -0.18236488103866577, -0.18624792993068695, -0.3268827795982361, -0.20205654203891754, 0.03045279160141945, 0.5300558805465698, -0.2307511568069458, -0.5189659595489502, -0.7730833292007446, -1.293248176574707, -0.15459752082824707, -0.20828168094158173, 0.35847973823547363, 1.186253547668457, -0.4853755831718445, 0.0449252650141716, 0.5636928677558899, -0.6632674336433411, 0.7187924981117249, 0.35802438855171204, -0.4889296889305115, -0.2418220192193985, 2.3020708560943604, -0.39104142785072327, 0.5569866895675659, -0.22336487472057343, -1.6816266775131226, 0.19868768751621246, -0.13112325966358185, 0.9005528688430786, 0.9280403256416321, 0.7295733690261841, -1.3937363624572754, -0.30933257937431335, 1.3856056928634644, 0.9664258360862732, 0.23339825868606567, 0.19692358374595642, 1.6727852821350098, -0.03956790640950203, 0.9549084305763245, 0.5650714635848999, 0.7006328105926514, -1.679450511932373, -0.12011326849460602, -0.07181399315595627, -0.095761738717556, 0.9956645369529724, 0.20117996633052826, 0.7465915679931641, 0.054858867079019547, -0.3142319321632385, -1.3896156549453735, 0.735740065574646, 0.1384950578212738, -0.4059663414955139, 0.3721447288990021, -0.03737235814332962, -0.8309456706047058, 0.07999075949192047, 0.9051183462142944, 0.9076669216156006, 0.20527943968772888, -1.2148289680480957, -1.5721007585525513, -1.3511238098144531, 1.501688838005066, -2.417823076248169, -0.36399009823799133, -0.10425714403390884, 0.07452982664108276, -0.5293601751327515, -0.06919694691896439, -0.1614091694355011, 0.5904160737991333, 0.04697698354721069, -0.3286530673503876, 0.8985664248466492, -0.08748085051774979, -0.10574781894683838, 0.4160977602005005, 0.6131457686424255, -2.0721256732940674, -0.5714304447174072, -0.9314391016960144, 0.09429476410150528, -0.04188128560781479, 2.515047788619995, 0.5679774284362793, -0.4359751343727112, -0.10675093531608582, -0.370906263589859, 1.4008195400238037, -0.4473104476928711, 1.3954212665557861, -1.357962965965271, -1.6320291757583618, 0.8624855875968933, -0.18342958390712738, 0.23686394095420837, -0.0035223113372921944, 0.03267010301351547, 0.09181872755289078, -0.4340704679489136, -0.1502363234758377, 0.18983256816864014, -2.9795610904693604, 0.29983365535736084, 0.04629768431186676, 0.7183228731155396, -0.10517413914203644, -0.018611129373311996, -0.20837152004241943, 1.8343058824539185, 0.27882927656173706, 0.21423611044883728, -0.1364619880914688, -0.16818080842494965, -0.4856620132923126, 1.0631418228149414], [-0.17628654837608337, 1.476844072341919, -1.6760046482086182, 0.09137891978025436, 0.21466317772865295, -0.5781102180480957, -0.619497537612915, -1.4771850109100342, -0.0979374572634697, 0.04549819976091385, 0.09964648634195328, -0.12637439370155334, 0.3125351667404175, 0.006998776458203793, -0.539435863494873, -0.07175379991531372, -2.5039825439453125, 1.369104027748108, 0.56317138671875, -0.03182085603475571, 0.02140149287879467, -0.29255232214927673, 0.5720559358596802, -2.630765914916992, 0.4332404136657715, -0.04423672705888748, 1.7427692413330078, 1.0213431119918823, -0.2679170072078705, 1.200044870376587, 0.2015710175037384, -2.808790922164917, 0.6567208766937256, 0.8719869256019592, 0.15906979143619537, 1.5291308164596558, -0.12826292216777802, 0.14333441853523254, 2.9116787910461426, 0.5098539590835571, -1.149522066116333, -0.551638662815094, -0.2978695333003998, -1.7646949291229248, 0.7442340850830078, 1.011267900466919, -0.12250945717096329, 0.8767743706703186, -0.014676305465400219, -1.1952900886535645, 0.795099675655365, 0.4421303868293762, -0.13760004937648773, -0.16341671347618103, -0.08363478630781174, 0.2842179834842682, -1.8681397438049316, -0.45521295070648193, 1.0981826782226562, 0.11967858672142029, 1.3112823963165283, 0.8237373232841492, -0.3819209337234497, -0.013756461441516876, -0.28267180919647217, -0.29335129261016846, -0.007855016738176346, 0.5782251954078674, 1.0755857229232788, -0.5747758150100708, 2.297083616256714, -0.5880208611488342, 0.549807608127594, -1.2722724676132202, -0.9162551760673523, -0.2527555823326111, 0.15593624114990234, 0.6339285969734192, 0.1170492097735405, 0.24101249873638153, -0.7991892099380493, 0.04764089360833168, -5.655276298522949, 0.3643959164619446, -0.17077530920505524, -0.3487250506877899, 0.9764660596847534, 0.008710210211575031, -0.10603255033493042, 1.7314069271087646, 1.1027168035507202, -0.9695038795471191, -0.7894955277442932, -2.1702725887298584, -0.281594455242157, -0.8727484941482544, -0.19195282459259033, -0.5156872868537903, -0.49731454253196716, 1.5877856016159058, 1.9232079982757568, -1.3861823081970215, 0.4130913317203522, 0.3523407578468323, 2.662386655807495, -0.7736746072769165, 2.140158176422119, 0.026879310607910156, -0.00440680468454957, 0.2907061278820038, 1.7210631370544434, 1.1196885108947754, 0.056816522032022476, -0.4017798602581024, -0.47986921668052673, 0.45305779576301575, -0.27335941791534424, -0.17035090923309326, -0.6256322860717773, 0.05631120502948761, -0.8437515497207642, -0.047675736248493195, -1.595766544342041, 0.5877350568771362, -0.7366887927055359, -0.19102969765663147, -0.0974818542599678, 0.5033135414123535, 0.019818516448140144], [-0.03806499391794205, 0.8213697671890259, -1.460453748703003, 0.1627344936132431, 0.1957814246416092, 5.988100528717041, 1.359055995941162, 0.5854145288467407, 0.1947263777256012, 0.2864713668823242, 0.14833436906337738, 0.2738310992717743, -0.20190244913101196, -0.11281663179397583, 0.21494121849536896, -0.2202516496181488, -2.1553239822387695, 1.754215955734253, 0.22767506539821625, -0.008755091577768326, -0.19731540977954865, 0.24581775069236755, 0.5117893815040588, -0.5240188241004944, 0.09310798346996307, -0.7128854990005493, 0.24222759902477264, -0.10102248936891556, 0.7733768820762634, 0.9750556945800781, -5.586664701695554e-05, -2.638571262359619, -0.7889673709869385, 2.280034065246582, 0.8817148208618164, -0.6609660983085632, 0.44567203521728516, 0.15932905673980713, -2.626291036605835, -0.7526084780693054, 0.21209560334682465, 0.11036696285009384, -0.32455575466156006, 1.9929187297821045, -1.4900106191635132, 0.8917707204818726, -0.9592253565788269, -2.179001808166504, 0.028612002730369568, 0.5652328133583069, 0.9294430017471313, 0.9420632719993591, -0.7815977334976196, 0.4942808747291565, -0.11687316745519638, -0.008070050738751888, 2.862661838531494, -0.4983847141265869, 1.0512919425964355, -0.016850043088197708, -0.6887585520744324, 0.36816316843032837, -1.046245813369751, 0.337167352437973, 0.30672743916511536, -0.3812905251979828, 0.14831602573394775, 0.7857125401496887, 0.8627690672874451, 0.8455008268356323, -1.6199537515640259, -0.29265934228897095, -3.0184905529022217, -2.8187637329101562, -0.6172412037849426, -0.3811071217060089, -0.8859377503395081, -0.5244885683059692, -0.061653975397348404, 0.017050916329026222, 2.7900893688201904, -0.5294638872146606, 2.0314865112304688, -0.24921725690364838, 0.323927104473114, -1.2326682806015015, 0.14245660603046417, 0.23396120965480804, 0.16747725009918213, -0.6421128511428833, -0.6916000247001648, -1.391787052154541, 0.7648162841796875, 1.4107880592346191, -0.21628041565418243, 0.8985542058944702, -2.7852981090545654, 0.0444054938852787, -0.4694114625453949, 0.8059676885604858, -0.5389729738235474, -1.0609726905822754, -0.317060261964798, -0.7952142953872681, 2.7218222618103027, -0.169514998793602, 0.502572238445282, 0.024118585512042046, -0.007197326514869928, -0.15604987740516663, -0.10384505987167358, -0.23099838197231293, -0.234622523188591, 0.17144516110420227, -2.3863096237182617, -1.8149909973144531, 0.1910640150308609, 1.0503000020980835, 0.7746813297271729, -0.0940731093287468, 0.04452824592590332, 0.20687457919120789, 1.042662262916565, 0.9192876815795898, -0.3212165832519531, 0.12386903911828995, 0.3569488525390625, -1.6748833656311035, -0.2918338477611542], [0.05606265738606453, 0.5252726674079895, 0.08659570664167404, 0.6427649855613708, 0.6166657209396362, 1.2936859130859375, 2.2728729248046875, -0.3557513654232025, -0.025572478771209717, -0.12496209144592285, 0.010238192975521088, -0.35325396060943604, 0.588062047958374, 0.1602753847837448, -0.5177093744277954, -0.14699549973011017, 0.2620077431201935, -0.2938741147518158, 3.0540294647216797, -0.01925363950431347, 0.3917958736419678, 0.4215249717235565, 1.1192413568496704, -2.3328511714935303, -0.11777864396572113, -1.0902020931243896, -0.6634044051170349, 0.3698951303958893, 1.0798081159591675, -3.610734224319458, 0.09736689180135727, -1.4277111291885376, 0.3840040862560272, 0.1499590277671814, -0.4696873426437378, 5.172450065612793, -0.09190171211957932, 0.2863578796386719, -1.2287122011184692, 1.3857007026672363, -0.8756941556930542, -0.33345699310302734, -6.082660675048828, 1.9337087869644165, 0.2712976038455963, -1.0792473554611206, -0.16485455632209778, -2.2213919162750244, 0.15265372395515442, 1.5997976064682007, 1.2895748615264893, 4.388551712036133, -0.7053502202033997, 0.5758656859397888, 0.03933396935462952, 0.24960924685001373, 2.0220284461975098, -0.1750229299068451, 0.2933049499988556, 0.04589925333857536, -0.43294087052345276, -1.1863727569580078, -1.1589691638946533, -0.0045164600014686584, 0.17046000063419342, -0.027983659878373146, 0.20623430609703064, -1.645642876625061, -0.09405524283647537, -2.6240386962890625, 1.0738023519515991, 0.14744949340820312, -0.705865740776062, 0.28588441014289856, -0.5963913202285767, -0.8471086621284485, 2.27665114402771, -0.2696826159954071, 0.0506834015250206, 0.09288223087787628, 0.037867818027734756, -1.1039609909057617, -4.515372276306152, 0.4368508756160736, -0.4681905508041382, -0.15499655902385712, 0.821081817150116, 0.052403539419174194, 0.0945555716753006, 0.9729653596878052, 0.25283104181289673, 1.157037377357483, 1.3897171020507812, 2.333430051803589, 0.061258796602487564, 0.9790990948677063, -4.006826877593994, 0.24377961456775665, -0.4479876160621643, -1.985675573348999, -1.8380244970321655, 0.6095698475837708, 2.238926887512207, -1.3113515377044678, -0.731497585773468, 2.3138136863708496, -1.6391137838363647, 0.08410181850194931, 0.8528345227241516, 0.21227435767650604, -2.0356876850128174, -0.15466836094856262, -0.4808872640132904, 1.4804574251174927, 1.6222898960113525, -0.6485834717750549, 0.012586175464093685, -0.7845848798751831, -0.9871829748153687, -0.22701531648635864, 1.2044641971588135, 0.055627983063459396, 2.243804693222046, -0.00881402287632227, -0.17299965023994446, -0.19464349746704102, -0.427154541015625, -0.4609963297843933, -0.46812179684638977], [-0.003720636013895273, 0.8284152150154114, 1.1910706758499146, -0.09046296775341034, -0.16047783195972443, -0.7946274876594543, -0.6471351981163025, 0.6180994510650635, -0.18214717507362366, -0.0774586945772171, -0.021554846316576004, -0.10986405611038208, 0.05869172513484955, 0.0066109648905694485, 0.9014901518821716, 0.05127710849046707, -1.879903793334961, 0.6613290905952454, 1.0749925374984741, 0.03896111994981766, -0.48720288276672363, 0.1764194220304489, -0.9561173915863037, 0.36257311701774597, -0.3114039897918701, 0.5955389738082886, 1.3359144926071167, 0.5091480016708374, -0.07867758721113205, -3.3312718868255615, -0.10235778987407684, 3.294708490371704, -0.43083375692367554, -0.048781681805849075, -0.4907585680484772, 2.042107105255127, 0.8709396719932556, 0.23367860913276672, 0.020744603127241135, -1.2096253633499146, 0.37331175804138184, -0.0595574751496315, -0.5461186766624451, -0.15151123702526093, 0.529987096786499, 1.733780860900879, 0.3676098585128784, 0.04011502489447594, 0.0948902890086174, -0.3356878459453583, 0.4732140898704529, -2.227717161178589, 1.1954256296157837, 0.6540054678916931, -0.09322930127382278, -0.21551428735256195, 0.5022290945053101, 0.24637645483016968, 0.9565318822860718, -0.17531704902648926, -1.7021057605743408, -0.1551474630832672, -0.9012625813484192, -0.0529022254049778, -0.17807698249816895, 0.34173375368118286, 0.0782497227191925, -0.3085848093032837, -0.6833405494689941, -0.21067258715629578, -1.4370747804641724, -0.18722310662269592, 0.992108941078186, -0.5733030438423157, -1.9387551546096802, 0.38758736848831177, 0.3953635096549988, 0.7051419615745544, 0.07564030587673187, -0.011499589309096336, -0.7344111800193787, -1.4367200136184692, -0.3011023700237274, 0.2993258535861969, -0.2122359424829483, -0.012241140939295292, 0.5630359649658203, 0.04216819629073143, -0.1512175351381302, 0.0983085036277771, 1.5144919157028198, -1.0529240369796753, -0.4729832708835602, -0.21153169870376587, 0.3783612549304962, -0.27915337681770325, 1.018807053565979, -0.09722959995269775, -0.32965898513793945, -1.2045321464538574, -0.10904873162508011, -2.2152202129364014, -0.45545634627342224, -0.16503547132015228, -1.2208889722824097, -1.5029655694961548, 1.4237022399902344, 0.09545550495386124, 0.5737892389297485, 0.03937804698944092, -0.3880525529384613, 0.9242033362388611, 0.9863802194595337, 0.6072009801864624, 1.6582183837890625, 1.8918251991271973, -0.039974480867385864, -0.675456166267395, 0.004143622238188982, -0.21711599826812744, -0.08341994881629944, 0.08011486381292343, 0.04900660365819931, 0.24432705342769623, 0.7148385047912598, 0.05041654780507088, 0.22222746908664703, -1.414098858833313, 0.04591970890760422], [0.04019083082675934, -0.09891010075807571, -0.1386488825082779, 0.03848949074745178, 0.3927687704563141, -0.5887942910194397, 0.6787564754486084, 0.5035748481750488, -0.0039564622566103935, -0.2284567803144455, -0.06461583077907562, 0.31802263855934143, 0.6949062943458557, -0.12014443427324295, -0.11139748990535736, 0.0021468002814799547, -1.214471697807312, -1.1362603902816772, 0.8714132308959961, 0.06263725459575653, 0.740021824836731, -0.5246379971504211, 1.0257641077041626, 1.0155248641967773, 0.054551802575588226, 0.23815403878688812, -0.7974706888198853, -0.27924367785453796, 0.5572526454925537, -0.875544011592865, -0.020605536177754402, -0.3692156970500946, 0.24511165916919708, -1.649715781211853, 0.3119262158870697, 0.8733823299407959, -0.788601279258728, -0.4293673634529114, 1.211135983467102, 0.30558785796165466, -0.9916355013847351, -0.49294912815093994, 0.9560343027114868, 1.762918472290039, 0.8512970209121704, -0.36260533332824707, -0.6370726227760315, 0.09365855902433395, -0.08368924260139465, 1.7260011434555054, -0.671154260635376, -1.0725051164627075, -0.3867338299751282, -0.7110177874565125, 0.17361579835414886, 0.5854679346084595, 0.010798734612762928, 0.05666210129857063, 0.9861482381820679, -0.24479526281356812, -0.07505100965499878, 0.4979169964790344, -0.8485740423202515, -0.027847526594996452, 0.05056067183613777, 0.1105206161737442, -0.12042909860610962, 0.19820702075958252, 2.2004401683807373, -3.0950798988342285, 1.1450331211090088, -0.7384231090545654, -1.3451389074325562, 1.4647420644760132, 0.10088571161031723, 0.7648840546607971, -0.8045049905776978, 1.0623679161071777, -0.05832904577255249, -0.4003477096557617, -0.04207434132695198, -0.5680311918258667, -0.9378676414489746, 0.37702643871307373, 0.005151156801730394, -0.4079444110393524, 1.1075162887573242, -0.020232198759913445, 0.09208366274833679, -2.0691075325012207, 1.6381291151046753, -0.42846041917800903, 2.0200889110565186, -0.6441536545753479, -0.13253171741962433, -0.9431294202804565, 0.6172470450401306, 0.2510577142238617, 1.16688871383667, -1.496221899986267, 0.16622675955295563, -2.1120290756225586, -1.5755723714828491, 2.3281121253967285, 0.30040231347084045, -2.0760674476623535, 0.9240081906318665, -0.11865639686584473, 0.593912661075592, -0.09895731508731842, 3.4784414768218994, -1.4585649967193604, -0.06375804543495178, 0.516490638256073, 0.12801307439804077, -0.2554466128349304, 0.06401710957288742, -0.416049987077713, 0.66741544008255, -0.16616351902484894, 0.07411245256662369, -0.13285920023918152, -0.8289580345153809, 0.3851568400859833, -1.2447954416275024, 0.06258595734834671, 0.1552056223154068, -0.4056275188922882, -0.8475683331489563], [-0.24701425433158875, -0.11836673319339752, 1.465451955795288, -0.20608535408973694, 0.10598985105752945, 1.5347280502319336, -0.9741737842559814, 2.2895708084106445, -0.17955869436264038, -0.40131130814552307, 0.027609579265117645, -0.15429562330245972, 0.5325514078140259, 0.1510210931301117, -0.3481495976448059, 0.14341509342193604, 2.0423121452331543, -0.2454397976398468, 2.9995217323303223, -0.1174934133887291, -0.19784533977508545, 0.28460052609443665, -1.4787167310714722, -0.9129260182380676, 1.6483237743377686, 0.6329361200332642, 0.4237835109233856, -1.557803750038147, 0.051247481256723404, -0.42995041608810425, 0.08623805642127991, -1.6547192335128784, -0.5543343424797058, 1.540480375289917, 0.4660821557044983, -2.1978085041046143, -0.10507724434137344, -0.0849849060177803, -1.212630271911621, 0.5758599638938904, 0.6362047791481018, 1.5642955303192139, -1.4395695924758911, -1.2817535400390625, -0.3129814863204956, 0.07041976600885391, -0.09416884183883667, -0.7825357913970947, -0.11756247282028198, -0.3143022656440735, 0.745230495929718, -0.03740513324737549, 0.2760408818721771, 0.16508832573890686, 0.5112714767456055, 0.33089107275009155, 0.41524675488471985, 0.3645992577075958, -1.0523548126220703, -0.07782673835754395, -0.7642395496368408, -0.2470533698797226, -1.3447662591934204, -0.21140041947364807, 0.033795639872550964, -0.04540640115737915, 0.42824846506118774, -1.0733709335327148, 1.790814995765686, 0.20894017815589905, -1.4286538362503052, -0.1580001413822174, 0.49132004380226135, -1.5825382471084595, -1.029407262802124, -0.5364757180213928, 1.2891453504562378, -0.1579761654138565, -0.03151732683181763, 0.19164708256721497, -0.5166141986846924, -1.3416956663131714, -3.1345090866088867, -0.6801435947418213, 0.23000755906105042, 0.3299517035484314, 0.23244665563106537, 0.053380876779556274, 0.14957110583782196, 1.0571445226669312, 6.116945266723633, 0.07862532883882523, 1.3969603776931763, 0.9056611061096191, 0.2626868784427643, -0.30898210406303406, -3.622000217437744, -0.07907894998788834, -0.38288378715515137, -0.02794727124273777, -0.8636686205863953, 0.9250494241714478, 2.139807939529419, 0.6770575046539307, -0.9578438997268677, -0.13308420777320862, -4.209856986999512, 0.05232241004705429, -0.04853931441903114, 0.16074073314666748, 0.6847737431526184, -0.6888925433158875, 1.036301851272583, -1.3250055313110352, 1.2096797227859497, 2.078200578689575, -0.18927358090877533, -0.12999612092971802, -0.620966374874115, -0.36249053478240967, -0.12462171167135239, -0.0002740802592597902, -0.7407612800598145, 0.2791787087917328, -0.4121473431587219, -0.19964031875133514, 0.13708315789699554, 0.6528196334838867, 1.2395483255386353], [-0.1396103948354721, -0.5439974665641785, -0.41540002822875977, -0.48359283804893494, -0.14915107190608978, -0.27205735445022583, -0.32458049058914185, 1.0778565406799316, -0.1594725400209427, 0.02694900892674923, -0.059851959347724915, 0.04434242472052574, 0.007216308265924454, 0.18577495217323303, -0.2931934893131256, -0.008871455676853657, -0.8651359677314758, -0.29809942841529846, -0.4278499186038971, -0.037068359553813934, 0.11856207251548767, 0.08540742099285126, -0.5841174125671387, -0.590224027633667, -0.5331916213035583, -0.16612961888313293, -1.6987797021865845, 0.15098492801189423, -0.984395444393158, 1.244834303855896, 0.12019829452037811, -1.500359058380127, -0.4034624695777893, -0.7487273216247559, 0.7007423639297485, 1.6599445343017578, 0.94336998462677, -0.013031970709562302, 3.304593563079834, 0.0660659670829773, -1.1751705408096313, 0.32145974040031433, -0.06577567011117935, 0.2886944115161896, 1.6257325410842896, -1.0499762296676636, 0.007879839278757572, -2.1155309677124023, -0.0004580934182740748, -0.28521111607551575, 0.8760079741477966, 0.6151689291000366, -0.3864641487598419, 0.4102422297000885, 0.14695687592029572, -0.07045888900756836, -0.5169547200202942, -0.07689432799816132, 0.14400726556777954, -0.11804629862308502, -0.23965215682983398, -0.35888999700546265, 0.02293628454208374, -0.11703083664178848, 0.012774123810231686, 0.20193184912204742, -0.0030291080474853516, 0.668535590171814, -0.6488555073738098, -1.79445996764116e-05, -2.445646047592163, 0.12024762481451035, 1.2671520709991455, 1.828934907913208, -1.0585718154907227, 0.42036110162734985, -3.7254574298858643, -0.004828677047044039, -0.03446506708860397, 0.03834120184183121, -0.6614184975624084, 0.5888440012931824, -1.8580580949783325, -0.5695195198059082, -0.12371745705604553, 0.4272565543651581, -0.26089558005332947, 0.029006950557231903, 0.13313904404640198, 0.6897817254066467, 2.20219087600708, 1.2815403938293457, 0.4695439040660858, 2.2390992641448975, -0.20882152020931244, -0.5215526819229126, -1.8936924934387207, -0.08477000892162323, -0.12847410142421722, 2.816542625427246, 0.37920865416526794, 0.4577719569206238, 0.7844069004058838, 0.21554110944271088, 0.5932334065437317, 0.12099254131317139, 1.5430738925933838, 0.04097209870815277, 0.5297232866287231, 0.25920620560646057, -3.846773862838745, -0.04781755059957504, 0.5421647429466248, 0.0174407921731472, 1.409951090812683, 2.395416021347046, -0.07217787951231003, -0.2946908473968506, -0.3732755780220032, -0.03847240284085274, -1.0242301225662231, -0.02505972608923912, -1.6222470998764038, -0.3201514482498169, 0.6557117104530334, 0.03436296433210373, 0.31863054633140564, -0.07204510271549225, 0.31663811206817627], [0.06979573518037796, -1.0833224058151245, -1.3068779706954956, -0.2811194360256195, -0.7902798652648926, 1.492449164390564, -1.8156400918960571, 2.5144689083099365, 0.04721274599432945, -0.17011404037475586, -0.2661724090576172, 0.17511703073978424, 0.42360830307006836, -0.11460710316896439, -0.818175733089447, -0.4109219014644623, 1.5982760190963745, 1.399665117263794, 0.4801231026649475, 0.1280006319284439, 0.5212827920913696, -0.2984790802001953, -0.3920240104198456, 0.7934267520904541, 0.1527044177055359, -0.18464665114879608, -0.19709201157093048, -0.11834187805652618, -0.13105855882167816, 0.67302006483078, -0.1835693120956421, -0.8304763436317444, 0.18098857998847961, -1.7304850816726685, -0.7811258435249329, 2.2979133129119873, 0.4025416672229767, 0.032125599682331085, 2.908221483230591, 0.9819471836090088, 1.2442232370376587, -0.5178927779197693, 1.5018467903137207, -0.5703843832015991, 0.9479216933250427, 0.48014625906944275, -1.442783236503601, 0.9317640066146851, -0.09736975282430649, 0.43184301257133484, 1.9789135456085205, 1.5981199741363525, 2.0233466625213623, -1.52993643283844, 0.08225182443857193, -0.08844573050737381, -1.033225655555725, 0.7634045481681824, 1.1124470233917236, -0.1075909286737442, 0.16828730702400208, 0.7711157202720642, -0.05396554246544838, 0.1649434119462967, -0.016306154429912567, -0.3711523413658142, -0.03881540149450302, 0.06764786690473557, 0.3435366749763489, -1.142883062362671, -1.6732439994812012, -1.2139909267425537, -2.228114128112793, -2.011474609375, 1.3454097509384155, -1.3911930322647095, 3.174085855484009, 0.7105765342712402, -0.1196729764342308, 0.062335480004549026, 3.469202995300293, 0.8934472799301147, 0.880012571811676, 0.1029425859451294, -0.3448740243911743, -0.808624804019928, 1.8217980861663818, -0.11661174893379211, 0.022321784868836403, 1.6655939817428589, -0.4663703739643097, 0.7099533081054688, -0.8593770861625671, -1.1248949766159058, -0.2540820837020874, 0.022994423285126686, 0.6875916719436646, -0.6109129786491394, -0.5873590111732483, -2.0371649265289307, 0.5615599155426025, 2.4213173389434814, 0.8061689734458923, -1.222074031829834, -0.19910387694835663, -1.6174577474594116, 1.2878122329711914, -0.18688492476940155, -0.032392241060733795, -0.2857729196548462, -2.8547286987304688, 0.450331449508667, 0.04597464203834534, -0.9943028092384338, -1.08685302734375, 1.6790891885757446, 0.05399421975016594, -1.0566163063049316, -0.8328641057014465, 0.021004408597946167, -1.0335228443145752, -0.11910766363143921, -4.397706031799316, -0.08329698443412781, -2.1288232803344727, -0.1830224096775055, -0.5644193291664124, -0.8904058337211609, -0.24756823480129242], [0.22999995946884155, 1.4830511808395386, 1.397629976272583, 0.6779066324234009, -0.27775558829307556, -0.6609821319580078, -0.5756904482841492, 2.9643921852111816, 0.15568533539772034, 0.037827711552381516, -0.22266711294651031, -0.061281949281692505, -1.484243392944336, 0.4082809388637543, 0.18049068748950958, -0.09310339391231537, 0.9114551544189453, -0.30486881732940674, -1.285269021987915, 0.02393694408237934, 1.2768844366073608, -0.40931200981140137, -0.2260465919971466, -0.6012176275253296, -0.805171012878418, -0.22283846139907837, -3.9157447814941406, 0.44777682423591614, 0.480169415473938, -0.8584684133529663, 0.4387179911136627, -0.5118985772132874, -0.28308945894241333, 0.1295420527458191, 0.23699337244033813, -0.43528130650520325, 0.14470882713794708, 0.4905015528202057, 2.1840248107910156, -0.07145363837480545, -0.34771928191185, -0.36020973324775696, 1.0403611660003662, 0.5497527122497559, 1.1351255178451538, 1.1941224336624146, -0.7362651824951172, 0.45331060886383057, -0.025635134428739548, 1.5159099102020264, -0.5794752836227417, -1.6306697130203247, -1.169958472251892, -1.2850580215454102, -0.6135380268096924, 0.7161180377006531, -0.0437583327293396, -0.35185706615448, -0.3463195264339447, -0.003196000587195158, 0.21231906116008759, 0.47522875666618347, -0.2812745273113251, -0.01621438004076481, 0.29339492321014404, 0.21997034549713135, 0.3211384117603302, -0.22990773618221283, 0.5181394219398499, -0.5033743977546692, -0.16398365795612335, 1.019172191619873, -0.3136070668697357, -2.5582659244537354, -3.838926315307617, -0.4290757477283478, 1.6146291494369507, -1.117267370223999, 0.32136404514312744, 0.10912680625915527, -2.684326410293579, -1.398826241493225, -3.3039345741271973, -0.7955396175384521, -0.070078045129776, 0.05327119678258896, 0.23617695271968842, -0.22496473789215088, 0.12763340771198273, 0.4181874394416809, 5.498349189758301, 2.6540489196777344, 0.6806718707084656, 1.7305902242660522, 0.2506582736968994, 1.0458500385284424, -0.27424126863479614, -0.07261470705270767, 2.161635160446167, -1.0870420932769775, 0.7332690954208374, -1.2640955448150635, 1.5615015029907227, 0.7803719639778137, 0.4399775564670563, -1.6183745861053467, 0.23380528390407562, 0.20060309767723083, 0.22546914219856262, 0.3887213170528412, -1.2830967903137207, -1.3353568315505981, 0.3809987008571625, -0.7357781529426575, 1.5096849203109741, -4.004298686981201, 0.5275111198425293, -0.23053564131259918, 1.8271678686141968, 0.2775409519672394, 0.23393434286117554, 0.5068566203117371, 0.37026605010032654, -0.1589612513780594, 0.00430001737549901, 0.12205405533313751, 0.10209571570158005, 0.06295955181121826, -0.5613148808479309], [0.17167258262634277, 0.5446089506149292, -1.3454079627990723, 0.21256248652935028, 0.3045550286769867, -1.7204039096832275, -0.8687085509300232, 0.4097996950149536, -0.13441140949726105, 0.3079221844673157, 0.5226256847381592, 0.09230620414018631, 0.07866502553224564, 0.16764771938323975, 0.23753860592842102, 0.14721770584583282, 3.3853611946105957, -0.7840539216995239, -2.2318217754364014, 0.1386529803276062, 0.14166975021362305, -0.39414912462234497, -0.11180427670478821, -0.9562745690345764, 0.26362571120262146, -0.1887647658586502, 0.4706938862800598, 1.0751023292541504, 0.10122864693403244, 0.9620828628540039, 0.07546382397413254, 1.0860588550567627, -0.41887161135673523, -1.8165401220321655, 0.22874583303928375, -0.7960203886032104, -0.3575783669948578, 0.2127048373222351, 1.014135718345642, -1.2924829721450806, -2.6143414974212646, -0.8768422603607178, -1.8916550874710083, 0.06868909299373627, -0.7309470176696777, 1.2009344100952148, -0.6137076616287231, -0.05347692593932152, 0.14670147001743317, 0.4692250192165375, -1.8142163753509521, -2.0244946479797363, -0.1767810583114624, -1.2653284072875977, -0.5729783177375793, -0.39822831749916077, 0.9459620714187622, 0.17006772756576538, 0.10907701402902603, 0.23689593374729156, 0.37101927399635315, 0.6151217818260193, 0.9924896359443665, 0.30148637294769287, 0.2772817015647888, -0.07469122111797333, 0.1758720427751541, 0.49061065912246704, 0.1913258284330368, -0.7483543157577515, -0.02036488614976406, -0.247394859790802, -0.9482871890068054, 0.5354555249214172, -2.081955671310425, -1.2720534801483154, -0.22512474656105042, -0.3766246736049652, 0.12698249518871307, 0.14414365589618683, 2.33770751953125, 1.0510703325271606, 1.1687591075897217, 0.09236646443605423, -0.2266491800546646, -0.6046252846717834, 0.18746258318424225, 0.039919521659612656, 0.05244133621454239, 0.4880337417125702, 1.8909825086593628, -3.6392934322357178, 1.0108916759490967, 0.24687296152114868, 0.05164707079529762, 0.029515225440263748, 1.3821079730987549, 0.4734328091144562, -0.22866244614124298, 1.5363582372665405, 0.6847094297409058, -2.970047950744629, -3.192807197570801, -0.024694763123989105, 2.664990186691284, -0.10737655311822891, 1.4054099321365356, 0.1275942325592041, -0.3534267842769623, 0.2140260636806488, -1.784207820892334, 0.9058375358581543, 0.6165213584899902, 0.38955697417259216, 3.475278854370117, 3.3416972160339355, 0.21821768581867218, -1.194525122642517, 0.8288279175758362, 0.028663288801908493, 0.43917548656463623, 0.15718534588813782, -0.30116915702819824, 0.44592469930648804, 0.8086153864860535, 0.17626149952411652, 0.3648151159286499, -1.1409075260162354, -1.0413316488265991], [-0.008853001520037651, 0.651168942451477, 0.9023881554603577, -0.021700454875826836, 0.26307713985443115, 2.0127227306365967, -1.7692747116088867, -0.390489786863327, 0.1891251653432846, 0.06511098146438599, -0.2393398880958557, -0.2567393481731415, 0.2404537945985794, 0.0007100391085259616, -0.775074303150177, -0.12578223645687103, 0.35892605781555176, -2.6698296070098877, 1.9749906063079834, -0.16236622631549835, 0.4431605935096741, -0.12555406987667084, -1.651471495628357, -0.861440122127533, -0.2859237790107727, -0.4583263695240021, 1.9642959833145142, -0.49498629570007324, -0.10392053425312042, 0.3713344633579254, -0.19483549892902374, -5.225024700164795, -0.854856014251709, 0.20072084665298462, 0.4314076602458954, 0.3550571799278259, -0.422209769487381, 0.2889494299888611, 3.850400686264038, -0.26695212721824646, 1.1781647205352783, -0.5573780536651611, 1.4292232990264893, 2.062518358230591, 0.564694881439209, -0.5668846964836121, -0.6976749897003174, -1.3183317184448242, -0.028626950457692146, 0.0323038175702095, -0.5915406346321106, -1.8806642293930054, -0.47629573941230774, -0.19397194683551788, 0.5106736421585083, -0.6620638370513916, 1.386476755142212, 0.8385674357414246, -1.6812682151794434, -0.1960315853357315, -0.6386162042617798, 1.2014154195785522, -0.06099801883101463, -0.437779039144516, -0.03722039982676506, -0.15546733140945435, -0.07797642052173615, 0.32203909754753113, -1.1601232290267944, -0.8164411187171936, 0.5390896797180176, -0.9019996523857117, 1.1394363641738892, -0.9963511824607849, -0.278766930103302, -1.5748298168182373, 5.096522808074951, -1.5730992555618286, -0.11885765939950943, 0.18020711839199066, -0.33949071168899536, -0.4607538878917694, -1.2522855997085571, -1.2480807304382324, -0.21904020011425018, 0.47138047218322754, 1.5627778768539429, -0.07893244177103043, 0.007748891599476337, -0.32466235756874084, -0.15154534578323364, 1.9896292686462402, -0.29873475432395935, 1.0078279972076416, -0.46303999423980713, 0.21819661557674408, 5.488229274749756, -0.16618528962135315, -0.8113391399383545, -1.3823983669281006, -0.9218422174453735, 0.7335275411605835, -0.2651239037513733, -0.9752187132835388, -0.43171682953834534, 0.9226723909378052, -0.02081264555454254, 0.07701421529054642, 0.30153125524520874, -0.1431514024734497, -3.959768772125244, -0.2329542487859726, 0.43926048278808594, 0.9649115800857544, 0.4648350775241852, 0.9086310267448425, -0.013353263959288597, 0.9951154589653015, 0.2644801735877991, -0.01313199196010828, 0.9529566168785095, 0.03189083933830261, 0.22594863176345825, -0.1608983874320984, 0.6943472623825073, 0.08213146775960922, -1.1676790714263916, 0.4981967508792877, 0.1977742314338684], [-0.01737838052213192, -1.5657111406326294, -2.2169110774993896, 0.41302868723869324, 0.1021587997674942, -0.9208091497421265, 0.6261081099510193, -1.1826924085617065, 0.11935083568096161, 0.018692363053560257, 0.3829996883869171, -0.05017653852701187, 0.716492772102356, 0.29790595173835754, 0.15892016887664795, 0.1540895700454712, 0.17906047403812408, 0.36230894923210144, -2.2730536460876465, 0.0055512916296720505, 0.3998474180698395, -0.02370462752878666, -0.03963543847203255, -0.3872579336166382, 0.6175994873046875, 0.13734519481658936, -4.963521957397461, 0.11224152892827988, 0.13513512909412384, 0.9331912994384766, -0.0313798151910305, -0.038575559854507446, 0.08290891349315643, 0.46981605887413025, 0.8098734021186829, -1.0155837535858154, -0.11751164495944977, 0.3037399351596832, -0.8730396628379822, 0.3427630364894867, -0.6736608743667603, 0.24978074431419373, 1.6230597496032715, 0.4685268998146057, -1.468207597732544, 0.4023374617099762, -0.5563939213752747, 0.2592886984348297, 0.2039358764886856, 0.776445746421814, -0.9016879796981812, -1.8051846027374268, -0.4291512966156006, 0.3455066978931427, -0.0570528581738472, -0.1705038696527481, -0.5640457272529602, -0.2231021225452423, -0.32346031069755554, 0.03027302585542202, -0.4992765188217163, 0.2089536190032959, -0.7155709266662598, 0.33270442485809326, -0.0693473294377327, -0.1340315341949463, 0.368611216545105, 0.1059059202671051, 0.6354960203170776, 0.6588734984397888, 1.953748345375061, -0.18618515133857727, -0.04329422861337662, 0.16268415749073029, 0.3666188418865204, 0.42385929822921753, -5.643313884735107, 0.48939597606658936, -0.0605398453772068, 0.11147736757993698, 1.8262308835983276, 0.687110185623169, -0.4067121744155884, 0.847814679145813, -0.13276390731334686, 1.5230382680892944, -0.7984990477561951, -0.03413411229848862, 0.20872029662132263, -2.2144715785980225, -0.21755416691303253, 2.038339853286743, -1.793951153755188, 0.8216073513031006, 0.10523105412721634, -0.21152052283287048, 0.9391148090362549, -0.8571028709411621, 0.6197536587715149, 0.4663441479206085, 0.48764169216156006, 0.32122015953063965, -0.6570725440979004, -0.77621990442276, 0.6999301910400391, -0.13088180124759674, -1.194676399230957, 0.23348532617092133, 0.7339673638343811, -0.23336589336395264, -0.35943493247032166, -0.9276047348976135, -0.4510619640350342, 0.7419958114624023, 1.7336856126785278, 1.9162185192108154, 0.3991400897502899, 1.6129618883132935, -0.18199017643928528, 0.4803704619407654, 0.7508967518806458, 0.2420491874217987, -1.4610072374343872, 0.25588780641555786, 1.796493649482727, 0.17950233817100525, 0.1407945454120636, 0.8522791862487793, 1.5605393648147583], [0.31887635588645935, 2.206085205078125, 0.8544390797615051, 0.5677688717842102, 0.06282773613929749, -0.012937871739268303, -0.0003575877344701439, -2.202684164047241, 0.019540002569556236, -0.15597547590732574, 0.03595735505223274, 0.5747653841972351, 0.3245244026184082, 0.12162953615188599, 0.8741055130958557, 0.1375693678855896, -1.0117764472961426, -0.18445661664009094, 0.5275986790657043, 0.2566491663455963, 0.23957668244838715, -0.14410823583602905, -0.596323549747467, -0.8876707553863525, 0.06091541424393654, 0.8113771677017212, 1.0055187940597534, 0.7286314368247986, 0.41462403535842896, 0.419706255197525, 0.2380513846874237, 1.0550848245620728, -0.007631860673427582, -1.6372178792953491, -0.41627636551856995, -2.2615807056427, 0.5462167859077454, 0.36495253443717957, 0.8054074048995972, -0.7177568674087524, 0.3395618796348572, 0.649624764919281, 0.9503315091133118, 1.9824063777923584, 0.5262270569801331, 1.0517864227294922, 0.3056253492832184, -2.2231850624084473, 0.1564052402973175, 0.8240147829055786, -1.408825397491455, 3.3944523334503174, 1.0472445487976074, 1.3797458410263062, 0.21490097045898438, 0.49336302280426025, 0.6943262219429016, 0.223302960395813, 0.44528675079345703, 0.36698463559150696, -1.3527510166168213, 0.10702338814735413, -0.22025690972805023, -0.008971606381237507, 0.19122089445590973, -0.01921888068318367, -0.004307324532419443, 0.7136062383651733, -1.688520908355713, -3.095346212387085, 0.22427798807621002, -0.2803078591823578, 1.9139817953109741, 0.19761492311954498, 0.34486812353134155, 0.32197967171669006, -4.285214424133301, -1.243249535560608, 0.16177575290203094, 0.23429274559020996, 0.7068929076194763, 1.21743643283844, -1.1343727111816406, 0.5315037965774536, -0.1404566764831543, 0.9722038507461548, 0.434000700712204, 0.029452519491314888, -0.03191632032394409, 1.3163681030273438, -1.6631643772125244, -0.4590189754962921, 0.6632707118988037, -4.2505950927734375, 0.31299296021461487, 0.027785660699009895, -0.2893383502960205, 0.36523956060409546, 0.8253524899482727, -0.5603843331336975, -0.07582372426986694, -3.101745843887329, -0.08169697970151901, 0.03163281828165054, -1.2516146898269653, 0.0975944846868515, 1.3720577955245972, 0.10533169656991959, -0.08351103961467743, -0.12513375282287598, -0.6589874029159546, -0.7484992742538452, 0.16905680298805237, -0.18133805692195892, -0.5582586526870728, -2.2347662448883057, 0.10809619724750519, 0.36888861656188965, -0.17391906678676605, -0.13436488807201385, 0.6160638332366943, 0.1640799641609192, -1.3495235443115234, 0.22398720681667328, 1.3066399097442627, -0.15576158463954926, 0.6636393666267395, -0.39437833428382874, 0.24727967381477356], [0.04285073280334473, 0.8170807361602783, -0.010874356143176556, -0.2351328432559967, 0.34379449486732483, 1.0173791646957397, 2.296264886856079, 0.5280527472496033, 0.20554155111312866, 0.0548650287091732, 0.09373302012681961, 0.3933815062046051, 0.3676489293575287, 0.007152484729886055, -0.08707571774721146, 0.02482806332409382, -0.0866478681564331, -1.4287959337234497, -0.03783366456627846, 0.03169127553701401, 0.24055178463459015, 0.2557853162288666, 0.42850014567375183, -0.9237242937088013, 0.6899623870849609, 0.5663363337516785, 2.306549072265625, -1.1801425218582153, 0.2899208664894104, -0.12149085849523544, 0.24980531632900238, -1.5741852521896362, 0.5964950323104858, -0.11169394850730896, -0.4281857907772064, 1.6165515184402466, -0.18755096197128296, 0.11423720419406891, -0.8236639499664307, 0.4644915759563446, 0.9137674570083618, 0.73323655128479, 2.2879273891448975, 1.102086067199707, -1.0302560329437256, 0.337256520986557, 0.12134233117103577, -1.6041051149368286, 0.015764323994517326, -1.5289936065673828, 0.22994141280651093, -0.23645082116127014, -1.139855146408081, -0.463706374168396, 0.3184971809387207, 0.24250881373882294, -0.36718225479125977, 0.4821885824203491, 0.28149107098579407, 0.02533107064664364, 0.23727373778820038, -0.019942183047533035, 0.05825691297650337, 0.4878425896167755, -0.0070299929939210415, 0.04536791145801544, 0.05331749469041824, -0.312394917011261, -1.1620533466339111, 0.2177986204624176, 0.04655824601650238, -0.2302590161561966, -1.5389074087142944, 0.7398092150688171, 0.4096863567829132, -0.09435009956359863, -0.7455437779426575, 0.3420548141002655, -3.4289889754290925e-06, -0.10526115447282791, -0.9832078218460083, 0.7787157893180847, -2.6587307453155518, -0.3230469822883606, 0.21053138375282288, -0.07174965739250183, -0.04271603375673294, 0.09079595655202866, 0.009554482065141201, 0.284768670797348, -0.16033236682415009, -1.7337220907211304, 0.4309636652469635, 2.8590939044952393, 0.1702149212360382, -0.8469800353050232, -0.7375919222831726, 0.3042958378791809, -0.3103748857975006, -1.218857765197754, 0.3760077655315399, -0.10780293494462967, 0.4073394536972046, -0.8512381315231323, 1.2469524145126343, 0.08228005468845367, -0.5013337731361389, 0.0188332200050354, 0.3314796984195709, 0.1821518987417221, -1.067321538925171, -1.7708815336227417, -0.32810354232788086, 2.219647169113159, -1.3171474933624268, -1.5669807195663452, -0.0782105028629303, -0.34027764201164246, -1.0643789768218994, -0.03176458925008774, 0.3662368059158325, 0.07108157128095627, 0.0014704500790685415, 0.4272555708885193, -0.1553008258342743, 0.1989538073539734, -0.20866258442401886, 0.7571594715118408, 0.7014676332473755], [-0.08359628170728683, 0.8233811855316162, -0.10279063880443573, 0.6313287615776062, 0.10400387644767761, 3.598923444747925, -1.3647388219833374, -1.2378876209259033, -0.12892676889896393, 0.3818522095680237, -0.07734838128089905, -0.25166478753089905, -0.2825808823108673, -0.055802930146455765, 0.19956302642822266, -0.16023224592208862, 1.2624115943908691, -0.6748443245887756, -0.2559399902820587, -0.133006751537323, -0.17983290553092957, 0.0019990357104688883, -0.023525280877947807, 1.6487399339675903, 0.32648035883903503, -1.3478038311004639, -1.8462153673171997, 0.23472121357917786, -0.17621128261089325, -0.7210219502449036, 0.16058897972106934, -1.173746943473816, -0.2298424392938614, -0.3708727955818176, 0.5728998780250549, -2.8510098457336426, 0.2583637833595276, 0.20104707777500153, 1.0989731550216675, -0.808624267578125, -0.1822318732738495, -0.43340495228767395, 1.0544993877410889, 0.38264182209968567, -0.461103618144989, -0.23142054677009583, 1.001372218132019, 1.7603551149368286, 0.16642381250858307, 0.32331034541130066, 0.5428033471107483, -1.0837507247924805, -1.0901089906692505, 0.03654338791966438, -0.06789063662290573, -0.11255625635385513, -1.3707022666931152, -0.49905189871788025, 1.4619896411895752, 0.03782568871974945, -1.487563967704773, 0.3418009281158447, 0.12619903683662415, 0.14818961918354034, 0.19790856540203094, 0.1498311311006546, 0.01879534311592579, -0.4798549711704254, -0.8211534023284912, -1.0781573057174683, 0.6310951113700867, 0.979016125202179, 0.1924278885126114, -0.8769491910934448, 1.8187912702560425, -0.7004902362823486, 0.6231797337532043, 0.6787405014038086, -0.11389680206775665, 0.3113446831703186, 0.800807535648346, 0.45185813307762146, -1.2270528078079224, 0.1806327998638153, -0.5226184129714966, -1.654603123664856, 1.076173186302185, -0.013705992139875889, -0.1718091368675232, 0.26156726479530334, -0.6043503284454346, -3.0945212841033936, 0.09985484927892685, 1.8086814880371094, 0.35704272985458374, 0.47747132182121277, -2.673173427581787, -0.006700946483761072, -0.3443213403224945, 0.916020929813385, 1.1868019104003906, -0.42775577306747437, -2.2269515991210938, 1.6606825590133667, 1.4208475351333618, 0.9585559368133545, -0.34258005023002625, 0.08994405716657639, 0.26318445801734924, -0.09826607257127762, -0.9384973049163818, -1.429000973701477, 0.5221923589706421, 0.06204581260681152, 1.2666243314743042, 0.1131693571805954, -0.11630063503980637, -1.7371981143951416, -0.2742733955383301, 0.7136834859848022, 1.6168218851089478, 0.34375059604644775, 0.6416745781898499, -0.12706591188907623, 0.27312031388282776, 0.43584585189819336, -0.2693900167942047, 0.03131220117211342, 1.4077616930007935], [0.018828442320227623, 0.6019241809844971, 0.9795476794242859, -0.22880315780639648, 0.35708221793174744, -1.3395403623580933, -0.025172749534249306, 0.7240504622459412, 0.3031439185142517, 0.32648852467536926, 0.191958948969841, 0.2377437800168991, -0.32395485043525696, -0.0076858061365783215, 0.6372935175895691, 0.03553041070699692, -1.1664271354675293, -1.6298444271087646, 0.6863265037536621, -0.20684948563575745, -0.1149338036775589, 0.5287379622459412, -0.01983901672065258, 1.2316511869430542, 0.833225429058075, 1.5436701774597168, 0.03418486937880516, 0.11712323874235153, -0.006407354027032852, 2.0737173557281494, -0.02366805076599121, 1.3208024501800537, -1.253753423690796, 1.452758550643921, -0.13246648013591766, 6.4682793617248535, -1.2543926239013672, -0.09578461199998856, 0.7785738110542297, -1.056725025177002, -0.27676767110824585, 0.2288437783718109, -0.5142080187797546, -0.2278377115726471, -0.44710859656333923, -0.23015150427818298, 0.45509839057922363, 1.087437391281128, 0.012016808614134789, -1.0316811800003052, -2.149367332458496, 1.5364422798156738, 0.2494564950466156, -1.282984972000122, -0.32538023591041565, 0.20982778072357178, -0.9086307883262634, 0.1866879016160965, 0.6003803610801697, -0.027692846953868866, 0.6724787354469299, -0.15555402636528015, -0.7875609993934631, 0.013164299540221691, 0.05964981019496918, 0.23541896045207977, -0.10236724466085434, -0.16048812866210938, -0.01694299653172493, 0.14183112978935242, 0.5077453255653381, -0.2872813045978546, 0.21208839118480682, -1.185497522354126, -0.3077969551086426, 0.29927635192871094, -3.54929780960083, -1.390773057937622, -0.0014552101492881775, -0.09653674066066742, 2.7275753021240234, -0.12359380722045898, 0.9356311559677124, 0.573398232460022, -0.29236969351768494, 0.8081194162368774, 1.6729861497879028, 0.02711278386414051, -0.14644023776054382, 0.758868396282196, -0.9855694770812988, -0.01926429010927677, -0.22669881582260132, 4.178370952606201, 0.19423165917396545, 0.28990331292152405, -5.429425239562988, -0.331714004278183, -0.36299046874046326, -0.5395621657371521, -0.39128708839416504, -0.9043957591056824, 1.3744609355926514, -2.5303280353546143, -1.5076884031295776, -0.17594900727272034, 3.159273386001587, -0.010612044483423233, 0.3871247470378876, -0.012860987335443497, -2.576289415359497, -0.4070312976837158, -0.13704243302345276, -1.1843169927597046, -1.1254258155822754, 0.9157490730285645, -0.20191709697246552, -1.2680541276931763, 0.5951904058456421, 0.14533644914627075, -0.9387496709823608, -0.031232789158821106, -0.6833348870277405, 0.3180077373981476, 1.3174923658370972, -0.10655650496482849, -0.41165485978126526, -0.7855508923530579, -0.37180233001708984], [-0.10439757257699966, 0.9012082815170288, -1.530429720878601, 0.1271541267633438, 0.4278159737586975, 5.230792999267578, 0.9968238472938538, -0.1954897791147232, 0.11457612365484238, -0.09540478885173798, -0.10934668034315109, -0.21923178434371948, -0.7573977112770081, -0.15591295063495636, 0.11647513508796692, -0.1433367282152176, -2.175562620162964, -1.0646997690200806, -1.2416962385177612, -0.21056737005710602, 0.28871044516563416, 0.1579008251428604, -0.2962433993816376, 0.18351145088672638, 0.9241405725479126, 0.7811986804008484, 1.6474792957305908, 0.7328240871429443, -0.5914701223373413, 0.9788691401481628, -0.11010201275348663, -0.2307480424642563, -0.9800623655319214, -0.30067211389541626, 0.13311702013015747, -0.3613594174385071, -1.2647379636764526, 0.055051736533641815, 2.266357898712158, 0.6190101504325867, -0.644366443157196, -0.03167611360549927, 1.7828080654144287, -0.9254751205444336, -1.2282596826553345, 0.8491527438163757, -0.5276558995246887, 0.6886511445045471, -0.02196776494383812, 0.028817113488912582, 0.41466963291168213, -2.8214504718780518, -1.2677834033966064, 0.34810009598731995, 0.25521355867385864, -0.019272128120064735, -0.7650722861289978, -0.22996683418750763, -1.0127513408660889, -0.14913831651210785, 0.14353728294372559, -0.8021081686019897, 1.0633217096328735, -0.034998245537281036, -0.2827625274658203, -0.1627553105354309, -0.08750354498624802, 0.8887932896614075, -1.2930138111114502, 2.164069890975952, -0.24586308002471924, 1.0586529970169067, 2.1654138565063477, 0.4669433832168579, -0.6890238523483276, -0.16689375042915344, -3.2021803855895996, -1.801234483718872, -0.05681965500116348, -0.013940311968326569, 1.2912710905075073, 0.9801925420761108, 2.66249680519104, -0.10721634328365326, -0.6005957722663879, 0.15117891132831573, 0.2899775803089142, -0.09240370243787766, -0.05563032254576683, -0.6533836722373962, 0.3056848347187042, -0.7900349497795105, -0.3855545222759247, 1.881506323814392, -0.473896324634552, -0.03729378804564476, -3.9565436840057373, 0.3424641788005829, -0.6853377223014832, 1.0759351253509521, -0.36842814087867737, -0.7177553772926331, -1.8633239269256592, -0.3845859169960022, -3.012953996658325, 0.1595085859298706, -0.6157386898994446, -0.057633399963378906, -0.512691080570221, 0.11224223673343658, -0.40304437279701233, -0.03807550296187401, 0.33670416474342346, -0.7934316992759705, 3.824706554412842, -0.5953578352928162, -0.13994517922401428, 0.5593236088752747, 0.5531140565872192, -0.11729416996240616, 0.3264213800430298, -0.08578917384147644, 3.7073471546173096, 0.26278287172317505, 0.20051661133766174, 0.14892923831939697, 0.07766395062208176, -0.09976860135793686, 0.015379812568426132], [0.10374730080366135, -0.3329313397407532, 0.2848753035068512, -0.11965926736593246, -0.06339003145694733, -1.6825135946273804, 0.6451454758644104, 2.0120372772216797, 0.2079315334558487, 0.16006796061992645, 0.2718905806541443, -0.09093879908323288, 0.3812810778617859, 0.11136376857757568, 0.04034363105893135, 0.007357062306255102, 0.8669719099998474, -0.8444980382919312, -0.6156588792800903, 0.05618685856461525, -0.32358139753341675, -0.07588375359773636, 0.7069322466850281, 0.8040850758552551, 1.0882514715194702, -1.0441867113113403, -3.287306785583496, 0.3658895194530487, 0.12214093655347824, 2.8931782245635986, 0.05236109346151352, 2.975327491760254, 0.10466256737709045, 1.976750373840332, -0.340840607881546, -3.345033884048462, -0.08247820287942886, 0.015279268845915794, -1.5519800186157227, 0.4130155146121979, -0.22861796617507935, -1.0229038000106812, -0.4586438834667206, 1.4106069803237915, -0.8479364514350891, 1.7175767421722412, 1.2228360176086426, -0.9603871703147888, 0.052178312093019485, -0.25267866253852844, 3.7476613521575928, 2.0094234943389893, -0.46412551403045654, 0.20509928464889526, 0.1117912083864212, -0.23069846630096436, 0.13415367901325226, -0.24302417039871216, 0.5878589749336243, 0.1457332819700241, -0.6809279918670654, -1.4743614196777344, 0.1120268702507019, -0.27659323811531067, 0.03967633098363876, -0.10470490902662277, 0.04910803958773613, -0.29861515760421753, -0.47118547558784485, -1.8584908246994019, -1.1920233964920044, -0.30968907475471497, -0.4347423315048218, 0.11029046028852463, -2.5410964488983154, 0.1920107901096344, -1.2332381010055542, 1.1110458374023438, 0.09464763104915619, -0.06204305961728096, 3.999692916870117, -0.8369084596633911, -2.9877917766571045, 0.3852371871471405, -0.08715417236089706, 0.3261982798576355, 0.6823087930679321, 0.14926908910274506, 0.2008162885904312, -0.47511425614356995, -1.3107080459594727, 0.1868264526128769, 0.4189930260181427, -0.9518678188323975, -0.09524936974048615, -0.1380390077829361, 2.8042092323303223, 0.49615734815597534, 1.3164879083633423, 1.4075613021850586, 0.12997181713581085, -0.39314645528793335, 1.2355536222457886, -1.2447447776794434, -3.695324182510376, 1.8262474536895752, -0.23504763841629028, 0.08619150519371033, -0.6693488955497742, -0.08251501619815826, -1.6368547677993774, 0.5466148257255554, 0.2559773325920105, -0.6967968940734863, -1.9343388080596924, 0.3285624384880066, 0.01957719214260578, 1.7424052953720093, -0.0985654965043068, 0.2534637153148651, 0.620911180973053, 0.08675134927034378, -1.3714841604232788, 0.017332496121525764, 0.06176668405532837, 0.018885312601923943, 0.5620024800300598, -1.3420506715774536, -0.1561126559972763], [0.01899721659719944, -0.37983572483062744, -0.4767351746559143, 0.16520968079566956, 0.6491696238517761, -1.1292798519134521, -0.862929105758667, 0.14277097582817078, -0.3603551983833313, -0.16402876377105713, -0.014077057130634785, -0.052194830030202866, 0.05298743396997452, -0.28399595618247986, 1.4529006481170654, -0.205768883228302, -2.1810855865478516, 1.314314603805542, -0.06333383917808533, -0.30160820484161377, -0.16166992485523224, -0.13153032958507538, -0.8405904173851013, -0.48674535751342773, -0.8542793989181519, 0.14682213962078094, -3.524824619293213, 0.652981162071228, -0.6605083346366882, 1.0994361639022827, -0.26241278648376465, 2.8021304607391357, -0.4144217073917389, 1.3397576808929443, -0.35177597403526306, 0.027000095695257187, -0.5070472359657288, -0.09743024408817291, 0.6243820786476135, -1.9767531156539917, 0.6766857504844666, -0.5204972624778748, 1.101406455039978, -1.9128527641296387, 0.5668472647666931, -0.13600201904773712, -0.1961355209350586, 1.2925913333892822, -0.14182737469673157, 0.5217185616493225, 4.227072238922119, 2.0286026000976562, -0.9833338856697083, -0.19962750375270844, -0.4458421766757965, 0.10309851169586182, -0.1066698208451271, 0.3749505877494812, 0.7410107254981995, -0.28347641229629517, -0.12789523601531982, -1.4204806089401245, 0.35868847370147705, -0.09657300263643265, 0.1392945647239685, -0.19922496378421783, -0.25616225600242615, 0.15101300179958344, -0.3963738679885864, 1.2324308156967163, 1.2910363674163818, 0.33678367733955383, -1.6822713613510132, 2.399249315261841, -2.473243236541748, 0.6971054077148438, 1.6008788347244263, -0.6824660301208496, -0.18811242282390594, -0.3645039498806, 1.2114286422729492, -0.7362061738967896, -0.6785412430763245, -0.6587095856666565, -0.35248368978500366, -0.4543793797492981, 1.1388375759124756, -0.16921523213386536, -0.0758022591471672, -2.9684014320373535, -1.8573780059814453, -0.9179519414901733, -0.000743997166864574, 2.171509265899658, -0.2535545825958252, 0.07618029415607452, 1.4033633470535278, 0.894787073135376, 0.22886240482330322, 0.3512078821659088, -1.2389179468154907, -1.437886357307434, 0.6427348852157593, -0.5593573451042175, -1.7942547798156738, 0.03052622824907303, -0.046604596078395844, -0.10596879571676254, -0.742419421672821, -0.15881727635860443, 2.10317063331604, 0.119615837931633, -0.15175095200538635, 0.4363032877445221, -0.18641698360443115, 2.915670156478882, -0.47017329931259155, 2.2466073036193848, -0.28806018829345703, -0.2835289239883423, -1.3475018739700317, -0.14933961629867554, 0.2911939322948456, 0.6302935481071472, -0.7493679523468018, -0.1681886613368988, 0.0777892991900444, 1.2188975811004639, 0.04240855202078819], [-0.07926690578460693, -1.2247899770736694, -0.3956975042819977, -0.2621757686138153, -0.20609821379184723, 0.9124937057495117, -0.4170236587524414, 1.5864135026931763, 0.16868577897548676, 0.05106763541698456, 0.0247568991035223, -0.0018552351975813508, -0.48978957533836365, -0.06367513537406921, -0.7017340660095215, -0.1823655366897583, 0.6779444217681885, -2.4649531841278076, -2.8687331676483154, -0.25611549615859985, 0.28747889399528503, -0.02845168486237526, 0.2914372980594635, 0.6042492985725403, 0.6043128371238708, -0.17850597202777863, -0.7271114587783813, 1.0963512659072876, 0.27039584517478943, 0.8236899375915527, -0.15137603878974915, 1.0661194324493408, 0.04970645159482956, -0.2639915645122528, -0.32433605194091797, 1.002861499786377, 0.9336718320846558, 0.05357910320162773, 1.228270173072815, -0.5296739935874939, 0.6092802882194519, 0.08033342659473419, -2.499131202697754, 1.3300789594650269, -1.369680404663086, -0.4595288038253784, 0.24491676688194275, 0.4862457811832428, -0.02506231889128685, -0.08742868155241013, 0.7154491543769836, 4.6415228843688965, 0.08219482004642487, -0.5138702988624573, -0.3136636018753052, -0.14107070863246918, 1.4751482009887695, -0.7007191777229309, 0.34943026304244995, -0.21112491190433502, -0.1628725826740265, 1.6138454675674438, -0.03191617131233215, 0.3553487956523895, -0.06979883462190628, -0.11092817783355713, -0.10759767889976501, -0.038729481399059296, -0.8238638043403625, 0.001587561098858714, 0.9142599701881409, -0.8612961173057556, -1.849278211593628, -0.7827911972999573, 0.6052398681640625, -0.6389967799186707, 1.9933500289916992, -0.9819329977035522, -0.10251551121473312, -0.04989749938249588, 0.8329364061355591, 0.031169431284070015, -0.12153967469930649, -0.5058732032775879, 0.20438271760940552, 0.5253620147705078, 0.9138326048851013, -0.10218898206949234, -0.1318797469139099, -1.399786114692688, -1.4521024227142334, 0.5883644819259644, -1.0425268411636353, -2.552783250808716, -0.16734366118907928, -0.6899268627166748, 1.4298831224441528, 0.4274977445602417, -0.290597140789032, 0.41698572039604187, 1.0535061359405518, 1.3117488622665405, 1.376880407333374, -0.6142106056213379, -1.5097516775131226, -3.7666101455688477, 0.425055593252182, -0.07827825099229813, -0.1366894394159317, -0.15391188859939575, 0.517542839050293, 2.449138879776001, 0.16340766847133636, -0.7874341607093811, -0.3692050278186798, 2.7004432678222656, 0.30454620718955994, 0.08368773013353348, 1.0793107748031616, 0.36623629927635193, -0.6653178334236145, -0.06468808650970459, -0.9820331335067749, -0.11024694889783859, -0.24887150526046753, -0.2896669805049896, -0.15944014489650726, -0.1338070034980774, -1.1555110216140747], [-0.06878090649843216, -0.36542779207229614, 0.4341524541378021, -0.1489301472902298, 0.4518781304359436, 1.1108659505844116, -0.21003025770187378, -0.36972787976264954, -0.038424164056777954, -0.14263363182544708, -0.03736638277769089, -0.20012715458869934, -0.031633518636226654, 0.21810586750507355, 0.2851024866104126, -0.07788980007171631, 0.6885054707527161, 0.17448188364505768, 1.3940881490707397, -0.047380268573760986, -0.8320233821868896, 0.12940071523189545, 0.878459095954895, 0.36959636211395264, -0.7129680514335632, 0.14141084253787994, -0.21593622863292694, 0.8578322529792786, 0.6403408646583557, 1.0582793951034546, -0.06089089810848236, -1.2643173933029175, 0.2797931730747223, -0.21744732558727264, 0.3575918972492218, -0.48691919445991516, -0.5114912390708923, 0.5695436000823975, 0.3084743022918701, 0.48858076333999634, 0.5172653198242188, 0.6697211861610413, -0.0073942444287240505, -2.1766464710235596, 0.09021274745464325, 0.5347647666931152, -1.1112819910049438, -2.067690372467041, -0.14148801565170288, -0.3005291819572449, -1.5038070678710938, 0.3392797112464905, 1.498867392539978, -0.1675546020269394, -0.1470068395137787, 0.5297964215278625, 0.10378824919462204, 0.7311477065086365, -0.21199020743370056, 0.019186044111847878, 0.9580753445625305, 0.21553897857666016, 0.5703146457672119, -0.16259440779685974, -0.12808138132095337, 0.3884355127811432, 0.29258033633232117, 0.5345585346221924, -0.04951413348317146, -0.5875779986381531, -0.8653926253318787, 0.37884485721588135, -1.9180469512939453, -1.786278486251831, -1.638681173324585, 0.3043866455554962, -5.9944939613342285, -0.0701790377497673, -0.010894815437495708, 0.07387329638004303, 2.8336803913116455, 0.11468496918678284, -0.8889702558517456, -0.4622219204902649, 0.34472423791885376, -0.1673639714717865, 0.3932710289955139, 0.014814866706728935, -0.13597175478935242, 0.0032851649448275566, -0.18922235071659088, 1.5761123895645142, 1.385080337524414, -1.8469173908233643, -0.16838695108890533, 0.5874801278114319, 1.1053926944732666, 0.24413198232650757, -0.03252359852194786, -0.9383725523948669, -0.26651349663734436, -0.5825647115707397, -1.2325031757354736, 0.790538489818573, 1.6993002891540527, 0.6373632550239563, 0.720450222492218, 0.051072485744953156, 0.21627438068389893, 0.030418986454606056, 0.8718319535255432, 0.03212083503603935, -0.7412211894989014, -0.5331074595451355, 2.253992795944214, 1.0702250003814697, 0.2590281367301941, -0.30696648359298706, -0.27911752462387085, -0.13623596727848053, 0.11856216192245483, -0.01818811520934105, -0.6543433666229248, -0.0882052332162857, 0.14433932304382324, 0.03079622983932495, 0.099625363945961, -1.5831363201141357, 1.123803734779358], [0.0697319433093071, -0.6193585395812988, -0.06401322782039642, 0.3938555121421814, -0.3270011246204376, -0.49465784430503845, -0.13612604141235352, 0.9499638080596924, 0.15619425475597382, 0.14615999162197113, -0.2816639542579651, 0.4727720618247986, 0.4191490709781647, -0.11261168867349625, 0.8848894834518433, -0.2855077087879181, -0.5136967301368713, 0.017623770982027054, -0.2292996197938919, -0.2267986238002777, -0.19891658425331116, 0.5795546174049377, 0.5630411505699158, 1.1639442443847656, 0.9900281429290771, 0.15176527202129364, 1.1675664186477661, 0.2606423497200012, -1.0705722570419312, 1.1123701333999634, 0.0601506307721138, -2.645035743713379, -0.44691699743270874, -0.2183835357427597, -1.0804119110107422, -2.924901247024536, 0.8005198836326599, 0.2676871418952942, -1.523422360420227, -0.11022236198186874, 1.32765793800354, -0.3557957410812378, 1.4845362901687622, -1.829493522644043, 1.2470831871032715, -0.7060309648513794, -0.6679642200469971, -0.5923486948013306, 0.03446633368730545, 0.21118351817131042, -0.43394947052001953, -1.9028602838516235, 1.6386477947235107, 2.285914182662964, 0.4380504786968231, 0.8421850204467773, -0.6313210129737854, -0.3934788703918457, 0.2722964584827423, -0.16806334257125854, -0.5983502268791199, 0.32924774289131165, 0.30285564064979553, -0.026425445452332497, -0.2690543532371521, -0.16903549432754517, -0.06160286068916321, -0.4166642129421234, -0.8522325158119202, 0.655642569065094, -2.4324893951416016, -0.19951693713665009, 0.4461188018321991, -1.0251991748809814, 2.013350009918213, 0.119888074696064, 2.1619722843170166, -1.3299493789672852, -0.018255895003676414, 0.004774675238877535, 1.426938533782959, 1.4519623517990112, -3.3922998905181885, -0.07522617280483246, -0.11597362160682678, 0.26020902395248413, -0.6908984184265137, -0.05670715123414993, 0.11982730031013489, -0.9149057269096375, 1.0321369171142578, 1.0300813913345337, 0.7182097434997559, -1.0108392238616943, 0.014494664035737514, 0.5900336503982544, -2.014169216156006, -0.010645939037203789, -0.44991350173950195, 1.2650244235992432, -0.39425376057624817, -2.0002360343933105, 1.85189688205719, -1.0713250637054443, 4.590750217437744, 1.1027370691299438, 0.9347047209739685, 0.03366946056485176, -0.6577056050300598, -0.2622387409210205, 0.7452477812767029, -0.017339644953608513, -0.8248512148857117, 1.3041671514511108, -0.6986098885536194, -0.849993884563446, -0.10078322887420654, -0.4515785574913025, -0.3692750334739685, -0.0362468920648098, 0.5200614333152771, 0.07321863621473312, -1.3440698385238647, 0.19383609294891357, 0.7687667012214661, -0.09194763004779816, -0.16358935832977295, -0.7452444434165955, 0.9290131330490112], [-0.02157217636704445, -1.5638329982757568, 0.3626631200313568, 0.44637104868888855, 0.1212206780910492, 2.540574550628662, -0.47488340735435486, 1.4567590951919556, 0.33116233348846436, -0.11182118207216263, 0.2350597381591797, 0.13781924545764923, 0.10714371502399445, 0.13935406506061554, -0.22646364569664001, 0.2810017168521881, -0.12956713140010834, -1.512778878211975, -0.116940937936306, 0.07075901329517365, 0.13193358480930328, 0.43708115816116333, -0.1989566832780838, 1.8075876235961914, 0.19607403874397278, -0.17584964632987976, 0.8143616318702698, -1.2242751121520996, -0.027121080085635185, 1.3295382261276245, -0.08246031403541565, -2.7808570861816406, 0.3112953305244446, 1.4117748737335205, 0.831180989742279, -2.0949273109436035, 0.6169148087501526, -0.30792108178138733, 3.0491409301757812, -0.7452843189239502, 0.5604604482650757, -0.3372494876384735, 2.9639642238616943, -1.2350269556045532, -1.1103205680847168, -0.2981428802013397, -0.16620491445064545, -0.5162111520767212, 0.033619463443756104, -0.27671822905540466, 1.1613184213638306, 0.7960224151611328, -0.5551595091819763, -1.462503433227539, 0.28212758898735046, 0.2579819858074188, 1.7985291481018066, 0.4291878342628479, 0.34041041135787964, 0.07248611748218536, -0.29490914940834045, -0.6208694577217102, 1.2996701002120972, -0.04923044145107269, -0.23496201634407043, -0.029217418283224106, 0.09290996938943863, -0.21675609052181244, -0.22027219831943512, -2.254485845565796, 1.4356383085250854, -0.7780121564865112, 1.9913464784622192, -1.4072802066802979, 0.36906418204307556, 0.25986817479133606, -0.35121238231658936, -0.545926034450531, -0.00214810436591506, -0.11932314187288284, 0.6904816627502441, 0.5317174196243286, -0.4893325865268707, -0.3361396789550781, 0.07069169729948044, -1.1465508937835693, -1.036084532737732, 0.06166018545627594, 0.039277054369449615, 2.6356024742126465, -2.070974826812744, -0.1554892510175705, 0.6503314971923828, 2.4328315258026123, 0.05581015348434448, 1.0413706302642822, -4.410695552825928, -0.02977302297949791, 1.001401424407959, -0.27931275963783264, 1.7026598453521729, -2.2604053020477295, -1.2722828388214111, -0.6775302290916443, -1.5124481916427612, -0.40161994099617004, 1.0315876007080078, 0.023857660591602325, -0.5388005971908569, -0.001509272027760744, -1.404732584953308, 0.07064170390367508, -0.19062833487987518, 0.37402915954589844, 0.1444096565246582, 0.865268349647522, -0.11207985132932663, -0.3796065151691437, -0.9795448184013367, 0.05643627420067787, -0.7165637016296387, 0.03925185650587082, -1.0112919807434082, 0.3076667785644531, 0.09841791540384293, -0.022690387442708015, 0.029137317091226578, 0.6006461977958679, 0.44717976450920105], [0.5331909656524658, -0.6447071433067322, 0.40318527817726135, 0.10130322724580765, -0.2563728988170624, 5.1375532150268555, 0.556210458278656, -1.3654223680496216, 0.34464868903160095, 0.28564074635505676, -0.08797526359558105, -0.47986140847206116, 0.9378122091293335, 0.2418385148048401, -0.9813202619552612, -0.026106882840394974, -0.08495271950960159, 0.5703777074813843, 1.0204945802688599, 0.1967640370130539, 0.007626065984368324, 0.03576048091053963, -0.2682269811630249, 0.5201194286346436, -0.1916799545288086, -0.11090810596942902, 1.005367398262024, -0.4908668100833893, -0.41680076718330383, -0.8044843673706055, 0.13722163438796997, -1.3214647769927979, 0.39292386174201965, 1.3978995084762573, 0.007088971324265003, -2.4628281593322754, 1.1873832941055298, 0.041761986911296844, 1.3897435665130615, 0.4742279648780823, 0.7923092246055603, 0.35616421699523926, 1.8380876779556274, -1.4563729763031006, -1.3603304624557495, -1.6587547063827515, 0.08848929405212402, 0.007368811871856451, 0.13749194145202637, -0.4110875427722931, 0.8995772004127502, 0.41571998596191406, -0.5043528079986572, -0.00022869152599014342, -0.1505759358406067, 0.11813589185476303, -0.608240008354187, -0.34581130743026733, -0.9741323590278625, -0.14987020194530487, -0.6670619249343872, -1.052492618560791, 0.8812575340270996, 0.17853917181491852, 0.29617032408714294, 0.13055624067783356, 0.13255730271339417, 0.4471081793308258, -0.49477043747901917, -1.9443033933639526, -2.4838485717773438, 0.3963330388069153, 0.5538288354873657, 0.11305874586105347, -1.2316631078720093, -0.4409027099609375, -4.007214069366455, 1.1357791423797607, 0.07936830818653107, -0.17891855537891388, -1.554806113243103, -0.48675382137298584, -1.0540423393249512, 0.8778093457221985, 0.047199200838804245, -0.6268578767776489, 0.7924959659576416, 0.0997447520494461, 0.10331766307353973, 0.03770580515265465, -1.5921440124511719, 0.4253219962120056, -0.11676784604787827, -0.8992390632629395, -0.1434759944677353, -0.10100608319044113, 5.454082489013672, 0.29729941487312317, -0.6522913575172424, -0.06906275451183319, 0.4398908019065857, 1.5356301069259644, 1.5019582509994507, 0.3574025630950928, -3.124199151992798, -1.0288630723953247, 0.21028733253479004, 0.0790410041809082, 0.16300755739212036, -0.1656818985939026, 0.06064349785447121, 0.4910888373851776, 0.5070945024490356, -0.45133715867996216, -0.23836049437522888, 1.3520623445510864, -0.1535119265317917, -0.3077557384967804, 0.3554932177066803, -0.3018737733364105, -0.16631323099136353, 0.0702868402004242, 1.6659263372421265, 0.10318531841039658, -1.067931056022644, 0.11556964367628098, 0.16248244047164917, 0.4768354594707489, 0.7102779746055603], [0.04787243530154228, 0.48704686760902405, -0.2112840861082077, -0.24493622779846191, 0.11476849764585495, 5.898085117340088, -0.11277229338884354, -0.055998917669057846, -0.12642621994018555, -0.16550590097904205, -0.22134381532669067, 0.30826133489608765, 0.27869340777397156, -0.08661573380231857, 0.6431539058685303, -0.031928133219480515, -1.4352370500564575, -2.5072906017303467, 0.3463011384010315, 0.16926254332065582, -0.4506538212299347, 0.1450863480567932, -0.8106467127799988, -2.783128023147583, 0.595896303653717, -0.29323044419288635, 3.0683839321136475, 0.09989789873361588, -0.8445011973381042, 1.2672592401504517, 0.23169110715389252, 1.2270036935806274, -1.1773356199264526, 1.077422022819519, -0.07303877919912338, -0.37195390462875366, 0.22178617119789124, -0.16848815977573395, 4.133424282073975, -1.5645064115524292, -0.14202547073364258, -0.5215990543365479, 2.47110915184021, -1.4170262813568115, -0.6325457096099854, -0.7863853573799133, -0.00954090990126133, 0.1700972020626068, -0.05384489521384239, -1.031632423400879, -0.39728137850761414, 0.18544146418571472, -0.22110015153884888, 0.49812471866607666, 0.07323548197746277, -0.3027355670928955, -0.200151726603508, -0.04985341057181358, 0.2344832420349121, -0.09989511221647263, 0.7995694875717163, -1.228623628616333, 0.20859260857105255, -0.15574760735034943, -0.18916329741477966, -0.3276287019252777, -0.15283629298210144, -0.2947028577327728, 1.2409451007843018, 1.1008403301239014, -0.06556187570095062, -0.2112882435321808, 0.09307002276182175, -3.923487424850464, 1.0150378942489624, -0.2708030641078949, -1.4168295860290527, -0.5725256204605103, -0.057583969086408615, 0.3875141143798828, 6.312264442443848, 1.6432344913482666, -0.5349057912826538, -0.05856809765100479, -0.48629188537597656, -0.29923126101493835, 0.3493836224079132, 0.08658980578184128, -0.14035142958164215, -0.6553012728691101, -0.7436988353729248, 0.4405210018157959, 0.8867663741111755, -4.4135026931762695, -0.015622933395206928, -0.6720712184906006, -0.19785457849502563, -0.3168089687824249, -0.49789735674858093, 0.4488520622253418, -0.11062242835760117, -0.33179908990859985, -1.6958577632904053, 0.33295702934265137, -0.26390567421913147, 1.4360466003417969, -2.167982578277588, -0.027238549664616585, -0.19462469220161438, 0.39119789004325867, 0.45334064960479736, -0.7846170663833618, -0.2718701660633087, 1.211190938949585, 1.2564417123794556, -1.6936886310577393, -0.5607016682624817, -0.10017897933721542, -0.43570825457572937, 0.023218747228384018, 1.0793159008026123, -0.20390167832374573, 0.0539139024913311, -0.021485229954123497, 0.3894237279891968, -0.2525840997695923, 0.26305726170539856, 0.51670902967453, 0.20431162416934967], [0.049048952758312225, -1.98673677444458, -0.6491953730583191, -0.027230892330408096, 0.3738291561603546, -0.6746649146080017, 1.162874698638916, -4.64877986907959, -0.12098168581724167, -0.32464104890823364, -0.15120871365070343, 0.31755372881889343, -1.1477514505386353, 0.04749399423599243, -0.20924343168735504, -0.12435543537139893, 5.226273536682129, 0.001009344239719212, -0.919286847114563, -0.15074795484542847, -0.5083242654800415, 0.15412965416908264, 0.37568432092666626, 1.6296947002410889, 0.3267134428024292, -0.21037642657756805, -2.308791399002075, -1.2379043102264404, -0.8154920935630798, -0.25585928559303284, 0.051905032247304916, -1.4139081239700317, 0.22917456924915314, -2.1327719688415527, 0.08565716445446014, 2.426945686340332, -0.3855292499065399, -0.04122132062911987, -2.3327407836914062, -0.37868067622184753, -2.220838785171509, -0.5405188202857971, -0.09348144382238388, 0.7281575798988342, -0.7823954224586487, -0.22346638143062592, 0.2563144564628601, 0.2520374357700348, 0.06696585565805435, 0.07941605895757675, -0.08577042073011398, -0.22479507327079773, -1.4509025812149048, 1.028070330619812, -0.19073180854320526, 0.7221746444702148, 0.934979259967804, 0.09776174277067184, 1.0929129123687744, -0.1161193996667862, 0.34080174565315247, -0.47443485260009766, 0.567315399646759, -0.07625821232795715, -0.013513908721506596, -0.5946230888366699, -0.11534491926431656, 0.8027315139770508, 0.3285616636276245, 0.6140924692153931, -0.16078586876392365, 0.2204156517982483, 0.24210003018379211, 0.9492537975311279, 4.789865970611572, 0.6553748250007629, -1.9507277011871338, -0.006297420244663954, -0.07437139004468918, 0.033710841089487076, 2.6692755222320557, 0.18897831439971924, -2.160027503967285, -0.2671270966529846, -0.00887609738856554, -0.20061984658241272, -0.18757760524749756, -0.05936400964856148, 0.2900523543357849, -1.85693359375, 1.2367217540740967, 1.160649299621582, 2.270268201828003, 0.8164419531822205, -0.2109134942293167, 0.32160574197769165, -0.6834445595741272, 0.7862702012062073, -1.4838180541992188, 0.8211963176727295, 1.8411738872528076, 0.13483025133609772, 1.4429703950881958, 0.9048264622688293, 1.517471432685852, 1.733987808227539, 0.7945795655250549, -0.2848922610282898, 0.38293352723121643, -0.37333956360816956, -1.397377610206604, -0.6609863042831421, -0.08767881989479065, 1.0462771654129028, -2.1274142265319824, -0.6358461380004883, -0.1766234040260315, -1.52809739112854, 0.9986476302146912, -0.57054203748703, -0.08234662562608719, -0.11973434686660767, -0.6711045503616333, 0.18657732009887695, -0.12146756052970886, -0.33070704340934753, 0.13802024722099304, 0.265690416097641, -0.3717864453792572], [-0.15725992619991302, -2.455923318862915, 0.11531781405210495, 0.36949580907821655, 0.4819040298461914, -1.7590372562408447, -0.5864117741584778, -0.35074350237846375, -0.03838489204645157, 0.11638100445270538, -0.0010813382687047124, -0.19535991549491882, -0.1908324956893921, -0.2418416440486908, -0.33870363235473633, -0.28614264726638794, 1.3728504180908203, 0.8145691156387329, -0.9686296582221985, -0.10414251685142517, 0.27385473251342773, -0.27504029870033264, -0.5345712304115295, 1.5409512519836426, 0.015888117253780365, -0.7043424844741821, -0.10336411744356155, 0.510134756565094, 0.08075184375047684, -0.5628542900085449, -0.03960642218589783, 1.3070402145385742, 1.8054758310317993, 1.1986379623413086, -0.09033109247684479, 2.029939889907837, -0.23988306522369385, 0.24803641438484192, -1.9928631782531738, -0.7701200246810913, -1.286858320236206, 0.5233444571495056, 0.6588724851608276, 0.8354142308235168, 0.39562010765075684, 0.5061135292053223, -1.1999139785766602, -1.5493255853652954, -0.37217551469802856, 0.7196940779685974, 1.0013072490692139, -0.11773046851158142, 1.1528400182724, 0.7842621803283691, -0.04549536854028702, 0.3181888163089752, 1.3967370986938477, -0.029316304251551628, 0.5287182331085205, -0.21126686036586761, -0.03585692122578621, 0.9016236066818237, -0.07646902650594711, -0.5103750228881836, -0.2333444058895111, -0.47656679153442383, -0.19389210641384125, -0.9655441045761108, -0.03603431582450867, 1.1152241230010986, -0.13530059158802032, 0.5956128835678101, 0.2677058279514313, -1.6828157901763916, 3.2255547046661377, -0.07966527342796326, 0.019347311928868294, -0.4872951805591583, -0.33297935128211975, -0.0865531787276268, -3.0287342071533203, 1.190437912940979, -0.5069401264190674, -1.0231108665466309, 0.1993413269519806, -0.044176649302244186, -0.20798258483409882, -0.21185638010501862, -0.13090656697750092, 0.4999343156814575, 0.06732403486967087, -0.13210634887218475, -0.49165505170822144, -6.794243335723877, -0.3374103605747223, -0.5670549869537354, -1.1342393159866333, 0.8936658501625061, -0.43254855275154114, 0.11599189788103104, 0.8290515542030334, 1.6430920362472534, 0.278123140335083, 0.28878337144851685, 2.509108781814575, -0.5755702257156372, 1.1789852380752563, -0.23917411267757416, -0.8964953422546387, -0.3461763560771942, -0.32856184244155884, 4.103075981140137, 0.07874849438667297, 0.06541959941387177, 1.52712881565094, -1.5864609479904175, -0.41709962487220764, 1.0591518878936768, -0.3942677676677704, -0.6052811741828918, 1.0662363767623901, -0.2464943528175354, 0.833436906337738, -0.2502596378326416, -1.0078043937683105, -0.10380329936742783, 0.015017518773674965, -0.16029973328113556, -0.3365057110786438], [-0.15115998685359955, -1.427630066871643, -0.4500628709793091, -0.23045574128627777, -0.5908588171005249, -5.5654988288879395, -0.1109083816409111, -0.403051495552063, 0.2264164388179779, 0.31518030166625977, 0.1345577985048294, -0.23375721275806427, -0.44833701848983765, 0.15670715272426605, 0.7985360026359558, -0.0698360949754715, 0.08842691034078598, -1.363946557044983, -1.5461479425430298, 0.13339701294898987, 0.3578914999961853, 0.448787659406662, -0.6575430631637573, 2.598727226257324, 0.6877123117446899, -0.5277925133705139, 1.5100436210632324, 1.5866678953170776, 1.8866815567016602, 1.3503998517990112, -0.030360447242856026, 0.881598711013794, 0.7490460276603699, -1.6157952547073364, -0.1484491527080536, 1.4108753204345703, 0.5699982047080994, 0.2945396304130554, -2.9990763664245605, 1.7975013256072998, 0.25808417797088623, -0.553605318069458, 1.8729252815246582, 0.4949185252189636, -0.24817371368408203, 0.08785763382911682, 0.3826526999473572, 1.5508038997650146, 0.019639430567622185, 1.5762176513671875, 2.626692056655884, -3.479966640472412, -0.8168089389801025, 0.027913236990571022, 0.012175205163657665, -0.7551546096801758, -0.7415941953659058, -0.11625026911497116, -0.5791686773300171, 0.11578529328107834, -0.6323036551475525, 0.8603032231330872, -0.48603591322898865, -0.39036574959754944, -0.23159624636173248, 0.2809436321258545, 0.022282693535089493, 0.39529937505722046, -1.3811814785003662, -1.3172857761383057, -0.7993928790092468, 1.1528741121292114, 1.182970404624939, 0.7890737652778625, -3.8441247940063477, 0.25797706842422485, 7.73614501953125, -0.5187629461288452, 0.10011989623308182, -0.2214997112751007, -4.129222393035889, -0.03164410591125488, 0.7842161059379578, 0.25715017318725586, 0.3950541913509369, 2.139631748199463, 0.38068240880966187, 0.0012926653726026416, -0.010361922904849052, -2.9096152782440186, 0.22717881202697754, 2.0267138481140137, -1.2560290098190308, -2.034217119216919, 0.3026079535484314, -0.256227046251297, -0.9393094778060913, 0.40544894337654114, -0.5826030373573303, 1.2350101470947266, 1.2289234399795532, 0.7530502676963806, 0.7836247086524963, -0.31932398676872253, 0.27842065691947937, 0.3552953898906708, -0.01042021345347166, 0.06189009174704552, 0.7250033020973206, -0.055389512330293655, -3.600482940673828, -1.6734941005706787, 0.23824630677700043, -1.5497102737426758, 1.8900469541549683, -0.6266117095947266, 0.20029745995998383, -0.40223294496536255, 0.41848552227020264, -0.1522124707698822, 0.7556842565536499, 0.02547430992126465, -0.19977526366710663, 0.16285917162895203, 0.36438509821891785, 0.06297788769006729, 0.028329074382781982, -1.0567526817321777, 1.2042475938796997], [-0.0663655549287796, -0.5173996090888977, -1.921581506729126, 0.16644032299518585, -0.017899489030241966, -0.04490112140774727, 3.732809066772461, 0.9996799230575562, -0.06264006346464157, 0.13287466764450073, 0.00800169724971056, -0.37292176485061646, -0.2681790590286255, 0.03209704905748367, -0.24481436610221863, 0.0744362622499466, 1.8240599632263184, -0.03997097909450531, -0.1849149465560913, 0.05466119199991226, 0.37654829025268555, -0.34280234575271606, 0.4201858639717102, 0.7377809882164001, 0.2520139217376709, 0.6020935773849487, 0.2239627242088318, 0.33866265416145325, 0.05551578477025032, -0.47725147008895874, 0.13058044016361237, -2.1333494186401367, -0.6631156206130981, 2.365752935409546, 0.4064571261405945, 0.05822928249835968, -0.1393214911222458, 0.1822480410337448, -2.6118409633636475, 0.6949602961540222, -1.0122947692871094, -0.054378170520067215, 1.9689534902572632, -0.5044837594032288, 1.0413742065429688, 0.21967443823814392, -0.5023317337036133, -0.14800944924354553, 0.03293178975582123, -0.4164668023586273, 1.4345521926879883, -0.0042390902526676655, -0.6042749881744385, 0.5443253517150879, -0.32072463631629944, -0.03877194598317146, 0.644127607345581, -0.07984452694654465, 0.4895687997341156, -0.045929405838251114, 0.7261548638343811, -0.19846387207508087, -1.3863319158554077, 0.09326677769422531, -0.2709580063819885, 0.22557863593101501, 0.017511451616883278, 1.4595736265182495, -1.5621956586837769, 0.9901867508888245, -1.9000977277755737, 1.3933054208755493, -0.8924223184585571, 0.9339641332626343, 0.7868220806121826, -0.37054213881492615, -0.5469282865524292, 0.03440142050385475, -0.04514987766742706, 0.2839767634868622, -0.15764671564102173, -1.4387824535369873, -0.16989417374134064, -0.725963294506073, -0.19724908471107483, -0.38689422607421875, -1.499221920967102, 0.032101746648550034, 0.15239544212818146, 0.6144447922706604, 2.5102739334106445, 1.8302829265594482, 0.8743929266929626, -0.730861485004425, 0.2611249089241028, -0.29205068945884705, 2.170128345489502, 0.4800463914871216, -1.1765191555023193, -2.2146918773651123, -0.41766923666000366, 2.052999496459961, -0.5593902468681335, 0.5821628570556641, -1.3058496713638306, 1.7737224102020264, 1.3804278373718262, -0.02353193238377571, -0.3026634752750397, -0.12038613855838776, 0.8005653619766235, 1.0359975099563599, -0.0668407678604126, -0.2899516224861145, -0.055467795580625534, -2.636152982711792, 0.11377014219760895, -0.7661877870559692, -0.9324208498001099, -0.029504671692848206, -2.316621780395508, 0.020327867940068245, -1.9298045635223389, 0.05445937439799309, -1.146763563156128, -0.15199415385723114, -0.39946091175079346, -1.6978272199630737, 1.2224469184875488], [-0.27158355712890625, 0.25440841913223267, -0.9068015217781067, -0.07057114690542221, 0.31072646379470825, 1.9448951482772827, -0.6660444140434265, 0.1985662281513214, -0.018475323915481567, -0.07215797156095505, 0.10686672478914261, -0.3607328236103058, -0.5195266008377075, 0.191108837723732, -0.13687846064567566, 0.09182561188936234, 0.8707687854766846, 1.019713044166565, 1.9524543285369873, -0.02013348415493965, 0.2426760345697403, -0.7286818623542786, 0.16084463894367218, 2.852412223815918, -0.26933905482292175, -0.4605660140514374, -1.371842384338379, -0.2849400043487549, -0.23051871359348297, 1.3874993324279785, -0.26340943574905396, 4.109212398529053, -0.6284108757972717, -1.113106369972229, 0.4357806444168091, 1.0262809991836548, 1.123856782913208, 0.1938471645116806, -0.8287633657455444, -0.7629349231719971, -0.07861785590648651, -0.474440336227417, 1.2043414115905762, 0.2762572467327118, 0.7280369997024536, 1.1574721336364746, 1.5277601480484009, -1.1513794660568237, -0.046023376286029816, -0.45722901821136475, -0.732283353805542, -0.07603544741868973, -0.0024887591134756804, -0.3838222324848175, 0.08848271518945694, 0.3790796101093292, 0.4363834857940674, -0.6584073305130005, -0.3463055193424225, -0.34480732679367065, -1.6842732429504395, -1.2637745141983032, -0.9599035978317261, -0.3216877281665802, 0.04780300334095955, 0.18416255712509155, -0.15705327689647675, 0.4434235095977783, 0.554606020450592, 0.5151313543319702, -3.1297268867492676, -0.678468644618988, -0.8990116119384766, -0.22792598605155945, -0.10207969695329666, 0.15538637340068817, -1.1682326793670654, 0.8753065466880798, -0.06410450488328934, -0.06953202188014984, -1.0803776979446411, -0.4114425778388977, -3.002558469772339, -0.39390820264816284, 0.2430706024169922, -0.02944720722734928, 0.6665186285972595, 0.1273614913225174, -0.04001503437757492, 1.7392948865890503, -1.024788498878479, 1.7486196756362915, -0.6021366119384766, 1.3590458631515503, -0.14422988891601562, -0.45223936438560486, 2.1415657997131348, 0.24164552986621857, -0.7861689925193787, -0.7326369285583496, 0.1952279955148697, 2.072967052459717, 0.9744855165481567, -0.11712246388196945, -1.500436544418335, -1.2350693941116333, -0.2576117515563965, -0.01208840124309063, 0.3930306136608124, 0.035337451845407486, 1.4817935228347778, 1.2491127252578735, 0.09548317641019821, -0.2630244195461273, -1.5548043251037598, -2.2402219772338867, -0.3435518145561218, -0.0186524149030447, -0.4192725121974945, 0.18902069330215454, -1.554841160774231, 0.09509127587080002, 1.4935283660888672, 0.09111089259386063, -0.15714916586875916, 0.08072496205568314, -0.27523669600486755, 0.19550347328186035, 0.15514837205410004], [-0.07031287252902985, 0.045889921486377716, 0.1911705732345581, -0.22946414351463318, -0.6213052868843079, -2.4480185508728027, -0.4955423176288605, -0.6041989326477051, 0.02668921649456024, -0.24562720954418182, 0.017043041065335274, -0.482360303401947, -0.4954071044921875, 0.026791507378220558, -0.8419163227081299, -0.22197262942790985, -1.074396014213562, -1.6209429502487183, 0.27558282017707825, -0.11397577077150345, -0.5735523700714111, -0.09742037206888199, -0.7195150852203369, -0.11018065363168716, 0.22945427894592285, -0.2842250466346741, 0.9747326374053955, -1.5679161548614502, -0.16924521327018738, 0.14065174758434296, -0.22789296507835388, -1.3596445322036743, -0.33830544352531433, 1.1952745914459229, -0.9554697871208191, 0.07394956797361374, 0.66823410987854, 0.08798026293516159, 2.6195156574249268, 1.4242969751358032, -0.08299281448125839, -0.04747873917222023, 0.9371619820594788, 0.5648910403251648, 0.4836901128292084, 0.2050577700138092, 1.466671347618103, 0.21967452764511108, -0.1741853505373001, 1.7364460229873657, 0.5264523029327393, 1.5140637159347534, -1.09432852268219, -0.3503674268722534, 0.32476404309272766, -0.08575636893510818, 0.00879750493913889, 0.3792256712913513, 1.587533950805664, -0.21219022572040558, -0.9661117196083069, -1.7927465438842773, -0.24926984310150146, 0.42423897981643677, 0.2142389863729477, -0.2339802384376526, -0.22278843820095062, -0.40430453419685364, 0.43908077478408813, -0.6805708408355713, -0.35730260610580444, 1.1186654567718506, -0.5570641160011292, 2.803849458694458, -1.588187575340271, -0.6494705677032471, -3.2783401012420654, -0.24063166975975037, -0.11537852883338928, -0.11221136152744293, -2.6282389163970947, -1.0288512706756592, 4.2280683517456055, 0.8248925805091858, 0.196165069937706, -0.34739992022514343, -0.6915847659111023, -0.14949972927570343, -0.06361135840415955, 0.7421876192092896, 0.40445592999458313, -1.8357304334640503, -1.5962474346160889, -2.4627325534820557, -0.2199328988790512, 1.0539265871047974, 4.417496681213379, -0.3515409231185913, 0.95106440782547, 0.26993605494499207, -0.2277563214302063, 1.2582050561904907, -2.5031776428222656, -1.090861201286316, -3.443331480026245, 1.4960280656814575, 1.9394054412841797, -0.1770947277545929, -0.995141327381134, 0.13515813648700714, 1.3718589544296265, 2.5352888107299805, 0.38108429312705994, -0.3768629729747772, 1.2188137769699097, 2.3089513778686523, -0.030856478959321976, -0.8663246631622314, 1.6352593898773193, -0.2857750952243805, -0.928910493850708, -0.15212443470954895, 4.138150691986084, -0.7713091969490051, -0.6814484596252441, -0.02141227386891842, -0.2645014226436615, -0.10834898799657822, -0.6930840015411377], [-0.24957548081874847, -0.7021312117576599, -0.4812960922718048, -0.45108386874198914, 0.003827817505225539, -0.24618731439113617, 0.2955237329006195, 1.797974944114685, 0.3274880349636078, -0.024335060268640518, -0.37206584215164185, -0.39054661989212036, -0.7198900580406189, -0.07674892991781235, 0.2894584536552429, 0.1761241853237152, -1.9112281799316406, 0.005746728274971247, -0.34820127487182617, -0.06644297391176224, 0.8219789862632751, -0.41373613476753235, 0.9728963971138, 0.20925070345401764, -0.30453166365623474, 0.6784562468528748, -0.9276901483535767, -0.3428463339805603, 0.04370487853884697, 1.0520309209823608, -0.1785215437412262, -2.3995959758758545, -0.05175498127937317, 1.0669718980789185, -0.13656958937644958, -4.6600446701049805, 0.49374353885650635, 0.060150858014822006, -0.7452894449234009, -0.5575781464576721, 0.04638002812862396, 0.3506154716014862, -1.8868818283081055, 0.8717283606529236, 1.5060371160507202, -0.421604186296463, 0.3622663915157318, 0.2299041748046875, -0.14889930188655853, -1.2099717855453491, -3.2491772174835205, -2.9929702281951904, -0.8232380747795105, -0.9529547095298767, 0.49692288041114807, 0.13079728186130524, -2.406107187271118, -0.513015627861023, -0.18110673129558563, -0.1109858825802803, 0.12581323087215424, 1.0174270868301392, -0.9571020007133484, 0.609713077545166, -0.2782702147960663, -0.18370415270328522, 0.03909473493695259, 0.3353158235549927, -1.196494460105896, 1.1073319911956787, -0.5164722204208374, 0.39787009358406067, 2.08201003074646, -0.11769455671310425, 0.9425973892211914, 0.14130274951457977, 2.4823832511901855, 0.23761218786239624, -0.013425094075500965, -0.1324150711297989, -1.6610437631607056, 0.8864697217941284, 1.416446566581726, 0.1701006442308426, 0.10723947733640671, 0.021577369421720505, -1.4665181636810303, -0.26818546652793884, 0.1160726323723793, 0.7795391082763672, 1.0840362310409546, 3.398061752319336, -0.7747924327850342, 3.5280141830444336, 0.10868314653635025, -0.17841145396232605, -1.539196491241455, 0.14068491756916046, 0.14126592874526978, 0.3069009482860565, -0.47070378065109253, -0.3487689197063446, -0.08710191398859024, 0.6694846749305725, 1.609418272972107, -0.7790285348892212, -0.40553605556488037, -0.16735069453716278, 0.15551134943962097, -0.12825745344161987, 1.6029351949691772, 0.5545305609703064, 0.3226344585418701, 1.5733253955841064, 0.6846660375595093, -1.0393728017807007, -0.058382488787174225, 0.3414827883243561, 1.2270770072937012, 0.20524382591247559, -0.12329497933387756, -0.18147993087768555, 1.587918758392334, 0.01804046705365181, -0.8470130562782288, 0.21057219803333282, -0.32544031739234924, 0.45276015996932983, 0.6711151599884033], [-0.33046913146972656, -0.6751675605773926, 0.2897074520587921, 0.848642885684967, 0.24647949635982513, -0.92704176902771, -0.3868409991264343, 1.4493870735168457, -0.3117542862892151, 0.11416158825159073, 0.25098496675491333, 0.42970651388168335, 0.9237728714942932, 0.2287520319223404, 0.14348344504833221, -0.0919661670923233, -1.516706943511963, -0.2912733554840088, 3.2753024101257324, -0.0748094841837883, 0.29400548338890076, 0.6801902651786804, 0.21443337202072144, -2.3484811782836914, -0.6694182753562927, 0.025012550875544548, 2.7553515434265137, 0.5363712310791016, -0.6407803297042847, 1.9842969179153442, -0.08367783576250076, 2.2240920066833496, -0.1378983110189438, 1.534577488899231, -0.39851659536361694, -0.5556352734565735, -1.255388855934143, -0.3770700991153717, -4.984570503234863, -0.33335956931114197, -0.5836265087127686, 0.8526816368103027, -1.0774585008621216, -1.785247564315796, -0.5552840828895569, 0.2360255867242813, 1.4822182655334473, -0.2583247721195221, -0.09488826245069504, -1.4682873487472534, -1.0917316675186157, -0.1451701670885086, 0.43279942870140076, 0.2525253891944885, 0.1310354471206665, -0.1613902598619461, -1.7785078287124634, 0.5754333138465881, -1.607231616973877, -0.15859954059123993, -0.22447168827056885, 0.8138037919998169, -0.12597833573818207, -0.09487943351268768, -0.13122886419296265, 0.017241915687918663, -0.0329001247882843, 0.37166982889175415, -0.8604367971420288, -1.7938092947006226, -2.486737012863159, 0.3462575078010559, 0.38646575808525085, -0.20217719674110413, 3.2692489624023438, 0.30912521481513977, 1.9647856950759888, -0.22851218283176422, -0.060681913048028946, 0.08818040043115616, 4.5297980308532715, -1.0649374723434448, 1.1635997295379639, 0.45202648639678955, -0.11672395467758179, -1.401664137840271, 1.5572001934051514, -0.10055924206972122, 0.11155607551336288, 0.49405279755592346, -2.945237874984741, 0.12296649068593979, 0.36804717779159546, -3.6578872203826904, -0.3620792329311371, -0.29711809754371643, 3.62507963180542, 0.9358311295509338, -0.7249822020530701, 3.3274447917938232, -0.31989556550979614, -0.1694738268852234, -1.0316519737243652, 0.08799802511930466, -2.1067497730255127, 0.21088233590126038, 0.5891051888465881, -0.05157698690891266, 0.506872832775116, 0.1023171916604042, 0.31662434339523315, 0.6550112962722778, -0.26497775316238403, 0.5294656157493591, -0.5675031542778015, -0.09283844381570816, 0.0777478888630867, -1.1838629245758057, 0.3137146234512329, -0.38095831871032715, -0.3722013831138611, -0.1033346951007843, 0.5996710658073425, -0.1592274010181427, 0.041370026767253876, 0.07163724303245544, 0.5234816074371338, -0.657541811466217, 0.23965904116630554], [-0.04074466601014137, -0.022965209558606148, 0.5409882068634033, -0.15429803729057312, 0.1789914071559906, 3.3111069202423096, 1.1750112771987915, -0.1566656082868576, -0.10685689002275467, 0.0014489858876913786, 0.14480799436569214, 0.6154371500015259, 0.6658300757408142, 0.018945490941405296, -0.713729202747345, 0.10736189037561417, 0.2544441819190979, 0.594723641872406, 0.6132434010505676, 0.016232788562774658, -0.3517880141735077, 0.7111960649490356, -0.5921401977539062, -0.007365118712186813, 0.33638831973075867, -0.1019783467054367, -1.5509878396987915, 0.5275760293006897, -0.6281065940856934, -2.8789076805114746, 0.25363683700561523, -0.8230447769165039, 0.3314211964607239, 0.3615788519382477, -0.47474929690361023, -2.5652050971984863, 0.6734845638275146, 0.017130790278315544, 1.092528223991394, 1.0447897911071777, -2.70581316947937, 0.07758070528507233, -0.437034547328949, 1.3331859111785889, -0.39390498399734497, 1.2368804216384888, -0.02280515804886818, 1.158602237701416, 0.012167243286967278, 0.5515391826629639, -0.46054205298423767, 1.6983636617660522, -0.1369895488023758, -1.0365216732025146, 0.2208021879196167, -0.025327999144792557, 1.100298285484314, -0.03197063133120537, -0.4823184907436371, 0.11986600607633591, -1.7055505514144897, -2.0821566581726074, -1.4402145147323608, 0.046333231031894684, 0.03532535210251808, 0.6948208212852478, 0.2576414942741394, 0.2522527575492859, 0.4872280955314636, -0.9909053444862366, 0.7871408462524414, -0.7683940529823303, -0.4365456700325012, -0.8226194977760315, 1.5321522951126099, 0.14876556396484375, -0.8505609035491943, -1.2553520202636719, -0.1162579357624054, 0.0014863621909171343, 2.3973352909088135, 0.8599458336830139, -2.0469300746917725, -0.27971041202545166, -0.08652429282665253, 0.005741910543292761, 0.9095735549926758, 0.08669119328260422, 0.1854717880487442, 0.18160273134708405, -3.745462656021118, 1.6718488931655884, -0.036630287766456604, 2.6119253635406494, 0.2232600301504135, -0.05426977202296257, -0.7490147352218628, 0.5693811774253845, -0.5480436682701111, -2.4262514114379883, -0.8239637613296509, 1.5849835872650146, -1.3963905572891235, 0.593920886516571, 2.2518019676208496, 1.1680442094802856, -1.5452969074249268, -0.05601857602596283, -0.12131793797016144, 0.2833392322063446, 0.6082017421722412, 0.37807485461235046, 0.10174226760864258, 0.2243749350309372, 1.4323550462722778, -0.24181438982486725, 0.1367531716823578, 1.0311949253082275, 1.465129017829895, -0.3534170091152191, -1.1614998579025269, 0.2654989957809448, -1.8438266515731812, -0.07816570997238159, -0.5102357864379883, 0.0721607506275177, -0.4866704046726227, -0.16133494675159454, -0.51698237657547], [0.10270462185144424, 0.1648174375295639, 0.6597670316696167, -0.2202390432357788, 0.06633369624614716, -0.3800217807292938, 0.3638356626033783, -0.8407556414604187, 0.29800939559936523, 0.17932875454425812, 0.22926095128059387, 0.23329584300518036, -0.9613171815872192, 0.18106676638126373, -0.7828989624977112, -0.08556649833917618, -2.258079767227173, 1.046751618385315, -0.8549844026565552, 0.13023993372917175, -0.3491261303424835, 0.17134422063827515, 0.49563324451446533, 0.3972293734550476, -0.8299375772476196, 1.002971887588501, 1.9119478464126587, 0.041057709604501724, 0.4780919551849365, 2.2296252250671387, -0.010996532626450062, 2.351557493209839, -0.13882803916931152, -0.8167668581008911, 0.27750954031944275, -2.7370293140411377, 0.39213278889656067, 0.06197763979434967, -1.1616084575653076, -0.09943602234125137, -0.5874858498573303, 0.35758036375045776, -1.1443830728530884, 2.862841844558716, -0.9639242887496948, 0.2509772777557373, -0.5008484721183777, -1.0443860292434692, 0.3343520164489746, -0.026320619508624077, -3.38970947265625, -2.036869525909424, 0.8025004267692566, 0.6935027241706848, -0.2197287380695343, -0.1609751284122467, -0.46503162384033203, -0.4633012115955353, 1.6648646593093872, 0.14247895777225494, 0.582644522190094, 0.4517054557800293, 0.8895398378372192, -0.10576439648866653, 0.29572680592536926, 0.17133012413978577, 0.21938776969909668, 0.15182049572467804, -0.06419257819652557, -2.3291614055633545, -2.019706964492798, 0.246120423078537, -0.8455138206481934, -0.5626439452171326, 2.4974350929260254, 0.09341946989297867, -3.025963068008423, 0.25235122442245483, 0.24217484891414642, 0.2579464614391327, -0.8102595806121826, 0.38395845890045166, -4.182840824127197, -0.41191366314888, 0.29166293144226074, 0.9760212302207947, -0.7814162969589233, 0.27111563086509705, 0.12742358446121216, 2.676532030105591, -0.24212810397148132, -1.6947226524353027, -1.0560269355773926, -0.034217022359371185, 0.017797239124774933, -0.081448495388031, 0.8030580878257751, -0.13448520004749298, -0.13847880065441132, 0.6431823372840881, -0.11222243309020996, 0.6931605935096741, 1.5933005809783936, 1.6192386150360107, -0.4751397669315338, 0.26117390394210815, -0.6370015740394592, 0.17818401753902435, -0.17102746665477753, 0.42817896604537964, -3.3471901416778564, 3.682190179824829, 0.7732896208763123, -0.6079585552215576, -0.36549484729766846, 2.7577364444732666, 0.20417186617851257, 1.3524196147918701, 0.14918304979801178, -0.1688770055770874, 0.77554851770401, 0.22035184502601624, 0.6567922234535217, 0.696968138217926, -0.7981516718864441, 0.15662027895450592, 0.042820051312446594, 0.7498131394386292, -1.3745813369750977], [-0.03504663705825806, 0.5618079304695129, -0.2843456268310547, 0.13320249319076538, 0.0959688276052475, 2.1911473274230957, -0.38090822100639343, -0.1765459179878235, -0.1427329182624817, -0.24038788676261902, 0.007703887764364481, -0.26198792457580566, 0.08145176619291306, -0.04201553761959076, -1.057780146598816, -0.38819485902786255, 1.4526259899139404, -0.39619338512420654, 0.39718958735466003, 0.018530918285250664, 0.09753188490867615, -0.09970756620168686, 0.6805931925773621, -0.3913101553916931, -0.9474334120750427, -0.40608879923820496, -1.6409318447113037, 0.17104467749595642, 0.43144920468330383, -1.6585419178009033, -0.15414637327194214, 1.696353554725647, 0.7242189645767212, -1.9670650959014893, 0.2904004156589508, -0.6844767928123474, -0.2929784059524536, 0.37971773743629456, 1.397234559059143, -0.8236666917800903, 0.10427023470401764, -0.7052855491638184, 2.7594194412231445, -0.18952490389347076, -0.3822559714317322, -0.7917112708091736, -0.19496755301952362, 1.9250504970550537, -0.005054605193436146, -0.8781962990760803, -0.681302547454834, 0.6210569739341736, -0.12304479628801346, -0.4002531170845032, -0.2976120710372925, -1.0484097003936768, -0.5777713656425476, -0.2900203764438629, 0.08680436760187149, -0.11316373199224472, -0.1868956983089447, -0.3585855960845947, -0.6965316534042358, 0.4814057946205139, 0.06919282674789429, -0.288766086101532, 0.061777930706739426, 0.2908095121383667, 0.9560976624488831, 0.6696457862854004, 1.8407245874404907, -0.5982611775398254, 0.5081382989883423, 0.0417727455496788, 3.794031858444214, -0.426321417093277, 3.0677595138549805, -1.0958540439605713, -0.20849908888339996, -0.08534612506628036, 2.328277587890625, -0.9827059507369995, 1.554386854171753, 0.27466800808906555, -0.15353544056415558, 0.5584047436714172, -1.1676149368286133, 0.0902269035577774, -0.20959582924842834, -0.27201852202415466, -0.07833528518676758, 0.2444811761379242, -1.2034862041473389, -2.309384346008301, 0.008627473376691341, -0.24868105351924896, 1.2052197456359863, -0.054550059139728546, -0.19924721121788025, -0.0548548549413681, -1.0923161506652832, 0.7360280752182007, -2.6893539428710938, 0.45446014404296875, -0.47789299488067627, 0.02253374271094799, 1.058334469795227, -0.20379279553890228, 0.23584824800491333, -0.44162407517433167, -1.17364501953125, -1.3165286779403687, -0.2829751968383789, -1.1420847177505493, 1.8767517805099487, -0.878768801689148, 0.020315397530794144, -0.1478046327829361, 0.8986849188804626, 0.0030239594634622335, -0.3698842525482178, 0.1959621161222458, -0.7533724308013916, -0.5350939631462097, 0.2023298144340515, -0.20384402573108673, -0.11810331046581268, 1.0721399784088135, 0.7303789854049683], [0.1296287477016449, -0.8213108777999878, -1.609808087348938, 0.4358203709125519, 0.9869047999382019, -1.3694343566894531, -1.3107949495315552, -0.43606865406036377, 0.12393058091402054, 0.467984676361084, 0.10947257280349731, 0.2987286448478699, 0.3046407401561737, 0.2502231001853943, 0.31659501791000366, 0.13533194363117218, 1.2291579246520996, -0.4009852409362793, 1.4861353635787964, 0.0336143895983696, 0.21866029500961304, 0.6651813387870789, 0.3080019950866699, 0.09832529723644257, 1.6290580034255981, 0.5359014272689819, -1.7746624946594238, 0.560174286365509, -0.1608283370733261, -1.8648293018341064, 0.09115826338529587, -3.2664060592651367, 0.10562919080257416, 0.40809381008148193, -0.6648594737052917, 1.4524736404418945, -0.5447374582290649, 0.07490544021129608, -0.8402799367904663, 0.5394641160964966, 1.3111976385116577, -0.9930965900421143, -1.432924747467041, -1.3111385107040405, 1.149013876914978, -0.3792583644390106, -0.01896561309695244, 0.7986770272254944, 0.12102783471345901, 0.5732575058937073, 0.8095898628234863, -2.234454870223999, 0.030992472544312477, -0.035215750336647034, 0.48749470710754395, -0.15615886449813843, -0.0499286949634552, 1.2398457527160645, -0.7115755677223206, -0.010742386803030968, 1.0366506576538086, 0.4039342701435089, -0.6984527707099915, 0.0055961012840271, 0.2940739691257477, -0.14844399690628052, 0.14058543741703033, 0.5812426209449768, -1.980364441871643, 0.686894953250885, 1.3864284753799438, -0.18720710277557373, 0.36935126781463623, 0.2153952270746231, 2.3467395305633545, -0.41421160101890564, -4.832309722900391, -0.4376285672187805, 0.11902253329753876, -0.17482250928878784, 2.3355093002319336, 1.349135398864746, 0.2799210846424103, 0.14395305514335632, -0.14108015596866608, 0.05404059588909149, 0.8941207528114319, 0.0765799880027771, -0.026312943547964096, 0.9116902351379395, -1.347792387008667, 3.398998498916626, 0.5885252952575684, 1.5130908489227295, 0.32417750358581543, -0.26147976517677307, 0.8676935434341431, -0.3626779615879059, 1.7049269676208496, -0.4873974025249481, 0.5669481754302979, -1.279579758644104, -0.7693847417831421, 0.9665427803993225, -1.0055533647537231, 0.39714688062667847, 0.3691793382167816, 0.11066997051239014, 0.6593864560127258, 0.09057549387216568, -0.7329395413398743, 0.8466424345970154, -0.3117084801197052, -1.9553500413894653, -1.8130935430526733, -0.3305148482322693, -0.018877573311328888, -0.2840737998485565, -0.2693863809108734, 0.11811590939760208, -2.612704038619995, 0.1039755642414093, -2.271332025527954, 0.3247447907924652, 0.2736145853996277, 0.12114838510751724, 0.47254639863967896, 0.18588122725486755, 0.4014556407928467], [-0.15716756880283356, -0.02006683126091957, 0.8427692651748657, -0.14788109064102173, -0.2686671018600464, -0.8893380761146545, 0.8826059699058533, 2.2383644580841064, 0.09178472310304642, -0.3302558958530426, 0.017913907766342163, 0.04642060026526451, 0.622339129447937, 0.22474955022335052, -0.2134707272052765, -0.08075527846813202, -1.5071885585784912, 0.10917099565267563, 1.2433897256851196, -0.03104853443801403, 0.020593710243701935, -0.05780559405684471, 0.061061643064022064, 0.6477189660072327, 0.7758669853210449, 0.4233684241771698, 1.1932947635650635, 0.006865830160677433, 0.5555155873298645, -0.8579598069190979, -0.13022524118423462, 2.800600051879883, 0.410201758146286, -0.6132562756538391, 1.558982014656067, 2.015343427658081, -0.5510202050209045, 0.23971526324748993, 2.2397818565368652, -0.20176534354686737, -0.08976015448570251, -0.17386870086193085, 1.5951958894729614, -0.4665546417236328, 0.8292369246482849, 0.7190241813659668, 0.8070052266120911, -1.1040525436401367, -0.1429782658815384, 0.09838390350341797, 0.8660832047462463, -0.8224740624427795, 0.55971360206604, 0.7961517572402954, -0.21330079436302185, -0.5613990426063538, -1.1033669710159302, -0.40652310848236084, 0.5037751197814941, 0.023567913100123405, 0.16970975697040558, 0.9711010456085205, -0.689873456954956, -0.5204901099205017, -0.10327935963869095, 0.1933298259973526, -0.14269384741783142, 0.2785988748073578, -0.0038810311816632748, -0.7220432162284851, 1.7181318998336792, 0.2428579479455948, 0.21267734467983246, -0.5041719079017639, -1.2026169300079346, -0.8102443218231201, 0.21037769317626953, -0.5090392827987671, -0.10777677595615387, -0.21869365870952606, -2.364410638809204, -0.4458031952381134, -0.5333402752876282, 0.009403768926858902, 0.00448519317433238, -0.1778627187013626, -1.4914225339889526, 0.02976219542324543, -0.08239494264125824, -0.9746015667915344, 0.7952393889427185, -0.9854543209075928, -0.618060827255249, -2.775178909301758, -0.2661457657814026, -0.21658815443515778, 0.4365513324737549, -0.7426719069480896, -1.2001596689224243, 0.3861337900161743, -1.5970925092697144, 0.41406917572021484, 1.020798683166504, -0.5823804140090942, 0.8846191167831421, -0.9786949753761292, 1.2976350784301758, 0.03948039561510086, -0.1963174194097519, -0.23980247974395752, 1.5227584838867188, 1.8628652095794678, -0.35512328147888184, -0.9770777821540833, -1.4375460147857666, 0.1637810617685318, 0.03182777762413025, 0.601524829864502, 0.09411054104566574, 0.0427279956638813, -2.1183300018310547, -0.05632598698139191, 0.5521291494369507, -0.5763550400733948, -0.010526210069656372, 0.10982465744018555, -0.2548660337924957, -1.886493444442749, 0.45491260290145874], [0.16673707962036133, -0.3494274914264679, 0.3417389690876007, -0.41700688004493713, -0.5404308438301086, -4.429222106933594, 0.03624233230948448, -0.15317198634147644, -0.11166439950466156, -0.29125678539276123, -0.11302374303340912, 0.40484899282455444, 0.9617515206336975, 0.04052289202809334, 0.06919350475072861, -0.030143000185489655, -1.3346514701843262, 0.9004654288291931, 0.7654846906661987, -0.28201425075531006, -0.025728758424520493, -0.1958106905221939, -0.3769267499446869, -0.12663541734218597, -0.17704685032367706, -0.41175511479377747, -2.3413918018341064, -1.1409180164337158, 0.19130797684192657, 1.0293340682983398, -0.14354269206523895, 1.502547025680542, 1.3117073774337769, -0.15380029380321503, -0.5544167160987854, -2.4010684490203857, -0.22640350461006165, 0.5747047066688538, -1.9974604845046997, 2.163255453109741, 1.1653938293457031, -0.34836986660957336, 0.3023912310600281, 1.4279534816741943, 0.6742902994155884, 0.2861570715904236, -0.3436155617237091, -0.587825357913971, -0.24865517020225525, 0.7334278225898743, -2.2615957260131836, -0.6776034832000732, -1.0215872526168823, -0.15932762622833252, -0.263860285282135, -0.16653282940387726, -0.1181376576423645, -0.23840035498142242, 0.08480360358953476, -0.0894835963845253, -0.0011824257671833038, -0.8126792907714844, -0.20043683052062988, -0.32098647952079773, 0.00785133894532919, 0.377382755279541, -0.1368432492017746, -0.21383748948574066, 0.14373047649860382, 1.344214916229248, 2.029022693634033, -0.30809760093688965, -0.24160224199295044, -0.7017326951026917, 3.11204195022583, 0.2469850331544876, 0.6436740756034851, 1.0413719415664673, 0.0741121768951416, -0.20413045585155487, 1.3679956197738647, 0.8899684548377991, -1.4977755546569824, -0.1897609531879425, 0.22000329196453094, -0.5457504987716675, -0.41830217838287354, 0.12283939123153687, -0.18645767867565155, -0.8760559558868408, 2.56329345703125, 0.3206312656402588, -0.07998232543468475, 1.3014475107192993, -0.1404573619365692, -0.2723756730556488, -3.9094667434692383, -0.36678561568260193, 0.13780635595321655, 0.4218815863132477, -0.5158619284629822, 0.14158694446086884, 1.9365423917770386, -0.3749990165233612, -2.1117305755615234, 2.2255818843841553, -0.2940824627876282, -0.015456097200512886, 0.07252301275730133, -0.1488901525735855, 2.8929903507232666, 0.7624396681785583, -0.1476208120584488, -0.05099094659090042, 2.4688234329223633, -0.6320428848266602, -0.0018101396271958947, -0.7793634533882141, -0.29298946261405945, -0.1634531021118164, -0.7032727003097534, -0.05116896331310272, 1.5778579711914062, 0.36143073439598083, 0.02215484157204628, -0.17690257728099823, -0.17011559009552002, -1.0742621421813965, 0.21718065440654755]], "dtype": "float32"}, {"__nd__": [-1.017633318901062, 16.139389038085938, -5.213844299316406, 0.6895219683647156, -9.882615089416504, -10.11305046081543, -4.538920879364014, -5.092029571533203, 7.310101509094238, 7.126828670501709, -2.564944267272949, 10.489529609680176, 6.73978328704834, 6.323063373565674, -7.675174713134766, -11.560545921325684, -3.042484760284424, 4.087190628051758, -1.970550298690796, 3.957828998565674, 8.221128463745117, 1.3524519205093384, 1.1450822353363037, -4.588761806488037, 7.607890605926514, 9.6243896484375, 1.0958361625671387, -13.64117431640625, 3.361182689666748, 3.9091053009033203, 11.466741561889648, -2.7373905181884766, 3.521488904953003, -1.7986466884613037, 11.321757316589355, 13.771201133728027, 3.255556344985962, -4.80408239364624, 4.741897106170654, -1.2233355045318604, 5.925196170806885, 0.4826645255088806, 1.7522333860397339, 8.696399688720703, -24.103004455566406, 0.6483152508735657, -19.143930435180664, 4.508553981781006, -1.837957739830017, 15.39115047454834, -11.672394752502441, 1.4233571290969849, 0.7185178995132446, -6.545590400695801, -16.534988403320312, -2.5192911624908447, 7.581478118896484, 14.0050630569458, -7.125672817230225, 4.985797882080078, 1.242724895477295, -2.8345956802368164, -2.841315984725952, 3.271568536758423, 4.349607467651367, -7.777151107788086, -5.16154146194458, 14.689675331115723, -14.897607803344727, 4.56957483291626, -1.6649082899093628, 0.5317465662956238, -11.762121200561523, 17.56475257873535, 5.6347198486328125, -1.14534592628479, -7.974218368530273, 10.17467212677002, -15.314709663391113, 2.4899938106536865, -12.489124298095703, -5.434810638427734, 9.903069496154785, -3.351816415786743, 3.8538005352020264, 5.320490837097168, 8.951800346374512, -7.486342430114746, -1.0168567895889282, 12.741317749023438, 0.8321151733398438, -2.3710124492645264, 17.820951461791992, 0.981171190738678, 2.4713187217712402, 17.599117279052734, 3.5825538635253906, -7.053457260131836, 8.930424690246582, -5.419361114501953, -12.239870071411133, 14.559043884277344, 11.726775169372559, -7.22300910949707, -11.572019577026367, -11.94239330291748, 12.45681381225586, -5.429043292999268, -15.410961151123047, 1.736181378364563, -7.541248798370361, -14.210844039916992, 11.674409866333008, 2.6390678882598877, -2.5774059295654297, -3.660966634750366, 3.051417350769043, 1.9536699056625366, 10.380590438842773, 1.207364797592163, -1.6506603956222534, 0.10266512632369995, 4.220750331878662, 6.336920261383057, -12.847222328186035, -6.092015743255615, 5.367743015289307, 9.29470443725586, 5.102808475494385], "dtype": "float32"}, {"__nd__": [10.263789176940918, -8.940621376037598, -0.5290946364402771, 14.508613586425781, 3.4206416606903076, 7.996506214141846, -5.62132453918457, -13.129281044006348, 15.942468643188477, 0.1628860980272293, 12.327191352844238, 3.5020077228546143, -6.794544219970703, -9.100462913513184, -10.889677047729492, 2.850569248199463, 1.5983651876449585, -14.231993675231934, 19.28929328918457, 8.953720092773438, 4.861202239990234, -6.74678373336792, -0.03485330939292908, -1.596327304840088, 2.261936664581299, -6.761653900146484, 2.3791441917419434, -7.920958995819092, 3.7381975650787354, 3.5468785762786865, -16.09535789489746, -2.912152051925659, -11.3673677444458, 4.193623065948486, 3.2088446617126465, 2.1593291759490967, -12.20125961303711, -23.96593475341797, 3.9929747581481934, -0.0718173086643219, -7.834157466888428, -10.449226379394531, -6.385990619659424, -0.4185856282711029, 1.676594614982605, -5.980463027954102, 9.474320411682129, -1.3074971437454224, 5.1100287437438965, 3.598734140396118, 3.857792854309082, 0.21275441348552704, 14.860081672668457, -7.041597843170166, -6.780176639556885, 5.002188682556152, -0.07488897442817688, -0.2660124599933624, 5.953913688659668, -18.198274612426758, 8.492718696594238, -4.214625835418701, 16.875940322875977, 2.885078191757202, -7.642999649047852, -7.930155277252197, 0.2305658757686615, 7.688595771789551, 5.732313632965088, -24.067785263061523, -16.70383644104004, 4.566401958465576, 4.710299968719482, 16.347705841064453, -5.02069091796875, -14.498189926147461, -6.604515552520752, 14.109125137329102, 12.348801612854004, -0.47876280546188354, 22.17209815979004, 6.348535060882568, 23.701093673706055, -1.6101757287979126, -16.577363967895508, 3.654048442840576, 12.787849426269531, 10.149518013000488, -4.363189697265625, 4.597609043121338, 12.8551025390625, 6.053181171417236, 10.958471298217773, -5.850303649902344, 4.722558498382568, -0.7693731188774109, -1.5577912330627441, -14.2835693359375, 1.073765754699707, 1.300105333328247, -13.703499794006348, -1.7145377397537231, -16.069408416748047, -3.234253406524658, 2.9677157402038574, 2.2080111503601074, -10.893784523010254, -12.425426483154297, -20.728240966796875, 22.33289909362793, 16.044696807861328, -3.2357828617095947, 0.47618770599365234, 2.9817991256713867, -16.16631507873535, -1.62875235080719, -4.498338222503662, -0.15253755450248718, -10.471731185913086, -0.25577566027641296, 3.4568238258361816, -7.952585220336914, 0.18090024590492249, 13.548715591430664, -9.691238403320312, 1.861559271812439, 0.7515601515769958, -16.3809814453125, -6.985550880432129], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(layer_norm_bwd, json.loads(_TESTS), 'triton', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")